# Project 3 — notebook gộp: `du-lieu-do-thi` + pipeline ảnh/video

Notebook này gộp hai pipeline của Project 3 (Truy vấn đa phương thức có giải thích bằng đồ thị) để chạy một lần trên Kaggle.

| Phần | Nguồn | Nội dung | Thư mục kết quả |
|---|---|---|---|
| **A** | `hieutrinh123/du-lieu-do-thi` (version 9/11) | Flickr8k (text → ảnh) và Action Genome (text → video): CLIP → đồ thị cảnh (node đối tượng, quan hệ, hành động/bối cảnh Charades) → GNN + đường đi + nhánh khớp truy vấn↔node → xếp hạng lại; giải thích, đối chứng, kiểm định cặp. Phần 4 (MSVD → MSR-VTT) cũ vẫn tắt (`%%script true`). | `/kaggle/working/results/`, `/kaggle/working/output/action_genome/results/` |
| **C** | `lamvo30ai/project03-level4b-v2-graphfix` (version 1), mục 4u trong phần B | Text → video MSVD + MSR-VTT: đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học, ConceptNet, xAI; tạo cache đặc trưng frame cho mục 4v. Chạy trong tiến trình riêng (papermill). | `/kaggle/working/project03_level4b_v2/`, `/kaggle/working/p3_out/video_v2/` |
| **B** | `lamvo30ai/project3-full-pipeline` (version 8) | Visual Genome + COCO (text → ảnh, scene graph GT; GraphSAGE; SCENIR), ARO + SugarCrepe (ảnh → caption, RelTR), COCO-spatial (cạnh hình học), XAI, video MSVD (Level4B v5). | `/kaggle/working/p3_out/` |

**Kaggle:** GPU T4, Internet On. Datasets: `hieutrinh123/action-genome`, `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`. Notebook nguồn: `hieutrinh123/du-lieu-do-thi` (cache CLIP), `lamvo30ai/project03-level4b-v2-graphfix` (cache video của phần B — nếu không có quyền truy cập, nhánh video của phần B ghi lỗi và phần còn lại vẫn chạy).

**Thời gian ước tính:** phần A khoảng 4–5 giờ (ViT-L/14), phần B khoảng 3 giờ ⇒ 7–8 giờ. Phần C: gần như 0 nếu output `lamvo30ai/project03-level4b-v2-graphfix` đã được Add Input (chế độ `auto` chỉ chép kết quả), ngược lại +2,4 giờ (tổng ~10 giờ, sát giới hạn 12 giờ của Kaggle).

**Thay đổi khi gộp:** ngoài việc nhúng phần C (mục 4u), chỉ hai chỗ — (1) phần A đặt backbone ViT-L/14 như mô hình cuối (version 9); (2) phần B đổi tên hàm `log` thành `p3_log` để không đè hàm `log` của phần A. Ngoài ra thêm một cell giải phóng bộ nhớ GPU giữa hai phần. Không đổi logic, tham số hay dữ liệu của phần nào.


---
# PHẦN A — `du-lieu-do-thi`: Flickr8k + Action Genome


# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị

**Pipeline:** `truy vấn → CLIP → top-K ứng viên → điểm đồ thị (GNN + đường đi) → xếp hạng lại → giải thích → fidelity`

Điểm số cuối của một ứng viên:

$$\text{score} = \alpha \cdot \cos_{\text{CLIP}} + (1-\alpha)\big[\lambda \cdot s_{\text{GNN}} + (1-\lambda)\cdot s_{\text{path}}\big]$$

* `s_GNN` — GraphSAGE/GAT tự cài trên đồ thị dị thể (ảnh / lớp đối tượng / khái niệm).
  Mục 1.7.2 thay nhánh này bằng **SCENIR** (ICML 2025): Graph Autoencoder tự giám sát trên scene graph
  của từng ảnh — phương pháp mới cho cấp độ 2 (`CFG.gnn_type = "scenir"`).
* `s_path` — tổng **cộng tính** theo từng đường đi nối khái niệm truy vấn tới node ảnh.
  Vì cộng tính nên bỏ một đường đi làm điểm giảm đúng bằng phần đóng góp của nó ⇒ giải thích
  đến thẳng từ cơ chế xếp hạng, không phải hợp lý hoá sau khi có kết quả.
* `α`, `λ` chọn trên **validation**, không bao giờ trên test.

**Chạy trên Kaggle:** Settings → Accelerator → GPU P100 (hoặc T4 x2), Internet → On.
Notebook tự dò dữ liệu trong `/kaggle/input`; không thấy thì chạy chế độ `SYNTHETIC` để smoke-test toàn bộ mã.

## Cấu trúc notebook

| Phần | Nội dung | Vị trí |
|---|---|---|
| **1** | Nền tảng + Flickr8k: cấu hình, đồ thị, GNN cổ điển, huấn luyện, giải thích, fidelity, ablation | mục 1.1 – 1.17 |
| **2** | GNN của SCENIR thay GNN cổ điển (Flickr8k) | Phần 2 |
| **3** | Action Genome (dataset thứ hai): liên kết khái niệm, ablation, SCENIR | mục 3.1 – 3.5 |
| **4** | Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT) | mục 4.1 – 4.16 |


> **Bản v5 (chạy lại Phần 1–3).** Thay đổi so với bản trước:
> 1. Node **bộ ba cấp ảnh** + mẫu đường đi `P6` (mục 1.2b); nhãn đối tượng Flickr8k bằng linear probe ngoài-fold.
> 2. Đối chứng `no_triples`, `shuffled_triples`; **bộ đánh giá quan hệ khó** (cùng đối tượng, khác quan hệ) cho Flickr8k và AG.
> 3. Sửa đối chứng fidelity: giải thích luôn nhỏ hơn toàn bộ đường đi (bản trước in `sparsity 0.00` cho các ca ≤ 5 đường đi).
> 4. Dùng dataset Kaggle thay Drive; cache CLIP của lần chạy trước được dùng lại nếu output cũ được Add Input.
> 5. **Phần 4 (MSVD → MSR-VTT) không chạy lại** (các cell có `%%script true`); kết quả của nó nằm ở version trước.
>
> **Bản v8 (version 6).** (a) AG: đồ thị không–thời gian — thêm node hành động, bối cảnh và bộ ba thời gian từ nhãn Charades (cell v8 đầu Phần 3), đối chứng `no_actions` / `shuffled_actions`, cấu hình `no_class_edges`. (b) Flickr8k: chế độ `oracle_grams` — gallery ĐÃ ĐƯỢC CHÚ THÍCH (caption 1–4 của ảnh làm nguồn node, cùng điều kiện với annotation của AG), node = từ + cụm 2 từ + bộ ba. Kết quả đồ thị BLIP ở version 5.
>
> **Bản v7 (version 5).** Flickr8k: đồ thị cảnh dựng từ mô hình mô tả ảnh ĐỘC LẬP với CLIP (BLIP), xem mục 1.2c; thêm kiểm định cặp CLIP+Graph vs CLIP, đối chứng `shuffled_vlm`, và backbone ViT-B/32 trên cùng đồ thị. Phần 2–4 không chạy lại (AG: version 4).
>
> **Bản v6 (version 3).** (a) Flickr8k chạy với backbone YẾU hơn — CLIP **ViT-B/32** — để đo hiệu quả của đồ thị theo độ mạnh của backbone (kết quả ViT-L/14 nằm ở version 2). (b) AG thêm 2 đối chứng: `composite_swap` (gán nhầm quan hệ, giữ nguyên vật) và **truy vấn phản thực tế** (đổi vị từ trong truy vấn). (c) Không chạy lại oracle, SCENIR, liên kết ngữ nghĩa AG (đã có ở version 2) để tiết kiệm GPU.


---
# PHẦN 1 — Nền tảng và Flickr8k

_Mục 1.1 – 1.17: toàn bộ mã dùng chung (đồ thị, mô hình, truy vấn, đánh giá, giải thích) rồi chạy trên Flickr8k._


In [ ]:
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
print("Kaggle environment:", RUN_ON_KAGGLE)

In [ ]:
# Cài đặt phụ thuộc. Trên Kaggle torch/numpy đã có sẵn; chỉ cần open_clip + nltk.
def pip_install(pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

if RUN_ON_KAGGLE:
    pip_install(["open_clip_torch", "nltk"])   # không ghim phiên bản: Kaggle đổi torch thường xuyên

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1.1 Cấu hình

Mọi siêu tham số nằm ở một chỗ. `CFG.n_images` là núm vặn chính cho thời gian chạy:
4000 ảnh chạy khoảng 12-15 phút trên P100 (gồm cả trích đặc trưng CLIP).

In [ ]:
class CFG:
    # --- dữ liệu ---
    dataset      = "flickr8k"
    n_images     = 8000
    min_obj_freq = 5           # lớp đối tượng hiếm hơn ngưỡng này bị loại — ĐẾM TRÊN TRAIN
    max_concepts = 4000

    # --- nhãn ảnh Flickr8k ---
    # "tagger": mọi ảnh gắn nhãn bằng CLIP zero-shot trên từ vựng rút từ caption TRAIN
    #           (không ảnh val/test nào được dùng caption của chính nó).
    # "oracle": nhãn rút từ caption 1..4 của chính ảnh (cách cũ) — chỉ để so sánh, KHÔNG báo cáo là kết quả chính.
    flickr_annot = "oracle_grams"   # v8: gallery đã chú thích (caption 1–4); "vlm" = v7 (BLIP), "probe" = v5/v6
    _flickr_annot_v7 = "vlm"       # v7: node từ mô hình mô tả ảnh độc lập với CLIP (1.2c); "probe" = v5/v6
    _flickr_annot_v5 = "probe"     # v5: linear probe khởi tạo từ văn bản CLIP, ngoài-fold (1.2b); "tagger" = zero-shot cũ
    tag_k_grid   = [3, 5, 8, 10, 12]          # số nhãn/ảnh — chọn trên VAL theo F1
    tag_prompts  = ["a photo of a {}.", "a photo of {}.", "a photo containing a {}."]

    # --- chia dữ liệu (chia theo ẢNH, không theo caption) ---
    split_seed   = 20240101
    val_ratio    = 0.15
    test_ratio   = 0.20

    # --- CLIP ---
    clip_model   = "ViT-L-14"                 # trước: ViT-B-32. Tagger + baseline mạnh hơn rõ rệt
    clip_weights = "laion2b_s32b_b82k"
    clip_bs      = 128                        # ViT-L tốn bộ nhớ hơn; L4 24GB chạy 128 ổn

    # --- đồ thị ---
    max_path_len      = 3
    scene_min_count   = 2      # cạnh scene phải xuất hiện ≥ ngưỡng này trong caption TRAIN (trước đây = 1)
    scene_top_k       = 10     # mỗi lớp đối tượng giữ tối đa k cạnh scene mạnh nhất
    cooc_min_count    = 10
    cooc_top_k        = 10
    use_wordnet       = True
    isa_min_depth     = 4      # bỏ khái niệm quá chung (entity, object, whole, ...)
    use_idf           = True   # trọng số đường đi = IDF của node đích (tính trên TRAIN)
    drop_redundant_indirect = True   # bỏ đường đi gián tiếp tới khái niệm ảnh đã chứa trực tiếp

    # --- v5: quan hệ CẤP ẢNH (node bộ ba, mẫu P6) — xem mục 1.2b ---
    use_triples      = True
    triple_min_count = 5                      # bộ ba phải xuất hiện ≥ ngưỡng này trong caption TRAIN
    triple_k_grid    = [1, 2, 3, 5, 8]        # số bộ ba/ảnh — chọn trên VAL theo F1
    triple_tau_grid  = [0.0, 0.05, 0.1, 0.2, 0.3]
    probe_folds      = 5                      # dự đoán ngoài-fold cho ảnh train
    probe_steps      = 300
    probe_lr         = 1e-3
    probe_wd_init    = 1e-3                   # phạt ‖W − text‖²
    rel_neg_min      = 3                      # bộ quan hệ khó: tối thiểu / tối đa số âm mỗi truy vấn
    rel_neg_max      = 30

    # --- sửa ablation âm ---
    # (1) IDF: "taxonomy" = đối tượng đếm df qua P0, khái niệm qua P2 (isa thật, trước khi xáo).
    #     "all" = cách cũ: đếm qua MỌI mẫu ⇒ thêm quan hệ làm df phình, IDF của khớp trực tiếp
    #     tụt về ≈0 — đây là lý do chính khiến 'none' thua 'no_relations'.
    idf_from        = "taxonomy"
    # (2) Cạnh related: xếp hạng theo NPMI thay vì đếm thô (đếm thô ⇒ mọi lớp nối vào các hub
    #     phổ biến như person/man/floor). Trọng số cạnh = P(đích | nguồn) trên TRAIN.
    cooc_score      = "npmi"         # "npmi" | "count" (cũ)
    cooc_min_npmi   = 0.10
    edge_weighting  = True           # trọng số đường đi = idf(c) × Π P(đích|nguồn); False = đếm (cũ)
    # (3) GNN tách theo loại cạnh + cổng học được, khởi tạo gần như đóng ⇒ mô hình bắt đầu
    #     từ 'no_relations' và chỉ mở quan hệ khi nó thực sự giảm loss.
    gnn_typed       = True
    gate_init       = -1.5           # sigmoid(-1.5) ≈ 0.18 cho cạnh scene/related/isa; contains mở sẵn
    path_init_indirect = -1.0        # path_w khởi tạo cho P1..P5 (softplus(-1) ≈ 0.31; P0 = 0.69)
    scalar_lr_mult  = 10.0           # lr của cổng + path_w = lr × hệ số này
    # (4) Huấn luyện giống lúc suy luận: mẫu âm khó lấy từ CLIP top-K trên gallery TRAIN,
    #     và có z(s_clip) trong logit ⇒ đồ thị học phần CLIP còn thiếu, không học lại CLIP.
    hard_neg_k      = 31             # 0 = tắt (chỉ dùng âm trong batch như cũ)
    train_with_clip = True
    # Hệ số của z(s_clip) trong logit HUẤN LUYỆN có học không. AG (CLIP yếu, MRR 0.12): học được
    # thì tốt hơn (clip_w → 0.07). Flickr (CLIP mạnh): học làm clip_w → 0.4 và MRR giảm.
    # CHỌN THEO VAL MRR cho từng bộ dữ liệu, không theo test.
    learn_clip_w    = True

    # --- GNN ---
    gnn_type     = "sage"
    gnn_dim      = 256
    gnn_layers   = 2
    gnn_dropout  = 0.1
    lr           = 1e-3
    weight_decay = 1e-4
    epochs       = 20
    train_bs     = 128
    temperature  = 0.07

    # --- truy vấn / xếp hạng ---
    rerank_k     = 100

    # --- thí nghiệm ---
    seeds        = [0, 1, 2]
    alpha_grid   = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid  = [0.0, 0.25, 0.5, 0.75]     # bỏ 1.0: λ=1 ⇒ s_path có trọng số 0 ⇒ giải thích không trung thực với điểm cuối
    ablations    = ("none", "random_edges", "no_relations", "no_scene", "no_related", "no_concept")

    adaptive_fusion = True
    fusion_steps    = 300
    fusion_tau      = 0.2
    fusion_wd       = 1e-2

    # --- xAI ---
    n_explain_queries = 20
    top_m_paths       = 5

    # --- đường dẫn ---
    out_dir = Path("/kaggle/working") if RUN_ON_KAGGLE else Path("./work")

# Flickr8k chạy thêm 2 đối chứng của node bộ ba; AG không có bộ ba dự đoán nên giữ CFG.ablations.
FLICKR_ABLATIONS = ("none", "no_class_edges", "no_relations")   # v6: rút gọn

CFG.out_dir.mkdir(parents=True, exist_ok=True)
(CFG.out_dir / "cache").mkdir(exist_ok=True)
(CFG.out_dir / "results").mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1.2 Nạp dữ liệu

Notebook dò tự động các tệp Visual Genome trong `/kaggle/input` theo tên
(`objects.json`, `relationships.json`, `image_data.json`, `region_descriptions.json`).
Không tìm thấy ⇒ tự sinh dữ liệu tổng hợp để toàn bộ mã vẫn chạy hết (smoke-test).

**Cấu trúc chuẩn hoá** dùng chung cho mọi nguồn:
```python
images = [{"image_id": int, "caption": str, "objects": [str], "relations": [(subj, pred, obj)]}]
```

In [ ]:
# Lớp khái niệm dự phòng: dùng khi WordNet/nltk không tải được (Kaggle tắt Internet).
# Đây là một MMKG tí hon viết tay, đủ để tạo đường đi gián tiếp cho smoke-test và
# vẫn hoạt động trên Visual Genome vì các lớp đối tượng phổ biến đều có mặt.
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
}


def find_vg_files():
    """Dò các tệp Visual Genome trong /kaggle/input theo tên tệp."""
    wanted = ["objects.json", "relationships.json", "image_data.json",
              "region_descriptions.json", "attributes.json"]
    found = {}
    if not Path("/kaggle/input").exists():
        return found
    for name in wanted:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            found[name.replace(".json", "")] = sorted(hits, key=lambda p: -os.path.getsize(p))[0]
    return found


def _norm(s):
    """Chuẩn hoá chuỗi nhãn: chữ thường, bỏ ký tự lạ, gộp khoảng trắng."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_visual_genome(paths, n_images):
    """Nạp VG. Trả về danh sách dict đã chuẩn hoá."""
    print("Nạp Visual Genome từ:", {k: v for k, v in paths.items()})
    with open(paths["objects"]) as f:
        objects = json.load(f)
    obj_by_img, objid2name = {}, {}
    for rec in objects:
        iid = rec.get("image_id") or rec.get("id")
        names = []
        for o in rec.get("objects", []):
            nm = _norm((o.get("names") or [o.get("name", "")])[0])
            if nm:
                names.append(nm)
                objid2name[o.get("object_id")] = nm
        if names:
            obj_by_img[iid] = names
    del objects

    rel_by_img = defaultdict(list)
    if "relationships" in paths:
        with open(paths["relationships"]) as f:
            rels = json.load(f)
        for rec in rels:
            iid = rec.get("image_id") or rec.get("id")
            for r in rec.get("relationships", []):
                pred = _norm(r.get("predicate", ""))
                s = r.get("subject", {}); o = r.get("object", {})
                sn = _norm((s.get("names") or [s.get("name", "")])[0]) or objid2name.get(s.get("object_id"), "")
                on = _norm((o.get("names") or [o.get("name", "")])[0]) or objid2name.get(o.get("object_id"), "")
                if pred and sn and on:
                    rel_by_img[iid].append((sn, pred, on))
        del rels

    cap_by_img = {}
    if "region_descriptions" in paths:
        with open(paths["region_descriptions"]) as f:
            regions = json.load(f)
        for rec in regions:
            iid = rec.get("id") or rec.get("image_id")
            phrases = [r.get("phrase", "").strip() for r in rec.get("regions", [])]
            phrases = [p for p in phrases if 15 <= len(p) <= 160]
            if phrases:
                # ghép 3 vùng dài nhất thành một caption cấp ảnh
                phrases = sorted(phrases, key=len, reverse=True)[:3]
                cap_by_img[iid] = ". ".join(phrases)
        del regions

    url_by_img = {}
    if "image_data" in paths:
        with open(paths["image_data"]) as f:
            meta = json.load(f)
        for m in meta:
            url_by_img[m.get("image_id") or m.get("id")] = m.get("url", "")
        del meta

    ids = [i for i in obj_by_img if i in cap_by_img] or list(obj_by_img)
    ids = sorted(ids)[:n_images]
    out = []
    for iid in ids:
        out.append(dict(image_id=int(iid),
                        caption=cap_by_img.get(iid, " ".join(obj_by_img[iid][:8])),
                        objects=sorted(set(obj_by_img[iid])),
                        relations=list({tuple(r) for r in rel_by_img.get(iid, [])}),
                        url=url_by_img.get(iid, "")))
    print(f"  → {len(out)} ảnh, "
          f"{np.mean([len(r['objects']) for r in out]):.1f} đối tượng/ảnh, "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/ảnh")
    return out


def make_synthetic(n_images, seed=0):
    """Dữ liệu tổng hợp có cấu trúc quan hệ thật, dùng để smoke-test toàn bộ pipeline.

    Thiết kế có chủ ý: mỗi ảnh thuộc một 'chủ đề', chủ đề quyết định đối tượng và
    quan hệ, nên đồ thị MANG thông tin — nếu pipeline đúng thì điểm đồ thị phải
    cải thiện được kết quả so với đặc trưng nhiễu.
    """
    rng = random.Random(seed)
    themes = {
        "football":  (["ball", "field", "child", "goal", "grass", "shoe"],
                      [("child", "plays", "ball"), ("ball", "on", "grass"), ("child", "wears", "shoe")]),
        "kitchen":   (["pot", "stove", "woman", "plate", "spoon", "table"],
                      [("woman", "cooks with", "pot"), ("pot", "on", "stove"), ("plate", "on", "table")]),
        "beach":     (["sand", "umbrella", "wave", "man", "towel", "boat"],
                      [("man", "sits on", "towel"), ("umbrella", "on", "sand"), ("boat", "in", "wave")]),
        "street":    (["car", "sign", "road", "building", "light", "person"],
                      [("car", "on", "road"), ("sign", "near", "light"), ("person", "crosses", "road")]),
        "zoo":       (["elephant", "fence", "tree", "keeper", "water", "rock"],
                      [("elephant", "near", "fence"), ("tree", "behind", "fence"), ("keeper", "feeds", "elephant")]),
        "bedroom":   (["bed", "lamp", "pillow", "window", "curtain", "book"],
                      [("lamp", "next to", "bed"), ("pillow", "on", "bed"), ("curtain", "on", "window")]),
    }
    names = list(themes)
    out = []
    for i in range(n_images):
        th = names[i % len(names)]
        objs, rels = themes[th]
        k = rng.randint(3, len(objs))
        chosen = rng.sample(objs, k)
        r = [t for t in rels if t[0] in chosen and t[2] in chosen]
        # nhiễu: thỉnh thoảng mượn một đối tượng của chủ đề khác
        if rng.random() < 0.3:
            other = themes[rng.choice(names)][0]
            chosen.append(rng.choice(other))
        # 40% truy vấn nhắc tới KHÁI NIỆM (không có trong danh sách đối tượng) thay vì
        # gọi thẳng tên đối tượng ⇒ tạo ra nhóm truy vấn gián tiếp cần đường đi P2/P4/P5.
        if rng.random() < 0.4:
            concepts = sorted({c for o in chosen for c in FALLBACK_ISA.get(o, [])}
                              - set(chosen))
            if concepts:
                cap = (f"a photo showing {' and '.join(rng.sample(concepts, min(2, len(concepts))))}"
                       f" in a {th} scene")
            else:
                cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        else:
            cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        out.append(dict(image_id=i, caption=cap, objects=sorted(set(chosen)),
                        relations=r, url="", theme=th))
    return out


def load_dataset():
    mode = CFG.dataset
    if mode in ("auto", "vg"):
        paths = find_vg_files()
        if "objects" in paths:
            return load_visual_genome(paths, CFG.n_images), "visual_genome"
        if mode == "vg":
            raise FileNotFoundError("Không tìm thấy objects.json của Visual Genome trong /kaggle/input")
    print("⚠ Không tìm thấy dữ liệu công khai → chạy chế độ SYNTHETIC (smoke-test).")
    return make_synthetic(CFG.n_images), "synthetic"

## 1.3 Chia dữ liệu và phòng rò rỉ

Yêu cầu mục 6 của đề, cài đặt cụ thể:

| Quy định | Cách thực hiện trong mã |
|---|---|
| Test không dùng để huấn luyện / chọn siêu tham số | Chia **theo ảnh**; caption của test không vào tập huấn luyện |
| Cạnh suy ra từ thống kê phải xây từ train | `build_graph(..., stat_image_ids=train_ids)`; cạnh đồng xuất hiện chỉ đếm trên train |
| α và siêu tham số chọn trên validation | `tune_on_val()` quét lưới trên val, test chỉ chạy một lần |
| Nộp split cố định hoặc seed + mã tạo split | `split.json` được ghi ra kèm seed và hash |

Node ảnh của test **vẫn nằm trong đồ thị** (chúng là ứng viên cần xếp hạng), nhưng
cạnh của chúng chỉ đến từ annotation của chính bức ảnh, không từ thống kê toàn cục,
và không có gradient nào chảy qua chúng khi huấn luyện.

In [ ]:
def make_split(records, seed=CFG.split_seed):
    ids = sorted(r["image_id"] for r in records)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_test = int(n * CFG.test_ratio)
    n_val  = int(n * CFG.val_ratio)
    split = dict(test=sorted(ids[:n_test]),
                 val=sorted(ids[n_test:n_test + n_val]),
                 train=sorted(ids[n_test + n_val:]),
                 seed=seed)
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split (seed={seed}, sha1={split['sha1']}): "
          f"train={len(split['train'])} val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])), "train/test chồng lấn!"
    assert not (set(split["val"]) & set(split["test"])), "val/test chồng lấn!"
    return split

## 1.4 Đặc trưng CLIP (đóng băng)

CLIP được **đóng băng hoàn toàn** và chỉ chạy một lần: đặc trưng lưu ra `.npy` rồi tái sử dụng
cho mọi thí nghiệm. Đây là baseline của Nhiệm vụ 1.

Thứ tự ưu tiên tìm ảnh: tệp ảnh trong `/kaggle/input` → tải theo URL → đặc trưng tổng hợp.

In [ ]:
def index_local_images():
    """Lập chỉ mục mọi tệp ảnh trong /kaggle/input theo tên tệp (không đuôi)."""
    idx = {}
    roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    for root in roots:
        for p in root.rglob("*"):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                idx.setdefault(p.stem, str(p))
    return idx


def encode_with_clip(records, tag):
    """Trả về (img_feat, txt_feat) đã chuẩn hoá L2, shape [N, D]."""
    cache = CFG.out_dir / "cache" / f"clip_{tag}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache)
        print(f"Dùng lại đặc trưng CLIP từ cache: {cache.name}")
        return z["img"], z["txt"]

    try:
        import open_clip
        from PIL import Image
    except ImportError:
        open_clip = None

    img_index = index_local_images()
    have_images = sum(1 for r in records if str(r["image_id"]) in img_index)
    print(f"Ảnh tìm thấy cục bộ: {have_images}/{len(records)}")

    if open_clip is None or have_images < 0.5 * len(records):
        print("⚠ Không đủ ảnh thật hoặc thiếu open_clip → sinh đặc trưng tổng hợp "
              "(vẫn chạy hết pipeline, nhưng con số không dùng để báo cáo).")
        return synthetic_features(records)

    model, _, preprocess = open_clip.create_model_and_transforms(
        CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters():          # đóng băng tuyệt đối
        p.requires_grad_(False)

    # --- ảnh ---
    feats = []
    from torch.utils.data import Dataset, DataLoader

    class ImgDS(Dataset):
        def __len__(self): return len(records)
        def __getitem__(self, i):
            p = img_index.get(str(records[i]["image_id"]))
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    dl = DataLoader(ImgDS(), batch_size=CFG.clip_bs, num_workers=2)
    with torch.no_grad():
        for batch in dl:
            f = model.encode_image(batch.to(DEVICE))
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    img_feat = np.concatenate(feats)

    # --- văn bản ---
    feats = []
    caps = [r["caption"] for r in records]
    with torch.no_grad():
        for i in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[i:i + CFG.clip_bs]).to(DEVICE)
            f = model.encode_text(tok)
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    txt_feat = np.concatenate(feats)

    del model
    torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img_feat, txt=txt_feat)
    return img_feat, txt_feat


def synthetic_features(records, dim=512, seed=0):
    """Đặc trưng giả có tín hiệu thật: ảnh cùng chủ đề/đối tượng thì gần nhau."""
    rng = np.random.RandomState(seed)
    vocab = sorted({o for r in records for o in r["objects"]})
    basis = rng.randn(len(vocab), dim).astype(np.float32)
    v2i = {v: i for i, v in enumerate(vocab)}
    img, txt = [], []
    for r in records:
        idx = [v2i[o] for o in r["objects"]]
        core = basis[idx].mean(0) if idx else rng.randn(dim).astype(np.float32)
        img.append(core + 0.9 * rng.randn(dim).astype(np.float32))
        txt.append(core + 0.9 * rng.randn(dim).astype(np.float32))
    f = lambda x: (lambda a: a / np.linalg.norm(a, axis=1, keepdims=True))(np.stack(x))
    return f(img), f(txt)

## 1.5 Xây đồ thị

**Đỉnh:** ảnh · lớp đối tượng · khái niệm.
Đối tượng là **lớp** (một node `dog` dùng chung mọi ảnh) chứ không phải từng thể hiện —
nhờ vậy đồ thị nối được các ảnh với nhau và message passing mới có tác dụng.

**Cạnh:**

| Loại | Nguồn | Có phải thống kê? |
|---|---|---|
| `contains` ảnh → lớp đối tượng | annotation của chính ảnh đó | không → an toàn cho test |
| `scene` đối tượng → đối tượng | scene graph, gộp trên **train** | có → chỉ train |
| `related` đối tượng → đối tượng | đồng xuất hiện trên **train** | có → chỉ train |
| `isa` đối tượng → khái niệm | WordNet hypernym | không (tri thức ngoài) |

**Mẫu đường đi** dùng cho `s_path` (từ node ảnh tới node khái niệm của truy vấn):

| Mẫu | Đường đi | Dài | Loại |
|---|---|---|---|
| P0 | `img –contains→ c` | 1 | trực tiếp |
| P1 | `img –contains→ o –scene→ c` | 2 | gián tiếp |
| P2 | `img –contains→ o –isa→ c` | 2 | gián tiếp |
| P3 | `img –contains→ o –related→ c` | 2 | gián tiếp |
| P4 | `img –contains→ o –scene→ o₂ –isa→ c` | 3 | gián tiếp |
| P5 | `img –contains→ o –related→ o₂ –isa→ c` | 3 | gián tiếp |

`child plays football` với ảnh chỉ nhận diện được `ball`, `field` được bắt qua P1/P2/P4.

In [ ]:
import scipy.sparse as sp

PATTERNS = ["P0_contains", "P1_scene", "P2_isa", "P3_related", "P4_scene_isa", "P5_related_isa",
            "P6_triple"]                      # v5: ảnh –thể_hiện→ ⟨s p o⟩ (quan hệ CẤP ẢNH)
PATTERN_LEN = [1, 2, 2, 2, 3, 3, 1]
PIDX = {p: i for i, p in enumerate(PATTERNS)}
PATTERN_DESC = {
    "P0_contains":  "ảnh –chứa→ {c}",
    "P1_scene":     "ảnh –chứa→ {o} –{rel}→ {c}",
    "P2_isa":       "ảnh –chứa→ {o} –là_một→ {c}",
    "P3_related":   "ảnh –chứa→ {o} –liên_quan→ {c}",
    "P4_scene_isa": "ảnh –chứa→ {o} –{rel}→ {o2} –là_một→ {c}",
    "P5_related_isa": "ảnh –chứa→ {o} –liên_quan→ {o2} –là_một→ {c}",
    "P6_triple":    "ảnh –thể_hiện→ ⟨{c}⟩",
}

# ---------------------------------------------------------------------------------
# Lọc WordNet: chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC.
#   - nghĩa danh từ phổ biến nhất thuộc VISUAL_LEX (hoặc là môn thể thao/trò chơi),
#   - và từ đó được dùng như danh từ ít nhất ngang như động từ (loại stand, shake, jump, walk...).
# Loại được các node rác ở bản trước: pink, blue, beige, snowy, walking, shake, stand, bit...
# ---------------------------------------------------------------------------------
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
# khái niệm "cấp cơ bản" — luôn thêm nếu nằm trên đường hypernym (dog→animal, car→vehicle...)
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment"}

_WN = None


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            try:
                import nltk
                nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
                from nltk.corpus import wordnet as wn
                wn.synsets("dog"); _WN = wn
            except Exception:
                _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE = {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True          # không có WordNet → không lọc được
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":       # football, soccer, tennis... vẫn giữ
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def get_wordnet_hypernyms(words):
    """IsA: cha + ông (đủ sâu, không quá chung) + mọi tổ tiên thuộc BASIC_CATEGORIES.
    Chỉ dùng nghĩa phổ biến nhất ⇒ không còn 'bicycle là_một ride', 'beige là_một brown'."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        fb = {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
        print(f"⚠ WordNet không dùng được → FALLBACK_ISA ({len(fb)}/{len(words)} lớp có khái niệm cha).")
        return fb
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in s.hypernym_paths()[:2]:
            for depth in range(len(path) - 2, -1, -1):        # path[-1] chính là w
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3                   # cha hoặc ông
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                        or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    print(f"WordNet(+fallback): {len(out)}/{len(words)} lớp đối tượng có khái niệm cha")
    return out


class MultimodalGraph:
    """Đồ thị dị thể ảnh–đối tượng–khái niệm, kèm ma trận đường đi (đã nhân IDF) theo mẫu."""

    def __init__(self, records, train_ids, ablation="none", seed=0):
        """ablation: none | no_relations | random_edges | no_scene | no_related | no_concept"""
        self.records = records
        self.ablation = ablation
        self.n_img = len(records)
        self.img_pos = {r["image_id"]: i for i, r in enumerate(records)}
        train_set = set(train_ids)
        rng = np.random.RandomState(seed)

        # ---------- từ vựng lớp đối tượng: tần suất CHỈ trên TRAIN ----------
        freq = Counter(o for r in records if r["image_id"] in train_set for o in r["objects"])
        objs = sorted([o for o, c in freq.items() if c >= CFG.min_obj_freq])
        self.obj2id = {o: self.n_img + i for i, o in enumerate(objs)}
        self.n_obj = len(objs)

        # ---------- khái niệm (WordNet) ----------
        hyper = {} if ablation in ("no_relations", "no_concept") else get_wordnet_hypernyms(objs)
        # Khái niệm trùng tên một lớp đối tượng (vd 'dog' là cha của 'terrier') được nối
        # THẲNG vào node đối tượng đó — bản trước tạo 2 node 'dog' và truy vấn 'dog' chỉ
        # khớp node khái niệm, làm mất đường đi P0.
        concepts = sorted({c for v in hyper.values() for c in v
                           if c not in self.obj2id})[:CFG.max_concepts]
        base = self.n_img + self.n_obj
        self.con2id = {c: base + i for i, c in enumerate(concepts)}
        self.n_con = len(concepts)
        # ---------- v5: node BỘ BA cấp ảnh (s, p, o) — bỏ khi no_relations / no_triples ----------
        self.tri_base = self.n_img + self.n_obj + self.n_con
        use_tri = (getattr(CFG, "use_triples", False) and ablation not in ("no_relations", "no_triples")
                   and any(r.get("img_triples") for r in records))
        tri_keys = sorted({k for r in records for k, _ in r.get("img_triples", [])
                           if k[0] in self.obj2id and k[2] in self.obj2id}) if use_tri else []
        self.tri2id = {k: self.tri_base + i for i, k in enumerate(tri_keys)}
        self.id2key = {v: k for k, v in self.tri2id.items()}
        self.n_tri = len(tri_keys)
        self.n_nodes = self.tri_base + self.n_tri

        self.id2name = {}
        for o, i in self.obj2id.items(): self.id2name[i] = o
        for c, i in self.con2id.items(): self.id2name[i] = c
        for k, i in self.tri2id.items(): self.id2name[i] = " ".join(k)

        # ---------- cạnh contains (nhãn riêng của mỗi ảnh) ----------
        rows, cols = [], []
        self.img_objects = []
        for i, r in enumerate(records):
            ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
            self.img_objects.append(ids)
            rows += [i] * len(ids); cols += ids
        A_con = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                              shape=(self.n_img, self.n_nodes))

        # ---------- cạnh scene: CHỈ train, có ngưỡng + top-k ----------
        scene = Counter(); pred_cnt = defaultdict(Counter)
        if ablation not in ("no_relations", "no_scene"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue                      # ← chốt chặn rò rỉ
                for s, p, o in r["relations"]:
                    if s in self.obj2id and o in self.obj2id and s != o:
                        key = (self.obj2id[s], self.obj2id[o])
                        scene[key] += 1; pred_cnt[key][p] += 1
            by_src = defaultdict(list)
            for (a, b), v in scene.items():
                if v >= CFG.scene_min_count:
                    by_src[a].append((v, b))
            scene = {(a, b): v for a, lst in by_src.items()
                     for v, b in sorted(lst, reverse=True)[:CFG.scene_top_k]}
        # vị từ hiển thị = vị từ PHỔ BIẾN NHẤT của cặp (bản trước lấy vị từ gặp đầu tiên)
        self.scene_pred = {k: pred_cnt[k].most_common(1)[0][0] for k in scene}
        # số ảnh TRAIN chứa mỗi lớp — mẫu số cho P(đích | nguồn)
        n_tr = max(sum(1 for r in records if r["image_id"] in train_set), 1)
        obj_df = Counter(self.obj2id[o] for r in records if r["image_id"] in train_set
                         for o in set(r["objects"]) if o in self.obj2id)
        scene_w = {(a, b): min(1.0, v / max(obj_df[a], 1)) for (a, b), v in scene.items()}
        A_scene = self._mat(scene_w)

        # ---------- cạnh related: đồng xuất hiện, CHỈ train ----------
        cooc = Counter()
        if ablation not in ("no_relations", "no_related"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue
                ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
                for a, b in itertools.combinations(ids, 2):
                    cooc[(a, b)] += 1; cooc[(b, a)] += 1
            cooc = {k: v for k, v in cooc.items() if v >= CFG.cooc_min_count}
            by_src = defaultdict(list)
            for (a, b), v in cooc.items():
                if CFG.cooc_score == "npmi":
                    # NPMI ∈ [-1, 1]: 0 = độc lập. Đếm thô ưu ái lớp phổ biến (hub) ⇒ mọi lớp
                    # đều nối tới person/man/floor và P3/P5 thành nhiễu phủ khắp gallery.
                    p_ab = v / n_tr
                    pmi = math.log(p_ab / ((obj_df[a] / n_tr) * (obj_df[b] / n_tr)))
                    s = pmi / max(-math.log(p_ab), 1e-6)
                    if s < CFG.cooc_min_npmi:
                        continue
                else:
                    s = v
                by_src[a].append((s, b))
            cooc = {(a, b): cooc[(a, b)] for a, lst in by_src.items()
                    for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]}
        cooc_w = {(a, b): v / max(obj_df[a], 1) for (a, b), v in cooc.items()}   # P(b | a)
        A_rel = self._mat(cooc_w)

        # ---------- cạnh isa (đích có thể là node khái niệm HOẶC node đối tượng) ----------
        isa = {}
        for o, cs in hyper.items():
            if o not in self.obj2id:
                continue
            for c in cs:
                t = self.obj2id.get(c, self.con2id.get(c))
                if t is not None and t != self.obj2id[o]:
                    isa[(self.obj2id[o], t)] = 1.0
        A_isa = self._mat(isa)
        A_isa_true = A_isa.copy()                 # cho IDF: không phụ thuộc ablation random_edges

        # ---------- ablation: cạnh ngẫu nhiên, GIỮ NGUYÊN số cạnh ----------
        if ablation == "random_edges":
            A_scene = self._randomize(A_scene, rng)
            A_rel   = self._randomize(A_rel, rng)
            A_isa   = self._randomize(A_isa, rng)
        if ablation in ("no_relations", "no_class_edges"):     # no_class_edges (v8): bỏ cạnh CẤP LỚP, GIỮ bộ ba cấp ảnh/video
            A_scene = A_rel = A_isa = self._mat({})

        # ---------- v5: cạnh ảnh –thể_hiện→ bộ ba, trọng số = xác suất probe của CHÍNH ảnh ----------
        # shuffled_triples / random_edges: ảnh i nhận bộ ba của ảnh π(i) ⇒ đối chứng gắn-với-ảnh.
        trows, tcols, tvals = [], [], []
        if self.n_tri:
            src_img = (rng.permutation(self.n_img) if ablation in ("shuffled_triples", "random_edges")
                       else np.arange(self.n_img))
            for i in range(self.n_img):
                for k, pr in records[int(src_img[i])].get("img_triples", []):
                    t = self.tri2id.get(k)
                    if t is not None:
                        trows.append(i); tcols.append(t); tvals.append(float(pr))
        A_tri = sp.csr_matrix((np.array(tvals, np.float32), (np.array(trows, int), np.array(tcols, int))),
                              shape=(self.n_img, self.n_nodes))
        A_tri.sum_duplicates()
        self.A_triple = A_tri

        self.scene_count = dict(scene)
        self.cooc_count = dict(cooc)
        for M in (A_scene, A_rel, A_isa):
            if CFG.edge_weighting:
                M.data[:] = np.clip(M.data, 0.0, 1.0)   # P(đích|nguồn); xáo cạnh có thể cộng dồn >1
            else:
                M.data[:] = 1.0                   # R_p[i,c] = SỐ đường đi thật sự (cách cũ)
        self.A_contains, self.A_scene, self.A_related, self.A_isa = A_con, A_scene, A_rel, A_isa

        # ---------- ma trận đếm đường đi theo mẫu ----------
        R = {
            "P0_contains":    A_con.tocsr(),
            "P1_scene":       (A_con @ A_scene).tocsr(),
            "P2_isa":         (A_con @ A_isa).tocsr(),
            "P3_related":     (A_con @ A_rel).tocsr(),
            "P4_scene_isa":   (A_con @ A_scene @ A_isa).tocsr(),
            "P5_related_isa": (A_con @ A_rel @ A_isa).tocsr(),
            "P6_triple":      A_tri.tocsr(),
        }
        # Bỏ đường đi gián tiếp tới khái niệm mà ảnh ĐÃ chứa trực tiếp: chúng chỉ đếm lại
        # tín hiệu P0 (vd 'dog –pushing→ terrier –là_một→ dog' ở bản trước).
        if CFG.drop_redundant_indirect:
            direct = (A_con > 0).astype(np.float32)
            for p in PATTERNS[1:]:
                R[p] = (R[p] - R[p].multiply(direct)).tocsr()
                R[p].eliminate_zeros()

        # IDF của node đích, tính CHỈ trên ảnh TRAIN: khái niệm mà hầu như ảnh nào cũng
        # với tới (person, man...) không phân biệt được ứng viên ⇒ trọng số ≈ 0.
        tr_rows = np.array(sorted(self.img_pos[i] for i in train_set if i in self.img_pos))
        if CFG.idf_from == "taxonomy":
            # df(đối tượng) = số ảnh train CHỨA nó; df(khái niệm) = số ảnh train chứa một hạ vị
            # của nó (isa thật). Không đếm scene/related ⇒ IDF của khớp trực tiếp giống hệt
            # nhau ở mọi ablation, và quan hệ không còn "pha loãng" P0.
            df_con = np.asarray((A_con[tr_rows] > 0).sum(0)).ravel()
            df_isa = np.asarray(((A_con @ A_isa_true)[tr_rows] > 0).sum(0)).ravel()
            df_tri = np.asarray((A_tri[tr_rows] > 0).sum(0)).ravel()      # v5: df của bộ ba = số ảnh train mang nó
            ar = np.arange(self.n_nodes)
            df = np.where(ar < self.n_img + self.n_obj, df_con, np.where(ar < self.tri_base, df_isa, df_tri))
        else:
            reach = None
            for p in PATTERNS:
                m = (R[p][tr_rows] > 0).astype(np.float32)
                reach = m if reach is None else reach + m
            df = np.asarray((reach > 0).sum(0)).ravel()
        idf = np.log((len(tr_rows) + 1.0) / (df + 1.0)).astype(np.float32)
        if not CFG.use_idf:
            idf = np.ones_like(idf)
        idf[:self.n_img] = 0.0
        self.idf = idf
        D = sp.diags(idf)
        # R[p][i, c] = Σ_{đường đi i→c theo mẫu p} idf(c). Liệt kê đường đi dùng ĐÚNG
        # trọng số này nên giải thích vẫn khớp điểm (self-test kiểm tra).
        self.R = {p: (R[p] @ D).tocsr() for p in PATTERNS}
        for p in PATTERNS:
            self.R[p].eliminate_zeros()

        self.name2id = dict(self.obj2id); self.name2id.update(self.con2id)
        self.edge_index, self.edge_type = self._build_edge_index()

    def _mat(self, d):
        if not d:
            return sp.csr_matrix((self.n_nodes, self.n_nodes), dtype=np.float32)
        keys = list(d.keys())
        r = np.fromiter((k[0] for k in keys), int, len(keys))
        c = np.fromiter((k[1] for k in keys), int, len(keys))
        v = np.fromiter(d.values(), np.float32, len(keys))
        return sp.csr_matrix((v, (r, c)), shape=(self.n_nodes, self.n_nodes))

    def _randomize(self, A, rng):
        """Giữ nguyên SỐ LƯỢNG cạnh, xáo trộn đích đến — đối chứng cho ablation."""
        A = A.tocoo()
        if A.nnz == 0:
            return A.tocsr()
        new_col = rng.randint(self.n_img, getattr(self, "tri_base", self.n_nodes), size=A.nnz)
        M = sp.csr_matrix((A.data, (A.row, new_col)), shape=A.shape)
        M.sum_duplicates()
        return M

    def _build_edge_index(self):
        parts, types = [], []
        for t, A in enumerate([self.A_contains.tocoo(), self.A_scene.tocoo(),
                               self.A_related.tocoo(), self.A_isa.tocoo()]):
            if A.nnz == 0:
                continue
            parts.append(np.stack([A.row, A.col])); types.append(np.full(A.nnz, t))
            parts.append(np.stack([A.col, A.row])); types.append(np.full(A.nnz, t + 4))
        if not parts:
            return (torch.zeros(2, 0, dtype=torch.long), torch.zeros(0, dtype=torch.long))
        ei = torch.tensor(np.concatenate(parts, 1), dtype=torch.long)
        et = torch.tensor(np.concatenate(types), dtype=torch.long)
        return ei, et

    def summary(self):
        return dict(nodes=self.n_nodes, images=self.n_img, object_classes=self.n_obj,
                    concepts=self.n_con, edges=int(self.edge_index.shape[1] // 2),
                    contains=int(self.A_contains.nnz), scene=int(self.A_scene.nnz),
                    related=int(self.A_related.nnz), isa=int(self.A_isa.nnz),
                    triple_nodes=int(getattr(self, "n_tri", 0)),
                    triple_edges=int(self.A_triple.nnz) if hasattr(self, "A_triple") else 0,
                    paths={p: int(self.R[p].nnz) for p in PATTERNS},
                    ablation=self.ablation)

## 1.6 Phân tích truy vấn → khái niệm trên đồ thị

Một truy vấn được quy về tập node (lớp đối tượng hoặc khái niệm) mà nó nhắc tới.
Đây là điểm neo để `s_path` tìm đường đi.

In [ ]:
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next".split())


def parse_query(text, graph):
    """Trả về [(node_id, tên, n-gram khớp)] cho các khái niệm xuất hiện trong truy vấn."""
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):                                    # ưu tiên cụm 2 từ
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


def path_score_matrix(graph, concept_ids, cand_idx):
    """[len(cand), n_patterns] — số đường đi (đã log1p) theo từng mẫu, cho mỗi ứng viên."""
    out = np.zeros((len(cand_idx), len(PATTERNS)), np.float32)
    if not concept_ids:
        return out
    cols = np.array(sorted(set(concept_ids)))
    for p, name in enumerate(PATTERNS):
        sub = graph.R[name][cand_idx][:, cols]
        out[:, p] = np.asarray(sub.sum(1)).ravel()
    return np.log1p(out)

## 1.7 Mô hình

* **1.7.1 GNN cổ điển (cấp độ 1)**: GraphSAGE/GAT + PathScorer.
* **1.7.2 GNN của SCENIR (nhiệm vụ 2)**: thay nhánh `s_GNN` bằng encoder của SCENIR, bật bằng `CFG.gnn_type = "scenir"`.

### 1.7.1 GNN cổ điển (cấp độ 1)

GNN **tự cài bằng PyTorch thuần** (không cần PyTorch Geometric — tránh hẳn rủi ro cài đặt
trên Kaggle, và là thứ nhóm phải tự giải thích được khi vấn đáp).

* **GraphSAGE** — `h' = W₁h + W₂·mean(h_neighbours)`. Inductive: node ảnh test chưa từng
  tham gia huấn luyện vẫn có biểu diễn hợp lệ.
* **GAT** — hệ số chú ý từng cạnh, softmax theo node đích (dùng `index_reduce_` để ổn định số học).
* **PathScorer** — một trọng số học được cho mỗi mẫu đường đi, nhân hệ số suy giảm theo độ dài.
  Vì điểm là **tổng cộng tính** nên đóng góp của từng đường đi tách bạch được ⇒ fidelity đo được.

In [ ]:
def scatter_mean(src, index, n):
    out = torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    cnt = torch.zeros(n, 1, device=src.device, dtype=src.dtype)
    cnt.index_add_(0, index, torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype))
    return out / cnt.clamp(min=1)


def scatter_softmax(logits, index, n):
    m = torch.full((n,), -1e30, device=logits.device, dtype=logits.dtype)
    m = m.index_reduce_(0, index, logits, "amax", include_self=True)
    e = (logits - m[index]).exp()
    s = torch.zeros(n, device=logits.device, dtype=logits.dtype).index_add_(0, index, e)
    return e / s[index].clamp(min=1e-16)


class SAGELayer(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.Linear(din, dout)

    def forward(self, x, edge_index, edge_weight=None):
        src, dst = edge_index
        msg = x[src] if edge_weight is None else x[src] * edge_weight.unsqueeze(-1)
        agg = scatter_mean(msg, dst, x.size(0))
        return self.self_w(x) + self.neigh_w(agg)


class TypedSAGELayer(nn.Module):
    """h' = W₀h + Σ_t σ(g_t)·W_t·mean_{u∈N_t(v)} h_u — mỗi loại cạnh một ma trận và một cổng.
    SAGELayer cũ gộp contains/scene/related/isa vào CHUNG một trung bình ⇒ cạnh nhiễu pha
    loãng tín hiệu ảnh–đối tượng, và mô hình không có cách nào tắt riêng một loại cạnh.
    Cổng các loại quan hệ khởi tạo gần đóng ⇒ điểm xuất phát ≈ no_relations."""

    N_TYPES = 8                                   # 4 loại × 2 chiều (xem _build_edge_index)

    def __init__(self, din, dout, gate_init=-1.5):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(self.N_TYPES)])
        g = torch.full((self.N_TYPES,), float(gate_init))
        g[0] = g[4] = 3.0                         # contains (hai chiều) mở sẵn
        self.gate = nn.Parameter(g)

    def forward(self, x, edge_index, edge_type):
        src, dst = edge_index
        out = self.self_w(x)
        for t in range(self.N_TYPES):
            m = edge_type == t
            if not bool(m.any()):
                continue
            agg = scatter_mean(x[src[m]], dst[m], x.size(0))
            out = out + torch.sigmoid(self.gate[t]) * self.neigh_w[t](agg)
        return out


class GATLayer(nn.Module):
    def __init__(self, din, dout, heads=4):
        super().__init__()
        assert dout % heads == 0
        self.h, self.dk = heads, dout // heads
        self.lin = nn.Linear(din, dout)
        self.att_src = nn.Parameter(torch.empty(heads, self.dk))
        self.att_dst = nn.Parameter(torch.empty(heads, self.dk))
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index, edge_weight=None):
        N = x.size(0)
        src, dst = edge_index
        z = self.lin(x).view(N, self.h, self.dk)
        a = (z * self.att_src).sum(-1)[src] + (z * self.att_dst).sum(-1)[dst]   # [E,H]
        a = F.leaky_relu(a, 0.2)
        if edge_weight is not None:
            a = a + edge_weight.log().clamp(min=-5).unsqueeze(-1)
        out = torch.zeros(N, self.h, self.dk, device=x.device, dtype=z.dtype)
        for hh in range(self.h):                      # softmax riêng từng head
            al = scatter_softmax(a[:, hh], dst, N)
            out[:, hh].index_add_(0, dst, z[src, hh] * al.unsqueeze(-1))
        return out.reshape(N, -1)


class GraphScorer(nn.Module):
    """Sinh biểu diễn node bằng GNN, chiếu CLIP-text vào cùng không gian, cộng điểm đường đi."""

    def __init__(self, n_nodes, clip_dim, img_feat, cfg=CFG, edge_type=None):
        super().__init__()
        self.n_img = img_feat.shape[0]
        self.typed = cfg.gnn_typed and cfg.gnn_type == "sage"
        # loại cạnh của đồ thị ĐẦY ĐỦ — dùng khi encode_graph được gọi không kèm edge_type
        self.register_buffer("full_et", (edge_type if edge_type is not None
                                         else torch.zeros(0, dtype=torch.long)).clone())
        # Node ẢNH: chiếu từ đặc trưng CLIP đóng băng ⇒ inductive, ảnh test chưa từng
        # tham gia huấn luyện vẫn có biểu diễn hợp lệ (không phải vector ngẫu nhiên).
        self.register_buffer("img_feat", img_feat.clone())
        self.img_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        # Node ĐỐI TƯỢNG / KHÁI NIỆM: embedding học được (chúng lặp lại giữa các ảnh).
        self.sym_emb = nn.Embedding(n_nodes - self.n_img, cfg.gnn_dim)
        nn.init.normal_(self.sym_emb.weight, std=0.02)
        if self.typed:
            self.layers = nn.ModuleList([TypedSAGELayer(cfg.gnn_dim, cfg.gnn_dim, cfg.gate_init)
                                         for _ in range(cfg.gnn_layers)])
        else:
            Layer = SAGELayer if cfg.gnn_type == "sage" else GATLayer
            self.layers = nn.ModuleList(
                [Layer(cfg.gnn_dim, cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(                     # CLIP-text → không gian đồ thị
            nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(),
            nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        # PathScorer: 1 trọng số / mẫu, softplus để luôn ≥0 (giải thích mới diễn giải được)
        # P0 khởi tạo 0 (softplus = 0.69), P1..P5 khởi tạo thấp: đường đi gián tiếp phải
        # "tự chứng minh" trên mẫu âm khó thì trọng số mới tăng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        # path_gain CỐ ĐỊNH (buffer, không học): sau znorm_per_row điểm bất biến theo tỉ lệ nên
        # gain không có tín hiệu thật — gradient chỉ đi qua eps=1e-6 của z-norm, rất nhỏ nhưng
        # cùng dấu, và Adam vẫn bước đủ lr ⇒ gain trôi về ~1e-7, nhánh path bị TẮT NGẦM và mọi
        # "đóng góp" trong giải thích hiện +0.0000 (đã gặp trên AG). Tắt nhánh path là việc của λ.
        self.register_buffer("path_gain", torch.tensor(1.0))
        # trọng số của z(s_clip) trong logit lúc HUẤN LUYỆN, học được (softplus, khởi tạo = 1):
        # trên AG CLIP rất yếu (MRR 0.12) nên mô hình cần hạ được trọng số này.
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        # Hệ số của z(s_path) trong logit HUẤN LUYỆN (softplus, khởi tạo = 1). Thiếu nó, nhánh path
        # có hệ số cứng = 1, nên khi nó gây hại mô hình tự tắt bằng đường thoái hoá: đợt 2 kéo
        # path_gain → 1e-7, đợt 3 dồn trọng số chuẩn hoá sang mẫu không có dữ liệu (P0 → 0.000).
        # Lúc suy luận vẫn trộn bằng λ chọn trên val; hệ số này chỉ dùng khi huấn luyện.
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        return torch.cat([self.img_proj(self.img_feat), self.sym_emb.weight], 0)

    def edge_gates(self):
        """σ(g_t) của từng lớp — in ra để biết mô hình có thực sự dùng quan hệ hay không."""
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers] if self.typed else []

    def encode_graph(self, edge_index, edge_type=None):
        h = self.node_features()
        if self.typed and edge_type is None:
            assert self.full_et.numel() == edge_index.size(1), \
                "encode_graph trên đồ thị con cần truyền edge_type tương ứng"
            edge_type = self.full_et
        for i, (l, n) in enumerate(zip(self.layers, self.norms)):
            h = l(h, edge_index, edge_type) if self.typed else l(h, edge_index)
            h = n(h)
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        """Trọng số hiệu dụng của mỗi mẫu: P0 NEO = 1, P_k = softplus(w_k) · decay^(len_k − 1).
        Neo P0 thay cho chuẩn hoá tổng = 1: tỉ lệ vẫn định danh được, nhưng trọng số không thể
        trôi sang mẫu vắng dữ liệu (no_relations seed 2 đợt 3: P0 = 0.000, nhánh path chết)."""
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)          # [B, d]
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])      # [B, K]

    def path_score(self, path_mat):
        """path_mat: [B, K, n_patterns] → [B, K]. Cộng tính theo mẫu ⇒ tách được đóng góp."""
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)

### 1.7.2 GNN của SCENIR thay cho GNN cổ điển

SCENIR (ICML 2025, [arXiv 2505.15867](https://arxiv.org/html/2505.15867v1)). Chỉ thay nhánh `s_GNN`; split, ứng viên CLIP,
`s_path`, chọn α/λ trên val, ablation và fidelity giữ nguyên. Bật bằng `CFG.gnn_type = "scenir"`.

* **Đầu vào:** mỗi ảnh một scene graph. Đỉnh = lớp đối tượng của ảnh + khái niệm IsA của chúng; cạnh = cạnh
  `scene`/`related` (xây chỉ từ train) và `isa` của `MultimodalGraph` nằm trong tập đỉnh đó. Đặc trưng đỉnh =
  CLIP-text của tên đỉnh.
* **Encoder:** SCENIR GAE — 3 lớp GIN với hai nhánh `mu`/`logstd` tách rời, feature decoder và edge decoder MLP,
  discriminator đối kháng. Huấn luyện tự giám sát trên scene graph của ảnh **train** rồi đóng băng.
* **Điểm:** `z_I = L2norm(Σ_v mu_v)`, `s_GNN = cos(q_proj(CLIP_text(q)), z_I)`. `q_proj` và trọng số đường đi học
  bằng đúng vòng InfoNCE của GNN cổ điển trong `train_graph_scorer`.

In [ ]:
from contextlib import contextmanager

# Siêu tham số SCENIR = config.json của repo gốc; chiều đặc trưng đỉnh lấy theo CLIP-text (512 thay vì 768).
for _k, _v in dict(scenir_latent=1000, scenir_layers=3, scenir_disc_hidden=32,
                   scenir_edge_hidden=256, scenir_edge_out=32, scenir_lr=1e-3,
                   scenir_lr_gamma=0.95, scenir_bs=64, scenir_disc_steps=5,
                   scenir_epochs=30, scenir_with_concepts=True,
                   scenir_proj_hidden=1024).items():
    setattr(CFG, _k, _v)


@contextmanager
def use_gnn(kind):
    """Tạm đổi CFG.gnn_type ('sage' | 'gat' | 'scenir'); khôi phục khi ra khỏi khối."""
    old = CFG.gnn_type
    CFG.gnn_type = kind
    try:
        yield
    finally:
        CFG.gnn_type = old


def image_scene_graphs(graph):
    """Hợp rời (disjoint union) scene graph của mọi ảnh, cache trên chính đồ thị.
    node_gid: id node toàn cục của từng đỉnh cục bộ; node_ptr/edge_ptr: đỉnh/cạnh của ảnh i nằm liền
    nhau trong [ptr[i], ptr[i+1]); edge_index vô hướng (mỗi cạnh lưu hai chiều)."""
    if hasattr(graph, "_scenir_sg"):
        return graph._scenir_sg
    A_obj = ((graph.A_scene + graph.A_related) > 0).tocsr()
    A_isa = graph.A_isa.tocsr()
    nbrs = lambda M, n: M.indices[M.indptr[n]:M.indptr[n + 1]]
    node_gid, node_ptr, edge_ptr, src, dst = [], [0], [0], [], []
    for i in range(graph.n_img):
        objs = [int(o) for o in graph.img_objects[i]]
        local = {o: k for k, o in enumerate(objs)}
        pairs = set()
        for o in objs:
            for t in nbrs(A_obj, o):
                t = int(t)
                if t in local and t != o:
                    pairs.add((min(local[o], local[t]), max(local[o], local[t])))
        if CFG.scenir_with_concepts:
            for o in objs:
                for c in nbrs(A_isa, o):
                    c = int(c)
                    if c == o:
                        continue
                    if c not in local:
                        local[c] = len(local)
                    pairs.add((min(local[o], local[c]), max(local[o], local[c])))
        base = len(node_gid)
        node_gid.extend(local)                 # dict giữ thứ tự chèn = thứ tự id cục bộ
        node_ptr.append(len(node_gid))
        for a, b in sorted(pairs):
            src += [base + a, base + b]; dst += [base + b, base + a]
        edge_ptr.append(len(src))
    sg = dict(node_gid=np.array(node_gid, np.int64), node_ptr=np.array(node_ptr, np.int64),
              edge_ptr=np.array(edge_ptr, np.int64),
              edge_index=np.array([src, dst], np.int64).reshape(2, -1))
    graph._scenir_sg = sg
    return sg


_SCENIR_NAME_EMB = {}


def _encode_node_names(names):
    """CLIP-text đóng băng (cùng encoder với truy vấn). Không có CLIP (synthetic/offline) thì dùng
    vector ngẫu nhiên cố định theo tên — chỉ để smoke-test."""
    try:
        return clip_text_features(names)
    except Exception as e:
        print(f"⚠ Không mã hoá được tên node bằng CLIP ({type(e).__name__}) → vector ngẫu nhiên theo tên.")
        out = []
        for n in names:
            v = np.random.RandomState(int(hashlib.md5(n.encode()).hexdigest()[:8], 16)).randn(512)
            out.append((v / np.linalg.norm(v)).astype(np.float32))
        return np.stack(out)


def node_name_features(graph):
    """[n_nodes, D] đặc trưng đỉnh; dòng của node ảnh để 0 (scene graph không chứa node ảnh)."""
    names = [graph.id2name[i] for i in range(graph.n_img, graph.n_nodes)]
    new = [n for n in dict.fromkeys(names) if n not in _SCENIR_NAME_EMB]
    if new:
        _SCENIR_NAME_EMB.update(zip(new, _encode_node_names(new)))
    dim = len(next(iter(_SCENIR_NAME_EMB.values()))) if _SCENIR_NAME_EMB else 512
    X = np.zeros((graph.n_nodes, dim), np.float32)
    if names:
        X[graph.n_img:] = np.stack([_SCENIR_NAME_EMB[n] for n in names])
    return torch.tensor(X)

In [ ]:
def scatter_sum(src, index, n):
    return torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype).index_add_(0, index, src)


class GINLayer(nn.Module):
    """GINConv(nn.Linear) của PyG với ε = 0: h'_v = W·(h_v + Σ_{u∈N(v)} h_u)."""

    def __init__(self, din, dout):
        super().__init__()
        self.lin = nn.Linear(din, dout)

    def forward(self, x, edge_index):
        src, dst = edge_index
        return self.lin(x + scatter_sum(x[src], dst, x.size(0)))


class ScenirEncoder(nn.Module):
    """MultiGINEncoder của SCENIR: hai nhánh GIN tách rời cho mu và logstd, không ReLU ở lớp cuối."""

    def __init__(self, din, dhid, dout, layers=3):
        super().__init__()
        dims = [din] + [dhid] * (layers - 1) + [dout]
        self.gnn_mu = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.gnn_logstd = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])

    @staticmethod
    def _run(layers, x, edge_index):
        for i, layer in enumerate(layers):
            x = layer(x, edge_index)
            if i < len(layers) - 1:
                x = F.relu(x)
        return x

    def forward(self, x, edge_index, mu_only=False):
        mu = self._run(self.gnn_mu, x, edge_index)
        return mu if mu_only else (mu, self._run(self.gnn_logstd, x, edge_index))


class ScenirMLP(nn.Module):
    """MLP 2 lớp — feature decoder và edge decoder của SCENIR."""

    def __init__(self, din, dhid, dout):
        super().__init__()
        self.layer_1, self.layer_2 = nn.Linear(din, dhid), nn.Linear(dhid, dout)

    def forward(self, x):
        return self.layer_2(F.relu(self.layer_1(x)))


class ScenirDiscriminator(nn.Module):
    def __init__(self, din, dhid):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din, dhid), nn.ReLU(),
                                 nn.Linear(dhid, dhid), nn.ReLU(), nn.Linear(dhid, 1))

    def forward(self, z):
        return self.net(z)


SCENIR_EPS, SCENIR_MAX_LOGSTD = 1e-15, 10


def scenir_negative_edges(edge_index, batch, n_samples):
    """Cặp (u, v) cùng một đồ thị trong batch, không tự nối, không trùng cạnh thật."""
    N = batch.numel()
    cnt = torch.bincount(batch)
    ptr = torch.cat([cnt.new_zeros(1), cnt.cumsum(0)[:-1]])
    u = torch.randint(0, N, (2 * n_samples,), device=batch.device)
    b = batch[u]
    off = torch.minimum((torch.rand(u.numel(), device=batch.device) * cnt[b]).long(), cnt[b] - 1)
    v = ptr[b] + off
    keep = (u != v) & ~torch.isin(u * N + v, edge_index[0] * N + edge_index[1])
    return torch.stack([u[keep], v[keep]])[:, :n_samples]


class ScenirGAE(nn.Module):
    """SCENIR = ARGVA (VGAE + điều chuẩn đối kháng) + feature decoder MLP + edge decoder MLP."""

    def __init__(self, din, cfg=CFG):
        super().__init__()
        L = cfg.scenir_latent
        self.encoder = ScenirEncoder(din, L, L, cfg.scenir_layers)
        self.discriminator = ScenirDiscriminator(L, cfg.scenir_disc_hidden)
        self.feature_decoder = ScenirMLP(L, din, din)
        self.edge_decoder = ScenirMLP(L, cfg.scenir_edge_hidden, cfg.scenir_edge_out)

    def encode(self, x, edge_index):
        mu, logstd = self.encoder(x, edge_index)
        logstd = logstd.clamp(max=SCENIR_MAX_LOGSTD)
        z = mu + torch.randn_like(logstd) * logstd.exp() if self.training else mu
        return z, mu, logstd

    @staticmethod
    def _inner(z, edge_index):
        return torch.sigmoid((z[edge_index[0]] * z[edge_index[1]]).sum(-1))

    @staticmethod
    def kl_loss(mu, logstd):
        return -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu ** 2 - logstd.exp() ** 2, dim=1))

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.discriminator(z)) + SCENIR_EPS).mean()

    def discriminator_loss(self, z):
        real = torch.sigmoid(self.discriminator(torch.randn_like(z)))
        fake = torch.sigmoid(self.discriminator(z.detach()))
        return -torch.log(real + SCENIR_EPS).mean() - torch.log(1 - fake + SCENIR_EPS).mean()

    def recon_loss(self, x, edge_index, batch):
        """recon_loss_batched của repo: MSE tái tạo đặc trưng + BCE cạnh dương/âm trên đầu ra edge decoder.
        Encode lại như bản gốc; trả về (loss, mu, logstd) để kl_loss dùng đúng lần encode này."""
        z, mu, logstd = self.encode(x, edge_index)
        loss = F.mse_loss(self.feature_decoder(z), x)
        if edge_index.size(1) > 0:
            z_e = self.edge_decoder(z)
            loss = loss - torch.log(self._inner(z_e, edge_index) + SCENIR_EPS).mean()
            neg = scenir_negative_edges(edge_index, batch, edge_index.size(1))
            if neg.size(1) > 0:
                loss = loss - torch.log(1 - self._inner(z_e, neg) + SCENIR_EPS).mean()
        return loss, mu, logstd

In [ ]:
def _sg_batch(sg, X, img_idx):
    """Gom scene graph của các ảnh img_idx thành một batch (x, edge_index, batch) trên DEVICE.
    X phải nằm sẵn trên DEVICE."""
    img_idx = np.asarray(img_idx)
    ranges = lambda p: np.concatenate([np.arange(p[i], p[i + 1]) for i in img_idx])
    nodes, edges = ranges(sg["node_ptr"]), ranges(sg["edge_ptr"])
    counts = sg["node_ptr"][img_idx + 1] - sg["node_ptr"][img_idx]
    remap = np.full(len(sg["node_gid"]), -1, np.int64)
    remap[nodes] = np.arange(len(nodes))
    ei = remap[sg["edge_index"][:, edges]]
    x = X[torch.from_numpy(sg["node_gid"][nodes]).to(X.device)]
    return (x, torch.from_numpy(ei).to(X.device),
            torch.from_numpy(np.repeat(np.arange(len(img_idx)), counts)).to(X.device))


def train_scenir_gae(sg, X, img_idx, seed=0, verbose=True):
    """Huấn luyện SCENIR GAE tự giám sát CHỈ trên scene graph của ảnh train (không truy vấn, không nhãn).
    loss = (1/6)·reg + 3·recon + (1/3)·(1/num_nodes)·kl; discriminator cập nhật disc_steps bước/lần."""
    set_seed(seed)
    # Giữ cả đồ thị không cạnh (recon_loss chịu được), nếu không ablation no_relations không còn gì để huấn luyện.
    n_nodes = np.diff(sg["node_ptr"])
    idx = np.array([i for i in img_idx if n_nodes[i] > 0], np.int64)
    assert len(idx) > 0, "Không ảnh train nào có scene graph — kiểm tra nhãn đối tượng."
    gae = ScenirGAE(X.shape[1]).to(DEVICE)
    disc_opt = torch.optim.AdamW(gae.discriminator.parameters(), lr=CFG.scenir_lr)
    enc_opt = torch.optim.AdamW(list(gae.encoder.parameters()) + list(gae.edge_decoder.parameters())
                                + list(gae.feature_decoder.parameters()), lr=CFG.scenir_lr)
    sched = torch.optim.lr_scheduler.ExponentialLR(enc_opt, gamma=CFG.scenir_lr_gamma)
    hist, t0 = [], time.time()
    for ep in range(CFG.scenir_epochs):
        gae.train()
        perm = np.random.permutation(idx)
        tot_e = tot_d = 0.0
        nb = 0
        for s in range(0, len(perm), CFG.scenir_bs):
            x, ei, b = _sg_batch(sg, X, perm[s:s + CFG.scenir_bs])
            enc_opt.zero_grad()
            z, _, _ = gae.encode(x, ei)
            d_acc = 0.0
            for _ in range(CFG.scenir_disc_steps):
                disc_opt.zero_grad()
                d_loss = gae.discriminator_loss(z)
                d_loss.backward()
                disc_opt.step()
                d_acc += d_loss.item()
            rec, mu, logstd = gae.recon_loss(x, ei, b)
            loss = ((1 / 6) * gae.reg_loss(z) + 3 * rec
                    + (1 / 3) * (1 / x.size(0)) * gae.kl_loss(mu, logstd))
            loss.backward()
            enc_opt.step()
            tot_e += loss.item(); tot_d += d_acc / CFG.scenir_disc_steps; nb += 1
        sched.step()
        hist.append((tot_e / max(nb, 1), tot_d / max(nb, 1)))
        if verbose and (ep % 5 == 0 or ep == CFG.scenir_epochs - 1):
            print(f"  [SCENIR GAE] epoch {ep:2d} | enc {hist[-1][0]:.4f} | disc {hist[-1][1]:.4f} "
                  f"| {len(idx)} đồ thị train | {time.time() - t0:.0f}s")
    gae.eval()
    for p in gae.parameters():
        p.requires_grad_(False)
    return gae, hist


@torch.no_grad()
def scenir_embed_images(gae, sg, X, chunk=1024):
    """z_I = Σ_v mu_v (eval ⇒ không lấy mẫu) cho MỌI ảnh; ảnh không có đỉnh nào → vector 0."""
    gae.eval()
    n_img = len(sg["node_ptr"]) - 1
    out = torch.zeros(n_img, CFG.scenir_latent)
    for s in range(0, n_img, chunk):
        idx = np.arange(s, min(s + chunk, n_img))
        x, ei, b = _sg_batch(sg, X, idx)
        if x.size(0) == 0:
            continue
        mu = gae.encoder(x, ei, mu_only=True)
        out[torch.from_numpy(idx)] = scatter_sum(mu, b, len(idx)).cpu()
    return out

In [ ]:
class ScenirGraphScorer(nn.Module):
    """Cùng giao diện với GraphScorer (encode_graph / gnn_score / path_score / pattern_weights), nên
    score_queries, tune_on_val, fidelity và demo phía sau dùng lại nguyên vẹn."""

    def __init__(self, gae, z_img, clip_dim, cfg=CFG):
        super().__init__()
        self.gae = gae
        self.register_buffer("z_img", F.normalize(z_img, dim=-1))      # sum-pool ⇒ bắt buộc L2
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.scenir_proj_hidden), nn.ReLU(),
                                    nn.Linear(cfg.scenir_proj_hidden, z_img.shape[1]))
        # Phần chấm đường đi khởi tạo giống GraphScorer (mục 7.1) để hai nhánh GNN so sánh công bằng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def encode_graph(self, edge_index=None, edge_type=None):
        """GAE đã đóng băng ⇒ nhúng ảnh tính một lần; edge_index/edge_type chỉ để khớp giao diện GraphScorer."""
        return self.z_img

    def edge_gates(self):
        return []                                # SCENIR không có cổng theo loại cạnh

    pattern_weights = GraphScorer.pattern_weights
    gnn_score = GraphScorer.gnn_score
    path_score = GraphScorer.path_score


def build_scenir_scorer(engine, train_img_idx, seed=0, verbose=True):
    """Dựng scene graph từng ảnh, huấn luyện GAE trên ảnh train, nhúng mọi ảnh."""
    g = engine.graph
    sg = image_scene_graphs(g)
    X = node_name_features(g).to(DEVICE)
    if verbose:
        nv, ne = np.diff(sg["node_ptr"]), np.diff(sg["edge_ptr"]) // 2
        print(f"  SCENIR scene graph/ảnh: {nv.mean():.1f} đỉnh, {ne.mean():.1f} cạnh | "
              f"ảnh không có cạnh: {(ne == 0).mean():.1%}")
    gae, _ = train_scenir_gae(sg, X, train_img_idx, seed=seed, verbose=verbose)
    return ScenirGraphScorer(gae, scenir_embed_images(gae, sg, X), engine.txt_feat.shape[1])

## 1.8 Bộ máy truy vấn: ứng viên, huấn luyện, xếp hạng lại

Để tính `s_path` cho mọi cặp (truy vấn, ảnh) mà không phải lặp Python, ta dựng
ma trận chỉ báo khái niệm `Q [n_query × n_node]` rồi nhân thưa một lần cho mỗi mẫu:
`S_p = R_p · Qᵀ` → `[n_img × n_query]`. Sau đó chỉ việc tra.

In [ ]:
def znorm_per_row(x, eps=1e-6):
    """Chuẩn hoá z theo từng truy vấn. Dùng CHUNG cho train và eval để α, λ chọn
    trên val vẫn đúng khi áp lên test."""
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


minmax_per_row = znorm_per_row      # giữ tên cũ cho tương thích


class RetrievalEngine:
    def __init__(self, records, graph, img_feat, txt_feat, queries):
        """queries: list[dict(text, gold_image_id, group)] — group ∈ {direct, indirect}"""
        self.records, self.graph = records, graph
        self.img_feat = torch.tensor(img_feat)
        self.txt_feat = torch.tensor(txt_feat)
        self.queries = queries
        self.q_concepts = [parse_query(q["text"], graph) for q in queries]
        self._build_S()

    def _build_S(self):
        g = self.graph
        rows, cols = [], []
        for j, hits in enumerate(self.q_concepts):
            for nid, _, _ in hits:
                rows.append(nid); cols.append(j)
        Q = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                          shape=(g.n_nodes, len(self.queries)))
        self.S = {p: (g.R[p] @ Q).tocsr() for p in PATTERNS}   # [n_img × n_query]

    def path_feats(self, img_idx, q_idx):
        """img_idx [B,K] hoặc [B]; q_idx [B] → [.., n_patterns]"""
        img_idx = np.asarray(img_idx); q_idx = np.asarray(q_idx)
        if img_idx.ndim == 1:
            img_idx = img_idx[:, None]
        B, K = img_idx.shape
        rr = img_idx.ravel(); cc = np.repeat(q_idx, K)
        out = np.zeros((B * K, len(PATTERNS)), np.float32)
        for p, name in enumerate(PATTERNS):
            out[:, p] = np.asarray(self.S[name][rr, cc]).ravel()
        return np.log1p(out).reshape(B, K, len(PATTERNS))   # log1p SAU khi cộng

    def clip_candidates(self, q_idx, gallery_idx, k):
        qf = self.txt_feat[q_idx]                       # [B, D]
        gf = self.img_feat[gallery_idx]                 # [G, D]
        sim = qf @ gf.T
        k = min(k, len(gallery_idx))
        top = sim.topk(k, dim=1)
        return gallery_idx[top.indices.numpy()], top.values.numpy()


def train_graph_scorer(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    """InfoNCE với mẫu âm trong batch. CLIP đóng băng; chỉ học GNN + trọng số đường đi."""
    set_seed(seed)
    g = engine.graph
    if CFG.gnn_type == "scenir":                 # mục 7.2: GAE tự giám sát trên ảnh train, rồi đóng băng
        model = build_scenir_scorer(engine, train_img_idx, seed=seed, verbose=verbose).to(DEVICE)
    else:
        model = GraphScorer(g.n_nodes, engine.txt_feat.shape[1],
                            engine.img_feat, edge_type=g.edge_type).to(DEVICE)
    # CFG.learn_clip_w = False ⇒ cố định hệ số z(s_clip) ở softplus(0.5413) = 1 (không vào optimizer).
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    # Cổng cạnh + trọng số mẫu đường đi chỉ là vài vô hướng: cho lr riêng (×CFG.scalar_lr_mult),
    # không weight decay — nếu không chúng gần như đứng yên ở giá trị khởi tạo.
    # Tham số đã đóng băng (GAE của SCENIR) không đưa vào optimizer.
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n, p in params
              if n.endswith("gate") or n in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW(
        [dict(params=[p for _, p in params if id(p) not in sid]),
         dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
        lr=CFG.lr, weight_decay=CFG.weight_decay)
    # INDUCTIVE: khi huấn luyện bỏ mọi cạnh chạm node ảnh KHÔNG thuộc train (val/test).
    # Node ảnh val/test thành node cô lập ⇒ không gửi thông điệp, không nhận gradient,
    # không ảnh hưởng tới biểu diễn đối tượng/khái niệm học được. Lúc đánh giá
    # (score_queries) mới dùng đồ thị đầy đủ — GraphSAGE inductive nên hợp lệ.
    keep_img = np.zeros(g.n_img, bool)
    keep_img[np.asarray(train_img_idx)] = True
    src, dst = g.edge_index.numpy()
    is_bad = lambda v: (v < g.n_img) & ~keep_img[np.minimum(v, g.n_img - 1)]
    mask = ~(is_bad(src) | is_bad(dst))
    tmask = torch.from_numpy(mask)
    ei = g.edge_index[:, tmask].to(DEVICE)
    et = g.edge_type[tmask].to(DEVICE)
    if verbose:
        print(f"  cạnh dùng khi huấn luyện: {int(mask.sum())}/{len(mask)} "
              f"(đã bỏ {int((~mask).sum())} cạnh chạm ảnh val/test)")
    hist = []
    tr_q, tr_img = np.asarray(train_q_idx), np.asarray(train_img_idx)
    img_dev = engine.img_feat.to(DEVICE)

    # Mẫu âm KHÓ: CLIP top-K trong gallery TRAIN (bỏ ảnh đúng). Lúc test đồ thị chỉ xếp lại
    # top-100 của CLIP, nên phải học phân biệt ĐÚNG những ứng viên đó — với âm ngẫu nhiên
    # trong batch, đường đi gián tiếp "dễ thắng" và học trọng số không dùng được ở test.
    hard = None
    if CFG.hard_neg_k > 0:
        cand_hn, _ = engine.clip_candidates(tr_q, tr_img, CFG.hard_neg_k + 1)
        hard = np.stack([np.concatenate([row[row != gold], row[:1]])[:CFG.hard_neg_k]
                         for row, gold in zip(cand_hn, tr_img)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.tensor(cand_np, device=DEVICE)
        pm = torch.tensor(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:      # CLIP không có gradient; chỉ trọng số clip_w học được
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, img_dev[cand]))
        return z / CFG.temperature

    for ep in range(CFG.epochs):
        model.train()
        order = np.random.permutation(len(train_q_idx))
        tot, nb = 0.0, 0
        for s in range(0, len(order), CFG.train_bs):
            sel = order[s:s + CFG.train_bs]
            if len(sel) < 8:
                continue
            qi = tr_q[sel]
            gi = tr_img[sel]                              # ảnh đúng của từng truy vấn
            qf = engine.txt_feat[qi].to(DEVICE)
            h = model.encode_graph(ei, et)

            # (a) âm trong batch như cũ: [B, B], đáp án trên đường chéo
            loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf),
                                   torch.arange(len(sel), device=DEVICE))
            # (b) âm khó từ CLIP: [B, 1+K], đáp án ở cột 0
            if hard is not None:
                cand_h = np.concatenate([gi[:, None], hard[sel]], 1)
                loss = loss + F.cross_entropy(logits_for(cand_h, qi, h, qf),
                                              torch.zeros(len(sel), dtype=torch.long, device=DEVICE))

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 3 == 0 or ep == CFG.epochs - 1):
            w = model.pattern_weights().detach().cpu().numpy()
            gates = model.edge_gates()
            # Cổng của LỚP 0: ở lớp cuối chỉ node ảnh được dùng, mà ảnh chỉ nhận cạnh contains
            # ⇒ cổng scene/related/isa lớp cuối không có gradient (luôn đứng ở giá trị khởi tạo).
            gs = (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4])) if gates else ""
            print(f"  epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.3f}" for x in w) + gs +
                  f" | clip_w {F.softplus(model.clip_logit).item():.2f}")
    return model, hist


@torch.no_grad()
def score_queries(model, engine, q_idx, gallery_idx, k=None):
    """Trả về (cand_idx, s_clip, s_gnn, s_path) — mọi điểm đã chuẩn hoá z theo hàng. Dùng đồ thị ĐẦY ĐỦ."""
    k = k or CFG.rerank_k
    model.eval()
    ei = engine.graph.edge_index.to(DEVICE)
    h = model.encode_graph(ei)
    cand, s_clip = engine.clip_candidates(q_idx, gallery_idx, k)
    pm = torch.tensor(engine.path_feats(cand, q_idx)).to(DEVICE)
    qf = engine.txt_feat[q_idx].to(DEVICE)
    s_gnn = model.gnn_score(h, qf, torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    return cand, znorm_per_row(s_clip), znorm_per_row(s_gnn), znorm_per_row(s_path)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)

## 1.9 Chỉ số đánh giá

In [ ]:
def retrieval_metrics(cand, scores, gold_pos, ks=(1, 5, 10)):
    """gold_pos: chỉ số ảnh đúng (trong không gian toàn cục) cho từng truy vấn."""
    order = np.argsort(-scores, axis=1)
    ranked = np.take_along_axis(cand, order, axis=1)
    hit = ranked == np.asarray(gold_pos)[:, None]
    has = hit.any(1)
    rank = np.where(has, hit.argmax(1) + 1, 10**9)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float(np.where(has, 1.0 / rank, 0.0).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    return out


def fmt_metrics(runs):
    """runs: list[dict] từ nhiều seed → chuỗi 'mean ± std'."""
    keys = runs[0].keys()
    return {k: f"{np.mean([r[k] for r in runs]):.4f} ± {np.std([r[k] for r in runs]):.4f}"
            for k in keys}

## 1.10 Bộ truy vấn đánh giá: trực tiếp vs gián tiếp

Đây là bằng chứng chính cho câu hỏi "tri thức quan hệ giúp ở đâu".

* **Trực tiếp** — truy vấn nhắc tới lớp đối tượng có sẵn trong annotation của ảnh đúng
  (tồn tại đường đi P0). CLIP thường đã làm tốt nhóm này.
* **Gián tiếp** — truy vấn nhắc tới khái niệm **không** nằm trong danh sách đối tượng của ảnh,
  nhưng nối được qua quan hệ (P1–P5). Đây là nhóm đồ thị phải chứng minh giá trị.

Cách phân nhóm này tự động và kiểm tra được, tránh việc gán nhãn cảm tính.

In [ ]:
def build_query_set(records, graph, img_ids, seed=0, verbose=True):
    pos = graph.img_pos
    queries = []
    for iid in img_ids:
        i = pos[iid]
        r = records[i]
        hits = parse_query(r["caption"], graph)
        # nhóm direct/indirect tính trên đối tượng/khái niệm như bản trước (không tính node bộ ba)
        hits = [h for h in hits if h[0] < getattr(graph, "tri_base", graph.n_nodes)]
        own = set(graph.img_objects[i])
        hit_ids = {n for n, _, _ in hits}
        direct = bool(hit_ids & own)
        indirect_ids = hit_ids - own
        reach = 0
        if indirect_ids:
            cols = np.array(sorted(indirect_ids))
            for p in PATTERNS[1:]:
                reach += int(graph.R[p][i][:, cols].nnz)
        group = "indirect" if (indirect_ids and reach > 0 and not direct) else \
                ("direct" if direct else "other")
        queries.append(dict(text=r["caption"], gold=iid, group=group,
                            n_concepts=len(hits), n_indirect=len(indirect_ids)))
    if verbose:
        print(f"Nhóm truy vấn: {dict(Counter(q['group'] for q in queries))}")
    return queries


def export_eval_queries(queries, graph, path, n=20, allowed_ids=None):
    """Xuất bộ ≥20 truy vấn đánh giá của cấp độ 2. allowed_ids = id ảnh TEST (bắt buộc)."""
    assert allowed_ids is not None, "Truyền allowed_ids=set(split['test']) — chỉ lấy truy vấn của tập test"
    allowed_ids = set(allowed_ids)
    pool = [q for q in queries if q["gold"] in allowed_ids]
    ind = [q for q in pool if q["group"] == "indirect"]
    dir_ = [q for q in pool if q["group"] == "direct"]
    rng = random.Random(0)
    sel = rng.sample(ind, min(n // 2, len(ind))) + rng.sample(dir_, min(n // 2, len(dir_)))
    payload = dict(
        n=len(sel),
        construction=("Truy vấn lấy từ caption của ảnh trong tập test. Nhóm 'indirect' là "
                      "truy vấn có khái niệm KHÔNG nằm trong annotation đối tượng của ảnh "
                      "đúng nhưng nối được qua đường đi P1-P5; nhóm 'direct' có ít nhất một "
                      "khái niệm trùng annotation (đường đi P0). Phân nhóm tự động, kiểm tra "
                      "lại được bằng hàm build_query_set."),
        usage="CHỈ dùng để đánh giá. Không tinh chỉnh mô hình trên bộ này.",
        queries=[{k: q[k] for k in ("text", "gold", "group", "n_concepts", "n_indirect")}
                 for q in sel])
    with open(path, "w") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Đã ghi {len(sel)} truy vấn đánh giá → {path}")
    return sel

## 1.11 Chọn siêu tham số trên VALIDATION

Quét lưới `(α, λ)` trên tập val. Test chưa hề được chạm vào ở bước này.

In [ ]:
def tune_on_val(model, engine, val_q, val_gallery, verbose=True):
    cand, sc, sg, sp_ = score_queries(model, engine, val_q["q_idx"], val_gallery)
    best = (-1, None)
    grid = []
    for a in CFG.alpha_grid:
        for l in CFG.lambda_grid:
            m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), val_q["gold_pos"])
            grid.append(dict(alpha=a, lam=l, **m))
            if m["MRR"] > best[0]:
                best = (m["MRR"], (a, l))
    if verbose:
        print(f"  Tốt nhất trên val: α={best[1][0]}, λ={best[1][1]} (MRR={best[0]:.4f})")
    return best[1], grid

## 1.12 Giải thích và đo fidelity (Nhiệm vụ 3)

Giải thích = danh sách **đường đi cụ thể** đóng góp nhiều nhất vào `s_path` của ứng viên đó.
Phần đóng góp không phải suy đoán: nó được đo bằng cách **bỏ đúng đường đi đó rồi tính lại điểm**.

| Chỉ số | Định nghĩa | Kỳ vọng |
|---|---|---|
| fidelity− (cần thiết) | bỏ đường đi giải thích → Δ điểm, Δ thứ hạng | điểm giảm rõ, hạng tụt |
| fidelity+ (đủ) | **chỉ giữ** đường đi giải thích → Δ điểm | điểm gần như không đổi |
| sparsity | 1 − |đường đi giải thích| / |toàn bộ đường đi| | càng cao càng gọn |
| validity | đường đi có thật và nối đúng khái niệm truy vấn ↔ ảnh | 100% |
| **đối chứng** | bỏ **ngẫu nhiên** cùng số đường đi | phải ÍT ảnh hưởng hơn |

Dòng đối chứng là dòng quan trọng nhất: nếu bỏ đường đi giải thích không tệ hơn bỏ ngẫu nhiên
thì giải thích vô nghĩa. Đây gần như chắc chắn là câu hỏi vấn đáp.

In [ ]:
def enumerate_paths(graph, img_pos_idx, concept_ids, cap=100000):
    """Liệt kê đường đi CỤ THỂ từ node ảnh tới các khái niệm truy vấn.
    Mỗi đường đi mang trọng số w = idf(node đích) × Π trọng số cạnh (P(đích|nguồn) nếu
    CFG.edge_weighting) — đúng trọng số trong graph.R, nên tổng theo mẫu khớp tuyệt đối
    với điểm tính bằng ma trận."""
    target = set(concept_ids)
    objs = graph.img_objects[img_pos_idx]
    own = set(objs)
    A_s, A_r, A_i = graph.A_scene, graph.A_related, graph.A_isa
    idf = graph.idf
    out = []

    def nbrs(M, n):
        sl = slice(M.indptr[n], M.indptr[n + 1])
        return zip(M.indices[sl], M.data[sl])

    def add(pattern, nodes, rel=None, ew=1.0):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return                                  # cùng quy tắc với ma trận R
        w = float(idf[c]) * float(ew)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=nodes, rel=rel, w=w))

    for o in objs:
        if o in target:
            add("P0_contains", [o])
        for o2, w1 in nbrs(A_s, o):
            rel = graph.scene_pred.get((o, o2), "quan hệ")
            if o2 in target:
                add("P1_scene", [o, o2], rel, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P4_scene_isa", [o, o2, c], rel, w1 * w2)
        for c, w1 in nbrs(A_i, o):
            if c in target:
                add("P2_isa", [o, c], None, w1)
        for o2, w1 in nbrs(A_r, o):
            if o2 in target:
                add("P3_related", [o, o2], None, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], None, w1 * w2)
        if len(out) > cap:
            break
    # v5: ảnh –thể_hiện→ bộ ba (trọng số = xác suất probe × idf, đúng như graph.R['P6_triple'])
    A_t = getattr(graph, "A_triple", None)
    if A_t is not None and A_t.nnz:
        sl = slice(A_t.indptr[img_pos_idx], A_t.indptr[img_pos_idx + 1])
        for t, v in zip(A_t.indices[sl], A_t.data[sl]):
            if int(t) in target:
                add("P6_triple", [int(t)], None, v)
    return out[:cap]


def paths_to_counts(paths):
    """Đặc trưng mẫu = log1p(Σ trọng số đường đi của mẫu) — giống hệt RetrievalEngine.path_feats."""
    v = np.zeros(len(PATTERNS), np.float64)
    for p in paths:
        v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_text(graph, p):
    n = [graph.id2name.get(x, f"#{x}") for x in p["nodes"]]
    t = PATTERN_DESC[p["pattern"]]
    d = dict(c=n[-1], o=n[0], rel=p.get("rel") or "quan_hệ")
    if len(n) >= 3:
        d["o2"] = n[1]
    return t.format(**d)


@torch.no_grad()
def explain_and_score(model, graph, img_idx, concept_ids, top_m=CFG.top_m_paths):
    """Trả về (giải thích, điểm đầy đủ, chi tiết fidelity) cho MỘT cặp (truy vấn, ảnh)."""
    paths = enumerate_paths(graph, img_idx, concept_ids)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts(ps) * w).sum())

    full = score(paths)
    # đóng góp biên của từng đường đi: bỏ đúng nó ra rồi tính lại
    contrib = []
    for i, p in enumerate(paths):
        rest = paths[:i] + paths[i + 1:]
        contrib.append(full - score(rest))
    order = np.argsort(-np.array(contrib)) if contrib else np.array([], int)
    # v5: giải thích phải NHỎ HƠN toàn bộ đường đi, nếu không đối chứng ngẫu nhiên trùng hệt giải thích
    # (bản trước in 'fidelity− = đối chứng, sparsity 0.00' cho mọi ca có ≤ top_m đường đi).
    m = min(top_m, len(paths) - 1) if len(paths) >= 2 else len(paths)
    expl_idx = list(order[:m])
    expl = [dict(text=path_text(graph, paths[i]), pattern=paths[i]["pattern"],
                 contribution=float(contrib[i])) for i in expl_idx]

    keep = [paths[i] for i in expl_idx]
    drop = [p for i, p in enumerate(paths) if i not in set(expl_idx)]
    # Đối chứng: bỏ NGẪU NHIÊN cùng số đường đi, lấy trung bình 10 lần bốc.
    # Bốc một lần thôi thì với truy vấn ít đường đi nó hay trùng đúng tập giải thích
    # và đối chứng mất ý nghĩa.
    rng = np.random.RandomState(0)
    rnd_scores = []
    for _ in range(10):
        ridx = set(rng.choice(len(paths), size=min(len(expl_idx), len(paths)),
                              replace=False).tolist()) if paths else set()
        rnd_scores.append(score([p for i, p in enumerate(paths) if i not in ridx]))
    rnd_score = float(np.mean(rnd_scores)) if (rnd_scores and len(paths) >= 2) else float("nan")

    detail = dict(
        n_paths=len(paths),
        score_full=full,
        score_without_explanation=score(drop),      # fidelity−
        score_only_explanation=score(keep),         # fidelity+
        score_without_random=rnd_score,             # ĐỐI CHỨNG (trung bình 10 lần)
        sparsity=1 - len(expl_idx) / max(len(paths), 1),
        validity=float(all(p["nodes"][-1] in set(concept_ids) for p in keep)) if keep else 0.0,
    )
    return expl, full, detail

## 1.13 Chạy thí nghiệm chính

Một hàm `run_experiment` cho cả 3 cấu hình ablation, mỗi cấu hình 3 seed.

In [ ]:
_GROUP_CACHE = {}
PER_QUERY = {}          # (ablation, seed) → MRR từng truy vấn test; khoá "<ab>+adaptive" cho bản thích nghi
ART_BY_ABL = {}         # ablation → mô hình + đồ thị của seed đầu
FUSION_LOG = []         # nhật ký kết hợp thích nghi theo (ablation, seed)


def reciprocal_ranks(cand, scores, gold_pos):
    order = np.argsort(-scores, axis=1)
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold_pos)[:, None]
    return np.where(hit.any(1), 1.0 / (hit.argmax(1) + 1), 0.0)


def prepare(records, split, img_feat, txt_feat, ablation="none", seed=0):
    graph = MultimodalGraph(records, split["train"], ablation=ablation, seed=seed)
    # Nhóm trực tiếp/gián tiếp chốt MỘT LẦN trên đồ thị đầy đủ rồi dùng lại cho mọi ablation.
    if "groups" not in _GROUP_CACHE:
        base = graph if ablation == "none" else MultimodalGraph(records, split["train"])
        _GROUP_CACHE["groups"] = {q["gold"]: q["group"] for q in
                                  build_query_set(records, base, [r["image_id"] for r in records])}
    queries = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    for q in queries:
        q["group"] = _GROUP_CACHE["groups"].get(q["gold"], q["group"])
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, queries)
    pos = graph.img_pos
    idx = lambda ids: np.array([pos[i] for i in ids])
    parts = {}
    for name in ("train", "val", "test"):
        ids = split[name]
        parts[name] = dict(q_idx=idx(ids), gold_pos=idx(ids), img_idx=idx(ids),
                           groups=[queries[pos[i]]["group"] for i in ids])
    return graph, eng, parts


def evaluate_split(model, engine, part, alpha, lam, by_group=True):
    gallery = part["img_idx"]
    cand, sc, sg, sp_ = score_queries(model, engine, part["q_idx"], gallery)
    res = {}
    full = combine(sc, sg, sp_, alpha, lam)
    res["CLIP"] = retrieval_metrics(cand, sc, part["gold_pos"])
    res["CLIP+Graph"] = retrieval_metrics(cand, full, part["gold_pos"])
    if by_group:
        g = np.array(part["groups"])
        for grp in ("direct", "indirect"):
            m = g == grp
            if m.sum() >= 5:
                res[f"CLIP [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], sc[m], np.array(part["gold_pos"])[m])
                res[f"CLIP+Graph [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], full[m], np.array(part["gold_pos"])[m])
    return res, (cand, sc, sg, sp_)


# ============================ KẾT HỢP ĐIỂM THÍCH NGHI ============================
FUSION_FEATS = ["has_concept", "log_n_concepts", "path_coverage", "clip_margin"]


def fusion_features(engine, q_idx, cand, sc):
    """Đặc trưng của TRUY VẤN, không dùng nhãn đúng: có khái niệm không, bao nhiêu khái niệm,
    tỉ lệ ứng viên top-K có ≥1 đường đi, độ chênh top1–top2 của CLIP (CLIP tự tin thì ít cần đồ thị)."""
    ncon = np.array([len(engine.q_concepts[q]) for q in q_idx], np.float32)
    cov = (engine.path_feats(cand, q_idx).sum(-1) > 0).mean(1)
    srt = -np.sort(-sc, axis=1)
    return np.stack([(ncon > 0).astype(np.float32), np.log1p(ncon), cov,
                     srt[:, 0] - srt[:, 1]], 1).astype(np.float32)


def _fuse(a, l, sc, sg, sp):
    return a[:, None] * sc + (1 - a[:, None]) * (l[:, None] * sg + (1 - l[:, None]) * sp)


class AdaptiveFusion(nn.Module):
    def __init__(self, d, a0, l0, mu, sd):
        super().__init__()
        self.wa, self.wl = nn.Linear(d, 1), nn.Linear(d, 1)
        for lin, v in ((self.wa, a0), (self.wl, l0)):
            v = min(max(float(v), 0.02), 0.98)
            nn.init.zeros_(lin.weight); nn.init.constant_(lin.bias, math.log(v / (1 - v)))
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(sd, dtype=torch.float32))

    def gates(self, Xn):
        return torch.sigmoid(self.wa(Xn)).squeeze(-1), torch.sigmoid(self.wl(Xn)).squeeze(-1)

    @torch.no_grad()
    def score(self, X, sc, sg, sp):
        a, l = self.gates((torch.tensor(X) - self.mu) / self.sd)
        a, l = a.numpy(), l.numpy()
        return _fuse(a, l, sc, sg, sp), a, l


def fit_fusion(X, sc, sg, sp, cand, gold, a0, l0):
    """Listwise CE trên top-K ứng viên của VAL. Khởi tạo w=0 ⇒ xuất phát đúng bằng (α, λ) toàn cục."""
    gold = np.asarray(gold); hit = cand == gold[:, None]; keep = hit.any(1)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    m = AdaptiveFusion(X.shape[1], a0, l0, mu, sd)
    m.wa.bias.requires_grad_(False); m.wl.bias.requires_grad_(False)   # α(q) chỉ lệch khỏi α toàn cục theo đặc trưng
    if keep.sum() < 10:
        return m
    t = lambda a: torch.tensor(a[keep], dtype=torch.float32)
    Xn, SC, SG, SP = t((X - mu) / sd), t(sc), t(sg), t(sp)
    y = torch.tensor(hit[keep].argmax(1))
    opt = torch.optim.Adam([m.wa.weight, m.wl.weight], lr=0.05)
    for _ in range(CFG.fusion_steps):
        a, l = m.gates(Xn)
        loss = F.cross_entropy(_fuse(a, l, SC, SG, SP) / CFG.fusion_tau, y) \
            + CFG.fusion_wd * (m.wa.weight.pow(2).sum() + m.wl.weight.pow(2).sum())
        opt.zero_grad(); loss.backward(); opt.step()
    return m


def cv_fusion(X, sc, sg, sp, cand, gold, a0, l0, k=5, seed=0):
    """MRR ngoài-fold trên VAL: thích nghi vs (α, λ) toàn cục. Chống overfit khi chỉ có ~100–1000 truy vấn val."""
    gold = np.asarray(gold); n = len(X)
    idx = np.random.RandomState(seed).permutation(n)
    rr_a, rr_g = np.zeros(n), np.zeros(n)
    for f in np.array_split(idx, k):
        tr = np.setdiff1d(idx, f)
        m = fit_fusion(X[tr], sc[tr], sg[tr], sp[tr], cand[tr], gold[tr], a0, l0)
        rr_a[f] = reciprocal_ranks(cand[f], m.score(X[f], sc[f], sg[f], sp[f])[0], gold[f])
        rr_g[f] = reciprocal_ranks(cand[f], combine(sc[f], sg[f], sp[f], a0, l0), gold[f])
    return float(rr_a.mean()), float(rr_g.mean())


def adaptive_rows(fus, engine, part, raw):
    cand, sc, sg, sp_ = raw
    X = fusion_features(engine, part["q_idx"], cand, sc)
    s, a, l = fus.score(X, sc, sg, sp_)
    gold = np.asarray(part["gold_pos"])
    res = {"CLIP+Graph (adaptive)": retrieval_metrics(cand, s, gold)}
    g = np.array(part["groups"])
    for grp in ("direct", "indirect"):
        m = g == grp
        if m.sum() >= 5:
            res[f"CLIP+Graph (adaptive) [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], s[m], gold[m])
    return res, s, a, l, X


def _adaptive_step(model, eng, parts, alpha, lam, test_raw):
    pv = parts["val"]
    vc, vsc, vsg, vsp = score_queries(model, eng, pv["q_idx"], pv["img_idx"])
    Xv = fusion_features(eng, pv["q_idx"], vc, vsc)
    cv_a, cv_g = cv_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    fus = fit_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    ad_res, s_ad, a_q, l_q, Xt = adaptive_rows(fus, eng, parts["test"], test_raw)
    has = Xt[:, 0] > 0
    flog = dict(alpha=alpha, lam=lam, cv_global=cv_g, cv_adaptive=cv_a,
                alpha_q_with_concept=float(a_q[has].mean()) if has.any() else float("nan"),
                alpha_q_no_concept=float(a_q[~has].mean()) if (~has).any() else float("nan"),
                w_alpha=dict(zip(FUSION_FEATS, fus.wa.weight.detach().numpy().ravel().round(3).tolist())))
    return fus, flog, ad_res, s_ad


def run_experiment(records, split, img_feat, txt_feat, ablations=("none",), seeds=None):
    seeds = seeds or CFG.seeds
    _GROUP_CACHE.clear(); PER_QUERY.clear(); ART_BY_ABL.clear(); FUSION_LOG.clear()
    all_res = defaultdict(list)
    artefacts = {}
    for ab in ablations:
        print(f"\n{'='*66}\nABLATION = {ab}\n{'='*66}")
        for si, seed in enumerate(seeds):
            graph, eng, parts = prepare(records, split, img_feat, txt_feat, ab, seed)
            if si == 0:
                print("Đồ thị:", graph.summary())
            model, hist = train_graph_scorer(eng, parts["train"]["q_idx"],
                                             parts["train"]["img_idx"], seed=seed,
                                             verbose=(si == 0))
            (alpha, lam), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"],
                                             verbose=(si == 0))
            res, raw = evaluate_split(model, eng, parts["test"], alpha, lam)
            cand, sc, sg, sp_ = raw
            gold_t, groups_t = parts["test"]["gold_pos"], np.array(parts["test"]["groups"])
            rr_clip = reciprocal_ranks(cand, sc, gold_t)
            PER_QUERY[(ab, seed)] = dict(rr=reciprocal_ranks(cand, combine(sc, sg, sp_, alpha, lam), gold_t),
                                         rr_clip=rr_clip, groups=groups_t, lam=lam)
            if "relprobe_hook" in globals():          # v5: bộ quan hệ khó (α, λ giữ nguyên từ VAL)
                relprobe_hook(ab, seed, model, eng, parts, alpha, lam)
            if "match_hook" in globals():             # v9: nhánh khớp truy vấn ↔ node (α, λ giữ nguyên từ VAL)
                match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res)
            fus, flog = None, None
            if CFG.adaptive_fusion:
                fus, flog, ad_res, s_ad = _adaptive_step(model, eng, parts, alpha, lam, raw)
                res.update(ad_res)
                FUSION_LOG.append(dict(ablation=ab, seed=seed, **flog))
                PER_QUERY[(f"{ab}+adaptive", seed)] = dict(rr=reciprocal_ranks(cand, s_ad, gold_t),
                                                           rr_clip=rr_clip, groups=groups_t, lam=lam)
            for k, v in res.items():
                all_res[f"{ab} | {k}"].append(v)
            if si == 0:
                ART_BY_ABL[ab] = dict(graph=graph, engine=eng, model=model, parts=parts, alpha=alpha,
                                      lam=lam, grid=grid, raw=raw, hist=hist, fusion=fus)
            if ab == "none" and si == 0:
                artefacts = ART_BY_ABL[ab]
            msg = (f"  seed {seed}: R@1={res['CLIP+Graph']['R@1']:.4f} "
                   f"MRR={res['CLIP+Graph']['MRR']:.4f} (α={alpha}, λ={lam})")
            if flog:
                msg += (f" | adaptive MRR={res['CLIP+Graph (adaptive)']['MRR']:.4f} "
                        f"[CV val {flog['cv_adaptive']:.4f} vs {flog['cv_global']:.4f}; "
                        f"α(q) có/không khái niệm {flog['alpha_q_with_concept']:.2f}/{flog['alpha_q_no_concept']:.2f}]")
            print(msg)
    return all_res, artefacts


def results_table(all_res):
    import pandas as pd
    rows = []
    for k, runs in all_res.items():
        ab, name = k.split(" | ", 1)
        rows.append(dict(ablation=ab, model=name, **fmt_metrics(runs)))
    return pd.DataFrame(rows)


def final_model_table(all_res, fusion_log):
    """MÔ HÌNH CUỐI = (cấu trúc đồ thị, global|adaptive) có MRR VAL cao nhất (adaptive tính ngoài-fold).
    Test chỉ được ĐỌC cho cấu hình đã chọn. Đây là con số chính đưa vào báo cáo."""
    import pandas as pd
    fl = pd.DataFrame(fusion_log)
    fl = fl[fl.ablation == "none"]          # ablation là phân tích, không phải ứng viên mô hình cuối
    sel = pd.concat([fl.groupby("ablation").cv_global.mean().rename("val_MRR").reset_index().assign(mode="global"),
                     fl.groupby("ablation").cv_adaptive.mean().rename("val_MRR").reset_index().assign(mode="adaptive")]
                    ).sort_values("val_MRR", ascending=False).reset_index(drop=True)
    best = sel.iloc[0]
    tab = results_table(all_res)
    pat = r"^CLIP\+Graph( \[|$)" if best["mode"] == "global" else r"^CLIP\+Graph \(adaptive\)"
    test = tab[(tab.ablation == best.ablation) & (tab.model.str.match(pat) | tab.model.str.match(r"^CLIP( \[|$)"))]
    print(f"→ Mô hình cuối (chọn trên VAL): đồ thị='{best.ablation}', kết hợp={best['mode']} "
          f"(VAL MRR {best.val_MRR:.4f})")
    return sel, test

In [ ]:
# =====================================================================================
# v9/v10 — NHÁNH KHỚP TRỰC TIẾP TRUY VẤN ↔ NODE ĐỒ THỊ (learned query → scene-graph matcher)
#   Vì sao: ở v8, một ánh xạ TUYẾN TÍNH truy vấn → vector node của video (trên CPU) đã cho MRR
#   cao hơn cả hệ GNN ⇒ GNN chưa tận dụng hết thông tin trong node. Nhánh này học (chỉ trên TRAIN)
#   một ridge (dạng đối ngẫu) từ đặc trưng truy vấn [TF-IDF 1–2 gram ⊕ CLIP-text] sang vector node
#   (đối tượng / hành động / bối cảnh / bộ ba, trọng số IDF) của ảnh–video đúng. Điểm khớp
#   s_match(q, v) = ŷ_q · y_v. Dùng s_match để (1) bổ sung ứng viên vào top-K (γ) và
#   (2) cộng vào điểm cuối (β). λ_ridge, γ, β chọn trên VAL; α, λ của CLIP+Graph giữ nguyên.
#   Ghi thêm 2 mô hình:  CLIP+Graph+Match  (đầy đủ)  và  CLIP+Match  (không GNN/path — đối chứng
#   để biết GNN còn đóng góp gì khi đã có nhánh khớp). Mọi ablation/đối chứng đồ thị (xáo node,
#   bỏ hành động, bỏ quan hệ…) tự áp vào nhánh khớp vì vector node lấy từ CHÍNH đồ thị đó.
# =====================================================================================
import time, traceback
import pandas as pd
import scipy.sparse as _sps
from sklearn.feature_extraction.text import TfidfVectorizer

MATCH_ON = True
MATCH_RIDGE = (0.3, 1.0, 3.0)
MATCH_GAMMA = (0.0, 0.25, 0.5, 1.0, 2.0)
MATCH_BETA = tuple(float(b) for b in np.round(np.arange(0, 1.01, 0.1), 2))
_MATCH_CORE, MATCH_LOG = {}, []


def _l2n(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)


def _match_core(eng, train_pos):
    tf = eng.txt_feat.cpu().numpy() if isinstance(eng.txt_feat, torch.Tensor) else np.asarray(eng.txt_feat)
    texts = [q["text"] for q in eng.queries]
    tr = np.asarray(train_pos, np.int64)
    key = (len(texts), len(tr), tf.shape[1], round(float(tf[0, 0]), 6), texts[0][:80], int(tr[:50].sum()))
    if key in _MATCH_CORE:
        return _MATCH_CORE[key]
    t0 = time.time()
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vec.fit([texts[i] for i in tr])
    Xt = vec.transform(texts).tocsr()
    Xc = _l2n(tf.astype(np.float32))
    K_all = (Xt @ Xt[tr].T).toarray().astype(np.float32) + Xc @ Xc[tr].T          # [n_img × n_tr]
    Ktr = torch.tensor(K_all[tr], dtype=torch.float64, device=DEVICE)
    ev, U = torch.linalg.eigh((Ktr + Ktr.T) / 2)
    M = {lr: ((U * (1.0 / (ev.clamp_min(0) + lr))) @ U.T).float() for lr in MATCH_RIDGE}
    del Ktr, U
    core = dict(vec=vec, tr=tr, Xt_tr=Xt[tr], Xc_tr=Xc[tr], M=M, K_all=torch.tensor(K_all, device=DEVICE))
    _MATCH_CORE.clear(); _MATCH_CORE[key] = core
    print(f"    [match] lõi ridge: {len(tr)} truy vấn train, TF-IDF {Xt.shape[1]} chiều, {time.time() - t0:.0f}s")
    return core


MATCH_TRI_W = (0.0, 0.25, 0.5, 1.0)        # v10: trọng số khối node BỘ BA trong vector node (chọn trên VAL)


def _idf_l2(Y, tr):
    df_ = np.asarray((Y[tr] > 0).sum(0)).ravel()
    Y = Y.multiply((np.log((1 + len(tr)) / (1 + df_)) + 1)[None, :]).tocsr()
    nrm = np.sqrt(np.asarray(Y.multiply(Y).sum(1)).ravel())
    return (_sps.diags(1.0 / (nrm + 1e-8)) @ Y).tocsr(), (nrm > 0).astype(np.float32)


def _match_G(graph, tr):
    """Trả về (G_obj, G_tri, có_bộ_ba[train], có_bộ_ba[tất cả]); y = [a, w·b] / ‖·‖ với a = node đối tượng/
    hành động/bối cảnh, b = node bộ ba (mỗi khối IDF + L2 riêng). v9 gộp chung ⇒ hàng nghìn node bộ ba
    thưa lấn át node đối tượng (match-only VAL trên AG: 0.32 có bộ ba vs 0.67 không bộ ba)."""
    n_img, ro, co, rt, ct = graph.n_img, [], [], [], []
    tri = getattr(graph, "tri2id", None) or {}
    n_obj_cols = max(1, graph.n_nodes - n_img)
    for i in range(n_img):
        ids = sorted(set(graph.img_objects[i])); ro += [i] * len(ids); co += [j - n_img for j in ids]
        if tri:
            t = sorted({tri[k] for k, _ in graph.records[i].get("img_triples", []) if k in tri})
            rt += [i] * len(t); ct += [j - n_img for j in t]
    A, _ = _idf_l2(_sps.csr_matrix((np.ones(len(ro), np.float32), (ro, co)), shape=(n_img, n_obj_cols)), tr)
    B, hb = _idf_l2(_sps.csr_matrix((np.ones(len(rt), np.float32), (rt, ct)), shape=(n_img, n_obj_cols)), tr)
    T = lambda X: torch.tensor((X[tr] @ X.T).toarray().astype(np.float32), device=DEVICE)
    return (T(A), T(B), torch.tensor(hb[tr], device=DEVICE), torch.tensor(hb, device=DEVICE))


def _G_w(Gs, w):
    Ga, Gb, ht, hv = Gs
    if w == 0:
        return Ga
    nt, nv = torch.sqrt(1 + w * w * ht), torch.sqrt(1 + w * w * hv)
    return (Ga + w * w * Gb) / (nt[:, None] * nv[None, :])


def _match_scores(core, G, lr, q_idx):
    qi = torch.as_tensor(np.asarray(q_idx, np.int64), device=DEVICE)
    return (core["K_all"][qi] @ core["M"][lr]) @ G                                     # [B × n_img]


def _match_text_scores(core, G, lr, text, clipfeat):
    k = (core["vec"].transform([text]) @ core["Xt_tr"].T).toarray().astype(np.float32)[0] + \
        _l2n(np.asarray(clipfeat, np.float32).reshape(1, -1))[0] @ core["Xc_tr"].T
    return ((torch.tensor(k, device=DEVICE)[None] @ core["M"][lr]) @ G)[0].cpu().numpy()


def _pick(S, cand):
    c = torch.as_tensor(np.asarray(cand, np.int64), device=DEVICE)
    return torch.gather(S, 1, c).cpu().numpy()


def _match_cands(eng, q_idx, gallery, S, gamma, k):
    gallery = np.asarray(gallery)
    sim = (eng.txt_feat[np.asarray(q_idx)] @ eng.img_feat[gallery].T).numpy()
    if gamma > 0:
        sim = znorm_per_row(sim) + gamma * znorm_per_row(S[:, torch.as_tensor(gallery, device=DEVICE)].cpu().numpy())
    k = min(k, len(gallery))
    return gallery[np.argpartition(-sim, k - 1, axis=1)[:, :k]]


def _graph_scores(model, eng, h, q_idx, cand):
    q_idx, cand = np.asarray(q_idx), np.asarray(cand)
    qf = eng.txt_feat[q_idx]
    s_clip = torch.einsum("bd,bkd->bk", qf, eng.img_feat[torch.from_numpy(cand)]).numpy()
    pm = torch.tensor(eng.path_feats(cand, q_idx)).to(DEVICE)
    s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    z = znorm_per_row
    return z(s_clip), z(s_gnn), z(s_path)


def _final(tag, b, sc, sg, spp, sm, alpha, lam):
    return (1 - b) * combine(sc, sg, spp, alpha, lam) + b * sm if tag == "gm" else (1 - b) * sc + b * sm


@torch.no_grad()
def _match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res):
    t0 = time.time()
    tr = parts["train"]["q_idx"]
    core = _match_core(eng, tr)
    Gs = _match_G(eng.graph, tr)
    pv_ = parts["val"]; gv_ = np.asarray(pv_["gold_pos"]); gal_ = np.tile(np.asarray(pv_["img_idx"]), (len(gv_), 1))
    tri_sel = {}
    for w in (MATCH_TRI_W if bool(getattr(eng.graph, "tri2id", None)) else (0.0,)):
        Gw = _G_w(Gs, w)
        for lr in MATCH_RIDGE:
            tri_sel[(w, lr)] = float(reciprocal_ranks(gal_, _pick(_match_scores(core, Gw, lr, pv_["q_idx"]), gal_), gv_).mean())
    w_tri = max(tri_sel, key=tri_sel.get)[0]          # chọn trên VAL bằng match-only (rẻ, không cần GNN)
    G = _G_w(Gs, w_tri); del Gs
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    K, z = CFG.rerank_k, znorm_per_row
    pv, pt = parts["val"], parts["test"]
    gv = np.asarray(pv["gold_pos"])
    best = {"gm": (-1.0, None), "cm": (-1.0, None)}
    val_match_only = {}
    for lr in MATCH_RIDGE:
        Sv = _match_scores(core, G, lr, pv["q_idx"])
        val_match_only[lr] = float(reciprocal_ranks(np.tile(np.asarray(pv["img_idx"]), (len(gv), 1)),
                                                    _pick(Sv, np.tile(np.asarray(pv["img_idx"]), (len(gv), 1))), gv).mean())
        for gm in MATCH_GAMMA:
            cand = _match_cands(eng, pv["q_idx"], pv["img_idx"], Sv, gm, K)
            sc, sg, spp = _graph_scores(model, eng, h, pv["q_idx"], cand)
            sm = z(_pick(Sv, cand))
            for b in MATCH_BETA:
                for tag in ("gm", "cm"):
                    m = float(reciprocal_ranks(cand, _final(tag, b, sc, sg, spp, sm, alpha, lam), gv).mean())
                    if m > best[tag][0] + 1e-9:
                        best[tag] = (m, (lr, gm, b))
    gt, groups = np.asarray(pt["gold_pos"]), np.array(pt["groups"])
    rr_clip = PER_QUERY[(ab, seed)]["rr_clip"]
    for tag, mname, suffix in (("gm", "CLIP+Graph+Match", "+match"), ("cm", "CLIP+Match", "+clipmatch")):
        lr, gm, b = best[tag][1]
        St = _match_scores(core, G, lr, pt["q_idx"])
        cand = _match_cands(eng, pt["q_idx"], pt["img_idx"], St, gm, K)
        sc, sg, spp = _graph_scores(model, eng, h, pt["q_idx"], cand)
        fin = _final(tag, b, sc, sg, spp, z(_pick(St, cand)), alpha, lam)
        res[mname] = retrieval_metrics(cand, fin, gt)
        for grp in ("direct", "indirect"):
            m = groups == grp
            if m.sum() >= 5:
                res[f"{mname} [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], fin[m], gt[m])
        PER_QUERY[(ab + suffix, seed)] = dict(rr=reciprocal_ranks(cand, fin, gt), rr_clip=rr_clip, groups=groups, lam=lam)
    # ---- bộ quan hệ khó + truy vấn phản thực tế (tham số giữ nguyên từ VAL) ----
    key = globals().get("REL_GAL_KEY")
    name = globals().get("REL_RENAME", {}).get(ab, ab)
    if key is not None and key in globals().get("REL_GALLERIES", {}) and (name, seed) in REL_PQ[key]:
        for tag, suffix in (("gm", "+match"), ("cm", "+clipmatch")):
            lr, gm, b = best[tag][1]
            rr = []
            for q, cand in REL_GALLERIES[key]:
                cand = np.asarray(cand, np.int64)[None]
                sc, sg, spp = _graph_scores(model, eng, h, np.array([int(q)]), cand)
                sm = z(_pick(_match_scores(core, G, lr, [int(q)]), cand))
                fin = _final(tag, b, sc, sg, spp, sm, alpha, lam)[0]
                rr.append(1.0 / (1 + int((fin > fin[0]).sum())))
            base_ = REL_PQ[key][(name, seed)]
            REL_PQ[key][(name + suffix, seed)] = dict(rr=np.array(rr), rr_clip=base_["rr_clip"], groups=base_["groups"], lam=lam)
        if key in globals().get("REL_CF", {}):
            for tag, suffix in (("gm", "+match"), ("cm", "+clipmatch")):
                lr, gm, b = best[tag][1]
                rows = []
                for x in REL_CF[key]:
                    cand = np.asarray(x["cand"], np.int64); row = {}
                    for t in ("text", "cf"):
                        qf = torch.tensor(REL_CF_FEATS[x[t]][None])
                        cids = [n for n, _, _ in parse_query(x[t], eng.graph)]
                        s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
                        pm = torch.tensor(path_score_matrix(eng.graph, cids, cand)[None]).to(DEVICE)
                        s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
                        s_path = model.path_score(pm).cpu().numpy()
                        sm = _match_text_scores(core, G, lr, x[t], REL_CF_FEATS[x[t]])[cand][None]
                        fin = _final(tag, b, z(s_clip), z(s_gnn), z(s_path), z(sm), alpha, lam)[0]
                        row[f"rank_{t}"] = 1 + int((fin > fin[0]).sum())
                    rows.append(row)
                REL_CF_RES[key][(name + suffix, seed)] = pd.DataFrame(rows)
    MATCH_LOG.append(dict(n_img=eng.graph.n_img, ablation=name, seed=seed, w_tri=w_tri,
                          ridge_gm=best["gm"][1][0], gamma_gm=best["gm"][1][1], beta_gm=best["gm"][1][2], val_gm=best["gm"][0],
                          ridge_cm=best["cm"][1][0], gamma_cm=best["cm"][1][1], beta_cm=best["cm"][1][2], val_cm=best["cm"][0],
                          **{f"val_match_only_r{lr}": v for lr, v in val_match_only.items()}, sec=time.time() - t0))
    print(f"    [match] {name} seed {seed}: test MRR CLIP+Graph+Match {res['CLIP+Graph+Match']['MRR']:.4f} "
          f"(λr,γ,β={best['gm'][1]}) | CLIP+Match {res['CLIP+Match']['MRR']:.4f} (λr,γ,β={best['cm'][1]}) "
          f"| match-only val {max(val_match_only.values()):.4f} | w_tri={w_tri} | {time.time() - t0:.0f}s")


def match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res):
    if not globals().get("MATCH_ON", False):
        return
    try:
        _match_hook(ab, seed, model, eng, parts, raw, alpha, lam, res)
    except Exception as ex:                       # nhánh phụ — không được làm hỏng thí nghiệm chính
        print("    [match] LỖI — bỏ qua nhánh khớp ở lần chạy này:", repr(ex)); traceback.print_exc()


## 1.14 Phân tích 5 thành công / 5 thất bại + fidelity trên ≥10 truy vấn

In [ ]:
def analyse_cases(art, records, n_success=5, n_fail=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, (cand, sc, sg, sp_) = art["parts"]["test"], art["raw"]
    full = combine(sc, sg, sp_, art["alpha"], art["lam"])
    gold = np.array(part["gold_pos"])

    def rank_of(S):
        o = np.argsort(-S, 1); r = np.take_along_axis(cand, o, 1)
        hit = r == gold[:, None]
        return np.where(hit.any(1), hit.argmax(1) + 1, 10**9)

    r_clip, r_full = rank_of(sc), rank_of(full)
    delta = r_clip - r_full                                   # >0 = đồ thị giúp
    succ = [i for i in np.argsort(-delta) if r_full[i] <= 10][:n_success]
    fail = [i for i in np.argsort(delta) if r_clip[i] < r_full[i]][:n_fail]

    cases = []
    for kind, idxs in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idxs:
            qi = part["q_idx"][i]
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            expl, s, det = explain_and_score(model, graph, int(gold[i]), cids)
            cases.append(dict(kind=kind, query=eng.queries[qi]["text"][:110],
                              group=eng.queries[qi]["group"],
                              rank_clip=int(r_clip[i]), rank_graph=int(r_full[i]),
                              explanation=expl, detail=det))
    return cases


def fidelity_study(art, n_queries=CFG.n_explain_queries, top_m_paths_min=CFG.top_m_paths):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part = art["parts"]["test"]
    rows = []
    for i in range(len(part["q_idx"])):
        qi = part["q_idx"][i]
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        expl, s, det = explain_and_score(model, graph, int(part["gold_pos"][i]), cids)
        if det["n_paths"] == 0:
            continue
        if det["n_paths"] <= top_m_paths_min:   # quá ít đường đi ⇒ giải thích = toàn bộ, đối chứng vô nghĩa
            continue
        rows.append(dict(group=eng.queries[qi]["group"], **det,
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"]))
        if len(rows) >= n_queries:
            break
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không có truy vấn nào có đường đi — kiểm tra lại bước dựng đồ thị.")
        return df, {}
    summary = dict(
        n_queries=len(df),
        mean_paths=float(df.n_paths.mean()),
        fidelity_minus=float(df.drop_explanation.mean()),
        fidelity_minus_random_control=float(df.drop_random.mean()),
        fidelity_plus_gap=float((df.score_full - df.score_only_explanation).mean()),
        sparsity=float(df.sparsity.mean()),
        validity=float(df.validity.mean()),
    )
    summary["necessity_ratio"] = (summary["fidelity_minus"] /
                                  max(summary["fidelity_minus_random_control"], 1e-9))
    return df, summary

In [ ]:
@torch.no_grad()
def final_rank_fidelity(art, n_queries=50, top_m=CFG.top_m_paths, n_rand=10):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, alpha, lam = art["parts"]["test"], art["alpha"], art["lam"]
    model.eval()
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    cand_all, clip_all = eng.clip_candidates(part["q_idx"], part["img_idx"], CFG.rerank_k)
    rng = np.random.RandomState(0)
    z = znorm_per_row
    sc = lambda ps: gain * float((paths_to_counts(ps) * w).sum())
    rows = []

    for i, qi in enumerate(part["q_idx"]):
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        gold = int(part["gold_pos"][i])
        cand = cand_all[i]
        hit = np.where(cand == gold)[0]
        if not cids or len(hit) == 0:
            continue
        paths = enumerate_paths(graph, gold, cids)
        if len(paths) <= top_m:          # ít đường đi quá thì đối chứng ngẫu nhiên vô nghĩa
            continue
        j = int(hit[0])
        pm = eng.path_feats(cand[None], np.array([qi]))[0]
        sp_raw = gain * (pm * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[qi]].to(DEVICE),
                                torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_clip = clip_all[i][None]

        def rank_with(kept):
            sp = sp_raw.copy()
            sp[j] = sc(kept)
            fin = combine(z(s_clip), z(s_gnn), z(sp[None]), alpha, lam)[0]
            return int((fin > fin[j]).sum()) + 1

        full = sc(paths)
        contrib = np.array([full - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib)[:top_m].tolist())

        r_rand = np.mean([
            rank_with([p for k, p in enumerate(paths)
                       if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
            for _ in range(n_rand)])
        rows.append(dict(group=eng.queries[qi]["group"], n_paths=len(paths),
                         rank_full=rank_with(paths),
                         rank_drop_expl=rank_with([p for k, p in enumerate(paths) if k not in top]),
                         rank_drop_rand=float(r_rand),
                         rank_only_expl=rank_with([paths[k] for k in top])))
        if len(rows) >= n_queries:
            break

    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không truy vấn nào có > top_m đường đi.")
        return df, {}
    summ = dict(n_queries=len(df),
                rank_full=df.rank_full.mean(),
                rank_after_drop_explanation=df.rank_drop_expl.mean(),
                rank_after_drop_random=df.rank_drop_rand.mean(),
                rank_with_only_explanation=df.rank_only_expl.mean(),
                frac_rank_worse_when_drop_expl=float((df.rank_drop_expl > df.rank_full).mean()),
                frac_rank_worse_when_drop_rand=float((df.rank_drop_rand > df.rank_full).mean()))
    return df, summ

## 1.15 Demo: nhập truy vấn → top-k + giải thích

In [ ]:
@torch.no_grad()
def demo_query(text, art, k=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    gallery = art["parts"]["test"]["img_idx"]
    hits = parse_query(text, graph)
    print(f"\nTruy vấn: {text!r}")
    print(f"Khái niệm nhận ra: {[h[1] for h in hits] or '(không có)'}")
    cids = [n for n, _, _ in hits]

    # mã hoá truy vấn tự do: dùng CLIP nếu có, nếu không lấy caption gần nhất làm proxy
    q_emb = encode_free_text(text, eng)
    sim = (q_emb @ eng.img_feat[gallery].T).numpy()[0]
    top = np.argsort(-sim)[:CFG.rerank_k]
    cand = gallery[top]
    pm = torch.tensor(np.stack([_path_feats_free(graph, cand, cids)])).to(DEVICE)
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    s_gnn = model.gnn_score(h, q_emb.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    s = combine(znorm_per_row(sim[top][None]), znorm_per_row(s_gnn),
                znorm_per_row(s_path), art["alpha"], art["lam"])[0]
    order = np.argsort(-s)[:k]
    for rank, j in enumerate(order, 1):
        img_i = int(cand[j])
        rec = eng.records[img_i]
        expl, _, det = explain_and_score(model, graph, img_i, cids)
        print(f"\n  #{rank}  image_id={rec['image_id']}  score={s[j]:.4f}")
        print(f"      caption: {rec['caption'][:90]}")
        print(f"      đối tượng: {rec['objects'][:8]}")
        if expl:
            print(f"      GIẢI THÍCH ({det['n_paths']} đường đi, hiển thị {len(expl)}):")
            for e in expl:
                print(f"        • {e['text']}   [đóng góp {e['contribution']:+.4f}]")
        else:
            print("      (không có đường đi đồ thị — xếp hạng chỉ dựa trên CLIP)")


def _proxy_text(text, eng):
    toks = set(_norm(text).split())
    best, bi = -1, 0
    for i, q in enumerate(eng.queries):
        ov = len(toks & set(_norm(q["text"]).split()))
        if ov > best:
            best, bi = ov, i
    return eng.txt_feat[bi:bi + 1]


def encode_free_text(text, eng):
    """CLIP thật chỉ dùng khi đặc trưng ảnh cũng là CLIP thật; synthetic thì dùng proxy."""
    if dataset_name == "synthetic":
        return _proxy_text(text, eng)
    try:
        import open_clip
        if not hasattr(encode_free_text, "_m"):
            m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model,
                                                            pretrained=CFG.clip_weights)
            encode_free_text._m = m.to(DEVICE).eval()
            encode_free_text._t = open_clip.get_tokenizer(CFG.clip_model)
        with torch.no_grad():
            f = encode_free_text._m.encode_text(encode_free_text._t([text]).to(DEVICE))
        return F.normalize(f, dim=-1).float().cpu()
    except Exception:
        return _proxy_text(text, eng)

def _path_feats_free(graph, cand, cids):
    out = np.zeros((len(cand), len(PATTERNS)), np.float32)
    if not cids:
        return out
    cols = np.array(sorted(set(cids)))
    for p, name in enumerate(PATTERNS):
        out[:, p] = np.asarray(graph.R[name][cand][:, cols].sum(1)).ravel()
    return np.log1p(out)

## 1.16 Self-test — chạy trước khi tin bất kỳ con số nào

Ba kiểm tra này chính là ba câu hỏi vấn đáp dễ bị hỏi nhất. Chạy được và in ✓ thì
nhóm có bằng chứng ngay tại chỗ.

In [ ]:
def self_test(records, split, img_feat, txt_feat):
    import copy
    ok = []

    # (1) RÒ RỈ — cấy một quan hệ chỉ tồn tại ở ảnh TEST, đồ thị train không được có
    rec2 = copy.deepcopy(records)
    pos = {r["image_id"]: i for i, r in enumerate(rec2)}
    A, B = "zzsecretobja", "zzsecretobjb"
    for iid in split["test"][:40]:
        r = rec2[pos[iid]]
        r["objects"] = sorted(set(r["objects"]) | {A, B})
        r["relations"] = list(r["relations"]) + [(A, "juggles", B)]
    gt = MultimodalGraph(rec2, split["train"])
    ga = MultimodalGraph(rec2, split["train"] + split["test"])
    e = lambda G, M: (M[G.obj2id[A], G.obj2id[B]] != 0) if A in G.obj2id else False
    c1 = (not e(gt, gt.A_scene)) and e(ga, ga.A_scene)
    c2 = (not e(gt, gt.A_related)) and e(ga, ga.A_related)
    ok.append(("Cạnh scene + đồng xuất hiện chỉ xây từ train", c1 and c2))

    # (2) TRUNG THỰC — điểm đường đi tính bằng ma trận phải khớp liệt kê từng đường đi
    graph = MultimodalGraph(records, split["train"])
    qs = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, qs)
    bad = n = 0
    for qi in range(min(300, len(qs))):
        cids = [x for x, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        n += 1
        m = eng.path_feats(np.array([[qi]]), np.array([qi]))[0, 0]
        v = paths_to_counts(enumerate_paths(graph, qi, cids))
        bad += int(not np.allclose(m, v, atol=1e-5))
    ok.append((f"Điểm đường đi khớp giải thích liệt kê ({n - bad}/{n} cặp)", bad == 0))

    # (3) SPLIT — không chồng lấn và tái lập được
    s2 = make_split(records, seed=CFG.split_seed)
    ok.append(("Split tái lập được từ seed", s2["sha1"] == split["sha1"]))

    # (4) v5 — bộ ba cấp ảnh: nhãn chỉ từ caption TRAIN, ảnh train nhận dự đoán ngoài-fold
    if globals().get("TRIPLE_INFO") and any(r.get("img_triples") for r in records):
        ok.append(("Bộ ba cấp ảnh: nhãn huấn luyện chỉ từ ảnh TRAIN, ảnh train dùng dự đoán ngoài-fold",
                   bool(TRIPLE_INFO.get("train_only")) and bool(TRIPLE_INFO.get("oof"))))
        g3 = MultimodalGraph(records, split["train"])
        ok.append((f"Node bộ ba có trong đồ thị ({g3.n_tri} node, {g3.A_triple.nnz} cạnh ảnh–bộ ba)",
                   g3.n_tri > 0 and g3.A_triple.nnz > 0))

    print()
    for name, passed in ok:
        print(f"  {'✓' if passed else '✗ THẤT BẠI'}  {name}")
    assert all(p for _, p in ok), "Self-test không đạt — dừng lại và sửa trước khi chạy tiếp."
    return True

---
## 1.17 Thực thi (Flickr8k)

Từ đây trở xuống là phần chạy thật. Muốn chạy nhanh để thử, giảm `CFG.n_images`.

In [ ]:
# ============ FLICKR8K: nạp dữ liệu, rút đối tượng/quan hệ, CLIP, split ============
import nltk
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    try: nltk.download(_pkg, quiet=True)
    except Exception: pass
from nltk.stem import WordNetLemmatizer

_LEM = WordNetLemmatizer(); _LEM_CACHE = {}
try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()          # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
except Exception as e:
    print("⚠ Không tải được POS tagger → dùng cách rút danh từ thô:", type(e).__name__)
    _TAGGER = None

def _lemma(w):
    r = _LEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "n")
        except Exception: r = w[:-1] if (w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[w] = r
    return r

NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())


def extract_objects_relations(captions):
    """Danh từ NHÌN THẤY ĐƯỢC (lọc WordNet) → lớp đối tượng. Quan hệ = (danh từ, động từ/giới từ,
    danh từ kế tiếp). Heuristic — nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    objs, rels = set(), set()
    for cap in captions:
        toks = re.findall(r"[A-Za-z]+", cap)
        if not toks:
            continue
        if _TAGGER is not None:
            tags = _TAGGER.tag(toks)
        else:
            tags = [(t, "NN" if len(t) > 2 and t.lower() not in STOP else "DT") for t in toks]
        last_n, preds = None, []
        for w, t in tags:
            wl = w.lower()
            if t in ("NN", "NNS") and len(wl) > 2 and wl not in NOT_OBJECTS and wl not in STOP:
                n = _lemma(wl)
                if n in NOT_OBJECTS or not is_visual_noun(n):
                    continue                     # tính từ/động từ bị tag nhầm NN: pink, walking, stand...
                objs.add(n)
                if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                    rels.add((last_n, " ".join(preds), n))
                last_n, preds = n, []
            elif (t.startswith("VB") or t in ("IN", "TO")) and wl not in _SKIP_P and last_n is not None:
                preds.append(wl)
    return objs, rels


def load_flickr8k():
    """Chỉ rút nhãn-từ-caption (cap_objects/cap_relations). objects/relations THẬT dùng cho
    đồ thị được gán sau bởi apply_flickr_annotation (tagger hoặc oracle)."""
    from datasets import load_dataset as hf_load
    dsd = hf_load("jxie/flickr8k")
    records, parts, iid = [], [], 0
    for hf_name, tag in (("train", "train"), ("validation", "val"), ("test", "test")):
        d = dsd[hf_name]; parts.append(d)
        for row in d.remove_columns("image"):
            allc = [str(row[f"caption_{k}"]).strip() for k in range(5)]
            objs, rels = extract_objects_relations(allc[1:])   # caption 1..4; caption_0 là truy vấn
            records.append(dict(image_id=iid, caption=allc[0], captions=allc,
                                cap_objects=sorted(objs), cap_relations=sorted(rels),
                                objects=[], relations=[], url="", split=tag))
            iid += 1
    print(f"Flickr8k: {len(records)} ảnh | {np.mean([len(r['cap_objects']) for r in records]):.1f} danh từ/ảnh | "
          f"{np.mean([len(r['cap_relations']) for r in records]):.1f} quan hệ/ảnh (rút từ caption)")
    for r in records[:3]:
        print("  •", r["caption"], "→", r["cap_objects"][:8], r["cap_relations"][:3])
    return records, parts


def encode_flickr8k(records, parts):
    cache = CFG.out_dir / "cache" / f"clip_flickr8k_{CFG.clip_model}_{CFG.clip_weights}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    img, txt = [], []
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), CFG.clip_bs):
                ims = d[s:s + CFG.clip_bs]["image"]
                x = torch.stack([preprocess(im.convert("RGB")) for im in ims]).to(DEVICE)
                img.append(F.normalize(model.encode_image(x), dim=-1).float().cpu().numpy())
        caps = [r["caption"] for r in records]
        for s in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[s:s + CFG.clip_bs]).to(DEVICE)
            txt.append(F.normalize(model.encode_text(tok), dim=-1).float().cpu().numpy())
    del model; torch.cuda.empty_cache()
    img, txt = np.concatenate(img), np.concatenate(txt)
    assert len(img) == len(records) == len(txt)
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


# =====================================================================================
# NHÃN ẢNH BẰNG CLIP ZERO-SHOT (thay cho nhãn rút từ caption của chính ảnh)
#   - Từ vựng: danh từ nhìn thấy được trong caption TRAIN, tần suất ≥ min_obj_freq.
#   - Điểm: cos(ảnh, prompt-ensemble); chuẩn hoá theo từng từ bằng mean/std trên ảnh TRAIN.
#   - (cách chuẩn hoá, k) chọn trên VAL theo F1 so với danh từ trong caption val.
#   - Caption của ảnh val/test KHÔNG bao giờ vào đồ thị; caption test chỉ dùng để BÁO CÁO P/R/F1.
# =====================================================================================
@torch.no_grad()
def clip_text_features(texts, bs=512):
    import open_clip
    if not hasattr(clip_text_features, "_m"):
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        clip_text_features._m = m.to(DEVICE).eval()
        clip_text_features._t = open_clip.get_tokenizer(CFG.clip_model)
    m, tok = clip_text_features._m, clip_text_features._t
    out = [F.normalize(m.encode_text(tok(texts[s:s + bs]).to(DEVICE)), dim=-1).float().cpu().numpy()
           for s in range(0, len(texts), bs)]
    return np.concatenate(out)


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1); R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def clip_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vset = set(vocab)

    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = img_feat @ T.T                                          # [N, V]
    mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên train
    scores = {"raw": S, "z": (S - mu) / sd, "raw+z": S / S[tr].std() + (S - mu) / sd}

    def tags(Sc, idx, k):
        top = np.argpartition(-Sc[idx], k, axis=1)[:, :k]
        return [{vocab[j] for j in row} for row in top]
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]

    grid, best = [], None
    for mode, Sc in scores.items():
        for k in CFG.tag_k_grid:
            P, R, F1 = _prf(tags(Sc, va, k), truth(va))
            grid.append(dict(norm=mode, k=k, val_P=P, val_R=R, val_F1=F1))
            if best is None or F1 > best[0]:
                best = (F1, mode, k)
    _, mode, k = best
    all_tags = tags(scores[mode], np.arange(len(records)), k)
    P, R, F1 = _prf([all_tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm=mode, k=k, test_P=P, test_R=R, test_F1=F1, grid=grid)
    return all_tags, info


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    """Gán records[i]['objects'] / ['relations'] dùng cho đồ thị.
    relations chỉ giữ cho ảnh TRAIN (dùng để xây cạnh scene thống kê)."""
    mode = mode or CFG.flickr_annot
    tr = set(split["train"])
    info = dict(annotation=mode)
    if mode == "tagger":
        global TAG_INFO
        if not all("tags" in r for r in records) or "TAG_INFO" not in globals():
            tags, TAG_INFO = clip_tagger(records, split, img_feat, text_encoder)
            for r, t in zip(records, tags):
                r["tags"] = sorted(t)
        info.update(TAG_INFO)
    for r in records:
        r["objects"] = list(r["tags"]) if mode == "tagger" else list(r["cap_objects"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    msg = f"Nhãn ảnh = {mode}"
    if mode == "tagger":
        msg += (f" | vocab={info['vocab_size']} | chuẩn hoá={info['norm']}, k={info['k']} (chọn trên val)"
                f" | TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    print(msg)
    return info


# Split chính thức của HF; bản ngẫu nhiên cũ giữ lại cho trường hợp synthetic.
if "_make_split_random" not in globals():
    _make_split_random = make_split

def make_split(records, seed=CFG.split_seed):
    if not all("split" in r for r in records):
        return _make_split_random(records, seed)
    split = {k: sorted(r["image_id"] for r in records if r["split"] == k) for k in ("train", "val", "test")}
    split["seed"] = "official-hf-split"
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split chính thức HF (sha1={split['sha1']}): train={len(split['train'])} "
          f"val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])) and not (set(split["val"]) & set(split["test"]))
    return split


# Truy vấn Flickr có số nhiều ("dogs") → phải lemmatize như lúc rút đối tượng.
def parse_query(text, graph):
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    # v5: bộ ba (s, p, o) tách từ truy vấn → node bộ ba cấp ảnh (mẫu P6)
    if getattr(graph, "n_tri", 0) and "query_triples" in globals():
        for key in query_triples(text):
            nid = graph.tri2id.get(key)
            if nid is not None and nid not in seen:
                seen.add(nid); hits.append((nid, " ".join(key), " ".join(key)))
    return hits

## 1.2b Quan hệ cấp ảnh (v5)

Lần chạy trước cho thấy `none ≈ no_relations ≈ random_edges` trên cả ba bộ dữ liệu. Lý do: mọi cạnh quan hệ (scene / related / isa) là **thống kê cấp lớp** gộp từ train, nên hai ảnh cùng chứa `dog` nhận đúng cùng một tập đường đi và quan hệ không phân biệt được ứng viên.

Bản này thêm **node bộ ba cấp ảnh** ⟨chủ thể, vị từ, đối tượng⟩: mỗi ảnh có tập bộ ba riêng, do một linear probe (khởi tạo từ văn bản CLIP, học trên caption **train**, ảnh train nhận dự đoán ngoài-fold) dự đoán từ đặc trưng CLIP của chính ảnh. Mẫu đường đi mới `P6: ảnh –thể_hiện→ ⟨s p o⟩` khớp với bộ ba tách từ truy vấn và vẫn cộng tính, nên giải thích và fidelity giữ nguyên cơ chế. Nhãn đối tượng cũng chuyển từ CLIP zero-shot sang probe cùng kiểu.

Đối chứng mới: `no_triples` (= `none` của bản trước), `shuffled_triples` (ảnh nhận bộ ba của ảnh khác). Đánh giá mới: **bộ quan hệ khó**, gallery gồm ảnh đúng và các ảnh test có cùng đối tượng nhưng không có quan hệ của truy vấn (theo caption test, chỉ dùng làm nhãn chấm).


In [ ]:
# =====================================================================================
# 1.2b  QUAN HỆ CẤP ẢNH (bản v5) — vì sao cần
#   Lần chạy trước: 'none' ≈ 'no_relations' ≈ 'random_edges' trên cả Flickr8k, AG, MSR-VTT.
#   Nguyên nhân: mọi cạnh quan hệ (scene / related / isa) là thống kê CẤP LỚP gộp từ train —
#   hai ảnh cùng có 'dog' nhận ĐÚNG cùng tập đường đi 'dog → …', nên quan hệ không phân biệt
#   được ứng viên; nó chỉ là một kiểu mở rộng truy vấn mà CLIP đã làm tốt hơn.
#   Bản này thêm:
#     (1) Nhãn đối tượng bằng LINEAR PROBE khởi tạo từ văn bản CLIP (học trên caption TRAIN,
#         ảnh train dùng dự đoán NGOÀI-FOLD) thay cho zero-shot (F1 test cũ 0.25).
#     (2) Node BỘ BA (chủ thể, vị từ, đối tượng) CẤP ẢNH: mỗi ảnh có tập bộ ba riêng do probe
#         dự đoán từ đặc trưng CLIP của CHÍNH ẢNH (không dùng caption val/test).
#         Mẫu đường đi mới P6: ảnh –thể_hiện→ ⟨s p o⟩, khớp với bộ ba tách từ truy vấn.
#     (3) Đối chứng mới: no_triples (= 'none' của bản trước), shuffled_triples (bộ ba của
#         ảnh KHÁC ⇒ kiểm tra tín hiệu có thật sự gắn với ảnh không).
#     (4) Bộ đánh giá QUAN HỆ KHÓ: gallery = ảnh đúng + ảnh test có CÙNG đối tượng nhưng
#         KHÔNG có quan hệ của truy vấn (theo caption test — chỉ dùng làm NHÃN CHẤM).
# =====================================================================================
import shutil

# --- dùng lại cache CLIP của lần chạy trước nếu output cũ được Add Input ---
for _f in glob.glob("/kaggle/input/**/cache/clip_flickr8k_*.npz", recursive=True):
    _dst = CFG.out_dir / "cache" / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)
        print("Dùng lại cache CLIP từ input:", _dst.name)

PREPS = set("in on at with by for from into onto over under near behind beside across through along "
            "around against toward towards up down out off past above below between inside outside "
            "beneath underneath atop upon".split())
TRIVIAL_PREDS = {"of", "for", "by"}          # 'body of water', 'pair of shoe' không phải quan hệ thị giác
_VLEM_CACHE = {}


def _verb_lemma(w):
    r = _VLEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "v")
        except Exception: r = w
        _VLEM_CACHE[w] = r
    return r


def canon_triple(s, p, o):
    """('dog', 'running after', 'ball') → ('dog', 'run', 'ball'); ('dog', 'in', 'snow') giữ giới từ."""
    toks = p.split()
    if not toks or s == o:
        return None
    t0 = toks[0]
    cp = t0 if t0 in PREPS else _verb_lemma(t0)
    if cp in TRIVIAL_PREDS:
        return None
    return (s, cp, o)


def caption_triples(caps):
    _, rels = extract_objects_relations(caps)
    return {t for t in (canon_triple(*r) for r in rels) if t is not None}


_QT_CACHE = {}


def query_triples(text):
    """Bộ ba chuẩn hoá tách từ VĂN BẢN TRUY VẤN — cùng bộ rút quan hệ dùng cho caption train."""
    if text not in _QT_CACHE:
        _QT_CACHE[text] = sorted(caption_triples([text]))
    return _QT_CACHE[text]


# ---------------------------------------------------------------------------------
# Linear probe khởi tạo từ văn bản CLIP: logit = e^s·cos(ảnh, w_v) + b_v, w_v khởi tạo = text_v,
# phạt ‖W − T‖² để từ hiếm vẫn bám prior văn bản. Ảnh TRAIN: dự đoán NGOÀI-FOLD (k-fold).
# ---------------------------------------------------------------------------------
def _fit_probe(X, Y, T, steps=None, lr=None, wd_init=None, seed=0):
    steps = steps or CFG.probe_steps; lr = lr or CFG.probe_lr
    wd_init = CFG.probe_wd_init if wd_init is None else wd_init
    torch.manual_seed(seed)
    X, Y, T = (torch.as_tensor(a, dtype=torch.float32, device=DEVICE) for a in (X, Y, T))
    W = nn.Parameter(T.clone())
    s = nn.Parameter(torch.tensor(math.log(30.0), device=DEVICE))
    prior = Y.mean(0).clamp(1e-4, 1 - 1e-4)
    with torch.no_grad():
        b0 = torch.log(prior / (1 - prior)) - (s.exp() * (X @ T.T)).mean(0)
    b = nn.Parameter(b0)
    opt = torch.optim.Adam([dict(params=[W], lr=lr), dict(params=[b, s], lr=lr * 10)])
    for _ in range(steps):
        logit = s.exp() * (X @ W.T) + b
        loss = F.binary_cross_entropy_with_logits(logit, Y) + wd_init * (W - T).pow(2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    W, b, s = W.detach(), b.detach(), s.detach()
    return lambda Xq: torch.sigmoid(s.exp() * (torch.as_tensor(Xq, dtype=torch.float32, device=DEVICE) @ W.T)
                                    + b).cpu().numpy()


def cross_fit_probe(X, Y_tr, T, tr, folds=None, seed=0):
    """X [N, D] mọi ảnh; Y_tr [len(tr), V] nhãn CHỈ của ảnh train. Trả P [N, V]:
    ảnh train ← mô hình KHÔNG thấy ảnh đó (ngoài-fold); ảnh val/test ← mô hình học trên toàn bộ train."""
    folds = folds or CFG.probe_folds
    tr = np.asarray(tr)
    P = np.zeros((len(X), T.shape[0]), np.float32)
    order = np.random.RandomState(seed).permutation(len(tr))
    for f in np.array_split(order, folds):
        keep = np.setdiff1d(order, f)
        pred = _fit_probe(X[tr[keep]], Y_tr[keep], T, seed=seed)
        P[tr[f]] = pred(X[tr[f]])
    rest = np.setdiff1d(np.arange(len(X)), tr)
    if len(rest):
        P[rest] = _fit_probe(X[tr], Y_tr, T, seed=seed)(X[rest])
    return P


def _multi_hot(sets, vocab_index, n_rows):
    Y = np.zeros((n_rows, len(vocab_index)), np.float32)
    for i, s in enumerate(sets):
        for t in s:
            j = vocab_index.get(t)
            if j is not None:
                Y[i, j] = 1.0
    return Y


def _topk_sets(P, idx, vocab, k, tau=0.0):
    out = []
    for i in idx:
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        out.append({vocab[j] for j in top if row[j] >= tau})
    return out


def probe_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vix = {w: j for j, w in enumerate(vocab)}
    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([records[i]["cap_objects"] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]
    grid, best = [], None
    for k in CFG.tag_k_grid:
        Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k), truth(va))
        grid.append(dict(norm="probe", k=k, val_P=Pv, val_R=Rv, val_F1=Fv))
        if best is None or Fv > best[0]:
            best = (Fv, k)
    k = best[1]
    tags = _topk_sets(P, np.arange(len(records)), vocab, k)
    Pt, Rt, Ft = _prf([tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm="probe(ngoài-fold)", k=k, test_P=Pt, test_R=Rt, test_F1=Ft,
                grid=grid, vocab=vocab)
    return tags, info


if "_apply_flickr_annotation_base" not in globals():
    _apply_flickr_annotation_base = apply_flickr_annotation


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    mode = mode or CFG.flickr_annot
    if mode != "probe":
        return _apply_flickr_annotation_base(records, split, img_feat, mode, text_encoder)
    global PROBE_INFO
    if not all("probe_tags" in r for r in records) or "PROBE_INFO" not in globals():
        tags, PROBE_INFO = probe_tagger(records, split, img_feat, text_encoder)
        for r, t in zip(records, tags):
            r["probe_tags"] = sorted(t)
    tr = set(split["train"])
    for r in records:
        r["objects"] = list(r["probe_tags"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    info = dict(annotation="probe", **{k: v for k, v in PROBE_INFO.items() if k != "vocab"})
    print(f"Nhãn ảnh = probe | vocab={info['vocab_size']} | k={info['k']} (chọn trên val) | "
          f"TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    return info


def fit_image_triples(records, split, img_feat, text_encoder=None):
    """Gán records[i]['img_triples'] = [((s, p, o), xác_suất)] cho MỌI ảnh.
    Nhãn huấn luyện = bộ ba chuẩn hoá trong caption 1..4 của ảnh TRAIN; ảnh train nhận dự đoán
    ngoài-fold; (k, τ) chọn trên VAL theo F1; caption test chỉ để BÁO CÁO P/R/F1."""
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    labels = {}
    cnt, surface = Counter(), defaultdict(Counter)
    for i in range(len(records)):
        ts = set()
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is None:
                continue
            ts.add(t)
        labels[i] = ts
    tr_set = set(tr.tolist())
    for i in tr:
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is not None:
                cnt[t] += 1; surface[t][p_] += 1
    obj_vocab = set(PROBE_INFO["vocab"]) if "PROBE_INFO" in globals() else \
        {o for i in tr for o in records[i]["cap_objects"]}
    vocab = sorted(t for t, c in cnt.items()
                   if c >= CFG.triple_min_count and t[0] in obj_vocab and t[2] in obj_vocab)
    vix = {t: j for j, t in enumerate(vocab)}
    prompts = [f"a photo of a {s} {surface[(s, p, o)].most_common(1)[0][0]} the {o}." for s, p, o in vocab]
    T = text_encoder(prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([labels[i] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [labels[i] & vset for i in idx]
    grid, best = [], None
    for k in CFG.triple_k_grid:
        for tau in CFG.triple_tau_grid:
            Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k, tau), truth(va))
            grid.append(dict(k=k, tau=tau, val_P=Pv, val_R=Rv, val_F1=Fv))
            if best is None or Fv > best[0]:
                best = (Fv, k, tau)
    _, k, tau = best
    # zero-shot (chỉ văn bản, chưa học) để so sánh
    S0 = img_feat @ T.T
    zs = _topk_sets((S0 - S0[tr].mean(0)) / (S0[tr].std(0) + 1e-6), te, vocab, k)
    for i in range(len(records)):
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        records[i]["img_triples"] = sorted(((vocab[j], float(row[j])) for j in top if row[j] >= tau),
                                           key=lambda x: -x[1])
    pred_te = [{t for t, _ in records[i]["img_triples"]} for i in te]
    Pt, Rt, Ft = _prf(pred_te, truth(te))
    _, _, Fz = _prf(zs, truth(te))
    qcov = np.mean([bool(set(query_triples(records[i]["caption"])) & vset) for i in te])
    info = dict(vocab_size=len(vocab), k=k, tau=tau, test_P=Pt, test_R=Rt, test_F1=Ft, zero_shot_test_F1=Fz,
                mean_triples_per_image=float(np.mean([len(r["img_triples"]) for r in records])),
                test_queries_with_vocab_triple=float(qcov), grid=grid,
                train_only=bool(Y_tr.shape[0] == len(tr_set)), oof=True,
                examples=[dict(caption=records[i]["caption"], triples=[" ".join(t) for t, _ in records[i]["img_triples"][:5]])
                          for i in te[:5]])
    print(f"Bộ ba cấp ảnh: vocab={len(vocab)} | k={k}, τ={tau} (chọn trên val) | "
          f"TEST P={Pt:.3f} R={Rt:.3f} F1={Ft:.3f} (zero-shot F1={Fz:.3f}) | "
          f"{info['mean_triples_per_image']:.2f} bộ ba/ảnh | truy vấn test có bộ ba trong vocab: {qcov:.1%}")
    return info


# ---------------------------------------------------------------------------------
# Bộ đánh giá QUAN HỆ KHÓ: cùng đối tượng, khác quan hệ.
# ---------------------------------------------------------------------------------
REL_GALLERIES, REL_PQ = {}, defaultdict(dict)
REL_GAL_KEY, REL_RENAME = None, {}


def _sample_negs(negs, rng):
    if len(negs) > CFG.rel_neg_max:
        negs = list(rng.choice(negs, CFG.rel_neg_max, replace=False))
    return negs


def build_rel_galleries_flickr(records, split, seed=0):
    """Truy vấn test có ≥1 bộ ba (s, p, o). Âm = ảnh test mà 5 caption nhắc ĐỦ mọi s, o của truy vấn
    nhưng KHÔNG có bộ ba nào của truy vấn. Caption test chỉ dùng làm NHÃN CHẤM."""
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    te = [pos[i] for i in split["test"]]
    nouns, tris = {}, {}
    for p in te:
        o, rels = extract_objects_relations(records[p]["captions"])
        nouns[p] = o
        tris[p] = {t for t in (canon_triple(*r) for r in rels) if t is not None}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        qt = set(query_triples(records[p]["caption"]))
        if not qt:
            continue
        n_q += 1
        need = {t[0] for t in qt} | {t[2] for t in qt}
        negs = [p2 for p2 in te if p2 != p and need <= nouns[p2] and not (qt & tris[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (Flickr8k): {len(gal)}/{n_q} truy vấn test có bộ ba đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


def build_rel_galleries_ag(recs, split, seed=0):
    """Truy vấn test nhắc tới một quan hệ người–vật 'v o' có trong annotation của video đúng.
    Âm = video test có vật o nhưng KHÔNG có quan hệ 'v o' (theo annotation)."""
    pos = {r["image_id"]: i for i, r in enumerate(recs)}
    te = [pos[i] for i in split["test"]]
    objs = {p: set(recs[p]["objects"]) for p in te}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        toks = [t for t in _norm(recs[p]["caption"]).split() if t not in STOP and len(t) > 1]
        verbs = set()
        for t in toks:
            v = _ag_lemma(t, "v"); verbs.add(AG_VERB_ALIAS.get(v, v))
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t), _ag_lemma(t)) for t in toks}
        comps = {c for c in objs[p] if " " in c and c.split(" ", 1)[0] in verbs and c.split(" ", 1)[1] in nouns}
        if not comps:
            continue
        n_q += 1
        need = {c.split(" ", 1)[1] for c in comps}
        negs = [p2 for p2 in te if p2 != p and need <= objs[p2] and not (comps & objs[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (AG): {len(gal)}/{n_q} truy vấn test có quan hệ người–vật đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


@torch.no_grad()
def rel_probe_rr(model, eng, gal, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    rr, rr_clip, groups = [], [], []
    for q, cand in gal:
        cand = np.asarray(cand, np.int64)
        qf = eng.txt_feat[[int(q)]]
        s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
        pm = torch.tensor(eng.path_feats(cand[None], np.array([q]))).to(DEVICE)
        s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_path = model.path_score(pm).cpu().numpy()
        fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
        rr.append(1.0 / (1 + int((fin > fin[0]).sum())))
        rr_clip.append(1.0 / (1 + int((s_clip[0] > s_clip[0, 0]).sum())))
        groups.append(eng.queries[int(q)]["group"])
    return np.array(rr), np.array(rr_clip), np.array(groups)


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    """Gọi trong run_experiment sau khi đánh giá test (α, λ đã chọn trên VAL — không chọn lại)."""
    if REL_GAL_KEY is None or REL_GAL_KEY not in REL_GALLERIES:
        return
    rr, rr_clip, groups = rel_probe_rr(model, eng, REL_GALLERIES[REL_GAL_KEY], alpha, lam)
    name = REL_RENAME.get(ab, ab)
    store = REL_PQ[REL_GAL_KEY]
    store[(name, seed)] = dict(rr=rr, rr_clip=rr_clip, groups=groups, lam=lam)
    store.setdefault(("CLIP", seed), dict(rr=rr_clip, rr_clip=rr_clip, groups=groups, lam=None))
    print(f"    [quan hệ khó] {name} seed {seed}: MRR {rr.mean():.4f} (CLIP {rr_clip.mean():.4f}) "
          f"| R@1 {(rr == 1).mean():.4f} | n={len(rr)}")


def rel_probe_report(key, base="none", others=None, out_csv=None):
    pq = REL_PQ[key]
    names = list(dict.fromkeys(a for a, _ in pq))
    rows = []
    for a in names:
        runs = [v["rr"] for (b, s), v in pq.items() if b == a]
        rows.append(dict(model=a, n_queries=len(runs[0]), seeds=len(runs),
                         MRR=f"{np.mean([r.mean() for r in runs]):.4f} ± {np.std([r.mean() for r in runs]):.4f}",
                         R1=f"{np.mean([(r == 1).mean() for r in runs]):.4f}"))
    tab = pd.DataFrame(rows)
    others = others or [a for a in names if a != base]
    sig_ = pd.concat([paired_test(pq, base, b) for b in others if any(k[0] == b for k in pq)])
    sig_ = sig_[sig_.group == "all"] if len(sig_) else sig_
    if out_csv is not None:
        tab.to_csv(out_csv, index=False)
        sig_.to_csv(str(out_csv).replace(".csv", "_paired.csv"), index=False)
    return tab, sig_


In [ ]:
# =====================================================================================
# 1.2c  ĐỒ THỊ CẢNH TỪ MÔ HÌNH MÔ TẢ ẢNH ĐỘC LẬP VỚI CLIP (bản v7)
#   Chẩn đoán (CPU, trên đặc trưng đã cache): mọi node của bản v5/v6 được dự đoán từ chính vector CLIP
#   của ảnh ⇒ đồ thị không mang thông tin mới (+0,003 MRR). Cận trên khi mỗi ảnh có mô tả văn bản độc lập
#   (caption người viết, khớp cả câu): +0,03…+0,05 (ViT-L/14), +0,08…+0,10 (ViT-B/32).
#   Bản này: mỗi ảnh (kể cả val/test — mô hình chỉ NHÌN ẢNH, không dùng caption của bộ dữ liệu) được một mô hình
#   mô tả ảnh (BLIP) sinh vài câu mô tả; từ đó rút node cho đồ thị:
#     • node nội dung: từ đã lemma + cụm 2 từ kề nhau (đối tượng, thuộc tính, hành động: 'dog', 'black dog', 'run')
#     • node bộ ba ⟨s p o⟩ cấp ảnh (mẫu P6) và cạnh scene (thống kê trên ảnh TRAIN) rút từ CHÍNH các mô tả đó
#   Truy vấn được tách đúng cùng cách (parse_query) ⇒ khớp node bằng chuỗi, giải thích/fidelity giữ nguyên cơ chế.
#   Caption người viết của TRAIN chỉ còn dùng làm truy vấn huấn luyện; của VAL/TEST chỉ là truy vấn + nhãn chấm.
# =====================================================================================
CFG.vlm_name     = "Salesforce/blip-image-captioning-large"
CFG.vlm_bs       = 32
CFG.vlm_n_sample = 3          # số mô tả lấy mẫu (nucleus) thêm vào 1 mô tả beam search
CFG.vlm_max_tok  = 40


def content_grams(text):
    """Cùng cách tách với parse_query của Flickr: lemma từng từ (bỏ stopword) + cụm 2 từ kề nhau."""
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    return set(toks) | {" ".join(p) for p in zip(toks, toks[1:])}


def vlm_captions(parts, n):
    cache = CFG.out_dir / "cache" / f"vlm_caps_flickr8k_{CFG.vlm_name.split('/')[-1]}_{n}.json"
    for f in glob.glob(f"/kaggle/input/**/cache/{cache.name}", recursive=True):
        if not cache.exists():
            shutil.copy2(f, cache)
    if cache.exists():
        print("Dùng lại mô tả ảnh đã sinh:", cache.name)
        return json.load(open(cache))
    from transformers import BlipProcessor, BlipForConditionalGeneration
    proc = BlipProcessor.from_pretrained(CFG.vlm_name)
    model = BlipForConditionalGeneration.from_pretrained(CFG.vlm_name).to(DEVICE).eval()
    if DEVICE == "cuda":
        model = model.half()
    torch.manual_seed(0)
    out, t0 = [], time.time()
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), CFG.vlm_bs):
                ims = [im.convert("RGB") for im in d[s:s + CFG.vlm_bs]["image"]]
                inp = proc(images=ims, return_tensors="pt").to(DEVICE)
                if DEVICE == "cuda":
                    inp["pixel_values"] = inp["pixel_values"].half()
                caps = [[] for _ in ims]
                gens = [model.generate(**inp, num_beams=3, max_new_tokens=CFG.vlm_max_tok)]
                gens += [model.generate(**inp, do_sample=True, top_p=0.9, max_new_tokens=CFG.vlm_max_tok)
                         for _ in range(CFG.vlm_n_sample)]
                for g in gens:
                    for i, t in enumerate(proc.batch_decode(g, skip_special_tokens=True)):
                        caps[i].append(t.strip())
                out += caps
                if (len(out) // CFG.vlm_bs) % 25 == 0:
                    print(f"  mô tả ảnh: {len(out)}/{n} | {time.time() - t0:.0f}s", flush=True)
    del model; torch.cuda.empty_cache()
    assert len(out) == n, (len(out), n)
    json.dump(out, open(cache, "w"))
    return out


if "_apply_flickr_annotation_v5" not in globals():
    _apply_flickr_annotation_v5 = apply_flickr_annotation


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    mode = mode or CFG.flickr_annot
    if mode not in ("vlm", "oracle_grams"):
        return _apply_flickr_annotation_v5(records, split, img_feat, mode, text_encoder)
    if not all(r.get("_node_src") == mode for r in records):
        # oracle_grams: gallery ĐÃ ĐƯỢC CHÚ THÍCH — caption 1..4 của ảnh (caption 0 là truy vấn, không bao giờ dùng làm node)
        caps = vlm_captions(hf_parts, len(records)) if mode == "vlm" else [r["captions"][1:] for r in records]
        for r, c in zip(records, caps):
            r["vlm_caps"] = c; r["_node_src"] = mode
            grams = set()
            for t in c:
                grams |= content_grams(t)
            _, rels = extract_objects_relations(c)
            r["vlm_nodes"], r["vlm_rels"] = sorted(grams), sorted(rels)
            r["vlm_triples"] = sorted({t for t in (canon_triple(*x) for x in rels) if t is not None})
    tr = set(split["train"])
    for r in records:
        r["objects"] = list(r["vlm_nodes"])
        r["relations"] = list(r["vlm_rels"]) if r["image_id"] in tr else []     # cạnh scene: thống kê CHỈ trên train
        r["img_triples"] = [(t, 1.0) for t in r["vlm_triples"]]                # bộ ba của CHÍNH ảnh (mô hình nhìn ảnh)
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    te = [pos[i] for i in split["test"]]
    # Độ phủ: bao nhiêu % từ/cụm từ của TRUY VẤN test có trong node của ảnh đúng (caption test chỉ dùng để ĐO)
    cov = np.mean([len(content_grams(records[i]["caption"]) & set(records[i]["vlm_nodes"])) /
                   max(len(content_grams(records[i]["caption"])), 1) for i in te])
    qt = [set(query_triples(records[i]["caption"])) for i in te]
    tcov = np.mean([bool(q & set(records[i]["vlm_triples"])) for q, i in zip(qt, te) if q] or [0.0])
    info = dict(annotation=mode, vlm=CFG.vlm_name if mode == "vlm" else "caption người viết 1–4", captions_per_image=len(records[0]["vlm_caps"]),
                mean_nodes_per_image=float(np.mean([len(r["vlm_nodes"]) for r in records])),
                mean_triples_per_image=float(np.mean([len(r["vlm_triples"]) for r in records])),
                test_query_gram_coverage=float(cov), test_query_triple_hit=float(tcov))
    print(f"Nhãn ảnh = {mode} ({info['vlm']}) | {info['captions_per_image']} mô tả/ảnh | "
          f"{info['mean_nodes_per_image']:.1f} node/ảnh | {info['mean_triples_per_image']:.1f} bộ ba/ảnh | "
          f"độ phủ từ/cụm từ của truy vấn test ở ảnh đúng: {cov:.1%} | truy vấn test có bộ ba khớp ảnh đúng: {tcov:.1%}")
    for r in records[-3:]:
        print(f"  • truy vấn : {r['caption'][:90]}\n      mô tả VLM: {r['vlm_caps'][:2]}\n"
              f"      bộ ba    : {[' '.join(t) for t in r['vlm_triples'][:4]]}")
    return info


def paired_vs_clip(pq, ab, n_boot=10000, seed=0):
    """Kiểm định cặp CLIP+Graph vs CLIP trên CÙNG truy vấn test (RR trung bình theo seed)."""
    seeds = sorted(s for a, s in pq if a == ab)
    rr = np.mean([pq[(ab, s)]["rr"] for s in seeds], 0); rc = pq[(ab, seeds[0])]["rr_clip"]
    groups = pq[(ab, seeds[0])]["groups"]; rng = np.random.RandomState(seed); rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(rr), bool) if grp == "all" else groups == grp
        d = (rr - rc)[m]
        if len(d) < 5:
            continue
        bs = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        rows.append(dict(config=ab, group=grp, n=int(m.sum()), MRR_clip=rc[m].mean(), MRR_graph=rr[m].mean(),
                         R1_clip=(rc[m] == 1).mean(), R1_graph=np.mean([(pq[(ab, s)]["rr"][m] == 1).mean() for s in seeds]),
                         diff=d.mean(), ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5)))
    return pd.DataFrame(rows)


In [ ]:
# HF_TOKEN (Kaggle Secrets) — không bắt buộc; thiếu secret thì vẫn tải ẩn danh.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    print("Không đọc được HF_TOKEN từ Kaggle Secrets → tải ẩn danh:", type(e).__name__)


In [ ]:
# v7: backbone chính ViT-L/14 (đặc trưng lấy từ cache của version trước); ViT-B/32 chạy ở cell đối chứng cuối Phần 1.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-L-14", "laion2b_s32b_b82k", 128   # mô hình cuối (version 9); đổi thành "ViT-H-14", "laion2b_s32b_b79k", 64 để chạy ViT-H
for _a in ("_m", "_t"):
    clip_text_features.__dict__.pop(_a, None)
import pandas as pd
records, hf_parts = load_flickr8k()
dataset_name = "flickr8k"
split = make_split(records)
img_feat, txt_feat = encode_flickr8k(records, hf_parts)
annot_info = apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)
print(f"\nDữ liệu: {dataset_name} | {len(records)} ảnh | đặc trưng CLIP {img_feat.shape}")

TRIPLE_INFO = fit_image_triples(records, split, img_feat) if (CFG.use_triples and CFG.flickr_annot not in ("vlm", "oracle_grams")) else {}
if CFG.flickr_annot in ("tagger", "probe"):
    print("\nChọn (chuẩn hoá, k) của tagger trên VALIDATION:")
    print(pd.DataFrame(annot_info["grid"]).pivot_table(index="k", columns="norm",
                                                         values="val_F1").round(3).to_string())
    for r in records[-3:]:                       # vài ảnh TEST: nhãn tagger vs danh từ trong caption
        print(f"  • {r['caption'][:70]}\n      nhãn   : {r['objects']}\n      caption: {r['cap_objects']}"
              f"\n      bộ ba  : {[' '.join(t) + f' ({p:.2f})' for t, p in r.get('img_triples', [])[:4]]}")
with open(CFG.out_dir / "results" / "annotation_info.json", "w") as f:
    json.dump({k: v for k, v in annot_info.items()}, f, indent=2)
with open(CFG.out_dir / "results" / "triple_info.json", "w") as f:
    json.dump({k: v for k, v in TRIPLE_INFO.items()}, f, indent=2, ensure_ascii=False)

In [ ]:
self_test(records, split, img_feat, txt_feat)

In [ ]:
# Chọn CFG.learn_clip_w trên VALIDATION (1 seed, cấu hình 'none'). Không nhìn test.
def pick_learn_clip_w(records, split, img_feat, txt_feat, seed=0):
    rows = []
    for flag in (False, True):
        CFG.learn_clip_w = flag
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(records, split, img_feat, txt_feat, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        rows.append(dict(learn_clip_w=flag, val_MRR=max(r["MRR"] for r in grid), alpha=a, lam=l,
                         clip_w=round(F.softplus(model.clip_logit).item(), 3),
                         path_w=round(F.softplus(model.path_logit).item(), 3)))
    df_ = pd.DataFrame(rows); display(df_)
    CFG.learn_clip_w = bool(df_.loc[df_.val_MRR.idxmax(), "learn_clip_w"])
    print("→ CFG.learn_clip_w =", CFG.learn_clip_w, "(chọn theo VAL MRR)")
    return df_

clipw_flickr = pick_learn_clip_w(records, split, img_feat, txt_feat)

In [ ]:
# Thí nghiệm chính: 8 cấu hình ablation × 3 seed (6 cũ + no_triples + shuffled_triples), nhãn ảnh = CFG.flickr_annot
REL_GALLERIES["flickr"] = build_rel_galleries_flickr(records, split)
REL_GAL_KEY, REL_RENAME = "flickr", {}
all_res, art = run_experiment(records, split, img_feat, txt_feat, ablations=FLICKR_ABLATIONS)
REL_GAL_KEY = None
df = results_table(all_res)
df.to_csv(CFG.out_dir / "results" / f"main_results_{CFG.flickr_annot}.csv", index=False)
df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
display(df)

In [ ]:
# =====================================================================================
# ABLATION THEO GIẢI THÍCH: giải thích có đúng với tri thức ĐỘC LẬP với đồ thị không?
#
# Với mỗi ablation (seed đầu), giải thích của ẢNH ĐÚNG trên mọi truy vấn test = top-m đường đi
# QUAN HỆ (P1..P5, bỏ P0) theo đóng góp biên — đúng định nghĩa explain_and_score. Mỗi CẠNH
# trong đường đi được chấm bằng nguồn KHÔNG dùng để dựng đồ thị:
#   • scene / related  : NPMI của hai đầu cạnh trên tập nhãn tham chiếu của ảnh VAL+TEST
#                        (Flickr: danh từ trong caption 1..4 — caption 0 là truy vấn nên bỏ;
#                         AG: annotation của video). Đồ thị chỉ thống kê trên TRAIN ⇒ độc lập.
#                        Cạnh "hợp lý" ⇔ NPMI > 0 (hai thứ hay đi cùng nhau hơn ngẫu nhiên).
#   • isa              : đúng ⇔ đích nằm trong bao đóng hypernym WordNet (hoặc FALLBACK_ISA).
# Chỉ số:
#   edge_ok    % cạnh hợp lý            path_ok  % đường đi có MỌI cạnh hợp lý
#   grounded   % đường đi mà mọi node ĐỐI TƯỢNG trung gian có trong nhãn tham chiếu của ảnh đó
#   rand_path_ok  path_ok khi chọn NGẪU NHIÊN cùng số đường đi quan hệ của chính ảnh đó
#                 ⇒ (path_ok − rand_path_ok) = cơ chế chọn giải thích có ưu tiên đường đi đúng không
# Không rò rỉ: nhãn tham chiếu chỉ dùng để CHẤM, không huấn luyện / chọn siêu tham số.
# Chạy NGAY sau cell thí nghiệm chính (cần ART_BY_ABL của cell 28 bản đợt 4).
# =====================================================================================
from functools import lru_cache


def reference_sets(records, ids):
    key = "cap_objects" if "cap_objects" in records[0] else "objects"
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    return key, {i: set(records[pos[i]][key]) for i in ids}


def make_npmi(docs):
    n = len(docs)
    c1 = Counter(x for d in docs for x in d)
    c2 = Counter(p for d in docs for p in itertools.combinations(sorted(d), 2))
    def npmi(a, b):
        ab = c2.get(tuple(sorted((a, b))), 0)
        if ab == 0 or c1[a] == 0 or c1[b] == 0:
            return -1.0
        p_ab = ab / n
        return math.log(p_ab / ((c1[a] / n) * (c1[b] / n))) / max(-math.log(p_ab), 1e-9)
    return npmi


@lru_cache(maxsize=None)
def _isa_ok(a, b):
    if b in FALLBACK_ISA.get(a, []):
        return True
    wn = _wordnet()
    if wn is None:
        return False
    for s in wn.synsets(a.replace(" ", "_"), pos="n")[:3]:
        for h in s.closure(lambda x: x.hypernyms()):
            if b in {l.name().replace("_", " ").lower() for l in h.lemmas()}:
                return True
    return False


_EDGE_KIND = {"P1_scene": ["scene"], "P2_isa": ["isa"], "P3_related": ["related"],
              "P4_scene_isa": ["scene", "isa"], "P5_related_isa": ["related", "isa"]}


def _edges(p):
    nodes = p["nodes"]                      # [o, c] hoặc [o, o2, c]
    return list(zip(_EDGE_KIND[p["pattern"]], nodes[:-1], nodes[1:]))


def _rel_contrib(paths, model):
    """Đóng góp biên của từng đường đi = đúng công thức explain_and_score, nhưng O(P)."""
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    pidx = np.array([PIDX[p["pattern"]] for p in paths])
    pw = np.array([p["w"] for p in paths])
    tot = np.bincount(pidx, weights=pw, minlength=len(PATTERNS))
    return gain * w[pidx] * (np.log1p(tot[pidx]) - np.log1p(np.maximum(tot[pidx] - pw, 0)))


def explanation_grounding(art_by_abl, records, split, top_m=CFG.top_m_paths, n_rand=10, seed=0):
    key, ref = reference_sets(records, split["val"] + split["test"])
    npmi = make_npmi(list(ref.values()))
    rng = np.random.RandomState(seed)
    rows = []
    for ab, art in art_by_abl.items():
        g, eng, model, part = art["graph"], art["engine"], art["model"], art["parts"]["test"]
        name = lambda n: g.id2name.get(n, "")
        is_obj = lambda n: g.n_img <= n < g.n_img + g.n_obj
        edge_cache = {}

        def edge_ok(kind, a, b):
            k = (kind, a, b)
            if k not in edge_cache:
                edge_cache[k] = (_isa_ok(name(a), name(b)) if kind == "isa"
                                 else npmi(name(a), name(b)) > 0)
            return edge_cache[k]

        def path_ok(p):
            return all(edge_ok(*e) for e in _edges(p))

        for i, qi in enumerate(part["q_idx"]):
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            if not cids:
                continue
            gold = int(part["gold_pos"][i])
            rel = [p for p in enumerate_paths(g, gold, cids) if p["pattern"] in _EDGE_KIND]   # P1..P5
            if not rel:
                continue
            top = [rel[j] for j in np.argsort(-_rel_contrib(rel, model))[:top_m]]
            truth = ref.get(records[gold]["image_id"], set())
            k = len(top)
            rand_ok = np.mean([np.mean([path_ok(rel[j]) for j in rng.choice(len(rel), k, replace=False)])
                               for _ in range(n_rand)])
            edges = [e for p in top for e in _edges(p)]
            rows.append(dict(
                ablation=ab, group=part["groups"][i], n_rel_paths=len(rel), n_expl=k,
                edge_ok=np.mean([edge_ok(*e) for e in edges]),
                path_ok=np.mean([path_ok(p) for p in top]),
                rand_path_ok=rand_ok,
                grounded=np.mean([all(name(n) in truth for n in p["nodes"][:-1] if is_obj(n))
                                  for p in top]),
                mean_npmi=np.mean([npmi(name(a), name(b)) for kd, a, b in edges if kd != "isa"] or [np.nan])))
    df = pd.DataFrame(rows)
    print(f"Nhãn tham chiếu: '{key}' của {len(ref)} ảnh VAL+TEST | top_m={top_m}")
    return df


def _boot_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = x[np.random.RandomState(seed).randint(0, len(x), (n_boot, len(x)))].mean(1)
    return f"{x.mean():.3f} [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]"


def grounding_table(df):
    out = []
    for grp in ("all", "direct", "indirect"):
        for ab in df.ablation.unique():
            d = df[(df.ablation == ab) & ((df.group == grp) | (grp == "all"))]
            if len(d) < 5:
                continue
            out.append(dict(ablation=ab, group=grp, n_queries=len(d),
                            rel_paths_per_q=round(d.n_rel_paths.mean(), 1),
                            edge_ok=_boot_ci(d.edge_ok), path_ok=_boot_ci(d.path_ok),
                            rand_path_ok=_boot_ci(d.rand_path_ok),
                            selection_lift=_boot_ci(d.path_ok - d.rand_path_ok),
                            grounded=_boot_ci(d.grounded), mean_npmi=round(d.mean_npmi.mean(), 3)))
    return pd.DataFrame(out)


# ------------------------------ Flickr8k ------------------------------
ART_FLICKR = dict(ART_BY_ABL)                     # giữ lại trước khi cell oracle / AG ghi đè
try:
    gdf = explanation_grounding(ART_FLICKR, records, split)
    gtab = grounding_table(gdf)
    gdf.to_csv(CFG.out_dir / "results" / "explanation_grounding_per_query.csv", index=False)
    gtab.to_csv(CFG.out_dir / "results" / "explanation_grounding.csv", index=False)
    display(gtab)
except Exception as e:                      # phân tích phụ — không để nó chặn phần còn lại
    print("⚠ explanation_grounding bỏ qua:", type(e).__name__, e)

In [ ]:
# Kiểm định CẶP giữa các ablation trên cùng truy vấn test (trung bình 3 seed / truy vấn).
# Chênh MRR 0.000x với std 0.002-0.003 thì KHÔNG kết luận được gì — cần CI cho hiệu số.
# Chạy NGAY sau cell thí nghiệm chính (cell oracle / AG sẽ ghi đè PER_QUERY).
def paired_test(pq, a="none", b="no_relations", n_boot=10000, seed=0):
    seeds = sorted({s for ab, s in pq if ab == a} & {s for ab, s in pq if ab == b})
    ra = np.mean([pq[(a, s)]["rr"] for s in seeds], 0)
    rb = np.mean([pq[(b, s)]["rr"] for s in seeds], 0)
    groups = pq[(a, seeds[0])]["groups"]
    rng = np.random.RandomState(seed)
    rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(ra), bool) if grp == "all" else groups == grp
        d = (ra - rb)[m]
        if len(d) < 5:
            continue
        boot = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(group=grp, n=int(m.sum()), MRR_a=ra[m].mean(), MRR_b=rb[m].mean(),
                         diff=d.mean(), ci_lo=lo, ci_hi=hi,
                         p_a_not_better=float((boot <= 0).mean()),
                         a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum())))
    return pd.DataFrame(rows).assign(a=a, b=b)


PQ_MAIN = dict(PER_QUERY)            # giữ lại trước khi cell sau ghi đè
sig = pd.concat([paired_test(PQ_MAIN, "none", b)
                 for b in ("no_triples", "shuffled_triples", "no_relations", "random_edges",
                           "no_scene", "no_related", "no_concept")
                 if any(k[0] == b for k in PQ_MAIN)])
sig.to_csv(CFG.out_dir / "results" / "paired_ablation_test.csv", index=False)
print("λ chọn trên val cho 'none':", [v["lam"] for k, v in PQ_MAIN.items() if k[0] == "none"],
      "(λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)")
display(sig.round(4))

In [ ]:
# v5 — BỘ QUAN HỆ KHÓ trên Flickr8k: gallery = ảnh đúng + ảnh test cùng đối tượng, khác quan hệ.
# Đây là chỗ quan hệ CẤP ẢNH phải tạo khác biệt: nhãn đối tượng không còn phân biệt được ứng viên.
rel_tab, rel_sig = rel_probe_report("flickr", base="none",
                                    others=["CLIP", "no_triples", "shuffled_triples", "no_relations", "random_edges"],
                                    out_csv=CFG.out_dir / "results" / "relational_hard_negatives.csv")
display(rel_tab); display(rel_sig.round(4))
print("diff > 0 và ci_lo > 0 ⇒ 'none' (có bộ ba cấp ảnh) tốt hơn cấu hình so sánh một cách có ý nghĩa.")


In [ ]:
# Mô hình cuối của Flickr8k chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_FLICKR = list(FUSION_LOG)
sel_f, final_f = final_model_table(all_res, FUSION_FLICKR)
display(sel_f.round(4)); display(final_f)
display(pd.DataFrame(FUSION_FLICKR).round(3))
sig_ad = paired_test(dict(PER_QUERY), "none+adaptive", "none")
display(sig_ad.round(4))
for name, obj in (("final_model_val_selection.csv", sel_f), ("final_model_test.csv", final_f),
                  ("fusion_log.csv", pd.DataFrame(FUSION_FLICKR)), ("paired_adaptive_vs_global.csv", sig_ad)):
    obj.to_csv(CFG.out_dir / "results" / name, index=False)

In [ ]:
# So sánh với nhãn ORACLE (rút từ caption 1..4 của chính ảnh — cách của bản trước).
# Chỉ để định lượng "rò rỉ" của cách cũ: chênh lệch oracle − tagger chính là phần điểm
# đến từ việc ứng viên mang sẵn mô tả văn bản của chính nó. KHÔNG báo cáo là kết quả chính.
RUN_ORACLE = False      # v6: đã chạy ở version 2 (ViT-L/14)
if RUN_ORACLE:
    apply_flickr_annotation(records, split, img_feat, "oracle")
    oracle_res, _ = run_experiment(records, split, img_feat, txt_feat,
                                   ablations=("none", "no_relations"))
    df_oracle = results_table(oracle_res)
    df_oracle.to_csv(CFG.out_dir / "results" / "main_results_oracle.csv", index=False)
    apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)   # KHÔI PHỤC nhãn chính
    cmp_annot = pd.concat([df[df.ablation.isin(["none", "no_relations"])].assign(annotation=CFG.flickr_annot),
                           df_oracle.assign(annotation="oracle")])
    cmp_annot = cmp_annot[cmp_annot.model.isin(["CLIP", "CLIP+Graph"])]
    cmp_annot.to_csv(CFG.out_dir / "results" / "annotation_comparison.csv", index=False)
    display(cmp_annot[["annotation", "ablation", "model", "R@1", "R@5", "MRR"]])

In [ ]:
# Fidelity đo trên THỨ HẠNG của điểm cuối (CLIP + GNN + path)
rdf, rsumm = final_rank_fidelity(art)
rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
for k, v in rsumm.items():
    print(f"  {k:36s} {v:.3f}" if isinstance(v, float) else f"  {k:36s} {v}")
with open(CFG.out_dir / "results" / "xai_final_rank_summary.json", "w") as f:
    json.dump(rsumm, f, indent=2)

**Cách đọc bảng.**

1. `none / CLIP` vs `none / CLIP+Graph` — đồ thị có giúp không (Nhiệm vụ 1).
2. `none` vs `random_edges` / `no_relations` — cải thiện có đến từ **cấu trúc quan hệ** không.
   Chỉ kết luận "quan hệ có ích" khi `none` > `random_edges` **và** `none` > `no_relations`
   vượt quá độ lệch chuẩn giữa các seed.
3. `no_scene` / `no_concept` — loại cạnh nào (scene graph rút từ caption hay IsA WordNet) đóng góp.
4. Dòng `[indirect]` — nhóm duy nhất mà đường đi P1–P5 có thể giúp; P0 không tồn tại cho nhóm này.

Nếu `no_relations` vẫn ≥ `none`, kết luận trung thực là: lợi ích đến từ nhãn đối tượng (P0)
và GNN, còn quan hệ trong đồ thị hiện tại chưa giúp — kèm phân tích lý do ở mục case study.

In [ ]:
# Độ nhạy theo α (yêu cầu ở cột Mở rộng của Nhiệm vụ 2)
import pandas as pd
gdf = pd.DataFrame(art["grid"])
piv = gdf.pivot_table(index="alpha", columns="lam", values="MRR")
print("MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:")
print(piv.round(4).to_string())
piv.to_csv(CFG.out_dir / "results" / "alpha_sensitivity.csv")

In [ ]:
# Phân tích 5 thành công + 5 thất bại
cases = analyse_cases(art, records)
for c in cases:
    print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → CLIP+Graph {c['rank_graph']}")
    print(f"  truy vấn: {c['query']}")
    for e in c["explanation"]:
        print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    d = c["detail"]
    ctrl = d['score_full'] - d['score_without_random']
    print(f"  fidelity−: {d['score_full'] - d['score_without_explanation']:+.4f} | "
          f"đối chứng ngẫu nhiên: {'n/a (1 đường đi)' if ctrl != ctrl else f'{ctrl:+.4f}'} | "
          f"sparsity {d['sparsity']:.2f} | {d['n_paths']} đường đi")
with open(CFG.out_dir / "results" / "case_analysis.json", "w") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)

In [ ]:
# Fidelity / validity / sparsity trên ≥10 truy vấn (cấp độ 3)
fdf, summ = fidelity_study(art)
fdf.to_csv(CFG.out_dir / "results" / "fidelity_per_query.csv", index=False)
print("Tổng hợp xAI:")
for k, v in summ.items():
    print(f"  {k:34s} {v:.4f}" if isinstance(v, float) else f"  {k:34s} {v}")
print("\nnecessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên")
print("cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.")
with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
    json.dump(summ, f, indent=2)

In [ ]:
# Bộ ≥20 truy vấn đánh giá (cấp độ 2) — CHỈ lấy truy vấn của ảnh test
eval_qs = export_eval_queries(art["engine"].queries, art["graph"],
                              CFG.out_dir / "results" / "eval_queries_20.json", n=20,
                              allowed_ids=split["test"])

In [ ]:
# Demo
demo_query("a child playing football on the grass", art, k=3)
demo_query("a person cooking food in a kitchen", art, k=3)

In [ ]:
# =====================================================================================
# v7 — (a) kiểm định cặp CLIP+Graph vs CLIP; (b) đối chứng XÁO node VLM giữa các ảnh; (c) backbone ViT-B/32
# Chạy SAU mọi phân tích của mô hình chính (run_experiment ghi đè PER_QUERY / ART_BY_ABL).
# =====================================================================================
import copy
R_F = CFG.out_dir / "results"
vs_clip = [paired_vs_clip(PQ_MAIN, ab).assign(backbone=CFG.clip_model) for ab in ("none", "no_class_edges", "no_relations", "no_triples")
           if any(k[0] == ab for k in PQ_MAIN)]

# (b) mỗi ảnh nhận node + bộ ba VLM của một ảnh KHÁC cùng split ⇒ nếu hiệu quả đến từ nội dung riêng của ảnh thì phải mất
_rng = np.random.RandomState(0)
records_sh = copy.deepcopy(records)
_pos = {r["image_id"]: i for i, r in enumerate(records)}
for _k in ("train", "val", "test"):
    _idx = np.array([_pos[i] for i in split[_k]]); _perm = _rng.permutation(_idx)
    for a, b in zip(_idx, _perm):
        for f in ("objects", "relations", "img_triples"):
            records_sh[a][f] = copy.deepcopy(records[b][f])
REL_GAL_KEY, REL_RENAME = "flickr", {"none": "shuffled_nodes"}
sh_res, _ = run_experiment(records_sh, split, img_feat, txt_feat, ablations=("none",))
REL_GAL_KEY, REL_RENAME = None, {}
PQ_SH = {("shuffled_nodes" + ab[4:], s): v for (ab, s), v in PER_QUERY.items() if ab == "none" or ab.startswith("none+")}
sh_df = results_table(sh_res).assign(ablation="shuffled_nodes")
vs_clip.append(paired_vs_clip(PQ_SH, "shuffled_nodes").assign(backbone=CFG.clip_model))
sig_sh = paired_test({**PQ_MAIN, **PQ_SH}, "none", "shuffled_nodes")
sig_sh.to_csv(R_F / "paired_none_vs_shuffled_nodes.csv", index=False)
pd.concat([df, sh_df]).to_csv(R_F / "main_results_with_controls.csv", index=False)
rel_tab, rel_sig = rel_probe_report("flickr", base="none",
                                    others=["CLIP", "shuffled_nodes", "no_class_edges", "no_relations"],
                                    out_csv=R_F / "relational_hard_negatives.csv")
print("Bộ quan hệ khó (Flickr8k), kèm đối chứng shuffled_nodes:"); display(rel_tab); display(rel_sig.round(4))

# (c) cùng đồ thị VLM, backbone CLIP yếu hơn (ViT-B/32) — dùng đặc trưng đã cache ở version trước
_b = sorted(glob.glob(f"/kaggle/input/**/cache/clip_flickr8k_ViT-B-32_*_{len(records)}.npz", recursive=True)) + \
     sorted((CFG.out_dir / "cache").glob(f"clip_flickr8k_ViT-B-32_*_{len(records)}.npz"))
if _b:
    _z = np.load(_b[0]); imgB, txtB = _z["img"], _z["txt"]
    REL_GALLERIES["flickr_b32"] = REL_GALLERIES["flickr"]
    REL_GAL_KEY, REL_RENAME = "flickr_b32", {}
    b_res, _ = run_experiment(records, split, imgB, txtB, ablations=("none", "no_class_edges"))
    REL_GAL_KEY = None
    PQ_B = dict(PER_QUERY)
    b_df = results_table(b_res); b_df.to_csv(R_F / "main_results_vitb32.csv", index=False)
    display(b_df[b_df.model.isin(["CLIP", "CLIP+Graph"])])
    vs_clip += [paired_vs_clip(PQ_B, ab).assign(backbone="ViT-B-32") for ab in ("none", "no_class_edges")]
    relb_tab, relb_sig = rel_probe_report("flickr_b32", base="none", others=["CLIP", "no_class_edges"],
                                          out_csv=R_F / "relational_hard_negatives_vitb32.csv")
    display(relb_tab); display(relb_sig.round(4))
else:
    print("Không thấy cache CLIP ViT-B/32 trong /kaggle/input → bỏ qua phần (c).")

vs_clip = pd.concat(vs_clip)
vs_clip.to_csv(R_F / "paired_graph_vs_clip.csv", index=False)
print("CLIP+Graph vs CLIP (kiểm định cặp, bootstrap theo truy vấn):")
print(vs_clip.round(4).to_string())
print("\nnone vs shuffled_nodes:"); print(sig_sh.round(4).to_string())


In [ ]:
# =====================================================================================
# v9 — BÁO CÁO NHÁNH KHỚP (Flickr8k): CLIP+Graph+Match / CLIP+Match vs CLIP, vs CLIP+Graph, vs đối chứng
# =====================================================================================
R_F = CFG.out_dir / "results"
def _vs_names(pq, names, **tags):
    out = [paired_vs_clip(pq, n).assign(**tags) for n in names if any(k[0] == n for k in pq)]
    return pd.concat(out) if out else pd.DataFrame()
_mf = [_vs_names(PQ_MAIN, ["none", "none+clipmatch", "none+match", "no_class_edges+match", "no_relations+match"], backbone=CFG.clip_model),
       _vs_names(PQ_SH, ["shuffled_nodes", "shuffled_nodes+clipmatch", "shuffled_nodes+match"], backbone=CFG.clip_model)]
if "PQ_B" in globals():
    _mf.append(_vs_names(PQ_B, ["none", "none+clipmatch", "none+match"], backbone="ViT-B-32"))
match_vs_clip_f = pd.concat(_mf); match_vs_clip_f.to_csv(R_F / "match_vs_clip.csv", index=False)
print("Flickr8k — kiểm định cặp vs CLIP:"); print(match_vs_clip_f[match_vs_clip_f.group == "all"].round(4).to_string())
_P = {**PQ_MAIN, **PQ_SH}
_pairs = [("none+match", "none"), ("none+match", "none+clipmatch"), ("none+match", "shuffled_nodes+match"),
          ("none+match", "no_relations+match"), ("none+match", "no_class_edges+match"), ("none+clipmatch", "none")]
_sig = [paired_test(_P, a, b).assign(a=a, b=b) for a, b in _pairs
        if any(k[0] == a for k in _P) and any(k[0] == b for k in _P)]
if "PQ_B" in globals():
    _sig += [paired_test(PQ_B, a, b).assign(a=a, b=b, backbone="ViT-B-32") for a, b in _pairs[:2]
             if any(k[0] == a for k in PQ_B) and any(k[0] == b for k in PQ_B)]
match_sig_f = pd.concat(_sig); match_sig_f.to_csv(R_F / "match_paired.csv", index=False)
print("\nKiểm định cặp giữa các cấu hình:"); print(match_sig_f[match_sig_f.group == "all"].round(4).to_string())
if "flickr" in REL_PQ and any(k[0] == "none+match" for k in REL_PQ["flickr"]):
    _rt, _rs = rel_probe_report("flickr", base="none+match",
                                others=["CLIP", "none", "none+clipmatch", "shuffled_nodes+match", "no_relations+match"],
                                out_csv=R_F / "relational_hard_negatives_match.csv")
    print("\nBộ quan hệ khó (Flickr8k) với nhánh khớp:"); print(_rt.to_string()); print(_rs.round(4).to_string())
pd.DataFrame(MATCH_LOG).to_csv(R_F / "match_log.csv", index=False)


---
# PHẦN 2 — GNN của SCENIR thay GNN cổ điển (Flickr8k)

_Nhiệm vụ 2 · dùng lại toàn bộ mã của Phần 1._

Cùng `run_experiment`, cùng split, nhãn tagger và ứng viên CLIP; chỉ đổi `CFG.gnn_type = "scenir"` (mục 1.7.2).

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
with use_gnn("scenir"):
    sc_res, sc_art = run_experiment(records, split, img_feat, txt_feat,
                                    ablations=("none", "random_edges", "no_relations"))
sc_df = results_table(sc_res)
sc_df.to_csv(CFG.out_dir / "results" / f"main_results_scenir_{CFG.flickr_annot}.csv", index=False)
display(sc_df)

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# GNN cổ điển vs SCENIR, tách nhóm trực tiếp / gián tiếp
cmp_gnn = pd.concat([df.assign(gnn=CFG.gnn_type), sc_df.assign(gnn="scenir")])
cmp_gnn = cmp_gnn[cmp_gnn.ablation.isin(["none", "random_edges", "no_relations"])]
cmp_gnn.to_csv(CFG.out_dir / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

In [ ]:
# Gói kết quả để nộp
print("\nTệp đã ghi:")
for p in sorted((CFG.out_dir / "results").glob("*")) + [CFG.out_dir / "split.json"]:
    print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")

---
# PHẦN 3 — Action Genome (dataset thứ hai)

_Cùng pipeline của Phần 1–2 trên video/frame Action Genome: liên kết khái niệm (dictionary vs semantic), 3 ablation × 3 seed, SCENIR._

In [ ]:
import shutil, os, glob, json
from pathlib import Path

# Dataset hieutrinh123/action-genome được Kaggle mount sẵn (chỉ đọc, đĩa cục bộ)
# ⇒ không cần chép/giải nén. Dò thư mục chứa ag_records.json; frame nằm ở <thư mục>/ag_frames/.
_ag_found = sorted(glob.glob("/kaggle/input/**/ag_records.json", recursive=True))
assert _ag_found, ("Không thấy ag_records.json trong /kaggle/input — Add Input → "
                   "dataset 'hieutrinh123/action-genome'.")
DATA_PATH = str(Path(_ag_found[0]).parent)
AG_CACHE_DIR = Path("/kaggle/working/output/action_genome/cache")   # = AG_OUT / "cache"
AG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Nếu có cache CLIP của lần chạy trước được Add Input thì dùng lại, khỏi mã hoá frame.
for _f in glob.glob("/kaggle/input/**/clip_ag_*.npz", recursive=True):
    _dst = AG_CACHE_DIR / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)

_n_jpg = len(os.listdir(Path(DATA_PATH) / "ag_frames"))
print("AG root:", DATA_PATH, "|", _n_jpg, "frame")

# v6: AG giữ backbone ViT-L/14 (dùng lại cache CLIP) và chỉ 3 ablation chính; đối chứng quan hệ chạy riêng ở cuối Phần 3.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-L-14", "laion2b_s32b_b82k", 128   # mô hình cuối (version 9); đổi thành "ViT-H-14", "laion2b_s32b_b79k", 64 để chạy ViT-H
for _f in (clip_text_features, encode_free_text):          # v6: bỏ model CLIP ViT-B/32 đã cache ở Phần 1
    for _a in ("_m", "_t"):
        _f.__dict__.pop(_a, None)
CFG.ablations = ("none", "no_class_edges", "no_relations")


In [ ]:
# =====================================================================================
# v6 — HAI ĐỐI CHỨNG KIỂM TRA "CHÍNH QUAN HỆ mang thông tin" trên AG
#  (1) composite_swap: mỗi node quan hệ 'v o' của MỌI video bị thay bằng 'v′ o' (v′ ≠ v, lấy theo
#      phân bố vị từ của vật o trên TRAIN). Giữ nguyên vật và số node ⇒ nếu hiệu quả trên bộ quan hệ
#      khó mất đi thì thông tin nằm ở QUAN HỆ ĐÚNG, không phải ở việc có thêm node.
#  (2) Truy vấn phản thực tế: đổi vị từ trong truy vấn ('drinking … cup' → 'holding … cup') sao cho
#      video đúng KHÔNG có quan hệ mới. Mô hình hiểu quan hệ phải hạ hạng video đúng; CLIP gần như không.
# =====================================================================================
import copy
AG_VO = defaultdict(Counter)                     # vật → phân bố vị từ (CHỈ train), dựng khi có ag_records


def _ag_vo():
    if not AG_VO:
        tr_ids = set(ag_split["train"])
        for r in ag_records:
            if r["image_id"] in tr_ids:
                for c in r["objects"]:
                    if " " in c:
                        v, o = c.split(" ", 1); AG_VO[o][v] += 1
    return AG_VO


# ---------------- truy vấn phản thực tế: đổi vị từ ----------------
VERB_ING = dict(look="looking", hold="holding", carry="carrying", drink="drinking", eat="eating",
                sit="sitting", stand="standing", lie="lying", lean="leaning", touch="touching",
                wear="wearing", wipe="wiping", write="writing", twist="twisting")


def build_ag_counterfactuals(recs, gal):
    out = []
    for q, cand in gal:
        text, objs = recs[q]["caption"], set(recs[q]["objects"])
        toks = re.findall(r"[A-Za-z]+|[^A-Za-z]+", text)
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t.lower()), _ag_lemma(t.lower())) for t in toks if t.isalpha()}
        for k, t in enumerate(toks):
            if not t.isalpha():
                continue
            v = _ag_lemma(t.lower(), "v"); v = AG_VERB_ALIAS.get(v, v)
            hit = sorted(c for c in objs if " " in c and c.split(" ", 1)[0] == v and c.split(" ", 1)[1] in nouns)
            if not hit:
                continue
            o = hit[0].split(" ", 1)[1]
            alts = [x for x, _ in _ag_vo()[o].most_common() if x != v and f"{x} {o}" not in objs and x in VERB_ING]
            if not alts:
                continue
            new = list(toks); new[k] = VERB_ING[alts[0]]
            out.append(dict(q=int(q), cand=np.asarray(cand), text=text, cf="".join(new), rel=hit[0],
                            cf_rel=f"{alts[0]} {o}"))
            break
    return out


REL_CF, REL_CF_FEATS, REL_CF_RES = {}, {}, defaultdict(dict)


def _init_ag_cf():
    """Dựng bộ truy vấn phản thực tế từ bộ quan hệ khó của AG (gọi một lần, lúc hook chạy lần đầu)."""
    _ag_vo()
    REL_CF["ag"] = build_ag_counterfactuals(ag_records, REL_GALLERIES["ag"])
    texts = sorted({x["text"] for x in REL_CF["ag"]} | {x["cf"] for x in REL_CF["ag"]})
    REL_CF_FEATS.update(zip(texts, clip_text_features(texts)))        # CÙNG bộ mã hoá cho cả 2 phiên bản
    log(f"truy vấn phản thực tế: {len(REL_CF['ag'])}/{len(REL_GALLERIES['ag'])} truy vấn của bộ quan hệ khó")
    for x in REL_CF["ag"][:4]:
        print(f"  • [{x['rel']} → {x['cf_rel']}]\n      gốc : {x['text'][:110]}\n      đổi : {x['cf'][:110]}")


@torch.no_grad()
def cf_ranks(model, eng, cfs, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    out = []
    for x in cfs:
        cand = x["cand"]
        row = {}
        for tag in ("text", "cf"):
            qf = torch.tensor(REL_CF_FEATS[x[tag]][None])
            cids = [n for n, _, _ in parse_query(x[tag], eng.graph)]
            s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
            pm = torch.tensor(path_score_matrix(eng.graph, cids, cand)[None]).to(DEVICE)
            s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
            s_path = model.path_score(pm).cpu().numpy()
            fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
            row[f"rank_{tag}"] = 1 + int((fin > fin[0]).sum())
            row[f"clip_rank_{tag}"] = 1 + int((s_clip[0] > s_clip[0, 0]).sum())
        out.append(row)
    return pd.DataFrame(out)


if "_relprobe_hook_v5" not in globals():
    _relprobe_hook_v5 = relprobe_hook


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    _relprobe_hook_v5(ab, seed, model, eng, parts, alpha, lam)
    if REL_GAL_KEY == "ag" and "ag" not in REL_CF and "ag" in REL_GALLERIES:
        _init_ag_cf()
    if REL_GAL_KEY in REL_CF:
        d = cf_ranks(model, eng, REL_CF[REL_GAL_KEY], alpha, lam)
        name = REL_RENAME.get(ab, ab)
        REL_CF_RES[REL_GAL_KEY][(name, seed)] = d
        REL_CF_RES[REL_GAL_KEY].setdefault(("CLIP", seed), d.assign(rank_text=d.clip_rank_text,
                                                                    rank_cf=d.clip_rank_cf))
        print(f"    [phản thực tế] {name} seed {seed}: video đúng bị hạ hạng ở "
              f"{(d.rank_cf > d.rank_text).mean():.1%} truy vấn (CLIP {(d.clip_rank_cf > d.clip_rank_text).mean():.1%})")


def cf_report(key, base="none", n_boot=10000):
    res = REL_CF_RES[key]
    names = list(dict.fromkeys(a for a, _ in res))
    rows, per = [], {}
    for a in names:
        ds = [d for (b, s), d in res.items() if b == a]
        drr = np.mean([1 / d.rank_text - 1 / d.rank_cf for d in ds], 0)      # RR(gốc) − RR(đổi vị từ), TB theo seed
        per[a] = drr
        rows.append(dict(model=a, n=len(drr), seeds=len(ds),
                         demoted=np.mean([(d.rank_cf > d.rank_text).mean() for d in ds]),
                         promoted=np.mean([(d.rank_cf < d.rank_text).mean() for d in ds]),
                         delta_RR=drr.mean()))
    tab = pd.DataFrame(rows)
    rng = np.random.RandomState(0); sig_rows = []
    for b in names:
        if b == base:
            continue
        d = per[base] - per[b]
        bs = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        sig_rows.append(dict(a=base, b=b, n=len(d), diff_delta_RR=d.mean(),
                             ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5)))
    return tab, pd.DataFrame(sig_rows)


In [ ]:
# =====================================================================================
# v8 — ĐỒ THỊ CẢNH KHÔNG–THỜI GIAN: Action Genome + nhãn hành động / bối cảnh của Charades
#   Chẩn đoán (CPU): chỉ với annotation AG (35 loại vật, 25 quan hệ), một ánh xạ tuyến tính truy vấn→annotation
#   đã đạt MRR 0,36–0,39 — ngang toàn bộ hệ thống GNN (0,37–0,38) ⇒ giới hạn là LƯỢNG THÔNG TIN trong đồ thị:
#   truy vấn mô tả hành động (mở cửa, đi lại, cầm lên…) mà annotation AG không có. AG được dựng trên Charades;
#   Charades có nhãn hành động (157 lớp, kèm mốc thời gian) và bối cảnh (phòng) do người chú thích cho từng video.
#   Thêm vào đồ thị, cho MỌI video (cùng điều kiện "gallery đã được chú thích" như annotation AG):
#     • node hành động 'động_từ vật' ('open door', 'put clothes'…) / động từ đơn ('sneeze') + vật của hành động
#     • node bối cảnh ('kitchen', 'living room'…)
#     • bộ ba THỜI GIAN cấp video ⟨a before b⟩ (a bắt đầu trước b ≥ 1 giây) — mẫu P6; truy vấn: thứ tự nhắc tới
#   KHÔNG dùng cột 'script' / 'descriptions' / 'objects' của Charades (văn bản gần với truy vấn).
#   Đối chứng (cell sau thí nghiệm chính): no_actions (đồ thị AG như version 4), shuffled_actions (nhãn Charades
#   của video KHÁC cùng split).
# =====================================================================================
import csv, zipfile, urllib.request

CH_URL = "https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip"
CH_SCENE = {"Kitchen": "kitchen", "Bedroom": "bedroom", "Living room": "living room", "Bathroom": "bathroom",
            "Home Office": "office", "Entryway": "entryway", "Dining room": "dining room", "Stairs": "stair",
            "Hallway": "hallway", "Laundry room": "laundry room", "Closet": "closet", "Recreation room": "recreation room",
            "Pantry": "pantry", "Garage": "garage", "Basement": "basement"}
CH, CH_ACTION_NODES = {}, set()


def _ch_dir():
    hits = glob.glob("/kaggle/input/**/Charades_v1_train.csv", recursive=True)
    if hits:
        return Path(hits[0]).parent
    d = (Path("/kaggle/working") if RUN_ON_KAGGLE else Path(".")) / "charades"
    if not (d / "Charades" / "Charades_v1_train.csv").exists():
        d.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(CH_URL, d / "Charades.zip")
        with zipfile.ZipFile(d / "Charades.zip") as z:
            z.extractall(d)
    return d / "Charades"


def _ch_init():
    """Nạp Charades một lần; mở rộng tập động từ của ag_parse_query và IsA 'động_từ vật' → 'động_từ'."""
    if CH:
        return
    d = _ch_dir()
    two = lambda fn: dict(l.strip().split(" ", 1) for l in open(d / fn, encoding="utf-8") if l.strip())
    verb = {k: AG_VERB_ALIAS.get(v.strip().lower(), v.strip().lower()) for k, v in two("Charades_v1_verbclasses.txt").items()}
    obj = {k: (None if v.strip() == "None" else ag_canon(v)) for k, v in two("Charades_v1_objectclasses.txt").items()}
    name = {}
    for l in open(d / "Charades_v1_mapping.txt", encoding="utf-8"):
        c, o, v = l.split()
        name[c] = (verb[v], obj[o])
    rows = {}
    for f in ("train", "test"):
        for row in csv.DictReader(open(d / f"Charades_v1_{f}.csv", encoding="utf-8")):
            rows[row["id"]] = row
    CH.update(rows=rows, name=name)
    _AG_VERBS.update(verb.values())
    for v, o in name.values():
        node = f"{v} {o}" if o else v
        CH_ACTION_NODES.add(node)
        if o:
            FALLBACK_ISA.setdefault(node, [v])
    print(f"Charades: {len(rows)} video | {len(CH_ACTION_NODES)} node hành động | tập động từ truy vấn: {len(_AG_VERBS)}")


def charades_info(video_id):
    row = CH["rows"].get(video_id)
    out = dict(nodes=set(), rels=set(), triples=[])
    if row is None:
        return out
    acts = []
    for x in (row.get("actions") or "").split(";"):
        p = x.split()
        if len(p) == 3 and p[0] in CH["name"]:
            v, o = CH["name"][p[0]]
            node = f"{v} {o}" if o else v
            acts.append((float(p[1]), node))
            out["nodes"].add(node)
            if o:
                out["nodes"].add(o); out["rels"].add((o, "afford", node))
    sc = next((v for k, v in CH_SCENE.items() if (row.get("scene") or "").startswith(k)), None)
    if sc:
        out["nodes"].add(sc)
    acts.sort()
    tri = {(a, "before", b) for i, (s1, a) in enumerate(acts) for s2, b in acts[i + 1:] if a != b and s2 >= s1 + 1.0}
    out["triples"] = sorted(tri)
    return out


def apply_ag_actions(recs, mode="full", seed=0):
    """Gán objects / relations / img_triples của từng video theo mode: full | none | shuffled (trong cùng split)."""
    src = list(range(len(recs)))
    if mode == "shuffled":
        rng = np.random.RandomState(seed)
        for k in ("train", "val", "test"):
            idx = np.array([i for i, r in enumerate(recs) if r["split"] == k])
            for a, b in zip(idx, rng.permutation(idx)):
                src[a] = int(b)
    for i, r in enumerate(recs):
        ch = recs[src[i]] if mode != "none" else None
        r["objects"] = sorted(set(r["ag_objects"]) | (set(ch["ch_nodes"]) if ch else set()))
        r["relations"] = sorted(set(map(tuple, r["ag_relations"])) | (set(map(tuple, ch["ch_rels"])) if ch else set()))
        r["img_triples"] = [(tuple(t), 1.0) for t in ch["ch_triples"]] if ch else []
    return recs


def ag_temporal_hits(names_in_order, graph, seen):
    """Bộ ba thời gian của TRUY VẤN: hai hành động được nhắc liên tiếp ⇒ ⟨trước before sau⟩ (nếu là node của đồ thị)."""
    hits = []
    if not getattr(graph, "n_tri", 0):
        return hits
    acts = [n for n in dict.fromkeys(names_in_order) if n in CH_ACTION_NODES]
    for a, b in zip(acts, acts[1:]):
        nid = graph.tri2id.get((a, "before", b))
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, f"{a} before {b}", f"{a} … {b}"))
    return hits


In [ ]:
# =====================================================================================
# CẤP ĐỘ 2 — DATASET THỨ HAI: ACTION GENOME (text-to-video)
# Chạy toàn bộ pipeline trên AG mà KHÔNG ghi đè kết quả Flickr8k.
# =====================================================================================
import os
import nltk
from contextlib import contextmanager
from nltk.stem import WordNetLemmatizer
import pandas as pd
for _p in ("wordnet", "omw-1.4"):
    try: nltk.download(_p, quiet=True)
    except Exception: pass

_AG_LEM = WordNetLemmatizer(); _AG_LEM_CACHE = {}

def _ag_lemma(w, pos="n"):
    k = (w, pos)
    if k not in _AG_LEM_CACHE:
        try: _AG_LEM_CACHE[k] = _AG_LEM.lemmatize(w, pos)
        except Exception: _AG_LEM_CACHE[k] = w
    return _AG_LEM_CACHE[k]

# ---- Từ vựng AG: phần lấy từ dữ liệu vs phần viết tay (ghi rõ trong báo cáo) ----
_ag_hits = glob.glob(f"{DATA_PATH}/**/ag_records.json", recursive=True)
assert _ag_hits, "Chưa Add Input output của notebook ag-download"
_ag_payload = json.load(open(_ag_hits[0]))

# (1) Quan hệ người–vật → động từ gốc. Ánh xạ thủ công 1-1 từ tên lớp AG; đã kiểm tra bên dưới.
#     Bỏ: quan hệ không gian, quan hệ phủ định/mơ hồ, covered_by (ngược chiều ngữ nghĩa).
AG_PRED2VERB = {"looking_at": "look", "holding": "hold", "carrying": "carry",
                "have_it_on_the_back": "carry", "drinking_from": "drink", "eating": "eat",
                "sitting_on": "sit", "standing_on": "stand", "lying_on": "lie", "leaning_on": "lean",
                "touching": "touch", "wearing": "wear", "wiping": "wipe", "writing_on": "write",
                "twisting": "twist"}
_pred_seen = Counter(p for r in _ag_payload["records"] for _, p, _ in r["triples"])
print("Predicate có trong dữ liệu nhưng KHÔNG dùng:",
      {p: n for p, n in _pred_seen.items() if p not in AG_PRED2VERB})
_bad = [p for p in AG_PRED2VERB if p not in _pred_seen]
assert not _bad, f"Khóa không tồn tại trong annotation AG: {_bad}"

# (2) Alias danh từ: SINH TỰ ĐỘNG từ tên lớp nhiều tên của AG (vd 'cup/glass/bottle')
_ag_classes = {c for r in _ag_payload["records"] for c in r["objects"]}
AG_NOUN_ALIAS = {_ag_lemma(alt.strip().lower()): _ag_lemma(c.split("/")[0].strip().lower())
                 for c in _ag_classes if "/" in c for alt in c.split("/")[1:]}
print("Alias danh từ sinh từ dữ liệu:", AG_NOUN_ALIAS)

# (3) Alias động từ: VIẾT TAY, căn cứ là tần suất động từ trong mô tả của tập TRAIN
AG_VERB_ALIAS = {"watch": "look", "grab": "hold"}
_AG_VERBS = set(AG_PRED2VERB.values())
_train_verbs = Counter(
    _ag_lemma(t, "v") for r in _ag_payload["records"] if r["split"] == "train"
    for t in _norm(r["caption"]).split()
    if t not in STOP and _ag_lemma(t, "v") != t)        # xấp xỉ: token có dạng biến tố động từ
print("Động từ phổ biến trên TRAIN chưa được ánh xạ:",
      [(v, n) for v, n in _train_verbs.most_common(40)
       if v not in _AG_VERBS and v not in AG_VERB_ALIAS][:20])


def ag_canon(cls):
    """'cup/glass/bottle' → 'cup'; 'groceries' → 'grocery' (khớp lemma phía truy vấn)."""
    return _ag_lemma(cls.split("/")[0].strip().lower().replace("_", " "))


def load_action_genome():
    root = Path(_ag_hits[0]).parent
    out = []
    for i, r in enumerate(_ag_payload["records"]):
        objs = {ag_canon(c) for c in r["objects"]} - {"person"}   # video nào cũng có người
        comps, rels = set(), set()
        for kind, pred, cls in r["triples"]:
            v, o = AG_PRED2VERB.get(pred), ag_canon(cls)
            if v is None or o == "person":
                continue
            c = f"{v} {o}"                       # node ghép, vd 'hold cup'
            comps.add(c)
            rels.add((o, "afford", c))           # cup –afford→ hold cup (gộp CHỈ trên train)
        out.append(dict(image_id=i, video_id=r["video_id"], split=r["split"],
                        caption=r["caption"], captions=r["descriptions"],
                        objects=sorted(objs | comps), relations=sorted(rels), url="",
                        frames=[str(root / "ag_frames" / f) for f in r["frames"]]))
    _ch_init()                                   # v8: nhãn hành động / bối cảnh / thứ tự thời gian của Charades
    for r_ in out:
        ch_ = charades_info(r_["video_id"])
        r_["ag_objects"], r_["ag_relations"] = list(r_["objects"]), list(r_["relations"])
        r_["ch_nodes"], r_["ch_rels"], r_["ch_triples"] = sorted(ch_["nodes"]), sorted(ch_["rels"]), ch_["triples"]
    apply_ag_actions(out, "full")
    print(f"  + Charades: {np.mean([len(r['ch_nodes']) for r in out]):.1f} node hành động/bối cảnh mỗi video | "
          f"{np.mean([len(r['ch_triples']) for r in out]):.1f} bộ ba thời gian mỗi video")
    print(f"Action Genome: {len(out)} video | "
          f"{np.mean([len(r['objects']) for r in out]):.1f} node/video | "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/video | "
          f"{np.mean([len(r['frames']) for r in out]):.1f} frame/video")
    for r in out[:3]:
        print("  •", r["caption"][:80], "→", r["objects"][:8])
    return out


def encode_action_genome(recs):
    """CLIP từng frame → mean-pool theo thời gian → L2. Văn bản = mô tả Charades."""
    cache = CFG.out_dir / "cache" / f"clip_ag_{CFG.clip_model}_{CFG.clip_weights}_{len(recs)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model,
                                                                 pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    items = [(i, p) for i, r in enumerate(recs) for p in r["frames"]]
    missing = [p for _, p in items if not os.path.exists(p)]
    if len(missing) > 0.01 * len(items) and "ag_ensure_frames" in globals():
        print(f"Thiếu {len(missing)}/{len(items)} frame (cache CLIP không khớp) → chép frame từ Drive…")
        _DONE.unlink(missing_ok=True)            # bỏ dấu 'chỉ annotation' để chép đủ
        ag_ensure_frames()
        missing = [p for _, p in items if not os.path.exists(p)]
    assert len(missing) <= 0.01 * len(items), \
        f"{len(missing)}/{len(items)} frame không tồn tại (vd {missing[:2]}) — kiểm tra DATA_PATH"
    class FrameDS(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, k):
            try: im = Image.open(items[k][1]).convert("RGB")
            except Exception: im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    feats = []
    with torch.no_grad():
        from tqdm.auto import tqdm
        log(f"CLIP: mã hoá {len(items)} frame của {len(recs)} video (batch {CFG.clip_bs})")
        for x in tqdm(DataLoader(FrameDS(), batch_size=CFG.clip_bs, num_workers=2),
                      desc="CLIP frame", unit="batch"):
            feats.append(F.normalize(model.encode_image(x.to(DEVICE)), dim=-1).float().cpu().numpy())
        feats = np.concatenate(feats)
        img = np.zeros((len(recs), feats.shape[1]), np.float32)
        np.add.at(img, np.array([i for i, _ in items]), feats)
        img /= np.linalg.norm(img, axis=1, keepdims=True).clip(1e-8)
        caps = [r["caption"] for r in recs]
        # Mô tả Charades thường dài hơn 77 token ⇒ open_clip cắt cụt. Tách theo câu, mã hoá từng câu,
        # lấy trung bình. Giữ cả bản cắt cụt; chọn bản nào bằng MRR CLIP trên VAL (không nhìn test).
        pieces, owner = [], []
        for i, t in enumerate(caps):
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip().split()) >= 2] or [t]
            pieces += sents; owner += [i] * len(sents)
        enc = lambda xs: np.concatenate([F.normalize(model.encode_text(tokenizer(xs[s:s + CFG.clip_bs]).to(DEVICE)),
                                                     dim=-1).float().cpu().numpy()
                                         for s in range(0, len(xs), CFG.clip_bs)])
        txt_trunc = enc(caps)
        E = enc(pieces); txt_chunk = np.zeros_like(txt_trunc)
        np.add.at(txt_chunk, np.array(owner), E)
        txt_chunk /= np.linalg.norm(txt_chunk, axis=1, keepdims=True)
        va = np.array([i for i, r in enumerate(recs) if r["split"] == "val"])
        mrr = lambda T: reciprocal_ranks(np.tile(va, (len(va), 1)), T[va] @ img[va].T, va).mean()
        m_t, m_c = mrr(txt_trunc), mrr(txt_chunk)
        assert max(m_t, m_c) > 0.02, \
            f"CLIP VAL MRR {max(m_t, m_c):.4f} ≈ ngẫu nhiên — frame hỏng hoặc ảnh/văn bản lệch thứ tự"
        txt = txt_chunk if m_c > m_t else txt_trunc
        log(f"Văn bản AG — VAL MRR CLIP: cắt cụt {m_t:.4f} | theo câu {m_c:.4f} → dùng "
            f"{'theo câu' if m_c > m_t else 'cắt cụt'}")
    log(f"CLIP xong: ảnh {img.shape}, văn bản {txt.shape} → cache {cache.name}")
    del model; torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


def ag_make_split(recs, seed=None):
    """Split lấy từ ag_records.json: train/test gốc của AG, val tách bằng seed cố định."""
    sp_ = {k: sorted(r["image_id"] for r in recs if r["split"] == k) for k in ("train", "val", "test")}
    sp_["seed"] = "ag-official-train-test + val seed 20240101"
    payload = json.dumps({k: sp_[k] for k in ("train", "val", "test")}, sort_keys=True)
    sp_["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(sp_, f)
    print(f"Split AG (sha1={sp_['sha1']}): train={len(sp_['train'])} "
          f"val={len(sp_['val'])} test={len(sp_['test'])}")
    assert not (set(sp_["train"]) & set(sp_["test"])) and not (set(sp_["val"]) & set(sp_["test"]))
    return sp_


def ag_parse_query(text, graph):
    """'A person is drinking from a glass' → ['drink', 'cup'] → khớp 'drink cup' và 'cup'."""
    toks = []
    for t in _norm(text).split():
        if t in STOP or len(t) < 2:
            continue
        v = _ag_lemma(t, "v"); v = AG_VERB_ALIAS.get(v, v)
        if v in _AG_VERBS:
            toks.append(v); continue
        n = _ag_lemma(t, "n"); toks.append(AG_NOUN_ALIAS.get(n, n))
    cands = [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    if "ag_temporal_hits" in globals():          # v8: bộ ba thời gian theo thứ tự nhắc tới trong truy vấn
        big = {i for i in range(len(toks) - 1) if " ".join(toks[i:i + 2]) in CH_ACTION_NODES}      # hành động dạng 'động_từ vật'
        order = sorted([(i, " ".join(toks[i:i + 2])) for i in big] +
                       [(i, toks[i]) for i in range(len(toks)) if i not in big and (i - 1) not in big])
        hits += ag_temporal_hits([g for _, g in order], graph, seen)
    return hits


@contextmanager
def use_dataset(out_dir, **overrides):
    """Tạm thay hàm toàn cục + thư mục output; khôi phục nguyên trạng khi ra khỏi khối."""
    g = globals()
    saved = {k: g.get(k) for k in overrides}
    saved_out = CFG.out_dir
    g.update(overrides)
    CFG.out_dir = Path(out_dir)
    for sub in ("cache", "results"):
        (CFG.out_dir / sub).mkdir(parents=True, exist_ok=True)
    _GROUP_CACHE.clear()
    try:
        yield
    finally:
        for k, v in saved.items():
            if v is None: g.pop(k, None)
            else: g[k] = v
        CFG.out_dir = saved_out
        _GROUP_CACHE.clear()


# (4) IsA cho node ghép 'động từ vật' → khái niệm hành động 'động từ' (vd 'hold cup' –là_một→ 'hold').
#     WordNet chỉ phủ 31/202 lớp AG nên tầng khái niệm gần như rỗng; đây là tri thức NGOÀI
#     (cấu trúc tên lớp), không phải thống kê ⇒ an toàn cho test. Truy vấn "someone holding
#     something" giờ có đường đi P2/P4/P5 tới 'hold' thay vì không khớp node nào.
#     Ghi vào FALLBACK_ISA toàn cục để mọi cell AG sau (semantic linking) dùng cùng một đồ thị;
#     tên 'động từ vật' không bao giờ xuất hiện trong Flickr8k nên không ảnh hưởng phần trước.
AG_ACTION_ISA = {f"{v} {ag_canon(cls)}": [v]
                 for r in _ag_payload["records"] for _, pred, cls in r["triples"]
                 for v in [AG_PRED2VERB.get(pred)] if v is not None and ag_canon(cls) != "person"}
FALLBACK_ISA.update(AG_ACTION_ISA)
print(f"IsA hành động: {len(AG_ACTION_ISA)} node ghép → {len(set(AG_PRED2VERB.values()))} khái niệm động từ")

# ------------------------------------ LOG TIẾN ĐỘ ------------------------------------
# Mọi dòng có [mm:ss từ đầu cell | +ss so với dòng trước] và flush ngay (Colab hay giữ buffer).
# Các hàm nặng của pipeline được BỌC lại qua use_dataset(...) — chỉ có hiệu lực trong khối
# with, tự khôi phục khi ra khỏi khối, không sửa cell nào khác.
_T0 = _TL = time.time()


def log(msg):
    global _TL
    now = time.time()
    m, s_ = divmod(int(now - _T0), 60)
    print(f"[{m:02d}:{s_:02d} | +{now - _TL:5.1f}s] {msg}", flush=True)
    _TL = now


@contextmanager
def step(title):
    log(f"▶ {title}")
    t = time.time()
    yield
    log(f"✔ {title} ({time.time() - t:.1f}s)")


_N_ABL = len(CFG.ablations); _N_SEED = len(CFG.seeds)
_PROG = dict(abl=None, i_abl=0, i_seed=0)
_orig = dict(prepare=prepare, train_graph_scorer=train_graph_scorer, tune_on_val=tune_on_val,
             evaluate_split=evaluate_split, self_test=self_test)


def _prepare_logged(records, split, img_feat, txt_feat, ablation="none", seed=0):
    if ablation != _PROG["abl"]:
        _PROG.update(abl=ablation, i_abl=_PROG["i_abl"] + 1, i_seed=0)
    _PROG["i_seed"] += 1
    tag = f"[ablation {_PROG['i_abl']}/{_N_ABL} '{ablation}' | seed {_PROG['i_seed']}/{_N_SEED}]"
    with step(f"{tag} dựng đồ thị + bộ truy vấn"):
        out = _orig["prepare"](records, split, img_feat, txt_feat, ablation, seed)
    g = out[0]
    log(f"    nodes={g.n_nodes} | scene={g.A_scene.nnz} related={g.A_related.nnz} isa={g.A_isa.nnz}"
        f" | đường đi P1..P5={sum(g.R[p].nnz for p in PATTERNS[1:])}")
    return out


def _train_logged(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    with step(f"  huấn luyện GNN seed={seed} ({CFG.epochs} epoch, {len(train_q_idx)} truy vấn)"):
        model, hist = _orig["train_graph_scorer"](engine, train_q_idx, train_img_idx,
                                                  seed=seed, verbose=verbose)
    gates = model.edge_gates()
    log(f"    loss {hist[0]:.3f} → {hist[-1]:.3f} | clip_w {F.softplus(model.clip_logit).item():.2f}"
        f" | path_w {F.softplus(model.path_logit).item():.2f}"
        f" | w_path " +
        " ".join(f"{x:.3f}" for x in model.pattern_weights().detach().cpu().numpy()) +
        (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4]) if gates else ""))
    return model, hist


def _tune_logged(model, engine, val_q, val_gallery, verbose=True):
    with step(f"  quét (α, λ) trên VAL ({len(CFG.alpha_grid) * len(CFG.lambda_grid)} cặp)"):
        (a, l), grid = _orig["tune_on_val"](model, engine, val_q, val_gallery, verbose=False)
    best = max(grid, key=lambda r: r["MRR"])
    log(f"    chọn α={a}, λ={l} | VAL MRR={best['MRR']:.4f} (CLIP thuần: "
        f"{next(r['MRR'] for r in grid if r['alpha'] == 1.0):.4f})"
        + ("  ⚠ λ=1 ⇒ s_path không tham gia điểm cuối" if l == 1.0 else ""))
    return (a, l), grid


def _eval_logged(model, engine, part, alpha, lam, by_group=True):
    res, raw = _orig["evaluate_split"](model, engine, part, alpha, lam, by_group)
    log(f"    TEST: CLIP MRR={res['CLIP']['MRR']:.4f} → CLIP+Graph MRR={res['CLIP+Graph']['MRR']:.4f}")
    return res, raw


def _self_test_logged(*a, **k):
    with step("self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu"):
        return _orig["self_test"](*a, **k)


# ------------------------------------ CHẠY ------------------------------------
AG_OUT = Path("/kaggle/working/output/action_genome")
log(f"Bắt đầu Action Genome | ablations={CFG.ablations} | seeds={CFG.seeds} | epochs={CFG.epochs}"
    f" | device={DEVICE}")
with use_dataset(AG_OUT, parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged,
                 self_test=_self_test_logged):
    R = CFG.out_dir / "results"

    with step("1/9 nạp annotation AG"):
        ag_records = load_action_genome()
    with step("2/9 split"):
        ag_split = make_split(ag_records)
    with step("3/9 đặc trưng CLIP (dùng cache nếu có)"):
        ag_img, ag_txt = encode_action_genome(ag_records)
        if "ag_save_clip_cache_to_drive" in globals():
            ag_save_clip_cache_to_drive()
    with step("4/9 self-test"):
        self_test(ag_records, ag_split, ag_img, ag_txt)
    if "pick_learn_clip_w" in globals():
        with step("4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0)"):
            clipw_ag = pick_learn_clip_w(ag_records, ag_split, ag_img, ag_txt)

    with step(f"5/9 thí nghiệm chính: {_N_ABL} ablation × {_N_SEED} seed = {_N_ABL * _N_SEED} lần huấn luyện"):
        _PROG.update(abl=None, i_abl=0, i_seed=0)
        REL_GALLERIES["ag"] = build_rel_galleries_ag(ag_records, ag_split)      # v5: bộ quan hệ khó
        REL_GAL_KEY, REL_RENAME = "ag", {}
        ag_all_res, ag_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                            ablations=CFG.ablations)
        REL_GAL_KEY = None
    ag_df = results_table(ag_all_res)
    import pickle
    with open(R / "per_query_all_ablations.pkl", "wb") as f:      # cho kiểm định cặp về sau
        pickle.dump(dict(PER_QUERY), f)
    ag_df.to_csv(R / "main_results.csv", index=False)
    display(ag_df)
    PQ_AG = dict(PER_QUERY) if "PER_QUERY" in globals() else {}   # cho cell kiểm định cặp

    ag_piv = pd.DataFrame(ag_art["grid"]).pivot_table(index="alpha", columns="lam", values="MRR")
    ag_piv.to_csv(R / "alpha_sensitivity.csv")
    print("\nMRR trên VALIDATION theo (α, λ):\n", ag_piv.round(4).to_string())

    with step("6/9 phân tích 5 thành công / 5 thất bại"):
        ag_cases = analyse_cases(ag_art, ag_records)
    for c in ag_cases:
        print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → "
              f"CLIP+Graph {c['rank_graph']}\n  truy vấn: {c['query']}")
        for e in c["explanation"]:
            print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    with open(R / "case_analysis.json", "w") as f:
        json.dump(ag_cases, f, ensure_ascii=False, indent=2)

    with step("7/9 fidelity trên điểm đường đi"):
        ag_fdf, ag_summ = fidelity_study(ag_art)
    ag_fdf.to_csv(R / "fidelity_per_query.csv", index=False)
    with open(R / "xai_summary.json", "w") as f:
        json.dump(ag_summ, f, indent=2)
    print("\nxAI (điểm đường đi):", {k: round(v, 4) if isinstance(v, float) else v
                                     for k, v in ag_summ.items()})

    if "final_rank_fidelity" in globals():
        with step("8/9 fidelity trên thứ hạng điểm cuối"):
            ag_rdf, ag_rsumm = final_rank_fidelity(ag_art)
        ag_rdf.to_csv(R / "fidelity_final_rank.csv", index=False)
        with open(R / "xai_final_rank_summary.json", "w") as f:
            json.dump(ag_rsumm, f, indent=2)
        print("xAI (thứ hạng điểm cuối):", {k: round(v, 3) if isinstance(v, float) else v
                                           for k, v in ag_rsumm.items()})

    with step("9/9 xuất bộ truy vấn đánh giá + demo"):
        export_eval_queries(ag_art["engine"].queries, ag_art["graph"],
                            R / "eval_queries_20.json", n=20, allowed_ids=ag_split["test"])
        demo_query("a person drinking from a glass while sitting on the couch", ag_art, k=3)
        demo_query("someone watching tv and holding a phone", ag_art, k=3)

# ------------------- Bảng tổng hợp cấp độ 2: hai bộ dữ liệu -------------------
if "df" in globals():
    both = pd.concat([df.assign(dataset="flickr8k"), ag_df.assign(dataset="action_genome")])
    both = both[both.ablation == "none"][["dataset", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]]
    both.to_csv(Path("/kaggle/working/results") / "level2_two_datasets.csv", index=False)
    display(both)
log("XONG toàn bộ cell Action Genome")

In [ ]:
# =====================================================================================
# v8 — ĐỐI CHỨNG cho đồ thị không–thời gian trên AG: no_actions (đồ thị AG như version 4) và
#       shuffled_actions (nhãn Charades của video KHÁC) + kiểm định cặp + bộ quan hệ khó + phản thực tế
# =====================================================================================
import copy
R_AG = AG_OUT / "results"
ag_vs = [paired_vs_clip(PQ_AG, ab).assign(dataset="AG") for ab in CFG.ablations]
ctrl_df, PQ_CTRL = [], {}
with use_dataset(AG_OUT / "controls", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    for _name, _mode in (("no_actions", "none"), ("shuffled_actions", "shuffled")):
        _recs = apply_ag_actions(copy.deepcopy(ag_records), _mode)
        _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
        REL_GAL_KEY, REL_RENAME = "ag", {"none": _name}
        _res, _ = run_experiment(_recs, ag_split, ag_img, ag_txt, ablations=("none",))
        REL_GAL_KEY, REL_RENAME = None, {}
        ctrl_df.append(results_table(_res).assign(ablation=_name))
        _pq = {(_name + ab[4:], s): v for (ab, s), v in PER_QUERY.items() if ab == "none" or ab.startswith("none+")}
        PQ_CTRL.update(_pq); ag_vs.append(paired_vs_clip(_pq, _name).assign(dataset="AG"))
_N_ABL = len(CFG.ablations)

pd.concat([ag_df] + ctrl_df).to_csv(R_AG / "main_results_with_controls.csv", index=False)
ag_vs = pd.concat(ag_vs); ag_vs.to_csv(R_AG / "paired_graph_vs_clip.csv", index=False)
print("AG — CLIP+Graph vs CLIP (kiểm định cặp):"); print(ag_vs[ag_vs.group == "all"].round(4).to_string())
sig_ctrl = pd.concat([paired_test({**PQ_AG, **PQ_CTRL}, "none", b) for b in ("no_actions", "shuffled_actions")] +
                     [paired_test(PQ_AG, "none", b) for b in CFG.ablations if b != "none"])
sig_ctrl.to_csv(R_AG / "paired_ablation_test.csv", index=False)
print("\nAG — đồ thị đầy đủ (none) vs từng cấu hình:"); print(sig_ctrl[sig_ctrl.group == "all"].round(4).to_string())
rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_actions", "shuffled_actions", "no_class_edges", "no_relations"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
print("\nBộ quan hệ khó (AG):"); print(rel_tab_ag.to_string()); print(rel_sig_ag.round(4).to_string())
if REL_CF_RES.get("ag"):
    cf_tab, cf_sig = cf_report("ag")
    cf_tab.to_csv(R_AG / "counterfactual_verb_swap.csv", index=False); cf_sig.to_csv(R_AG / "counterfactual_verb_swap_paired.csv", index=False)
    print("\nTruy vấn phản thực tế:"); print(cf_tab.round(4).to_string()); print(cf_sig.round(4).to_string())


In [ ]:
# =====================================================================================
# v9 — BÁO CÁO NHÁNH KHỚP (Action Genome)
# =====================================================================================
R_AG = AG_OUT / "results"
_P = {**PQ_AG, **PQ_CTRL}
_names = ["none", "none+clipmatch", "none+match", "no_class_edges+match", "no_relations+match",
          "no_actions", "no_actions+match", "shuffled_actions", "shuffled_actions+match"]
match_vs_clip_ag = pd.concat([paired_vs_clip(_P, n).assign(dataset="AG") for n in _names if any(k[0] == n for k in _P)])
match_vs_clip_ag.to_csv(R_AG / "match_vs_clip.csv", index=False)
print("AG — kiểm định cặp vs CLIP:"); print(match_vs_clip_ag[match_vs_clip_ag.group == "all"].round(4).to_string())
_pairs = [("none+match", "none"), ("none+match", "none+clipmatch"), ("none+match", "no_actions+match"),
          ("none+match", "shuffled_actions+match"), ("none+match", "no_relations+match"),
          ("none+match", "no_class_edges+match"), ("none+clipmatch", "none")]
match_sig_ag = pd.concat([paired_test(_P, a, b).assign(a=a, b=b) for a, b in _pairs
                          if any(k[0] == a for k in _P) and any(k[0] == b for k in _P)])
match_sig_ag.to_csv(R_AG / "match_paired.csv", index=False)
print("\nKiểm định cặp giữa các cấu hình:"); print(match_sig_ag[match_sig_ag.group == "all"].round(4).to_string())
if any(k[0] == "none+match" for k in REL_PQ.get("ag", {})):
    _rt, _rs = rel_probe_report("ag", base="none+match",
                                others=["CLIP", "none", "none+clipmatch", "no_actions+match", "shuffled_actions+match",
                                        "no_relations+match", "no_class_edges+match"],
                                out_csv=R_AG / "relational_hard_negatives_match.csv")
    print("\nBộ quan hệ khó (AG) với nhánh khớp:"); print(_rt.to_string()); print(_rs.round(4).to_string())
if any(k[0] == "none+match" for k in REL_CF_RES.get("ag", {})):
    _ct, _cs = cf_report("ag", base="none+match")
    _ct.to_csv(R_AG / "counterfactual_verb_swap_match.csv", index=False)
    _cs.to_csv(R_AG / "counterfactual_verb_swap_match_paired.csv", index=False)
    print("\nTruy vấn phản thực tế với nhánh khớp:"); print(_ct.round(4).to_string()); print(_cs.round(4).to_string())
pd.DataFrame(MATCH_LOG).to_csv(R_AG / "match_log.csv", index=False)


In [ ]:
%%script true
# v7: không chạy lại ở bản này — kết quả AG xem ở version 4 của notebook.
# Mô hình cuối của Action Genome chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_AG = list(FUSION_LOG)
sel_ag, final_ag = final_model_table(ag_all_res, FUSION_AG)
display(sel_ag.round(4)); display(final_ag)
display(pd.DataFrame(FUSION_AG).round(3))
sig_ad_ag = paired_test(PQ_AG, "none+adaptive", "none")
display(sig_ad_ag.round(4))
R_AG = AG_OUT / "results"
for name, obj in (("final_model_val_selection.csv", sel_ag), ("final_model_test.csv", final_ag),
                  ("fusion_log.csv", pd.DataFrame(FUSION_AG)), ("paired_adaptive_vs_global.csv", sig_ad_ag)):
    obj.to_csv(R_AG / name, index=False)

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# =====================================================================================
# LIÊN KẾT TRUY VẤN → NODE BẰNG NGỮ NGHĨA (CLIP-text), KHÔNG DÙNG TỪ ĐIỂN VIẾT TAY
# τ chọn trên VALIDATION. So sánh: dictionary (3 danh sách) | exact (khớp chuỗi) | semantic@τ
# =====================================================================================
import pandas as pd

class SemanticLinker:
    """CLIP-text đóng băng; cache embedding theo chuỗi để mọi lần dựng đồ thị dùng lại."""
    def __init__(self):
        import open_clip
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        self.m = m.to(DEVICE).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.tok = open_clip.get_tokenizer(CFG.clip_model)
        self.cache = {}
        self.dim = self.embed(["x"]).shape[1]

    @torch.no_grad()
    def embed(self, texts):
        new = [t for t in dict.fromkeys(texts) if t not in self.cache]
        for s in range(0, len(new), 1024):
            b = new[s:s + 1024]
            f = F.normalize(self.m.encode_text(self.tok(b).to(DEVICE)), dim=-1).float().cpu().numpy()
            self.cache.update(zip(b, f))
        if not texts:
            return np.zeros((0, self.dim), np.float32)
        return np.stack([self.cache[t] for t in texts])


if "LINKER" not in globals():
    LINKER = SemanticLinker()


def query_ngrams(text):
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    return [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]


def _node_index(graph):
    """Embedding tên node, gắn vào chính đồ thị (mỗi ablation có tập node khác nhau)."""
    if not hasattr(graph, "_sem"):
        names = sorted(graph.name2id)
        graph._sem = (names, np.array([graph.name2id[n] for n in names]), LINKER.embed(names))
    return graph._sem


def make_semantic_parse(tau):
    """Mỗi n-gram nối tới node gần nhất nếu cosine ≥ tau. Trả về (node_id, tên node, n-gram)."""
    def parse(text, graph):
        grams = query_ngrams(text)
        if not grams:
            return []
        names, ids, E = _node_index(graph)
        S = LINKER.embed(grams) @ E.T
        best = S.argmax(1)
        sc = S[np.arange(len(grams)), best]
        hits, seen = [], set()
        for g in np.argsort(-sc):
            if sc[g] < tau:
                break
            nid = int(ids[best[g]])
            if nid not in seen:
                seen.add(nid); hits.append((nid, names[best[g]], grams[g]))
        return hits
    parse.tau = tau
    return parse


# Mã hóa trước mọi n-gram của toàn bộ truy vấn (theo batch) để vòng thí nghiệm chạy nhanh
LINKER.embed(sorted({g for r in ag_records for g in query_ngrams(r["caption"])}))
_AG_COMPOSITES = {o for r in ag_records for o in r["objects"] if " " in o}   # node ghép 'động từ vật'


def val_score(parse_fn, tag, seed=0):
    """Huấn luyện 1 seed với cách liên kết cho trước, chỉ đo trên VAL (không chạm test)."""
    with use_dataset(AG_OUT / "semantic_tune" / tag, parse_query=parse_fn, make_split=ag_make_split):
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(ag_records, ag_split, ag_img, ag_txt, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), _ = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        cand, sc, sg, sp_ = score_queries(model, eng, parts["val"]["q_idx"], parts["val"]["img_idx"])
        m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), parts["val"]["gold_pos"])
        clip_m = retrieval_metrics(cand, sc, parts["val"]["gold_pos"])
        vq = parts["val"]["q_idx"]
        grp = Counter(parts["val"]["groups"])
    return dict(method=tag, val_MRR=m["MRR"], val_R1=m["R@1"], val_MRR_CLIP=clip_m["MRR"],
                alpha=a, lam=l,
                coverage=float(np.mean([len(eng.q_concepts[q]) > 0 for q in vq])),
                rel_coverage=float(np.mean([any(h[1] in _AG_COMPOSITES for h in eng.q_concepts[q])
                                            for q in vq])),
                concepts_per_q=float(np.mean([len(eng.q_concepts[q]) for q in vq])),
                n_direct=grp.get("direct", 0), n_indirect=grp.get("indirect", 0))

### 3.1 Chọn τ trên VALIDATION

### 3.2 Đánh giá (Evaluation)

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
TAU_GRID = [0.80, 0.84, 0.88, 0.92, 0.96]
rows = [val_score(ag_parse_query, "dictionary"),
        val_score(make_semantic_parse(0.999), "exact")]          # khớp chuỗi thuần, không danh sách
for tau in TAU_GRID:
    rows.append(val_score(make_semantic_parse(tau), f"semantic@{tau:.2f}"))
link_df = pd.DataFrame(rows)
(AG_OUT / "results").mkdir(parents=True, exist_ok=True)
link_df.to_csv(AG_OUT / "results" / "linking_tau_val.csv", index=False)
display(link_df.round(4))

sem_rows = link_df[link_df.method.str.startswith("semantic")]
BEST_TAU = float(sem_rows.loc[sem_rows.val_MRR.idxmax(), "method"].split("@")[1])
best_parse = make_semantic_parse(BEST_TAU)
print(f"τ tốt nhất trên val: {BEST_TAU}")

In [ ]:
%%script true
# v7: không chạy lại ở bản này — kết quả AG xem ở version 4 của notebook.
ART_AG = dict(ART_BY_ABL)
ag_gdf = explanation_grounding(ART_AG, ag_records, ag_split)
ag_gtab = grounding_table(ag_gdf)
ag_gdf.to_csv(AG_OUT / "results" / "explanation_grounding_per_query.csv", index=False)
ag_gtab.to_csv(AG_OUT / "results" / "explanation_grounding.csv", index=False)
display(ag_gtab)

In [ ]:
%%script true
# v7: không chạy lại ở bản này — kết quả AG xem ở version 4 của notebook.
# =====================================================================================
# ABLATION ĐÚNG NGHĨA CHO AG: bỏ NODE QUAN HỆ người–vật ('hold cup', 'sit sofa'...).
# Trên AG, 'no_relations' KHÔNG bỏ quan hệ: các bộ ba (người, holding, cup) đã được "biên dịch"
# thành node ghép và nối vào video bằng cạnh contains (P0). 'no_relations' chỉ bỏ cạnh
# vật–vật (afford/related/isa). Muốn hỏi "tri thức quan hệ có giúp không" thì phải bỏ chính
# các node ghép này: đồ thị chỉ còn video –chứa→ lớp đồ vật.
# Chạy NGAY sau cell 54 (cần ag_records, ag_split, ag_img, ag_txt, PQ_AG).
# =====================================================================================
import copy
ag_records_nc = copy.deepcopy(ag_records)
for r in ag_records_nc:
    r["objects"] = [o for o in r["objects"] if " " not in o]     # bỏ node ghép 'động từ vật'
    r["relations"] = []
log(f"no_composite: {np.mean([len(r['objects']) for r in ag_records_nc]):.1f} node/video "
    f"(đầy đủ: {np.mean([len(r['objects']) for r in ag_records]):.1f})")

with use_dataset(AG_OUT / "no_composite", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
    REL_GAL_KEY, REL_RENAME = "ag", {"none": "no_composite"}      # cùng gallery quan hệ khó, bỏ node quan hệ
    nc_res, _ = run_experiment(ag_records_nc, ag_split, ag_img, ag_txt, ablations=("none",))
    REL_GAL_KEY, REL_RENAME = None, {}
    nc_df = results_table(nc_res).assign(ablation="no_composite")
    PQ_NC = {("no_composite", s): v for (ab, s), v in PER_QUERY.items() if ab == "none"}
_N_ABL = len(CFG.ablations)

R_AG = AG_OUT / "results"
ag_full = pd.concat([ag_df, nc_df])
ag_full.to_csv(R_AG / "main_results_with_no_composite.csv", index=False)
display(ag_full[ag_full.model.str.startswith("CLIP+Graph")][["ablation", "model", "R@1", "R@5", "MRR"]])

PQ_ALL = {**PQ_AG, **PQ_NC}           # nhóm direct/indirect lấy theo đồ thị ĐẦY ĐỦ ('none')
sig_ag = pd.concat([paired_test(PQ_ALL, "none", b)
                    for b in ("no_composite", "no_relations", "random_edges", "no_scene",
                              "no_related", "no_concept") if any(k[0] == b for k in PQ_ALL)])
sig_ag.to_csv(R_AG / "paired_ablation_test.csv", index=False)
display(sig_ag.round(4))

In [ ]:
%%script true
# v7: không chạy lại ở bản này — kết quả AG xem ở version 4 của notebook.
# v5 — BỘ QUAN HỆ KHÓ trên AG: cùng vật, khác quan hệ người–vật (gallery cố định, α/λ từ VAL)
rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_composite", "no_relations", "random_edges"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
display(rel_tab_ag); display(rel_sig_ag.round(4))


In [ ]:
%%script true
# v7: không chạy lại ở bản này — kết quả AG xem ở version 4 của notebook.
# v6 — ĐỐI CHỨNG composite_swap: gán nhầm quan hệ, giữ nguyên vật (xem mô tả ở cell định nghĩa đầu Phần 3)
_ag_vo()
_rng_sw = np.random.RandomState(0)
ag_records_sw = copy.deepcopy(ag_records)
_n_sw = _n_drop = 0
for r in ag_records_sw:
    comps = [c for c in r["objects"] if " " in c]
    plain = [c for c in r["objects"] if " " not in c]
    new = set()
    for c in comps:
        v, o = c.split(" ", 1)
        alts = sorted(x for x in _ag_vo()[o] if x != v and f"{x} {o}" not in comps and f"{x} {o}" not in new)
        if alts:
            new.add(f"{alts[_rng_sw.randint(len(alts))]} {o}"); _n_sw += 1
        else:
            _n_drop += 1                         # không có vị từ thay thế ⇒ bỏ (không giữ quan hệ thật)
    r["objects"] = sorted(set(plain) | new)
    r["relations"] = sorted((c.split(" ", 1)[1], "afford", c) for c in new)
log(f"composite_swap: đổi {_n_sw} node quan hệ, bỏ {_n_drop} (vật không có vị từ thay thế) | "
    f"{np.mean([len(r['objects']) for r in ag_records_sw]):.1f} node/video (gốc "
    f"{np.mean([len(r['objects']) for r in ag_records]):.1f})")


# v6 — chạy đối chứng composite_swap (3 seed) + đánh giá phản thực tế cho mọi cấu hình đã có
with use_dataset(AG_OUT / "composite_swap", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
    REL_GAL_KEY, REL_RENAME = "ag", {"none": "composite_swap"}
    sw_res, _ = run_experiment(ag_records_sw, ag_split, ag_img, ag_txt, ablations=("none",))
    REL_GAL_KEY, REL_RENAME = None, {}
    sw_df = results_table(sw_res).assign(ablation="composite_swap")
    PQ_SW = {("composite_swap", s): v for (ab, s), v in PER_QUERY.items() if ab == "none"}
_N_ABL = len(CFG.ablations)

R_AG = AG_OUT / "results"
pd.concat([ag_full, sw_df]).to_csv(R_AG / "main_results_with_controls.csv", index=False)
sig_sw = paired_test({**PQ_AG, **PQ_SW}, "none", "composite_swap")
sig_sw.to_csv(R_AG / "paired_none_vs_composite_swap.csv", index=False)
display(sig_sw.round(4))

rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_composite", "composite_swap", "no_relations", "random_edges"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
print("Bộ quan hệ khó (AG):"); display(rel_tab_ag); display(rel_sig_ag.round(4))

cf_tab, cf_sig = cf_report("ag")
cf_tab.to_csv(R_AG / "counterfactual_verb_swap.csv", index=False)
cf_sig.to_csv(R_AG / "counterfactual_verb_swap_paired.csv", index=False)
print("Truy vấn phản thực tế (đổi vị từ): demoted = % truy vấn mà video đúng bị hạ hạng; "
      "delta_RR = RR(gốc) − RR(đổi vị từ), càng lớn càng nhạy với quan hệ")
display(cf_tab.round(4)); display(cf_sig.round(4))


### 3.3 Ví dụ liên kết (đọc tay để kiểm tra chất lượng)

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
_g = MultimodalGraph(ag_records, ag_split["train"])
names, ids, E = _node_index(_g)
for iid in ag_split["val"][:8]:
    text = ag_records[iid]["caption"]
    print(f"\n{text[:90]}")
    for nid, node, gram in best_parse(text, _g):
        cos = float(LINKER.embed([gram])[0] @ E[names.index(node)])
        print(f"   '{gram}' → {node}   (cos {cos:.3f})")

### 3.4 Chạy đầy đủ với τ tốt nhất: 3 ablation × 3 seed

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
with use_dataset(AG_OUT / "semantic", parse_query=best_parse, make_split=ag_make_split):
    ag_sem_all_res, ag_sem_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                                ablations=("none", "random_edges", "no_relations"))
    ag_sem_df = results_table(ag_sem_all_res)
    ag_sem_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
    ag_sem_fdf, ag_sem_summ = fidelity_study(ag_sem_art)
    with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
        json.dump(ag_sem_summ, f, indent=2)
    if "final_rank_fidelity" in globals():
        ag_sem_rdf, ag_sem_rsumm = final_rank_fidelity(ag_sem_art)
        ag_sem_rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
    demo_query("a person drinking from a glass while sitting on the couch", ag_sem_art, k=3)

### 3.5 So sánh dictionary vs semantic trên TEST (cùng split)

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
cmp_ = pd.concat([ag_df.assign(linking="dictionary"),
                  ag_sem_df.assign(linking=f"semantic@{BEST_TAU}")])
cmp_.to_csv(AG_OUT / "results" / "linking_comparison_test.csv", index=False)
display(cmp_[cmp_.ablation == "none"][["linking", "model", "R@1", "R@5", "MRR", "nDCG@10"]])

In [ ]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# Nhiệm vụ 2 — GNN của SCENIR trên dataset thứ hai (Action Genome)
with use_dataset(AG_OUT / "scenir", parse_query=ag_parse_query, make_split=ag_make_split), use_gnn("scenir"):
    ag_sc_res, ag_sc_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                          ablations=("none", "random_edges", "no_relations"))
    ag_sc_df = results_table(ag_sc_res)
    ag_sc_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)

ag_cmp_gnn = pd.concat([ag_df.assign(gnn=CFG.gnn_type), ag_sc_df.assign(gnn="scenir")])
ag_cmp_gnn.to_csv(AG_OUT / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(ag_cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

---
# PHẦN 4 — Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT)

> **Gộp từ `project03_level4b.ipynb`.** Phần này là một pipeline độc lập với Phần 1–3: nó **định nghĩa lại** `CFG`, `make_split`, `scatter_sum`, `znorm_per_row`… nên **chỉ chạy sau khi Phần 1–3 đã chạy xong** (hoặc khởi động lại runtime rồi chạy riêng từ cell `import` đầu tiên của Phần 4). Không chạy lại các cell của Phần 1–3 sau khi đã chạy Phần 4.


> **v5:** phần này không chạy lại trong bản này (mọi cell code có `%%script true`). Xoá dòng đó ở đầu cell để chạy.

### Cấp độ 4(b) — Truy vấn văn bản → video có xếp hạng lại và giải thích bằng đồ thị

**Phạm vi.** Notebook này chỉ làm **Level 4(b): mở rộng text-to-video** của hệ thống truy vấn có đồ thị. Không đề xuất thuật toán mới (Level 4(a)), không khảo sát người dùng (Level 4(c)), không làm image-to-text. Phần nền tảng text→image nằm ở `image/` (notebook của nhóm, Flickr8k + Action Genome).

**Pipeline** (mục 3.3 của master instruction):
1. truy vấn văn bản;
2. CLIP text encoder đóng băng;
3. lấy mẫu 12 khung hình / video;
4. CLIP image encoder đóng băng;
5. gộp khung hình thành embedding video (cách gộp chọn trên VAL);
6. top-K = 50 video ứng viên theo cosine;
7. xếp hạng lại bằng điểm đồ thị;
8. trả top-K cùng giải thích là các số hạng đồ thị đã thực sự cộng vào điểm.

**Mô hình đồ thị được tái sử dụng, không phải đóng góp mới:**
* **QC-RGAT** (v4) là mô hình chính vì điểm của nó tách chính xác thành các số hạng.
* **GraphSAGE có kiểu cạnh** (mô hình cuối của v3) là baseline GNN đã học.
* Cả hai được đánh giá, **không mặc định** cái nào hơn CLIP-only hay chỉ-contains.

**Dữ liệu và trạng thái từng benchmark:**

| Tập | Vai trò | Đã nhìn trước | Trạng thái kết quả |
|---|---|---|---|
| MSVD train (1200 video) | dựng đồ thị, huấn luyện GNN | — | — |
| MSVD val (100 video) | chọn cấu hình | — | phát triển |
| MSVD test (670 video) | đánh giá | 4 lần (v1–v4) | **EXPLORATORY** |
| MSR-VTT 1k-A | — | 1 lần (v3) | không dùng |
| MSR-VTT fresh1k (seed 20261025) | — | 1 lần (v4) | không dùng (đặt `MSVD_V4_CONFIRM_SET=fresh1k` thì chạy lại, ghi EXPLORATORY) |
| **MSR-VTT fresh1k-B** (seed 20261026) | **xác nhận**, đánh giá **một lần** sau `freeze()` | 0 | **CONFIRMATORY** |

fresh1k-B gồm 1000 video train-9k, rời 1k-A, rời 200 video dev và rời fresh1k. Nó cùng miền với fresh1k, mà số liệu fresh1k đã ảnh hưởng tới thiết kế notebook này; điều đó được khai báo trong báo cáo.

**Phần thêm so với v4** (các ô "Level 4(b)"):
* Ablation **huấn luyện lại** theo họ quan hệ:
  - G7: bỏ cạnh vai `agent_of`/`object_of`;
  - G8: bỏ cạnh `svo`/`prep`.
* Bộ truy vấn **trực tiếp / gián tiếp ≥ 20 / nhóm**: chọn theo luật đăng ký trước, dựng **trước** khi chấm mô hình, báo metric riêng.
* xAI theo **thứ hạng**: giải thích là các số hạng đồ thị ủng hộ (contains và đường đi quan hệ). Xoá chúng rồi đo Δ điểm, Δ hạng, top-1 và top-10, so với xoá ngẫu nhiên.
* Phân tích **5 thành công + 5 thất bại**: chọn theo luật, phân tầng nguyên nhân.
* **Demo**: nhập văn bản tự do, nhận top-K với điểm CLIP, điểm đồ thị, đường đi giải thích và thumbnail.
* `data_inventory.json`, `data_leakage_audit.md`, `audit_report.md`, `level4b_conclusions.md` và `artifacts_manifest.json`, **tự sinh** từ chính lần chạy.

**Chạy trên Kaggle:**
1. Settings: GPU T4, **Internet On**.
2. *(Khuyến nghị)* Add Input output của lần chạy v4 để dùng lại cache đặc trưng MSVD.
3. Save & Run All.
4. Tải `/kaggle/working/project03_level4b_review.zip`.

**Biến môi trường:**
* `MSVD_V4_STAGES` (mặc định `V1,V7`);
* `MSVD_L4_SEEDS` (mặc định `0,1,2,3,4`);
* `MSVD_L4_EPOCHS`;
* `MSVD_V4_CONFIRM_SET`: `fresh1kB` (mặc định), `fresh1k` hoặc `1kA`. Hai giá trị sau ⇒ EXPLORATORY.

**Nguyên tắc.**
* Test và tập ngoài chỉ được mở sau `freeze()`.
* Caption của test và tập ngoài không bao giờ dùng để dựng từ vựng hay quan hệ, huấn luyện hoặc chọn cấu hình.
* PASS của kiểm thử phần mềm **không** phải kết luận khoa học.
* Thí nghiệm không chạy được thì ghi `NOT RUN` kèm lý do.


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re, io, socket
import shutil, zipfile, tempfile, platform, copy, threading, contextlib
from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
NB_PID = os.getpid()
try:
    _EXEC_COUNT_START = get_ipython().execution_count      # = 1 khi chạy từ kernel mới (Restart & Run All)
except NameError:
    _EXEC_COUNT_START = None
STAGES = []


def mark_stage(name, requires=()):
    """Ghi thứ tự chạy các cell chính. Cell chạy lệch thứ tự (hoặc dựa vào biến sót từ phiên trước)
    dừng ngay với thông báo rõ ràng thay vì cho ra số liệu sai."""
    done = {s["name"] for s in STAGES}
    missing = [r for r in requires if r not in done]
    assert not missing, f"Cell '{name}' phải chạy sau {missing} — hãy Restart & Run All."
    assert os.getpid() == NB_PID, "PID kernel đã đổi giữa chừng — hãy Restart & Run All."
    STAGES.append(dict(name=name, t=time.time(), pid=os.getpid()))


def has_internet(host="huggingface.co", port=443, timeout=5.0):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


INTERNET = has_internet()
mark_stage("env")
print(f"Kaggle: {RUN_ON_KAGGLE} | Internet: {INTERNET} | Python {platform.python_version()} | "
      f"execution_count đầu = {_EXEC_COUNT_START}")
if not INTERNET:
    print("⚠ Không có Internet: chỉ chạy được nếu open_clip, trọng số CLIP, dữ liệu NLTK và MSVD đã có sẵn "
          "(Add Input). Bật: Kaggle → Settings → Internet → On.")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# Phụ thuộc tối thiểu. Kaggle đã có torch/torchvision/opencv/nltk/pandas/scipy; chỉ cài open_clip khi thiếu.
# Cài thất bại thì DỪNG kèm log (không bỏ qua lỗi cài đặt).
def ensure_package(module, pip_spec):
    import importlib
    try:
        return importlib.import_module(module)
    except ImportError:
        pass
    if not INTERNET:
        raise RuntimeError(f"Thiếu gói '{pip_spec}' và không có Internet — bật Internet hoặc Add Input wheel.")
    print(f"→ pip install {pip_spec}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"pip install {pip_spec} thất bại (mã {r.returncode}) — xem log phía trên.")
    importlib.invalidate_caches()
    return importlib.import_module(module)


cv2 = ensure_package("cv2", "opencv-python-headless")
open_clip = ensure_package("open_clip", "open_clip_torch==3.2.0")
nltk = ensure_package("nltk", "nltk")

import scipy
import scipy.sparse as sp
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV_INFO = dict(python=platform.python_version(), platform=platform.platform(), torch=torch.__version__,
                cuda=torch.version.cuda, device=DEVICE, open_clip=getattr(open_clip, "__version__", "?"),
                opencv=cv2.__version__, numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,
                nltk=nltk.__version__, kaggle=RUN_ON_KAGGLE, internet=INTERNET)
if DEVICE == "cuda":
    ENV_INFO["gpu"] = torch.cuda.get_device_name(0)
    print(f"GPU: {ENV_INFO['gpu']} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ KHÔNG có GPU: trích đặc trưng ViT-L-14 cho ~23,6k frame + 80,8k caption trên CPU rất chậm (hàng giờ).\n"
          "  Kaggle: Settings → Accelerator → GPU T4/P100. Chạy thử trên CPU: MSVD_L4_CLIP=ViT-B-32, "
          "MSVD_L4_CLIP_W=laion2b_s34b_b79k.")
print(json.dumps(ENV_INFO, indent=1))
mark_stage("deps", requires=("env",))

## 4.1 Cấu hình

Mọi siêu tham số nằm ở cell dưới. Phần dữ liệu, CLIP, lấy mẫu frame và tagger giữ nguyên như v1/v2, nên cache đặc trưng MSVD của v2 dùng lại được. Các khối mới của v3 gồm:
* vị từ: họ có vị từ tự do, prompt, ngưỡng support;
* người chấm: mẫu, ngưỡng precision, file nhãn;
* benchmark xác nhận: MSR-VTT ở commit ghim;
* thống kê: Holm.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
_env = os.environ.get


class CFG:
    # ---------------- chế độ chạy ----------------
    # "msvd": dữ liệu thật (mặc định). "synthetic": smoke-test mã bằng video tổng hợp — phải chọn TƯỜNG MINH.
    # Notebook KHÔNG bao giờ tự rơi về synthetic khi thiếu MSVD: chỉ có caption thì không phải text-to-video.
    dataset  = _env("MSVD_L4_DATASET", "msvd")
    # dev_mode: chạy đủ pipeline trên dữ liệu thật nhưng KHOÁ test; phân tích cuối chạy trên val, gắn nhãn DEV.
    dev_mode = _env("MSVD_L4_DEV", "0") == "1"

    # ---------------- dữ liệu MSVD ----------------
    hf_repo      = "friedrichor/MSVD"
    hf_revision  = "9aa95a3039c106ee9e0969d82d7ce5a9efab1196"     # ghim commit ⇒ tái lập được
    msvd_root    = _env("MSVD_ROOT", "")             # thư mục chứa msvd_*.json + YouTubeClips/ hoặc MSVD_Videos.zip
    video_source = _env("MSVD_VIDEO_SOURCE", "auto")  # auto | dir | zip
    expected     = dict(train=(1200, 48774), val=(100, 4290), test=(670, 27763))   # (video, caption) — dataset card
    max_bad_video_frac = 0.0                         # MSVD thật: không chấp nhận video nào giải mã lỗi

    # ---------------- lấy mẫu khung hình ----------------
    n_frames_max   = 12
    frame_policy   = "uniform-segment-midpoint"      # frame k = floor((k + 0.5) / F · n_frame_giải_mã_được)
    n_frames_grid  = [4, 8, 12]                       # chọn trên VAL; F=1 chỉ in ra để tham chiếu
    pool_grid      = ["mean", "centrality"]           # chọn trên VAL
    centrality_tau = 0.1
    decode_workers = 4
    thumb_width    = 160

    # ---------------- CLIP (đóng băng hoàn toàn) ----------------
    clip_model   = _env("MSVD_L4_CLIP", "ViT-L-14")
    clip_weights = _env("MSVD_L4_CLIP_W", "laion2b_s32b_b82k")
    clip_bs      = 128
    text_bs      = 512

    # ---------------- từ vựng (TRAIN) + tagger (chọn trên VAL) ----------------
    vocab_min_caps  = 2        # một từ được tính cho video nếu xuất hiện trong ≥ 2 caption của video đó ...
    min_term_df     = 3        # ... và ở ≥ 3 video TRAIN
    truth_min_caps  = 3        # nhãn tham chiếu (chỉ để chấm F1 của tagger trên VAL)
    tag_norms       = ["raw", "z", "raw+z"]
    tag_pools       = ["video", "frame_max"]
    tag_k_noun_grid = [3, 5, 8, 10, 12]
    tag_k_verb_grid = [1, 2, 3, 5]
    noun_prompts = ["a photo of a {}.", "a video frame showing a {}.", "a photo containing a {}."]
    verb_prompts = ["a photo of someone {}.", "a video frame of a person {}.", "someone is {}."]
    min_tag_freq = 2           # lớp phải được gắn cho ≥ 2 video TRAIN mới thành node
    max_concepts = 4000

    # ---------------- đồ thị (cùng ý nghĩa với notebook gốc) ----------------
    max_path_len = 3
    scene_min_count, scene_top_k = 2, 10
    cooc_min_count, cooc_top_k = 5, 10
    cooc_score, cooc_min_npmi = "npmi", 0.10
    use_wordnet, isa_min_depth = True, 4
    use_idf, idf_from = True, "taxonomy"
    drop_redundant_indirect = True
    edge_weighting = True

    # ---------------- GNN ----------------
    gnn_type, gnn_typed = "sage", True
    gate_init, path_init_indirect, scalar_lr_mult = -1.5, -1.0, 10.0
    hard_neg_k = 31
    train_with_clip = True
    learn_clip_w = False       # chọn lại trên VAL
    gnn_dim, gnn_layers, gnn_dropout = 256, 2, 0.1
    lr, weight_decay, temperature = 1e-3, 1e-4, 0.07
    epochs = int(_env("MSVD_L4_EPOCHS", "15"))
    caption_passes = 4         # mỗi epoch: 4 lượt × (1 caption ngẫu nhiên / video TRAIN), batch không trùng video
    train_bs = 128

    # ---------------- xếp hạng lại ----------------
    rerank_k_grid = [10, 20, 50]                       # chọn trên VAL (cấu hình 'none', seed đầu)
    rerank_k      = None
    alpha_grid    = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid   = [0.0, 0.25, 0.5, 0.75]             # bỏ λ=1: khi α<1, s_path luôn có trọng số > 0
    eval_chunk    = 2048

    # ---------------- thí nghiệm (v2: theo stage, mỗi stage một câu hỏi) ----------------
    seeds  = [int(s) for s in _env("MSVD_L4_SEEDS", "0,1,2,3,4").split(",")]
    stages = [s for s in _env("MSVD_V4_STAGES", "V1,V7").split(",") if s]
    # Luật CHẤP NHẬN một thay đổi (đăng ký trước, chỉ dùng VAL): Δ MRR val trung bình seed ≥ adopt_min_delta
    # VÀ bootstrap theo cụm video trên val cho P(Δ ≤ 0) < adopt_max_p. Không đạt ⇒ giữ thiết kế đang dùng.
    adopt_min_delta, adopt_max_p = 0.002, 0.10
    save_checkpoints = _env("MSVD_G2_CKPT", "1") == "1"

    # ---------------- đồ thị v2 ----------------
    reliable_min_support = 3          # quan hệ 'đáng tin' = được ≥ 3 video TRAIN hỗ trợ (không có nhãn người ⇒ chỉ là lọc theo tần suất)
    tax_hub_frac = 0.15               # taxonomy 'lọc hub': bỏ khái niệm mà > 15% video TRAIN với tới
    tax_min_depth_filtered = 6        # ... và khái niệm danh từ quá nông (độ sâu WordNet < 6)
    sense_max_candidates, sense_margin = 6, 1.0   # chọn sense kiểu Lesk trên ngữ cảnh caption TRAIN
    conf_modes = ["sigmoid", "minmax"]            # chuẩn hoá trọng số contains — chọn trên VAL theo MRR
    conf_floor = 0.05
    rewire_swaps_per_edge = 10        # số lần thử swap / cạnh trong constrained rewiring
    rewire_min_changed = 0.5          # đối chứng hợp lệ ⇔ ≥ 50% cạnh của mỗi loại (≥ 10 cạnh) bị đổi
    n_audit_captions = 200            # mẫu caption TRAIN cho audit quan hệ (để người chấm)

    # ---------------- v3: vị từ (predicate) là thành phần của cạnh ----------------
    # Họ có vị từ TỰ DO: svo (vị từ = cụm động từ), prep (vị từ = giới từ). Ở agent_of / object_of vị từ chính là node
    # động từ đích ⇒ không mang thông tin riêng của cạnh; vẫn ghi vào schema nhưng không đưa embedding vị từ vào message.
    pred_families = ("svo", "prep")
    pred_min_support = 5              # chính sách 'support': (họ, vị từ) xuất hiện ở ≥ 5 video TRAIN
    pred_prompts = {"svo": ["someone {} something.", "a video of a person {} something."],
                    "prep": ["something {} something else.", "a photo of an object {} another object."]}
    pred_adapt_hidden = 128           # phương án B: MLP dư DÙNG CHUNG cho mọi vị từ (không có ID riêng cho từng vị từ)
    # ---------------- chất lượng quan hệ: kiểm chứng TỰ ĐỘNG (mặc định) + nhãn người chấm (TUỲ CHỌN) ----------------
    dep_model = "en_core_web_sm"      # parser phụ thuộc cú pháp độc lập (spaCy) để đối chiếu với parser heuristic
    agree_min = 0.5                   # chính sách 'agreement': giữ (họ, nguồn, vị từ, đích) nếu ≥ 50% caption TRAIN của nó được spaCy xác nhận
    annot_n_per_cell, annot_n_double_per_cell, annot_seed = 30, 10, 20261024
    annot_strata = ((2, 2, "low"), (3, 5, "mid"), (6, 10 ** 9, "high"))   # support = số video TRAIN có quan hệ
    annot_file = _env("MSVD_G3_LABELS", "")                               # trống ⇒ tự tìm relation_annotation*.csv trong /kaggle/input
    prec_min_lower = 0.60             # chính sách 'precision': giữ ô (họ × mức phổ biến) có cận dưới Wilson 95% ≥ 0.60
    # ---------------- benchmark xác nhận ĐỘC LẬP (chưa dùng ở v1/v2) ----------------
    confirm = _env("MSVD_G3_CONFIRM", "msrvtt")                           # "msrvtt" | "none"
    msrvtt_repo = "friedrichor/MSR-VTT"
    msrvtt_revision = "c1af215a96934854f42683c19c51391aaee6f962"
    msrvtt_dev_n = 200                # dev_mode: 200 video TRAIN-9k (KHÔNG thuộc test 1k-A) chỉ để thử đường ống
    msrvtt_root = _env("MSRVTT_ROOT", "")  # thư mục có sẵn msrvtt_*.json / MSRVTT_Videos.zip (trống ⇒ tải từ HF)
    confirm_set = _env("MSVD_V4_CONFIRM_SET", "fresh1kB")  # "fresh1kB" (mặc định, CHƯA nhìn) | "fresh1k" (đã nhìn ở v4) | "1kA" (đã nhìn ở v3)
    fresh_n, fresh_seed = 1000, 20261025  # fresh1k: 1000 video train-9k, ngoài 1k-A và ngoài 200 video dev — chọn ngẫu nhiên cố định
    fresh_b_seed = 20261026           # fresh1k-B (Level 4(b)): 1000 video train-9k KHÁC — rời 1k-A, rời dev, rời fresh1k
    # ---------------- Level 4(b) ----------------
    n_eval_queries = 20               # bộ truy vấn đăng ký trước: số truy vấn / nhóm (trực tiếp, gián tiếp) / tập đánh giá
    eval_query_seed = 20261026
    n_cases = 5                       # 5 thành công + 5 thất bại
    demo_topk = 5
    demo_queries = ["a man is playing a guitar", "a child is playing football on the grass", "a woman is slicing a tomato",
                    "a dog is running on the beach", "two people are dancing on a stage"]   # cố định trước, không chỉnh theo kết quả
    # ---------------- v4: QC-RGAT ----------------
    v4_dim = 64                       # chiều embedding của QC-RGAT
    v4_lr = float(_env("MSVD_V4_LR", "2e-3"))
    v4_score_bs = 256                 # số truy vấn / lô khi chấm điểm
    v4_rerank_k = 50                  # k chọn trên VAL ở v3 (Kaggle), giữ cố định
    # ---------------- thống kê ----------------
    holm_alpha = 0.05                 # Holm–Bonferroni cho họ so sánh chính đăng ký trước

    # ---------------- xAI ----------------
    n_explain_queries, top_m_paths = 20, 5
    n_rank_fidelity, n_edge_deletion = 50, 15
    n_seed_sensitivity = 20

    # ---------------- đường dẫn ----------------
    _base      = Path("/kaggle/working") if RUN_ON_KAGGLE else Path(".")
    out_dir    = Path(_env("MSVD_L4_OUT", str(_base / ("project03_level4b" if dataset == "msvd" else "project03_level4b_synthetic"))))
    data_cache = Path(_env("MSVD_L4_DATA_CACHE", "/tmp/msvd_hf" if RUN_ON_KAGGLE else str(_base / "msvd_hf")))
    extra_cache_dirs = [d for d in _env("MSVD_L4_CACHE_DIRS", "").split(":") if d]


assert CFG.dataset in ("msvd", "synthetic"), f"CFG.dataset không hợp lệ: {CFG.dataset}"
RESULTS = CFG.out_dir / "results"
for _d in (CFG.out_dir / "cache", RESULTS, CFG.out_dir / "thumbs", CFG.out_dir / "checkpoints"):
    _d.mkdir(parents=True, exist_ok=True)
RES_TAG = "DEV_val" if CFG.dev_mode else ("test" if CFG.dataset == "msvd" else "SYNTHETIC_test")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (set, frozenset)): return sorted(o)
    return str(o)


def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_jsonable)
    return path


def sha1_of(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_jsonable).encode()).hexdigest()[:12]


def cfg_snapshot():
    return {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}


CHECKS = []


def check(cid, desc, passed, evidence="", hard=True, status=None):
    """Ghi một kiểm tra vào sổ audit. hard=True ⇒ FAIL dừng notebook ngay (không chạy tiếp trên dữ liệu sai)."""
    st = status or ("PASS" if passed else "FAIL")
    CHECKS.append(dict(id=cid, check=desc, status=st, evidence=str(evidence)[:900]))
    print(f"  [{st}] {cid} — {desc}: {evidence}")
    if hard and st == "FAIL":
        raise AssertionError(f"Kiểm tra {cid} THẤT BẠI: {desc} — {evidence}")
    return st == "PASS"


save_json(dict(cfg=cfg_snapshot(), env=ENV_INFO), RESULTS / "config_initial.json")
print(f"dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | CLIP={CFG.clip_model}/{CFG.clip_weights} | "
      f"seeds={CFG.seeds} | epochs={CFG.epochs} | out={CFG.out_dir.resolve()}")
if CFG.dataset == "synthetic":
    print("⚠⚠ CHẾ ĐỘ SYNTHETIC: chỉ smoke-test mã. Mọi con số dưới đây KHÔNG phải kết quả MSVD.")
if CFG.dev_mode:
    print("⚠ dev_mode: test bị KHOÁ; mọi phân tích cuối chạy trên VAL, gắn nhãn DEV — không dùng để báo cáo.")
mark_stage("config", requires=("deps",))

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Gói kết quả để TẢI VỀ xem xét: MỘT file zip nhẹ <out_dir>_review.zip, nằm cạnh thư mục output. ---
# Gồm: results/** (mọi CSV/JSON/MD, log truy cập, pip freeze), notebook đã chạy (nếu tìm được), pack_manifest.json.
# KHÔNG gồm dữ liệu và các file nặng suy ra từ dữ liệu: video MSVD (CFG.data_cache), cache/ (đặc trưng CLIP),
# thumbs/ (khung hình video), checkpoints/, synthetic_videos/. Manifest vẫn ghi số file và dung lượng của chúng.
# Cell cuối gọi pack_review(); nếu một cell bị LỖI, hook bên dưới gói ngay những gì đã có, kèm traceback.
import traceback

REVIEW_ZIP = CFG.out_dir.parent / f"{CFG.out_dir.name}_review.zip"
PACK_MAX_FILE_MB = 50


def _find_executed_notebook():
    """Kaggle chạy bằng papermill (input = output = __notebook__.ipynb) và lưu notebook sau mỗi cell, nên bản này
    có output của mọi cell đã chạy xong. Chỉ nhận file được ghi SAU khi kernel này khởi động (tránh bản cũ)."""
    t0 = STAGES[0]["t"] if STAGES else 0.0
    cands = [Path.cwd() / "__notebook__.ipynb", *sorted(Path("/kaggle").glob("*/__notebook__.ipynb")),
             *sorted(Path("/kaggle/src").glob("*.ipynb"))]
    for p in cands:
        try:
            if p.is_file() and p.stat().st_mtime >= t0 and json.loads(p.read_text()).get("cells"):
                return p
        except Exception:
            continue
    return None


def _dir_size(d):
    fs = [f for f in d.rglob("*") if f.is_file()] if d.is_dir() else [d]
    return dict(n_files=len(fs), mb=round(sum(f.stat().st_size for f in fs) / 2**20, 1))


def pack_review(reason, error=None):
    """Ghi lại REVIEW_ZIP từ đầu (gọi nhiều lần được). Không bao giờ ném lỗi ra ngoài: trả về None nếu thất bại."""
    try:
        root, inc, too_large = REVIEW_ZIP.stem, [], []
        nb_path = _find_executed_notebook()
        tmp = REVIEW_ZIP.with_name(REVIEW_ZIP.name + ".tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for p in sorted(p for p in RESULTS.rglob("*") if p.is_file()):
                rel, size = p.relative_to(CFG.out_dir).as_posix(), p.stat().st_size
                if size > PACK_MAX_FILE_MB * 2**20:
                    too_large.append(dict(path=rel, mb=round(size / 2**20, 1)))
                    continue
                z.write(p, f"{root}/{rel}")
                inc.append(dict(path=rel, bytes=size))
            if nb_path is not None:
                z.write(nb_path, f"{root}/executed_notebook.ipynb")
            excluded = {d.name + ("/" if d.is_dir() else ""): _dir_size(d) for d in sorted(CFG.out_dir.iterdir()) if d != RESULTS}
            excluded[f"video MSVD ({CFG.data_cache})"] = "không gói"
            manifest = dict(created=time.strftime("%Y-%m-%d %H:%M:%S"), reason=reason, status="error" if error else "ok",
                            error_traceback=error, dataset=CFG.dataset, dev_mode=CFG.dev_mode, res_tag=RES_TAG, kaggle=RUN_ON_KAGGLE,
                            stages_done=[s["name"] for s in STAGES], checks=dict(Counter(c["status"] for c in CHECKS)),
                            checks_fail=[c["id"] for c in CHECKS if c["status"] == "FAIL"],
                            executed_notebook=str(nb_path) if nb_path else None, included=inc, skipped_too_large=too_large,
                            excluded=excluded)
            z.writestr(f"{root}/pack_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2, default=_jsonable))
        tmp.replace(REVIEW_ZIP)
        return REVIEW_ZIP
    except Exception as e:
        print(f"⚠ Không đóng gói được {REVIEW_ZIP}: {type(e).__name__}: {e}")
        return None


def _pack_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is None:
        return
    tb = "".join(traceback.format_exception(type(err), err, err.__traceback__))[-8000:]
    z = pack_review(f"cell lỗi (execution_count={result.execution_count})", error=tb)
    if z is not None:
        print(f"⚠ Notebook dừng vì lỗi — đã gói những gì đã có (kèm traceback) → {z} ({z.stat().st_size / 2**20:.1f} MB)")


try:
    _IP = get_ipython()
    for _cb in [c for c in _IP.events.callbacks["post_run_cell"] if getattr(c, "__name__", "") == "_pack_on_error"]:
        _IP.events.unregister("post_run_cell", _cb)          # chạy lại cell này thì không đăng ký trùng
    _IP.events.register("post_run_cell", _pack_on_error)
except NameError:
    pass
print(f"Gói xem xét: {REVIEW_ZIP} (tạo ở cell cuối; tự tạo nếu một cell bị lỗi)")

## 4.2 Giao thức đăng ký trước (viết TRƯỚC khi chạm MSR-VTT fresh1k-B)

**Đã biết trước khi viết notebook này** (v4, Kaggle, fresh1k):
* v4 hơn CLIP-only: ΔMRR +0,0109, CI [+0,0072; +0,0145].
* v4 **không** hơn chỉ-contains, không vị từ, vị từ hoán vị, rewired, attention tĩnh hay v3.
* Bỏ mọi cạnh quan hệ lúc suy luận làm MRR **tăng** nhẹ.
* Xoá giải thích (chỉ đường đi quan hệ) làm điểm giảm nhưng **không** đổi thứ hạng nào.

Vì vậy notebook này **không kỳ vọng** quan hệ giúp. Mục tiêu là đo lại một cách trung thực trên tập chưa nhìn, kèm các đối chứng còn thiếu.

**Họ so sánh chính** (một phía, `a > b`, MRR trên mọi caption, bootstrap theo video, Holm–Bonferroni α = 0,05). Chỉ **fresh1k-B** mới là xác nhận; test MSVD là thăm dò.

| | So sánh | Câu hỏi |
|---|---|---|
| H1 | G3 vs G0 | đồ thị (QC-RGAT) có hơn CLIP-only? |
| H2 | G3 vs G1 | cạnh quan hệ có đóng góp ngoài contains? |
| H3 | G3 vs G2 | khớp vị từ có ích? |
| H4 | G3 vs G4 | ghép **đúng** vị từ vào **đúng** cạnh có ích? |
| H5 | G3 vs G5 | topology thật có hơn rewiring có ràng buộc? |
| H6 | G3 vs G6 | attention theo truy vấn có hơn attention tĩnh? |
| H7 | G3 vs R3 | QC-RGAT có hơn GraphSAGE có kiểu cạnh (v3)? |
| H8 | G3 vs G7 | họ cạnh **vai** (`agent_of`, `object_of`) có giá trị? (G7 huấn luyện lại không có họ này) |
| H9 | G3 vs G8 | họ cạnh **`svo`/`prep`** có giá trị? (G8 huấn luyện lại không có họ này) |

**Luật kết luận:**
* "Quan hệ có giá trị thật" chỉ được kết luận khi **H2, H4 và H5 cùng được ủng hộ** và đối chứng đạt audit.
* Một họ cạnh "có giá trị" khi H8 hoặc H9 tương ứng được ủng hộ.

**So sánh phụ** (không vào họ Holm, chỉ báo cáo kèm CI):
* G3 vs G0 theo nhóm truy vấn trực tiếp / gián tiếp;
* trên bộ truy vấn đăng ký trước (≥ 20 / nhóm).

**Chọn cấu hình** (chỉ VAL):
* Incumbent là T = 1 (đơn giản hơn).
* T = 2 chỉ được nhận khi Δ MRR val ≥ 0,002 **và** p Holm < 0,10.
* Cố định trước: d = 64, lr = 2e-3, k = 50.
* α (trọng số CLIP trong điểm cuối) chọn trên VAL trong lưới có α = 1 (CLIP-only). Nếu α = 1 thắng trên VAL thì đồ thị không được dùng ở điểm cuối, và kết quả được báo đúng như vậy.

**Ngân sách:** 5 seed × 15 epoch cho mọi thí nghiệm. Trên cùng một tập đánh giá, mọi thí nghiệm dùng chung tập ứng viên CLIP top-k.

**Bộ truy vấn đăng ký trước** (mục 7.3). Lấy từ caption của tập đánh giá, chỉ dựa vào hai thứ: nhãn CLIP zero-shot của video đúng và cạnh đồ thị dựng từ train. **Không** dựa vào điểm của mô hình.
* **Trực tiếp:** mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.
* **Gián tiếp:** truy vấn nêu một quan hệ (bộ ba chủ thể–vị từ–đối tượng); nhãn video chỉ phủ **một phần** khái niệm của truy vấn; có ít nhất một cạnh quan hệ nối khái niệm còn thiếu với một nhãn của video. Đó là các quan hệ dự kiến phải khai thác.
* Mỗi nhóm lấy 20 truy vấn (mỗi video tối đa 1) bằng seed cố định.


## 4.3 Hàng rào chống rò rỉ

* **`CaptionStore`**: caption không được lưu trong `RECORDS`. Mọi lần đọc caption, hoặc dùng đặc trưng của caption, đều phải nêu **mục đích**; một ma trận quyết định split nào được dùng cho mục đích nào.
  * `vocab`, `relations`, `annotation` và `gnn_train` chỉ dùng train; `tune` và `tagger_select` chỉ dùng val.
  * Caption test MSVD và tập ngoài (`ext`, MSR-VTT) chỉ dùng để `encode`, `query_input` và (sau đóng băng) `eval`, `report`, `display`.
  * Truy cập sai sẽ ném `PermissionError` ngay.
* **`TestGate`**: test MSVD và tập ngoài chỉ mở sau `freeze()`, và mọi lần chạm đều được ghi log kèm tên split. Ở `dev_mode`, test MSVD bị khoá; tập ngoài khi đó là video **train-9k** của MSR-VTT, rời hẳn 1k-A.
* **`provenance()`**: mỗi artifact được ghi lại nguồn, split caption và split video đã dùng; khai báo sai loại thì `assert` thất bại.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
class TestGate:
    """Test chỉ mở SAU khi cấu hình đã đóng băng trên validation. Mọi lần chạm test được ghi log."""

    def __init__(self):
        self.frozen, self.frozen_sha, self.frozen_at, self.log = None, None, None, []

    def freeze(self, config):
        assert self.frozen is None, "Cấu hình đã đóng băng — không được chọn lại sau khi mở test."
        payload = json.dumps(config, sort_keys=True, default=_jsonable)
        self.frozen, self.frozen_at = json.loads(payload), time.time()
        self.frozen_sha = hashlib.sha1(payload.encode()).hexdigest()[:12]
        save_json(dict(config=self.frozen, sha1=self.frozen_sha, frozen_at=self.frozen_at,
                       frozen_at_str=time.strftime("%Y-%m-%d %H:%M:%S")), RESULTS / "frozen_config.json")
        print(f"🔒 Đã đóng băng cấu hình (sha1={self.frozen_sha}) → {RESULTS / 'frozen_config.json'}")

    def check(self, what, split="test"):
        """split 'test' = test MSVD (đã nhìn ở v1/v2 ⇒ ở v3 chỉ là THĂM DÒ); 'ext' = benchmark xác nhận độc lập (MSR-VTT).
        Cả hai chỉ mở sau khi đóng băng. dev_mode: test MSVD bị khoá; 'ext' khi đó là video TRAIN-9k (không phải tập xác nhận)."""
        if self.frozen is None:
            raise RuntimeError(f"Truy cập {split.upper()} ('{what}') trước khi đóng băng cấu hình — vi phạm giao thức.")
        if CFG.dev_mode and split == "test":
            raise RuntimeError(f"dev_mode: test bị khoá ('{what}')")
        self.log.append(dict(what=what, split=split, t=time.time(), frozen_sha=self.frozen_sha, dev=bool(CFG.dev_mode)))


TEST_GATE = TestGate()


class CaptionStore:
    """Nơi DUY NHẤT giữ caption. Mỗi lần đọc phải nêu mục đích; split không được phép ⇒ PermissionError."""
    ALLOWED = {
        "vocab":         {"train"},                  # từ vựng khái niệm
        "relations":     {"train"},                  # cạnh quan hệ thống kê từ caption
        "gnn_train":     {"train"},                  # truy vấn huấn luyện GNN
        "tagger_select": {"val"},                    # chọn (pool, norm, k) của tagger theo F1
        "tune":          {"val"},                    # chọn n_frames/pooling/learn_clip_w/k/α/λ
        "encode":        {"train", "val", "test", "ext"},   # CLIP-text đóng băng (không học) — chỉ để cache đặc trưng
        "query_input":   {"train", "val", "test", "ext"},   # đọc văn bản của CHÍNH truy vấn đang được trả lời
        "eval":          {"val", "test", "ext"},
        "display":       {"train", "val", "test", "ext"},   # chỉ hiển thị SAU khi đã xếp hạng
        "report":        {"val", "test", "ext"},            # báo cáo P/R/F1 của tagger sau khi đóng băng
    }
    GATED = {"eval", "report", "display"}            # với split test / ext: chỉ sau khi đóng băng

    def __init__(self, caps_by_vid, split_by_vid):
        self._caps = {int(v): list(c) for v, c in caps_by_vid.items()}
        self.split_of = {int(v): s for v, s in split_by_vid.items()}
        vid_of_cap, self._text = [], []
        for v in sorted(self._caps):
            for c in self._caps[v]:
                vid_of_cap.append(v); self._text.append(c)
        self.vid_of_cap = np.array(vid_of_cap, np.int64)
        self.split_of_cap = np.array([self.split_of[v] for v in vid_of_cap])
        self.log = Counter()

    def _authorize(self, splits, purpose):
        assert purpose in self.ALLOWED, f"mục đích lạ: {purpose}"
        for s in splits:
            if s not in self.ALLOWED[purpose]:
                raise PermissionError(f"RÒ RỈ bị chặn: caption split '{s}' không được dùng cho mục đích '{purpose}'")
            if s in ("test", "ext") and purpose in self.GATED:
                TEST_GATE.check(f"captions:{purpose}", s)
        for s, n in splits.items():
            self.log[(purpose, s)] += int(n)

    def by_video(self, vids, purpose):
        vids = [int(v) for v in np.atleast_1d(vids)]
        self._authorize(Counter(self.split_of[v] for v in vids), purpose)
        return {v: list(self._caps[v]) for v in vids}

    def texts(self, cap_idx, purpose):
        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))
        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)
        return [self._text[i] for i in cap_idx]

    def use(self, cap_idx, purpose):
        """Khai báo dùng ĐẶC TRƯNG của caption (không lấy văn bản) — cùng luật cho phép."""
        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))
        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)

    def replaced(self, new_caps, new_split=None):
        """Bản sao với caption (và tuỳ chọn split) của một số video bị thay — CHỈ dùng trong self-test."""
        caps = {v: list(c) for v, c in self._caps.items()}
        caps.update({int(v): list(c) for v, c in new_caps.items()})
        return CaptionStore(caps, {**self.split_of, **(new_split or {})})

    def access_table(self):
        return pd.DataFrame([dict(purpose=p, split=s, n=n) for (p, s), n in sorted(self.log.items())])


PROVENANCE = []
_PROV_RULES = {            # loại artifact → split CAPTION được phép dùng
    "learned":  {"train"},             # thống kê / trọng số học được
    "selected": {"val"},               # siêu tham số chọn trên validation
    "frames":   set(),                 # chỉ từ frame (không caption)
    "external": set(),                 # tri thức ngoài / mô hình tiền huấn luyện
    "encoded":  {"train", "val", "test", "ext"},   # mã hoá bằng encoder đóng băng, không học gì
    "evaluated": {"val", "test", "ext"},
}


def provenance(artifact, kind, source, caption_splits=(), video_splits=(), note=""):
    cs = set(caption_splits)
    bad = cs - _PROV_RULES[kind]
    assert not bad, f"RÒ RỈ: '{artifact}' (loại {kind}) dùng caption của {sorted(bad)}"
    PROVENANCE.append(dict(artifact=artifact, kind=kind, source=source,
                           caption_splits=",".join(sorted(cs)) or "—",
                           video_splits=",".join(sorted(set(video_splits))) or "—",
                           note=note, stage=STAGES[-1]["name"], time=time.strftime("%H:%M:%S")))

## 4.4 Dữ liệu: MSVD (phát triển) + MSR-VTT 1k-A (xác nhận) — Cổng A

**MSVD.** Như v1/v2. Video thật nằm trong `MSVD_Videos.zip`. Notebook dò `MSVD_ROOT`, rồi `/kaggle/input`, rồi cache, rồi tải từ HF ở commit ghim. Không có video thì **dừng**.

**MSR-VTT** (`friedrichor/MSR-VTT`, commit ghim). Dùng `msrvtt_test_1k.json` (1000 video, caption chuẩn JSFusion), `raw_data/MSRVTT_data.json` (20 caption/video) và `MSRVTT_Videos.zip` (2,2 GB, mp4).
* Video tập ngoài được nối **sau** MSVD trong `RECORDS` với split `ext`. Vì vậy cache MSVD giữ nguyên tên và thứ tự.
* Truy vấn chính là mọi caption của 1000 video; truy vấn phụ là caption chuẩn 1k-A. Nếu caption chuẩn không nằm trong 20 caption thì nó được thêm vào và đếm riêng.
* Không nạp được MSR-VTT thì tập xác nhận ghi **NOT RUN** kèm lý do; notebook không thay bằng tập khác.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
HF_JSON = {"train": "msvd_train.json", "val": "msvd_val.json", "test": "msvd_test.json"}
HF_ZIP = "MSVD_Videos.zip"


class DirVideoSource:
    kind = "dir"

    def __init__(self, root):
        self.root = Path(root)
        self.files = {p.stem: p for p in self.root.rglob("*") if p.suffix.lower() == ".avi"}
        self.label = str(self.root)

    def has(self, vid): return vid in self.files
    def size(self, vid): return self.files[vid].stat().st_size

    @contextmanager
    def open(self, vid):
        yield self.files[vid]


class ZipVideoSource:
    """Đọc video thẳng từ zip cục bộ (không giải nén toàn bộ): mỗi video được ghi ra file tạm, giải mã, rồi xoá."""
    kind = "zip"

    def __init__(self, zip_file, label, exts=(".avi",)):
        self.z = zipfile.ZipFile(zip_file)
        self.members = {Path(i.filename).stem: i for i in self.z.infolist()
                        if i.filename.lower().endswith(exts) and not Path(i.filename).name.startswith("._")}
        self.lock = threading.Lock()
        self.tmp = Path(tempfile.mkdtemp(prefix="msvd_zip_"))
        self.label = label

    def has(self, vid): return vid in self.members
    def size(self, vid): return self.members[vid].file_size

    @contextmanager
    def open(self, vid):
        with self.lock:                                   # ZipFile không an toàn khi đọc song song
            data = self.z.read(self.members[vid])
        p = self.tmp / f"{vid}{Path(self.members[vid].filename).suffix.lower()}"
        p.write_bytes(data)
        try:
            yield p
        finally:
            p.unlink(missing_ok=True)


def _first(root, name):
    hits = sorted(root.rglob(name), key=lambda p: len(str(p)))
    return hits[0] if hits else None


def locate_msvd():
    roots = [Path(CFG.msvd_root)] if CFG.msvd_root else []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.append(CFG.data_cache)
    found = dict(json={}, video_dir=None, zip=None, roots=[str(r) for r in roots])
    for root in roots:
        if not root.exists():
            continue
        for s, name in HF_JSON.items():
            if s not in found["json"]:
                p = _first(root, name)
                if p is not None:
                    found["json"][s] = p
        if found["video_dir"] is None:
            for d in root.rglob("YouTubeClips"):
                if d.is_dir() and any(d.glob("*.avi")):
                    found["video_dir"] = d
                    break
        if found["zip"] is None:
            found["zip"] = _first(root, HF_ZIP)
    return found


def hf_download(filename):
    from huggingface_hub import hf_hub_download
    CFG.data_cache.mkdir(parents=True, exist_ok=True)
    print(f"  tải {CFG.hf_repo}@{CFG.hf_revision[:8]}/{filename} → {CFG.data_cache}")
    return Path(hf_hub_download(CFG.hf_repo, filename, repo_type="dataset", revision=CFG.hf_revision,
                                local_dir=str(CFG.data_cache)))


def open_video_source(found):
    mode = CFG.video_source
    assert mode in ("auto", "dir", "zip"), mode
    if mode in ("auto", "dir") and found["video_dir"] is not None:
        return DirVideoSource(found["video_dir"])
    if mode in ("auto", "zip") and found["zip"] is not None:
        return ZipVideoSource(found["zip"], str(found["zip"]))
    if mode in ("auto", "zip") and INTERNET:
        return ZipVideoSource(hf_download(HF_ZIP), "hf_hub_download")
    raise FileNotFoundError(
        "KHÔNG tìm thấy file video MSVD (YouTubeClips/*.avi hoặc MSVD_Videos.zip) và không tải được. DỪNG: chỉ có "
        "caption thì KHÔNG được chạy/gọi là thí nghiệm text-to-video. Khắc phục: bật Internet, hoặc Add Input "
        f"dataset chứa video, hoặc đặt MSVD_ROOT. Đã dò: {found['roots']}")


def load_msvd():
    found = locate_msvd()
    for s, name in HF_JSON.items():
        if s not in found["json"]:
            if not INTERNET:
                raise FileNotFoundError(f"Thiếu {name} trong {found['roots']} và không có Internet để tải.")
            found["json"][s] = hf_download(name)
    source = open_video_source(found)
    records, caps = [], {}
    report = dict(source=source.kind, source_label=source.label, n_video_files=len(
        source.files if source.kind == "dir" else source.members),
        json={k: str(v) for k, v in found["json"].items()}, splits={})
    for s in ("train", "val", "test"):
        rows = json.load(open(found["json"][s]))
        fields = sorted({k for r in rows for k in r})
        bad = [r.get("video_id") for r in rows if not r.get("video_id") or not isinstance(r.get("caption"), list)]
        assert not bad, f"Schema {s} sai (thiếu video_id hoặc caption không phải list): {bad[:5]}"
        n_cap, missing, ncaps = 0, [], []
        for r in rows:
            vid = len(records)
            c = [str(x).strip() for x in r["caption"] if str(x).strip()]
            records.append(dict(vid=vid, video_id=r["video_id"], file=r.get("video") or f"{r['video_id']}.avi",
                                split=s, n_captions=len(c)))
            caps[vid] = c; n_cap += len(c); ncaps.append(len(c))
            if not source.has(r["video_id"]):
                missing.append(r["video_id"])
        report["splits"][s] = dict(videos=len(rows), captions=n_cap, fields=fields, missing_files=len(missing),
                                   missing_examples=missing[:5],
                                   captions_per_video=[int(min(ncaps)), round(float(np.mean(ncaps)), 1), int(max(ncaps))],
                                   video_field_is_filename=all(str(r.get("video", "")).endswith(".avi") for r in rows))
    ids = [r["video_id"] for r in records]
    assert len(set(ids)) == len(ids), "video_id trùng lặp giữa các bản ghi"
    return records, caps, source, report


def probe_video(path):
    """Codec / fps / kích thước / số frame giải mã thật sự (đếm bằng grab, không tin FRAME_COUNT)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return dict(ok=False, error="cv2 không mở được")
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    info = dict(codec="".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)), fps=round(cap.get(cv2.CAP_PROP_FPS), 3),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                frames_meta=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), backend=cap.getBackendName())
    n = 0
    while cap.grab():
        n += 1
    cap.release()
    info.update(ok=n > 0, frames_decoded=n, duration_s=round(n / info["fps"], 2) if info["fps"] > 0 else None)
    return info

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Level 4(b): benchmark xác nhận ĐỘC LẬP = MSR-VTT fresh1k-B (CHƯA dùng ở v1–v4); fresh1k (v4) và 1k-A (v3) đã nhìn ⇒ chỉ THĂM DÒ.
# (Ghi chú lịch sử của v4:) Benchmark xác nhận cho v4: MSR-VTT fresh1k (CHƯA dùng ở v1–v3). Video nối vào đồ thị theo giao thức inductive. ---
# 1k-A (JSFusion) đã được nhìn MỘT lần ở v3 ⇒ v4 dùng 1000 video KHÁC: lấy ngẫu nhiên (seed cố định) trong train-9k, loại 1k-A và
# loại 200 video dev (đã chạy ở máy local). Truy vấn CHÍNH = mọi caption (MSRVTT_data.json); truy vấn PHỤ = caption đầu tiên.
# CFG.confirm_set = "1kA" chạy lại 1k-A (lần nhìn thứ 2 — phải khai báo). Chỉ đánh giá MỘT lần, sau khi đóng băng.
# dev_mode: msrvtt_dev_n video TRAIN-9k đầu theo id — chỉ để thử đường ống, không phải kết quả xác nhận.
MSRVTT_FILES = dict(test="msrvtt_test_1k.json", train="msrvtt_train_9k.json", all_caps="raw_data/MSRVTT_data.json",
                    zip="MSRVTT_Videos.zip")


def msrvtt_file(key):
    name = MSRVTT_FILES[key]
    roots = ([Path(CFG.msrvtt_root)] if CFG.msrvtt_root else []) + ([Path("/kaggle/input")] if RUN_ON_KAGGLE else []) \
        + [CFG.data_cache / "msrvtt"]
    for root in roots:
        if root.exists():
            p = _first(root, Path(name).name)
            if p is not None:
                return p
    if not INTERNET:
        raise FileNotFoundError(f"Thiếu {name} (đã dò {roots}) và không có Internet.")
    from huggingface_hub import hf_hub_download
    d = CFG.data_cache / "msrvtt"
    d.mkdir(parents=True, exist_ok=True)
    print(f"  tải {CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}/{name} → {d}")
    return Path(hf_hub_download(CFG.msrvtt_repo, name, repo_type="dataset", revision=CFG.msrvtt_revision, local_dir=str(d)))


def load_msrvtt(vid_offset):
    test = json.load(open(msrvtt_file("test")))
    test_ids = sorted({r["video_id"] for r in test}, key=lambda x: int(x[5:]))
    tr_all = sorted([r for r in json.load(open(msrvtt_file("train"))) if r["video_id"] not in set(test_ids)],
                    key=lambda r: int(r["video_id"][5:]))
    dev_ids = {r["video_id"] for r in tr_all[:CFG.msrvtt_dev_n]}
    fresh1k_ids = set()
    set_name = "dev" if CFG.dev_mode else CFG.confirm_set
    if CFG.dev_mode:
        rows = tr_all[:CFG.msrvtt_dev_n]
        caps_of = {r["video_id"]: [str(c).strip() for c in r["caption"] if str(c).strip()] for r in rows}
        std = {v: c[0] for v, c in caps_of.items()}
        label = f"MSR-VTT train-9k, {len(rows)} video đầu theo id (DEV — không phải tập xác nhận)"
    else:
        assert CFG.confirm_set in ("fresh1kB", "fresh1k", "1kA"), CFG.confirm_set
        pool = [r for r in tr_all if r["video_id"] not in dev_ids]
        pick = np.random.RandomState(CFG.fresh_seed).choice(len(pool), CFG.fresh_n, replace=False)
        fresh1k_ids = {pool[i]["video_id"] for i in pick}          # tập xác nhận của v4 — ĐÃ nhìn một lần
        if CFG.confirm_set == "fresh1kB":
            pool_b = [r for r in pool if r["video_id"] not in fresh1k_ids]
            pick_b = np.random.RandomState(CFG.fresh_b_seed).choice(len(pool_b), CFG.fresh_n, replace=False)
            rows = sorted([pool_b[i] for i in pick_b], key=lambda r: int(r["video_id"][5:]))
            label = (f"MSR-VTT fresh1k-B: {len(rows)} video train-9k (ngoài 1k-A, ngoài {CFG.msrvtt_dev_n} video dev, ngoài fresh1k của v4; "
                     f"seed {CFG.fresh_b_seed}) — CHƯA dùng ở v1–v4")
        elif CFG.confirm_set == "fresh1k":
            rows = sorted([pool[i] for i in pick], key=lambda r: int(r["video_id"][5:]))
            label = (f"MSR-VTT fresh1k: {len(rows)} video train-9k (seed {CFG.fresh_seed}) — ĐÃ nhìn ở v4 ⇒ chỉ THĂM DÒ")
        else:
            rows = sorted(test, key=lambda r: int(r["video_id"][5:]))
            label = "MSR-VTT test 1k-A (JSFusion) — LẦN NHÌN THỨ 2 (đã dùng ở v3)"
        data = json.load(open(msrvtt_file("all_caps")))
        allc = defaultdict(list)
        for x in sorted(data["sentences"], key=lambda x: int(x["sen_id"])):
            allc[x["video_id"]].append(str(x["caption"]).strip())
        caps_of = {r["video_id"]: [c for c in allc[r["video_id"]] if c] for r in rows}
        std = ({r["video_id"]: str(r["caption"]).strip() for r in rows} if CFG.confirm_set == "1kA"
               else {v: c[0] for v, c in caps_of.items()})
    n_std_added = 0
    for v, c in caps_of.items():                       # caption chuẩn phải nằm trong tập truy vấn chính
        if std[v] not in c:
            c.append(std[v]); n_std_added += 1
    source = ZipVideoSource(msrvtt_file("zip"), f"{CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}", exts=(".mp4",))
    records, caps, missing = [], {}, []
    for j, r in enumerate(rows):
        vid, c = vid_offset + j, caps_of[r["video_id"]]
        records.append(dict(vid=vid, video_id=r["video_id"], file=r.get("video") or f"{r['video_id']}.mp4", split="ext",
                            n_captions=len(c), std_caption=c.index(std[r["video_id"]]), dataset="msrvtt"))
        caps[vid] = c
        if not source.has(r["video_id"]):
            missing.append(r["video_id"])
    ids = {r["video_id"] for r in records}
    report = dict(label=label, set=set_name, dev=bool(CFG.dev_mode), videos=len(records), captions=sum(len(c) for c in caps.values()),
                  captions_per_video=[min(map(len, caps.values())), round(float(np.mean(list(map(len, caps.values())))), 1),
                                      max(map(len, caps.values()))],
                  std_captions_added=n_std_added, missing_files=len(missing), missing_examples=missing[:5],
                  n_test_1kA_ids=len(test_ids), overlap_with_test_1kA=len(ids & set(test_ids)),
                  overlap_with_dev=len(ids & dev_ids) if not CFG.dev_mode else None,
                  overlap_with_fresh1k=len(ids & fresh1k_ids) if not CFG.dev_mode else None, zip_members=len(source.members))
    if set_name in ("fresh1k", "fresh1kB"):
        assert report["overlap_with_test_1kA"] == 0 and report["overlap_with_dev"] == 0, "fresh1k / fresh1k-B phải rời 1k-A và tập dev"
    if set_name == "fresh1kB":
        assert report["overlap_with_fresh1k"] == 0 and report["videos"] == CFG.fresh_n, "fresh1k-B phải rời fresh1k của v4"
    return records, caps, source, report


def make_synthetic_ext(vid_offset, n_videos=30, seed=7):
    """Tập 'ext' tổng hợp cho smoke-test: cùng thế giới đối tượng, câu chữ khác (kiểm tra đường ống, không phải kết quả)."""
    recs, caps, source, _ = make_synthetic_msvd(n_videos=n_videos, seed=seed, prefix="ext", all_split="ext")
    out_r, out_c = [], {}
    for j, r in enumerate(recs):
        out_r.append(dict(r, vid=vid_offset + j, split="ext", std_caption=0, dataset="synthetic_ext"))
        out_c[vid_offset + j] = caps[r["vid"]]
    report = dict(label="synthetic ext", dev=True, videos=len(out_r), captions=sum(len(c) for c in out_c.values()),
                  missing_files=0, overlap_with_test_1kA=0, std_captions_added=0)
    return out_r, out_c, source, report

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# Video TỔNG HỢP — chỉ dùng khi CFG.dataset == "synthetic" (smoke-test mã, không phải kết quả).
# Mỗi video có 3 cảnh (chủ thể → hành động → đồ vật), chữ in trên frame để CLIP đọc được; được ghi
# thành AVI (MJPG) thật để kiểm tra luôn đường ống đọc video bằng OpenCV.
SYN_SUBJ = {"man": "person", "woman": "person", "child": "person", "dog": "animal", "cat": "animal"}
SYN_ACT = {"play": (["guitar", "piano", "ball"], "playing"), "ride": (["horse", "bicycle"], "riding"),
           "eat": (["apple", "pizza"], "eating"), "cut": (["onion", "bread"], "cutting")}
SYN_COLORS = [(40, 40, 160), (40, 140, 40), (160, 60, 40), (120, 40, 120), (30, 120, 150), (150, 150, 30)]


def make_synthetic_msvd(n_videos=150, seed=0, prefix="syn", all_split=None):
    rng = random.Random(seed)
    vdir = CFG.out_dir / "synthetic_videos"
    vdir.mkdir(parents=True, exist_ok=True)
    records, caps = [], {}
    n_tr, n_va = int(n_videos * 0.6), int(n_videos * 0.15)
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    for i in range(n_videos):
        s = rng.choice(sorted(SYN_SUBJ)); a = rng.choice(sorted(SYN_ACT))
        objs, ing = SYN_ACT[a]; o = rng.choice(objs)
        split_name = all_split or ("train" if i < n_tr else ("val" if i < n_tr + n_va else "test"))
        name = f"{prefix}{i:04d}_{s}_{a}_{o}"
        path = vdir / f"{name}.avi"
        if not path.exists():
            w = cv2.VideoWriter(str(path), fourcc, 12.0, (224, 224))
            assert w.isOpened(), "cv2.VideoWriter không mở được (MJPG/AVI)"
            for k in range(36):
                word = (s, ing, o)[k // 12]
                fr = np.full((224, 224, 3), SYN_COLORS[(i + k // 12) % len(SYN_COLORS)], np.uint8)
                cv2.putText(fr, word, (8 + (k % 12) * 2, 122), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3)
                w.write(fr)
            w.release()
        c = [f"a {s} is {ing} a {o}", f"a {s} {a}s the {o}", f"someone is {ing} a {o}", f"a {s} with a {o}",
             f"the {s} is {ing}", f"a {s} is {ing} a {o} slowly", f"a {SYN_SUBJ[s]} is {ing}", f"a {o} and a {s}"]
        if all_split == "ext":                         # tập ngoài: câu chữ khác với tập huấn luyện
            c = [f"there is a {s} {ing} the {o}", f"{s} {ing} {o}", f"video of a {o} being used by a {s}",
                 f"the {SYN_SUBJ[s]} keeps {ing} a {o}"]
        rng.shuffle(c)
        records.append(dict(vid=i, video_id=name, file=path.name, split=split_name, n_captions=len(c)))
        caps[i] = c
    source = DirVideoSource(vdir)
    report = dict(source="synthetic-dir", source_label=str(vdir), n_video_files=len(source.files), json={}, splits={
        s: dict(videos=sum(r["split"] == s for r in records),
                captions=sum(len(caps[r["vid"]]) for r in records if r["split"] == s),
                fields=["caption", "video", "video_id"], missing_files=sum(
                    not source.has(r["video_id"]) for r in records if r["split"] == s))
        for s in ("train", "val", "test")})
    return records, caps, source, report

## 4.5 Split chính thức và manifest

Split MSVD giữ đúng `msvd_{train,val,test}.json` (1200 / 100 / 670 video), chia **theo video**. sha1 chỉ tính trên ba split MSVD, nên **trùng với v2**; kiểm tra C7 xác nhận điều này. Tập ngoài `ext` có sha1 riêng và không được trùng video nào với MSVD.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
SPLITS_MSVD = ("train", "val", "test")


def make_split(records, write=True):
    """Split MSVD giữ NGUYÊN như v1/v2 (sha1 chỉ tính trên train/val/test MSVD ⇒ so sánh được với v2);
    'ext' = benchmark xác nhận độc lập (MSR-VTT), có sha1 riêng."""
    keys = SPLITS_MSVD + ("ext",)
    split = {k: [r["vid"] for r in records if r["split"] == k] for k in keys}
    ids = {k: [records[v]["video_id"] for v in split[k]] for k in keys}
    for a, b in itertools.combinations(keys, 2):
        assert not set(split[a]) & set(split[b]) and not set(ids[a]) & set(ids[b]), f"{a}/{b} chồng lấn video!"
    payload = {k: sorted(ids[k]) for k in SPLITS_MSVD}
    split["sha1"] = sha1_of(payload)
    split["ext_sha1"] = sha1_of(sorted(ids["ext"])) if ids["ext"] else None
    if write:
        manifest = dict(
            dataset=CFG.dataset, hf_repo=CFG.hf_repo, hf_revision=CFG.hf_revision, sha1=split["sha1"],
            policy=("split chính thức của MSVD theo msvd_{train,val,test}.json; chia THEO VIDEO (không theo caption); "
                    "1 video = 1 ứng viên trong gallery của split đó; mỗi caption = 1 truy vấn có đúng 1 video dương"),
            counts={k: dict(videos=len(split[k]), captions=int(sum(records[v]["n_captions"] for v in split[k]))) for k in keys},
            video_ids=payload, ext=dict(sha1=split["ext_sha1"], source=EXT_REPORT.get("label") if EXT_REPORT else None,
                                        video_ids=sorted(ids["ext"])))
        save_json(manifest, RESULTS / "split_manifest.json")
        print(f"Split (sha1={split['sha1']}): " + " | ".join(
            f"{k}={manifest['counts'][k]['videos']} video / {manifest['counts'][k]['captions']} caption" for k in keys))
    return split

## 4.6 Lấy mẫu khung hình và đặc trưng CLIP đóng băng (Cổng B)

* **Chính sách lấy mẫu `uniform-segment-midpoint`:** video được chia thành `F_max = 12` đoạn bằng nhau, mỗi đoạn lấy frame ở giữa: `idx_k = floor((k + 0.5)/12 · n)`, với `n` là số frame **giải mã được thực sự**. Số này được đếm bằng `grab()`, vì `CAP_PROP_FRAME_COUNT` của AVI có thể sai. Ở lượt đọc thứ hai, notebook chỉ `retrieve()` đúng 12 frame cần lấy, nên bộ nhớ chỉ chứa 12 frame/video.
* Mọi frame được mã hoá **một lần** bằng CLIP-image đóng băng, chuẩn hoá L2, và lưu cache `[N_video, 12, D]` (float16) cùng chỉ số frame, timestamp, fps và thời lượng. Cache được dò ở `msvd_l4/cache` và ở `/kaggle/input/**/cache`.
* Mọi caption được mã hoá **một lần** bằng CLIP-text của cùng mô hình. Bước này không học gì, chỉ để cache đặc trưng; việc dùng đặc trưng caption về sau vẫn phải theo luật của `CaptionStore`.
* Thumbnail của frame giữa video (rộng 160 px) được lưu để demo hiển thị kết quả.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
_CLIP = {}


def get_clip():
    """CLIP/OpenCLIP tiền huấn luyện, đóng băng tuyệt đối (eval + requires_grad=False). Nạp một lần."""
    if "m" not in _CLIP:
        m, _, pre = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        m = m.to(DEVICE).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        _CLIP.update(m=m, pre=pre, tok=open_clip.get_tokenizer(CFG.clip_model))
    return _CLIP["m"], _CLIP["pre"], _CLIP["tok"]


def _amp():
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE == "cuda" else contextlib.nullcontext()


@torch.no_grad()
def clip_encode_images(x):
    m, _, _ = get_clip()
    with _amp():
        f = m.encode_image(x.to(DEVICE))
    return F.normalize(f.float(), dim=-1).cpu().numpy()


@torch.no_grad()
def clip_encode_texts(texts, bs=None):
    m, _, tok = get_clip()
    bs = bs or CFG.text_bs
    out = []
    for s in range(0, len(texts), bs):
        with _amp():
            f = m.encode_text(tok(texts[s:s + bs]).to(DEVICE))
        out.append(F.normalize(f.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def segment_midpoints(n, n_frames):
    """Chỉ số frame tại điểm giữa của n_frames đoạn đều nhau trên n frame giải mã được."""
    return np.minimum(np.floor((np.arange(n_frames) + 0.5) / n_frames * n).astype(int), n - 1)


def read_sampled_frames(path, n_frames):
    """Lượt 1: đếm frame GIẢI MÃ ĐƯỢC. Lượt 2: chỉ retrieve các frame cần lấy (RGB). Trả về (frames, meta)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError("cv2 không mở được video")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    n_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    n = 0
    while cap.grab():
        n += 1
    cap.release()
    if n == 0:
        raise IOError("0 frame giải mã được")
    idx = segment_midpoints(n, n_frames)
    want, got = set(idx.tolist()), {}
    cap = cv2.VideoCapture(str(path))
    i = 0
    while i <= idx[-1] and cap.grab():
        if i in want:
            ok, fr = cap.retrieve()
            if ok:
                got[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()
    lost = sorted(j for j in want if j not in got)
    if lost:
        raise IOError(f"không đọc được frame {lost[:5]}")
    fps_eff = fps if fps > 0 else 30.0
    meta = dict(n_decoded=n, n_meta=n_meta, fps=round(fps, 3), fps_assumed=fps <= 0,
                duration_s=round(n / fps_eff, 3), frame_idx=idx.tolist(),
                t_sec=[round(float(j) / fps_eff, 3) for j in idx])
    return [got[int(j)] for j in idx], meta


def _cache_name(prefix, n, data_tag=None):
    data_tag = data_tag or f"{CFG.dataset}"
    rev = CFG.hf_revision[:8] if data_tag == CFG.dataset else ""
    tag = f"{data_tag}_{CFG.clip_model}_{CFG.clip_weights}_F{CFG.n_frames_max}_{CFG.frame_policy}_{rev}_{n}".replace("__", "_")
    return f"{prefix}_" + re.sub(r"[^A-Za-z0-9_-]", "-", tag)


def find_cache(name):
    """Cache của v2, cache của v1 (MSVD_L4_CACHE_DIRS, ví dụ output v1 đã Add Input) và /kaggle/input/**/cache."""
    dirs = [CFG.out_dir / "cache"] + [Path(d) for d in CFG.extra_cache_dirs]
    if RUN_ON_KAGGLE:
        dirs += [p for p in Path("/kaggle/input").glob("*/**/cache") if p.is_dir()]
    for d in dirs:
        if (d / name).exists():
            return d / name
    return None


def extract_frame_features(records, source, allow_encode=True, data_tag=None):
    """[N_video, F_max, D] float16 (mỗi frame đã chuẩn hoá L2) + meta lấy mẫu. Dùng cache nếu có.
    Hàng thứ j ứng với records[j] (không giả định vid = j: tập ngoài nối sau MSVD)."""
    name = _cache_name("frames", len(records), data_tag) + ".npz"
    vids = [r["video_id"] for r in records]
    p = find_cache(name)
    if p is not None:
        z = np.load(p, allow_pickle=False)
        assert z["video_ids"].tolist() == vids, "cache đặc trưng frame không khớp thứ tự video"
        meta = json.load(open(p.with_suffix(".json")))
        print(f"Dùng lại cache đặc trưng frame: {p} {z['feat'].shape}")
        return z["feat"].astype(np.float32), meta, p
    if not allow_encode:
        raise FileNotFoundError(name)
    _, pre, _ = get_clip()
    Fm, N = CFG.n_frames_max, len(records)
    feats, metas, bad = None, [None] * N, {}
    thumbs = CFG.out_dir / "thumbs"

    def work(j):
        r = records[j]
        try:
            with source.open(r["video_id"]) as path:
                frames, meta = read_sampled_frames(path, Fm)
            mid = frames[Fm // 2]
            h, w = mid.shape[:2]
            th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(thumbs / f"{r['video_id']}.jpg"), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))
            return j, torch.stack([pre(Image.fromarray(f)) for f in frames]), meta
        except Exception as e:                          # ghi lại, quyết định dừng hay không ở dưới
            return j, None, f"{type(e).__name__}: {e}"

    from concurrent.futures import ThreadPoolExecutor
    t0, chunk = time.time(), max(8, 4 * CFG.clip_bs // Fm)
    with ThreadPoolExecutor(CFG.decode_workers) as ex:
        for s in tqdm(range(0, N, chunk), desc=f"giải mã + CLIP ({N} video)", unit="lô"):
            out = list(ex.map(work, range(s, min(s + chunk, N))))     # theo lô ⇒ bộ nhớ có chặn trên
            ok = [(v, x, m) for v, x, m in out if x is not None]
            bad.update({v: m for v, x, m in out if x is None})
            if not ok:
                continue
            X = torch.cat([x for _, x, _ in ok])
            E = np.concatenate([clip_encode_images(X[i:i + CFG.clip_bs]) for i in range(0, len(X), CFG.clip_bs)])
            if feats is None:
                feats = np.zeros((N, Fm, E.shape[1]), np.float16)
            for j, (v, _, m) in enumerate(ok):
                feats[v] = E[j * Fm:(j + 1) * Fm]
                metas[v] = m
    if bad:
        print(f"⚠ {len(bad)} video giải mã lỗi:", list(bad.items())[:5])
    assert feats is not None, "Không giải mã được video nào"
    assert len(bad) / N <= CFG.max_bad_video_frac, (
        f"{len(bad)}/{N} video lỗi > ngưỡng {CFG.max_bad_video_frac:.0%} — dừng, không chạy thí nghiệm trên dữ liệu hỏng")
    meta = dict(policy=CFG.frame_policy, n_frames_max=Fm, clip=f"{CFG.clip_model}/{CFG.clip_weights}",
                source=source.kind, seconds=round(time.time() - t0, 1),
                per_video={records[v]["video_id"]: m for v, m in enumerate(metas)},
                bad={records[v]["video_id"]: e for v, e in bad.items()})
    p = CFG.out_dir / "cache" / name
    np.savez(p, feat=feats, video_ids=np.array(vids))
    save_json(meta, p.with_suffix(".json"))
    print(f"Đặc trưng frame {feats.shape} sau {meta['seconds']}s → {p}")
    return feats.astype(np.float32), meta, p


def extract_text_features(store, cap_idx=None, allow_encode=True, data_tag=None):
    """[N_caption, D]: CLIP-text đóng băng của các caption cap_idx (mặc định: mọi caption; mục đích 'encode' — không học gì)."""
    cap_idx = np.arange(len(store.vid_of_cap)) if cap_idx is None else np.asarray(cap_idx)
    texts = store.texts(cap_idx, "encode")
    digest = hashlib.sha1("\n".join(texts).encode()).hexdigest()[:10]
    name = _cache_name("text", len(texts), data_tag) + f"_{digest}.npz"
    p = find_cache(name)
    if p is not None:
        feat = np.load(p)["feat"].astype(np.float32)
        assert feat.shape[0] == len(texts)
        print(f"Dùng lại cache đặc trưng caption: {p} {feat.shape}")
        return feat, p
    if not allow_encode:
        raise FileNotFoundError(name)
    t0 = time.time()
    feat = clip_encode_texts(texts).astype(np.float16)
    p = CFG.out_dir / "cache" / name
    np.savez(p, feat=feat)
    print(f"Đặc trưng caption {feat.shape} sau {time.time() - t0:.0f}s → {p}")
    return feat.astype(np.float32), p


def ext_data_tag():
    if not EXT_RECORDS:
        return None
    if CFG.dataset != "msvd":
        return "synthetic_ext"
    return f"msrvtt_{CFG.msrvtt_revision[:8]}_{'dev' + str(len(EXT_RECORDS)) if EXT_REPORT['dev'] else EXT_REPORT.get('set', '1kA')}"


def all_features(allow_encode=True):
    """MSVD (cache v1/v2 dùng lại được: cùng tên, cùng thứ tự) + tập ngoài (cache riêng), nối theo thứ tự RECORDS / caption."""
    fm, mm, pm = extract_frame_features(RECORDS[:N_MSVD], SOURCE, allow_encode)
    in_msvd = CAP_VID < N_MSVD
    assert in_msvd[:in_msvd.sum()].all(), "caption MSVD phải đứng trước caption tập ngoài"
    tm, ptm = extract_text_features(CAPS, np.where(in_msvd)[0], allow_encode)
    caches = [pm, ptm]
    if EXT_RECORDS:
        fe, me, pe = extract_frame_features(RECORDS[N_MSVD:], EXT_SOURCE, allow_encode, data_tag=ext_data_tag())
        te, pte = extract_text_features(CAPS, np.where(~in_msvd)[0], allow_encode, data_tag=ext_data_tag())
        fm, tm = np.concatenate([fm, fe]), np.concatenate([tm, te])
        mm = dict(mm, per_video={**mm["per_video"], **me["per_video"]}, bad={**mm["bad"], **me["bad"]},
                  seconds=round(mm["seconds"] + me["seconds"], 1) if "seconds" in mm and "seconds" in me else None,
                  source=f"{mm['source']}+{me['source']}")
        caches += [pe, pte]
    return fm, mm, tm, caches

### 4.6b Gộp theo thời gian (chọn trên VAL)

Notebook dùng một embedding duy nhất cho mỗi video, gộp từ `F ∈ {4, 8, 12}` frame con, cách đều nhau trong 12 frame đã lấy mẫu. Có hai cách gộp:

* `mean`: chuẩn hoá L2 từng frame, lấy trung bình, rồi chuẩn hoá lại.
* `centrality`: trọng số `softmax(cos(frame, trung bình)/τ)`. Cách này giảm trọng số các frame lạc chủ đề, chẳng hạn frame chuyển cảnh hoặc frame tối; trọng số không phụ thuộc truy vấn, nên mỗi video vẫn chỉ có một vector.

Cặp `(F, cách gộp)` được chọn theo **MRR text→video của CLIP thuần trên VAL**, xếp hạng trong toàn bộ gallery val. Dòng `F=1` (frame giữa) chỉ được in ra để tham chiếu và không được chọn, để bảo đảm embedding thật sự gộp từ nhiều frame.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def frame_subset(n_frames):
    """F frame con cách đều trong F_max frame đã lấy mẫu (F=4 ⇒ đúng điểm giữa của 4 đoạn: 1, 4, 7, 10)."""
    n_frames = min(int(n_frames), CFG.n_frames_max)
    pos = np.round((np.arange(n_frames) + 0.5) * CFG.n_frames_max / n_frames - 0.5)
    return np.unique(np.clip(pos, 0, CFG.n_frames_max - 1).astype(int))


def pool_frames(frame_feat, n_frames, mode):
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    if mode == "mean" or X.shape[1] == 1:
        v = X.mean(1)
    elif mode == "centrality":
        c = X.mean(1, keepdims=True)
        c /= np.linalg.norm(c, axis=-1, keepdims=True).clip(1e-8)
        a = (X * c).sum(-1) / CFG.centrality_tau
        w = np.exp(a - a.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
        v = (w[..., None] * X).sum(1)
    else:
        raise ValueError(mode)
    return (v / np.linalg.norm(v, axis=-1, keepdims=True).clip(1e-8)).astype(np.float32)


def full_gallery_ranks(txt, vid, q_idx, gallery, gold):
    """Thứ hạng (1 = tốt nhất) của video đúng trong TOÀN BỘ gallery theo cosine CLIP."""
    G = torch.from_numpy(np.ascontiguousarray(vid[np.asarray(gallery)]))
    pos = {int(v): i for i, v in enumerate(gallery)}
    gi = torch.tensor([pos[int(v)] for v in gold], dtype=torch.long)
    out = np.empty(len(q_idx), np.int64)
    for s in range(0, len(q_idx), CFG.eval_chunk):
        S = torch.from_numpy(np.ascontiguousarray(txt[np.asarray(q_idx[s:s + CFG.eval_chunk])])) @ G.T
        g = S[torch.arange(S.shape[0]), gi[s:s + CFG.eval_chunk]]
        out[s:s + CFG.eval_chunk] = (S > g[:, None]).sum(1).numpy() + 1
    return out


def select_pooling_on_val(frame_feat, txt, split):
    va = np.where(CAP_SPLIT == "val")[0]
    CAPS.use(va, "tune")
    gal, gold = np.array(split["val"]), CAP_VID[va]
    rows = []
    for nf in sorted(set(CFG.n_frames_grid) | {1}):
        for mode in CFG.pool_grid:
            if nf == 1 and mode != "mean":
                continue
            m = metrics_from_ranks(full_gallery_ranks(txt, pool_frames(frame_feat, nf, mode), va, gal, gold), gold)
            rows.append(dict(n_frames=nf, pooling=mode, frames=frame_subset(nf).tolist(),
                             selectable=nf in CFG.n_frames_grid, **{k: m[k] for k in ("R@1", "R@5", "R@10", "MRR")}))
    df = pd.DataFrame(rows)
    best = df[df.selectable].sort_values(["MRR", "n_frames"], ascending=[False, True]).iloc[0]
    return dict(n_frames=int(best.n_frames), pooling=str(best.pooling), val_MRR=float(best.MRR)), df


def gate_b_checks(frame_feat, frame_meta, vid_feat, txt, split):
    N, Fm, D = frame_feat.shape
    ff = frame_feat.reshape(-1, D)
    fn = np.linalg.norm(ff, axis=1)
    m, _, _ = get_clip()
    check("B0", "CLIP đóng băng (eval, không tham số nào requires_grad)",
          not m.training and not any(p.requires_grad for p in m.parameters()), f"{CFG.clip_model}/{CFG.clip_weights}")
    check("B1", "đặc trưng frame có shape [N_video, F_max, D]", (N, Fm) == (len(RECORDS), CFG.n_frames_max),
          f"{frame_feat.shape}")
    check("B2", "đặc trưng frame hữu hạn, chuẩn L2 ≈ 1", bool(np.isfinite(ff).all() and np.abs(fn - 1).max() < 2e-2),
          f"max|‖f‖−1| = {np.abs(fn - 1).max():.2e}")
    vn = np.linalg.norm(vid_feat, axis=1)
    check("B3", "đúng 1 embedding / video, hữu hạn, chuẩn 1, video_id duy nhất",
          vid_feat.shape == (N, D) and bool(np.isfinite(vid_feat).all()) and np.abs(vn - 1).max() < 1e-4
          and len({r["video_id"] for r in RECORDS}) == N, f"{vid_feat.shape}")
    check("B4", "đúng 1 embedding / caption, hữu hạn, cùng chiều với video",
          txt.shape == (len(CAP_VID), D) and bool(np.isfinite(txt).all()), f"{txt.shape}")
    qs = np.random.RandomState(0).choice(len(txt), min(3000, len(txt)), replace=False)
    S = txt[qs] @ vid_feat.T
    check("B5", "cosine văn bản–video nằm trong [−1, 1]", S.min() >= -1 - 1e-3 and S.max() <= 1 + 1e-3,
          f"[{S.min():.3f}, {S.max():.3f}]")
    idx = frame_subset(POOL_SEL["n_frames"])
    Xn = frame_feat[:, idx] / np.linalg.norm(frame_feat[:, idx], axis=-1, keepdims=True).clip(1e-8)
    cos_fv = (Xn * vid_feat[:, None]).sum(-1)
    frac_single = float((cos_fv.max(1) > 0.9999).mean())
    pair = np.einsum("nfd,ngd->nfg", Xn, Xn)
    off = pair[:, ~np.eye(len(idx), dtype=bool)].mean() if len(idx) > 1 else 1.0
    check("B6", "embedding video thật sự gộp từ NHIỀU frame (không vô tình dùng 1 frame)",
          len(idx) > 1 and frac_single < 0.05,
          f"F={len(idx)} frame {idx.tolist()} ({POOL_SEL['pooling']}); tỉ lệ video ≡ 1 frame = {frac_single:.3f}; "
          f"cos TB giữa các frame = {off:.3f}")
    bad_ts = []
    for name, mt in frame_meta["per_video"].items():
        if mt is None:
            bad_ts.append(name); continue
        t = np.array(mt["t_sec"])
        ok = (len(t) == Fm and (t >= 0).all() and (np.diff(t) >= 0).all() and t.max() < mt["duration_s"] + 1e-6
              and max(mt["frame_idx"]) < mt["n_decoded"])
        if not ok:
            bad_ts.append(name)
    check("B7", "lấy mẫu ổn định: đủ F_max frame / video, timestamp tăng dần trong [0, thời lượng)", not bad_ts,
          f"{len(frame_meta['per_video'])} video; lỗi {len(bad_ts)} {bad_ts[:3]}")
    cnt = np.bincount(CAP_VID, minlength=N)
    vsplit = np.array([r["split"] for r in RECORDS])
    probe = np.random.RandomState(1).choice(len(CAP_VID), min(500, len(CAP_VID)), replace=False)
    ok_text = all(CAPS._text[q] in CAPS._caps[int(CAP_VID[q])] for q in probe)        # kiểm tra chỉ số nội bộ
    galleries_ok = all(len(set(split[k])) == len(split[k]) and set(CAP_VID[CAP_SPLIT == k]) == set(split[k])
                       for k in ("train", "val", "test", "ext"))
    check("B8", "mọi caption-truy vấn ánh xạ đúng 1 video dương cùng split; gallery mỗi split không nhân bản video",
          all(cnt[r["vid"]] == r["n_captions"] for r in RECORDS) and bool((vsplit[CAP_VID] == CAP_SPLIT).all())
          and ok_text and galleries_ok,
          f"{len(CAP_VID)} truy vấn → {N} video; caption/video TB {cnt.mean():.1f}; gallery = "
          + ", ".join(f"{k}:{len(split[k])}" for k in ("train", "val", "test", "ext")))
    f2, _, t2, paths = all_features(allow_encode=False)
    check("B9", "cache đặc trưng nạp lại được mà KHÔNG chạy encoder, giống hệt bản trong bộ nhớ",
          np.array_equal(f2, frame_feat) and np.array_equal(t2, txt), " | ".join(p.name for p in paths))

## 4.7 Từ vựng (chỉ TRAIN), quan hệ, tagger và mẫu cho người chấm

Parser quan hệ của v2 được **giữ nguyên**, để đo đúng chất lượng của nó. Mỗi quan hệ là `(nguồn, họ, đích, vị từ)`:
* `agent_of`: chủ thể → động từ;
* `object_of`: đồ vật → động từ. Ở hai họ này vị từ chính là động từ đích;
* `svo`: danh từ –cụm động từ→ danh từ, ví dụ `man –ride→ horse`;
* `prep`: danh từ –giới từ→ danh từ, ví dụ `cat –on→ table`.

**Kiểm chứng tự động (Nhiệm vụ B), không cần người chấm.**
* spaCy `en_core_web_sm` là parser phụ thuộc cú pháp, độc lập với heuristic POS. Nó trích lại quan hệ cùng schema trên **chính caption train**:
  * chủ ngữ (`nsubj`, kể cả kế thừa qua `conj`, `xcomp`, `acl`) → `agent_of`;
  * tân ngữ (`dobj`, `nsubjpass`) → `object_of`;
  * chủ–động–tân → `svo`;
  * giới từ gắn vào **danh từ** → `prep`.
* Một quan hệ heuristic được xác nhận khi spaCy trích cùng (nguồn, họ, đích), cùng động từ đầu (svo) hoặc cùng giới từ (prep). Tỉ lệ xác nhận được tính cho từng quan hệ, từng họ và từng ô (họ × mức phổ biến), kèm khoảng Wilson.
* Độ phủ ngược (spaCy tìm được mà heuristic cũng tìm được) chỉ là so sánh tương đối, **không** phải recall thật.
* Đây là nhãn **silver**: hai parser có thể cùng sai, nên báo cáo không gọi nó là precision.

**Mẫu phân tầng** (họ × mức phổ biến; 360 dòng) vẫn được xuất kèm cột `auto_parser_agrees`. Nếu có file nhãn người chấm (`relation_annotation*`, **tuỳ chọn**), notebook tính thêm precision, Wilson và Cohen's kappa.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- NLTK: WordNet + POS tagger. Không tải được thì cảnh báo rõ và dùng phương án dự phòng (ghi vào ENV_INFO). ---
NLTK_STATUS = {}
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"):
    try:
        NLTK_STATUS[_pkg] = bool(nltk.download(_pkg, quiet=True, raise_on_error=False))
    except Exception as e:
        NLTK_STATUS[_pkg] = f"lỗi {type(e).__name__}"
from nltk.stem import WordNetLemmatizer

_LEM, _LEM_CACHE = WordNetLemmatizer(), {}


def _verb_lemma(w):
    """WordNetLemmatizer trả về ứng viên NGẮN NHẤT ('riding' → 'rid', 'shining' → 'shin'); ở đây chọn ứng viên
    có tần suất WordNet cao nhất ('riding' → 'ride')."""
    wn = _wordnet()
    cands = wn._morphy(w, "v") if wn is not None and hasattr(wn, "_morphy") else []
    if len(cands) > 1:
        cnt = lambda c: sum(l.count() for s in wn.synsets(c, pos="v") for l in s.lemmas() if l.name().lower() == c)
        return max(sorted(cands), key=lambda c: (cnt(c), len(c)))      # sorted: không phụ thuộc PYTHONHASHSEED
    return sorted(cands)[0] if cands else _LEM.lemmatize(w, "v")


def lemma(w, pos="n"):
    k = (w, pos)
    r = _LEM_CACHE.get(k)
    if r is None:
        try:
            if pos == "v":
                r = _verb_lemma(w)
            else:                                   # như NLTK (ứng viên ngắn nhất) nhưng hoà thì theo thứ tự chữ
                wn_ = _wordnet()
                cands = wn_._morphy(w, pos) if wn_ is not None and hasattr(wn_, "_morphy") else []
                r = min(sorted(cands), key=len) if cands else _LEM.lemmatize(w, pos)
        except LookupError:
            r = w[:-1] if (pos == "n" and w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[k] = r
    return r


try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()                    # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
    _TAGGER.tag(["a", "man", "is", "playing"])
except Exception as e:
    _TAGGER = None
    print("⚠ Không nạp được POS tagger NLTK → phân loại từ theo WordNet (kém chính xác hơn):", type(e).__name__)

# --- Lớp khái niệm dự phòng (notebook gốc) — dùng khi WordNet không có ---
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
    "guitar": ["musical instrument"], "piano": ["musical instrument"], "onion": ["vegetable"],
    "apple": ["fruit"], "bread": ["food"], "bicycle": ["vehicle"],
}
FALLBACK_VERB_ISA = {"slice": ["cut"], "chop": ["cut"], "dice": ["cut"], "strum": ["play"], "gallop": ["ride"],
                     "jog": ["run"], "sprint": ["run"], "stir": ["mix"], "whisk": ["mix"], "fry": ["cook"],
                     "bake": ["cook"], "boil": ["cook"], "grill": ["cook"], "sip": ["drink"], "munch": ["eat"]}

# --- Lọc WordNet (notebook gốc): chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC ---
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment", "vegetable", "fruit"}
GENERIC_VERBS = {"act", "move", "change", "make", "be", "have", "do", "get", "go", "cause", "interact",
                 "transfer", "use", "take", "give", "put", "perform", "modify", "remove", "create", "travel",
                 "come", "displace", "connect", "communicate", "express", "refer", "think", "consume"}

_WN = None

_syn_key = lambda x: x.name()
_path_key = lambda path: [x.name() for x in path]      # NLTK lưu con trỏ WordNet trong set ⇒ thứ tự phụ thuộc PYTHONHASHSEED


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE, _VERB_CACHE = {}, {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def is_action_verb(v):
    if v not in _VERB_CACHE:
        wn = _wordnet()
        _VERB_CACHE[v] = True if wn is None else bool(wn.synsets(v, pos="v"))
    return _VERB_CACHE[v]


def get_wordnet_hypernyms(words):
    """IsA danh từ (notebook gốc): cha + ông (đủ sâu, không quá chung) + tổ tiên thuộc BASIC_CATEGORIES."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        return {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
            for depth in range(len(path) - 2, -1, -1):
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    return out


def get_verb_hypernyms(verbs):
    """IsA động từ: cha/ông của nghĩa động từ phổ biến nhất (slice → cut, gallop → ride), bỏ động từ quá chung."""
    if not CFG.use_wordnet:
        return {}
    wn, out = _wordnet(), {}
    for v in verbs:
        hy = []
        syns = wn.synsets(v, pos="v") if wn is not None else []
        if syns:
            for path in sorted(syns[0].hypernym_paths(), key=_path_key)[:2]:
                for depth in (len(path) - 2, len(path) - 3):
                    if depth < 0:
                        continue
                    name = path[depth].lemmas()[0].name().replace("_", " ").lower()
                    if name != v and name not in GENERIC_VERBS and " " not in name and name not in hy:
                        hy.append(name)
        for h in FALLBACK_VERB_ISA.get(v, []):
            if h != v and h not in hy:
                hy.append(h)
        if hy:
            out[v] = hy[:3]
    return out


def get_hypernyms(classes, types):
    nouns = [c for c in classes if types.get(c, "noun") == "noun"]
    verbs = [c for c in classes if types.get(c) == "verb"]
    out = get_wordnet_hypernyms(nouns)
    out.update(get_verb_hypernyms(verbs))
    return out


ENV_INFO.update(nltk_status=NLTK_STATUS, wordnet=_wordnet() is not None, pos_tagger=_TAGGER is not None)
print("NLTK:", NLTK_STATUS, "| WordNet:", _wordnet() is not None, "| POS tagger:", _TAGGER is not None)

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next video clip".split())
NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone somebody anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type video clip "
                  "footage scene time minute second".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())
AUX_VERBS = set("be have do can will would shall should may might must seem appear become begin start try "
                "keep continue let get go come".split())


def _norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def _fallback_pos(toks):
    wn, out = _wordnet(), []
    for t in toks:
        if wn is None:
            out.append((t, "NN" if len(t) > 2 and t not in STOP else "DT"))
        elif t.endswith("ing") and wn.synsets(lemma(t, "v"), pos="v"):
            out.append((t, "VBG"))
        elif wn.synsets(t, pos="n"):
            out.append((t, "NN"))
        elif wn.synsets(lemma(t, "v"), pos="v"):
            out.append((t, "VB"))
        else:
            out.append((t, "DT"))
    return out


_PARSE_CACHE = {}


def parse_caption(cap):
    """(danh từ nhìn thấy được, {động từ: dạng V-ing}, bộ ba quan hệ) của MỘT caption — heuristic POS.
    Bộ ba gồm (danh từ, vị từ, danh từ) như notebook gốc + (chủ thể, agent_of, động từ), (đồ vật, object_of,
    động từ) và chiều ngược (done_by, acts_on). Nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    if cap in _PARSE_CACHE:
        return _PARSE_CACHE[cap]
    toks = re.findall(r"[a-z]+", str(cap).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    nouns, verbs, triples = set(), {}, set()
    subj, last_n, preds, cur_v = None, None, [], None
    for w, t in tags:
        if t in ("NN", "NNS") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:
            n = lemma(w, "n")
            if n in NOT_OBJECTS or not is_visual_noun(n):
                continue
            nouns.add(n)
            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                triples.add((last_n, " ".join(preds), n))
            if cur_v is not None and n != subj:
                triples.add((n, "object_of", cur_v)); triples.add((cur_v, "acts_on", n))
                cur_v = None
            subj = subj or n
            last_n, preds = n, []
        elif t.startswith("VB"):
            v = lemma(w, "v")
            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):
                continue
            if t == "VBG":
                verbs[v] = w
            else:
                verbs.setdefault(v, None)
            if subj is not None and v != subj:
                triples.add((subj, "agent_of", v)); triples.add((v, "done_by", subj))
            if last_n is not None:
                preds.append(v)
            cur_v = v
        elif t in ("IN", "TO", "RP") and w not in _SKIP_P and last_n is not None:
            preds.append(w)
    res = (frozenset(nouns), dict(verbs), frozenset(triples))
    _PARSE_CACHE[cap] = res
    return res


def naive_gerund(v):
    if v.endswith("ie"):
        return v[:-2] + "ying"
    if v.endswith("e") and not v.endswith(("ee", "ye", "oe")):
        return v[:-1] + "ing"
    return v + "ing"


def build_vocab(store, split):
    """Từ vựng khái niệm từ caption TRAIN (mục đích 'vocab' — CaptionStore chặn mọi split khác)."""
    tr = split["train"]
    caps = store.by_video(tr, "vocab")
    df, pos_cnt, ger = Counter(), defaultdict(Counter), defaultdict(Counter)
    for v in tr:
        cnt = Counter()
        for c in caps[v]:
            nouns, verbs, _ = parse_caption(c)
            for n in nouns:
                cnt[n] += 1; pos_cnt[n]["noun"] += 1
            for vb, surf in verbs.items():
                cnt[vb] += 1; pos_cnt[vb]["verb"] += 1
                if surf:
                    ger[vb][surf] += 1
        df.update(t for t, k in cnt.items() if k >= CFG.vocab_min_caps)
    terms = sorted(t for t, k in df.items() if k >= CFG.min_term_df)
    types = {t: ("verb" if pos_cnt[t]["verb"] > pos_cnt[t]["noun"] else "noun") for t in terms}
    gerund = {t: (ger[t].most_common(1)[0][0] if ger[t] else naive_gerund(t)) for t in terms if types[t] == "verb"}
    return dict(terms=terms, types=types, gerund=gerund, df={t: int(df[t]) for t in terms},
                built_from="train", n_train_videos=len(tr), sha1=sha1_of([terms, types]))


def build_relations(store, split, vocab):
    """Bộ ba quan hệ của từng video TRAIN (mục đích 'relations'). Video val/test KHÔNG có quan hệ từ caption."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "relations")
    rel = {}
    for v in split["train"]:
        trip = set()
        for c in caps[v]:
            trip |= {t for t in parse_caption(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        rel[v] = sorted(trip)
    return rel


def caption_truth(store, vids, vocab, purpose):
    """Nhãn tham chiếu: từ của từ vựng xuất hiện ≥ truth_min_caps caption của video (chỉ để CHẤM tagger)."""
    vset, caps, out = set(vocab["terms"]), store.by_video(vids, purpose), {}
    for v in vids:
        cnt = Counter()
        for c in caps[v]:
            n, vb, _ = parse_caption(c)
            cnt.update(set(n) | set(vb))
        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}
    return out


_TERM_FEAT = {}


def term_text_features(vocab):
    """CLIP-text của prompt cho từng từ (ensemble 3 prompt; danh từ / động từ dạng V-ing)."""
    key = vocab["sha1"] + sha1_of(vocab["gerund"])
    if key not in _TERM_FEAT:
        terms, T = vocab["terms"], None
        for typ, prompts in (("noun", CFG.noun_prompts), ("verb", CFG.verb_prompts)):
            idx = [i for i, t in enumerate(terms) if vocab["types"][t] == typ]
            if not idx:
                continue
            words = [terms[i] if typ == "noun" else vocab["gerund"][terms[i]] for i in idx]
            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)
            if T is None:
                T = np.zeros((len(terms), E.shape[1]), np.float32)
            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
        _TERM_FEAT[key] = T
    return _TERM_FEAT[key]


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1)
    R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def run_tagger(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):
    """Gắn nhãn MỌI video bằng CLIP zero-shot trên frame với từ vựng TRAIN — không đọc caption của video được gắn.
    select=None ⇒ chọn (pool, norm, k_noun, k_verb) theo F1 trên VAL. Trả về (tags theo vid, lựa chọn, lưới)."""
    terms = vocab["terms"]
    T = term_text_features(vocab)
    tr = np.array(split["train"])
    N, D = vid_feat.shape
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    S_raw = {"video": vid_feat @ T.T, "frame_max": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}
    cols = {t: np.array([i for i, w in enumerate(terms) if vocab["types"][w] == t], int) for t in ("noun", "verb")}

    def normed(S, mode):
        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên video TRAIN
        return S if mode == "raw" else ((S - mu) / sd if mode == "z" else S / S[tr].std() + (S - mu) / sd)

    def topk(Sc, rows, kn, kv):
        out = [set() for _ in rows]
        for typ, k in (("noun", kn), ("verb", kv)):
            c = cols[typ]
            if k <= 0 or len(c) == 0:
                continue
            k = min(k, len(c))
            top = np.argpartition(-Sc[np.ix_(rows, c)], k - 1, axis=1)[:, :k]
            for j, row in enumerate(top):
                out[j].update(terms[c[i]] for i in row)
        return out

    grid = []
    if select is None:
        va = np.array(split["val"])
        truth = caption_truth(store, va, vocab, "tagger_select")
        tv = [truth[v] for v in va]
        for pool in CFG.tag_pools:
            for mode in CFG.tag_norms:
                Sc = normed(S_raw[pool], mode)
                for kn in CFG.tag_k_noun_grid:
                    for kv in CFG.tag_k_verb_grid:
                        P, R, F1 = _prf(topk(Sc, va, kn, kv), tv)
                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))
        best = max(grid, key=lambda g: (round(g["val_F1"], 9), -(g["k_noun"] + g["k_verb"])))
        select = {k: best[k] for k in ("pool", "norm", "k_noun", "k_verb", "val_P", "val_R", "val_F1")}
    Sc = normed(S_raw[select["pool"]], select["norm"])
    tags = topk(Sc, np.arange(N), select["k_noun"], select["k_verb"])
    return {v: sorted(t) for v, t in enumerate(tags)}, select, grid


def tagger_report(tags, vids, vocab, purpose):
    truth = caption_truth(CAPS, vids, vocab, purpose)
    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])
    return dict(P=P, R=R, F1=F1, n_videos=len(vids))

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
REL_CATS = ("agent_of", "object_of", "svo", "prep")
_PARSE2_CACHE = {}


def ckey(lemma_, pos):
    """Khoá node lớp: 'dog#n', 'play#v'."""
    return f"{lemma_}#{pos}"


def parse_caption_v2(cap):
    """Như parse_caption (v1) nhưng (1) khoá theo (lemma, POS), (2) mỗi quan hệ giữ LOẠI và VỊ TỪ.
    Trả về (danh từ, {động từ: dạng V-ing}, {(nguồn, loại, đích, vị từ)})."""
    if cap in _PARSE2_CACHE:
        return _PARSE2_CACHE[cap]
    toks = re.findall(r"[a-z]+", str(cap).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    nouns, verbs, triples = set(), {}, set()
    subj, last_n, preds, pred_has_verb, cur_v = None, None, [], False, None
    for w, t in tags:
        if t in ("NN", "NNS") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:
            n = lemma(w, "n")
            if n in NOT_OBJECTS or not is_visual_noun(n):
                continue
            nouns.add(n)
            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                triples.add((ckey(last_n, "n"), "svo" if pred_has_verb else "prep", ckey(n, "n"), " ".join(preds)))
            if cur_v is not None and n != subj:
                triples.add((ckey(n, "n"), "object_of", ckey(cur_v, "v"), cur_v))
                cur_v = None
            subj = subj or n
            last_n, preds, pred_has_verb = n, [], False
        elif t.startswith("VB"):
            v = lemma(w, "v")
            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):
                continue
            if t == "VBG":
                verbs[v] = w
            else:
                verbs.setdefault(v, None)
            if subj is not None:
                triples.add((ckey(subj, "n"), "agent_of", ckey(v, "v"), v))
            if last_n is not None:
                preds.append(v); pred_has_verb = True
            cur_v = v
        elif t in ("IN", "TO", "RP") and w not in _SKIP_P and last_n is not None:
            preds.append(w)
    res = (frozenset(nouns), dict(verbs), frozenset(triples))
    _PARSE2_CACHE[cap] = res
    return res


def build_vocab_v2(store, split):
    """Từ vựng khoá (lemma, POS) từ caption TRAIN: một lemma có thể là cả 'x#n' và 'x#v' nếu cả hai đạt ngưỡng."""
    tr = split["train"]
    caps = store.by_video(tr, "vocab")
    df, ger = Counter(), defaultdict(Counter)
    for v in tr:
        cnt = Counter()
        for c in caps[v]:
            nouns, verbs, _ = parse_caption_v2(c)
            cnt.update(ckey(n, "n") for n in nouns)
            for vb, surf in verbs.items():
                cnt[ckey(vb, "v")] += 1
                if surf:
                    ger[vb][surf] += 1
        df.update(k for k, m in cnt.items() if m >= CFG.vocab_min_caps)
    terms = sorted(k for k, m in df.items() if m >= CFG.min_term_df)
    lem = {k: k.rsplit("#", 1)[0] for k in terms}
    types = {k: ("verb" if k.endswith("#v") else "noun") for k in terms}
    gerund = {k: (ger[lem[k]].most_common(1)[0][0] if ger[lem[k]] else naive_gerund(lem[k])) for k in terms if types[k] == "verb"}
    both = sorted({lem[k] for k in terms if types[k] == "noun"} & {lem[k] for k in terms if types[k] == "verb"})
    return dict(terms=terms, types=types, lemma=lem, gerund=gerund, df={k: int(df[k]) for k in terms},
                lemmas_with_two_pos=both, built_from="train", n_train_videos=len(tr), sha1=sha1_of([terms, types]))


def build_relations_v2(store, split, vocab):
    """{vid TRAIN: [(nguồn, loại, đích, vị từ)]} — chỉ video TRAIN (mục đích 'relations')."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "relations")
    rel = {}
    for v in split["train"]:
        trip = set()
        for c in caps[v]:
            trip |= {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        rel[v] = sorted(trip)
    return rel


def caption_truth_v2(store, vids, vocab, purpose):
    vset, caps, out = set(vocab["terms"]), store.by_video(vids, purpose), {}
    for v in vids:
        cnt = Counter()
        for c in caps[v]:
            n, vb, _ = parse_caption_v2(c)
            cnt.update({ckey(x, "n") for x in n} | {ckey(x, "v") for x in vb})
        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}
    return out


_TERM_FEAT2 = {}


def term_text_features_v2(vocab):
    """CLIP-text (ensemble 3 prompt) cho từng khoá: danh từ dùng lemma, động từ dùng dạng V-ing."""
    key = vocab["sha1"] + sha1_of(vocab["gerund"])
    if key not in _TERM_FEAT2:
        terms, T = vocab["terms"], None
        for typ, prompts in (("noun", CFG.noun_prompts), ("verb", CFG.verb_prompts)):
            idx = [i for i, t in enumerate(terms) if vocab["types"][t] == typ]
            if not idx:
                continue
            words = [vocab["lemma"][terms[i]] if typ == "noun" else vocab["gerund"][terms[i]] for i in idx]
            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)
            if T is None:
                T = np.zeros((len(terms), E.shape[1]), np.float32)
            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
        _TERM_FEAT2[key] = T
    return _TERM_FEAT2[key]


def run_tagger_v2(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):
    """Như run_tagger (v1) với khoá (lemma, POS). Trả thêm điểm đã chuẩn hoá của từng nhãn được gắn
    (dùng làm trọng số contains ở S3). Thống kê chuẩn hoá chỉ trên video TRAIN; lựa chọn theo F1 trên VAL."""
    terms = vocab["terms"]
    T = term_text_features_v2(vocab)
    tr = np.array(split["train"])
    N, D = vid_feat.shape
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    S_raw = {"video": vid_feat @ T.T, "frame_max": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}
    cols = {t: np.array([i for i, w in enumerate(terms) if vocab["types"][w] == t], int) for t in ("noun", "verb")}

    def normed(S, mode):
        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6
        return S if mode == "raw" else ((S - mu) / sd if mode == "z" else S / S[tr].std() + (S - mu) / sd)

    def topk(Sc, rows, kn, kv):
        out = [dict() for _ in rows]
        for typ, k in (("noun", kn), ("verb", kv)):
            c = cols[typ]
            if k <= 0 or len(c) == 0:
                continue
            k = min(k, len(c))
            sub = Sc[np.ix_(rows, c)]
            top = np.argpartition(-sub, k - 1, axis=1)[:, :k]
            for j, row in enumerate(top):
                out[j].update({terms[c[i]]: float(sub[j, i]) for i in row})
        return out

    grid = []
    if select is None:
        va = np.array(split["val"])
        truth = caption_truth_v2(store, va, vocab, "tagger_select")
        tv = [truth[v] for v in va]
        for pool in CFG.tag_pools:
            for mode in CFG.tag_norms:
                Sc = normed(S_raw[pool], mode)
                for kn in CFG.tag_k_noun_grid:
                    for kv in CFG.tag_k_verb_grid:
                        P, R, F1 = _prf([set(d) for d in topk(Sc, va, kn, kv)], tv)
                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))
        best = max(grid, key=lambda g: (round(g["val_F1"], 9), -(g["k_noun"] + g["k_verb"])))
        select = {k: best[k] for k in ("pool", "norm", "k_noun", "k_verb", "val_P", "val_R", "val_F1")}
    Sc = normed(S_raw[select["pool"]], select["norm"])
    tagged = topk(Sc, np.arange(N), select["k_noun"], select["k_verb"])
    tags = {v: sorted(d) for v, d in enumerate(tagged)}
    scores = {v: d for v, d in enumerate(tagged)}
    return tags, scores, select, grid


def tagger_report_v2(tags, vids, vocab, purpose):
    truth = caption_truth_v2(CAPS, vids, vocab, purpose)
    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])
    return dict(P=P, R=R, F1=F1, n_videos=len(vids))

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Chất lượng quan hệ (Nhiệm vụ B): mẫu cho NGƯỜI chấm + đọc nhãn đã chấm. Chỉ caption TRAIN. ---
# Mẫu KHÔNG phụ thuộc CLIP/tagger (chỉ parser + từ vựng train) ⇒ giống nhau giữa máy local và Kaggle.
# Đơn vị mẫu = MỘT quan hệ (họ, nguồn, vị từ, đích) đủ điều kiện thành cạnh (support ≥ scene_min_count video TRAIN),
# kèm MỘT caption TRAIN chọn ngẫu nhiên có chứa nó; người chấm trả lời: caption này có diễn đạt đúng quan hệ đó không?
CaptionStore.ALLOWED["annotation"] = {"train"}
ANNOT_LABEL_COLS = ("annotator_1", "annotator_2", "adjudicated")


def relation_instances(store, split, vocab):
    """[(vid, caption, (nguồn, họ, đích, vị từ))] trên caption TRAIN; support = số video TRAIN có quan hệ (đếm 1 lần / video);
    pred_support[(họ, vị từ)] = số video TRAIN có ít nhất một quan hệ thuộc họ đó với vị từ đó."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "annotation")
    inst, sup, psup = [], Counter(), Counter()
    for v in split["train"]:
        seen = set()
        for c in caps[v]:
            for t in sorted(parse_caption_v2(c)[2]):
                if t[0] in vset and t[2] in vset and t[0] != t[2]:
                    inst.append((v, c, t)); seen.add(t)
        sup.update(seen)
        psup.update({(t[1], t[3]) for t in seen})
    return inst, sup, psup


def support_stratum(s):
    for lo, hi, name in CFG.annot_strata:
        if lo <= s <= hi:
            return name
    return "below_threshold"


def _display_relation(src, rel, dst, pred):
    a, b = src.rsplit("#", 1)[0], dst.rsplit("#", 1)[0]
    if rel == "agent_of":
        return a, pred, "", f"'{a}' là CHỦ THỂ thực hiện hành động '{pred}'"
    if rel == "object_of":
        return "", pred, a, f"'{a}' là ĐỐI TƯỢNG chịu hành động '{pred}'"
    return a, pred, b, f"{a} —{pred}→ {b}"


def build_annotation_sample(store, split, vocab, records):
    """Phân tầng: họ quan hệ × mức phổ biến (support); trong mỗi ô chọn luân phiên theo vị từ để đa dạng vị từ."""
    inst, sup, psup = relation_instances(store, split, vocab)
    by_trip = defaultdict(list)
    for v, c, t in inst:
        if sup[t] >= CFG.scene_min_count:
            by_trip[t].append((v, c))
    rng = np.random.RandomState(CFG.annot_seed)
    rows, cells = [], []
    for rel in REL_CATS:
        for lo, hi, name in CFG.annot_strata:
            trips = sorted(t for t in by_trip if t[1] == rel and lo <= sup[t] <= hi)
            by_pred = defaultdict(list)
            for t in trips:
                by_pred[t[3]].append(t)
            preds = sorted(by_pred)
            rng.shuffle(preds)
            for p in preds:
                rng.shuffle(by_pred[p])
            chosen = []
            while len(chosen) < CFG.annot_n_per_cell and any(by_pred[p] for p in preds):
                for p in preds:                              # luân phiên vị từ ⇒ không vị từ nào chiếm cả ô
                    if by_pred[p] and len(chosen) < CFG.annot_n_per_cell:
                        chosen.append(by_pred[p].pop())
            dbl = set(rng.choice(len(chosen), min(CFG.annot_n_double_per_cell, len(chosen)), replace=False).tolist()) if chosen else set()
            cells.append(dict(family=rel, stratum=name, support_lo=lo, support_hi=hi, n_relations_in_cell=len(trips),
                              n_predicates_in_cell=len(preds), n_sampled=len(chosen)))
            for j, t in enumerate(chosen):
                v, c = by_trip[t][rng.randint(len(by_trip[t]))]
                s_, p_, o_, txt = _display_relation(t[0], t[1], t[2], t[3])
                vid = records[v]["video_id"]
                rows.append(dict(sample_id=hashlib.sha1("|".join([vid, c, t[1], t[0], t[3], t[2]]).encode()).hexdigest()[:12],
                                 family=rel, caption=c, subject=s_, predicate=p_, object=o_, relation=txt,
                                 double_annotate=int(j in dbl), annotator_1="", annotator_2="", adjudicated="", note="",
                                 video_id=vid, src_node=t[0], dst_node=t[2], stratum=name, relation_support_videos=int(sup[t]),
                                 predicate_support_videos=int(psup[(t[1], t[3])])))
    df = pd.DataFrame(rows)
    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)      # trộn thứ tự: người chấm không thấy khối theo ô
    return df, pd.DataFrame(cells), sup, psup


def _label(x):
    x = str(x).strip().lower()
    return 1 if x in ("1", "1.0", "đúng", "dung", "true", "yes", "y") else (0 if x in ("0", "0.0", "sai", "false", "no", "n") else None)


def _read_table(p):
    if p.suffix.lower() == ".xlsx":
        return pd.read_excel(p, dtype=str, keep_default_na=False, sheet_name=0)
    for enc in ("utf-8-sig", "cp1252"):                # Excel có thể lưu ';' hoặc mã hoá khác — nhãn chỉ là 1/0/?
        try:
            return pd.read_csv(p, dtype=str, keep_default_na=False, sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"không đọc được {p}")


def find_label_file():
    """CFG.annot_file, hoặc relation_annotation*.(csv|xlsx) trong /kaggle/input có ít nhất một nhãn ở annotator_1."""
    cands = [Path(CFG.annot_file)] if CFG.annot_file else []
    if RUN_ON_KAGGLE:
        cands += sorted(p for p in Path("/kaggle/input").rglob("relation_annotation*") if p.suffix.lower() in (".csv", ".xlsx"))
    for p in cands:
        try:
            df = _read_table(p)
            if "annotator_1" in df and df["annotator_1"].map(_label).notna().any():
                return p, df
        except Exception as e:
            print(f"⚠ Không đọc được {p}: {type(e).__name__}: {e}")
    return None, None


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h


def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return float("nan")
    po = float((a == b).mean())
    pe = float(a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean()))
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")


def relation_quality(labels):
    """Precision theo họ và theo ô (họ × mức phổ biến) từ nhãn NGƯỜI chấm. Nhãn cuối = adjudicated nếu có; nếu không:
    annotator_1 khi chỉ có một người chấm hoặc hai người đồng ý; hai người bất đồng mà chưa phân xử ⇒ loại khỏi precision."""
    L = labels.copy()
    for c in ANNOT_LABEL_COLS:
        L[c + "_v"] = L[c].map(_label) if c in L else None
    final, status = [], []
    for a1, a2, ad in zip(L["annotator_1_v"], L["annotator_2_v"], L["adjudicated_v"]):
        if not pd.isna(ad):
            final.append(ad); status.append("adjudicated")
        elif pd.isna(a1):
            final.append(None); status.append("unlabeled_or_unclear")
        elif pd.isna(a2):
            final.append(a1); status.append("single")
        elif a1 == a2:
            final.append(a1); status.append("agree")
        else:
            final.append(None); status.append("disagree_unresolved")
    L["final"], L["label_status"] = final, status
    both = L[L["annotator_1_v"].notna() & L["annotator_2_v"].notna()]
    rows = []
    for keys, sub in [(("ALL", "ALL"), L)] + [((f, "ALL"), g) for f, g in L.groupby("family")] + \
            [((f, s), g) for (f, s), g in L.groupby(["family", "stratum"])]:
        lab = sub[sub["final"].notna()]
        k, n = int(lab["final"].sum()), len(lab)
        p, lo, hi = wilson(k, n)
        rows.append(dict(family=keys[0], stratum=keys[1], n_rows=len(sub), n_labeled=n, n_correct=k, precision=p, wilson_lo=lo,
                         wilson_hi=hi, n_unclear_or_unlabeled=int((sub["label_status"] == "unlabeled_or_unclear").sum()),
                         n_disagree_unresolved=int((sub["label_status"] == "disagree_unresolved").sum()),
                         verified=bool(keys[1] != "ALL" and n > 0 and lo >= CFG.prec_min_lower)))
    q = pd.DataFrame(rows)
    agree = dict(n_double=len(both), kappa=cohen_kappa(both["annotator_1_v"].astype(int), both["annotator_2_v"].astype(int))
                 if len(both) else float("nan"), raw_agreement=float((both["annotator_1_v"] == both["annotator_2_v"]).mean())
                 if len(both) else float("nan"))
    return q, agree, L

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Kiểm chứng TỰ ĐỘNG chất lượng quan hệ (không cần người chấm): đồng thuận với một parser phụ thuộc cú pháp ĐỘC LẬP. ---
# Parser của đồ thị là heuristic POS (v2). spaCy (dependency parse, mô hình thống kê khác hẳn) trích lại quan hệ CÙNG schema trên
# CHÍNH caption TRAIN đó; một quan hệ được "xác nhận" nếu spaCy cũng trích ra nó. Đây là nhãn SILVER (đồng thuận giữa hai
# phương pháp) — KHÔNG phải precision do người chấm: hai parser có thể cùng sai. Chỉ caption TRAIN (mục đích 'agreement').
import importlib

CaptionStore.ALLOWED["agreement"] = {"train"}
_NLP, _DEP_CACHE, DEP_STATUS = None, {}, "chưa nạp"
AGREE, AGREE_COVER = None, None                 # gán ở cell chạy từ vựng (chỉ TRAIN)
_NOUN_POS, _SUBJ, _OBJ = ("NOUN", "PROPN"), ("nsubj",), ("dobj", "obj", "nsubjpass")


def dep_parser():
    """Nạp spaCy + en_core_web_sm (tải nếu thiếu và có Internet). Không được ⇒ None (kiểm chứng tự động = NOT RUN)."""
    global _NLP, DEP_STATUS
    if _NLP is None:
        try:
            spacy = ensure_package("spacy", "spacy")
            try:
                _NLP = spacy.load(CFG.dep_model, disable=["ner", "lemmatizer"])
            except OSError:
                if not INTERNET:
                    raise
                r = subprocess.run([sys.executable, "-m", "spacy", "download", CFG.dep_model], capture_output=True, text=True)
                if r.returncode != 0:
                    raise RuntimeError(r.stderr[-500:])
                importlib.invalidate_caches()
                _NLP = spacy.load(CFG.dep_model, disable=["ner", "lemmatizer"])
            DEP_STATUS = f"spaCy {spacy.__version__} / {CFG.dep_model} {_NLP.meta.get('version')}"
        except Exception as e:
            _NLP, DEP_STATUS = False, f"NOT RUN — không nạp được spaCy/{CFG.dep_model}: {type(e).__name__}: {str(e)[:200]}"
        ENV_INFO["dep_parser"] = DEP_STATUS
    return _NLP or None


def _nkey(t):
    return ckey(lemma(t.text.lower(), "n"), "n")


def _vkey(t):
    return ckey(lemma(t.text.lower(), "v"), "v")


def _with_conj(t):
    return [t] + [c for c in t.conjuncts if c.pos_ in _NOUN_POS]


def _subjects(v, depth=0):
    s = [c for c in v.children if c.dep_ in _SUBJ and c.pos_ in _NOUN_POS]
    if not s and depth < 3:
        if v.dep_ in ("conj", "xcomp", "advcl") and v.head.pos_ in ("VERB", "AUX"):
            return _subjects(v.head, depth + 1)                  # "sitting and wagging", "tries to ride"
        if v.dep_ in ("acl", "relcl") and v.head.pos_ in _NOUN_POS:
            return _with_conj(v.head)                            # "a lady making a dish"
    return [x for t in s for x in _with_conj(t)]


def dep_relations(doc):
    """Quan hệ theo CÙNG schema với parser heuristic: agent {(danh, động)}, object {(danh, động)},
    svo {(chủ, đối tượng, động từ)}, prep {(danh, danh, giới từ)} — giới từ phải gắn vào DANH TỪ (không vào động từ)."""
    out = dict(agent=set(), object=set(), svo=set(), prep=set())
    for t in doc:
        if t.pos_ == "VERB":
            v = lemma(t.text.lower(), "v")
            subs = _subjects(t)
            objs = [x for c in t.children if c.dep_ in _OBJ and c.pos_ in _NOUN_POS for x in _with_conj(c)]
            pobjs = [(p.text, x) for p in t.children if p.dep_ == "prep" for c in p.children
                     if c.dep_ == "pobj" and c.pos_ in _NOUN_POS for x in _with_conj(c)]
            out["agent"] |= {(_nkey(s), _vkey(t)) for s in subs}
            out["object"] |= {(_nkey(o), _vkey(t)) for o in objs}
            out["svo"] |= {(_nkey(s), _nkey(o), v) for s in subs for o in objs + [x for _, x in pobjs]}
        elif t.dep_ == "prep" and t.head.pos_ in _NOUN_POS:
            for c in t.children:
                if c.dep_ == "pobj" and c.pos_ in _NOUN_POS:
                    out["prep"] |= {(_nkey(t.head), _nkey(x), t.text.lower()) for x in _with_conj(c)}
    return out


def dep_parse_all(texts):
    nlp = dep_parser()
    if nlp is None:
        return False
    todo = sorted({x.lower() for x in texts} - set(_DEP_CACHE))
    for txt, doc in zip(todo, nlp.pipe(todo, batch_size=512)):
        _DEP_CACHE[txt] = dep_relations(doc)
    return True


def triple_confirmed(t, dep):
    """Quan hệ heuristic (nguồn, họ, đích, vị từ) được spaCy xác nhận trên cùng caption?
    svo: cùng (chủ, đối tượng) và CÙNG động từ đầu của vị từ; prep: cùng (danh, danh) và giới từ thuộc vị từ."""
    s, rel, o, p = t
    if rel == "agent_of":
        return (s, o) in dep["agent"]
    if rel == "object_of":
        return (s, o) in dep["object"]
    if rel == "svo":
        return (s, o, p.split()[0]) in dep["svo"]
    return any((s, o, w) in dep["prep"] for w in p.split())


def build_agreement(store, split, vocab):
    """AGREE[(họ, nguồn, đích, vị từ)] = [số caption TRAIN spaCy xác nhận, số caption TRAIN có quan hệ]; cùng điều kiện từ
    vựng với build_relations_v2. Kèm độ phủ ngược: tỉ lệ quan hệ spaCy (trong từ vựng) mà heuristic cũng trích được."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "agreement")
    texts = [c for v in split["train"] for c in caps[v]]
    if not dep_parse_all(texts):
        return None, None
    agree, cover = defaultdict(lambda: [0, 0]), Counter()
    for c in texts:
        dep = _DEP_CACHE[c.lower()]
        heur = {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        for t in heur:
            a = agree[(t[1], t[0], t[2], t[3])]
            a[0] += int(triple_confirmed(t, dep)); a[1] += 1
        hk = {(t[1], t[0], t[2]) for t in heur}
        for fam, rel in (("agent", "agent_of"), ("object", "object_of")):
            for s, o in dep[fam]:
                if s in vset and o in vset:
                    cover[(rel, "total")] += 1; cover[(rel, "found")] += int((rel, s, o) in hk)
        for fam in ("svo", "prep"):
            for s, o, _ in dep[fam]:
                if s in vset and o in vset and s != o:
                    cover[(fam, "total")] += 1; cover[(fam, "found")] += int((fam, s, o) in hk)
    return {k: tuple(v) for k, v in agree.items()}, dict(cover)


def agreement_rate(rel, s, o, p):
    a = AGREE.get((rel, s, o, p)) if AGREE else None
    return a[0] / a[1] if a and a[1] else float("nan")


def agreement_quality(agree, cover, sup):
    """Bảng SILVER theo họ và theo ô (họ × mức phổ biến): tỉ lệ caption-instance được parser độc lập xác nhận (Wilson 95%)."""
    rows = []
    cells = defaultdict(lambda: [0, 0])
    for (rel, s, o, p), (k, n) in agree.items():
        st = support_stratum(sup.get((s, rel, o, p), 0))
        for key in ((rel, "ALL"), (rel, st), ("ALL", "ALL")):
            cells[key][0] += k; cells[key][1] += n
    for (rel, st), (k, n) in sorted(cells.items()):
        p_, lo, hi = wilson(k, n)
        cv = (cover.get((rel, "found"), 0) / cover[(rel, "total")]) if st == "ALL" and cover.get((rel, "total")) else float("nan")
        rows.append(dict(kind="silver_parser_agreement", family=rel, stratum=st, n_instances=n, n_confirmed=k, agreement=p_,
                         wilson_lo=lo, wilson_hi=hi, heuristic_covers_spacy=cv,
                         note="đồng thuận heuristic ↔ spaCy trên caption TRAIN; KHÔNG phải precision do người chấm"))
    return pd.DataFrame(rows)

## 4.8 Đặc tả đồ thị v3 (schema)

**Node.** Canonical inventory **cố định**: video MSVD + video tập ngoài, lớp `(lemma, POS)` từ train, khái niệm theo synset WordNet. Ablation chỉ **tắt cạnh**.

**Cạnh.** Mỗi bản ghi cạnh trong `graph_edges_v3.csv` gồm `(src, relation_family, relation, predicate, dst, direction, weight, confidence, source, status)`. Mỗi dòng là **một** quan hệ: một cặp có nhiều vị từ thành nhiều dòng.
* Trường `predicate` **không bao giờ** chứa tên loại quan hệ; có kiểm tra G2 cho điều này.
* `weight` là support (tần suất), **không** phải độ tin cậy ngữ nghĩa.
* `confidence` là `silver <tỉ lệ> (k/n)`: tỉ lệ caption train của quan hệ được parser độc lập xác nhận. Nếu có nhãn người chấm thì là `human <precision>`; không có cả hai thì là `unverified`.
* Cạnh bị loại vẫn được ghi với `status = excluded:<lý do>`.

**Mẫu đường đi.** Gồm 12 mẫu của v2 cộng `P1m_svo` và `P1m_prep`: đường đi `video → a –f[p]→ b` chỉ được tính khi truy vấn nhắc tới vị từ `p`.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
FAMILIES = ("agent", "object", "svo", "prep")
FAMILY_REL = {"agent": "agent_of", "object": "object_of", "svo": "svo", "prep": "prep"}
_SCENE_SRC = "caption TRAIN (POS parser heuristic) — đếm số video TRAIN hỗ trợ; giữ nếu ≥ scene_min_count video, ≤ scene_top_k cạnh / (nguồn, loại)"
_SCENE_W = ("support weight = min(1, support / df_train(nguồn)); chiều _inv: min(1, support / df_train(đích)). "
            "Đây là TẦN SUẤT, KHÔNG phải độ tin cậy ngữ nghĩa")
_CONF_REL = "precision của ô (họ × mức phổ biến) theo nhãn NGƯỜI chấm (Wilson 95%); không có nhãn ⇒ 'unverified'"
_PRED_VERB = "lemma động từ = CHÍNH node đích/nguồn động từ ⇒ không mang thông tin riêng của cạnh; ghi lại, KHÔNG đưa vào message"
_PRED_FREE = ("phân phối vị từ của cặp (vị từ → số video TRAIN); embedding vị từ = CLIP-text đóng băng theo prompt "
              "(pred_mode 'clip') hoặc + MLP dư dùng chung (pred_mode 'clip_adapt'); vào message của GNN và mẫu đường đi P1m")
REL_SPEC = {
    "contains":      dict(src="video", tgt="class", direction="video → lớp được gắn", family="contains", inverse=None,
                          source="CLIP zero-shot trên frame của CHÍNH video × từ vựng train (tagger chọn trên val)",
                          weight="1 (binary) | độ tin cậy tagger đã chuẩn hoá (contains = conf, chọn trên val)",
                          predicate="—", confidence="điểm tagger CLIP (chỉ khi contains = conf)",
                          split="frame mọi video; từ vựng + thống kê chuẩn hoá: train", usage="gnn+path"),
    "contains_inv":  dict(src="class", tgt="video", direction="lớp → video chứa nó", family="contains", inverse="contains",
                          source="chiều ngược của contains", weight="= contains", predicate="—", confidence="= contains",
                          split="như contains", usage="gnn"),
    "agent_of":      dict(src="class:noun", tgt="class:verb", direction="chủ thể → hành động", family="agent", inverse=None,
                          source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL, split="train", usage="gnn+path"),
    "agent_of_inv":  dict(src="class:verb", tgt="class:noun", direction="hành động → chủ thể (done_by)", family="agent",
                          inverse="agent_of", source="chiều ngược của agent_of", weight=_SCENE_W, predicate=_PRED_VERB,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "object_of":     dict(src="class:noun", tgt="class:verb", direction="đồ vật → hành động tác động lên nó", family="object",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "object_of_inv": dict(src="class:verb", tgt="class:noun", direction="hành động → đồ vật (acts_on)", family="object",
                          inverse="object_of", source="chiều ngược của object_of", weight=_SCENE_W, predicate=_PRED_VERB,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "svo":           dict(src="class:noun", tgt="class:noun", direction="chủ thể –vị từ→ đối tượng", family="svo",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "svo_inv":       dict(src="class:noun", tgt="class:noun", direction="đối tượng –bị vị từ bởi→ chủ thể", family="svo",
                          inverse="svo", source="chiều ngược của svo", weight=_SCENE_W, predicate=_PRED_FREE,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "prep":          dict(src="class:noun", tgt="class:noun", direction="danh từ –giới từ→ danh từ", family="prep",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "prep_inv":      dict(src="class:noun", tgt="class:noun", direction="chiều ngược của prep", family="prep", inverse="prep",
                          source="chiều ngược của prep", weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "related":       dict(src="class", tgt="class", direction="đối xứng (lưu cả hai chiều)", family="related", inverse=None,
                          source="NPMI ≥ ngưỡng của nhãn tagger trên video TRAIN; ≥ min_count; ≤ cooc_top_k / nguồn",
                          weight="P(đích | nguồn) = count / df_train(nguồn)", predicate="—",
                          confidence="— (đồng xuất hiện ≠ quan hệ ngữ nghĩa trực tiếp)", split="train (nhãn tagger từ frame)",
                          usage="gnn+path"),
    "isa":           dict(src="class", tgt="concept|class (cùng POS)", direction="lớp → khái niệm tổng quát hơn",
                          family="isa", inverse=None, source="WordNet (hypernym theo biến thể taxonomy)", weight="1",
                          predicate="—", confidence="— (sense mơ hồ ⇒ KHÔNG nối)",
                          split="tri thức ngoài (chọn sense: ngữ cảnh caption TRAIN)", usage="gnn+path"),
    "isa_inv":       dict(src="concept|class", tgt="class", direction="khái niệm → lớp cụ thể", family="isa", inverse="isa",
                          source="chiều ngược của isa", weight="1", predicate="—", confidence="—", split="như isa", usage="gnn"),
}
EDGE_TYPES = list(REL_SPEC)
ETYPE_ID = {t: i for i, t in enumerate(EDGE_TYPES)}
OPEN_TYPES = ("contains", "contains_inv")            # cổng mở sẵn (như v1)
RELATION_TYPES = [t for t in EDGE_TYPES if REL_SPEC[t]["family"] != "contains"]
PRED_FAMILIES = tuple(CFG.pred_families)             # họ có vị từ tự do (svo, prep)
PRED_TYPES = [t for f in PRED_FAMILIES for t in (f, f + "_inv")]
MASKS = {
    "none": (),
    "no_scene": tuple(t for t in EDGE_TYPES if REL_SPEC[t]["family"] in FAMILIES),
    "no_related": ("related",),
    "no_taxonomy": ("isa", "isa_inv"),
    "contains_only": tuple(RELATION_TYPES),
    "no_roles": ("agent_of", "agent_of_inv", "object_of", "object_of_inv"),   # Level 4(b): họ cạnh vai
    "no_svo_prep": tuple(PRED_TYPES),                                          # Level 4(b): họ cạnh có vị từ tự do
}
# P1m_f: đường đi video → lớp a –f[vị từ p]→ lớp b CHỈ được tính khi truy vấn nhắc tới p (khớp vị từ); dùng ở path_mode 'pred_match'
PATTERNS = (["P0_contains"] + [f"P1_{f}" for f in FAMILIES] + ["P2_isa", "P3_related"]
            + [f"P4_{f}_isa" for f in FAMILIES] + ["P5_related_isa"] + [f"P1m_{f}" for f in PRED_FAMILIES])
PATTERN_LEN = [1] + [2] * 4 + [2, 2] + [3] * 4 + [3] + [2] * len(PRED_FAMILIES)
PIDX = {p: i for i, p in enumerate(PATTERNS)}
REL_TEXT = {"contains": "chứa", "agent_of": "là_chủ_thể_của", "agent_of_inv": "do", "object_of": "là_đối_tượng_của",
            "object_of_inv": "tác_động_lên", "svo": "{pred}", "svo_inv": "bị_{pred}_bởi", "prep": "{pred}",
            "prep_inv": "có_{pred}", "related": "đồng_xuất_hiện", "isa": "là_một"}

### 4.8b Taxonomy (giữ nguyên cách dựng của v2)
Khái niệm được khoá theo synset WordNet (ví dụ `animal.n.01`), nên danh từ và động từ không bao giờ bị trộn với nhau. Có bốn biến thể trên **cùng** một tập node (tập node là hợp của các biến thể):

| Biến thể | Cách chọn hypernym |
|---|---|
| `heuristic` | quy tắc v1, lấy sense đầu tiên |
| `filtered` | `heuristic` bỏ các khái niệm "hub" (hơn 15% video train với tới) và khái niệm danh từ quá nông |
| `sense` | chọn sense bằng Extended Lesk trên ngữ cảnh caption **train**; mapping **mơ hồ** thì không nối `isa` và được ghi rõ |
| `none` | không có `isa` |

Mọi mapping (synset, gloss, điểm Lesk, cờ mơ hồ, hub) được xuất ra `graph_taxonomy_audit.csv`.

Ở v3, `filtered` thắng val nhưng thua trên test đã nhìn, nên **không** còn là ứng viên. Stage V4 so sánh `none` (mặc định, đơn giản hơn) với taxonomy không lọc (`heuristic`). `heuristic` chỉ được giữ khi thắng theo luật đăng ký trước. Kiểm tra mơ hồ, tương thích danh/động từ, hypernym quá chung, hub và chu trình vẫn được ghi trong audit.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
CaptionStore.ALLOWED["taxonomy"] = {"train"}          # chọn sense WordNet theo ngữ cảnh caption TRAIN
_WTOK = re.compile(r"[a-z]+")


def _head(s):
    return s.lemmas()[0].name().replace("_", " ").lower()


def _noun_targets(s, word):
    """Quy tắc hypernym danh từ của v1 (cha/ông đủ sâu, nhìn thấy được + BASIC_CATEGORIES) — trả về synset."""
    hy = []
    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
        for depth in range(len(path) - 2, -1, -1):
            h = path[depth]
            name = _head(h)
            near = depth >= len(path) - 3
            if name == word or name in GENERIC_CONCEPTS or h in hy:
                continue
            if (near and depth >= CFG.isa_min_depth and (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                    or name in BASIC_CATEGORIES:
                hy.append(h)
    return hy[:5]


def _verb_targets(s, word):
    """Quy tắc hypernym động từ của v1 (cha/ông, bỏ động từ quá chung)."""
    hy = []
    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
        for depth in (len(path) - 2, len(path) - 3):
            if depth < 0:
                continue
            h = path[depth]
            name = _head(h)
            if name != word and name not in GENERIC_VERBS and " " not in name and h not in hy:
                hy.append(h)
    return hy[:3]


def _signature(s):
    """Extended Lesk: từ trong định nghĩa, ví dụ, lemma của synset + cha + 10 con (đã lemma hoá)."""
    words = set(_WTOK.findall(s.definition().lower()))
    for e in s.examples():
        words |= set(_WTOK.findall(e.lower()))
    for r in [s] + sorted(s.hypernyms(), key=_syn_key) + sorted(s.hyponyms(), key=_syn_key)[:10]:
        words |= {l.name().lower() for l in r.lemmas() if "_" not in l.name()}
    return ({lemma(w, "n") for w in words} | {lemma(w, "v") for w in words}) - STOP


def build_sense_context(store, split, vset):
    """Ngữ cảnh của mỗi lớp = các danh từ/động từ khác cùng xuất hiện trong caption TRAIN chứa lớp đó."""
    caps = store.by_video(split["train"], "taxonomy")
    ctx = defaultdict(Counter)
    for v in split["train"]:
        for c in caps[v]:
            nouns, verbs, _ = parse_caption_v2(c)
            words = set(nouns) | set(verbs)
            for k in {ckey(n, "n") for n in nouns} | {ckey(x, "v") for x in verbs}:
                if k in vset:
                    ctx[k].update(words - {k.rsplit("#", 1)[0]})
    return ctx


def choose_sense(lem, pos, ctx):
    wn = _wordnet()
    syns = wn.synsets(lem.replace(" ", "_"), pos=pos)[:CFG.sense_max_candidates] if wn is not None else []
    if not syns:
        return None, True, []
    scores = [sum(math.log1p(ctx[w]) for w in sorted(_signature(s)) if w in ctx) for s in syns]   # thứ tự cố định
    order = np.argsort(-np.array(scores), kind="stable")
    b = scores[order[0]]
    s2 = scores[order[1]] if len(syns) > 1 else -1e9
    ambiguous = len(syns) > 1 and (b <= 0 or b - s2 < CFG.sense_margin)
    return syns[order[0]], ambiguous, [(syns[i].name(), round(scores[i], 2)) for i in order[:3]]


def build_taxonomy_v2(vocab, store, split, train_tags):
    """Ba biến thể taxonomy trên cùng một tập khái niệm (hợp) ⇒ inventory không đổi giữa các biến thể:
    heuristic (= quy tắc v1, sense đầu tiên), filtered (bỏ hub/khái niệm nông), sense (chọn sense theo ngữ cảnh
    caption TRAIN; mơ hồ ⇒ KHÔNG nối isa và đánh dấu), none (không isa)."""
    wn = _wordnet()
    vset = set(vocab["terms"])
    ctx = build_sense_context(store, split, vset)
    concept_meta, var = {}, {"heuristic": {}, "sense": {}, "filtered": {}, "none": {}}
    rows = []

    def target_key(h, pos):
        k = ckey(_head(h), pos)
        if k in vset:
            return k                                   # hypernym trùng một lớp cùng POS ⇒ nối thẳng vào lớp đó
        concept_meta.setdefault(h.name(), dict(key=h.name(), lemma=_head(h), pos=pos, synset=h.name(),
                                               gloss=h.definition(), depth=h.min_depth(), source="wordnet"))
        return h.name()

    def fallback(lem, pos):
        names = (FALLBACK_ISA if pos == "n" else FALLBACK_VERB_ISA).get(lem, [])
        out = []
        for nm in names:
            k = ckey(nm, pos)
            if k not in vset:
                k = f"{nm}.{pos}.fallback"
                concept_meta.setdefault(k, dict(key=k, lemma=nm, pos=pos, synset="", gloss="", depth=-1, source="fallback"))
            out.append(k)
        return out

    for k in vocab["terms"]:
        lem, pos = k.rsplit("#", 1)
        syns = wn.synsets(lem.replace(" ", "_"), pos=pos) if wn is not None else []
        tf = _noun_targets if pos == "n" else _verb_targets
        heur = [target_key(h, pos) for h in tf(syns[0], lem)] if syns else fallback(lem, pos)
        s_syn, amb, scores = choose_sense(lem, pos, ctx[k])
        sense = [] if (amb or s_syn is None) else [target_key(h, pos) for h in tf(s_syn, lem)]
        heur = list(dict.fromkeys(t for t in heur if t != k))
        sense = list(dict.fromkeys(t for t in sense if t != k))
        var["heuristic"][k], var["sense"][k] = heur, sense
        rows.append(dict(class_key=k, pos=pos, train_df=vocab["df"][k], n_synsets=len(syns),
                         heuristic_synset=syns[0].name() if syns else "", heuristic_gloss=syns[0].definition() if syns else "",
                         sense_synset=s_syn.name() if s_syn is not None else "", sense_ambiguous=bool(amb),
                         sense_differs=bool(syns) and s_syn is not None and s_syn != syns[0], lesk_top3=scores,
                         heuristic_targets=heur, sense_targets=sense))
    # filtered: bỏ khái niệm 'hub' (với tới > tax_hub_frac video TRAIN) và khái niệm danh từ quá nông
    tr = list(split["train"])
    reach = defaultdict(set)
    for v in tr:
        for c in train_tags.get(v, []):
            for t in var["heuristic"].get(c, []):
                reach[t].add(v)
    hub = {t for t, s in reach.items() if len(s) / max(len(tr), 1) > CFG.tax_hub_frac}
    shallow = {t for t, m in concept_meta.items() if m["pos"] == "n" and 0 <= m["depth"] < CFG.tax_min_depth_filtered}
    for row in rows:
        k = row["class_key"]
        var["filtered"][k] = [t for t in var["heuristic"][k] if t not in hub and t not in shallow]
        row["filtered_targets"] = var["filtered"][k]
        row["dropped_hub"] = [t for t in var["heuristic"][k] if t in hub]
        row["dropped_shallow"] = [t for t in var["heuristic"][k] if t in shallow]
    var["none"] = {k: [] for k in vocab["terms"]}
    hub_rows = sorted(((t, len(s), len(s) / max(len(tr), 1)) for t, s in reach.items()), key=lambda x: -x[1])
    cycles = {}
    for name, tax in var.items():                       # chu trình trên các cạnh lớp → lớp
        nxt = {k: [t for t in ts if t in vset] for k, ts in tax.items()}
        seen, stack, cyc = set(), set(), 0

        def dfs(u):
            nonlocal cyc
            seen.add(u); stack.add(u)
            for w in nxt.get(u, []):
                if w in stack:
                    cyc += 1
                elif w not in seen:
                    dfs(w)
            stack.discard(u)
        for u in nxt:
            if u not in seen:
                dfs(u)
        cycles[name] = cyc
    stats = dict(n_concepts=len(concept_meta), hubs=[dict(target=t, train_videos=n, frac=round(f, 3)) for t, n, f in hub_rows[:25]],
                 n_hub=len(hub), n_shallow=len(shallow), cycles=cycles,
                 n_ambiguous=int(sum(r["sense_ambiguous"] for r in rows)),
                 n_sense_differs=int(sum(r["sense_differs"] for r in rows)),
                 isa_edges={n: int(sum(len(v) for v in t.values())) for n, t in var.items()})
    return var, concept_meta, pd.DataFrame(rows), stats

### 4.8c Đồ thị v3 và hai đối chứng ngẫu nhiên có ràng buộc
**`GraphV3`** giữ topology cặp của v2, nên relation-family-only (G2) trùng với `v2_base`. Mỗi cặp (nguồn, họ, đích) mang **phân phối vị từ** `{vị từ: số video train}`.

**Chính sách cạnh:**
* `all`: như v2;
* `support`: chỉ vị từ có ≥ 5 video train;
* `agreement`: chỉ vị từ của quan hệ được parser độc lập xác nhận ở ≥ 50% caption train của nó (kiểm chứng tự động);
* `precision`: chỉ ô đã kiểm chứng bằng nhãn người (tuỳ chọn).

Một cặp không còn vị từ hợp lệ thì bị loại và lưu lại.

**Đối chứng:**
* **Rewiring** (G5): directed double-edge swap trong từng họ và nhóm kiểu node. Phép này giữ bậc vào/ra, trọng số và **đa tập phân phối vị từ**, vì vị từ đi theo cạnh. `contains` không đổi.
* **Hoán vị vị từ** (G4): hoán vị phân phối vị từ giữa các cạnh **cùng họ** svo/prep. Topology, trọng số và bậc giống hệt; chỉ việc ghép vị từ ↔ cạnh bị phá.

`path_mode` gồm `weighted`, `count`, `degnorm`, `dedup` như v2, và `pred_match` (= `weighted` cộng mẫu P1m khớp vị từ). Engine, `compute_R` và phần giải thích dùng **chung** các ma trận này.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
class NodeInventory:
    """Tập node CỐ ĐỊNH cho mọi cấu hình so sánh được: video | lớp (lemma, POS) | khái niệm (synset).
    Ablation chỉ tắt cạnh ⇒ node id, số node, kích thước embedding không đổi."""

    def __init__(self, records, vocab, concept_meta):
        self.n_vid = len(records)
        self.class_keys = list(vocab["terms"])
        self.concept_keys = sorted(concept_meta)
        self.n_cls, self.n_con = len(self.class_keys), len(self.concept_keys)
        self.n_nodes = self.n_vid + self.n_cls + self.n_con
        self.key2id = {k: self.n_vid + i for i, k in enumerate(self.class_keys)}
        self.key2id.update({k: self.n_vid + self.n_cls + i for i, k in enumerate(self.concept_keys)})
        self.id2key = {i: k for k, i in self.key2id.items()}
        self.node_type = np.array(["video"] * self.n_vid + ["class"] * self.n_cls + ["concept"] * self.n_con)
        self.pos = np.array([""] * self.n_vid + [k.rsplit("#", 1)[1] for k in self.class_keys]
                            + [concept_meta[k]["pos"] for k in self.concept_keys])
        self.lemma = {**{self.key2id[k]: vocab["lemma"][k] for k in self.class_keys},
                      **{self.key2id[k]: concept_meta[k]["lemma"] for k in self.concept_keys}}
        self.name = {i: (records[i]["video_id"] if i < self.n_vid else self.id2key[i]) for i in range(self.n_nodes)}
        self.concepts_by_lemma = defaultdict(list)
        for k in self.concept_keys:
            self.concepts_by_lemma[(concept_meta[k]["lemma"], concept_meta[k]["pos"])].append(self.key2id[k])
        self.concept_meta = concept_meta
        self.sha1 = sha1_of([self.n_vid, self.class_keys, self.concept_keys])

    def semantic_type(self, i):
        t = self.node_type[i]
        return "video" if t == "video" else ({"n": "noun", "v": "verb"}[self.pos[i]] + ("_concept" if t == "concept" else ""))

    def is_class(self, i):
        return self.n_vid <= i < self.n_vid + self.n_cls


def contains_calibration(tag_scores, train_vids, inv):
    """Thống kê chuẩn hoá điểm tagger của các nhãn ĐÃ GẮN trên video TRAIN, tách danh từ / động từ."""
    by = defaultdict(list)
    for v in train_vids:
        for k, s in tag_scores[v].items():
            by[k.rsplit("#", 1)[1]].append(s)
    return {p: dict(mu=float(np.mean(x)), sd=float(np.std(x) + 1e-6), lo=float(np.percentile(x, 1)),
                    hi=float(np.percentile(x, 99))) for p, x in by.items()}


def _conf_weight(s, pos, cal, mode):
    c = cal[pos]
    if mode == "sigmoid":                      # z bị chặn ±3 để hạn chế ngoại lai
        w = 1.0 / (1.0 + math.exp(-float(np.clip((s - c["mu"]) / c["sd"], -3, 3))))
    else:                                      # minmax giữa phân vị 1–99 của train
        w = (s - c["lo"]) / max(c["hi"] - c["lo"], 1e-6)
    return float(np.clip(w, CFG.conf_floor, 1.0))


def _csr(n_rows, n_cols, rows, cols, vals):
    if len(rows) == 0:
        return sp.csr_matrix((n_rows, n_cols), dtype=np.float32)
    M = sp.csr_matrix((np.asarray(vals, np.float32), (np.asarray(rows), np.asarray(cols))), shape=(n_rows, n_cols))
    M.sum_duplicates()
    return M


def _symnorm(M):
    if M.nnz == 0:
        return M.copy()
    dout = np.asarray(M.sum(1)).ravel(); din = np.asarray(M.sum(0)).ravel()
    a = np.where(dout > 0, 1 / np.sqrt(np.maximum(dout, 1e-12)), 0)
    b = np.where(din > 0, 1 / np.sqrt(np.maximum(din, 1e-12)), 0)
    return (sp.diags(a) @ M @ sp.diags(b)).tocsr().astype(np.float32)


def _maxprod(C, B):
    """out[v, c] = max_o C[v, o] · B[o, c] (thưa) — tổ hợp 'max' thay cho tổng (path_dedup)."""
    C, B = C.tocsr(), B.tocsr()
    rows = []
    for v in range(C.shape[0]):
        sl = slice(C.indptr[v], C.indptr[v + 1])
        idx, val = C.indices[sl], C.data[sl]
        if len(idx) == 0:
            rows.append(sp.csr_matrix((1, B.shape[1]), dtype=np.float32))
            continue
        sub = (sp.diags(val) @ B[idx]).tocsr()
        rows.append(sp.csr_matrix(sub.max(axis=0)))
    return sp.vstack(rows).tocsr()




PATH_TYPES = ["agent_of", "agent_of_inv", "object_of", "object_of_inv", "svo", "svo_inv", "prep", "prep_inv", "related", "isa"]
REL_POLICIES = ("all", "support", "agreement", "precision")


def predicate_tokens(p):
    return tuple(p.split())


class GraphV3:
    """Đồ thị dị thể có kiểu, có hướng, có trọng số trên NodeInventory cố định; cạnh quan hệ mang VỊ TỪ.
    Cạnh quan hệ = cặp (nguồn, họ, đích) như v2 (cùng topology) + phân phối vị từ của cặp {vị từ: số video TRAIN}.
      rel_policy : 'all' | 'support' (vị từ có ≥ pred_min_support video TRAIN) | 'agreement' (≥ agree_min caption TRAIN được
                   parser độc lập xác nhận — kiểm chứng TỰ ĐỘNG) | 'precision' (ô đã kiểm chứng bằng nhãn người, tuỳ chọn)
                   — cặp không còn vị từ hợp lệ bị LOẠI và lưu trong self.excluded
      path_mode  : 'weighted' | 'count' | 'degnorm' | 'dedup' | 'pred_match' (= weighted + mẫu P1m khớp vị từ truy vấn)
      rewire_seed: constrained rewiring (vị từ, trọng số đi theo cạnh) ; pred_shuffle_seed: hoán vị phân phối vị từ giữa
                   các cạnh CÙNG họ (topology, trọng số giữ nguyên) ; related_cfg: (min_count, min_npmi)."""

    def __init__(self, inv, records, train_vids, relations, taxonomy, *, tag_scores=None, contains="binary",
                 conf_mode="sigmoid", conf_cal=None, rel_policy="all", path_mode="weighted", masked=(),
                 rewire_seed=None, pred_shuffle_seed=None, tags_override=None, related_cfg=None, verified_cells=None, name=""):
        assert rel_policy in REL_POLICIES, rel_policy
        if rel_policy == "precision" and verified_cells is None:
            raise ValueError("rel_policy = precision cần nhãn người chấm (verified_cells) — NOT RUN khi chưa có nhãn")
        if rel_policy == "agreement" and not AGREE:
            raise ValueError("rel_policy = agreement cần kiểm chứng tự động (spaCy) — NOT RUN khi không nạp được parser")
        self.inv, self.name, self.path_mode, self.masked = inv, name, path_mode, tuple(sorted(set(masked)))
        self.contains_mode, self.conf_mode, self.rel_policy = contains, conf_mode, rel_policy
        self.rewire_seed, self.pred_shuffle_seed = rewire_seed, pred_shuffle_seed
        self.related_cfg = tuple(related_cfg or (CFG.cooc_min_count, CFG.cooc_min_npmi))
        n, N = inv.n_nodes, inv.n_vid
        self.n_nodes, self.n_vid = n, N
        train_set = set(int(v) for v in train_vids)
        tr_rows = np.array(sorted(train_set))
        n_tr = max(len(tr_rows), 1)
        tags = tags_override if tags_override is not None else {r["vid"]: r["tags"] for r in records}
        freq = Counter(t for v in train_set for t in tags[v])
        self.active = {inv.key2id[k] for k, c in freq.items() if c >= CFG.min_tag_freq and k in inv.key2id}

        # ---------- contains (video → lớp), từ nhãn tagger của chính video ----------
        rows, cols, vals, self.vid_tags = [], [], [], []
        for v in range(N):
            ids = []
            for k in tags[v]:
                i = inv.key2id.get(k)
                if i is None or i not in self.active:
                    continue
                w = 1.0 if contains == "binary" else _conf_weight(tag_scores[v][k], k.rsplit("#", 1)[1], conf_cal, conf_mode)
                rows.append(v); cols.append(i); vals.append(w); ids.append(i)
            self.vid_tags.append(sorted(ids))
        self.A_con = _csr(N, n, rows, cols, vals)
        Cb = (self.A_con > 0).astype(np.float32)
        cls_df = np.asarray(Cb[tr_rows].sum(0)).ravel()
        self.cls_df = cls_df

        # ---------- quan hệ từ caption TRAIN, TÁCH theo họ, giữ phân phối vị từ ----------
        sup, preds = defaultdict(Counter), defaultdict(Counter)
        for v in train_set:
            seen = set()
            for s, rel, o, pred in relations.get(v, []):
                a, b = inv.key2id.get(s), inv.key2id.get(o)
                if a is None or b is None or a not in self.active or b not in self.active or a == b:
                    continue
                preds[(rel, a, b)][pred] += 1             # = số video TRAIN có đúng (nguồn, họ, đích, vị từ)
                if (rel, a, b) not in seen:
                    sup[rel][(a, b)] += 1; seen.add((rel, a, b))
        self.preds = preds

        def kept_preds(rel, a, b):
            pc = preds[(rel, a, b)]
            if rel_policy == "support":
                return {p: c for p, c in pc.items() if PRED_SUPPORT.get((rel, p), 0) >= CFG.pred_min_support}
            if rel_policy == "precision":
                return {p: c for p, c in pc.items() if (rel, support_stratum(c)) in verified_cells}
            if rel_policy == "agreement":
                ka, kb = inv.id2key[a], inv.id2key[b]
                return {p: c for p, c in pc.items() if agreement_rate(rel, ka, kb, p) >= CFG.agree_min}
            return dict(pc)

        fwd, self.excluded = {}, []
        for rel in REL_CATS:
            by_src = defaultdict(list)
            for (a, b), c in sup[rel].items():
                if c < CFG.scene_min_count:
                    continue
                kp = kept_preds(rel, a, b)
                if not kp:
                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=dict(preds[(rel, a, b)]),
                                              reason=f"policy:{rel_policy}"))
                    continue
                by_src[a].append((c, b, kp))
            E = [(a, b, c, kp) for a, lst in by_src.items() for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[:CFG.scene_top_k]]
            for a, lst in by_src.items():
                for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[CFG.scene_top_k:]:
                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=kp, reason="top_k"))
            f = dict(src=np.array([e[0] for e in E], int), dst=np.array([e[1] for e in E], int),
                     support=np.array([e[2] for e in E], float))
            f["w"] = np.minimum(1.0, f["support"] / np.maximum(cls_df[f["src"]], 1)) if E else np.zeros(0)
            f["w_inv"] = np.minimum(1.0, f["support"] / np.maximum(cls_df[f["dst"]], 1)) if E else np.zeros(0)
            f["pdist"] = [e[3] for e in E]
            fwd[rel] = f
        self.n_candidate_relations = {r: len(sup[r]) for r in REL_CATS}

        # ---------- related: NPMI đồng xuất hiện nhãn tagger trên video TRAIN (đối xứng) ----------
        min_count, min_npmi = self.related_cfg
        cooc = Counter()
        for v in train_set:
            for a, b in itertools.combinations(sorted(self.vid_tags[v]), 2):
                cooc[(a, b)] += 1
        und = {}
        for (a, b), c in cooc.items():
            if c < min_count:
                continue
            p_ab = c / n_tr
            npmi = math.log(p_ab / ((cls_df[a] / n_tr) * (cls_df[b] / n_tr))) / max(-math.log(p_ab), 1e-6)
            if npmi >= min_npmi:
                und[(a, b)] = (c, npmi)
        keep, by_src = set(), defaultdict(list)
        for (a, b), (c, s) in und.items():
            by_src[a].append((s, b)); by_src[b].append((s, a))
        for a, lst in by_src.items():
            for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]:
                keep.add((min(a, b), max(a, b)))
        RE = sorted(keep)
        related = dict(src=np.array([e[0] for e in RE], int), dst=np.array([e[1] for e in RE], int),
                       count=np.array([und[e][0] for e in RE], float))
        related["w_ab"] = np.minimum(1.0, related["count"] / np.maximum(cls_df[related["src"]], 1)) if RE else np.zeros(0)
        related["w_ba"] = np.minimum(1.0, related["count"] / np.maximum(cls_df[related["dst"]], 1)) if RE else np.zeros(0)

        # ---------- isa (biến thể taxonomy) ----------
        IE = sorted({(inv.key2id[c], inv.key2id[t]) for c, ts in taxonomy.items() if c in inv.key2id
                     and inv.key2id[c] in self.active for t in ts if t in inv.key2id
                     and (inv.node_type[inv.key2id[t]] == "concept" or inv.key2id[t] in self.active)})
        isa = dict(src=np.array([e[0] for e in IE], int), dst=np.array([e[1] for e in IE], int))
        isa_true = _csr(n, n, isa["src"], isa["dst"], np.ones(len(IE)))

        # ---------- đối chứng 1: constrained rewiring (contains giữ nguyên; vị từ + trọng số đi theo cạnh) ----------
        self.rewire_audit = []
        if rewire_seed is not None:
            rng = np.random.RandomState(rewire_seed)
            for rel in REL_CATS:
                fwd[rel], st = rewire_directed(fwd[rel], inv, rng, ["w", "w_inv", "support", "pdist"], rel)
                self.rewire_audit.append(st)
            related, st = rewire_undirected(related, inv, rng, "related")
            self.rewire_audit.append(st)
            isa, st = rewire_directed(isa, inv, rng, [], "isa")
            self.rewire_audit.append(st)

        # ---------- đối chứng 2: hoán vị phân phối vị từ giữa các cạnh CÙNG họ (topology, trọng số giữ nguyên) ----------
        self.shuffle_audit = []
        if pred_shuffle_seed is not None:
            rng = np.random.RandomState(pred_shuffle_seed)
            for rel in PRED_FAMILIES:
                old = fwd[rel]["pdist"]
                perms = [rng.permutation(len(old)) for _ in range(10)]     # chọn hoán vị đổi được nhiều cạnh nhất
                perm = max(perms, key=lambda pm: sum(old[i] != old[j] for i, j in enumerate(pm)))
                new = [old[i] for i in perm]
                fwd[rel] = dict(fwd[rel], pdist=new)
                ms = lambda L: sorted(json.dumps(sorted(d.items())) for d in L)
                self.shuffle_audit.append(dict(family=rel, n_edges=len(old),
                                               frac_edges_predicates_changed=float(np.mean([a != b for a, b in zip(old, new)])) if old else 0.0,
                                               predicate_multiset_equal=ms(old) == ms(new)))
        self._fwd, self._related, self._isa = fwd, related, isa

        # ---------- ma trận theo từng loại cạnh + bảng vị từ theo cặp ----------
        A = {}
        for rel in REL_CATS:
            f = fwd[rel]
            A[rel] = _csr(n, n, f["src"], f["dst"], f["w"])
            A[rel + "_inv"] = _csr(n, n, f["dst"], f["src"], f["w_inv"])
        rs, rd = related["src"], related["dst"]
        A["related"] = _csr(n, n, np.r_[rs, rd], np.r_[rd, rs], np.r_[related["w_ab"], related["w_ba"]])
        A["isa"] = _csr(n, n, isa["src"], isa["dst"], np.ones(len(isa["src"])))
        A["isa_inv"] = A["isa"].T.tocsr()
        for t in self.masked:
            if t != "contains" and t != "contains_inv":
                A[t] = sp.csr_matrix((n, n), dtype=np.float32)
        self.A = A
        if "contains" in self.masked:
            raise ValueError("không tắt contains trong ablation cạnh quan hệ")
        # pdist[(họ, a, b)] = {vị từ: số video}; pshare[họ][(a, b)] = {pred_id: tỉ phần} cho họ có vị từ tự do
        self.pdist = {(rel, int(a), int(b)): d for rel in REL_CATS for a, b, d in zip(fwd[rel]["src"], fwd[rel]["dst"], fwd[rel]["pdist"])}
        self.pshare = {}
        for rel in PRED_FAMILIES:
            sh = {}
            for (r_, a, b), d in self.pdist.items():
                if r_ == rel and A[rel][a, b] != 0:
                    tot = float(sum(d.values()))
                    sh[(a, b)] = {PRED_ID[(rel, p)]: c / tot for p, c in sorted(d.items()) if (rel, p) in PRED_ID}
            self.pshare[rel] = sh

        # ---------- IDF (taxonomy, chỉ TRAIN) — KHÔNG phụ thuộc mask/rewiring ----------
        df_i = np.asarray(((Cb @ isa_true)[tr_rows] > 0).sum(0)).ravel()
        df = np.where(np.arange(n) < N + inv.n_cls, cls_df, df_i)
        idf = np.log((n_tr + 1.0) / (df + 1.0)).astype(np.float32)
        idf[:N] = 0.0
        self.idf = idf
        self.PM = self.path_mats()
        self.R = self.compute_R()

    # ---------- đường đi ----------
    def pred_expand(self, A, rel):
        """X[u, pid·n + v] = A_t[u, v] · tỉ phần vị từ pid của cặp, cho t ∈ {rel, rel_inv} (đường đi khớp vị từ)."""
        n, rows, cols, vals = self.n_nodes, [], [], []
        sh = self.pshare.get(rel, {})
        for t, inv_dir in ((rel, False), (rel + "_inv", True)):
            m = A[t].tocoo()
            for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):
                for pid, s in sh.get((v, u) if inv_dir else (u, v), {}).items():
                    rows.append(u); cols.append(pid * n + v); vals.append(w * s)
        return _csr(n, len(PRED_VOCAB) * n, rows, cols, vals)

    def path_mats(self, A=None, A_con=None):
        A = A or self.A
        C = (A_con if A_con is not None else self.A_con).tocsr().astype(np.float32)
        if self.path_mode == "count":
            C = (C > 0).astype(np.float32)
        PM = {"C": C}
        for t in PATH_TYPES:
            M = A[t].tocsr().astype(np.float32)
            if self.path_mode == "count":
                M = (M > 0).astype(np.float32)
            elif self.path_mode == "degnorm":
                M = _symnorm(M)
            PM[t] = M
        if self.path_mode == "pred_match":
            for rel in PRED_FAMILIES:
                PM[f"X_{rel}"] = self.pred_expand(A, rel)
        return PM

    def _fam(self, PM, f):
        a, b = PM[FAMILY_REL[f]], PM[FAMILY_REL[f] + "_inv"]
        return a.maximum(b) if self.path_mode == "dedup" else (a + b)

    def compute_R(self, rows=None, PM=None):
        PM = PM or self.PM
        C = PM["C"] if rows is None else PM["C"][np.asarray(rows)]
        I, Rl = PM["isa"], PM["related"]
        prod = _maxprod if self.path_mode == "dedup" else (lambda X, Y: (X @ Y).tocsr())
        R = {"P0_contains": C.tocsr()}
        for f in FAMILIES:
            CF = prod(C, self._fam(PM, f))
            R[f"P1_{f}"] = CF
            R[f"P4_{f}_isa"] = prod(CF, I)
        R["P2_isa"] = prod(C, I)
        CR = prod(C, Rl)
        R["P3_related"] = CR
        R["P5_related_isa"] = prod(CR, I)
        direct = (C > 0).astype(np.float32)
        if CFG.drop_redundant_indirect:
            for p in PATTERNS[1:]:
                if p in R:
                    R[p] = (R[p] - R[p].multiply(direct)).tocsr()
        D = sp.diags(self.idf)
        out = {}
        for p in PATTERNS:
            if p.startswith("P1m_"):
                continue
            M = (R[p] @ D).tocsr(); M.eliminate_zeros(); out[p] = M
        n, n_p = self.n_nodes, len(PRED_VOCAB)
        for rel in PRED_FAMILIES:
            key = f"X_{rel}"
            if key not in PM:
                out[f"P1m_{rel}"] = sp.csr_matrix((C.shape[0], n_p * n), dtype=np.float32)
                continue
            M = (C @ PM[key]).tocoo()
            b = M.col % n
            keep = np.ones(M.nnz, bool)
            if CFG.drop_redundant_indirect and M.nnz:
                keep = np.asarray(direct[M.row, b]).ravel() == 0
            X = sp.csr_matrix(((M.data * self.idf[b])[keep], (M.row[keep], M.col[keep])), shape=M.shape, dtype=np.float32)
            X.eliminate_zeros()
            out[f"P1m_{rel}"] = X
        return out

    # ---------- cạnh cho GNN (có loại, có trọng số, có chỉ số vị từ) ----------
    def edge_arrays(self):
        """(src, dst, type, weight, pidx): pidx = hàng trong self.pmix (phân phối vị từ của cạnh) hoặc −1."""
        if not hasattr(self, "_edges"):
            parts, mix_rows = [], []
            c = self.A_con.tocoo()
            parts.append((c.row, c.col, np.full(c.nnz, ETYPE_ID["contains"]), c.data, np.full(c.nnz, -1)))
            parts.append((c.col, c.row, np.full(c.nnz, ETYPE_ID["contains_inv"]), c.data, np.full(c.nnz, -1)))
            for t in RELATION_TYPES:
                m = self.A[t].tocoo()
                if not m.nnz:
                    continue
                pidx = np.full(m.nnz, -1)
                if t in PRED_TYPES:
                    base, inv_dir = (t[:-4], True) if t.endswith("_inv") else (t, False)
                    for k, (u, v) in enumerate(zip(m.row.tolist(), m.col.tolist())):
                        d = self.pshare[base].get((v, u) if inv_dir else (u, v), {})
                        if d:
                            pidx[k] = len(mix_rows); mix_rows.append(d)
                parts.append((m.row, m.col, np.full(m.nnz, ETYPE_ID[t]), m.data, pidx))
            s, d_, e, w, pi = (np.concatenate([p[i] for p in parts]) for i in range(5))
            r_, c_, v_ = [], [], []
            for i, dd in enumerate(mix_rows):
                for pid, sh in dd.items():
                    r_.append(i); c_.append(pid); v_.append(sh)
            self.pmix = _csr(len(mix_rows), len(PRED_VOCAB), r_, c_, v_)
            self._edges = (s.astype(np.int64), d_.astype(np.int64), e.astype(np.int64), w.astype(np.float32), pi.astype(np.int64))
        return self._edges

    def edge_counts(self):
        out = {"contains": int(self.A_con.nnz), "contains_inv": int(self.A_con.nnz)}
        out.update({t: int(self.A[t].nnz) for t in RELATION_TYPES})
        return out

    def predicates_of(self, rel, a, b):
        """{vị từ: số video} của cạnh (rel, a → b) THEO ĐÚNG CHIỀU lưu trong đồ thị (đã tính rewiring / hoán vị vị từ)."""
        base = rel[:-4] if rel.endswith("_inv") else rel
        return self.pdist.get((base, b, a) if rel.endswith("_inv") else (base, a, b), {})

    def predicate(self, rel, a, b):
        d = self.predicates_of(rel, a, b)
        return max(sorted(d), key=lambda p: d[p]) if d else ""

    def summary(self):
        return dict(name=self.name, nodes=self.n_nodes, videos=self.n_vid, classes=self.inv.n_cls, concepts=self.inv.n_con,
                    active_classes=len(self.active), edges=self.edge_counts(), contains=self.contains_mode,
                    path_mode=self.path_mode, rel_policy=self.rel_policy, related_cfg=list(self.related_cfg),
                    masked=list(self.masked), rewired=self.rewire_seed is not None,
                    predicate_shuffled=self.pred_shuffle_seed is not None, excluded=len(self.excluded),
                    paths={p: int(self.R[p].nnz) for p in PATTERNS})

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def rewire_directed(E, inv, rng, attrs, rel):
    """Directed double-edge swap TRONG một loại quan hệ: (a→b),(c→d) ⇒ (a→d),(c→b) khi a,c cùng kiểu và b,d cùng
    kiểu (semantic type: noun / verb / noun_concept / verb_concept). Giữ out-degree, in-degree, kiểu đầu/cuối,
    số cạnh; cấm self-loop và cạnh trùng. Thuộc tính (trọng số, support, phân phối vị từ) gắn với cạnh theo NGUỒN."""
    src, dst = E["src"].copy(), E["dst"].copy()
    m = len(src)
    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)
    if m < 2:
        st.update(frac_changed=0.0, success_rate=0.0)
        return dict(E, src=src, dst=dst), st
    key = np.array([f"{inv.semantic_type(a)}>{inv.semantic_type(b)}" for a, b in zip(src, dst)])
    edges = set(zip(src.tolist(), dst.tolist()))
    for g in np.unique(key):
        I = np.where(key == g)[0]
        if len(I) < 2:
            continue
        for _ in range(CFG.rewire_swaps_per_edge * len(I)):
            i, j = rng.choice(I, 2, replace=False)
            a, b, c, d = src[i], dst[i], src[j], dst[j]
            st["attempts"] += 1
            if a == d or c == b:
                st["fail_self"] += 1; continue
            if (a, d) in edges or (c, b) in edges:
                st["fail_dup"] += 1; continue
            edges -= {(a, b), (c, d)}; edges |= {(a, d), (c, b)}
            dst[i], dst[j] = d, b
            st["swaps"] += 1
    orig = set(zip(E["src"].tolist(), E["dst"].tolist()))
    st["frac_changed"] = 1 - len(orig & set(zip(src.tolist(), dst.tolist()))) / m
    st["success_rate"] = st["swaps"] / max(st["attempts"], 1)
    return dict(E, src=src, dst=dst), st


def rewire_undirected(E, inv, rng, rel):
    """Double-edge swap cho quan hệ ĐỐI XỨNG (related): giữ bậc vô hướng, cặp kiểu, không self-loop/trùng.
    Cạnh lưu (a, b) có kiểu(a) ≤ kiểu(b); số đếm đồng xuất hiện gắn với cạnh."""
    src, dst = E["src"].copy(), E["dst"].copy()
    wab, wba = E["w_ab"].copy(), E["w_ba"].copy()            # P(b|a), P(a|b) gắn với cạnh, đổi chỗ khi lật hướng
    m = len(src)
    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)
    if m < 2:
        st.update(frac_changed=0.0, success_rate=0.0)
        return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st
    T = lambda x: inv.semantic_type(x)

    def flip(k):
        src[k], dst[k] = dst[k], src[k]
        wab[k], wba[k] = wba[k], wab[k]

    for k in range(m):                                  # định hướng theo kiểu để swap giữ cặp kiểu
        if T(src[k]) > T(dst[k]):
            flip(k)
    key = np.array([f"{T(a)}~{T(b)}" for a, b in zip(src, dst)])
    edges = {(min(a, b), max(a, b)) for a, b in zip(src.tolist(), dst.tolist())}
    for g in np.unique(key):
        I = np.where(key == g)[0]
        if len(I) < 2:
            continue
        same_type = g.split("~")[0] == g.split("~")[1]
        for _ in range(CFG.rewire_swaps_per_edge * len(I)):
            i, j = rng.choice(I, 2, replace=False)
            if same_type and rng.rand() < 0.5:
                flip(j)                                 # cùng kiểu ⇒ cho phép cả hai cách ghép
            a, b, c, d = src[i], dst[i], src[j], dst[j]
            st["attempts"] += 1
            n1, n2 = (min(a, d), max(a, d)), (min(c, b), max(c, b))
            if a == d or c == b or n1 == n2:
                st["fail_self"] += 1; continue
            if n1 in edges or n2 in edges:
                st["fail_dup"] += 1; continue
            edges -= {(min(a, b), max(a, b)), (min(c, d), max(c, d))}; edges |= {n1, n2}
            dst[i], dst[j] = d, b                       # (a, d) và (c, b); trọng số theo nguồn a, c
            st["swaps"] += 1
    orig = {(min(a, b), max(a, b)) for a, b in zip(E["src"].tolist(), E["dst"].tolist())}
    st["frac_changed"] = 1 - len(orig & edges) / m
    st["success_rate"] = st["swaps"] / max(st["attempts"], 1)
    return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st


def _pred_multiset(g, rel):
    """Đa tập phân phối vị từ của các cạnh thuộc họ rel (chỉ cạnh còn trong đồ thị)."""
    if rel not in REL_CATS:
        return []
    return sorted(json.dumps(sorted(d.items())) for (r_, a, b), d in g.pdist.items() if r_ == rel and g.A[rel][a, b] != 0)


def randomization_audit(g0, g1):
    """So sánh đồ thị gốc và đồ thị rewired theo từng loại cạnh; áp tiêu chí chấp nhận đăng ký trước."""
    inv, rows = g0.inv, []
    stats = {s["relation"]: s for s in g1.rewire_audit}
    for t in ["contains"] + RELATION_TYPES:
        M0 = g0.A_con if t == "contains" else g0.A[t]
        M1 = g1.A_con if t == "contains" else g1.A[t]
        c0, c1 = M0.tocoo(), M1.tocoo()
        e0, e1 = set(zip(c0.row.tolist(), c0.col.tolist())), set(zip(c1.row.tolist(), c1.col.tolist()))
        tp = lambda c: Counter(f"{inv.semantic_type(a) if t != 'contains' else 'video'}>{inv.semantic_type(b)}"
                               for a, b in zip(c.row, c.col))
        base = t[:-4] if t.endswith("_inv") else t
        s = stats.get(base, {})
        row = dict(relation=t, edges_orig=len(e0), edges_rand=len(e1),
                   out_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(1)).ravel(), np.asarray((M1 > 0).sum(1)).ravel())),
                   in_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(0)).ravel(), np.asarray((M1 > 0).sum(0)).ravel())),
                   type_pairs_equal=tp(c0) == tp(c1), weights_multiset_equal=bool(np.allclose(np.sort(c0.data), np.sort(c1.data))),
                   self_loops=int(sum(a == b for a, b in e1)), duplicates=int(c1.nnz - len(e1)),
                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),
                   frac_changed=1 - len(e0 & e1) / max(len(e0), 1), swaps=s.get("swaps"), attempts=s.get("attempts"),
                   success_rate=s.get("success_rate"), fail_dup=s.get("fail_dup"), fail_self=s.get("fail_self"))
        n_s, n_t = len({a for a, _ in e0}), len({b for _, b in e0})
        row["density"] = len(e0) / max(n_s * n_t, 1)      # ≈ 1 ⇒ gần như KHÔNG có cách sắp xếp khác giữ nguyên bậc
        must_change = t != "contains" and len(e0) >= 10
        row["accepted"] = bool(row["edges_orig"] == row["edges_rand"] and row["out_degree_equal"] and row["in_degree_equal"]
                               and row["type_pairs_equal"] and row["weights_multiset_equal"] and row["self_loops"] == 0
                               and row["duplicates"] == 0 and row["predicates_multiset_equal"]
                               and (row["frac_changed"] >= CFG.rewire_min_changed or not must_change)
                               and (t != "contains" or row["frac_changed"] == 0))
        row["note"] = ("contains phải giữ nguyên" if t == "contains" else "< 10 cạnh: không yêu cầu đổi" if not must_change
                       else ("đổi < ngưỡng: đồ thị quá đặc (mật độ cao) — không có đủ cách sắp xếp giữ bậc"
                             if not row["accepted"] and row["density"] > 0.5 else ""))
        rows.append(row)
    df = pd.DataFrame(rows)
    return df, bool(df.accepted.all())


def predicate_shuffle_audit(g0, g1):
    """Đối chứng hoán vị vị từ: topology, trọng số, bậc của MỌI loại cạnh phải giống hệt; chỉ phân phối vị từ của cạnh
    thuộc họ có vị từ tự do được đổi chỗ (đa tập giữ nguyên)."""
    rows = []
    stats = {s["family"]: s for s in g1.shuffle_audit}
    for t in ["contains"] + RELATION_TYPES:
        M0 = g0.A_con if t == "contains" else g0.A[t]
        M1 = g1.A_con if t == "contains" else g1.A[t]
        base = t[:-4] if t.endswith("_inv") else t
        same_topology = (M0 != M1).nnz == 0
        moved = 0
        if base in PRED_FAMILIES:
            c = M0.tocoo()
            for u, v in zip(c.row.tolist(), c.col.tolist()):
                moved += int(g0.predicates_of(t, u, v) != g1.predicates_of(t, u, v))
        st = stats.get(base, {})
        row = dict(relation=t, n_edges=int(M0.nnz), topology_and_weights_equal=bool(same_topology),
                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),
                   edges_with_changed_predicates=moved, frac_edges_predicates_changed=moved / max(int(M0.nnz), 1),
                   shuffled=base in PRED_FAMILIES, family_frac_changed=st.get("frac_edges_predicates_changed"))
        row["accepted"] = bool(same_topology and row["predicates_multiset_equal"]
                               and (moved == 0 if base not in PRED_FAMILIES else (M0.nnz < 10 or row["frac_edges_predicates_changed"] >= CFG.rewire_min_changed)))
        rows.append(row)
    df = pd.DataFrame(rows)
    return df, bool(df.accepted.all())

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def query_tokens_v2(text):
    """[(POS dự đoán, lemma danh từ, lemma động từ)] — POS tagger trên CHÍNH câu truy vấn (input của hệ thống)."""
    toks = re.findall(r"[a-z]+", str(text).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    out = []
    for w, t in tags:
        if w in STOP or len(w) < 2:
            continue
        p = "v" if t.startswith("VB") else ("n" if t.startswith("NN") else "x")
        out.append((p, lemma(w, "n"), lemma(w, "v")))
    return out


def parse_tokens_v2(toks, inv):
    """Khớp token với node: lớp đúng POS trước (danh từ → 'x#n', động từ → 'x#v'), không có thì POS còn lại,
    không có lớp thì khái niệm taxonomy cùng lemma + POS. Trả về [(node_id, tên node, token)]."""
    hits, seen = [], set()
    for p, ln, lv in toks:
        order = [("v", lv), ("n", ln)] if p == "v" else [("n", ln), ("v", lv)]
        found = None
        for pos, lm in order:
            i = inv.key2id.get(ckey(lm, pos))
            if i is not None:
                found = [i]; break
        if found is None:
            for pos, lm in order:
                if inv.concepts_by_lemma.get((lm, pos)):
                    found = inv.concepts_by_lemma[(lm, pos)]; break
        for i in found or []:
            if i not in seen:
                seen.add(i); hits.append((i, inv.name[i], lv if p == "v" else ln))
    return hits


def query_pred_tokens(text):
    """Tập token của truy vấn để khớp VỊ TỪ: từ gốc + lemma động từ (giới từ giữ nguyên; KHÔNG bỏ stop-word)."""
    w = re.findall(r"[a-z]+", str(text).lower())
    return frozenset(w) | frozenset(lemma(x, "v") for x in w)


_PRED_BY_FIRST = {}


def match_predicates(qset, rel):
    """pred_id của họ rel có MỌI token nằm trong truy vấn ('ride on' khớp 'a man is riding on a horse')."""
    if rel not in _PRED_BY_FIRST:
        idx = defaultdict(list)
        for i, (r, p) in enumerate(PRED_VOCAB):
            if r == rel:
                t = predicate_tokens(p)
                idx[t[0]].append((i, t))
        _PRED_BY_FIRST[rel] = idx
    idx = _PRED_BY_FIRST[rel]
    return sorted({i for tok in qset for i, t in idx.get(tok, ()) if all(x in qset for x in t)})

## 4.9 Mô hình v3

**`TypedSAGEv3`.** Công thức là `h'_v = W₀h_v + Σ_t σ(g_t)·[W_t·AGG_t(h_u) + U_t·AGG_t(e_uv)]`.
* `e_uv` là biểu diễn vị từ của cạnh. Nó chỉ có ở các loại cạnh svo, svo_inv, prep, prep_inv, và chỉ khi `pred_mode ≠ none`.
* `AGG_t` dùng **cùng hệ số** cho `h_u` và `e_uv`, nên trọng số cạnh chỉ nhân một lần.
* Vị từ của agent_of/object_of chính là node động từ đích, nên không đưa vào message, tránh trùng thông tin.

**`pred_mode`:**
* `none` = relation-family-only, đúng mô hình v2;
* `clip` (phương án A) = CLIP-text **đóng băng** của vị từ theo prompt, qua một phép chiếu tuyến tính;
* `clip_adapt` (phương án B) = A cộng MLP dư **dùng chung** cho mọi vị từ, lớp cuối khởi tạo 0. Không có ID riêng cho từng vị từ, nên vị từ hiếm hoặc chưa gặp vẫn mã hoá được.

`mp_weight` (`binary` / `weighted_mean` / `weighted_symnorm`) và `init` như v2.
### 4.9b Mô hình v4 — QC-RGAT (GNN quan hệ có attention điều kiện theo truy vấn)

**Vì sao đổi mô hình.** Ở v3, truy vấn chỉ đi vào GNN dưới dạng **một vector CLIP toàn cục**. Cạnh quan hệ nối lớp với lớp và là thống kê **dùng chung** cho mọi video, nên mọi video có cùng nhãn đều nhận cùng "ngữ cảnh quan hệ". Vì vậy quan hệ chỉ làm mượt embedding lớp, và kết quả Kaggle v3 cho G3 ≈ G2 ≈ G4 ≈ G5. Ở v4, **chính truy vấn quyết định cạnh quan hệ nào được truyền tin**.

**Công thức** (T = 1 hoặc 2 lớp; chọn trên VAL):
* `H_q⁽⁰⁾(c) = E_c` và `H_q⁽ᵗ⁺¹⁾(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q⁽ᵗ⁾(u)`. Ở đây `z_{t,τ}` là toán tử của loại quan hệ τ, phép nhân theo chiều kiểu CompGCN/DistMult.
* `a_q = σ(β_τ + κ_τ·log w + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ'_τ·cos(e_p, e_P(q)))` gồm năm thành phần:
  * cổng theo loại quan hệ do truy vấn chọn (`W_g u_q`);
  * **vai** của node nguồn trong truy vấn (chủ thể / động từ / đối tượng / tân ngữ giới từ, do parser caption áp lên chính truy vấn);
  * **khớp vị từ** giữa cạnh và truy vấn, cả khớp đúng lẫn khớp mềm bằng CLIP-text;
  * support của cạnh (`κ_τ·log w`);
  * độ lệch `β_τ`.
* `s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v,c] · H_q⁽ᵀ⁾(c)⟩`. Video **chỉ nhận** qua nhãn của chính nó (inductive). Điểm cuối = `α·z(cos CLIP) + (1−α)·z(s)`, với α chọn trên VAL.

**Khởi tạo và học các cổng.** γ_τ khởi tạo bằng 1, tức prior "ưu tiên cạnh có vị từ khớp truy vấn"; dữ liệu có thể làm γ giảm. Các cổng vô hướng (β, κ, ρ, γ, γ') học với lr × `scalar_lr_mult`, như cổng của v3. Nếu khởi tạo γ = 0 và học với lr thường, γ gần như không đổi sau huấn luyện. Khi đó G2, G4, G6 cho kết quả trùng G3, và các đối chứng không còn đo được gì; lần thử dev đã gặp đúng trường hợp này.

**Giải thích từ chính cơ chế.** Attention chỉ phụ thuộc truy vấn và thuộc tính tĩnh của cạnh, nên phép truyền là **tuyến tính theo E**. Vì vậy `s` tách **chính xác** thành tổng các số hạng: thị giác + từng nhãn contains + từng **đường đi quan hệ** (`u –τ[p]→ c ←chứa– v`, có thêm một bước khi T = 2). Giải thích là top-m số hạng quan hệ. Xoá một cạnh làm điểm đổi **đúng bằng** tổng đóng góp của các đường đi qua cạnh đó (T = 1); điều này có kiểm thử Q3 và X3.

**Đối chứng tách từng thành phần** (cùng mô hình, cùng giao thức huấn luyện):
* G1 `v4_rel=False`: không cạnh quan hệ;
* G2 `v4_pred=False`: không khớp vị từ;
* G4: hoán vị vị từ giữa các cạnh cùng họ;
* G5: rewiring có ràng buộc;
* G6 `v4_qcond=False`: attention tĩnh, không theo truy vấn — tức là quan hệ như v3;
* R3: chính mô hình v3 cuối.

Liên hệ tài liệu: NBFNet (Zhu et al., NeurIPS 2021) với message điều kiện theo truy vấn; KGAT (Wang et al., KDD 2019) với attention nhận biết quan hệ; CompGCN (Vashishth et al., ICLR 2020) với toán tử quan hệ. Thuộc họ **GAT quan hệ** của cấp độ 1.


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def scatter_sum(src, index, n):
    out = torch.zeros((n,) + tuple(src.shape[1:]), device=src.device, dtype=src.dtype)
    return out.index_add_(0, index, src)


PRED_MODES = ("none", "clip", "clip_adapt")


class TypedSAGEv3(nn.Module):
    """h'_v = W₀ h_v + Σ_t σ(g_t) · [ W_t · AGG_t(h_u) + U_t · AGG_t(e_uv) ]
    e_uv = embedding vị từ của cạnh (chỉ loại cạnh thuộc họ có vị từ tự do, khi pred_mode ≠ 'none'); AGG_t dùng CÙNG hệ số
    chuẩn hoá cho h_u và e_uv ⇒ message(u → v) = W_t h_u + U_t e_uv đúng như thiết kế, trọng số cạnh chỉ nhân MỘT lần.
    AGG theo mp_weight: 'binary' (trung bình, bỏ trọng số) | 'weighted_mean' (Σ w·x / Σ w) | 'weighted_symnorm' (Σ w·x / √(d_u d_v)).
    Nhánh W₀ h_v giữ biểu diễn của node không có hàng xóm (không sinh NaN)."""

    def __init__(self, din, dout, n_types, open_types, gate_init, mp_weight, pred_type_ids=()):
        super().__init__()
        assert mp_weight in ("binary", "weighted_mean", "weighted_symnorm"), mp_weight
        self.mp_weight, self.n_types = mp_weight, n_types
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(n_types)])
        self.pred_type_ids = list(pred_type_ids)
        self.pred_w = nn.ModuleDict({str(t): nn.Linear(din, dout, bias=False) for t in self.pred_type_ids})
        g = torch.full((n_types,), float(gate_init))
        for t in open_types:
            g[t] = 3.0
        self.gate = nn.Parameter(g)

    def coef(self, src, dst, w, n):
        """Hệ số của từng cạnh trong AGG (cùng cho h_u và e_uv)."""
        if self.mp_weight == "binary":
            w = torch.ones_like(w)
        if self.mp_weight in ("binary", "weighted_mean"):
            return w / scatter_sum(w, dst, n)[dst].clamp(min=1e-12)
        d_out, d_in = scatter_sum(w, src, n), scatter_sum(w, dst, n)
        return w / torch.sqrt(d_out[src] * d_in[dst]).clamp(min=1e-12)

    def forward(self, x, edge_index, edge_type, edge_weight, edge_pidx=None, pred_h=None):
        src, dst = edge_index
        n = x.size(0)
        out = self.self_w(x)
        for t in range(self.n_types):
            m = edge_type == t
            if not bool(m.any()):
                continue
            s, d = src[m], dst[m]
            c = self.coef(s, d, edge_weight[m], n).unsqueeze(-1)
            msg = self.neigh_w[t](scatter_sum(x[s] * c, d, n))
            if pred_h is not None and t in self.pred_type_ids:
                pi = edge_pidx[m]
                has = pi >= 0
                if bool(has.any()):
                    msg = msg + self.pred_w[str(t)](scatter_sum(pred_h[pi[has]] * c[has], d[has], n))
            out = out + torch.sigmoid(self.gate[t]) * msg
        return out


class PredicateEncoder(nn.Module):
    """Biểu diễn vị từ của cạnh = Σ_p tỉ phần(p) · φ(e_p), e_p = CLIP-text ĐÓNG BĂNG của vị từ theo prompt.
      'clip'       (phương án A): φ = phép chiếu tuyến tính học được (về chiều GNN)
      'clip_adapt' (phương án B): φ(e) = chiếu(e + MLP(e)); MLP DÙNG CHUNG cho mọi vị từ, lớp cuối khởi tạo 0
                                  ⇒ không có tham số riêng cho vị từ hiếm; vị từ mới (ngoài từ vựng) vẫn mã hoá được."""

    def __init__(self, clip_dim, dout, mode, hidden):
        super().__init__()
        self.mode = mode
        self.proj = nn.Linear(clip_dim, dout)
        if mode == "clip_adapt":
            self.res = nn.Sequential(nn.Linear(clip_dim, hidden), nn.GELU(), nn.Linear(hidden, clip_dim))
            nn.init.zeros_(self.res[2].weight); nn.init.zeros_(self.res[2].bias)

    def forward(self, pmix, pemb):
        e = pemb + self.res(pemb) if self.mode == "clip_adapt" else pemb
        return torch.sparse.mm(pmix, self.proj(e)) if pmix.is_sparse else pmix @ self.proj(e)


class GraphScorerV3(nn.Module):
    """Như GraphScorer v2 (12 mẫu đường đi + 2 mẫu khớp vị từ; khởi tạo node 'learned' | 'clip_frozen' | 'clip_residual')
    cộng nhánh vị từ (pred_mode). pred_mode = 'none' ⇒ đúng mô hình v2 (đối chứng relation-family-only)."""

    def __init__(self, n_nodes, clip_dim, vid_feat, sym_feat, pred_emb=None, cfg=CFG, mp_weight="binary", init="learned",
                 pred_mode="none"):
        super().__init__()
        assert pred_mode in PRED_MODES, pred_mode
        self.n_vid, self.init, self.pred_mode = vid_feat.shape[0], init, pred_mode
        self.register_buffer("vid_feat", vid_feat.clone())
        self.vid_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        n_sym = n_nodes - self.n_vid
        if init == "learned":
            self.sym_emb = nn.Embedding(n_sym, cfg.gnn_dim)
            nn.init.normal_(self.sym_emb.weight, std=0.02)
        else:
            self.register_buffer("sym_feat", torch.as_tensor(sym_feat, dtype=torch.float32).clone())
            self.sym_proj = nn.Linear(sym_feat.shape[1], cfg.gnn_dim)
            if init == "clip_residual":
                self.sym_res = nn.Embedding(n_sym, cfg.gnn_dim)
                nn.init.zeros_(self.sym_res.weight)
        pred_ids = [ETYPE_ID[t] for t in PRED_TYPES] if pred_mode != "none" else []
        if pred_mode != "none":
            self.register_buffer("pred_emb", torch.as_tensor(pred_emb, dtype=torch.float32).clone())
            self.pred_enc = PredicateEncoder(self.pred_emb.shape[1], cfg.gnn_dim, pred_mode, cfg.pred_adapt_hidden)
        open_ids = [ETYPE_ID[t] for t in OPEN_TYPES]
        self.layers = nn.ModuleList([TypedSAGEv3(cfg.gnn_dim, cfg.gnn_dim, len(EDGE_TYPES), open_ids, cfg.gate_init, mp_weight, pred_ids)
                                     for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(), nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        init_w = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init_w[0] = 0.0
        self.path_w = nn.Parameter(init_w)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        if self.init == "learned":
            sym = self.sym_emb.weight
        else:
            sym = self.sym_proj(self.sym_feat)
            if self.init == "clip_residual":
                sym = sym + self.sym_res.weight
        return torch.cat([self.vid_proj(self.vid_feat), sym], 0)

    def edge_gates(self):
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers]

    def predicate_features(self, pmix):
        return self.pred_enc(pmix, self.pred_emb) if self.pred_mode != "none" and pmix is not None else None

    def encode_graph(self, edge_index, edge_type, edge_weight, edge_pidx=None, pmix=None):
        h = self.node_features()
        ph = self.predicate_features(pmix)
        for i, (l, nrm) in enumerate(zip(self.layers, self.norms)):
            h = nrm(l(h, edge_index, edge_type, edge_weight, edge_pidx, ph))
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])

    def path_score(self, path_mat):
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)


_SYM_FEAT = {}


def symbolic_features(inv, vocab):
    """CLIP-text của node lớp (= prompt ensemble của tagger) và khái niệm ('lemma: định nghĩa WordNet')."""
    key = inv.sha1
    if key not in _SYM_FEAT:
        T = term_text_features_v2(vocab)
        cls = np.stack([T[vocab["terms"].index(k)] for k in inv.class_keys])
        texts = [f"{inv.concept_meta[k]['lemma']}: {inv.concept_meta[k]['gloss']}".strip(": ") for k in inv.concept_keys]
        con = clip_encode_texts(texts) if texts else np.zeros((0, cls.shape[1]), np.float32)
        _SYM_FEAT[key] = np.concatenate([cls, con]).astype(np.float32)
    return _SYM_FEAT[key]


def predicate_phrase(rel, p, vocab):
    """Cụm chữ cho prompt: svo → dạng V-ing của động từ đầu + phần còn lại ('ride on' → 'riding on'); prep → nguyên văn."""
    toks = p.split()
    if rel == "svo" and toks:
        g = vocab["gerund"].get(ckey(toks[0], "v")) or naive_gerund(toks[0])
        toks = [g] + toks[1:]
    return " ".join(toks)


def predicate_embeddings(pred_vocab, vocab):
    """[n_vị_từ, D] CLIP-text ĐÓNG BĂNG (trung bình các prompt CFG.pred_prompts[họ], chuẩn L2)."""
    out = None
    for rel in PRED_FAMILIES:
        idx = [i for i, (r, _) in enumerate(pred_vocab) if r == rel]
        if not idx:
            continue
        words = [predicate_phrase(rel, pred_vocab[i][1], vocab) for i in idx]
        E = sum(clip_encode_texts([pr.format(w) for w in words]) for pr in CFG.pred_prompts[rel])
        if out is None:
            out = np.zeros((len(pred_vocab), E.shape[1]), np.float32)
        out[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
    return out if out is not None else np.zeros((len(pred_vocab), 1), np.float32)

# ---------------------------------- v4: QC-RGAT (GNN quan hệ có attention ĐIỀU KIỆN THEO TRUY VẤN) ----------------------------------
# H_q^(0)(c) = E_c ;  H_q^(t+1)(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q^(t)(u)            (cạnh giữa lớp/khái niệm)
# a_q = σ( β_τ + κ_τ·log w_uc + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ'_τ·cos(e_p, e_P(q)) )
# s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v, c] · H_q^(T)(c)⟩        (Ā_con = contains chuẩn hoá theo hàng, video CHỈ nhận)
# Phép truyền TUYẾN TÍNH theo E (attention chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh) ⇒ s = Σ đóng góp của từng
# thành phần: thị giác + từng nhãn contains + từng ĐƯỜNG ĐI quan hệ — phân rã CHÍNH XÁC, giải thích = đúng các số hạng của điểm.
V4_REL_TYPES = list(RELATION_TYPES)
V4_TID = {ETYPE_ID[t]: i for i, t in enumerate(V4_REL_TYPES)}
Q_ROLES = ("none", "subj", "verb", "obj", "pobj", "other")


def query_roles_v4(texts, inv):
    """Vai của node lớp trong CHÍNH truy vấn (parser caption áp lên văn bản truy vấn lúc suy luận — mục đích query_input)."""
    out = []
    for text in texts:
        _, _, trip = parse_caption_v2(text)
        role = {}
        for s_, rel, o_, _p in trip:
            a, b = inv.key2id.get(s_), inv.key2id.get(o_)
            if a is None or b is None:
                continue
            if rel == "agent_of":
                role.setdefault(a, "subj"); role.setdefault(b, "verb")
            elif rel == "object_of":
                role.setdefault(a, "obj"); role.setdefault(b, "verb")
            elif rel == "svo":
                role.setdefault(a, "subj")
                role[b] = role[b] if role.get(b) not in (None, "other") else "obj"
            elif rel == "prep":
                role.setdefault(b, "pobj")
        out.append({k: Q_ROLES.index(r) for k, r in role.items()})
    return out


class RelDataV4:
    """Tensor của một đồ thị cho QC-RGAT: cạnh quan hệ (giữa node lớp/khái niệm), vị từ của cạnh, contains chuẩn hoá theo hàng."""

    def __init__(self, graph, drop_edges=None):
        self.graph = graph
        self.N, self.n = graph.n_vid, graph.n_nodes
        self.ns = self.n - self.N
        s, d, e, w, pi = graph.edge_arrays()
        keep = np.isin(e, list(V4_TID))
        assert (s[keep] >= self.N).all() and (d[keep] >= self.N).all(), "cạnh quan hệ chỉ được nối node lớp/khái niệm"
        self.gidx = np.where(keep)[0]                       # chỉ số cạnh trong graph.edge_arrays() (để giải thích / xoá cạnh)
        if drop_edges is not None:
            self.gidx = np.setdiff1d(self.gidx, np.asarray(list(drop_edges), np.int64))
        s, d, e, w, pi = s[self.gidx], d[self.gidx], e[self.gidx], w[self.gidx], pi[self.gidx]
        self.E = len(self.gidx)
        self.src_np, self.dst_np, self.pidx_np = s - self.N, d - self.N, pi
        self.src = torch.from_numpy(self.src_np).to(DEVICE)
        self.dst = torch.from_numpy(self.dst_np).to(DEVICE)
        self.et = torch.tensor([V4_TID[int(x)] for x in e], dtype=torch.long, device=DEVICE)
        self.logw = torch.from_numpy(np.log(np.maximum(w, 1e-3)).astype(np.float32)).to(DEVICE)
        PE = np.zeros((self.E, PRED_EMB.shape[1]), np.float32)
        rows = np.where(pi >= 0)[0]
        if len(rows):
            X = np.asarray(graph.pmix[pi[rows]] @ PRED_EMB)
            PE[rows] = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
        self.PE = torch.from_numpy(PE).to(DEVICE)
        M = (graph.pmix[np.maximum(pi, 0)].multiply((pi >= 0)[:, None]) if self.E and (pi >= 0).any()
             else sp.csr_matrix((self.E, len(PRED_VOCAB)), dtype=np.float32))   # G8: không cạnh nào mang vị từ
        self.Mex = sp.csr_matrix(M).T.tocsr()               # [n_vị_từ, E]: phần vị từ p trên cạnh e
        A = (graph.A_con.tocsr()[:, self.N:] > 0).astype(np.float32).toarray()
        self.Acon = torch.from_numpy(A / np.maximum(A.sum(1, keepdims=True), 1)).to(DEVICE)

    def query_batch(self, q_idx, use_pred=True):
        """Đầu vào phía truy vấn: node khớp + vai, vị từ truy vấn (khớp đúng + CLIP mềm) trên TỪNG cạnh, CLIP-text."""
        q_idx = np.asarray(q_idx, np.int64)
        B = len(q_idx)
        bi, ci, ri = [], [], []
        role_src = torch.zeros(B, self.ns, dtype=torch.long)
        for b, q in enumerate(q_idx):
            for h in V4_HITS[q]:
                r = V4_ROLES[q].get(h, Q_ROLES.index("verb" if INV.pos[h] == "v" else "other"))
                bi.append(b); ci.append(h - self.N); ri.append(r)
                role_src[b, h - self.N] = r
        ex = np.zeros((B, self.E), np.float32)
        qp = np.zeros((B, PRED_EMB.shape[1]), np.float32)
        if use_pred and self.E:
            for b, q in enumerate(q_idx):
                ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]
                if ids:
                    ex[b] = np.minimum(np.asarray(self.Mex[ids].sum(0)).ravel(), 1.0)
                    m = PRED_EMB[ids].mean(0)
                    qp[b] = m / max(np.linalg.norm(m), 1e-8)
        qp_t = torch.from_numpy(qp).to(DEVICE)
        return dict(q=q_idx, bi=torch.tensor(bi, dtype=torch.long, device=DEVICE), ci=torch.tensor(ci, dtype=torch.long, device=DEVICE),
                    ri=torch.tensor(ri, dtype=torch.long, device=DEVICE), role_src=role_src.to(DEVICE),
                    ex=torch.from_numpy(ex).to(DEVICE), soft=qp_t @ self.PE.T,
                    tq=torch.from_numpy(np.ascontiguousarray(TXT_FEAT[q_idx], dtype=np.float32)).to(DEVICE))


class QCRGAT(nn.Module):
    """rel=False ⇒ G1 (chỉ contains); use_pred=False ⇒ G2 (không vị từ); qcond=False ⇒ G6 (attention tĩnh, không theo truy vấn)."""

    SCALARS = ("beta", "kappa", "rgate", "g_ex", "g_soft")      # cổng vô hướng: lr × CFG.scalar_lr_mult (như cổng của v3)

    def __init__(self, data, clip_dim, d=64, T=1, rel=True, qcond=True, use_pred=True):
        super().__init__()
        assert T in (1, 2), "phân rã đường đi hiện cài cho T = 1 hoặc 2"
        self.D, self.T, self.rel, self.qcond, self.use_pred = data, T, rel, qcond, use_pred
        nt = len(V4_REL_TYPES)
        self.vid = nn.Linear(clip_dim, d)
        self.q = nn.Sequential(nn.Linear(clip_dim, d), nn.GELU(), nn.Linear(d, d))
        self.qo = nn.Linear(d, d)
        self.cls = nn.Embedding(data.ns, d)
        nn.init.normal_(self.cls.weight, std=0.02)
        self.z = nn.Parameter(1 + 0.1 * torch.randn(T, nt, d))
        self.beta = nn.Parameter(torch.full((nt,), -1.0))
        self.kappa = nn.Parameter(torch.zeros(nt))
        self.qgate = nn.Linear(d, nt)
        self.rgate = nn.Parameter(torch.zeros(len(Q_ROLES), nt))
        self.g_ex = nn.Parameter(torch.ones(nt))         # prior: cạnh có vị từ KHỚP truy vấn được ưu tiên (học tiếp từ dữ liệu)
        self.g_soft = nn.Parameter(torch.zeros(nt))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))

    def attention(self, qb, u, D=None):
        """a_q(cạnh) ∈ (0, 1), [B, E]. Chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh (KHÔNG phụ thuộc trạng thái node)."""
        D = D or self.D
        a = (self.beta[D.et] + self.kappa[D.et] * D.logw)[None].expand(u.shape[0], -1)
        if self.qcond:
            a = a + self.qgate(u)[:, D.et] + self.rgate[qb["role_src"][:, D.src], D.et]
            if self.use_pred:
                a = a + self.g_ex[D.et][None] * qb["ex"] + self.g_soft[D.et][None] * qb["soft"]
        return torch.sigmoid(a)

    def class_states(self, qb, u, D=None):
        D = D or self.D
        E = self.cls.weight[None].expand(u.shape[0], -1, -1)
        if not self.rel or D.E == 0:
            return E, None
        a = self.attention(qb, u, D)
        H = E
        for t in range(self.T):
            H = E.index_add(1, D.dst, a[:, :, None] * self.z[t][D.et][None] * H[:, D.src, :])
        return H, a

    def score(self, qb, cand, D=None):
        """cand: [B, K] video id (tensor) ⇒ [B, K]."""
        D = D or self.D
        u = self.q(qb["tq"])
        H, _ = self.class_states(qb, u, D)
        V = self.vid(VID_T[cand]) + torch.einsum("bkc,bcd->bkd", D.Acon[cand], H)
        return torch.einsum("bd,bkd->bk", F.normalize(self.qo(u), dim=-1), V)

    @torch.no_grad()
    def decompose(self, q, vid, D=None):
        """Phân rã CHÍNH XÁC s(q, vid) = thị giác + Σ nhãn + Σ đường đi quan hệ. Mỗi đường đi: list chỉ số cạnh (trong D)
        theo chiều đi từ node nguồn tới node nhãn c của video, kèm đóng góp."""
        D = D or self.D
        qb = D.query_batch([q], self.use_pred)
        u = self.q(qb["tq"])
        uo = F.normalize(self.qo(u), dim=-1)[0]
        E = self.cls.weight
        vt = int(vid)
        terms = [dict(kind="visual", node=None, edges=(), contrib=float(uo @ self.vid(VID_T[vt])))]
        acon = D.Acon[vt]
        tags = torch.nonzero(acon > 0).flatten().tolist()
        for c in tags:
            terms.append(dict(kind="contains", node=c, edges=(), contrib=float(acon[c] * (uo @ E[c]))))
        if self.rel and D.E:
            a = self.attention(qb, u, D)[0]
            inc = defaultdict(list)
            for k in range(D.E):
                inc[int(D.dst_np[k])].append(k)
            for c in tags:
                for e1 in inc.get(c, []):                     # đường đi 1 bước: u –e1→ c   (dùng z của lớp cuối)
                    zz = self.z[self.T - 1][D.et[e1]]
                    terms.append(dict(kind="relation", node=c, edges=(e1,),
                                      contrib=float(acon[c] * a[e1] * (uo @ (zz * E[D.src_np[e1]])))))
                    if self.T == 2:                           # đường đi 2 bước: w –e0→ u –e1→ c
                        for e0 in inc.get(int(D.src_np[e1]), []):
                            zz2 = self.z[1][D.et[e1]] * self.z[0][D.et[e0]]
                            terms.append(dict(kind="relation", node=c, edges=(e0, e1),
                                              contrib=float(acon[c] * a[e1] * a[e0] * (uo @ (zz2 * E[D.src_np[e0]])))))
        return terms


VID_T = None


def v4_tensors():
    global VID_T
    VID_T = torch.from_numpy(np.ascontiguousarray(VID_FEAT, dtype=np.float32)).to(DEVICE)


## 4.10 Engine và hai giao thức suy luận
* **strict inductive** (mặc định): video val, test và tập ngoài **chỉ nhận** message qua `contains_inv`, nên không ảnh hưởng node dùng chung hay video khác. Có unit test cho cả video test MSVD và video MSR-VTT.
* **transductive**: chỉ để báo cáo.

Engine mang thêm chỉ số vị từ của từng cạnh (`edge_pidx`) và bảng phân phối vị từ (`pmix`) cho GNN. Nó cũng có ma trận truy vấn × (vị từ, node) cho mẫu P1m. Vị từ của truy vấn được khớp khi mọi token của vị từ có mặt trong truy vấn, ví dụ `ride on` khớp "a man is riding on a horse".

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def znorm_per_row(x, eps=1e-6):
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)


PROTOCOLS = ("inductive", "transductive")


def _torch_sparse(M):
    M = M.tocoo()
    idx = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))
    return torch.sparse_coo_tensor(idx, torch.from_numpy(M.data.astype(np.float32)), M.shape).coalesce()


class EngineV3:
    """Truy vấn = caption; ứng viên = video. Ba tập cạnh cho GNN (như v2):
      train        : bỏ MỌI cạnh chạm video không thuộc train;
      inductive    : cạnh của train + cạnh lớp → video (contains_inv) của video val/test/ext, KHÔNG có chiều video → lớp
                     ⇒ video mới chỉ NHẬN message, không ảnh hưởng node lớp/khái niệm dùng chung hay video khác;
      transductive : đồ thị đầy đủ.
    Thêm ở v3: chỉ số vị từ của từng cạnh (edge_pidx) + bảng phân phối vị từ (pmix) cho GNN; ma trận truy vấn × (vị từ, node)
    cho mẫu đường đi khớp vị từ P1m."""

    _QCACHE = {}

    def __init__(self, graph, vid_feat, txt_feat, qtok, qpreds):
        self.graph = graph
        self.vid_feat = torch.from_numpy(np.ascontiguousarray(vid_feat, dtype=np.float32))
        self.txt_feat = torch.from_numpy(np.ascontiguousarray(txt_feat, dtype=np.float32))
        n = graph.n_nodes
        key = (graph.inv.sha1, len(qtok), id(qtok), id(qpreds))
        if key not in EngineV3._QCACHE:                      # chỉ phụ thuộc inventory + truy vấn ⇒ dùng chung mọi biến thể đồ thị
            qc = [parse_tokens_v2(t, graph.inv) for t in qtok]
            rows, cols = [], []
            for j, hits in enumerate(qc):
                for nid, _, _ in hits:
                    rows.append(j); cols.append(nid)
            QT = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(qtok), n))
            QPB = {}
            for rel in PRED_FAMILIES:
                r_, c_ = [], []
                for j, (hits, qp) in enumerate(zip(qc, qpreds)):
                    for pid in qp[rel]:
                        for nid, _, _ in hits:
                            r_.append(j); c_.append(pid * n + nid)
                QPB[rel] = sp.csr_matrix((np.ones(len(r_), np.float32), (r_, c_)), shape=(len(qtok), len(PRED_VOCAB) * n))
            EngineV3._QCACHE[key] = (qc, QT, QPB)
        self.q_concepts, self.QT, self.QPB = EngineV3._QCACHE[key]
        self.q_preds = qpreds
        s, d, e, w, pi = graph.edge_arrays()
        N = graph.n_vid
        is_train_vid = np.zeros(n, bool)
        is_train_vid[np.array(SPLIT["train"])] = True
        other_vid = (np.arange(n) < N) & ~is_train_vid
        touch = other_vid[s] | other_vid[d]
        keep = {"train": ~touch, "transductive": np.ones(len(s), bool),
                "inductive": ~touch | ((e == ETYPE_ID["contains_inv"]) & other_vid[d] & ~other_vid[s])}
        self.edges = {k: (torch.from_numpy(np.stack([s[m], d[m]])), torch.from_numpy(e[m]), torch.from_numpy(w[m]),
                          torch.from_numpy(pi[m])) for k, m in keep.items()}
        self.edge_stats = {k: int(m.sum()) for k, m in keep.items()}
        self.pmix = _torch_sparse(graph.pmix)

    def edges_on(self, name):
        ei, et, ew, epi = self.edges[name]
        return ei.to(DEVICE), et.to(DEVICE), ew.to(DEVICE), epi.to(DEVICE), self.pmix.to(DEVICE)

    def path_feats(self, cand, q_idx):
        cand, q_idx = np.asarray(cand), np.asarray(q_idx)
        if cand.ndim == 1:
            cand = cand[:, None]
        B, K = cand.shape
        out = np.zeros((B, K, len(PATTERNS)), np.float32)
        for s0 in range(0, B, CFG.eval_chunk):
            c, q = cand[s0:s0 + CFG.eval_chunk], q_idx[s0:s0 + CFG.eval_chunk]
            rows, inv_ = np.unique(c.ravel(), return_inverse=True)
            inv_ = inv_.reshape(c.shape)
            col = np.arange(len(q))[:, None]
            Qb = self.QT[q].T.tocsr()
            for p, name in enumerate(PATTERNS):
                if name.startswith("P1m_"):
                    M = self.graph.R[name]
                    if M.nnz == 0:
                        continue
                    out[s0:s0 + len(q), :, p] = (M[rows] @ self.QPB[name[4:]][q].T).toarray()[inv_, col]
                else:
                    out[s0:s0 + len(q), :, p] = (self.graph.R[name][rows] @ Qb).toarray()[inv_, col]
        return np.log1p(out)

    def clip_candidates(self, q_idx, gallery, k):
        gallery, q_idx = np.asarray(gallery), np.asarray(q_idx)
        G = self.vid_feat[torch.from_numpy(gallery)]
        k = min(int(k), len(gallery))
        C, S = [], []
        for s0 in range(0, len(q_idx), CFG.eval_chunk):
            top = (self.txt_feat[torch.from_numpy(q_idx[s0:s0 + CFG.eval_chunk])] @ G.T).topk(k, dim=1)
            C.append(gallery[top.indices.numpy()]); S.append(top.values.numpy())
        return np.concatenate(C), np.concatenate(S)


def make_model(engine, sym_feat, mp_weight="binary", init="learned", pred_mode="none"):
    return GraphScorerV3(engine.graph.n_nodes, engine.txt_feat.shape[1], engine.vid_feat, sym_feat, pred_emb=PRED_EMB,
                         mp_weight=mp_weight, init=init, pred_mode=pred_mode).to(DEVICE)


def train_graph_scorer_v3(engine, train_q, sym_feat, seed=0, mp_weight="binary", init="learned", pred_mode="none", verbose=False):
    """InfoNCE như v1/v2 (âm trong batch không trùng video + âm khó CLIP trong gallery TRAIN); chỉ dùng cạnh 'train'."""
    train_q = np.asarray(train_q)
    CAPS.use(train_q, "gnn_train")
    set_seed(seed)
    gold_all = CAP_VID[train_q]
    train_vids = np.unique(gold_all)
    assert all(SPLIT_OF[int(v)] == "train" for v in train_vids)
    model = make_model(engine, sym_feat, mp_weight, init, pred_mode)
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    params = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n_, p in params if n_.endswith("gate") or n_ in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW([dict(params=[p for _, p in params if id(p) not in sid]),
                             dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
                            lr=CFG.lr, weight_decay=CFG.weight_decay)
    edges = engine.edges_on("train")
    vid_dev = engine.vid_feat.to(DEVICE)
    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)
    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.from_numpy(cand_np).to(DEVICE)
        pm = torch.from_numpy(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, vid_dev[cand]))
        return z / CFG.temperature

    by_vid = defaultdict(list)
    for j, v in enumerate(gold_all):
        by_vid[int(v)].append(j)
    rng, hist = np.random.RandomState(seed), []
    for ep in range(CFG.epochs):
        model.train()
        tot, nb = 0.0, 0
        for _ in range(CFG.caption_passes):
            perm = rng.permutation(train_vids)
            for s0 in range(0, len(perm), CFG.train_bs):
                bv = perm[s0:s0 + CFG.train_bs]
                if len(bv) < 8:
                    continue
                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])
                qi, gi = train_q[sel], gold_all[sel]
                qf = engine.txt_feat[torch.from_numpy(qi)].to(DEVICE)
                h = model.encode_graph(*edges)
                loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf), torch.arange(len(gi), device=DEVICE))
                loss = loss + F.cross_entropy(logits_for(np.concatenate([gi[:, None], hard[sel]], 1), qi, h, qf),
                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):
            print(f"    epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.2f}" for x in model.pattern_weights().detach().cpu().numpy()))
    model.eval()
    return model, hist


@torch.no_grad()
def score_queries_v3(model, engine, q_idx, gallery, k, protocol="inductive"):
    if isinstance(model, QCRGAT):                       # v4: cùng đầu ra, s_path = 0 (λ = 1)
        return score_queries_v4(model, engine, q_idx, gallery, k, protocol)
    model.eval()
    h = model.encode_graph(*engine.edges_on(protocol))
    q_idx = np.asarray(q_idx)
    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)
    s_gnn, s_path = np.zeros_like(s_clip), np.zeros_like(s_clip)
    for s0 in range(0, len(q_idx), CFG.eval_chunk):
        sl = slice(s0, s0 + CFG.eval_chunk)
        c = torch.from_numpy(cand[sl]).to(DEVICE)
        qf = engine.txt_feat[torch.from_numpy(q_idx[sl])].to(DEVICE)
        s_gnn[sl] = model.gnn_score(h, qf, c).cpu().numpy()
        s_path[sl] = model.path_score(torch.from_numpy(engine.path_feats(cand[sl], q_idx[sl])).to(DEVICE)).cpu().numpy()
    return dict(cand=cand, clip_raw=s_clip, path_raw=s_path, gnn_raw=s_gnn,
                s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s_gnn), s_path=znorm_per_row(s_path))


def tune_on_val_v3(model, engine, part, k, clip_rank, lam_grid=None, protocol="inductive"):
    """(α, λ) tối đa MRR trên VAL; lam_grid cố định cho ablation path-only (λ=0) / GNN-only (λ=1)."""
    assert part["name"] == "val", "tune chỉ nhận VAL"
    CAPS.use(part["q_idx"], "tune")
    sc = score_queries_v3(model, engine, part["q_idx"], part["gallery"], k, protocol)
    grid, best = [], None
    for a in CFG.alpha_grid:
        for l in (lam_grid or CFG.lambda_grid):
            r = ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], a, l), part["gold"], clip_rank)
            m = metrics_from_ranks(r, part["gold"])
            grid.append(dict(alpha=a, lam=l, k=k, **{x: m[x] for x in ("R@1", "R@5", "R@10", "MRR")}, rank=r))
            if best is None or m["MRR"] > best["MRR"]:
                best = grid[-1]
    return (best["alpha"], best["lam"]), [{k_: v for k_, v in g_.items() if k_ != "rank"} for g_ in grid], best

# ---------------------------------- v4: huấn luyện + chấm điểm QC-RGAT (cùng giao thức v3: InfoNCE, âm khó CLIP, chỉ video TRAIN) ----------------------------------
_RELDATA = {}


def reldata_v4(engine):
    if id(engine) not in _RELDATA:
        _RELDATA[id(engine)] = RelDataV4(engine.graph)
    return _RELDATA[id(engine)]


def make_model_v4(engine, design):
    return QCRGAT(reldata_v4(engine), TXT_FEAT.shape[1], d=CFG.v4_dim, T=design["v4_layers"], rel=design["v4_rel"],
                  qcond=design["v4_qcond"], use_pred=design["v4_pred"]).to(DEVICE)


def train_v4(engine, train_q, design, seed=0, verbose=False):
    """Như train_graph_scorer_v3: in-batch (video không trùng) + 31 âm khó CLIP trong gallery TRAIN; logit = z(s) + c·z(cos CLIP)."""
    train_q = np.asarray(train_q)
    CAPS.use(train_q, "gnn_train")
    set_seed(seed)
    gold_all = CAP_VID[train_q]
    train_vids = np.unique(gold_all)
    assert all(SPLIT_OF[int(v)] == "train" for v in train_vids)
    model = make_model_v4(engine, design)
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    named = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]
    scal = [p for n_, p in named if n_ in QCRGAT.SCALARS]
    opt = torch.optim.AdamW([dict(params=[p for n_, p in named if n_ not in QCRGAT.SCALARS]),
                             dict(params=scal, lr=CFG.v4_lr * CFG.scalar_lr_mult, weight_decay=0.0)],
                            lr=CFG.v4_lr, weight_decay=CFG.weight_decay)
    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)
    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])
    txt_dev = torch.from_numpy(np.ascontiguousarray(TXT_FEAT, dtype=np.float32)).to(DEVICE)

    def logits_for(qb, cand_np):
        cand = torch.from_numpy(cand_np).to(DEVICE)
        z = znorm_per_row(model.score(qb, cand))
        if CFG.train_with_clip:
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", txt_dev[qb["q"]], VID_T[cand]))
        return z / CFG.temperature

    by_vid = defaultdict(list)
    for j, v in enumerate(gold_all):
        by_vid[int(v)].append(j)
    rng, hist = np.random.RandomState(seed), []
    for ep in range(CFG.epochs):
        model.train()
        tot, nb = 0.0, 0
        for _ in range(CFG.caption_passes):
            perm = rng.permutation(train_vids)
            for s0 in range(0, len(perm), CFG.train_bs):
                bv = perm[s0:s0 + CFG.train_bs]
                if len(bv) < 8:
                    continue
                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])
                gi = gold_all[sel]
                qb = model.D.query_batch(train_q[sel], model.use_pred)
                loss = F.cross_entropy(logits_for(qb, np.tile(gi, (len(gi), 1))), torch.arange(len(gi), device=DEVICE))
                loss = loss + F.cross_entropy(logits_for(qb, np.concatenate([gi[:, None], hard[sel]], 1)),
                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):
            print(f"    epoch {ep:2d} | loss {hist[-1]:.4f}")
    model.eval()
    return model, hist


@torch.no_grad()
def score_queries_v4(model, engine, q_idx, gallery, k, protocol="inductive", D=None):
    """Cùng đầu ra với score_queries_v3; s_gnn = điểm QC-RGAT, s_path = 0 (λ cố định = 1)."""
    model.eval()
    D = D or model.D
    q_idx = np.asarray(q_idx)
    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)
    s = np.zeros_like(s_clip)
    bs = max(1, CFG.v4_score_bs)
    for s0 in range(0, len(q_idx), bs):
        qb = D.query_batch(q_idx[s0:s0 + bs], model.use_pred)
        s[s0:s0 + bs] = model.score(qb, torch.from_numpy(cand[s0:s0 + bs]).to(DEVICE), D).cpu().numpy()
    z = np.zeros_like(s)
    return dict(cand=cand, clip_raw=s_clip, path_raw=z, gnn_raw=s, s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s), s_path=z)


## 4.11 Chỉ số đánh giá

* Mỗi **caption là một truy vấn**, có đúng một video dương. Notebook báo cáo R@1, R@5, R@10, MRR, nDCG@10, MedR và `MRR_video_macro` (macro-MRR theo video).
* Thứ hạng tính trên **toàn bộ gallery**: top-K được sắp lại, ngoài top-K giữ thứ hạng CLIP.
* Kiểm định cặp dùng **bootstrap theo cụm video**. Họ so sánh chính được hiệu chỉnh **Holm–Bonferroni**; luật chọn từng stage cũng hiệu chỉnh Holm trên các ứng viên của stage.
* Notebook báo cáo thêm: candidate recall, nhóm direct/indirect/other, khái niệm hiếm, vị từ hiếm/phổ biến/không có vị từ, số node/cạnh/bậc, thời gian chạy và bộ nhớ GPU.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def ranks_with_fallback(cand, scores, gold, clip_rank):
    """Thứ hạng trên TOÀN gallery: trong top-K theo điểm xếp lại; ngoài top-K giữ thứ hạng CLIP (> K)."""
    order = np.argsort(-scores, axis=1, kind="stable")
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold)[:, None]
    return np.where(hit.any(1), hit.argmax(1) + 1, np.maximum(clip_rank, cand.shape[1] + 1))


def metrics_from_ranks(rank, gold=None, ks=(1, 5, 10)):
    rank = np.asarray(rank, np.float64)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float((1.0 / rank).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    out["MedR"] = float(np.median(rank))
    out["n_queries"] = int(len(rank))
    if gold is not None:
        _, inv = np.unique(np.asarray(gold), return_inverse=True)
        out["MRR_video_macro"] = float((np.bincount(inv, 1.0 / rank) / np.bincount(inv)).mean())
        out["n_videos"] = int(inv.max() + 1) if len(inv) else 0
    return out


METRIC_KEYS = ["R@1", "R@5", "R@10", "MRR", "nDCG@10", "MedR", "MRR_video_macro"]


def fmt_mean_std(vals):
    vals = np.asarray(vals, float)
    return f"{vals.mean():.4f} ± {vals.std():.4f}"


def cluster_bootstrap_diff(rr_a, rr_b, gold, n_boot=5000, seed=0):
    """Δ = mean(rr_a − rr_b) theo truy vấn; khoảng tin cậy 95% bằng bootstrap THEO VIDEO (cụm caption)."""
    d = np.asarray(rr_a) - np.asarray(rr_b)
    _, inv = np.unique(np.asarray(gold), return_inverse=True)
    S, C = np.bincount(inv, d), np.bincount(inv).astype(float)
    idx = np.random.RandomState(seed).randint(0, len(S), (n_boot, len(S)))
    boot = S[idx].sum(1) / C[idx].sum(1)
    return dict(diff=float(d.mean()), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                p_not_better=float((boot <= 0).mean()), a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum()))


def holm(pvals):
    """p đã hiệu chỉnh Holm–Bonferroni (đơn điệu), cùng thứ tự với đầu vào."""
    p = np.asarray(pvals, float)
    m = len(p)
    if m == 0:
        return p
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i]))
        adj[i] = run
    return adj

## 4.12 Giải thích v3
* Đường đi được liệt kê theo **chiều** cạnh, kèm loại cạnh và **vị từ** ở bước quan hệ.
  * Mẫu P1m chỉ dùng vị từ có trong truy vấn.
  * Giải thích hợp lệ khi: cạnh có thật, đúng loại, đúng chiều, đúng kiểu node; vị từ thuộc **đúng** cạnh theo **đúng** chiều và có trong truy vấn.
  * Có đối chứng âm: sửa pred_id thì phải bị phát hiện.
* Điểm liệt kê phải khớp ma trận ở mọi `path_mode`.
* Fidelity báo riêng ba mức: thay đổi **điểm**, **thứ hạng**, **quyết định** top-1 / top-10.
* Có phần đóng góp theo mẫu đường đi và độ nhạy theo seed (Jaccard).
* Xoá cạnh giải thích được so với xoá cạnh **đối chứng** cùng loại.
* Notebook báo cáo trọng số hiệu dụng `(1−α)(1−λ)` của nhánh đường đi.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def enumerate_paths_v3(g, vid, concept_ids, qpred=None, PM=None):
    """Đường đi CỤ THỂ từ video tới khái niệm truy vấn, đi theo CHIỀU cạnh, ghi loại cạnh từng bước và VỊ TỪ của bước
    quan hệ. Trọng số = idf(đích) × Π trọng số trong ma trận đường đi (PM) của chế độ path hiện tại ⇒ khớp graph.R.
    Mẫu P1m_f (path_mode 'pred_match'): bước a –f→ b chỉ tính phần vị từ có trong truy vấn (qpred[f] = tập pred_id)."""
    PM = PM or g.PM
    target = {int(c) for c in concept_ids}
    C = PM["C"]
    own = set(C.indices[C.indptr[vid]:C.indptr[vid + 1]].tolist())
    out = []

    def nbrs(M, x):
        sl = slice(M.indptr[x], M.indptr[x + 1])
        return zip(M.indices[sl].tolist(), M.data[sl].tolist())

    def add(pattern, nodes, rels, w, pred=None):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return
        w = float(g.idf[c]) * float(w)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=[int(x) for x in nodes], rels=list(rels), w=w, pred=pred))

    for o, w0 in nbrs(C, vid):
        if o in target:
            add("P0_contains", [o], ["contains"], w0)
        for f in FAMILIES:
            for t in (FAMILY_REL[f], FAMILY_REL[f] + "_inv"):
                for o2, w1 in nbrs(PM[t], o):
                    if o2 in target:
                        add(f"P1_{f}", [o, o2], ["contains", t], w0 * w1)
                    for c, w2 in nbrs(PM["isa"], o2):
                        if c in target:
                            add(f"P4_{f}_isa", [o, o2, c], ["contains", t, "isa"], w0 * w1 * w2)
        for c, w1 in nbrs(PM["isa"], o):
            if c in target:
                add("P2_isa", [o, c], ["contains", "isa"], w0 * w1)
        for o2, w1 in nbrs(PM["related"], o):
            if o2 in target:
                add("P3_related", [o, o2], ["contains", "related"], w0 * w1)
            for c, w2 in nbrs(PM["isa"], o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], ["contains", "related", "isa"], w0 * w1 * w2)
        if g.path_mode == "pred_match" and qpred:
            for rel in PRED_FAMILIES:
                want = set(qpred.get(rel, ()))
                if not want:
                    continue
                for t in (rel, rel + "_inv"):
                    for o2, w1 in nbrs(PM[t], o):
                        if o2 not in target:
                            continue
                        a, b = (o2, o) if t.endswith("_inv") else (o, o2)
                        for pid, s in g.pshare[rel].get((a, b), {}).items():
                            if pid in want:
                                add(f"P1m_{rel}", [o, o2], ["contains", t], w0 * w1 * s, pred=(rel, int(pid), float(s)))
    return out


def paths_to_counts_v3(paths, mode):
    """log1p(tổng theo mẫu) — với path_dedup: tổng theo khái niệm của đường đi MẠNH NHẤT (khớp _maxprod)."""
    v = np.zeros(len(PATTERNS), np.float64)
    if mode == "dedup":
        best = defaultdict(float)
        for p in paths:
            k = (p["pattern"], p["nodes"][-1])
            best[k] = max(best[k], p["w"])
        for (pat, _), w in best.items():
            v[PIDX[pat]] += w
    else:
        for p in paths:
            v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_edges_v3(p, vid):
    """[(loại, u, v)] gồm cạnh contains đầu tiên."""
    nodes = [int(vid)] + p["nodes"]
    return list(zip(p["rels"], nodes[:-1], nodes[1:]))


def edge_valid_v3(g, rel, u, v):
    """Cạnh có thật trong ma trận GỐC đúng loại + đúng chiều, và kiểu đầu/cuối khớp REL_SPEC."""
    M = g.A_con if rel == "contains" else g.A[rel]
    if M[u, v] == 0:
        return False
    ok_src = REL_SPEC[rel]["src"].split(":")[0].split("|")
    ok_tgt = REL_SPEC[rel]["tgt"].split(" ")[0].split(":")[0].split("|")
    return g.inv.node_type[u] in ok_src and g.inv.node_type[v] in ok_tgt


def pred_valid_v3(g, p, vid, qtokens):
    """Đường đi khớp vị từ: vị từ ghi trong giải thích phải (1) thuộc phân phối vị từ của ĐÚNG cạnh theo ĐÚNG chiều và
    (2) có mặt trong truy vấn. Đường đi không mang vị từ ⇒ True."""
    if p.get("pred") is None:
        return True
    rel, pid, share = p["pred"]
    t, u, v = path_edges_v3(p, vid)[1]
    a, b = (v, u) if t.endswith("_inv") else (u, v)
    ok_edge = abs(g.pshare[rel].get((a, b), {}).get(pid, -1.0) - share) < 1e-9
    return bool(ok_edge and all(x in qtokens for x in predicate_tokens(PRED_VOCAB[pid][1])) and PRED_VOCAB[pid][0] == rel)


def path_text_v3(g, p):
    nm = lambda i: g.inv.name[i]
    s = "video"
    nodes = p["nodes"]
    for rel, a, b in zip(p["rels"], [None] + nodes[:-1], nodes):
        lab = REL_TEXT.get(rel, rel)
        if "{pred}" in lab:
            if p.get("pred") is not None:
                pr = f"{PRED_VOCAB[p['pred'][1]][1]}({p['pred'][2]:.2f})"
            else:
                d = g.predicates_of(rel, a, b)
                tot = max(sum(d.values()), 1)
                pr = "/".join(f"{k}({c / tot:.2f})" for k, c in sorted(d.items(), key=lambda x: (-x[1], x[0]))[:3]) or "?"
            lab = lab.format(pred=pr.replace(" ", "_"))
        s += f" –{lab}→ {nm(b)}"
    return s


def q_args(eng, q):
    """(node khái niệm của truy vấn, {họ: pred_id khớp}, tập token truy vấn)."""
    return [nid for nid, _, _ in eng.q_concepts[q]], eng.q_preds[q], QPRED_TOK[q]


@torch.no_grad()
def explain_and_score_v3(model, g, vid, concept_ids, qpred=None, qtokens=frozenset(), top_m=CFG.top_m_paths, rng_seed=0):
    paths = enumerate_paths_v3(g, vid, concept_ids, qpred)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts_v3(ps, g.path_mode) * w).sum())
    full = score(paths)
    contrib = [full - score(paths[:i] + paths[i + 1:]) for i in range(len(paths))]
    order = np.argsort(-np.array(contrib), kind="stable") if contrib else np.array([], int)
    ex = [int(i) for i in order[:top_m]]
    expl = [dict(text=path_text_v3(g, paths[i]), pattern=paths[i]["pattern"], contribution=float(contrib[i]),
                 nodes=paths[i]["nodes"], rels=paths[i]["rels"], pred=paths[i]["pred"]) for i in ex]
    keep = [paths[i] for i in ex]
    drop = [p for i, p in enumerate(paths) if i not in set(ex)]
    rng, rnd = np.random.RandomState(rng_seed), []
    for _ in range(10):
        r = set(rng.choice(len(paths), size=min(len(ex), len(paths)), replace=False).tolist()) if paths else set()
        rnd.append(score([p for i, p in enumerate(paths) if i not in r]))
    cset = {int(c) for c in concept_ids}
    valid = [p["nodes"][-1] in cset and all(edge_valid_v3(g, *e) for e in path_edges_v3(p, vid)) for p in keep]
    pvalid = [pred_valid_v3(g, p, vid, qtokens) for p in keep]
    share = np.zeros(len(PATTERNS))
    for i in range(len(paths)):
        share[PIDX[paths[i]["pattern"]]] += max(contrib[i], 0)
    detail = dict(n_paths=len(paths), score_full=full, score_without_explanation=score(drop),
                  score_only_explanation=score(keep), score_without_random=float(np.mean(rnd)) if rnd else full,
                  sparsity=1 - len(ex) / max(len(paths), 1), validity=float(np.mean(valid)) if valid else 0.0,
                  predicate_validity=float(np.mean(pvalid)) if pvalid else 1.0,
                  n_predicate_paths=int(sum(p.get("pred") is not None for p in keep)),
                  pattern_share={p: float(s / max(share.sum(), 1e-12)) for p, s in zip(PATTERNS, share) if s > 0})
    return expl, full, detail


def _rank_in(fin, j):
    return int((fin > fin[j]).sum()) + 1


def path_score_from_R(g, R, cids, qpred, gain, w):
    """s_path (chưa chuẩn hoá) từ một R đã tính lại (hàng = ứng viên) — cùng công thức với engine.path_feats."""
    n, cols = g.n_nodes, np.array(sorted({int(c) for c in cids}))
    feats = []
    for p in PATTERNS:
        if p.startswith("P1m_"):
            pc = np.array(sorted(pid * n + c for pid in qpred.get(p[4:], ()) for c in cols), np.int64)
            feats.append(np.asarray(R[p][:, pc].sum(1)).ravel() if len(pc) and R[p].nnz else np.zeros(R["P0_contains"].shape[0]))
        else:
            feats.append(np.asarray(R[p][:, cols].sum(1)).ravel())
    return gain * (np.log1p(np.stack(feats, 1)) * w).sum(1)


@torch.no_grad()
def edge_deletion_v3(run, part, groups, n=CFG.n_edge_deletion, seed=0, protocol="inductive"):
    """Với mỗi truy vấn (video đúng trong top-K): xoá THẬT (a) cạnh then chốt của đường đi giải thích top-1 và
    (b) đường đi QUAN HỆ đứng đầu; đối chứng: xoá một cạnh CÙNG LOẠI, KHÔNG nằm trên đường đi nào tới khái niệm
    của truy vấn. Tính lại R cho mọi ứng viên từ ma trận đã sửa (cùng công thức), ghi Δ s_path, Δ điểm cuối, hạng,
    và quyết định top-1 / top-10. GNN giữ cố định để cô lập nhánh đường đi."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    a, l, z = run["alpha"], run["lam"], znorm_per_row
    h = model.encode_graph(*eng.edges_on(protocol))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    rng = np.random.RandomState(seed)
    rows, used, mism = [], set(), 0

    def delete(rel, u, v):
        A2 = dict(g.A)
        Acon = g.A_con
        if rel == "contains":
            M = g.A_con.tolil(copy=True); M[u, v] = 0; Acon = M.tocsr(); Acon.eliminate_zeros()
        else:
            M = g.A[rel].tolil(copy=True); M[u, v] = 0; A2[rel] = M.tocsr(); A2[rel].eliminate_zeros()
        return g.path_mats(A=A2, A_con=Acon)

    for i in rng.permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng, q)
        if gold in used or not cids:
            continue
        cand, s_clip = eng.clip_candidates(np.array([q]), part["gallery"], CFG.rerank_k)
        cand, s_clip = cand[0], s_clip[0]
        hit = np.where(cand == gold)[0]
        if len(hit) == 0:
            continue
        expl, _, _ = explain_and_score_v3(model, g, gold, cids, qpred, qtok)
        expl = [e for e in expl if e["contribution"] > 0]
        if not expl:
            continue
        j = int(hit[0])
        sp0 = path_score_from_R(g, g.compute_R(rows=cand), cids, qpred, gain, w)
        sp_eng = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)
        mism += int(not np.allclose(sp0, sp_eng, atol=1e-4))
        sg = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()
        f0 = combine(z(s_clip[None]), z(sg), z(sp0[None]), a, l)[0]
        r0, top0 = _rank_in(f0, j), int(cand[np.argmax(f0)])
        targets = [("top1", expl[0])]
        relp = next((e for e in expl if e["pattern"] != "P0_contains"), None)
        if relp is not None and relp is not expl[0]:
            targets.append(("top_relational", relp))
        on_path = {(t_, x, y) for p in enumerate_paths_v3(g, gold, cids, qpred) for t_, x, y in path_edges_v3(p, gold)}
        for which, p in targets:
            edges = path_edges_v3(p, gold)
            rel, u, v = edges[1] if len(edges) > 1 else edges[0]
            ctrl = None
            if rel != "contains":                        # cạnh đối chứng cùng loại, không nằm trên đường đi nào
                cand_e = [(x, y) for x, y in zip(*g.A[rel].nonzero()) if (rel, x, y) not in on_path]
                if cand_e:
                    ctrl = cand_e[rng.randint(len(cand_e))]
            todo = [("explanation", (u, v))] + ([("control", ctrl)] if ctrl is not None else [])
            for kind, (uu, vv) in todo:
                sp1 = path_score_from_R(g, g.compute_R(rows=cand, PM=delete(rel, uu, vv)), cids, qpred, gain, w)
                f1 = combine(z(s_clip[None]), z(sg), z(sp1[None]), a, l)[0]
                r1 = _rank_in(f1, j)
                rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], group=str(groups[i]), which=which, deleted=kind,
                                 explanation=p["text"], pattern=p["pattern"], relation=rel,
                                 predicate=PRED_VOCAB[p["pred"][1]][1] if p.get("pred") else "",
                                 edge=f"{g.inv.name[uu]} → {g.inv.name[vv]}", edge_valid=edge_valid_v3(g, rel, uu, vv),
                                 path_contribution=p["contribution"], s_path_before=float(sp0[j]), s_path_after=float(sp1[j]),
                                 d_s_path=float(sp1[j] - sp0[j]), d_final=float(f1[j] - f0[j]), rank_before=r0, rank_after=r1,
                                 rank_changed=r1 != r0, top1_changed=int(cand[np.argmax(f1)]) != top0,
                                 top10_changed=(r0 <= 10) != (r1 <= 10),
                                 w_path_effective=(1 - a) * (1 - l), alpha=a, lam=l))
        used.add(gold)
        if len(used) >= n:
            break
    return pd.DataFrame(rows), mism


@torch.no_grad()
def path_fidelity_v3(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0, protocol="inductive"):
    """Fidelity mức đường đi (score) và mức thứ hạng của điểm cuối cho cùng một tập truy vấn; đóng góp theo mẫu đường đi."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    a, l, z = run["alpha"], run["lam"], znorm_per_row
    h = model.encode_graph(*eng.edges_on(protocol))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    sc = lambda ps: gain * float((paths_to_counts_v3(ps, g.path_mode) * w).sum())
    rng, rows, used = np.random.RandomState(seed), [], set()
    for i in rng.permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng, q)
        if gold in used or not cids:
            continue
        cand, s_clip = eng.clip_candidates(np.array([q]), part["gallery"], CFG.rerank_k)
        cand, s_clip = cand[0], s_clip[0]
        hit = np.where(cand == gold)[0]
        expl, s, det = explain_and_score_v3(model, g, gold, cids, qpred, qtok, top_m=top_m)
        if len(hit) == 0 or det["n_paths"] <= top_m:
            continue
        paths = enumerate_paths_v3(g, gold, cids, qpred)
        j = int(hit[0])
        sp_raw = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()

        def rank_with(kept):
            s_ = sp_raw.copy(); s_[j] = sc(kept)
            return _rank_in(combine(z(s_clip[None]), z(s_gnn), z(s_[None]), a, l)[0], j)

        contrib = np.array([sc(paths) - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib, kind="stable")[:top_m].tolist())
        rr = [rank_with([p for k, p in enumerate(paths) if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
              for _ in range(10)]
        rf, rd = rank_with(paths), rank_with([p for k, p in enumerate(paths) if k not in top])
        rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], group=str(groups[i]),
                         **{k_: v for k_, v in det.items() if k_ != "pattern_share"},
                         **{f"share_{k_}": v for k_, v in det["pattern_share"].items()},
                         matrix_vs_enumeration_ok=bool(abs(sc(paths) - sp_raw[j]) < 1e-4),
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"],
                         rank_full=rf, rank_drop_expl=rd, rank_drop_rand=float(np.mean(rr)),
                         rank_only_expl=rank_with([paths[k] for k in top]),
                         top10_changed_drop_expl=(rf <= 10) != (rd <= 10), w_path_effective=(1 - a) * (1 - l),
                         explanation=" || ".join(e["text"] for e in expl[:3])))
        used.add(gold)
        if len(rows) >= n_queries:
            break
    return pd.DataFrame(rows)


@torch.no_grad()
def explanation_sensitivity(runs_by_seed, part, n=CFG.n_seed_sensitivity, top_m=CFG.top_m_paths, seed=0):
    """Jaccard của tập đường đi giải thích top-m cho CÙNG (truy vấn, video đúng) giữa các seed / cấu hình."""
    keys = list(runs_by_seed)
    if len(keys) < 2:
        return pd.DataFrame()
    eng0 = runs_by_seed[keys[0]]["engine"]
    rows, used = [], set()
    for i in np.random.RandomState(seed).permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng0, q)
        if gold in used or not cids:
            continue
        sets = {}
        for k in keys:
            r = runs_by_seed[k]
            ex, _, _ = explain_and_score_v3(r["model"], r["graph"], gold, cids, qpred, qtok, top_m=top_m)
            sets[k] = {e["text"] for e in ex}
        if not any(sets.values()):
            continue
        js = [len(sets[x] & sets[y]) / max(len(sets[x] | sets[y]), 1) for x, y in itertools.combinations(keys, 2)]
        rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], n_configs=len(keys), mean_jaccard=float(np.mean(js)),
                         min_jaccard=float(np.min(js))))
        used.add(gold)
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)

# ---------------------------------- v4: giải thích = CHÍNH các số hạng của điểm (phân rã chính xác) ----------------------------------
def v4_edge_info(D, k):
    """(nguồn, loại, đích, vị từ) của cạnh k trong RelDataV4 — đọc lại từ đồ thị gốc."""
    g = D.graph
    s, d, e, w, pi = g.edge_arrays()
    j = int(D.gidx[k])
    preds = {}
    if pi[j] >= 0:
        row = g.pmix[pi[j]]
        preds = {PRED_VOCAB[c][1]: float(v) for c, v in zip(row.indices, row.data)}
    return dict(src=int(s[j]), rel=EDGE_TYPES[int(e[j])], dst=int(d[j]), weight=float(w[j]), predicates=preds)


def v4_path_text(D, t, vid):
    inv = D.graph.inv
    if t["kind"] == "visual":
        return f"[thị giác] {inv.name[vid]}"
    c = D.N + t["node"]
    if t["kind"] == "contains":
        return f"{inv.name[vid]} –chứa→ {inv.name[c]}"
    hops = []
    for k in t["edges"]:
        ei = v4_edge_info(D, k)
        p = max(ei["predicates"], key=ei["predicates"].get) if ei["predicates"] else ""
        hops.append((inv.name[ei["src"]], ei["rel"] + (f"[{p}]" if p else ""), inv.name[ei["dst"]]))
    chain = hops[0][0] + "".join(f" –{r}→ {b}" for _, r, b in hops)
    return f"{chain} ←chứa– {inv.name[vid]}"


def v4_edge_valid(D, k, q):
    """Cạnh có thật (đúng loại, chiều, kiểu node theo REL_SPEC); nếu truy vấn có vị từ khớp cạnh thì vị từ đó thuộc ĐÚNG cạnh."""
    ei = v4_edge_info(D, k)
    g, inv = D.graph, D.graph.inv
    ok = ei["rel"] in g.A and g.A[ei["rel"]][ei["src"], ei["dst"]] > 0
    ok = ok and inv.node_type[ei["src"]] != "video" and inv.node_type[ei["dst"]] != "video"
    ids = {PRED_VOCAB[p][1] for rel in PRED_FAMILIES for p in QPREDS[q][rel]}
    matched = ids & set(ei["predicates"])
    return bool(ok), bool(matched), bool(not matched or matched <= set(ei["predicates"]))


@torch.no_grad()
def explain_v4(model, q, vid, top_m=CFG.top_m_paths):
    """Giải thích "vì sao video được trả về" = top-m đường đi quan hệ ỦNG HỘ (đóng góp > 0); đường đi PHẢN ĐỐI (< 0) ghi riêng.
    Σ mọi số hạng (thị giác + nhãn + mọi đường đi) = điểm mô hình."""
    terms = model.decompose(q, vid)
    tot = {k: sum(t["contrib"] for t in terms if t["kind"] == k) for k in ("visual", "contains", "relation")}
    rel = [t for t in terms if t["kind"] == "relation"]
    sup = sorted([t for t in rel if t["contrib"] > 0], key=lambda t: -t["contrib"])[:top_m]
    opp = sorted([t for t in rel if t["contrib"] < 0], key=lambda t: t["contrib"])[:top_m]
    return dict(terms=terms, top_relation=sup, opposing=opp, totals=tot, score=sum(t["contrib"] for t in terms))


@torch.no_grad()
def v4_rank_after(run, part_q, gold, cand, s_clip_z, D2):
    """Thứ hạng video đúng trong top-K (điểm cuối với α đã đóng băng) khi mô hình dùng dữ liệu đồ thị D2."""
    m = run["model"]
    qb = D2.query_batch([part_q], m.use_pred)
    s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE), D2).cpu().numpy()
    fin = run["alpha"] * s_clip_z[None] + (1 - run["alpha"]) * znorm_per_row(s)
    j = int(np.where(cand == gold)[0][0])
    return int((fin[0] > fin[0, j]).sum()) + 1, s[0, j], fin[0]


def xai_v4(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0):
    """Với ≥ n truy vấn có video đúng trong top-K và có đường đi quan hệ ỦNG HỘ: (1) hợp lệ (cạnh thật, vị từ đúng cạnh), (2) độ thưa,
    (3) fidelity: xoá THẬT các cạnh của top-m đường đi giải thích ⇒ Δ điểm mô hình so với dự đoán từ phân rã (khớp = chính xác),
    so với xoá cùng số cạnh quan hệ NGẪU NHIÊN nối vào nhãn của video; (4) thứ hạng / top-1 / top-10 đổi (báo riêng)."""
    m, eng = run["model"], run["engine"]
    D = m.D
    rng = np.random.RandomState(seed)
    q_all = part["q_idx"]
    order = rng.permutation(len(q_all))
    cand_all, s_clip = eng.clip_candidates(q_all, part["gallery"], CFG.rerank_k)
    rows = []
    for i in order:
        if len(rows) >= n_queries:
            break
        q, gold = int(q_all[i]), int(part["gold"][i])
        cand = cand_all[i]
        if gold not in cand:
            continue
        ex = explain_v4(m, q, gold, top_m)
        if not ex["top_relation"]:
            continue
        zc = znorm_per_row(s_clip[i:i + 1])[0]
        r_full, s_full, _ = v4_rank_after(run, q, gold, cand, zc, D)
        e_del = sorted({k for t in ex["top_relation"] for k in t["edges"]})
        D_ex = RelDataV4(D.graph, drop_edges=D.gidx[e_del])
        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, D_ex)
        tags = set(torch.nonzero(D.Acon[gold] > 0).flatten().tolist())
        pool = [k for k in range(D.E) if int(D.dst_np[k]) in tags and k not in e_del]
        e_rand = list(rng.choice(pool, min(len(e_del), len(pool)), replace=False)) if pool else []
        D_rd = RelDataV4(D.graph, drop_edges=D.gidx[e_rand]) if e_rand else D
        r_rd, s_rd, _ = v4_rank_after(run, q, gold, cand, zc, D_rd)
        expect = sum(t["contrib"] for t in ex["terms"] if t["kind"] == "relation" and set(t["edges"]) & set(e_del))
        val = [v4_edge_valid(D, k, q) for k in e_del]
        _, _, fin_full = v4_rank_after(run, q, gold, cand, zc, D)
        rows.append(dict(q=q, gold=gold, group=groups[i], n_terms=len(ex["terms"]), n_relation_terms=sum(t["kind"] == "relation" for t in ex["terms"]),
                         n_expl_paths=len(ex["top_relation"]), n_expl_edges=len(e_del), sparsity=len(ex["top_relation"]) / max(1, sum(t["kind"] == "relation" for t in ex["terms"])),
                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[2] for v in val])),
                         n_predicate_edges=int(sum(v[1] for v in val)),
                         share_relation=ex["totals"]["relation"] / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         s_full=s_full, d_s_expl=s_ex - s_full, d_s_expected=-expect, d_s_rand=s_rd - s_full,
                         decomposition_ok=float(abs(ex["score"] - s_full) < 1e-4), deletion_matches_decomposition=float(abs((s_ex - s_full) + expect) < 1e-4) if m.T == 1 else float("nan"),
                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,
                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),
                         top10_changed_expl=float(set(np.argsort(-fin_ex)[:10]) != set(np.argsort(-fin_full)[:10])),
                         explanation=" || ".join(f"{t['contrib']:+.4f} {v4_path_text(D, t, gold)}" for t in ex["top_relation"]),
                         opposing=" || ".join(f"{t['contrib']:+.4f} {v4_path_text(D, t, gold)}" for t in ex["opposing"][:3])))
    return pd.DataFrame(rows)


@torch.no_grad()
def relation_analysis_v4(run, part, max_q=3000, seed=0):
    """Theo họ quan hệ: phần đóng góp vào điểm của video đúng, knockout lúc suy luận (bỏ mọi cạnh của họ) ⇒ ΔMRR;
    attention trung bình trên cạnh có vị từ KHỚP truy vấn vs không khớp; theo vai của node nguồn trong truy vấn."""
    m, eng = run["model"], run["engine"]
    D = m.D
    q = part["q_idx"]
    if len(q) > max_q:
        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))
    gold = CAP_VID[q]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k)
    base = metrics_from_ranks(ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], 1.0),
                                                  gold, clip_rank))["MRR"]
    rows = []
    fam_of = np.array([REL_SPEC[V4_REL_TYPES[int(t)]]["family"] for t in D.et.cpu().numpy()])
    for fam in sorted(set(fam_of)):
        D2 = RelDataV4(D.graph, drop_edges=D.gidx[np.where(fam_of == fam)[0]])
        sc2 = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k, D=D2)
        mrr = metrics_from_ranks(ranks_with_fallback(sc2["cand"], combine(sc2["s_clip"], sc2["s_gnn"], sc2["s_path"], run["alpha"], 1.0),
                                                     gold, clip_rank))["MRR"]
        rows.append(dict(section="family_knockout", split=part["name"], item=fam, n_edges=int((fam_of == fam).sum()),
                         knockout_MRR=mrr, d_MRR_knockout=mrr - base))
    D0 = RelDataV4(D.graph, drop_edges=D.gidx)
    sc0 = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k, D=D0)
    mrr0 = metrics_from_ranks(ranks_with_fallback(sc0["cand"], combine(sc0["s_clip"], sc0["s_gnn"], sc0["s_path"], run["alpha"], 1.0),
                                                  gold, clip_rank))["MRR"]
    rows.append(dict(section="family_knockout", split=part["name"], item="ALL_relations", n_edges=D.E, knockout_MRR=mrr0, d_MRR_knockout=mrr0 - base))
    att_m, att_n, by_role = [], [], defaultdict(list)
    for s0 in range(0, min(len(q), 1000), 128):
        qb = D.query_batch(q[s0:s0 + 128], m.use_pred)
        a = m.attention(qb, m.q(qb["tq"])) if m.rel and D.E else None
        if a is None:
            break
        pe = (D.pidx_np >= 0)
        exm = qb["ex"].cpu().numpy() > 0
        A = a.cpu().numpy()
        att_m.extend(A[exm & pe[None]].tolist()); att_n.extend(A[~exm & pe[None]].tolist())
        rs = qb["role_src"][:, D.src].cpu().numpy()
        for r in range(len(Q_ROLES)):
            if (rs == r).any():
                by_role[Q_ROLES[r]].extend(A[rs == r].tolist())
    rows.append(dict(section="predicate_attention", split=part["name"], item="pred_edges",
                     att_pred_match=float(np.mean(att_m)) if att_m else float("nan"),
                     att_pred_nomatch=float(np.mean(att_n)) if att_n else float("nan"), n_match=len(att_m)))
    for r, v in by_role.items():
        rows.append(dict(section="role_attention", split=part["name"], item=r, att_mean=float(np.mean(v)), n=len(v)))
    return pd.DataFrame(rows), dict(base_MRR=base, all_relations_knockout_MRR=mrr0,
                                    att_pred_match=float(np.mean(att_m)) if att_m else float("nan"),
                                    att_pred_nomatch=float(np.mean(att_n)) if att_n else float("nan"))


## 4.13 Khung thí nghiệm
Mỗi thí nghiệm có: `experiment_id`, stage, giả thuyết, thiết kế đầy đủ, seed, checkpoint fp16, số cạnh theo loại, số node, số tham số, (α, λ) chọn trên val, thời gian chạy, đỉnh bộ nhớ GPU, trạng thái. Thí nghiệm không chạy được ghi `NOT RUN` kèm lý do.

`decide()` áp luật đăng ký trước trên **val**; incumbent là phương án đơn giản hơn. `PRIMARY_COMPARISONS` là họ so sánh chính ở mục 4.2.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
DESIGN_BASE = dict(taxonomy="heuristic", contains="binary", conf_mode="sigmoid", rel_policy="all", path_mode="weighted",
                   mp_weight="binary", init="learned", mask="none", rewire=False, pred_shuffle=False, pred_mode="none",
                   related="base", protocol="inductive", tags="base",
                   model="v3", v4_layers=1, v4_rel=True, v4_qcond=True, v4_pred=True)        # model: v3 (TypedSAGE) | v4 (QC-RGAT)
V3_FINAL = dict(DESIGN_BASE, taxonomy="none", mask="no_related")   # cấu hình cuối của v3 (E44, chọn trên VAL ở Kaggle) = đồ thị của v4
RELATED_CFGS = {"base": (CFG.cooc_min_count, CFG.cooc_min_npmi), "npmi20": (CFG.cooc_min_count, 0.20),
                "count10": (10, CFG.cooc_min_npmi)}
GRAPHS, EXPS, DECISIONS, NOT_RUN = {}, {}, [], []
REWIRE_SEED_BASE, SHUFFLE_SEED_BASE = 1000, 2000
VERIFIED_CELLS = None            # {(họ, mức phổ biến)} đạt ngưỡng precision — chỉ có khi có nhãn người chấm


def masked_types(mask):
    """mask = tên trong MASKS, có thể ghép bằng '+' (ví dụ 'no_related+no_taxonomy')."""
    return tuple(sorted({t for m in str(mask).split("+") for t in MASKS[m]}))


def add_mask(mask, extra):
    parts = sorted({m for m in str(mask).split("+") + [extra] if m != "none"})
    return "+".join(parts) or "none"


def graph_key(d, seed):
    return (d["taxonomy"], d["contains"], d["conf_mode"] if d["contains"] == "conf" else "", d["rel_policy"],
            d["path_mode"], d["mask"], (REWIRE_SEED_BASE + seed) if d["rewire"] else None,
            (SHUFFLE_SEED_BASE + seed) if d["pred_shuffle"] else None, d["tags"], d["related"])


def graph_for(d, seed):
    key = graph_key(d, seed)
    if key not in GRAPHS:
        tags = ALT_TAGS[d["tags"]] if d["tags"] != "base" else None
        scores = ALT_SCORES[d["tags"]] if d["tags"] != "base" else TAG_SCORES
        g = GraphV3(INV, RECORDS, SPLIT["train"], RELATIONS2, TAXONOMY[d["taxonomy"]], tag_scores=scores,
                    contains=d["contains"], conf_mode=d["conf_mode"], conf_cal=CONF_CAL, rel_policy=d["rel_policy"],
                    path_mode=d["path_mode"], masked=masked_types(d["mask"]), rewire_seed=key[6], pred_shuffle_seed=key[7],
                    tags_override=tags, related_cfg=RELATED_CFGS[d["related"]], verified_cells=VERIFIED_CELLS,
                    name="|".join(str(x) for x in key))
        assert g.n_nodes == INV.n_nodes and g.inv is INV, "inventory phải cố định"
        GRAPHS[key] = (g, EngineV3(g, VID_FEAT, TXT_FEAT, QTOK2, QPREDS))
    return GRAPHS[key]


def make_part(name, std_only=False):
    """Tập truy vấn + gallery. 'test' (MSVD, thăm dò) và 'ext' (xác nhận) chỉ mở sau khi đóng băng.
    std_only: chỉ caption chuẩn 1k-A của tập ngoài (truy vấn PHỤ)."""
    if name in ("test", "ext"):
        TEST_GATE.check(f"make_part({name})", name)
    q = np.where((CAP_SPLIT == name) & (CAP_STD if std_only else True))[0]
    return dict(name=name + ("_std" if std_only else ""), split=name, q_idx=q, gold=CAP_VID[q], gallery=np.array(SPLIT[name]))


def _mean_std(vals):
    return float(np.mean(vals)), float(np.std(vals))


def not_run(exp_id, stage, desc, reason, design=None, hypothesis=""):
    """Ghi thí nghiệm KHÔNG chạy được (kèm lý do) vào manifest — không tạo số liệu thay thế."""
    EXPS[exp_id] = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=design, seeds=[],
                        runs={}, status="NOT RUN", reason=reason, val_MRR=(float("nan"), float("nan")),
                        val_R1=(float("nan"), float("nan")), edges={}, n_params=0, seconds=0.0)
    NOT_RUN.append(dict(stage=stage, experiment_id=exp_id, reason=reason))
    print(f"  [{stage}] {exp_id:28s} NOT RUN — {reason}")
    return EXPS[exp_id]


def run_exp(exp_id, stage, design, desc, hypothesis="", seeds=None, lam_fixed=None, reuse=None):
    """Huấn luyện (hoặc tái dùng mô hình của `reuse`) cho từng seed; chọn (α, λ) trên VAL; ghi manifest."""
    seeds = seeds or CFG.seeds
    rec = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=dict(design),
               seeds=list(seeds), lam_fixed=lam_fixed, reuse=reuse, status="RUNNING", runs={})
    EXPS[exp_id] = rec
    for seed in seeds:
        g, eng = graph_for(design, seed)
        t0 = time.time()
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
        if reuse is not None:
            model, hist = EXPS[reuse]["runs"][seed]["model"], EXPS[reuse]["runs"][seed]["hist"]
        elif design.get("model") == "v4":
            model, hist = train_v4(eng, TRAIN_Q, design, seed=seed)
        else:
            model, hist = train_graph_scorer_v3(eng, TRAIN_Q, SYM_FEAT, seed=seed, mp_weight=design["mp_weight"],
                                                init=design["init"], pred_mode=design["pred_mode"])
        lam_v = 1.0 if design.get("model") == "v4" else lam_fixed           # v4: một điểm đồ thị duy nhất (s_path = 0)
        (a, l), grid, best = tune_on_val_v3(model, eng, PV, CFG.rerank_k, CLIP_RANK_VAL,
                                            lam_grid=[lam_v] if lam_v is not None else None, protocol=design["protocol"])
        ck = None
        if CFG.save_checkpoints and reuse is None:
            ck = CFG.out_dir / "checkpoints" / f"{exp_id}_s{seed}.pt"
            torch.save({k_: v.half() if v.is_floating_point() else v for k_, v in model.state_dict().items()}, ck)
        rec["runs"][seed] = dict(graph=g, engine=eng, model=model, hist=hist, alpha=a, lam=l, grid=grid,
                                 val={k_: best[k_] for k_ in ("R@1", "R@5", "R@10", "MRR")}, rr_val=1.0 / best["rank"],
                                 seconds=round(time.time() - t0, 1), checkpoint=str(ck) if ck else None,
                                 gpu_peak_mb=round(torch.cuda.max_memory_allocated() / 2 ** 20, 1) if DEVICE == "cuda" else None)
    r0 = rec["runs"][seeds[0]]
    rec.update(status="DONE", val_MRR=_mean_std([r["val"]["MRR"] for r in rec["runs"].values()]),
               val_R1=_mean_std([r["val"]["R@1"] for r in rec["runs"].values()]),
               rr_val=np.mean([r["rr_val"] for r in rec["runs"].values()], 0),
               edges=r0["graph"].edge_counts(), n_nodes=r0["graph"].n_nodes,
               n_params=int(sum(p.numel() for p in r0["model"].parameters())),
               alpha_lam=[(r["alpha"], r["lam"]) for r in rec["runs"].values()],
               seconds=float(sum(r["seconds"] for r in rec["runs"].values())),
               gpu_peak_mb=max([r["gpu_peak_mb"] or 0 for r in rec["runs"].values()]) or None)
    m, s = rec["val_MRR"]
    print(f"  [{stage}] {exp_id:28s} VAL MRR {m:.4f} ± {s:.4f} | α,λ {rec['alpha_lam']} | "
          f"{rec['seconds']:.0f}s | cạnh quan hệ {sum(v for k_, v in rec['edges'].items() if not k_.startswith('contains'))}")
    return rec


def design_value(d, field):
    return "/".join(str((d or {}).get(f)) for f in field.split("|"))


def decide(stage, incumbent, candidates, field):
    """Luật đăng ký trước: incumbent = phương án ĐƠN GIẢN hơn (hoặc thiết kế đang dùng). Ứng viên chỉ được chấp nhận khi
    Δ MRR val (trung bình seed) ≥ adopt_min_delta VÀ p(Δ ≤ 0) — bootstrap theo video, hiệu chỉnh Holm trên các ứng viên
    của stage — < adopt_max_p. Nhiều ứng viên đạt ⇒ chọn Δ lớn nhất. Ứng viên NOT RUN bị bỏ qua (ghi lại)."""
    base = EXPS[incumbent]
    cands = [c for c in candidates if EXPS[c]["status"] == "DONE"]
    rows = []
    for c in cands:
        e = EXPS[c]
        d = e["val_MRR"][0] - base["val_MRR"][0]
        bs = cluster_bootstrap_diff(e["rr_val"], base["rr_val"], PV["gold"])
        rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=design_value(e["design"], field),
                         delta_val_MRR=d, ci_lo=bs["ci_lo"], ci_hi=bs["ci_hi"], p_not_better=bs["p_not_better"]))
    adj = holm([r["p_not_better"] for r in rows])
    best = None
    for r, pa in zip(rows, adj):
        r["p_holm"] = float(pa)
        r["passes"] = bool(r["delta_val_MRR"] >= CFG.adopt_min_delta and pa < CFG.adopt_max_p)
        if r["passes"] and (best is None or r["delta_val_MRR"] > best[1]):
            best = (r["candidate"], r["delta_val_MRR"])
    for r in rows:
        r["adopted"] = best is not None and r["candidate"] == best[0]
    for c in candidates:
        if EXPS[c]["status"] != "DONE":
            rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=None, delta_val_MRR=float("nan"),
                             ci_lo=float("nan"), ci_hi=float("nan"), p_not_better=float("nan"), p_holm=float("nan"),
                             passes=False, adopted=False, note=f"NOT RUN: {EXPS[c].get('reason', '')}"))
    DECISIONS.extend(rows)
    winner = best[0] if best else incumbent
    print(f"  → {stage}: {'CHẤP NHẬN ' + winner if best else 'giữ ' + incumbent} ({field} = {design_value(EXPS[winner]['design'], field)})")
    return winner


@torch.no_grad()
def evaluate_exp(exp_id, part, clip_rank, protocol=None):
    """Đánh giá MỘT lần mỗi seed với (α, λ) đã đóng băng; trả về thứ hạng theo truy vấn."""
    e = EXPS[exp_id]
    out = {}
    for seed, r in e["runs"].items():
        sc = score_queries_v3(r["model"], r["engine"], part["q_idx"], part["gallery"], CFG.rerank_k,
                              protocol or e["design"]["protocol"])
        fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], r["alpha"], r["lam"])
        out[seed] = dict(rank=ranks_with_fallback(sc["cand"], fin, part["gold"], clip_rank),
                         cand_hash=hashlib.sha1(sc["cand"].tobytes()).hexdigest()[:12])
    return out


def query_groups_v3(engine, q_idx):
    """direct | indirect | other (như v1/v2) trên đồ thị của engine."""
    g, q_idx = engine.graph, np.asarray(q_idx)
    gold = CAP_VID[q_idx]
    pf = engine.path_feats(gold[:, None], q_idx)[:, 0]
    out = []
    for j, q in enumerate(q_idx):
        cids = {n_ for n_, _, _ in engine.q_concepts[q]}
        out.append("direct" if cids & set(g.vid_tags[gold[j]]) else ("indirect" if pf[j, 1:].sum() > 0 else "other"))
    return np.array(out)


def rare_mask(engine, q_idx, thr=5):
    """Truy vấn 'khái niệm hiếm': mọi node lớp khớp được có tần suất gắn nhãn trên video TRAIN ≤ thr (hoặc chỉ khớp khái niệm)."""
    g = engine.graph
    out = []
    for q in q_idx:
        cls = [n_ for n_, _, _ in engine.q_concepts[q] if g.inv.is_class(n_)]
        out.append(bool(engine.q_concepts[q]) and (not cls or max(g.cls_df[c] for c in cls) <= thr))
    return np.array(out)


def predicate_groups(engine, q_idx, thr=None):
    """Chỉ xét vị từ của truy vấn CÓ trên cạnh của đồ thị (svo/prep): 'no_pred' nếu không có; 'pred_rare' nếu MỌI vị từ khớp
    đều có support TRAIN ≤ thr (truy vấn chỉ dựa vào vị từ hiếm); ngược lại 'pred_common'."""
    thr = thr or 2 * CFG.pred_min_support
    used = {pid for rel in PRED_FAMILIES for d in engine.graph.pshare[rel].values() for pid in d}
    out = []
    for q in q_idx:
        pids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel] if p in used]
        if not pids:
            out.append("no_pred")
        else:
            out.append("pred_rare" if max(PRED_SUPPORT.get(PRED_VOCAB[p], 0) for p in pids) <= thr else "pred_common")
    return np.array(out)


@torch.no_grad()
def path_analysis(run, part, protocol="inductive", max_q=4000, seed=0):
    """Theo mẫu: phần đóng góp, knockout từng mẫu và từng HỌ quan hệ lúc suy luận; theo node: bậc, số đường đi đi qua,
    phạm vi video; tổng hợp: ρ(degree, s_path), ρ(degree, điểm cuối), P(degree video đúng > video âm)."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    q = part["q_idx"]
    if len(q) > max_q:
        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))
    gold = CAP_VID[q]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v3(model, eng, q, part["gallery"], CFG.rerank_k, protocol)
    pf = eng.path_feats(sc["cand"], q)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    contrib = gain * pf * w
    tot = contrib.sum(-1, keepdims=True)
    share = np.where(tot > 0, contrib / np.maximum(tot, 1e-12), 0).mean((0, 1))
    reldeg_out = sum(np.asarray((g.A[t] > 0).sum(1)).ravel() for t in PATH_TYPES)
    reldeg_in = sum(np.asarray((g.A[t] > 0).sum(0)).ravel() for t in PATH_TYPES)
    con_in = np.asarray((g.A_con > 0).sum(0)).ravel()
    vdeg = np.array([reldeg_out[g.vid_tags[v]].sum() for v in range(g.n_vid)])
    cand_deg = vdeg[sc["cand"]]
    from scipy.stats import spearmanr
    is_gold = (sc["cand"] == gold[:, None])
    base_fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], run["lam"])
    base_mrr = metrics_from_ranks(ranks_with_fallback(sc["cand"], base_fin, gold, clip_rank))["MRR"]

    def knock(idx):
        pf2 = pf.copy(); pf2[..., idx] = 0
        sp2 = znorm_per_row(gain * (pf2 * w).sum(-1))
        return metrics_from_ranks(ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sp2, run["alpha"], run["lam"]),
                                                      gold, clip_rank))["MRR"]

    rows = []
    for p_i, p in enumerate(PATTERNS):
        mrr = knock([p_i])
        reach = np.asarray((g.R[p] > 0).sum(0)).ravel()
        rows.append(dict(section="pattern", split=part["name"], item=p, weight=float(w[p_i]), nnz=int(g.R[p].nnz),
                         frac_pairs_nonzero=float((pf[..., p_i] > 0).mean()), contribution_share=float(share[p_i]),
                         knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr, max_video_reach=int(reach.max()) if reach.size else 0))
    groups = {f"family:{f}": [PIDX[p] for p in PATTERNS if p.startswith(f"P1_{f}") or p.startswith(f"P4_{f}")
                              or p == f"P1m_{FAMILY_REL[f]}"] for f in FAMILIES}
    groups.update({"family:related": [PIDX["P3_related"], PIDX["P5_related_isa"]],
                   "family:isa": [PIDX["P2_isa"]] + [PIDX[f"P4_{f}_isa"] for f in FAMILIES] + [PIDX["P5_related_isa"]],
                   "all_indirect": list(range(1, len(PATTERNS)))})
    for nm, idx in groups.items():
        mrr = knock(idx)
        rows.append(dict(section="family_knockout", split=part["name"], item=nm, knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr,
                         contribution_share=float(share[idx].sum())))
    reach_all = sum(np.asarray((g.R[p] > 0).sum(0)).ravel()[:g.n_nodes] for p in PATTERNS[1:] if not p.startswith("P1m_"))
    through = con_in * reldeg_out                         # số đường đi 2 bước video → node → lớp/khái niệm đi QUA node
    for i in np.argsort(-through)[:50]:
        if through[i] <= 0:
            break
        rows.append(dict(section="node", split=part["name"], item=g.inv.name[i], node_type=g.inv.semantic_type(i),
                         videos_tagged=int(con_in[i]), rel_out_degree=int(reldeg_out[i]), rel_in_degree=int(reldeg_in[i]),
                         paths_through=int(through[i]), videos_reached_indirect=int(reach_all[i]), idf=float(g.idf[i])))
    flat_deg, flat_sp, flat_fin = cand_deg.ravel(), sc["path_raw"].ravel(), base_fin.ravel()
    gold_rank = ranks_with_fallback(sc["cand"], base_fin, gold, clip_rank)
    gold_deg = vdeg[gold]
    deg_auc = float(np.mean([(cand_deg[k][is_gold[k]] > cand_deg[k][~is_gold[k]]).mean()
                              for k in range(len(q)) if is_gold[k].any()])) if is_gold.any() else float("nan")
    summary = dict(split=part["name"], n_queries=len(q), base_MRR=base_mrr,
                   spearman_degree_vs_s_path=float(spearmanr(flat_deg, flat_sp).correlation),
                   spearman_degree_vs_final=float(spearmanr(flat_deg, flat_fin).correlation),
                   spearman_gold_degree_vs_gold_rank=float(spearmanr(gold_deg, gold_rank).correlation),
                   p_gold_degree_gt_negative=deg_auc, w_path_effective=(1 - run["alpha"]) * (1 - run["lam"]),
                   hubs=[dict(node=g.inv.name[i], paths_through=int(through[i])) for i in np.argsort(-through)[:10]])
    rows.append(dict(section="summary", split=part["name"], item="degree_bias",
                     **{k: v for k, v in summary.items() if k not in ("hubs", "split")}))
    return pd.DataFrame(rows), summary


# ---- họ so sánh CHÍNH (đăng ký trước, một phía: a > b), hiệu chỉnh Holm; chỉ trên tập ngoài mới là XÁC NHẬN ----
PRIMARY_COMPARISONS = [("H1_graph_vs_clip", "G3", "G0"), ("H2_relations_vs_contains_only", "G3", "G1"),
                       ("H3_predicate_vs_family_only", "G3", "G2"), ("H4_true_vs_shuffled_predicates", "G3", "G4"),
                       ("H5_true_vs_rewired_topology", "G3", "G5"), ("H6_query_conditioned_vs_static", "G3", "G6"),
                       ("H7_v4_vs_v3", "G3", "V3REF"),
                       ("H8_role_edges", "G3", "G7"), ("H9_svo_prep_edges", "G3", "G8")]
H_V4 = ("quan hệ được dùng CÓ ĐIỀU KIỆN THEO TRUY VẤN (loại cạnh, vị từ khớp truy vấn, vai của node trong truy vấn) cải thiện "
        "retrieval so với cùng mô hình không có quan hệ / quan hệ sai")

## 4.14 Self-test v3 (mục 12)
| mục 12 | kiểm tra | id |
|---|---|---|
| 1 | embedding vị từ tác động forward; pred_mode none (≡ v2) **không** bị ảnh hưởng | P1 |
| 2 | hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác | P2 |
| 3 | rewiring giữ kiểu node, họ, bậc, trọng số, đa tập vị từ | G6 |
| 4 | hoán vị vị từ không đổi topology / trọng số | G7 |
| 5 | mỗi ablation tắt đúng thành phần; inventory cố định | AB, G1 |
| 6 | trọng số chỉ nhân một lần (khớp tính tay, bất biến tỉ lệ) | G4 |
| 7 | node không có hàng xóm không sinh NaN; cô lập inductive | G4, G5 |
| 8 | không rò rỉ caption train/val/test/tập ngoài | C1, C1b, C2, C3, C8, C9 |
| 9 | giải thích đúng vị từ, đúng chiều; điểm khớp ma trận | G3, X1 |
| 10 | số trong báo cáo khớp CSV đầu ra | R1 |

Một kiểm thử phần mềm PASS **không** có nghĩa giả thuyết khoa học đã được xác nhận.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
CANARY_NOUNS = ["flamingo", "accordion", "kayak", "harmonica", "trombone", "anvil", "walrus", "xylophone",
                "gondola", "hedgehog", "unicycle", "periscope", "tuba", "iguana", "llama"]
CANARY_VERBS = ["juggle", "knit", "yodel", "whittle", "embroider", "lasso"]


def _same_graph(g1, g2):
    if g1.n_nodes != g2.n_nodes or g1.inv.sha1 != g2.inv.sha1:
        return False
    if (g1.A_con != g2.A_con).nnz:
        return False
    return (all((g1.A[t] != g2.A[t]).nnz == 0 for t in RELATION_TYPES) and np.array_equal(g1.idf, g2.idf)
            and g1.pdist == g2.pdist)


def _pred_vocab_of(rel):
    s = Counter()
    for v in SPLIT["train"]:
        s.update({(r, p) for _, r, _, p in rel[v]})
    return sorted(k for k in s if k[0] in PRED_FAMILIES)


def test_canary(splits=("test", "ext"), n_videos=40):
    """Canary chỉ có trong caption TEST / tập ngoài không được lọt vào từ vựng, quan hệ, từ vựng vị từ; đối chứng dương:
    đặt canary vào TRAIN thì phải phát hiện được."""
    terms = {k.rsplit("#", 1)[0] for k in VOCAB2["terms"]}
    a, b = [w for w in CANARY_NOUNS if w not in terms and is_visual_noun(w)][:2]
    seen_pred = {t for _, p in PRED_VOCAB for t in p.split()}           # động từ canary cũng không được là vị từ train
    v = next((w for w in CANARY_VERBS if w not in terms and w not in seen_pred and is_action_verb(w)),
             next(w for w in CANARY_VERBS if w not in terms and is_action_verb(w)))
    canary = [f"a {a} is {naive_gerund(v)} a {b}"] * 3
    tv = [x for s in splits for x in SPLIT[s][:n_videos]]
    store = CAPS.replaced({x: CAPS._caps[x] + canary for x in tv})
    voc = build_vocab_v2(store, SPLIT)
    rel = build_relations_v2(store, SPLIT, voc)
    keys = {ckey(a, "n"), ckey(b, "n"), ckey(v, "v")}
    leak_v = sorted(keys & set(voc["terms"]))
    leak_r = [t for trip in rel.values() for t in trip if {t[0], t[2]} & keys]
    leak_p = sorted(set(_pred_vocab_of(rel)) - set(PRED_VOCAB))          # vị từ MỚI so với từ vựng vị từ train
    fake = dict(SPLIT, train=list(SPLIT["train"]) + list(tv))
    store_pos = store.replaced({}, new_split={x: "train" for x in tv})
    voc_pos = build_vocab_v2(store_pos, fake)
    rel_pos = build_relations_v2(store_pos, fake, voc_pos)
    detected = {ckey(a, "n"), ckey(b, "n")} <= set(voc_pos["terms"]) and any({t[0], t[2]} & keys for trip in rel_pos.values() for t in trip)
    return dict(canary=f"{a} –{v}→ {b}", splits=list(splits), n_videos=len(tv), leak_vocab=leak_v, leak_relations=leak_r[:3],
                leak_predicates=leak_p[:3], positive_control_detected=bool(detected))


def test_invariance():
    """Xáo / thay caption TEST và tập ngoài rồi dựng lại TOÀN BỘ: từ vựng, quan hệ, vị từ, tagger, taxonomy, đồ thị."""
    tv = list(SPLIT["test"]) + list(SPLIT["ext"])
    perm = np.random.RandomState(0).permutation(tv)
    junk = ["a purple elephant is skiing on the moon", "zebra piano astronaut volcano juggling"] * 4
    store = CAPS.replaced({x: (CAPS._caps[int(p)] if k % 2 == 0 else junk) for k, (x, p) in enumerate(zip(tv, perm))})
    voc = build_vocab_v2(store, SPLIT)
    rel = build_relations_v2(store, SPLIT, voc)
    tags, scores, sel, _ = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL["n_frames"], voc, SPLIT, store)
    tax, cmeta, _, _ = build_taxonomy_v2(voc, store, SPLIT, {v: tags[v] for v in SPLIT["train"]})
    recs = [dict(r, tags=tags[r["vid"]]) for r in RECORDS]
    inv = NodeInventory(recs, voc, cmeta)
    g2 = GraphV3(inv, recs, SPLIT["train"], rel, tax["heuristic"], tag_scores=scores)
    g1 = graph_for(DESIGN_BASE, CFG.seeds[0])[0]
    return dict(vocab=voc["terms"] == VOCAB2["terms"], relations=rel == RELATIONS2, predicate_vocab=_pred_vocab_of(rel) == PRED_VOCAB,
                tagger_selection=sel == TAG_SELECT2, tags=tags == TAGS2, taxonomy=tax == TAXONOMY, inventory=inv.sha1 == INV.sha1,
                graph=_same_graph(g1, g2), annotation_sample=build_annotation_sample(store, SPLIT, voc, RECORDS)[0]
                .sample_id.tolist() == ANN_SAMPLE.sample_id.tolist(),
                automatic_verification=(build_agreement(store, SPLIT, voc)[0] == AGREE) if AGREE else True)


VARIANT_DESIGNS = ([dict(DESIGN_BASE, mask=m) for m in MASKS] + [dict(DESIGN_BASE, rewire=True), dict(DESIGN_BASE, pred_shuffle=True),
                   dict(DESIGN_BASE, rel_policy="support"), dict(DESIGN_BASE, taxonomy="none"), dict(DESIGN_BASE, contains="conf"),
                   dict(DESIGN_BASE, related="npmi20"), dict(DESIGN_BASE, path_mode="pred_match")])


def test_inventory_constant():
    """Mọi biến thể đồ thị dùng CÙNG inventory; với CÙNG cấu hình mô hình, số tham số không đổi giữa các biến thể đồ thị."""
    gs = [graph_for(d, CFG.seeds[0]) for d in VARIANT_DESIGNS]
    params = {pm: sorted({sum(p.numel() for p in make_model(e, SYM_FEAT, pred_mode=pm).parameters()) for _, e in gs[:4]})
              for pm in PRED_MODES}
    return dict(n_graphs=len(gs), node_counts=sorted({g.n_nodes for g, _ in gs}), same_inventory=all(g.inv is INV for g, _ in gs),
                param_counts_by_pred_mode=params)


def test_schema_conformance(g):
    """Mọi cạnh: kiểu nút đầu/cuối + POS đúng REL_SPEC; không self-loop; trọng số trong (0, 1]; vị từ đúng họ:
    svo/prep ⇒ mọi vị từ thuộc từ vựng vị từ của họ đó (không lẫn tên loại quan hệ); agent/object ⇒ vị từ = lemma động từ."""
    bad = Counter()
    inv = g.inv
    fam_names = set(REL_SPEC) | set(FAMILIES)
    for t in ["contains"] + RELATION_TYPES:
        M = (g.A_con if t == "contains" else g.A[t]).tocoo()
        spec = REL_SPEC[t]
        base = t[:-4] if t.endswith("_inv") else t
        for u, v, w in zip(M.row.tolist(), M.col.tolist(), M.data.tolist()):
            src_ok = spec["src"].split(":")[0].split("|")
            tgt_ok = spec["tgt"].split(" ")[0].split(":")[0].split("|")
            if inv.node_type[u] not in src_ok or inv.node_type[v] not in tgt_ok:
                bad[f"{t}:type"] += 1
            if ":" in spec["src"] and inv.semantic_type(u) != spec["src"].split(":")[1]:
                bad[f"{t}:src_pos"] += 1
            if ":" in spec["tgt"] and inv.semantic_type(v) != spec["tgt"].split(":")[1]:
                bad[f"{t}:tgt_pos"] += 1
            if t in ("isa", "isa_inv") and inv.pos[u] != inv.pos[v]:
                bad[f"{t}:pos_mismatch"] += 1
            if u == v:
                bad[f"{t}:self_loop"] += 1
            if not (0 < w <= 1 + 1e-6):
                bad[f"{t}:weight"] += 1
            if base in REL_CATS:
                d = g.predicates_of(t, u, v)
                if not d:
                    bad[f"{t}:no_predicate"] += 1
                if any(p in fam_names for p in d):
                    bad[f"{t}:predicate_is_relation_name"] += 1
                if base in PRED_FAMILIES and any((base, p) not in PRED_ID for p in d):
                    bad[f"{t}:predicate_not_in_vocab"] += 1
                if base in ("agent_of", "object_of"):
                    verb = inv.lemma[v if not t.endswith("_inv") else u]
                    if set(d) != {verb}:
                        bad[f"{t}:predicate_not_verb_node"] += 1
    return dict(violations=dict(bad), n_checked=int(g.A_con.nnz + sum(g.A[t].nnz for t in RELATION_TYPES)))


def test_path_identity(n=200):
    """Điểm ma trận (engine) = tổng/max liệt kê đường đi ở MỌI path_mode (kể cả pred_match); tính lại R khớp graph.R;
    mọi đường đi khớp vị từ ghi đúng vị từ của đúng cạnh theo đúng chiều; đối chứng âm: sửa pred_id ⇒ bị phát hiện."""
    out = {}
    rng = np.random.RandomState(0)
    pool = np.where(np.isin(CAP_SPLIT, ["train", "val"]))[0]
    qs = rng.choice(pool, min(n, len(pool)), replace=False)
    pairs = [(q, int(CAP_VID[q])) for q in qs] + [(q, int(v)) for q in qs for v in rng.choice(SPLIT["train"], 2)]
    for mode in ("weighted", "count", "degnorm", "dedup", "pred_match"):
        g, eng = graph_for(dict(DESIGN_BASE, path_mode=mode), CFG.seeds[0])
        bad = cnt = npred = pbad = neg = 0
        for q, v in pairs:                               # video đúng + video ngẫu nhiên (có nhiều đường đi gián tiếp hơn)
            cids, qpred, qtok = q_args(eng, q)
            if not cids:
                continue
            cnt += 1
            paths = enumerate_paths_v3(g, v, cids, qpred)
            bad += int(not np.allclose(eng.path_feats(np.array([[v]]), np.array([q]))[0, 0],
                                       paths_to_counts_v3(paths, mode), atol=1e-5))
            for p in paths:
                if p.get("pred") is not None:
                    npred += 1
                    pbad += int(not pred_valid_v3(g, p, v, qtok))
                    wrong = dict(p, pred=(p["pred"][0], (p["pred"][1] + 1) % len(PRED_VOCAB), p["pred"][2]))
                    neg += int(not pred_valid_v3(g, wrong, v, qtok))
        R2 = g.compute_R(rows=np.arange(g.n_vid), PM=g.path_mats())
        diff = max(float(abs(R2[p] - g.R[p]).max()) if (R2[p] - g.R[p]).nnz else 0.0 for p in PATTERNS)
        out[mode] = dict(pairs=cnt, mismatches=bad, recompute_maxdiff=diff, predicate_paths=npred, predicate_invalid=pbad,
                         negative_control_detected=neg)
    return out


def test_mp_weights():
    """Unit test TypedSAGEv3: trọng số cạnh dùng trong forward (weighted), bị bỏ qua (binary), w=0 ≡ không cạnh, node cô lập
    hữu hạn, bất biến thứ tự cạnh, gradient hữu hạn; trọng số chỉ nhân MỘT lần: (i) khớp công thức tính tay, (ii) weighted_mean
    bất biến khi nhân mọi trọng số với hằng số (nhân lặp sẽ làm đổi), (iii) cùng hệ số cho h_u và vị từ e_uv."""
    torch.manual_seed(0)
    res = {}
    x = torch.randn(4, 8)
    ei = torch.tensor([[1, 2, 3], [0, 0, 0]])
    et = torch.zeros(3, dtype=torch.long)
    pidx = torch.tensor([0, 1, 2])
    ph = torch.randn(3, 8)
    for mode in ("binary", "weighted_mean", "weighted_symnorm"):
        L = TypedSAGEv3(8, 8, 1, [0], 0.0, mode, pred_type_ids=[0])
        w1 = torch.tensor([1.0, 1.0, 1.0]); w2 = torch.tensor([1.0, 0.0, 1.0]); w3 = torch.tensor([3.0, 1.0, 1.0])
        o1, o2, o3 = L(x, ei, et, w1, pidx, ph), L(x, ei, et, w2, pidx, ph), L(x, ei, et, w3, pidx, ph)
        perm = torch.tensor([2, 0, 1])
        o_perm = L(x, ei[:, perm], et[perm], w3[perm], pidx[perm], ph)
        o_drop = L(x, ei[:, [0, 2]], et[[0, 2]], torch.tensor([1.0, 1.0]), pidx[[0, 2]], ph)
        o_scaled = L(x, ei, et, 5 * w3, pidx, ph)
        ww = torch.ones(3) if mode == "binary" else w3
        coef = ww / ww.sum() if mode != "weighted_symnorm" else ww / torch.sqrt(ww * ww.sum())
        manual = L.self_w(x[0]) + torch.sigmoid(L.gate[0]) * (L.neigh_w[0]((coef[:, None] * x[1:4]).sum(0))
                                                               + L.pred_w["0"]((coef[:, None] * ph).sum(0)))
        xg = x.clone().requires_grad_(True)
        L(xg, ei, et, w3, pidx, ph).sum().backward()
        changes = bool((o1[0] - o3[0]).abs().max() > 1e-6)
        zero_ok = bool((o2[0] - o_drop[0]).abs().max() < 1e-5)
        res[mode] = dict(weight_changes_output=changes, zero_weight_equals_edge_removed=zero_ok,
                         isolated_nodes_finite=bool(torch.isfinite(o1[1:]).all()),
                         edge_order_invariant=bool((o3 - o_perm).abs().max() < 1e-5),
                         grad_finite=bool(torch.isfinite(xg.grad).all()),
                         matches_formula=bool((o3[0] - manual).abs().max() < 1e-5),
                         scale_invariant=bool((o3[0] - o_scaled[0]).abs().max() < 1e-5),
                         as_designed=((not changes) if mode == "binary" else (changes and zero_ok))
                         and bool((o3[0] - manual).abs().max() < 1e-5)
                         and (mode != "weighted_mean" or bool((o3[0] - o_scaled[0]).abs().max() < 1e-5)))
    return res


def test_predicate_forward():
    """(1) Embedding vị từ thật sự tác động forward: đổi embedding của các vị từ đang dùng ⇒ biểu diễn node đổi khi
    pred_mode ∈ {clip, clip_adapt}; pred_mode = none (relation-family-only, ≡ v2) PHẢI không đổi (đối chứng âm).
    (2) Hai vị từ khác nhau trên CÙNG cặp node tạo thông điệp khác nhau (trường hợp kiểm soát, embedding CLIP thật)."""
    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])
    used = sorted({pid for rel in PRED_FAMILIES for d in g.pshare[rel].values() for pid in d})
    out = {}
    for pm in PRED_MODES:
        torch.manual_seed(0)
        model = make_model(eng, SYM_FEAT, pred_mode=pm).eval()
        if pm == "clip_adapt":                         # MLP dư khởi tạo 0 ⇒ cho trọng số khác 0 để kiểm tra cả nhánh dư
            with torch.no_grad():
                for prm in model.pred_enc.res.parameters():
                    prm.normal_(0, 0.05)
        with torch.no_grad():
            h1 = model.encode_graph(*eng.edges_on("train"))
            if pm != "none" and used:
                old = model.pred_emb.clone()
                model.pred_emb[used] = torch.roll(model.pred_emb[used], 1, 0)
                h2 = model.encode_graph(*eng.edges_on("train"))
                model.pred_emb.copy_(old)
            else:
                ed = list(eng.edges_on("train"))
                ed[3] = torch.full_like(ed[3], -1)                       # gỡ mọi chỉ số vị từ
                h2 = model.encode_graph(*ed)
        out[pm] = dict(max_change=float((h1 - h2).abs().max()), finite=bool(torch.isfinite(h1).all()))
    out["predicate_used_as_designed"] = (out["clip"]["max_change"] > 1e-5 and out["clip_adapt"]["max_change"] > 1e-5
                                         and out["none"]["max_change"] < 1e-7)
    pick = {}
    for want in ("ride", "feed", "play", "hold"):
        if ("svo", want) in PRED_ID:
            pick[want] = PRED_ID[("svo", want)]
    pair = list(pick.items())[:2] if len(pick) >= 2 else [(PRED_VOCAB[i][1], i) for i in used[:2]]
    torch.manual_seed(0)
    L = TypedSAGEv3(8, 8, 1, [0], 0.0, "binary", pred_type_ids=[0])
    enc = PredicateEncoder(PRED_EMB.shape[1], 8, "clip", 16)
    pe = enc(torch.eye(len(PRED_VOCAB))[[pair[0][1], pair[1][1]]], torch.from_numpy(PRED_EMB))
    x = torch.randn(2, 8)
    ei, et, w = torch.tensor([[0], [1]]), torch.zeros(1, dtype=torch.long), torch.ones(1)
    with torch.no_grad():
        a = L(x, ei, et, w, torch.tensor([0]), pe)[1]
        b = L(x, ei, et, w, torch.tensor([1]), pe)[1]
    out["two_predicates"] = dict(predicates=[pair[0][0], pair[1][0]], message_diff=float((a - b).abs().max()),
                                 differ=bool((a - b).abs().max() > 1e-6))
    return out


def test_ablations(g0):
    """Mỗi ablation tắt ĐÚNG thành phần đã mô tả: mask ⇒ loại cạnh bị tắt = 0, loại khác giữ nguyên; support ⇒ mọi vị từ còn
    lại đạt ngưỡng; pred_mode none ⇒ mô hình không có tham số vị từ; contains_only ⇒ chỉ còn contains."""
    out = {}
    for m, types in MASKS.items():
        g = graph_for(dict(DESIGN_BASE, mask=m), CFG.seeds[0])[0]
        off_ok = all(g.A[t].nnz == 0 for t in types)
        rest_ok = all((g.A[t] != g0.A[t]).nnz == 0 for t in RELATION_TYPES if t not in types) and (g.A_con != g0.A_con).nnz == 0
        out[f"mask:{m}"] = bool(off_ok and rest_ok)
    gs = graph_for(dict(DESIGN_BASE, rel_policy="support"), CFG.seeds[0])[0]
    out["policy:support"] = all(PRED_SUPPORT[(r_, p)] >= CFG.pred_min_support for (r_, a, b), d in gs.pdist.items()
                                if gs.A[r_][a, b] != 0 for p in d)
    if AGREE:
        ga = graph_for(dict(DESIGN_BASE, rel_policy="agreement"), CFG.seeds[0])[0]
        out["policy:agreement"] = all(agreement_rate(r_, INV.id2key[a], INV.id2key[b], p) >= CFG.agree_min
                                      for (r_, a, b), d in ga.pdist.items() if ga.A[r_][a, b] != 0 for p in d)
    m0 = make_model(E_BASE, SYM_FEAT, pred_mode="none")
    out["pred_mode:none_has_no_predicate_params"] = not any("pred" in n_ for n_, _ in m0.named_parameters())
    m1 = make_model(E_BASE, SYM_FEAT, pred_mode="clip")
    out["pred_mode:clip_has_predicate_params"] = any("pred" in n_ for n_, _ in m1.named_parameters())
    out["pred_mode:clip_no_adapter"] = not any("pred_enc.res" in n_ for n_, _ in m1.named_parameters())
    gp = graph_for(dict(DESIGN_BASE, path_mode="pred_match"), CFG.seeds[0])[0]
    out["path:weighted_has_no_P1m"] = all(g0.R[f"P1m_{r}"].nnz == 0 for r in PRED_FAMILIES)
    out["path:pred_match_has_P1m"] = any(gp.R[f"P1m_{r}"].nnz > 0 for r in PRED_FAMILIES)
    out["path:pred_match_same_other_patterns"] = all((gp.R[p] != g0.R[p]).nnz == 0 for p in PATTERNS if not p.startswith("P1m_"))
    return out


def test_inductive_isolation():
    """Strict inductive: đổi embedding của MỘT video đánh giá không làm đổi node lớp/khái niệm hay video khác;
    transductive thì có (để chứng minh phép thử nhạy). Chạy cho cả video test MSVD và video tập ngoài."""
    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])
    model = make_model(eng, SYM_FEAT, pred_mode="clip").eval()
    out = {}
    for s in ("test", "ext"):
        tv = np.array(SPLIT[s])
        if len(tv) < 2:
            continue
        v0, v1 = int(tv[0]), int(tv[1])
        for proto in PROTOCOLS:
            with torch.no_grad():
                h1 = model.encode_graph(*eng.edges_on(proto))
                old = model.vid_feat[v0].clone()
                model.vid_feat[v0] = torch.randn_like(old)
                h2 = model.encode_graph(*eng.edges_on(proto))
                model.vid_feat[v0] = old
            sym = slice(g.n_vid, g.n_nodes)
            out[f"{s}:{proto}"] = dict(symbolic_changed=float((h1[sym] - h2[sym]).abs().max()),
                                       other_video_changed=float((h1[v1] - h2[v1]).abs().max()),
                                       own_video_changed=float((h1[v0] - h2[v0]).abs().max()),
                                       all_finite=bool(torch.isfinite(h1).all()))
    out["edges"] = eng.edge_stats
    return out


def test_guards():
    out = {}
    for s in ("test", "ext"):
        if not SPLIT[s]:
            continue
        try:
            make_part(s); out[f"{s}_before_freeze_blocked"] = False
        except RuntimeError:
            out[f"{s}_before_freeze_blocked"] = True
        sq = np.where(CAP_SPLIT == s)[0][:3]
        for name, fn in ((f"{s}→gnn_train", lambda: CAPS.use(sq, "gnn_train")), (f"{s}→tune", lambda: CAPS.use(sq, "tune")),
                         (f"{s}→vocab", lambda: CAPS.by_video(SPLIT[s][:2], "vocab")),
                         (f"{s}→relations", lambda: CAPS.by_video(SPLIT[s][:2], "relations")),
                         (f"{s}→annotation", lambda: CAPS.by_video(SPLIT[s][:2], "annotation")),
                         (f"{s}→agreement", lambda: CAPS.by_video(SPLIT[s][:2], "agreement"))):
            try:
                fn(); out[name + "_blocked"] = False
            except PermissionError:
                out[name + "_blocked"] = True
        try:
            CAPS.use(sq, "eval"); out[f"{s}_eval_before_freeze_blocked"] = False
        except RuntimeError:
            out[f"{s}_eval_before_freeze_blocked"] = True
    vq = np.where(CAP_SPLIT == "val")[0][:3]
    for name, fn in (("val→gnn_train", lambda: CAPS.use(vq, "gnn_train")),
                     ("val→relations", lambda: CAPS.by_video(SPLIT["val"][:2], "relations")),
                     ("train→tune", lambda: CAPS.use(np.where(CAP_SPLIT == "train")[0][:3], "tune"))):
        try:
            fn(); out[name + "_blocked"] = False
        except PermissionError:
            out[name + "_blocked"] = True
    return out

# ---------------------------------- Self-test v4 (QC-RGAT) — nhanh (< 1 phút CPU), không huấn luyện ----------------------------------
def _v4_model_for_test(design, seed=0):
    """Mô hình KHỞI TẠO NGẪU NHIÊN nhưng mọi cổng đều khác 0 (để phép thử không qua vì tham số bằng 0)."""
    g, eng = graph_for(design, CFG.seeds[0])
    set_seed(seed)
    m = make_model_v4(eng, design).eval()
    with torch.no_grad():
        for p in (m.beta, m.kappa, m.g_ex, m.g_soft, m.rgate):
            p.copy_(torch.randn_like(p))
    return g, eng, m


def _queries_with_pred_edges(D, n, seed=0):
    """Truy vấn TRAIN có ≥ 1 cạnh mang vị từ khớp vị từ của truy vấn (để thử nhánh vị từ)."""
    rng = np.random.RandomState(seed)
    out = []
    for q in rng.permutation(TRAIN_Q)[:4000]:
        ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]
        if ids and D.E and np.asarray(D.Mex[ids].sum(0)).ravel().max() > 0 and V4_HITS[q]:
            out.append(int(q))
            if len(out) >= n:
                break
    return out


def test_v4_decomposition(n=30):
    """Σ đóng góp (thị giác + nhãn + đường đi) = điểm của mô hình, cho T = 1 và 2; và xoá một cạnh ⇒ điểm giảm ĐÚNG bằng tổng đóng
    góp các đường đi qua cạnh đó (T = 1: tuyệt đối; T = 2: tính lại trên đồ thị đã xoá)."""
    out = {}
    for T in (1, 2):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=T))
        qs = _queries_with_pred_edges(m.D, n)
        maxdiff, del_diff, n_rel, n_del = 0.0, 0.0, 0, 0
        vids = [int(v) for v in SPLIT["train"] if m.D.Acon[int(v)].gt(0).any()]
        for q in qs:
            v = vids[q % len(vids)]
            terms = m.decompose(q, v)
            with torch.no_grad():
                s = float(m.score(m.D.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE))[0, 0])
            maxdiff = max(maxdiff, abs(s - sum(t["contrib"] for t in terms)))
            rel = [t for t in terms if t["kind"] == "relation"]
            n_rel += len(rel)
            if rel:
                e = max(rel, key=lambda t: abs(t["contrib"]))["edges"][-1]
                D2 = RelDataV4(g, drop_edges=[int(m.D.gidx[e])])
                with torch.no_grad():
                    s2 = float(m.score(D2.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE), D2)[0, 0])
                expect = sum(t["contrib"] for t in rel if e in t["edges"])
                del_diff = max(del_diff, abs((s - s2) - expect)); n_del += 1
        out[f"T{T}"] = dict(queries=len(qs), relation_terms=n_rel, max_abs_diff=maxdiff, deletions=n_del, deletion_max_abs_diff=del_diff)
    return out


def test_v4_predicate_and_query():
    """(1) Đổi vị từ của truy vấn ⇒ attention trên cạnh mang vị từ đó đổi; use_pred = False ⇒ KHÔNG đổi (đối chứng âm).
    (2) Hai truy vấn khác nhau ⇒ attention khác (qcond); qcond = False ⇒ giống hệt. (3) Hoán vị vị từ (G4) đổi tín hiệu khớp."""
    out = {}
    for use_pred in (True, False):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=1, v4_pred=use_pred))
        qs = _queries_with_pred_edges(m.D, 10)
        diffs = []
        for q in qs:
            qb = m.D.query_batch([q], True)
            qb2 = dict(qb, ex=torch.zeros_like(qb["ex"]), soft=torch.zeros_like(qb["soft"]))
            with torch.no_grad():
                u = m.q(qb["tq"])
                diffs.append(float((m.attention(qb, u) - m.attention(qb2, u)).abs().max()))
        out[f"pred_change_use_pred={use_pred}"] = max(diffs) if diffs else float("nan")
    for qcond in (True, False):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=1, v4_qcond=qcond))
        qs = _queries_with_pred_edges(m.D, 2)
        with torch.no_grad():
            qb = m.D.query_batch(qs, True)
            a = m.attention(qb, m.q(qb["tq"]))
        out[f"two_queries_attention_diff_qcond={qcond}"] = float((a[0] - a[1]).abs().max())
    g0 = graph_for(dict(V3_FINAL, model="v4"), CFG.seeds[0])[0]
    gs = graph_for(dict(V3_FINAL, model="v4", pred_shuffle=True), CFG.seeds[0])[0]
    D0, Ds = RelDataV4(g0), RelDataV4(gs)
    qs = _queries_with_pred_edges(D0, 50)
    out["pred_shuffle_changes_match"] = float((D0.query_batch(qs)["ex"] - Ds.query_batch(qs)["ex"]).abs().sum())
    out["pred_shuffle_same_topology"] = bool(np.array_equal(D0.src_np, Ds.src_np) and np.array_equal(D0.dst_np, Ds.dst_np))
    return out


def test_v4_inductive_and_masks():
    """Video chỉ NHẬN (node lớp không phụ thuộc video) ⇒ đổi đặc trưng / nhãn của MỘT video test (hoặc tập ngoài) không đổi điểm
    của video khác; rel = False ⇒ không có số hạng quan hệ; điểm hữu hạn, gradient hữu hạn.
    So với dung sai 1e-6 (như G5 của v3): trên GPU, index_add_ cộng dồn bằng atomic nên hai lần gọi cùng đầu vào có thể lệch ~1e-8."""
    g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=2))
    out = {}
    q = _queries_with_pred_edges(m.D, 1)[0]
    qb = m.D.query_batch([q])
    for s in ("test", "ext"):
        tv = [int(v) for v in SPLIT.get(s, [])]
        if len(tv) < 2:
            continue
        v0, v1 = tv[0], tv[1]
        cand = torch.tensor([[v0, v1]], device=DEVICE)
        with torch.no_grad():
            s1 = m.score(qb, cand)
            s1b = m.score(qb, cand)                          # cùng đầu vào ⇒ đo sai số số học khi chạy lại (GPU không tất định)
            old_x, old_a = VID_T[v0].clone(), m.D.Acon[v0].clone()
            VID_T[v0] = torch.randn_like(old_x); m.D.Acon[v0] = torch.roll(old_a, 7)
            s2 = m.score(qb, cand)
            VID_T[v0] = old_x; m.D.Acon[v0] = old_a
        out[s] = dict(other_video_changed=float((s1[0, 1] - s2[0, 1]).abs()), own_video_changed=float((s1[0, 0] - s2[0, 0]).abs()),
                      rerun_noise=float((s1 - s1b).abs().max()))
    g1, e1, m1 = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_rel=False))
    v = int(SPLIT["train"][0])
    out["no_relation_terms_when_rel_false"] = not any(t["kind"] == "relation" for t in m1.decompose(q, v))
    m.train()
    qb = m.D.query_batch(TRAIN_Q[:16])
    loss = m.score(qb, torch.from_numpy(np.tile(CAP_VID[TRAIN_Q[:16]], (16, 1))).to(DEVICE)).logsumexp(1).mean()
    loss.backward()
    out["grad_finite"] = bool(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))
    out["score_finite"] = bool(torch.isfinite(loss).item())
    m.eval()
    return out


## 4.15 Level 4(b): bộ truy vấn đăng ký trước, xAI theo thứ hạng, phân tích ca, demo, kiểm kê

Các hàm dưới đây chỉ là **định nghĩa**, được gọi ở phần THỰC THI. Mọi file đầu ra sinh từ chính lần chạy; không có số liệu điền tay.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b): công cụ bổ sung (chỉ ĐỊNH NGHĨA — chạy ở phần THỰC THI) ==============================
import copy as _copy

# Nguồn / giấy phép: dataset card HF (kiểm qua API ngày 2026-10-25) KHÔNG khai báo license ⇒ ghi đúng như vậy, không tự điền.
DATA_SOURCES = {
    "msvd": dict(name="MSVD (Microsoft Research Video Description Corpus)", paper="Chen & Dolan, ACL 2011",
                 repo=CFG.hf_repo, revision=CFG.hf_revision,
                 license="KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc (clip YouTube) phát hành cho nghiên cứu",
                 role="train: từ vựng, quan hệ, huấn luyện GNN | val: chọn cấu hình | test: THĂM DÒ"),
    "msrvtt": dict(name="MSR-VTT", paper="Xu et al., CVPR 2016", repo=CFG.msrvtt_repo, revision=CFG.msrvtt_revision,
                   license="KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc của Microsoft Research cho nghiên cứu",
                   role="chỉ đánh giá (chuyển giao zero-shot): không caption nào dùng để học hay chọn"),
}
# Số lần mỗi benchmark đã được nhìn TRƯỚC notebook này (lịch sử v1–v4, xem README). Đã nhìn ⇒ kết quả chỉ là THĂM DÒ.
BENCHMARK_LOOKS_BEFORE = {"msvd_test": 4, "msrvtt_1kA": 1, "msrvtt_fresh1k": 1, "msrvtt_fresh1kB": 0}
_PROV_RULES["evaluation"] = {"val", "test", "ext"}   # bộ truy vấn đánh giá: dựng từ văn bản truy vấn của tập đánh giá, không học / chọn gì
FINDINGS = []          # kết quả KHOA HỌC được báo cáo (không phải kiểm thử phần mềm): id, câu hỏi, kết quả, diễn giải


def finding(fid, question, result, interpretation):
    FINDINGS.append(dict(id=fid, question=question, result=str(result)[:900], interpretation=interpretation))
    print(f"  [FINDING] {fid} — {question}: {result}\n            ⇒ {interpretation}")


def benchmark_status():
    """Trạng thái từng tập đánh giá của lần chạy này: CONFIRMATORY chỉ khi tập CHƯA từng được nhìn."""
    ext_key = {"fresh1kB": "msrvtt_fresh1kB", "fresh1k": "msrvtt_fresh1k", "1kA": "msrvtt_1kA"}.get(EXT_REPORT.get("set"))
    out = {"msvd_test": dict(looks_before=BENCHMARK_LOOKS_BEFORE["msvd_test"], evaluated=not CFG.dev_mode,
                             status="EXPLORATORY" if not CFG.dev_mode else "NOT RUN (dev_mode: test khoá)")}
    if EXT_RECORDS and not EXT_REPORT.get("dev"):
        lb = BENCHMARK_LOOKS_BEFORE[ext_key]
        out[ext_key] = dict(looks_before=lb, evaluated=True, status="CONFIRMATORY" if lb == 0 else "EXPLORATORY",
                            label=EXT_REPORT.get("label"))
    elif EXT_RECORDS:
        out["msrvtt_dev"] = dict(looks_before=None, evaluated=True, status="DEV (thử đường ống)", label=EXT_REPORT.get("label"))
    else:
        out["msrvtt"] = dict(evaluated=False, status=EXT_STATUS)
    return out


def node_name(i):
    return INV.name[int(i)]


def edges_into_tags(g, src, tags):
    """Cạnh quan hệ src –t→ c với c ∈ tags (đúng chiều QC-RGAT truyền tin: nguồn → nhãn của video)."""
    out = []
    for t in RELATION_TYPES:
        A = g.A[t]
        if A.nnz == 0:
            continue
        row = A.getrow(int(src))
        for c in row.indices:
            if int(c) in tags:
                base = t[:-4] if t.endswith("_inv") else t
                key = (base, int(c), int(src)) if t.endswith("_inv") else (base, int(src), int(c))
                pd_ = g.pdist.get(key, {}) if base in PRED_FAMILIES else {}
                p = max(pd_, key=pd_.get) if pd_ else ""
                out.append((int(src), t, int(c), p))
    return out


def edge_str(e):
    s_, t, d_, p = e
    return f"{node_name(s_)} –{t}{'[' + p + ']' if p else ''}→ {node_name(d_)}"


def build_eval_queries(part, g, n_per_group=None, seed=None):
    """Bộ truy vấn ĐĂNG KÝ TRƯỚC (mục 7.3). Chỉ dùng: văn bản truy vấn (mục đích query_input), nhãn CLIP zero-shot của video đúng,
    cạnh đồ thị dựng từ caption TRAIN. KHÔNG dùng điểm hay thứ hạng của bất kỳ mô hình nào.
    trực tiếp : mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.
    gián tiếp : truy vấn nêu ≥ 1 quan hệ (bộ ba), nhãn video phủ MỘT PHẦN khái niệm truy vấn, và có cạnh quan hệ nối một khái
                niệm còn thiếu vào một nhãn của video (= quan hệ dự kiến phải khai thác)."""
    n_per_group = n_per_group or CFG.n_eval_queries
    seed = CFG.eval_query_seed if seed is None else seed
    q_all, gold_all = part["q_idx"], part["gold"]
    texts = CAPS.texts(q_all, "query_input")
    pools = {"direct": [], "indirect": []}
    for j, q in enumerate(q_all):
        v, hits = int(gold_all[j]), set(V4_HITS[int(q)])
        tags = set(g.vid_tags[v])
        if not hits:
            continue
        seen, miss = hits & tags, hits - tags
        base = dict(q_idx=int(q), text=texts[j], gold_vid=v, gold_video_id=RECORDS[v]["video_id"],
                    query_concepts="; ".join(sorted(node_name(h) for h in hits)),
                    concepts_seen_in_video="; ".join(sorted(node_name(h) for h in seen)),
                    concepts_missing="; ".join(sorted(node_name(h) for h in miss)))
        if not miss:
            pools["direct"].append(dict(base, type="direct", expected_relations="",
                                        reason=f"mọi khái niệm của truy vấn ({base['query_concepts']}) là nhãn CLIP zero-shot của video đúng"))
            continue
        trip = [t for t in parse_caption_v2(texts[j])[2] if INV.key2id.get(t[0]) is not None and INV.key2id.get(t[2]) is not None]
        if not trip or not seen:
            continue
        bridge = [e for m_ in sorted(miss) for e in edges_into_tags(g, m_, tags)]
        if not bridge:
            continue
        rel_txt = "; ".join(f"{s_.split('#')[0]} –{r_}{'[' + p_ + ']' if p_ else ''}→ {o_.split('#')[0]}" for s_, r_, o_, p_ in sorted(trip))
        pools["indirect"].append(dict(base, type="indirect", expected_relations=" || ".join(edge_str(e) for e in bridge[:5]),
                                      reason=(f"truy vấn nêu quan hệ ({rel_txt}); video chỉ có nhãn {base['concepts_seen_in_video']}, "
                                              f"thiếu {base['concepts_missing']}; {len(bridge)} cạnh quan hệ nối khái niệm thiếu vào nhãn video")))
    rows, info = [], {}
    for k, (grp, pool) in enumerate(pools.items()):
        order = np.random.RandomState(seed + k).permutation(len(pool))
        used, take = set(), []
        for i in order:
            if pool[i]["gold_vid"] in used:
                continue
            used.add(pool[i]["gold_vid"]); take.append(pool[i])
            if len(take) >= n_per_group:
                break
        for n_, r in enumerate(take, 1):
            rows.append(dict(query_id=f"{part['name']}-{grp[0].upper()}{n_:02d}", part=part["name"], **r))
        info[grp] = dict(eligible=len(pool), eligible_videos=len({r["gold_vid"] for r in pool}), selected=len(take))
    return pd.DataFrame(rows), info


def eval_query_metrics(E, QS):
    """R@1/5/10, MRR (trung bình ± std theo seed) của MỌI thí nghiệm trên bộ truy vấn đăng ký trước, theo nhóm."""
    part = E["part"]
    pos = {int(q): i for i, q in enumerate(part["q_idx"])}
    rows, pairs = [], []
    for grp in ("all", "direct", "indirect"):
        sel = QS if grp == "all" else QS[QS.type == grp]
        if not len(sel):
            continue
        idx = np.array([pos[int(q)] for q in sel.q_idx])
        gold = part["gold"][idx]
        for k, ev_k in E["ev"].items():
            ms = [metrics_from_ranks(v["rank"][idx], gold) for v in ev_k.values()]
            rows.append(dict(part=part["name"], kind=E["info"]["kind"], experiment_id=k, group=grp, n_queries=len(idx),
                             n_videos=len(set(gold.tolist())), n_seeds=len(ms),
                             **{f"{x}_mean": float(np.mean([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR")},
                             **{f"{x}_std": float(np.std([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR")}))
        for a in (G_IDS.get("G3"), G_IDS.get("V3REF")):                 # QC-RGAT và GraphSAGE có kiểu cạnh, mỗi cái vs CLIP-only
            if a in E["ev"]:
                rr = lambda k_: np.mean([1.0 / v["rank"][idx] for v in E["ev"][k_].values()], 0)
                bs = cluster_bootstrap_diff(rr(a), rr("G0_clip_only"), gold)
                pairs.append(dict(part=part["name"], kind=E["info"]["kind"], group=grp, a=a, b="G0_clip_only", n=len(idx),
                                  MRR_a=float(rr(a).mean()), MRR_b=float(rr("G0_clip_only").mean()), **bs))
    return pd.DataFrame(rows), pd.DataFrame(pairs)


def drop_graph_parts(D, vid, cls_drop=(), rel_drop=()):
    """Bản sao dữ liệu đồ thị với (1) cạnh contains video –chứa→ c (c ∈ cls_drop) của RIÊNG video `vid` bị xoá (chuẩn hoá lại hàng
    như lúc dựng), (2) cạnh quan hệ (chỉ số trong D) bị xoá. Mô hình và tham số giữ nguyên."""
    D2 = RelDataV4(D.graph, drop_edges=D.gidx[list(rel_drop)]) if len(rel_drop) else _copy.copy(D)
    if len(cls_drop):
        A = D2.Acon.clone()
        row = (A[vid] > 0).float()
        row[list(cls_drop)] = 0.0
        A[vid] = row / row.sum().clamp(min=1.0)
        D2 = _copy.copy(D2)
        D2.Acon = A
    return D2


def term_text(D, t, vid):
    if t["kind"] == "contains":
        return f"{RECORDS[vid]['video_id']} –chứa→ {node_name(D.N + t['node'])}"
    return v4_path_text(D, t, vid)


def term_valid(D, t, vid, q):
    """Cạnh của số hạng có thật trong đồ thị đang chạy, đúng loại, đúng chiều, đúng kiểu node; vị từ khớp (nếu có) nằm trên ĐÚNG cạnh."""
    if t["kind"] == "contains":
        c = D.N + t["node"]
        ok = D.graph.A_con[vid, c] > 0 and INV.node_type[c] != "video" and INV.node_type[vid] == "video"
        return bool(ok), True
    v = [v4_edge_valid(D, k, q) for k in t["edges"]]
    return all(x[0] for x in v), all(x[2] for x in v)


@torch.no_grad()
def explain_full(model, q, vid, top_m=None):
    """Giải thích ĐẦY ĐỦ "vì sao video được xếp hạng như vậy" = top-m số hạng ĐỒ THỊ ủng hộ (contains + đường đi quan hệ).
    Số hạng thị giác (W_v x_v) không phải đồ thị nên được báo riêng. Σ mọi số hạng = điểm đồ thị của mô hình."""
    top_m = top_m or CFG.top_m_paths
    terms = model.decompose(q, vid)
    graph_terms = [t for t in terms if t["kind"] != "visual"]
    sup = sorted([t for t in graph_terms if t["contrib"] > 0], key=lambda t: -t["contrib"])[:top_m]
    opp = sorted([t for t in graph_terms if t["contrib"] < 0], key=lambda t: t["contrib"])[:top_m]
    tot = {k: sum(t["contrib"] for t in terms if t["kind"] == k) for k in ("visual", "contains", "relation")}
    return dict(terms=terms, support=sup, oppose=opp, totals=tot, score=sum(t["contrib"] for t in terms),
                n_graph_terms=len(graph_terms), pos_graph=sum(t["contrib"] for t in graph_terms if t["contrib"] > 0))


def _deletion_sets(sup):
    return (sorted({t["node"] for t in sup if t["kind"] == "contains"}),
            sorted({k for t in sup if t["kind"] == "relation" for k in t["edges"]}))


@torch.no_grad()
def xai_rank_fidelity(run, part, n_queries=None, top_m=None, seed=0):
    """Fidelity theo QUYẾT ĐỊNH xếp hạng (mục 8.3): với ≥ n truy vấn có video đúng trong top-K ứng viên, xoá THẬT các cạnh của
    giải thích đầy đủ (contains của video + cạnh quan hệ) rồi chấm lại TOÀN BỘ ứng viên ⇒ Δ điểm đồ thị, Δ điểm cuối, Δ hạng,
    đổi top-1 / top-10; đối chứng = xoá cùng số cạnh contains khác của video + cùng số cạnh quan hệ ngẫu nhiên vào nhãn của video.
    Báo riêng: hạng không đổi thì ghi đúng là không đổi."""
    n_queries, top_m = n_queries or CFG.n_explain_queries, top_m or CFG.top_m_paths
    m, eng = run["model"], run["engine"]
    D = m.D
    rng = np.random.RandomState(seed)
    q_all = part["q_idx"]
    cand_all, s_clip = eng.clip_candidates(q_all, part["gallery"], CFG.rerank_k)
    rows = []
    for i in rng.permutation(len(q_all)):
        if len(rows) >= n_queries:
            break
        q, gold = int(q_all[i]), int(part["gold"][i])
        cand = cand_all[i]
        if gold not in cand:
            continue
        ex = explain_full(m, q, gold, top_m)
        if not ex["support"]:
            continue
        zc = znorm_per_row(s_clip[i:i + 1])[0]
        r_full, s_full, fin_full = v4_rank_after(run, q, gold, cand, zc, D)
        j = int(np.where(cand == gold)[0][0])
        con_del, rel_del = _deletion_sets(ex["support"])
        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, con_del, rel_del))
        tags = torch.nonzero(D.Acon[gold] > 0).flatten().tolist()
        other = [c for c in tags if c not in con_del]
        rc = [int(x) for x in rng.choice(other, min(len(con_del), len(other)), replace=False)] if (other and con_del) else []
        pool = [k for k in range(D.E) if int(D.dst_np[k]) in set(tags) and k not in rel_del]
        rr = [int(x) for x in rng.choice(pool, min(len(rel_del), len(pool)), replace=False)] if (pool and rel_del) else []
        r_rd, s_rd, fin_rd = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, rc, rr))
        val = [term_valid(D, t, gold, q) for t in ex["support"]]
        expl_edges = len(con_del) + len(rel_del)
        top10 = lambda f: set(np.argsort(-f)[:10].tolist())
        rows.append(dict(q=q, gold=gold, gold_video_id=RECORDS[gold]["video_id"], n_graph_terms=ex["n_graph_terms"],
                         n_expl_terms=len(ex["support"]), n_expl_contains=sum(t["kind"] == "contains" for t in ex["support"]),
                         n_expl_relation=sum(t["kind"] == "relation" for t in ex["support"]), n_expl_edges=expl_edges,
                         n_rand_edges=len(rc) + len(rr), sparsity=len(ex["support"]) / max(1, ex["n_graph_terms"]),
                         coverage=sum(t["contrib"] for t in ex["support"]) / max(1e-9, ex["pos_graph"]),
                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[1] for v in val])),
                         decomposition_ok=float(abs(ex["score"] - s_full) < 1e-4),
                         s_graph=float(s_full), d_s_expl=float(s_ex - s_full), d_s_rand=float(s_rd - s_full),
                         d_final_expl=float(fin_ex[j] - fin_full[j]), d_final_rand=float(fin_rd[j] - fin_full[j]),
                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,
                         rank_worse_expl=float(r_ex > r_full), rank_worse_rand=float(r_rd > r_full),
                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),
                         top1_changed_rand=float(np.argmax(fin_rd) != np.argmax(fin_full)),
                         top10_changed_expl=float(top10(fin_ex) != top10(fin_full)), top10_changed_rand=float(top10(fin_rd) != top10(fin_full)),
                         share_graph=(ex["totals"]["contains"] + ex["totals"]["relation"]) / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         share_relation=ex["totals"]["relation"] / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         explanation=" || ".join(f"{t['contrib']:+.4f} {term_text(D, t, gold)}" for t in ex["support"]),
                         opposing=" || ".join(f"{t['contrib']:+.4f} {term_text(D, t, gold)}" for t in ex["oppose"][:3])))
    return pd.DataFrame(rows)


def diagnose_case(run, q, gold, cand, rank, clip_rank, text):
    """Nguyên nhân (luật cố định): candidate_recall_low | tag_missing | missing_edge | predicate_not_in_graph | wrong_predicate |
    noisy_relation | distractor_shares_concepts | graph_helped | clip_already_correct."""
    g, D = run["graph"], run["model"].D
    hits, tags = set(V4_HITS[q]), set(g.vid_tags[gold])
    causes, notes = [], []
    if gold not in cand:
        causes.append("candidate_recall_low"); notes.append(f"video đúng ở hạng CLIP {clip_rank} > k = {CFG.rerank_k}")
    if hits and not hits & tags:
        causes.append("tag_missing"); notes.append("không khái niệm nào của truy vấn là nhãn của video đúng")
    for s_, r_, o_, p_ in sorted(parse_caption_v2(text)[2]):
        a, b = INV.key2id.get(s_), INV.key2id.get(o_)
        if a is None or b is None or a == b:
            continue
        on = [t for t in RELATION_TYPES if g.A[t][a, b] != 0 or g.A[t][b, a] != 0]
        if not on:
            causes.append("missing_edge"); notes.append(f"không có cạnh nào giữa {s_} và {o_} (quan hệ '{r_}' của truy vấn)")
        elif r_ in PRED_FAMILIES and p_:
            if (r_, p_) not in PRED_ID:
                causes.append("predicate_not_in_graph"); notes.append(f"vị từ '{p_}' không có trong từ vựng vị từ (train)")
            else:
                pd_ = g.pdist.get((r_, a, b), {}) or g.pdist.get((r_, b, a), {})
                if pd_ and p_ not in pd_:
                    causes.append("wrong_predicate")
                    notes.append(f"cạnh {s_}–{r_}–{o_} mang vị từ {sorted(pd_)[:3]}, không có '{p_}' của truy vấn")
    if gold in cand:
        ex = explain_full(run["model"], q, gold)
        if ex["totals"]["relation"] < 0 and rank > clip_rank:
            causes.append("noisy_relation"); notes.append(f"tổng đóng góp quan hệ cho video đúng âm ({ex['totals']['relation']:+.4f})")
        if rank < clip_rank:
            causes.append("graph_helped"); notes.append(f"đồ thị kéo video đúng từ hạng {clip_rank} lên {rank}")
    if clip_rank == 1 and rank == 1:
        causes.append("clip_already_correct")
    return list(dict.fromkeys(causes)), notes


FAIL_CAUSES = ("candidate_recall_low", "missing_edge", "wrong_predicate", "noisy_relation", "tag_missing", "predicate_not_in_graph")


@torch.no_grad()
def case_analysis_l4b(run, part, n_each=None, seed=0, max_q=4000):
    """5 thành công (hạng 1) + 5 thất bại (hạng > 10), chọn theo LUẬT: thành công ưu tiên ca đồ thị kéo hạng lên rồi tới ca CLIP đã đúng;
    thất bại lấy mỗi nguyên nhân trong FAIL_CAUSES một ca (nếu có) rồi bổ sung ngẫu nhiên — không chọn ví dụ đẹp."""
    n_each = n_each or CFG.n_cases
    m, eng = run["model"], run["engine"]
    rng = np.random.RandomState(seed)
    q = part["q_idx"]
    sel = np.sort(rng.choice(len(q), min(max_q, len(q)), replace=False))
    q, gold = q[sel], part["gold"][sel]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k)
    fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], 1.0)
    rank = ranks_with_fallback(sc["cand"], fin, gold, clip_rank)
    texts = CAPS.texts(q, "report")
    order = rng.permutation(len(q))
    succ = [i for i in order if rank[i] == 1 and clip_rank[i] > 1][:3]
    succ += [i for i in order if rank[i] == 1 and i not in succ][:n_each - len(succ)]
    fail_pool = [i for i in order if rank[i] > 10]
    diag = {}
    for i in fail_pool[:400]:
        diag[i] = diagnose_case(run, int(q[i]), int(gold[i]), sc["cand"][i], int(rank[i]), int(clip_rank[i]), texts[i])
    fail = []
    for c in FAIL_CAUSES:
        hit = next((i for i in fail_pool[:400] if c in diag[i][0] and i not in fail), None)
        if hit is not None and len(fail) < n_each:
            fail.append(hit)
    fail += [i for i in fail_pool if i not in fail][:n_each - len(fail)]
    cases = []
    for kind, idx in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idx:
            qi, gi = int(q[i]), int(gold[i])
            causes, notes = diag.get(i) or diagnose_case(run, qi, gi, sc["cand"][i], int(rank[i]), int(clip_rank[i]), texts[i])
            top = sc["cand"][i][np.argsort(-fin[i])[:3]].tolist()
            ex = explain_full(m, qi, gi) if gi in sc["cand"][i] else None
            ex_top1 = explain_full(m, qi, top[0]) if top and top[0] != gi else None
            cases.append(dict(kind=kind, q=qi, query=texts[i], gold_vid=gi, gold_video_id=RECORDS[gi]["video_id"],
                              rank_clip=int(clip_rank[i]), rank_final=int(rank[i]), in_candidates=bool(gi in sc["cand"][i]),
                              top3=[RECORDS[v]["video_id"] for v in top], causes=causes, notes=notes,
                              gold_totals={k_: round(v_, 4) for k_, v_ in ex["totals"].items()} if ex else None,
                              gold_explanation=[f"{t['contrib']:+.4f} {term_text(m.D, t, gi)}" for t in ex["support"]] if ex else [],
                              gold_opposing=[f"{t['contrib']:+.4f} {term_text(m.D, t, gi)}" for t in ex["oppose"][:3]] if ex else [],
                              top1_explanation=[f"{t['contrib']:+.4f} {term_text(m.D, t, top[0])}" for t in ex_top1["support"]] if ex_top1 else []))
    stats = dict(n_queries=len(q), top1=float(np.mean(rank == 1)), fail_gt10=float(np.mean(rank > 10)),
                 cause_counts_in_fail_sample=dict(Counter(c for i in diag for c in diag[i][0])), n_fail_diagnosed=len(diag))
    return cases, stats


def cases_markdown(cases, stats, part_name):
    L = [f"# Phân tích 5 thành công + 5 thất bại — {part_name}", "",
         "Chọn theo luật cố định (xem `case_analysis_l4b`), KHÔNG chọn tay. Thành công = video đúng ở hạng 1; thất bại = hạng > 10.",
         f"Thống kê trên {stats['n_queries']} truy vấn mẫu: hạng 1 = {stats['top1']:.1%}, hạng > 10 = {stats['fail_gt10']:.1%}; "
         f"nguyên nhân trong mẫu thất bại đã chẩn đoán ({stats['n_fail_diagnosed']}): {stats['cause_counts_in_fail_sample']}", ""]
    for n_, c in enumerate(cases, 1):
        L += [f"## {n_}. [{c['kind']}] \"{c['query']}\"", "",
              f"- Video đúng: `{c['gold_video_id']}` — hạng CLIP **{c['rank_clip']}** → hạng cuối **{c['rank_final']}**"
              f"{'' if c['in_candidates'] else ' (KHÔNG nằm trong top-k ứng viên CLIP ⇒ đồ thị không thể cứu)'}",
              f"- Top-3 trả về: {', '.join('`' + v + '`' for v in c['top3'])}",
              f"- Chẩn đoán (luật cố định{'; nguyên nhân thất bại' if c['kind'] == 'THẤT BẠI' else ''}): {', '.join(c['causes']) or '—'}"] + \
            [f"  - {x}" for x in c["notes"]]
        if c["gold_totals"]:
            L.append(f"- Phân rã điểm đồ thị của video đúng: {c['gold_totals']}")
        L += ["- Đường đi / quan hệ ỦNG HỘ video đúng (đóng góp thật vào điểm):"] + [f"  - {x}" for x in c["gold_explanation"] or ["(không có số hạng đồ thị dương)"]]
        if c["gold_opposing"]:
            L += ["- Số hạng PHẢN ĐỐI:"] + [f"  - {x}" for x in c["gold_opposing"]]
        if c["top1_explanation"]:
            L += [f"- Vì sao `{c['top3'][0]}` (hạng 1, sai) được xếp trên:"] + [f"  - {x}" for x in c["top1_explanation"]]
        L.append("")
    return "\n".join(L)


# ---------------------------------- Demo: văn bản tự do → top-K video + giải thích ----------------------------------
def thumb_path(vid):
    """Thumbnail khung giữa: dùng file đã lưu lúc trích đặc trưng; nếu không có (dùng cache) thì đọc lại từ video."""
    r = RECORDS[vid]
    p = CFG.out_dir / "thumbs" / f"{r['video_id']}.jpg"
    if p.exists():
        return p
    try:
        src = SOURCE if vid < N_MSVD else EXT_SOURCE
        with src.open(r["video_id"]) as path:
            frames, _ = read_sampled_frames(path, 3)
        mid = frames[1]
        h, w = mid.shape[:2]
        th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(p), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))
        return p
    except Exception:
        return None


@torch.no_grad()
def demo_query(text, run=None, part=None, k=None, show=True, save_png=None):
    """Nhập văn bản tự do ⇒ top-k video của gallery `part` (mặc định tập xác nhận, nếu không thì tập thăm dò) với điểm CLIP, điểm đồ thị,
    điểm cuối (α đã đóng băng) và giải thích = số hạng đồ thị ủng hộ. Văn bản demo KHÔNG đi vào CaptionStore và không được lưu làm dữ liệu."""
    global TXT_FEAT
    run = run or EXPS[G_IDS["G3"]]["runs"][CFG.seeds[0]]
    part = part or (EVALS["confirm"]["part"] if "confirm" in EVALS else EVALS["explore"]["part"])
    k = k or CFG.demo_topk
    m = run["model"]
    D = m.D
    t = clip_encode_texts([text])[0].astype(TXT_FEAT.dtype)
    q = len(TXT_FEAT)
    toks = query_tokens_v2(text)
    saved = (TXT_FEAT,)
    TXT_FEAT = np.vstack([TXT_FEAT, t[None]])
    V4_HITS.append(sorted({nid for nid, _, _ in parse_tokens_v2(toks, INV)}))
    V4_ROLES.append(query_roles_v4([text], INV)[0])
    QPREDS.append({rel: match_predicates(query_pred_tokens(text), rel) for rel in PRED_FAMILIES})
    try:
        gal = np.asarray(part["gallery"])
        cos = VID_FEAT[gal] @ t
        top = np.argsort(-cos)[:CFG.rerank_k]
        cand, s_clip = gal[top], cos[top]
        qb = D.query_batch([q], m.use_pred)
        s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()[0]
        fin = run["alpha"] * znorm_per_row(s_clip[None])[0] + (1 - run["alpha"]) * znorm_per_row(s[None])[0]
        order = np.argsort(-fin)[:k]
        out = dict(query=text, gallery=part["name"], alpha=run["alpha"], matched_concepts=[node_name(h) for h in V4_HITS[q]],
                   results=[])
        for r_, j in enumerate(order, 1):
            v = int(cand[j])
            ex = explain_full(m, q, v)
            out["results"].append(dict(rank=r_, video_id=RECORDS[v]["video_id"], clip_cos=float(s_clip[j]), graph_score=float(s[j]),
                                       final=float(fin[j]), clip_rank=int(j) + 1,          # cand đã xếp theo cosine giảm dần
                                       totals={k_: round(v_, 4) for k_, v_ in ex["totals"].items()},
                                       explanation=[f"{tt['contrib']:+.4f} {term_text(D, tt, v)}" for tt in ex["support"]],
                                       note="" if ex["support"] else "KHÔNG có đường đi giải thích hợp lệ (không số hạng đồ thị nào ủng hộ)"))
    finally:
        TXT_FEAT = saved[0]
        V4_HITS.pop(); V4_ROLES.pop(); QPREDS.pop()
    if show:
        print(f"\nTruy vấn: '{text}' | gallery {out['gallery']} ({len(gal)} video) | khái niệm khớp: {out['matched_concepts'] or '—'} | α = {out['alpha']}")
        for r in out["results"]:
            print(f"  #{r['rank']} {r['video_id']:>12s} | CLIP cos {r['clip_cos']:.3f} (hạng CLIP {r['clip_rank']}) | đồ thị {r['graph_score']:+.3f} "
                  f"| cuối {r['final']:+.3f} | phân rã {r['totals']}")
            for x in r["explanation"]:
                print(f"       • {x}")
            if r["note"]:
                print(f"       ⚠ {r['note']}")
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, len(out["results"]), figsize=(3 * len(out["results"]), 2.6))
            for ax, r in zip(np.atleast_1d(axes), out["results"]):
                v = next(x["vid"] for x in RECORDS if x["video_id"] == r["video_id"])
                p = thumb_path(v)
                if p is not None:
                    ax.imshow(plt.imread(str(p)))
                ax.set_title(f"#{r['rank']} {r['video_id']}\nCLIP {r['clip_cos']:.2f} | cuối {r['final']:+.2f}", fontsize=8)
                ax.axis("off")
            fig.suptitle(text, fontsize=9)
            fig.tight_layout()
            if save_png:
                fig.savefig(save_png, dpi=110)
            plt.show()
            plt.close(fig)
        except Exception as e:
            print(f"  (không vẽ được thumbnail: {type(e).__name__}: {e})")
    return out


def demo_widget():
    """Ô nhập truy vấn tương tác (Jupyter / Colab / Kaggle interactive). Không tự chạy khi Save & Run All."""
    import ipywidgets as w
    box, btn, out = w.Text(placeholder="ví dụ: a man is riding a horse", layout=w.Layout(width="60%")), w.Button(description="Tìm"), w.Output()

    def go(_):
        with out:
            out.clear_output()
            demo_query(box.value)
    btn.on_click(go)
    display(w.HBox([box, btn]), out)


# ---------------------------------- Kiểm kê, audit, kết luận, manifest ----------------------------------
def file_sha1(p, n=1 << 20):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        while True:
            b = f.read(n)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def split_counts():
    out = {}
    for s in sorted(set(CAP_SPLIT.tolist())):
        vids = sorted(set(CAP_VID[CAP_SPLIT == s].tolist()))
        out[s] = dict(videos=len(vids), captions=int((CAP_SPLIT == s).sum()),
                      video_ids_sha1=sha1_of([RECORDS[v]["video_id"] for v in vids]))
    return out


def graph_inventory(g):
    s, d, e, w, pi = g.edge_arrays()
    et = Counter(EDGE_TYPES[int(x)] for x in e)
    nt = Counter(INV.node_type)
    deg = np.bincount(np.concatenate([s, d]), minlength=INV.n_nodes) if len(s) else np.zeros(INV.n_nodes, int)
    nonvid = np.array([t != "video" for t in INV.node_type])
    return dict(nodes_by_type=dict(nt), edges_by_type=dict(et), n_edges=int(len(e)), self_loops=int((s == d).sum()),
                duplicate_edges=int(len(e) - len(set(zip(s.tolist(), d.tolist(), e.tolist())))),
                isolated_non_video_nodes=int(((deg == 0) & nonvid).sum()), videos_without_tag=int(sum(len(x) == 0 for x in g.vid_tags)),
                degree_non_video=dict(mean=float(deg[nonvid].mean()), median=float(np.median(deg[nonvid])), max=int(deg[nonvid].max())),
                edges_with_predicate=int((pi >= 0).sum()), schema_sha1=SCHEMA["sha1"], inventory_sha1=INV.sha1)

---
## 4.16 Thực thi

Thứ tự chạy:
1. Dữ liệu: MSVD và MSR-VTT → split → đặc trưng → pooling.
2. Từ vựng, vị từ, tagger, mẫu cho người chấm và nhãn.
3. Đồ thị v3, schema, audit → self-test.
4. Các stage V0–V7 trên VAL → **đóng băng**.
5. **Bộ truy vấn đăng ký trước** (trực tiếp / gián tiếp) → đánh giá test MSVD (thăm dò) và MSR-VTT fresh1k-B (xác nhận).
6. Phân tích quan hệ → xAI → metric bộ truy vấn → xAI theo thứ hạng → 5 + 5 ca → demo.
7. Xuất file → báo cáo, kết luận → kiểm kê dữ liệu, audit rò rỉ, nghiệm thu, 7 câu kết luận, manifest → nén zip.

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG A: dữ liệu thật, đọc được ==============================
mark_stage("data", requires=("config",))
if CFG.dataset == "msvd":
    RECORDS, _caps, SOURCE, DATA_REPORT = load_msvd()
else:
    RECORDS, _caps, SOURCE, DATA_REPORT = make_synthetic_msvd()
N_MSVD = len(RECORDS)

# ---- benchmark xác nhận độc lập (MSR-VTT, tập theo CFG.confirm_set — mặc định fresh1k); lỗi tải ⇒ NOT RUN, ghi rõ lý do, KHÔNG thay bằng tập khác ----
EXT_RECORDS, EXT_SOURCE, EXT_REPORT, EXT_STATUS = [], None, {}, "NOT RUN — CFG.confirm = none"
if CFG.confirm == "msrvtt":
    try:
        if CFG.dataset == "msvd":
            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = load_msrvtt(N_MSVD)
        else:
            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = make_synthetic_ext(N_MSVD)
        _caps.update(_ext_caps)
        EXT_STATUS = "DEV (không phải tập xác nhận)" if EXT_REPORT["dev"] else "OK"
    except Exception as e:
        EXT_RECORDS, EXT_SOURCE, EXT_STATUS = [], None, f"NOT RUN — không nạp được MSR-VTT: {type(e).__name__}: {e}"
        print("⚠", EXT_STATUS)
RECORDS = RECORDS + EXT_RECORDS
SPLIT_OF = {r["vid"]: r["split"] for r in RECORDS}
CAPS = CaptionStore(_caps, SPLIT_OF)
del _caps                                                  # caption chỉ còn trong CAPS
CAP_VID, CAP_SPLIT = CAPS.vid_of_cap, CAPS.split_of_cap
CAP_STD = np.zeros(len(CAP_VID), bool)                     # caption chuẩn 1k-A (truy vấn PHỤ của tập ngoài)
_first_cap = {v: i for i, v in reversed(list(enumerate(CAP_VID.tolist())))}
for r in EXT_RECORDS:
    CAP_STD[_first_cap[r["vid"]] + r["std_caption"]] = True

rep = DATA_REPORT["splits"]
print(f"Nguồn video: {DATA_REPORT['source']} ({DATA_REPORT['source_label']}) | {DATA_REPORT['n_video_files']} file video")
display(pd.DataFrame(rep).T)
if CFG.dataset == "msvd":
    got = {s: (rep[s]["videos"], rep[s]["captions"]) for s in rep}
    check("A1", "số video / caption theo split khớp dataset card MSVD", got == {s: tuple(v) for s, v in CFG.expected.items()},
          f"thực tế {got} | kỳ vọng {CFG.expected}")
    check("A1b", "cột 'video' của HF chỉ là tên file .avi (video thật lấy từ zip/thư mục)",
          all(rep[s]["video_field_is_filename"] for s in rep), DATA_REPORT["source_label"])
else:
    check("A1", "số video / caption khớp MSVD", False, "dữ liệu SYNTHETIC — không áp dụng", hard=False, status="NOT RUN")
check("A2", "mỗi video_id có đúng một file video (không thiếu file)",
      all(rep[s]["missing_files"] == 0 for s in rep), {s: rep[s]["missing_files"] for s in rep})
print(f"Tập ngoài: {EXT_STATUS} | {EXT_REPORT.get('label', '—')} | {EXT_REPORT.get('videos', 0)} video / "
      f"{EXT_REPORT.get('captions', 0)} caption")
if EXT_RECORDS:
    if EXT_REPORT["dev"]:
        ok_ext = EXT_REPORT["overlap_with_test_1kA"] == 0
    elif EXT_REPORT.get("set") in ("fresh1k", "fresh1kB"):
        ok_ext = (EXT_REPORT["videos"] == CFG.fresh_n and EXT_REPORT["overlap_with_test_1kA"] == 0
                  and EXT_REPORT["overlap_with_dev"] == 0
                  and (EXT_REPORT["set"] == "fresh1k" or EXT_REPORT["overlap_with_fresh1k"] == 0))
    else:
        ok_ext = EXT_REPORT["videos"] == 1000 and EXT_REPORT["overlap_with_test_1kA"] == 1000
    check("A6", "tập ngoài: đủ file video; fresh1k / fresh1k-B = đúng CFG.fresh_n video, rời 1k-A, rời 200 video dev (fresh1k-B rời cả fresh1k); "
                "1kA = đúng 1000 video test 1k-A; dev = rời hẳn 1k-A",
          EXT_REPORT["missing_files"] == 0 and ok_ext,
          {k: EXT_REPORT.get(k) for k in ("label", "set", "videos", "captions", "captions_per_video", "std_captions_added",
                                          "missing_files", "overlap_with_test_1kA", "overlap_with_dev", "overlap_with_fresh1k")})
else:
    check("A6", "tập ngoài (benchmark xác nhận độc lập)", False, EXT_STATUS, hard=False, status="NOT RUN")

probes = []
for s in ("train", "val", "test", "ext"):
    src = EXT_SOURCE if s == "ext" else SOURCE
    for r in [r for r in RECORDS if r["split"] == s][:3]:
        with src.open(r["video_id"]) as p:
            probes.append(dict(split=s, video_id=r["video_id"], **probe_video(p)))
display(pd.DataFrame(probes))
check("A3", "giải mã được video mẫu của TỪNG split (3 video/split, đếm frame bằng grab)",
      all(p["ok"] for p in probes),
      f"{sum(p['ok'] for p in probes)}/{len(probes)} video | codec {sorted({str(p.get('codec')) for p in probes})} | "
      f"frame {[p.get('frames_decoded') for p in probes]}")
DATA_REPORT["probes"], DATA_REPORT["ext"] = probes, dict(EXT_REPORT, status=EXT_STATUS)
save_json(DATA_REPORT, RESULTS / "gateA_dataset_report.json")
provenance("RECORDS + CaptionStore", "external", f"{CFG.hf_repo}@{CFG.hf_revision[:8]}" if CFG.dataset == "msvd"
           else "synthetic", note="metadata + caption gốc; caption chỉ đọc qua CaptionStore")
if EXT_RECORDS:
    provenance("tập ngoài (MSR-VTT)", "external", EXT_REPORT["label"], note="chỉ frame (encode) trước đóng băng; caption chỉ để đánh giá")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
mark_stage("split", requires=("data",))
SPLIT = make_split(RECORDS)
check("A5", "split THEO VIDEO, không video nào ở hai split; manifest đã lưu",
      sum(len(SPLIT[k]) for k in ("train", "val", "test", "ext")) == len(RECORDS), f"sha1={SPLIT['sha1']} → split_manifest.json")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG B (1/2): frame + đặc trưng ==============================
mark_stage("features", requires=("split",))
FRAME_FEAT, FRAME_META, TXT_FEAT, FEAT_CACHES = all_features()
n_bad = len(FRAME_META["bad"])
check("A4", "TOÀN BỘ video giải mã được và lấy đủ F_max frame", n_bad == 0,
      f"{len(RECORDS) - n_bad}/{len(RECORDS)} video đọc được | nguồn {FRAME_META['source']} | {FRAME_META['seconds']}s")
save_json(dict(policy=CFG.frame_policy, n_frames_max=CFG.n_frames_max,
               formula="idx_k = floor((k + 0.5) / F_max · n_decoded), t_k = idx_k / fps",
               per_video=FRAME_META["per_video"], bad=FRAME_META["bad"]), RESULTS / "frame_sampling.json")
provenance("FRAME_FEAT [N,12,D]", "frames", f"CLIP-image đóng băng {CFG.clip_model}/{CFG.clip_weights} trên 12 frame/video",
           video_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]), note=" | ".join(map(str, FEAT_CACHES)))
provenance("TXT_FEAT [N_caption,D]", "encoded", "CLIP-text đóng băng — mã hoá để cache, không học",
           caption_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]), note="việc DÙNG tuân theo CaptionStore")
dur = np.array([m["duration_s"] for m in FRAME_META["per_video"].values() if m])
print(f"Thời lượng video: min {dur.min():.1f}s | trung vị {np.median(dur):.1f}s | max {dur.max():.1f}s")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG B (2/2): gộp theo thời gian + kiểm tra ==============================
mark_stage("pooling", requires=("features",))
POOL_SEL, POOL_TABLE = select_pooling_on_val(FRAME_FEAT, TXT_FEAT, SPLIT)
VID_FEAT = pool_frames(FRAME_FEAT, POOL_SEL["n_frames"], POOL_SEL["pooling"])
POOL_TABLE.to_csv(RESULTS / "pooling_selection_val.csv", index=False)
display(POOL_TABLE.round(4))
print(f"→ chọn trên VAL: {POOL_SEL}")
provenance("n_frames + pooling", "selected", "MRR CLIP-only text→video trên VAL", caption_splits=("val",))
provenance("VID_FEAT [N,D]", "frames", f"gộp {POOL_SEL['n_frames']} frame ({POOL_SEL['pooling']})",
           video_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]))
gate_b_checks(FRAME_FEAT, FRAME_META, VID_FEAT, TXT_FEAT, SPLIT)
save_json([c for c in CHECKS if c["id"].startswith("B")], RESULTS / "gateB_checks.json")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
mark_stage("vocab_tagger", requires=("pooling",))
t0 = time.time()
VOCAB2 = build_vocab_v2(CAPS, SPLIT)
RELATIONS2 = build_relations_v2(CAPS, SPLIT, VOCAB2)
TAGS2, TAG_SCORES, TAG_SELECT2, TAG_GRID2 = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL["n_frames"], VOCAB2, SPLIT, CAPS)
for r in RECORDS:
    r["tags"] = TAGS2[r["vid"]]
TAXONOMY, CONCEPT_META, TAX_AUDIT, TAX_STATS = build_taxonomy_v2(VOCAB2, CAPS, SPLIT, {v: TAGS2[v] for v in SPLIT["train"]})
INV = NodeInventory(RECORDS, VOCAB2, CONCEPT_META)
CONF_CAL = contains_calibration(TAG_SCORES, SPLIT["train"], INV)
_qtexts = CAPS.texts(np.arange(len(CAP_VID)), "query_input")
QTOK2 = [query_tokens_v2(t) for t in _qtexts]
SYM_FEAT = symbolic_features(INV, VOCAB2)
TRAIN_Q = np.where(CAP_SPLIT == "train")[0]

# ---- vị từ (v3): từ vựng + support chỉ từ quan hệ caption TRAIN; embedding = CLIP-text đóng băng theo prompt ----
PRED_SUPPORT = Counter()
for v in SPLIT["train"]:
    PRED_SUPPORT.update({(rel, p) for _, rel, _, p in RELATIONS2[v]})
PRED_VOCAB = sorted(k for k in PRED_SUPPORT if k[0] in PRED_FAMILIES)
PRED_ID = {k: i for i, k in enumerate(PRED_VOCAB)}
PRED_EMB = predicate_embeddings(PRED_VOCAB, VOCAB2)
QPRED_TOK = [query_pred_tokens(t) for t in _qtexts]
QPREDS = [{rel: match_predicates(qs, rel) for rel in PRED_FAMILIES} for qs in QPRED_TOK]
del _qtexts
n_v = sum(t == "verb" for t in VOCAB2["types"].values())
print(f"Từ vựng: {len(VOCAB2['terms'])} khoá = {len(VOCAB2['terms']) - n_v} danh từ + {n_v} động từ; "
      f"{len(VOCAB2['lemmas_with_two_pos'])} lemma có CẢ hai POS")
print(f"Tagger (chọn trên VAL): {TAG_SELECT2}")
print(f"Taxonomy: {TAX_STATS['n_concepts']} khái niệm | isa theo biến thể {TAX_STATS['isa_edges']} | mơ hồ {TAX_STATS['n_ambiguous']} | "
      f"sense khác sense đầu {TAX_STATS['n_sense_differs']} | hub {TAX_STATS['n_hub']} | chu trình {TAX_STATS['cycles']}")
print(f"Vị từ: {len(PRED_VOCAB)} (họ có vị từ tự do {PRED_FAMILIES}; "
      f"{ {f: sum(r == f for r, _ in PRED_VOCAB) for f in PRED_FAMILIES} }); ≥ {CFG.pred_min_support} video TRAIN: "
      f"{ {f: sum(r == f and PRED_SUPPORT[(r, p)] >= CFG.pred_min_support for r, p in PRED_VOCAB) for f in PRED_FAMILIES} } | "
      f"truy vấn khớp ≥ 1 vị từ: {np.mean([any(q.values()) for q in QPREDS]):.1%}")
print(f"Inventory: {INV.n_nodes} node = {INV.n_vid} video ({N_MSVD} MSVD + {INV.n_vid - N_MSVD} tập ngoài) + {INV.n_cls} lớp + "
      f"{INV.n_con} khái niệm (sha1 {INV.sha1}) | {time.time() - t0:.0f}s")

# ---- Nhiệm vụ B: kiểm chứng TỰ ĐỘNG (parser độc lập, chỉ TRAIN) + mẫu phân tầng; nhãn người chấm chỉ là TUỲ CHỌN ----
t1 = time.time()
AGREE, AGREE_COVER = build_agreement(CAPS, SPLIT, VOCAB2)
ANN_SAMPLE, ANN_CELLS, TRIPLE_SUP, _ = build_annotation_sample(CAPS, SPLIT, VOCAB2, RECORDS)
if AGREE is not None:
    ANN_SAMPLE.insert(7, "auto_parser_agrees", [int(triple_confirmed((r.src_node, r.family, r.dst_node, r.predicate),
                                                                       _DEP_CACHE[r.caption.lower()])) for r in ANN_SAMPLE.itertuples()])
    REL_QUALITY_AUTO = agreement_quality(AGREE, AGREE_COVER, TRIPLE_SUP)
    _fam = REL_QUALITY_AUTO[(REL_QUALITY_AUTO.stratum == "ALL") & (REL_QUALITY_AUTO.family != "ALL")]
    AUTO_STATUS = (f"kiểm chứng TỰ ĐỘNG ({DEP_STATUS}): tỉ lệ quan hệ heuristic được parser độc lập xác nhận — "
                   + "; ".join(f"{r.family} {r.agreement:.2f} [{r.wilson_lo:.2f}, {r.wilson_hi:.2f}]" for r in _fam.itertuples())
                   + f" ({time.time() - t1:.0f}s). SILVER — không phải precision do người chấm.")
else:
    REL_QUALITY_AUTO = pd.DataFrame([dict(kind="silver_parser_agreement", family="ALL", stratum="ALL", status="NOT RUN", note=DEP_STATUS)])
    AUTO_STATUS = DEP_STATUS
print("Kiểm chứng tự động:", AUTO_STATUS)
ANN_SAMPLE.to_csv(RESULTS / "relation_annotation_sample.csv", index=False, encoding="utf-8-sig")
ANN_CELLS.to_csv(RESULTS / "relation_annotation_cells.csv", index=False)
LABEL_FILE, LABELS = find_label_file()           # tuỳ chọn: chỉ dùng nếu có file nhãn trong /kaggle/input hoặc MSVD_G3_LABELS
if LABELS is not None:
    REL_QUALITY, KAPPA, LABELED = relation_quality(LABELS)
    VERIFIED_CELLS = {(r.family, r.stratum) for r in REL_QUALITY.itertuples() if r.verified}
    match = float(LABELS["sample_id"].isin(set(ANN_SAMPLE["sample_id"])).mean()) if "sample_id" in LABELS else float("nan")
    REL_QUALITY_STATUS = (f"nhãn người chấm: {LABEL_FILE} | {int(REL_QUALITY.iloc[0].n_labeled)} dòng có nhãn | "
                          f"kappa {KAPPA['kappa']:.3f} trên {KAPPA['n_double']} dòng chấm đôi | ô đạt ngưỡng: {sorted(VERIFIED_CELLS)} | "
                          f"sample_id khớp mẫu tái tạo: {match:.1%}")
    LABELED.to_csv(RESULTS / "relation_annotation_labeled.csv", index=False)
else:
    KAPPA, REL_QUALITY = dict(n_double=0, kappa=float("nan"), raw_agreement=float("nan")), None
    REL_QUALITY_STATUS = "không có nhãn người chấm (tuỳ chọn) — chỉ có kiểm chứng tự động"
pd.concat([REL_QUALITY_AUTO] + ([REL_QUALITY.assign(kind="human_precision")] if REL_QUALITY is not None else []),
          ignore_index=True).to_csv(RESULTS / "relation_quality_audit.csv", index=False)
print("Nhãn người chấm:", REL_QUALITY_STATUS)

pd.DataFrame(TAG_GRID2).to_csv(RESULTS / "tagger_selection_val_v2.csv", index=False)
save_json(dict(VOCAB2, relations_example={RECORDS[v]["video_id"]: RELATIONS2[v][:8] for v in SPLIT["train"][:5]}),
          RESULTS / "vocab_v2.json")
save_json(dict(selection=TAG_SELECT2, calibration=CONF_CAL), RESULTS / "tagger_info_v2.json")
TAX_AUDIT.to_csv(RESULTS / "graph_taxonomy_audit.csv", index=False)
save_json(TAX_STATS, RESULTS / "graph_taxonomy_stats.json")
provenance("từ vựng (lemma, POS)", "learned", "caption TRAIN (POS + WordNet)", caption_splits=("train",))
provenance("quan hệ (loại + vị từ)", "learned", "caption TRAIN", caption_splits=("train",))
provenance("từ vựng + support vị từ", "learned", "quan hệ caption TRAIN", caption_splits=("train",))
provenance("embedding vị từ (CLIP-text đóng băng)", "external", "prompt CFG.pred_prompts × vị từ train")
provenance("mẫu quan hệ phân tầng", "learned", "caption TRAIN (phân tầng họ × mức phổ biến)", caption_splits=("train",))
if AGREE is not None:
    provenance("đồng thuận heuristic ↔ spaCy (kiểm chứng tự động)", "learned", f"caption TRAIN; {DEP_STATUS}", caption_splits=("train",))
if LABELS is not None:
    provenance("precision quan hệ + ô đã kiểm chứng", "external", f"nhãn NGƯỜI chấm trên caption TRAIN ({LABEL_FILE.name})")
provenance("tham số tagger", "selected", "F1 trên caption VAL", caption_splits=("val",))
provenance("thống kê hiệu chỉnh contains (μ, σ, phân vị)", "frames", "điểm tagger trên video TRAIN", video_splits=("train",))
provenance("taxonomy: sense theo ngữ cảnh + hub", "learned", "caption TRAIN + nhãn tagger TRAIN + WordNet",
           caption_splits=("train",), video_splits=("train",))
provenance("đặc trưng ngữ nghĩa node (CLIP-text)", "external", "prompt tagger / 'lemma: định nghĩa WordNet'")
# ---- v4: node khớp + VAI của từng node trong CHÍNH truy vấn (đọc văn bản truy vấn — mục đích query_input, như QTOK2) ----
V4_HITS = [sorted({nid for nid, _, _ in parse_tokens_v2(t, INV)}) for t in QTOK2]
V4_ROLES = query_roles_v4(CAPS.texts(np.arange(len(CAP_VID)), "query_input"), INV)
print(f"v4: truy vấn có ≥ 1 node khớp {np.mean([bool(h) for h in V4_HITS]):.3f}; có vai chủ thể/đối tượng "
      f"{np.mean([any(r in (1, 3) for r in d.values()) for d in V4_ROLES]):.3f}")


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐỒ THỊ v3: schema · node · cạnh có vị từ · audit cấu trúc · audit vị từ ==============================
mark_stage("graph_v3", requires=("vocab_tagger",))
G_BASE, E_BASE = graph_for(DESIGN_BASE, CFG.seeds[0])
print("Đồ thị v3 (thiết kế gốc = v2_base):", json.dumps(G_BASE.summary(), ensure_ascii=False))
SCHEMA = dict(version="v3", node_types=dict(video="video MSVD / tập ngoài (embedding CLIP gộp frame, đóng băng)",
                                            **{"class": "lớp (lemma, POS) từ caption TRAIN", "concept": "synset WordNet"}),
              semantic_types=["video", "noun", "verb", "noun_concept", "verb_concept"], relations=REL_SPEC,
              edge_record=["src", "relation_family", "relation", "predicate", "dst", "direction", "weight", "confidence",
                           "source", "status"],
              predicate_families=list(PRED_FAMILIES), predicate_vocab_size=len(PRED_VOCAB),
              predicate_prompts=CFG.pred_prompts, pred_modes=list(PRED_MODES), rel_policies=list(REL_POLICIES),
              patterns=dict(zip(PATTERNS, PATTERN_LEN)), masks={k: list(v) for k, v in MASKS.items()},
              inventory=dict(n_nodes=INV.n_nodes, videos=INV.n_vid, msvd_videos=N_MSVD, ext_videos=INV.n_vid - N_MSVD,
                             classes=INV.n_cls, concepts=INV.n_con, sha1=INV.sha1),
              path_modes=["weighted", "count", "degnorm", "dedup", "pred_match"],
              mp_weight_modes=["binary", "weighted_mean", "weighted_symnorm"], init_modes=["learned", "clip_frozen", "clip_residual"],
              related_cfgs=RELATED_CFGS, protocols=list(PROTOCOLS))
SCHEMA["sha1"] = sha1_of(SCHEMA)
save_json(SCHEMA, RESULTS / "graph_schema_v3.json")

deg_out = {t: np.asarray((G_BASE.A[t] > 0).sum(1)).ravel() for t in RELATION_TYPES}
deg_in = {t: np.asarray((G_BASE.A[t] > 0).sum(0)).ravel() for t in RELATION_TYPES}
con_in = np.asarray((G_BASE.A_con > 0).sum(0)).ravel()
node_rows = []
for i in range(INV.n_nodes):
    t = INV.node_type[i]
    node_rows.append(dict(node_id=i, node_type=t, canonical_name=INV.name[i], semantic_type=INV.semantic_type(i),
                          lemma=INV.lemma.get(i, ""), split_provenance=SPLIT_OF.get(i, "") if t == "video" else
                          ("train" if t == "class" else "external"),
                          synset=INV.concept_meta.get(INV.name[i], {}).get("synset", ""),
                          train_df=int(G_BASE.cls_df[i]) if t == "class" else "", active=i in G_BASE.active,
                          videos_tagged=int(con_in[i]), rel_out_degree=int(sum(deg_out[x][i] for x in RELATION_TYPES)),
                          rel_in_degree=int(sum(deg_in[x][i] for x in RELATION_TYPES))))
NODES_DF = pd.DataFrame(node_rows)
NODES_DF.to_csv(RESULTS / "graph_nodes_v3.csv", index=False)


def _confidence(rel, c, s=None, o=None, p=None):
    """Nhãn người chấm (nếu có) của ô (họ × mức phổ biến); ngược lại tỉ lệ đồng thuận SILVER với parser độc lập của đúng
    quan hệ đó; không có cả hai ⇒ 'unverified'."""
    if rel not in REL_CATS:
        return ""
    if REL_QUALITY is not None:
        r = REL_QUALITY[(REL_QUALITY.family == rel) & (REL_QUALITY.stratum == support_stratum(c))]
        if len(r) and r.iloc[0].n_labeled:
            return f"human {r.iloc[0].precision:.3f} [{r.iloc[0].wilson_lo:.3f}, {r.iloc[0].wilson_hi:.3f}]"
    a = AGREE.get((rel, s, o, p)) if AGREE and s is not None else None
    return f"silver {a[0] / a[1]:.2f} ({a[0]}/{a[1]})" if a and a[1] else "unverified"


def edge_table(g):
    """Một dòng = MỘT quan hệ (nguồn, họ, vị từ, đích, chiều): cạnh quan hệ tách theo từng vị từ của cặp; cạnh bị loại
    (chính sách / top-k) vẫn được ghi với status = excluded:<lý do>."""
    rows = []
    c = g.A_con.tocoo()
    for u, v, w in zip(c.row.tolist(), c.col.tolist(), c.data.tolist()):
        for rel, a, b, dr in (("contains", u, v, "forward"), ("contains_inv", v, u, "inverse")):
            rows.append(dict(src=INV.name[a], relation_family="contains", relation=rel, predicate="", dst=INV.name[b],
                             direction=dr, weight=w, confidence="tagger" if g.contains_mode == "conf" else "",
                             source=REL_SPEC[rel]["source"][:60], status="kept", pair_support="", predicate_videos="",
                             predicate_share=""))
    for t in RELATION_TYPES:
        m = g.A[t].tocoo()
        base = t[:-4] if t.endswith("_inv") else t
        dr = "symmetric" if t == "related" else ("inverse" if t.endswith("_inv") else "forward")
        for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):
            d = g.predicates_of(t, u, v) if base in REL_CATS else {}
            tot = float(sum(d.values())) or 1.0
            ka, kb = (INV.name[v], INV.name[u]) if t.endswith("_inv") else (INV.name[u], INV.name[v])
            for p, cnt in (sorted(d.items(), key=lambda x: (-x[1], x[0])) or [("", "")]):
                rows.append(dict(src=INV.name[u], relation_family=REL_SPEC[t]["family"], relation=t, predicate=p,
                                 dst=INV.name[v], direction=dr, weight=w, confidence=_confidence(base, cnt, ka, kb, p) if p else "",
                                 source=REL_SPEC[t]["source"][:60], status="kept",
                                 pair_support=int(sum(d.values())) if d else "", predicate_videos=cnt,
                                 predicate_share=round(cnt / tot, 4) if p else ""))
    for x in g.excluded:
        for p, cnt in sorted(x["predicates"].items(), key=lambda y: (-y[1], y[0])):
            rows.append(dict(src=INV.name[x["src"]], relation_family=REL_SPEC[x["family"]]["family"], relation=x["family"],
                             predicate=p, dst=INV.name[x["dst"]], direction="forward", weight="",
                             confidence=_confidence(x["family"], cnt, INV.name[x["src"]], INV.name[x["dst"]], p),
                             source=REL_SPEC[x["family"]]["source"][:60],
                             status=f"excluded:{x['reason']}", pair_support=x["support"], predicate_videos=cnt, predicate_share=""))
    return pd.DataFrame(rows)


EDGES_DF = edge_table(G_BASE)
EDGES_DF.to_csv(RESULTS / "graph_edges_v3.csv", index=False)

struct = []
for t in ["contains"] + RELATION_TYPES:
    M = G_BASE.A_con if t == "contains" else G_BASE.A[t]
    m = M.tocoo()
    do, di = np.asarray((M > 0).sum(1)).ravel(), np.asarray((M > 0).sum(0)).ravel()
    srcs, tgts = np.unique(m.row), np.unique(m.col)
    base = t[:-4] if t.endswith("_inv") else t
    npred = [len(G_BASE.predicates_of(t, u, v)) for u, v in zip(m.row.tolist(), m.col.tolist())] if base in REL_CATS else []
    struct.append(dict(section="v3_relation", relation=t, family=REL_SPEC[t]["family"], usage=REL_SPEC[t]["usage"], n_edges=int(M.nnz),
                       src_types=dict(Counter(INV.semantic_type(i) for i in srcs)) if t != "contains" else {"video": len(srcs)},
                       tgt_types=dict(Counter(INV.semantic_type(i) for i in tgts)),
                       w_mean=float(m.data.mean()) if m.nnz else None, w_nonbinary_frac=float((m.data < 1 - 1e-9).mean()) if m.nnz else 0,
                       out_deg_mean=float(do[do > 0].mean()) if (do > 0).any() else 0, out_deg_max=int(do.max()) if do.size else 0,
                       in_deg_mean=float(di[di > 0].mean()) if (di > 0).any() else 0, in_deg_max=int(di.max()) if di.size else 0,
                       predicates_per_edge_mean=float(np.mean(npred)) if npred else None,
                       frac_edges_multi_predicate=float(np.mean([k > 1 for k in npred])) if npred else None,
                       predicate_in_message=t in PRED_TYPES, self_loops=int((m.row == m.col).sum()),
                       duplicates=int(m.nnz - len(set(zip(m.row, m.col))))))
act = np.array(sorted(G_BASE.active))
rel_deg = sum(deg_out[t] + deg_in[t] for t in RELATION_TYPES)
vid_no_tag = int(sum(len(x) == 0 for x in G_BASE.vid_tags))
struct.append(dict(section="v3_nodes", relation="—", note=(
    f"lớp active {len(act)}/{INV.n_cls}; lớp active KHÔNG có cạnh quan hệ: {int((rel_deg[act] == 0).sum())}; video không có contains: "
    f"{vid_no_tag}; khái niệm không có isa vào: {int((deg_in['isa'][INV.n_vid + INV.n_cls:] == 0).sum())}/{INV.n_con}; "
    f"cặp quan hệ bị loại (chính sách/top-k): {len(G_BASE.excluded)}")))
STRUCT_DF = pd.DataFrame(struct)
STRUCT_DF.to_csv(RESULTS / "graph_structure_audit.csv", index=False)
display(STRUCT_DF[STRUCT_DF.section == "v3_relation"][["relation", "n_edges", "w_mean", "predicates_per_edge_mean",
                                                       "frac_edges_multi_predicate", "predicate_in_message", "self_loops", "duplicates"]])

# ---- audit embedding vị từ: vị từ nào dùng trong đồ thị gốc, support, láng giềng gần nhất (kiểm tra ngữ nghĩa) ----
_used = Counter()
for rel in PRED_FAMILIES:
    for d in G_BASE.pshare[rel].values():
        _used.update({PRED_VOCAB[pid] for pid in d})
_E = PRED_EMB / np.linalg.norm(PRED_EMB, axis=1, keepdims=True).clip(1e-8)
pe_rows = []
for rel in PRED_FAMILIES:
    idx = np.array([i for i, (r, _) in enumerate(PRED_VOCAB) if r == rel])
    if not len(idx):
        continue
    used_idx = np.array([i for i in idx if _used[PRED_VOCAB[i]] > 0])
    S = _E[idx] @ _E[idx].T
    off = S[~np.eye(len(idx), dtype=bool)] if len(idx) > 1 else np.array([1.0])
    for i in (used_idx if len(used_idx) else idx[:0]):
        s = _E[idx] @ _E[i]
        nn_ = [PRED_VOCAB[idx[j]][1] for j in np.argsort(-s) if idx[j] != i][:3]
        pe_rows.append(dict(family=rel, predicate=PRED_VOCAB[i][1], phrase=predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2),
                            prompt=CFG.pred_prompts[rel][0].format(predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2)),
                            train_videos=int(PRED_SUPPORT[PRED_VOCAB[i]]), graph_edges=int(_used[PRED_VOCAB[i]]),
                            passes_support_policy=PRED_SUPPORT[PRED_VOCAB[i]] >= CFG.pred_min_support,
                            nearest=", ".join(nn_), family_mean_cos=float(off.mean()), family_min_cos=float(off.min()),
                            family_max_cos=float(off.max())))
PRED_AUDIT = pd.DataFrame(pe_rows)
PRED_AUDIT.to_csv(RESULTS / "predicate_embedding_audit.csv", index=False)
if len(PRED_AUDIT):
    print("Embedding vị từ:", PRED_AUDIT.groupby("family").agg(n=("predicate", "size"), mean_cos=("family_mean_cos", "first"),
                                                              min_cos=("family_min_cos", "first")).to_dict("index"))

# ---- thống kê quan hệ caption TRAIN ----
tq = np.where(CAP_SPLIT == "train")[0]
tr_texts = CAPS.texts(tq, "relations")
all_trip, oov = Counter(), 0
vset = set(VOCAB2["terms"])
for txt in tr_texts:
    for s, rel, o, pred in parse_caption_v2(txt)[2]:
        all_trip[rel] += 1
        oov += int(s not in vset or o not in vset)
REL_STATS = dict(train_captions=len(tq), triples_extracted=int(sum(all_trip.values())), by_category=dict(all_trip),
                 frac_dropped_oov=oov / max(sum(all_trip.values()), 1),
                 kept_relation_pairs=sum(int(G_BASE.A[rel].nnz) for rel in REL_CATS),
                 kept_relation_predicate_rows=int(((EDGES_DF.status == "kept") & EDGES_DF.relation.isin(REL_CATS)).sum()),
                 excluded_pairs=len(G_BASE.excluded), annotation_sample_rows=len(ANN_SAMPLE),
                 automatic_verification=AUTO_STATUS, human_precision=REL_QUALITY_STATUS,
                 heuristic_covers_spacy={k: round(AGREE_COVER.get((k, "found"), 0) / max(AGREE_COVER.get((k, "total"), 0), 1), 3)
                                         for k in REL_CATS} if AGREE_COVER else "NOT RUN",
                 recall_f1="NOT RUN — không có tập quan hệ đúng đầy đủ; 'heuristic_covers_spacy' chỉ là độ phủ tương đối so với spaCy")
save_json(REL_STATS, RESULTS / "graph_relation_quality_stats.json")
print("Quan hệ caption TRAIN:", json.dumps(REL_STATS, ensure_ascii=False))
provenance("cạnh quan hệ v3 (agent/object/svo/prep + _inv, phân phối vị từ)", "learned", "caption TRAIN", caption_splits=("train",))
provenance("cạnh related (NPMI)", "learned", "nhãn tagger của video TRAIN", video_splits=("train",))
provenance("IDF", "learned", "contains + isa thật của video TRAIN", video_splits=("train",))
provenance("cạnh isa", "learned", "WordNet + chọn sense từ ngữ cảnh caption TRAIN", caption_splits=("train",))

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== SELF-TEST v3 (mục 12: kiểm thử 1–10) ==============================
# PASS ở đây chỉ xác nhận PHẦN MỀM làm đúng như mô tả — KHÔNG xác nhận giả thuyết khoa học.
mark_stage("self_tests", requires=("graph_v3",))
ST = {}
t = ST["canary"] = test_canary()
check("C1", "[12.8] canary chỉ có trong caption TEST / tập ngoài không lọt vào từ vựng, quan hệ, từ vựng vị từ",
      not t["leak_vocab"] and not t["leak_relations"] and not t["leak_predicates"], t)
check("C1b", "đối chứng dương: canary đặt vào TRAIN thì bị phát hiện", t["positive_control_detected"], t)
t = ST["invariance"] = test_invariance()
check("C2", "[12.8] xáo/thay caption TEST + tập ngoài: từ vựng, quan hệ, vị từ, tagger, taxonomy, inventory, đồ thị, mẫu gán nhãn GIỐNG HỆT",
      all(t.values()), t)
t = ST["guards"] = test_guards()
check("C3", "[12.8] hàng rào: chặn mở test / tập ngoài trước đóng băng + caption sai mục đích", all(t.values()), t)
t = ST["inventory"] = test_inventory_constant()
check("G1", "[12.5] inventory CỐ ĐỊNH giữa mọi biến thể đồ thị; cùng cấu hình mô hình ⇒ cùng số tham số",
      len(t["node_counts"]) == 1 and t["same_inventory"] and all(len(v) == 1 for v in t["param_counts_by_pred_mode"].values()), t)
t = ST["schema"] = test_schema_conformance(G_BASE)
check("G2", "mọi cạnh tuân thủ schema (kiểu node, POS, trọng số, vị từ đúng họ, không lẫn tên loại quan hệ)", not t["violations"], t)
t = ST["path_identity"] = test_path_identity()
check("G3", "[12.9] điểm đường đi ma trận = liệt kê ở cả 5 path_mode; tính lại R khớp; đường đi khớp vị từ ghi đúng vị từ + chiều",
      all(v["mismatches"] == 0 and v["recompute_maxdiff"] < 1e-5 and v["pairs"] > 0 and v["predicate_invalid"] == 0
          and v["negative_control_detected"] == v["predicate_paths"] for v in t.values())
      and t["pred_match"]["predicate_paths"] > 0, t)
t = ST["mp_weights"] = test_mp_weights()
check("G4", "[12.6, 12.7] message passing: trọng số dùng đúng công thức, chỉ nhân MỘT lần (khớp tính tay, bất biến tỉ lệ), "
      "w=0 ≡ không cạnh, node cô lập hữu hạn, bất biến thứ tự, gradient hữu hạn",
      all(v["as_designed"] and v["isolated_nodes_finite"] and v["edge_order_invariant"] and v["grad_finite"] for v in t.values()), t)
t = ST["predicate_forward"] = test_predicate_forward()
check("P1", "[12.1] embedding vị từ TÁC ĐỘNG forward khi pred_mode ≠ none; pred_mode none (≡ v2) KHÔNG dùng vị từ",
      t["predicate_used_as_designed"] and all(t[m]["finite"] for m in PRED_MODES), t)
check("P2", "[12.2] hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác nhau", t["two_predicates"]["differ"], t["two_predicates"])
t = ST["ablations"] = test_ablations(G_BASE)
check("AB", "[12.5] mỗi ablation tắt ĐÚNG thành phần mô tả (mask, chính sách support, pred_mode, path_mode)", all(t.values()), t)
t = ST["inductive"] = test_inductive_isolation()
ind = [v for k, v in t.items() if k.endswith(":inductive")]
tra = [v for k, v in t.items() if k.endswith(":transductive")]
check("G5", "[12.7] strict inductive (video test MSVD + tập ngoài): KHÔNG đổi node lớp/khái niệm hay video khác; transductive thì đổi; "
      "không NaN", all(v["symbolic_changed"] < 1e-6 and v["other_video_changed"] < 1e-6 and v["own_video_changed"] > 1e-6
                        and v["all_finite"] for v in ind) and all(v["symbolic_changed"] > 1e-6 for v in tra), t)
R_AUD, ok_list = [], []
for s in CFG.seeds:
    df_, ok_ = randomization_audit(G_BASE, graph_for(dict(DESIGN_BASE, rewire=True), s)[0])
    R_AUD.append(df_.assign(control="rewire", seed=s, rand_seed=REWIRE_SEED_BASE + s)); ok_list.append(ok_)
S_AUD, ok_s = [], []
for s in CFG.seeds:
    df_, ok_ = predicate_shuffle_audit(G_BASE, graph_for(dict(DESIGN_BASE, pred_shuffle=True), s)[0])
    S_AUD.append(df_.assign(control="predicate_shuffle", seed=s, rand_seed=SHUFFLE_SEED_BASE + s)); ok_s.append(ok_)
RAND_AUDIT = pd.concat(R_AUD + S_AUD, ignore_index=True)
RAND_AUDIT.to_csv(RESULTS / "graph_randomization_audit.csv", index=False)
RANDOM_VALID, SHUFFLE_VALID = all(ok_list), all(ok_s)
display(RAND_AUDIT[(RAND_AUDIT.seed == CFG.seeds[0]) & (RAND_AUDIT.control == "rewire")][
    ["relation", "edges_orig", "edges_rand", "out_degree_equal", "in_degree_equal", "type_pairs_equal", "weights_multiset_equal",
     "predicates_multiset_equal", "duplicates", "self_loops", "frac_changed", "accepted"]].round(3))
check("G6", "[12.3] đối chứng rewiring giữ kiểu node, họ quan hệ, bậc, trọng số, đa tập vị từ; đổi ≥ ngưỡng (nếu không: KHÔNG kết luận về topology)",
      RANDOM_VALID, f"{int((~RAND_AUDIT[RAND_AUDIT.control == 'rewire'].accepted).sum())} dòng không đạt", hard=False)
check("G7", "[12.4] đối chứng hoán vị vị từ KHÔNG đổi topology / trọng số; chỉ đổi vị từ trong họ svo/prep (nếu không: KHÔNG kết luận về vị từ)",
      SHUFFLE_VALID, RAND_AUDIT[RAND_AUDIT.control == "predicate_shuffle"].groupby("relation").frac_edges_predicates_changed.mean()
      .round(3).to_dict(), hard=False)
check("C7", "split MSVD tái lập được (sha1 giống v2)", make_split(RECORDS, write=False)["sha1"] == SPLIT["sha1"], SPLIT["sha1"])
v4_tensors()
t = ST["v4_decomposition"] = test_v4_decomposition()
check("Q3", "[v4] Σ số hạng (thị giác + nhãn + đường đi quan hệ) = điểm mô hình (T = 1, 2); xoá một cạnh ⇒ điểm đổi ĐÚNG bằng phần "
      "đóng góp các đường đi qua cạnh đó", all(v["max_abs_diff"] < 1e-4 and v["deletion_max_abs_diff"] < 1e-4 and v["relation_terms"] > 0
                                             for v in t.values()), t)
t = ST["v4_predicate_query"] = test_v4_predicate_and_query()
check("Q1", "[v4] vị từ của truy vấn tác động attention trên ĐÚNG cạnh mang vị từ đó; v4_pred = False thì không (đối chứng âm); "
      "hoán vị vị từ đổi tín hiệu khớp nhưng giữ topology",
      t["pred_change_use_pred=True"] > 1e-6 and t["pred_change_use_pred=False"] < 1e-6 and t["pred_shuffle_changes_match"] > 0
      and t["pred_shuffle_same_topology"], t)
check("Q2", "[v4] attention phụ thuộc truy vấn khi v4_qcond = True; giống hệt giữa các truy vấn khi False (đối chứng âm)",
      t["two_queries_attention_diff_qcond=True"] > 1e-6 and t["two_queries_attention_diff_qcond=False"] < 1e-6, t)
t = ST["v4_inductive"] = test_v4_inductive_and_masks()
check("Q4", "[v4] inductive: đổi đặc trưng/nhãn của MỘT video test (tập ngoài) không đổi điểm video khác (dung sai 1e-6 như G5: GPU cộng dồn atomic không tất định); rel = False ⇒ không số hạng "
      "quan hệ; điểm + gradient hữu hạn",
      all(v["other_video_changed"] < 1e-6 and v["own_video_changed"] > 1e-6 for k, v in t.items() if k in ("test", "ext"))
      and t["no_relation_terms_when_rel_false"] and t["grad_finite"] and t["score_finite"], t)
save_json(ST, RESULTS / "self_test_v4.json")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V0 — CLIP-only (G0) + v3 cuối (tham chiếu R3) ==============================
mark_stage("V0", requires=("self_tests",))
PV = make_part("val")
CLIP_RANK_VAL = full_gallery_ranks(TXT_FEAT, VID_FEAT, PV["q_idx"], PV["gallery"], PV["gold"])
CLIP_VAL = metrics_from_ranks(CLIP_RANK_VAL, PV["gold"])
ALT_TAGS, ALT_SCORES = {}, {}
EXPS["G0_clip_only"] = dict(experiment_id="G0_clip_only", stage="V0", description="CLIP-only (không xếp lại bằng đồ thị, α = 1)",
                            design=None, hypothesis="baseline chính", seeds=[], runs={}, status="DONE",
                            val_MRR=(CLIP_VAL["MRR"], 0.0), val_R1=(CLIP_VAL["R@1"], 0.0), rr_val=1.0 / CLIP_RANK_VAL,
                            edges={}, n_params=0, seconds=0.0)
print(f"CLIP-only VAL: MRR {CLIP_VAL['MRR']:.4f} R@1 {CLIP_VAL['R@1']:.4f}")
CFG.rerank_k = CFG.v4_rerank_k                       # k = 50: chọn trên VAL ở v3 (Kaggle), giữ cố định ở v4 — không nhìn test
CAND_RECALL_VAL = float(np.mean(CLIP_RANK_VAL <= CFG.rerank_k))
provenance("k (số ứng viên rerank)", "selected", "chọn trên VAL ở v3 (k = 50), giữ nguyên ở v4", caption_splits=("val",))
v4_tensors()
run_exp("R3_v3_final", "V0", V3_FINAL, "v3 cuối (TypedSAGE + 12 mẫu đường đi, đồ thị E44_no_related) — tham chiếu", "v4 tốt hơn v3")



In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V1 — v4 (QC-RGAT): chọn số lớp trên VAL ==============================
mark_stage("V1", requires=("V0",))
V4_T1 = dict(V3_FINAL, model="v4", v4_layers=1)
run_exp("E10_v4_T1", "V1", V4_T1, "QC-RGAT 1 lớp (đường đi quan hệ ≤ 1 bước tới nhãn của video)", H_V4)
if "V1" in CFG.stages:
    run_exp("E11_v4_T2", "V1", dict(V4_T1, v4_layers=2), "QC-RGAT 2 lớp (đường đi ≤ 2 bước)", "đường đi 2 bước có thêm thông tin")
    FINAL = decide("V1", "E10_v4_T1", ["E11_v4_T2"], "v4_layers")
else:
    NOT_RUN.append(dict(stage="V1", experiment_id="E11_v4_T2", reason="không có trong CFG.stages"))
    FINAL = "E10_v4_T1"
CURRENT = dict(EXPS[FINAL]["design"])



In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V7 — ma trận đối chứng dưới CÙNG cấu hình v4 (G3 = v4 đầy đủ) ==============================
mark_stage("V7", requires=("V1",))
G3_DESIGN = dict(CURRENT)
G_IDS = {"G0": "G0_clip_only", "V3REF": "R3_v3_final"}


def control(gid, design, desc, hyp, same_as=None):
    G_IDS[gid.split("_")[0]] = gid
    src = same_as or next((k for k, e in EXPS.items() if e["status"] == "DONE" and e["design"] == design
                           and sorted(e["runs"]) == sorted(CFG.seeds)), None)
    if src is not None:
        return run_exp(gid, "V7", EXPS[src]["design"], f"{desc} (≡ {src}, dùng lại mô hình)", hyp, reuse=src)
    return run_exp(gid, "V7", design, desc, hyp)


control("G3_v4_full", G3_DESIGN, "v4 đầy đủ: quan hệ thật + vị từ + attention theo truy vấn", H_V4)
if "V7" in CFG.stages:
    control("G1_contains_only", dict(G3_DESIGN, v4_rel=False), "cùng mô hình, KHÔNG cạnh quan hệ (chỉ contains + thị giác)",
            "quan hệ đóng góp ngoài contains")
    control("G2_no_predicate", dict(G3_DESIGN, v4_pred=False), "cùng topology + vai + cổng theo truy vấn, KHÔNG khớp vị từ", H_V4)
    control("G4_pred_shuffled", dict(G3_DESIGN, pred_shuffle=True), "phân phối vị từ hoán vị giữa các cạnh cùng họ",
            "ghép ĐÚNG vị từ vào ĐÚNG cạnh có ích")
    control("G5_rewired", dict(G3_DESIGN, rewire=True), "constrained rewiring (giữ kiểu node, họ, bậc, trọng số, đa tập vị từ)",
            "cấu trúc liên kết thật có ích hơn ngẫu nhiên có ràng buộc")
    control("G6_static_attention", dict(G3_DESIGN, v4_qcond=False), "attention tĩnh (chỉ loại cạnh + trọng số), KHÔNG theo truy vấn",
            "điều kiện theo truy vấn làm quan hệ có ích")
    control("G7_no_role_edges", dict(G3_DESIGN, mask=add_mask(G3_DESIGN["mask"], "no_roles")),
            "HUẤN LUYỆN LẠI không có họ cạnh vai (agent_of, object_of); cùng inventory, các họ khác, giao thức", "họ cạnh vai có giá trị")
    control("G8_no_svo_prep_edges", dict(G3_DESIGN, mask=add_mask(G3_DESIGN["mask"], "no_svo_prep")),
            "HUẤN LUYỆN LẠI không có họ cạnh svo / prep (có vị từ); cùng inventory, các họ khác, giao thức", "họ cạnh svo/prep có giá trị")
    control("V3REF", V3_FINAL, "v3 cuối (tham chiếu)", "v4 tốt hơn v3", same_as="R3_v3_final")
    g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]
    a1, RANDOM_VALID_G3 = randomization_audit(g3, graph_for(dict(G3_DESIGN, rewire=True), CFG.seeds[0])[0])
    a2, SHUFFLE_VALID_G3 = predicate_shuffle_audit(g3, graph_for(dict(G3_DESIGN, pred_shuffle=True), CFG.seeds[0])[0])
    RAND_AUDIT = pd.concat([RAND_AUDIT, a1.assign(control="rewire", config="G3", seed=CFG.seeds[0]),
                            a2.assign(control="predicate_shuffle", config="G3", seed=CFG.seeds[0])], ignore_index=True)
    RAND_AUDIT.to_csv(RESULTS / "graph_randomization_audit.csv", index=False)
else:
    NOT_RUN.append(dict(stage="V7", experiment_id="G1–G6", reason="không có trong CFG.stages"))
    RANDOM_VALID_G3, SHUFFLE_VALID_G3 = RANDOM_VALID, SHUFFLE_VALID
DEC_DF = pd.DataFrame(DECISIONS)
DEC_DF.to_csv(RESULTS / "graph_stage_decisions.csv", index=False)
VAL_TABLE = pd.DataFrame([dict(experiment_id=k, stage=e["stage"], status=e["status"], description=e["description"],
                               val_MRR=e["val_MRR"][0], val_MRR_std=e["val_MRR"][1], val_R1=e["val_R1"][0],
                               n_seeds=len(e["seeds"]) or (1 if e["status"] == "DONE" else 0), seconds=e.get("seconds"))
                          for k, e in EXPS.items()])
display(VAL_TABLE.round(4))


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐÓNG BĂNG (trước mọi lần chạm test MSVD / tập ngoài của v4) ==============================
mark_stage("freeze", requires=("V7",))
untouched = not TEST_GATE.log and all(CAPS.log.get((p, s), 0) == 0 for p in ("eval", "report", "display") for s in ("test", "ext"))
FROZEN = dict(version="level4b", model="QC-RGAT", v4=dict(dim=CFG.v4_dim, lr=CFG.v4_lr), v3_final=V3_FINAL, final_experiment=FINAL, final_design=CURRENT, g3_design=G3_DESIGN, g_ids=G_IDS, design_base=DESIGN_BASE,
              rerank_k=CFG.rerank_k, learn_clip_w=CFG.learn_clip_w, seeds=CFG.seeds, split_sha1=SPLIT["sha1"],
              ext_sha1=SPLIT["ext_sha1"], ext_status=EXT_STATUS, inventory_sha1=INV.sha1, schema_sha1=SCHEMA["sha1"],
              clip=f"{CFG.clip_model}/{CFG.clip_weights}", n_frames=POOL_SEL["n_frames"], pooling=POOL_SEL["pooling"],
              tagger=TAG_SELECT2, predicate_vocab=len(PRED_VOCAB), relation_quality=dict(automatic=AUTO_STATUS, human=REL_QUALITY_STATUS),
              verified_cells=sorted(VERIFIED_CELLS) if VERIFIED_CELLS is not None else None,
              random_control_valid=dict(base=RANDOM_VALID, g3=RANDOM_VALID_G3),
              predicate_shuffle_valid=dict(base=SHUFFLE_VALID, g3=SHUFFLE_VALID_G3),
              adoption_rule=dict(min_delta=CFG.adopt_min_delta, max_p_holm=CFG.adopt_max_p, default="phương án đơn giản hơn"),
              confirmatory=PRIMARY_COMPARISONS, holm_alpha=CFG.holm_alpha,
              decisions=DECISIONS, experiments={k: dict(stage=e["stage"], status=e["status"], design=e["design"],
                                                        alpha_lam=e.get("alpha_lam"), val_MRR=e["val_MRR"]) for k, e in EXPS.items()},
              stages_not_run=NOT_RUN, msvd_test_looks_before=BENCHMARK_LOOKS_BEFORE["msvd_test"], benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,
              confirm_set=CFG.confirm_set, eval_query_rule=dict(n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed),
              note=("test MSVD đã được nhìn ở v1–v4 ⇒ ở đây chỉ là THĂM DÒ (lần 5), không dùng để chọn; "
                    "xác nhận = MSR-VTT fresh1k-B (1000 video CHƯA dùng ở v1–v4), đánh giá MỘT lần sau đóng băng"))
TEST_GATE.freeze(FROZEN)
check("D1", "mọi lựa chọn của v4 chỉ dựa trên VAL và đóng băng TRƯỚC khi chạm test MSVD / tập ngoài", untouched,
      f"sha1={TEST_GATE.frozen_sha}; final={FINAL}; G3={G_IDS.get('G3')}; quyết định={[(d['stage'], d['candidate'], d['adopted']) for d in DECISIONS]}")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC (dựng sau đóng băng, TRƯỚC khi chấm bất kỳ mô hình nào) ==============================
mark_stage("eval_queries", requires=("freeze",))
_scored_before = sum(CAPS.log.get(("eval", s_), 0) for s_ in ("val", "test", "ext"))
_G3_GRAPH = graph_for(G3_DESIGN, CFG.seeds[0])[0]
_qs_parts = [("explore", make_part("val" if CFG.dev_mode else "test"))] + ([("confirm", make_part("ext"))] if EXT_RECORDS else [])
QS_ALL, QS_INFO = [], {}
for _k, _p in _qs_parts:
    _df, _inf = build_eval_queries(_p, _G3_GRAPH)
    QS_ALL.append(_df.assign(part_key=_k))
    QS_INFO[_k] = dict(part=_p["name"], **_inf)
QS = pd.concat(QS_ALL, ignore_index=True)
QS.to_csv(RESULTS / "eval_queries_level4b.csv", index=False)
QS_SHA1 = sha1_of(QS[["query_id", "q_idx", "type"]].values.tolist())
save_json(dict(rule=build_eval_queries.__doc__, n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed, sha1=QS_SHA1,
               model_scores_used=False, eval_calls_before_build=_scored_before, graph="G3 (đồ thị cuối, cạnh từ caption TRAIN)",
               tags="CLIP zero-shot trên frame của chính video (tagger chọn trên VAL)", info=QS_INFO), RESULTS / "eval_queries_level4b.json")
provenance("bộ truy vấn đánh giá Level 4(b)", "evaluation", "luật cố định: nhãn CLIP của video đúng + cạnh đồ thị train; không dùng điểm mô hình",
           caption_splits=tuple(sorted({p_["split"] for _, p_ in _qs_parts})))
print(f"Bộ truy vấn đăng ký trước (sha1 {QS_SHA1}): {QS_INFO}")
display(QS[["query_id", "type", "text", "gold_video_id", "concepts_seen_in_video", "concepts_missing", "expected_relations"]].head(8))
_main_k = "confirm" if "confirm" in QS_INFO else "explore"
check("QS1", f"bộ truy vấn trực tiếp / gián tiếp ≥ {CFG.n_eval_queries} / nhóm trên tập đánh giá chính, theo luật đăng ký trước, "
             "dựng TRƯỚC khi chấm mô hình (không dùng điểm mô hình)",
      _scored_before == 0 and all(QS_INFO[_main_k][g_]["selected"] >= CFG.n_eval_queries for g_ in ("direct", "indirect")),
      {k_: {g_: v_[g_]["selected"] for g_ in ("direct", "indirect")} for k_, v_ in QS_INFO.items()}, hard=False)

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐÁNH GIÁ sau đóng băng: test MSVD (THĂM DÒ) + tập ngoài (XÁC NHẬN) ==============================
# dev_mode: test MSVD bị khoá ⇒ phần 'thăm dò' chạy trên VAL (gắn nhãn DEV); tập ngoài = video TRAIN-9k của MSR-VTT (DEV).
mark_stage("evaluate", requires=("freeze",))
EVAL_SPLIT = "val" if CFG.dev_mode else "test"
_, E_G3 = graph_for(G3_DESIGN, CFG.seeds[0])
resolve = lambda x: FINAL if x == "FINAL" else G_IDS.get(x, x)


def eval_part(part, kind):
    """Đánh giá MỌI thí nghiệm đã chạy trên một tập truy vấn; trả về thứ hạng, nhóm truy vấn, bảng metric."""
    CAPS.use(part["q_idx"], "eval")
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, part["q_idx"], part["gallery"], part["gold"])
    groups = query_groups_v3(E_G3, part["q_idx"])
    pg = predicate_groups(E_G3, part["q_idx"])
    masks = {"all": np.ones(len(groups), bool), "direct": groups == "direct", "indirect": groups == "indirect",
             "other": groups == "other", "rare_concepts": rare_mask(E_G3, part["q_idx"]),
             "pred_rare": pg == "pred_rare", "pred_common": pg == "pred_common", "no_pred": pg == "no_pred"}
    ev, hashes = {"G0_clip_only": {None: dict(rank=clip_rank)}}, {}
    t0 = time.time()
    for k, e in EXPS.items():
        if e["runs"]:
            ev[k] = evaluate_exp(k, part, clip_rank)
            hashes[k] = {r["cand_hash"] for r in ev[k].values()}
    rows = []
    for k, ev_k in ev.items():
        e = EXPS[k]
        for grp, m in masks.items():
            if m.sum() < 5:
                continue
            ms = [metrics_from_ranks(v["rank"][m], part["gold"][m]) for v in ev_k.values()]
            rows.append(dict(part=part["name"], kind=kind, experiment_id=k, stage=e["stage"], description=e["description"], group=grp,
                             n_queries=int(m.sum()), n_videos=ms[0]["n_videos"], n_seeds=len(ms),
                             **{f"{x}_mean": float(np.mean([mm[x] for mm in ms])) for x in METRIC_KEYS},
                             **{f"{x}_std": float(np.std([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR", "MRR_video_macro")},
                             val_MRR=e["val_MRR"][0]))
    info = dict(part=part["name"], kind=kind, n_queries=len(part["q_idx"]), n_videos=len(part["gallery"]),
                candidate_recall=float(np.mean(clip_rank <= CFG.rerank_k)), groups=dict(Counter(groups.tolist())),
                predicate_groups=dict(Counter(pg.tolist())), seconds=round(time.time() - t0, 1),
                cand_hashes=sorted(set().union(*hashes.values())) if hashes else [])
    return dict(part=part, ev=ev, masks=masks, rows=pd.DataFrame(rows), info=info)


def rr_of(E, k, m=None):
    rr = np.mean([1.0 / v["rank"] for v in E["ev"][k].values()], 0)
    return rr if m is None else rr[m]


def paired(E, a, b, grp="all"):
    m = E["masks"][grp]
    bs = cluster_bootstrap_diff(rr_of(E, a, m), rr_of(E, b, m), E["part"]["gold"][m])
    return dict(part=E["part"]["name"], kind=E["info"]["kind"], a=a, b=b, group=grp, n=int(m.sum()),
                MRR_a=float(rr_of(E, a, m).mean()), MRR_b=float(rr_of(E, b, m).mean()), **bs)


EVALS = {}
EVALS["explore"] = eval_part(make_part(EVAL_SPLIT), "DEV_val" if CFG.dev_mode else "exploratory_test_5th_look")
if EXT_RECORDS:
    EVALS["confirm"] = eval_part(make_part("ext"), "DEV_ext_train9k" if EXT_REPORT["dev"] else
                                  ("confirmatory" if EXT_REPORT.get("set") == "fresh1kB" else "exploratory_seen_benchmark"))
    EVALS["confirm_std"] = eval_part(make_part("ext", std_only=True), "DEV_ext_train9k_std" if EXT_REPORT["dev"] else
                                      ("confirmatory_secondary" if EXT_REPORT.get("set") == "fresh1kB" else "exploratory_seen_benchmark_std"))
for k_, E in EVALS.items():
    print(f"[{k_}] {E['info']}")
check("D2", "mọi thí nghiệm dùng chung split, gallery, truy vấn, tập ứng viên CLIP và hàm metric (từng tập đánh giá)",
      all(len(E["info"]["cand_hashes"]) == 1 for E in EVALS.values()), {k_: E["info"]["cand_hashes"] for k_, E in EVALS.items()})

COMPARE = []
for d in DECISIONS:
    if d.get("adopted") is not None and d["candidate"] in EXPS and EXPS[d["candidate"]]["runs"] and EXPS[d["incumbent"]].get("runs") is not None:
        COMPARE.append((f"{d['stage']}_variant", d["candidate"], d["incumbent"]))
for gid in ("G1", "G2", "G4", "G5", "G6", "G7", "G8", "V3REF"):
    if gid in G_IDS and EXPS.get(G_IDS[gid], {}).get("runs"):
        COMPARE.append(("G3_vs_control", G_IDS["G3"], G_IDS[gid]))
if "G3" in G_IDS:
    COMPARE.append(("G3_vs_clip", G_IDS["G3"], "G0_clip_only"))       # so sánh PHỤ theo nhóm trực tiếp / gián tiếp
SIG_ROWS, CONF_ROWS = [], []
for k_, E in EVALS.items():
    for c, a, b in COMPARE:
        if a in E["ev"] and b in E["ev"]:
            for g_ in ("all", "direct", "indirect", "rare_concepts", "pred_rare", "pred_common"):
                if E["masks"][g_].sum() >= 5:
                    SIG_ROWS.append(dict(comparison=c, **paired(E, a, b, g_)))
    fam = [(h, resolve(a), resolve(b)) for h, a, b in PRIMARY_COMPARISONS if not (h == "H0_final_vs_clip" and FINAL == G_IDS.get("G3"))]
    ok = [(h, a, b) for h, a, b in fam if a in E["ev"] and b in E["ev"]]
    res = [dict(hypothesis=h, **paired(E, a, b)) for h, a, b in ok]
    adj = holm([r["p_not_better"] for r in res])
    for r, pa in zip(res, adj):
        r.update(p_holm=float(pa), supported=bool(pa < CFG.holm_alpha and r["diff"] > 0),
                 status="CONFIRMATORY" if E["info"]["kind"] == "confirmatory" else
                 ("EXPLORATORY (benchmark đã được nhìn trước đây)" if E["info"]["kind"].startswith("exploratory") else "DEV"))
        CONF_ROWS.append(r)
    for h, a, b in fam:
        if (h, a, b) not in ok:
            CONF_ROWS.append(dict(hypothesis=h, part=E["part"]["name"], kind=E["info"]["kind"], a=a, b=b, status="NOT RUN",
                                  note=f"{a if a not in E['ev'] else b}: {EXPS.get(a if a not in E['ev'] else b, {}).get('reason', 'không có')}"))
if not EXT_RECORDS:
    for h, a, b in PRIMARY_COMPARISONS:
        CONF_ROWS.append(dict(hypothesis=h, part="ext", kind="confirmatory", a=resolve(a), b=resolve(b), status="NOT RUN", note=EXT_STATUS))
SIG_T = pd.DataFrame(SIG_ROWS)
CONF_T = pd.DataFrame(CONF_ROWS)
RES_ALL = pd.concat([E["rows"] for E in EVALS.values()], ignore_index=True)
RES_ALL.to_csv(RESULTS / "graph_experiment_results.csv", index=False)
SIG_T.to_csv(RESULTS / "graph_paired_tests.csv", index=False)
CONF_T.to_csv(RESULTS / "graph_confirmatory_tests.csv", index=False)
save_json({k_: E["info"] for k_, E in EVALS.items()}, RESULTS / "graph_eval_parts.json")
show = ["part", "experiment_id", "n_seeds", "R@1_mean", "R@5_mean", "R@10_mean", "MRR_mean", "MRR_std", "MRR_video_macro_mean", "val_MRR"]
print("\nKẾT QUẢ (nhóm 'all'):")
display(RES_ALL[(RES_ALL.group == "all") & RES_ALL.experiment_id.isin(list(G_IDS.values()) + [FINAL])][show].round(4))
print("Họ so sánh chính (một phía, Holm):")
display(CONF_T[[c for c in ("hypothesis", "part", "status", "a", "b", "MRR_a", "MRR_b", "diff", "ci_lo", "ci_hi", "p_not_better",
                            "p_holm", "supported") if c in CONF_T]].round(4))
TAGGER_REPORT = {k_: tagger_report_v2(TAGS2, SPLIT[E["part"]["split"]], VOCAB2, "report")
                 for k_, E in EVALS.items() if k_ != "confirm_std"}
print("Tagger trên tập đánh giá:", TAGGER_REPORT)
check("D3", "R@1/5/10, MRR, macro-MRR theo video + kiểm định cặp cho G0–G6 + v3 trên tập xác nhận độc lập",
      "confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory",
      f"{len(RES_ALL)} dòng → graph_experiment_results.csv | xác nhận: {EXT_STATUS}", hard=False,
      status=None if ("confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory") else "NOT RUN")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Phân tích quan hệ v4: knockout theo họ, attention theo vị từ / vai ==============================
mark_stage("relations", requires=("evaluate",))
P_MAIN = EVALS["confirm"]["part"] if "confirm" in EVALS else EVALS["explore"]["part"]
G3_ID = G_IDS["G3"]
RA_ROWS, RA_SUMM = [], []
for part in (PV, P_MAIN):
    df_, s_ = relation_analysis_v4(EXPS[G3_ID]["runs"][CFG.seeds[0]], part)
    RA_ROWS.append(df_.assign(experiment_id=G3_ID)); RA_SUMM.append(dict(experiment_id=G3_ID, split=part["name"], **s_))
REL_DF = pd.concat(RA_ROWS, ignore_index=True)
REL_DF.to_csv(RESULTS / "graph_relation_analysis.csv", index=False)
save_json(RA_SUMM, RESULTS / "graph_relation_analysis_summary.json")
display(REL_DF[REL_DF.split == P_MAIN["name"]].round(4))
for s_ in RA_SUMM:
    print(f"{s_['split']:8s} MRR {s_['base_MRR']:.4f} | bỏ MỌI cạnh quan hệ lúc suy luận ⇒ {s_['all_relations_knockout_MRR']:.4f} | "
          f"attention cạnh có vị từ KHỚP truy vấn {s_['att_pred_match']:.3f} vs không khớp {s_['att_pred_nomatch']:.3f}")



In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== xAI v4: giải thích = số hạng của điểm; fidelity bằng xoá cạnh THẬT ==============================
mark_stage("xai", requires=("relations",))
RUN_F = EXPS[G3_ID]["runs"][CFG.seeds[0]]
GROUPS_X = query_groups_v3(RUN_F["engine"], P_MAIN["q_idx"])
XAI_DF = xai_v4(RUN_F, P_MAIN, GROUPS_X)
XAI_DF.to_csv(RESULTS / "graph_xai_fidelity.csv", index=False)
SENS = []
if len(CFG.seeds) > 1 and len(XAI_DF):
    for _, r in XAI_DF.head(CFG.n_seed_sensitivity).iterrows():
        sets = [{tuple(t["edges"]) for t in explain_v4(EXPS[G3_ID]["runs"][s]["model"], int(r.q), int(r.gold))["top_relation"]}
                for s in CFG.seeds]
        js = [len(a & b) / max(1, len(a | b)) for a, b in itertools.combinations(sets, 2)]
        SENS.append(dict(q=int(r.q), gold=int(r.gold), mean_jaccard=float(np.mean(js))))
SENS_DF = pd.DataFrame(SENS)
SENS_DF.to_csv(RESULTS / "graph_xai_seed_sensitivity.csv", index=False)
_f = lambda d, c: float(d[c].mean()) if len(d) and c in d else float("nan")
XAI_SUMM = dict(experiment_id=G3_ID, part=P_MAIN["name"], n_queries=len(XAI_DF), validity=_f(XAI_DF, "validity"),
                predicate_validity=_f(XAI_DF, "predicate_validity"), predicate_edges=int(XAI_DF.n_predicate_edges.sum()) if len(XAI_DF) else 0,
                sparsity=_f(XAI_DF, "sparsity"), share_relation=_f(XAI_DF, "share_relation"),
                decomposition_ok=_f(XAI_DF, "decomposition_ok"), deletion_matches_decomposition=_f(XAI_DF, "deletion_matches_decomposition"),
                d_s_expl=_f(XAI_DF, "d_s_expl"), d_s_rand=_f(XAI_DF, "d_s_rand"),
                frac_score_decreased_expl=float((XAI_DF.d_s_expl < -1e-9).mean()) if len(XAI_DF) else float("nan"),
                rank_full=_f(XAI_DF, "rank_full"), rank_drop_expl=_f(XAI_DF, "rank_drop_expl"), rank_drop_rand=_f(XAI_DF, "rank_drop_rand"),
                frac_rank_changed_expl=float((XAI_DF.rank_drop_expl != XAI_DF.rank_full).mean()) if len(XAI_DF) else float("nan"),
                frac_rank_changed_rand=float((XAI_DF.rank_drop_rand != XAI_DF.rank_full).mean()) if len(XAI_DF) else float("nan"),
                frac_top1_changed_expl=_f(XAI_DF, "top1_changed_expl"), frac_top10_changed_expl=_f(XAI_DF, "top10_changed_expl"),
                seed_sensitivity_mean_jaccard=_f(SENS_DF, "mean_jaccard"), alpha=RUN_F["alpha"])
save_json(XAI_SUMM, RESULTS / "graph_xai_summary.json")
print(json.dumps(XAI_SUMM, indent=1, ensure_ascii=False))
if len(XAI_DF):
    for _, r in XAI_DF.head(3).iterrows():
        print(f"  q={r.q} (nhóm {r.group}) hạng {r.rank_full}→{r.rank_drop_expl} khi xoá giải thích: {r.explanation[:300]}")
check("X1", "[12.9] giải thích hợp lệ (cạnh thật đúng loại/chiều; vị từ khớp nằm trên ĐÚNG cạnh); Σ số hạng = điểm; ≥ 10 truy vấn",
      len(XAI_DF) >= 10 and XAI_SUMM["validity"] == 1.0 and XAI_SUMM["predicate_validity"] == 1.0 and XAI_SUMM["decomposition_ok"] == 1.0,
      XAI_SUMM, hard=False)
check("X2", "xoá cạnh của các đường đi ỦNG HỘ (giải thích) làm điểm mô hình giảm nhiều hơn xoá cùng số cạnh quan hệ ngẫu nhiên (vào nhãn của cùng video)",
      XAI_SUMM["d_s_expl"] < XAI_SUMM["d_s_rand"], f"{XAI_SUMM['d_s_expl']:.4f} vs {XAI_SUMM['d_s_rand']:.4f}", hard=False)
check("X3", "T = 1: Δ điểm khi xoá cạnh = đúng tổng đóng góp đã báo (fidelity chính xác theo cấu tạo)",
      RUN_F["model"].T != 1 or XAI_SUMM["deletion_matches_decomposition"] == 1.0, XAI_SUMM["deletion_matches_decomposition"], hard=False)
# X4 (thứ hạng có đổi khi xoá giải thích?) là KẾT QUẢ khoa học, không phải kiểm thử phần mềm ⇒ báo ở FINDING F-X4 (ô xAI theo thứ hạng).


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — METRIC TRÊN BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC + nhóm trực tiếp / gián tiếp ==============================
mark_stage("eval_queries_metrics", requires=("evaluate", "eval_queries"))
QM, QP = [], []
for _k, E in EVALS.items():
    _q = QS[QS.part_key == _k] if _k in ("explore", "confirm") else QS.iloc[:0]
    if len(_q):
        _a, _b = eval_query_metrics(E, _q)
        QM.append(_a); QP.append(_b)
QS_METRICS = pd.concat(QM, ignore_index=True) if QM else pd.DataFrame()
QS_PAIRED = pd.concat(QP, ignore_index=True) if QP else pd.DataFrame()
QS_METRICS.to_csv(RESULTS / "eval_queries_metrics.csv", index=False)
QS_PAIRED.to_csv(RESULTS / "eval_queries_paired.csv", index=False)
_show = [G_IDS[g_] for g_ in ("G0", "G1", "G3", "G7", "G8", "V3REF") if g_ in G_IDS]
for _k in ("explore", "confirm"):
    if _k in EVALS and len(QS_METRICS):
        _t = QS_METRICS[(QS_METRICS.part == EVALS[_k]["part"]["name"]) & QS_METRICS.experiment_id.isin(_show)]
        print(f"\n[{_k}] bộ truy vấn đăng ký trước — MRR (trung bình theo seed):")
        display(_t.pivot_table(index="experiment_id", columns="group", values="MRR_mean").round(4))
if len(QS_PAIRED):
    display(QS_PAIRED[["part", "group", "a", "b", "n", "MRR_a", "MRR_b", "diff", "ci_lo", "ci_hi"]].round(4))
_mk = "confirm" if "confirm" in EVALS else "explore"
_sig = SIG_T[(SIG_T.comparison == "G3_vs_clip") & (SIG_T.part == EVALS[_mk]["part"]["name"])] if len(SIG_T) else SIG_T
for _, r in _sig[_sig.group.isin(["all", "direct", "indirect"])].iterrows():
    finding(f"F-GRP-{r.group}", f"QC-RGAT (G3) vs CLIP-only trên nhóm '{r.group}' của {r.part} (mọi caption; so sánh PHỤ, không Holm)",
            f"n={r.n}: ΔMRR {r['diff']:+.4f} [CI95 theo video {r.ci_lo:+.4f}, {r.ci_hi:+.4f}]",
            "đồ thị giúp nhóm này" if r.ci_lo > 0 else ("đồ thị làm kém nhóm này" if r.ci_hi < 0 else "không khác biệt có ý nghĩa"))
_qi = QS_PAIRED[(QS_PAIRED.group == "indirect") & (QS_PAIRED.a == G_IDS.get("G3")) & (QS_PAIRED.part == EVALS[_mk]["part"]["name"])] \
    if len(QS_PAIRED) else QS_PAIRED
if len(_qi):
    r = _qi.iloc[0]
    finding("F-QS-indirect", f"G3 vs CLIP-only trên {r.n} truy vấn GIÁN TIẾP đăng ký trước ({r.part})",
            f"MRR {r.MRR_b:.4f} → {r.MRR_a:.4f} (Δ {r['diff']:+.4f}, CI95 [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}])",
            "n nhỏ ⇒ CI rộng; chỉ là bằng chứng bổ sung cho kết quả trên mọi caption")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — xAI theo THỨ HẠNG: giải thích đầy đủ (contains + đường đi quan hệ) ==============================
mark_stage("xai_rank", requires=("xai",))
XR = xai_rank_fidelity(RUN_F, P_MAIN)
XR.to_csv(RESULTS / "xai_rank_fidelity.csv", index=False)
_m = lambda c: float(XR[c].mean()) if len(XR) else float("nan")
XR_SUMM = dict(experiment_id=G3_ID, part=P_MAIN["name"], n_queries=len(XR), top_m=CFG.top_m_paths, alpha=RUN_F["alpha"],
               validity=_m("validity"), predicate_validity=_m("predicate_validity"), decomposition_ok=_m("decomposition_ok"),
               size_edges=_m("n_expl_edges"), sparsity=_m("sparsity"), coverage=_m("coverage"),
               expl_contains_terms=_m("n_expl_contains"), expl_relation_terms=_m("n_expl_relation"),
               d_s_expl=_m("d_s_expl"), d_s_rand=_m("d_s_rand"), d_final_expl=_m("d_final_expl"), d_final_rand=_m("d_final_rand"),
               rank_full=_m("rank_full"), rank_drop_expl=_m("rank_drop_expl"), rank_drop_rand=_m("rank_drop_rand"),
               frac_rank_worse_expl=_m("rank_worse_expl"), frac_rank_worse_rand=_m("rank_worse_rand"),
               frac_top1_changed_expl=_m("top1_changed_expl"), frac_top1_changed_rand=_m("top1_changed_rand"),
               frac_top10_changed_expl=_m("top10_changed_expl"), frac_top10_changed_rand=_m("top10_changed_rand"),
               share_graph=_m("share_graph"), share_relation=_m("share_relation"),
               definitions=dict(explanation="top-m số hạng ĐỒ THỊ có đóng góp dương (contains: video –chứa→ nhãn; quan hệ: u –r→ nhãn ←chứa– video)",
                                validity="tỉ lệ số hạng giải thích mà MỌI cạnh có thật trong đồ thị đang chạy, đúng loại, chiều, kiểu node",
                                sparsity="số hạng giải thích / số hạng đồ thị khác 0 của video (nhỏ = gọn)", size_edges="số cạnh bị xoá",
                                coverage="Σ đóng góp của giải thích / Σ đóng góp đồ thị dương",
                                fidelity="xoá THẬT các cạnh của giải thích rồi chấm lại mọi ứng viên: Δ điểm đồ thị, Δ điểm cuối, Δ hạng, top-1/top-10; "
                                         "đối chứng = xoá cùng số cạnh contains khác + cạnh quan hệ ngẫu nhiên vào nhãn của cùng video"))
save_json(XR_SUMM, RESULTS / "xai_level4b_summary.json")
print(json.dumps({k_: (round(v_, 4) if isinstance(v_, float) else v_) for k_, v_ in XR_SUMM.items() if k_ != "definitions"}, ensure_ascii=False))
if len(XR):
    display(XR[["gold_video_id", "rank_full", "rank_drop_expl", "rank_drop_rand", "d_s_expl", "d_s_rand", "d_final_expl", "explanation"]].head(6))
check("X5", "giải thích đầy đủ hợp lệ (mọi cạnh contains / quan hệ có thật, đúng loại, chiều, kiểu node; vị từ đúng cạnh); "
            "Σ số hạng = điểm; ≥ 10 truy vấn", len(XR) >= 10 and XR_SUMM["validity"] == 1.0 and XR_SUMM["predicate_validity"] == 1.0
      and XR_SUMM["decomposition_ok"] == 1.0, {k_: XR_SUMM[k_] for k_ in ("n_queries", "validity", "predicate_validity", "decomposition_ok")},
      hard=False)
finding("F-X4", "xoá các đường đi QUAN HỆ của giải thích (v4) có đổi thứ hạng video đúng?",
        f"{XAI_SUMM['n_queries']} truy vấn: hạng đổi {XAI_SUMM['frac_rank_changed_expl']:.0%} (xoá ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}); "
        f"Δ điểm đồ thị {XAI_SUMM['d_s_expl']:+.4f} vs ngẫu nhiên {XAI_SUMM['d_s_rand']:+.4f}",
        "điểm giảm đúng như phân rã nhưng quyết định xếp hạng KHÔNG đổi ⇒ quan hệ không phải nguyên nhân của thứ hạng"
        if XAI_SUMM["frac_rank_changed_expl"] == 0 else "xoá quan hệ của giải thích có làm đổi thứ hạng ở một phần truy vấn")
_better = XR_SUMM["frac_rank_worse_expl"] > XR_SUMM["frac_rank_worse_rand"] and XR_SUMM["d_final_expl"] < XR_SUMM["d_final_rand"]
finding("F-X6", "xoá giải thích ĐẦY ĐỦ (contains + quan hệ) có làm video đúng tụt hạng nhiều hơn xoá ngẫu nhiên cùng số cạnh?",
        f"{XR_SUMM['n_queries']} truy vấn: tụt hạng {XR_SUMM['frac_rank_worse_expl']:.0%} vs {XR_SUMM['frac_rank_worse_rand']:.0%}; "
        f"hạng TB {XR_SUMM['rank_full']:.2f} → {XR_SUMM['rank_drop_expl']:.2f} (ngẫu nhiên {XR_SUMM['rank_drop_rand']:.2f}); "
        f"Δ điểm cuối {XR_SUMM['d_final_expl']:+.3f} vs {XR_SUMM['d_final_rand']:+.3f}; top-1 đổi {XR_SUMM['frac_top1_changed_expl']:.0%}",
        "giải thích chỉ ra đúng phần đồ thị quyết định thứ hạng (mạnh hơn ngẫu nhiên)" if _better else
        "KHÔNG mạnh hơn xoá ngẫu nhiên ⇒ không kết luận được giải thích phản ánh nguyên nhân của thứ hạng")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — 5 THÀNH CÔNG + 5 THẤT BẠI (chọn theo luật, có đường đi thật + nguyên nhân) ==============================
mark_stage("cases", requires=("xai_rank",))
CASES, CASE_STATS = case_analysis_l4b(RUN_F, P_MAIN)
save_json(dict(experiment_id=G3_ID, part=P_MAIN["name"], alpha=RUN_F["alpha"], stats=CASE_STATS, cases=CASES), RESULTS / "case_analysis.json")
(RESULTS / "case_analysis.md").write_text(cases_markdown(CASES, CASE_STATS, P_MAIN["name"]))
for c in CASES:
    print(f"[{c['kind']}] hạng CLIP {c['rank_clip']} → {c['rank_final']} | {c['query'][:70]} | nguyên nhân: {', '.join(c['causes']) or '—'}")
    for x in c["gold_explanation"][:3]:
        print(f"      • {x}")
_ns, _nf = sum(c["kind"] == "THÀNH CÔNG" for c in CASES), sum(c["kind"] == "THẤT BẠI" for c in CASES)
check("K1", f"phân tích ≥ {CFG.n_cases} thành công + ≥ {CFG.n_cases} thất bại, chọn theo luật, có đường đi thật, thứ hạng và nguyên nhân",
      _ns >= CFG.n_cases and _nf >= CFG.n_cases,
      f"{_ns} thành công, {_nf} thất bại; nguyên nhân có mặt: {sorted({x for c in CASES for x in c['causes']})}", hard=False)

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — DEMO: văn bản tự do → top-K video + điểm CLIP / đồ thị / cuối + giải thích ==============================
mark_stage("demo", requires=("cases",))
DEMO_OUT = []
for _i, _t in enumerate(CFG.demo_queries, 1):
    DEMO_OUT.append(demo_query(_t, save_png=RESULTS / f"demo_{_i}.png"))
for _kind in ("THÀNH CÔNG", "THẤT BẠI"):                     # một ca có đáp án mỗi loại (văn bản của tập đánh giá, chỉ hiển thị)
    c = next((c for c in CASES if c["kind"] == _kind), None)
    if c is not None:
        _o = demo_query(c["query"], save_png=RESULTS / f"demo_case_{'ok' if _kind == 'THÀNH CÔNG' else 'fail'}.png")
        _o.update(gold_video_id=c["gold_video_id"], case_kind=_kind)
        DEMO_OUT.append(_o)
save_json(DEMO_OUT, RESULTS / "demo_outputs.json")
_ok = all(len(o["results"]) == CFG.demo_topk and all(r["explanation"] or r["note"] for r in o["results"]) for o in DEMO_OUT)
check("DM1", "demo: văn bản tự do → top-k video + điểm CLIP / đồ thị / cuối + giải thích (hoặc báo rõ không có đường đi hợp lệ)",
      _ok and len(DEMO_OUT) >= len(CFG.demo_queries), f"{len(DEMO_OUT)} truy vấn demo → demo_outputs.json, demo_*.png", hard=False)
print("\nThử truy vấn khác: demo_query('a man is riding a horse')  |  ô nhập tương tác: demo_widget()")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== XUẤT KẾT QUẢ THEO CÂU HỎI + MANIFEST ==============================
mark_stage("export", requires=("xai",))
PARTS = [k_ for k_ in ("explore", "confirm", "confirm_std") if k_ in EVALS]


def result_table(ids, groups=("all", "direct", "indirect")):
    """Một dòng / thí nghiệm: thiết kế, trạng thái, VAL, và metric trên từng tập đánh giá (thăm dò / xác nhận) theo nhóm."""
    rows = []
    label = {v: k for k, v in G_IDS.items()}
    for k in dict.fromkeys(ids):
        if k not in EXPS:
            continue
        e = EXPS[k]
        base = dict(experiment_id=k, control=label.get(k, ""), is_final=k == FINAL, stage=e["stage"], status=e["status"],
                    reason=e.get("reason", ""), description=e["description"], hypothesis=e.get("hypothesis", ""),
                    design=json.dumps(e["design"], ensure_ascii=False), n_seeds=len(e["seeds"]),
                    edges=json.dumps(e.get("edges", {})), n_params=e.get("n_params"), runtime_s=e.get("seconds"),
                    gpu_peak_mb=e.get("gpu_peak_mb"), val_MRR=e["val_MRR"][0], val_MRR_std=e["val_MRR"][1], val_R1=e["val_R1"][0])
        for pk in PARTS:
            sub = EVALS[pk]["rows"]
            sub = sub[sub.experiment_id == k]
            for grp in groups:
                r = sub[sub.group == grp]
                if len(r):
                    r = r.iloc[0]
                    tag = f"{pk}_{grp}"
                    base.update({f"{tag}_{x}": r[f"{x}_mean"] for x in ("R@1", "R@5", "R@10", "MRR", "MRR_video_macro")})
                    base[f"{tag}_MRR_std"] = r["MRR_std"]
        rows.append(base)
    return pd.DataFrame(rows)


G_ORDER = [G_IDS[g] for g in ("G0", "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "V3REF") if g in G_IDS]
ABL_DF = result_table(G_ORDER + [FINAL] + [k for k, e in EXPS.items() if e["stage"] == "V1"],
                      ("all", "direct", "indirect", "pred_rare", "pred_common", "no_pred"))
ABL_DF.to_csv(RESULTS / "graph_ablation_results.csv", index=False)

MANIFEST = []
for k, e in EXPS.items():
    MANIFEST.append(dict(experiment_id=k, control={v: g for g, v in G_IDS.items()}.get(k, ""), stage=e["stage"], status=e["status"],
                         reason=e.get("reason"), description=e["description"], hypothesis=e.get("hypothesis", ""), design=e["design"],
                         seeds=e["seeds"], reuse=e.get("reuse"), lam_fixed=e.get("lam_fixed"),
                         checkpoints=[r.get("checkpoint") for r in e["runs"].values()], edges_by_type=e.get("edges"),
                         n_nodes=e.get("n_nodes"), n_params=e.get("n_params"), alpha_lam=e.get("alpha_lam"),
                         val=dict(MRR=e["val_MRR"], R1=e["val_R1"]), runtime_s=e.get("seconds"), gpu_peak_mb=e.get("gpu_peak_mb"),
                         eval={pk: {r_.group: dict(MRR=r_["MRR_mean"], MRR_std=r_["MRR_std"], R1=r_["R@1_mean"])
                                    for _, r_ in EVALS[pk]["rows"][EVALS[pk]["rows"].experiment_id == k].iterrows()} for pk in PARTS},
                         per_seed={str(s): dict(val=r["val"], alpha=r["alpha"], lam=r["lam"], seconds=r["seconds"],
                                                final_loss=r["hist"][-1] if r["hist"] else None) for s, r in e["runs"].items()}))
save_json(dict(version="level4b", final=FINAL, benchmark_status=benchmark_status(), g_ids=G_IDS, rerank_k=CFG.rerank_k, eval_parts={k_: E["info"] for k_, E in EVALS.items()},
               candidate_recall_val=CAND_RECALL_VAL, not_run=NOT_RUN, primary_comparisons=PRIMARY_COMPARISONS,
               experiments=MANIFEST, env=ENV_INFO), RESULTS / "experiment_manifest.json")
PROV = pd.DataFrame(PROVENANCE)
PROV.to_csv(RESULTS / "provenance.csv", index=False)
ACCESS = CAPS.access_table()
ACCESS.to_csv(RESULTS / "caption_access_log.csv", index=False)
bad_access = ACCESS[ACCESS.split.isin(["test", "ext"]) & ~ACCESS.purpose.isin(["encode", "query_input", "eval", "display", "report"])]
check("C8", "[12.8] nhật ký: caption TEST / tập ngoài chỉ dùng cho encode / query_input / eval / report", bad_access.empty,
      {f"{r.purpose}:{r.split}": int(r.n) for r in ACCESS.itertuples()})
bad_prov = PROV[((PROV.kind == "learned") & PROV.caption_splits.str.contains("val|test|ext"))
                | ((PROV.kind == "selected") & PROV.caption_splits.str.contains("test|ext"))]
check("C9", "[12.8] provenance: artifact học được chỉ từ caption TRAIN; lựa chọn chỉ từ VAL", bad_prov.empty, f"{len(PROV)} artifact")
save_json(TEST_GATE.log, RESULTS / "test_access_log.json")
first = min((x["t"] for x in TEST_GATE.log), default=None)
check("D4", "mọi lần chạm test MSVD / tập ngoài sau thời điểm đóng băng", first is None or first >= TEST_GATE.frozen_at,
      f"{len(TEST_GATE.log)} lần ({dict(Counter(x['split'] for x in TEST_GATE.log))})")
r_ = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(RESULTS / "pip_freeze.txt").write_text(r_.stdout if r_.returncode == 0 else r_.stderr[-2000:])
save_json(dict(ENV_INFO, cfg=cfg_snapshot(), stages=STAGES), RESULTS / "env_versions.json")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== BÁO CÁO AUDIT + KẾT LUẬN v4 (tự sinh từ số liệu) ==============================
mark_stage("report", requires=("export",))
official = CFG.dataset == "msvd" and not CFG.dev_mode
_by = {c["id"]: c for c in CHECKS}


def _st(*ids):
    rows = [_by[i] for i in ids if i in _by]
    if not rows:
        return "NOT RUN", "không chạy"
    st = "FAIL" if any(r["status"] == "FAIL" for r in rows) else ("NOT RUN" if any(r["status"] == "NOT RUN" for r in rows) else "PASS")
    return st, " | ".join(f"{r['id']}={r['status']}" for r in rows)


def _verdict(lo, hi):
    return "TỐT HƠN có ý nghĩa" if lo > 0 else ("KÉM HƠN có ý nghĩa" if hi < 0 else "không khác biệt có ý nghĩa")


def _conf(hyp, part_key):
    if part_key not in EVALS:
        return "NOT RUN" + (f" ({EXT_STATUS})" if part_key.startswith("confirm") else "")
    r = CONF_T[(CONF_T.hypothesis == hyp) & (CONF_T.part == EVALS[part_key]["part"]["name"])]
    if r.empty or r.iloc[0].get("status") == "NOT RUN":
        return "NOT RUN" + (f" — {r.iloc[0].get('note', '')}" if len(r) else "")
    r = r.iloc[0]
    return (f"{r.a} vs {r.b}: ΔMRR {r['diff']:+.4f} (CI95 theo video [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}], p Holm {r.p_holm:.3f}) "
            f"⇒ {_verdict(r.ci_lo, r.ci_hi)}{' — ỦNG HỘ giả thuyết' if r.supported else ''}")


def _val(a, b):
    if not (EXPS.get(a, {}).get("runs") or a == "G0_clip_only") or not (EXPS.get(b, {}).get("runs") or b == "G0_clip_only"):
        return "NOT RUN"
    bs = cluster_bootstrap_diff(EXPS[a]["rr_val"], EXPS[b]["rr_val"], PV["gold"])
    return f"{a} vs {b} (VAL): ΔMRR {bs['diff']:+.4f} [{bs['ci_lo']:+.4f}, {bs['ci_hi']:+.4f}]"


CK = "confirm" if "confirm" in EVALS else None
EX = "explore"
L = []
if not official:
    L.append(f"⚠ KHÔNG phải lần chạy chính thức (dataset={CFG.dataset}, dev_mode={CFG.dev_mode}) — số liệu chỉ để kiểm tra mã.")
L.append(f"⚠ Khai báo: test MSVD đã được nhìn ở v1–v4 ⇒ số test MSVD ở đây là THĂM DÒ (lần nhìn thứ 5). Xác nhận độc lập: "
         f"{EXT_STATUS} ({EXT_REPORT.get('label', '—')}).")
L.append(f"Mô hình v4 = QC-RGAT ({CFG.v4_dim} chiều, {CURRENT['v4_layers']} lớp), đồ thị = cấu hình cuối của v3 "
         f"({json.dumps({k: V3_FINAL[k] for k in ('taxonomy', 'mask', 'rel_policy', 'contains')}, ensure_ascii=False)}); k = {CFG.rerank_k}; "
         f"chọn số lớp trên VAL: {FINAL}.")
L.append(f"Chất lượng quan hệ — {AUTO_STATUS}")
HYPS = (("H1_graph_vs_clip", "v4 (G3) vs CLIP-only (G0)"), ("H2_relations_vs_contains_only", "Quan hệ vs chỉ contains (G3 vs G1)"),
        ("H3_predicate_vs_family_only", "Khớp vị từ vs không vị từ (G3 vs G2)"),
        ("H4_true_vs_shuffled_predicates", "Vị từ đúng cạnh vs hoán vị (G3 vs G4)"),
        ("H5_true_vs_rewired_topology", "Topology thật vs rewiring có ràng buộc (G3 vs G5)"),
        ("H6_query_conditioned_vs_static", "Attention theo truy vấn vs tĩnh (G3 vs G6)"),
        ("H7_v4_vs_v3", "QC-RGAT vs GraphSAGE có kiểu cạnh (G3 vs R3)"),
        ("H8_role_edges", "Họ cạnh vai agent_of / object_of (G3 vs G7 huấn luyện lại)"),
        ("H9_svo_prep_edges", "Họ cạnh svo / prep (G3 vs G8 huấn luyện lại)"))
for hyp, txt in HYPS:
    a, b = PRIMARY_COMPARISONS[[h for h, _, _ in PRIMARY_COMPARISONS].index(hyp)][1:]
    L.append(f"{hyp} — {txt}:")
    L.append(f"     VAL (phát triển): {_val(resolve(a), resolve(b))}")
    L.append(f"     {'test MSVD (THĂM DÒ)' if not CFG.dev_mode else 'VAL lặp lại (DEV)'}: {_conf(hyp, EX)}")
    L.append(f"     tập ngoài {'(XÁC NHẬN)' if CK and EVALS[CK]['info']['kind'] == 'confirmatory' else '(DEV)' if CK else ''}: "
             f"{_conf(hyp, CK) if CK else 'NOT RUN — ' + EXT_STATUS}")
L.append(f"Đối chứng (dưới cấu hình G3): rewiring hợp lệ: {RANDOM_VALID_G3}; hoán vị vị từ hợp lệ: {SHUFFLE_VALID_G3}. "
         "Đối chứng không hợp lệ ⇒ KHÔNG rút kết luận tương ứng.")
for s_ in RA_SUMM:
    L.append(f"Phân tích quan hệ ({s_['split']}): bỏ mọi cạnh quan hệ lúc suy luận: MRR {s_['base_MRR']:.4f} → "
             f"{s_['all_relations_knockout_MRR']:.4f}; attention cạnh có vị từ khớp truy vấn {s_['att_pred_match']:.3f} vs không khớp "
             f"{s_['att_pred_nomatch']:.3f}.")
L.append(f"Giải thích ({XAI_SUMM['experiment_id']}, {XAI_SUMM['part']}, {XAI_SUMM['n_queries']} truy vấn): hợp lệ {XAI_SUMM['validity']:.2f}, "
         f"vị từ đúng cạnh {XAI_SUMM['predicate_validity']:.2f}, Σ số hạng = điểm {XAI_SUMM['decomposition_ok']:.2f}, xoá cạnh khớp phân rã "
         f"{XAI_SUMM['deletion_matches_decomposition']:.2f}; Δ điểm khi xoá giải thích {XAI_SUMM['d_s_expl']:+.4f} vs ngẫu nhiên "
         f"{XAI_SUMM['d_s_rand']:+.4f}; hạng đổi {XAI_SUMM['frac_rank_changed_expl']:.0%} (ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}); "
         f"phần điểm từ quan hệ {XAI_SUMM['share_relation']:.1%}; Jaccard theo seed {XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}.")
conf_sup = {r.hypothesis: bool(r.supported) for r in CONF_T.itertuples()
            if CK and r.part == EVALS[CK]["part"]["name"] and getattr(r, "status", "") == "CONFIRMATORY"}
REL_KEYS = ("H2_relations_vs_contains_only", "H4_true_vs_shuffled_predicates", "H5_true_vs_rewired_topology")
if conf_sup and all(conf_sup.get(h) for h in REL_KEYS) and RANDOM_VALID_G3 and SHUFFLE_VALID_G3:
    VERDICT = ("Trên tập xác nhận độc lập, v4 vượt có ý nghĩa (Holm) cả đối chứng chỉ-contains, vị từ hoán vị và topology rewired ⇒ "
               "quan hệ (đúng cạnh, đúng vị từ) có giá trị thật cho xếp hạng.")
elif conf_sup:
    VERDICT = ("Quan hệ CHƯA được xác nhận đầy đủ: trên tập xác nhận, v4 không vượt có ý nghĩa mọi đối chứng quan hệ "
               f"(ủng hộ: {[h for h, v in conf_sup.items() if v] or '—'}). Cơ chế dùng quan hệ có thật (P/Q/X-test) nhưng tác động lên MRR "
               "không đủ để kết luận.")
else:
    VERDICT = "Chưa có kết luận xác nhận: tập xác nhận độc lập NOT RUN hoặc DEV. Số VAL / test MSVD chỉ là phát triển / thăm dò."
L.append("KẾT LUẬN: " + VERDICT)
_tf1 = ", ".join(f"{k_} {v_['F1']:.3f}" for k_, v_ in TAGGER_REPORT.items())
L.append(f"Giới hạn: nhãn video từ CLIP zero-shot (F1 {_tf1}); quan hệ là thống kê cấp LỚP từ caption train (không có quan hệ cấp video "
         "cho video mới); parser quan hệ heuristic; val chỉ 100 video; PASS của kiểm thử phần mềm không xác nhận giả thuyết khoa học.")
CONCLUSION = "\n".join(L)
(RESULTS / "graph_conclusion.md").write_text("# Kết luận đồ thị — Level 4(b) (tự sinh từ số liệu)\n\n" + "\n".join(f"- {x}" for x in L) + "\n")

_csv_conf = pd.read_csv(RESULTS / "graph_confirmatory_tests.csv")
_csv_abl = pd.read_csv(RESULTS / "graph_ablation_results.csv")
_m1 = CONF_T.dropna(subset=["diff"]) if "diff" in CONF_T else CONF_T.iloc[:0]
_m2 = _csv_conf.dropna(subset=["diff"]) if "diff" in _csv_conf else _csv_conf.iloc[:0]
ok_conf = len(_m1) == len(_m2) and np.allclose(_m1["diff"].values, _m2["diff"].values) and \
    np.allclose(_m1["p_holm"].values, _m2["p_holm"].values)
ok_abl = all(abs(float(_csv_abl.loc[_csv_abl.experiment_id == k, "val_MRR"].iloc[0]) - EXPS[k]["val_MRR"][0]) < 1e-9
             for k in _csv_abl.experiment_id if EXPS[k]["status"] == "DONE")
ok_text = all(f"{r['diff']:+.4f}" in CONCLUSION for _, r in _m1.iterrows()
              if r["hypothesis"] in {h for h, _, _ in PRIMARY_COMPARISONS} and r["part"] in {EVALS[k_]["part"]["name"] for k_ in (EX, CK) if k_})
check("R1", "[12.10] kết quả trong báo cáo / kết luận khớp các file CSV đầu ra (đọc lại từ đĩa)", ok_conf and ok_abl and ok_text,
      dict(confirmatory=ok_conf, ablation_val=ok_abl, conclusion_text=ok_text))

REQUIRED = ["graph_schema_v3.json", "graph_edges_v3.csv", "relation_quality_audit.csv", "graph_randomization_audit.csv",
            "graph_ablation_results.csv", "graph_relation_analysis.csv", "graph_xai_fidelity.csv", "frozen_config.json",
            "experiment_manifest.json", "graph_conclusion.md"]


def _nonempty(p):
    if not p.exists() or p.stat().st_size == 0:
        return False
    if p.suffix == ".csv":
        try:
            return len(pd.read_csv(p)) > 0
        except Exception:
            return False
    return True


files = {f: _nonempty(RESULTS / f) for f in REQUIRED}
criteria = [
    ("1", "Quan hệ được DÙNG có điều kiện theo truy vấn (vị từ, vai, loại cạnh tác động attention)", *_st("Q1", "Q2")),
    ("2", "Điểm = Σ số hạng giải thích (phân rã chính xác); xoá cạnh ⇒ đúng phần đóng góp", *_st("Q3", "X1", "X3")),
    ("3", "Có đối chứng tách quan hệ / vị từ / topology / điều kiện truy vấn (G1, G2, G4, G5, G6 + audit)", *_st("G6", "G7")),
    ("4", "Ablation công bằng (inventory cố định, cùng ứng viên, cùng giao thức huấn luyện)", *_st("G1", "D2")),
    ("5", "Không dùng test đã xem để chọn mô hình", *_st("D1", "D4", "C3")),
    ("6", "Đủ bằng chứng xác định tốt hơn / tương đương / kém hơn (tập xác nhận độc lập, Holm)", *_st("D3")),
    ("7", "Mọi kết luận có nguồn số liệu kiểm chứng", *_st("R1")),
]
md = ["# Audit đồ thị — Level 4(b): QC-RGAT + đối chứng cho text→video (MSVD → MSR-VTT)", "",
      f"{time.strftime('%Y-%m-%d %H:%M:%S')} | dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | Kaggle={RUN_ON_KAGGLE} | "
      f"{ENV_INFO.get('gpu', DEVICE)} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | tập ngoài: {EXT_STATUS}", "",
      "## Tiêu chí nghiệm thu", "", "| # | Tiêu chí | Trạng thái | Bằng chứng |", "|---|---|---|---|"]
md += [f"| {a} | {b} | **{c}** | {d} |" for a, b, c, d in criteria]
md += ["", "## File bàn giao", "", "| File | Có dữ liệu |", "|---|---|"] + [f"| {f} | {'PASS' if ok else 'FAIL'} |" for f, ok in files.items()]
md += ["", "## Quyết định theo stage (chỉ VAL; incumbent = phương án đơn giản hơn)", "",
       "| stage | incumbent | ứng viên | Δ val MRR | p Holm | chấp nhận |", "|---|---|---|---|---|---|"]
md += [f"| {d['stage']} | {d['incumbent']} | {d['candidate']} | {d['delta_val_MRR']:+.4f} | {d['p_holm']:.3f} | {d['adopted']} |"
       if d.get("value") is not None else f"| {d['stage']} | {d['incumbent']} | {d['candidate']} | NOT RUN | — | False |" for d in DECISIONS]
md += ["", "## Thí nghiệm NOT RUN", ""] + ([f"- {n['stage']} {n.get('experiment_id', '')}: {n['reason']}" for n in NOT_RUN] or ["- (không có)"])
md += ["", "## Kết luận", ""] + [f"- {x}" for x in L]
md += ["", "## Mọi kiểm tra", "", "| id | kiểm tra | trạng thái |", "|---|---|---|"] + [f"| {c['id']} | {c['check']} | {c['status']} |" for c in CHECKS]
(RESULTS / "graph_audit_report.md").write_text("\n".join(md) + "\n")
display(pd.DataFrame(criteria, columns=["#", "tiêu chí", "trạng thái", "bằng chứng"]))
print("\n" + CONCLUSION)
print(f"\nFile bàn giao có dữ liệu: {sum(files.values())}/{len(files)} | kiểm tra FAIL: {sum(c['status'] == 'FAIL' for c in CHECKS)} "
      f"→ {RESULTS / 'graph_audit_report.md'}")


In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — KIỂM KÊ DỮ LIỆU · AUDIT RÒ RỈ · NGHIỆM THU · KẾT LUẬN · MANIFEST (tự sinh từ lần chạy này) ==============================
mark_stage("level4b_report", requires=("report", "demo"))
BENCH = benchmark_status()
_g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]
GRAPH_INV = graph_inventory(_g3)
_cache_dirs = [CFG.out_dir / "cache"] + [Path(d) for d in CFG.extra_cache_dirs] + \
    ([p for p in Path("/kaggle/input").glob("*/**/cache") if p.is_dir()] if RUN_ON_KAGGLE else [])
_caches = [dict(file=p.name, dir=str(p.parent), mb=round(p.stat().st_size / 2 ** 20, 1), used_by_this_run=CFG.clip_model in p.name and
                CFG.clip_weights in p.name) for d in _cache_dirs if d.exists() for p in sorted(d.glob("*.npz"))]
_ck = sorted((CFG.out_dir / "checkpoints").glob("*")) if (CFG.out_dir / "checkpoints").exists() else []
DATA_INVENTORY = dict(
    created=time.strftime("%Y-%m-%d %H:%M:%S"), official_run=CFG.dataset == "msvd" and not CFG.dev_mode, dev_mode=CFG.dev_mode,
    datasets=[dict(key="msvd", **DATA_SOURCES["msvd"], videos_loaded=N_MSVD, video_source=DATA_REPORT.get("source_label"),
                   splits={s: DATA_REPORT["splits"][s] for s in DATA_REPORT["splits"]}),
              dict(key="msrvtt", **DATA_SOURCES["msrvtt"], status=EXT_STATUS,
                   report={k_: v_ for k_, v_ in EXT_REPORT.items() if k_ != "missing_examples"})],
    loaded_counts_by_split=split_counts(), benchmark_status=BENCH, benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,
    split_manifest=dict(file="split_manifest.json", msvd_sha1=SPLIT["sha1"], ext_sha1=SPLIT.get("ext_sha1"),
                        rule="split CHÍNH THỨC của MSVD theo video; tập ngoài theo luật seed cố định (load_msrvtt)"),
    features=dict(clip=f"{CFG.clip_model}/{CFG.clip_weights} (đóng băng)", frames_per_video=CFG.n_frames_max,
                  frame_policy=CFG.frame_policy, pooling_selected_on_val=POOL_SEL, video_dim=int(VID_FEAT.shape[1]),
                  n_video_embeddings=int(VID_FEAT.shape[0]), n_text_embeddings=int(TXT_FEAT.shape[0]), caches=_caches),
    tagger=dict(selected_on_val=TAG_SELECT2, report=TAGGER_REPORT),
    graph=dict(design=G3_DESIGN, **GRAPH_INV), checkpoints=dict(n_files=len(_ck), mb=round(sum(p.stat().st_size for p in _ck) / 2 ** 20, 1),
                                                               names=[p.name for p in _ck][:60]),
    environment=ENV_INFO)
save_json(DATA_INVENTORY, RESULTS / "data_inventory.json")
check("I1", "kiểm kê: số video / caption theo split khớp dữ liệu đã nạp; manifest split có sha1; cache có metadata mô hình",
      all(DATA_INVENTORY["loaded_counts_by_split"][s]["videos"] == len(SPLIT[s]) for s in DATA_INVENTORY["loaded_counts_by_split"] if s in SPLIT)
      and bool(SPLIT["sha1"]), {s: (v["videos"], v["captions"]) for s, v in DATA_INVENTORY["loaded_counts_by_split"].items()}, hard=False)
check("I2", "đồ thị đang chạy: không self-loop, không cạnh trùng (cùng nguồn, đích, loại); có thống kê bậc và node cô lập",
      GRAPH_INV["self_loops"] == 0 and GRAPH_INV["duplicate_edges"] == 0,
      {k_: GRAPH_INV[k_] for k_ in ("n_edges", "self_loops", "duplicate_edges", "isolated_non_video_nodes", "videos_without_tag")}, hard=False)

# ---------------- data_leakage_audit.md ----------------
_by = {c["id"]: c for c in CHECKS}
_LEAK = [("A5", "split theo video, không video nào ở hai split"), ("C7", "split MSVD tái lập được"),
         ("C1", "canary chỉ có ở caption test / tập ngoài KHÔNG lọt vào từ vựng, quan hệ, vị từ"), ("C1b", "đối chứng dương của canary"),
         ("C2", "xáo caption test / tập ngoài ⇒ mọi artifact học được GIỐNG HỆT"), ("C3", "hàng rào chặn mở test / dùng caption sai mục đích"),
         ("C8", "nhật ký: caption test / tập ngoài chỉ dùng cho encode / query_input / eval / report"),
         ("C9", "provenance: học chỉ từ train, chọn chỉ từ val"), ("D1", "mọi lựa chọn trước đóng băng, chỉ dựa trên val"),
         ("D4", "mọi lần chạm test / tập ngoài sau đóng băng"), ("A6", "tập ngoài rời các tập đã dùng"), ("QS1", "bộ truy vấn dựng trước khi chấm mô hình")]
_acc = CAPS.access_table()
L = ["# Audit rò rỉ dữ liệu — Level 4(b) (tự sinh)", "",
     f"{DATA_INVENTORY['created']} | chạy chính thức: {DATA_INVENTORY['official_run']} | tập ngoài: {EXT_STATUS}", "",
     "## Trạng thái từng benchmark", "", "| Tập | Đã nhìn trước | Trạng thái |", "|---|---|---|"]
L += [f"| {k_} | {v_.get('looks_before', '—')} | **{v_['status']}** |" for k_, v_ in BENCH.items()]
L += ["", "## Kiểm tra chống rò rỉ", "", "| id | nội dung | trạng thái | bằng chứng |", "|---|---|---|---|"]
L += [f"| {i} | {d} | {_by[i]['status'] if i in _by else 'NOT RUN'} | {(_by[i]['evidence'][:220] if i in _by else '').replace('|', '/')} |" for i, d in _LEAK]
L += ["", "## Nhật ký dùng caption (mục đích × split)", "", "| mục đích | split | số lần |", "|---|---|---|"]
L += [f"| {r.purpose} | {r.split} | {int(r.n)} |" for r in _acc.itertuples()]
L += ["", "## Chất lượng ngữ nghĩa của cạnh", "",
      f"- Kiểm chứng tự động (parser độc lập, SILVER — không phải precision): {AUTO_STATUS}",
      f"- Nhãn người chấm: {REL_QUALITY_STATUS}",
      "- ⇒ Chất lượng ngữ nghĩa của cạnh quan hệ **CHƯA được xác nhận bởi người**. Tỉ lệ đồng thuận giữa hai parser không phải precision thật.", "",
      "## Nguồn và giấy phép", ""] + [f"- **{v_['name']}** ({v_['paper']}) — `{v_['repo']}@{v_['revision'][:8]}` — {v_['license']} — vai trò: {v_['role']}"
                                     for v_ in DATA_SOURCES.values()]
(RESULTS / "data_leakage_audit.md").write_text("\n".join(L) + "\n")

# ---------------- audit_report.md: ma trận nghiệm thu (mục 12 của master instruction) ----------------
def _req_st(*ids):
    st, ev = _st(*ids)
    return st, ev


def _done(*gids):
    ks = [G_IDS.get(g_, g_) for g_ in gids]
    ok = all(EXPS.get(k_, {}).get("status") == "DONE" and (EXPS[k_].get("runs") or k_ == "G0_clip_only") for k_ in ks)   # CLIP-only không huấn luyện
    return ("PASS" if ok else "NOT RUN"), ", ".join(f"{k_}={EXPS.get(k_, {}).get('status', 'không có')}" for k_ in ks)


def _file(*names):
    ok = all(_nonempty(RESULTS / n) for n in names)
    return ("PASS" if ok else "FAIL"), ", ".join(names)


def _all(*pairs):
    sts = [p[0] for p in pairs]
    st = "FAIL" if "FAIL" in sts else ("NOT RUN" if "NOT RUN" in sts else "PASS")
    return st, " ; ".join(p[1] for p in pairs)


_n_seeds_ok = ("PASS" if len(CFG.seeds) >= 3 else "FAIL", f"{len(CFG.seeds)} seed {CFG.seeds}")
_conf_ok = ("PASS" if "confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory" else "NOT RUN",
            f"tập ngoài: {EXT_STATUS}; kind={EVALS['confirm']['info']['kind'] if 'confirm' in EVALS else '—'}")
REQ = [
    ("A1", "Baseline CLIP đóng băng", *_all(_req_st("B0"), _done("G0_clip_only"))),
    ("A2", "Đồ thị từ nguồn hợp lệ (caption TRAIN + CLIP zero-shot trên frame), đúng schema", *_req_st("G2", "C1", "C9")),
    ("A3", "Ít nhất một GNN đã học (GraphSAGE có kiểu cạnh = R3; QC-RGAT = G3)", *_done("V3REF", "G3")),
    ("A4", "CLIP-only và CLIP + Graph so sánh trên cùng split, cùng ứng viên", *_all(_req_st("D2"), _done("G0_clip_only", "G3"))),
    ("A5", "Ablation quan hệ (G1, G7, G8 huấn luyện lại) + đồ thị ngẫu nhiên hợp lệ (G4, G5)", *_all(_done("G1", "G4", "G5", "G7", "G8"), _req_st("G6", "G7", "AB"))),
    ("A6", "R@1, R@5, R@10, MRR; ≥ 3 seed cho thí nghiệm chính", *_all(_n_seeds_ok, _req_st("D3"))),
    ("B1", "Schema node / cạnh rõ ràng", *_file("graph_schema_v3.json")),
    ("B2", "Nguồn gốc cạnh được ghi (bảng cạnh + provenance)", *_file("graph_edges_v3.csv", "provenance.csv")),
    ("B3", "Không rò rỉ test", *_req_st("A5", "C1", "C1b", "C2", "C3", "C8", "C9", "D1", "D4")),
    ("B4", "Audit cấu trúc (inventory cố định, schema, trọng số, self-loop / trùng / bậc)", *_req_st("G1", "G2", "G4", "I2")),
    ("B5", "Đối chứng ngẫu nhiên đúng thuộc tính bảo toàn đã công bố", *_req_st("G6", "G7")),
    ("B6", "Chất lượng ngữ nghĩa của cạnh được người xác minh", "NOT RUN",
     "chưa có nhãn người chấm ⇒ chưa xác nhận; chỉ có kiểm chứng tự động SILVER (spaCy)"),
    ("C1", "Pipeline text-to-video thật (giải mã video, lấy mẫu frame, gộp, gallery)", *_req_st("A3", "A4", "B1", "B3", "B6", "B7", "B8")),
    ("C2", "Encoder tiền huấn luyện đóng băng + lấy mẫu frame", *_req_st("B0", "B6")),
    ("C3", "Gallery, bộ truy vấn và metric xác định rõ (gồm bộ ≥ 20 truy vấn / nhóm)", *_req_st("B8", "QS1")),
    ("C4", "Xếp hạng lại bằng đồ thị + baseline, đánh giá trên tập CHƯA nhìn", *_all(_req_st("D3"), _conf_ok)),
    ("C5", "Giải thích lấy từ chính cơ chế xếp hạng", *_req_st("Q3", "X1", "X3", "X5")),
    ("C6", "Phân tích thành công / thất bại và giới hạn", *_req_st("K1")),
    ("D1", "Đường đi / quan hệ giải thích là thành phần thật của phép tính", *_req_st("Q3", "X3")),
    ("D2", "Validity, sparsity, fidelity (điểm + thứ hạng) trên ≥ 10 truy vấn", *_all(_req_st("X1", "X5"), _file("xai_rank_fidelity.csv"))),
    ("D3", "≥ 5 thành công + ≥ 5 thất bại được phân tích", *_req_st("K1")),
    ("D4", "Demo: nhập truy vấn → top-k + điểm + đường đi giải thích", *_req_st("DM1")),
    ("E1", "Notebook chạy hết (hoặc báo rõ giới hạn)", "PASS" if CFG.dataset == "msvd" and not CFG.dev_mode else "NOT RUN",
     "chạy chính thức tới ô này" if CFG.dataset == "msvd" and not CFG.dev_mode else f"dataset={CFG.dataset}, dev_mode={CFG.dev_mode}"),
    ("E2", "File môi trường + manifest", *_file("pip_freeze.txt", "env_versions.json", "experiment_manifest.json", "data_inventory.json")),
    ("E3", "Kết quả báo cáo khớp file nguồn", *_req_st("R1")),
]
REQ_DF = pd.DataFrame(REQ, columns=["id", "yêu cầu", "trạng thái", "bằng chứng"])
REQ_DF.to_csv(RESULTS / "requirements_status.csv", index=False)
md = ["# Audit nghiệm thu — Project 3 · Level 4(b) text-to-video (tự sinh)", "",
      f"{DATA_INVENTORY['created']} | chính thức: {DATA_INVENTORY['official_run']} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | "
      f"tập ngoài: {EXT_STATUS} | Kaggle={RUN_ON_KAGGLE}", "",
      "PASS ở đây là PASS của **kiểm thử / sản phẩm**, không phải kết luận khoa học. Kết luận khoa học nằm ở `level4b_conclusions.md`.", "",
      "| id | yêu cầu | trạng thái | bằng chứng |", "|---|---|---|---|"]
md += [f"| {a} | {b} | **{c}** | {str(d)[:240].replace('|', '/')} |" for a, b, c, d in REQ]
md += ["", "## Kết quả khoa học được báo cáo (FINDINGS)", "", "| id | câu hỏi | kết quả | diễn giải |", "|---|---|---|---|"]
md += [f"| {f['id']} | {f['question']} | {f['result'].replace('|', '/')} | {f['interpretation']} |" for f in FINDINGS]
md += ["", "## Thí nghiệm NOT RUN", ""] + ([f"- {n['stage']} {n.get('experiment_id', '')}: {n['reason']}" for n in NOT_RUN] or ["- (không có)"])
(RESULTS / "audit_report.md").write_text("\n".join(md) + "\n")
display(REQ_DF)

# ---------------- level4b_conclusions.md: 7 câu hỏi bắt buộc (mục 13) ----------------
_P = EVALS[CK]["part"]["name"] if CK else EVALS[EX]["part"]["name"]
_kind = EVALS[CK]["info"]["kind"] if CK else EVALS[EX]["info"]["kind"]


def _row(h):
    r = CONF_T[(CONF_T.hypothesis == h) & (CONF_T.part == _P)]
    return r.iloc[0] if len(r) and "diff" in r and pd.notna(r.iloc[0].get("diff")) else None


def _fmt(h):
    r = _row(h)
    if r is None:
        return "NOT RUN"
    return (f"ΔMRR {r['diff']:+.4f} (CI95 theo video, chưa hiệu chỉnh bội [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}] ⇒ {_verdict(r.ci_lo, r.ci_hi)}; "
            f"giả thuyết một phía sau Holm: p {r.p_holm:.3f} ⇒ {'ỦNG HỘ' if r.supported else 'KHÔNG ủng hộ'})")


def _mrr(k_, part=_P, grp="all"):
    r = RES_ALL[(RES_ALL.experiment_id == k_) & (RES_ALL.part == part) & (RES_ALL.group == grp)]
    return f"{r.iloc[0]['MRR_mean']:.4f} ± {r.iloc[0]['MRR_std']:.4f}" if len(r) else "NOT RUN"


_fam = REL_DF[(REL_DF.section == "family_knockout") & (REL_DF.split == P_MAIN["name"])] if len(REL_DF) else REL_DF
Q = []
Q.append(("1. CLIP + Graph có tốt hơn CLIP-only không?",
          f"Trên {_P} ({_kind}): CLIP-only MRR {_mrr('G0_clip_only')}; QC-RGAT (G3) {_mrr(G_IDS.get('G3'))}; GraphSAGE có kiểu cạnh (R3) "
          f"{_mrr(G_IDS.get('V3REF'))}. H1 (G3 > CLIP): {_fmt('H1_graph_vs_clip')}."))
_gi = SIG_T[(SIG_T.comparison == "G3_vs_clip") & (SIG_T.part == _P) & (SIG_T.group == "indirect")] if len(SIG_T) else SIG_T
_qs = QS_PAIRED[(QS_PAIRED.part == _P) & (QS_PAIRED.group == "indirect") & (QS_PAIRED.a == G_IDS.get("G3"))] if len(QS_PAIRED) else QS_PAIRED
Q.append(("2. Đồ thị có giúp nhóm truy vấn gián tiếp không?",
          (f"Mọi caption nhóm gián tiếp (n={int(_gi.iloc[0].n)}): ΔMRR {_gi.iloc[0]['diff']:+.4f} [CI {_gi.iloc[0].ci_lo:+.4f}, {_gi.iloc[0].ci_hi:+.4f}]"
           if len(_gi) else "NOT RUN") + "; " +
          (f"bộ {int(_qs.iloc[0].n)} truy vấn gián tiếp đăng ký trước: ΔMRR {_qs.iloc[0]['diff']:+.4f} [CI {_qs.iloc[0].ci_lo:+.4f}, {_qs.iloc[0].ci_hi:+.4f}]"
           if len(_qs) else "bộ truy vấn: NOT RUN") + ". So sánh PHỤ (không Holm)."))
Q.append(("3. Những loại cạnh nào thực sự có giá trị?",
          f"Quan hệ vs chỉ-contains (H2): {_fmt('H2_relations_vs_contains_only')}. Họ vai agent_of/object_of (H8, huấn luyện lại không có họ): "
          f"{_fmt('H8_role_edges')}. Họ svo/prep (H9): {_fmt('H9_svo_prep_edges')}. Vị từ đúng cạnh (H4): {_fmt('H4_true_vs_shuffled_predicates')}. "
          f"Topology thật (H5): {_fmt('H5_true_vs_rewired_topology')}. Knockout lúc suy luận ({P_MAIN['name']}): "
          + ("; ".join(f"{r.item} {r.d_MRR_knockout:+.4f}" for r in _fam.itertuples()) if len(_fam) else "NOT RUN")
          + ". Một họ cạnh chỉ được coi là có giá trị khi H tương ứng được ủng hộ."))
Q.append(("4. Dùng đồ thị có làm lời giải thích đáng tin cậy hơn không?",
          f"Giải thích = số hạng thật của điểm (Σ = điểm: {XR_SUMM['decomposition_ok']:.2f}; hợp lệ {XR_SUMM['validity']:.2f}). Xoá giải thích đầy đủ: "
          f"tụt hạng {XR_SUMM['frac_rank_worse_expl']:.0%} truy vấn vs ngẫu nhiên {XR_SUMM['frac_rank_worse_rand']:.0%}; chỉ xoá quan hệ: hạng đổi "
          f"{XAI_SUMM['frac_rank_changed_expl']:.0%} (ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}). Jaccard giải thích giữa seed "
          f"{XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}. ⇒ Giải thích trung thực với ĐIỂM theo cấu tạo (phân rã chính xác). Về QUYẾT ĐỊNH xếp hạng: "
          + ("xoá giải thích đầy đủ làm tụt hạng nhiều hơn xoá ngẫu nhiên, nên giải thích có chỉ ra phần đồ thị gây ra thứ hạng"
             if _better else "xoá giải thích đầy đủ KHÔNG mạnh hơn xoá ngẫu nhiên, nên không kết luận được giải thích phản ánh nguyên nhân thứ hạng")
          + ("; riêng phần QUAN HỆ hầu như không đổi thứ hạng" if XAI_SUMM["frac_rank_changed_expl"] <= XAI_SUMM["frac_rank_changed_rand"] + 0.05
             else "; phần quan hệ có đổi thứ hạng ở một số truy vấn")
          + f". Độ ổn định giữa seed thấp (Jaccard {XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}) ⇒ đường đi cụ thể phụ thuộc lần huấn luyện."))
Q.append(("5. Phần video tái sử dụng kiến trúc retrieval và graph explanation đến mức nào?",
          "Tái sử dụng nguyên: CLIP đóng băng (ảnh → frame), top-k theo cosine, điểm cuối α·z(CLIP) + (1−α)·z(đồ thị) chọn α trên val, "
          "schema node / cạnh (video ~ ảnh), GNN (GraphSAGE có kiểu cạnh, QC-RGAT), giải thích bằng phân rã điểm và xoá cạnh. "
          "Phần riêng của video: lấy mẫu 12 frame, gộp theo thời gian (chọn trên val), nhãn video = CLIP zero-shot trên frame, "
          f"quan hệ từ caption train cấp lớp. Bằng chứng chạy: {_st('A4', 'B6', 'B8')[1]}."))
_cnt = Counter(REQ_DF["trạng thái"])
Q.append(("6. Yêu cầu nào PASS, FAIL hay NOT RUN?",
          f"{dict(_cnt)}. FAIL: {', '.join(REQ_DF[REQ_DF['trạng thái'] == 'FAIL'].id) or '—'}; NOT RUN: "
          f"{', '.join(REQ_DF[REQ_DF['trạng thái'] == 'NOT RUN'].id) or '—'} (chi tiết: audit_report.md)."))
Q.append(("7. Level 4(b) được thực hiện đến đâu và giới hạn là gì?",
          f"Đã có pipeline text→video đầy đủ, xếp hạng lại bằng đồ thị, đối chứng, bộ truy vấn trực tiếp/gián tiếp, xAI theo thứ hạng, 5+5 ca, demo. "
          f"Tập xác nhận: {EXT_STATUS} ({EXT_REPORT.get('label', '—')}). Giới hạn: nhãn video là CLIP zero-shot (F1 "
          + ", ".join(f"{k_} {v_['F1']:.3f}" for k_, v_ in TAGGER_REPORT.items()) +
          "); quan hệ là thống kê cấp lớp từ caption train (video mới không có quan hệ riêng); parser heuristic; chất lượng ngữ nghĩa "
          "cạnh chưa được người xác minh; val chỉ 100 video; test MSVD đã nhìn 4 lần trước (thăm dò); fresh1k-B cùng miền với fresh1k đã ảnh "
          "hưởng tới thiết kế. PASS kiểm thử phần mềm không phải kết luận khoa học."))
CONCL = ["# Kết luận Level 4(b) — trả lời 7 câu hỏi bắt buộc (tự sinh từ số liệu của lần chạy này)", "",
         f"Tập chính: **{_P}** ({_kind}). Kết luận chung về quan hệ: {VERDICT}", ""]
for q_, a_ in Q:
    CONCL += [f"## {q_}", "", a_, ""]
(RESULTS / "level4b_conclusions.md").write_text("\n".join(CONCL))
print("\n".join(CONCL))

# ---------------- artifacts_manifest.json ----------------
_desc = {"eval_queries_level4b.csv": "bộ truy vấn trực tiếp / gián tiếp đăng ký trước", "eval_queries_metrics.csv": "metric trên bộ truy vấn",
         "xai_rank_fidelity.csv": "fidelity theo thứ hạng (giải thích đầy đủ)", "case_analysis.md": "5 thành công + 5 thất bại",
         "demo_outputs.json": "đầu ra demo", "data_inventory.json": "kiểm kê dữ liệu", "data_leakage_audit.md": "audit rò rỉ",
         "audit_report.md": "ma trận nghiệm thu", "level4b_conclusions.md": "7 câu kết luận", "graph_ablation_results.csv": "ablation (1 dòng / thí nghiệm)",
         "graph_experiment_results.csv": "metric mọi thí nghiệm × tập × nhóm", "graph_confirmatory_tests.csv": "họ so sánh chính (Holm)",
         "graph_paired_tests.csv": "so sánh phụ theo nhóm", "experiment_manifest.json": "cấu hình, seed, checkpoint mọi thí nghiệm",
         "split_manifest.json": "split tái lập", "graph_schema_v3.json": "schema đồ thị", "provenance.csv": "nguồn gốc artifact"}
ARTIFACTS = [dict(path=p.relative_to(CFG.out_dir).as_posix(), bytes=p.stat().st_size, sha1=file_sha1(p), description=_desc.get(p.name, ""))
             for p in sorted(RESULTS.rglob("*")) if p.is_file() and p.name != "artifacts_manifest.json"]
save_json(dict(created=time.strftime("%Y-%m-%d %H:%M:%S"), out_dir=str(CFG.out_dir), n_files=len(ARTIFACTS),
               not_packed=dict(thumbs=len(list((CFG.out_dir / "thumbs").glob("*.jpg"))), checkpoints=len(_ck)), files=ARTIFACTS),
          RESULTS / "artifacts_manifest.json")
print(f"\nartifacts_manifest.json: {len(ARTIFACTS)} file | yêu cầu: {dict(_cnt)} | FINDINGS: {len(FINDINGS)}")

In [ ]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== GÓI KẾT QUẢ ĐỂ TẢI VỀ (không gồm dữ liệu) ==============================
mark_stage("pack", requires=("report",))
_z = pack_review("hoàn tất")
assert _z is not None, "Không tạo được file zip xem xét — xem cảnh báo phía trên."
with zipfile.ZipFile(_z) as _zf:
    _names = _zf.namelist()
    _man = json.loads(_zf.read(f"{_z.stem}/pack_manifest.json"))
print(f"📦 {_z} — {_z.stat().st_size / 2**20:.1f} MB, {len(_names)} file. Tải file này về (Kaggle: Output → {_z.name}).")
print("   Notebook đã chạy:", "có (executed_notebook.ipynb, đủ output tới cell trước)" if _man["executed_notebook"]
      else "KHÔNG tìm thấy (chạy ngoài Kaggle/papermill) — tải notebook riêng nếu cần")
print("   Không gói:", "; ".join(f"{k} {v['n_files']} file, {v['mb']} MB" if isinstance(v, dict) else k for k, v in _man["excluded"].items()))

In [ ]:
# ============ CHUYỂN PHẦN: giải phóng bộ nhớ GPU/RAM của phần A trước khi chạy phần B ============
import gc
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as _e:
    print("Bỏ qua dọn GPU:", _e)
gc.collect()
print("Phần A xong. Kết quả: /kaggle/working/results/ và /kaggle/working/output/action_genome/results/")


---
# PHẦN B — pipeline ảnh + video (Visual Genome, ARO, SugarCrepe, COCO-spatial, MSVD)


# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị (notebook final)

Notebook tự chứa, chạy một lần từ đầu đến cuối trên Kaggle (GPU T4, khoảng 3 giờ: video ~2–2,5 giờ + ảnh ~30 phút), bao phủ các yêu cầu cấp 1, 2, 3 (phần giải thích) và cấp 4 hướng (a), (b) của đề cương.
**Ngoài phạm vi phần B:** bộ 20 truy vấn gián tiếp nằm ở phần A; MMKG tự xây nằm ở `mmkg/` (notebook riêng).

| Mục | Nội dung | Yêu cầu đề cương |
|---|---|---|
| 1–5 | Thiết lập · dữ liệu nhúng · lõi đồ thị · dữ liệu + luật chống rò rỉ · mô hình đóng băng (CLIP ViT-L/14, RelTR) | Mục 4, 6 |
| 4v | **Video** MSVD (text-to-video): notebook Level4B v5 — CLIP + PE-Core + đồ thị bằng chứng (hub, cầu nối) + Qwen2.5-VL | Cấp 4 (b) |
| 6 | **TN1** văn bản → ảnh, full-gallery: CLIP top-100 → điểm đồ thị (đường đi trên scene graph VG) → xếp hạng lại α | Cấp 1 (nhánh A) |
| 6b | **TN1b** GraphSAGE (GNN đã học) làm điểm đồ thị, cùng pipeline; ablation cạnh ngẫu nhiên / bỏ cạnh | Cấp 1 (thuật toán đã học) |
| 6c | **TN1c** SCENIR (ICML 2025), Graph Autoencoder không giám sát, cùng pipeline; ablation | Cấp 2 (phương pháp từ 2024) |
| 7 | **TN2** ARO VG-Relation + SugarCrepe: ảnh → caption đúng, scene graph RelTR từ điểm ảnh | Cấp 2 (≥ 2 tập), vai trò đồ thị |
| 7b | **TN2b** cạnh hình học suy từ box trên COCO-spatial (What'sUp) | Cấp 4 (a): tập công khai thứ hai |
| 8 | **XAI**: fidelity (xoá bằng chứng, lật ảnh), validity (khớp GT), sufficiency, sparsity + 10 ví dụ | Cấp 3 |
| 10 | Demo `search()`: nhập truy vấn → top-k ảnh + quan hệ giải thích | Mục 8 |
| 10b | 5 truy vấn thành công + 5 thất bại (văn bản → ảnh), luật chọn cố định | Cấp 1 |
| 11a | Độ nhạy α · so sánh cách tạo cạnh · xuất file split | Cấp 2 (mở rộng), mục 8 |
| 11 | Bảng tổng hợp · đối chiếu tái lập · **bảng đối chiếu yêu cầu đề cương** | — |

**Nguyên tắc chống rò rỉ:** không huấn luyện CLIP / RelTR; GNN (GraphSAGE, SCENIR) chỉ học trên ảnh không thuộc tập đánh giá; α chọn trên phần TUNE;
mọi ảnh Visual Genome thuộc VAL (RelTR chưa huấn luyện trên chúng), **không dùng heldout**; ngưỡng, lưới tham số và quy tắc quyết định đã đăng ký trước
(`v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md`, `v6_compositional_benchmarks/PREREGISTRATION_VI.md`, `XAI_PREREGISTRATION_VI.md`,
`final_pipeline/GNN_PREREGISTRATION_VI.md`, `SCENIR_VERIFICATION_VI.md`, `LEVEL4A_PREREGISTRATION_VI.md`).

**Kaggle:** gắn datasets `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`; bật GPU và Internet
(cài `open_clip`, `sentence-transformers`; tải trọng số CLIP, mã + checkpoint RelTR, JSON của ARO / SugarCrepe / What'sUp).
Chạy thử không cần GPU: đặt `P3_SMOKE=1` (dữ liệu giả — không phải kết quả).

## 1. Thiết lập

In [ ]:
# ============================== 1. THIẾT LẬP: thư viện · chế độ chạy · hằng số đã đăng ký trước · tiện ích ==============================
import os, sys, json, csv, time, re, random, hashlib, subprocess, argparse, warnings, glob
from collections import Counter, defaultdict
import numpy as np
warnings.filterwarnings("ignore")

T0 = time.time()
SMOKE = os.environ.get("P3_SMOKE") == "1"            # chạy thử với dữ liệu giả + mô hình giả (KHÔNG phải kết quả)
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.environ.get("P3_OUT", "/kaggle/working/p3_out" if ON_KAGGLE else "./p3_out")
WORK = os.environ.get("P3_WORK", "/kaggle/working/p3_work" if ON_KAGGLE else "./p3_work")
FIG = f"{OUT}/figures"
for _d in (OUT, WORK, FIG): os.makedirs(_d, exist_ok=True)
_LOG = open(f"{OUT}/log.txt", "w", encoding="utf-8")

# Hằng số khoá trước khi chạy (giống hệt các bản đăng ký trước của v5, v6, v6-XAI)
SEEDS = (0, 1, 2)                     # 3 seed: lần rút caption (TN1) / cách chia TUNE-CHECK (TN2)
K_TOP = 100                           # CLIP top-K trước khi xếp hạng lại
ALPHAS = [round(i / 10, 1) for i in range(11)]          # lưới α, có α = 1 (chỉ CLIP)
W_GRID = sorted((a / 4, b / 4, (4 - a - b) / 4) for a in range(5) for b in range(5 - a))   # trọng số 3 kiểu đường đi, bước 0.25
W_OBJ = (0.0, 0.0, 1.0)
TAU = 0.05                            # độ dốc sigmoid của cạnh hình học
FLOOR = 1e-30                         # sàn log như v3
N_BOOT = 10000
SEED_RAND = 20261116
N_RAND_DEL = 10                       # số lần xoá ngẫu nhiên (đối chứng fidelity)
assert len(W_GRID) == 15 and len(ALPHAS) == 11


def elapsed():
    return f"{time.time() - T0:.0f}s"


def p3_log(*a):
    s = f"[{time.time() - T0:8.1f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True); _LOG.write(s + "\n"); _LOG.flush()


def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    p3_log("$", " ".join(cmd)[:150], "->", r.returncode, (r.stderr or "")[-200:].replace("\n", " "))
    return r.returncode


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()


def stream_array(path, chunk=1 << 22):
    """đọc tuần tự một mảng JSON rất lớn (relationships.json ~743 MB) mà không nạp cả file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = f.read(chunk); pos = 0
        while pos < len(buf) and buf[pos] in " \t\r\n﻿": pos += 1
        assert buf[pos] == "[", "top-level JSON is not an array"
        pos += 1
        while True:
            while True:
                while pos < len(buf) and buf[pos] in " \t\r\n,": pos += 1
                if pos < len(buf): break
                more = f.read(chunk)
                if not more: return
                buf = more; pos = 0
            if buf[pos] == "]": return
            while True:
                try:
                    obj, end = dec.raw_decode(buf, pos); break
                except json.JSONDecodeError:
                    more = f.read(chunk)
                    if not more: raise
                    buf = buf[pos:] + more; pos = 0
            pos = end
            yield obj


def fetch(dst, url=None, gdrive=None):
    """tải file công khai (benchmark JSON, checkpoint RelTR) một lần."""
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        if gdrive:
            try:
                import gdown  # noqa
            except ImportError:
                sh([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            sh([sys.executable, "-m", "gdown", gdrive, "-O", dst])
        else:
            sh(["wget", "-q", "-O", dst, url])
    assert os.path.exists(dst) and os.path.getsize(dst) > 0, f"tải thất bại: {url or gdrive}"
    return dst


p3_log(f"Chế độ {'SMOKE (dữ liệu giả)' if SMOKE else 'REAL'} | out={OUT}")


## 2. Dữ liệu nhúng (sinh tự động bởi `build_notebook.py`)
Danh sách id cho luật chống rò rỉ, bảng VG150, số tham chiếu của các lần chạy riêng trước (đối chiếu tái lập).

In [ ]:
# ============================== 2. DỮ LIỆU NHÚNG (sinh tự động — không sửa tay) ==============================
TABLES = {"VG_CLASSES": ["N/A", "airplane", "animal", "arm", "bag", "banana", "basket", "beach", "bear", "bed", "bench", "bike", "bird", "board", "boat", "book", "boot", "bottle", "bowl", "box", "boy", "branch", "building", "bus", "cabinet", "cap", "car", "cat", "chair", "child", "clock", "coat", "counter", "cow", "cup", "curtain", "desk", "dog", "door", "drawer", "ear", "elephant", "engine", "eye", "face", "fence", "finger", "flag", "flower", "food", "fork", "fruit", "giraffe", "girl", "glass", "glove", "guy", "hair", "hand", "handle", "hat", "head", "helmet", "hill", "horse", "house", "jacket", "jean", "kid", "kite", "lady", "lamp", "laptop", "leaf", "leg", "letter", "light", "logo", "man", "men", "motorcycle", "mountain", "mouth", "neck", "nose", "number", "orange", "pant", "paper", "paw", "people", "person", "phone", "pillow", "pizza", "plane", "plant", "plate", "player", "pole", "post", "pot", "racket", "railing", "rock", "roof", "room", "screen", "seat", "sheep", "shelf", "shirt", "shoe", "short", "sidewalk", "sign", "sink", "skateboard", "ski", "skier", "sneaker", "snow", "sock", "stand", "street", "surfboard", "table", "tail", "tie", "tile", "tire", "toilet", "towel", "tower", "track", "train", "tree", "truck", "trunk", "umbrella", "vase", "vegetable", "vehicle", "wave", "wheel", "window", "windshield", "wing", "wire", "woman", "zebra"], "VG_PREDS": ["__background__", "above", "across", "against", "along", "and", "at", "attached to", "behind", "belonging to", "between", "carrying", "covered in", "covering", "eating", "flying in", "for", "from", "growing on", "hanging from", "has", "holding", "in", "in front of", "laying on", "looking at", "lying on", "made of", "mounted on", "near", "of", "on", "on back of", "over", "painted on", "parked on", "part of", "playing", "riding", "says", "sitting on", "standing on", "to", "under", "using", "walking in", "walking on", "watching", "wearing", "wears", "with"]}
CLIP_MODEL, CLIP_WEIGHTS, RELTR_GDRIVE = 'ViT-L-14', 'laion2b_s32b_b82k', '1id6oD_iwiNDD6HyCn2ORgRTIKkPD3tUD'
ARO_GDRIVE = '1kX2iCHEv0CADL8dSO1nMdW-V0NqIAiP3'   # visual_genome_relation.json (kho ARO, Yuksekgonul et al., ICLR 2023)
SC_URL = 'https://raw.githubusercontent.com/RAIVNLab/sugar-crepe/main/data/'   # SugarCrepe (NeurIPS 2023)
VAL_IDS = [2315353, 2315357, 2315358, 2315359, 2315360, 2315361, 2315362, 2315363, 2315366, 2315368, 2315369, 2315370, 2315371, 2315373, 2315374, 2315376, 2315377, 2315378, 2315379, 2315380, 2315381, 2315382, 2315383, 2315384, 2315388, 2315391, 2315392, 2315393, 2315394, 2315395, 2315397, 2315398, 2315399, 2315402, 2315403, 2315405, 2315406, 2315409, 2315410, 2315411, 2315414, 2315415, 2315416, 2315417, 2315422, 2315425, 2315426, 2315427, 2315428, 2315429, 2315432, 2315433, 2315434, 2315439, 2315442, 2315443, 2315444, 2315445, 2315446, 2315447, 2315448, 2315450, 2315451, 2315452, 2315453, 2315455, 2315456, 2315458, 2315459, 2315460, 2315463, 2315465, 2315466, 2315468, 2315470, 2315472, 2315473, 2315476, 2315477, 2315480, 2315482, 2315484, 2315486, 2315489, 2315490, 2315497, 2315499, 2315500, 2315503, 2315504, 2315505, 2315509, 2315510, 2315511, 2315512, 2315513, 2315514, 2315519, 2315520, 2315523, 2315526, 2315528, 2315530, 2315534, 2315539, 2315543, 2315544, 2315546, 2315547, 2315548, 2315549, 2315550, 2315552, 2315553, 2315555, 2315558, 2315559, 2315563, 2315566, 2315568, 2315569, 2315570, 2315575, 2315576, 2315578, 2315579, 2315580, 2315582, 2315584, 2315586, 2315587, 2315590, 2315591, 2315592, 2315593, 2315597, 2315598, 2315600, 2315601, 2315604, 2315611, 2315613, 2315617, 2315618, 2315619, 2315621, 2315622, 2315626, 2315628, 2315629, 2315630, 2315632, 2315634, 2315639, 2315640, 2315643, 2315644, 2315646, 2315647, 2315651, 2315654, 2315656, 2315657, 2315658, 2315661, 2315662, 2315663, 2315665, 2315667, 2315671, 2315673, 2315675, 2315676, 2315677, 2315678, 2315679, 2315683, 2315686, 2315687, 2315691, 2315696, 2315698, 2315699, 2315705, 2315710, 2315712, 2315718, 2315721, 2315724, 2315729, 2315730, 2315733, 2315735, 2315737, 2315739, 2315741, 2315742, 2315743, 2315744, 2315745, 2315746, 2315747, 2315749, 2315750, 2315753, 2315761, 2315762, 2315763, 2315764, 2315765, 2315767, 2315771, 2315773, 2315774, 2315776, 2315779, 2315782, 2315784, 2315785, 2315786, 2315787, 2315788, 2315790, 2315791, 2315792, 2315794, 2315795, 2315797, 2315801, 2315803, 2315804, 2315805, 2315809, 2315810, 2315812, 2315814, 2315815, 2315816, 2315817, 2315818, 2315819, 2315821, 2315823, 2315824, 2315825, 2315826, 2315827, 2315830, 2315833, 2315834, 2315835, 2315836, 2315837, 2315842, 2315843, 2315844, 2315846, 2315847, 2315849, 2315852, 2315853, 2315857, 2315862, 2315865, 2315866, 2315870, 2315872, 2315873, 2315874, 2315878, 2315879, 2315880, 2315881, 2315882, 2315883, 2315887, 2315889, 2315890, 2315891, 2315892, 2315893, 2315894, 2315895, 2315897, 2315898, 2315901, 2315902, 2315903, 2315904, 2315905, 2315908, 2315909, 2315910, 2315913, 2315916, 2315917, 2315919, 2315922, 2315924, 2315925, 2315930, 2315933, 2315935, 2315936, 2315937, 2315938, 2315939, 2315940, 2315941, 2315942, 2315943, 2315947, 2315948, 2315950, 2315951, 2315953, 2315954, 2315955, 2315957, 2315959, 2315960, 2315962, 2315963, 2315964, 2315965, 2315967, 2315969, 2315971, 2315973, 2315975, 2315976, 2315979, 2315980, 2315981, 2315982, 2315983, 2315984, 2315985, 2315986, 2315987, 2315990, 2315992, 2315993, 2315995, 2315996, 2315998, 2316001, 2316002, 2316004, 2316006, 2316010, 2316011, 2316012, 2316013, 2316015, 2316017, 2316021, 2316025, 2316026, 2316028, 2316029, 2316033, 2316035, 2316036, 2316037, 2316040, 2316041, 2316042, 2316043, 2316044, 2316046, 2316047, 2316048, 2316049, 2316050, 2316054, 2316056, 2316058, 2316061, 2316063, 2316065, 2316066, 2316068, 2316070, 2316071, 2316072, 2316073, 2316074, 2316075, 2316076, 2316079, 2316080, 2316082, 2316083, 2316084, 2316090, 2316093, 2316094, 2316095, 2316097, 2316098, 2316099, 2316101, 2316102, 2316103, 2316104, 2316105, 2316107, 2316112, 2316114, 2316115, 2316123, 2316125, 2316128, 2316129, 2316130, 2316131, 2316132, 2316133, 2316135, 2316136, 2316137, 2316138, 2316140, 2316141, 2316142, 2316148, 2316149, 2316155, 2316157, 2316158, 2316160, 2316161, 2316162, 2316164, 2316166, 2316168, 2316169, 2316170, 2316173, 2316174, 2316175, 2316176, 2316177, 2316181, 2316182, 2316183, 2316185, 2316186, 2316187, 2316188, 2316190, 2316191, 2316194, 2316195, 2316197, 2316198, 2316201, 2316202, 2316203, 2316204, 2316205, 2316210, 2316211, 2316214, 2316215, 2316216, 2316218, 2316220, 2316221, 2316222, 2316223, 2316224, 2316226, 2316228, 2316229, 2316230, 2316231, 2316233, 2316234, 2316237, 2316239, 2316241, 2316242, 2316243, 2316244, 2316245, 2316247, 2316251, 2316252, 2316253, 2316254, 2316258, 2316260, 2316261, 2316264, 2316266, 2316268, 2316269, 2316270, 2316271, 2316272, 2316274, 2316275, 2316277, 2316280, 2316281, 2316282, 2316284, 2316287, 2316289, 2316290, 2316293, 2316294, 2316295, 2316297, 2316299, 2316300, 2316303, 2316304, 2316305, 2316306, 2316307, 2316308, 2316311, 2316312, 2316313, 2316314, 2316316, 2316317, 2316318, 2316320, 2316322, 2316325, 2316327, 2316329, 2316330, 2316331, 2316336, 2316337, 2316338, 2316340, 2316344, 2316345, 2316348, 2316350, 2316353, 2316355, 2316356, 2316357, 2316359, 2316360, 2316361, 2316362, 2316363, 2316364, 2316365, 2316366, 2316368, 2316369, 2316372, 2316374, 2316376, 2316377, 2316379, 2316380, 2316381, 2316386, 2316387, 2316391, 2316392, 2316394, 2316395, 2316397, 2316398, 2316401, 2316402, 2316409, 2316410, 2316411, 2316412, 2316413, 2316415, 2316417, 2316419, 2316423, 2316425, 2316426, 2316429, 2316431, 2316432, 2316433, 2316435, 2316436, 2316439, 2316440, 2316443, 2316444, 2316445, 2316446, 2316447, 2316448, 2316450, 2316452, 2316453, 2316455, 2316456, 2316458, 2316459, 2316460, 2316461, 2316462, 2316466, 2316469, 2316471, 2316473, 2316476, 2316478, 2316480, 2316482, 2316484, 2316485, 2316486, 2316487, 2316488, 2316489, 2316490, 2316492, 2316493, 2316494, 2316495, 2316497, 2316499, 2316500, 2316501, 2316503, 2316504, 2316505, 2316506, 2316508, 2316510, 2316511, 2316512, 2316513, 2316514, 2316517, 2316518, 2316519, 2316520, 2316521, 2316522, 2316528, 2316529, 2316530, 2316531, 2316532, 2316534, 2316536, 2316539, 2316540, 2316541, 2316543, 2316544, 2316545, 2316546, 2316547, 2316549, 2316551, 2316553, 2316555, 2316557, 2316558, 2316559, 2316562, 2316563, 2316567, 2316569, 2316571, 2316572, 2316573, 2316574, 2316575, 2316576, 2316583, 2316584, 2316585, 2316587, 2316588, 2316591, 2316592, 2316593, 2316594, 2316595, 2316596, 2316597, 2316598, 2316600, 2316601, 2316604, 2316605, 2316607, 2316608, 2316610, 2316611, 2316612, 2316615, 2316617, 2316619, 2316621, 2316623, 2316624, 2316625, 2316626, 2316630, 2316631, 2316632, 2316635, 2316641, 2316642, 2316644, 2316645, 2316647, 2316648, 2316649, 2316651, 2316652, 2316653, 2316655, 2316656, 2316658, 2316659, 2316661, 2316663, 2316664, 2316666, 2316667, 2316669, 2316672, 2316673, 2316674, 2316675, 2316676, 2316678, 2316679, 2316681, 2316682, 2316684, 2316686, 2316687, 2316688, 2316689, 2316690, 2316691, 2316692, 2316696, 2316700, 2316703, 2316707, 2316708, 2316709, 2316711, 2316716, 2316721, 2316723, 2316726, 2316727, 2316728, 2316730, 2316732, 2316733, 2316734, 2316735, 2316736, 2316739, 2316740, 2316743, 2316745, 2316747, 2316752, 2316756, 2316758, 2316759, 2316763, 2316765, 2316770, 2316771, 2316773, 2316774, 2316775, 2316776, 2316778, 2316779, 2316780, 2316781, 2316783, 2316786, 2316787, 2316789, 2316790, 2316793, 2316797, 2316798, 2316799, 2316800, 2316802, 2316803, 2316804, 2316805, 2316806, 2316808, 2316810, 2316811, 2316812, 2316815, 2316816, 2316818, 2316820, 2316823, 2316824, 2316829, 2316830, 2316831, 2316832, 2316834, 2316836, 2316837, 2316838, 2316840, 2316844, 2316845, 2316846, 2316848, 2316849, 2316852, 2316855, 2316856, 2316858, 2316861, 2316862, 2316864, 2316868, 2316869, 2316870, 2316871, 2316873, 2316874, 2316875, 2316876, 2316879, 2316881, 2316882, 2316884, 2316885, 2316886, 2316887, 2316891, 2316892, 2316894, 2316896, 2316897, 2316899, 2316900, 2316901, 2316902, 2316905, 2316907, 2316913, 2316914, 2316915, 2316917, 2316918, 2316922, 2316923, 2316924, 2316925, 2316926, 2316928, 2316929, 2316930, 2316931, 2316932, 2316933, 2316934, 2316938, 2316940, 2316941, 2316943, 2316945, 2316948, 2316949, 2316950, 2316951, 2316952, 2316953, 2316954, 2316955, 2316956, 2316957, 2316958, 2316959, 2316960, 2316961, 2316963, 2316964, 2316965, 2316966, 2316967, 2316970, 2316971, 2316972, 2316974, 2316975, 2316977, 2316978, 2316980, 2316982, 2316984, 2316985, 2316986, 2316987, 2316988, 2316991, 2316992, 2316993, 2316994, 2316996, 2317000, 2317004, 2317005, 2317006, 2317008, 2317013, 2317016, 2317017, 2317018, 2317019, 2317020, 2317021, 2317022, 2317023, 2317024, 2317026, 2317027, 2317030, 2317033, 2317034, 2317035, 2317038, 2317039, 2317041, 2317043, 2317045, 2317046, 2317048, 2317050, 2317051, 2317052, 2317053, 2317057, 2317060, 2317061, 2317062, 2317064, 2317065, 2317071, 2317072, 2317073, 2317074, 2317076, 2317077, 2317079, 2317081, 2317082, 2317083, 2317084, 2317085, 2317086, 2317087, 2317088, 2317089, 2317090, 2317092, 2317093, 2317094, 2317095, 2317097, 2317102, 2317103, 2317104, 2317105, 2317107, 2317109, 2317110, 2317112, 2317114, 2317115, 2317116, 2317118, 2317119, 2317120, 2317121, 2317122, 2317123, 2317124, 2317125, 2317126, 2317127, 2317128, 2317131, 2317133, 2317139, 2317140, 2317142, 2317145, 2317146, 2317147, 2317148, 2317150, 2317151, 2317152, 2317155, 2317159, 2317162, 2317163, 2317164, 2317167, 2317169, 2317170, 2317171, 2317174, 2317175, 2317176, 2317182, 2317183, 2317185, 2317186, 2317187, 2317188, 2317190, 2317191, 2317193, 2317194, 2317195, 2317196, 2317197, 2317199, 2317200, 2317202, 2317203, 2317207, 2317208, 2317210, 2317211, 2317212, 2317214, 2317216, 2317217, 2317218, 2317219, 2317220, 2317223, 2317224, 2317226, 2317227, 2317229, 2317231, 2317232, 2317234, 2317236, 2317237, 2317238, 2317240, 2317241, 2317244, 2317246, 2317250, 2317251, 2317254, 2317257, 2317259, 2317263, 2317265, 2317267, 2317270, 2317272, 2317273, 2317275, 2317277, 2317278, 2317280, 2317281, 2317283, 2317284, 2317287, 2317288, 2317290, 2317292, 2317293, 2317295, 2317296, 2317299, 2317300, 2317301, 2317302, 2317303, 2317304, 2317305, 2317306, 2317307, 2317310, 2317311, 2317313, 2317315, 2317317, 2317318, 2317319, 2317320, 2317322, 2317323, 2317325, 2317329, 2317330, 2317331, 2317332, 2317333, 2317334, 2317335, 2317336, 2317338, 2317339, 2317340, 2317342, 2317343, 2317345, 2317348, 2317350, 2317351, 2317354, 2317355, 2317356, 2317358, 2317361, 2317362, 2317363, 2317364, 2317365, 2317366, 2317367, 2317369, 2317370, 2317371, 2317372, 2317373, 2317376, 2317380, 2317381, 2317382, 2317386, 2317387, 2317388, 2317389, 2317390, 2317391, 2317393, 2317394, 2317397, 2317399, 2317403, 2317406, 2317407, 2317409, 2317412, 2317413, 2317416, 2317417, 2317420, 2317421, 2317422, 2317423, 2317426, 2317428, 2317429, 2317434, 2317435, 2317436, 2317438, 2317439, 2317440, 2317443, 2317446, 2317448, 2317451, 2317452, 2317455, 2317457, 2317458, 2317459, 2317464, 2317468, 2317469, 2317470, 2317472, 2317473, 2317476, 2317479, 2317480, 2317481, 2317485, 2317486, 2317487, 2317491, 2317492, 2317493, 2317495, 2317497, 2317498, 2317500, 2317502, 2317503, 2317504, 2317506, 2317507, 2317508, 2317509, 2317510, 2317512, 2317513, 2317514, 2317515, 2317516, 2317517, 2317518, 2317519, 2317520, 2317522, 2317523, 2317527, 2317529, 2317531, 2317532, 2317533, 2317534, 2317535, 2317536, 2317537, 2317538, 2317540, 2317541, 2317542, 2317544, 2317548, 2317549, 2317551, 2317552, 2317553, 2317554, 2317555, 2317556, 2317557, 2317558, 2317559, 2317560, 2317561, 2317563, 2317564, 2317565, 2317566, 2317567, 2317568, 2317571, 2317573, 2317574, 2317575, 2317576, 2317577, 2317578, 2317579, 2317580, 2317581, 2317582, 2317584, 2317586, 2317588, 2317589, 2317590, 2317591, 2317592, 2317594, 2317595, 2317597, 2317598, 2317603, 2317605, 2317606, 2317607, 2317610, 2317612, 2317613, 2317618, 2317620, 2317621, 2317622, 2317623, 2317630, 2317632, 2317634, 2317636, 2317637, 2317639, 2317640, 2317644, 2317645, 2317648, 2317649, 2317654, 2317656, 2317657, 2317659, 2317660, 2317663, 2317664, 2317666, 2317668, 2317669, 2317671, 2317672, 2317673, 2317675, 2317676, 2317677, 2317678, 2317679, 2317683, 2317684, 2317685, 2317686, 2317687, 2317688, 2317692, 2317694, 2317695, 2317696, 2317697, 2317698, 2317700, 2317701, 2317703, 2317705, 2317708, 2317710, 2317711, 2317712, 2317713, 2317715, 2317716, 2317717, 2317723, 2317724, 2317727, 2317729, 2317730, 2317731, 2317732, 2317733, 2317734, 2317735, 2317737, 2317738, 2317740, 2317741, 2317742, 2317744, 2317745, 2317748, 2317749, 2317754, 2317755, 2317759, 2317760, 2317761, 2317764, 2317767, 2317768, 2317772, 2317773, 2317774, 2317775, 2317778, 2317782, 2317785, 2317786, 2317790, 2317791, 2317800, 2317802, 2317804, 2317807, 2317808, 2317809, 2317814, 2317818, 2317822, 2317823, 2317825, 2317827, 2317829, 2317833, 2317834, 2317839, 2317840, 2317841, 2317843, 2317844, 2317845, 2317846, 2317848, 2317849, 2317851, 2317852, 2317854, 2317857, 2317858, 2317859, 2317860, 2317861, 2317862, 2317864, 2317866, 2317868, 2317871, 2317872, 2317873, 2317875, 2317879, 2317880, 2317882, 2317886, 2317887, 2317888, 2317889, 2317891, 2317892, 2317893, 2317894, 2317896, 2317897, 2317898, 2317900, 2317901, 2317903, 2317905, 2317906, 2317907, 2317908, 2317910, 2317913, 2317914, 2317916, 2317917, 2317918, 2317919, 2317921, 2317923, 2317926, 2317927, 2317928, 2317929, 2317930, 2317931, 2317932, 2317933, 2317934, 2317935, 2317937, 2317940, 2317942, 2317943, 2317944, 2317946, 2317948, 2317949, 2317950, 2317951, 2317953, 2317955, 2317956, 2317957, 2317958, 2317959, 2317962, 2317963, 2317964, 2317965, 2317969, 2317972, 2317973, 2317975, 2317976, 2317977, 2317978, 2317979, 2317981, 2317982, 2317983, 2317987, 2317989, 2317990, 2317992, 2317996, 2317998, 2318002, 2318003, 2318005, 2318007, 2318008, 2318009, 2318010, 2318011, 2318012, 2318014, 2318015, 2318016, 2318019, 2318020, 2318021, 2318022, 2318024, 2318025, 2318027, 2318028, 2318029, 2318030, 2318031, 2318033, 2318036, 2318039, 2318041, 2318042, 2318045, 2318046, 2318047, 2318048, 2318049, 2318051, 2318052, 2318053, 2318054, 2318056, 2318057, 2318058, 2318059, 2318060, 2318062, 2318063, 2318067, 2318068, 2318069, 2318071, 2318072, 2318076, 2318077, 2318079, 2318084, 2318085, 2318086, 2318087, 2318088, 2318089, 2318091, 2318092, 2318093, 2318094, 2318095, 2318096, 2318097, 2318102, 2318105, 2318106, 2318107, 2318108, 2318109, 2318111, 2318113, 2318115, 2318116, 2318117, 2318118, 2318119, 2318123, 2318125, 2318126, 2318127, 2318128, 2318129, 2318134, 2318140, 2318144, 2318145, 2318146, 2318147, 2318148, 2318149, 2318151, 2318154, 2318155, 2318156, 2318157, 2318158, 2318159, 2318160, 2318163, 2318165, 2318167, 2318168, 2318169, 2318170, 2318171, 2318172, 2318174, 2318175, 2318178, 2318179, 2318181, 2318182, 2318185, 2318188, 2318190, 2318194, 2318197, 2318198, 2318199, 2318200, 2318203, 2318204, 2318207, 2318210, 2318212, 2318213, 2318214, 2318216, 2318218, 2318220, 2318222, 2318224, 2318228, 2318229, 2318230, 2318232, 2318234, 2318236, 2318237, 2318239, 2318241, 2318242, 2318244, 2318245, 2318246, 2318252, 2318255, 2318256, 2318257, 2318259, 2318260, 2318262, 2318266, 2318267, 2318268, 2318269, 2318270, 2318273, 2318276, 2318279, 2318281, 2318282, 2318283, 2318285, 2318286, 2318287, 2318289, 2318290, 2318291, 2318292, 2318297, 2318298, 2318300, 2318304, 2318305, 2318306, 2318307, 2318308, 2318311, 2318314, 2318315, 2318316, 2318317, 2318318, 2318319, 2318320, 2318321, 2318322, 2318323, 2318324, 2318326, 2318328, 2318330, 2318331, 2318332, 2318333, 2318334, 2318335, 2318336, 2318337, 2318339, 2318342, 2318343, 2318344, 2318346, 2318347, 2318348, 2318349, 2318350, 2318351, 2318352, 2318353, 2318354, 2318356, 2318357, 2318358, 2318359, 2318361, 2318362, 2318363, 2318364, 2318366, 2318367, 2318371, 2318375, 2318378, 2318379, 2318383, 2318384, 2318385, 2318386, 2318387, 2318388, 2318391, 2318392, 2318395, 2318396, 2318397, 2318399, 2318400, 2318401, 2318402, 2318403, 2318405, 2318406, 2318407, 2318408, 2318410, 2318411, 2318414, 2318415, 2318419, 2318420, 2318421, 2318423, 2318424, 2318426, 2318427, 2318428, 2318429, 2318430, 2318431, 2318433, 2318434, 2318435, 2318437, 2318438, 2318440, 2318442, 2318446, 2318447, 2318450, 2318452, 2318455, 2318458, 2318459, 2318460, 2318461, 2318462, 2318463, 2318464, 2318465, 2318466, 2318467, 2318472, 2318473, 2318474, 2318475, 2318476, 2318480, 2318481, 2318483, 2318487, 2318489, 2318490, 2318491, 2318492, 2318493, 2318494, 2318495, 2318496, 2318497, 2318498, 2318499, 2318500, 2318504, 2318509, 2318513, 2318515, 2318518, 2318520, 2318521, 2318522, 2318523, 2318524, 2318527, 2318528, 2318534, 2318535, 2318536, 2318537, 2318538, 2318539, 2318540, 2318542, 2318543, 2318546, 2318547, 2318549, 2318550, 2318551, 2318552, 2318553, 2318554, 2318555, 2318557, 2318560, 2318561, 2318565, 2318566, 2318568, 2318569, 2318570, 2318573, 2318574, 2318575, 2318580, 2318581, 2318583, 2318586, 2318588, 2318589, 2318590, 2318591, 2318592, 2318593, 2318596, 2318597, 2318598, 2318599, 2318601, 2318602, 2318604, 2318605, 2318607, 2318608, 2318609, 2318611, 2318612, 2318616, 2318618, 2318620, 2318622, 2318623, 2318624, 2318625, 2318627, 2318628, 2318629, 2318630, 2318632, 2318633, 2318635, 2318636, 2318637, 2318640, 2318641, 2318642, 2318645, 2318646, 2318647, 2318648, 2318650, 2318651, 2318652, 2318653, 2318654, 2318656, 2318657, 2318659, 2318660, 2318661, 2318663, 2318666, 2318667, 2318668, 2318669, 2318670, 2318671, 2318673, 2318675, 2318678, 2318679, 2318680, 2318681, 2318683, 2318685, 2318686, 2318687, 2318690, 2318692, 2318693, 2318694, 2318695, 2318699, 2318700, 2318702, 2318704, 2318705, 2318707, 2318711, 2318712, 2318713, 2318719, 2318720, 2318725, 2318727, 2318728, 2318729, 2318730, 2318732, 2318734, 2318736, 2318738, 2318741, 2318742, 2318743, 2318744, 2318745, 2318746, 2318747, 2318749, 2318751, 2318752, 2318753, 2318757, 2318758, 2318759, 2318760, 2318761, 2318762, 2318764, 2318766, 2318768, 2318769, 2318772, 2318778, 2318779, 2318780, 2318781, 2318785, 2318787, 2318788, 2318790, 2318791, 2318792, 2318793, 2318794, 2318796, 2318798, 2318801, 2318804, 2318805, 2318807, 2318808, 2318810, 2318811, 2318812, 2318813, 2318814, 2318815, 2318817, 2318818, 2318826, 2318828, 2318829, 2318831, 2318833, 2318834, 2318835, 2318842, 2318843, 2318844, 2318845, 2318849, 2318854, 2318857, 2318858, 2318860, 2318861, 2318864, 2318866, 2318867, 2318868, 2318869, 2318870, 2318871, 2318873, 2318874, 2318875, 2318877, 2318880, 2318881, 2318883, 2318884, 2318885, 2318887, 2318889, 2318890, 2318892, 2318895, 2318898, 2318901, 2318902, 2318903, 2318904, 2318905, 2318906, 2318907, 2318910, 2318911, 2318913, 2318915, 2318916, 2318917, 2318918, 2318919, 2318920, 2318921, 2318922, 2318924, 2318929, 2318930, 2318934, 2318935, 2318937, 2318939, 2318940, 2318941, 2318943, 2318946, 2318947, 2318948, 2318949, 2318950, 2318955, 2318956, 2318959, 2318960, 2318961, 2318962, 2318963, 2318964, 2318970, 2318972, 2318974, 2318976, 2318978, 2318979, 2318981, 2318984, 2318986, 2318987, 2318988, 2318989, 2318990, 2318992, 2318993, 2318994, 2318995, 2318996, 2318997, 2318999, 2319000, 2319001, 2319002, 2319005, 2319006, 2319007, 2319008, 2319010, 2319012, 2319013, 2319014, 2319015, 2319016, 2319017, 2319022, 2319023, 2319024, 2319026, 2319027, 2319032, 2319037, 2319038, 2319039, 2319042, 2319043, 2319049, 2319050, 2319051, 2319053, 2319054, 2319055, 2319056, 2319057, 2319058, 2319060, 2319061, 2319062, 2319064, 2319065, 2319066, 2319071, 2319072, 2319073, 2319074, 2319075, 2319077, 2319078, 2319079, 2319085, 2319087, 2319088, 2319089, 2319092, 2319094, 2319095, 2319096, 2319098, 2319101, 2319102, 2319104, 2319105, 2319106, 2319107, 2319110, 2319111, 2319113, 2319114, 2319117, 2319118, 2319119, 2319123, 2319125, 2319126, 2319129, 2319131, 2319133, 2319135, 2319136, 2319137, 2319139, 2319142, 2319143, 2319144, 2319145, 2319147, 2319148, 2319155, 2319159, 2319160, 2319162, 2319163, 2319164, 2319165, 2319166, 2319169, 2319170, 2319171, 2319172, 2319175, 2319176, 2319178, 2319180, 2319182, 2319184, 2319186, 2319187, 2319189, 2319193, 2319195, 2319196, 2319197, 2319199, 2319200, 2319203, 2319204, 2319205, 2319206, 2319209, 2319212, 2319213, 2319214, 2319215, 2319216, 2319218, 2319220, 2319221, 2319222, 2319224, 2319225, 2319228, 2319230, 2319231, 2319232, 2319235, 2319238, 2319239, 2319240, 2319242, 2319246, 2319247, 2319248, 2319249, 2319253, 2319255, 2319256, 2319260, 2319263, 2319264, 2319265, 2319267, 2319268, 2319273, 2319275, 2319276, 2319277, 2319278, 2319279, 2319281, 2319283, 2319286, 2319287, 2319288, 2319290, 2319297, 2319298, 2319299, 2319302, 2319304, 2319305, 2319308, 2319310, 2319312, 2319313, 2319314, 2319319, 2319320, 2319321, 2319323, 2319325, 2319326, 2319327, 2319328, 2319331, 2319332, 2319334, 2319336, 2319337, 2319339, 2319342, 2319343, 2319344, 2319346, 2319349, 2319350, 2319351, 2319355, 2319356, 2319358, 2319359, 2319360, 2319362, 2319365, 2319366, 2319367, 2319368, 2319370, 2319371, 2319372, 2319373, 2319374, 2319375, 2319377, 2319378, 2319379, 2319381, 2319382, 2319383, 2319386, 2319387, 2319388, 2319389, 2319391, 2319393, 2319394, 2319395, 2319396, 2319399, 2319400, 2319402, 2319405, 2319407, 2319409, 2319410, 2319411, 2319412, 2319413, 2319414, 2319415, 2319417, 2319418, 2319419, 2319420, 2319421, 2319426, 2319430, 2319431, 2319432, 2319434, 2319437, 2319438, 2319440, 2319441, 2319442, 2319443, 2319444, 2319445, 2319448, 2319449, 2319454, 2319455, 2319457, 2319458, 2319459, 2319460, 2319461, 2319464, 2319465, 2319466, 2319467, 2319468, 2319471, 2319472, 2319473, 2319474, 2319476, 2319477, 2319480, 2319482, 2319484, 2319486, 2319489, 2319490, 2319491, 2319493, 2319494, 2319495, 2319496, 2319497, 2319500, 2319502, 2319503, 2319505, 2319506, 2319507, 2319514, 2319517, 2319518, 2319522, 2319524, 2319525, 2319529, 2319530, 2319531, 2319532, 2319535, 2319539, 2319541, 2319544, 2319545, 2319548, 2319549, 2319550, 2319551, 2319552, 2319554, 2319556, 2319560, 2319564, 2319565, 2319569, 2319572, 2319573, 2319574, 2319575, 2319576, 2319578, 2319580, 2319581, 2319582, 2319583, 2319584, 2319585, 2319587, 2319588, 2319590, 2319591, 2319592, 2319594, 2319595, 2319598, 2319599, 2319600, 2319601, 2319603, 2319605, 2319606, 2319607, 2319608, 2319613, 2319614, 2319615, 2319616, 2319617, 2319618, 2319619, 2319621, 2319622, 2319623, 2319624, 2319625, 2319627, 2319628, 2319629, 2319631, 2319634, 2319635, 2319636, 2319637, 2319639, 2319641, 2319642, 2319645, 2319646, 2319649, 2319650, 2319651, 2319654, 2319656, 2319657, 2319659, 2319661, 2319662, 2319666, 2319669, 2319671, 2319672, 2319674, 2319675, 2319677, 2319678, 2319679, 2319680, 2319683, 2319685, 2319686, 2319689, 2319691, 2319694, 2319695, 2319696, 2319699, 2319700, 2319701, 2319706, 2319707, 2319709, 2319713, 2319714, 2319715, 2319718, 2319719, 2319720, 2319722, 2319723, 2319724, 2319727, 2319729, 2319732, 2319733, 2319734, 2319735, 2319736, 2319737, 2319738, 2319741, 2319742, 2319743, 2319745, 2319746, 2319750, 2319751, 2319752, 2319753, 2319754, 2319755, 2319757, 2319760, 2319763, 2319764, 2319765, 2319766, 2319767, 2319770, 2319771, 2319772, 2319773, 2319776, 2319781, 2319782, 2319784, 2319785, 2319787, 2319789, 2319790, 2319793, 2319794, 2319798, 2319799, 2319801, 2319803, 2319804, 2319808, 2319809, 2319810, 2319813, 2319814, 2319816, 2319817, 2319820, 2319821, 2319822, 2319825, 2319826, 2319828, 2319829, 2319832, 2319833, 2319834, 2319835, 2319836, 2319838, 2319839, 2319840, 2319841, 2319844, 2319848, 2319849, 2319850, 2319852, 2319856, 2319859, 2319862, 2319864, 2319869, 2319870, 2319871, 2319872, 2319873, 2319874, 2319875, 2319876, 2319877, 2319878, 2319880, 2319881, 2319887, 2319888, 2319889, 2319890, 2319892, 2319894, 2319896, 2319897, 2319898, 2319899, 2319900, 2319901, 2319903, 2319904, 2319905, 2319906, 2319912, 2319913, 2319916, 2319919, 2319923, 2319926, 2319927, 2319928, 2319930, 2319932, 2319934, 2319935, 2319936, 2319937, 2319938, 2319940, 2319943, 2319945, 2319946, 2319948, 2319949, 2319951, 2319954, 2319955, 2319957, 2319958, 2319961, 2319965, 2319966, 2319968, 2319969, 2319971, 2319973, 2319974, 2319975, 2319977, 2319978, 2319979, 2319981, 2319983, 2319987, 2319990, 2319991, 2319992, 2319993, 2319994, 2319995, 2319997, 2320000, 2320006, 2320007, 2320008, 2320009, 2320010, 2320013, 2320014, 2320015, 2320016, 2320019, 2320021, 2320022, 2320024, 2320027, 2320028, 2320029, 2320031, 2320033, 2320036, 2320037, 2320040, 2320041, 2320044, 2320045, 2320047, 2320049, 2320050, 2320053, 2320056, 2320059, 2320060, 2320061, 2320062, 2320063, 2320064, 2320066, 2320067, 2320069, 2320071, 2320072, 2320073, 2320074, 2320075, 2320076, 2320077, 2320078, 2320086, 2320089, 2320090, 2320091, 2320092, 2320094, 2320096, 2320097, 2320102, 2320103, 2320104, 2320108, 2320112, 2320113, 2320115, 2320116, 2320119, 2320120, 2320121, 2320123, 2320125, 2320129, 2320130, 2320133, 2320134, 2320137, 2320138, 2320141, 2320142, 2320143, 2320144, 2320146, 2320147, 2320148, 2320150, 2320154, 2320156, 2320157, 2320158, 2320159, 2320160, 2320161, 2320163, 2320164, 2320169, 2320171, 2320172, 2320173, 2320174, 2320175, 2320177, 2320179, 2320180, 2320182, 2320183, 2320184, 2320185, 2320190, 2320191, 2320193, 2320194, 2320195, 2320196, 2320197, 2320198, 2320199, 2320201, 2320203, 2320204, 2320205, 2320207, 2320211, 2320215, 2320216, 2320217, 2320218, 2320219, 2320220, 2320221, 2320225, 2320226, 2320228, 2320230, 2320231, 2320233, 2320234, 2320235, 2320236, 2320237, 2320238, 2320239, 2320240, 2320241, 2320242, 2320246, 2320247, 2320248, 2320251, 2320252, 2320256, 2320257, 2320258, 2320262, 2320264, 2320265, 2320266, 2320268, 2320269, 2320270, 2320271, 2320277, 2320278, 2320279, 2320280, 2320285, 2320293, 2320295, 2320297, 2320302, 2320303, 2320304, 2320306, 2320308, 2320311, 2320313, 2320314, 2320315, 2320317, 2320318, 2320320, 2320321, 2320322, 2320325, 2320326, 2320328, 2320331, 2320333, 2320334, 2320335, 2320337, 2320345, 2320346, 2320347, 2320348, 2320349, 2320350, 2320351, 2320353, 2320354, 2320359, 2320360, 2320362, 2320364, 2320365, 2320366, 2320367, 2320369, 2320371, 2320373, 2320374, 2320375, 2320376, 2320377, 2320378, 2320379, 2320380, 2320381, 2320382, 2320384, 2320387, 2320389, 2320392, 2320395, 2320398, 2320399, 2320404, 2320407, 2320408, 2320414, 2320415, 2320416, 2320418, 2320419, 2320422, 2320423, 2320425, 2320426, 2320427, 2320428, 2320429, 2320431, 2320432, 2320434, 2320439, 2320440, 2320441, 2320443, 2320445, 2320446, 2320450, 2320451, 2320453, 2320454, 2320456, 2320457, 2320458, 2320460, 2320464, 2320465, 2320466, 2320470, 2320471, 2320473, 2320474, 2320475, 2320480, 2320484, 2320488, 2320490, 2320493, 2320494, 2320497, 2320500, 2320503, 2320505, 2320507, 2320508, 2320509, 2320511, 2320512, 2320514, 2320517, 2320518, 2320519, 2320522, 2320523, 2320527, 2320529, 2320530, 2320534, 2320535, 2320536, 2320538, 2320539, 2320541, 2320542, 2320543, 2320545, 2320546, 2320548, 2320550, 2320551, 2320553, 2320555, 2320556, 2320559, 2320561, 2320562, 2320565, 2320566, 2320567, 2320568, 2320569, 2320571, 2320572, 2320573, 2320574, 2320575, 2320579, 2320580, 2320581, 2320582, 2320583, 2320584, 2320586, 2320587, 2320588, 2320590, 2320591, 2320595, 2320597, 2320598, 2320600, 2320602, 2320606, 2320607, 2320613, 2320614, 2320617, 2320620, 2320621, 2320622, 2320624, 2320625, 2320627, 2320628, 2320632, 2320633, 2320634, 2320635, 2320638, 2320639, 2320640, 2320641, 2320645, 2320646, 2320650, 2320651, 2320654, 2320655, 2320656, 2320657, 2320658, 2320660, 2320665, 2320667, 2320668, 2320669, 2320672, 2320674, 2320675, 2320676, 2320678, 2320679, 2320681, 2320683, 2320686, 2320687, 2320688, 2320690, 2320691, 2320693, 2320697, 2320699, 2320700, 2320703, 2320704, 2320705, 2320707, 2320708, 2320709, 2320712, 2320715, 2320716, 2320717, 2320718, 2320719, 2320720, 2320723, 2320725, 2320726, 2320728, 2320729, 2320730, 2320731, 2320732, 2320733, 2320734, 2320737, 2320740, 2320742, 2320743, 2320744, 2320745, 2320750, 2320752, 2320753, 2320756, 2320758, 2320759, 2320760, 2320762, 2320764, 2320765, 2320766, 2320768, 2320770, 2320774, 2320775, 2320776, 2320777, 2320779, 2320780, 2320782, 2320783, 2320787, 2320790, 2320791, 2320792, 2320794, 2320796, 2320798, 2320799, 2320800, 2320801, 2320802, 2320803, 2320806, 2320807, 2320808, 2320809, 2320811, 2320812, 2320813, 2320815, 2320816, 2320817, 2320820, 2320823, 2320824, 2320825, 2320826, 2320827, 2320828, 2320829, 2320832, 2320833, 2320834, 2320835, 2320838, 2320840, 2320841, 2320843, 2320846, 2320847, 2320848, 2320849, 2320851, 2320852, 2320855, 2320858, 2320860, 2320862, 2320864, 2320866, 2320867, 2320869, 2320871, 2320872, 2320873, 2320874, 2320875, 2320876, 2320877, 2320882, 2320883, 2320884, 2320886, 2320888, 2320889, 2320890, 2320891, 2320892, 2320893, 2320894, 2320895, 2320897, 2320899, 2320901, 2320902, 2320907, 2320908, 2320909, 2320911, 2320912, 2320915, 2320916, 2320917, 2320918, 2320920, 2320924, 2320926, 2320928, 2320930, 2320931, 2320933, 2320935, 2320937, 2320938, 2320939, 2320940, 2320948, 2320950, 2320954, 2320956, 2320959, 2320960, 2320961, 2320962, 2320963, 2320966, 2320968, 2320969, 2320976, 2320978, 2320981, 2320982, 2320983, 2320985, 2320987, 2320989, 2320990, 2320991, 2320992, 2320993, 2320995, 2320997, 2320998, 2320999, 2321001, 2321005, 2321007, 2321008, 2321012, 2321015, 2321016, 2321017, 2321018, 2321019, 2321022, 2321023, 2321025, 2321026, 2321028, 2321029, 2321030, 2321034, 2321037, 2321038, 2321040, 2321041, 2321042, 2321043, 2321044, 2321045, 2321047, 2321050, 2321052, 2321056, 2321057, 2321058, 2321059, 2321060, 2321061, 2321064, 2321065, 2321066, 2321067, 2321070, 2321072, 2321073, 2321076, 2321080, 2321083, 2321089, 2321090, 2321091, 2321092, 2321093, 2321094, 2321096, 2321098, 2321099, 2321102, 2321103, 2321104, 2321105, 2321107, 2321108, 2321110, 2321111, 2321112, 2321115, 2321116, 2321117, 2321120, 2321121, 2321123, 2321124, 2321125, 2321126, 2321127, 2321134, 2321137, 2321141, 2321142, 2321143, 2321144, 2321146, 2321151, 2321152, 2321154, 2321156, 2321159, 2321160, 2321161, 2321163, 2321164, 2321167, 2321170, 2321172, 2321174, 2321175, 2321179, 2321183, 2321187, 2321188, 2321190, 2321191, 2321192, 2321193, 2321195, 2321197, 2321198, 2321199, 2321201, 2321204, 2321207, 2321208, 2321210, 2321213, 2321218, 2321220, 2321221, 2321222, 2321223, 2321225, 2321228, 2321229, 2321234, 2321235, 2321238, 2321240, 2321242, 2321243, 2321244, 2321247, 2321251, 2321255, 2321257, 2321258, 2321260, 2321261, 2321263, 2321264, 2321265, 2321266, 2321267, 2321268, 2321270, 2321272, 2321273, 2321274, 2321275, 2321277, 2321278, 2321279, 2321281, 2321283, 2321284, 2321286, 2321288, 2321290, 2321291, 2321295, 2321296, 2321298, 2321299, 2321303, 2321304, 2321307, 2321310, 2321311, 2321313, 2321314, 2321315, 2321316, 2321320, 2321322, 2321324, 2321325, 2321329, 2321330, 2321331, 2321332, 2321334, 2321338, 2321342, 2321343, 2321344, 2321345, 2321347, 2321350, 2321353, 2321354, 2321356, 2321358, 2321360, 2321361, 2321362, 2321363, 2321364, 2321367, 2321368, 2321369, 2321372, 2321375, 2321376, 2321378, 2321379, 2321380, 2321385, 2321386, 2321387, 2321390, 2321391, 2321394, 2321397, 2321398, 2321399, 2321400, 2321401, 2321403, 2321404, 2321405, 2321406, 2321407, 2321409, 2321411, 2321414, 2321416, 2321417, 2321418, 2321419, 2321421, 2321424, 2321427, 2321428, 2321430, 2321431, 2321432, 2321434, 2321435, 2321436, 2321437, 2321440, 2321442, 2321444, 2321445, 2321447, 2321449, 2321450, 2321455, 2321456, 2321458, 2321461, 2321462, 2321463, 2321464, 2321465, 2321466, 2321467, 2321468, 2321469, 2321470, 2321471, 2321472, 2321473, 2321474, 2321475, 2321477, 2321478, 2321480, 2321481, 2321483, 2321484, 2321487, 2321489, 2321492, 2321493, 2321494, 2321495, 2321496, 2321497, 2321498, 2321499, 2321500, 2321502, 2321503, 2321504, 2321505, 2321506, 2321510, 2321511, 2321512, 2321514, 2321515, 2321516, 2321519, 2321520, 2321522, 2321523, 2321524, 2321526, 2321528, 2321530, 2321532, 2321534, 2321537, 2321538, 2321539, 2321540, 2321541, 2321543, 2321544, 2321545, 2321547, 2321552, 2321553, 2321555, 2321557, 2321558, 2321562, 2321565, 2321566, 2321567, 2321570, 2321575, 2321576, 2321577, 2321578, 2321580, 2321582, 2321583, 2321584, 2321591, 2321592, 2321593, 2321595, 2321596, 2321597, 2321598, 2321601, 2321602, 2321603, 2321604, 2321605, 2321606, 2321609, 2321615, 2321616, 2321617, 2321620, 2321621, 2321622, 2321625, 2321626, 2321628, 2321629, 2321630, 2321633, 2321636, 2321638, 2321640, 2321643, 2321644, 2321647, 2321648, 2321650, 2321653, 2321654, 2321655, 2321656, 2321657, 2321659, 2321660, 2321663, 2321665, 2321666, 2321669, 2321670, 2321671, 2321674, 2321677, 2321678, 2321679, 2321680, 2321682, 2321684, 2321686, 2321687, 2321688, 2321689, 2321690, 2321692, 2321693, 2321696, 2321698, 2321700, 2321702, 2321703, 2321704, 2321707, 2321708, 2321711, 2321713, 2321714, 2321716, 2321719, 2321724, 2321725, 2321728, 2321729, 2321732, 2321733, 2321734, 2321739, 2321740, 2321741, 2321743, 2321744, 2321745, 2321752, 2321754, 2321755, 2321756, 2321757, 2321758, 2321759, 2321760, 2321762, 2321764, 2321765, 2321766, 2321767, 2321768, 2321769, 2321770, 2321771, 2321772, 2321773, 2321775, 2321776, 2321777, 2321778, 2321779, 2321782, 2321783, 2321785, 2321786, 2321787, 2321788, 2321790, 2321793, 2321794, 2321795, 2321796, 2321797, 2321798, 2321799, 2321800, 2321802, 2321803, 2321807, 2321808, 2321809, 2321810, 2321811, 2321812, 2321814, 2321816, 2321817, 2321818, 2321820, 2321821, 2321822, 2321823, 2321827, 2321828, 2321830, 2321831, 2321833, 2321834, 2321835, 2321840, 2321841, 2321846, 2321847, 2321848, 2321851, 2321852, 2321853, 2321854, 2321857, 2321859, 2321860, 2321861, 2321863, 2321867, 2321868, 2321869, 2321870, 2321871, 2321872, 2321874, 2321875, 2321878, 2321879, 2321882, 2321883, 2321885, 2321886, 2321889, 2321894, 2321895, 2321896, 2321899, 2321900, 2321904, 2321905, 2321908, 2321911, 2321912, 2321914, 2321915, 2321917, 2321918, 2321922, 2321923, 2321924, 2321928, 2321929, 2321930, 2321931, 2321932, 2321933, 2321935, 2321937, 2321939, 2321941, 2321942, 2321943, 2321944, 2321947, 2321948, 2321949, 2321950, 2321952, 2321954, 2321956, 2321957, 2321959, 2321960, 2321962, 2321964, 2321966, 2321967, 2321968, 2321970, 2321973, 2321975, 2321976, 2321977, 2321980, 2321981, 2321983, 2321984, 2321985, 2321990, 2321991, 2321992, 2321994, 2321995, 2321998, 2321999, 2322002, 2322003, 2322006, 2322007, 2322008, 2322012, 2322014, 2322018, 2322019, 2322020, 2322021, 2322023, 2322025, 2322027, 2322030, 2322032, 2322033, 2322034, 2322035, 2322037, 2322038, 2322040, 2322041, 2322043, 2322044, 2322046, 2322047, 2322048, 2322049, 2322052, 2322053, 2322054, 2322055, 2322056, 2322060, 2322064, 2322066, 2322067, 2322069, 2322071, 2322072, 2322074, 2322076, 2322079, 2322080, 2322081, 2322083, 2322084, 2322088, 2322089, 2322091, 2322092, 2322093, 2322094, 2322096, 2322097, 2322098, 2322101, 2322103, 2322104, 2322105, 2322106, 2322107, 2322111, 2322112, 2322115, 2322116, 2322117, 2322122, 2322125, 2322126, 2322129, 2322131, 2322133, 2322134, 2322135, 2322138, 2322139, 2322141, 2322145, 2322146, 2322147, 2322149, 2322150, 2322152, 2322153, 2322154, 2322157, 2322158, 2322160, 2322161, 2322165, 2322167, 2322168, 2322169, 2322170, 2322172, 2322173, 2322174, 2322175, 2322177, 2322178, 2322180, 2322181, 2322186, 2322187, 2322190, 2322193, 2322195, 2322196, 2322197, 2322198, 2322201, 2322203, 2322204, 2322208, 2322214, 2322215, 2322216, 2322220, 2322223, 2322226, 2322228, 2322231, 2322232, 2322233, 2322234, 2322236, 2322237, 2322245, 2322246, 2322248, 2322251, 2322255, 2322257, 2322260, 2322261, 2322262, 2322263, 2322264, 2322266, 2322267, 2322270, 2322274, 2322280, 2322283, 2322287, 2322290, 2322292, 2322293, 2322294, 2322295, 2322296, 2322297, 2322299, 2322300, 2322302, 2322303, 2322307, 2322308, 2322309, 2322310, 2322311, 2322312, 2322313, 2322314, 2322315, 2322316, 2322319, 2322322, 2322323, 2322324, 2322325, 2322327, 2322328, 2322330, 2322332, 2322333, 2322337, 2322338, 2322342, 2322343, 2322347, 2322348, 2322350, 2322353, 2322354, 2322355, 2322358, 2322361, 2322363, 2322367, 2322370, 2322371, 2322372, 2322375, 2322376, 2322377, 2322379, 2322380, 2322382, 2322385, 2322386, 2322389, 2322390, 2322391, 2322392, 2322394, 2322395, 2322397, 2322398, 2322400, 2322401, 2322405, 2322406, 2322407, 2322408, 2322409, 2322411, 2322413, 2322414, 2322418, 2322420, 2322421, 2322423, 2322424, 2322426, 2322427, 2322431, 2322434, 2322435, 2322436, 2322438, 2322439, 2322441, 2322443, 2322444, 2322445, 2322447, 2322448, 2322449, 2322451, 2322453, 2322455, 2322456, 2322457, 2322458, 2322459, 2322461, 2322462, 2322463, 2322464, 2322465, 2322466, 2322468, 2322469, 2322474, 2322479, 2322483, 2322485, 2322487, 2322488, 2322490, 2322492, 2322493, 2322495, 2322497, 2322498, 2322499, 2322503, 2322505, 2322506, 2322509, 2322511, 2322516, 2322517, 2322519, 2322520, 2322524, 2322525, 2322529, 2322530, 2322534, 2322535, 2322536, 2322537, 2322538, 2322539, 2322540, 2322542, 2322544, 2322545, 2322546, 2322547, 2322548, 2322549, 2322550, 2322552, 2322553, 2322554, 2322555, 2322556, 2322557, 2322558, 2322559, 2322561, 2322563, 2322567, 2322568, 2322569, 2322570, 2322572, 2322573, 2322576, 2322577, 2322578, 2322579, 2322580, 2322583, 2322584, 2322585, 2322589, 2322590, 2322592, 2322596, 2322597, 2322598, 2322600, 2322601, 2322602, 2322603, 2322605, 2322606, 2322608, 2322609, 2322610, 2322611, 2322612, 2322614, 2322615, 2322616, 2322619, 2322620, 2322622, 2322624, 2322625, 2322627, 2322630, 2322633, 2322635, 2322636, 2322637, 2322640, 2322641, 2322642, 2322646, 2322648, 2322649, 2322651, 2322652, 2322653, 2322654, 2322655, 2322659, 2322664, 2322665, 2322666, 2322669, 2322670, 2322672, 2322675, 2322677, 2322678, 2322679, 2322680, 2322682, 2322683, 2322684, 2322686, 2322687, 2322688, 2322689, 2322690, 2322691, 2322692, 2322693, 2322694, 2322696, 2322698, 2322699, 2322700, 2322702, 2322704, 2322705, 2322706, 2322709, 2322711, 2322712, 2322713, 2322715, 2322716, 2322717, 2322718, 2322720, 2322721, 2322722, 2322723, 2322724, 2322726, 2322730, 2322731, 2322732, 2322734, 2322735, 2322737, 2322738, 2322740, 2322744, 2322746, 2322747, 2322749, 2322750, 2322752, 2322753, 2322754, 2322755, 2322756, 2322758, 2322761, 2322762, 2322764, 2322765, 2322767, 2322768, 2322769, 2322771, 2322772, 2322779, 2322781, 2322782, 2322783, 2322785, 2322789, 2322790, 2322792, 2322795, 2322796, 2322798, 2322801, 2322802, 2322803, 2322807, 2322808, 2322809, 2322810, 2322811, 2322812, 2322813, 2322814, 2322815, 2322816, 2322817, 2322818, 2322820, 2322822, 2322824, 2322827, 2322828, 2322829, 2322831, 2322832, 2322834, 2322836, 2322837, 2322840, 2322841, 2322842, 2322843, 2322844, 2322845, 2322846, 2322847, 2322848, 2322851, 2322853, 2322854, 2322855, 2322856, 2322857, 2322858, 2322860, 2322861, 2322862, 2322863, 2322864, 2322865, 2322870, 2322871, 2322873, 2322874, 2322875, 2322877, 2322879, 2322882, 2322883, 2322884, 2322885, 2322886, 2322887, 2322888, 2322889, 2322892, 2322893, 2322895, 2322896, 2322897, 2322898, 2322899, 2322900, 2322903, 2322906, 2322912, 2322913, 2322914, 2322915, 2322917, 2322918, 2322919, 2322921, 2322923, 2322924, 2322926, 2322927, 2322928, 2322929, 2322930, 2322932, 2322935, 2322938, 2322939, 2322940, 2322942, 2322943, 2322944, 2322946, 2322947, 2322948, 2322949, 2322950, 2322952, 2322953, 2322955, 2322956, 2322959, 2322960, 2322961, 2322963, 2322964, 2322966, 2322967, 2322968, 2322969, 2322970, 2322971, 2322972, 2322973, 2322975, 2322976, 2322977, 2322978, 2322980, 2322981, 2322982, 2322984, 2322985, 2322986, 2322987, 2322989, 2322990, 2322991, 2322992, 2322993, 2322994, 2322995, 2322997, 2322998, 2323003, 2323005, 2323006, 2323010, 2323011, 2323012, 2323013, 2323014, 2323016, 2323019, 2323020, 2323022, 2323028, 2323029, 2323030, 2323031, 2323032, 2323033, 2323034, 2323037, 2323038, 2323042, 2323046, 2323048, 2323049, 2323050, 2323052, 2323055, 2323056, 2323057, 2323058, 2323059, 2323060, 2323061, 2323062, 2323063, 2323065, 2323067, 2323073, 2323075, 2323076, 2323077, 2323078, 2323080, 2323081, 2323083, 2323087, 2323089, 2323091, 2323092, 2323093, 2323095, 2323096, 2323097, 2323099, 2323100, 2323102, 2323110, 2323111, 2323117, 2323118, 2323120, 2323122, 2323123, 2323125, 2323126, 2323127, 2323129, 2323131, 2323133, 2323137, 2323139, 2323142, 2323143, 2323145, 2323147, 2323148, 2323150, 2323153, 2323154, 2323155, 2323156, 2323157, 2323158, 2323159, 2323164, 2323166, 2323170, 2323172, 2323173, 2323174, 2323176, 2323177, 2323180, 2323183, 2323185, 2323187, 2323188, 2323192, 2323194, 2323196, 2323197, 2323200, 2323201, 2323204, 2323205, 2323208, 2323209, 2323210, 2323211, 2323212, 2323214, 2323215, 2323218, 2323219, 2323220, 2323222, 2323225, 2323227, 2323228, 2323229, 2323230, 2323234, 2323235, 2323236, 2323238, 2323240, 2323242, 2323243, 2323244, 2323251, 2323255, 2323256, 2323257, 2323258, 2323259, 2323260, 2323264, 2323265, 2323268, 2323270, 2323272, 2323273, 2323275, 2323276, 2323277, 2323278, 2323279, 2323281, 2323282, 2323283, 2323287, 2323289, 2323292, 2323294, 2323295, 2323296, 2323297, 2323298, 2323300, 2323302, 2323303, 2323305, 2323307, 2323308, 2323310, 2323313, 2323314, 2323318, 2323319, 2323320, 2323321, 2323323, 2323325, 2323327, 2323328, 2323331, 2323334, 2323337, 2323338, 2323340, 2323342, 2323343, 2323344, 2323345, 2323346, 2323347, 2323348, 2323349, 2323350, 2323352, 2323354, 2323355, 2323357, 2323358, 2323360, 2323361, 2323363, 2323365, 2323366, 2323367, 2323369, 2323370, 2323371, 2323373, 2323374, 2323375, 2323376, 2323377, 2323380, 2323381, 2323383, 2323385, 2323386, 2323387, 2323388, 2323389, 2323390, 2323392, 2323393, 2323396, 2323400, 2323402, 2323403, 2323404, 2323406, 2323408, 2323409, 2323411, 2323413, 2323416, 2323420, 2323421, 2323426, 2323430, 2323431, 2323432, 2323434, 2323435, 2323436, 2323438, 2323439, 2323441, 2323442, 2323444, 2323445, 2323447, 2323449, 2323451, 2323452, 2323455, 2323457, 2323462, 2323466, 2323467, 2323470, 2323471, 2323472, 2323473, 2323474, 2323475, 2323478, 2323481, 2323485, 2323487, 2323488, 2323489, 2323490, 2323491, 2323493, 2323495, 2323496, 2323499, 2323501, 2323503, 2323504, 2323506, 2323510, 2323511, 2323514, 2323516, 2323517, 2323519, 2323520, 2323523, 2323525, 2323527, 2323528, 2323529, 2323530, 2323536, 2323538, 2323540, 2323541, 2323544, 2323546, 2323547, 2323549, 2323550, 2323556, 2323557, 2323559, 2323560, 2323563, 2323564, 2323565, 2323570, 2323573, 2323574, 2323578, 2323579, 2323580, 2323581, 2323582, 2323583, 2323584, 2323585, 2323589, 2323593, 2323594, 2323595, 2323597, 2323598, 2323599, 2323602, 2323604, 2323605, 2323607, 2323608, 2323611, 2323613, 2323616, 2323617, 2323618, 2323621, 2323622, 2323623, 2323624, 2323625, 2323626, 2323628, 2323629, 2323632, 2323633, 2323634, 2323635, 2323636, 2323639, 2323644, 2323646, 2323648, 2323651, 2323652, 2323653, 2323656, 2323657, 2323659, 2323661, 2323662, 2323665, 2323668, 2323671, 2323672, 2323673, 2323674, 2323675, 2323676, 2323679, 2323682, 2323685, 2323687, 2323688, 2323689, 2323691, 2323692, 2323695, 2323696, 2323697, 2323698, 2323700, 2323701, 2323703, 2323704, 2323708, 2323713, 2323714, 2323715, 2323718, 2323719, 2323720, 2323721, 2323722, 2323723, 2323724, 2323725, 2323726, 2323727, 2323729, 2323730, 2323731, 2323732, 2323734, 2323738, 2323739, 2323742, 2323743, 2323745, 2323746, 2323751, 2323753, 2323756, 2323758, 2323759, 2323760, 2323761, 2323762, 2323763, 2323764, 2323765, 2323767, 2323769, 2323771, 2323772, 2323773, 2323777, 2323778, 2323781, 2323782, 2323787, 2323789, 2323790, 2323791, 2323794, 2323795, 2323796, 2323797, 2323798, 2323799, 2323802, 2323804, 2323806, 2323809, 2323810, 2323811, 2323812, 2323813, 2323815, 2323816, 2323817, 2323821, 2323822, 2323825, 2323826, 2323827, 2323828, 2323829, 2323831, 2323832, 2323836, 2323837, 2323840, 2323841, 2323842, 2323846, 2323847, 2323848, 2323850, 2323851, 2323852, 2323853, 2323854, 2323855, 2323858, 2323859, 2323860, 2323861, 2323863, 2323864, 2323865, 2323867, 2323868, 2323869, 2323871, 2323872, 2323875, 2323876, 2323879, 2323880, 2323882, 2323883, 2323885, 2323886, 2323888, 2323891, 2323893, 2323894, 2323895, 2323896, 2323897, 2323899, 2323900, 2323901, 2323902, 2323903, 2323904, 2323907, 2323908, 2323910, 2323911, 2323912, 2323913, 2323916, 2323918, 2323920, 2323924, 2323925, 2323927, 2323929, 2323930, 2323933, 2323935, 2323937, 2323938, 2323940, 2323941, 2323943, 2323945, 2323947, 2323948, 2323949, 2323950, 2323951, 2323957, 2323959, 2323962, 2323964, 2323966, 2323968, 2323969, 2323971, 2323972, 2323977, 2323979, 2323980, 2323981, 2323982, 2323984, 2323986, 2323987, 2323988, 2323990, 2323991, 2323992, 2323996, 2323997, 2323998, 2323999, 2324000, 2324001, 2324002, 2324008, 2324012, 2324013, 2324016, 2324018, 2324019, 2324020, 2324023, 2324024, 2324025, 2324027, 2324029, 2324031, 2324033, 2324035, 2324038, 2324039, 2324040, 2324041, 2324042, 2324043, 2324045, 2324046, 2324047, 2324048, 2324049, 2324050, 2324051, 2324052, 2324056, 2324057, 2324059, 2324060, 2324062, 2324064, 2324065, 2324067, 2324069, 2324070, 2324072, 2324074, 2324076, 2324078, 2324082, 2324083, 2324084, 2324085, 2324087, 2324088, 2324089, 2324090, 2324091, 2324092, 2324093, 2324094, 2324097, 2324101, 2324104, 2324107, 2324109, 2324110, 2324111, 2324112, 2324116, 2324117, 2324118, 2324119, 2324121, 2324122, 2324124, 2324125, 2324126, 2324128, 2324129, 2324130, 2324131, 2324132, 2324133, 2324134, 2324135, 2324137, 2324140, 2324143, 2324144, 2324145, 2324150, 2324151, 2324154, 2324158, 2324159, 2324160, 2324161, 2324163, 2324165, 2324169, 2324171, 2324172, 2324176, 2324177, 2324179, 2324180, 2324181, 2324182, 2324183, 2324185, 2324189, 2324190, 2324192, 2324193, 2324194, 2324197, 2324199, 2324201, 2324203, 2324204, 2324205, 2324206, 2324207, 2324208, 2324209, 2324213, 2324214, 2324215, 2324217, 2324218, 2324219, 2324221, 2324223, 2324224, 2324225, 2324226, 2324228, 2324229, 2324231, 2324232, 2324234, 2324237, 2324239, 2324240, 2324246, 2324247, 2324248, 2324250, 2324251, 2324256, 2324258, 2324259, 2324260, 2324262, 2324263, 2324264, 2324265, 2324266, 2324269, 2324270, 2324271, 2324272, 2324273, 2324274, 2324275, 2324276, 2324278, 2324279, 2324281, 2324282, 2324283, 2324285, 2324286, 2324287, 2324289, 2324290, 2324293, 2324294, 2324295, 2324296, 2324297, 2324299, 2324300, 2324301, 2324307, 2324308, 2324311, 2324313, 2324314, 2324315, 2324316, 2324317, 2324318, 2324319, 2324320, 2324321, 2324322, 2324324, 2324325, 2324327, 2324333, 2324338, 2324339, 2324340, 2324341, 2324343, 2324345, 2324347, 2324349, 2324350, 2324352, 2324353, 2324354, 2324355, 2324357, 2324358, 2324361, 2324362, 2324363, 2324364, 2324365, 2324366, 2324368, 2324370, 2324372, 2324376, 2324379, 2324382, 2324383, 2324385, 2324387, 2324389, 2324391, 2324392, 2324395, 2324397, 2324398, 2324399, 2324401, 2324402, 2324403, 2324404, 2324405, 2324407, 2324409, 2324410, 2324411, 2324412, 2324414, 2324415, 2324416, 2324417, 2324418, 2324420, 2324421, 2324422, 2324423, 2324424, 2324425, 2324427, 2324428, 2324429, 2324430, 2324431, 2324433, 2324435, 2324437, 2324440, 2324441, 2324442, 2324444, 2324446, 2324447, 2324450, 2324451, 2324453, 2324456, 2324457, 2324461, 2324466, 2324468, 2324469, 2324471, 2324472, 2324473, 2324474, 2324475, 2324476, 2324477, 2324479, 2324484, 2324486, 2324487, 2324488, 2324489, 2324490, 2324492, 2324493, 2324494, 2324496, 2324500, 2324501, 2324502, 2324505, 2324506, 2324507, 2324511, 2324512, 2324513, 2324515, 2324516, 2324517, 2324520, 2324521, 2324522, 2324525, 2324527, 2324528, 2324529, 2324530, 2324531, 2324533, 2324534, 2324537, 2324540, 2324541, 2324542, 2324543, 2324545, 2324548, 2324549, 2324551, 2324556, 2324557, 2324559, 2324560, 2324561, 2324562, 2324563, 2324564, 2324566, 2324568, 2324571, 2324572, 2324574, 2324575, 2324577, 2324579, 2324582, 2324583, 2324584, 2324585, 2324586, 2324587, 2324590, 2324591, 2324593, 2324594, 2324597, 2324599, 2324600, 2324602, 2324604, 2324610, 2324613, 2324615, 2324617, 2324619, 2324620, 2324622, 2324623, 2324624, 2324628, 2324630, 2324631, 2324633, 2324634, 2324635, 2324637, 2324638, 2324639, 2324640, 2324642, 2324645, 2324648, 2324650, 2324651, 2324652, 2324654, 2324655, 2324656, 2324659, 2324661, 2324663, 2324664, 2324665, 2324666, 2324669, 2324674, 2324676, 2324677, 2324678, 2324681, 2324683, 2324684, 2324687, 2324688, 2324689, 2324691, 2324692, 2324693, 2324695, 2324696, 2324697, 2324700, 2324702, 2324703, 2324704, 2324705, 2324706, 2324708, 2324710, 2324711, 2324712, 2324715, 2324718, 2324719, 2324720, 2324721, 2324723, 2324726, 2324729, 2324730, 2324731, 2324732, 2324733, 2324735, 2324737, 2324740, 2324741, 2324744, 2324746, 2324749, 2324751, 2324752, 2324753, 2324754, 2324756, 2324757, 2324760, 2324762, 2324765, 2324766, 2324769, 2324770, 2324772, 2324776, 2324778, 2324780, 2324783, 2324786, 2324789, 2324791, 2324798, 2324799, 2324803, 2324804, 2324807, 2324808, 2324809, 2324811, 2324812, 2324818, 2324819, 2324822, 2324823, 2324825, 2324826, 2324827, 2324828, 2324833, 2324834, 2324836, 2324837, 2324838, 2324842, 2324843, 2324844, 2324845, 2324846, 2324847, 2324848, 2324850, 2324852, 2324853, 2324854, 2324855, 2324856, 2324857, 2324858, 2324859, 2324861, 2324862, 2324864, 2324866, 2324868, 2324869, 2324870, 2324871, 2324872, 2324873, 2324874, 2324876, 2324879, 2324880, 2324881, 2324882, 2324883, 2324887, 2324888, 2324889, 2324892, 2324894, 2324896, 2324899, 2324901, 2324903, 2324904, 2324905, 2324907, 2324908, 2324909, 2324911, 2324913, 2324914, 2324915, 2324916, 2324917, 2324918, 2324920, 2324921, 2324922, 2324923, 2324928, 2324929, 2324931, 2324934, 2324935, 2324936, 2324938, 2324940, 2324941, 2324946, 2324948, 2324949, 2324952, 2324953, 2324954, 2324955, 2324956, 2324957, 2324958, 2324962, 2324963, 2324964, 2324965, 2324966, 2324967, 2324968, 2324969, 2324970, 2324974, 2324977, 2324984, 2324986, 2324987, 2324989, 2324990, 2324992, 2324993, 2324994, 2324995, 2324998, 2324999, 2325000, 2325001, 2325003, 2325005, 2325009, 2325010, 2325012, 2325013, 2325014, 2325017, 2325018, 2325019, 2325021, 2325022, 2325024, 2325027, 2325028, 2325030, 2325033, 2325034, 2325035, 2325036, 2325040, 2325042, 2325043, 2325044, 2325045, 2325047, 2325048, 2325049, 2325050, 2325051, 2325052, 2325053, 2325054, 2325055, 2325057, 2325059, 2325062, 2325063, 2325067, 2325070, 2325071, 2325078, 2325080, 2325082, 2325087, 2325088, 2325089, 2325090, 2325091, 2325092, 2325094, 2325095, 2325096, 2325097, 2325098, 2325100, 2325101, 2325102, 2325103, 2325106, 2325108, 2325110, 2325112, 2325115, 2325116, 2325118, 2325119, 2325120, 2325121, 2325122, 2325123, 2325124, 2325125, 2325126, 2325129, 2325131, 2325133, 2325137, 2325138, 2325140, 2325142, 2325143, 2325145, 2325146, 2325148, 2325149, 2325154, 2325155, 2325156, 2325157, 2325164, 2325168, 2325170, 2325175, 2325176, 2325177, 2325178, 2325180, 2325181, 2325183, 2325184, 2325186, 2325187, 2325188, 2325189, 2325190, 2325191, 2325192, 2325193, 2325194, 2325195, 2325197, 2325199, 2325200, 2325201, 2325202, 2325203, 2325207, 2325209, 2325211, 2325213, 2325214, 2325215, 2325218, 2325219, 2325220, 2325221, 2325222, 2325226, 2325229, 2325231, 2325232, 2325233, 2325234, 2325235, 2325236, 2325237, 2325238, 2325241, 2325242, 2325244, 2325245, 2325246, 2325248, 2325249, 2325251, 2325252, 2325254, 2325255, 2325257, 2325258, 2325259, 2325263, 2325266, 2325267, 2325268, 2325269, 2325270, 2325271, 2325273, 2325276, 2325277, 2325279, 2325280, 2325286, 2325287, 2325288, 2325290, 2325291, 2325293, 2325296, 2325297, 2325301, 2325303, 2325306, 2325307, 2325310, 2325312, 2325314, 2325315, 2325318, 2325319, 2325320, 2325321, 2325322, 2325324, 2325326, 2325327, 2325329, 2325330, 2325331, 2325332, 2325334, 2325335, 2325339, 2325340, 2325342, 2325343, 2325344, 2325347, 2325349, 2325350, 2325351, 2325352, 2325354, 2325355, 2325357, 2325358, 2325359, 2325361, 2325364, 2325365, 2325367, 2325371, 2325379, 2325381, 2325382, 2325383, 2325384, 2325386, 2325389, 2325390, 2325391, 2325392, 2325394, 2325397, 2325398, 2325399, 2325403, 2325404, 2325405, 2325406, 2325407, 2325408, 2325409, 2325413, 2325416, 2325418, 2325419, 2325422, 2325423, 2325424, 2325425, 2325426, 2325427, 2325428, 2325429, 2325430, 2325431, 2325432, 2325433, 2325434, 2325437, 2325438, 2325439, 2325442, 2325443, 2325445, 2325451, 2325452, 2325455, 2325456, 2325461, 2325462, 2325463, 2325465, 2325467, 2325470, 2325471, 2325472, 2325473, 2325475, 2325480, 2325481, 2325483, 2325485, 2325486, 2325493, 2325494, 2325495, 2325496, 2325497, 2325498, 2325499, 2325502, 2325503, 2325504, 2325505, 2325506, 2325507, 2325509, 2325510, 2325513, 2325514, 2325515, 2325517, 2325519, 2325522, 2325523, 2325525, 2325526, 2325534, 2325536, 2325537, 2325538, 2325541, 2325543, 2325544, 2325546, 2325549, 2325550, 2325551, 2325552, 2325553, 2325554, 2325556, 2325557, 2325558, 2325563, 2325564, 2325565, 2325567, 2325569, 2325572, 2325573, 2325574, 2325579, 2325581, 2325582, 2325584, 2325585, 2325586, 2325587, 2325589, 2325591, 2325592, 2325595, 2325596, 2325597, 2325599, 2325600, 2325601, 2325604, 2325605, 2325608, 2325609, 2325610, 2325615, 2325618, 2325619, 2325620, 2325621, 2325622, 2325623, 2325626, 2325631, 2325632, 2325633, 2325634, 2325635, 2325636, 2325637, 2325640, 2325642, 2325643, 2325644, 2325645, 2325646, 2325650, 2325651, 2325652, 2325654, 2325656, 2325657, 2325665, 2325667, 2325669, 2325670, 2325671, 2325673, 2325675, 2325676, 2325677, 2325678, 2325679, 2325682, 2325685, 2325686, 2325687, 2325689, 2325692, 2325694, 2325695, 2325696, 2325697, 2325703, 2325704, 2325705, 2325707, 2325709, 2325710, 2325712, 2325714, 2325718, 2325719, 2325720, 2325722, 2325723, 2325724, 2325725, 2325726, 2325727, 2325728, 2325733, 2325735, 2325736, 2325737, 2325738, 2325739, 2325741, 2325742, 2325743, 2325744, 2325745, 2325747, 2325748, 2325751, 2325752, 2325754, 2325756, 2325757, 2325758, 2325759, 2325760, 2325762, 2325764, 2325765, 2325767, 2325768, 2325770, 2325771, 2325773, 2325775, 2325777, 2325778, 2325779, 2325780, 2325781, 2325782, 2325788, 2325789, 2325792, 2325793, 2325795, 2325797, 2325800, 2325801, 2325802, 2325805, 2325807, 2325811, 2325812, 2325813, 2325814, 2325818, 2325819, 2325822, 2325825, 2325828, 2325830, 2325832, 2325833, 2325834, 2325835, 2325840, 2325841, 2325842, 2325844, 2325845, 2325846, 2325847, 2325848, 2325850, 2325853, 2325854, 2325858, 2325860, 2325862, 2325864, 2325865, 2325866, 2325869, 2325870, 2325871, 2325872, 2325873, 2325874, 2325875, 2325876, 2325878, 2325880, 2325881, 2325882, 2325883, 2325884, 2325885, 2325886, 2325888, 2325891, 2325894, 2325898, 2325899, 2325901, 2325903, 2325904, 2325905, 2325906, 2325910, 2325911, 2325913, 2325917, 2325918, 2325920, 2325921, 2325922, 2325923, 2325925, 2325926, 2325928, 2325929, 2325930, 2325932, 2325934, 2325937, 2325939, 2325940, 2325941, 2325942, 2325948, 2325949, 2325950, 2325951, 2325952, 2325953, 2325954, 2325955, 2325957, 2325959, 2325961, 2325962, 2325963, 2325964, 2325965, 2325966, 2325969, 2325970, 2325971, 2325972, 2325973, 2325975, 2325979, 2325983, 2325984, 2325985, 2325988, 2325989, 2325991, 2325992, 2325994, 2325996, 2325997, 2325998, 2325999, 2326000, 2326002, 2326005, 2326006, 2326007, 2326008, 2326012, 2326013, 2326014, 2326016, 2326018, 2326020, 2326024, 2326025, 2326026, 2326027, 2326030, 2326033, 2326034, 2326035, 2326036, 2326038, 2326039, 2326041, 2326043, 2326045, 2326046, 2326048, 2326050, 2326056, 2326057, 2326058, 2326059, 2326060, 2326061, 2326063, 2326065, 2326067, 2326068, 2326071, 2326075, 2326076, 2326077, 2326078, 2326081, 2326082, 2326083, 2326086, 2326087, 2326088, 2326090, 2326091, 2326092, 2326094, 2326095, 2326096, 2326098, 2326100, 2326105, 2326106, 2326107, 2326110, 2326112, 2326113, 2326114, 2326115, 2326116, 2326117, 2326118, 2326119, 2326121, 2326122, 2326123, 2326125, 2326127, 2326129, 2326133, 2326134, 2326135, 2326137, 2326138, 2326143, 2326144, 2326145, 2326148, 2326149, 2326150, 2326152, 2326153, 2326154, 2326157, 2326158, 2326160, 2326161, 2326167, 2326168, 2326169, 2326170, 2326171, 2326173, 2326175, 2326177, 2326178, 2326179, 2326181, 2326182, 2326183, 2326184, 2326185, 2326186, 2326188, 2326189, 2326190, 2326192, 2326193, 2326194, 2326195, 2326196, 2326198, 2326199, 2326200, 2326201, 2326202, 2326204, 2326206, 2326208, 2326209, 2326211, 2326212, 2326213, 2326219, 2326220, 2326221, 2326222, 2326223, 2326224, 2326225, 2326227, 2326228, 2326229, 2326230, 2326231, 2326233, 2326234, 2326235, 2326237, 2326238, 2326239, 2326240, 2326246, 2326249, 2326250, 2326252, 2326253, 2326255, 2326256, 2326257, 2326258, 2326259, 2326260, 2326262, 2326264, 2326265, 2326266, 2326267, 2326268, 2326269, 2326270, 2326272, 2326274, 2326276, 2326278, 2326282, 2326286, 2326287, 2326288, 2326290, 2326295, 2326299, 2326301, 2326304, 2326307, 2326309, 2326310, 2326311, 2326312, 2326314, 2326316, 2326317, 2326318, 2326319, 2326320, 2326322, 2326323, 2326324, 2326329, 2326330, 2326331, 2326333, 2326335, 2326336, 2326337, 2326338, 2326344, 2326345, 2326346, 2326348, 2326351, 2326361, 2326364, 2326366, 2326368, 2326370, 2326373, 2326375, 2326376, 2326377, 2326378, 2326379, 2326380, 2326383, 2326386, 2326387, 2326390, 2326392, 2326393, 2326394, 2326396, 2326397, 2326398, 2326399, 2326400, 2326401, 2326402, 2326403, 2326405, 2326406, 2326407, 2326408, 2326409, 2326410, 2326411, 2326412, 2326413, 2326414, 2326415, 2326417, 2326418, 2326419, 2326420, 2326422, 2326423, 2326424, 2326425, 2326426, 2326427, 2326428, 2326429, 2326433, 2326434, 2326435, 2326436, 2326437, 2326438, 2326439, 2326441, 2326442, 2326444, 2326445, 2326446, 2326447, 2326450, 2326453, 2326454, 2326455, 2326457, 2326458, 2326460, 2326461, 2326462, 2326463, 2326464, 2326465, 2326466, 2326468, 2326469, 2326471, 2326474, 2326475, 2326477, 2326478, 2326480, 2326482, 2326483, 2326485, 2326487, 2326490, 2326492, 2326493, 2326495, 2326496, 2326497, 2326499, 2326500, 2326501, 2326502, 2326506, 2326507, 2326509, 2326510, 2326511, 2326512, 2326513, 2326516, 2326518, 2326519, 2326521, 2326522, 2326524, 2326525, 2326528, 2326535, 2326538, 2326539, 2326540, 2326541, 2326543, 2326544, 2326545, 2326546, 2326547, 2326549, 2326550, 2326551, 2326552, 2326553, 2326554, 2326556, 2326558, 2326560, 2326561, 2326562, 2326564, 2326565, 2326566, 2326567, 2326568, 2326570, 2326571, 2326573, 2326574, 2326575, 2326576, 2326577, 2326580, 2326581, 2326582, 2326583, 2326587, 2326588, 2326589, 2326590, 2326593, 2326597, 2326599, 2326601, 2326602, 2326607, 2326608, 2326610, 2326611, 2326613, 2326616, 2326617, 2326619, 2326620, 2326621, 2326622, 2326625, 2326626, 2326627, 2326629, 2326630, 2326631, 2326635, 2326637, 2326639, 2326640, 2326641, 2326642, 2326643, 2326644, 2326646, 2326647, 2326648, 2326649, 2326650, 2326652, 2326655, 2326657, 2326658, 2326660, 2326661, 2326665, 2326666, 2326667, 2326669, 2326670, 2326671, 2326673, 2326674, 2326677, 2326679, 2326680, 2326682, 2326683, 2326686, 2326689, 2326690, 2326691, 2326693, 2326694, 2326698, 2326699, 2326700, 2326701, 2326704, 2326705, 2326707, 2326709, 2326711, 2326712, 2326713, 2326714, 2326717, 2326720, 2326721, 2326723, 2326725, 2326728, 2326729, 2326730, 2326731, 2326732, 2326734, 2326737, 2326739, 2326740, 2326742, 2326743, 2326744, 2326745, 2326746, 2326747, 2326749, 2326750, 2326753, 2326754, 2326755, 2326756, 2326759, 2326763, 2326765, 2326770, 2326772, 2326773, 2326774, 2326775, 2326776, 2326777, 2326779, 2326780, 2326781, 2326782, 2326783, 2326788, 2326789, 2326790, 2326791, 2326792, 2326793, 2326795, 2326798, 2326800, 2326801, 2326803, 2326804, 2326805, 2326806, 2326807, 2326808, 2326809, 2326816, 2326818, 2326819, 2326821, 2326827, 2326828, 2326830, 2326831, 2326833, 2326834, 2326835, 2326837, 2326838, 2326839, 2326841, 2326842, 2326846, 2326847, 2326849, 2326851, 2326852, 2326855, 2326856, 2326857, 2326860, 2326861, 2326862, 2326863, 2326864, 2326865, 2326866, 2326867, 2326871, 2326872, 2326874, 2326875, 2326876, 2326877, 2326878, 2326879, 2326880, 2326881, 2326882, 2326883, 2326884, 2326885, 2326886, 2326887, 2326888, 2326889, 2326892, 2326893, 2326896, 2326897, 2326898, 2326900, 2326902, 2326904, 2326906, 2326908, 2326909, 2326910, 2326911, 2326912, 2326915, 2326916, 2326917, 2326918, 2326921, 2326922, 2326924, 2326926, 2326930, 2326933, 2326934, 2326937, 2326938, 2326939, 2326940, 2326941, 2326942, 2326946, 2326949, 2326953, 2326954, 2326955, 2326957, 2326958, 2326960, 2326962, 2326965, 2326966, 2326967, 2326968, 2326970, 2326974, 2326975, 2326976, 2326978, 2326980, 2326981, 2326982, 2326983, 2326984, 2326985, 2326986, 2326987, 2326988, 2326989, 2326991, 2326993, 2326995, 2326996, 2326998, 2326999, 2327002, 2327003, 2327004, 2327006, 2327009, 2327011, 2327012, 2327015, 2327016, 2327017, 2327020, 2327028, 2327029, 2327031, 2327032, 2327033, 2327034, 2327035, 2327037, 2327039, 2327041, 2327042, 2327043, 2327045, 2327047, 2327050, 2327053, 2327054, 2327059, 2327060, 2327061, 2327062, 2327064, 2327065, 2327067, 2327069, 2327070, 2327071, 2327074, 2327076, 2327077, 2327078, 2327079, 2327080, 2327082, 2327083, 2327085, 2327086, 2327090, 2327092, 2327094, 2327095, 2327096, 2327098, 2327100, 2327101, 2327103, 2327104, 2327105, 2327106, 2327108, 2327110, 2327111, 2327112, 2327113, 2327114, 2327115, 2327116, 2327117, 2327118, 2327119, 2327120, 2327122, 2327123, 2327124, 2327126, 2327127, 2327129, 2327131, 2327132, 2327135, 2327138, 2327140, 2327143, 2327145, 2327147, 2327149, 2327150, 2327152, 2327153, 2327154, 2327157, 2327159, 2327163, 2327164, 2327167, 2327168, 2327177, 2327179, 2327181, 2327182, 2327184, 2327186, 2327187, 2327188, 2327189, 2327190, 2327191, 2327192, 2327193, 2327194, 2327195, 2327196, 2327198, 2327199, 2327201, 2327202, 2327204, 2327206, 2327207, 2327210, 2327211, 2327213, 2327216, 2327217, 2327218, 2327219, 2327220, 2327221, 2327224, 2327226, 2327227, 2327229, 2327230, 2327231, 2327232, 2327233, 2327234, 2327235, 2327240, 2327241, 2327242, 2327243, 2327247, 2327248, 2327250, 2327252, 2327254, 2327256, 2327258, 2327259, 2327260, 2327268, 2327271, 2327274, 2327275, 2327276, 2327277, 2327279, 2327280, 2327281, 2327282, 2327283, 2327284, 2327285, 2327286, 2327288, 2327291, 2327293, 2327294, 2327297, 2327299, 2327300, 2327301, 2327305, 2327306, 2327309, 2327310, 2327312, 2327313, 2327314, 2327317, 2327318, 2327319, 2327320, 2327321, 2327322, 2327323, 2327324, 2327325, 2327327, 2327328, 2327332, 2327333, 2327335, 2327338, 2327339, 2327343, 2327346, 2327347, 2327351, 2327353, 2327354, 2327356, 2327357, 2327360, 2327363, 2327364, 2327365, 2327366, 2327367, 2327370, 2327372, 2327376, 2327378, 2327379, 2327381, 2327386, 2327388, 2327389, 2327390, 2327391, 2327392, 2327394, 2327395, 2327397, 2327398, 2327399, 2327401, 2327402, 2327404, 2327406, 2327408, 2327410, 2327411, 2327412, 2327414, 2327416, 2327417, 2327418, 2327419, 2327420, 2327421, 2327425, 2327426, 2327427, 2327428, 2327429, 2327430, 2327433, 2327434, 2327436, 2327439, 2327440, 2327442, 2327444, 2327445, 2327448, 2327449, 2327454, 2327455, 2327456, 2327459, 2327460, 2327461, 2327462, 2327463, 2327465, 2327469, 2327472, 2327473, 2327475, 2327476, 2327477, 2327478, 2327479, 2327480, 2327482, 2327483, 2327484, 2327488, 2327489, 2327490, 2327491, 2327493, 2327494, 2327495, 2327496, 2327497, 2327498, 2327499, 2327501, 2327502, 2327503, 2327505, 2327508, 2327510, 2327513, 2327514, 2327516, 2327517, 2327518, 2327519, 2327520, 2327521, 2327522, 2327523, 2327524, 2327525, 2327526, 2327528, 2327530, 2327531, 2327532, 2327533, 2327536, 2327543, 2327544, 2327549, 2327550, 2327552, 2327554, 2327555, 2327558, 2327560, 2327563, 2327564, 2327565, 2327566, 2327567, 2327568, 2327570, 2327573, 2327574, 2327576, 2327577, 2327579, 2327582, 2327587, 2327588, 2327591, 2327592, 2327594, 2327595, 2327597, 2327598, 2327599, 2327600, 2327602, 2327604, 2327605, 2327608, 2327610, 2327611, 2327613, 2327614, 2327618, 2327620, 2327623, 2327624, 2327625, 2327628, 2327629, 2327632, 2327633, 2327640, 2327642, 2327643, 2327645, 2327646, 2327648, 2327649, 2327650, 2327651, 2327654, 2327655, 2327656, 2327657, 2327658, 2327659, 2327661, 2327663, 2327667, 2327669, 2327672, 2327673, 2327674, 2327675, 2327677, 2327679, 2327680, 2327681, 2327682, 2327683, 2327684, 2327686, 2327689, 2327691, 2327692, 2327693, 2327694, 2327696, 2327704, 2327705, 2327708, 2327709, 2327711, 2327714, 2327716, 2327717, 2327718, 2327721, 2327722, 2327723, 2327724, 2327725, 2327728, 2327729, 2327730, 2327732, 2327733, 2327734, 2327735, 2327736, 2327737, 2327739, 2327743, 2327745, 2327746, 2327750, 2327751, 2327752, 2327753, 2327755, 2327758, 2327760, 2327761, 2327764, 2327766, 2327767, 2327768, 2327769, 2327770, 2327771, 2327772, 2327773, 2327774, 2327775, 2327776, 2327778, 2327781, 2327782, 2327783, 2327787, 2327789, 2327791, 2327792, 2327793, 2327794, 2327795, 2327796, 2327799, 2327800, 2327803, 2327804, 2327805, 2327806, 2327807, 2327808, 2327811, 2327812, 2327813, 2327816, 2327817, 2327818, 2327819, 2327821, 2327822, 2327824, 2327828, 2327829, 2327832, 2327834, 2327836, 2327837, 2327839, 2327840, 2327841, 2327843, 2327844, 2327846, 2327847, 2327848, 2327850, 2327851, 2327854, 2327855, 2327857, 2327859, 2327860, 2327865, 2327866, 2327868, 2327870, 2327871, 2327872, 2327873, 2327874, 2327875, 2327876, 2327877, 2327878, 2327879, 2327880, 2327883, 2327884, 2327885, 2327886, 2327889, 2327890, 2327892, 2327894, 2327895, 2327896, 2327897, 2327899, 2327903, 2327904, 2327908, 2327909, 2327910, 2327911, 2327916, 2327917, 2327918, 2327920, 2327921, 2327922, 2327924, 2327925, 2327927, 2327928, 2327930, 2327931, 2327932, 2327935, 2327936, 2327939, 2327941, 2327942, 2327944, 2327945, 2327946, 2327947, 2327950, 2327951, 2327956, 2327960, 2327961, 2327962, 2327964, 2327965, 2327966, 2327970, 2327973, 2327974, 2327975, 2327977, 2327978, 2327980, 2327982, 2327983, 2327985, 2327986, 2327987, 2327989, 2327990, 2327991, 2327992, 2327993, 2327994, 2327998, 2327999, 2328000, 2328002, 2328006, 2328008, 2328010, 2328011, 2328012, 2328013, 2328018, 2328019, 2328020, 2328021, 2328022, 2328025, 2328028, 2328029, 2328030, 2328032, 2328033, 2328034, 2328035, 2328037, 2328039, 2328040, 2328042, 2328043, 2328044, 2328045, 2328046, 2328050, 2328051, 2328054, 2328056, 2328058, 2328060, 2328061, 2328062, 2328063, 2328067, 2328068, 2328069, 2328070, 2328072, 2328074, 2328075, 2328079, 2328082, 2328083, 2328085, 2328088, 2328091, 2328093, 2328094, 2328097, 2328098, 2328100, 2328101, 2328103, 2328104, 2328105, 2328107, 2328109, 2328111, 2328113, 2328114, 2328115, 2328116, 2328117, 2328119, 2328120, 2328121, 2328126, 2328127, 2328128, 2328131, 2328133, 2328135, 2328136, 2328138, 2328140, 2328141, 2328144, 2328147, 2328149, 2328150, 2328153, 2328154, 2328158, 2328164, 2328165, 2328166, 2328167, 2328168, 2328170, 2328171, 2328172, 2328173, 2328174, 2328176, 2328178, 2328181, 2328182, 2328183, 2328184, 2328188, 2328190, 2328191, 2328192, 2328193, 2328194, 2328196, 2328197, 2328198, 2328199, 2328200, 2328202, 2328204, 2328205, 2328206, 2328207, 2328209, 2328210, 2328211, 2328213, 2328214, 2328217, 2328218, 2328220, 2328222, 2328224, 2328225, 2328227, 2328229, 2328230, 2328232, 2328237, 2328238, 2328241, 2328242, 2328243, 2328244, 2328245, 2328247, 2328248, 2328256, 2328257, 2328258, 2328260, 2328261, 2328265, 2328266, 2328267, 2328268, 2328270, 2328271, 2328272, 2328274, 2328275, 2328276, 2328278, 2328279, 2328280, 2328281, 2328282, 2328283, 2328284, 2328289, 2328290, 2328292, 2328293, 2328294, 2328295, 2328297, 2328298, 2328299, 2328301, 2328302, 2328305, 2328310, 2328311, 2328314, 2328317, 2328319, 2328320, 2328321, 2328322, 2328323, 2328324, 2328325, 2328327, 2328328, 2328329, 2328330, 2328331, 2328333, 2328334, 2328335, 2328336, 2328338, 2328339, 2328340, 2328341, 2328342, 2328344, 2328347, 2328350, 2328351, 2328353, 2328354, 2328355, 2328361, 2328364, 2328365, 2328367, 2328368, 2328369, 2328371, 2328373, 2328375, 2328378, 2328379, 2328380, 2328381, 2328382, 2328385, 2328387, 2328390, 2328391, 2328393, 2328394, 2328395, 2328397, 2328400, 2328401, 2328402, 2328403, 2328406, 2328407, 2328408, 2328412, 2328416, 2328417, 2328420, 2328421, 2328422, 2328423, 2328424, 2328425, 2328426, 2328428, 2328430, 2328433, 2328434, 2328438, 2328439, 2328440, 2328442, 2328444, 2328447, 2328451, 2328453, 2328454, 2328456, 2328457, 2328461, 2328462, 2328465, 2328468, 2328473, 2328475, 2328477, 2328480, 2328481, 2328482, 2328484, 2328485, 2328487, 2328488, 2328491, 2328495, 2328497, 2328498, 2328501, 2328502, 2328503, 2328504, 2328506, 2328507, 2328508, 2328509, 2328510, 2328512, 2328513, 2328515, 2328517, 2328519, 2328524, 2328525, 2328526, 2328529, 2328530, 2328531, 2328535, 2328539, 2328540, 2328541, 2328542, 2328543, 2328544, 2328545, 2328547, 2328550, 2328551, 2328552, 2328554, 2328556, 2328557, 2328558, 2328560, 2328561, 2328563, 2328564, 2328565, 2328567, 2328569, 2328570, 2328573, 2328574, 2328576, 2328577, 2328581, 2328583, 2328586, 2328588, 2328589, 2328590, 2328594, 2328595, 2328598, 2328599, 2328600, 2328601, 2328602, 2328606, 2328607, 2328608, 2328609, 2328614, 2328616, 2328618, 2328620, 2328621, 2328624, 2328625, 2328628, 2328631, 2328632, 2328633, 2328634, 2328635, 2328637, 2328638, 2328639, 2328640, 2328641, 2328642, 2328644, 2328645, 2328647, 2328649, 2328652, 2328653, 2328654, 2328655, 2328656, 2328657, 2328658, 2328659, 2328660, 2328661, 2328662, 2328663, 2328665, 2328666, 2328669, 2328674, 2328678, 2328681, 2328682, 2328683, 2328684, 2328685, 2328687, 2328688, 2328691, 2328693, 2328695, 2328697, 2328700, 2328701, 2328704, 2328708, 2328711, 2328712, 2328714, 2328715, 2328716, 2328718, 2328719, 2328722, 2328723, 2328724, 2328725, 2328726, 2328727, 2328728, 2328731, 2328732, 2328733, 2328734, 2328735, 2328736, 2328738, 2328740, 2328741, 2328743, 2328746, 2328749, 2328750, 2328754, 2328755, 2328757, 2328758, 2328759, 2328760, 2328761, 2328762, 2328763, 2328764, 2328767, 2328769, 2328770, 2328773, 2328774, 2328776, 2328779, 2328781, 2328782, 2328784, 2328786, 2328787, 2328788, 2328789, 2328790, 2328792, 2328795, 2328796, 2328797, 2328798, 2328799, 2328802, 2328804, 2328806, 2328807, 2328809, 2328814, 2328815, 2328822, 2328824, 2328825, 2328826, 2328827, 2328828, 2328829, 2328831, 2328833, 2328834, 2328839, 2328840, 2328841, 2328844, 2328845, 2328847, 2328848, 2328849, 2328850, 2328851, 2328854, 2328858, 2328862, 2328863, 2328864, 2328865, 2328867, 2328869, 2328870, 2328871, 2328876, 2328878, 2328880, 2328882, 2328883, 2328884, 2328891, 2328892, 2328894, 2328896, 2328897, 2328899, 2328900, 2328901, 2328902, 2328903, 2328904, 2328905, 2328906, 2328908, 2328909, 2328912, 2328914, 2328915, 2328917, 2328918, 2328920, 2328921, 2328924, 2328925, 2328929, 2328932, 2328934, 2328935, 2328937, 2328940, 2328942, 2328943, 2328944, 2328947, 2328948, 2328949, 2328950, 2328953, 2328954, 2328956, 2328957, 2328958, 2328960, 2328961, 2328963, 2328965, 2328966, 2328968, 2328969, 2328971, 2328976, 2328977, 2328978, 2328981, 2328982, 2328986, 2328987, 2328988, 2328989, 2328991, 2328992, 2328994, 2328995, 2328996, 2328998, 2329000, 2329001, 2329004, 2329006, 2329007, 2329008, 2329009, 2329011, 2329012, 2329014, 2329015, 2329016, 2329017, 2329020, 2329021, 2329023, 2329027, 2329028, 2329030, 2329032, 2329033, 2329035, 2329038, 2329039, 2329042, 2329045, 2329046, 2329048, 2329053, 2329054, 2329058, 2329059, 2329060, 2329061, 2329062, 2329063, 2329064, 2329066, 2329070, 2329073, 2329074, 2329075, 2329078, 2329079, 2329080, 2329082, 2329083, 2329084, 2329086, 2329089, 2329091, 2329092, 2329093, 2329096, 2329099, 2329100, 2329101, 2329107, 2329109, 2329110, 2329112, 2329114, 2329115, 2329117, 2329118, 2329120, 2329122, 2329123, 2329125, 2329126, 2329127, 2329128, 2329129, 2329130, 2329132, 2329133, 2329134, 2329140, 2329142, 2329149, 2329150, 2329151, 2329152, 2329153, 2329155, 2329156, 2329157, 2329158, 2329159, 2329160, 2329161, 2329163, 2329165, 2329166, 2329168, 2329171, 2329173, 2329175, 2329178, 2329179, 2329181, 2329182, 2329184, 2329187, 2329188, 2329189, 2329192, 2329193, 2329194, 2329195, 2329198, 2329201, 2329203, 2329204, 2329208, 2329210, 2329211, 2329213, 2329214, 2329218, 2329219, 2329221, 2329222, 2329224, 2329225, 2329226, 2329228, 2329230, 2329233, 2329234, 2329235, 2329236, 2329237, 2329241, 2329242, 2329243, 2329244, 2329245, 2329246, 2329247, 2329249, 2329250, 2329251, 2329252, 2329253, 2329256, 2329257, 2329259, 2329260, 2329261, 2329262, 2329265, 2329266, 2329267, 2329268, 2329270, 2329271, 2329272, 2329273, 2329275, 2329276, 2329277, 2329278, 2329279, 2329282, 2329283, 2329286, 2329287, 2329288, 2329289, 2329290, 2329292, 2329295, 2329296, 2329297, 2329298, 2329299, 2329301, 2329302, 2329303, 2329304, 2329305, 2329306, 2329307, 2329308, 2329309, 2329310, 2329311, 2329312, 2329313, 2329316, 2329317, 2329318, 2329319, 2329320, 2329324, 2329327, 2329328, 2329331, 2329332, 2329334, 2329338, 2329340, 2329341, 2329342, 2329343, 2329344, 2329351, 2329353, 2329354, 2329355, 2329356, 2329358, 2329359, 2329361, 2329362, 2329366, 2329367, 2329369, 2329371, 2329374, 2329377, 2329378, 2329380, 2329381, 2329382, 2329384, 2329386, 2329387, 2329388, 2329389, 2329390, 2329391, 2329392, 2329394, 2329395, 2329398, 2329399, 2329400, 2329401, 2329404, 2329405, 2329407, 2329411, 2329414, 2329416, 2329417, 2329419, 2329420, 2329421, 2329426, 2329428, 2329429, 2329430, 2329431, 2329432, 2329433, 2329434, 2329435, 2329437, 2329438, 2329441, 2329443, 2329444, 2329445, 2329446, 2329449, 2329450, 2329453, 2329456, 2329458, 2329461, 2329462, 2329464, 2329465, 2329466, 2329469, 2329470, 2329471, 2329472, 2329473, 2329474, 2329476, 2329480, 2329482, 2329484, 2329486, 2329487, 2329488, 2329490, 2329492, 2329495, 2329496, 2329500, 2329501, 2329502, 2329503, 2329505, 2329506, 2329507, 2329508, 2329513, 2329514, 2329516, 2329517, 2329521, 2329523, 2329528, 2329529, 2329533, 2329534, 2329536, 2329537, 2329538, 2329539, 2329540, 2329541, 2329542, 2329543, 2329545, 2329546, 2329548, 2329551, 2329553, 2329555, 2329556, 2329557, 2329558, 2329562, 2329563, 2329564, 2329565, 2329566, 2329568, 2329569, 2329570, 2329571, 2329572, 2329573, 2329575, 2329576, 2329581, 2329582, 2329583, 2329587, 2329589, 2329592, 2329595, 2329596, 2329597, 2329598, 2329599, 2329600, 2329601, 2329607, 2329608, 2329609, 2329610, 2329614, 2329616, 2329620, 2329621, 2329623, 2329624, 2329625, 2329626, 2329630, 2329631, 2329633, 2329635, 2329637, 2329638, 2329641, 2329642, 2329644, 2329645, 2329647, 2329648, 2329650, 2329652, 2329653, 2329655, 2329656, 2329657, 2329658, 2329659, 2329661, 2329663, 2329664, 2329665, 2329668, 2329669, 2329671, 2329674, 2329675, 2329676, 2329677, 2329678, 2329679, 2329680, 2329681, 2329682, 2329683, 2329685, 2329686, 2329687, 2329688, 2329691, 2329692, 2329693, 2329695, 2329697, 2329698, 2329699, 2329702, 2329703, 2329706, 2329707, 2329708, 2329709, 2329711, 2329712, 2329713, 2329714, 2329716, 2329717, 2329720, 2329721, 2329724, 2329725, 2329726, 2329728, 2329731, 2329732, 2329736, 2329739, 2329742, 2329743, 2329746, 2329748, 2329750, 2329754, 2329755, 2329758, 2329761, 2329762, 2329763, 2329764, 2329766, 2329767, 2329769, 2329770, 2329771, 2329772, 2329773, 2329775, 2329776, 2329779, 2329780, 2329781, 2329782, 2329783, 2329784, 2329785, 2329786, 2329794, 2329795, 2329797, 2329801, 2329804, 2329805, 2329806, 2329807, 2329808, 2329810, 2329811, 2329812, 2329813, 2329815, 2329816, 2329819, 2329820, 2329821, 2329823, 2329824, 2329825, 2329828, 2329829, 2329832, 2329833, 2329834, 2329837, 2329838, 2329839, 2329840, 2329842, 2329843, 2329844, 2329846, 2329849, 2329850, 2329851, 2329852, 2329853, 2329854, 2329856, 2329857, 2329862, 2329865, 2329866, 2329867, 2329869, 2329870, 2329871, 2329872, 2329873, 2329874, 2329875, 2329878, 2329879, 2329880, 2329881, 2329885, 2329887, 2329888, 2329889, 2329890, 2329892, 2329893, 2329894, 2329895, 2329896, 2329897, 2329898, 2329899, 2329900, 2329901, 2329902, 2329903, 2329904, 2329906, 2329907, 2329908, 2329910, 2329911, 2329914, 2329915, 2329918, 2329920, 2329921, 2329924, 2329925, 2329926, 2329927, 2329928, 2329931, 2329933, 2329935, 2329936, 2329937, 2329939, 2329943, 2329945, 2329946, 2329947, 2329948, 2329949, 2329955, 2329957, 2329959, 2329962, 2329963, 2329964, 2329965, 2329966, 2329968, 2329974, 2329976, 2329977, 2329978, 2329980, 2329981, 2329982, 2329983, 2329984, 2329985, 2329986, 2329987, 2329989, 2329990, 2329991, 2329993, 2329994, 2329995, 2329996, 2329999, 2330000, 2330001, 2330003, 2330005, 2330006, 2330009, 2330011, 2330014, 2330015, 2330016, 2330019, 2330021, 2330023, 2330024, 2330025, 2330026, 2330027, 2330029, 2330030, 2330032, 2330033, 2330035, 2330036, 2330040, 2330042, 2330044, 2330048, 2330049, 2330051, 2330052, 2330054, 2330056, 2330059, 2330061, 2330062, 2330063, 2330065, 2330066, 2330067, 2330068, 2330069, 2330070, 2330071, 2330072, 2330073, 2330075, 2330076, 2330077, 2330080, 2330082, 2330084, 2330086, 2330087, 2330089, 2330090, 2330091, 2330093, 2330094, 2330095, 2330096, 2330098, 2330099, 2330100, 2330102, 2330105, 2330106, 2330107, 2330109, 2330112, 2330113, 2330114, 2330115, 2330116, 2330118, 2330119, 2330120, 2330121, 2330123, 2330124, 2330125, 2330126, 2330127, 2330128, 2330130, 2330131, 2330132, 2330133, 2330134, 2330136, 2330138, 2330140, 2330141, 2330144, 2330145, 2330146, 2330149, 2330150, 2330153, 2330155, 2330156, 2330157, 2330158, 2330160, 2330161, 2330163, 2330164, 2330167, 2330168, 2330169, 2330171, 2330172, 2330174, 2330175, 2330176, 2330177, 2330178, 2330179, 2330180, 2330181, 2330183, 2330185, 2330187, 2330189, 2330190, 2330191, 2330194, 2330196, 2330198, 2330199, 2330200, 2330202, 2330204, 2330206, 2330207, 2330208, 2330212, 2330213, 2330217, 2330219, 2330220, 2330221, 2330222, 2330223, 2330225, 2330226, 2330228, 2330229, 2330230, 2330231, 2330234, 2330235, 2330236, 2330238, 2330239, 2330242, 2330244, 2330245, 2330246, 2330248, 2330251, 2330252, 2330253, 2330255, 2330257, 2330262, 2330263, 2330264, 2330266, 2330268, 2330276, 2330277, 2330278, 2330280, 2330281, 2330282, 2330284, 2330288, 2330290, 2330291, 2330292, 2330293, 2330295, 2330296, 2330297, 2330299, 2330300, 2330301, 2330302, 2330303, 2330308, 2330312, 2330313, 2330316, 2330317, 2330319, 2330321, 2330322, 2330325, 2330326, 2330328, 2330329, 2330331, 2330332, 2330333, 2330334, 2330335, 2330336, 2330338, 2330339, 2330340, 2330343, 2330346, 2330347, 2330348, 2330353, 2330357, 2330358, 2330360, 2330361, 2330362, 2330363, 2330365, 2330366, 2330367, 2330370, 2330371, 2330372, 2330374, 2330377, 2330378, 2330379, 2330380, 2330383, 2330384, 2330386, 2330387, 2330388, 2330389, 2330391, 2330392, 2330393, 2330395, 2330396, 2330398, 2330399, 2330400, 2330401, 2330403, 2330405, 2330406, 2330407, 2330408, 2330410, 2330413, 2330414, 2330415, 2330417, 2330418, 2330419, 2330420, 2330421, 2330424, 2330425, 2330428, 2330430, 2330431, 2330432, 2330434, 2330435, 2330436, 2330438, 2330439, 2330442, 2330444, 2330445, 2330446, 2330448, 2330449, 2330451, 2330452, 2330453, 2330454, 2330458, 2330460, 2330461, 2330462, 2330463, 2330464, 2330466, 2330467, 2330468, 2330469, 2330471, 2330472, 2330473, 2330474, 2330475, 2330476, 2330479, 2330480, 2330483, 2330485, 2330488, 2330491, 2330493, 2330497, 2330499, 2330501, 2330502, 2330503, 2330504, 2330505, 2330507, 2330508, 2330511, 2330513, 2330514, 2330515, 2330517, 2330518, 2330519, 2330520, 2330522, 2330523, 2330524, 2330526, 2330528, 2330529, 2330530, 2330532, 2330536, 2330537, 2330541, 2330542, 2330543, 2330545, 2330547, 2330548, 2330549, 2330550, 2330551, 2330555, 2330559, 2330560, 2330561, 2330562, 2330567, 2330568, 2330570, 2330573, 2330575, 2330576, 2330579, 2330580, 2330582, 2330583, 2330584, 2330585, 2330586, 2330587, 2330588, 2330589, 2330590, 2330591, 2330593, 2330594, 2330597, 2330599, 2330605, 2330606, 2330607, 2330608, 2330610, 2330611, 2330613, 2330616, 2330620, 2330621, 2330626, 2330628, 2330629, 2330631, 2330634, 2330637, 2330639, 2330643, 2330648, 2330649, 2330652, 2330653, 2330654, 2330658, 2330659, 2330661, 2330663, 2330666, 2330668, 2330672, 2330675, 2330676, 2330679, 2330681, 2330682, 2330684, 2330685, 2330686, 2330688, 2330689, 2330690, 2330692, 2330696, 2330700, 2330703, 2330704, 2330705, 2330708, 2330709, 2330710, 2330717, 2330719, 2330721, 2330722, 2330724, 2330726, 2330729, 2330730, 2330731, 2330733, 2330735, 2330737, 2330738, 2330741, 2330742, 2330744, 2330746, 2330748, 2330749, 2330750, 2330754, 2330756, 2330758, 2330759, 2330760, 2330761, 2330763, 2330764, 2330765, 2330769, 2330770, 2330771, 2330772, 2330774, 2330777, 2330778, 2330779, 2330780, 2330781, 2330782, 2330783, 2330784, 2330785, 2330788, 2330789, 2330792, 2330793, 2330794, 2330795, 2330796, 2330798, 2330800, 2330802, 2330805, 2330809, 2330811, 2330813, 2330814, 2330816, 2330819, 2330820, 2330823, 2330824, 2330825, 2330827, 2330828, 2330829, 2330830, 2330832, 2330833, 2330835, 2330836, 2330837, 2330838, 2330839, 2330841, 2330842, 2330845, 2330846, 2330847, 2330848, 2330849, 2330851, 2330852, 2330853, 2330856, 2330857, 2330858, 2330859, 2330860, 2330861, 2330862, 2330863, 2330864, 2330866, 2330870, 2330872, 2330875, 2330880, 2330881, 2330883, 2330884, 2330886, 2330888, 2330889, 2330890, 2330891, 2330896, 2330897, 2330898, 2330899, 2330900, 2330902, 2330904, 2330907, 2330909, 2330911, 2330916, 2330920, 2330921, 2330922, 2330923, 2330925, 2330927, 2330929, 2330930, 2330931, 2330934, 2330936, 2330937, 2330939, 2330940, 2330942, 2330943, 2330944, 2330945, 2330946, 2330947, 2330949, 2330950, 2330951, 2330952, 2330954, 2330956, 2330957, 2330958, 2330960, 2330961, 2330962, 2330964, 2330965, 2330966, 2330967, 2330968, 2330969, 2330970, 2330971, 2330972, 2330974, 2330975, 2330976, 2330977, 2330981, 2330982, 2330983, 2330984, 2330986, 2330987, 2330990, 2330991, 2330993, 2330994, 2330995, 2330997, 2330998, 2331000, 2331002, 2331003, 2331004, 2331005, 2331008, 2331009, 2331012, 2331014, 2331015, 2331018, 2331020, 2331021, 2331022, 2331023, 2331024, 2331025, 2331026, 2331027, 2331028, 2331029, 2331031, 2331034, 2331035, 2331037, 2331038, 2331041, 2331042, 2331043, 2331044, 2331045, 2331046, 2331047, 2331048, 2331049, 2331050, 2331051, 2331052, 2331054, 2331056, 2331058, 2331060, 2331064, 2331065, 2331066, 2331068, 2331069, 2331070, 2331072, 2331074, 2331076, 2331077, 2331084, 2331087, 2331088, 2331089, 2331090, 2331092, 2331093, 2331094, 2331095, 2331099, 2331101, 2331102, 2331103, 2331104, 2331105, 2331106, 2331107, 2331111, 2331112, 2331113, 2331114, 2331115, 2331116, 2331117, 2331118, 2331119, 2331120, 2331121, 2331122, 2331127, 2331128, 2331129, 2331130, 2331134, 2331135, 2331136, 2331138, 2331140, 2331141, 2331143, 2331144, 2331145, 2331146, 2331150, 2331153, 2331159, 2331161, 2331163, 2331164, 2331166, 2331167, 2331168, 2331171, 2331174, 2331175, 2331176, 2331177, 2331178, 2331179, 2331181, 2331183, 2331184, 2331186, 2331187, 2331188, 2331189, 2331190, 2331191, 2331192, 2331194, 2331196, 2331197, 2331198, 2331200, 2331201, 2331202, 2331203, 2331204, 2331206, 2331207, 2331209, 2331211, 2331212, 2331213, 2331215, 2331216, 2331217, 2331218, 2331219, 2331220, 2331221, 2331225, 2331227, 2331229, 2331230, 2331231, 2331232, 2331233, 2331236, 2331237, 2331239, 2331242, 2331244, 2331245, 2331246, 2331247, 2331248, 2331250, 2331252, 2331254, 2331255, 2331256, 2331258, 2331259, 2331261, 2331262, 2331263, 2331264, 2331268, 2331269, 2331270, 2331272, 2331274, 2331275, 2331276, 2331278, 2331281, 2331283, 2331285, 2331286, 2331287, 2331288, 2331290, 2331291, 2331294, 2331295, 2331297, 2331298, 2331299, 2331300, 2331301, 2331302, 2331303, 2331304, 2331305, 2331306, 2331310, 2331311, 2331312, 2331314, 2331316, 2331317, 2331319, 2331321, 2331322, 2331324, 2331325, 2331326, 2331327, 2331329, 2331330, 2331332, 2331335, 2331336, 2331340, 2331343, 2331344, 2331345, 2331346, 2331347, 2331352, 2331353, 2331354, 2331355, 2331356, 2331359, 2331360, 2331361, 2331363, 2331364, 2331368, 2331373, 2331374, 2331376, 2331379, 2331382, 2331384, 2331387, 2331388, 2331394, 2331395, 2331396, 2331397, 2331398, 2331399, 2331400, 2331401, 2331402, 2331404, 2331412, 2331418, 2331419, 2331421, 2331423, 2331430, 2331432, 2331433, 2331434, 2331435, 2331437, 2331439, 2331441, 2331442, 2331443, 2331444, 2331445, 2331447, 2331448, 2331449, 2331453, 2331454, 2331455, 2331457, 2331459, 2331460, 2331462, 2331464, 2331468, 2331469, 2331470, 2331472, 2331473, 2331475, 2331476, 2331481, 2331482, 2331483, 2331485, 2331487, 2331488, 2331490, 2331491, 2331492, 2331499, 2331500, 2331501, 2331502, 2331503, 2331505, 2331506, 2331508, 2331510, 2331516, 2331517, 2331519, 2331520, 2331521, 2331523, 2331524, 2331526, 2331528, 2331529, 2331531, 2331533, 2331537, 2331538, 2331539, 2331540, 2331542, 2331543, 2331544, 2331546, 2331548, 2331549, 2331550, 2331551, 2331552, 2331555, 2331556, 2331558, 2331559, 2331560, 2331562, 2331564, 2331565, 2331568, 2331569, 2331571, 2331575, 2331576, 2331577, 2331579, 2331580, 2331581, 2331582, 2331583, 2331585, 2331588, 2331589, 2331590, 2331591, 2331595, 2331596, 2331598, 2331599, 2331601, 2331603, 2331605, 2331606, 2331609, 2331610, 2331611, 2331613, 2331614, 2331616, 2331617, 2331619, 2331620, 2331621, 2331622, 2331623, 2331624, 2331627, 2331628, 2331629, 2331630, 2331631, 2331632, 2331633, 2331634, 2331635, 2331638, 2331639, 2331641, 2331642, 2331643, 2331645, 2331648, 2331649, 2331650, 2331656, 2331658, 2331659, 2331660, 2331662, 2331663, 2331664, 2331666, 2331667, 2331668, 2331673, 2331675, 2331677, 2331679, 2331684, 2331688, 2331689, 2331690, 2331695, 2331697, 2331699, 2331700, 2331702, 2331704, 2331705, 2331706, 2331707, 2331710, 2331711, 2331715, 2331718, 2331719, 2331722, 2331724, 2331726, 2331727, 2331732, 2331733, 2331734, 2331735, 2331736, 2331739, 2331740, 2331741, 2331742, 2331746, 2331747, 2331748, 2331749, 2331751, 2331752, 2331753, 2331756, 2331757, 2331758, 2331759, 2331760, 2331761, 2331762, 2331764, 2331765, 2331766, 2331767, 2331768, 2331769, 2331771, 2331773, 2331774, 2331775, 2331776, 2331778, 2331780, 2331781, 2331782, 2331784, 2331786, 2331789, 2331790, 2331793, 2331794, 2331795, 2331797, 2331800, 2331802, 2331803, 2331804, 2331807, 2331808, 2331811, 2331814, 2331815, 2331817, 2331818, 2331819, 2331820, 2331822, 2331823, 2331824, 2331825, 2331826, 2331827, 2331828, 2331829, 2331831, 2331832, 2331833, 2331834, 2331835, 2331839, 2331840, 2331841, 2331842, 2331844, 2331845, 2331847, 2331848, 2331849, 2331851, 2331852, 2331854, 2331856, 2331857, 2331858, 2331859, 2331861, 2331863, 2331864, 2331865, 2331866, 2331868, 2331872, 2331875, 2331876, 2331877, 2331878, 2331879, 2331880, 2331882, 2331883, 2331884, 2331885, 2331887, 2331888, 2331890, 2331891, 2331894, 2331897, 2331898, 2331899, 2331900, 2331903, 2331905, 2331907, 2331908, 2331909, 2331911, 2331914, 2331915, 2331916, 2331917, 2331918, 2331923, 2331924, 2331926, 2331927, 2331930, 2331932, 2331935, 2331937, 2331939, 2331940, 2331941, 2331942, 2331946, 2331949, 2331950, 2331951, 2331954, 2331956, 2331957, 2331959, 2331964, 2331965, 2331966, 2331968, 2331969, 2331971, 2331973, 2331974, 2331979, 2331980, 2331982, 2331984, 2331985, 2331986, 2331990, 2331991, 2331994, 2331995, 2331996, 2331998, 2331999, 2332001, 2332002, 2332004, 2332006, 2332007, 2332012, 2332017, 2332018, 2332020, 2332021, 2332023, 2332025, 2332029, 2332030, 2332033, 2332034, 2332037, 2332038, 2332042, 2332043, 2332044, 2332045, 2332048, 2332051, 2332052, 2332054, 2332057, 2332058, 2332059, 2332060, 2332062, 2332064, 2332065, 2332066, 2332068, 2332071, 2332074, 2332075, 2332076, 2332077, 2332078, 2332079, 2332081, 2332082, 2332085, 2332087, 2332088, 2332092, 2332093, 2332099, 2332101, 2332105, 2332107, 2332108, 2332109, 2332111, 2332113, 2332117, 2332118, 2332119, 2332122, 2332124, 2332125, 2332126, 2332127, 2332128, 2332129, 2332130, 2332131, 2332132, 2332135, 2332136, 2332138, 2332139, 2332140, 2332145, 2332146, 2332147, 2332148, 2332149, 2332150, 2332155, 2332159, 2332161, 2332163, 2332166, 2332167, 2332173, 2332174, 2332176, 2332177, 2332179, 2332182, 2332183, 2332184, 2332185, 2332186, 2332187, 2332188, 2332189, 2332190, 2332192, 2332195, 2332196, 2332197, 2332198, 2332199, 2332202, 2332203, 2332204, 2332205, 2332206, 2332208, 2332211, 2332212, 2332218, 2332219, 2332221, 2332223, 2332226, 2332227, 2332232, 2332236, 2332237, 2332238, 2332242, 2332243, 2332244, 2332245, 2332246, 2332247, 2332248, 2332251, 2332252, 2332253, 2332254, 2332255, 2332258, 2332260, 2332262, 2332263, 2332264, 2332266, 2332267, 2332268, 2332269, 2332270, 2332271, 2332272, 2332273, 2332275, 2332277, 2332278, 2332279, 2332281, 2332282, 2332283, 2332284, 2332285, 2332286, 2332288, 2332289, 2332291, 2332292, 2332293, 2332294, 2332295, 2332296, 2332297, 2332299, 2332301, 2332303, 2332304, 2332305, 2332306, 2332308, 2332309, 2332312, 2332319, 2332320, 2332323, 2332324, 2332325, 2332326, 2332327, 2332328, 2332329, 2332331, 2332332, 2332333, 2332335, 2332339, 2332341, 2332345, 2332350, 2332351, 2332352, 2332355, 2332356, 2332357, 2332360, 2332362, 2332363, 2332364, 2332365, 2332366, 2332367, 2332368, 2332372, 2332373, 2332375, 2332377, 2332379, 2332380, 2332382, 2332385, 2332388, 2332389, 2332396, 2332398, 2332399, 2332401, 2332402, 2332403, 2332404, 2332406, 2332407, 2332408, 2332410, 2332411, 2332413, 2332414, 2332417, 2332420, 2332422, 2332423, 2332424, 2332427, 2332428, 2332430, 2332431, 2332433, 2332434, 2332436, 2332437, 2332438, 2332440, 2332441, 2332442, 2332443, 2332446, 2332448, 2332449, 2332451, 2332452, 2332453, 2332457, 2332459, 2332460, 2332461, 2332462, 2332463, 2332464, 2332465, 2332466, 2332467, 2332469, 2332470, 2332471, 2332472, 2332478, 2332479, 2332480, 2332481, 2332484, 2332485, 2332489, 2332490, 2332491, 2332492, 2332495, 2332498, 2332501, 2332502, 2332506, 2332507, 2332509, 2332510, 2332511, 2332512, 2332513, 2332516, 2332517, 2332518, 2332521, 2332524, 2332525, 2332526, 2332530, 2332532, 2332534, 2332536, 2332538, 2332539, 2332540, 2332542, 2332544, 2332545, 2332547, 2332549, 2332550, 2332551, 2332552, 2332553, 2332554, 2332556, 2332557, 2332560, 2332561, 2332567, 2332568, 2332571, 2332574, 2332576, 2332577, 2332578, 2332580, 2332585, 2332586, 2332587, 2332588, 2332589, 2332593, 2332594, 2332595, 2332596, 2332597, 2332598, 2332599, 2332603, 2332604, 2332605, 2332606, 2332607, 2332609, 2332611, 2332612, 2332613, 2332614, 2332615, 2332616, 2332618, 2332619, 2332621, 2332622, 2332624, 2332625, 2332627, 2332629, 2332631, 2332632, 2332634, 2332635, 2332637, 2332639, 2332641, 2332644, 2332647, 2332653, 2332654, 2332655, 2332658, 2332659, 2332660, 2332661, 2332668, 2332670, 2332671, 2332672, 2332673, 2332674, 2332675, 2332678, 2332679, 2332682, 2332683, 2332684, 2332685, 2332691, 2332693, 2332695, 2332696, 2332697, 2332698, 2332701, 2332704, 2332705, 2332707, 2332709, 2332712, 2332713, 2332714, 2332715, 2332717, 2332718, 2332719, 2332720, 2332721, 2332723, 2332725, 2332726, 2332727, 2332729, 2332730, 2332731, 2332732, 2332733, 2332734, 2332736, 2332738, 2332739, 2332741, 2332743, 2332744, 2332748, 2332749, 2332752, 2332757, 2332758, 2332759, 2332760, 2332761, 2332762, 2332763, 2332764, 2332767, 2332768, 2332770, 2332772, 2332773, 2332775, 2332779, 2332780, 2332783, 2332784, 2332787, 2332789, 2332790, 2332791, 2332792, 2332793, 2332797, 2332798, 2332801, 2332802, 2332803, 2332804, 2332807, 2332808, 2332809, 2332810, 2332811, 2332816, 2332818, 2332820, 2332823, 2332825, 2332826, 2332827, 2332828, 2332829, 2332830, 2332831, 2332832, 2332833, 2332834, 2332836, 2332837, 2332839, 2332842, 2332844, 2332846, 2332848, 2332852, 2332853, 2332854, 2332855, 2332857, 2332860, 2332861, 2332862, 2332863, 2332864, 2332865, 2332869, 2332870, 2332871, 2332872, 2332874, 2332875, 2332876, 2332877, 2332878, 2332881, 2332882, 2332883, 2332884, 2332885, 2332886, 2332887, 2332888, 2332890, 2332892, 2332893, 2332894, 2332896, 2332897, 2332898, 2332899, 2332900, 2332901, 2332903, 2332906, 2332908, 2332910, 2332911, 2332913, 2332914, 2332915, 2332916, 2332917, 2332918, 2332919, 2332920, 2332921, 2332922, 2332923, 2332924, 2332926, 2332927, 2332928, 2332929, 2332930, 2332934, 2332935, 2332936, 2332937, 2332943, 2332945, 2332947, 2332948, 2332949, 2332950, 2332952, 2332953, 2332954, 2332956, 2332958, 2332960, 2332961, 2332962, 2332964, 2332965, 2332967, 2332968, 2332970, 2332971, 2332972, 2332974, 2332980, 2332981, 2332984, 2332985, 2332986, 2332987, 2332989, 2332990, 2332993, 2332995, 2332997, 2332998, 2332999, 2333002, 2333003, 2333005, 2333007, 2333010, 2333012, 2333016, 2333017, 2333018, 2333019, 2333020, 2333021, 2333022, 2333024, 2333028, 2333029, 2333033, 2333036, 2333037, 2333038, 2333040, 2333042, 2333044, 2333045, 2333046, 2333048, 2333049, 2333050, 2333051, 2333053, 2333054, 2333055, 2333056, 2333057, 2333061, 2333062, 2333063, 2333064, 2333065, 2333067, 2333070, 2333071, 2333073, 2333075, 2333076, 2333077, 2333078, 2333082, 2333083, 2333084, 2333085, 2333086, 2333091, 2333092, 2333093, 2333094, 2333095, 2333096, 2333102, 2333103, 2333105, 2333106, 2333110, 2333111, 2333112, 2333115, 2333116, 2333117, 2333119, 2333120, 2333122, 2333124, 2333125, 2333126, 2333127, 2333128, 2333129, 2333132, 2333138, 2333142, 2333143, 2333147, 2333149, 2333151, 2333152, 2333153, 2333154, 2333155, 2333157, 2333158, 2333161, 2333162, 2333163, 2333164, 2333165, 2333167, 2333168, 2333169, 2333172, 2333174, 2333176, 2333177, 2333178, 2333179, 2333180, 2333185, 2333186, 2333187, 2333188, 2333189, 2333192, 2333195, 2333196, 2333197, 2333203, 2333204, 2333205, 2333206, 2333208, 2333209, 2333211, 2333212, 2333213, 2333215, 2333216, 2333217, 2333218, 2333221, 2333224, 2333225, 2333226, 2333227, 2333228, 2333229, 2333231, 2333232, 2333233, 2333236, 2333237, 2333240, 2333241, 2333242, 2333243, 2333245, 2333247, 2333248, 2333249, 2333250, 2333253, 2333255, 2333256, 2333261, 2333263, 2333264, 2333266, 2333270, 2333271, 2333272, 2333273, 2333275, 2333276, 2333279, 2333281, 2333282, 2333283, 2333288, 2333290, 2333293, 2333294, 2333295, 2333296, 2333299, 2333300, 2333301, 2333302, 2333306, 2333307, 2333308, 2333309, 2333310, 2333311, 2333312, 2333317, 2333318, 2333323, 2333324, 2333328, 2333329, 2333334, 2333335, 2333338, 2333339, 2333343, 2333346, 2333350, 2333351, 2333352, 2333353, 2333355, 2333356, 2333357, 2333358, 2333359, 2333360, 2333361, 2333363, 2333364, 2333365, 2333366, 2333368, 2333371, 2333373, 2333374, 2333375, 2333376, 2333378, 2333379, 2333380, 2333382, 2333383, 2333386, 2333387, 2333388, 2333389, 2333390, 2333391, 2333395, 2333396, 2333397, 2333398, 2333401, 2333403, 2333411, 2333412, 2333414, 2333416, 2333417, 2333420, 2333421, 2333422, 2333423, 2333425, 2333427, 2333428, 2333430, 2333431, 2333432, 2333435, 2333436, 2333442, 2333443, 2333445, 2333447, 2333448, 2333450, 2333451, 2333452, 2333456, 2333459, 2333460, 2333464, 2333465, 2333466, 2333469, 2333474, 2333475, 2333476, 2333478, 2333479, 2333483, 2333484, 2333485, 2333486, 2333489, 2333490, 2333491, 2333493, 2333495, 2333498, 2333500, 2333501, 2333502, 2333506, 2333507, 2333508, 2333511, 2333512, 2333514, 2333518, 2333520, 2333521, 2333522, 2333524, 2333527, 2333528, 2333529, 2333531, 2333532, 2333533, 2333537, 2333540, 2333546, 2333547, 2333548, 2333549, 2333550, 2333551, 2333553, 2333554, 2333558, 2333559, 2333560, 2333561, 2333562, 2333563, 2333566, 2333567, 2333569, 2333570, 2333573, 2333574, 2333577, 2333579, 2333580, 2333582, 2333583, 2333584, 2333585, 2333587, 2333588, 2333591, 2333592, 2333593, 2333594, 2333595, 2333597, 2333604, 2333605, 2333606, 2333607, 2333608, 2333609, 2333612, 2333614, 2333615, 2333617, 2333618, 2333619, 2333620, 2333622, 2333623, 2333624, 2333625, 2333627, 2333630, 2333632, 2333633, 2333634, 2333635, 2333641, 2333644, 2333646, 2333647, 2333648, 2333653, 2333655, 2333656, 2333659, 2333660, 2333661, 2333664, 2333665, 2333671, 2333672, 2333673, 2333675, 2333676, 2333678, 2333679, 2333680, 2333681, 2333684, 2333685, 2333686, 2333688, 2333690, 2333691, 2333693, 2333695, 2333696, 2333699, 2333700, 2333701, 2333702, 2333703, 2333704, 2333707, 2333710, 2333712, 2333713, 2333714, 2333715, 2333716, 2333717, 2333718, 2333719, 2333721, 2333722, 2333727, 2333728, 2333731, 2333733, 2333734, 2333736, 2333740, 2333741, 2333742, 2333743, 2333744, 2333745, 2333746, 2333747, 2333749, 2333751, 2333752, 2333753, 2333754, 2333755, 2333756, 2333757, 2333758, 2333760, 2333761, 2333764, 2333766, 2333771, 2333772, 2333775, 2333776, 2333777, 2333778, 2333779, 2333781, 2333782, 2333785, 2333786, 2333787, 2333789, 2333791, 2333793, 2333794, 2333795, 2333797, 2333799, 2333800, 2333801, 2333803, 2333805, 2333806, 2333807, 2333808, 2333809, 2333811, 2333812, 2333813, 2333814, 2333816, 2333818, 2333819, 2333820, 2333821, 2333823, 2333824, 2333825, 2333826, 2333829, 2333830, 2333832, 2333833, 2333835, 2333836, 2333837, 2333838, 2333839, 2333841, 2333844, 2333845, 2333848, 2333850, 2333851, 2333852, 2333853, 2333854, 2333855, 2333859, 2333860, 2333861, 2333862, 2333864, 2333865, 2333867, 2333872, 2333874, 2333875, 2333876, 2333877, 2333878, 2333879, 2333880, 2333881, 2333882, 2333883, 2333884, 2333885, 2333887, 2333888, 2333889, 2333891, 2333894, 2333898, 2333899, 2333900, 2333901, 2333902, 2333906, 2333907, 2333908, 2333910, 2333911, 2333912, 2333913, 2333914, 2333915, 2333916, 2333918, 2333920, 2333922, 2333923, 2333924, 2333925, 2333927, 2333928, 2333933, 2333935, 2333939, 2333940, 2333941, 2333942, 2333944, 2333946, 2333949, 2333950, 2333952, 2333955, 2333956, 2333957, 2333958, 2333960, 2333962, 2333964, 2333966, 2333971, 2333972, 2333973, 2333975, 2333977, 2333981, 2333982, 2333983, 2333984, 2333986, 2333989, 2333990, 2333991, 2333993, 2333997, 2333999, 2334000, 2334001, 2334002, 2334005, 2334006, 2334008, 2334010, 2334011, 2334012, 2334015, 2334017, 2334019, 2334023, 2334024, 2334028, 2334029, 2334031, 2334033, 2334036, 2334038, 2334039, 2334042, 2334043, 2334045, 2334046, 2334047, 2334048, 2334049, 2334050, 2334051, 2334054, 2334055, 2334058, 2334059, 2334060, 2334061, 2334062, 2334063, 2334064, 2334066, 2334068, 2334070, 2334072, 2334074, 2334076, 2334078, 2334080, 2334081, 2334083, 2334087, 2334089, 2334090, 2334091, 2334092, 2334093, 2334095, 2334096, 2334098, 2334100, 2334101, 2334103, 2334104, 2334106, 2334107, 2334108, 2334113, 2334115, 2334116, 2334117, 2334118, 2334119, 2334120, 2334121, 2334122, 2334123, 2334124, 2334125, 2334126, 2334129, 2334130, 2334135, 2334138, 2334140, 2334142, 2334143, 2334144, 2334146, 2334147, 2334148, 2334149, 2334150, 2334152, 2334158, 2334159, 2334162, 2334165, 2334166, 2334167, 2334168, 2334169, 2334170, 2334171, 2334172, 2334173, 2334177, 2334180, 2334184, 2334185, 2334186, 2334187, 2334188, 2334189, 2334190, 2334191, 2334192, 2334193, 2334196, 2334201, 2334203, 2334206, 2334207, 2334208, 2334213, 2334215, 2334217, 2334218, 2334219, 2334220, 2334221, 2334222, 2334223, 2334224, 2334225, 2334226, 2334231, 2334236, 2334237, 2334238, 2334243, 2334245, 2334246, 2334248, 2334249, 2334254, 2334256, 2334257, 2334262, 2334263, 2334266, 2334267, 2334268, 2334269, 2334270, 2334272, 2334274, 2334275, 2334278, 2334280, 2334281, 2334284, 2334285, 2334288, 2334291, 2334293, 2334295, 2334298, 2334300, 2334302, 2334303, 2334307, 2334309, 2334310, 2334311, 2334312, 2334313, 2334314, 2334315, 2334316, 2334317, 2334318, 2334319, 2334320, 2334324, 2334326, 2334327, 2334328, 2334330, 2334331, 2334333, 2334334, 2334335, 2334336, 2334337, 2334340, 2334341, 2334342, 2334343, 2334345, 2334348, 2334349, 2334351, 2334352, 2334354, 2334356, 2334360, 2334361, 2334362, 2334363, 2334364, 2334365, 2334367, 2334369, 2334371, 2334372, 2334373, 2334375, 2334377, 2334378, 2334382, 2334383, 2334384, 2334387, 2334390, 2334391, 2334392, 2334393, 2334395, 2334396, 2334398, 2334399, 2334401, 2334403, 2334404, 2334405, 2334406, 2334410, 2334416, 2334417, 2334419, 2334422, 2334425, 2334426, 2334429, 2334431, 2334434, 2334435, 2334436, 2334438, 2334439, 2334440, 2334441, 2334444, 2334447, 2334449, 2334451, 2334452, 2334453, 2334454, 2334455, 2334458, 2334460, 2334461, 2334463, 2334464, 2334467, 2334469, 2334470, 2334471, 2334472, 2334473, 2334474, 2334478, 2334479, 2334483, 2334484, 2334488, 2334489, 2334490, 2334491, 2334492, 2334501, 2334502, 2334505, 2334506, 2334507, 2334508, 2334509, 2334511, 2334513, 2334514, 2334517, 2334518, 2334520, 2334521, 2334522, 2334523, 2334524, 2334525, 2334526, 2334528, 2334529, 2334530, 2334531, 2334532, 2334534, 2334535, 2334536, 2334541, 2334542, 2334543, 2334544, 2334548, 2334550, 2334553, 2334554, 2334555, 2334557, 2334558, 2334559, 2334564, 2334565, 2334568, 2334571, 2334572, 2334574, 2334576, 2334578, 2334579, 2334582, 2334586, 2334589, 2334591, 2334592, 2334593, 2334597, 2334598, 2334600, 2334602, 2334603, 2334604, 2334605, 2334606, 2334608, 2334611, 2334613, 2334614, 2334615, 2334616, 2334617, 2334619, 2334621, 2334623, 2334624, 2334625, 2334628, 2334629, 2334631, 2334632, 2334634, 2334636, 2334638, 2334641, 2334642, 2334645, 2334646, 2334647, 2334650, 2334651, 2334653, 2334654, 2334655, 2334656, 2334657, 2334658, 2334659, 2334661, 2334663, 2334664, 2334666, 2334667, 2334668, 2334669, 2334670, 2334672, 2334675, 2334677, 2334678, 2334681, 2334685, 2334690, 2334691, 2334692, 2334698, 2334699, 2334700, 2334701, 2334702, 2334709, 2334714, 2334715, 2334717, 2334720, 2334721, 2334723, 2334724, 2334727, 2334728, 2334729, 2334730, 2334731, 2334733, 2334735, 2334737, 2334739, 2334740, 2334741, 2334742, 2334744, 2334746, 2334747, 2334748, 2334751, 2334752, 2334753, 2334755, 2334757, 2334759, 2334760, 2334761, 2334762, 2334763, 2334764, 2334765, 2334768, 2334770, 2334771, 2334773, 2334774, 2334775, 2334778, 2334780, 2334781, 2334782, 2334783, 2334784, 2334785, 2334786, 2334787, 2334792, 2334796, 2334797, 2334798, 2334800, 2334804, 2334805, 2334806, 2334807, 2334808, 2334809, 2334813, 2334815, 2334816, 2334817, 2334821, 2334822, 2334823, 2334824, 2334825, 2334827, 2334829, 2334831, 2334834, 2334836, 2334837, 2334838, 2334839, 2334842, 2334846, 2334847, 2334848, 2334849, 2334850, 2334851, 2334853, 2334854, 2334856, 2334858, 2334860, 2334861, 2334863, 2334864, 2334869, 2334870, 2334871, 2334872, 2334873, 2334874, 2334875, 2334876, 2334880, 2334882, 2334884, 2334885, 2334887, 2334889, 2334891, 2334893, 2334895, 2334896, 2334898, 2334900, 2334901, 2334903, 2334904, 2334906, 2334907, 2334908, 2334911, 2334912, 2334913, 2334915, 2334918, 2334919, 2334920, 2334921, 2334923, 2334924, 2334927, 2334929, 2334930, 2334931, 2334932, 2334933, 2334936, 2334940, 2334941, 2334942, 2334943, 2334945, 2334946, 2334947, 2334949, 2334950, 2334951, 2334954, 2334956, 2334958, 2334960, 2334962, 2334963, 2334964, 2334965, 2334966, 2334968, 2334970, 2334971, 2334972, 2334973, 2334976, 2334977, 2334978, 2334979, 2334980, 2334981, 2334988, 2334989, 2334990, 2334992, 2334993, 2334994, 2334995, 2334998, 2335000, 2335003, 2335005, 2335006, 2335010, 2335011, 2335014, 2335016, 2335017, 2335018, 2335019, 2335020, 2335021, 2335023, 2335024, 2335026, 2335028, 2335034, 2335038, 2335039, 2335041, 2335043, 2335044, 2335045, 2335046, 2335049, 2335051, 2335052, 2335055, 2335057, 2335058, 2335059, 2335060, 2335061, 2335062, 2335066, 2335067, 2335068, 2335070, 2335072, 2335074, 2335077, 2335078, 2335079, 2335082, 2335083, 2335088, 2335089, 2335092, 2335094, 2335095, 2335096, 2335099, 2335100, 2335101, 2335102, 2335103, 2335104, 2335105, 2335106, 2335108, 2335109, 2335111, 2335119, 2335121, 2335124, 2335125, 2335126, 2335127, 2335131, 2335132, 2335136, 2335137, 2335138, 2335139, 2335143, 2335145, 2335146, 2335148, 2335150, 2335151, 2335152, 2335153, 2335154, 2335160, 2335161, 2335162, 2335163, 2335164, 2335166, 2335168, 2335170, 2335171, 2335176, 2335177, 2335179, 2335180, 2335182, 2335186, 2335188, 2335189, 2335191, 2335192, 2335193, 2335195, 2335196, 2335197, 2335198, 2335200, 2335201, 2335202, 2335204, 2335205, 2335207, 2335208, 2335210, 2335211, 2335212, 2335213, 2335214, 2335218, 2335220, 2335221, 2335223, 2335226, 2335227, 2335228, 2335229, 2335230, 2335231, 2335232, 2335234, 2335235, 2335237, 2335238, 2335239, 2335242, 2335245, 2335247, 2335250, 2335253, 2335254, 2335255, 2335256, 2335258, 2335259, 2335260, 2335262, 2335263, 2335264, 2335265, 2335267, 2335268, 2335270, 2335271, 2335275, 2335276, 2335278, 2335279, 2335280, 2335282, 2335284, 2335285, 2335286, 2335289, 2335290, 2335293, 2335296, 2335298, 2335300, 2335304, 2335307, 2335308, 2335310, 2335311, 2335312, 2335316, 2335317, 2335321, 2335324, 2335326, 2335327, 2335329, 2335332, 2335334, 2335335, 2335337, 2335338, 2335339, 2335345, 2335348, 2335349, 2335350, 2335351, 2335353, 2335354, 2335356, 2335359, 2335360, 2335361, 2335363, 2335365, 2335366, 2335367, 2335371, 2335373, 2335374, 2335375, 2335377, 2335378, 2335380, 2335381, 2335382, 2335383, 2335384, 2335385, 2335387, 2335388, 2335390, 2335391, 2335394, 2335398, 2335399, 2335400, 2335404, 2335405, 2335406, 2335408, 2335409, 2335410, 2335411, 2335412, 2335413, 2335415, 2335416, 2335419, 2335420, 2335421, 2335422, 2335424, 2335425, 2335427, 2335428, 2335429, 2335430, 2335431, 2335433, 2335437, 2335438, 2335442, 2335443, 2335444, 2335446, 2335450, 2335451, 2335453, 2335454, 2335455, 2335457, 2335458, 2335460, 2335462, 2335463, 2335464, 2335465, 2335466, 2335467, 2335468, 2335469, 2335470, 2335471, 2335475, 2335476, 2335477, 2335481, 2335483, 2335486, 2335487, 2335491, 2335492, 2335495, 2335496, 2335497, 2335498, 2335502, 2335504, 2335507, 2335508, 2335509, 2335513, 2335514, 2335517, 2335518, 2335519, 2335520, 2335523, 2335525, 2335526, 2335528, 2335529, 2335532, 2335537, 2335539, 2335540, 2335542, 2335545, 2335546, 2335548, 2335549, 2335551, 2335552, 2335553, 2335554, 2335556, 2335557, 2335558, 2335559, 2335561, 2335563, 2335566, 2335567, 2335568, 2335570, 2335573, 2335574, 2335575, 2335577, 2335578, 2335579, 2335582, 2335584, 2335585, 2335587, 2335588, 2335589, 2335590, 2335593, 2335595, 2335596, 2335599, 2335600, 2335603, 2335604, 2335605, 2335606, 2335608, 2335609, 2335613, 2335615, 2335616, 2335621, 2335622, 2335623, 2335624, 2335626, 2335627, 2335629, 2335630, 2335631, 2335633, 2335636, 2335637, 2335639, 2335640, 2335641, 2335642, 2335646, 2335647, 2335648, 2335649, 2335651, 2335652, 2335654, 2335655, 2335656, 2335658, 2335659, 2335662, 2335664, 2335668, 2335669, 2335670, 2335671, 2335672, 2335675, 2335676, 2335677, 2335679, 2335681, 2335685, 2335686, 2335687, 2335689, 2335693, 2335695, 2335698, 2335699, 2335700, 2335701, 2335702, 2335704, 2335705, 2335706, 2335708, 2335709, 2335710, 2335711, 2335712, 2335713, 2335714, 2335715, 2335716, 2335718, 2335719, 2335722, 2335723, 2335724, 2335726, 2335728, 2335729, 2335730, 2335734, 2335736, 2335737, 2335738, 2335739, 2335740, 2335743, 2335744, 2335745, 2335747, 2335748, 2335749, 2335750, 2335751, 2335753, 2335754, 2335755, 2335756, 2335757, 2335761, 2335762, 2335765, 2335766, 2335767, 2335769, 2335773, 2335774, 2335775, 2335776, 2335777, 2335779, 2335780, 2335782, 2335785, 2335786, 2335787, 2335788, 2335790, 2335791, 2335796, 2335797, 2335799, 2335800, 2335806, 2335808, 2335810, 2335811, 2335812, 2335813, 2335817, 2335819, 2335820, 2335821, 2335823, 2335824, 2335825, 2335826, 2335828, 2335831, 2335832, 2335833, 2335834, 2335835, 2335836, 2335838, 2335840, 2335842, 2335846, 2335848, 2335849, 2335850, 2335851, 2335853, 2335854, 2335855, 2335857, 2335859, 2335862, 2335865, 2335868, 2335869, 2335870, 2335871, 2335872, 2335873, 2335877, 2335879, 2335882, 2335883, 2335884, 2335885, 2335888, 2335889, 2335891, 2335895, 2335896, 2335897, 2335899, 2335900, 2335902, 2335903, 2335904, 2335905, 2335909, 2335911, 2335913, 2335923, 2335925, 2335926, 2335928, 2335929, 2335930, 2335934, 2335935, 2335939, 2335940, 2335941, 2335942, 2335946, 2335947, 2335948, 2335949, 2335950, 2335954, 2335955, 2335956, 2335957, 2335958, 2335959, 2335961, 2335962, 2335963, 2335964, 2335965, 2335971, 2335972, 2335973, 2335974, 2335976, 2335977, 2335979, 2335980, 2335981, 2335982, 2335983, 2335984, 2335985, 2335986, 2335987, 2335989, 2335993, 2335999, 2336000, 2336001, 2336002, 2336004, 2336005, 2336006, 2336007, 2336009, 2336011, 2336012, 2336013, 2336014, 2336017, 2336020, 2336021, 2336022, 2336023, 2336025, 2336026, 2336027, 2336030, 2336031, 2336032, 2336034, 2336035, 2411494, 2411502, 2411503, 2411506, 2411514, 2411515, 2411518, 2411522, 2411523, 2411525, 2411526, 2411527, 2411528, 2411534, 2411535, 2411537, 2411538, 2411539, 2411542, 2411543, 2411544, 2411545, 2411549, 2411551, 2411552, 2411555, 2411556, 2411560, 2411561, 2411563, 2411565, 2411567, 2411571, 2411574, 2411575, 2411577, 2411581, 2411582, 2411584, 2411587, 2411588, 2411592, 2411593, 2411594, 2411598, 2411599, 2411600, 2411601, 2411605, 2411606, 2411608, 2411609, 2411610, 2411611, 2411612, 2411613, 2411616, 2411617, 2411619, 2411620, 2411625, 2411627, 2411628, 2411633, 2411641, 2411642, 2411644, 2411646, 2411647, 2411649, 2411652, 2411653, 2411655, 2411659, 2411661, 2411666, 2411669, 2411677, 2411680, 2411681, 2411682, 2411686, 2411687, 2411691, 2411692, 2411693, 2411694, 2411695, 2411696, 2411697, 2411698, 2411699, 2411700, 2411701, 2411702, 2411703, 2411704, 2411709, 2411710, 2411712, 2411713, 2411717, 2411722, 2411723, 2411731, 2411732, 2411733, 2411734, 2411739, 2411740, 2411743, 2411744, 2411746, 2411747, 2411748, 2411750, 2411756, 2411758, 2411760, 2411761, 2411762, 2411764, 2411765, 2411766, 2411767, 2411770, 2411774, 2411779, 2411783, 2411784, 2411786, 2411790, 2411792, 2411793, 2411794, 2411795, 2411796, 2411798, 2411799, 2411800, 2411801, 2411802, 2411811, 2411812, 2411817, 2411821, 2411822, 2411823, 2411824, 2411828, 2411829, 2411830, 2411837, 2411840, 2411841, 2411842, 2411850, 2411852, 2411853, 2411854, 2411855, 2411856, 2411857, 2411858, 2411869, 2411871, 2411872, 2411873, 2411881, 2411883, 2411887, 2411889, 2411891, 2411893, 2411894, 2411895, 2411896, 2411898, 2411899, 2411900, 2411902, 2411903, 2411905, 2411909, 2411910, 2411912, 2411913, 2411915, 2411916, 2411918, 2411919, 2411921, 2411922, 2411923, 2411926, 2411927, 2411928, 2411931, 2411933, 2411934, 2411936, 2411937, 2411938, 2411940, 2411942, 2411947, 2411952, 2411953, 2411954, 2411957, 2411958, 2411959, 2411960, 2411961, 2411962, 2411963, 2411964, 2411965, 2411966, 2411967, 2411968, 2411980, 2411981, 2411983, 2411984, 2411985, 2411986, 2411987, 2411988, 2411990, 2411995, 2411997, 2411998, 2412002, 2412003, 2412004, 2412006, 2412009, 2412012, 2412013, 2412014, 2412025, 2412026, 2412027, 2412028, 2412035, 2412036, 2412037, 2412038, 2412039, 2412045, 2412046, 2412050, 2412052, 2412062, 2412063, 2412065, 2412066, 2412075, 2412078, 2412082, 2412083, 2412088, 2412090, 2412092, 2412093, 2412094, 2412095, 2412096, 2412104, 2412105, 2412107, 2412109, 2412110, 2412111, 2412114, 2412120, 2412121, 2412124, 2412125, 2412127, 2412128, 2412134, 2412140, 2412141, 2412142, 2412143, 2412147, 2412148, 2412157, 2412158, 2412159, 2412160, 2412161, 2412162, 2412164, 2412165, 2412166, 2412169, 2412172, 2412176, 2412178, 2412188, 2412190, 2412193, 2412195, 2412196, 2412197, 2412198, 2412199, 2412203, 2412204, 2412215, 2412216, 2412221, 2412222, 2412224, 2412226, 2412229, 2412231, 2412233, 2412245, 2412246, 2412248, 2412258, 2412259, 2412261, 2412265, 2412268, 2412269, 2412271, 2412272, 2412273, 2412277, 2412278, 2412279, 2412280, 2412281, 2412282, 2412283, 2412285, 2412286, 2412287, 2412288, 2412290, 2412291, 2412292, 2412294, 2412296, 2412297, 2412299, 2412300, 2412302, 2412307, 2412308, 2412309, 2412310, 2412311, 2412312, 2412313, 2412318, 2412321, 2412323, 2412324, 2412325, 2412326, 2412336, 2412337, 2412342, 2412343, 2412353, 2412360, 2412362, 2412365, 2412366, 2412369, 2412371, 2412372, 2412375, 2412376, 2412382, 2412383, 2412387, 2412388, 2412398, 2412402, 2412404, 2412405, 2412406, 2412408, 2412409, 2412410, 2412411, 2412413, 2412414, 2412415, 2412416, 2412417, 2412418, 2412419, 2412422, 2412423, 2412425, 2412429, 2412430, 2412431, 2412436, 2412440, 2412441, 2412446, 2412448, 2412450, 2412453, 2412455, 2412457, 2412458, 2412462, 2412463, 2412464, 2412465, 2412466, 2412467, 2412468, 2412469, 2412470, 2412471, 2412476, 2412477, 2412478, 2412479, 2412480, 2412481, 2412482, 2412483, 2412487, 2412494, 2412495, 2412496, 2412497, 2412498, 2412499, 2412500, 2412502, 2412503, 2412505, 2412518, 2412525, 2412527, 2412528, 2412530, 2412538, 2412542, 2412548, 2412553, 2412556, 2412559, 2412561, 2412567, 2412568, 2412569, 2412570, 2412571, 2412572, 2412573, 2412574, 2412578, 2412580, 2412581, 2412588, 2412590, 2412591, 2412592, 2412596, 2412597, 2412598, 2412599, 2412601, 2412603, 2412605, 2412609, 2412610, 2412612, 2412614, 2412622, 2412624, 2412626, 2412627, 2412628, 2412629, 2412630, 2412631, 2412633, 2412639, 2412641, 2412643, 2412644, 2412645, 2412646, 2412647, 2412650, 2412652, 2412653, 2412657, 2412658, 2412664, 2412665, 2412666, 2412668, 2412669, 2412671, 2412674, 2412679, 2412681, 2412682, 2412683, 2412685, 2412686, 2412690, 2412692, 2412693, 2412697, 2412698, 2412701, 2412708, 2412715, 2412716, 2412717, 2412719, 2412721, 2412722, 2412723, 2412724, 2412725, 2412727, 2412729, 2412730, 2412734, 2412736, 2412737, 2412738, 2412739, 2412740, 2412741, 2412742, 2412745, 2412746, 2412747, 2412748, 2412752, 2412753, 2412754, 2412757, 2412759, 2412764, 2412768, 2412770, 2412771, 2412772, 2412777, 2412779, 2412780, 2412781, 2412783, 2412784, 2412791, 2412792, 2412793, 2412797, 2412801, 2412803, 2412804, 2412805, 2412806, 2412808, 2412809, 2412810, 2412811, 2412812, 2412818, 2412822, 2412826, 2412829, 2412833, 2412842, 2412845, 2412848, 2412849, 2412850, 2412851, 2412853, 2412854, 2412856, 2412857, 2412858, 2412859, 2412860, 2412862, 2412866, 2412870, 2412871, 2412872, 2412876, 2412877, 2412882, 2412884, 2412888, 2412889, 2412890, 2412891, 2412892, 2412895, 2412896, 2412897, 2412898, 2412902, 2412906, 2412907, 2412914, 2412915, 2412925, 2412930, 2412931, 2412936, 2412940, 2412941, 2412943, 2412946, 2412947, 2412949, 2412950, 2412953, 2412956, 2412958, 2412960, 2412961, 2412963, 2412965, 2412966, 2412969, 2412970, 2412972, 2412975, 2412981, 2412982, 2412983, 2412991, 2412992, 2412994, 2412996, 2412997, 2413009, 2413011, 2413012, 2413013, 2413016, 2413017, 2413019, 2413020, 2413021, 2413028, 2413031, 2413032, 2413033, 2413034, 2413036, 2413037, 2413039, 2413042, 2413045, 2413046, 2413047, 2413048, 2413049, 2413050, 2413051, 2413054, 2413055, 2413058, 2413059, 2413063, 2413064, 2413066, 2413067, 2413073, 2413074, 2413075, 2413076, 2413078, 2413079, 2413084, 2413087, 2413088, 2413089, 2413090, 2413094, 2413100, 2413101, 2413103, 2413104, 2413105, 2413106, 2413110, 2413111, 2413116, 2413117, 2413123, 2413127, 2413128, 2413129, 2413130, 2413135, 2413137, 2413140, 2413143, 2413144, 2413147, 2413148, 2413149, 2413150, 2413151, 2413152, 2413153, 2413155, 2413156, 2413157, 2413159, 2413160, 2413161, 2413162, 2413163, 2413164, 2413166, 2413167, 2413177, 2413181, 2413182, 2413184, 2413185, 2413186, 2413187, 2413189, 2413191, 2413197, 2413198, 2413201, 2413203, 2413205, 2413207, 2413212, 2413213, 2413214, 2413215, 2413216, 2413217, 2413219, 2413222, 2413224, 2413229, 2413230, 2413233, 2413234, 2413237, 2413238, 2413239, 2413240, 2413247, 2413248, 2413249, 2413250, 2413252, 2413253, 2413259, 2413260, 2413261, 2413262, 2413265, 2413266, 2413267, 2413269, 2413271, 2413273, 2413274, 2413275, 2413277, 2413278, 2413279, 2413280, 2413283, 2413287, 2413288, 2413291, 2413292, 2413294, 2413296, 2413302, 2413303, 2413309, 2413311, 2413319, 2413320, 2413321, 2413322, 2413323, 2413324, 2413325, 2413326, 2413327, 2413328, 2413329, 2413330, 2413332, 2413335, 2413336, 2413337, 2413340, 2413341, 2413342, 2413343, 2413344, 2413345, 2413346, 2413347, 2413349, 2413350, 2413351, 2413353, 2413358, 2413359, 2413360, 2413367, 2413368, 2413369, 2413371, 2413372, 2413382, 2413386, 2413387, 2413393, 2413394, 2413395, 2413397, 2413398, 2413401, 2413402, 2413408, 2413409, 2413410, 2413412, 2413415, 2413417, 2413427, 2413428, 2413429, 2413430, 2413433, 2413434, 2413436, 2413437, 2413438, 2413439, 2413440, 2413441, 2413442, 2413446, 2413447, 2413448, 2413449, 2413451, 2413457, 2413458, 2413459, 2413460, 2413465, 2413468, 2413469, 2413474, 2413478, 2413479, 2413484, 2413486, 2413487, 2413488, 2413490, 2413492, 2413493, 2413497, 2413499, 2413508, 2413509, 2413510, 2413514, 2413515, 2413518, 2413527, 2413528, 2413529, 2413530, 2413531, 2413533, 2413534, 2413536, 2413538, 2413539, 2413540, 2413544, 2413548, 2413550, 2413555, 2413558, 2413559, 2413560, 2413567, 2413569, 2413576, 2413577, 2413578, 2413579, 2413581, 2413585, 2413586, 2413587, 2413595, 2413597, 2413598, 2413599, 2413600, 2413601, 2413602, 2413603, 2413605, 2413607, 2413609, 2413610, 2413621, 2413622, 2413623, 2413624, 2413626, 2413628, 2413631, 2413634, 2413644, 2413647, 2413648, 2413649, 2413650, 2413653, 2413662, 2413663, 2413664, 2413666, 2413667, 2413668, 2413669, 2413670, 2413676, 2413677, 2413678, 2413679, 2413686, 2413691, 2413693, 2413694, 2413695, 2413697, 2413701, 2413712, 2413713, 2413715, 2413716, 2413719, 2413721, 2413722, 2413723, 2413724, 2413725, 2413730, 2413731, 2413732, 2413733, 2413736, 2413741, 2413742, 2413745, 2413746, 2413747, 2413748, 2413749, 2413755, 2413756, 2413760, 2413761, 2413762, 2413763, 2413764, 2413765, 2413767, 2413773, 2413779, 2413780, 2413781, 2413782, 2413783, 2413784, 2413786, 2413788, 2413793, 2413794, 2413795, 2413799, 2413800, 2413804, 2413806, 2413807, 2413812, 2413813, 2413814, 2413817, 2413822, 2413823, 2413824, 2413826, 2413829, 2413830, 2413831, 2413832, 2413833, 2413836, 2413837, 2413838, 2413839, 2413841, 2413843, 2413844, 2413847, 2413859, 2413861, 2413862, 2413863, 2413864, 2413872, 2413873, 2413874, 2413875, 2413876, 2413880, 2413881, 2413882, 2413885, 2413886, 2413887, 2413889, 2413892, 2413893, 2413894, 2413895, 2413896, 2413898, 2413903, 2413905, 2413911, 2413916, 2413918, 2413919, 2413921, 2413922, 2413923, 2413924, 2413926, 2413927, 2413928, 2413930, 2413931, 2413932, 2413933, 2413935, 2413936, 2413938, 2413939, 2413940, 2413942, 2413951, 2413955, 2413956, 2413957, 2413958, 2413959, 2413960, 2413966, 2413967, 2413968, 2413970, 2413974, 2413978, 2413982, 2413988, 2413990, 2413991, 2413992, 2413993, 2413994, 2413996, 2413997, 2413998, 2414003, 2414004, 2414009, 2414010, 2414011, 2414012, 2414013, 2414021, 2414026, 2414031, 2414038, 2414045, 2414046, 2414047, 2414051, 2414058, 2414059, 2414060, 2414062, 2414065, 2414067, 2414068, 2414069, 2414070, 2414075, 2414077, 2414078, 2414079, 2414081, 2414082, 2414083, 2414085, 2414087, 2414091, 2414092, 2414093, 2414096, 2414098, 2414099, 2414106, 2414107, 2414108, 2414109, 2414110, 2414111, 2414113, 2414115, 2414116, 2414123, 2414124, 2414125, 2414127, 2414128, 2414129, 2414130, 2414134, 2414135, 2414139, 2414140, 2414141, 2414143, 2414144, 2414145, 2414147, 2414148, 2414150, 2414153, 2414154, 2414155, 2414166, 2414167, 2414170, 2414174, 2414177, 2414180, 2414181, 2414185, 2414186, 2414189, 2414190, 2414192, 2414193, 2414197, 2414199, 2414204, 2414206, 2414207, 2414209, 2414213, 2414218, 2414221, 2414223, 2414226, 2414227, 2414229, 2414231, 2414236, 2414237, 2414238, 2414239, 2414240, 2414241, 2414249, 2414255, 2414256, 2414257, 2414258, 2414259, 2414262, 2414263, 2414264, 2414265, 2414266, 2414268, 2414270, 2414271, 2414272, 2414273, 2414277, 2414279, 2414280, 2414286, 2414288, 2414295, 2414298, 2414300, 2414301, 2414302, 2414305, 2414307, 2414308, 2414309, 2414310, 2414311, 2414312, 2414313, 2414314, 2414316, 2414317, 2414318, 2414319, 2414320, 2414322, 2414325, 2414326, 2414327, 2414330, 2414337, 2414339, 2414341, 2414342, 2414348, 2414353, 2414354, 2414361, 2414362, 2414363, 2414364, 2414365, 2414366, 2414371, 2414372, 2414373, 2414374, 2414378, 2414380, 2414381, 2414383, 2414384, 2414385, 2414391, 2414395, 2414396, 2414399, 2414400, 2414401, 2414402, 2414406, 2414408, 2414409, 2414414, 2414416, 2414424, 2414432, 2414433, 2414435, 2414439, 2414444, 2414449, 2414450, 2414462, 2414463, 2414464, 2414466, 2414467, 2414468, 2414472, 2414473, 2414475, 2414476, 2414477, 2414481, 2414484, 2414486, 2414487, 2414489, 2414493, 2414494, 2414495, 2414496, 2414498, 2414500, 2414501, 2414502, 2414503, 2414505, 2414507, 2414509, 2414515, 2414518, 2414519, 2414520, 2414521, 2414522, 2414523, 2414526, 2414528, 2414531, 2414536, 2414538, 2414539, 2414540, 2414543, 2414547, 2414548, 2414551, 2414552, 2414554, 2414555, 2414556, 2414557, 2414558, 2414561, 2414562, 2414564, 2414567, 2414571, 2414577, 2414578, 2414580, 2414581, 2414582, 2414585, 2414586, 2414587, 2414588, 2414590, 2414592, 2414593, 2414597, 2414604, 2414606, 2414610, 2414612, 2414615, 2414620, 2414621, 2414622, 2414628, 2414633, 2414634, 2414635, 2414640, 2414643, 2414644, 2414645, 2414648, 2414650, 2414652, 2414658, 2414659, 2414661, 2414662, 2414663, 2414664, 2414665, 2414667, 2414673, 2414674, 2414675, 2414677, 2414679, 2414682, 2414683, 2414685, 2414688, 2414691, 2414694, 2414701, 2414702, 2414705, 2414707, 2414710, 2414714, 2414716, 2414718, 2414722, 2414723, 2414724, 2414729, 2414736, 2414737, 2414739, 2414740, 2414747, 2414749, 2414763, 2414764, 2414765, 2414766, 2414767, 2414769, 2414770, 2414771, 2414772, 2414775, 2414776, 2414779, 2414780, 2414781, 2414789, 2414790, 2414791, 2414792, 2414794, 2414799, 2414802, 2414806, 2414807, 2414810, 2414815, 2414816, 2414818, 2414819, 2414820, 2414821, 2414825, 2414826, 2414828, 2414829, 2414830, 2414835, 2414839, 2414840, 2414845, 2414847, 2414848, 2414849, 2414850, 2414851, 2414852, 2414856, 2414858, 2414859, 2414860, 2414861, 2414862, 2414866, 2414867, 2414868, 2414869, 2414871, 2414872, 2414874, 2414875, 2414876, 2414881, 2414882, 2414883, 2414884, 2414886, 2414887, 2414888, 2414889, 2414890, 2414891, 2414898, 2414900, 2414901, 2414902, 2414903, 2414910, 2414911, 2414912, 2414913, 2414914, 2414918, 2414920, 2414923, 2414925, 2414929, 2414935, 2414937, 2414941, 2414942, 2414943, 2414944, 2414945, 2414946, 2414952, 2414953, 2414954, 2414955, 2414956, 2414961, 2414966, 2414968, 2414971, 2414973, 2414976, 2414977, 2414979, 2414980, 2414985, 2414989, 2414991, 2414992, 2414993, 2414994, 2414996, 2414997, 2415004, 2415005, 2415006, 2415011, 2415021, 2415023, 2415025, 2415029, 2415031, 2415033, 2415034, 2415035, 2415041, 2415042, 2415043, 2415044, 2415048, 2415049, 2415050, 2415052, 2415055, 2415061, 2415063, 2415064, 2415066, 2415067, 2415068, 2415069, 2415070, 2415071, 2415072, 2415073, 2415189, 2415191, 2415192, 2415193, 2415194, 2415195, 2415196, 2415197, 2415198, 2415202, 2415203, 2415204, 2415206, 2415207, 2415208, 2415211, 2415215, 2415216, 2415217, 2415218, 2415219, 2415220, 2415221, 2415222, 2415223, 2415224, 2415227, 2415229, 2415230, 2415231, 2415232, 2415233, 2415234, 2415235, 2415236, 2415237, 2415239, 2415240, 2415241, 2415243, 2415244, 2415245, 2415246, 2415248, 2415249, 2415250, 2415251, 2415253, 2415254, 2415255, 2415256, 2415257, 2415258, 2415260, 2415263, 2415264, 2415267, 2415269, 2415270, 2415272, 2415273, 2415274, 2415275, 2415276, 2415277, 2415278, 2415280, 2415282, 2415283, 2415286, 2415288, 2415290, 2415298, 2415299, 2415300, 2415302, 2415305, 2415308, 2415310, 2415311, 2415314, 2415315, 2415316, 2415317, 2415320, 2415321, 2415324, 2415325, 2415326, 2415328, 2415329, 2415330, 2415332, 2415335, 2415338, 2415339, 2415340, 2415341, 2415342, 2415343, 2415346, 2415348, 2415349, 2415353, 2415354, 2415357, 2415358, 2415359, 2415362, 2415363, 2415366, 2415368, 2415371, 2415372, 2415373, 2415375, 2415378, 2415380, 2415384, 2415386, 2415387, 2415388, 2415389, 2415390, 2415393, 2415395, 2415399, 2415400, 2415401, 2415402, 2415406, 2415408, 2415409, 2415411, 2415412, 2415413, 2415414, 2415415, 2415417, 2415418, 2415419, 2415423, 2415424, 2415429, 2415430, 2415431, 2415433, 2415439, 2415440, 2415441, 2415442, 2415443, 2415445, 2415446, 2415447, 2415448, 2415449, 2415451, 2415452, 2415453, 2415454, 2415457, 2415458, 2415459, 2415460, 2415461, 2415462, 2415463, 2415465, 2415466, 2415467, 2415468, 2415469, 2415470, 2415471, 2415472, 2415474, 2415475, 2415476, 2415477, 2415479, 2415480, 2415481, 2415484, 2415486, 2415488, 2415489, 2415490, 2415491, 2415493, 2415494, 2415499, 2415500, 2415501, 2415502, 2415503, 2415505, 2415506, 2415507, 2415508, 2415512, 2415513, 2415514, 2415515, 2415517, 2415518, 2415521, 2415522, 2415523, 2415524, 2415527, 2415528, 2415529, 2415531, 2415534, 2415536, 2415537, 2415538, 2415539, 2415541, 2415542, 2415543, 2415545, 2415546, 2415549, 2415551, 2415552, 2415554, 2415559, 2415560, 2415564, 2415565, 2415566, 2415567, 2415569, 2415570, 2415571, 2415573, 2415575, 2415576, 2415581, 2415582, 2415585, 2415587, 2415588, 2415589, 2415590, 2415591, 2415594, 2415595, 2415596, 2415603, 2415604, 2415606, 2415607, 2415609, 2415610, 2415611, 2415612, 2415614, 2415615, 2415616, 2415617, 2415622, 2415623, 2415625, 2415626, 2415628, 2415631, 2415633, 2415635, 2415636, 2415637, 2415638, 2415639, 2415640, 2415642, 2415644, 2415645, 2415647, 2415648, 2415651, 2415653, 2415656, 2415657, 2415659, 2415660, 2415661, 2415662, 2415663, 2415664, 2415665, 2415666, 2415667, 2415669, 2415670, 2415671, 2415672, 2415673, 2415676, 2415677, 2415679, 2415680, 2415683, 2415684, 2415685, 2415686, 2415687, 2415688, 2415694, 2415695, 2415697, 2415698, 2415699, 2415700, 2415701, 2415706, 2415707, 2415708, 2415710, 2415711, 2415712, 2415713, 2415717, 2415718, 2415719, 2415721, 2415722, 2415723, 2415724, 2415725, 2415726, 2415730, 2415731, 2415735, 2415737, 2415739, 2415742, 2415747, 2415751, 2415753, 2415755, 2415756, 2415763, 2415765, 2415766, 2415768, 2415770, 2415772, 2415774, 2415775, 2415776, 2415777, 2415779, 2415780, 2415782, 2415783, 2415784, 2415785, 2415786, 2415787, 2415789, 2415791, 2415792, 2415794, 2415796, 2415797, 2415798, 2415799, 2415800, 2415801, 2415802, 2415804, 2415807, 2415808, 2415809, 2415813, 2415816, 2415817, 2415820, 2415822, 2415824, 2415826, 2415827, 2415828, 2415829, 2415830, 2415833, 2415836, 2415837, 2415840, 2415843, 2415844, 2415845, 2415846, 2415847, 2415848, 2415852, 2415855, 2415857, 2415859, 2415861, 2415863, 2415865, 2415866, 2415867, 2415868, 2415869, 2415871, 2415875, 2415876, 2415879, 2415880, 2415881, 2415883, 2415884, 2415886, 2415890, 2415893, 2415894, 2415897, 2415900, 2415901, 2415902, 2415904, 2415905, 2415906, 2415908, 2415909, 2415911, 2415912, 2415916, 2415917, 2415919, 2415920, 2415921, 2415924, 2415925, 2415928, 2415931, 2415932, 2415934, 2415935, 2415936, 2415937, 2415941, 2415942, 2415943, 2415944, 2415945, 2415947, 2415948, 2415951, 2415953, 2415955, 2415958, 2415959, 2415964, 2415965, 2415968, 2415971, 2415973, 2415975, 2415978, 2415980, 2415981, 2415982, 2415983, 2415984, 2415986, 2415988, 2415992, 2415994, 2415995, 2415996, 2415997, 2415998, 2416002, 2416003, 2416005, 2416007, 2416008, 2416009, 2416012, 2416014, 2416016, 2416024, 2416025, 2416027, 2416028, 2416030, 2416031, 2416033, 2416034, 2416035, 2416036, 2416039, 2416041, 2416042, 2416044, 2416045, 2416047, 2416050, 2416051, 2416053, 2416056, 2416058, 2416059, 2416060, 2416061, 2416065, 2416066, 2416068, 2416071, 2416072, 2416074, 2416075, 2416076, 2416077, 2416078, 2416079, 2416080, 2416081, 2416084, 2416085, 2416087, 2416088, 2416089, 2416092, 2416093, 2416095, 2416096, 2416097, 2416099, 2416101, 2416105, 2416107, 2416108, 2416109, 2416110, 2416111, 2416113, 2416114, 2416117, 2416118, 2416120, 2416121, 2416122, 2416123, 2416124, 2416126, 2416128, 2416129, 2416132, 2416134, 2416135, 2416137, 2416138, 2416141, 2416142, 2416144, 2416148, 2416149, 2416150, 2416151, 2416152, 2416153, 2416154, 2416157, 2416158, 2416160, 2416163, 2416164, 2416168, 2416169, 2416170, 2416171, 2416172, 2416174, 2416177, 2416178, 2416179, 2416180, 2416181, 2416182, 2416184, 2416186, 2416188, 2416190, 2416192, 2416193, 2416194, 2416196, 2416198, 2416201, 2416202, 2416203, 2416208, 2416209, 2416210, 2416211, 2416212, 2416215, 2416216, 2416217, 2416220, 2416221, 2416225, 2416227, 2416228, 2416229, 2416231, 2416233, 2416249, 2416252, 2416253, 2416256, 2416258, 2416259, 2416260, 2416261, 2416262, 2416263, 2416264, 2416265, 2416266, 2416268, 2416269, 2416270, 2416272, 2416273, 2416274, 2416275, 2416276, 2416277, 2416278, 2416279, 2416281, 2416282, 2416283, 2416287, 2416291, 2416292, 2416294, 2416298, 2416299, 2416301, 2416302, 2416303, 2416305, 2416306, 2416307, 2416309, 2416311, 2416312, 2416315, 2416319, 2416323, 2416324, 2416326, 2416327, 2416328, 2416329, 2416331, 2416333, 2416334, 2416335, 2416336, 2416337, 2416340, 2416341, 2416343, 2416345, 2416351, 2416356, 2416358, 2416359, 2416361, 2416362, 2416365, 2416366, 2416369, 2416371, 2416373, 2416374, 2416375, 2416376, 2416378, 2416380, 2416381, 2416382, 2416383, 2416384, 2416386, 2416388, 2416389, 2416390, 2416393, 2416397, 2416398, 2416399, 2416400, 2416401, 2416402, 2416403, 2416405, 2416407, 2416409, 2416413, 2416414, 2416416, 2416417, 2416419, 2416423, 2416424, 2416426, 2416428, 2416432, 2416433, 2416437, 2416438, 2416439, 2416442, 2416444, 2416446, 2416447, 2416448, 2416449, 2416450, 2416453, 2416457, 2416458, 2416459, 2416460, 2416463, 2416464, 2416465, 2416467, 2416468, 2416472, 2416473, 2416474, 2416477, 2416478, 2416479, 2416482, 2416484, 2416485, 2416487, 2416488, 2416490, 2416492, 2416493, 2416494, 2416495, 2416497, 2416498, 2416504, 2416505, 2416506, 2416508, 2416509, 2416511, 2416513, 2416516, 2416518, 2416520, 2416521, 2416523, 2416525, 2416527, 2416529, 2416531, 2416533, 2416535, 2416536, 2416538, 2416543, 2416552, 2416554, 2416555, 2416556, 2416557, 2416560, 2416561, 2416562, 2416563, 2416565, 2416572, 2416574, 2416575, 2416576, 2416578, 2416580, 2416581, 2416582, 2416585, 2416586, 2416588, 2416589, 2416590, 2416596, 2416600, 2416601, 2416604, 2416606, 2416607, 2416610, 2416618, 2416619, 2416623, 2416624, 2416627, 2416628, 2416629, 2416631, 2416632, 2416634, 2416637, 2416638, 2416639, 2416641, 2416643, 2416645, 2416646, 2416648, 2416649, 2416650, 2416651, 2416652, 2416654, 2416658, 2416661, 2416662, 2416663, 2416665, 2416668, 2416670, 2416672, 2416673, 2416674, 2416676, 2416677, 2416678, 2416680, 2416683, 2416684, 2416685, 2416690, 2416693, 2416694, 2416695, 2416696, 2416699, 2416701, 2416704, 2416708, 2416709, 2416710, 2416713, 2416714, 2416716, 2416717, 2416718, 2416719, 2416721, 2416724, 2416728, 2416729, 2416730, 2416734, 2416735, 2416736, 2416737, 2416742, 2416743, 2416744, 2416745, 2416746, 2416747, 2416749, 2416752, 2416753, 2416754, 2416756, 2416757, 2416758, 2416760, 2416762, 2416765, 2416766, 2416767, 2416768, 2416769, 2416771, 2416774, 2416778, 2416779, 2416782, 2416783, 2416786, 2416787, 2416788, 2416790, 2416791, 2416792, 2416793, 2416794, 2416796, 2416801, 2416804, 2416806, 2416807, 2416812, 2416813, 2416814, 2416816, 2416818, 2416819, 2416820, 2416823, 2416825, 2416826, 2416827, 2416828, 2416830, 2416832, 2416833, 2416834, 2416835, 2416836, 2416840, 2416843, 2416847, 2416849, 2416850, 2416853, 2416854, 2416855, 2416856, 2416862, 2416863, 2416865, 2416867, 2416869, 2416871, 2416873, 2416874, 2416875, 2416882, 2416884, 2416885, 2416886, 2416887, 2416889, 2416890, 2416891, 2416895, 2416897, 2416898, 2416900, 2416901, 2416902, 2416904, 2416905, 2416906, 2416907, 2416909, 2416910, 2416911, 2416913, 2416914, 2416917, 2416918, 2416919, 2416920, 2416921, 2416923, 2416924, 2416925, 2416926, 2416927, 2416928, 2416929, 2416930, 2416931, 2416932, 2416933, 2416934, 2416937, 2416939, 2416940, 2416941, 2416942, 2416945, 2416948, 2416949, 2416953, 2416955, 2416961, 2416962, 2416963, 2416964, 2416965, 2416967, 2416969, 2416970, 2416972, 2416973, 2416974, 2416975, 2416976, 2416977, 2416979, 2416980, 2416982, 2416985, 2416987, 2416991, 2416993, 2416995, 2416996, 2416997, 2416998, 2417000, 2417001, 2417002, 2417003, 2417005, 2417006, 2417007, 2417008, 2417009, 2417012, 2417013, 2417015, 2417016, 2417019, 2417020, 2417022, 2417023, 2417024, 2417026, 2417029, 2417030, 2417033, 2417036, 2417038, 2417042, 2417045, 2417047, 2417048, 2417050, 2417051, 2417052, 2417053, 2417054, 2417059, 2417060, 2417061, 2417062, 2417063, 2417064, 2417065, 2417067, 2417068, 2417070, 2417071, 2417072, 2417073, 2417074, 2417075, 2417077, 2417078, 2417082, 2417083, 2417084, 2417085, 2417086, 2417088, 2417089, 2417094, 2417095, 2417097, 2417098, 2417100, 2417103, 2417104, 2417106, 2417108, 2417109, 2417111, 2417112, 2417113, 2417114, 2417116, 2417119, 2417120, 2417121, 2417123, 2417124, 2417125, 2417126, 2417127, 2417128, 2417129, 2417130, 2417131, 2417132, 2417135, 2417137, 2417139, 2417141, 2417143, 2417145, 2417146, 2417147, 2417149, 2417151, 2417152, 2417154, 2417161, 2417162, 2417163, 2417164, 2417166, 2417167, 2417168, 2417169, 2417170, 2417171, 2417172, 2417173, 2417174, 2417176, 2417177, 2417180, 2417181, 2417182, 2417186, 2417188, 2417191, 2417192, 2417195, 2417196, 2417197, 2417199, 2417200, 2417201, 2417204, 2417206, 2417208, 2417210, 2417212, 2417216, 2417217, 2417219, 2417222, 2417224, 2417225, 2417226, 2417227, 2417228, 2417229, 2417232, 2417233, 2417235, 2417236, 2417238, 2417239, 2417240, 2417241, 2417243, 2417244, 2417245, 2417246, 2417247, 2417252, 2417253, 2417254, 2417255, 2417256, 2417257, 2417258, 2417262, 2417264, 2417265, 2417268, 2417272, 2417273, 2417274, 2417275, 2417276, 2417278, 2417282, 2417285, 2417287, 2417288, 2417290, 2417291, 2417295, 2417296, 2417301, 2417303, 2417304, 2417306, 2417307, 2417308, 2417314, 2417319, 2417320, 2417322, 2417323, 2417326, 2417327, 2417328, 2417330, 2417334, 2417337, 2417341, 2417342, 2417345, 2417346, 2417347, 2417349, 2417352, 2417353, 2417354, 2417355, 2417359, 2417360, 2417361, 2417362, 2417363, 2417364, 2417366, 2417367, 2417369, 2417370, 2417371, 2417372, 2417374, 2417376, 2417377, 2417378, 2417379, 2417380, 2417381, 2417384, 2417389, 2417391, 2417397, 2417398, 2417401, 2417406, 2417407, 2417409, 2417410, 2417411, 2417412, 2417413, 2417416, 2417419, 2417421, 2417422, 2417423, 2417424, 2417425, 2417427, 2417434, 2417435, 2417436, 2417437, 2417438, 2417441, 2417446, 2417452, 2417453, 2417454, 2417456, 2417460, 2417461, 2417462, 2417465, 2417466, 2417467, 2417468, 2417469, 2417470, 2417472, 2417473, 2417476, 2417478, 2417481, 2417486, 2417487, 2417490, 2417491, 2417493, 2417497, 2417498, 2417500, 2417501, 2417503, 2417504, 2417505, 2417506, 2417507, 2417508, 2417509, 2417511, 2417512, 2417523, 2417524, 2417525, 2417526, 2417531, 2417532, 2417533, 2417536, 2417541, 2417542, 2417543, 2417544, 2417545, 2417546, 2417547, 2417549, 2417550, 2417553, 2417556, 2417558, 2417559, 2417560, 2417561, 2417562, 2417564, 2417565, 2417566, 2417568, 2417569, 2417570, 2417575, 2417581, 2417582, 2417584, 2417585, 2417586, 2417587, 2417588, 2417589, 2417590, 2417591, 2417592, 2417594, 2417596, 2417597, 2417598, 2417599, 2417600, 2417601, 2417602, 2417603, 2417605, 2417607, 2417608, 2417610, 2417611, 2417613, 2417614, 2417615, 2417617, 2417620, 2417622, 2417623, 2417625, 2417626, 2417627, 2417628, 2417629, 2417631, 2417633, 2417634, 2417635, 2417636, 2417640, 2417641, 2417644, 2417645, 2417646, 2417649, 2417653, 2417654, 2417655, 2417660, 2417661, 2417662, 2417663, 2417664, 2417667, 2417669, 2417670, 2417677, 2417678, 2417681, 2417682, 2417684, 2417687, 2417689, 2417690, 2417693, 2417694, 2417696, 2417698, 2417700, 2417701, 2417704, 2417705, 2417706, 2417710, 2417712, 2417713, 2417717, 2417718, 2417719, 2417720, 2417722, 2417723, 2417725, 2417727, 2417729, 2417730, 2417732, 2417733, 2417736, 2417737, 2417738, 2417739, 2417740, 2417742, 2417743, 2417744, 2417745, 2417746, 2417748, 2417749, 2417750, 2417752, 2417753, 2417754, 2417755, 2417756, 2417757, 2417759, 2417760, 2417761, 2417763, 2417767, 2417768, 2417769, 2417772, 2417773, 2417775, 2417778, 2417779, 2417780, 2417786, 2417789, 2417791, 2417792, 2417793, 2417794, 2417795, 2417797, 2417800, 2417801, 2417804, 2417805, 2417807, 2417813, 2417816, 2417817, 2417818, 2417819, 2417820, 2417821, 2417822, 2417825, 2417827, 2417829, 2417830, 2417831, 2417834, 2417838, 2417839, 2417840, 2417841, 2417844, 2417846, 2417847, 2417849, 2417851, 2417852, 2417854, 2417855, 2417859, 2417860, 2417861, 2417862, 2417863, 2417864, 2417865, 2417868, 2417872, 2417873, 2417876, 2417879, 2417880, 2417882, 2417883, 2417884, 2417887, 2417888, 2417889, 2417890, 2417891, 2417892, 2417893, 2417894, 2417895, 2417896, 2417897, 2417898, 2417899, 2417902, 2417903, 2417904, 2417906, 2417907, 2417909, 2417910, 2417912, 2417913, 2417914, 2417915, 2417916, 2417917, 2417918, 2417919, 2417920, 2417921, 2417922, 2417924, 2417925, 2417928, 2417929, 2417931, 2417932, 2417934, 2417935, 2417937, 2417941, 2417943, 2417944, 2417946, 2417947, 2417948, 2417951, 2417952, 2417953, 2417954, 2417956, 2417957, 2417958, 2417961, 2417962, 2417964, 2417965, 2417966, 2417967, 2417970, 2417971, 2417974, 2417977, 2417978, 2417980, 2417981, 2417983, 2417985, 2417988, 2417989, 2417993, 2417994, 2417995, 2417997]
VAL_MD5 = '7fd4612d059eb7a7cd3bad7bf0f5e931'
G0_IDS = [2315353, 2315357, 2315358, 2315359, 2315361, 2315362, 2315363, 2315366, 2315368, 2315369, 2315370, 2315371, 2315373, 2315374, 2315376, 2315377, 2315378, 2315379, 2315380, 2315381, 2315382, 2315383, 2315384, 2315388, 2315391, 2315393, 2315394, 2315395, 2315397, 2315398, 2315399, 2315402, 2315403, 2315405, 2315406, 2315409, 2315410, 2315411, 2315414, 2315415, 2315416, 2315417, 2315422, 2315425, 2315427, 2315428, 2315429, 2315433, 2315434, 2315439, 2315442, 2315443, 2315444, 2315445, 2315446, 2315447, 2315448, 2315450, 2315451, 2315452, 2315453, 2315455, 2315456, 2315458, 2315459, 2315460, 2315463, 2315465, 2315466, 2315468, 2315470, 2315472, 2315473, 2315476, 2315477, 2315480, 2315484, 2315486, 2315490, 2315497, 2315499, 2315500, 2315503, 2315504, 2315505, 2315509, 2315510, 2315512, 2315513, 2315519, 2315520, 2315523, 2315526, 2315528, 2315530, 2315534, 2315539, 2315543, 2315546, 2315547, 2315548, 2315549, 2315550, 2315552, 2315553, 2315555, 2315559, 2315563, 2315566, 2315568, 2315570, 2315576, 2315578, 2315579, 2315580, 2315586, 2315587, 2315590, 2315591, 2315592, 2315593, 2315597, 2315598, 2315600, 2315601, 2315604, 2315611, 2315613, 2315617, 2315618, 2315619, 2315621, 2315622, 2315626, 2315628, 2315629, 2315630, 2315632, 2315634, 2315639, 2315644, 2315646, 2315647, 2315651, 2315654, 2315656, 2315657, 2315658, 2315661, 2315662, 2315663, 2315665, 2315667, 2315671, 2315673, 2315675, 2315677, 2315678, 2315679, 2315683, 2315686, 2315687, 2315691, 2315696, 2315698, 2315699, 2315705, 2315712, 2315718, 2315721, 2315724, 2315729, 2315730, 2315733, 2315735, 2315737, 2315739, 2315741, 2315742, 2315743, 2315744, 2315745, 2315747, 2315749, 2315750, 2315753, 2315761, 2315762, 2315763, 2315764, 2315765, 2315767, 2315771, 2315773, 2315774, 2315776, 2315779, 2315782, 2315787, 2315788, 2315790, 2315791, 2315792, 2315794, 2315795, 2315797, 2315803, 2315804, 2315805, 2315809, 2315810, 2315812, 2315814, 2315815, 2315816, 2315817, 2315821, 2315823, 2315824, 2315826, 2315827, 2315830, 2315833, 2315834, 2315835, 2315837, 2315842, 2315843, 2315844, 2315846, 2315847, 2315849, 2315852, 2315857, 2315862, 2315865, 2315866, 2315870, 2315872, 2315873, 2315874, 2315881, 2315882, 2315883, 2315887, 2315889, 2315890, 2315891, 2315892, 2315893, 2315895, 2315897, 2315898, 2315901, 2315902, 2315903, 2315904, 2315905, 2315909, 2315910, 2315913, 2315916, 2315917, 2315919, 2315922, 2315924, 2315925, 2315930, 2315933, 2315935, 2315936, 2315938, 2315939, 2315940, 2315941, 2315942, 2315943, 2315947, 2315948, 2315951, 2315953, 2315954, 2315955, 2315960, 2315962, 2315963, 2315964, 2315965, 2315967, 2315969, 2315973, 2315975, 2315976, 2315979, 2315980, 2315981, 2315982, 2315983, 2315984, 2315985, 2315986, 2315987, 2315990, 2315992, 2315995, 2315996, 2315998, 2316001, 2316002, 2316004, 2316006, 2316010, 2316011, 2316012, 2316013, 2316015, 2316021, 2316025, 2316026, 2316028, 2316033, 2316035, 2316036, 2316037, 2316040, 2316041, 2316042, 2316043, 2316044, 2316048, 2316049, 2316054, 2316056, 2316058, 2316061, 2316063, 2316065, 2316066, 2316068, 2316071, 2316072, 2316074, 2316075, 2316076, 2316079, 2316082, 2316083, 2316084, 2316090, 2316093, 2316094, 2316095, 2316097, 2316098, 2316099, 2316101, 2316102, 2316103, 2316104, 2316105, 2316107, 2316112, 2316115, 2316123, 2316125, 2316128, 2316129, 2316130, 2316131, 2316132, 2316133, 2316135, 2316136, 2316137, 2316138, 2316140, 2316141, 2316142, 2316148, 2316149, 2316155, 2316157, 2316158, 2316160, 2316161, 2316162, 2316164, 2316166, 2316168, 2316169, 2316170, 2316173, 2316174, 2316175, 2316176, 2316177, 2316181, 2316182, 2316183, 2316185, 2316186, 2316190, 2316191, 2316195, 2316197, 2316198, 2316201, 2316202, 2316204, 2316205, 2316210, 2316214, 2316216, 2316218, 2316220, 2316221, 2316222, 2316223, 2316224, 2316226, 2316228, 2316229, 2316230, 2316231, 2316233, 2316234, 2316239, 2316241, 2316242, 2316243, 2316244, 2316245, 2316247, 2316251, 2316252, 2316253, 2316254, 2316260, 2316261, 2316266, 2316268, 2316269, 2316271, 2316272, 2316274, 2316275, 2316277, 2316280, 2316281, 2316282, 2316287, 2316289, 2316290, 2316293, 2316294, 2316295, 2316297, 2316299, 2316300, 2316303, 2316304, 2316305, 2316306, 2316307, 2316308, 2316311, 2316312, 2316313, 2316314, 2316316, 2316317, 2316318, 2316320, 2316322, 2316325, 2316327, 2316329, 2316330, 2316331, 2316336, 2316337, 2316340, 2316344, 2316345, 2316348, 2316350, 2316353, 2316355, 2316356, 2316357, 2316359, 2316360, 2316362, 2316363, 2316365, 2316366, 2316369, 2316372, 2316374, 2316376, 2316377, 2316379, 2316380, 2316381, 2316387, 2316391, 2316394, 2316395, 2316397, 2316401, 2316402, 2316409, 2316410, 2316412, 2316413, 2316415, 2316417, 2316419, 2316423, 2316425, 2316426, 2316431, 2316432, 2316433, 2316435, 2316436, 2316439, 2316440, 2316443, 2316444, 2316445, 2316446, 2316447, 2316448, 2316450, 2316452, 2316453, 2316455, 2316456, 2316458, 2316459, 2316460, 2316461, 2316462, 2316466, 2316469, 2316471, 2316473, 2316476, 2316478, 2316480, 2316482, 2316484, 2316485, 2316486, 2316487, 2316488, 2316489, 2316490, 2316492, 2316493, 2316494, 2316495, 2316499, 2316500, 2316501, 2316503, 2316504, 2316505, 2316506, 2316508, 2316510, 2316512, 2316513, 2316514, 2316517, 2316518, 2316519, 2316522, 2316528, 2316529, 2316530, 2316531, 2316534, 2316539, 2316540, 2316541, 2316543, 2316544, 2316545, 2316546, 2316547, 2316549, 2316551, 2316553, 2316555, 2316557, 2316558, 2316562, 2316563, 2316567, 2316569, 2316572, 2316573, 2316574, 2316575, 2316576, 2316583, 2316584, 2316585, 2316587, 2316588, 2316591, 2316592, 2316593, 2316594, 2316595, 2316596, 2316597, 2316598, 2316600, 2316604, 2316605, 2316607, 2316610, 2316611, 2316612, 2316617, 2316619, 2316621, 2316623, 2316624, 2316625, 2316626, 2316630, 2316635, 2316641, 2316642, 2316644, 2316645, 2316648, 2316649, 2316651, 2316652, 2316653, 2316655, 2316656, 2316658, 2316659, 2316661, 2316666, 2316667, 2316669, 2316672, 2316673, 2316674, 2316675, 2316682, 2316684, 2316686, 2316687, 2316688, 2316689, 2316691, 2316692, 2316696, 2316700, 2316703, 2316707, 2316708, 2316709, 2316711, 2316716, 2316721, 2316723, 2316726, 2316727, 2316728, 2316730, 2316732, 2316734, 2316736, 2316739, 2316740, 2316743, 2316745, 2316747, 2316752, 2316756, 2316758, 2316759, 2316763, 2316770, 2316773, 2316774, 2316775, 2316779, 2316780, 2316781, 2316783, 2316789, 2316790, 2316793, 2316797, 2316798, 2316799, 2316800, 2316802, 2316803, 2316804, 2316805, 2316806, 2316808, 2316811, 2316812, 2316815, 2316816, 2316818, 2316820, 2316823, 2316824, 2316829, 2316831, 2316832, 2316834, 2316836, 2316837, 2316844, 2316845, 2316846, 2316848, 2316849, 2316852, 2316855, 2316856, 2316858, 2316861, 2316862, 2316864, 2316869, 2316870, 2316871, 2316873, 2316874, 2316875, 2316876, 2316879, 2316881, 2316882, 2316884, 2316885, 2316886, 2316887, 2316891, 2316892, 2316894, 2316896, 2316897, 2316899, 2316900, 2316901, 2316902, 2316905, 2316907, 2316913, 2316914, 2316915, 2316918, 2316922, 2316923, 2316925, 2316926, 2316928, 2316929, 2316930, 2316931, 2316932, 2316933, 2316934, 2316938, 2316940, 2316941, 2316943, 2316948, 2316949, 2316951, 2316952, 2316953, 2316954, 2316956, 2316957, 2316958, 2316959, 2316960, 2316961, 2316964, 2316965, 2316966, 2316970, 2316971, 2316974, 2316975, 2316977, 2316978, 2316980, 2316982, 2316984, 2316985, 2316986, 2316987, 2316988, 2316991, 2316992, 2316993, 2316994, 2316996, 2317000, 2317004, 2317005, 2317006, 2317008, 2317013, 2317016, 2317017, 2317018, 2317021, 2317022, 2317023, 2317024, 2317026, 2317027, 2317033, 2317034, 2317035, 2317038, 2317039, 2317041, 2317043, 2317045, 2317046, 2317048, 2317050, 2317051, 2317053, 2317061, 2317064, 2317065, 2317071, 2317072, 2317073, 2317074, 2317076, 2317077, 2317079, 2317081, 2317082, 2317083, 2317084, 2317085, 2317086, 2317087, 2317088, 2317089, 2317090, 2317092, 2317093, 2317094, 2317097, 2317102, 2317103, 2317104, 2317105, 2317107, 2317109, 2317110, 2317112, 2317114, 2317115, 2317116, 2317118, 2317119, 2317120, 2317121, 2317123, 2317125, 2317126, 2317127, 2317128, 2317131, 2317133, 2317139, 2317140, 2317142, 2317145, 2317146, 2317147, 2317148, 2317150, 2317151, 2317152, 2317155, 2317159, 2317162, 2317163, 2317164, 2317167, 2317169, 2317170, 2317174, 2317175, 2317176, 2317182, 2317183, 2317185, 2317186, 2317187, 2317188, 2317190, 2317191, 2317193, 2317194, 2317195, 2317196, 2317197, 2317199, 2317200, 2317203, 2317207, 2317208, 2317210, 2317212, 2317214, 2317216, 2317217, 2317218, 2317219, 2317220, 2317223, 2317224, 2317226, 2317227, 2317229, 2317231, 2317232, 2317234, 2317236, 2317237, 2317238, 2317240, 2317241, 2317250, 2317251, 2317254, 2317257, 2317265, 2317267, 2317270, 2317272, 2317277, 2317278, 2317281, 2317283, 2317284, 2317287, 2317288, 2317292, 2317295, 2317296, 2317299, 2317300, 2317301, 2317302, 2317303, 2317304, 2317305, 2317306, 2317307, 2317317, 2317318, 2317319, 2317320, 2317322, 2317323, 2317325, 2317329, 2317330, 2317331, 2317332, 2317335, 2317338, 2317339, 2317340, 2317342, 2317343, 2317345, 2317348, 2317350, 2317351, 2317354, 2317356, 2317358, 2317361, 2317362, 2317364, 2317365, 2317366, 2317367, 2317371, 2317372, 2317376, 2317380, 2317386, 2317387, 2317388, 2317389, 2317391, 2317393, 2317394, 2317397, 2317399, 2317403, 2317406, 2317407, 2317412, 2317413, 2317416, 2317417, 2317420, 2317421, 2317422, 2317423, 2317426, 2317429, 2317434, 2317435, 2317436, 2317438, 2317439, 2317440, 2317443, 2317446, 2317448, 2317451, 2317452, 2317455, 2317457, 2317458, 2317459, 2317464, 2317468, 2317469, 2317470, 2317472, 2317473, 2317476, 2317479, 2317481, 2317485, 2317486, 2317487, 2317492, 2317493, 2317495, 2317497, 2317498, 2317500, 2317502, 2317503, 2317504, 2317506, 2317507, 2317508, 2317510, 2317512, 2317513, 2317514, 2317515, 2317516, 2317517, 2317518, 2317520, 2317523, 2317529, 2317531, 2317533, 2317534, 2317535, 2317536, 2317537, 2317538, 2317540, 2317541, 2317542, 2317548, 2317549, 2317551, 2317552, 2317553, 2317554, 2317555, 2317556, 2317557, 2317558, 2317559, 2317560, 2317561, 2317563, 2317564, 2317565, 2317566, 2317567, 2317568, 2317573, 2317574, 2317575, 2317576, 2317577, 2317580, 2317581, 2317582, 2317584, 2317586, 2317588, 2317589, 2317590, 2317591, 2317592, 2317594, 2317595, 2317597, 2317598, 2317603, 2317605, 2317607, 2317610, 2317612, 2317613, 2317618, 2317620, 2317621, 2317623, 2317630, 2317632, 2317634, 2317636, 2317637, 2317639, 2317640, 2317648, 2317649, 2317654, 2317657, 2317659, 2317660, 2317664, 2317668, 2317671, 2317672, 2317673, 2317675, 2317676, 2317677, 2317679, 2317683, 2317684, 2317685, 2317686, 2317687, 2317688, 2317692, 2317694, 2317695, 2317696, 2317697, 2317698, 2317700, 2317701, 2317703, 2317705, 2317710, 2317713, 2317715, 2317716, 2317717, 2317724, 2317727, 2317729, 2317730, 2317731, 2317732, 2317733, 2317735, 2317737, 2317738, 2317740, 2317741, 2317742, 2317744, 2317745, 2317748, 2317749, 2317754, 2317755, 2317759, 2317760, 2317761, 2317764, 2317767, 2317768, 2317772, 2317773, 2317774, 2317775, 2317782, 2317785, 2317786, 2317790, 2317791, 2317800, 2317802, 2317804, 2317807, 2317808, 2317809, 2317814, 2317818, 2317822, 2317825, 2317827, 2317829, 2317833, 2317834, 2317839, 2317841, 2317843, 2317845, 2317846, 2317848, 2317849, 2317851, 2317852, 2317854, 2317857, 2317858, 2317860, 2317861, 2317866, 2317868, 2317871, 2317872, 2317873, 2317875, 2317879, 2317880, 2317882, 2317886, 2317888, 2317889, 2317891, 2317893, 2317894, 2317898, 2317900, 2317901, 2317903, 2317905, 2317906, 2317907, 2317908, 2317910, 2317913, 2317914, 2317916, 2317917, 2317919, 2317921, 2317923, 2317926, 2317927, 2317928, 2317929, 2317930, 2317931, 2317932, 2317933, 2317934, 2317937, 2317940, 2317942, 2317943, 2317946, 2317948, 2317949, 2317950, 2317951, 2317953, 2317955, 2317956, 2317957, 2317958, 2317959, 2317962, 2317963, 2317964, 2317965, 2317972, 2317973, 2317975, 2317976, 2317977, 2317978, 2317979, 2317981, 2317982, 2317983, 2317987, 2317989, 2317990, 2317992, 2317996, 2317998, 2318002, 2318003, 2318005, 2318007, 2318008, 2318009, 2318011, 2318012, 2318015, 2318016, 2318019, 2318020, 2318021, 2318022, 2318024, 2318028, 2318030, 2318031, 2318033, 2318036, 2318041, 2318042, 2318045, 2318046, 2318047, 2318048, 2318049, 2318051, 2318052, 2318053, 2318054, 2318056, 2318057, 2318058, 2318059, 2318060, 2318062, 2318063, 2318067, 2318069, 2318071, 2318072, 2318076, 2318077, 2318079, 2318084, 2318087, 2318088, 2318089, 2318091, 2318092, 2318093, 2318094, 2318095, 2318096, 2318097, 2318102, 2318105, 2318106, 2318107, 2318109, 2318111, 2318113, 2318115, 2318116, 2318117, 2318118, 2318119, 2318123, 2318125, 2318126, 2318127, 2318128, 2318129, 2318134, 2318140, 2318144, 2318146, 2318147, 2318148, 2318149, 2318151, 2318154, 2318156, 2318157, 2318158, 2318159, 2318163, 2318165, 2318167, 2318168, 2318169, 2318170, 2318171, 2318172, 2318174, 2318175, 2318178, 2318179, 2318181, 2318182, 2318185, 2318188, 2318190, 2318194, 2318197, 2318198, 2318199, 2318200, 2318204, 2318207, 2318210, 2318212, 2318213, 2318214, 2318216, 2318218, 2318220, 2318222, 2318228, 2318229, 2318230, 2318232, 2318234, 2318237, 2318241, 2318244, 2318245, 2318252, 2318256, 2318257, 2318259, 2318260, 2318262, 2318267, 2318268, 2318269, 2318270, 2318273, 2318281, 2318282, 2318283, 2318285, 2318286, 2318287, 2318289, 2318290, 2318291, 2318292, 2318297, 2318298, 2318300, 2318304, 2318305, 2318306, 2318307, 2318308, 2318311, 2318314, 2318315, 2318316, 2318318, 2318319, 2318320, 2318321, 2318323, 2318324, 2318326, 2318328, 2318330, 2318331, 2318332, 2318333, 2318334, 2318335, 2318336, 2318337, 2318339, 2318342, 2318344, 2318346, 2318347, 2318348, 2318349, 2318351, 2318352, 2318353, 2318354, 2318356, 2318357, 2318358, 2318359, 2318361, 2318362, 2318364, 2318366, 2318367, 2318371, 2318375, 2318378, 2318379, 2318384, 2318385, 2318386, 2318388, 2318391, 2318392, 2318395, 2318396, 2318397, 2318399, 2318400, 2318401, 2318402, 2318403, 2318405, 2318406, 2318407, 2318408, 2318410, 2318414, 2318415, 2318419, 2318420, 2318421, 2318423, 2318424, 2318426, 2318427, 2318428, 2318429, 2318431, 2318433, 2318434, 2318435, 2318437, 2318438, 2318440, 2318442, 2318446, 2318447, 2318450, 2318452, 2318455, 2318458, 2318459, 2318460, 2318461, 2318462, 2318463, 2318465, 2318466, 2318467, 2318472, 2318473, 2318475, 2318476, 2318480, 2318481, 2318483, 2318489, 2318490, 2318493, 2318494, 2318496, 2318497, 2318498, 2318499, 2318504, 2318509, 2318513, 2318515, 2318520, 2318521, 2318522, 2318523, 2318524, 2318527, 2318528, 2318534, 2318535, 2318536, 2318538, 2318539, 2318540, 2318542, 2318543, 2318546, 2318547, 2318549, 2318550, 2318551, 2318552, 2318553, 2318554, 2318555, 2318557, 2318560, 2318561, 2318565, 2318566, 2318568, 2318570, 2318573, 2318574, 2318575, 2318580, 2318581, 2318583, 2318586, 2318588, 2318589, 2318590, 2318591, 2318592, 2318593, 2318596, 2318597, 2318598, 2318599, 2318602, 2318604, 2318605, 2318607, 2318608, 2318609, 2318611, 2318612, 2318616, 2318618, 2318620, 2318622, 2318623, 2318624, 2318625, 2318627, 2318629, 2318630, 2318632, 2318633, 2318637, 2318641, 2318642, 2318645, 2318646, 2318647, 2318648, 2318651, 2318652, 2318653, 2318654, 2318656, 2318657, 2318659, 2318660, 2318661, 2318663, 2318666, 2318667, 2318668, 2318669, 2318670, 2318671, 2318673, 2318675, 2318678, 2318679, 2318680, 2318681, 2318683, 2318685, 2318686, 2318687, 2318690, 2318692, 2318693, 2318694, 2318699, 2318700, 2318702, 2318704, 2318705, 2318707, 2318711, 2318713, 2318719, 2318720, 2318725, 2318727, 2318728, 2318729, 2318730, 2318732, 2318734, 2318736, 2318738, 2318741, 2318742, 2318743, 2318744, 2318745, 2318747, 2318749, 2318751, 2318752, 2318753, 2318757, 2318758, 2318759, 2318760, 2318762, 2318764, 2318766, 2318768, 2318769, 2318778, 2318779, 2318780, 2318781, 2318785, 2318788, 2318790, 2318791, 2318792, 2318794, 2318796, 2318798, 2318801, 2318804, 2318805, 2318810, 2318811, 2318812, 2318813, 2318814, 2318815, 2318817, 2318818, 2318826, 2318828, 2318829, 2318831, 2318833, 2318834, 2318835, 2318842, 2318843, 2318845, 2318849, 2318854, 2318857, 2318858, 2318860, 2318861, 2318864, 2318866, 2318867, 2318868, 2318870, 2318871, 2318873, 2318874, 2318875, 2318877, 2318880, 2318883, 2318884, 2318885, 2318887, 2318889, 2318890, 2318892, 2318895, 2318898, 2318901, 2318902, 2318903, 2318905, 2318906, 2318907, 2318910, 2318911, 2318913, 2318915, 2318917, 2318918, 2318919, 2318921, 2318922, 2318924, 2318929, 2318930, 2318934, 2318935, 2318937, 2318939, 2318940, 2318943, 2318946, 2318947, 2318948, 2318949, 2318950, 2318955, 2318956, 2318960, 2318961, 2318962, 2318963, 2318964, 2318970, 2318972, 2318974, 2318978, 2318979, 2318981, 2318984, 2318986, 2318987, 2318988, 2318989, 2318990, 2318992, 2318993, 2318994, 2318995, 2318996, 2318997, 2319000, 2319001, 2319002, 2319005, 2319007, 2319008, 2319010, 2319012, 2319013, 2319014, 2319015, 2319016, 2319017, 2319022, 2319023, 2319024, 2319026, 2319027, 2319037, 2319038, 2319039, 2319042, 2319049, 2319050, 2319051, 2319053, 2319054, 2319055, 2319056, 2319057, 2319058, 2319060, 2319061, 2319062, 2319064, 2319065, 2319066, 2319072, 2319073, 2319074, 2319075, 2319077, 2319078, 2319079, 2319087, 2319088, 2319089, 2319092, 2319095, 2319096, 2319098, 2319101, 2319102, 2319105, 2319106, 2319107, 2319110, 2319111, 2319113, 2319114, 2319117, 2319118, 2319119, 2319123, 2319125, 2319126, 2319129, 2319131, 2319133, 2319135, 2319136, 2319137, 2319142, 2319143, 2319144, 2319147, 2319148, 2319155, 2319159, 2319160, 2319162, 2319163, 2319164, 2319165, 2319166, 2319169, 2319170, 2319171, 2319172, 2319175, 2319176, 2319178, 2319180, 2319182, 2319184, 2319186, 2319187, 2319193, 2319195, 2319196, 2319197, 2319199, 2319200, 2319203, 2319204, 2319205, 2319206, 2319209, 2319212, 2319213, 2319214, 2319215, 2319216, 2319218, 2319220, 2319221, 2319222, 2319224, 2319225, 2319228, 2319230, 2319231, 2319232, 2319235, 2319238, 2319239, 2319240, 2319242, 2319246, 2319247, 2319248, 2319249, 2319253, 2319256, 2319265, 2319267, 2319268, 2319273, 2319276, 2319277, 2319278, 2319279, 2319281, 2319283, 2319286, 2319288, 2319290, 2319297, 2319298, 2319299, 2319302, 2319304, 2319305, 2319308, 2319310, 2319312, 2319313, 2319314, 2319319, 2319320, 2319323, 2319325, 2319326, 2319327, 2319328, 2319331, 2319332, 2319334, 2319336, 2319337, 2319339, 2319343, 2319344, 2319346, 2319349, 2319350, 2319351, 2319355, 2319356, 2319358, 2319359, 2319360, 2319362, 2319365, 2319366, 2319368, 2319370, 2319371, 2319372, 2319373, 2319374, 2319375, 2319377, 2319378, 2319379, 2319381, 2319382, 2319386, 2319387, 2319388, 2319389, 2319391, 2319393, 2319394, 2319395, 2319396, 2319399, 2319400, 2319405, 2319407, 2319410, 2319411, 2319412, 2319413, 2319414, 2319415, 2319417, 2319418, 2319419, 2319420, 2319421, 2319426, 2319430, 2319431, 2319432, 2319437, 2319438, 2319440, 2319441, 2319442, 2319443, 2319444, 2319445, 2319448, 2319449, 2319455, 2319458, 2319459, 2319461, 2319464, 2319465, 2319466, 2319467, 2319468, 2319473, 2319474, 2319476, 2319477, 2319480, 2319482, 2319486, 2319489, 2319490, 2319493, 2319494, 2319495, 2319496, 2319497, 2319500, 2319502, 2319503, 2319505, 2319506, 2319507, 2319514, 2319517, 2319518, 2319522, 2319524, 2319525, 2319529, 2319530, 2319531, 2319532, 2319535, 2319539, 2319541, 2319544, 2319545, 2319548, 2319549, 2319550, 2319551, 2319554, 2319556, 2319560, 2319564, 2319565, 2319572, 2319573, 2319574, 2319575, 2319578, 2319580, 2319581, 2319582, 2319583, 2319584, 2319585, 2319587, 2319588, 2319590, 2319591, 2319592, 2319594, 2319598, 2319599, 2319601, 2319606, 2319607, 2319608, 2319613, 2319614, 2319615, 2319616, 2319617, 2319618, 2319619, 2319621, 2319622, 2319624, 2319625, 2319627, 2319628, 2319629, 2319631, 2319634, 2319635, 2319636, 2319637, 2319641, 2319642, 2319645, 2319646, 2319649, 2319650, 2319651, 2319654, 2319656, 2319659, 2319661, 2319662, 2319666, 2319669, 2319671, 2319672, 2319674, 2319675, 2319677, 2319678, 2319680, 2319683, 2319685, 2319686, 2319689, 2319691, 2319694, 2319695, 2319696, 2319699, 2319700, 2319701, 2319706, 2319707, 2319709, 2319713, 2319714, 2319715, 2319718, 2319719, 2319720, 2319722, 2319723, 2319724, 2319727, 2319729, 2319732, 2319733, 2319734, 2319735, 2319736, 2319737, 2319738, 2319741, 2319742, 2319743, 2319745, 2319746, 2319750, 2319752, 2319755, 2319757, 2319760, 2319763, 2319764, 2319765, 2319766, 2319767, 2319770, 2319771, 2319772, 2319773, 2319776, 2319781, 2319782, 2319784, 2319785, 2319787, 2319789, 2319790, 2319794, 2319798, 2319799, 2319801, 2319803, 2319804, 2319808, 2319809, 2319810, 2319813, 2319814, 2319816, 2319817, 2319822, 2319825, 2319826, 2319828, 2319829, 2319832, 2319833, 2319834, 2319835, 2319836, 2319838, 2319839, 2319844, 2319848, 2319849, 2319850, 2319852, 2319856, 2319859, 2319862, 2319864, 2319869, 2319870, 2319871, 2319872, 2319873, 2319874, 2319875, 2319876, 2319877, 2319880, 2319881, 2319887, 2319890, 2319892, 2319894, 2319896, 2319897, 2319898, 2319899, 2319900, 2319901, 2319904, 2319905, 2319906, 2319912, 2319916, 2319919, 2319923, 2319926, 2319928, 2319930, 2319934, 2319936, 2319937, 2319940, 2319943, 2319946, 2319948, 2319949, 2319951, 2319954, 2319955, 2319957, 2319958, 2319961, 2319965, 2319966, 2319968, 2319969, 2319971, 2319974, 2319975, 2319977, 2319978, 2319979, 2319981, 2319987, 2319990, 2319991, 2319992, 2319993, 2319994, 2319995, 2319997, 2320000, 2320006, 2320007, 2320008, 2320009, 2320013, 2320014, 2320015, 2320019, 2320021, 2320022, 2320024, 2320027, 2320029, 2320033, 2320036, 2320037, 2320040, 2320041, 2320044, 2320045, 2320047, 2320049, 2320050, 2320053, 2320056, 2320059, 2320060, 2320062, 2320063, 2320066, 2320067, 2320069, 2320071, 2320072, 2320073, 2320074, 2320075, 2320076, 2320077, 2320078, 2320086, 2320089, 2320094, 2320096, 2320097, 2320102, 2320103, 2320104, 2320108, 2320112, 2320113, 2320116, 2320120, 2320121, 2320123, 2320125, 2320129, 2320130, 2320133, 2320134, 2320137, 2320138, 2320141, 2320142, 2320143, 2320144, 2320146, 2320147, 2320148, 2320150, 2320154, 2320156, 2320157, 2320158, 2320159, 2320160, 2320161, 2320164, 2320169, 2320171, 2320172, 2320173, 2320174, 2320175, 2320177, 2320179, 2320180, 2320182, 2320183, 2320184, 2320185, 2320191, 2320193, 2320194, 2320195, 2320196, 2320197, 2320198, 2320199, 2320201, 2320203, 2320204, 2320205, 2320207, 2320211, 2320215, 2320216, 2320217, 2320219, 2320220, 2320225, 2320226, 2320228, 2320230, 2320231, 2320233, 2320234, 2320235, 2320236, 2320237, 2320238, 2320239, 2320240, 2320241, 2320242, 2320248, 2320251, 2320252, 2320256, 2320258, 2320262, 2320264, 2320265, 2320266, 2320268, 2320270, 2320271, 2320277, 2320278, 2320279, 2320280, 2320285, 2320293, 2320297, 2320302, 2320304, 2320306, 2320308, 2320311, 2320313, 2320314, 2320315, 2320317, 2320318, 2320320, 2320322, 2320325, 2320326, 2320331, 2320333, 2320334, 2320335, 2320337, 2320345, 2320346, 2320347, 2320348, 2320349, 2320350, 2320351, 2320353, 2320354, 2320359, 2320360, 2320364, 2320365, 2320366, 2320367, 2320369, 2320371, 2320373, 2320374, 2320375, 2320376, 2320377, 2320378, 2320379, 2320380, 2320381, 2320382, 2320384, 2320387, 2320389, 2320392, 2320395, 2320398, 2320399, 2320404, 2320407, 2320408, 2320414, 2320415, 2320416, 2320419, 2320422, 2320423, 2320425, 2320426, 2320427, 2320428, 2320429, 2320431, 2320432, 2320434, 2320440, 2320441, 2320443, 2320445, 2320446, 2320450, 2320451, 2320453, 2320454, 2320456, 2320457, 2320458, 2320464, 2320465, 2320466, 2320470, 2320471, 2320473, 2320474, 2320475, 2320480, 2320484, 2320488, 2320490, 2320493, 2320494, 2320497, 2320500, 2320503, 2320505, 2320511, 2320512, 2320514, 2320517, 2320518, 2320519, 2320523, 2320527, 2320529, 2320530, 2320534, 2320535, 2320536, 2320541, 2320543, 2320545, 2320546, 2320548, 2320550, 2320551, 2320553, 2320555, 2320556, 2320559, 2320561, 2320562, 2320565, 2320567, 2320569, 2320571, 2320572, 2320573, 2320574, 2320575, 2320579, 2320581, 2320582, 2320583, 2320584, 2320587, 2320588, 2320591, 2320597, 2320598, 2320600, 2320602, 2320606, 2320607, 2320613, 2320614, 2320617, 2320620, 2320622, 2320624, 2320625, 2320627, 2320628, 2320632, 2320633, 2320634, 2320635, 2320639, 2320640, 2320641, 2320645, 2320646, 2320650, 2320651, 2320654, 2320656, 2320657, 2320658, 2320660, 2320665, 2320667, 2320668, 2320669, 2320674, 2320675, 2320676, 2320678, 2320679, 2320681, 2320683, 2320686, 2320687, 2320688, 2320690, 2320691, 2320693, 2320697, 2320699, 2320700, 2320703, 2320704, 2320705, 2320707, 2320708, 2320709, 2320712, 2320715, 2320716, 2320717, 2320720, 2320725, 2320726, 2320728, 2320729, 2320731, 2320732, 2320733, 2320737, 2320740, 2320742, 2320743, 2320744, 2320750, 2320752, 2320753, 2320758, 2320759, 2320760, 2320762, 2320764, 2320765, 2320766, 2320768, 2320770, 2320774, 2320775, 2320776, 2320777, 2320779, 2320780, 2320782, 2320783, 2320787, 2320790, 2320792, 2320794, 2320796, 2320800, 2320801, 2320802, 2320803, 2320806, 2320807, 2320808, 2320809, 2320811, 2320812, 2320813, 2320815, 2320816, 2320817, 2320820, 2320824, 2320825, 2320826, 2320827, 2320828, 2320829, 2320833, 2320834, 2320835, 2320838, 2320840, 2320841, 2320843, 2320846, 2320847, 2320848, 2320851, 2320852, 2320855, 2320860, 2320864, 2320866, 2320867, 2320869, 2320871, 2320872, 2320873, 2320874, 2320875, 2320876, 2320882, 2320883, 2320884, 2320886, 2320888, 2320889, 2320890, 2320891, 2320892, 2320894, 2320895, 2320897, 2320899, 2320902, 2320907, 2320908, 2320909, 2320911, 2320912, 2320915, 2320916, 2320917, 2320918, 2320920, 2320924, 2320926, 2320928, 2320930, 2320933, 2320935, 2320937, 2320938, 2320939, 2320940, 2320948, 2320950, 2320954, 2320956, 2320959, 2320960, 2320961, 2320962, 2320963, 2320966, 2320968, 2320969, 2320976, 2320978, 2320981, 2320982, 2320983, 2320985, 2320987, 2320989, 2320990, 2320991, 2320992, 2320993, 2320995, 2320997, 2320998, 2320999, 2321001, 2321005, 2321007, 2321008, 2321012, 2321015, 2321016, 2321017, 2321018, 2321022, 2321023, 2321025, 2321026, 2321028, 2321029, 2321030, 2321034, 2321037, 2321038, 2321040, 2321041, 2321042, 2321043, 2321044, 2321045, 2321047, 2321050, 2321052, 2321056, 2321057, 2321059, 2321061, 2321064, 2321065, 2321067, 2321070, 2321072, 2321076, 2321080, 2321089, 2321091, 2321092, 2321093, 2321094, 2321096, 2321098, 2321099, 2321102, 2321103, 2321104, 2321105, 2321107, 2321108, 2321110, 2321111, 2321112, 2321115, 2321117, 2321120, 2321121, 2321123, 2321125, 2321127, 2321134, 2321137, 2321141, 2321142, 2321143, 2321144, 2321146, 2321152, 2321154, 2321156, 2321159, 2321160, 2321161, 2321163, 2321167, 2321170, 2321172, 2321174, 2321175, 2321179, 2321183, 2321187, 2321188, 2321191, 2321193, 2321195, 2321197, 2321198, 2321199, 2321201, 2321204, 2321207, 2321208, 2321213, 2321218, 2321220, 2321221, 2321222, 2321223, 2321225, 2321228, 2321229, 2321235, 2321238, 2321240, 2321242, 2321244, 2321247, 2321251, 2321255, 2321257, 2321258, 2321261, 2321263, 2321264, 2321265, 2321266, 2321267, 2321268, 2321270, 2321272, 2321273, 2321274, 2321275, 2321277, 2321279, 2321281, 2321283, 2321284, 2321288, 2321290, 2321295, 2321296, 2321298, 2321299, 2321304, 2321307, 2321311, 2321313, 2321315, 2321316, 2321320, 2321322, 2321325, 2321329, 2321330, 2321331, 2321332, 2321338, 2321342, 2321343, 2321344, 2321345, 2321347, 2321350, 2321353, 2321354, 2321356, 2321358, 2321360, 2321361, 2321362, 2321363, 2321364, 2321367, 2321368, 2321372, 2321375, 2321376, 2321378, 2321379, 2321380, 2321385, 2321386, 2321391, 2321394, 2321397, 2321398, 2321400, 2321401, 2321403, 2321404, 2321405, 2321406, 2321407, 2321409, 2321411, 2321414, 2321417, 2321418, 2321419, 2321421, 2321424, 2321427, 2321428, 2321430, 2321431, 2321432, 2321434, 2321435, 2321436, 2321437, 2321440, 2321442, 2321444, 2321445, 2321447, 2321450, 2321455, 2321456, 2321458, 2321461, 2321462, 2321463, 2321464, 2321465, 2321466, 2321467, 2321468, 2321469, 2321470, 2321471, 2321472, 2321473, 2321474, 2321475, 2321477, 2321478, 2321480, 2321481, 2321483, 2321484, 2321487, 2321489, 2321492, 2321493, 2321494, 2321495, 2321496, 2321498, 2321499, 2321500, 2321502, 2321503, 2321504, 2321506, 2321510, 2321511, 2321514, 2321515, 2321516, 2321519, 2321520, 2321522, 2321523, 2321524, 2321526, 2321528, 2321530, 2321532, 2321534, 2321537, 2321538, 2321539, 2321540, 2321541, 2321543, 2321544, 2321545, 2321547, 2321552, 2321553, 2321555, 2321558, 2321562, 2321565, 2321566, 2321567, 2321570, 2321575, 2321576, 2321577, 2321578, 2321580, 2321582, 2321583, 2321584, 2321591, 2321592, 2321595, 2321596, 2321597, 2321601, 2321602, 2321603, 2321604, 2321605, 2321606, 2321609, 2321615, 2321616, 2321617, 2321620, 2321621, 2321622, 2321626, 2321628, 2321629, 2321633, 2321638, 2321640, 2321643, 2321644, 2321647, 2321648, 2321653, 2321654, 2321656, 2321657, 2321659, 2321663, 2321665, 2321666, 2321669, 2321670, 2321671, 2321674, 2321677, 2321678, 2321679, 2321680, 2321682, 2321686, 2321687, 2321688, 2321689, 2321692, 2321693, 2321696, 2321698, 2321700, 2321702, 2321703, 2321704, 2321707, 2321708, 2321711, 2321713, 2321714, 2321716, 2321719, 2321724, 2321725, 2321728, 2321729, 2321732, 2321734, 2321739, 2321741, 2321744, 2321745, 2321754, 2321755, 2321756, 2321758, 2321759, 2321760, 2321762, 2321764, 2321765, 2321766, 2321767, 2321769, 2321770, 2321771, 2321772, 2321773, 2321775, 2321776, 2321777, 2321778, 2321779, 2321782, 2321783, 2321785, 2321786, 2321790, 2321793, 2321794, 2321795, 2321796, 2321797, 2321798, 2321799, 2321800, 2321802, 2321803, 2321808, 2321809, 2321810, 2321811, 2321812, 2321814, 2321817, 2321818, 2321820, 2321821, 2321822, 2321823, 2321827, 2321828, 2321830, 2321831, 2321833, 2321834, 2321835, 2321840, 2321841, 2321846, 2321847, 2321848, 2321851, 2321853, 2321854, 2321857, 2321859, 2321860, 2321861, 2321863, 2321867, 2321868, 2321870, 2321871, 2321872, 2321874, 2321875, 2321878, 2321879, 2321882, 2321883, 2321885, 2321886, 2321889, 2321894, 2321895, 2321896, 2321899, 2321900, 2321904, 2321905, 2321908, 2321911, 2321912, 2321914, 2321915, 2321917, 2321918, 2321922, 2321923, 2321924, 2321928, 2321929, 2321931, 2321932, 2321933, 2321935, 2321937, 2321939, 2321941, 2321942, 2321943, 2321944, 2321947, 2321948, 2321949, 2321950, 2321952, 2321954, 2321956, 2321957, 2321959, 2321960, 2321962, 2321964, 2321966, 2321967, 2321968, 2321970, 2321973, 2321975, 2321976, 2321977, 2321980, 2321981, 2321983, 2321984, 2321990, 2321991, 2321994, 2321995, 2321998, 2321999, 2322002, 2322003, 2322007, 2322008, 2322014, 2322018, 2322019, 2322020, 2322021, 2322023, 2322025, 2322027, 2322030, 2322033, 2322034, 2322035, 2322037, 2322038, 2322040, 2322043, 2322044, 2322046, 2322047, 2322048, 2322049, 2322052, 2322053, 2322054, 2322055, 2322056, 2322064, 2322066, 2322067, 2322069, 2322071, 2322072, 2322074, 2322079, 2322080, 2322081, 2322083, 2322084, 2322088, 2322089, 2322091, 2322092, 2322093, 2322096, 2322097, 2322098, 2322101, 2322103, 2322104, 2322105, 2322106, 2322107, 2322111, 2322112, 2322115, 2322116, 2322117, 2322125, 2322126, 2322129, 2322131, 2322133, 2322135, 2322138, 2322139, 2322141, 2322145, 2322146, 2322149, 2322150, 2322152, 2322153, 2322154, 2322157, 2322158, 2322160, 2322161, 2322165, 2322167, 2322168, 2322169, 2322170, 2322172, 2322173, 2322174, 2322175, 2322177, 2322178, 2322180, 2322181, 2322186, 2322187, 2322190, 2322193, 2322195, 2322196, 2322197, 2322198, 2322201, 2322203, 2322208, 2322214, 2322215, 2322216, 2322220, 2322223, 2322228, 2322231, 2322232, 2322233, 2322234, 2322237, 2322245, 2322246, 2322248, 2322251, 2322255, 2322257, 2322260, 2322261, 2322262, 2322264, 2322266, 2322270, 2322274, 2322280, 2322283, 2322287, 2322292, 2322293, 2322294, 2322295, 2322296, 2322297, 2322299, 2322300, 2322302, 2322303, 2322307, 2322308, 2322309, 2322310, 2322311, 2322312, 2322313, 2322315, 2322319, 2322322, 2322323, 2322324, 2322325, 2322327, 2322328, 2322330, 2322332, 2322333, 2322338, 2322342, 2322343, 2322347, 2322348, 2322350, 2322354, 2322355, 2322358, 2322361, 2322363, 2322367, 2322370, 2322371, 2322372, 2322375, 2322376, 2322377, 2322379, 2322380, 2322382, 2322385, 2322386, 2322389, 2322390, 2322391, 2322392, 2322394, 2322395, 2322398, 2322401, 2322405, 2322406, 2322407, 2322408, 2322409, 2322411, 2322413, 2322414, 2322418, 2322421, 2322423, 2322424, 2322426, 2322427, 2322431, 2322434, 2322435, 2322436, 2322438, 2322441, 2322443, 2322444, 2322445, 2322447, 2322448, 2322449, 2322451, 2322455, 2322456, 2322457, 2322458, 2322459, 2322461, 2322462, 2322463, 2322465, 2322468, 2322469, 2322474, 2322479, 2322483, 2322485, 2322487, 2322488, 2322490, 2322492, 2322493, 2322495, 2322497, 2322498, 2322499, 2322503, 2322505, 2322506, 2322509, 2322511, 2322516, 2322519, 2322520, 2322524, 2322525, 2322529, 2322530, 2322536, 2322537, 2322538, 2322539, 2322540, 2322542, 2322544, 2322545, 2322546, 2322547, 2322548, 2322549, 2322550, 2322552, 2322553, 2322554, 2322555, 2322556, 2322557, 2322558, 2322559, 2322561, 2322563, 2322567, 2322568, 2322569, 2322570, 2322572, 2322573, 2322576, 2322577, 2322578, 2322579, 2322580, 2322583, 2322584, 2322585, 2322589, 2322590, 2322592, 2322596, 2322597, 2322598, 2322600, 2322601, 2322602, 2322603, 2322605, 2322606, 2322608, 2322609, 2322611, 2322612, 2322614, 2322615, 2322616, 2322619, 2322620, 2322622, 2322624, 2322625, 2322627, 2322630, 2322633, 2322635, 2322636, 2322637, 2322640, 2322641, 2322642, 2322646, 2322648, 2322649, 2322651, 2322652, 2322653, 2322654, 2322655, 2322659, 2322664, 2322665, 2322666, 2322669, 2322670, 2322672, 2322675, 2322677, 2322678, 2322679, 2322680, 2322682, 2322684, 2322686, 2322687, 2322689, 2322690, 2322693, 2322694, 2322696, 2322698, 2322699, 2322700, 2322702, 2322704, 2322705, 2322706, 2322709, 2322711, 2322712, 2322713, 2322715, 2322716, 2322717, 2322718, 2322720, 2322721, 2322722, 2322724, 2322726, 2322730, 2322731, 2322732, 2322734, 2322735, 2322737, 2322738, 2322740, 2322746, 2322749, 2322750, 2322752, 2322753, 2322754, 2322755, 2322756, 2322761, 2322762, 2322764, 2322765, 2322767, 2322769, 2322771, 2322772, 2322779, 2322781, 2322782, 2322783, 2322785, 2322789, 2322790, 2322792, 2322795, 2322796, 2322798, 2322801, 2322802, 2322803, 2322807, 2322809, 2322810, 2322811, 2322812, 2322813, 2322814, 2322815, 2322816, 2322817, 2322820, 2322822, 2322824, 2322827, 2322828, 2322829, 2322831, 2322832, 2322834, 2322836, 2322837, 2322840, 2322841, 2322842, 2322843, 2322844, 2322845, 2322846, 2322847, 2322848, 2322851, 2322853, 2322854, 2322855, 2322856, 2322857, 2322858, 2322860, 2322861, 2322862, 2322863, 2322864, 2322865, 2322870, 2322871, 2322873, 2322874, 2322875, 2322877, 2322879, 2322882, 2322883, 2322884, 2322885, 2322886, 2322887, 2322888, 2322889, 2322892, 2322893, 2322895, 2322896, 2322897, 2322898, 2322899, 2322900, 2322906, 2322912, 2322914, 2322915, 2322917, 2322918, 2322919, 2322921, 2322923, 2322924, 2322926, 2322927, 2322928, 2322929, 2322930, 2322932, 2322935, 2322938, 2322939, 2322940, 2322942, 2322943, 2322946, 2322947, 2322948, 2322949, 2322952, 2322953, 2322955, 2322959, 2322961, 2322963, 2322964, 2322966, 2322967, 2322968, 2322969, 2322970, 2322971, 2322972, 2322973, 2322975, 2322976, 2322977, 2322978, 2322980, 2322981, 2322982, 2322984, 2322985, 2322986, 2322987, 2322989, 2322990, 2322991, 2322992, 2322993, 2322994, 2322995, 2322997, 2322998, 2323003, 2323005, 2323006, 2323010, 2323011, 2323012, 2323013, 2323014, 2323016, 2323019, 2323020, 2323028, 2323029, 2323030, 2323031, 2323032, 2323033, 2323034, 2323037, 2323038, 2323042, 2323046, 2323048, 2323049, 2323050, 2323052, 2323055, 2323056, 2323057, 2323058, 2323059, 2323060, 2323061, 2323062, 2323063, 2323065, 2323067, 2323073, 2323075, 2323076, 2323077, 2323078, 2323080, 2323081, 2323083, 2323087, 2323091, 2323092, 2323093, 2323095, 2323096, 2323099, 2323100, 2323102, 2323110, 2323111, 2323117, 2323118, 2323120, 2323122, 2323123, 2323125, 2323126, 2323127, 2323129, 2323131, 2323133, 2323137, 2323139, 2323142, 2323145, 2323147, 2323148, 2323150, 2323153, 2323155, 2323156, 2323157, 2323158, 2323159, 2323164, 2323166, 2323170, 2323172, 2323173, 2323174, 2323176, 2323177, 2323180, 2323183, 2323185, 2323187, 2323188, 2323194, 2323196, 2323200, 2323201, 2323204, 2323205, 2323208, 2323209, 2323210, 2323211, 2323212, 2323214, 2323215, 2323218, 2323219, 2323220, 2323222, 2323225, 2323227, 2323228, 2323229, 2323230, 2323234, 2323235, 2323236, 2323240, 2323242, 2323243, 2323244, 2323251, 2323255, 2323256, 2323257, 2323258, 2323259, 2323260, 2323264, 2323265, 2323270, 2323272, 2323273, 2323275, 2323276, 2323277, 2323278, 2323279, 2323281, 2323282, 2323283, 2323287, 2323289, 2323292, 2323294, 2323295, 2323298, 2323300, 2323302, 2323303, 2323305, 2323308, 2323310, 2323314, 2323318, 2323319, 2323320, 2323321, 2323323, 2323325, 2323327, 2323328, 2323331, 2323334, 2323338, 2323340, 2323342, 2323344, 2323345, 2323346, 2323347, 2323348, 2323349, 2323355, 2323357, 2323358, 2323360, 2323361, 2323363, 2323365, 2323366, 2323367, 2323369, 2323370, 2323371, 2323373, 2323374, 2323375, 2323376, 2323377, 2323380, 2323381, 2323383, 2323385, 2323386, 2323387, 2323388, 2323389, 2323390, 2323392, 2323393, 2323396, 2323400, 2323402, 2323403, 2323404, 2323406, 2323409, 2323411, 2323413, 2323416, 2323420, 2323421, 2323426, 2323430, 2323431, 2323432, 2323434, 2323435, 2323436, 2323438, 2323439, 2323441, 2323442, 2323445, 2323452, 2323455, 2323457, 2323462, 2323466, 2323467, 2323470, 2323471, 2323472, 2323473, 2323474, 2323475, 2323478, 2323481, 2323485, 2323488, 2323489, 2323491, 2323493, 2323495, 2323496, 2323499, 2323501, 2323503, 2323504, 2323511, 2323514, 2323516, 2323517, 2323519, 2323520, 2323523, 2323527, 2323528, 2323536, 2323538, 2323540, 2323541, 2323544, 2323546, 2323547, 2323549, 2323550, 2323556, 2323557, 2323559, 2323560, 2323563, 2323565, 2323570, 2323573, 2323574, 2323578, 2323580, 2323581, 2323582, 2323583, 2323584, 2323585, 2323589, 2323593, 2323594, 2323595, 2323597, 2323598, 2323599, 2323602, 2323604, 2323605, 2323607, 2323608, 2323611, 2323613, 2323616, 2323617, 2323621, 2323622, 2323623, 2323624, 2323625, 2323626, 2323629, 2323632, 2323633, 2323634, 2323635, 2323636, 2323639, 2323644, 2323646, 2323648, 2323651, 2323656, 2323657, 2323659, 2323661, 2323662, 2323665, 2323668, 2323671, 2323672, 2323673, 2323674, 2323675, 2323676, 2323679, 2323682, 2323685, 2323687, 2323689, 2323691, 2323692, 2323695, 2323696, 2323697, 2323698, 2323704, 2323708, 2323713, 2323714, 2323715, 2323718, 2323719, 2323720, 2323721, 2323722, 2323723, 2323724, 2323725, 2323726, 2323727, 2323729, 2323730, 2323731, 2323732, 2323734, 2323738, 2323739, 2323742, 2323743, 2323745, 2323746, 2323751, 2323753, 2323756, 2323759, 2323760, 2323761, 2323762, 2323763, 2323764, 2323765, 2323769, 2323771, 2323772, 2323773, 2323778, 2323781, 2323782, 2323787, 2323789, 2323790, 2323791, 2323796, 2323797, 2323798, 2323799, 2323802, 2323804, 2323806, 2323809, 2323810, 2323811, 2323813, 2323815, 2323816, 2323817, 2323821, 2323822, 2323825, 2323826, 2323827, 2323828, 2323829, 2323831, 2323832, 2323836, 2323837, 2323840, 2323841, 2323842, 2323847, 2323848, 2323850, 2323851, 2323853, 2323854, 2323855, 2323858, 2323859, 2323860, 2323861, 2323863, 2323864, 2323865, 2323867, 2323868, 2323869, 2323871, 2323872, 2323875, 2323876, 2323880, 2323882, 2323883, 2323885, 2323886, 2323891, 2323893, 2323894, 2323895, 2323896, 2323897, 2323899, 2323900, 2323901, 2323902, 2323903, 2323904, 2323907, 2323908, 2323910, 2323911, 2323912, 2323913, 2323916, 2323918, 2323920, 2323924, 2323925, 2323927, 2323929, 2323930, 2323933, 2323935, 2323937, 2323938, 2323940, 2323941, 2323943, 2323945, 2323947, 2323948, 2323949, 2323951, 2323957, 2323959, 2323962, 2323964, 2323966, 2323968, 2323969, 2323971, 2323977, 2323979, 2323980, 2323981, 2323982, 2323984, 2323986, 2323988, 2323990, 2323991, 2323992, 2323996, 2323997, 2323998, 2323999, 2324000, 2324001, 2324002, 2324008, 2324012, 2324016, 2324018, 2324020, 2324023, 2324025, 2324027, 2324029, 2324031, 2324033, 2324035, 2324038, 2324039, 2324040, 2324041, 2324042, 2324043, 2324045, 2324046, 2324047, 2324049, 2324050, 2324051, 2324052, 2324057, 2324059, 2324060, 2324062, 2324064, 2324067, 2324069, 2324070, 2324072, 2324074, 2324076, 2324078, 2324082, 2324084, 2324085, 2324088, 2324090, 2324091, 2324092, 2324094, 2324097, 2324101, 2324104, 2324107, 2324110, 2324111, 2324112, 2324116, 2324117, 2324118, 2324119, 2324122, 2324124, 2324125, 2324126, 2324129, 2324131, 2324132, 2324135, 2324137, 2324140, 2324143, 2324144, 2324145, 2324150, 2324151, 2324154, 2324158, 2324159, 2324160, 2324161, 2324163, 2324165, 2324169, 2324171, 2324172, 2324177, 2324179, 2324181, 2324182, 2324183, 2324185, 2324189, 2324190, 2324192, 2324193, 2324194, 2324197, 2324199, 2324201, 2324204, 2324205, 2324206, 2324207, 2324208, 2324209, 2324214, 2324215, 2324217, 2324218, 2324219, 2324221, 2324223, 2324224, 2324225, 2324226, 2324228, 2324229, 2324232, 2324234, 2324237, 2324239, 2324240, 2324246, 2324247, 2324248, 2324251, 2324256, 2324258, 2324259, 2324262, 2324263, 2324265, 2324269, 2324270, 2324271, 2324272, 2324273, 2324274, 2324275, 2324276, 2324278, 2324279, 2324281, 2324282, 2324283, 2324285, 2324286, 2324287, 2324289, 2324290, 2324293, 2324294, 2324295, 2324296, 2324297, 2324299, 2324300, 2324301, 2324307, 2324308, 2324311, 2324314, 2324315, 2324316, 2324317, 2324318, 2324319, 2324320, 2324321, 2324322, 2324324, 2324325, 2324333, 2324338, 2324339, 2324340, 2324341, 2324343, 2324345, 2324347, 2324349, 2324350, 2324352, 2324353, 2324354, 2324355, 2324357, 2324358, 2324361, 2324362, 2324364, 2324365, 2324366, 2324370, 2324372, 2324376, 2324382, 2324383, 2324385, 2324387, 2324391, 2324392, 2324398, 2324399, 2324401, 2324404, 2324405, 2324407, 2324409, 2324410, 2324411, 2324414, 2324415, 2324416, 2324417, 2324418, 2324420, 2324421, 2324422, 2324423, 2324425, 2324427, 2324428, 2324429, 2324430, 2324431, 2324433, 2324435, 2324437, 2324440, 2324441, 2324442, 2324444, 2324446, 2324447, 2324451, 2324453, 2324456, 2324457, 2324461, 2324466, 2324468, 2324469, 2324471, 2324472, 2324473, 2324474, 2324475, 2324476, 2324477, 2324479, 2324484, 2324486, 2324487, 2324488, 2324489, 2324490, 2324492, 2324493, 2324494, 2324496, 2324500, 2324501, 2324502, 2324505, 2324506, 2324507, 2324511, 2324512, 2324513, 2324515, 2324516, 2324517, 2324520, 2324521, 2324522, 2324525, 2324527, 2324528, 2324529, 2324530, 2324531, 2324534, 2324540, 2324541, 2324542, 2324543, 2324548, 2324549, 2324551, 2324556, 2324559, 2324560, 2324561, 2324562, 2324564, 2324566, 2324568, 2324571, 2324572, 2324574, 2324575, 2324577, 2324579, 2324583, 2324584, 2324585, 2324586, 2324587, 2324590, 2324594, 2324597, 2324599, 2324600, 2324602, 2324604, 2324610, 2324615, 2324617, 2324619, 2324620, 2324623, 2324624, 2324628, 2324630, 2324631, 2324633, 2324634, 2324635, 2324637, 2324638, 2324639, 2324642, 2324645, 2324648, 2324650, 2324651, 2324652, 2324654, 2324655, 2324656, 2324659, 2324661, 2324663, 2324664, 2324665, 2324666, 2324669, 2324674, 2324676, 2324677, 2324681, 2324683, 2324684, 2324687, 2324688, 2324689, 2324691, 2324692, 2324693, 2324695, 2324696, 2324697, 2324702, 2324703, 2324704, 2324705, 2324706, 2324708, 2324710, 2324711, 2324712, 2324718, 2324719, 2324720, 2324721, 2324723, 2324726, 2324729, 2324730, 2324731, 2324732, 2324733, 2324735, 2324737, 2324740, 2324741, 2324746, 2324749, 2324751, 2324752, 2324753, 2324756, 2324757, 2324760, 2324762, 2324765, 2324766, 2324769, 2324770, 2324772, 2324776, 2324778, 2324783, 2324786, 2324789, 2324791, 2324798, 2324799, 2324803, 2324804, 2324807, 2324808, 2324811, 2324812, 2324818, 2324819, 2324822, 2324823, 2324825, 2324826, 2324827, 2324828, 2324833, 2324834, 2324836, 2324837, 2324838, 2324842, 2324843, 2324844, 2324845, 2324846, 2324848, 2324850, 2324852, 2324853, 2324854, 2324855, 2324856, 2324857, 2324859, 2324861, 2324862, 2324864, 2324866, 2324868, 2324869, 2324870, 2324871, 2324873, 2324876, 2324879, 2324880, 2324881, 2324883, 2324887, 2324888, 2324889, 2324892, 2324894, 2324896, 2324899, 2324901, 2324903, 2324904, 2324905, 2324907, 2324908, 2324909, 2324911, 2324913, 2324915, 2324916, 2324917, 2324918, 2324920, 2324923, 2324928, 2324929, 2324931, 2324934, 2324935, 2324936, 2324938, 2324940, 2324941, 2324946, 2324948, 2324952, 2324953, 2324955, 2324956, 2324958, 2324962, 2324963, 2324965, 2324966, 2324968, 2324969, 2324970, 2324974, 2324977, 2324984, 2324986, 2324987, 2324989, 2324992, 2324993, 2324994, 2324998, 2324999, 2325000, 2325001, 2325003, 2325005, 2325010, 2325012, 2325013, 2325014, 2325017, 2325018, 2325019, 2325021, 2325022, 2325027, 2325028, 2325030, 2325034, 2325035, 2325036, 2325040, 2325042, 2325043, 2325044, 2325045, 2325047, 2325048, 2325049, 2325050, 2325051, 2325052, 2325053, 2325054, 2325055, 2325059, 2325062, 2325063, 2325067, 2325078, 2325080, 2325082, 2325087, 2325088, 2325089, 2325090, 2325092, 2325094, 2325095, 2325096, 2325097, 2325098, 2325100, 2325102, 2325103, 2325106, 2325110, 2325115, 2325116, 2325118, 2325119, 2325120, 2325121, 2325122, 2325124, 2325125, 2325126, 2325129, 2325131, 2325133, 2325137, 2325138, 2325140, 2325142, 2325143, 2325145, 2325148, 2325149, 2325154, 2325155, 2325156, 2325157, 2325164, 2325168, 2325170, 2325175, 2325176, 2325177, 2325178, 2325180, 2325181, 2325183, 2325184, 2325186, 2325187, 2325188, 2325189, 2325190, 2325191, 2325193, 2325194, 2325195, 2325199, 2325201, 2325202, 2325203, 2325207, 2325209, 2325211, 2325213, 2325214, 2325215, 2325218, 2325219, 2325220, 2325221, 2325222, 2325229, 2325231, 2325232, 2325233, 2325234, 2325235, 2325236, 2325237, 2325238, 2325241, 2325242, 2325244, 2325245, 2325248, 2325249, 2325251, 2325252, 2325254, 2325255, 2325257, 2325259, 2325263, 2325267, 2325268, 2325269, 2325270, 2325271, 2325273, 2325276, 2325277, 2325279, 2325280, 2325286, 2325287, 2325288, 2325290, 2325291, 2325293, 2325296, 2325301, 2325306, 2325307, 2325310, 2325314, 2325315, 2325318, 2325319, 2325320, 2325321, 2325322, 2325324, 2325326, 2325327, 2325329, 2325330, 2325332, 2325334, 2325335, 2325339, 2325340, 2325342, 2325347, 2325349, 2325351, 2325352, 2325354, 2325355, 2325357, 2325358, 2325361, 2325364, 2325365, 2325367, 2325371, 2325379, 2325381, 2325382, 2325383, 2325384, 2325386, 2325389, 2325390, 2325392, 2325394, 2325397, 2325399, 2325403, 2325404, 2325405, 2325407, 2325408, 2325409, 2325413, 2325416, 2325418, 2325419, 2325422, 2325423, 2325424, 2325425, 2325426, 2325427, 2325429, 2325430, 2325431, 2325432, 2325433, 2325434, 2325437, 2325438, 2325439, 2325442, 2325443, 2325445, 2325451, 2325455, 2325456, 2325461, 2325462, 2325463, 2325465, 2325467, 2325470, 2325471, 2325473, 2325475, 2325480, 2325481, 2325483, 2325485, 2325486, 2325493, 2325494, 2325495, 2325496, 2325497, 2325498, 2325499, 2325502, 2325503, 2325504, 2325505, 2325506, 2325509, 2325510, 2325513, 2325515, 2325517, 2325519, 2325522, 2325523, 2325525, 2325534, 2325536, 2325537, 2325538, 2325541, 2325543, 2325544, 2325546, 2325549, 2325550, 2325551, 2325552, 2325553, 2325554, 2325557, 2325563, 2325564, 2325565, 2325567, 2325569, 2325572, 2325573, 2325574, 2325579, 2325581, 2325582, 2325584, 2325585, 2325586, 2325587, 2325589, 2325592, 2325595, 2325596, 2325599, 2325600, 2325601, 2325605, 2325608, 2325609, 2325610, 2325615, 2325618, 2325619, 2325620, 2325621, 2325622, 2325623, 2325626, 2325631, 2325632, 2325633, 2325634, 2325635, 2325636, 2325637, 2325640, 2325642, 2325643, 2325644, 2325645, 2325646, 2325650, 2325651, 2325652, 2325654, 2325656, 2325657, 2325667, 2325669, 2325670, 2325671, 2325673, 2325675, 2325676, 2325677, 2325678, 2325679, 2325682, 2325687, 2325689, 2325692, 2325694, 2325695, 2325696, 2325697, 2325703, 2325704, 2325705, 2325707, 2325709, 2325710, 2325712, 2325714, 2325718, 2325720, 2325722, 2325723, 2325724, 2325725, 2325726, 2325727, 2325728, 2325733, 2325735, 2325736, 2325737, 2325738, 2325739, 2325741, 2325742, 2325743, 2325744, 2325745, 2325748, 2325751, 2325752, 2325754, 2325756, 2325758, 2325759, 2325760, 2325762, 2325764, 2325765, 2325767, 2325770, 2325771, 2325775, 2325777, 2325778, 2325779, 2325780, 2325781, 2325782, 2325788, 2325789, 2325792, 2325793, 2325800, 2325801, 2325802, 2325805, 2325807, 2325811, 2325812, 2325813, 2325814, 2325818, 2325819, 2325822, 2325828, 2325830, 2325832, 2325833, 2325834, 2325835, 2325840, 2325841, 2325842, 2325844, 2325846, 2325847, 2325848, 2325850, 2325853, 2325854, 2325858, 2325860, 2325862, 2325864, 2325865, 2325866, 2325869, 2325870, 2325871, 2325872, 2325874, 2325875, 2325876, 2325880, 2325881, 2325882, 2325883, 2325884, 2325885, 2325886, 2325888, 2325894, 2325898, 2325899, 2325901, 2325903, 2325904, 2325906, 2325910, 2325911, 2325913, 2325917, 2325918, 2325920, 2325921, 2325922, 2325923, 2325925, 2325926, 2325928, 2325929, 2325930, 2325932, 2325934, 2325937, 2325939, 2325940, 2325941, 2325942, 2325948, 2325949, 2325951, 2325952, 2325953, 2325954, 2325955, 2325957, 2325959, 2325961, 2325963, 2325964, 2325965, 2325969, 2325970, 2325971, 2325972, 2325973, 2325975, 2325979, 2325983, 2325984, 2325985, 2325988, 2325991, 2325992, 2325994, 2325996, 2325997, 2325998, 2325999, 2326000, 2326002, 2326005, 2326006, 2326007, 2326008, 2326012, 2326013, 2326014, 2326016, 2326018, 2326020, 2326024, 2326025, 2326026, 2326027, 2326030, 2326033, 2326034, 2326035, 2326036, 2326038, 2326039, 2326041, 2326043, 2326045, 2326046, 2326048, 2326050, 2326056, 2326057, 2326058, 2326059, 2326060, 2326061, 2326063, 2326067, 2326071, 2326075, 2326076, 2326077, 2326078, 2326081, 2326082, 2326083, 2326086, 2326090, 2326091, 2326092, 2326094, 2326095, 2326098, 2326100, 2326105, 2326106, 2326107, 2326110, 2326112, 2326113, 2326114, 2326115, 2326116, 2326118, 2326119, 2326121, 2326122, 2326123, 2326127, 2326129, 2326134, 2326135, 2326137, 2326138, 2326143, 2326144, 2326145, 2326149, 2326150, 2326152, 2326153, 2326154, 2326157, 2326158, 2326160, 2326161, 2326167, 2326168, 2326169, 2326170, 2326171, 2326175, 2326177, 2326178, 2326179, 2326181, 2326182, 2326183, 2326184, 2326185, 2326186, 2326188, 2326189, 2326190, 2326192, 2326193, 2326194, 2326196, 2326198, 2326199, 2326200, 2326202, 2326204, 2326206, 2326208, 2326211, 2326212, 2326213, 2326219, 2326220, 2326221, 2326222, 2326223, 2326224, 2326225, 2326228, 2326229, 2326230, 2326231, 2326233, 2326235, 2326237, 2326238, 2326239, 2326240, 2326249, 2326250, 2326252, 2326253, 2326255, 2326257, 2326258, 2326259, 2326260, 2326262, 2326264, 2326265, 2326266, 2326267, 2326268, 2326269, 2326270, 2326272, 2326274, 2326276, 2326278, 2326282, 2326286, 2326287, 2326288, 2326290, 2326295, 2326299, 2326301, 2326304, 2326307, 2326309, 2326310, 2326311, 2326312, 2326314, 2326316, 2326317, 2326318, 2326319, 2326320, 2326322, 2326323, 2326324, 2326329, 2326330, 2326331, 2326333, 2326335, 2326336, 2326338, 2326344, 2326345, 2326346, 2326348, 2326351, 2326361, 2326364, 2326366, 2326368, 2326370, 2326376, 2326377, 2326378, 2326379, 2326380, 2326383, 2326386, 2326387, 2326390, 2326392, 2326394, 2326396, 2326397, 2326398, 2326399, 2326400, 2326401, 2326402, 2326403, 2326405, 2326406, 2326407, 2326408, 2326409, 2326410, 2326411, 2326412, 2326413, 2326414, 2326415, 2326417, 2326418, 2326419, 2326420, 2326422, 2326423, 2326424, 2326425, 2326426, 2326427, 2326428, 2326429, 2326434, 2326436, 2326437, 2326438, 2326441, 2326442, 2326444, 2326445, 2326446, 2326447, 2326450, 2326453, 2326454, 2326455, 2326457, 2326458, 2326460, 2326461, 2326462, 2326463, 2326464, 2326465, 2326466, 2326469, 2326471, 2326474, 2326475, 2326477, 2326478, 2326480, 2326482, 2326483, 2326485, 2326487, 2326492, 2326493, 2326495, 2326497, 2326499, 2326500, 2326501, 2326502, 2326506, 2326507, 2326509, 2326510, 2326511, 2326513, 2326516, 2326519, 2326521, 2326522, 2326524, 2326525, 2326528, 2326535, 2326538, 2326539, 2326540, 2326541, 2326543, 2326545, 2326546, 2326547, 2326549, 2326550, 2326551, 2326552, 2326554, 2326558, 2326560, 2326561, 2326562, 2326564, 2326565, 2326566, 2326567, 2326570, 2326571, 2326573, 2326574, 2326575, 2326576, 2326577, 2326580, 2326582, 2326583, 2326587, 2326588, 2326590, 2326593, 2326597, 2326599, 2326602, 2326607, 2326608, 2326610, 2326611, 2326613, 2326616, 2326617, 2326619, 2326620, 2326621, 2326622, 2326625, 2326626, 2326627, 2326629, 2326630, 2326635, 2326637, 2326640, 2326641, 2326642, 2326644, 2326646, 2326647, 2326648, 2326649, 2326650, 2326652, 2326655, 2326657, 2326658, 2326660, 2326661, 2326665, 2326666, 2326667, 2326669, 2326671, 2326673, 2326677, 2326679, 2326680, 2326682, 2326683, 2326686, 2326689, 2326690, 2326691, 2326693, 2326694, 2326698, 2326699, 2326700, 2326701, 2326704, 2326705, 2326709, 2326711, 2326712, 2326713, 2326714, 2326717, 2326720, 2326721, 2326723, 2326725, 2326728, 2326729, 2326730, 2326731, 2326732, 2326734, 2326739, 2326742, 2326743, 2326744, 2326745, 2326746, 2326747, 2326750, 2326753, 2326754, 2326755, 2326756, 2326759, 2326763, 2326765, 2326770, 2326772, 2326773, 2326774, 2326775, 2326776, 2326777, 2326779, 2326780, 2326781, 2326782, 2326783, 2326788, 2326789, 2326791, 2326792, 2326793, 2326795, 2326798, 2326800, 2326803, 2326804, 2326805, 2326806, 2326807, 2326808, 2326809, 2326816, 2326818, 2326819, 2326821, 2326827, 2326828, 2326830, 2326831, 2326833, 2326834, 2326835, 2326837, 2326838, 2326839, 2326841, 2326842, 2326846, 2326847, 2326851, 2326852, 2326855, 2326856, 2326857, 2326860, 2326861, 2326862, 2326863, 2326864, 2326865, 2326866, 2326867, 2326871, 2326872, 2326874, 2326875, 2326877, 2326878, 2326879, 2326880, 2326881, 2326882, 2326883, 2326884, 2326886, 2326887, 2326888, 2326889, 2326893, 2326896, 2326898, 2326900, 2326902, 2326904, 2326906, 2326908, 2326909, 2326910, 2326911, 2326912, 2326915, 2326917, 2326918, 2326921, 2326922, 2326924, 2326926, 2326930, 2326933, 2326934, 2326937, 2326938, 2326939, 2326941, 2326942, 2326946, 2326949, 2326954, 2326955, 2326957, 2326958, 2326960, 2326962, 2326965, 2326966, 2326967, 2326970, 2326974, 2326975, 2326976, 2326978, 2326980, 2326981, 2326982, 2326983, 2326984, 2326985, 2326986, 2326987, 2326988, 2326989, 2326991, 2326993, 2326995, 2326996, 2326999, 2327002, 2327003, 2327004, 2327006, 2327009, 2327011, 2327012, 2327015, 2327016, 2327017, 2327020, 2327028, 2327029, 2327031, 2327032, 2327033, 2327034, 2327035, 2327037, 2327041, 2327042, 2327047, 2327050, 2327053, 2327054, 2327059, 2327060, 2327061, 2327062, 2327064, 2327065, 2327067, 2327069, 2327070, 2327071, 2327074, 2327076, 2327077, 2327078, 2327079, 2327080, 2327082, 2327083, 2327085, 2327086, 2327090, 2327092, 2327094, 2327095, 2327096, 2327098, 2327100, 2327101, 2327103, 2327104, 2327106, 2327110, 2327111, 2327112, 2327114, 2327115, 2327116, 2327117, 2327118, 2327120, 2327122, 2327123, 2327124, 2327126, 2327127, 2327129, 2327131, 2327132, 2327135, 2327138, 2327140, 2327143, 2327145, 2327147, 2327149, 2327150, 2327153, 2327154, 2327157, 2327159, 2327163, 2327164, 2327167, 2327168, 2327177, 2327179, 2327181, 2327182, 2327184, 2327186, 2327187, 2327188, 2327189, 2327190, 2327191, 2327192, 2327193, 2327194, 2327195, 2327196, 2327198, 2327199, 2327201, 2327202, 2327206, 2327207, 2327210, 2327211, 2327213, 2327216, 2327217, 2327218, 2327219, 2327220, 2327221, 2327224, 2327226, 2327227, 2327229, 2327230, 2327231, 2327232, 2327233, 2327234, 2327240, 2327241, 2327242, 2327247, 2327248, 2327252, 2327254, 2327256, 2327258, 2327259, 2327260, 2327271, 2327274, 2327275, 2327276, 2327277, 2327280, 2327281, 2327282, 2327283, 2327284, 2327285, 2327286, 2327288, 2327291, 2327294, 2327297, 2327299, 2327301, 2327305, 2327306, 2327309, 2327310, 2327312, 2327313, 2327314, 2327317, 2327318, 2327319, 2327320, 2327321, 2327323, 2327324, 2327325, 2327327, 2327328, 2327332, 2327333, 2327338, 2327339, 2327343, 2327347, 2327351, 2327353, 2327354, 2327356, 2327357, 2327360, 2327363, 2327364, 2327365, 2327366, 2327367, 2327370, 2327372, 2327376, 2327378, 2327379, 2327381, 2327386, 2327388, 2327389, 2327390, 2327391, 2327392, 2327394, 2327395, 2327397, 2327398, 2327399, 2327401, 2327402, 2327404, 2327406, 2327408, 2327410, 2327411, 2327412, 2327416, 2327417, 2327418, 2327420, 2327421, 2327425, 2327426, 2327427, 2327428, 2327429, 2327430, 2327433, 2327434, 2327436, 2327439, 2327440, 2327442, 2327444, 2327445, 2327448, 2327449, 2327454, 2327455, 2327456, 2327459, 2327460, 2327461, 2327462, 2327463, 2327465, 2327472, 2327473, 2327475, 2327476, 2327477, 2327478, 2327479, 2327480, 2327482, 2327483, 2327484, 2327488, 2327490, 2327491, 2327494, 2327495, 2327496, 2327497, 2327498, 2327499, 2327501, 2327502, 2327503, 2327505, 2327508, 2327510, 2327513, 2327514, 2327516, 2327517, 2327518, 2327519, 2327521, 2327523, 2327524, 2327525, 2327528, 2327530, 2327532, 2327533, 2327536, 2327543, 2327544, 2327549, 2327550, 2327552, 2327554, 2327555, 2327558, 2327560, 2327563, 2327565, 2327566, 2327567, 2327568, 2327570, 2327573, 2327574, 2327576, 2327577, 2327579, 2327582, 2327587, 2327588, 2327591, 2327592, 2327595, 2327597, 2327598, 2327599, 2327600, 2327602, 2327604, 2327605, 2327608, 2327610, 2327611, 2327613, 2327614, 2327618, 2327620, 2327623, 2327624, 2327628, 2327629, 2327632, 2327633, 2327640, 2327642, 2327643, 2327645, 2327646, 2327648, 2327649, 2327650, 2327651, 2327654, 2327656, 2327657, 2327658, 2327659, 2327661, 2327663, 2327667, 2327669, 2327672, 2327673, 2327674, 2327675, 2327677, 2327679, 2327680, 2327681, 2327682, 2327684, 2327686, 2327689, 2327691, 2327692, 2327693, 2327694, 2327696, 2327704, 2327705, 2327708, 2327709, 2327711, 2327714, 2327716, 2327717, 2327718, 2327721, 2327722, 2327723, 2327724, 2327725, 2327728, 2327729, 2327730, 2327732, 2327733, 2327734, 2327735, 2327736, 2327737, 2327739, 2327743, 2327745, 2327746, 2327750, 2327751, 2327752, 2327753, 2327755, 2327758, 2327760, 2327761, 2327766, 2327768, 2327769, 2327770, 2327771, 2327772, 2327773, 2327774, 2327775, 2327776, 2327781, 2327782, 2327783, 2327787, 2327789, 2327791, 2327792, 2327794, 2327795, 2327799, 2327800, 2327803, 2327804, 2327805, 2327806, 2327807, 2327808, 2327811, 2327812, 2327813, 2327816, 2327817, 2327818, 2327819, 2327821, 2327822, 2327824, 2327828, 2327829, 2327832, 2327834, 2327836, 2327837, 2327839, 2327840, 2327841, 2327844, 2327846, 2327848, 2327850, 2327851, 2327854, 2327855, 2327857, 2327860, 2327865, 2327866, 2327868, 2327870, 2327872, 2327873, 2327874, 2327875, 2327876, 2327877, 2327878, 2327879, 2327880, 2327883, 2327884, 2327885, 2327889, 2327890, 2327894, 2327895, 2327896, 2327897, 2327899, 2327903, 2327904, 2327908, 2327909, 2327910, 2327911, 2327916, 2327917, 2327918, 2327920, 2327921, 2327922, 2327924, 2327925, 2327927, 2327928, 2327930, 2327931, 2327932, 2327935, 2327936, 2327941, 2327942, 2327944, 2327945, 2327946, 2327947, 2327950, 2327951, 2327956, 2327960, 2327961, 2327962, 2327965, 2327966, 2327970, 2327973, 2327974, 2327975, 2327977, 2327978, 2327980, 2327985, 2327986, 2327987, 2327989, 2327990, 2327991, 2327992, 2327993, 2327994, 2327998, 2327999, 2328000, 2328002, 2328006, 2328008, 2328010, 2328011, 2328013, 2328018, 2328019, 2328020, 2328021, 2328025, 2328028, 2328030, 2328032, 2328033, 2328034, 2328035, 2328037, 2328039, 2328040, 2328042, 2328043, 2328044, 2328045, 2328046, 2328050, 2328051, 2328054, 2328056, 2328058, 2328060, 2328061, 2328062, 2328063, 2328067, 2328068, 2328069, 2328070, 2328075, 2328079, 2328083, 2328085, 2328088, 2328091, 2328093, 2328094, 2328097, 2328098, 2328100, 2328101, 2328103, 2328104, 2328105, 2328107, 2328111, 2328113, 2328114, 2328115, 2328116, 2328117, 2328119, 2328120, 2328121, 2328126, 2328128, 2328131, 2328133, 2328135, 2328138, 2328140, 2328141, 2328144, 2328147, 2328149, 2328150, 2328153, 2328154, 2328158, 2328164, 2328165, 2328166, 2328167, 2328168, 2328170, 2328171, 2328172, 2328176, 2328178, 2328182, 2328183, 2328184, 2328188, 2328190, 2328191, 2328192, 2328193, 2328194, 2328196, 2328197, 2328198, 2328199, 2328200, 2328202, 2328204, 2328205, 2328206, 2328207, 2328209, 2328210, 2328211, 2328213, 2328214, 2328217, 2328218, 2328220, 2328222, 2328224, 2328225, 2328227, 2328229, 2328230, 2328232, 2328237, 2328238, 2328241, 2328242, 2328243, 2328244, 2328245, 2328247, 2328248, 2328256, 2328257, 2328258, 2328260, 2328261, 2328265, 2328266, 2328267, 2328268, 2328270, 2328271, 2328272, 2328275, 2328276, 2328278, 2328279, 2328282, 2328283, 2328284, 2328289, 2328290, 2328292, 2328293, 2328294, 2328295, 2328298, 2328299, 2328301, 2328305, 2328310, 2328311, 2328314, 2328319, 2328320, 2328321, 2328322, 2328323, 2328324, 2328325, 2328327, 2328328, 2328329, 2328330, 2328331, 2328333, 2328334, 2328335, 2328336, 2328340, 2328341, 2328342, 2328344, 2328350, 2328351, 2328353, 2328354, 2328355, 2328361, 2328364, 2328365, 2328367, 2328368, 2328369, 2328371, 2328373, 2328375, 2328378, 2328379, 2328380, 2328381, 2328382, 2328385, 2328390, 2328391, 2328393, 2328394, 2328395, 2328397, 2328400, 2328401, 2328402, 2328403, 2328406, 2328407, 2328408, 2328412, 2328416, 2328417, 2328420, 2328422, 2328423, 2328424, 2328426, 2328428, 2328430, 2328433, 2328434, 2328438, 2328440, 2328442, 2328444, 2328447, 2328451, 2328453, 2328454, 2328456, 2328457, 2328461, 2328462, 2328465, 2328468, 2328473, 2328475, 2328477, 2328480, 2328481, 2328484, 2328485, 2328487, 2328488, 2328491, 2328495, 2328497, 2328498, 2328501, 2328502, 2328503, 2328504, 2328506, 2328507, 2328509, 2328510, 2328512, 2328513, 2328515, 2328517, 2328519, 2328525, 2328526, 2328529, 2328530, 2328531, 2328535, 2328539, 2328540, 2328541, 2328542, 2328543, 2328544, 2328545, 2328550, 2328551, 2328552, 2328557, 2328558, 2328560, 2328563, 2328564, 2328565, 2328567, 2328573, 2328576, 2328577, 2328581, 2328583, 2328586, 2328588, 2328589, 2328590, 2328594, 2328595, 2328598, 2328599, 2328600, 2328601, 2328602, 2328607, 2328608, 2328609, 2328614, 2328616, 2328618, 2328621, 2328624, 2328625, 2328628, 2328632, 2328633, 2328634, 2328635, 2328637, 2328639, 2328640, 2328641, 2328644, 2328645, 2328647, 2328649, 2328654, 2328656, 2328657, 2328658, 2328659, 2328660, 2328661, 2328662, 2328663, 2328665, 2328666, 2328669, 2328674, 2328678, 2328681, 2328682, 2328683, 2328684, 2328685, 2328687, 2328688, 2328691, 2328693, 2328695, 2328697, 2328700, 2328701, 2328704, 2328708, 2328711, 2328712, 2328715, 2328716, 2328718, 2328719, 2328723, 2328724, 2328725, 2328726, 2328727, 2328728, 2328731, 2328732, 2328733, 2328734, 2328735, 2328736, 2328738, 2328740, 2328741, 2328743, 2328746, 2328749, 2328750, 2328754, 2328758, 2328759, 2328760, 2328761, 2328762, 2328763, 2328764, 2328769, 2328770, 2328773, 2328776, 2328779, 2328781, 2328784, 2328786, 2328787, 2328789, 2328790, 2328792, 2328795, 2328796, 2328797, 2328798, 2328802, 2328804, 2328806, 2328807, 2328809, 2328815, 2328822, 2328824, 2328825, 2328826, 2328827, 2328828, 2328829, 2328831, 2328834, 2328839, 2328840, 2328841, 2328844, 2328845, 2328847, 2328848, 2328849, 2328850, 2328851, 2328854, 2328862, 2328863, 2328864, 2328865, 2328867, 2328869, 2328870, 2328871, 2328876, 2328878, 2328880, 2328882, 2328883, 2328884, 2328891, 2328892, 2328894, 2328896, 2328897, 2328899, 2328900, 2328902, 2328903, 2328904, 2328905, 2328906, 2328908, 2328909, 2328912, 2328914, 2328918, 2328920, 2328921, 2328924, 2328925, 2328929, 2328932, 2328934, 2328935, 2328937, 2328940, 2328942, 2328943, 2328944, 2328947, 2328948, 2328949, 2328950, 2328953, 2328954, 2328956, 2328957, 2328958, 2328960, 2328961, 2328963, 2328966, 2328968, 2328969, 2328971, 2328976, 2328977, 2328978, 2328981, 2328982, 2328986, 2328987, 2328988, 2328989, 2328991, 2328994, 2328995, 2328996, 2328998, 2329000, 2329001, 2329004, 2329006, 2329007, 2329008, 2329009, 2329011, 2329012, 2329014, 2329015, 2329016, 2329017, 2329020, 2329021, 2329023, 2329027, 2329028, 2329030, 2329032, 2329035, 2329038, 2329039, 2329042, 2329045, 2329048, 2329053, 2329054, 2329058, 2329059, 2329060, 2329061, 2329062, 2329063, 2329066, 2329070, 2329073, 2329074, 2329075, 2329078, 2329079, 2329080, 2329082, 2329083, 2329084, 2329086, 2329089, 2329091, 2329092, 2329093, 2329096, 2329099, 2329100, 2329101, 2329107, 2329109, 2329110, 2329114, 2329115, 2329117, 2329118, 2329120, 2329122, 2329123, 2329125, 2329126, 2329127, 2329128, 2329129, 2329132, 2329133, 2329134, 2329140, 2329142, 2329151, 2329152, 2329153, 2329155, 2329156, 2329157, 2329158, 2329159, 2329160, 2329161, 2329163, 2329165, 2329166, 2329168, 2329171, 2329175, 2329178, 2329179, 2329181, 2329182, 2329187, 2329188, 2329189, 2329192, 2329193, 2329194, 2329195, 2329198, 2329201, 2329203, 2329204, 2329208, 2329210, 2329211, 2329213, 2329214, 2329218, 2329219, 2329221, 2329222, 2329225, 2329226, 2329228, 2329233, 2329234, 2329235, 2329237, 2329241, 2329242, 2329243, 2329244, 2329245, 2329246, 2329247, 2329249, 2329250, 2329252, 2329253, 2329256, 2329257, 2329259, 2329260, 2329261, 2329262, 2329265, 2329266, 2329267, 2329268, 2329270, 2329271, 2329272, 2329273, 2329275, 2329276, 2329277, 2329278, 2329279, 2329282, 2329283, 2329286, 2329287, 2329288, 2329290, 2329292, 2329295, 2329296, 2329297, 2329298, 2329299, 2329302, 2329304, 2329305, 2329306, 2329307, 2329308, 2329309, 2329310, 2329311, 2329312, 2329313, 2329316, 2329317, 2329318, 2329319, 2329320, 2329324, 2329327, 2329328, 2329331, 2329332, 2329334, 2329338, 2329340, 2329341, 2329342, 2329343, 2329344, 2329351, 2329353, 2329354, 2329356, 2329358, 2329359, 2329361, 2329362, 2329366, 2329367, 2329369, 2329371, 2329374, 2329377, 2329380, 2329381, 2329382, 2329384, 2329386, 2329387, 2329388, 2329389, 2329390, 2329391, 2329392, 2329394, 2329395, 2329398, 2329399, 2329400, 2329401, 2329404, 2329405, 2329407, 2329411, 2329414, 2329416, 2329417, 2329419, 2329420, 2329421, 2329426, 2329428, 2329429, 2329430, 2329431, 2329432, 2329433, 2329434, 2329435, 2329437, 2329441, 2329443, 2329444, 2329445, 2329446, 2329449, 2329450, 2329453, 2329456, 2329458, 2329461, 2329462, 2329464, 2329465, 2329466, 2329469, 2329470, 2329471, 2329472, 2329473, 2329474, 2329476, 2329480, 2329482, 2329484, 2329486, 2329487, 2329490, 2329492, 2329495, 2329496, 2329501, 2329502, 2329503, 2329505, 2329506, 2329507, 2329508, 2329513, 2329514, 2329516, 2329517, 2329521, 2329528, 2329529, 2329533, 2329534, 2329536, 2329537, 2329538, 2329539, 2329540, 2329542, 2329543, 2329545, 2329546, 2329548, 2329551, 2329553, 2329555, 2329556, 2329557, 2329558, 2329562, 2329563, 2329564, 2329565, 2329566, 2329568, 2329569, 2329570, 2329571, 2329572, 2329573, 2329575, 2329576, 2329581, 2329582, 2329583, 2329587, 2329589, 2329595, 2329596, 2329597, 2329598, 2329599, 2329600, 2329601, 2329607, 2329608, 2329609, 2329610, 2329614, 2329616, 2329620, 2329621, 2329623, 2329624, 2329625, 2329626, 2329630, 2329631, 2329633, 2329635, 2329637, 2329638, 2329641, 2329642, 2329645, 2329647, 2329648, 2329650, 2329652, 2329653, 2329655, 2329656, 2329657, 2329658, 2329659, 2329661, 2329663, 2329664, 2329668, 2329671, 2329674, 2329675, 2329676, 2329677, 2329678, 2329679, 2329680, 2329681, 2329682, 2329683, 2329685, 2329686, 2329687, 2329688, 2329691, 2329693, 2329695, 2329697, 2329698, 2329699, 2329702, 2329703, 2329706, 2329707, 2329708, 2329709, 2329711, 2329712, 2329713, 2329714, 2329716, 2329717, 2329720, 2329721, 2329725, 2329726, 2329728, 2329731, 2329732, 2329736, 2329739, 2329742, 2329743, 2329746, 2329748, 2329750, 2329754, 2329755, 2329758, 2329761, 2329762, 2329763, 2329764, 2329766, 2329767, 2329769, 2329770, 2329771, 2329772, 2329773, 2329775, 2329776, 2329779, 2329780, 2329781, 2329782, 2329784, 2329785, 2329786, 2329794, 2329795, 2329797, 2329804, 2329805, 2329806, 2329807, 2329808, 2329810, 2329811, 2329812, 2329813, 2329815, 2329816, 2329819, 2329820, 2329821, 2329823, 2329824, 2329825, 2329828, 2329829, 2329832, 2329833, 2329834, 2329837, 2329838, 2329839, 2329842, 2329843, 2329844, 2329846, 2329849, 2329850, 2329851, 2329852, 2329853, 2329854, 2329856, 2329857, 2329862, 2329866, 2329869, 2329870, 2329871, 2329872, 2329873, 2329874, 2329875, 2329878, 2329879, 2329880, 2329881, 2329885, 2329887, 2329888, 2329889, 2329890, 2329892, 2329893, 2329894, 2329895, 2329896, 2329897, 2329898, 2329899, 2329900, 2329901, 2329902, 2329903, 2329904, 2329906, 2329907, 2329908, 2329910, 2329911, 2329914, 2329915, 2329918, 2329920, 2329921, 2329924, 2329925, 2329931, 2329933, 2329935, 2329936, 2329937, 2329939, 2329943, 2329945, 2329946, 2329947, 2329948, 2329949, 2329955, 2329957, 2329959, 2329962, 2329963, 2329964, 2329965, 2329966, 2329968, 2329974, 2329976, 2329977, 2329978, 2329980, 2329981, 2329982, 2329983, 2329984, 2329985, 2329986, 2329987, 2329989, 2329990, 2329991, 2329993, 2329994, 2329995, 2329996, 2329999, 2330000, 2330001, 2330003, 2330006, 2330009, 2330011, 2330014, 2330015, 2330016, 2330019, 2330021, 2330023, 2330024, 2330025, 2330026, 2330027, 2330029, 2330030, 2330032, 2330033, 2330040, 2330042, 2330044, 2330048, 2330049, 2330051, 2330052, 2330054, 2330056, 2330059, 2330061, 2330062, 2330063, 2330065, 2330066, 2330067, 2330068, 2330069, 2330070, 2330071, 2330072, 2330073, 2330075, 2330076, 2330077, 2330080, 2330082, 2330084, 2330086, 2330087, 2330089, 2330090, 2330091, 2330093, 2330094, 2330095, 2330096, 2330098, 2330099, 2330100, 2330102, 2330105, 2330106, 2330107, 2330109, 2330112, 2330113, 2330114, 2330115, 2330116, 2330118, 2330119, 2330120, 2330121, 2330123, 2330124, 2330125, 2330126, 2330127, 2330128, 2330130, 2330131, 2330132, 2330133, 2330134, 2330136, 2330138, 2330144, 2330145, 2330146, 2330149, 2330150, 2330153, 2330155, 2330156, 2330157, 2330158, 2330160, 2330161, 2330163, 2330164, 2330167, 2330168, 2330171, 2330172, 2330174, 2330175, 2330176, 2330177, 2330178, 2330179, 2330180, 2330181, 2330183, 2330185, 2330187, 2330189, 2330190, 2330191, 2330196, 2330198, 2330199, 2330200, 2330202, 2330204, 2330206, 2330207, 2330208, 2330212, 2330213, 2330217, 2330219, 2330220, 2330221, 2330222, 2330223, 2330225, 2330226, 2330228, 2330229, 2330230, 2330231, 2330234, 2330235, 2330236, 2330238, 2330239, 2330242, 2330244, 2330245, 2330246, 2330248, 2330251, 2330252, 2330253, 2330255, 2330257, 2330262, 2330263, 2330264, 2330266, 2330268, 2330276, 2330277, 2330278, 2330280, 2330281, 2330282, 2330284, 2330288, 2330290, 2330291, 2330292, 2330293, 2330295, 2330296, 2330297, 2330299, 2330300, 2330301, 2330302, 2330303, 2330308, 2330312, 2330313, 2330316, 2330317, 2330319, 2330321, 2330322, 2330325, 2330326, 2330328, 2330331, 2330332, 2330333, 2330334, 2330335, 2330338, 2330339, 2330340, 2330343, 2330346, 2330347, 2330348, 2330353, 2330357, 2330361, 2330362, 2330363, 2330365, 2330367, 2330370, 2330371, 2330372, 2330374, 2330377, 2330378, 2330379, 2330380, 2330383, 2330384, 2330386, 2330387, 2330388, 2330389, 2330391, 2330393, 2330395, 2330396, 2330398, 2330399, 2330400, 2330403, 2330405, 2330406, 2330408, 2330410, 2330413, 2330414, 2330415, 2330417, 2330418, 2330419, 2330420, 2330421, 2330424, 2330425, 2330428, 2330430, 2330431, 2330432, 2330434, 2330435, 2330438, 2330439, 2330444, 2330445, 2330448, 2330449, 2330451, 2330452, 2330453, 2330454, 2330458, 2330460, 2330461, 2330462, 2330463, 2330464, 2330466, 2330467, 2330468, 2330469, 2330471, 2330472, 2330473, 2330474, 2330475, 2330476, 2330479, 2330480, 2330483, 2330485, 2330488, 2330491, 2330493, 2330497, 2330499, 2330501, 2330502, 2330503, 2330504, 2330507, 2330508, 2330511, 2330513, 2330514, 2330515, 2330518, 2330520, 2330522, 2330523, 2330524, 2330526, 2330528, 2330529, 2330530, 2330532, 2330536, 2330537, 2330541, 2330542, 2330545, 2330547, 2330548, 2330550, 2330551, 2330555, 2330559, 2330560, 2330561, 2330562, 2330568, 2330570, 2330573, 2330575, 2330576, 2330579, 2330580, 2330582, 2330583, 2330584, 2330585, 2330586, 2330587, 2330588, 2330589, 2330590, 2330591, 2330593, 2330594, 2330597, 2330599, 2330605, 2330607, 2330608, 2330610, 2330611, 2330613, 2330616, 2330620, 2330621, 2330626, 2330628, 2330629, 2330631, 2330634, 2330637, 2330639, 2330643, 2330648, 2330649, 2330652, 2330653, 2330658, 2330659, 2330661, 2330663, 2330666, 2330668, 2330675, 2330676, 2330679, 2330681, 2330682, 2330684, 2330685, 2330686, 2330688, 2330689, 2330690, 2330692, 2330696, 2330700, 2330703, 2330704, 2330705, 2330708, 2330709, 2330710, 2330717, 2330719, 2330721, 2330722, 2330724, 2330726, 2330729, 2330730, 2330731, 2330733, 2330735, 2330737, 2330738, 2330741, 2330742, 2330744, 2330746, 2330748, 2330749, 2330750, 2330754, 2330756, 2330758, 2330759, 2330760, 2330761, 2330763, 2330764, 2330765, 2330769, 2330770, 2330771, 2330772, 2330774, 2330777, 2330778, 2330779, 2330780, 2330781, 2330782, 2330783, 2330784, 2330785, 2330788, 2330789, 2330792, 2330793, 2330794, 2330795, 2330796, 2330798, 2330800, 2330802, 2330805, 2330809, 2330811, 2330813, 2330814, 2330816, 2330819, 2330820, 2330823, 2330824, 2330825, 2330827, 2330828, 2330829, 2330830, 2330832, 2330833, 2330835, 2330836, 2330837, 2330838, 2330839, 2330841, 2330842, 2330845, 2330846, 2330847, 2330848, 2330849, 2330851, 2330852, 2330853, 2330856, 2330857, 2330858, 2330859, 2330860, 2330861, 2330863, 2330864, 2330866, 2330870, 2330872, 2330875, 2330880, 2330881, 2330884, 2330886, 2330888, 2330889, 2330890, 2330891, 2330896, 2330898, 2330900, 2330904, 2330907, 2330909, 2330916, 2330920, 2330921, 2330922, 2330925, 2330927, 2330929, 2330930, 2330931, 2330934, 2330936, 2330937, 2330939, 2330940, 2330942, 2330943, 2330944, 2330945, 2330946, 2330947, 2330949, 2330950, 2330951, 2330952, 2330954, 2330956, 2330957, 2330958, 2330960, 2330961, 2330962, 2330964, 2330965, 2330966, 2330967, 2330969, 2330970, 2330971, 2330972, 2330974, 2330975, 2330976, 2330977, 2330981, 2330982, 2330983, 2330984, 2330986, 2330987, 2330990, 2330991, 2330993, 2330994, 2330995, 2330997, 2330998, 2331000, 2331002, 2331003, 2331004, 2331005, 2331008, 2331009, 2331012, 2331014, 2331015, 2331018, 2331020, 2331021, 2331022, 2331023, 2331024, 2331025, 2331026, 2331027, 2331028, 2331029, 2331031, 2331034, 2331035, 2331037, 2331038, 2331041, 2331042, 2331043, 2331044, 2331045, 2331046, 2331047, 2331048, 2331049, 2331050, 2331052, 2331054, 2331056, 2331058, 2331060, 2331064, 2331065, 2331066, 2331068, 2331069, 2331070, 2331072, 2331076, 2331084, 2331087, 2331088, 2331089, 2331090, 2331092, 2331093, 2331095, 2331099, 2331101, 2331102, 2331103, 2331104, 2331105, 2331106, 2331107, 2331111, 2331112, 2331113, 2331114, 2331115, 2331116, 2331117, 2331118, 2331119, 2331121, 2331122, 2331127, 2331128, 2331129, 2331130, 2331135, 2331136, 2331138, 2331140, 2331141, 2331143, 2331144, 2331145, 2331146, 2331150, 2331153, 2331159, 2331161, 2331163, 2331164, 2331166, 2331167, 2331168, 2331171, 2331174, 2331175, 2331176, 2331177, 2331179, 2331184, 2331186, 2331187, 2331188, 2331189, 2331190, 2331191, 2331192, 2331194, 2331196, 2331197, 2331198, 2331200, 2331201, 2331202, 2331203, 2331204, 2331206, 2331207, 2331211, 2331212, 2331213, 2331215, 2331216, 2331217, 2331218, 2331219, 2331221, 2331225, 2331229, 2331230, 2331231, 2331232, 2331233, 2331236, 2331237, 2331239, 2331242, 2331244, 2331245, 2331246, 2331247, 2331248, 2331250, 2331252, 2331254, 2331255, 2331256, 2331258, 2331259, 2331261, 2331262, 2331263, 2331264, 2331268, 2331269, 2331270, 2331272, 2331274, 2331275, 2331276, 2331278, 2331281, 2331283, 2331285, 2331286, 2331287, 2331288, 2331290, 2331291, 2331294, 2331295, 2331297, 2331298, 2331299, 2331300, 2331301, 2331302, 2331303, 2331305, 2331306, 2331310, 2331311, 2331312, 2331314, 2331316, 2331317, 2331319, 2331321, 2331322, 2331324, 2331325, 2331326, 2331327, 2331329, 2331330, 2331332, 2331335, 2331336, 2331340, 2331343, 2331345, 2331346, 2331347, 2331352, 2331353, 2331355, 2331356, 2331359, 2331360, 2331361, 2331363, 2331364, 2331368, 2331373, 2331374, 2331376, 2331379, 2331384, 2331387, 2331388, 2331394, 2331397, 2331398, 2331399, 2331400, 2331401, 2331402, 2331404, 2331412, 2331419, 2331421, 2331423, 2331430, 2331435, 2331437, 2331439, 2331441, 2331443, 2331444, 2331445, 2331447, 2331448, 2331449, 2331453, 2331455, 2331457, 2331459, 2331460, 2331462, 2331464, 2331468, 2331469, 2331470, 2331472, 2331473, 2331475, 2331476, 2331481, 2331483, 2331485, 2331487, 2331488, 2331490, 2331491, 2331492, 2331499, 2331501, 2331502, 2331503, 2331505, 2331506, 2331508, 2331510, 2331516, 2331517, 2331519, 2331520, 2331521, 2331523, 2331524, 2331526, 2331528, 2331529, 2331531, 2331533, 2331537, 2331538, 2331539, 2331540, 2331542, 2331543, 2331544, 2331546, 2331548, 2331549, 2331550, 2331551, 2331552, 2331555, 2331556, 2331558, 2331559, 2331560, 2331562, 2331564, 2331565, 2331568, 2331569, 2331571, 2331575, 2331576, 2331579, 2331580, 2331581, 2331582, 2331583, 2331585, 2331588, 2331589, 2331590, 2331591, 2331595, 2331596, 2331598, 2331599, 2331601, 2331603, 2331605, 2331606, 2331609, 2331610, 2331611, 2331613, 2331614, 2331616, 2331617, 2331619, 2331620, 2331621, 2331622, 2331623, 2331624, 2331627, 2331628, 2331629, 2331630, 2331631, 2331632, 2331633, 2331634, 2331635, 2331638, 2331639, 2331641, 2331642, 2331645, 2331648, 2331649, 2331650, 2331656, 2331658, 2331659, 2331660, 2331662, 2331663, 2331664, 2331666, 2331667, 2331668, 2331675, 2331677, 2331684, 2331688, 2331689, 2331690, 2331695, 2331697, 2331699, 2331700, 2331702, 2331704, 2331705, 2331706, 2331707, 2331710, 2331711, 2331715, 2331718, 2331719, 2331724, 2331726, 2331727, 2331732, 2331733, 2331734, 2331735, 2331736, 2331739, 2331740, 2331741, 2331742, 2331746, 2331747, 2331748, 2331749, 2331751, 2331752, 2331753, 2331756, 2331757, 2331758, 2331759, 2331760, 2331761, 2331764, 2331766, 2331767, 2331768, 2331771, 2331773, 2331774, 2331775, 2331776, 2331778, 2331780, 2331781, 2331782, 2331784, 2331786, 2331789, 2331790, 2331793, 2331794, 2331795, 2331797, 2331800, 2331802, 2331803, 2331804, 2331807, 2331808, 2331811, 2331814, 2331815, 2331817, 2331818, 2331819, 2331820, 2331822, 2331823, 2331824, 2331825, 2331826, 2331827, 2331828, 2331829, 2331831, 2331832, 2331833, 2331835, 2331839, 2331840, 2331841, 2331842, 2331844, 2331845, 2331847, 2331848, 2331849, 2331851, 2331852, 2331854, 2331856, 2331857, 2331858, 2331859, 2331861, 2331864, 2331865, 2331866, 2331868, 2331872, 2331875, 2331876, 2331877, 2331878, 2331879, 2331880, 2331882, 2331883, 2331884, 2331885, 2331887, 2331888, 2331890, 2331891, 2331894, 2331897, 2331898, 2331900, 2331903, 2331905, 2331907, 2331908, 2331909, 2331911, 2331914, 2331915, 2331916, 2331917, 2331918, 2331923, 2331924, 2331926, 2331927, 2331930, 2331932, 2331935, 2331937, 2331939, 2331940, 2331941, 2331942, 2331946, 2331949, 2331950, 2331951, 2331954, 2331956, 2331957, 2331959, 2331964, 2331965, 2331966, 2331968, 2331969, 2331971, 2331973, 2331974, 2331979, 2331980, 2331982, 2331984, 2331985, 2331986, 2331990, 2331991, 2331994, 2331995, 2331996, 2331998, 2331999, 2332001, 2332002, 2332004, 2332006, 2332007, 2332012, 2332017, 2332018, 2332020, 2332021, 2332023, 2332029, 2332030, 2332033, 2332034, 2332037, 2332038, 2332042, 2332043, 2332044, 2332045, 2332048, 2332051, 2332052, 2332054, 2332057, 2332058, 2332059, 2332060, 2332062, 2332064, 2332065, 2332068, 2332071, 2332074, 2332075, 2332076, 2332077, 2332078, 2332079, 2332081, 2332082, 2332085, 2332087, 2332088, 2332092, 2332093, 2332099, 2332101, 2332105, 2332107, 2332108, 2332111, 2332113, 2332117, 2332118, 2332119, 2332124, 2332125, 2332126, 2332127, 2332128, 2332129, 2332130, 2332131, 2332132, 2332135, 2332136, 2332138, 2332139, 2332140, 2332145, 2332146, 2332147, 2332148, 2332149, 2332150, 2332155, 2332159, 2332161, 2332163, 2332166, 2332167, 2332173, 2332174, 2332176, 2332177, 2332183, 2332184, 2332185, 2332186, 2332187, 2332188, 2332189, 2332190, 2332192, 2332196, 2332197, 2332198, 2332199, 2332202, 2332203, 2332204, 2332205, 2332206, 2332208, 2332211, 2332218, 2332219, 2332221, 2332232, 2332236, 2332237, 2332238, 2332242, 2332244, 2332245, 2332246, 2332247, 2332251, 2332252, 2332254, 2332255, 2332258, 2332260, 2332262, 2332263, 2332264, 2332266, 2332267, 2332268, 2332269, 2332270, 2332271, 2332272, 2332273, 2332275, 2332277, 2332278, 2332279, 2332281, 2332282, 2332283, 2332284, 2332285, 2332286, 2332288, 2332289, 2332291, 2332292, 2332294, 2332295, 2332296, 2332297, 2332299, 2332301, 2332303, 2332305, 2332306, 2332308, 2332309, 2332312, 2332319, 2332320, 2332323, 2332324, 2332325, 2332326, 2332327, 2332328, 2332331, 2332332, 2332333, 2332335, 2332339, 2332341, 2332345, 2332350, 2332351, 2332352, 2332355, 2332356, 2332357, 2332360, 2332362, 2332363, 2332364, 2332365, 2332366, 2332367, 2332372, 2332373, 2332375, 2332377, 2332379, 2332380, 2332382, 2332385, 2332388, 2332389, 2332396, 2332398, 2332399, 2332401, 2332402, 2332404, 2332406, 2332407, 2332408, 2332410, 2332411, 2332413, 2332414, 2332417, 2332420, 2332423, 2332424, 2332427, 2332428, 2332430, 2332433, 2332434, 2332436, 2332437, 2332438, 2332440, 2332441, 2332442, 2332443, 2332446, 2332448, 2332449, 2332452, 2332453, 2332457, 2332459, 2332460, 2332461, 2332462, 2332463, 2332464, 2332465, 2332469, 2332470, 2332471, 2332472, 2332478, 2332479, 2332480, 2332481, 2332484, 2332485, 2332489, 2332491, 2332492, 2332495, 2332498, 2332501, 2332502, 2332506, 2332507, 2332509, 2332510, 2332511, 2332512, 2332513, 2332516, 2332517, 2332518, 2332521, 2332524, 2332526, 2332530, 2332532, 2332534, 2332536, 2332538, 2332539, 2332540, 2332542, 2332544, 2332545, 2332547, 2332549, 2332550, 2332551, 2332552, 2332553, 2332554, 2332556, 2332557, 2332560, 2332567, 2332568, 2332571, 2332574, 2332576, 2332577, 2332578, 2332580, 2332585, 2332586, 2332587, 2332588, 2332589, 2332593, 2332594, 2332595, 2332596, 2332597, 2332598, 2332599, 2332604, 2332605, 2332606, 2332607, 2332609, 2332611, 2332612, 2332613, 2332614, 2332615, 2332616, 2332618, 2332619, 2332621, 2332622, 2332624, 2332625, 2332627, 2332629, 2332631, 2332634, 2332635, 2332637, 2332639, 2332641, 2332644, 2332647, 2332653, 2332654, 2332655, 2332658, 2332659, 2332661, 2332668, 2332670, 2332671, 2332672, 2332674, 2332675, 2332678, 2332679, 2332682, 2332684, 2332685, 2332691, 2332693, 2332695, 2332696, 2332697, 2332698, 2332701, 2332704, 2332705, 2332707, 2332709, 2332712, 2332713, 2332714, 2332717, 2332718, 2332719, 2332720, 2332721, 2332723, 2332725, 2332726, 2332727, 2332729, 2332730, 2332731, 2332732, 2332733, 2332734, 2332736, 2332738, 2332739, 2332741, 2332743, 2332744, 2332748, 2332749, 2332752, 2332757, 2332758, 2332759, 2332760, 2332761, 2332762, 2332763, 2332764, 2332767, 2332768, 2332770, 2332772, 2332773, 2332775, 2332779, 2332783, 2332784, 2332787, 2332789, 2332790, 2332791, 2332792, 2332793, 2332797, 2332798, 2332801, 2332802, 2332803, 2332804, 2332807, 2332808, 2332809, 2332810, 2332816, 2332818, 2332820, 2332823, 2332825, 2332826, 2332827, 2332828, 2332829, 2332830, 2332831, 2332832, 2332833, 2332834, 2332836, 2332837, 2332839, 2332844, 2332846, 2332848, 2332852, 2332853, 2332854, 2332855, 2332857, 2332860, 2332861, 2332862, 2332863, 2332869, 2332870, 2332871, 2332872, 2332874, 2332875, 2332876, 2332877, 2332878, 2332881, 2332882, 2332883, 2332884, 2332885, 2332886, 2332887, 2332888, 2332890, 2332892, 2332893, 2332894, 2332896, 2332897, 2332898, 2332899, 2332900, 2332901, 2332903, 2332906, 2332908, 2332910, 2332911, 2332913, 2332914, 2332916, 2332917, 2332918, 2332919, 2332920, 2332921, 2332923, 2332924, 2332926, 2332927, 2332928, 2332929, 2332930, 2332934, 2332935, 2332936, 2332937, 2332943, 2332945, 2332947, 2332948, 2332949, 2332950, 2332952, 2332953, 2332954, 2332956, 2332958, 2332960, 2332961, 2332962, 2332964, 2332965, 2332967, 2332968, 2332970, 2332971, 2332972, 2332974, 2332980, 2332981, 2332984, 2332986, 2332989, 2332990, 2332993, 2332995, 2332998, 2332999, 2333002, 2333003, 2333005, 2333007, 2333010, 2333012, 2333016, 2333018, 2333019, 2333020, 2333021, 2333022, 2333024, 2333028, 2333029, 2333036, 2333037, 2333038, 2333040, 2333042, 2333044, 2333045, 2333046, 2333048, 2333050, 2333051, 2333053, 2333055, 2333056, 2333057, 2333061, 2333062, 2333063, 2333064, 2333065, 2333067, 2333070, 2333071, 2333073, 2333075, 2333077, 2333078, 2333082, 2333083, 2333084, 2333085, 2333086, 2333091, 2333092, 2333093, 2333094, 2333095, 2333096, 2333102, 2333106, 2333110, 2333111, 2333112, 2333115, 2333116, 2333117, 2333119, 2333120, 2333124, 2333125, 2333126, 2333127, 2333128, 2333129, 2333132, 2333138, 2333142, 2333147, 2333149, 2333151, 2333152, 2333153, 2333154, 2333155, 2333157, 2333158, 2333161, 2333162, 2333164, 2333165, 2333167, 2333168, 2333169, 2333172, 2333174, 2333176, 2333177, 2333178, 2333179, 2333180, 2333186, 2333187, 2333188, 2333189, 2333192, 2333195, 2333196, 2333197, 2333203, 2333204, 2333205, 2333206, 2333209, 2333211, 2333212, 2333213, 2333215, 2333216, 2333217, 2333218, 2333221, 2333224, 2333225, 2333226, 2333227, 2333228, 2333229, 2333231, 2333232, 2333233, 2333236, 2333240, 2333241, 2333242, 2333243, 2333245, 2333247, 2333248, 2333249, 2333253, 2333255, 2333256, 2333261, 2333263, 2333264, 2333266, 2333270, 2333271, 2333272, 2333273, 2333275, 2333276, 2333279, 2333282, 2333283, 2333288, 2333290, 2333293, 2333294, 2333295, 2333296, 2333299, 2333300, 2333301, 2333302, 2333306, 2333307, 2333308, 2333309, 2333310, 2333311, 2333312, 2333317, 2333318, 2333323, 2333324, 2333328, 2333329, 2333334, 2333335, 2333338, 2333343, 2333346, 2333350, 2333351, 2333352, 2333355, 2333356, 2333357, 2333358, 2333359, 2333360, 2333361, 2333363, 2333364, 2333365, 2333366, 2333368, 2333371, 2333373, 2333374, 2333375, 2333376, 2333378, 2333379, 2333382, 2333386, 2333387, 2333388, 2333389, 2333390, 2333391, 2333396, 2333397, 2333398, 2333401, 2333403, 2333411, 2333412, 2333414, 2333416, 2333417, 2333421, 2333423, 2333425, 2333427, 2333430, 2333431, 2333432, 2333435, 2333442, 2333443, 2333445, 2333447, 2333448, 2333450, 2333452, 2333456, 2333459, 2333460, 2333464, 2333465, 2333466, 2333469, 2333474, 2333475, 2333476, 2333478, 2333483, 2333485, 2333486, 2333490, 2333491, 2333495, 2333498, 2333500, 2333501, 2333502, 2333506, 2333507, 2333508, 2333511, 2333514, 2333518, 2333520, 2333521, 2333522, 2333524, 2333527, 2333528, 2333529, 2333531, 2333532, 2333533, 2333537, 2333540, 2333546, 2333547, 2333548, 2333549, 2333550, 2333551, 2333553, 2333554, 2333558, 2333559, 2333560, 2333561, 2333562, 2333563, 2333566, 2333567, 2333569, 2333570, 2333573, 2333574, 2333577, 2333579, 2333580, 2333582, 2333583, 2333584, 2333585, 2333587, 2333588, 2333591, 2333592, 2333594, 2333595, 2333597, 2333604, 2333605, 2333606, 2333607, 2333608, 2333609, 2333614, 2333617, 2333618, 2333619, 2333620, 2333622, 2333623, 2333624, 2333625, 2333627, 2333630, 2333632, 2333633, 2333634, 2333635, 2333641, 2333644, 2333646, 2333647, 2333648, 2333653, 2333655, 2333659, 2333660, 2333661, 2333664, 2333665, 2333672, 2333673, 2333675, 2333676, 2333678, 2333679, 2333680, 2333681, 2333685, 2333688, 2333690, 2333691, 2333693, 2333695, 2333696, 2333699, 2333700, 2333701, 2333703, 2333704, 2333710, 2333712, 2333713, 2333714, 2333715, 2333716, 2333717, 2333718, 2333719, 2333721, 2333722, 2333727, 2333728, 2333731, 2333733, 2333734, 2333736, 2333740, 2333742, 2333743, 2333745, 2333747, 2333749, 2333751, 2333752, 2333754, 2333755, 2333756, 2333757, 2333758, 2333761, 2333764, 2333766, 2333771, 2333772, 2333775, 2333776, 2333777, 2333778, 2333779, 2333781, 2333782, 2333785, 2333786, 2333787, 2333789, 2333791, 2333793, 2333794, 2333795, 2333797, 2333799, 2333800, 2333801, 2333803, 2333805, 2333806, 2333808, 2333809, 2333811, 2333812, 2333813, 2333814, 2333816, 2333818, 2333820, 2333821, 2333823, 2333825, 2333826, 2333829, 2333830, 2333832, 2333833, 2333835, 2333836, 2333837, 2333839, 2333841, 2333844, 2333845, 2333848, 2333850, 2333851, 2333852, 2333853, 2333854, 2333855, 2333859, 2333860, 2333862, 2333864, 2333865, 2333867, 2333872, 2333874, 2333875, 2333876, 2333877, 2333878, 2333879, 2333880, 2333881, 2333883, 2333884, 2333887, 2333888, 2333889, 2333891, 2333894, 2333898, 2333899, 2333900, 2333906, 2333907, 2333908, 2333910, 2333912, 2333913, 2333915, 2333916, 2333918, 2333920, 2333922, 2333923, 2333924, 2333925, 2333927, 2333928, 2333933, 2333935, 2333939, 2333940, 2333941, 2333942, 2333944, 2333946, 2333949, 2333950, 2333952, 2333955, 2333956, 2333957, 2333958, 2333960, 2333962, 2333964, 2333966, 2333972, 2333973, 2333975, 2333977, 2333981, 2333983, 2333984, 2333986, 2333990, 2333991, 2333993, 2333997, 2333999, 2334000, 2334001, 2334005, 2334006, 2334008, 2334010, 2334011, 2334012, 2334015, 2334019, 2334024, 2334028, 2334029, 2334031, 2334033, 2334036, 2334038, 2334039, 2334042, 2334043, 2334045, 2334046, 2334047, 2334048, 2334049, 2334050, 2334051, 2334054, 2334055, 2334058, 2334059, 2334060, 2334061, 2334062, 2334063, 2334064, 2334066, 2334068, 2334072, 2334074, 2334076, 2334078, 2334080, 2334081, 2334083, 2334087, 2334090, 2334091, 2334092, 2334093, 2334095, 2334096, 2334098, 2334100, 2334101, 2334103, 2334104, 2334106, 2334107, 2334108, 2334113, 2334115, 2334116, 2334117, 2334118, 2334119, 2334120, 2334121, 2334122, 2334123, 2334124, 2334125, 2334126, 2334129, 2334130, 2334135, 2334138, 2334140, 2334142, 2334143, 2334146, 2334147, 2334148, 2334149, 2334150, 2334152, 2334158, 2334159, 2334162, 2334165, 2334166, 2334167, 2334168, 2334170, 2334171, 2334172, 2334173, 2334180, 2334184, 2334185, 2334186, 2334187, 2334189, 2334190, 2334192, 2334193, 2334196, 2334201, 2334206, 2334207, 2334208, 2334215, 2334217, 2334218, 2334219, 2334220, 2334221, 2334222, 2334223, 2334224, 2334226, 2334231, 2334236, 2334237, 2334238, 2334243, 2334248, 2334249, 2334256, 2334257, 2334262, 2334263, 2334267, 2334268, 2334269, 2334270, 2334272, 2334274, 2334275, 2334278, 2334280, 2334281, 2334284, 2334285, 2334288, 2334293, 2334295, 2334298, 2334300, 2334302, 2334303, 2334307, 2334309, 2334311, 2334312, 2334313, 2334314, 2334315, 2334317, 2334318, 2334319, 2334320, 2334324, 2334327, 2334328, 2334330, 2334331, 2334333, 2334334, 2334335, 2334336, 2334337, 2334340, 2334341, 2334342, 2334343, 2334345, 2334348, 2334349, 2334351, 2334352, 2334354, 2334356, 2334360, 2334361, 2334362, 2334363, 2334364, 2334365, 2334367, 2334369, 2334371, 2334372, 2334375, 2334377, 2334378, 2334382, 2334383, 2334384, 2334390, 2334391, 2334392, 2334393, 2334395, 2334396, 2334398, 2334399, 2334401, 2334403, 2334404, 2334405, 2334406, 2334410, 2334416, 2334417, 2334419, 2334422, 2334425, 2334426, 2334429, 2334431, 2334434, 2334436, 2334438, 2334439, 2334440, 2334441, 2334444, 2334447, 2334451, 2334452, 2334454, 2334458, 2334460, 2334461, 2334463, 2334467, 2334469, 2334470, 2334471, 2334472, 2334473, 2334474, 2334478, 2334479, 2334483, 2334484, 2334488, 2334489, 2334490, 2334491, 2334492, 2334501, 2334502, 2334505, 2334506, 2334507, 2334508, 2334509, 2334511, 2334513, 2334514, 2334517, 2334518, 2334520, 2334521, 2334522, 2334523, 2334524, 2334525, 2334526, 2334528, 2334529, 2334530, 2334531, 2334532, 2334535, 2334536, 2334542, 2334543, 2334548, 2334550, 2334553, 2334554, 2334555, 2334557, 2334558, 2334559, 2334565, 2334568, 2334571, 2334572, 2334574, 2334576, 2334578, 2334579, 2334582, 2334586, 2334589, 2334591, 2334592, 2334593, 2334597, 2334598, 2334600, 2334602, 2334604, 2334605, 2334606, 2334608, 2334611, 2334613, 2334614, 2334615, 2334616, 2334617, 2334619, 2334621, 2334623, 2334624, 2334628, 2334629, 2334631, 2334632, 2334634, 2334636, 2334638, 2334641, 2334642, 2334645, 2334646, 2334647, 2334650, 2334651, 2334653, 2334654, 2334655, 2334656, 2334657, 2334658, 2334659, 2334661, 2334663, 2334664, 2334667, 2334668, 2334669, 2334675, 2334677, 2334678, 2334681, 2334685, 2334690, 2334691, 2334692, 2334698, 2334699, 2334700, 2334702, 2334714, 2334715, 2334717, 2334720, 2334721, 2334723, 2334724, 2334727, 2334728, 2334729, 2334730, 2334731, 2334733, 2334735, 2334737, 2334739, 2334740, 2334741, 2334742, 2334744, 2334746, 2334747, 2334748, 2334751, 2334752, 2334753, 2334757, 2334759, 2334760, 2334761, 2334762, 2334764, 2334765, 2334768, 2334771, 2334773, 2334774, 2334775, 2334780, 2334781, 2334782, 2334783, 2334784, 2334785, 2334786, 2334787, 2334792, 2334796, 2334797, 2334798, 2334800, 2334806, 2334807, 2334808, 2334809, 2334813, 2334815, 2334816, 2334817, 2334821, 2334822, 2334823, 2334824, 2334825, 2334827, 2334831, 2334834, 2334837, 2334838, 2334839, 2334842, 2334846, 2334847, 2334848, 2334849, 2334850, 2334851, 2334854, 2334856, 2334858, 2334860, 2334861, 2334864, 2334869, 2334870, 2334872, 2334873, 2334875, 2334880, 2334884, 2334885, 2334887, 2334891, 2334893, 2334895, 2334896, 2334898, 2334900, 2334901, 2334903, 2334904, 2334907, 2334908, 2334911, 2334912, 2334913, 2334915, 2334918, 2334919, 2334920, 2334921, 2334923, 2334924, 2334927, 2334929, 2334930, 2334931, 2334932, 2334933, 2334936, 2334940, 2334941, 2334942, 2334943, 2334945, 2334946, 2334947, 2334949, 2334950, 2334956, 2334958, 2334960, 2334962, 2334963, 2334964, 2334965, 2334966, 2334968, 2334970, 2334972, 2334973, 2334976, 2334977, 2334978, 2334979, 2334981, 2334988, 2334989, 2334990, 2334992, 2334993, 2334994, 2334995, 2334998, 2335000, 2335003, 2335005, 2335006, 2335010, 2335011, 2335014, 2335016, 2335017, 2335018, 2335019, 2335020, 2335021, 2335023, 2335024, 2335026, 2335028, 2335034, 2335038, 2335039, 2335041, 2335043, 2335044, 2335045, 2335046, 2335049, 2335052, 2335055, 2335057, 2335058, 2335061, 2335062, 2335066, 2335067, 2335068, 2335070, 2335072, 2335074, 2335077, 2335079, 2335082, 2335083, 2335088, 2335089, 2335092, 2335094, 2335095, 2335096, 2335099, 2335100, 2335101, 2335102, 2335103, 2335105, 2335106, 2335108, 2335109, 2335111, 2335121, 2335124, 2335125, 2335127, 2335131, 2335132, 2335137, 2335138, 2335139, 2335143, 2335145, 2335146, 2335148, 2335150, 2335151, 2335152, 2335153, 2335154, 2335160, 2335161, 2335162, 2335163, 2335164, 2335166, 2335168, 2335170, 2335171, 2335176, 2335177, 2335179, 2335180, 2335186, 2335188, 2335189, 2335191, 2335192, 2335193, 2335195, 2335196, 2335197, 2335198, 2335201, 2335202, 2335204, 2335205, 2335207, 2335208, 2335210, 2335212, 2335213, 2335214, 2335218, 2335220, 2335221, 2335223, 2335226, 2335227, 2335228, 2335229, 2335230, 2335232, 2335234, 2335235, 2335237, 2335238, 2335239, 2335242, 2335245, 2335247, 2335250, 2335253, 2335254, 2335255, 2335256, 2335258, 2335259, 2335260, 2335262, 2335263, 2335264, 2335265, 2335267, 2335268, 2335270, 2335271, 2335276, 2335279, 2335280, 2335282, 2335284, 2335285, 2335286, 2335289, 2335290, 2335293, 2335296, 2335298, 2335300, 2335304, 2335307, 2335308, 2335310, 2335311, 2335312, 2335316, 2335317, 2335321, 2335324, 2335326, 2335327, 2335329, 2335332, 2335334, 2335335, 2335337, 2335338, 2335339, 2335345, 2335348, 2335349, 2335350, 2335351, 2335353, 2335354, 2335356, 2335359, 2335360, 2335361, 2335365, 2335366, 2335367, 2335371, 2335373, 2335374, 2335375, 2335377, 2335380, 2335381, 2335382, 2335383, 2335384, 2335385, 2335387, 2335388, 2335390, 2335391, 2335394, 2335398, 2335399, 2335400, 2335404, 2335405, 2335406, 2335408, 2335409, 2335410, 2335411, 2335412, 2335413, 2335415, 2335419, 2335421, 2335424, 2335425, 2335427, 2335429, 2335430, 2335431, 2335433, 2335437, 2335438, 2335443, 2335444, 2335446, 2335450, 2335451, 2335453, 2335455, 2335457, 2335460, 2335462, 2335463, 2335464, 2335465, 2335466, 2335467, 2335468, 2335469, 2335470, 2335475, 2335476, 2335481, 2335483, 2335486, 2335487, 2335491, 2335492, 2335496, 2335497, 2335498, 2335502, 2335504, 2335508, 2335509, 2335513, 2335514, 2335517, 2335518, 2335519, 2335520, 2335523, 2335526, 2335528, 2335529, 2335532, 2335537, 2335539, 2335540, 2335542, 2335545, 2335546, 2335548, 2335549, 2335551, 2335552, 2335554, 2335556, 2335557, 2335558, 2335559, 2335561, 2335563, 2335566, 2335567, 2335568, 2335570, 2335575, 2335578, 2335579, 2335582, 2335584, 2335585, 2335587, 2335588, 2335589, 2335593, 2335595, 2335596, 2335599, 2335600, 2335603, 2335604, 2335605, 2335606, 2335608, 2335609, 2335613, 2335615, 2335616, 2335621, 2335622, 2335624, 2335626, 2335627, 2335629, 2335630, 2335631, 2335633, 2335637, 2335639, 2335640, 2335642, 2335646, 2335647, 2335649, 2335651, 2335652, 2335654, 2335655, 2335656, 2335658, 2335662, 2335664, 2335668, 2335669, 2335670, 2335671, 2335675, 2335676, 2335677, 2335681, 2335685, 2335686, 2335687, 2335689, 2335693, 2335695, 2335698, 2335699, 2335700, 2335701, 2335702, 2335704, 2335705, 2335706, 2335708, 2335710, 2335711, 2335712, 2335713, 2335714, 2335715, 2335716, 2335718, 2335719, 2335722, 2335723, 2335726, 2335728, 2335729, 2335730, 2335734, 2335736, 2335737, 2335738, 2335739, 2335740, 2335743, 2335745, 2335748, 2335749, 2335751, 2335753, 2335754, 2335755, 2335756, 2335757, 2335761, 2335762, 2335765, 2335766, 2335767, 2335769, 2335773, 2335774, 2335775, 2335776, 2335777, 2335779, 2335780, 2335782, 2335785, 2335786, 2335787, 2335788, 2335790, 2335791, 2335796, 2335797, 2335800, 2335806, 2335808, 2335810, 2335811, 2335812, 2335813, 2335817, 2335819, 2335820, 2335821, 2335824, 2335825, 2335826, 2335828, 2335831, 2335832, 2335833, 2335834, 2335835, 2335836, 2335838, 2335840, 2335846, 2335848, 2335849, 2335850, 2335851, 2335853, 2335854, 2335855, 2335857, 2335859, 2335862, 2335865, 2335870, 2335871, 2335877, 2335879, 2335882, 2335883, 2335884, 2335885, 2335889, 2335891, 2335896, 2335897, 2335899, 2335900, 2335902, 2335903, 2335904, 2335905, 2335909, 2335911, 2335913, 2335923, 2335925, 2335928, 2335930, 2335935, 2335940, 2335941, 2335942, 2335947, 2335948, 2335949, 2335950, 2335954, 2335955, 2335957, 2335958, 2335959, 2335961, 2335962, 2335963, 2335964, 2335971, 2335972, 2335973, 2335974, 2335977, 2335980, 2335981, 2335982, 2335983, 2335984, 2335985, 2335986, 2335987, 2335989, 2335993, 2335999, 2336000, 2336001, 2336002, 2336005, 2336006, 2336007, 2336009, 2336011, 2336012, 2336013, 2336014, 2336017, 2336020, 2336021, 2336022, 2336023, 2336025, 2336026, 2336027, 2336030, 2336031, 2336032, 2336034, 2336035, 2411494, 2411503, 2411506, 2411514, 2411515, 2411518, 2411522, 2411523, 2411525, 2411526, 2411528, 2411534, 2411535, 2411537, 2411538, 2411542, 2411543, 2411544, 2411545, 2411551, 2411552, 2411555, 2411556, 2411560, 2411561, 2411563, 2411574, 2411575, 2411581, 2411582, 2411584, 2411587, 2411588, 2411592, 2411593, 2411598, 2411599, 2411600, 2411601, 2411605, 2411606, 2411608, 2411609, 2411610, 2411613, 2411616, 2411617, 2411619, 2411620, 2411627, 2411628, 2411633, 2411641, 2411642, 2411644, 2411646, 2411647, 2411649, 2411652, 2411653, 2411661, 2411669, 2411677, 2411680, 2411681, 2411686, 2411687, 2411691, 2411692, 2411693, 2411694, 2411695, 2411696, 2411697, 2411698, 2411700, 2411701, 2411702, 2411703, 2411704, 2411709, 2411710, 2411712, 2411713, 2411717, 2411722, 2411723, 2411731, 2411732, 2411733, 2411734, 2411739, 2411740, 2411744, 2411746, 2411748, 2411758, 2411760, 2411761, 2411762, 2411764, 2411766, 2411767, 2411774, 2411779, 2411783, 2411784, 2411786, 2411792, 2411793, 2411794, 2411795, 2411796, 2411798, 2411799, 2411800, 2411801, 2411802, 2411811, 2411812, 2411817, 2411821, 2411822, 2411823, 2411824, 2411829, 2411837, 2411840, 2411841, 2411842, 2411850, 2411852, 2411853, 2411855, 2411857, 2411858, 2411869, 2411871, 2411872, 2411873, 2411881, 2411883, 2411889, 2411891, 2411893, 2411894, 2411896, 2411899, 2411900, 2411902, 2411903, 2411909, 2411910, 2411912, 2411913, 2411915, 2411916, 2411918, 2411919, 2411921, 2411922, 2411923, 2411926, 2411928, 2411931, 2411933, 2411934, 2411938, 2411940, 2411942, 2411947, 2411952, 2411953, 2411954, 2411957, 2411962, 2411964, 2411966, 2411967, 2411968, 2411980, 2411983, 2411985, 2411986, 2411987, 2411990, 2411997, 2411998, 2412002, 2412003, 2412004, 2412006, 2412012, 2412025, 2412026, 2412027, 2412028, 2412036, 2412037, 2412038, 2412039, 2412046, 2412050, 2412052, 2412062, 2412063, 2412065, 2412066, 2412075, 2412078, 2412082, 2412083, 2412088, 2412090, 2412094, 2412095, 2412096, 2412104, 2412105, 2412107, 2412109, 2412111, 2412114, 2412121, 2412124, 2412125, 2412127, 2412134, 2412140, 2412141, 2412142, 2412143, 2412147, 2412148, 2412157, 2412159, 2412160, 2412161, 2412162, 2412164, 2412165, 2412166, 2412169, 2412172, 2412176, 2412178, 2412188, 2412190, 2412195, 2412197, 2412198, 2412199, 2412203, 2412204, 2412215, 2412216, 2412221, 2412224, 2412226, 2412229, 2412231, 2412245, 2412246, 2412248, 2412259, 2412261, 2412265, 2412268, 2412269, 2412271, 2412272, 2412273, 2412277, 2412279, 2412280, 2412282, 2412283, 2412285, 2412286, 2412287, 2412290, 2412294, 2412296, 2412297, 2412300, 2412302, 2412308, 2412310, 2412311, 2412318, 2412321, 2412323, 2412324, 2412325, 2412326, 2412336, 2412337, 2412343, 2412360, 2412362, 2412365, 2412369, 2412372, 2412375, 2412376, 2412382, 2412383, 2412387, 2412388, 2412398, 2412402, 2412404, 2412405, 2412406, 2412408, 2412409, 2412410, 2412413, 2412414, 2412415, 2412416, 2412418, 2412419, 2412423, 2412425, 2412429, 2412430, 2412431, 2412436, 2412440, 2412441, 2412448, 2412453, 2412457, 2412458, 2412462, 2412463, 2412464, 2412466, 2412468, 2412469, 2412470, 2412471, 2412477, 2412478, 2412479, 2412480, 2412481, 2412483, 2412487, 2412494, 2412495, 2412498, 2412500, 2412502, 2412518, 2412525, 2412527, 2412528, 2412530, 2412538, 2412542, 2412553, 2412556, 2412559, 2412561, 2412567, 2412568, 2412569, 2412570, 2412571, 2412572, 2412573, 2412574, 2412578, 2412580, 2412581, 2412590, 2412591, 2412592, 2412596, 2412597, 2412598, 2412599, 2412601, 2412603, 2412605, 2412609, 2412610, 2412612, 2412614, 2412624, 2412626, 2412627, 2412628, 2412629, 2412630, 2412631, 2412633, 2412641, 2412643, 2412645, 2412646, 2412647, 2412650, 2412652, 2412653, 2412657, 2412658, 2412664, 2412665, 2412668, 2412669, 2412674, 2412679, 2412681, 2412683, 2412685, 2412686, 2412690, 2412693, 2412697, 2412698, 2412708, 2412717, 2412719, 2412721, 2412722, 2412724, 2412725, 2412727, 2412729, 2412730, 2412736, 2412738, 2412740, 2412741, 2412742, 2412745, 2412746, 2412748, 2412752, 2412753, 2412754, 2412757, 2412759, 2412764, 2412768, 2412770, 2412771, 2412772, 2412779, 2412780, 2412781, 2412784, 2412791, 2412792, 2412793, 2412797, 2412801, 2412803, 2412804, 2412805, 2412806, 2412808, 2412810, 2412811, 2412812, 2412818, 2412822, 2412826, 2412829, 2412833, 2412842, 2412845, 2412848, 2412849, 2412850, 2412851, 2412853, 2412854, 2412856, 2412857, 2412859, 2412860, 2412862, 2412866, 2412870, 2412871, 2412872, 2412876, 2412877, 2412882, 2412884, 2412889, 2412890, 2412891, 2412895, 2412896, 2412897, 2412898, 2412902, 2412906, 2412907, 2412914, 2412915, 2412925, 2412930, 2412931, 2412936, 2412940, 2412943, 2412946, 2412947, 2412949, 2412953, 2412956, 2412958, 2412960, 2412961, 2412963, 2412965, 2412966, 2412969, 2412970, 2412972, 2412975, 2412981, 2412982, 2412983, 2412991, 2412992, 2412996, 2412997, 2413009, 2413011, 2413012, 2413013, 2413016, 2413017, 2413019, 2413020, 2413021, 2413028, 2413031, 2413032, 2413033, 2413034, 2413036, 2413046, 2413047, 2413048, 2413051, 2413058, 2413063, 2413064, 2413073, 2413076, 2413078, 2413079, 2413087, 2413088, 2413089, 2413090, 2413094, 2413100, 2413101, 2413103, 2413105, 2413106, 2413110, 2413111, 2413116, 2413117, 2413123, 2413127, 2413128, 2413129, 2413135, 2413137, 2413140, 2413144, 2413147, 2413148, 2413149, 2413150, 2413151, 2413152, 2413153, 2413155, 2413157, 2413159, 2413160, 2413161, 2413162, 2413163, 2413164, 2413166, 2413167, 2413177, 2413181, 2413182, 2413184, 2413185, 2413187, 2413189, 2413191, 2413197, 2413198, 2413201, 2413203, 2413205, 2413207, 2413212, 2413213, 2413214, 2413215, 2413217, 2413219, 2413222, 2413224, 2413230, 2413233, 2413234, 2413237, 2413238, 2413239, 2413240, 2413247, 2413248, 2413249, 2413250, 2413253, 2413259, 2413260, 2413261, 2413262, 2413265, 2413266, 2413267, 2413269, 2413271, 2413273, 2413274, 2413277, 2413278, 2413279, 2413280, 2413283, 2413287, 2413288, 2413292, 2413294, 2413296, 2413302, 2413303, 2413309, 2413311, 2413319, 2413320, 2413321, 2413323, 2413324, 2413325, 2413326, 2413327, 2413328, 2413329, 2413330, 2413332, 2413335, 2413336, 2413337, 2413340, 2413341, 2413342, 2413343, 2413344, 2413345, 2413346, 2413347, 2413349, 2413350, 2413351, 2413353, 2413358, 2413359, 2413360, 2413367, 2413368, 2413369, 2413372, 2413382, 2413386, 2413387, 2413394, 2413395, 2413397, 2413398, 2413401, 2413402, 2413408, 2413410, 2413412, 2413415, 2413417, 2413427, 2413428, 2413429, 2413430, 2413433, 2413434, 2413436, 2413439, 2413440, 2413441, 2413442, 2413446, 2413447, 2413448, 2413449, 2413451, 2413457, 2413458, 2413459, 2413460, 2413465, 2413469, 2413478, 2413479, 2413484, 2413490, 2413492, 2413493, 2413497, 2413499, 2413508, 2413509, 2413510, 2413514, 2413515, 2413518, 2413528, 2413530, 2413531, 2413533, 2413534, 2413538, 2413539, 2413540, 2413544, 2413548, 2413550, 2413555, 2413558, 2413559, 2413560, 2413567, 2413569, 2413576, 2413577, 2413578, 2413579, 2413581, 2413585, 2413587, 2413595, 2413597, 2413598, 2413599, 2413600, 2413601, 2413602, 2413603, 2413605, 2413607, 2413609, 2413610, 2413621, 2413622, 2413623, 2413624, 2413626, 2413628, 2413631, 2413644, 2413647, 2413648, 2413649, 2413650, 2413653, 2413662, 2413663, 2413666, 2413667, 2413668, 2413669, 2413670, 2413676, 2413677, 2413678, 2413679, 2413686, 2413691, 2413693, 2413694, 2413695, 2413697, 2413701, 2413712, 2413715, 2413716, 2413721, 2413722, 2413724, 2413725, 2413730, 2413731, 2413732, 2413736, 2413741, 2413742, 2413746, 2413747, 2413748, 2413749, 2413755, 2413756, 2413762, 2413764, 2413765, 2413767, 2413773, 2413779, 2413780, 2413781, 2413782, 2413783, 2413784, 2413786, 2413788, 2413794, 2413800, 2413804, 2413806, 2413807, 2413812, 2413813, 2413814, 2413817, 2413822, 2413823, 2413824, 2413826, 2413829, 2413830, 2413831, 2413832, 2413833, 2413836, 2413837, 2413838, 2413839, 2413841, 2413843, 2413847, 2413859, 2413862, 2413863, 2413864, 2413872, 2413873, 2413874, 2413875, 2413876, 2413880, 2413881, 2413882, 2413885, 2413886, 2413887, 2413892, 2413893, 2413894, 2413895, 2413896, 2413898, 2413903, 2413905, 2413911, 2413916, 2413918, 2413919, 2413921, 2413922, 2413923, 2413924, 2413926, 2413927, 2413928, 2413930, 2413931, 2413932, 2413933, 2413935, 2413936, 2413938, 2413939, 2413940, 2413942, 2413951, 2413955, 2413956, 2413957, 2413958, 2413959, 2413960, 2413966, 2413967, 2413968, 2413970, 2413974, 2413978, 2413988, 2413990, 2413991, 2413992, 2413993, 2413994, 2413996, 2413997, 2413998, 2414003, 2414004, 2414009, 2414010, 2414011, 2414012, 2414013, 2414021, 2414026, 2414031, 2414038, 2414045, 2414046, 2414047, 2414051, 2414059, 2414060, 2414062, 2414065, 2414067, 2414068, 2414069, 2414075, 2414077, 2414078, 2414079, 2414081, 2414082, 2414083, 2414085, 2414087, 2414091, 2414092, 2414093, 2414096, 2414098, 2414099, 2414106, 2414107, 2414108, 2414109, 2414110, 2414111, 2414113, 2414115, 2414116, 2414123, 2414124, 2414125, 2414127, 2414128, 2414129, 2414130, 2414135, 2414139, 2414140, 2414141, 2414143, 2414144, 2414145, 2414147, 2414150, 2414153, 2414154, 2414155, 2414167, 2414170, 2414174, 2414177, 2414180, 2414181, 2414185, 2414186, 2414189, 2414190, 2414192, 2414193, 2414197, 2414204, 2414206, 2414207, 2414209, 2414213, 2414218, 2414221, 2414223, 2414227, 2414229, 2414231, 2414236, 2414237, 2414238, 2414239, 2414240, 2414241, 2414249, 2414255, 2414256, 2414257, 2414259, 2414263, 2414264, 2414265, 2414266, 2414268, 2414270, 2414271, 2414272, 2414277, 2414279, 2414280, 2414286, 2414288, 2414295, 2414298, 2414300, 2414301, 2414302, 2414307, 2414308, 2414309, 2414310, 2414311, 2414312, 2414313, 2414314, 2414316, 2414318, 2414319, 2414320, 2414322, 2414325, 2414327, 2414337, 2414341, 2414342, 2414348, 2414353, 2414354, 2414362, 2414363, 2414364, 2414365, 2414366, 2414371, 2414372, 2414373, 2414374, 2414378, 2414380, 2414381, 2414383, 2414391, 2414395, 2414399, 2414400, 2414401, 2414402, 2414406, 2414409, 2414414, 2414416, 2414424, 2414433, 2414435, 2414439, 2414449, 2414450, 2414462, 2414463, 2414466, 2414467, 2414468, 2414472, 2414475, 2414476, 2414477, 2414481, 2414484, 2414486, 2414487, 2414489, 2414493, 2414494, 2414495, 2414496, 2414500, 2414502, 2414503, 2414505, 2414507, 2414509, 2414515, 2414518, 2414519, 2414520, 2414522, 2414523, 2414526, 2414528, 2414531, 2414536, 2414538, 2414540, 2414543, 2414547, 2414548, 2414551, 2414552, 2414554, 2414555, 2414556, 2414557, 2414558, 2414561, 2414562, 2414564, 2414567, 2414571, 2414578, 2414580, 2414582, 2414585, 2414586, 2414587, 2414590, 2414597, 2414604, 2414606, 2414610, 2414615, 2414621, 2414622, 2414633, 2414634, 2414635, 2414640, 2414643, 2414644, 2414645, 2414648, 2414650, 2414652, 2414659, 2414661, 2414662, 2414663, 2414664, 2414665, 2414667, 2414675, 2414677, 2414679, 2414682, 2414683, 2414685, 2414691, 2414694, 2414701, 2414702, 2414705, 2414707, 2414710, 2414714, 2414716, 2414718, 2414722, 2414723, 2414724, 2414729, 2414736, 2414737, 2414739, 2414740, 2414747, 2414749, 2414763, 2414764, 2414765, 2414766, 2414767, 2414770, 2414771, 2414772, 2414775, 2414776, 2414779, 2414781, 2414789, 2414790, 2414791, 2414794, 2414802, 2414806, 2414807, 2414810, 2414815, 2414816, 2414818, 2414819, 2414820, 2414821, 2414825, 2414828, 2414830, 2414839, 2414840, 2414845, 2414847, 2414848, 2414850, 2414851, 2414852, 2414856, 2414859, 2414860, 2414861, 2414862, 2414866, 2414867, 2414868, 2414869, 2414871, 2414872, 2414874, 2414875, 2414876, 2414881, 2414882, 2414883, 2414884, 2414886, 2414887, 2414888, 2414889, 2414890, 2414898, 2414900, 2414901, 2414902, 2414903, 2414910, 2414911, 2414912, 2414913, 2414914, 2414923, 2414925, 2414929, 2414935, 2414937, 2414941, 2414942, 2414943, 2414944, 2414945, 2414946, 2414952, 2414953, 2414955, 2414956, 2414961, 2414966, 2414968, 2414971, 2414973, 2414976, 2414977, 2414979, 2414980, 2414985, 2414989, 2414991, 2414993, 2414994, 2414996, 2414997, 2415006, 2415011, 2415021, 2415023, 2415025, 2415029, 2415031, 2415033, 2415035, 2415041, 2415042, 2415044, 2415048, 2415049, 2415050, 2415052, 2415055, 2415061, 2415064, 2415066, 2415067, 2415068, 2415069, 2415070, 2415071, 2415072, 2415073, 2415189, 2415191, 2415192, 2415193, 2415194, 2415195, 2415196, 2415197, 2415198, 2415202, 2415203, 2415204, 2415206, 2415207, 2415208, 2415211, 2415215, 2415216, 2415217, 2415218, 2415219, 2415220, 2415221, 2415222, 2415223, 2415224, 2415227, 2415229, 2415230, 2415231, 2415232, 2415234, 2415235, 2415236, 2415237, 2415239, 2415240, 2415241, 2415243, 2415244, 2415245, 2415248, 2415249, 2415250, 2415251, 2415255, 2415256, 2415257, 2415258, 2415260, 2415263, 2415264, 2415267, 2415269, 2415270, 2415272, 2415273, 2415274, 2415275, 2415278, 2415280, 2415282, 2415283, 2415286, 2415288, 2415298, 2415299, 2415300, 2415302, 2415305, 2415308, 2415310, 2415311, 2415314, 2415315, 2415316, 2415317, 2415320, 2415321, 2415324, 2415326, 2415328, 2415329, 2415330, 2415332, 2415335, 2415338, 2415339, 2415340, 2415341, 2415342, 2415343, 2415346, 2415348, 2415349, 2415353, 2415354, 2415357, 2415358, 2415359, 2415362, 2415363, 2415366, 2415371, 2415372, 2415373, 2415375, 2415378, 2415380, 2415384, 2415386, 2415387, 2415388, 2415389, 2415390, 2415395, 2415399, 2415400, 2415401, 2415402, 2415406, 2415409, 2415411, 2415413, 2415414, 2415415, 2415417, 2415418, 2415419, 2415423, 2415424, 2415429, 2415430, 2415431, 2415433, 2415439, 2415441, 2415442, 2415443, 2415445, 2415447, 2415448, 2415449, 2415451, 2415452, 2415454, 2415457, 2415458, 2415460, 2415461, 2415463, 2415465, 2415466, 2415467, 2415468, 2415469, 2415470, 2415471, 2415472, 2415474, 2415475, 2415476, 2415477, 2415479, 2415480, 2415481, 2415484, 2415486, 2415488, 2415489, 2415490, 2415491, 2415493, 2415494, 2415499, 2415500, 2415502, 2415503, 2415505, 2415506, 2415507, 2415508, 2415512, 2415513, 2415515, 2415517, 2415521, 2415522, 2415523, 2415527, 2415528, 2415531, 2415534, 2415536, 2415538, 2415539, 2415541, 2415542, 2415543, 2415545, 2415546, 2415551, 2415552, 2415554, 2415559, 2415560, 2415564, 2415565, 2415566, 2415567, 2415569, 2415570, 2415571, 2415573, 2415575, 2415576, 2415581, 2415582, 2415585, 2415587, 2415588, 2415589, 2415590, 2415591, 2415594, 2415595, 2415596, 2415603, 2415604, 2415606, 2415609, 2415610, 2415612, 2415614, 2415615, 2415617, 2415622, 2415623, 2415625, 2415626, 2415628, 2415631, 2415633, 2415635, 2415636, 2415637, 2415638, 2415639, 2415640, 2415642, 2415644, 2415645, 2415647, 2415648, 2415651, 2415653, 2415656, 2415657, 2415659, 2415660, 2415661, 2415662, 2415663, 2415664, 2415665, 2415666, 2415667, 2415669, 2415670, 2415671, 2415672, 2415673, 2415676, 2415677, 2415679, 2415683, 2415684, 2415685, 2415686, 2415687, 2415688, 2415694, 2415695, 2415697, 2415698, 2415699, 2415700, 2415701, 2415706, 2415707, 2415708, 2415710, 2415711, 2415712, 2415713, 2415717, 2415718, 2415719, 2415721, 2415722, 2415723, 2415724, 2415730, 2415731, 2415735, 2415737, 2415739, 2415747, 2415751, 2415755, 2415756, 2415763, 2415765, 2415766, 2415768, 2415770, 2415772, 2415774, 2415775, 2415776, 2415777, 2415779, 2415780, 2415782, 2415783, 2415784, 2415785, 2415786, 2415787, 2415789, 2415791, 2415792, 2415794, 2415796, 2415797, 2415798, 2415800, 2415801, 2415802, 2415804, 2415807, 2415808, 2415809, 2415813, 2415816, 2415817, 2415820, 2415822, 2415824, 2415826, 2415827, 2415828, 2415829, 2415830, 2415833, 2415836, 2415837, 2415840, 2415843, 2415844, 2415845, 2415846, 2415847, 2415848, 2415852, 2415857, 2415859, 2415861, 2415863, 2415865, 2415866, 2415867, 2415868, 2415869, 2415871, 2415875, 2415880, 2415881, 2415883, 2415884, 2415886, 2415890, 2415893, 2415894, 2415897, 2415900, 2415902, 2415904, 2415905, 2415906, 2415908, 2415911, 2415912, 2415916, 2415917, 2415919, 2415921, 2415924, 2415925, 2415928, 2415932, 2415934, 2415935, 2415936, 2415937, 2415941, 2415942, 2415943, 2415944, 2415945, 2415947, 2415948, 2415951, 2415953, 2415955, 2415958, 2415959, 2415965, 2415968, 2415975, 2415978, 2415980, 2415981, 2415982, 2415983, 2415984, 2415986, 2415988, 2415992, 2415994, 2415995, 2415996, 2415997, 2415998, 2416002, 2416003, 2416005, 2416008, 2416009, 2416012, 2416014, 2416016, 2416024, 2416025, 2416027, 2416028, 2416030, 2416031, 2416033, 2416034, 2416035, 2416036, 2416039, 2416041, 2416042, 2416044, 2416045, 2416047, 2416050, 2416051, 2416053, 2416056, 2416058, 2416059, 2416060, 2416065, 2416066, 2416068, 2416072, 2416074, 2416075, 2416076, 2416077, 2416078, 2416079, 2416080, 2416081, 2416084, 2416085, 2416087, 2416088, 2416089, 2416092, 2416093, 2416095, 2416096, 2416097, 2416099, 2416101, 2416105, 2416107, 2416108, 2416109, 2416110, 2416111, 2416113, 2416114, 2416117, 2416118, 2416120, 2416122, 2416123, 2416124, 2416126, 2416128, 2416129, 2416132, 2416134, 2416135, 2416137, 2416138, 2416141, 2416144, 2416148, 2416149, 2416150, 2416151, 2416153, 2416154, 2416157, 2416160, 2416163, 2416164, 2416168, 2416169, 2416170, 2416171, 2416172, 2416174, 2416177, 2416178, 2416179, 2416180, 2416181, 2416182, 2416184, 2416186, 2416188, 2416193, 2416194, 2416196, 2416201, 2416202, 2416203, 2416208, 2416209, 2416210, 2416211, 2416212, 2416215, 2416216, 2416217, 2416220, 2416221, 2416225, 2416227, 2416228, 2416229, 2416231, 2416249, 2416252, 2416253, 2416256, 2416258, 2416259, 2416260, 2416261, 2416262, 2416263, 2416264, 2416265, 2416266, 2416268, 2416269, 2416270, 2416272, 2416273, 2416274, 2416275, 2416276, 2416278, 2416279, 2416281, 2416282, 2416283, 2416287, 2416291, 2416292, 2416298, 2416299, 2416301, 2416302, 2416303, 2416305, 2416306, 2416307, 2416309, 2416311, 2416312, 2416315, 2416319, 2416323, 2416326, 2416327, 2416328, 2416329, 2416333, 2416334, 2416335, 2416336, 2416337, 2416341, 2416343, 2416351, 2416356, 2416359, 2416361, 2416362, 2416365, 2416366, 2416369, 2416371, 2416373, 2416374, 2416375, 2416376, 2416378, 2416380, 2416381, 2416382, 2416383, 2416384, 2416386, 2416389, 2416390, 2416397, 2416398, 2416399, 2416400, 2416401, 2416402, 2416403, 2416405, 2416407, 2416409, 2416413, 2416414, 2416416, 2416417, 2416419, 2416423, 2416424, 2416426, 2416428, 2416432, 2416433, 2416437, 2416438, 2416439, 2416442, 2416444, 2416446, 2416447, 2416448, 2416449, 2416450, 2416453, 2416457, 2416458, 2416459, 2416460, 2416463, 2416464, 2416465, 2416467, 2416468, 2416472, 2416473, 2416474, 2416477, 2416478, 2416479, 2416482, 2416484, 2416485, 2416487, 2416488, 2416490, 2416492, 2416493, 2416494, 2416495, 2416497, 2416498, 2416504, 2416505, 2416506, 2416508, 2416511, 2416513, 2416516, 2416518, 2416520, 2416521, 2416523, 2416525, 2416527, 2416529, 2416531, 2416533, 2416535, 2416536, 2416538, 2416543, 2416554, 2416556, 2416557, 2416560, 2416561, 2416562, 2416563, 2416565, 2416572, 2416574, 2416575, 2416576, 2416578, 2416580, 2416581, 2416582, 2416585, 2416586, 2416588, 2416589, 2416590, 2416596, 2416600, 2416601, 2416604, 2416606, 2416607, 2416610, 2416618, 2416619, 2416623, 2416624, 2416627, 2416628, 2416629, 2416631, 2416632, 2416634, 2416637, 2416638, 2416639, 2416641, 2416645, 2416646, 2416648, 2416649, 2416650, 2416651, 2416652, 2416654, 2416658, 2416661, 2416662, 2416663, 2416665, 2416668, 2416670, 2416672, 2416673, 2416674, 2416676, 2416677, 2416680, 2416683, 2416684, 2416685, 2416690, 2416693, 2416694, 2416695, 2416696, 2416699, 2416701, 2416704, 2416708, 2416710, 2416713, 2416714, 2416716, 2416717, 2416718, 2416721, 2416724, 2416728, 2416729, 2416730, 2416734, 2416735, 2416736, 2416737, 2416743, 2416744, 2416745, 2416747, 2416749, 2416752, 2416753, 2416754, 2416757, 2416758, 2416760, 2416762, 2416765, 2416766, 2416767, 2416768, 2416769, 2416771, 2416774, 2416778, 2416779, 2416782, 2416783, 2416786, 2416787, 2416788, 2416791, 2416793, 2416794, 2416796, 2416801, 2416804, 2416806, 2416807, 2416812, 2416813, 2416816, 2416818, 2416819, 2416820, 2416823, 2416825, 2416826, 2416827, 2416828, 2416830, 2416832, 2416833, 2416834, 2416835, 2416836, 2416840, 2416843, 2416847, 2416849, 2416854, 2416855, 2416856, 2416862, 2416863, 2416867, 2416869, 2416871, 2416873, 2416874, 2416875, 2416882, 2416884, 2416885, 2416886, 2416887, 2416889, 2416891, 2416895, 2416900, 2416901, 2416904, 2416905, 2416906, 2416907, 2416909, 2416910, 2416911, 2416913, 2416914, 2416917, 2416918, 2416919, 2416920, 2416921, 2416923, 2416924, 2416925, 2416926, 2416928, 2416929, 2416930, 2416931, 2416932, 2416933, 2416934, 2416937, 2416939, 2416940, 2416942, 2416945, 2416948, 2416949, 2416953, 2416955, 2416961, 2416962, 2416963, 2416964, 2416965, 2416967, 2416969, 2416970, 2416972, 2416973, 2416974, 2416975, 2416976, 2416977, 2416979, 2416982, 2416985, 2416991, 2416993, 2416995, 2416996, 2416997, 2416998, 2417000, 2417001, 2417002, 2417003, 2417005, 2417006, 2417008, 2417009, 2417012, 2417015, 2417016, 2417019, 2417020, 2417022, 2417023, 2417024, 2417029, 2417030, 2417033, 2417036, 2417038, 2417042, 2417045, 2417047, 2417048, 2417050, 2417051, 2417052, 2417053, 2417054, 2417059, 2417060, 2417061, 2417062, 2417063, 2417064, 2417065, 2417067, 2417068, 2417070, 2417071, 2417072, 2417073, 2417074, 2417075, 2417077, 2417078, 2417082, 2417083, 2417084, 2417085, 2417086, 2417088, 2417089, 2417094, 2417095, 2417097, 2417098, 2417100, 2417103, 2417104, 2417106, 2417108, 2417109, 2417111, 2417112, 2417113, 2417114, 2417116, 2417119, 2417120, 2417121, 2417123, 2417124, 2417125, 2417126, 2417127, 2417128, 2417129, 2417131, 2417132, 2417135, 2417137, 2417139, 2417141, 2417143, 2417145, 2417146, 2417147, 2417151, 2417154, 2417161, 2417162, 2417163, 2417164, 2417166, 2417167, 2417168, 2417169, 2417170, 2417171, 2417172, 2417173, 2417174, 2417176, 2417180, 2417181, 2417182, 2417186, 2417188, 2417191, 2417192, 2417195, 2417196, 2417197, 2417199, 2417200, 2417201, 2417204, 2417210, 2417212, 2417216, 2417217, 2417222, 2417224, 2417225, 2417226, 2417227, 2417228, 2417229, 2417232, 2417233, 2417235, 2417236, 2417238, 2417239, 2417240, 2417241, 2417243, 2417244, 2417245, 2417246, 2417247, 2417252, 2417254, 2417255, 2417256, 2417257, 2417258, 2417262, 2417264, 2417265, 2417268, 2417272, 2417273, 2417274, 2417275, 2417276, 2417278, 2417282, 2417285, 2417287, 2417288, 2417290, 2417291, 2417295, 2417296, 2417301, 2417303, 2417304, 2417306, 2417307, 2417308, 2417314, 2417322, 2417323, 2417326, 2417328, 2417330, 2417334, 2417337, 2417341, 2417342, 2417345, 2417346, 2417347, 2417352, 2417353, 2417354, 2417355, 2417359, 2417360, 2417361, 2417362, 2417363, 2417364, 2417366, 2417367, 2417370, 2417371, 2417372, 2417374, 2417376, 2417377, 2417378, 2417379, 2417380, 2417381, 2417384, 2417389, 2417391, 2417397, 2417398, 2417401, 2417406, 2417407, 2417409, 2417410, 2417411, 2417412, 2417413, 2417419, 2417421, 2417422, 2417423, 2417424, 2417425, 2417427, 2417434, 2417435, 2417436, 2417437, 2417438, 2417441, 2417446, 2417452, 2417454, 2417456, 2417460, 2417461, 2417462, 2417466, 2417467, 2417468, 2417469, 2417470, 2417473, 2417476, 2417478, 2417481, 2417486, 2417487, 2417490, 2417491, 2417493, 2417497, 2417500, 2417501, 2417503, 2417504, 2417505, 2417507, 2417508, 2417509, 2417511, 2417512, 2417523, 2417525, 2417526, 2417531, 2417532, 2417533, 2417536, 2417541, 2417542, 2417543, 2417544, 2417545, 2417546, 2417547, 2417549, 2417550, 2417553, 2417556, 2417558, 2417559, 2417561, 2417564, 2417565, 2417566, 2417568, 2417569, 2417570, 2417575, 2417581, 2417582, 2417584, 2417586, 2417587, 2417588, 2417589, 2417591, 2417592, 2417594, 2417596, 2417598, 2417599, 2417600, 2417601, 2417602, 2417603, 2417605, 2417607, 2417608, 2417610, 2417611, 2417613, 2417614, 2417617, 2417620, 2417622, 2417625, 2417626, 2417627, 2417631, 2417633, 2417635, 2417636, 2417640, 2417641, 2417644, 2417645, 2417646, 2417649, 2417653, 2417655, 2417660, 2417661, 2417662, 2417663, 2417664, 2417667, 2417669, 2417670, 2417677, 2417678, 2417681, 2417682, 2417684, 2417687, 2417689, 2417690, 2417693, 2417694, 2417696, 2417698, 2417700, 2417701, 2417704, 2417705, 2417706, 2417710, 2417712, 2417713, 2417717, 2417718, 2417719, 2417720, 2417722, 2417723, 2417725, 2417727, 2417729, 2417732, 2417733, 2417736, 2417737, 2417739, 2417740, 2417742, 2417743, 2417744, 2417745, 2417748, 2417749, 2417750, 2417752, 2417753, 2417755, 2417756, 2417757, 2417759, 2417761, 2417763, 2417767, 2417768, 2417769, 2417772, 2417773, 2417775, 2417778, 2417779, 2417780, 2417786, 2417789, 2417791, 2417792, 2417793, 2417794, 2417795, 2417797, 2417800, 2417804, 2417805, 2417807, 2417813, 2417816, 2417817, 2417819, 2417820, 2417821, 2417822, 2417827, 2417830, 2417834, 2417838, 2417839, 2417840, 2417841, 2417844, 2417846, 2417847, 2417849, 2417851, 2417852, 2417855, 2417859, 2417860, 2417862, 2417863, 2417864, 2417865, 2417868, 2417872, 2417873, 2417879, 2417882, 2417883, 2417887, 2417888, 2417889, 2417890, 2417891, 2417892, 2417893, 2417894, 2417895, 2417896, 2417897, 2417898, 2417899, 2417902, 2417903, 2417904, 2417906, 2417907, 2417909, 2417910, 2417912, 2417913, 2417915, 2417916, 2417917, 2417918, 2417919, 2417920, 2417921, 2417922, 2417924, 2417925, 2417928, 2417929, 2417931, 2417932, 2417934, 2417935, 2417937, 2417941, 2417943, 2417944, 2417946, 2417947, 2417948, 2417951, 2417952, 2417953, 2417954, 2417956, 2417957, 2417958, 2417961, 2417962, 2417964, 2417965, 2417966, 2417967, 2417970, 2417971, 2417974, 2417977, 2417978, 2417980, 2417981, 2417983, 2417988, 2417989, 2417994, 2417995, 2417997]
G0_MD5 = '78c23bf3c338564adf36a9bf0020292d'
V3_BENCH_IDS = [2315360, 2315392, 2315426, 2315432, 2315482, 2315489, 2315511, 2315514, 2315544, 2315558, 2315569, 2315575, 2315582, 2315584, 2315640, 2315643, 2315676, 2315710, 2315746, 2315784, 2315785, 2315786, 2315801, 2315818, 2315819, 2315825, 2315836, 2315853, 2315878, 2315879, 2315880, 2315894, 2315908, 2315937, 2315950, 2315957, 2315959, 2315971, 2315993, 2316017, 2316029, 2316046, 2316047, 2316050, 2316070, 2316073, 2316080, 2316114, 2316187, 2316188, 2316194, 2316203, 2316211, 2316215, 2316237, 2316258, 2316264, 2316270, 2316284, 2316338, 2316361, 2316364, 2316368, 2316386, 2316392, 2316398, 2316411, 2316429, 2316497, 2316511, 2316520, 2316521, 2316532, 2316536, 2316559, 2316571, 2316601, 2316608, 2316615, 2316631, 2316632, 2316647, 2316663, 2316664, 2316676, 2316678, 2316679, 2316681, 2316690, 2316733, 2316735, 2316765, 2316771, 2316776, 2316778, 2316786, 2316787, 2316810, 2316830, 2316838, 2316840, 2316868, 2316917, 2316924, 2316945, 2316950, 2316955, 2316963, 2316967, 2316972, 2317019, 2317020, 2317030, 2317052, 2317057, 2317060, 2317062, 2317095, 2317122, 2317124, 2317171, 2317202, 2317211, 2317244, 2317246, 2317259, 2317263, 2317273, 2317275, 2317280, 2317290, 2317293, 2317310, 2317311, 2317313, 2317315, 2317333, 2317334, 2317336, 2317355, 2317363, 2317369, 2317370, 2317373, 2317381, 2317382, 2317390, 2317409, 2317428, 2317480, 2317491, 2317509, 2317519, 2317522, 2317527, 2317532, 2317544, 2317571, 2317578, 2317579, 2317606, 2317622, 2317644, 2317645, 2317656, 2317663, 2317666, 2317669, 2317678, 2317708, 2317711, 2317712, 2317723, 2317734, 2317778, 2317823, 2317840, 2317844, 2317859, 2317862, 2317864, 2317887, 2317892, 2317896, 2317897, 2317918, 2317935, 2317944, 2317969, 2318010, 2318014, 2318025, 2318027, 2318029, 2318039, 2318068, 2318085, 2318086, 2318108, 2318145, 2318155, 2318160, 2318203, 2318224, 2318236, 2318239, 2318242, 2318246, 2318255, 2318266, 2318276, 2318279, 2318317, 2318322, 2318343, 2318350, 2318363, 2318383, 2318387, 2318411, 2318430, 2318464, 2318474, 2318487, 2318491, 2318492, 2318495, 2318500, 2318518, 2318537, 2318569, 2318601, 2318628, 2318635, 2318636, 2318640, 2318650, 2318695, 2318712, 2318746, 2318761, 2318772, 2318787, 2318793, 2318807, 2318808, 2318844, 2318869, 2318881, 2318904, 2318916, 2318920, 2318941, 2318959, 2318976, 2318999, 2319006, 2319032, 2319043, 2319071, 2319085, 2319094, 2319104, 2319139, 2319145, 2319189, 2319255, 2319260, 2319263, 2319264, 2319275, 2319287, 2319321, 2319342, 2319367, 2319383, 2319402, 2319409, 2319434, 2319454, 2319457, 2319460, 2319471, 2319472, 2319484, 2319491, 2319552, 2319569, 2319576, 2319595, 2319600, 2319603, 2319605, 2319623, 2319639, 2319657, 2319679, 2319751, 2319753, 2319754, 2319793, 2319820, 2319821, 2319840, 2319841, 2319878, 2319888, 2319889, 2319903, 2319913, 2319927, 2319932, 2319935, 2319938, 2319945, 2319973, 2319983, 2320010, 2320016, 2320028, 2320031, 2320061, 2320064, 2320090, 2320091, 2320092, 2320115, 2320119, 2320163, 2320190, 2320218, 2320221, 2320246, 2320247, 2320257, 2320269, 2320295, 2320303, 2320321, 2320328, 2320362, 2320418, 2320439, 2320460, 2320507, 2320508, 2320509, 2320522, 2320538, 2320539, 2320542, 2320566, 2320568, 2320580, 2320586, 2320590, 2320595, 2320621, 2320638, 2320655, 2320672, 2320718, 2320719, 2320723, 2320730, 2320734, 2320745, 2320756, 2320791, 2320798, 2320799, 2320823, 2320832, 2320849, 2320858, 2320862, 2320877, 2320893, 2320901, 2320931, 2321019, 2321058, 2321060, 2321066, 2321073, 2321083, 2321090, 2321116, 2321124, 2321126, 2321151, 2321164, 2321190, 2321192, 2321210, 2321234, 2321243, 2321260, 2321278, 2321286, 2321291, 2321303, 2321310, 2321314, 2321324, 2321334, 2321369, 2321387, 2321390, 2321399, 2321416, 2321449, 2321497, 2321505, 2321512, 2321557, 2321593, 2321598, 2321625, 2321630, 2321636, 2321650, 2321655, 2321660, 2321684, 2321690, 2321733, 2321740, 2321743, 2321752, 2321757, 2321768, 2321787, 2321788, 2321807, 2321816, 2321852, 2321869, 2321930, 2321985, 2321992, 2322006, 2322012, 2322032, 2322041, 2322060, 2322076, 2322094, 2322122, 2322134, 2322147, 2322204, 2322226, 2322236, 2322263, 2322267, 2322290, 2322314, 2322316, 2322337, 2322353, 2322397, 2322400, 2322420, 2322439, 2322453, 2322464, 2322466, 2322517, 2322534, 2322535, 2322610, 2322683, 2322688, 2322691, 2322692, 2322723, 2322744, 2322747, 2322758, 2322768, 2322808, 2322818, 2322903, 2322913, 2322944, 2322950, 2322956, 2322960, 2323022, 2323089, 2323097, 2323143, 2323154, 2323192, 2323197, 2323238, 2323268, 2323296, 2323297, 2323307, 2323313, 2323337, 2323343, 2323350, 2323352, 2323354, 2323408, 2323444, 2323447, 2323449, 2323451, 2323487, 2323490, 2323506, 2323510, 2323525, 2323529, 2323530, 2323564, 2323579, 2323618, 2323628, 2323652, 2323653, 2323688, 2323700, 2323701, 2323703, 2323758, 2323767, 2323777, 2323794, 2323795, 2323812, 2323846, 2323852, 2323879, 2323888, 2323950, 2323972, 2323987, 2324013, 2324019, 2324024, 2324048, 2324056, 2324065, 2324083, 2324087, 2324089, 2324093, 2324109, 2324121, 2324128, 2324130, 2324133, 2324134, 2324176, 2324180, 2324203, 2324213, 2324231, 2324250, 2324260, 2324264, 2324266, 2324313, 2324327, 2324363, 2324368, 2324379, 2324389, 2324395, 2324397, 2324402, 2324403, 2324412, 2324424, 2324450, 2324533, 2324537, 2324545, 2324557, 2324563, 2324582, 2324591, 2324593, 2324613, 2324622, 2324640, 2324678, 2324700, 2324715, 2324744, 2324754, 2324780, 2324809, 2324847, 2324858, 2324872, 2324874, 2324882, 2324914, 2324921, 2324922, 2324949, 2324954, 2324957, 2324964, 2324967, 2324990, 2324995, 2325009, 2325024, 2325033, 2325057, 2325070, 2325071, 2325091, 2325101, 2325108, 2325112, 2325123, 2325146, 2325192, 2325197, 2325200, 2325226, 2325246, 2325258, 2325266, 2325297, 2325303, 2325312, 2325331, 2325343, 2325344, 2325350, 2325359, 2325391, 2325398, 2325406, 2325428, 2325452, 2325472, 2325507, 2325514, 2325526, 2325556, 2325558, 2325591, 2325597, 2325604, 2325665, 2325685, 2325686, 2325719, 2325747, 2325757, 2325768, 2325773, 2325795, 2325797, 2325825, 2325845, 2325873, 2325878, 2325891, 2325905, 2325950, 2325962, 2325966, 2325989, 2326065, 2326068, 2326087, 2326088, 2326096, 2326117, 2326125, 2326133, 2326148, 2326173, 2326195, 2326201, 2326209, 2326227, 2326234, 2326246, 2326256, 2326337, 2326373, 2326375, 2326393, 2326433, 2326435, 2326439, 2326468, 2326490, 2326496, 2326512, 2326518, 2326544, 2326553, 2326556, 2326568, 2326581, 2326589, 2326601, 2326631, 2326639, 2326643, 2326670, 2326674, 2326707, 2326737, 2326740, 2326749, 2326790, 2326801, 2326849, 2326876, 2326885, 2326892, 2326897, 2326916, 2326940, 2326953, 2326968, 2326998, 2327039, 2327043, 2327045, 2327105, 2327108, 2327113, 2327119, 2327152, 2327204, 2327235, 2327243, 2327250, 2327268, 2327279, 2327293, 2327300, 2327322, 2327335, 2327346, 2327414, 2327419, 2327469, 2327489, 2327493, 2327520, 2327522, 2327526, 2327531, 2327564, 2327594, 2327625, 2327655, 2327683, 2327764, 2327767, 2327778, 2327793, 2327796, 2327843, 2327847, 2327859, 2327871, 2327886, 2327892, 2327939, 2327964, 2327982, 2327983, 2328012, 2328022, 2328029, 2328072, 2328074, 2328082, 2328109, 2328127, 2328136, 2328173, 2328174, 2328181, 2328274, 2328280, 2328281, 2328297, 2328302, 2328317, 2328338, 2328339, 2328347, 2328387, 2328421, 2328425, 2328439, 2328482, 2328508, 2328524, 2328547, 2328554, 2328556, 2328561, 2328569, 2328570, 2328574, 2328606, 2328620, 2328631, 2328638, 2328642, 2328652, 2328653, 2328655, 2328714, 2328722, 2328755, 2328757, 2328767, 2328774, 2328782, 2328788, 2328799, 2328814, 2328833, 2328858, 2328901, 2328915, 2328917, 2328965, 2328992, 2329033, 2329046, 2329064, 2329112, 2329130, 2329149, 2329150, 2329173, 2329184, 2329224, 2329230, 2329236, 2329251, 2329289, 2329301, 2329303, 2329355, 2329378, 2329438, 2329488, 2329500, 2329523, 2329541, 2329592, 2329644, 2329665, 2329669, 2329692, 2329724, 2329783, 2329801, 2329840, 2329865, 2329867, 2329926, 2329927, 2329928, 2330005, 2330035, 2330036, 2330140, 2330141, 2330169, 2330194, 2330329, 2330336, 2330358, 2330360, 2330366, 2330392, 2330401, 2330407, 2330436, 2330442, 2330446, 2330505, 2330517, 2330519, 2330543, 2330549, 2330567, 2330606, 2330654, 2330672, 2330862, 2330883, 2330897, 2330899, 2330902, 2330911, 2330923, 2330968, 2331051, 2331074, 2331077, 2331094, 2331120, 2331134, 2331178, 2331181, 2331183, 2331209, 2331220, 2331227, 2331304, 2331344, 2331354, 2331382, 2331395, 2331396, 2331418, 2331432, 2331433, 2331434, 2331442, 2331454, 2331482, 2331500, 2331577, 2331643, 2331673, 2331679, 2331722, 2331762, 2331765, 2331769, 2331834, 2331863, 2331899, 2332025, 2332066, 2332109, 2332122, 2332179, 2332182, 2332195, 2332212, 2332223, 2332226, 2332227, 2332243, 2332248, 2332253, 2332293, 2332304, 2332329, 2332368, 2332403, 2332422, 2332431, 2332451, 2332466, 2332467, 2332490, 2332525, 2332561, 2332603, 2332632, 2332660, 2332673, 2332683, 2332715, 2332780, 2332811, 2332842, 2332864, 2332865, 2332915, 2332922, 2332985, 2332987, 2332997, 2333017, 2333033, 2333049, 2333054, 2333076, 2333103, 2333105, 2333122, 2333143, 2333163, 2333185, 2333208, 2333237, 2333250, 2333281, 2333339, 2333353, 2333380, 2333383, 2333395, 2333420, 2333422, 2333428, 2333436, 2333451, 2333479, 2333484, 2333489, 2333493, 2333512, 2333593, 2333612, 2333615, 2333656, 2333671, 2333684, 2333686, 2333702, 2333707, 2333741, 2333744, 2333746, 2333753, 2333760, 2333807, 2333819, 2333824, 2333838, 2333861, 2333882, 2333885, 2333901, 2333902, 2333911, 2333914, 2333971, 2333982, 2333989, 2334002, 2334017, 2334023, 2334070, 2334089, 2334144, 2334169, 2334177, 2334188, 2334191, 2334203, 2334213, 2334225, 2334245, 2334246, 2334254, 2334266, 2334291, 2334310, 2334316, 2334326, 2334373, 2334387, 2334435, 2334449, 2334453, 2334455, 2334464, 2334534, 2334541, 2334544, 2334564, 2334603, 2334625, 2334666, 2334670, 2334672, 2334701, 2334709, 2334755, 2334763, 2334770, 2334778, 2334804, 2334805, 2334829, 2334836, 2334853, 2334863, 2334871, 2334874, 2334876, 2334882, 2334889, 2334906, 2334951, 2334954, 2334971, 2334980, 2335051, 2335059, 2335060, 2335078, 2335104, 2335119, 2335126, 2335136, 2335182, 2335200, 2335211, 2335231, 2335275, 2335278, 2335363, 2335378, 2335416, 2335420, 2335422, 2335428, 2335442, 2335454, 2335458, 2335471, 2335477, 2335495, 2335507, 2335525, 2335553, 2335573, 2335574, 2335577, 2335590, 2335623, 2335636, 2335641, 2335648, 2335659, 2335672, 2335679, 2335709, 2335724, 2335744, 2335747, 2335750, 2335799, 2335823, 2335842, 2335868, 2335869, 2335872, 2335873, 2335888, 2335895, 2335926, 2335929, 2335934, 2335939, 2335946, 2335956, 2335965, 2335976, 2335979, 2336004, 2411502, 2411527, 2411539, 2411549, 2411565, 2411567, 2411571, 2411577, 2411594, 2411611, 2411612, 2411625, 2411655, 2411659, 2411666, 2411682, 2411699, 2411743, 2411747, 2411750, 2411756, 2411765, 2411770, 2411790, 2411828, 2411830, 2411854, 2411856, 2411887, 2411895, 2411898, 2411905, 2411927, 2411936, 2411937, 2411958, 2411959, 2411960, 2411961, 2411963, 2411965, 2411981, 2411984, 2411988, 2411995, 2412009, 2412013, 2412014, 2412035, 2412045, 2412092, 2412093, 2412110, 2412120, 2412128, 2412158, 2412193, 2412196, 2412222, 2412233, 2412258, 2412278, 2412281, 2412288, 2412291, 2412292, 2412299, 2412307, 2412309, 2412312, 2412313, 2412342, 2412353, 2412366, 2412371, 2412411, 2412417, 2412422, 2412446, 2412450, 2412455, 2412465, 2412467, 2412476, 2412482, 2412496, 2412497, 2412499, 2412503, 2412505, 2412548, 2412588, 2412622, 2412639, 2412644, 2412666, 2412671, 2412682, 2412692, 2412701, 2412715, 2412716, 2412723, 2412734, 2412737, 2412739, 2412747, 2412777, 2412783, 2412809, 2412858, 2412888, 2412892, 2412941, 2412950, 2412994, 2413037, 2413039, 2413042, 2413045, 2413049, 2413050, 2413054, 2413055, 2413059, 2413066, 2413067, 2413074, 2413075, 2413084, 2413104, 2413130, 2413143, 2413156, 2413186, 2413216, 2413229, 2413252, 2413275, 2413291, 2413322, 2413371, 2413393, 2413409, 2413437, 2413438, 2413468, 2413474, 2413486, 2413487, 2413488, 2413527, 2413529, 2413536, 2413586, 2413634, 2413664, 2413713, 2413719, 2413723, 2413733, 2413745, 2413760, 2413761, 2413763, 2413793, 2413795, 2413799, 2413844, 2413861, 2413889, 2413982, 2414058, 2414070, 2414134, 2414148, 2414166, 2414199, 2414226, 2414258, 2414262, 2414273, 2414305, 2414317, 2414326, 2414330, 2414339, 2414361, 2414384, 2414385, 2414396, 2414408, 2414432, 2414444, 2414464, 2414473, 2414498, 2414501, 2414521, 2414539, 2414577, 2414581, 2414588, 2414592, 2414593, 2414612, 2414620, 2414628, 2414658, 2414673, 2414674, 2414688, 2414769, 2414780, 2414792, 2414799, 2414826, 2414829, 2414835, 2414849, 2414858, 2414891, 2414918, 2414920, 2414954, 2414992, 2415004, 2415005, 2415034, 2415043, 2415063, 2415233, 2415246, 2415253, 2415254, 2415276, 2415277, 2415290, 2415325, 2415368, 2415393, 2415408, 2415412, 2415440, 2415446, 2415453, 2415459, 2415462, 2415501, 2415514, 2415518, 2415524, 2415529, 2415537, 2415549, 2415607, 2415611, 2415616, 2415680, 2415725, 2415726, 2415742, 2415753, 2415799, 2415855, 2415876, 2415879, 2415901, 2415909, 2415920, 2415931, 2415964, 2415971, 2415973, 2416007, 2416061, 2416071, 2416121, 2416142, 2416152, 2416158, 2416190, 2416192, 2416198, 2416233, 2416277, 2416294, 2416324, 2416331, 2416340, 2416345, 2416358, 2416388, 2416393, 2416509, 2416552, 2416555, 2416643, 2416678, 2416709, 2416719, 2416742, 2416746, 2416756, 2416790, 2416792, 2416814, 2416850, 2416853, 2416865, 2416890, 2416897, 2416898, 2416902, 2416927, 2416941, 2416980, 2416987, 2417007, 2417013, 2417026, 2417130, 2417149, 2417152, 2417177, 2417206, 2417208, 2417219, 2417253, 2417319, 2417320, 2417327, 2417349, 2417369, 2417416, 2417453, 2417465, 2417472, 2417498, 2417506, 2417524, 2417560, 2417562, 2417585, 2417590, 2417597, 2417615, 2417623, 2417628, 2417629, 2417634, 2417654, 2417730, 2417738, 2417746, 2417754, 2417760, 2417801, 2417818, 2417825, 2417829, 2417831, 2417854, 2417861, 2417876, 2417880, 2417884, 2417914, 2417985, 2417993]
G0_SHA = {"image_data": "93d0053d7e8f451497646fa513e8fb9b1a889df6e757010b88e51e2e21803487", "rel": "9a7d5f396bd9f1d150e6977c2896ca4feb6024068c6e4c51d571d385f5ab4e64", "obj_dw": "b91429c777881d48708cce0b0bdb2048904c14da3e063babd618d215dd174cfa"}
VIDEO_NB_JSON = '{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "id": "c000",\n   "metadata": {},\n   "source": [\n    "# Project 3 · Cấp độ 4(b) · v5 — đồ thị bằng chứng đa nguồn (CLIP + PE + hub + cầu nối TRAIN + VLM)\\n",\n    "\\n",\n    "**Vì sao có v5**\\n",\n    "* **Cạnh cũ vô ích.** Cạnh quan hệ cấp lớp (đếm từ caption train) không hơn đồ thị rewired trên 4 tập xác nhận.\\n",\n    "* **Probe v4 (VLM).** Đồ thị tách node thua VLM chấm cả câu:\\n",\n    "  * tách node: +0,028 MRR;\\n",\n    "  * VLM chấm cả câu: +0,053 MRR.\\n",\n    "* **Tài liệu** (đã kiểm chứng) cho cùng kết luận: tách truy vấn thành câu hỏi nguyên tử kém chấm cả câu (VQAScore, ECCV 2024).\\n",\n    "  * Đòn bẩy lớn và rẻ nhất là bằng chứng từ một encoder đóng băng mạnh hơn: PE-Core-L, NeurIPS 2025; MSVD R@1 40,5 → 57,2 với cùng giao thức.\\n",\n    "  * Sau đó đến hiệu chỉnh hub bằng caption TRAIN (NNN, EMNLP 2024) và học trọng số kết hợp.\\n",\n    "\\n",\n    "**Đồ thị v5** — mỗi cạnh là một sự thật về một mẫu cụ thể, không còn cạnh lớp–lớp.\\n",\n    "\\n",\n    "| Cạnh | Nối | Nguồn |\\n",\n    "|---|---|---|\\n",\n    "| `clip`, `pe` | truy vấn → video ứng viên | cosine của CLIP ViT-L/14 và PE-Core-L/14-336, đóng băng, cùng 12 frame |\\n",\n    "| `hub_*` | video ứng viên → k caption TRAIN gần nhất | NNN: video \\"hub\\" gần với rất nhiều caption sẽ bị trừ điểm |\\n",\n    "| `br_*` | truy vấn → caption TRAIN → video TRAIN mà caption đó **mô tả** (nhãn thật) → ứng viên | đường cầu nối 3 bước: caption giống truy vấn, rồi video train của nó, rồi video giống về hình |\\n",\n    "| node VLM | truy vấn → ứng viên (top-M) | P(yes) của Qwen2.5-VL chấm **cả câu** trên lưới 3×3 frame của chính video |\\n",\n    "\\n",\n    "* **Ứng viên:** top-K = 30 của CLIP, đúng quy trình của đề.\\n",\n    "* **Điểm giai đoạn 1** = Σ w_t · z(cạnh_t): tuyến tính nên phân rã chính xác theo loại cạnh và theo từng đường cầu nối.\\n",\n    "* **Giai đoạn 2:** top-M = 10 được xếp lại bằng (1−β)·z(giai đoạn 1) + β·z(VLM).\\n",\n    "* **Không có cạnh tới video train cùng nguồn YouTube** với ứng viên (dedup).\\n",\n    "\\n",\n    "**Dữ liệu và chống rò rỉ**\\n",\n    "* Ngân hàng = caption + video **TRAIN**.\\n",\n    "* w, hub_k, bridge_m, τ, β chọn trên **VAL** (2-fold theo video).\\n",\n    "* Đóng băng trước khi mở test.\\n",\n    "* Test MSVD được chấm **một lần**: giai đoạn 1 trên mọi caption, giai đoạn 2 trên 1 caption / video. Đây là lần nhìn thứ 8 ⇒ nhãn **THĂM DÒ**.\\n",\n    "\\n",\n    "**Kaggle**\\n",\n    "1. GPU T4 × 2, Internet On.\\n",\n    "2. Add Input output của Level 4(b) v2 để dùng lại cache CLIP.\\n",\n    "3. Save & Run All.\\n",\n    "4. Tải `project03_level4b_v5_review.zip`.\\n",\n    "\\n",\n    "Ước lượng thời gian:\\n",\n    "* khoảng 20 phút đặc trưng PE;\\n",\n    "* khoảng 25 phút VLM trên VAL;\\n",\n    "* tối đa 120 phút VLM trên test.\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c001",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re, io, socket\\n",\n    "import shutil, zipfile, tempfile, platform, copy, threading, contextlib\\n",\n    "from pathlib import Path\\n",\n    "from collections import defaultdict, Counter\\n",\n    "from contextlib import contextmanager\\n",\n    "\\n",\n    "import numpy as np\\n",\n    "\\n",\n    "RUN_ON_KAGGLE = Path(\\"/kaggle/input\\").exists()\\n",\n    "NB_PID = os.getpid()\\n",\n    "try:\\n",\n    "    _EXEC_COUNT_START = get_ipython().execution_count      # = 1 khi chạy từ kernel mới (Restart & Run All)\\n",\n    "except NameError:\\n",\n    "    _EXEC_COUNT_START = None\\n",\n    "STAGES = []\\n",\n    "\\n",\n    "\\n",\n    "def mark_stage(name, requires=()):\\n",\n    "    \\"\\"\\"Ghi thứ tự chạy các cell chính. Cell chạy lệch thứ tự (hoặc dựa vào biến sót từ phiên trước)\\n",\n    "    dừng ngay với thông báo rõ ràng thay vì cho ra số liệu sai.\\"\\"\\"\\n",\n    "    done = {s[\\"name\\"] for s in STAGES}\\n",\n    "    missing = [r for r in requires if r not in done]\\n",\n    "    assert not missing, f\\"Cell \'{name}\' phải chạy sau {missing} — hãy Restart & Run All.\\"\\n",\n    "    assert os.getpid() == NB_PID, \\"PID kernel đã đổi giữa chừng — hãy Restart & Run All.\\"\\n",\n    "    STAGES.append(dict(name=name, t=time.time(), pid=os.getpid()))\\n",\n    "\\n",\n    "\\n",\n    "def has_internet(host=\\"huggingface.co\\", port=443, timeout=5.0):\\n",\n    "    try:\\n",\n    "        socket.create_connection((host, port), timeout=timeout).close()\\n",\n    "        return True\\n",\n    "    except OSError:\\n",\n    "        return False\\n",\n    "\\n",\n    "\\n",\n    "INTERNET = has_internet()\\n",\n    "mark_stage(\\"env\\")\\n",\n    "print(f\\"Kaggle: {RUN_ON_KAGGLE} | Internet: {INTERNET} | Python {platform.python_version()} | \\"\\n",\n    "      f\\"execution_count đầu = {_EXEC_COUNT_START}\\")\\n",\n    "if not INTERNET:\\n",\n    "    print(\\"⚠ Không có Internet: chỉ chạy được nếu open_clip, trọng số CLIP, dữ liệu NLTK và MSVD đã có sẵn \\"\\n",\n    "          \\"(Add Input). Bật: Kaggle → Settings → Internet → On.\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c002",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# Phụ thuộc tối thiểu. Kaggle đã có torch/torchvision/opencv/nltk/pandas/scipy; chỉ cài open_clip khi thiếu.\\n",\n    "# Cài thất bại thì DỪNG kèm log (không bỏ qua lỗi cài đặt).\\n",\n    "def ensure_package(module, pip_spec):\\n",\n    "    import importlib\\n",\n    "    try:\\n",\n    "        return importlib.import_module(module)\\n",\n    "    except ImportError:\\n",\n    "        pass\\n",\n    "    if not INTERNET:\\n",\n    "        raise RuntimeError(f\\"Thiếu gói \'{pip_spec}\' và không có Internet — bật Internet hoặc Add Input wheel.\\")\\n",\n    "    print(f\\"→ pip install {pip_spec}\\")\\n",\n    "    r = subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", pip_spec], capture_output=True, text=True)\\n",\n    "    if r.returncode != 0:\\n",\n    "        print(r.stdout[-3000:]); print(r.stderr[-3000:])\\n",\n    "        raise RuntimeError(f\\"pip install {pip_spec} thất bại (mã {r.returncode}) — xem log phía trên.\\")\\n",\n    "    importlib.invalidate_caches()\\n",\n    "    return importlib.import_module(module)\\n",\n    "\\n",\n    "\\n",\n    "cv2 = ensure_package(\\"cv2\\", \\"opencv-python-headless\\")\\n",\n    "open_clip = ensure_package(\\"open_clip\\", \\"open_clip_torch==3.2.0\\")\\n",\n    "nltk = ensure_package(\\"nltk\\", \\"nltk\\")\\n",\n    "\\n",\n    "import scipy\\n",\n    "import scipy.sparse as sp\\n",\n    "import pandas as pd\\n",\n    "import torch\\n",\n    "import torch.nn as nn\\n",\n    "import torch.nn.functional as F\\n",\n    "from PIL import Image\\n",\n    "from tqdm.auto import tqdm\\n",\n    "from IPython.display import display\\n",\n    "\\n",\n    "DEVICE = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n",\n    "ENV_INFO = dict(python=platform.python_version(), platform=platform.platform(), torch=torch.__version__,\\n",\n    "                cuda=torch.version.cuda, device=DEVICE, open_clip=getattr(open_clip, \\"__version__\\", \\"?\\"),\\n",\n    "                opencv=cv2.__version__, numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,\\n",\n    "                nltk=nltk.__version__, kaggle=RUN_ON_KAGGLE, internet=INTERNET)\\n",\n    "if DEVICE == \\"cuda\\":\\n",\n    "    ENV_INFO[\\"gpu\\"] = torch.cuda.get_device_name(0)\\n",\n    "    print(f\\"GPU: {ENV_INFO[\'gpu\']} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\\")\\n",\n    "else:\\n",\n    "    print(\\"⚠ KHÔNG có GPU: trích đặc trưng ViT-L-14 cho ~23,6k frame + 80,8k caption trên CPU rất chậm (hàng giờ).\\\\n\\"\\n",\n    "          \\"  Kaggle: Settings → Accelerator → GPU T4/P100. Chạy thử trên CPU: MSVD_L4_CLIP=ViT-B-32, \\"\\n",\n    "          \\"MSVD_L4_CLIP_W=laion2b_s34b_b79k.\\")\\n",\n    "print(json.dumps(ENV_INFO, indent=1))\\n",\n    "mark_stage(\\"deps\\", requires=(\\"env\\",))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c003",\n   "metadata": {},\n   "source": [\n    "## 0. Cấu hình\\n",\n    "\\n",\n    "Mọi siêu tham số nằm ở cell dưới. Phần dữ liệu, CLIP, lấy mẫu frame và tagger giữ nguyên như v1/v2, nên cache đặc trưng MSVD của v2 dùng lại được. Các khối mới của v3 gồm:\\n",\n    "* vị từ: họ có vị từ tự do, prompt, ngưỡng support;\\n",\n    "* người chấm: mẫu, ngưỡng precision, file nhãn;\\n",\n    "* benchmark xác nhận: MSR-VTT ở commit ghim;\\n",\n    "* thống kê: Holm."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c004",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "_env = os.environ.get\\n",\n    "\\n",\n    "\\n",\n    "class CFG:\\n",\n    "    # ---------------- chế độ chạy ----------------\\n",\n    "    # \\"msvd\\": dữ liệu thật (mặc định). \\"synthetic\\": smoke-test mã bằng video tổng hợp — phải chọn TƯỜNG MINH.\\n",\n    "    # Notebook KHÔNG bao giờ tự rơi về synthetic khi thiếu MSVD: chỉ có caption thì không phải text-to-video.\\n",\n    "    dataset  = _env(\\"MSVD_L4_DATASET\\", \\"msvd\\")\\n",\n    "    # dev_mode: chạy đủ pipeline trên dữ liệu thật nhưng KHOÁ test; phân tích cuối chạy trên val, gắn nhãn DEV.\\n",\n    "    dev_mode = _env(\\"MSVD_L4_DEV\\", \\"0\\") == \\"1\\"\\n",\n    "\\n",\n    "    # ---------------- dữ liệu MSVD ----------------\\n",\n    "    hf_repo      = \\"friedrichor/MSVD\\"\\n",\n    "    hf_revision  = \\"9aa95a3039c106ee9e0969d82d7ce5a9efab1196\\"     # ghim commit ⇒ tái lập được\\n",\n    "    msvd_root    = _env(\\"MSVD_ROOT\\", \\"\\")             # thư mục chứa msvd_*.json + YouTubeClips/ hoặc MSVD_Videos.zip\\n",\n    "    video_source = _env(\\"MSVD_VIDEO_SOURCE\\", \\"auto\\")  # auto | dir | zip\\n",\n    "    expected     = dict(train=(1200, 48774), val=(100, 4290), test=(670, 27763))   # (video, caption) — dataset card\\n",\n    "    max_bad_video_frac = 0.0                         # MSVD thật: không chấp nhận video nào giải mã lỗi\\n",\n    "\\n",\n    "    # ---------------- lấy mẫu khung hình ----------------\\n",\n    "    n_frames_max   = 12\\n",\n    "    frame_policy   = \\"uniform-segment-midpoint\\"      # frame k = floor((k + 0.5) / F · n_frame_giải_mã_được)\\n",\n    "    n_frames_grid  = [4, 8, 12]                       # chọn trên VAL; F=1 chỉ in ra để tham chiếu\\n",\n    "    pool_grid      = [\\"mean\\", \\"centrality\\"]           # chọn trên VAL\\n",\n    "    centrality_tau = 0.1\\n",\n    "    decode_workers = 4\\n",\n    "    thumb_width    = 160\\n",\n    "\\n",\n    "    # ---------------- CLIP (đóng băng hoàn toàn) ----------------\\n",\n    "    clip_model   = _env(\\"MSVD_L4_CLIP\\", \\"ViT-L-14\\")\\n",\n    "    clip_weights = _env(\\"MSVD_L4_CLIP_W\\", \\"laion2b_s32b_b82k\\")\\n",\n    "    clip_bs      = 128\\n",\n    "    text_bs      = 512\\n",\n    "\\n",\n    "    # ---------------- từ vựng (TRAIN) + tagger (chọn trên VAL) ----------------\\n",\n    "    vocab_min_caps  = 2        # một từ được tính cho video nếu xuất hiện trong ≥ 2 caption của video đó ...\\n",\n    "    min_term_df     = 3        # ... và ở ≥ 3 video TRAIN\\n",\n    "    truth_min_caps  = 3        # nhãn tham chiếu (chỉ để chấm F1 của tagger trên VAL)\\n",\n    "    tag_norms       = [\\"raw\\", \\"z\\", \\"raw+z\\"]\\n",\n    "    tag_pools       = [\\"video\\", \\"frame_max\\"]\\n",\n    "    tag_k_noun_grid = [3, 5, 8, 10, 12]\\n",\n    "    tag_k_verb_grid = [1, 2, 3, 5]\\n",\n    "    noun_prompts = [\\"a photo of a {}.\\", \\"a video frame showing a {}.\\", \\"a photo containing a {}.\\"]\\n",\n    "    verb_prompts = [\\"a photo of someone {}.\\", \\"a video frame of a person {}.\\", \\"someone is {}.\\"]\\n",\n    "    min_tag_freq = 2           # lớp phải được gắn cho ≥ 2 video TRAIN mới thành node\\n",\n    "    max_concepts = 4000\\n",\n    "\\n",\n    "    # ---------------- đồ thị (cùng ý nghĩa với notebook gốc) ----------------\\n",\n    "    max_path_len = 3\\n",\n    "    scene_min_count, scene_top_k = 2, 10\\n",\n    "    cooc_min_count, cooc_top_k = 5, 10\\n",\n    "    cooc_score, cooc_min_npmi = \\"npmi\\", 0.10\\n",\n    "    use_wordnet, isa_min_depth = True, 4\\n",\n    "    use_idf, idf_from = True, \\"taxonomy\\"\\n",\n    "    drop_redundant_indirect = True\\n",\n    "    edge_weighting = True\\n",\n    "\\n",\n    "    # ---------------- GNN ----------------\\n",\n    "    gnn_type, gnn_typed = \\"sage\\", True\\n",\n    "    gate_init, path_init_indirect, scalar_lr_mult = -1.5, -1.0, 10.0\\n",\n    "    hard_neg_k = 31\\n",\n    "    train_with_clip = True\\n",\n    "    learn_clip_w = False       # chọn lại trên VAL\\n",\n    "    gnn_dim, gnn_layers, gnn_dropout = 256, 2, 0.1\\n",\n    "    lr, weight_decay, temperature = 1e-3, 1e-4, 0.07\\n",\n    "    epochs = int(_env(\\"MSVD_L4_EPOCHS\\", \\"15\\"))\\n",\n    "    caption_passes = 4         # mỗi epoch: 4 lượt × (1 caption ngẫu nhiên / video TRAIN), batch không trùng video\\n",\n    "    train_bs = 128\\n",\n    "\\n",\n    "    # ---------------- xếp hạng lại ----------------\\n",\n    "    rerank_k_grid = [10, 20, 50]                       # chọn trên VAL (cấu hình \'none\', seed đầu)\\n",\n    "    rerank_k      = None\\n",\n    "    alpha_grid    = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]\\n",\n    "    lambda_grid   = [0.0, 0.25, 0.5, 0.75]             # bỏ λ=1: khi α<1, s_path luôn có trọng số > 0\\n",\n    "    eval_chunk    = 2048\\n",\n    "\\n",\n    "    # ---------------- thí nghiệm (v2: theo stage, mỗi stage một câu hỏi) ----------------\\n",\n    "    seeds  = [int(s) for s in _env(\\"MSVD_L4_SEEDS\\", \\"0,1,2,3,4\\").split(\\",\\")]\\n",\n    "    stages = [s for s in _env(\\"MSVD_V4_STAGES\\", \\"V1,V2,V7,V8\\").split(\\",\\") if s]\\n",\n    "    # Luật CHẤP NHẬN một thay đổi (đăng ký trước, chỉ dùng VAL): Δ MRR val trung bình seed ≥ adopt_min_delta\\n",\n    "    # VÀ bootstrap theo cụm video trên val cho P(Δ ≤ 0) < adopt_max_p. Không đạt ⇒ giữ thiết kế đang dùng.\\n",\n    "    adopt_min_delta, adopt_max_p = 0.002, 0.10\\n",\n    "    save_checkpoints = _env(\\"MSVD_G2_CKPT\\", \\"1\\") == \\"1\\"\\n",\n    "\\n",\n    "    # ---------------- đồ thị v2 ----------------\\n",\n    "    reliable_min_support = 3          # quan hệ \'đáng tin\' = được ≥ 3 video TRAIN hỗ trợ (không có nhãn người ⇒ chỉ là lọc theo tần suất)\\n",\n    "    tax_hub_frac = 0.15               # taxonomy \'lọc hub\': bỏ khái niệm mà > 15% video TRAIN với tới\\n",\n    "    tax_min_depth_filtered = 6        # ... và khái niệm danh từ quá nông (độ sâu WordNet < 6)\\n",\n    "    sense_max_candidates, sense_margin = 6, 1.0   # chọn sense kiểu Lesk trên ngữ cảnh caption TRAIN\\n",\n    "    conf_modes = [\\"sigmoid\\", \\"minmax\\"]            # chuẩn hoá trọng số contains — chọn trên VAL theo MRR\\n",\n    "    conf_floor = 0.05\\n",\n    "    rewire_swaps_per_edge = 10        # số lần thử swap / cạnh trong constrained rewiring\\n",\n    "    rewire_min_changed = 0.5          # đối chứng hợp lệ ⇔ ≥ 50% cạnh của mỗi loại (≥ 10 cạnh) bị đổi\\n",\n    "    n_audit_captions = 200            # mẫu caption TRAIN cho audit quan hệ (để người chấm)\\n",\n    "\\n",\n    "    # ---------------- v3: vị từ (predicate) là thành phần của cạnh ----------------\\n",\n    "    # Họ có vị từ TỰ DO: svo (vị từ = cụm động từ), prep (vị từ = giới từ). Ở agent_of / object_of vị từ chính là node\\n",\n    "    # động từ đích ⇒ không mang thông tin riêng của cạnh; vẫn ghi vào schema nhưng không đưa embedding vị từ vào message.\\n",\n    "    pred_families = (\\"svo\\", \\"prep\\")\\n",\n    "    pred_min_support = 5              # chính sách \'support\': (họ, vị từ) xuất hiện ở ≥ 5 video TRAIN\\n",\n    "    pred_prompts = {\\"svo\\": [\\"someone {} something.\\", \\"a video of a person {} something.\\"],\\n",\n    "                    \\"prep\\": [\\"something {} something else.\\", \\"a photo of an object {} another object.\\"]}\\n",\n    "    pred_adapt_hidden = 128           # phương án B: MLP dư DÙNG CHUNG cho mọi vị từ (không có ID riêng cho từng vị từ)\\n",\n    "    # ---------------- chất lượng quan hệ: kiểm chứng TỰ ĐỘNG (mặc định) + nhãn người chấm (TUỲ CHỌN) ----------------\\n",\n    "    dep_model = \\"en_core_web_sm\\"      # parser phụ thuộc cú pháp độc lập (spaCy) để đối chiếu với parser heuristic\\n",\n    "    agree_min = 0.5                   # chính sách \'agreement\': giữ (họ, nguồn, vị từ, đích) nếu ≥ 50% caption TRAIN của nó được spaCy xác nhận\\n",\n    "    annot_n_per_cell, annot_n_double_per_cell, annot_seed = 30, 10, 20261024\\n",\n    "    annot_strata = ((2, 2, \\"low\\"), (3, 5, \\"mid\\"), (6, 10 ** 9, \\"high\\"))   # support = số video TRAIN có quan hệ\\n",\n    "    annot_file = _env(\\"MSVD_G3_LABELS\\", \\"\\")                               # trống ⇒ tự tìm relation_annotation*.csv trong /kaggle/input\\n",\n    "    prec_min_lower = 0.60             # chính sách \'precision\': giữ ô (họ × mức phổ biến) có cận dưới Wilson 95% ≥ 0.60\\n",\n    "    # ---------------- benchmark xác nhận ĐỘC LẬP (chưa dùng ở v1/v2) ----------------\\n",\n    "    confirm = _env(\\"MSVD_G3_CONFIRM\\", \\"none\\")                           # \\"msrvtt\\" | \\"none\\"\\n",\n    "    msrvtt_repo = \\"friedrichor/MSR-VTT\\"\\n",\n    "    msrvtt_revision = \\"c1af215a96934854f42683c19c51391aaee6f962\\"\\n",\n    "    msrvtt_dev_n = 200                # dev_mode: 200 video TRAIN-9k (KHÔNG thuộc test 1k-A) chỉ để thử đường ống\\n",\n    "    msrvtt_root = _env(\\"MSRVTT_ROOT\\", \\"\\")  # thư mục có sẵn msrvtt_*.json / MSRVTT_Videos.zip (trống ⇒ tải từ HF)\\n",\n    "    confirm_set = _env(\\"MSVD_V4_CONFIRM_SET\\", \\"fresh1kC\\")  # \\"fresh1kC\\" (mặc định, CHƯA nhìn) | \\"fresh1kB\\" (Level 4(b) v1) | \\"fresh1k\\" (v4) | \\"1kA\\" (v3)\\n",\n    "    fresh_n, fresh_seed = 1000, 20261025  # fresh1k: 1000 video train-9k, ngoài 1k-A và ngoài 200 video dev — chọn ngẫu nhiên cố định\\n",\n    "    fresh_b_seed = 20261026           # fresh1k-B (Level 4(b)): 1000 video train-9k KHÁC — rời 1k-A, rời dev, rời fresh1k\\n",\n    "    fresh_c_seed = 20261031           # fresh1k-C (Level 4(b) v2): rời 1k-A, dev, fresh1k VÀ fresh1k-B\\n",\n    "    # ---------------- Level 4(b) v2: tagger học + ConceptNet ----------------\\n",\n    "    probe_wd_grid = [1e-4, 1e-3, 1e-2, 1e-1]   # weight decay của tagger học — chọn trên VAL (F1)\\n",\n    "    probe_epochs, probe_lr, probe_folds = 300, 1e-2, 5\\n",\n    "    kg_version = \\"5.7.0\\"\\n",\n    "    kg_url = \\"https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz\\"\\n",\n    "    kg_file = _env(\\"MSVD_KG_FILE\\", \\"\\")       # file assertions .csv.gz có sẵn (trống ⇒ /kaggle/input hoặc tải từ kg_url)\\n",\n    "    kg_max_lines = int(_env(\\"MSVD_KG_MAX_LINES\\", \\"0\\"))   # 0 = đọc hết (chỉ giới hạn khi thử mã)\\n",\n    "    # ---------------- Level 4(b) ----------------\\n",\n    "    n_eval_queries = 20               # bộ truy vấn đăng ký trước: số truy vấn / nhóm (trực tiếp, gián tiếp) / tập đánh giá\\n",\n    "    eval_query_seed = 20261026\\n",\n    "    n_cases = 5                       # 5 thành công + 5 thất bại\\n",\n    "    demo_topk = 5\\n",\n    "    demo_queries = [\\"a man is playing a guitar\\", \\"a child is playing football on the grass\\", \\"a woman is slicing a tomato\\",\\n",\n    "                    \\"a dog is running on the beach\\", \\"two people are dancing on a stage\\"]   # cố định trước, không chỉnh theo kết quả\\n",\n    "    # ---------------- v4: QC-RGAT ----------------\\n",\n    "    v4_dim = 64                       # chiều embedding của QC-RGAT\\n",\n    "    v4_lr = float(_env(\\"MSVD_V4_LR\\", \\"2e-3\\"))\\n",\n    "    v4_score_bs = 256                 # số truy vấn / lô khi chấm điểm\\n",\n    "    v4_rerank_k = 50                  # k chọn trên VAL ở v3 (Kaggle), giữ cố định\\n",\n    "    # ---------------- thống kê ----------------\\n",\n    "    holm_alpha = 0.05                 # Holm–Bonferroni cho họ so sánh chính đăng ký trước\\n",\n    "\\n",\n    "    # ---------------- xAI ----------------\\n",\n    "    n_explain_queries, top_m_paths = 20, 5\\n",\n    "    n_rank_fidelity, n_edge_deletion = 50, 15\\n",\n    "    n_seed_sensitivity = 20\\n",\n    "\\n",\n    "    # ---------------- đường dẫn ----------------\\n",\n    "    _base      = Path(\\"/kaggle/working\\") if RUN_ON_KAGGLE else Path(\\".\\")\\n",\n    "    out_dir    = Path(_env(\\"MSVD_L4_OUT\\", str(_base / (\\"project03_level4b_v5\\" if dataset == \\"msvd\\" else \\"project03_level4b_v5_synthetic\\"))))\\n",\n    "    data_cache = Path(_env(\\"MSVD_L4_DATA_CACHE\\", \\"/tmp/msvd_hf\\" if RUN_ON_KAGGLE else str(_base / \\"msvd_hf\\")))\\n",\n    "    extra_cache_dirs = [d for d in _env(\\"MSVD_L4_CACHE_DIRS\\", \\"\\").split(\\":\\") if d]\\n",\n    "\\n",\n    "\\n",\n    "assert CFG.dataset in (\\"msvd\\", \\"synthetic\\"), f\\"CFG.dataset không hợp lệ: {CFG.dataset}\\"\\n",\n    "RESULTS = CFG.out_dir / \\"results\\"\\n",\n    "for _d in (CFG.out_dir / \\"cache\\", RESULTS, CFG.out_dir / \\"thumbs\\", CFG.out_dir / \\"checkpoints\\"):\\n",\n    "    _d.mkdir(parents=True, exist_ok=True)\\n",\n    "RES_TAG = \\"DEV_val\\" if CFG.dev_mode else (\\"test\\" if CFG.dataset == \\"msvd\\" else \\"SYNTHETIC_test\\")\\n",\n    "\\n",\n    "\\n",\n    "def set_seed(seed):\\n",\n    "    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\\n",\n    "    if torch.cuda.is_available():\\n",\n    "        torch.cuda.manual_seed_all(seed)\\n",\n    "\\n",\n    "\\n",\n    "def _jsonable(o):\\n",\n    "    if isinstance(o, np.integer): return int(o)\\n",\n    "    if isinstance(o, np.floating): return float(o)\\n",\n    "    if isinstance(o, np.bool_): return bool(o)\\n",\n    "    if isinstance(o, np.ndarray): return o.tolist()\\n",\n    "    if isinstance(o, (set, frozenset)): return sorted(o)\\n",\n    "    return str(o)\\n",\n    "\\n",\n    "\\n",\n    "def save_json(obj, path):\\n",\n    "    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\\n",\n    "    with open(path, \\"w\\") as f:\\n",\n    "        json.dump(obj, f, ensure_ascii=False, indent=2, default=_jsonable)\\n",\n    "    return path\\n",\n    "\\n",\n    "\\n",\n    "def sha1_of(obj):\\n",\n    "    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_jsonable).encode()).hexdigest()[:12]\\n",\n    "\\n",\n    "\\n",\n    "def cfg_snapshot():\\n",\n    "    return {k: v for k, v in vars(CFG).items() if not k.startswith(\\"_\\") and not callable(v)}\\n",\n    "\\n",\n    "\\n",\n    "CHECKS = []\\n",\n    "\\n",\n    "\\n",\n    "def check(cid, desc, passed, evidence=\\"\\", hard=True, status=None):\\n",\n    "    \\"\\"\\"Ghi một kiểm tra vào sổ audit. hard=True ⇒ FAIL dừng notebook ngay (không chạy tiếp trên dữ liệu sai).\\"\\"\\"\\n",\n    "    st = status or (\\"PASS\\" if passed else \\"FAIL\\")\\n",\n    "    CHECKS.append(dict(id=cid, check=desc, status=st, evidence=str(evidence)[:900]))\\n",\n    "    print(f\\"  [{st}] {cid} — {desc}: {evidence}\\")\\n",\n    "    if hard and st == \\"FAIL\\":\\n",\n    "        raise AssertionError(f\\"Kiểm tra {cid} THẤT BẠI: {desc} — {evidence}\\")\\n",\n    "    return st == \\"PASS\\"\\n",\n    "\\n",\n    "\\n",\n    "save_json(dict(cfg=cfg_snapshot(), env=ENV_INFO), RESULTS / \\"config_initial.json\\")\\n",\n    "print(f\\"dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | CLIP={CFG.clip_model}/{CFG.clip_weights} | \\"\\n",\n    "      f\\"seeds={CFG.seeds} | epochs={CFG.epochs} | out={CFG.out_dir.resolve()}\\")\\n",\n    "if CFG.dataset == \\"synthetic\\":\\n",\n    "    print(\\"⚠⚠ CHẾ ĐỘ SYNTHETIC: chỉ smoke-test mã. Mọi con số dưới đây KHÔNG phải kết quả MSVD.\\")\\n",\n    "if CFG.dev_mode:\\n",\n    "    print(\\"⚠ dev_mode: test bị KHOÁ; mọi phân tích cuối chạy trên VAL, gắn nhãn DEV — không dùng để báo cáo.\\")\\n",\n    "mark_stage(\\"config\\", requires=(\\"deps\\",))"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c005",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# --- Gói kết quả để TẢI VỀ xem xét: MỘT file zip nhẹ <out_dir>_review.zip, nằm cạnh thư mục output. ---\\n",\n    "# Gồm: results/** (mọi CSV/JSON/MD, log truy cập, pip freeze), notebook đã chạy (nếu tìm được), pack_manifest.json.\\n",\n    "# KHÔNG gồm dữ liệu và các file nặng suy ra từ dữ liệu: video MSVD (CFG.data_cache), cache/ (đặc trưng CLIP),\\n",\n    "# thumbs/ (khung hình video), checkpoints/, synthetic_videos/. Manifest vẫn ghi số file và dung lượng của chúng.\\n",\n    "# Cell cuối gọi pack_review(); nếu một cell bị LỖI, hook bên dưới gói ngay những gì đã có, kèm traceback.\\n",\n    "import traceback\\n",\n    "\\n",\n    "REVIEW_ZIP = CFG.out_dir.parent / f\\"{CFG.out_dir.name}_review.zip\\"\\n",\n    "PACK_MAX_FILE_MB = 50\\n",\n    "\\n",\n    "\\n",\n    "def _find_executed_notebook():\\n",\n    "    \\"\\"\\"Kaggle chạy bằng papermill (input = output = __notebook__.ipynb) và lưu notebook sau mỗi cell, nên bản này\\n",\n    "    có output của mọi cell đã chạy xong. Chỉ nhận file được ghi SAU khi kernel này khởi động (tránh bản cũ).\\"\\"\\"\\n",\n    "    t0 = STAGES[0][\\"t\\"] if STAGES else 0.0\\n",\n    "    cands = [Path.cwd() / \\"__notebook__.ipynb\\", *sorted(Path(\\"/kaggle\\").glob(\\"*/__notebook__.ipynb\\")),\\n",\n    "             *sorted(Path(\\"/kaggle/src\\").glob(\\"*.ipynb\\"))]\\n",\n    "    for p in cands:\\n",\n    "        try:\\n",\n    "            if p.is_file() and p.stat().st_mtime >= t0 and json.loads(p.read_text()).get(\\"cells\\"):\\n",\n    "                return p\\n",\n    "        except Exception:\\n",\n    "            continue\\n",\n    "    return None\\n",\n    "\\n",\n    "\\n",\n    "def _dir_size(d):\\n",\n    "    fs = [f for f in d.rglob(\\"*\\") if f.is_file()] if d.is_dir() else [d]\\n",\n    "    return dict(n_files=len(fs), mb=round(sum(f.stat().st_size for f in fs) / 2**20, 1))\\n",\n    "\\n",\n    "\\n",\n    "def pack_review(reason, error=None):\\n",\n    "    \\"\\"\\"Ghi lại REVIEW_ZIP từ đầu (gọi nhiều lần được). Không bao giờ ném lỗi ra ngoài: trả về None nếu thất bại.\\"\\"\\"\\n",\n    "    try:\\n",\n    "        root, inc, too_large = REVIEW_ZIP.stem, [], []\\n",\n    "        nb_path = _find_executed_notebook()\\n",\n    "        tmp = REVIEW_ZIP.with_name(REVIEW_ZIP.name + \\".tmp\\")\\n",\n    "        with zipfile.ZipFile(tmp, \\"w\\", zipfile.ZIP_DEFLATED, compresslevel=6) as z:\\n",\n    "            for p in sorted(p for p in RESULTS.rglob(\\"*\\") if p.is_file()):\\n",\n    "                rel, size = p.relative_to(CFG.out_dir).as_posix(), p.stat().st_size\\n",\n    "                if size > PACK_MAX_FILE_MB * 2**20:\\n",\n    "                    too_large.append(dict(path=rel, mb=round(size / 2**20, 1)))\\n",\n    "                    continue\\n",\n    "                z.write(p, f\\"{root}/{rel}\\")\\n",\n    "                inc.append(dict(path=rel, bytes=size))\\n",\n    "            if nb_path is not None:\\n",\n    "                z.write(nb_path, f\\"{root}/executed_notebook.ipynb\\")\\n",\n    "            excluded = {d.name + (\\"/\\" if d.is_dir() else \\"\\"): _dir_size(d) for d in sorted(CFG.out_dir.iterdir()) if d != RESULTS}\\n",\n    "            excluded[f\\"video MSVD ({CFG.data_cache})\\"] = \\"không gói\\"\\n",\n    "            manifest = dict(created=time.strftime(\\"%Y-%m-%d %H:%M:%S\\"), reason=reason, status=\\"error\\" if error else \\"ok\\",\\n",\n    "                            error_traceback=error, dataset=CFG.dataset, dev_mode=CFG.dev_mode, res_tag=RES_TAG, kaggle=RUN_ON_KAGGLE,\\n",\n    "                            stages_done=[s[\\"name\\"] for s in STAGES], checks=dict(Counter(c[\\"status\\"] for c in CHECKS)),\\n",\n    "                            checks_fail=[c[\\"id\\"] for c in CHECKS if c[\\"status\\"] == \\"FAIL\\"],\\n",\n    "                            executed_notebook=str(nb_path) if nb_path else None, included=inc, skipped_too_large=too_large,\\n",\n    "                            excluded=excluded)\\n",\n    "            z.writestr(f\\"{root}/pack_manifest.json\\", json.dumps(manifest, ensure_ascii=False, indent=2, default=_jsonable))\\n",\n    "        tmp.replace(REVIEW_ZIP)\\n",\n    "        return REVIEW_ZIP\\n",\n    "    except Exception as e:\\n",\n    "        print(f\\"⚠ Không đóng gói được {REVIEW_ZIP}: {type(e).__name__}: {e}\\")\\n",\n    "        return None\\n",\n    "\\n",\n    "\\n",\n    "def _pack_on_error(result):\\n",\n    "    err = result.error_in_exec or result.error_before_exec\\n",\n    "    if err is None:\\n",\n    "        return\\n",\n    "    tb = \\"\\".join(traceback.format_exception(type(err), err, err.__traceback__))[-8000:]\\n",\n    "    z = pack_review(f\\"cell lỗi (execution_count={result.execution_count})\\", error=tb)\\n",\n    "    if z is not None:\\n",\n    "        print(f\\"⚠ Notebook dừng vì lỗi — đã gói những gì đã có (kèm traceback) → {z} ({z.stat().st_size / 2**20:.1f} MB)\\")\\n",\n    "\\n",\n    "\\n",\n    "try:\\n",\n    "    _IP = get_ipython()\\n",\n    "    for _cb in [c for c in _IP.events.callbacks[\\"post_run_cell\\"] if getattr(c, \\"__name__\\", \\"\\") == \\"_pack_on_error\\"]:\\n",\n    "        _IP.events.unregister(\\"post_run_cell\\", _cb)          # chạy lại cell này thì không đăng ký trùng\\n",\n    "    _IP.events.register(\\"post_run_cell\\", _pack_on_error)\\n",\n    "except NameError:\\n",\n    "    pass\\n",\n    "print(f\\"Gói xem xét: {REVIEW_ZIP} (tạo ở cell cuối; tự tạo nếu một cell bị lỗi)\\")"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c006",\n   "metadata": {},\n   "source": [\n    "### Đăng ký trước — v5 (viết TRƯỚC khi chạy)\\n",\n    "\\n",\n    "**Biến thể** (trọng số học trên VAL, áp nguyên lên test)\\n",\n    "\\n",\n    "| | Cạnh | Vai trò |\\n",\n    "|---|---|---|\\n",\n    "| M0 | clip | baseline CLIP |\\n",\n    "| G1 | clip + hub_clip + br_clip | **đồ thị trên pipeline chỉ-CLIP** (không encoder mới) |\\n",\n    "| M1 | clip + pe | **baseline phẳng, không đồ thị** (chỉ thêm encoder) |\\n",\n    "| M2 | M1 + hub | |\\n",\n    "| M3 | M1 + cầu nối | |\\n",\n    "| M4 | clip + pe + hub + cầu nối | đồ thị đầy đủ, giai đoạn 1 |\\n",\n    "| M5 | M4 + node VLM (top-10) | hệ thống cuối |\\n",\n    "| M5b | CLIP top-10 + VLM | kiểu probe v4, tham chiếu |\\n",\n    "\\n",\n    "**Đối chứng cấu trúc** (học lại trọng số trên VAL như biến thể thật)\\n",\n    "* `ctrl_hub_random`: mỗi video nối tới k caption TRAIN **ngẫu nhiên** (cùng bậc).\\n",\n    "* `ctrl_bridge_shuffled`: hoán vị cạnh \\"mô tả\\" caption → video train (giữ số caption mỗi video).\\n",\n    "* `ctrl_both`: cả hai.\\n",\n    "* `M5_ctrl_vlm_shuffled`: bằng chứng VLM hoán vị giữa các ứng viên.\\n",\n    "\\n",\n    "**So sánh** (test, CI95 bootstrap theo video, Holm trên T1–T8)\\n",\n    "\\n",\n    "| | So sánh | Câu hỏi |\\n",\n    "|---|---|---|\\n",\n    "| T1 | M4 vs M0 | tổng mức tăng giai đoạn 1 |\\n",\n    "| T2 | G1 vs M0 | cấu trúc đồ thị giúp ngay trên pipeline chỉ-CLIP? |\\n",\n    "| **T3** | **M4 vs M1** | **cấu trúc (hub + cầu nối) thêm gì ngoài fusion phẳng CLIP+PE?** (phép thử \\"do đồ thị\\" chính) |\\n",\n    "| **T4** | **M4 vs M4_ctrl_both** | **thay cạnh thật bằng cạnh ngẫu nhiên thì mất mức tăng?** (ablation đề yêu cầu); T4b tương tự cho G1 |\\n",\n    "| T5 | M5 vs M4 | node VLM thêm gì (tập con test) |\\n",\n    "| T6 | M5 vs M0 | hệ thống cuối vs CLIP (tập con test) |\\n",\n    "| T7 | M5 vs M5_ctrl_vlm_shuffled | cạnh VLM thật vs xáo |\\n",\n    "| T8 | M5 vs M5b | đồ thị + VLM vs VLM trên top-10 CLIP |\\n",\n    "\\n",\n    "**Cách đọc kết quả** (cố định trước)\\n",\n    "* **Phần tăng \\"do đồ thị\\"** chỉ được nhận nếu cả hai điều sau đều đúng:\\n",\n    "  * T3 > 0, CI95 > 0;\\n",\n    "  * T4 > 0, CI95 > 0.\\n",\n    "* **Nếu M4 ≈ M1:** mức tăng là do encoder PE, báo cáo đúng như vậy.\\n",\n    "* **Fidelity:** xoá 3 đường cầu nối đóng góp nhiều nhất của video đúng, so với xoá 3 đường ngẫu nhiên, trên ≤ 60 truy vấn.\\n",\n    "\\n",\n    "**Rủi ro**\\n",\n    "* Dữ liệu huấn luyện của PE / Qwen có thể chứa MSVD.\\n",\n    "* Test MSVD đã nhìn 7 lần.\\n",\n    "* Tập con VLM = 670 truy vấn, có thể ít hơn nếu hết ngân sách (ghi rõ).\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c007",\n   "metadata": {},\n   "source": [\n    "## 0b. Hàng rào chống rò rỉ\\n",\n    "\\n",\n    "* **`CaptionStore`**: caption không được lưu trong `RECORDS`. Mọi lần đọc caption, hoặc dùng đặc trưng của caption, đều phải nêu **mục đích**; một ma trận quyết định split nào được dùng cho mục đích nào.\\n",\n    "  * `vocab`, `relations`, `annotation` và `gnn_train` chỉ dùng train; `tune` và `tagger_select` chỉ dùng val.\\n",\n    "  * Caption test MSVD và tập ngoài (`ext`, MSR-VTT) chỉ dùng để `encode`, `query_input` và (sau đóng băng) `eval`, `report`, `display`.\\n",\n    "  * Truy cập sai sẽ ném `PermissionError` ngay.\\n",\n    "* **`TestGate`**: test MSVD và tập ngoài chỉ mở sau `freeze()`, và mọi lần chạm đều được ghi log kèm tên split. Ở `dev_mode`, test MSVD bị khoá; tập ngoài khi đó là video **train-9k** của MSR-VTT, rời hẳn 1k-A.\\n",\n    "* **`provenance()`**: mỗi artifact được ghi lại nguồn, split caption và split video đã dùng; khai báo sai loại thì `assert` thất bại."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c008",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "class TestGate:\\n",\n    "    \\"\\"\\"Test chỉ mở SAU khi cấu hình đã đóng băng trên validation. Mọi lần chạm test được ghi log.\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self):\\n",\n    "        self.frozen, self.frozen_sha, self.frozen_at, self.log = None, None, None, []\\n",\n    "\\n",\n    "    def freeze(self, config):\\n",\n    "        assert self.frozen is None, \\"Cấu hình đã đóng băng — không được chọn lại sau khi mở test.\\"\\n",\n    "        payload = json.dumps(config, sort_keys=True, default=_jsonable)\\n",\n    "        self.frozen, self.frozen_at = json.loads(payload), time.time()\\n",\n    "        self.frozen_sha = hashlib.sha1(payload.encode()).hexdigest()[:12]\\n",\n    "        save_json(dict(config=self.frozen, sha1=self.frozen_sha, frozen_at=self.frozen_at,\\n",\n    "                       frozen_at_str=time.strftime(\\"%Y-%m-%d %H:%M:%S\\")), RESULTS / \\"frozen_config.json\\")\\n",\n    "        print(f\\"🔒 Đã đóng băng cấu hình (sha1={self.frozen_sha}) → {RESULTS / \'frozen_config.json\'}\\")\\n",\n    "\\n",\n    "    def check(self, what, split=\\"test\\"):\\n",\n    "        \\"\\"\\"split \'test\' = test MSVD (đã nhìn ở v1/v2 ⇒ ở v3 chỉ là THĂM DÒ); \'ext\' = benchmark xác nhận độc lập (MSR-VTT).\\n",\n    "        Cả hai chỉ mở sau khi đóng băng. dev_mode: test MSVD bị khoá; \'ext\' khi đó là video TRAIN-9k (không phải tập xác nhận).\\"\\"\\"\\n",\n    "        if self.frozen is None:\\n",\n    "            raise RuntimeError(f\\"Truy cập {split.upper()} (\'{what}\') trước khi đóng băng cấu hình — vi phạm giao thức.\\")\\n",\n    "        if CFG.dev_mode and split == \\"test\\":\\n",\n    "            raise RuntimeError(f\\"dev_mode: test bị khoá (\'{what}\')\\")\\n",\n    "        self.log.append(dict(what=what, split=split, t=time.time(), frozen_sha=self.frozen_sha, dev=bool(CFG.dev_mode)))\\n",\n    "\\n",\n    "\\n",\n    "TEST_GATE = TestGate()\\n",\n    "\\n",\n    "\\n",\n    "class CaptionStore:\\n",\n    "    \\"\\"\\"Nơi DUY NHẤT giữ caption. Mỗi lần đọc phải nêu mục đích; split không được phép ⇒ PermissionError.\\"\\"\\"\\n",\n    "    ALLOWED = {\\n",\n    "        \\"vocab\\":         {\\"train\\"},                  # từ vựng khái niệm\\n",\n    "        \\"relations\\":     {\\"train\\"},                  # cạnh quan hệ thống kê từ caption\\n",\n    "        \\"gnn_train\\":     {\\"train\\"},                  # truy vấn huấn luyện GNN\\n",\n    "        \\"tagger_select\\": {\\"val\\"},                    # chọn (pool, norm, k) của tagger theo F1\\n",\n    "        \\"tune\\":          {\\"val\\"},                    # chọn n_frames/pooling/learn_clip_w/k/α/λ\\n",\n    "        \\"encode\\":        {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # CLIP-text đóng băng (không học) — chỉ để cache đặc trưng\\n",\n    "        \\"query_input\\":   {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # đọc văn bản của CHÍNH truy vấn đang được trả lời\\n",\n    "        \\"eval\\":          {\\"val\\", \\"test\\", \\"ext\\"},\\n",\n    "        \\"display\\":       {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # chỉ hiển thị SAU khi đã xếp hạng\\n",\n    "        \\"report\\":        {\\"val\\", \\"test\\", \\"ext\\"},            # báo cáo P/R/F1 của tagger sau khi đóng băng\\n",\n    "    }\\n",\n    "    GATED = {\\"eval\\", \\"report\\", \\"display\\"}            # với split test / ext: chỉ sau khi đóng băng\\n",\n    "\\n",\n    "    def __init__(self, caps_by_vid, split_by_vid):\\n",\n    "        self._caps = {int(v): list(c) for v, c in caps_by_vid.items()}\\n",\n    "        self.split_of = {int(v): s for v, s in split_by_vid.items()}\\n",\n    "        vid_of_cap, self._text = [], []\\n",\n    "        for v in sorted(self._caps):\\n",\n    "            for c in self._caps[v]:\\n",\n    "                vid_of_cap.append(v); self._text.append(c)\\n",\n    "        self.vid_of_cap = np.array(vid_of_cap, np.int64)\\n",\n    "        self.split_of_cap = np.array([self.split_of[v] for v in vid_of_cap])\\n",\n    "        self.log = Counter()\\n",\n    "\\n",\n    "    def _authorize(self, splits, purpose):\\n",\n    "        assert purpose in self.ALLOWED, f\\"mục đích lạ: {purpose}\\"\\n",\n    "        for s in splits:\\n",\n    "            if s not in self.ALLOWED[purpose]:\\n",\n    "                raise PermissionError(f\\"RÒ RỈ bị chặn: caption split \'{s}\' không được dùng cho mục đích \'{purpose}\'\\")\\n",\n    "            if s in (\\"test\\", \\"ext\\") and purpose in self.GATED:\\n",\n    "                TEST_GATE.check(f\\"captions:{purpose}\\", s)\\n",\n    "        for s, n in splits.items():\\n",\n    "            self.log[(purpose, s)] += int(n)\\n",\n    "\\n",\n    "    def by_video(self, vids, purpose):\\n",\n    "        vids = [int(v) for v in np.atleast_1d(vids)]\\n",\n    "        self._authorize(Counter(self.split_of[v] for v in vids), purpose)\\n",\n    "        return {v: list(self._caps[v]) for v in vids}\\n",\n    "\\n",\n    "    def texts(self, cap_idx, purpose):\\n",\n    "        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))\\n",\n    "        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)\\n",\n    "        return [self._text[i] for i in cap_idx]\\n",\n    "\\n",\n    "    def use(self, cap_idx, purpose):\\n",\n    "        \\"\\"\\"Khai báo dùng ĐẶC TRƯNG của caption (không lấy văn bản) — cùng luật cho phép.\\"\\"\\"\\n",\n    "        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))\\n",\n    "        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)\\n",\n    "\\n",\n    "    def replaced(self, new_caps, new_split=None):\\n",\n    "        \\"\\"\\"Bản sao với caption (và tuỳ chọn split) của một số video bị thay — CHỈ dùng trong self-test.\\"\\"\\"\\n",\n    "        caps = {v: list(c) for v, c in self._caps.items()}\\n",\n    "        caps.update({int(v): list(c) for v, c in new_caps.items()})\\n",\n    "        return CaptionStore(caps, {**self.split_of, **(new_split or {})})\\n",\n    "\\n",\n    "    def access_table(self):\\n",\n    "        return pd.DataFrame([dict(purpose=p, split=s, n=n) for (p, s), n in sorted(self.log.items())])\\n",\n    "\\n",\n    "\\n",\n    "PROVENANCE = []\\n",\n    "_PROV_RULES = {            # loại artifact → split CAPTION được phép dùng\\n",\n    "    \\"learned\\":  {\\"train\\"},             # thống kê / trọng số học được\\n",\n    "    \\"selected\\": {\\"val\\"},               # siêu tham số chọn trên validation\\n",\n    "    \\"frames\\":   set(),                 # chỉ từ frame (không caption)\\n",\n    "    \\"external\\": set(),                 # tri thức ngoài / mô hình tiền huấn luyện\\n",\n    "    \\"encoded\\":  {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # mã hoá bằng encoder đóng băng, không học gì\\n",\n    "    \\"evaluated\\": {\\"val\\", \\"test\\", \\"ext\\"},\\n",\n    "}\\n",\n    "\\n",\n    "\\n",\n    "def provenance(artifact, kind, source, caption_splits=(), video_splits=(), note=\\"\\"):\\n",\n    "    cs = set(caption_splits)\\n",\n    "    bad = cs - _PROV_RULES[kind]\\n",\n    "    assert not bad, f\\"RÒ RỈ: \'{artifact}\' (loại {kind}) dùng caption của {sorted(bad)}\\"\\n",\n    "    PROVENANCE.append(dict(artifact=artifact, kind=kind, source=source,\\n",\n    "                           caption_splits=\\",\\".join(sorted(cs)) or \\"—\\",\\n",\n    "                           video_splits=\\",\\".join(sorted(set(video_splits))) or \\"—\\",\\n",\n    "                           note=note, stage=STAGES[-1][\\"name\\"], time=time.strftime(\\"%H:%M:%S\\")))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c009",\n   "metadata": {},\n   "source": [\n    "## 1. Dữ liệu: MSVD (phát triển) + MSR-VTT 1k-A (xác nhận) — Cổng A\\n",\n    "\\n",\n    "**MSVD.** Như v1/v2. Video thật nằm trong `MSVD_Videos.zip`. Notebook dò `MSVD_ROOT`, rồi `/kaggle/input`, rồi cache, rồi tải từ HF ở commit ghim. Không có video thì **dừng**.\\n",\n    "\\n",\n    "**MSR-VTT** (`friedrichor/MSR-VTT`, commit ghim). Dùng `msrvtt_test_1k.json` (1000 video, caption chuẩn JSFusion), `raw_data/MSRVTT_data.json` (20 caption/video) và `MSRVTT_Videos.zip` (2,2 GB, mp4).\\n",\n    "* Video tập ngoài được nối **sau** MSVD trong `RECORDS` với split `ext`. Vì vậy cache MSVD giữ nguyên tên và thứ tự.\\n",\n    "* Truy vấn chính là mọi caption của 1000 video; truy vấn phụ là caption chuẩn 1k-A. Nếu caption chuẩn không nằm trong 20 caption thì nó được thêm vào và đếm riêng.\\n",\n    "* Không nạp được MSR-VTT thì tập xác nhận ghi **NOT RUN** kèm lý do; notebook không thay bằng tập khác."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c010",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "HF_JSON = {\\"train\\": \\"msvd_train.json\\", \\"val\\": \\"msvd_val.json\\", \\"test\\": \\"msvd_test.json\\"}\\n",\n    "HF_ZIP = \\"MSVD_Videos.zip\\"\\n",\n    "\\n",\n    "\\n",\n    "class DirVideoSource:\\n",\n    "    kind = \\"dir\\"\\n",\n    "\\n",\n    "    def __init__(self, root):\\n",\n    "        self.root = Path(root)\\n",\n    "        self.files = {p.stem: p for p in self.root.rglob(\\"*\\") if p.suffix.lower() == \\".avi\\"}\\n",\n    "        self.label = str(self.root)\\n",\n    "\\n",\n    "    def has(self, vid): return vid in self.files\\n",\n    "    def size(self, vid): return self.files[vid].stat().st_size\\n",\n    "\\n",\n    "    @contextmanager\\n",\n    "    def open(self, vid):\\n",\n    "        yield self.files[vid]\\n",\n    "\\n",\n    "\\n",\n    "class ZipVideoSource:\\n",\n    "    \\"\\"\\"Đọc video thẳng từ zip cục bộ (không giải nén toàn bộ): mỗi video được ghi ra file tạm, giải mã, rồi xoá.\\"\\"\\"\\n",\n    "    kind = \\"zip\\"\\n",\n    "\\n",\n    "    def __init__(self, zip_file, label, exts=(\\".avi\\",)):\\n",\n    "        self.z = zipfile.ZipFile(zip_file)\\n",\n    "        self.members = {Path(i.filename).stem: i for i in self.z.infolist()\\n",\n    "                        if i.filename.lower().endswith(exts) and not Path(i.filename).name.startswith(\\"._\\")}\\n",\n    "        self.lock = threading.Lock()\\n",\n    "        self.tmp = Path(tempfile.mkdtemp(prefix=\\"msvd_zip_\\"))\\n",\n    "        self.label = label\\n",\n    "\\n",\n    "    def has(self, vid): return vid in self.members\\n",\n    "    def size(self, vid): return self.members[vid].file_size\\n",\n    "\\n",\n    "    @contextmanager\\n",\n    "    def open(self, vid):\\n",\n    "        with self.lock:                                   # ZipFile không an toàn khi đọc song song\\n",\n    "            data = self.z.read(self.members[vid])\\n",\n    "        p = self.tmp / f\\"{vid}{Path(self.members[vid].filename).suffix.lower()}\\"\\n",\n    "        p.write_bytes(data)\\n",\n    "        try:\\n",\n    "            yield p\\n",\n    "        finally:\\n",\n    "            p.unlink(missing_ok=True)\\n",\n    "\\n",\n    "\\n",\n    "def _first(root, name):\\n",\n    "    hits = sorted(root.rglob(name), key=lambda p: len(str(p)))\\n",\n    "    return hits[0] if hits else None\\n",\n    "\\n",\n    "\\n",\n    "def locate_msvd():\\n",\n    "    roots = [Path(CFG.msvd_root)] if CFG.msvd_root else []\\n",\n    "    if Path(\\"/kaggle/input\\").exists():\\n",\n    "        roots.append(Path(\\"/kaggle/input\\"))\\n",\n    "    roots.append(CFG.data_cache)\\n",\n    "    found = dict(json={}, video_dir=None, zip=None, roots=[str(r) for r in roots])\\n",\n    "    for root in roots:\\n",\n    "        if not root.exists():\\n",\n    "            continue\\n",\n    "        for s, name in HF_JSON.items():\\n",\n    "            if s not in found[\\"json\\"]:\\n",\n    "                p = _first(root, name)\\n",\n    "                if p is not None:\\n",\n    "                    found[\\"json\\"][s] = p\\n",\n    "        if found[\\"video_dir\\"] is None:\\n",\n    "            for d in root.rglob(\\"YouTubeClips\\"):\\n",\n    "                if d.is_dir() and any(d.glob(\\"*.avi\\")):\\n",\n    "                    found[\\"video_dir\\"] = d\\n",\n    "                    break\\n",\n    "        if found[\\"zip\\"] is None:\\n",\n    "            found[\\"zip\\"] = _first(root, HF_ZIP)\\n",\n    "    return found\\n",\n    "\\n",\n    "\\n",\n    "def hf_download(filename):\\n",\n    "    from huggingface_hub import hf_hub_download\\n",\n    "    CFG.data_cache.mkdir(parents=True, exist_ok=True)\\n",\n    "    print(f\\"  tải {CFG.hf_repo}@{CFG.hf_revision[:8]}/{filename} → {CFG.data_cache}\\")\\n",\n    "    return Path(hf_hub_download(CFG.hf_repo, filename, repo_type=\\"dataset\\", revision=CFG.hf_revision,\\n",\n    "                                local_dir=str(CFG.data_cache)))\\n",\n    "\\n",\n    "\\n",\n    "def open_video_source(found):\\n",\n    "    mode = CFG.video_source\\n",\n    "    assert mode in (\\"auto\\", \\"dir\\", \\"zip\\"), mode\\n",\n    "    if mode in (\\"auto\\", \\"dir\\") and found[\\"video_dir\\"] is not None:\\n",\n    "        return DirVideoSource(found[\\"video_dir\\"])\\n",\n    "    if mode in (\\"auto\\", \\"zip\\") and found[\\"zip\\"] is not None:\\n",\n    "        return ZipVideoSource(found[\\"zip\\"], str(found[\\"zip\\"]))\\n",\n    "    if mode in (\\"auto\\", \\"zip\\") and INTERNET:\\n",\n    "        return ZipVideoSource(hf_download(HF_ZIP), \\"hf_hub_download\\")\\n",\n    "    raise FileNotFoundError(\\n",\n    "        \\"KHÔNG tìm thấy file video MSVD (YouTubeClips/*.avi hoặc MSVD_Videos.zip) và không tải được. DỪNG: chỉ có \\"\\n",\n    "        \\"caption thì KHÔNG được chạy/gọi là thí nghiệm text-to-video. Khắc phục: bật Internet, hoặc Add Input \\"\\n",\n    "        f\\"dataset chứa video, hoặc đặt MSVD_ROOT. Đã dò: {found[\'roots\']}\\")\\n",\n    "\\n",\n    "\\n",\n    "def load_msvd():\\n",\n    "    found = locate_msvd()\\n",\n    "    for s, name in HF_JSON.items():\\n",\n    "        if s not in found[\\"json\\"]:\\n",\n    "            if not INTERNET:\\n",\n    "                raise FileNotFoundError(f\\"Thiếu {name} trong {found[\'roots\']} và không có Internet để tải.\\")\\n",\n    "            found[\\"json\\"][s] = hf_download(name)\\n",\n    "    source = open_video_source(found)\\n",\n    "    records, caps = [], {}\\n",\n    "    report = dict(source=source.kind, source_label=source.label, n_video_files=len(\\n",\n    "        source.files if source.kind == \\"dir\\" else source.members),\\n",\n    "        json={k: str(v) for k, v in found[\\"json\\"].items()}, splits={})\\n",\n    "    for s in (\\"train\\", \\"val\\", \\"test\\"):\\n",\n    "        rows = json.load(open(found[\\"json\\"][s]))\\n",\n    "        fields = sorted({k for r in rows for k in r})\\n",\n    "        bad = [r.get(\\"video_id\\") for r in rows if not r.get(\\"video_id\\") or not isinstance(r.get(\\"caption\\"), list)]\\n",\n    "        assert not bad, f\\"Schema {s} sai (thiếu video_id hoặc caption không phải list): {bad[:5]}\\"\\n",\n    "        n_cap, missing, ncaps = 0, [], []\\n",\n    "        for r in rows:\\n",\n    "            vid = len(records)\\n",\n    "            c = [str(x).strip() for x in r[\\"caption\\"] if str(x).strip()]\\n",\n    "            records.append(dict(vid=vid, video_id=r[\\"video_id\\"], file=r.get(\\"video\\") or f\\"{r[\'video_id\']}.avi\\",\\n",\n    "                                split=s, n_captions=len(c)))\\n",\n    "            caps[vid] = c; n_cap += len(c); ncaps.append(len(c))\\n",\n    "            if not source.has(r[\\"video_id\\"]):\\n",\n    "                missing.append(r[\\"video_id\\"])\\n",\n    "        report[\\"splits\\"][s] = dict(videos=len(rows), captions=n_cap, fields=fields, missing_files=len(missing),\\n",\n    "                                   missing_examples=missing[:5],\\n",\n    "                                   captions_per_video=[int(min(ncaps)), round(float(np.mean(ncaps)), 1), int(max(ncaps))],\\n",\n    "                                   video_field_is_filename=all(str(r.get(\\"video\\", \\"\\")).endswith(\\".avi\\") for r in rows))\\n",\n    "    ids = [r[\\"video_id\\"] for r in records]\\n",\n    "    assert len(set(ids)) == len(ids), \\"video_id trùng lặp giữa các bản ghi\\"\\n",\n    "    return records, caps, source, report\\n",\n    "\\n",\n    "\\n",\n    "def probe_video(path):\\n",\n    "    \\"\\"\\"Codec / fps / kích thước / số frame giải mã thật sự (đếm bằng grab, không tin FRAME_COUNT).\\"\\"\\"\\n",\n    "    cap = cv2.VideoCapture(str(path))\\n",\n    "    if not cap.isOpened():\\n",\n    "        return dict(ok=False, error=\\"cv2 không mở được\\")\\n",\n    "    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))\\n",\n    "    info = dict(codec=\\"\\".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)), fps=round(cap.get(cv2.CAP_PROP_FPS), 3),\\n",\n    "                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),\\n",\n    "                frames_meta=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), backend=cap.getBackendName())\\n",\n    "    n = 0\\n",\n    "    while cap.grab():\\n",\n    "        n += 1\\n",\n    "    cap.release()\\n",\n    "    info.update(ok=n > 0, frames_decoded=n, duration_s=round(n / info[\\"fps\\"], 2) if info[\\"fps\\"] > 0 else None)\\n",\n    "    return info"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c011",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# --- Level 4(b): benchmark xác nhận ĐỘC LẬP = MSR-VTT fresh1k-B (CHƯA dùng ở v1–v4); fresh1k (v4) và 1k-A (v3) đã nhìn ⇒ chỉ THĂM DÒ.\\n",\n    "# (Ghi chú lịch sử của v4:) Benchmark xác nhận cho v4: MSR-VTT fresh1k (CHƯA dùng ở v1–v3). Video nối vào đồ thị theo giao thức inductive. ---\\n",\n    "# 1k-A (JSFusion) đã được nhìn MỘT lần ở v3 ⇒ v4 dùng 1000 video KHÁC: lấy ngẫu nhiên (seed cố định) trong train-9k, loại 1k-A và\\n",\n    "# loại 200 video dev (đã chạy ở máy local). Truy vấn CHÍNH = mọi caption (MSRVTT_data.json); truy vấn PHỤ = caption đầu tiên.\\n",\n    "# CFG.confirm_set = \\"1kA\\" chạy lại 1k-A (lần nhìn thứ 2 — phải khai báo). Chỉ đánh giá MỘT lần, sau khi đóng băng.\\n",\n    "# dev_mode: msrvtt_dev_n video TRAIN-9k đầu theo id — chỉ để thử đường ống, không phải kết quả xác nhận.\\n",\n    "MSRVTT_FILES = dict(test=\\"msrvtt_test_1k.json\\", train=\\"msrvtt_train_9k.json\\", all_caps=\\"raw_data/MSRVTT_data.json\\",\\n",\n    "                    zip=\\"MSRVTT_Videos.zip\\")\\n",\n    "\\n",\n    "\\n",\n    "def msrvtt_file(key):\\n",\n    "    name = MSRVTT_FILES[key]\\n",\n    "    roots = ([Path(CFG.msrvtt_root)] if CFG.msrvtt_root else []) + ([Path(\\"/kaggle/input\\")] if RUN_ON_KAGGLE else []) \\\\\\n",\n    "        + [CFG.data_cache / \\"msrvtt\\"]\\n",\n    "    for root in roots:\\n",\n    "        if root.exists():\\n",\n    "            p = _first(root, Path(name).name)\\n",\n    "            if p is not None:\\n",\n    "                return p\\n",\n    "    if not INTERNET:\\n",\n    "        raise FileNotFoundError(f\\"Thiếu {name} (đã dò {roots}) và không có Internet.\\")\\n",\n    "    from huggingface_hub import hf_hub_download\\n",\n    "    d = CFG.data_cache / \\"msrvtt\\"\\n",\n    "    d.mkdir(parents=True, exist_ok=True)\\n",\n    "    print(f\\"  tải {CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}/{name} → {d}\\")\\n",\n    "    return Path(hf_hub_download(CFG.msrvtt_repo, name, repo_type=\\"dataset\\", revision=CFG.msrvtt_revision, local_dir=str(d)))\\n",\n    "\\n",\n    "\\n",\n    "def load_msrvtt(vid_offset):\\n",\n    "    test = json.load(open(msrvtt_file(\\"test\\")))\\n",\n    "    test_ids = sorted({r[\\"video_id\\"] for r in test}, key=lambda x: int(x[5:]))\\n",\n    "    tr_all = sorted([r for r in json.load(open(msrvtt_file(\\"train\\"))) if r[\\"video_id\\"] not in set(test_ids)],\\n",\n    "                    key=lambda r: int(r[\\"video_id\\"][5:]))\\n",\n    "    dev_ids = {r[\\"video_id\\"] for r in tr_all[:CFG.msrvtt_dev_n]}\\n",\n    "    fresh1k_ids, fresh1kB_ids = set(), set()\\n",\n    "    set_name = \\"dev\\" if CFG.dev_mode else CFG.confirm_set\\n",\n    "    if CFG.dev_mode:\\n",\n    "        rows = tr_all[:CFG.msrvtt_dev_n]\\n",\n    "        caps_of = {r[\\"video_id\\"]: [str(c).strip() for c in r[\\"caption\\"] if str(c).strip()] for r in rows}\\n",\n    "        std = {v: c[0] for v, c in caps_of.items()}\\n",\n    "        label = f\\"MSR-VTT train-9k, {len(rows)} video đầu theo id (DEV — không phải tập xác nhận)\\"\\n",\n    "    else:\\n",\n    "        assert CFG.confirm_set in (\\"fresh1kC\\", \\"fresh1kB\\", \\"fresh1k\\", \\"1kA\\"), CFG.confirm_set\\n",\n    "        pool = [r for r in tr_all if r[\\"video_id\\"] not in dev_ids]\\n",\n    "        pick = np.random.RandomState(CFG.fresh_seed).choice(len(pool), CFG.fresh_n, replace=False)\\n",\n    "        fresh1k_ids = {pool[i][\\"video_id\\"] for i in pick}          # tập xác nhận của v4 — ĐÃ nhìn một lần\\n",\n    "        pool_b = [r for r in pool if r[\\"video_id\\"] not in fresh1k_ids]\\n",\n    "        pick_b = np.random.RandomState(CFG.fresh_b_seed).choice(len(pool_b), CFG.fresh_n, replace=False)\\n",\n    "        fresh1kB_ids = {pool_b[i][\\"video_id\\"] for i in pick_b}      # tập xác nhận của Level 4(b) v1 — ĐÃ nhìn một lần\\n",\n    "        if CFG.confirm_set == \\"fresh1kC\\":\\n",\n    "            pool_c = [r for r in pool_b if r[\\"video_id\\"] not in fresh1kB_ids]\\n",\n    "            pick_c = np.random.RandomState(CFG.fresh_c_seed).choice(len(pool_c), CFG.fresh_n, replace=False)\\n",\n    "            rows = sorted([pool_c[i] for i in pick_c], key=lambda r: int(r[\\"video_id\\"][5:]))\\n",\n    "            label = (f\\"MSR-VTT fresh1k-C: {len(rows)} video train-9k (ngoài 1k-A, dev, fresh1k, fresh1k-B; seed {CFG.fresh_c_seed}) \\"\\n",\n    "                     \\"— CHƯA dùng ở v1–v4 và Level 4(b) v1\\")\\n",\n    "        elif CFG.confirm_set == \\"fresh1kB\\":\\n",\n    "            rows = sorted([pool_b[i] for i in pick_b], key=lambda r: int(r[\\"video_id\\"][5:]))\\n",\n    "            label = f\\"MSR-VTT fresh1k-B (seed {CFG.fresh_b_seed}) — ĐÃ nhìn ở Level 4(b) v1 ⇒ chỉ THĂM DÒ\\"\\n",\n    "        elif CFG.confirm_set == \\"fresh1k\\":\\n",\n    "            rows = sorted([pool[i] for i in pick], key=lambda r: int(r[\\"video_id\\"][5:]))\\n",\n    "            label = (f\\"MSR-VTT fresh1k: {len(rows)} video train-9k (seed {CFG.fresh_seed}) — ĐÃ nhìn ở v4 ⇒ chỉ THĂM DÒ\\")\\n",\n    "        else:\\n",\n    "            rows = sorted(test, key=lambda r: int(r[\\"video_id\\"][5:]))\\n",\n    "            label = \\"MSR-VTT test 1k-A (JSFusion) — LẦN NHÌN THỨ 2 (đã dùng ở v3)\\"\\n",\n    "        data = json.load(open(msrvtt_file(\\"all_caps\\")))\\n",\n    "        allc = defaultdict(list)\\n",\n    "        for x in sorted(data[\\"sentences\\"], key=lambda x: int(x[\\"sen_id\\"])):\\n",\n    "            allc[x[\\"video_id\\"]].append(str(x[\\"caption\\"]).strip())\\n",\n    "        caps_of = {r[\\"video_id\\"]: [c for c in allc[r[\\"video_id\\"]] if c] for r in rows}\\n",\n    "        std = ({r[\\"video_id\\"]: str(r[\\"caption\\"]).strip() for r in rows} if CFG.confirm_set == \\"1kA\\"\\n",\n    "               else {v: c[0] for v, c in caps_of.items()})\\n",\n    "    n_std_added = 0\\n",\n    "    for v, c in caps_of.items():                       # caption chuẩn phải nằm trong tập truy vấn chính\\n",\n    "        if std[v] not in c:\\n",\n    "            c.append(std[v]); n_std_added += 1\\n",\n    "    source = ZipVideoSource(msrvtt_file(\\"zip\\"), f\\"{CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}\\", exts=(\\".mp4\\",))\\n",\n    "    records, caps, missing = [], {}, []\\n",\n    "    for j, r in enumerate(rows):\\n",\n    "        vid, c = vid_offset + j, caps_of[r[\\"video_id\\"]]\\n",\n    "        records.append(dict(vid=vid, video_id=r[\\"video_id\\"], file=r.get(\\"video\\") or f\\"{r[\'video_id\']}.mp4\\", split=\\"ext\\",\\n",\n    "                            n_captions=len(c), std_caption=c.index(std[r[\\"video_id\\"]]), dataset=\\"msrvtt\\"))\\n",\n    "        caps[vid] = c\\n",\n    "        if not source.has(r[\\"video_id\\"]):\\n",\n    "            missing.append(r[\\"video_id\\"])\\n",\n    "    ids = {r[\\"video_id\\"] for r in records}\\n",\n    "    report = dict(label=label, set=set_name, dev=bool(CFG.dev_mode), videos=len(records), captions=sum(len(c) for c in caps.values()),\\n",\n    "                  captions_per_video=[min(map(len, caps.values())), round(float(np.mean(list(map(len, caps.values())))), 1),\\n",\n    "                                      max(map(len, caps.values()))],\\n",\n    "                  std_captions_added=n_std_added, missing_files=len(missing), missing_examples=missing[:5],\\n",\n    "                  n_test_1kA_ids=len(test_ids), overlap_with_test_1kA=len(ids & set(test_ids)),\\n",\n    "                  overlap_with_dev=len(ids & dev_ids) if not CFG.dev_mode else None,\\n",\n    "                  overlap_with_fresh1k=len(ids & fresh1k_ids) if not CFG.dev_mode else None,\\n",\n    "                  overlap_with_fresh1kB=len(ids & fresh1kB_ids) if not CFG.dev_mode else None, zip_members=len(source.members))\\n",\n    "    if set_name in (\\"fresh1k\\", \\"fresh1kB\\", \\"fresh1kC\\"):\\n",\n    "        assert report[\\"overlap_with_test_1kA\\"] == 0 and report[\\"overlap_with_dev\\"] == 0, \\"fresh1k / fresh1k-B phải rời 1k-A và tập dev\\"\\n",\n    "    if set_name in (\\"fresh1kB\\", \\"fresh1kC\\"):\\n",\n    "        assert report[\\"overlap_with_fresh1k\\"] == 0 and report[\\"videos\\"] == CFG.fresh_n, \\"fresh1k-B/C phải rời fresh1k của v4\\"\\n",\n    "    if set_name == \\"fresh1kC\\":\\n",\n    "        assert report[\\"overlap_with_fresh1kB\\"] == 0, \\"fresh1k-C phải rời fresh1k-B\\"\\n",\n    "    return records, caps, source, report\\n",\n    "\\n",\n    "\\n",\n    "def make_synthetic_ext(vid_offset, n_videos=30, seed=7):\\n",\n    "    \\"\\"\\"Tập \'ext\' tổng hợp cho smoke-test: cùng thế giới đối tượng, câu chữ khác (kiểm tra đường ống, không phải kết quả).\\"\\"\\"\\n",\n    "    recs, caps, source, _ = make_synthetic_msvd(n_videos=n_videos, seed=seed, prefix=\\"ext\\", all_split=\\"ext\\")\\n",\n    "    out_r, out_c = [], {}\\n",\n    "    for j, r in enumerate(recs):\\n",\n    "        out_r.append(dict(r, vid=vid_offset + j, split=\\"ext\\", std_caption=0, dataset=\\"synthetic_ext\\"))\\n",\n    "        out_c[vid_offset + j] = caps[r[\\"vid\\"]]\\n",\n    "    report = dict(label=\\"synthetic ext\\", dev=True, videos=len(out_r), captions=sum(len(c) for c in out_c.values()),\\n",\n    "                  missing_files=0, overlap_with_test_1kA=0, std_captions_added=0)\\n",\n    "    return out_r, out_c, source, report"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c012",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# Video TỔNG HỢP — chỉ dùng khi CFG.dataset == \\"synthetic\\" (smoke-test mã, không phải kết quả).\\n",\n    "# Mỗi video có 3 cảnh (chủ thể → hành động → đồ vật), chữ in trên frame để CLIP đọc được; được ghi\\n",\n    "# thành AVI (MJPG) thật để kiểm tra luôn đường ống đọc video bằng OpenCV.\\n",\n    "SYN_SUBJ = {\\"man\\": \\"person\\", \\"woman\\": \\"person\\", \\"child\\": \\"person\\", \\"dog\\": \\"animal\\", \\"cat\\": \\"animal\\"}\\n",\n    "SYN_ACT = {\\"play\\": ([\\"guitar\\", \\"piano\\", \\"ball\\"], \\"playing\\"), \\"ride\\": ([\\"horse\\", \\"bicycle\\"], \\"riding\\"),\\n",\n    "           \\"eat\\": ([\\"apple\\", \\"pizza\\"], \\"eating\\"), \\"cut\\": ([\\"onion\\", \\"bread\\"], \\"cutting\\")}\\n",\n    "SYN_COLORS = [(40, 40, 160), (40, 140, 40), (160, 60, 40), (120, 40, 120), (30, 120, 150), (150, 150, 30)]\\n",\n    "\\n",\n    "\\n",\n    "def make_synthetic_msvd(n_videos=150, seed=0, prefix=\\"syn\\", all_split=None):\\n",\n    "    rng = random.Random(seed)\\n",\n    "    vdir = CFG.out_dir / \\"synthetic_videos\\"\\n",\n    "    vdir.mkdir(parents=True, exist_ok=True)\\n",\n    "    records, caps = [], {}\\n",\n    "    n_tr, n_va = int(n_videos * 0.6), int(n_videos * 0.15)\\n",\n    "    fourcc = cv2.VideoWriter_fourcc(*\\"MJPG\\")\\n",\n    "    for i in range(n_videos):\\n",\n    "        s = rng.choice(sorted(SYN_SUBJ)); a = rng.choice(sorted(SYN_ACT))\\n",\n    "        objs, ing = SYN_ACT[a]; o = rng.choice(objs)\\n",\n    "        split_name = all_split or (\\"train\\" if i < n_tr else (\\"val\\" if i < n_tr + n_va else \\"test\\"))\\n",\n    "        name = f\\"{prefix}{i:04d}_{s}_{a}_{o}\\"\\n",\n    "        path = vdir / f\\"{name}.avi\\"\\n",\n    "        if not path.exists():\\n",\n    "            w = cv2.VideoWriter(str(path), fourcc, 12.0, (224, 224))\\n",\n    "            assert w.isOpened(), \\"cv2.VideoWriter không mở được (MJPG/AVI)\\"\\n",\n    "            for k in range(36):\\n",\n    "                word = (s, ing, o)[k // 12]\\n",\n    "                fr = np.full((224, 224, 3), SYN_COLORS[(i + k // 12) % len(SYN_COLORS)], np.uint8)\\n",\n    "                cv2.putText(fr, word, (8 + (k % 12) * 2, 122), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3)\\n",\n    "                w.write(fr)\\n",\n    "            w.release()\\n",\n    "        c = [f\\"a {s} is {ing} a {o}\\", f\\"a {s} {a}s the {o}\\", f\\"someone is {ing} a {o}\\", f\\"a {s} with a {o}\\",\\n",\n    "             f\\"the {s} is {ing}\\", f\\"a {s} is {ing} a {o} slowly\\", f\\"a {SYN_SUBJ[s]} is {ing}\\", f\\"a {o} and a {s}\\"]\\n",\n    "        if all_split == \\"ext\\":                         # tập ngoài: câu chữ khác với tập huấn luyện\\n",\n    "            c = [f\\"there is a {s} {ing} the {o}\\", f\\"{s} {ing} {o}\\", f\\"video of a {o} being used by a {s}\\",\\n",\n    "                 f\\"the {SYN_SUBJ[s]} keeps {ing} a {o}\\"]\\n",\n    "        rng.shuffle(c)\\n",\n    "        records.append(dict(vid=i, video_id=name, file=path.name, split=split_name, n_captions=len(c)))\\n",\n    "        caps[i] = c\\n",\n    "    source = DirVideoSource(vdir)\\n",\n    "    report = dict(source=\\"synthetic-dir\\", source_label=str(vdir), n_video_files=len(source.files), json={}, splits={\\n",\n    "        s: dict(videos=sum(r[\\"split\\"] == s for r in records),\\n",\n    "                captions=sum(len(caps[r[\\"vid\\"]]) for r in records if r[\\"split\\"] == s),\\n",\n    "                fields=[\\"caption\\", \\"video\\", \\"video_id\\"], missing_files=sum(\\n",\n    "                    not source.has(r[\\"video_id\\"]) for r in records if r[\\"split\\"] == s))\\n",\n    "        for s in (\\"train\\", \\"val\\", \\"test\\")})\\n",\n    "    return records, caps, source, report"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c013",\n   "metadata": {},\n   "source": [\n    "## 2. Split chính thức và manifest\\n",\n    "\\n",\n    "Split MSVD giữ đúng `msvd_{train,val,test}.json` (1200 / 100 / 670 video), chia **theo video**. sha1 chỉ tính trên ba split MSVD, nên **trùng với v2**; kiểm tra C7 xác nhận điều này. Tập ngoài `ext` có sha1 riêng và không được trùng video nào với MSVD."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c014",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "SPLITS_MSVD = (\\"train\\", \\"val\\", \\"test\\")\\n",\n    "\\n",\n    "\\n",\n    "def make_split(records, write=True):\\n",\n    "    \\"\\"\\"Split MSVD giữ NGUYÊN như v1/v2 (sha1 chỉ tính trên train/val/test MSVD ⇒ so sánh được với v2);\\n",\n    "    \'ext\' = benchmark xác nhận độc lập (MSR-VTT), có sha1 riêng.\\"\\"\\"\\n",\n    "    keys = SPLITS_MSVD + (\\"ext\\",)\\n",\n    "    split = {k: [r[\\"vid\\"] for r in records if r[\\"split\\"] == k] for k in keys}\\n",\n    "    ids = {k: [records[v][\\"video_id\\"] for v in split[k]] for k in keys}\\n",\n    "    for a, b in itertools.combinations(keys, 2):\\n",\n    "        assert not set(split[a]) & set(split[b]) and not set(ids[a]) & set(ids[b]), f\\"{a}/{b} chồng lấn video!\\"\\n",\n    "    payload = {k: sorted(ids[k]) for k in SPLITS_MSVD}\\n",\n    "    split[\\"sha1\\"] = sha1_of(payload)\\n",\n    "    split[\\"ext_sha1\\"] = sha1_of(sorted(ids[\\"ext\\"])) if ids[\\"ext\\"] else None\\n",\n    "    if write:\\n",\n    "        manifest = dict(\\n",\n    "            dataset=CFG.dataset, hf_repo=CFG.hf_repo, hf_revision=CFG.hf_revision, sha1=split[\\"sha1\\"],\\n",\n    "            policy=(\\"split chính thức của MSVD theo msvd_{train,val,test}.json; chia THEO VIDEO (không theo caption); \\"\\n",\n    "                    \\"1 video = 1 ứng viên trong gallery của split đó; mỗi caption = 1 truy vấn có đúng 1 video dương\\"),\\n",\n    "            counts={k: dict(videos=len(split[k]), captions=int(sum(records[v][\\"n_captions\\"] for v in split[k]))) for k in keys},\\n",\n    "            video_ids=payload, ext=dict(sha1=split[\\"ext_sha1\\"], source=EXT_REPORT.get(\\"label\\") if EXT_REPORT else None,\\n",\n    "                                        video_ids=sorted(ids[\\"ext\\"])))\\n",\n    "        save_json(manifest, RESULTS / \\"split_manifest.json\\")\\n",\n    "        print(f\\"Split (sha1={split[\'sha1\']}): \\" + \\" | \\".join(\\n",\n    "            f\\"{k}={manifest[\'counts\'][k][\'videos\']} video / {manifest[\'counts\'][k][\'captions\']} caption\\" for k in keys))\\n",\n    "    return split"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c015",\n   "metadata": {},\n   "source": [\n    "## 3. Lấy mẫu khung hình và đặc trưng CLIP đóng băng (Cổng B)\\n",\n    "\\n",\n    "* **Chính sách lấy mẫu `uniform-segment-midpoint`:** video được chia thành `F_max = 12` đoạn bằng nhau, mỗi đoạn lấy frame ở giữa: `idx_k = floor((k + 0.5)/12 · n)`, với `n` là số frame **giải mã được thực sự**. Số này được đếm bằng `grab()`, vì `CAP_PROP_FRAME_COUNT` của AVI có thể sai. Ở lượt đọc thứ hai, notebook chỉ `retrieve()` đúng 12 frame cần lấy, nên bộ nhớ chỉ chứa 12 frame/video.\\n",\n    "* Mọi frame được mã hoá **một lần** bằng CLIP-image đóng băng, chuẩn hoá L2, và lưu cache `[N_video, 12, D]` (float16) cùng chỉ số frame, timestamp, fps và thời lượng. Cache được dò ở `msvd_l4/cache` và ở `/kaggle/input/**/cache`.\\n",\n    "* Mọi caption được mã hoá **một lần** bằng CLIP-text của cùng mô hình. Bước này không học gì, chỉ để cache đặc trưng; việc dùng đặc trưng caption về sau vẫn phải theo luật của `CaptionStore`.\\n",\n    "* Thumbnail của frame giữa video (rộng 160 px) được lưu để demo hiển thị kết quả."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c016",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "_CLIP = {}\\n",\n    "\\n",\n    "\\n",\n    "def get_clip():\\n",\n    "    \\"\\"\\"CLIP/OpenCLIP tiền huấn luyện, đóng băng tuyệt đối (eval + requires_grad=False). Nạp một lần.\\"\\"\\"\\n",\n    "    if \\"m\\" not in _CLIP:\\n",\n    "        m, _, pre = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)\\n",\n    "        m = m.to(DEVICE).eval()\\n",\n    "        for p in m.parameters():\\n",\n    "            p.requires_grad_(False)\\n",\n    "        _CLIP.update(m=m, pre=pre, tok=open_clip.get_tokenizer(CFG.clip_model))\\n",\n    "    return _CLIP[\\"m\\"], _CLIP[\\"pre\\"], _CLIP[\\"tok\\"]\\n",\n    "\\n",\n    "\\n",\n    "def _amp():\\n",\n    "    return torch.autocast(\\"cuda\\", dtype=torch.float16) if DEVICE == \\"cuda\\" else contextlib.nullcontext()\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def clip_encode_images(x):\\n",\n    "    m, _, _ = get_clip()\\n",\n    "    with _amp():\\n",\n    "        f = m.encode_image(x.to(DEVICE))\\n",\n    "    return F.normalize(f.float(), dim=-1).cpu().numpy()\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def clip_encode_texts(texts, bs=None):\\n",\n    "    m, _, tok = get_clip()\\n",\n    "    bs = bs or CFG.text_bs\\n",\n    "    out = []\\n",\n    "    for s in range(0, len(texts), bs):\\n",\n    "        with _amp():\\n",\n    "            f = m.encode_text(tok(texts[s:s + bs]).to(DEVICE))\\n",\n    "        out.append(F.normalize(f.float(), dim=-1).cpu().numpy())\\n",\n    "    return np.concatenate(out)\\n",\n    "\\n",\n    "\\n",\n    "def segment_midpoints(n, n_frames):\\n",\n    "    \\"\\"\\"Chỉ số frame tại điểm giữa của n_frames đoạn đều nhau trên n frame giải mã được.\\"\\"\\"\\n",\n    "    return np.minimum(np.floor((np.arange(n_frames) + 0.5) / n_frames * n).astype(int), n - 1)\\n",\n    "\\n",\n    "\\n",\n    "def read_sampled_frames(path, n_frames):\\n",\n    "    \\"\\"\\"Lượt 1: đếm frame GIẢI MÃ ĐƯỢC. Lượt 2: chỉ retrieve các frame cần lấy (RGB). Trả về (frames, meta).\\"\\"\\"\\n",\n    "    cap = cv2.VideoCapture(str(path))\\n",\n    "    if not cap.isOpened():\\n",\n    "        raise IOError(\\"cv2 không mở được video\\")\\n",\n    "    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)\\n",\n    "    n_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\\n",\n    "    n = 0\\n",\n    "    while cap.grab():\\n",\n    "        n += 1\\n",\n    "    cap.release()\\n",\n    "    if n == 0:\\n",\n    "        raise IOError(\\"0 frame giải mã được\\")\\n",\n    "    idx = segment_midpoints(n, n_frames)\\n",\n    "    want, got = set(idx.tolist()), {}\\n",\n    "    cap = cv2.VideoCapture(str(path))\\n",\n    "    i = 0\\n",\n    "    while i <= idx[-1] and cap.grab():\\n",\n    "        if i in want:\\n",\n    "            ok, fr = cap.retrieve()\\n",\n    "            if ok:\\n",\n    "                got[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)\\n",\n    "        i += 1\\n",\n    "    cap.release()\\n",\n    "    lost = sorted(j for j in want if j not in got)\\n",\n    "    if lost:\\n",\n    "        raise IOError(f\\"không đọc được frame {lost[:5]}\\")\\n",\n    "    fps_eff = fps if fps > 0 else 30.0\\n",\n    "    meta = dict(n_decoded=n, n_meta=n_meta, fps=round(fps, 3), fps_assumed=fps <= 0,\\n",\n    "                duration_s=round(n / fps_eff, 3), frame_idx=idx.tolist(),\\n",\n    "                t_sec=[round(float(j) / fps_eff, 3) for j in idx])\\n",\n    "    return [got[int(j)] for j in idx], meta\\n",\n    "\\n",\n    "\\n",\n    "def _cache_name(prefix, n, data_tag=None):\\n",\n    "    data_tag = data_tag or f\\"{CFG.dataset}\\"\\n",\n    "    rev = CFG.hf_revision[:8] if data_tag == CFG.dataset else \\"\\"\\n",\n    "    tag = f\\"{data_tag}_{CFG.clip_model}_{CFG.clip_weights}_F{CFG.n_frames_max}_{CFG.frame_policy}_{rev}_{n}\\".replace(\\"__\\", \\"_\\")\\n",\n    "    return f\\"{prefix}_\\" + re.sub(r\\"[^A-Za-z0-9_-]\\", \\"-\\", tag)\\n",\n    "\\n",\n    "\\n",\n    "def find_cache(name):\\n",\n    "    \\"\\"\\"Cache của v2, cache của v1 (MSVD_L4_CACHE_DIRS, ví dụ output v1 đã Add Input) và /kaggle/input/**/cache.\\"\\"\\"\\n",\n    "    dirs = [CFG.out_dir / \\"cache\\"] + [Path(d) for d in CFG.extra_cache_dirs]\\n",\n    "    if RUN_ON_KAGGLE:\\n",\n    "        dirs += [p for p in Path(\\"/kaggle/input\\").glob(\\"*/**/cache\\") if p.is_dir()]\\n",\n    "    for d in dirs:\\n",\n    "        if (d / name).exists():\\n",\n    "            return d / name\\n",\n    "    return None\\n",\n    "\\n",\n    "\\n",\n    "def extract_frame_features(records, source, allow_encode=True, data_tag=None):\\n",\n    "    \\"\\"\\"[N_video, F_max, D] float16 (mỗi frame đã chuẩn hoá L2) + meta lấy mẫu. Dùng cache nếu có.\\n",\n    "    Hàng thứ j ứng với records[j] (không giả định vid = j: tập ngoài nối sau MSVD).\\"\\"\\"\\n",\n    "    name = _cache_name(\\"frames\\", len(records), data_tag) + \\".npz\\"\\n",\n    "    vids = [r[\\"video_id\\"] for r in records]\\n",\n    "    p = find_cache(name)\\n",\n    "    if p is not None:\\n",\n    "        z = np.load(p, allow_pickle=False)\\n",\n    "        assert z[\\"video_ids\\"].tolist() == vids, \\"cache đặc trưng frame không khớp thứ tự video\\"\\n",\n    "        meta = json.load(open(p.with_suffix(\\".json\\")))\\n",\n    "        print(f\\"Dùng lại cache đặc trưng frame: {p} {z[\'feat\'].shape}\\")\\n",\n    "        return z[\\"feat\\"].astype(np.float32), meta, p\\n",\n    "    if not allow_encode:\\n",\n    "        raise FileNotFoundError(name)\\n",\n    "    _, pre, _ = get_clip()\\n",\n    "    Fm, N = CFG.n_frames_max, len(records)\\n",\n    "    feats, metas, bad = None, [None] * N, {}\\n",\n    "    thumbs = CFG.out_dir / \\"thumbs\\"\\n",\n    "\\n",\n    "    def work(j):\\n",\n    "        r = records[j]\\n",\n    "        try:\\n",\n    "            with source.open(r[\\"video_id\\"]) as path:\\n",\n    "                frames, meta = read_sampled_frames(path, Fm)\\n",\n    "            mid = frames[Fm // 2]\\n",\n    "            h, w = mid.shape[:2]\\n",\n    "            th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)\\n",\n    "            cv2.imwrite(str(thumbs / f\\"{r[\'video_id\']}.jpg\\"), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))\\n",\n    "            return j, torch.stack([pre(Image.fromarray(f)) for f in frames]), meta\\n",\n    "        except Exception as e:                          # ghi lại, quyết định dừng hay không ở dưới\\n",\n    "            return j, None, f\\"{type(e).__name__}: {e}\\"\\n",\n    "\\n",\n    "    from concurrent.futures import ThreadPoolExecutor\\n",\n    "    t0, chunk = time.time(), max(8, 4 * CFG.clip_bs // Fm)\\n",\n    "    with ThreadPoolExecutor(CFG.decode_workers) as ex:\\n",\n    "        for s in tqdm(range(0, N, chunk), desc=f\\"giải mã + CLIP ({N} video)\\", unit=\\"lô\\"):\\n",\n    "            out = list(ex.map(work, range(s, min(s + chunk, N))))     # theo lô ⇒ bộ nhớ có chặn trên\\n",\n    "            ok = [(v, x, m) for v, x, m in out if x is not None]\\n",\n    "            bad.update({v: m for v, x, m in out if x is None})\\n",\n    "            if not ok:\\n",\n    "                continue\\n",\n    "            X = torch.cat([x for _, x, _ in ok])\\n",\n    "            E = np.concatenate([clip_encode_images(X[i:i + CFG.clip_bs]) for i in range(0, len(X), CFG.clip_bs)])\\n",\n    "            if feats is None:\\n",\n    "                feats = np.zeros((N, Fm, E.shape[1]), np.float16)\\n",\n    "            for j, (v, _, m) in enumerate(ok):\\n",\n    "                feats[v] = E[j * Fm:(j + 1) * Fm]\\n",\n    "                metas[v] = m\\n",\n    "    if bad:\\n",\n    "        print(f\\"⚠ {len(bad)} video giải mã lỗi:\\", list(bad.items())[:5])\\n",\n    "    assert feats is not None, \\"Không giải mã được video nào\\"\\n",\n    "    assert len(bad) / N <= CFG.max_bad_video_frac, (\\n",\n    "        f\\"{len(bad)}/{N} video lỗi > ngưỡng {CFG.max_bad_video_frac:.0%} — dừng, không chạy thí nghiệm trên dữ liệu hỏng\\")\\n",\n    "    meta = dict(policy=CFG.frame_policy, n_frames_max=Fm, clip=f\\"{CFG.clip_model}/{CFG.clip_weights}\\",\\n",\n    "                source=source.kind, seconds=round(time.time() - t0, 1),\\n",\n    "                per_video={records[v][\\"video_id\\"]: m for v, m in enumerate(metas)},\\n",\n    "                bad={records[v][\\"video_id\\"]: e for v, e in bad.items()})\\n",\n    "    p = CFG.out_dir / \\"cache\\" / name\\n",\n    "    np.savez(p, feat=feats, video_ids=np.array(vids))\\n",\n    "    save_json(meta, p.with_suffix(\\".json\\"))\\n",\n    "    print(f\\"Đặc trưng frame {feats.shape} sau {meta[\'seconds\']}s → {p}\\")\\n",\n    "    return feats.astype(np.float32), meta, p\\n",\n    "\\n",\n    "\\n",\n    "def extract_text_features(store, cap_idx=None, allow_encode=True, data_tag=None):\\n",\n    "    \\"\\"\\"[N_caption, D]: CLIP-text đóng băng của các caption cap_idx (mặc định: mọi caption; mục đích \'encode\' — không học gì).\\"\\"\\"\\n",\n    "    cap_idx = np.arange(len(store.vid_of_cap)) if cap_idx is None else np.asarray(cap_idx)\\n",\n    "    texts = store.texts(cap_idx, \\"encode\\")\\n",\n    "    digest = hashlib.sha1(\\"\\\\n\\".join(texts).encode()).hexdigest()[:10]\\n",\n    "    name = _cache_name(\\"text\\", len(texts), data_tag) + f\\"_{digest}.npz\\"\\n",\n    "    p = find_cache(name)\\n",\n    "    if p is not None:\\n",\n    "        feat = np.load(p)[\\"feat\\"].astype(np.float32)\\n",\n    "        assert feat.shape[0] == len(texts)\\n",\n    "        print(f\\"Dùng lại cache đặc trưng caption: {p} {feat.shape}\\")\\n",\n    "        return feat, p\\n",\n    "    if not allow_encode:\\n",\n    "        raise FileNotFoundError(name)\\n",\n    "    t0 = time.time()\\n",\n    "    feat = clip_encode_texts(texts).astype(np.float16)\\n",\n    "    p = CFG.out_dir / \\"cache\\" / name\\n",\n    "    np.savez(p, feat=feat)\\n",\n    "    print(f\\"Đặc trưng caption {feat.shape} sau {time.time() - t0:.0f}s → {p}\\")\\n",\n    "    return feat.astype(np.float32), p\\n",\n    "\\n",\n    "\\n",\n    "def ext_data_tag():\\n",\n    "    if not EXT_RECORDS:\\n",\n    "        return None\\n",\n    "    if CFG.dataset != \\"msvd\\":\\n",\n    "        return \\"synthetic_ext\\"\\n",\n    "    return f\\"msrvtt_{CFG.msrvtt_revision[:8]}_{\'dev\' + str(len(EXT_RECORDS)) if EXT_REPORT[\'dev\'] else EXT_REPORT.get(\'set\', \'1kA\')}\\"\\n",\n    "\\n",\n    "\\n",\n    "def all_features(allow_encode=True):\\n",\n    "    \\"\\"\\"MSVD (cache v1/v2 dùng lại được: cùng tên, cùng thứ tự) + tập ngoài (cache riêng), nối theo thứ tự RECORDS / caption.\\"\\"\\"\\n",\n    "    fm, mm, pm = extract_frame_features(RECORDS[:N_MSVD], SOURCE, allow_encode)\\n",\n    "    in_msvd = CAP_VID < N_MSVD\\n",\n    "    assert in_msvd[:in_msvd.sum()].all(), \\"caption MSVD phải đứng trước caption tập ngoài\\"\\n",\n    "    tm, ptm = extract_text_features(CAPS, np.where(in_msvd)[0], allow_encode)\\n",\n    "    caches = [pm, ptm]\\n",\n    "    if EXT_RECORDS:\\n",\n    "        fe, me, pe = extract_frame_features(RECORDS[N_MSVD:], EXT_SOURCE, allow_encode, data_tag=ext_data_tag())\\n",\n    "        te, pte = extract_text_features(CAPS, np.where(~in_msvd)[0], allow_encode, data_tag=ext_data_tag())\\n",\n    "        fm, tm = np.concatenate([fm, fe]), np.concatenate([tm, te])\\n",\n    "        mm = dict(mm, per_video={**mm[\\"per_video\\"], **me[\\"per_video\\"]}, bad={**mm[\\"bad\\"], **me[\\"bad\\"]},\\n",\n    "                  seconds=round(mm[\\"seconds\\"] + me[\\"seconds\\"], 1) if \\"seconds\\" in mm and \\"seconds\\" in me else None,\\n",\n    "                  source=f\\"{mm[\'source\']}+{me[\'source\']}\\")\\n",\n    "        caches += [pe, pte]\\n",\n    "    return fm, mm, tm, caches"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c017",\n   "metadata": {},\n   "source": [\n    "### 3b. Gộp theo thời gian (chọn trên VAL)\\n",\n    "\\n",\n    "Notebook dùng một embedding duy nhất cho mỗi video, gộp từ `F ∈ {4, 8, 12}` frame con, cách đều nhau trong 12 frame đã lấy mẫu. Có hai cách gộp:\\n",\n    "\\n",\n    "* `mean`: chuẩn hoá L2 từng frame, lấy trung bình, rồi chuẩn hoá lại.\\n",\n    "* `centrality`: trọng số `softmax(cos(frame, trung bình)/τ)`. Cách này giảm trọng số các frame lạc chủ đề, chẳng hạn frame chuyển cảnh hoặc frame tối; trọng số không phụ thuộc truy vấn, nên mỗi video vẫn chỉ có một vector.\\n",\n    "\\n",\n    "Cặp `(F, cách gộp)` được chọn theo **MRR text→video của CLIP thuần trên VAL**, xếp hạng trong toàn bộ gallery val. Dòng `F=1` (frame giữa) chỉ được in ra để tham chiếu và không được chọn, để bảo đảm embedding thật sự gộp từ nhiều frame."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c018",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def frame_subset(n_frames):\\n",\n    "    \\"\\"\\"F frame con cách đều trong F_max frame đã lấy mẫu (F=4 ⇒ đúng điểm giữa của 4 đoạn: 1, 4, 7, 10).\\"\\"\\"\\n",\n    "    n_frames = min(int(n_frames), CFG.n_frames_max)\\n",\n    "    pos = np.round((np.arange(n_frames) + 0.5) * CFG.n_frames_max / n_frames - 0.5)\\n",\n    "    return np.unique(np.clip(pos, 0, CFG.n_frames_max - 1).astype(int))\\n",\n    "\\n",\n    "\\n",\n    "def pool_frames(frame_feat, n_frames, mode):\\n",\n    "    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n",\n    "    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n",\n    "    if mode == \\"mean\\" or X.shape[1] == 1:\\n",\n    "        v = X.mean(1)\\n",\n    "    elif mode == \\"centrality\\":\\n",\n    "        c = X.mean(1, keepdims=True)\\n",\n    "        c /= np.linalg.norm(c, axis=-1, keepdims=True).clip(1e-8)\\n",\n    "        a = (X * c).sum(-1) / CFG.centrality_tau\\n",\n    "        w = np.exp(a - a.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)\\n",\n    "        v = (w[..., None] * X).sum(1)\\n",\n    "    else:\\n",\n    "        raise ValueError(mode)\\n",\n    "    return (v / np.linalg.norm(v, axis=-1, keepdims=True).clip(1e-8)).astype(np.float32)\\n",\n    "\\n",\n    "\\n",\n    "def full_gallery_ranks(txt, vid, q_idx, gallery, gold):\\n",\n    "    \\"\\"\\"Thứ hạng (1 = tốt nhất) của video đúng trong TOÀN BỘ gallery theo cosine CLIP.\\"\\"\\"\\n",\n    "    G = torch.from_numpy(np.ascontiguousarray(vid[np.asarray(gallery)]))\\n",\n    "    pos = {int(v): i for i, v in enumerate(gallery)}\\n",\n    "    gi = torch.tensor([pos[int(v)] for v in gold], dtype=torch.long)\\n",\n    "    out = np.empty(len(q_idx), np.int64)\\n",\n    "    for s in range(0, len(q_idx), CFG.eval_chunk):\\n",\n    "        S = torch.from_numpy(np.ascontiguousarray(txt[np.asarray(q_idx[s:s + CFG.eval_chunk])])) @ G.T\\n",\n    "        g = S[torch.arange(S.shape[0]), gi[s:s + CFG.eval_chunk]]\\n",\n    "        out[s:s + CFG.eval_chunk] = (S > g[:, None]).sum(1).numpy() + 1\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def select_pooling_on_val(frame_feat, txt, split):\\n",\n    "    va = np.where(CAP_SPLIT == \\"val\\")[0]\\n",\n    "    CAPS.use(va, \\"tune\\")\\n",\n    "    gal, gold = np.array(split[\\"val\\"]), CAP_VID[va]\\n",\n    "    rows = []\\n",\n    "    for nf in sorted(set(CFG.n_frames_grid) | {1}):\\n",\n    "        for mode in CFG.pool_grid:\\n",\n    "            if nf == 1 and mode != \\"mean\\":\\n",\n    "                continue\\n",\n    "            m = metrics_from_ranks(full_gallery_ranks(txt, pool_frames(frame_feat, nf, mode), va, gal, gold), gold)\\n",\n    "            rows.append(dict(n_frames=nf, pooling=mode, frames=frame_subset(nf).tolist(),\\n",\n    "                             selectable=nf in CFG.n_frames_grid, **{k: m[k] for k in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}))\\n",\n    "    df = pd.DataFrame(rows)\\n",\n    "    best = df[df.selectable].sort_values([\\"MRR\\", \\"n_frames\\"], ascending=[False, True]).iloc[0]\\n",\n    "    return dict(n_frames=int(best.n_frames), pooling=str(best.pooling), val_MRR=float(best.MRR)), df\\n",\n    "\\n",\n    "\\n",\n    "def gate_b_checks(frame_feat, frame_meta, vid_feat, txt, split):\\n",\n    "    N, Fm, D = frame_feat.shape\\n",\n    "    ff = frame_feat.reshape(-1, D)\\n",\n    "    fn = np.linalg.norm(ff, axis=1)\\n",\n    "    m, _, _ = get_clip()\\n",\n    "    check(\\"B0\\", \\"CLIP đóng băng (eval, không tham số nào requires_grad)\\",\\n",\n    "          not m.training and not any(p.requires_grad for p in m.parameters()), f\\"{CFG.clip_model}/{CFG.clip_weights}\\")\\n",\n    "    check(\\"B1\\", \\"đặc trưng frame có shape [N_video, F_max, D]\\", (N, Fm) == (len(RECORDS), CFG.n_frames_max),\\n",\n    "          f\\"{frame_feat.shape}\\")\\n",\n    "    check(\\"B2\\", \\"đặc trưng frame hữu hạn, chuẩn L2 ≈ 1\\", bool(np.isfinite(ff).all() and np.abs(fn - 1).max() < 2e-2),\\n",\n    "          f\\"max|‖f‖−1| = {np.abs(fn - 1).max():.2e}\\")\\n",\n    "    vn = np.linalg.norm(vid_feat, axis=1)\\n",\n    "    check(\\"B3\\", \\"đúng 1 embedding / video, hữu hạn, chuẩn 1, video_id duy nhất\\",\\n",\n    "          vid_feat.shape == (N, D) and bool(np.isfinite(vid_feat).all()) and np.abs(vn - 1).max() < 1e-4\\n",\n    "          and len({r[\\"video_id\\"] for r in RECORDS}) == N, f\\"{vid_feat.shape}\\")\\n",\n    "    check(\\"B4\\", \\"đúng 1 embedding / caption, hữu hạn, cùng chiều với video\\",\\n",\n    "          txt.shape == (len(CAP_VID), D) and bool(np.isfinite(txt).all()), f\\"{txt.shape}\\")\\n",\n    "    qs = np.random.RandomState(0).choice(len(txt), min(3000, len(txt)), replace=False)\\n",\n    "    S = txt[qs] @ vid_feat.T\\n",\n    "    check(\\"B5\\", \\"cosine văn bản–video nằm trong [−1, 1]\\", S.min() >= -1 - 1e-3 and S.max() <= 1 + 1e-3,\\n",\n    "          f\\"[{S.min():.3f}, {S.max():.3f}]\\")\\n",\n    "    idx = frame_subset(POOL_SEL[\\"n_frames\\"])\\n",\n    "    Xn = frame_feat[:, idx] / np.linalg.norm(frame_feat[:, idx], axis=-1, keepdims=True).clip(1e-8)\\n",\n    "    cos_fv = (Xn * vid_feat[:, None]).sum(-1)\\n",\n    "    frac_single = float((cos_fv.max(1) > 0.9999).mean())\\n",\n    "    pair = np.einsum(\\"nfd,ngd->nfg\\", Xn, Xn)\\n",\n    "    off = pair[:, ~np.eye(len(idx), dtype=bool)].mean() if len(idx) > 1 else 1.0\\n",\n    "    check(\\"B6\\", \\"embedding video thật sự gộp từ NHIỀU frame (không vô tình dùng 1 frame)\\",\\n",\n    "          len(idx) > 1 and frac_single < 0.05,\\n",\n    "          f\\"F={len(idx)} frame {idx.tolist()} ({POOL_SEL[\'pooling\']}); tỉ lệ video ≡ 1 frame = {frac_single:.3f}; \\"\\n",\n    "          f\\"cos TB giữa các frame = {off:.3f}\\")\\n",\n    "    bad_ts = []\\n",\n    "    for name, mt in frame_meta[\\"per_video\\"].items():\\n",\n    "        if mt is None:\\n",\n    "            bad_ts.append(name); continue\\n",\n    "        t = np.array(mt[\\"t_sec\\"])\\n",\n    "        ok = (len(t) == Fm and (t >= 0).all() and (np.diff(t) >= 0).all() and t.max() < mt[\\"duration_s\\"] + 1e-6\\n",\n    "              and max(mt[\\"frame_idx\\"]) < mt[\\"n_decoded\\"])\\n",\n    "        if not ok:\\n",\n    "            bad_ts.append(name)\\n",\n    "    check(\\"B7\\", \\"lấy mẫu ổn định: đủ F_max frame / video, timestamp tăng dần trong [0, thời lượng)\\", not bad_ts,\\n",\n    "          f\\"{len(frame_meta[\'per_video\'])} video; lỗi {len(bad_ts)} {bad_ts[:3]}\\")\\n",\n    "    cnt = np.bincount(CAP_VID, minlength=N)\\n",\n    "    vsplit = np.array([r[\\"split\\"] for r in RECORDS])\\n",\n    "    probe = np.random.RandomState(1).choice(len(CAP_VID), min(500, len(CAP_VID)), replace=False)\\n",\n    "    ok_text = all(CAPS._text[q] in CAPS._caps[int(CAP_VID[q])] for q in probe)        # kiểm tra chỉ số nội bộ\\n",\n    "    galleries_ok = all(len(set(split[k])) == len(split[k]) and set(CAP_VID[CAP_SPLIT == k]) == set(split[k])\\n",\n    "                       for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\"))\\n",\n    "    check(\\"B8\\", \\"mọi caption-truy vấn ánh xạ đúng 1 video dương cùng split; gallery mỗi split không nhân bản video\\",\\n",\n    "          all(cnt[r[\\"vid\\"]] == r[\\"n_captions\\"] for r in RECORDS) and bool((vsplit[CAP_VID] == CAP_SPLIT).all())\\n",\n    "          and ok_text and galleries_ok,\\n",\n    "          f\\"{len(CAP_VID)} truy vấn → {N} video; caption/video TB {cnt.mean():.1f}; gallery = \\"\\n",\n    "          + \\", \\".join(f\\"{k}:{len(split[k])}\\" for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\")))\\n",\n    "    f2, _, t2, paths = all_features(allow_encode=False)\\n",\n    "    check(\\"B9\\", \\"cache đặc trưng nạp lại được mà KHÔNG chạy encoder, giống hệt bản trong bộ nhớ\\",\\n",\n    "          np.array_equal(f2, frame_feat) and np.array_equal(t2, txt), \\" | \\".join(p.name for p in paths))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c019",\n   "metadata": {},\n   "source": [\n    "## 4. Từ vựng (chỉ TRAIN), quan hệ, tagger và mẫu cho người chấm\\n",\n    "\\n",\n    "Parser quan hệ của v2 được **giữ nguyên**, để đo đúng chất lượng của nó. Mỗi quan hệ là `(nguồn, họ, đích, vị từ)`:\\n",\n    "* `agent_of`: chủ thể → động từ;\\n",\n    "* `object_of`: đồ vật → động từ. Ở hai họ này vị từ chính là động từ đích;\\n",\n    "* `svo`: danh từ –cụm động từ→ danh từ, ví dụ `man –ride→ horse`;\\n",\n    "* `prep`: danh từ –giới từ→ danh từ, ví dụ `cat –on→ table`.\\n",\n    "\\n",\n    "**Kiểm chứng tự động (Nhiệm vụ B), không cần người chấm.**\\n",\n    "* spaCy `en_core_web_sm` là parser phụ thuộc cú pháp, độc lập với heuristic POS. Nó trích lại quan hệ cùng schema trên **chính caption train**:\\n",\n    "  * chủ ngữ (`nsubj`, kể cả kế thừa qua `conj`, `xcomp`, `acl`) → `agent_of`;\\n",\n    "  * tân ngữ (`dobj`, `nsubjpass`) → `object_of`;\\n",\n    "  * chủ–động–tân → `svo`;\\n",\n    "  * giới từ gắn vào **danh từ** → `prep`.\\n",\n    "* Một quan hệ heuristic được xác nhận khi spaCy trích cùng (nguồn, họ, đích), cùng động từ đầu (svo) hoặc cùng giới từ (prep). Tỉ lệ xác nhận được tính cho từng quan hệ, từng họ và từng ô (họ × mức phổ biến), kèm khoảng Wilson.\\n",\n    "* Độ phủ ngược (spaCy tìm được mà heuristic cũng tìm được) chỉ là so sánh tương đối, **không** phải recall thật.\\n",\n    "* Đây là nhãn **silver**: hai parser có thể cùng sai, nên báo cáo không gọi nó là precision.\\n",\n    "\\n",\n    "**Mẫu phân tầng** (họ × mức phổ biến; 360 dòng) vẫn được xuất kèm cột `auto_parser_agrees`. Nếu có file nhãn người chấm (`relation_annotation*`, **tuỳ chọn**), notebook tính thêm precision, Wilson và Cohen\'s kappa."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c020",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# --- NLTK: WordNet + POS tagger. Không tải được thì cảnh báo rõ và dùng phương án dự phòng (ghi vào ENV_INFO). ---\\n",\n    "NLTK_STATUS = {}\\n",\n    "for _pkg in (\\"wordnet\\", \\"omw-1.4\\", \\"averaged_perceptron_tagger_eng\\", \\"averaged_perceptron_tagger\\"):\\n",\n    "    try:\\n",\n    "        NLTK_STATUS[_pkg] = bool(nltk.download(_pkg, quiet=True, raise_on_error=False))\\n",\n    "    except Exception as e:\\n",\n    "        NLTK_STATUS[_pkg] = f\\"lỗi {type(e).__name__}\\"\\n",\n    "from nltk.stem import WordNetLemmatizer\\n",\n    "\\n",\n    "_LEM, _LEM_CACHE = WordNetLemmatizer(), {}\\n",\n    "\\n",\n    "\\n",\n    "def _verb_lemma(w):\\n",\n    "    \\"\\"\\"WordNetLemmatizer trả về ứng viên NGẮN NHẤT (\'riding\' → \'rid\', \'shining\' → \'shin\'); ở đây chọn ứng viên\\n",\n    "    có tần suất WordNet cao nhất (\'riding\' → \'ride\').\\"\\"\\"\\n",\n    "    wn = _wordnet()\\n",\n    "    cands = wn._morphy(w, \\"v\\") if wn is not None and hasattr(wn, \\"_morphy\\") else []\\n",\n    "    if len(cands) > 1:\\n",\n    "        cnt = lambda c: sum(l.count() for s in wn.synsets(c, pos=\\"v\\") for l in s.lemmas() if l.name().lower() == c)\\n",\n    "        return max(sorted(cands), key=lambda c: (cnt(c), len(c)))      # sorted: không phụ thuộc PYTHONHASHSEED\\n",\n    "    return sorted(cands)[0] if cands else _LEM.lemmatize(w, \\"v\\")\\n",\n    "\\n",\n    "\\n",\n    "def lemma(w, pos=\\"n\\"):\\n",\n    "    k = (w, pos)\\n",\n    "    r = _LEM_CACHE.get(k)\\n",\n    "    if r is None:\\n",\n    "        try:\\n",\n    "            if pos == \\"v\\":\\n",\n    "                r = _verb_lemma(w)\\n",\n    "            else:                                   # như NLTK (ứng viên ngắn nhất) nhưng hoà thì theo thứ tự chữ\\n",\n    "                wn_ = _wordnet()\\n",\n    "                cands = wn_._morphy(w, pos) if wn_ is not None and hasattr(wn_, \\"_morphy\\") else []\\n",\n    "                r = min(sorted(cands), key=len) if cands else _LEM.lemmatize(w, pos)\\n",\n    "        except LookupError:\\n",\n    "            r = w[:-1] if (pos == \\"n\\" and w.endswith(\\"s\\") and len(w) > 3) else w\\n",\n    "        _LEM_CACHE[k] = r\\n",\n    "    return r\\n",\n    "\\n",\n    "\\n",\n    "try:\\n",\n    "    from nltk.tag import PerceptronTagger\\n",\n    "    _TAGGER = PerceptronTagger()                    # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)\\n",\n    "    _TAGGER.tag([\\"a\\", \\"man\\", \\"is\\", \\"playing\\"])\\n",\n    "except Exception as e:\\n",\n    "    _TAGGER = None\\n",\n    "    print(\\"⚠ Không nạp được POS tagger NLTK → phân loại từ theo WordNet (kém chính xác hơn):\\", type(e).__name__)\\n",\n    "\\n",\n    "# --- Lớp khái niệm dự phòng (notebook gốc) — dùng khi WordNet không có ---\\n",\n    "FALLBACK_ISA = {\\n",\n    "    \\"ball\\": [\\"sports equipment\\", \\"toy\\"], \\"field\\": [\\"open area\\", \\"sports venue\\"],\\n",\n    "    \\"grass\\": [\\"vegetation\\", \\"ground\\"], \\"goal\\": [\\"sports equipment\\"],\\n",\n    "    \\"shoe\\": [\\"clothing\\", \\"footwear\\"], \\"child\\": [\\"person\\"], \\"boy\\": [\\"person\\", \\"child\\"],\\n",\n    "    \\"girl\\": [\\"person\\", \\"child\\"], \\"man\\": [\\"person\\"], \\"woman\\": [\\"person\\"],\\n",\n    "    \\"person\\": [\\"human\\"], \\"keeper\\": [\\"person\\"],\\n",\n    "    \\"pot\\": [\\"cookware\\", \\"container\\"], \\"stove\\": [\\"appliance\\", \\"cooking device\\"],\\n",\n    "    \\"plate\\": [\\"tableware\\", \\"container\\"], \\"spoon\\": [\\"tableware\\", \\"utensil\\"],\\n",\n    "    \\"table\\": [\\"furniture\\"], \\"bed\\": [\\"furniture\\"], \\"lamp\\": [\\"furniture\\", \\"light source\\"],\\n",\n    "    \\"pillow\\": [\\"bedding\\"], \\"curtain\\": [\\"window covering\\"], \\"window\\": [\\"building part\\"],\\n",\n    "    \\"book\\": [\\"reading material\\"], \\"sand\\": [\\"ground\\"], \\"umbrella\\": [\\"shade device\\"],\\n",\n    "    \\"wave\\": [\\"water\\"], \\"towel\\": [\\"cloth\\"], \\"boat\\": [\\"vehicle\\", \\"watercraft\\"],\\n",\n    "    \\"car\\": [\\"vehicle\\"], \\"truck\\": [\\"vehicle\\"], \\"bus\\": [\\"vehicle\\"], \\"bike\\": [\\"vehicle\\"],\\n",\n    "    \\"sign\\": [\\"signage\\"], \\"road\\": [\\"open area\\", \\"transport way\\"],\\n",\n    "    \\"building\\": [\\"structure\\"], \\"light\\": [\\"light source\\"], \\"elephant\\": [\\"animal\\"],\\n",\n    "    \\"dog\\": [\\"animal\\"], \\"cat\\": [\\"animal\\"], \\"horse\\": [\\"animal\\"], \\"bird\\": [\\"animal\\"],\\n",\n    "    \\"tree\\": [\\"vegetation\\", \\"plant\\"], \\"fence\\": [\\"barrier\\", \\"structure\\"],\\n",\n    "    \\"water\\": [\\"liquid\\"], \\"rock\\": [\\"natural object\\"], \\"shirt\\": [\\"clothing\\"],\\n",\n    "    \\"hat\\": [\\"clothing\\"], \\"chair\\": [\\"furniture\\"], \\"food\\": [\\"edible\\"],\\n",\n    "    \\"pizza\\": [\\"food\\", \\"edible\\"], \\"plane\\": [\\"vehicle\\"], \\"train\\": [\\"vehicle\\"],\\n",\n    "    \\"guitar\\": [\\"musical instrument\\"], \\"piano\\": [\\"musical instrument\\"], \\"onion\\": [\\"vegetable\\"],\\n",\n    "    \\"apple\\": [\\"fruit\\"], \\"bread\\": [\\"food\\"], \\"bicycle\\": [\\"vehicle\\"],\\n",\n    "}\\n",\n    "FALLBACK_VERB_ISA = {\\"slice\\": [\\"cut\\"], \\"chop\\": [\\"cut\\"], \\"dice\\": [\\"cut\\"], \\"strum\\": [\\"play\\"], \\"gallop\\": [\\"ride\\"],\\n",\n    "                     \\"jog\\": [\\"run\\"], \\"sprint\\": [\\"run\\"], \\"stir\\": [\\"mix\\"], \\"whisk\\": [\\"mix\\"], \\"fry\\": [\\"cook\\"],\\n",\n    "                     \\"bake\\": [\\"cook\\"], \\"boil\\": [\\"cook\\"], \\"grill\\": [\\"cook\\"], \\"sip\\": [\\"drink\\"], \\"munch\\": [\\"eat\\"]}\\n",\n    "\\n",\n    "# --- Lọc WordNet (notebook gốc): chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC ---\\n",\n    "VISUAL_LEX = {\\"noun.person\\", \\"noun.animal\\", \\"noun.artifact\\", \\"noun.object\\", \\"noun.food\\",\\n",\n    "              \\"noun.plant\\", \\"noun.substance\\", \\"noun.body\\", \\"noun.location\\", \\"noun.group\\",\\n",\n    "              \\"noun.phenomenon\\"}\\n",\n    "SPORT_ROOTS = {\\"sport.n.01\\", \\"game.n.01\\"}\\n",\n    "GENERIC_CONCEPTS = {\\"entity\\", \\"physical entity\\", \\"abstraction\\", \\"object\\", \\"whole\\", \\"thing\\",\\n",\n    "                    \\"matter\\", \\"artifact\\", \\"instrumentality\\", \\"organism\\", \\"living thing\\",\\n",\n    "                    \\"causal agent\\", \\"location\\", \\"region\\", \\"group\\", \\"act\\", \\"event\\",\\n",\n    "                    \\"unit\\", \\"part\\", \\"piece\\", \\"material\\", \\"substance\\", \\"device\\", \\"commodity\\",\\n",\n    "                    \\"consumer goods\\", \\"covering\\", \\"creation\\", \\"structure\\", \\"area\\"}\\n",\n    "BASIC_CATEGORIES = {\\"person\\", \\"animal\\", \\"dog\\", \\"bird\\", \\"vehicle\\", \\"boat\\", \\"aircraft\\",\\n",\n    "                    \\"clothing\\", \\"footwear\\", \\"headdress\\", \\"food\\", \\"plant\\", \\"tree\\", \\"flower\\",\\n",\n    "                    \\"furniture\\", \\"building\\", \\"toy\\", \\"sport\\", \\"ball\\", \\"musical instrument\\",\\n",\n    "                    \\"body of water\\", \\"geological formation\\", \\"precipitation\\", \\"child\\",\\n",\n    "                    \\"game equipment\\", \\"sports equipment\\", \\"vegetable\\", \\"fruit\\"}\\n",\n    "GENERIC_VERBS = {\\"act\\", \\"move\\", \\"change\\", \\"make\\", \\"be\\", \\"have\\", \\"do\\", \\"get\\", \\"go\\", \\"cause\\", \\"interact\\",\\n",\n    "                 \\"transfer\\", \\"use\\", \\"take\\", \\"give\\", \\"put\\", \\"perform\\", \\"modify\\", \\"remove\\", \\"create\\", \\"travel\\",\\n",\n    "                 \\"come\\", \\"displace\\", \\"connect\\", \\"communicate\\", \\"express\\", \\"refer\\", \\"think\\", \\"consume\\"}\\n",\n    "\\n",\n    "_WN = None\\n",\n    "\\n",\n    "_syn_key = lambda x: x.name()\\n",\n    "_path_key = lambda path: [x.name() for x in path]      # NLTK lưu con trỏ WordNet trong set ⇒ thứ tự phụ thuộc PYTHONHASHSEED\\n",\n    "\\n",\n    "\\n",\n    "def _wordnet():\\n",\n    "    \\"\\"\\"Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA).\\"\\"\\"\\n",\n    "    global _WN\\n",\n    "    if _WN is None:\\n",\n    "        try:\\n",\n    "            from nltk.corpus import wordnet as wn\\n",\n    "            wn.synsets(\\"dog\\"); _WN = wn\\n",\n    "        except Exception:\\n",\n    "            _WN = False\\n",\n    "    return _WN or None\\n",\n    "\\n",\n    "\\n",\n    "def _main_noun_synset(w):\\n",\n    "    wn = _wordnet()\\n",\n    "    if wn is None:\\n",\n    "        return None\\n",\n    "    syns = wn.synsets(w.replace(\\" \\", \\"_\\"), pos=\\"n\\")\\n",\n    "    return syns[0] if syns else None\\n",\n    "\\n",\n    "\\n",\n    "_NOUN_CACHE, _VERB_CACHE = {}, {}\\n",\n    "\\n",\n    "\\n",\n    "def is_visual_noun(w):\\n",\n    "    if w in _NOUN_CACHE:\\n",\n    "        return _NOUN_CACHE[w]\\n",\n    "    wn = _wordnet()\\n",\n    "    if wn is None:\\n",\n    "        _NOUN_CACHE[w] = True\\n",\n    "        return True\\n",\n    "    s, ok = _main_noun_synset(w), False\\n",\n    "    if s is not None:\\n",\n    "        lex = s.lexname()\\n",\n    "        if lex in VISUAL_LEX:\\n",\n    "            ok = True\\n",\n    "        elif lex == \\"noun.act\\":\\n",\n    "            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}\\n",\n    "            ok = bool(anc & SPORT_ROOTS)\\n",\n    "        if ok:\\n",\n    "            key = w.replace(\\" \\", \\"_\\")\\n",\n    "            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)\\n",\n    "                                  for l in x.lemmas() if l.name().lower() == key)\\n",\n    "            ok = cnt(\\"n\\") >= cnt(\\"v\\")\\n",\n    "    _NOUN_CACHE[w] = ok\\n",\n    "    return ok\\n",\n    "\\n",\n    "\\n",\n    "def is_action_verb(v):\\n",\n    "    if v not in _VERB_CACHE:\\n",\n    "        wn = _wordnet()\\n",\n    "        _VERB_CACHE[v] = True if wn is None else bool(wn.synsets(v, pos=\\"v\\"))\\n",\n    "    return _VERB_CACHE[v]\\n",\n    "\\n",\n    "\\n",\n    "def get_wordnet_hypernyms(words):\\n",\n    "    \\"\\"\\"IsA danh từ (notebook gốc): cha + ông (đủ sâu, không quá chung) + tổ tiên thuộc BASIC_CATEGORIES.\\"\\"\\"\\n",\n    "    if not CFG.use_wordnet:\\n",\n    "        return {}\\n",\n    "    wn = _wordnet()\\n",\n    "    if wn is None:\\n",\n    "        return {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}\\n",\n    "    out = {}\\n",\n    "    for w in words:\\n",\n    "        s = _main_noun_synset(w)\\n",\n    "        if s is None:\\n",\n    "            continue\\n",\n    "        hy = []\\n",\n    "        for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n",\n    "            for depth in range(len(path) - 2, -1, -1):\\n",\n    "                h = path[depth]\\n",\n    "                name = h.lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n",\n    "                near = depth >= len(path) - 3\\n",\n    "                if name == w or name in GENERIC_CONCEPTS or name in hy:\\n",\n    "                    continue\\n",\n    "                if (near and depth >= CFG.isa_min_depth and\\n",\n    "                        (h.lexname() in VISUAL_LEX or h.lexname() == \\"noun.Tops\\")) or name in BASIC_CATEGORIES:\\n",\n    "                    hy.append(name)\\n",\n    "        if hy:\\n",\n    "            out[w] = hy[:5]\\n",\n    "    for w in words:\\n",\n    "        if w not in out and w in FALLBACK_ISA:\\n",\n    "            out[w] = FALLBACK_ISA[w]\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def get_verb_hypernyms(verbs):\\n",\n    "    \\"\\"\\"IsA động từ: cha/ông của nghĩa động từ phổ biến nhất (slice → cut, gallop → ride), bỏ động từ quá chung.\\"\\"\\"\\n",\n    "    if not CFG.use_wordnet:\\n",\n    "        return {}\\n",\n    "    wn, out = _wordnet(), {}\\n",\n    "    for v in verbs:\\n",\n    "        hy = []\\n",\n    "        syns = wn.synsets(v, pos=\\"v\\") if wn is not None else []\\n",\n    "        if syns:\\n",\n    "            for path in sorted(syns[0].hypernym_paths(), key=_path_key)[:2]:\\n",\n    "                for depth in (len(path) - 2, len(path) - 3):\\n",\n    "                    if depth < 0:\\n",\n    "                        continue\\n",\n    "                    name = path[depth].lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n",\n    "                    if name != v and name not in GENERIC_VERBS and \\" \\" not in name and name not in hy:\\n",\n    "                        hy.append(name)\\n",\n    "        for h in FALLBACK_VERB_ISA.get(v, []):\\n",\n    "            if h != v and h not in hy:\\n",\n    "                hy.append(h)\\n",\n    "        if hy:\\n",\n    "            out[v] = hy[:3]\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def get_hypernyms(classes, types):\\n",\n    "    nouns = [c for c in classes if types.get(c, \\"noun\\") == \\"noun\\"]\\n",\n    "    verbs = [c for c in classes if types.get(c) == \\"verb\\"]\\n",\n    "    out = get_wordnet_hypernyms(nouns)\\n",\n    "    out.update(get_verb_hypernyms(verbs))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "ENV_INFO.update(nltk_status=NLTK_STATUS, wordnet=_wordnet() is not None, pos_tagger=_TAGGER is not None)\\n",\n    "print(\\"NLTK:\\", NLTK_STATUS, \\"| WordNet:\\", _wordnet() is not None, \\"| POS tagger:\\", _TAGGER is not None)"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c021",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "STOP = set(\\"a an the of on in at to with and or is are was were this that these those \\"\\n",\n    "           \\"there here it its his her their some any for from by as be being been very \\"\\n",\n    "           \\"photo image picture showing shows scene view near next video clip\\".split())\\n",\n    "NOT_OBJECTS = set(\\"front background foreground top bottom side end corner air group area region place space \\"\\n",\n    "                  \\"picture photo image one other something someone somebody anything thing stuff view middle edge \\"\\n",\n    "                  \\"part way lot pair couple bunch number while each both several many few kind type video clip \\"\\n",\n    "                  \\"footage scene time minute second\\".split())\\n",\n    "_SKIP_P = set(\\"is are was were be been being has have had while as that who which and or\\".split())\\n",\n    "AUX_VERBS = set(\\"be have do can will would shall should may might must seem appear become begin start try \\"\\n",\n    "                \\"keep continue let get go come\\".split())\\n",\n    "\\n",\n    "\\n",\n    "def _norm(s):\\n",\n    "    s = str(s).lower().strip()\\n",\n    "    s = re.sub(r\\"[^a-z0-9 ]+\\", \\" \\", s)\\n",\n    "    return re.sub(r\\"\\\\s+\\", \\" \\", s).strip()\\n",\n    "\\n",\n    "\\n",\n    "def _fallback_pos(toks):\\n",\n    "    wn, out = _wordnet(), []\\n",\n    "    for t in toks:\\n",\n    "        if wn is None:\\n",\n    "            out.append((t, \\"NN\\" if len(t) > 2 and t not in STOP else \\"DT\\"))\\n",\n    "        elif t.endswith(\\"ing\\") and wn.synsets(lemma(t, \\"v\\"), pos=\\"v\\"):\\n",\n    "            out.append((t, \\"VBG\\"))\\n",\n    "        elif wn.synsets(t, pos=\\"n\\"):\\n",\n    "            out.append((t, \\"NN\\"))\\n",\n    "        elif wn.synsets(lemma(t, \\"v\\"), pos=\\"v\\"):\\n",\n    "            out.append((t, \\"VB\\"))\\n",\n    "        else:\\n",\n    "            out.append((t, \\"DT\\"))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "_PARSE_CACHE = {}\\n",\n    "\\n",\n    "\\n",\n    "def parse_caption(cap):\\n",\n    "    \\"\\"\\"(danh từ nhìn thấy được, {động từ: dạng V-ing}, bộ ba quan hệ) của MỘT caption — heuristic POS.\\n",\n    "    Bộ ba gồm (danh từ, vị từ, danh từ) như notebook gốc + (chủ thể, agent_of, động từ), (đồ vật, object_of,\\n",\n    "    động từ) và chiều ngược (done_by, acts_on). Nên đọc thử ~30 mẫu để báo cáo độ chính xác.\\"\\"\\"\\n",\n    "    if cap in _PARSE_CACHE:\\n",\n    "        return _PARSE_CACHE[cap]\\n",\n    "    toks = re.findall(r\\"[a-z]+\\", str(cap).lower())\\n",\n    "    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n",\n    "    nouns, verbs, triples = set(), {}, set()\\n",\n    "    subj, last_n, preds, cur_v = None, None, [], None\\n",\n    "    for w, t in tags:\\n",\n    "        if t in (\\"NN\\", \\"NNS\\") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:\\n",\n    "            n = lemma(w, \\"n\\")\\n",\n    "            if n in NOT_OBJECTS or not is_visual_noun(n):\\n",\n    "                continue\\n",\n    "            nouns.add(n)\\n",\n    "            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:\\n",\n    "                triples.add((last_n, \\" \\".join(preds), n))\\n",\n    "            if cur_v is not None and n != subj:\\n",\n    "                triples.add((n, \\"object_of\\", cur_v)); triples.add((cur_v, \\"acts_on\\", n))\\n",\n    "                cur_v = None\\n",\n    "            subj = subj or n\\n",\n    "            last_n, preds = n, []\\n",\n    "        elif t.startswith(\\"VB\\"):\\n",\n    "            v = lemma(w, \\"v\\")\\n",\n    "            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):\\n",\n    "                continue\\n",\n    "            if t == \\"VBG\\":\\n",\n    "                verbs[v] = w\\n",\n    "            else:\\n",\n    "                verbs.setdefault(v, None)\\n",\n    "            if subj is not None and v != subj:\\n",\n    "                triples.add((subj, \\"agent_of\\", v)); triples.add((v, \\"done_by\\", subj))\\n",\n    "            if last_n is not None:\\n",\n    "                preds.append(v)\\n",\n    "            cur_v = v\\n",\n    "        elif t in (\\"IN\\", \\"TO\\", \\"RP\\") and w not in _SKIP_P and last_n is not None:\\n",\n    "            preds.append(w)\\n",\n    "    res = (frozenset(nouns), dict(verbs), frozenset(triples))\\n",\n    "    _PARSE_CACHE[cap] = res\\n",\n    "    return res\\n",\n    "\\n",\n    "\\n",\n    "def naive_gerund(v):\\n",\n    "    if v.endswith(\\"ie\\"):\\n",\n    "        return v[:-2] + \\"ying\\"\\n",\n    "    if v.endswith(\\"e\\") and not v.endswith((\\"ee\\", \\"ye\\", \\"oe\\")):\\n",\n    "        return v[:-1] + \\"ing\\"\\n",\n    "    return v + \\"ing\\"\\n",\n    "\\n",\n    "\\n",\n    "def build_vocab(store, split):\\n",\n    "    \\"\\"\\"Từ vựng khái niệm từ caption TRAIN (mục đích \'vocab\' — CaptionStore chặn mọi split khác).\\"\\"\\"\\n",\n    "    tr = split[\\"train\\"]\\n",\n    "    caps = store.by_video(tr, \\"vocab\\")\\n",\n    "    df, pos_cnt, ger = Counter(), defaultdict(Counter), defaultdict(Counter)\\n",\n    "    for v in tr:\\n",\n    "        cnt = Counter()\\n",\n    "        for c in caps[v]:\\n",\n    "            nouns, verbs, _ = parse_caption(c)\\n",\n    "            for n in nouns:\\n",\n    "                cnt[n] += 1; pos_cnt[n][\\"noun\\"] += 1\\n",\n    "            for vb, surf in verbs.items():\\n",\n    "                cnt[vb] += 1; pos_cnt[vb][\\"verb\\"] += 1\\n",\n    "                if surf:\\n",\n    "                    ger[vb][surf] += 1\\n",\n    "        df.update(t for t, k in cnt.items() if k >= CFG.vocab_min_caps)\\n",\n    "    terms = sorted(t for t, k in df.items() if k >= CFG.min_term_df)\\n",\n    "    types = {t: (\\"verb\\" if pos_cnt[t][\\"verb\\"] > pos_cnt[t][\\"noun\\"] else \\"noun\\") for t in terms}\\n",\n    "    gerund = {t: (ger[t].most_common(1)[0][0] if ger[t] else naive_gerund(t)) for t in terms if types[t] == \\"verb\\"}\\n",\n    "    return dict(terms=terms, types=types, gerund=gerund, df={t: int(df[t]) for t in terms},\\n",\n    "                built_from=\\"train\\", n_train_videos=len(tr), sha1=sha1_of([terms, types]))\\n",\n    "\\n",\n    "\\n",\n    "def build_relations(store, split, vocab):\\n",\n    "    \\"\\"\\"Bộ ba quan hệ của từng video TRAIN (mục đích \'relations\'). Video val/test KHÔNG có quan hệ từ caption.\\"\\"\\"\\n",\n    "    vset = set(vocab[\\"terms\\"])\\n",\n    "    caps = store.by_video(split[\\"train\\"], \\"relations\\")\\n",\n    "    rel = {}\\n",\n    "    for v in split[\\"train\\"]:\\n",\n    "        trip = set()\\n",\n    "        for c in caps[v]:\\n",\n    "            trip |= {t for t in parse_caption(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n",\n    "        rel[v] = sorted(trip)\\n",\n    "    return rel\\n",\n    "\\n",\n    "\\n",\n    "def caption_truth(store, vids, vocab, purpose):\\n",\n    "    \\"\\"\\"Nhãn tham chiếu: từ của từ vựng xuất hiện ≥ truth_min_caps caption của video (chỉ để CHẤM tagger).\\"\\"\\"\\n",\n    "    vset, caps, out = set(vocab[\\"terms\\"]), store.by_video(vids, purpose), {}\\n",\n    "    for v in vids:\\n",\n    "        cnt = Counter()\\n",\n    "        for c in caps[v]:\\n",\n    "            n, vb, _ = parse_caption(c)\\n",\n    "            cnt.update(set(n) | set(vb))\\n",\n    "        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "_TERM_FEAT = {}\\n",\n    "\\n",\n    "\\n",\n    "def term_text_features(vocab):\\n",\n    "    \\"\\"\\"CLIP-text của prompt cho từng từ (ensemble 3 prompt; danh từ / động từ dạng V-ing).\\"\\"\\"\\n",\n    "    key = vocab[\\"sha1\\"] + sha1_of(vocab[\\"gerund\\"])\\n",\n    "    if key not in _TERM_FEAT:\\n",\n    "        terms, T = vocab[\\"terms\\"], None\\n",\n    "        for typ, prompts in ((\\"noun\\", CFG.noun_prompts), (\\"verb\\", CFG.verb_prompts)):\\n",\n    "            idx = [i for i, t in enumerate(terms) if vocab[\\"types\\"][t] == typ]\\n",\n    "            if not idx:\\n",\n    "                continue\\n",\n    "            words = [terms[i] if typ == \\"noun\\" else vocab[\\"gerund\\"][terms[i]] for i in idx]\\n",\n    "            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)\\n",\n    "            if T is None:\\n",\n    "                T = np.zeros((len(terms), E.shape[1]), np.float32)\\n",\n    "            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n",\n    "        _TERM_FEAT[key] = T\\n",\n    "    return _TERM_FEAT[key]\\n",\n    "\\n",\n    "\\n",\n    "def _prf(pred, truth):\\n",\n    "    tp = sum(len(p & t) for p, t in zip(pred, truth))\\n",\n    "    P = tp / max(sum(map(len, pred)), 1)\\n",\n    "    R = tp / max(sum(map(len, truth)), 1)\\n",\n    "    return P, R, 2 * P * R / max(P + R, 1e-9)\\n",\n    "\\n",\n    "\\n",\n    "def run_tagger(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):\\n",\n    "    \\"\\"\\"Gắn nhãn MỌI video bằng CLIP zero-shot trên frame với từ vựng TRAIN — không đọc caption của video được gắn.\\n",\n    "    select=None ⇒ chọn (pool, norm, k_noun, k_verb) theo F1 trên VAL. Trả về (tags theo vid, lựa chọn, lưới).\\"\\"\\"\\n",\n    "    terms = vocab[\\"terms\\"]\\n",\n    "    T = term_text_features(vocab)\\n",\n    "    tr = np.array(split[\\"train\\"])\\n",\n    "    N, D = vid_feat.shape\\n",\n    "    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n",\n    "    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n",\n    "    S_raw = {\\"video\\": vid_feat @ T.T, \\"frame_max\\": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}\\n",\n    "    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n",\n    "\\n",\n    "    def normed(S, mode):\\n",\n    "        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên video TRAIN\\n",\n    "        return S if mode == \\"raw\\" else ((S - mu) / sd if mode == \\"z\\" else S / S[tr].std() + (S - mu) / sd)\\n",\n    "\\n",\n    "    def topk(Sc, rows, kn, kv):\\n",\n    "        out = [set() for _ in rows]\\n",\n    "        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n",\n    "            c = cols[typ]\\n",\n    "            if k <= 0 or len(c) == 0:\\n",\n    "                continue\\n",\n    "            k = min(k, len(c))\\n",\n    "            top = np.argpartition(-Sc[np.ix_(rows, c)], k - 1, axis=1)[:, :k]\\n",\n    "            for j, row in enumerate(top):\\n",\n    "                out[j].update(terms[c[i]] for i in row)\\n",\n    "        return out\\n",\n    "\\n",\n    "    grid = []\\n",\n    "    if select is None:\\n",\n    "        va = np.array(split[\\"val\\"])\\n",\n    "        truth = caption_truth(store, va, vocab, \\"tagger_select\\")\\n",\n    "        tv = [truth[v] for v in va]\\n",\n    "        for pool in CFG.tag_pools:\\n",\n    "            for mode in CFG.tag_norms:\\n",\n    "                Sc = normed(S_raw[pool], mode)\\n",\n    "                for kn in CFG.tag_k_noun_grid:\\n",\n    "                    for kv in CFG.tag_k_verb_grid:\\n",\n    "                        P, R, F1 = _prf(topk(Sc, va, kn, kv), tv)\\n",\n    "                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))\\n",\n    "        best = max(grid, key=lambda g: (round(g[\\"val_F1\\"], 9), -(g[\\"k_noun\\"] + g[\\"k_verb\\"])))\\n",\n    "        select = {k: best[k] for k in (\\"pool\\", \\"norm\\", \\"k_noun\\", \\"k_verb\\", \\"val_P\\", \\"val_R\\", \\"val_F1\\")}\\n",\n    "    Sc = normed(S_raw[select[\\"pool\\"]], select[\\"norm\\"])\\n",\n    "    tags = topk(Sc, np.arange(N), select[\\"k_noun\\"], select[\\"k_verb\\"])\\n",\n    "    return {v: sorted(t) for v, t in enumerate(tags)}, select, grid\\n",\n    "\\n",\n    "\\n",\n    "def tagger_report(tags, vids, vocab, purpose):\\n",\n    "    truth = caption_truth(CAPS, vids, vocab, purpose)\\n",\n    "    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])\\n",\n    "    return dict(P=P, R=R, F1=F1, n_videos=len(vids))"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c022",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "REL_CATS = (\\"agent_of\\", \\"object_of\\", \\"svo\\", \\"prep\\")\\n",\n    "_PARSE2_CACHE = {}\\n",\n    "\\n",\n    "\\n",\n    "def ckey(lemma_, pos):\\n",\n    "    \\"\\"\\"Khoá node lớp: \'dog#n\', \'play#v\'.\\"\\"\\"\\n",\n    "    return f\\"{lemma_}#{pos}\\"\\n",\n    "\\n",\n    "\\n",\n    "def parse_caption_v2(cap):\\n",\n    "    \\"\\"\\"Như parse_caption (v1) nhưng (1) khoá theo (lemma, POS), (2) mỗi quan hệ giữ LOẠI và VỊ TỪ.\\n",\n    "    Trả về (danh từ, {động từ: dạng V-ing}, {(nguồn, loại, đích, vị từ)}).\\"\\"\\"\\n",\n    "    if cap in _PARSE2_CACHE:\\n",\n    "        return _PARSE2_CACHE[cap]\\n",\n    "    toks = re.findall(r\\"[a-z]+\\", str(cap).lower())\\n",\n    "    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n",\n    "    nouns, verbs, triples = set(), {}, set()\\n",\n    "    subj, last_n, preds, pred_has_verb, cur_v = None, None, [], False, None\\n",\n    "    for w, t in tags:\\n",\n    "        if t in (\\"NN\\", \\"NNS\\") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:\\n",\n    "            n = lemma(w, \\"n\\")\\n",\n    "            if n in NOT_OBJECTS or not is_visual_noun(n):\\n",\n    "                continue\\n",\n    "            nouns.add(n)\\n",\n    "            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:\\n",\n    "                triples.add((ckey(last_n, \\"n\\"), \\"svo\\" if pred_has_verb else \\"prep\\", ckey(n, \\"n\\"), \\" \\".join(preds)))\\n",\n    "            if cur_v is not None and n != subj:\\n",\n    "                triples.add((ckey(n, \\"n\\"), \\"object_of\\", ckey(cur_v, \\"v\\"), cur_v))\\n",\n    "                cur_v = None\\n",\n    "            subj = subj or n\\n",\n    "            last_n, preds, pred_has_verb = n, [], False\\n",\n    "        elif t.startswith(\\"VB\\"):\\n",\n    "            v = lemma(w, \\"v\\")\\n",\n    "            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):\\n",\n    "                continue\\n",\n    "            if t == \\"VBG\\":\\n",\n    "                verbs[v] = w\\n",\n    "            else:\\n",\n    "                verbs.setdefault(v, None)\\n",\n    "            if subj is not None:\\n",\n    "                triples.add((ckey(subj, \\"n\\"), \\"agent_of\\", ckey(v, \\"v\\"), v))\\n",\n    "            if last_n is not None:\\n",\n    "                preds.append(v); pred_has_verb = True\\n",\n    "            cur_v = v\\n",\n    "        elif t in (\\"IN\\", \\"TO\\", \\"RP\\") and w not in _SKIP_P and last_n is not None:\\n",\n    "            preds.append(w)\\n",\n    "    res = (frozenset(nouns), dict(verbs), frozenset(triples))\\n",\n    "    _PARSE2_CACHE[cap] = res\\n",\n    "    return res\\n",\n    "\\n",\n    "\\n",\n    "def build_vocab_v2(store, split):\\n",\n    "    \\"\\"\\"Từ vựng khoá (lemma, POS) từ caption TRAIN: một lemma có thể là cả \'x#n\' và \'x#v\' nếu cả hai đạt ngưỡng.\\"\\"\\"\\n",\n    "    tr = split[\\"train\\"]\\n",\n    "    caps = store.by_video(tr, \\"vocab\\")\\n",\n    "    df, ger = Counter(), defaultdict(Counter)\\n",\n    "    for v in tr:\\n",\n    "        cnt = Counter()\\n",\n    "        for c in caps[v]:\\n",\n    "            nouns, verbs, _ = parse_caption_v2(c)\\n",\n    "            cnt.update(ckey(n, \\"n\\") for n in nouns)\\n",\n    "            for vb, surf in verbs.items():\\n",\n    "                cnt[ckey(vb, \\"v\\")] += 1\\n",\n    "                if surf:\\n",\n    "                    ger[vb][surf] += 1\\n",\n    "        df.update(k for k, m in cnt.items() if m >= CFG.vocab_min_caps)\\n",\n    "    terms = sorted(k for k, m in df.items() if m >= CFG.min_term_df)\\n",\n    "    lem = {k: k.rsplit(\\"#\\", 1)[0] for k in terms}\\n",\n    "    types = {k: (\\"verb\\" if k.endswith(\\"#v\\") else \\"noun\\") for k in terms}\\n",\n    "    gerund = {k: (ger[lem[k]].most_common(1)[0][0] if ger[lem[k]] else naive_gerund(lem[k])) for k in terms if types[k] == \\"verb\\"}\\n",\n    "    both = sorted({lem[k] for k in terms if types[k] == \\"noun\\"} & {lem[k] for k in terms if types[k] == \\"verb\\"})\\n",\n    "    return dict(terms=terms, types=types, lemma=lem, gerund=gerund, df={k: int(df[k]) for k in terms},\\n",\n    "                lemmas_with_two_pos=both, built_from=\\"train\\", n_train_videos=len(tr), sha1=sha1_of([terms, types]))\\n",\n    "\\n",\n    "\\n",\n    "def build_relations_v2(store, split, vocab):\\n",\n    "    \\"\\"\\"{vid TRAIN: [(nguồn, loại, đích, vị từ)]} — chỉ video TRAIN (mục đích \'relations\').\\"\\"\\"\\n",\n    "    vset = set(vocab[\\"terms\\"])\\n",\n    "    caps = store.by_video(split[\\"train\\"], \\"relations\\")\\n",\n    "    rel = {}\\n",\n    "    for v in split[\\"train\\"]:\\n",\n    "        trip = set()\\n",\n    "        for c in caps[v]:\\n",\n    "            trip |= {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n",\n    "        rel[v] = sorted(trip)\\n",\n    "    return rel\\n",\n    "\\n",\n    "\\n",\n    "def caption_truth_v2(store, vids, vocab, purpose):\\n",\n    "    vset, caps, out = set(vocab[\\"terms\\"]), store.by_video(vids, purpose), {}\\n",\n    "    for v in vids:\\n",\n    "        cnt = Counter()\\n",\n    "        for c in caps[v]:\\n",\n    "            n, vb, _ = parse_caption_v2(c)\\n",\n    "            cnt.update({ckey(x, \\"n\\") for x in n} | {ckey(x, \\"v\\") for x in vb})\\n",\n    "        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "_TERM_FEAT2 = {}\\n",\n    "\\n",\n    "\\n",\n    "def term_text_features_v2(vocab):\\n",\n    "    \\"\\"\\"CLIP-text (ensemble 3 prompt) cho từng khoá: danh từ dùng lemma, động từ dùng dạng V-ing.\\"\\"\\"\\n",\n    "    key = vocab[\\"sha1\\"] + sha1_of(vocab[\\"gerund\\"])\\n",\n    "    if key not in _TERM_FEAT2:\\n",\n    "        terms, T = vocab[\\"terms\\"], None\\n",\n    "        for typ, prompts in ((\\"noun\\", CFG.noun_prompts), (\\"verb\\", CFG.verb_prompts)):\\n",\n    "            idx = [i for i, t in enumerate(terms) if vocab[\\"types\\"][t] == typ]\\n",\n    "            if not idx:\\n",\n    "                continue\\n",\n    "            words = [vocab[\\"lemma\\"][terms[i]] if typ == \\"noun\\" else vocab[\\"gerund\\"][terms[i]] for i in idx]\\n",\n    "            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)\\n",\n    "            if T is None:\\n",\n    "                T = np.zeros((len(terms), E.shape[1]), np.float32)\\n",\n    "            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n",\n    "        _TERM_FEAT2[key] = T\\n",\n    "    return _TERM_FEAT2[key]\\n",\n    "\\n",\n    "\\n",\n    "def run_tagger_v2(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):\\n",\n    "    \\"\\"\\"Như run_tagger (v1) với khoá (lemma, POS). Trả thêm điểm đã chuẩn hoá của từng nhãn được gắn\\n",\n    "    (dùng làm trọng số contains ở S3). Thống kê chuẩn hoá chỉ trên video TRAIN; lựa chọn theo F1 trên VAL.\\"\\"\\"\\n",\n    "    terms = vocab[\\"terms\\"]\\n",\n    "    T = term_text_features_v2(vocab)\\n",\n    "    tr = np.array(split[\\"train\\"])\\n",\n    "    N, D = vid_feat.shape\\n",\n    "    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n",\n    "    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n",\n    "    S_raw = {\\"video\\": vid_feat @ T.T, \\"frame_max\\": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}\\n",\n    "    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n",\n    "\\n",\n    "    def normed(S, mode):\\n",\n    "        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6\\n",\n    "        return S if mode == \\"raw\\" else ((S - mu) / sd if mode == \\"z\\" else S / S[tr].std() + (S - mu) / sd)\\n",\n    "\\n",\n    "    def topk(Sc, rows, kn, kv):\\n",\n    "        out = [dict() for _ in rows]\\n",\n    "        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n",\n    "            c = cols[typ]\\n",\n    "            if k <= 0 or len(c) == 0:\\n",\n    "                continue\\n",\n    "            k = min(k, len(c))\\n",\n    "            sub = Sc[np.ix_(rows, c)]\\n",\n    "            top = np.argpartition(-sub, k - 1, axis=1)[:, :k]\\n",\n    "            for j, row in enumerate(top):\\n",\n    "                out[j].update({terms[c[i]]: float(sub[j, i]) for i in row})\\n",\n    "        return out\\n",\n    "\\n",\n    "    grid = []\\n",\n    "    if select is None:\\n",\n    "        va = np.array(split[\\"val\\"])\\n",\n    "        truth = caption_truth_v2(store, va, vocab, \\"tagger_select\\")\\n",\n    "        tv = [truth[v] for v in va]\\n",\n    "        for pool in CFG.tag_pools:\\n",\n    "            for mode in CFG.tag_norms:\\n",\n    "                Sc = normed(S_raw[pool], mode)\\n",\n    "                for kn in CFG.tag_k_noun_grid:\\n",\n    "                    for kv in CFG.tag_k_verb_grid:\\n",\n    "                        P, R, F1 = _prf([set(d) for d in topk(Sc, va, kn, kv)], tv)\\n",\n    "                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))\\n",\n    "        best = max(grid, key=lambda g: (round(g[\\"val_F1\\"], 9), -(g[\\"k_noun\\"] + g[\\"k_verb\\"])))\\n",\n    "        select = {k: best[k] for k in (\\"pool\\", \\"norm\\", \\"k_noun\\", \\"k_verb\\", \\"val_P\\", \\"val_R\\", \\"val_F1\\")}\\n",\n    "    Sc = normed(S_raw[select[\\"pool\\"]], select[\\"norm\\"])\\n",\n    "    tagged = topk(Sc, np.arange(N), select[\\"k_noun\\"], select[\\"k_verb\\"])\\n",\n    "    tags = {v: sorted(d) for v, d in enumerate(tagged)}\\n",\n    "    scores = {v: d for v, d in enumerate(tagged)}\\n",\n    "    return tags, scores, select, grid\\n",\n    "\\n",\n    "\\n",\n    "def tagger_report_v2(tags, vids, vocab, purpose):\\n",\n    "    truth = caption_truth_v2(CAPS, vids, vocab, purpose)\\n",\n    "    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])\\n",\n    "    return dict(P=P, R=R, F1=F1, n_videos=len(vids))"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c023",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# --- Chất lượng quan hệ (Nhiệm vụ B): mẫu cho NGƯỜI chấm + đọc nhãn đã chấm. Chỉ caption TRAIN. ---\\n",\n    "# Mẫu KHÔNG phụ thuộc CLIP/tagger (chỉ parser + từ vựng train) ⇒ giống nhau giữa máy local và Kaggle.\\n",\n    "# Đơn vị mẫu = MỘT quan hệ (họ, nguồn, vị từ, đích) đủ điều kiện thành cạnh (support ≥ scene_min_count video TRAIN),\\n",\n    "# kèm MỘT caption TRAIN chọn ngẫu nhiên có chứa nó; người chấm trả lời: caption này có diễn đạt đúng quan hệ đó không?\\n",\n    "CaptionStore.ALLOWED[\\"annotation\\"] = {\\"train\\"}\\n",\n    "ANNOT_LABEL_COLS = (\\"annotator_1\\", \\"annotator_2\\", \\"adjudicated\\")\\n",\n    "\\n",\n    "\\n",\n    "def relation_instances(store, split, vocab):\\n",\n    "    \\"\\"\\"[(vid, caption, (nguồn, họ, đích, vị từ))] trên caption TRAIN; support = số video TRAIN có quan hệ (đếm 1 lần / video);\\n",\n    "    pred_support[(họ, vị từ)] = số video TRAIN có ít nhất một quan hệ thuộc họ đó với vị từ đó.\\"\\"\\"\\n",\n    "    vset = set(vocab[\\"terms\\"])\\n",\n    "    caps = store.by_video(split[\\"train\\"], \\"annotation\\")\\n",\n    "    inst, sup, psup = [], Counter(), Counter()\\n",\n    "    for v in split[\\"train\\"]:\\n",\n    "        seen = set()\\n",\n    "        for c in caps[v]:\\n",\n    "            for t in sorted(parse_caption_v2(c)[2]):\\n",\n    "                if t[0] in vset and t[2] in vset and t[0] != t[2]:\\n",\n    "                    inst.append((v, c, t)); seen.add(t)\\n",\n    "        sup.update(seen)\\n",\n    "        psup.update({(t[1], t[3]) for t in seen})\\n",\n    "    return inst, sup, psup\\n",\n    "\\n",\n    "\\n",\n    "def support_stratum(s):\\n",\n    "    for lo, hi, name in CFG.annot_strata:\\n",\n    "        if lo <= s <= hi:\\n",\n    "            return name\\n",\n    "    return \\"below_threshold\\"\\n",\n    "\\n",\n    "\\n",\n    "def _display_relation(src, rel, dst, pred):\\n",\n    "    a, b = src.rsplit(\\"#\\", 1)[0], dst.rsplit(\\"#\\", 1)[0]\\n",\n    "    if rel == \\"agent_of\\":\\n",\n    "        return a, pred, \\"\\", f\\"\'{a}\' là CHỦ THỂ thực hiện hành động \'{pred}\'\\"\\n",\n    "    if rel == \\"object_of\\":\\n",\n    "        return \\"\\", pred, a, f\\"\'{a}\' là ĐỐI TƯỢNG chịu hành động \'{pred}\'\\"\\n",\n    "    return a, pred, b, f\\"{a} —{pred}→ {b}\\"\\n",\n    "\\n",\n    "\\n",\n    "def build_annotation_sample(store, split, vocab, records):\\n",\n    "    \\"\\"\\"Phân tầng: họ quan hệ × mức phổ biến (support); trong mỗi ô chọn luân phiên theo vị từ để đa dạng vị từ.\\"\\"\\"\\n",\n    "    inst, sup, psup = relation_instances(store, split, vocab)\\n",\n    "    by_trip = defaultdict(list)\\n",\n    "    for v, c, t in inst:\\n",\n    "        if sup[t] >= CFG.scene_min_count:\\n",\n    "            by_trip[t].append((v, c))\\n",\n    "    rng = np.random.RandomState(CFG.annot_seed)\\n",\n    "    rows, cells = [], []\\n",\n    "    for rel in REL_CATS:\\n",\n    "        for lo, hi, name in CFG.annot_strata:\\n",\n    "            trips = sorted(t for t in by_trip if t[1] == rel and lo <= sup[t] <= hi)\\n",\n    "            by_pred = defaultdict(list)\\n",\n    "            for t in trips:\\n",\n    "                by_pred[t[3]].append(t)\\n",\n    "            preds = sorted(by_pred)\\n",\n    "            rng.shuffle(preds)\\n",\n    "            for p in preds:\\n",\n    "                rng.shuffle(by_pred[p])\\n",\n    "            chosen = []\\n",\n    "            while len(chosen) < CFG.annot_n_per_cell and any(by_pred[p] for p in preds):\\n",\n    "                for p in preds:                              # luân phiên vị từ ⇒ không vị từ nào chiếm cả ô\\n",\n    "                    if by_pred[p] and len(chosen) < CFG.annot_n_per_cell:\\n",\n    "                        chosen.append(by_pred[p].pop())\\n",\n    "            dbl = set(rng.choice(len(chosen), min(CFG.annot_n_double_per_cell, len(chosen)), replace=False).tolist()) if chosen else set()\\n",\n    "            cells.append(dict(family=rel, stratum=name, support_lo=lo, support_hi=hi, n_relations_in_cell=len(trips),\\n",\n    "                              n_predicates_in_cell=len(preds), n_sampled=len(chosen)))\\n",\n    "            for j, t in enumerate(chosen):\\n",\n    "                v, c = by_trip[t][rng.randint(len(by_trip[t]))]\\n",\n    "                s_, p_, o_, txt = _display_relation(t[0], t[1], t[2], t[3])\\n",\n    "                vid = records[v][\\"video_id\\"]\\n",\n    "                rows.append(dict(sample_id=hashlib.sha1(\\"|\\".join([vid, c, t[1], t[0], t[3], t[2]]).encode()).hexdigest()[:12],\\n",\n    "                                 family=rel, caption=c, subject=s_, predicate=p_, object=o_, relation=txt,\\n",\n    "                                 double_annotate=int(j in dbl), annotator_1=\\"\\", annotator_2=\\"\\", adjudicated=\\"\\", note=\\"\\",\\n",\n    "                                 video_id=vid, src_node=t[0], dst_node=t[2], stratum=name, relation_support_videos=int(sup[t]),\\n",\n    "                                 predicate_support_videos=int(psup[(t[1], t[3])])))\\n",\n    "    df = pd.DataFrame(rows)\\n",\n    "    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)      # trộn thứ tự: người chấm không thấy khối theo ô\\n",\n    "    return df, pd.DataFrame(cells), sup, psup\\n",\n    "\\n",\n    "\\n",\n    "def _label(x):\\n",\n    "    x = str(x).strip().lower()\\n",\n    "    return 1 if x in (\\"1\\", \\"1.0\\", \\"đúng\\", \\"dung\\", \\"true\\", \\"yes\\", \\"y\\") else (0 if x in (\\"0\\", \\"0.0\\", \\"sai\\", \\"false\\", \\"no\\", \\"n\\") else None)\\n",\n    "\\n",\n    "\\n",\n    "def _read_table(p):\\n",\n    "    if p.suffix.lower() == \\".xlsx\\":\\n",\n    "        return pd.read_excel(p, dtype=str, keep_default_na=False, sheet_name=0)\\n",\n    "    for enc in (\\"utf-8-sig\\", \\"cp1252\\"):                # Excel có thể lưu \';\' hoặc mã hoá khác — nhãn chỉ là 1/0/?\\n",\n    "        try:\\n",\n    "            return pd.read_csv(p, dtype=str, keep_default_na=False, sep=None, engine=\\"python\\", encoding=enc)\\n",\n    "        except UnicodeDecodeError:\\n",\n    "            continue\\n",\n    "    raise ValueError(f\\"không đọc được {p}\\")\\n",\n    "\\n",\n    "\\n",\n    "def find_label_file():\\n",\n    "    \\"\\"\\"CFG.annot_file, hoặc relation_annotation*.(csv|xlsx) trong /kaggle/input có ít nhất một nhãn ở annotator_1.\\"\\"\\"\\n",\n    "    cands = [Path(CFG.annot_file)] if CFG.annot_file else []\\n",\n    "    if RUN_ON_KAGGLE:\\n",\n    "        cands += sorted(p for p in Path(\\"/kaggle/input\\").rglob(\\"relation_annotation*\\") if p.suffix.lower() in (\\".csv\\", \\".xlsx\\"))\\n",\n    "    for p in cands:\\n",\n    "        try:\\n",\n    "            df = _read_table(p)\\n",\n    "            if \\"annotator_1\\" in df and df[\\"annotator_1\\"].map(_label).notna().any():\\n",\n    "                return p, df\\n",\n    "        except Exception as e:\\n",\n    "            print(f\\"⚠ Không đọc được {p}: {type(e).__name__}: {e}\\")\\n",\n    "    return None, None\\n",\n    "\\n",\n    "\\n",\n    "def wilson(k, n, z=1.96):\\n",\n    "    if n == 0:\\n",\n    "        return float(\\"nan\\"), float(\\"nan\\"), float(\\"nan\\")\\n",\n    "    p = k / n\\n",\n    "    d = 1 + z * z / n\\n",\n    "    c = (p + z * z / (2 * n)) / d\\n",\n    "    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d\\n",\n    "    return p, c - h, c + h\\n",\n    "\\n",\n    "\\n",\n    "def cohen_kappa(a, b):\\n",\n    "    a, b = np.asarray(a), np.asarray(b)\\n",\n    "    if len(a) == 0:\\n",\n    "        return float(\\"nan\\")\\n",\n    "    po = float((a == b).mean())\\n",\n    "    pe = float(a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean()))\\n",\n    "    return (po - pe) / (1 - pe) if pe < 1 else float(\\"nan\\")\\n",\n    "\\n",\n    "\\n",\n    "def relation_quality(labels):\\n",\n    "    \\"\\"\\"Precision theo họ và theo ô (họ × mức phổ biến) từ nhãn NGƯỜI chấm. Nhãn cuối = adjudicated nếu có; nếu không:\\n",\n    "    annotator_1 khi chỉ có một người chấm hoặc hai người đồng ý; hai người bất đồng mà chưa phân xử ⇒ loại khỏi precision.\\"\\"\\"\\n",\n    "    L = labels.copy()\\n",\n    "    for c in ANNOT_LABEL_COLS:\\n",\n    "        L[c + \\"_v\\"] = L[c].map(_label) if c in L else None\\n",\n    "    final, status = [], []\\n",\n    "    for a1, a2, ad in zip(L[\\"annotator_1_v\\"], L[\\"annotator_2_v\\"], L[\\"adjudicated_v\\"]):\\n",\n    "        if not pd.isna(ad):\\n",\n    "            final.append(ad); status.append(\\"adjudicated\\")\\n",\n    "        elif pd.isna(a1):\\n",\n    "            final.append(None); status.append(\\"unlabeled_or_unclear\\")\\n",\n    "        elif pd.isna(a2):\\n",\n    "            final.append(a1); status.append(\\"single\\")\\n",\n    "        elif a1 == a2:\\n",\n    "            final.append(a1); status.append(\\"agree\\")\\n",\n    "        else:\\n",\n    "            final.append(None); status.append(\\"disagree_unresolved\\")\\n",\n    "    L[\\"final\\"], L[\\"label_status\\"] = final, status\\n",\n    "    both = L[L[\\"annotator_1_v\\"].notna() & L[\\"annotator_2_v\\"].notna()]\\n",\n    "    rows = []\\n",\n    "    for keys, sub in [((\\"ALL\\", \\"ALL\\"), L)] + [((f, \\"ALL\\"), g) for f, g in L.groupby(\\"family\\")] + \\\\\\n",\n    "            [((f, s), g) for (f, s), g in L.groupby([\\"family\\", \\"stratum\\"])]:\\n",\n    "        lab = sub[sub[\\"final\\"].notna()]\\n",\n    "        k, n = int(lab[\\"final\\"].sum()), len(lab)\\n",\n    "        p, lo, hi = wilson(k, n)\\n",\n    "        rows.append(dict(family=keys[0], stratum=keys[1], n_rows=len(sub), n_labeled=n, n_correct=k, precision=p, wilson_lo=lo,\\n",\n    "                         wilson_hi=hi, n_unclear_or_unlabeled=int((sub[\\"label_status\\"] == \\"unlabeled_or_unclear\\").sum()),\\n",\n    "                         n_disagree_unresolved=int((sub[\\"label_status\\"] == \\"disagree_unresolved\\").sum()),\\n",\n    "                         verified=bool(keys[1] != \\"ALL\\" and n > 0 and lo >= CFG.prec_min_lower)))\\n",\n    "    q = pd.DataFrame(rows)\\n",\n    "    agree = dict(n_double=len(both), kappa=cohen_kappa(both[\\"annotator_1_v\\"].astype(int), both[\\"annotator_2_v\\"].astype(int))\\n",\n    "                 if len(both) else float(\\"nan\\"), raw_agreement=float((both[\\"annotator_1_v\\"] == both[\\"annotator_2_v\\"]).mean())\\n",\n    "                 if len(both) else float(\\"nan\\"))\\n",\n    "    return q, agree, L"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c024",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# --- Kiểm chứng TỰ ĐỘNG chất lượng quan hệ (không cần người chấm): đồng thuận với một parser phụ thuộc cú pháp ĐỘC LẬP. ---\\n",\n    "# Parser của đồ thị là heuristic POS (v2). spaCy (dependency parse, mô hình thống kê khác hẳn) trích lại quan hệ CÙNG schema trên\\n",\n    "# CHÍNH caption TRAIN đó; một quan hệ được \\"xác nhận\\" nếu spaCy cũng trích ra nó. Đây là nhãn SILVER (đồng thuận giữa hai\\n",\n    "# phương pháp) — KHÔNG phải precision do người chấm: hai parser có thể cùng sai. Chỉ caption TRAIN (mục đích \'agreement\').\\n",\n    "import importlib\\n",\n    "\\n",\n    "CaptionStore.ALLOWED[\\"agreement\\"] = {\\"train\\"}\\n",\n    "_NLP, _DEP_CACHE, DEP_STATUS = None, {}, \\"chưa nạp\\"\\n",\n    "AGREE, AGREE_COVER = None, None                 # gán ở cell chạy từ vựng (chỉ TRAIN)\\n",\n    "_NOUN_POS, _SUBJ, _OBJ = (\\"NOUN\\", \\"PROPN\\"), (\\"nsubj\\",), (\\"dobj\\", \\"obj\\", \\"nsubjpass\\")\\n",\n    "\\n",\n    "\\n",\n    "def dep_parser():\\n",\n    "    \\"\\"\\"Nạp spaCy + en_core_web_sm (tải nếu thiếu và có Internet). Không được ⇒ None (kiểm chứng tự động = NOT RUN).\\"\\"\\"\\n",\n    "    global _NLP, DEP_STATUS\\n",\n    "    if _NLP is None:\\n",\n    "        try:\\n",\n    "            spacy = ensure_package(\\"spacy\\", \\"spacy\\")\\n",\n    "            try:\\n",\n    "                _NLP = spacy.load(CFG.dep_model, disable=[\\"ner\\", \\"lemmatizer\\"])\\n",\n    "            except OSError:\\n",\n    "                if not INTERNET:\\n",\n    "                    raise\\n",\n    "                r = subprocess.run([sys.executable, \\"-m\\", \\"spacy\\", \\"download\\", CFG.dep_model], capture_output=True, text=True)\\n",\n    "                if r.returncode != 0:\\n",\n    "                    raise RuntimeError(r.stderr[-500:])\\n",\n    "                importlib.invalidate_caches()\\n",\n    "                _NLP = spacy.load(CFG.dep_model, disable=[\\"ner\\", \\"lemmatizer\\"])\\n",\n    "            DEP_STATUS = f\\"spaCy {spacy.__version__} / {CFG.dep_model} {_NLP.meta.get(\'version\')}\\"\\n",\n    "        except Exception as e:\\n",\n    "            _NLP, DEP_STATUS = False, f\\"NOT RUN — không nạp được spaCy/{CFG.dep_model}: {type(e).__name__}: {str(e)[:200]}\\"\\n",\n    "        ENV_INFO[\\"dep_parser\\"] = DEP_STATUS\\n",\n    "    return _NLP or None\\n",\n    "\\n",\n    "\\n",\n    "def _nkey(t):\\n",\n    "    return ckey(lemma(t.text.lower(), \\"n\\"), \\"n\\")\\n",\n    "\\n",\n    "\\n",\n    "def _vkey(t):\\n",\n    "    return ckey(lemma(t.text.lower(), \\"v\\"), \\"v\\")\\n",\n    "\\n",\n    "\\n",\n    "def _with_conj(t):\\n",\n    "    return [t] + [c for c in t.conjuncts if c.pos_ in _NOUN_POS]\\n",\n    "\\n",\n    "\\n",\n    "def _subjects(v, depth=0):\\n",\n    "    s = [c for c in v.children if c.dep_ in _SUBJ and c.pos_ in _NOUN_POS]\\n",\n    "    if not s and depth < 3:\\n",\n    "        if v.dep_ in (\\"conj\\", \\"xcomp\\", \\"advcl\\") and v.head.pos_ in (\\"VERB\\", \\"AUX\\"):\\n",\n    "            return _subjects(v.head, depth + 1)                  # \\"sitting and wagging\\", \\"tries to ride\\"\\n",\n    "        if v.dep_ in (\\"acl\\", \\"relcl\\") and v.head.pos_ in _NOUN_POS:\\n",\n    "            return _with_conj(v.head)                            # \\"a lady making a dish\\"\\n",\n    "    return [x for t in s for x in _with_conj(t)]\\n",\n    "\\n",\n    "\\n",\n    "def dep_relations(doc):\\n",\n    "    \\"\\"\\"Quan hệ theo CÙNG schema với parser heuristic: agent {(danh, động)}, object {(danh, động)},\\n",\n    "    svo {(chủ, đối tượng, động từ)}, prep {(danh, danh, giới từ)} — giới từ phải gắn vào DANH TỪ (không vào động từ).\\"\\"\\"\\n",\n    "    out = dict(agent=set(), object=set(), svo=set(), prep=set())\\n",\n    "    for t in doc:\\n",\n    "        if t.pos_ == \\"VERB\\":\\n",\n    "            v = lemma(t.text.lower(), \\"v\\")\\n",\n    "            subs = _subjects(t)\\n",\n    "            objs = [x for c in t.children if c.dep_ in _OBJ and c.pos_ in _NOUN_POS for x in _with_conj(c)]\\n",\n    "            pobjs = [(p.text, x) for p in t.children if p.dep_ == \\"prep\\" for c in p.children\\n",\n    "                     if c.dep_ == \\"pobj\\" and c.pos_ in _NOUN_POS for x in _with_conj(c)]\\n",\n    "            out[\\"agent\\"] |= {(_nkey(s), _vkey(t)) for s in subs}\\n",\n    "            out[\\"object\\"] |= {(_nkey(o), _vkey(t)) for o in objs}\\n",\n    "            out[\\"svo\\"] |= {(_nkey(s), _nkey(o), v) for s in subs for o in objs + [x for _, x in pobjs]}\\n",\n    "        elif t.dep_ == \\"prep\\" and t.head.pos_ in _NOUN_POS:\\n",\n    "            for c in t.children:\\n",\n    "                if c.dep_ == \\"pobj\\" and c.pos_ in _NOUN_POS:\\n",\n    "                    out[\\"prep\\"] |= {(_nkey(t.head), _nkey(x), t.text.lower()) for x in _with_conj(c)}\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def dep_parse_all(texts):\\n",\n    "    nlp = dep_parser()\\n",\n    "    if nlp is None:\\n",\n    "        return False\\n",\n    "    todo = sorted({x.lower() for x in texts} - set(_DEP_CACHE))\\n",\n    "    for txt, doc in zip(todo, nlp.pipe(todo, batch_size=512)):\\n",\n    "        _DEP_CACHE[txt] = dep_relations(doc)\\n",\n    "    return True\\n",\n    "\\n",\n    "\\n",\n    "def triple_confirmed(t, dep):\\n",\n    "    \\"\\"\\"Quan hệ heuristic (nguồn, họ, đích, vị từ) được spaCy xác nhận trên cùng caption?\\n",\n    "    svo: cùng (chủ, đối tượng) và CÙNG động từ đầu của vị từ; prep: cùng (danh, danh) và giới từ thuộc vị từ.\\"\\"\\"\\n",\n    "    s, rel, o, p = t\\n",\n    "    if rel == \\"agent_of\\":\\n",\n    "        return (s, o) in dep[\\"agent\\"]\\n",\n    "    if rel == \\"object_of\\":\\n",\n    "        return (s, o) in dep[\\"object\\"]\\n",\n    "    if rel == \\"svo\\":\\n",\n    "        return (s, o, p.split()[0]) in dep[\\"svo\\"]\\n",\n    "    return any((s, o, w) in dep[\\"prep\\"] for w in p.split())\\n",\n    "\\n",\n    "\\n",\n    "def build_agreement(store, split, vocab):\\n",\n    "    \\"\\"\\"AGREE[(họ, nguồn, đích, vị từ)] = [số caption TRAIN spaCy xác nhận, số caption TRAIN có quan hệ]; cùng điều kiện từ\\n",\n    "    vựng với build_relations_v2. Kèm độ phủ ngược: tỉ lệ quan hệ spaCy (trong từ vựng) mà heuristic cũng trích được.\\"\\"\\"\\n",\n    "    vset = set(vocab[\\"terms\\"])\\n",\n    "    caps = store.by_video(split[\\"train\\"], \\"agreement\\")\\n",\n    "    texts = [c for v in split[\\"train\\"] for c in caps[v]]\\n",\n    "    if not dep_parse_all(texts):\\n",\n    "        return None, None\\n",\n    "    agree, cover = defaultdict(lambda: [0, 0]), Counter()\\n",\n    "    for c in texts:\\n",\n    "        dep = _DEP_CACHE[c.lower()]\\n",\n    "        heur = {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n",\n    "        for t in heur:\\n",\n    "            a = agree[(t[1], t[0], t[2], t[3])]\\n",\n    "            a[0] += int(triple_confirmed(t, dep)); a[1] += 1\\n",\n    "        hk = {(t[1], t[0], t[2]) for t in heur}\\n",\n    "        for fam, rel in ((\\"agent\\", \\"agent_of\\"), (\\"object\\", \\"object_of\\")):\\n",\n    "            for s, o in dep[fam]:\\n",\n    "                if s in vset and o in vset:\\n",\n    "                    cover[(rel, \\"total\\")] += 1; cover[(rel, \\"found\\")] += int((rel, s, o) in hk)\\n",\n    "        for fam in (\\"svo\\", \\"prep\\"):\\n",\n    "            for s, o, _ in dep[fam]:\\n",\n    "                if s in vset and o in vset and s != o:\\n",\n    "                    cover[(fam, \\"total\\")] += 1; cover[(fam, \\"found\\")] += int((fam, s, o) in hk)\\n",\n    "    return {k: tuple(v) for k, v in agree.items()}, dict(cover)\\n",\n    "\\n",\n    "\\n",\n    "def agreement_rate(rel, s, o, p):\\n",\n    "    a = AGREE.get((rel, s, o, p)) if AGREE else None\\n",\n    "    return a[0] / a[1] if a and a[1] else float(\\"nan\\")\\n",\n    "\\n",\n    "\\n",\n    "def agreement_quality(agree, cover, sup):\\n",\n    "    \\"\\"\\"Bảng SILVER theo họ và theo ô (họ × mức phổ biến): tỉ lệ caption-instance được parser độc lập xác nhận (Wilson 95%).\\"\\"\\"\\n",\n    "    rows = []\\n",\n    "    cells = defaultdict(lambda: [0, 0])\\n",\n    "    for (rel, s, o, p), (k, n) in agree.items():\\n",\n    "        st = support_stratum(sup.get((s, rel, o, p), 0))\\n",\n    "        for key in ((rel, \\"ALL\\"), (rel, st), (\\"ALL\\", \\"ALL\\")):\\n",\n    "            cells[key][0] += k; cells[key][1] += n\\n",\n    "    for (rel, st), (k, n) in sorted(cells.items()):\\n",\n    "        p_, lo, hi = wilson(k, n)\\n",\n    "        cv = (cover.get((rel, \\"found\\"), 0) / cover[(rel, \\"total\\")]) if st == \\"ALL\\" and cover.get((rel, \\"total\\")) else float(\\"nan\\")\\n",\n    "        rows.append(dict(kind=\\"silver_parser_agreement\\", family=rel, stratum=st, n_instances=n, n_confirmed=k, agreement=p_,\\n",\n    "                         wilson_lo=lo, wilson_hi=hi, heuristic_covers_spacy=cv,\\n",\n    "                         note=\\"đồng thuận heuristic ↔ spaCy trên caption TRAIN; KHÔNG phải precision do người chấm\\"))\\n",\n    "    return pd.DataFrame(rows)"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c025",\n   "metadata": {},\n   "source": [\n    "## 5. Đặc tả đồ thị v3 (schema)\\n",\n    "\\n",\n    "**Node.** Canonical inventory **cố định**: video MSVD + video tập ngoài, lớp `(lemma, POS)` từ train, khái niệm theo synset WordNet. Ablation chỉ **tắt cạnh**.\\n",\n    "\\n",\n    "**Cạnh.** Mỗi bản ghi cạnh trong `graph_edges_v3.csv` gồm `(src, relation_family, relation, predicate, dst, direction, weight, confidence, source, status)`. Mỗi dòng là **một** quan hệ: một cặp có nhiều vị từ thành nhiều dòng.\\n",\n    "* Trường `predicate` **không bao giờ** chứa tên loại quan hệ; có kiểm tra G2 cho điều này.\\n",\n    "* `weight` là support (tần suất), **không** phải độ tin cậy ngữ nghĩa.\\n",\n    "* `confidence` là `silver <tỉ lệ> (k/n)`: tỉ lệ caption train của quan hệ được parser độc lập xác nhận. Nếu có nhãn người chấm thì là `human <precision>`; không có cả hai thì là `unverified`.\\n",\n    "* Cạnh bị loại vẫn được ghi với `status = excluded:<lý do>`.\\n",\n    "\\n",\n    "**Mẫu đường đi.** Gồm 12 mẫu của v2 cộng `P1m_svo` và `P1m_prep`: đường đi `video → a –f[p]→ b` chỉ được tính khi truy vấn nhắc tới vị từ `p`."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c026",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "FAMILIES = (\\"agent\\", \\"object\\", \\"svo\\", \\"prep\\")\\n",\n    "FAMILY_REL = {\\"agent\\": \\"agent_of\\", \\"object\\": \\"object_of\\", \\"svo\\": \\"svo\\", \\"prep\\": \\"prep\\"}\\n",\n    "_SCENE_SRC = \\"caption TRAIN (POS parser heuristic) — đếm số video TRAIN hỗ trợ; giữ nếu ≥ scene_min_count video, ≤ scene_top_k cạnh / (nguồn, loại)\\"\\n",\n    "_SCENE_W = (\\"support weight = min(1, support / df_train(nguồn)); chiều _inv: min(1, support / df_train(đích)). \\"\\n",\n    "            \\"Đây là TẦN SUẤT, KHÔNG phải độ tin cậy ngữ nghĩa\\")\\n",\n    "_CONF_REL = \\"precision của ô (họ × mức phổ biến) theo nhãn NGƯỜI chấm (Wilson 95%); không có nhãn ⇒ \'unverified\'\\"\\n",\n    "_PRED_VERB = \\"lemma động từ = CHÍNH node đích/nguồn động từ ⇒ không mang thông tin riêng của cạnh; ghi lại, KHÔNG đưa vào message\\"\\n",\n    "_PRED_FREE = (\\"phân phối vị từ của cặp (vị từ → số video TRAIN); embedding vị từ = CLIP-text đóng băng theo prompt \\"\\n",\n    "              \\"(pred_mode \'clip\') hoặc + MLP dư dùng chung (pred_mode \'clip_adapt\'); vào message của GNN và mẫu đường đi P1m\\")\\n",\n    "REL_SPEC = {\\n",\n    "    \\"contains\\":      dict(src=\\"video\\", tgt=\\"class\\", direction=\\"video → lớp được gắn\\", family=\\"contains\\", inverse=None,\\n",\n    "                          source=\\"CLIP zero-shot trên frame của CHÍNH video × từ vựng train (tagger chọn trên val)\\",\\n",\n    "                          weight=\\"1 (binary) | độ tin cậy tagger đã chuẩn hoá (contains = conf, chọn trên val)\\",\\n",\n    "                          predicate=\\"—\\", confidence=\\"điểm tagger CLIP (chỉ khi contains = conf)\\",\\n",\n    "                          split=\\"frame mọi video; từ vựng + thống kê chuẩn hoá: train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"contains_inv\\":  dict(src=\\"class\\", tgt=\\"video\\", direction=\\"lớp → video chứa nó\\", family=\\"contains\\", inverse=\\"contains\\",\\n",\n    "                          source=\\"chiều ngược của contains\\", weight=\\"= contains\\", predicate=\\"—\\", confidence=\\"= contains\\",\\n",\n    "                          split=\\"như contains\\", usage=\\"gnn\\"),\\n",\n    "    \\"agent_of\\":      dict(src=\\"class:noun\\", tgt=\\"class:verb\\", direction=\\"chủ thể → hành động\\", family=\\"agent\\", inverse=None,\\n",\n    "                          source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"agent_of_inv\\":  dict(src=\\"class:verb\\", tgt=\\"class:noun\\", direction=\\"hành động → chủ thể (done_by)\\", family=\\"agent\\",\\n",\n    "                          inverse=\\"agent_of\\", source=\\"chiều ngược của agent_of\\", weight=_SCENE_W, predicate=_PRED_VERB,\\n",\n    "                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"object_of\\":     dict(src=\\"class:noun\\", tgt=\\"class:verb\\", direction=\\"đồ vật → hành động tác động lên nó\\", family=\\"object\\",\\n",\n    "                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL,\\n",\n    "                          split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"object_of_inv\\": dict(src=\\"class:verb\\", tgt=\\"class:noun\\", direction=\\"hành động → đồ vật (acts_on)\\", family=\\"object\\",\\n",\n    "                          inverse=\\"object_of\\", source=\\"chiều ngược của object_of\\", weight=_SCENE_W, predicate=_PRED_VERB,\\n",\n    "                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"svo\\":           dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"chủ thể –vị từ→ đối tượng\\", family=\\"svo\\",\\n",\n    "                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n",\n    "                          split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"svo_inv\\":       dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"đối tượng –bị vị từ bởi→ chủ thể\\", family=\\"svo\\",\\n",\n    "                          inverse=\\"svo\\", source=\\"chiều ngược của svo\\", weight=_SCENE_W, predicate=_PRED_FREE,\\n",\n    "                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"prep\\":          dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"danh từ –giới từ→ danh từ\\", family=\\"prep\\",\\n",\n    "                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n",\n    "                          split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"prep_inv\\":      dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"chiều ngược của prep\\", family=\\"prep\\", inverse=\\"prep\\",\\n",\n    "                          source=\\"chiều ngược của prep\\", weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n",\n    "                          split=\\"train\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"related\\":       dict(src=\\"class\\", tgt=\\"class\\", direction=\\"đối xứng (lưu cả hai chiều)\\", family=\\"related\\", inverse=None,\\n",\n    "                          source=\\"NPMI ≥ ngưỡng của nhãn tagger trên video TRAIN; ≥ min_count; ≤ cooc_top_k / nguồn | biến thể related=conceptnet: ConceptNet 5.7 (tri thức NGOÀI, quan hệ dương en–en, hai đầu ∈ từ vựng train, ≤ cooc_top_k / nguồn theo weight)\\",\\n",\n    "                          weight=\\"P(đích | nguồn) = count / df_train(nguồn)\\", predicate=\\"—\\",\\n",\n    "                          confidence=\\"— (đồng xuất hiện ≠ quan hệ ngữ nghĩa trực tiếp)\\", split=\\"train (nhãn tagger từ frame)\\",\\n",\n    "                          usage=\\"gnn+path\\"),\\n",\n    "    \\"isa\\":           dict(src=\\"class\\", tgt=\\"concept|class (cùng POS)\\", direction=\\"lớp → khái niệm tổng quát hơn\\",\\n",\n    "                          family=\\"isa\\", inverse=None, source=\\"WordNet (hypernym theo biến thể taxonomy)\\", weight=\\"1\\",\\n",\n    "                          predicate=\\"—\\", confidence=\\"— (sense mơ hồ ⇒ KHÔNG nối)\\",\\n",\n    "                          split=\\"tri thức ngoài (chọn sense: ngữ cảnh caption TRAIN)\\", usage=\\"gnn+path\\"),\\n",\n    "    \\"isa_inv\\":       dict(src=\\"concept|class\\", tgt=\\"class\\", direction=\\"khái niệm → lớp cụ thể\\", family=\\"isa\\", inverse=\\"isa\\",\\n",\n    "                          source=\\"chiều ngược của isa\\", weight=\\"1\\", predicate=\\"—\\", confidence=\\"—\\", split=\\"như isa\\", usage=\\"gnn\\"),\\n",\n    "}\\n",\n    "EDGE_TYPES = list(REL_SPEC)\\n",\n    "ETYPE_ID = {t: i for i, t in enumerate(EDGE_TYPES)}\\n",\n    "OPEN_TYPES = (\\"contains\\", \\"contains_inv\\")            # cổng mở sẵn (như v1)\\n",\n    "RELATION_TYPES = [t for t in EDGE_TYPES if REL_SPEC[t][\\"family\\"] != \\"contains\\"]\\n",\n    "PRED_FAMILIES = tuple(CFG.pred_families)             # họ có vị từ tự do (svo, prep)\\n",\n    "PRED_TYPES = [t for f in PRED_FAMILIES for t in (f, f + \\"_inv\\")]\\n",\n    "MASKS = {\\n",\n    "    \\"none\\": (),\\n",\n    "    \\"no_scene\\": tuple(t for t in EDGE_TYPES if REL_SPEC[t][\\"family\\"] in FAMILIES),\\n",\n    "    \\"no_related\\": (\\"related\\",),\\n",\n    "    \\"no_taxonomy\\": (\\"isa\\", \\"isa_inv\\"),\\n",\n    "    \\"contains_only\\": tuple(RELATION_TYPES),\\n",\n    "    \\"no_roles\\": (\\"agent_of\\", \\"agent_of_inv\\", \\"object_of\\", \\"object_of_inv\\"),   # Level 4(b): họ cạnh vai\\n",\n    "    \\"no_svo_prep\\": tuple(PRED_TYPES),                                          # Level 4(b): họ cạnh có vị từ tự do\\n",\n    "}\\n",\n    "# P1m_f: đường đi video → lớp a –f[vị từ p]→ lớp b CHỈ được tính khi truy vấn nhắc tới p (khớp vị từ); dùng ở path_mode \'pred_match\'\\n",\n    "PATTERNS = ([\\"P0_contains\\"] + [f\\"P1_{f}\\" for f in FAMILIES] + [\\"P2_isa\\", \\"P3_related\\"]\\n",\n    "            + [f\\"P4_{f}_isa\\" for f in FAMILIES] + [\\"P5_related_isa\\"] + [f\\"P1m_{f}\\" for f in PRED_FAMILIES])\\n",\n    "PATTERN_LEN = [1] + [2] * 4 + [2, 2] + [3] * 4 + [3] + [2] * len(PRED_FAMILIES)\\n",\n    "PIDX = {p: i for i, p in enumerate(PATTERNS)}\\n",\n    "REL_TEXT = {\\"contains\\": \\"chứa\\", \\"agent_of\\": \\"là_chủ_thể_của\\", \\"agent_of_inv\\": \\"do\\", \\"object_of\\": \\"là_đối_tượng_của\\",\\n",\n    "            \\"object_of_inv\\": \\"tác_động_lên\\", \\"svo\\": \\"{pred}\\", \\"svo_inv\\": \\"bị_{pred}_bởi\\", \\"prep\\": \\"{pred}\\",\\n",\n    "            \\"prep_inv\\": \\"có_{pred}\\", \\"related\\": \\"đồng_xuất_hiện\\", \\"isa\\": \\"là_một\\"}"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c027",\n   "metadata": {},\n   "source": [\n    "### 5b. Taxonomy (giữ nguyên cách dựng của v2)\\n",\n    "Khái niệm được khoá theo synset WordNet (ví dụ `animal.n.01`), nên danh từ và động từ không bao giờ bị trộn với nhau. Có bốn biến thể trên **cùng** một tập node (tập node là hợp của các biến thể):\\n",\n    "\\n",\n    "| Biến thể | Cách chọn hypernym |\\n",\n    "|---|---|\\n",\n    "| `heuristic` | quy tắc v1, lấy sense đầu tiên |\\n",\n    "| `filtered` | `heuristic` bỏ các khái niệm \\"hub\\" (hơn 15% video train với tới) và khái niệm danh từ quá nông |\\n",\n    "| `sense` | chọn sense bằng Extended Lesk trên ngữ cảnh caption **train**; mapping **mơ hồ** thì không nối `isa` và được ghi rõ |\\n",\n    "| `none` | không có `isa` |\\n",\n    "\\n",\n    "Mọi mapping (synset, gloss, điểm Lesk, cờ mơ hồ, hub) được xuất ra `graph_taxonomy_audit.csv`.\\n",\n    "\\n",\n    "Ở v3, `filtered` thắng val nhưng thua trên test đã nhìn, nên **không** còn là ứng viên. Stage V4 so sánh `none` (mặc định, đơn giản hơn) với taxonomy không lọc (`heuristic`). `heuristic` chỉ được giữ khi thắng theo luật đăng ký trước. Kiểm tra mơ hồ, tương thích danh/động từ, hypernym quá chung, hub và chu trình vẫn được ghi trong audit."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c028",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "CaptionStore.ALLOWED[\\"taxonomy\\"] = {\\"train\\"}          # chọn sense WordNet theo ngữ cảnh caption TRAIN\\n",\n    "_WTOK = re.compile(r\\"[a-z]+\\")\\n",\n    "\\n",\n    "\\n",\n    "def _head(s):\\n",\n    "    return s.lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n",\n    "\\n",\n    "\\n",\n    "def _noun_targets(s, word):\\n",\n    "    \\"\\"\\"Quy tắc hypernym danh từ của v1 (cha/ông đủ sâu, nhìn thấy được + BASIC_CATEGORIES) — trả về synset.\\"\\"\\"\\n",\n    "    hy = []\\n",\n    "    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n",\n    "        for depth in range(len(path) - 2, -1, -1):\\n",\n    "            h = path[depth]\\n",\n    "            name = _head(h)\\n",\n    "            near = depth >= len(path) - 3\\n",\n    "            if name == word or name in GENERIC_CONCEPTS or h in hy:\\n",\n    "                continue\\n",\n    "            if (near and depth >= CFG.isa_min_depth and (h.lexname() in VISUAL_LEX or h.lexname() == \\"noun.Tops\\")) \\\\\\n",\n    "                    or name in BASIC_CATEGORIES:\\n",\n    "                hy.append(h)\\n",\n    "    return hy[:5]\\n",\n    "\\n",\n    "\\n",\n    "def _verb_targets(s, word):\\n",\n    "    \\"\\"\\"Quy tắc hypernym động từ của v1 (cha/ông, bỏ động từ quá chung).\\"\\"\\"\\n",\n    "    hy = []\\n",\n    "    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n",\n    "        for depth in (len(path) - 2, len(path) - 3):\\n",\n    "            if depth < 0:\\n",\n    "                continue\\n",\n    "            h = path[depth]\\n",\n    "            name = _head(h)\\n",\n    "            if name != word and name not in GENERIC_VERBS and \\" \\" not in name and h not in hy:\\n",\n    "                hy.append(h)\\n",\n    "    return hy[:3]\\n",\n    "\\n",\n    "\\n",\n    "def _signature(s):\\n",\n    "    \\"\\"\\"Extended Lesk: từ trong định nghĩa, ví dụ, lemma của synset + cha + 10 con (đã lemma hoá).\\"\\"\\"\\n",\n    "    words = set(_WTOK.findall(s.definition().lower()))\\n",\n    "    for e in s.examples():\\n",\n    "        words |= set(_WTOK.findall(e.lower()))\\n",\n    "    for r in [s] + sorted(s.hypernyms(), key=_syn_key) + sorted(s.hyponyms(), key=_syn_key)[:10]:\\n",\n    "        words |= {l.name().lower() for l in r.lemmas() if \\"_\\" not in l.name()}\\n",\n    "    return ({lemma(w, \\"n\\") for w in words} | {lemma(w, \\"v\\") for w in words}) - STOP\\n",\n    "\\n",\n    "\\n",\n    "def build_sense_context(store, split, vset):\\n",\n    "    \\"\\"\\"Ngữ cảnh của mỗi lớp = các danh từ/động từ khác cùng xuất hiện trong caption TRAIN chứa lớp đó.\\"\\"\\"\\n",\n    "    caps = store.by_video(split[\\"train\\"], \\"taxonomy\\")\\n",\n    "    ctx = defaultdict(Counter)\\n",\n    "    for v in split[\\"train\\"]:\\n",\n    "        for c in caps[v]:\\n",\n    "            nouns, verbs, _ = parse_caption_v2(c)\\n",\n    "            words = set(nouns) | set(verbs)\\n",\n    "            for k in {ckey(n, \\"n\\") for n in nouns} | {ckey(x, \\"v\\") for x in verbs}:\\n",\n    "                if k in vset:\\n",\n    "                    ctx[k].update(words - {k.rsplit(\\"#\\", 1)[0]})\\n",\n    "    return ctx\\n",\n    "\\n",\n    "\\n",\n    "def choose_sense(lem, pos, ctx):\\n",\n    "    wn = _wordnet()\\n",\n    "    syns = wn.synsets(lem.replace(\\" \\", \\"_\\"), pos=pos)[:CFG.sense_max_candidates] if wn is not None else []\\n",\n    "    if not syns:\\n",\n    "        return None, True, []\\n",\n    "    scores = [sum(math.log1p(ctx[w]) for w in sorted(_signature(s)) if w in ctx) for s in syns]   # thứ tự cố định\\n",\n    "    order = np.argsort(-np.array(scores), kind=\\"stable\\")\\n",\n    "    b = scores[order[0]]\\n",\n    "    s2 = scores[order[1]] if len(syns) > 1 else -1e9\\n",\n    "    ambiguous = len(syns) > 1 and (b <= 0 or b - s2 < CFG.sense_margin)\\n",\n    "    return syns[order[0]], ambiguous, [(syns[i].name(), round(scores[i], 2)) for i in order[:3]]\\n",\n    "\\n",\n    "\\n",\n    "def build_taxonomy_v2(vocab, store, split, train_tags):\\n",\n    "    \\"\\"\\"Ba biến thể taxonomy trên cùng một tập khái niệm (hợp) ⇒ inventory không đổi giữa các biến thể:\\n",\n    "    heuristic (= quy tắc v1, sense đầu tiên), filtered (bỏ hub/khái niệm nông), sense (chọn sense theo ngữ cảnh\\n",\n    "    caption TRAIN; mơ hồ ⇒ KHÔNG nối isa và đánh dấu), none (không isa).\\"\\"\\"\\n",\n    "    wn = _wordnet()\\n",\n    "    vset = set(vocab[\\"terms\\"])\\n",\n    "    ctx = build_sense_context(store, split, vset)\\n",\n    "    concept_meta, var = {}, {\\"heuristic\\": {}, \\"sense\\": {}, \\"filtered\\": {}, \\"none\\": {}}\\n",\n    "    rows = []\\n",\n    "\\n",\n    "    def target_key(h, pos):\\n",\n    "        k = ckey(_head(h), pos)\\n",\n    "        if k in vset:\\n",\n    "            return k                                   # hypernym trùng một lớp cùng POS ⇒ nối thẳng vào lớp đó\\n",\n    "        concept_meta.setdefault(h.name(), dict(key=h.name(), lemma=_head(h), pos=pos, synset=h.name(),\\n",\n    "                                               gloss=h.definition(), depth=h.min_depth(), source=\\"wordnet\\"))\\n",\n    "        return h.name()\\n",\n    "\\n",\n    "    def fallback(lem, pos):\\n",\n    "        names = (FALLBACK_ISA if pos == \\"n\\" else FALLBACK_VERB_ISA).get(lem, [])\\n",\n    "        out = []\\n",\n    "        for nm in names:\\n",\n    "            k = ckey(nm, pos)\\n",\n    "            if k not in vset:\\n",\n    "                k = f\\"{nm}.{pos}.fallback\\"\\n",\n    "                concept_meta.setdefault(k, dict(key=k, lemma=nm, pos=pos, synset=\\"\\", gloss=\\"\\", depth=-1, source=\\"fallback\\"))\\n",\n    "            out.append(k)\\n",\n    "        return out\\n",\n    "\\n",\n    "    for k in vocab[\\"terms\\"]:\\n",\n    "        lem, pos = k.rsplit(\\"#\\", 1)\\n",\n    "        syns = wn.synsets(lem.replace(\\" \\", \\"_\\"), pos=pos) if wn is not None else []\\n",\n    "        tf = _noun_targets if pos == \\"n\\" else _verb_targets\\n",\n    "        heur = [target_key(h, pos) for h in tf(syns[0], lem)] if syns else fallback(lem, pos)\\n",\n    "        s_syn, amb, scores = choose_sense(lem, pos, ctx[k])\\n",\n    "        sense = [] if (amb or s_syn is None) else [target_key(h, pos) for h in tf(s_syn, lem)]\\n",\n    "        heur = list(dict.fromkeys(t for t in heur if t != k))\\n",\n    "        sense = list(dict.fromkeys(t for t in sense if t != k))\\n",\n    "        var[\\"heuristic\\"][k], var[\\"sense\\"][k] = heur, sense\\n",\n    "        rows.append(dict(class_key=k, pos=pos, train_df=vocab[\\"df\\"][k], n_synsets=len(syns),\\n",\n    "                         heuristic_synset=syns[0].name() if syns else \\"\\", heuristic_gloss=syns[0].definition() if syns else \\"\\",\\n",\n    "                         sense_synset=s_syn.name() if s_syn is not None else \\"\\", sense_ambiguous=bool(amb),\\n",\n    "                         sense_differs=bool(syns) and s_syn is not None and s_syn != syns[0], lesk_top3=scores,\\n",\n    "                         heuristic_targets=heur, sense_targets=sense))\\n",\n    "    # filtered: bỏ khái niệm \'hub\' (với tới > tax_hub_frac video TRAIN) và khái niệm danh từ quá nông\\n",\n    "    tr = list(split[\\"train\\"])\\n",\n    "    reach = defaultdict(set)\\n",\n    "    for v in tr:\\n",\n    "        for c in train_tags.get(v, []):\\n",\n    "            for t in var[\\"heuristic\\"].get(c, []):\\n",\n    "                reach[t].add(v)\\n",\n    "    hub = {t for t, s in reach.items() if len(s) / max(len(tr), 1) > CFG.tax_hub_frac}\\n",\n    "    shallow = {t for t, m in concept_meta.items() if m[\\"pos\\"] == \\"n\\" and 0 <= m[\\"depth\\"] < CFG.tax_min_depth_filtered}\\n",\n    "    for row in rows:\\n",\n    "        k = row[\\"class_key\\"]\\n",\n    "        var[\\"filtered\\"][k] = [t for t in var[\\"heuristic\\"][k] if t not in hub and t not in shallow]\\n",\n    "        row[\\"filtered_targets\\"] = var[\\"filtered\\"][k]\\n",\n    "        row[\\"dropped_hub\\"] = [t for t in var[\\"heuristic\\"][k] if t in hub]\\n",\n    "        row[\\"dropped_shallow\\"] = [t for t in var[\\"heuristic\\"][k] if t in shallow]\\n",\n    "    var[\\"none\\"] = {k: [] for k in vocab[\\"terms\\"]}\\n",\n    "    hub_rows = sorted(((t, len(s), len(s) / max(len(tr), 1)) for t, s in reach.items()), key=lambda x: -x[1])\\n",\n    "    cycles = {}\\n",\n    "    for name, tax in var.items():                       # chu trình trên các cạnh lớp → lớp\\n",\n    "        nxt = {k: [t for t in ts if t in vset] for k, ts in tax.items()}\\n",\n    "        seen, stack, cyc = set(), set(), 0\\n",\n    "\\n",\n    "        def dfs(u):\\n",\n    "            nonlocal cyc\\n",\n    "            seen.add(u); stack.add(u)\\n",\n    "            for w in nxt.get(u, []):\\n",\n    "                if w in stack:\\n",\n    "                    cyc += 1\\n",\n    "                elif w not in seen:\\n",\n    "                    dfs(w)\\n",\n    "            stack.discard(u)\\n",\n    "        for u in nxt:\\n",\n    "            if u not in seen:\\n",\n    "                dfs(u)\\n",\n    "        cycles[name] = cyc\\n",\n    "    stats = dict(n_concepts=len(concept_meta), hubs=[dict(target=t, train_videos=n, frac=round(f, 3)) for t, n, f in hub_rows[:25]],\\n",\n    "                 n_hub=len(hub), n_shallow=len(shallow), cycles=cycles,\\n",\n    "                 n_ambiguous=int(sum(r[\\"sense_ambiguous\\"] for r in rows)),\\n",\n    "                 n_sense_differs=int(sum(r[\\"sense_differs\\"] for r in rows)),\\n",\n    "                 isa_edges={n: int(sum(len(v) for v in t.values())) for n, t in var.items()})\\n",\n    "    return var, concept_meta, pd.DataFrame(rows), stats"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c029",\n   "metadata": {},\n   "source": [\n    "### 5c. Đồ thị v3 và hai đối chứng ngẫu nhiên có ràng buộc\\n",\n    "**`GraphV3`** giữ topology cặp của v2, nên relation-family-only (G2) trùng với `v2_base`. Mỗi cặp (nguồn, họ, đích) mang **phân phối vị từ** `{vị từ: số video train}`.\\n",\n    "\\n",\n    "**Chính sách cạnh:**\\n",\n    "* `all`: như v2;\\n",\n    "* `support`: chỉ vị từ có ≥ 5 video train;\\n",\n    "* `agreement`: chỉ vị từ của quan hệ được parser độc lập xác nhận ở ≥ 50% caption train của nó (kiểm chứng tự động);\\n",\n    "* `precision`: chỉ ô đã kiểm chứng bằng nhãn người (tuỳ chọn).\\n",\n    "\\n",\n    "Một cặp không còn vị từ hợp lệ thì bị loại và lưu lại.\\n",\n    "\\n",\n    "**Đối chứng:**\\n",\n    "* **Rewiring** (G5): directed double-edge swap trong từng họ và nhóm kiểu node. Phép này giữ bậc vào/ra, trọng số và **đa tập phân phối vị từ**, vì vị từ đi theo cạnh. `contains` không đổi.\\n",\n    "* **Hoán vị vị từ** (G4): hoán vị phân phối vị từ giữa các cạnh **cùng họ** svo/prep. Topology, trọng số và bậc giống hệt; chỉ việc ghép vị từ ↔ cạnh bị phá.\\n",\n    "\\n",\n    "`path_mode` gồm `weighted`, `count`, `degnorm`, `dedup` như v2, và `pred_match` (= `weighted` cộng mẫu P1m khớp vị từ). Engine, `compute_R` và phần giải thích dùng **chung** các ma trận này."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c030",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "class NodeInventory:\\n",\n    "    \\"\\"\\"Tập node CỐ ĐỊNH cho mọi cấu hình so sánh được: video | lớp (lemma, POS) | khái niệm (synset).\\n",\n    "    Ablation chỉ tắt cạnh ⇒ node id, số node, kích thước embedding không đổi.\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, records, vocab, concept_meta):\\n",\n    "        self.n_vid = len(records)\\n",\n    "        self.class_keys = list(vocab[\\"terms\\"])\\n",\n    "        self.concept_keys = sorted(concept_meta)\\n",\n    "        self.n_cls, self.n_con = len(self.class_keys), len(self.concept_keys)\\n",\n    "        self.n_nodes = self.n_vid + self.n_cls + self.n_con\\n",\n    "        self.key2id = {k: self.n_vid + i for i, k in enumerate(self.class_keys)}\\n",\n    "        self.key2id.update({k: self.n_vid + self.n_cls + i for i, k in enumerate(self.concept_keys)})\\n",\n    "        self.id2key = {i: k for k, i in self.key2id.items()}\\n",\n    "        self.node_type = np.array([\\"video\\"] * self.n_vid + [\\"class\\"] * self.n_cls + [\\"concept\\"] * self.n_con)\\n",\n    "        self.pos = np.array([\\"\\"] * self.n_vid + [k.rsplit(\\"#\\", 1)[1] for k in self.class_keys]\\n",\n    "                            + [concept_meta[k][\\"pos\\"] for k in self.concept_keys])\\n",\n    "        self.lemma = {**{self.key2id[k]: vocab[\\"lemma\\"][k] for k in self.class_keys},\\n",\n    "                      **{self.key2id[k]: concept_meta[k][\\"lemma\\"] for k in self.concept_keys}}\\n",\n    "        self.name = {i: (records[i][\\"video_id\\"] if i < self.n_vid else self.id2key[i]) for i in range(self.n_nodes)}\\n",\n    "        self.concepts_by_lemma = defaultdict(list)\\n",\n    "        for k in self.concept_keys:\\n",\n    "            self.concepts_by_lemma[(concept_meta[k][\\"lemma\\"], concept_meta[k][\\"pos\\"])].append(self.key2id[k])\\n",\n    "        self.concept_meta = concept_meta\\n",\n    "        self.sha1 = sha1_of([self.n_vid, self.class_keys, self.concept_keys])\\n",\n    "\\n",\n    "    def semantic_type(self, i):\\n",\n    "        t = self.node_type[i]\\n",\n    "        return \\"video\\" if t == \\"video\\" else ({\\"n\\": \\"noun\\", \\"v\\": \\"verb\\"}[self.pos[i]] + (\\"_concept\\" if t == \\"concept\\" else \\"\\"))\\n",\n    "\\n",\n    "    def is_class(self, i):\\n",\n    "        return self.n_vid <= i < self.n_vid + self.n_cls\\n",\n    "\\n",\n    "\\n",\n    "def contains_calibration(tag_scores, train_vids, inv):\\n",\n    "    \\"\\"\\"Thống kê chuẩn hoá điểm tagger của các nhãn ĐÃ GẮN trên video TRAIN, tách danh từ / động từ.\\"\\"\\"\\n",\n    "    by = defaultdict(list)\\n",\n    "    for v in train_vids:\\n",\n    "        for k, s in tag_scores[v].items():\\n",\n    "            by[k.rsplit(\\"#\\", 1)[1]].append(s)\\n",\n    "    return {p: dict(mu=float(np.mean(x)), sd=float(np.std(x) + 1e-6), lo=float(np.percentile(x, 1)),\\n",\n    "                    hi=float(np.percentile(x, 99))) for p, x in by.items()}\\n",\n    "\\n",\n    "\\n",\n    "def _conf_weight(s, pos, cal, mode):\\n",\n    "    c = cal[pos]\\n",\n    "    if mode == \\"sigmoid\\":                      # z bị chặn ±3 để hạn chế ngoại lai\\n",\n    "        w = 1.0 / (1.0 + math.exp(-float(np.clip((s - c[\\"mu\\"]) / c[\\"sd\\"], -3, 3))))\\n",\n    "    else:                                      # minmax giữa phân vị 1–99 của train\\n",\n    "        w = (s - c[\\"lo\\"]) / max(c[\\"hi\\"] - c[\\"lo\\"], 1e-6)\\n",\n    "    return float(np.clip(w, CFG.conf_floor, 1.0))\\n",\n    "\\n",\n    "\\n",\n    "def _csr(n_rows, n_cols, rows, cols, vals):\\n",\n    "    if len(rows) == 0:\\n",\n    "        return sp.csr_matrix((n_rows, n_cols), dtype=np.float32)\\n",\n    "    M = sp.csr_matrix((np.asarray(vals, np.float32), (np.asarray(rows), np.asarray(cols))), shape=(n_rows, n_cols))\\n",\n    "    M.sum_duplicates()\\n",\n    "    return M\\n",\n    "\\n",\n    "\\n",\n    "def _symnorm(M):\\n",\n    "    if M.nnz == 0:\\n",\n    "        return M.copy()\\n",\n    "    dout = np.asarray(M.sum(1)).ravel(); din = np.asarray(M.sum(0)).ravel()\\n",\n    "    a = np.where(dout > 0, 1 / np.sqrt(np.maximum(dout, 1e-12)), 0)\\n",\n    "    b = np.where(din > 0, 1 / np.sqrt(np.maximum(din, 1e-12)), 0)\\n",\n    "    return (sp.diags(a) @ M @ sp.diags(b)).tocsr().astype(np.float32)\\n",\n    "\\n",\n    "\\n",\n    "def _maxprod(C, B):\\n",\n    "    \\"\\"\\"out[v, c] = max_o C[v, o] · B[o, c] (thưa) — tổ hợp \'max\' thay cho tổng (path_dedup).\\"\\"\\"\\n",\n    "    C, B = C.tocsr(), B.tocsr()\\n",\n    "    rows = []\\n",\n    "    for v in range(C.shape[0]):\\n",\n    "        sl = slice(C.indptr[v], C.indptr[v + 1])\\n",\n    "        idx, val = C.indices[sl], C.data[sl]\\n",\n    "        if len(idx) == 0:\\n",\n    "            rows.append(sp.csr_matrix((1, B.shape[1]), dtype=np.float32))\\n",\n    "            continue\\n",\n    "        sub = (sp.diags(val) @ B[idx]).tocsr()\\n",\n    "        rows.append(sp.csr_matrix(sub.max(axis=0)))\\n",\n    "    return sp.vstack(rows).tocsr()\\n",\n    "\\n",\n    "\\n",\n    "\\n",\n    "\\n",\n    "PATH_TYPES = [\\"agent_of\\", \\"agent_of_inv\\", \\"object_of\\", \\"object_of_inv\\", \\"svo\\", \\"svo_inv\\", \\"prep\\", \\"prep_inv\\", \\"related\\", \\"isa\\"]\\n",\n    "REL_POLICIES = (\\"all\\", \\"support\\", \\"agreement\\", \\"precision\\")\\n",\n    "\\n",\n    "\\n",\n    "def predicate_tokens(p):\\n",\n    "    return tuple(p.split())\\n",\n    "\\n",\n    "\\n",\n    "class GraphV3:\\n",\n    "    \\"\\"\\"Đồ thị dị thể có kiểu, có hướng, có trọng số trên NodeInventory cố định; cạnh quan hệ mang VỊ TỪ.\\n",\n    "    Cạnh quan hệ = cặp (nguồn, họ, đích) như v2 (cùng topology) + phân phối vị từ của cặp {vị từ: số video TRAIN}.\\n",\n    "      rel_policy : \'all\' | \'support\' (vị từ có ≥ pred_min_support video TRAIN) | \'agreement\' (≥ agree_min caption TRAIN được\\n",\n    "                   parser độc lập xác nhận — kiểm chứng TỰ ĐỘNG) | \'precision\' (ô đã kiểm chứng bằng nhãn người, tuỳ chọn)\\n",\n    "                   — cặp không còn vị từ hợp lệ bị LOẠI và lưu trong self.excluded\\n",\n    "      path_mode  : \'weighted\' | \'count\' | \'degnorm\' | \'dedup\' | \'pred_match\' (= weighted + mẫu P1m khớp vị từ truy vấn)\\n",\n    "      rewire_seed: constrained rewiring (vị từ, trọng số đi theo cạnh) ; pred_shuffle_seed: hoán vị phân phối vị từ giữa\\n",\n    "                   các cạnh CÙNG họ (topology, trọng số giữ nguyên) ; related_cfg: (min_count, min_npmi).\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, inv, records, train_vids, relations, taxonomy, *, tag_scores=None, contains=\\"binary\\",\\n",\n    "                 conf_mode=\\"sigmoid\\", conf_cal=None, rel_policy=\\"all\\", path_mode=\\"weighted\\", masked=(),\\n",\n    "                 rewire_seed=None, pred_shuffle_seed=None, tags_override=None, related_cfg=None, verified_cells=None, name=\\"\\"):\\n",\n    "        assert rel_policy in REL_POLICIES, rel_policy\\n",\n    "        if rel_policy == \\"precision\\" and verified_cells is None:\\n",\n    "            raise ValueError(\\"rel_policy = precision cần nhãn người chấm (verified_cells) — NOT RUN khi chưa có nhãn\\")\\n",\n    "        if rel_policy == \\"agreement\\" and not AGREE:\\n",\n    "            raise ValueError(\\"rel_policy = agreement cần kiểm chứng tự động (spaCy) — NOT RUN khi không nạp được parser\\")\\n",\n    "        self.inv, self.name, self.path_mode, self.masked = inv, name, path_mode, tuple(sorted(set(masked)))\\n",\n    "        self.contains_mode, self.conf_mode, self.rel_policy = contains, conf_mode, rel_policy\\n",\n    "        self.rewire_seed, self.pred_shuffle_seed = rewire_seed, pred_shuffle_seed\\n",\n    "        self.related_cfg = tuple(related_cfg or (CFG.cooc_min_count, CFG.cooc_min_npmi))\\n",\n    "        n, N = inv.n_nodes, inv.n_vid\\n",\n    "        self.n_nodes, self.n_vid = n, N\\n",\n    "        train_set = set(int(v) for v in train_vids)\\n",\n    "        tr_rows = np.array(sorted(train_set))\\n",\n    "        n_tr = max(len(tr_rows), 1)\\n",\n    "        tags = tags_override if tags_override is not None else {r[\\"vid\\"]: r[\\"tags\\"] for r in records}\\n",\n    "        freq = Counter(t for v in train_set for t in tags[v])\\n",\n    "        self.active = {inv.key2id[k] for k, c in freq.items() if c >= CFG.min_tag_freq and k in inv.key2id}\\n",\n    "\\n",\n    "        # ---------- contains (video → lớp), từ nhãn tagger của chính video ----------\\n",\n    "        rows, cols, vals, self.vid_tags = [], [], [], []\\n",\n    "        for v in range(N):\\n",\n    "            ids = []\\n",\n    "            for k in tags[v]:\\n",\n    "                i = inv.key2id.get(k)\\n",\n    "                if i is None or i not in self.active:\\n",\n    "                    continue\\n",\n    "                w = 1.0 if contains == \\"binary\\" else _conf_weight(tag_scores[v][k], k.rsplit(\\"#\\", 1)[1], conf_cal, conf_mode)\\n",\n    "                rows.append(v); cols.append(i); vals.append(w); ids.append(i)\\n",\n    "            self.vid_tags.append(sorted(ids))\\n",\n    "        self.A_con = _csr(N, n, rows, cols, vals)\\n",\n    "        Cb = (self.A_con > 0).astype(np.float32)\\n",\n    "        cls_df = np.asarray(Cb[tr_rows].sum(0)).ravel()\\n",\n    "        self.cls_df = cls_df\\n",\n    "\\n",\n    "        # ---------- quan hệ từ caption TRAIN, TÁCH theo họ, giữ phân phối vị từ ----------\\n",\n    "        sup, preds = defaultdict(Counter), defaultdict(Counter)\\n",\n    "        for v in train_set:\\n",\n    "            seen = set()\\n",\n    "            for s, rel, o, pred in relations.get(v, []):\\n",\n    "                a, b = inv.key2id.get(s), inv.key2id.get(o)\\n",\n    "                if a is None or b is None or a not in self.active or b not in self.active or a == b:\\n",\n    "                    continue\\n",\n    "                preds[(rel, a, b)][pred] += 1             # = số video TRAIN có đúng (nguồn, họ, đích, vị từ)\\n",\n    "                if (rel, a, b) not in seen:\\n",\n    "                    sup[rel][(a, b)] += 1; seen.add((rel, a, b))\\n",\n    "        self.preds = preds\\n",\n    "\\n",\n    "        def kept_preds(rel, a, b):\\n",\n    "            pc = preds[(rel, a, b)]\\n",\n    "            if rel_policy == \\"support\\":\\n",\n    "                return {p: c for p, c in pc.items() if PRED_SUPPORT.get((rel, p), 0) >= CFG.pred_min_support}\\n",\n    "            if rel_policy == \\"precision\\":\\n",\n    "                return {p: c for p, c in pc.items() if (rel, support_stratum(c)) in verified_cells}\\n",\n    "            if rel_policy == \\"agreement\\":\\n",\n    "                ka, kb = inv.id2key[a], inv.id2key[b]\\n",\n    "                return {p: c for p, c in pc.items() if agreement_rate(rel, ka, kb, p) >= CFG.agree_min}\\n",\n    "            return dict(pc)\\n",\n    "\\n",\n    "        fwd, self.excluded = {}, []\\n",\n    "        for rel in REL_CATS:\\n",\n    "            by_src = defaultdict(list)\\n",\n    "            for (a, b), c in sup[rel].items():\\n",\n    "                if c < CFG.scene_min_count:\\n",\n    "                    continue\\n",\n    "                kp = kept_preds(rel, a, b)\\n",\n    "                if not kp:\\n",\n    "                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=dict(preds[(rel, a, b)]),\\n",\n    "                                              reason=f\\"policy:{rel_policy}\\"))\\n",\n    "                    continue\\n",\n    "                by_src[a].append((c, b, kp))\\n",\n    "            E = [(a, b, c, kp) for a, lst in by_src.items() for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[:CFG.scene_top_k]]\\n",\n    "            for a, lst in by_src.items():\\n",\n    "                for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[CFG.scene_top_k:]:\\n",\n    "                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=kp, reason=\\"top_k\\"))\\n",\n    "            f = dict(src=np.array([e[0] for e in E], int), dst=np.array([e[1] for e in E], int),\\n",\n    "                     support=np.array([e[2] for e in E], float))\\n",\n    "            f[\\"w\\"] = np.minimum(1.0, f[\\"support\\"] / np.maximum(cls_df[f[\\"src\\"]], 1)) if E else np.zeros(0)\\n",\n    "            f[\\"w_inv\\"] = np.minimum(1.0, f[\\"support\\"] / np.maximum(cls_df[f[\\"dst\\"]], 1)) if E else np.zeros(0)\\n",\n    "            f[\\"pdist\\"] = [e[3] for e in E]\\n",\n    "            fwd[rel] = f\\n",\n    "        self.n_candidate_relations = {r: len(sup[r]) for r in REL_CATS}\\n",\n    "\\n",\n    "        # ---------- related: NPMI đồng xuất hiện nhãn tagger trên video TRAIN (đối xứng) ----------\\n",\n    "        if self.related_cfg[0] == \\"conceptnet\\":                   # tri thức NGOÀI: không dùng caption hay nhãn\\n",\n    "            und = {}\\n",\n    "            for e in KG_EDGES:\\n",\n    "                a_, b_ = inv.key2id.get(e[\\"src\\"]), inv.key2id.get(e[\\"dst\\"])\\n",\n    "                if a_ is None or b_ is None or a_ == b_ or a_ not in self.active or b_ not in self.active:\\n",\n    "                    continue\\n",\n    "                k_ = (min(a_, b_), max(a_, b_))\\n",\n    "                w_ = max(und.get(k_, (0.0, 0.0))[0], float(e[\\"weight\\"]))\\n",\n    "                und[k_] = (w_, w_)\\n",\n    "        else:\\n",\n    "            min_count, min_npmi = self.related_cfg\\n",\n    "            cooc = Counter()\\n",\n    "            for v in train_set:\\n",\n    "                for a, b in itertools.combinations(sorted(self.vid_tags[v]), 2):\\n",\n    "                    cooc[(a, b)] += 1\\n",\n    "            und = {}\\n",\n    "            for (a, b), c in cooc.items():\\n",\n    "                if c < min_count:\\n",\n    "                    continue\\n",\n    "                p_ab = c / n_tr\\n",\n    "                npmi = math.log(p_ab / ((cls_df[a] / n_tr) * (cls_df[b] / n_tr))) / max(-math.log(p_ab), 1e-6)\\n",\n    "                if npmi >= min_npmi:\\n",\n    "                    und[(a, b)] = (c, npmi)\\n",\n    "        keep, by_src = set(), defaultdict(list)\\n",\n    "        for (a, b), (c, s) in und.items():\\n",\n    "            by_src[a].append((s, b)); by_src[b].append((s, a))\\n",\n    "        for a, lst in by_src.items():\\n",\n    "            for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]:\\n",\n    "                keep.add((min(a, b), max(a, b)))\\n",\n    "        RE = sorted(keep)\\n",\n    "        related = dict(src=np.array([e[0] for e in RE], int), dst=np.array([e[1] for e in RE], int),\\n",\n    "                       count=np.array([und[e][0] for e in RE], float))\\n",\n    "        if self.related_cfg[0] == \\"conceptnet\\":\\n",\n    "            related[\\"w_ab\\"] = np.minimum(1.0, related[\\"count\\"] / 2.0) if RE else np.zeros(0)   # weight ConceptNet / 2, chặn 1\\n",\n    "            related[\\"w_ba\\"] = related[\\"w_ab\\"].copy()\\n",\n    "        else:\\n",\n    "            related[\\"w_ab\\"] = np.minimum(1.0, related[\\"count\\"] / np.maximum(cls_df[related[\\"src\\"]], 1)) if RE else np.zeros(0)\\n",\n    "            related[\\"w_ba\\"] = np.minimum(1.0, related[\\"count\\"] / np.maximum(cls_df[related[\\"dst\\"]], 1)) if RE else np.zeros(0)\\n",\n    "\\n",\n    "        # ---------- isa (biến thể taxonomy) ----------\\n",\n    "        IE = sorted({(inv.key2id[c], inv.key2id[t]) for c, ts in taxonomy.items() if c in inv.key2id\\n",\n    "                     and inv.key2id[c] in self.active for t in ts if t in inv.key2id\\n",\n    "                     and (inv.node_type[inv.key2id[t]] == \\"concept\\" or inv.key2id[t] in self.active)})\\n",\n    "        isa = dict(src=np.array([e[0] for e in IE], int), dst=np.array([e[1] for e in IE], int))\\n",\n    "        isa_true = _csr(n, n, isa[\\"src\\"], isa[\\"dst\\"], np.ones(len(IE)))\\n",\n    "\\n",\n    "        # ---------- đối chứng 1: constrained rewiring (contains giữ nguyên; vị từ + trọng số đi theo cạnh) ----------\\n",\n    "        self.rewire_audit = []\\n",\n    "        if rewire_seed is not None:\\n",\n    "            rng = np.random.RandomState(rewire_seed)\\n",\n    "            for rel in REL_CATS:\\n",\n    "                fwd[rel], st = rewire_directed(fwd[rel], inv, rng, [\\"w\\", \\"w_inv\\", \\"support\\", \\"pdist\\"], rel)\\n",\n    "                self.rewire_audit.append(st)\\n",\n    "            related, st = rewire_undirected(related, inv, rng, \\"related\\")\\n",\n    "            self.rewire_audit.append(st)\\n",\n    "            isa, st = rewire_directed(isa, inv, rng, [], \\"isa\\")\\n",\n    "            self.rewire_audit.append(st)\\n",\n    "\\n",\n    "        # ---------- đối chứng 2: hoán vị phân phối vị từ giữa các cạnh CÙNG họ (topology, trọng số giữ nguyên) ----------\\n",\n    "        self.shuffle_audit = []\\n",\n    "        if pred_shuffle_seed is not None:\\n",\n    "            rng = np.random.RandomState(pred_shuffle_seed)\\n",\n    "            for rel in PRED_FAMILIES:\\n",\n    "                old = fwd[rel][\\"pdist\\"]\\n",\n    "                perms = [rng.permutation(len(old)) for _ in range(10)]     # chọn hoán vị đổi được nhiều cạnh nhất\\n",\n    "                perm = max(perms, key=lambda pm: sum(old[i] != old[j] for i, j in enumerate(pm)))\\n",\n    "                new = [old[i] for i in perm]\\n",\n    "                fwd[rel] = dict(fwd[rel], pdist=new)\\n",\n    "                ms = lambda L: sorted(json.dumps(sorted(d.items())) for d in L)\\n",\n    "                self.shuffle_audit.append(dict(family=rel, n_edges=len(old),\\n",\n    "                                               frac_edges_predicates_changed=float(np.mean([a != b for a, b in zip(old, new)])) if old else 0.0,\\n",\n    "                                               predicate_multiset_equal=ms(old) == ms(new)))\\n",\n    "        self._fwd, self._related, self._isa = fwd, related, isa\\n",\n    "\\n",\n    "        # ---------- ma trận theo từng loại cạnh + bảng vị từ theo cặp ----------\\n",\n    "        A = {}\\n",\n    "        for rel in REL_CATS:\\n",\n    "            f = fwd[rel]\\n",\n    "            A[rel] = _csr(n, n, f[\\"src\\"], f[\\"dst\\"], f[\\"w\\"])\\n",\n    "            A[rel + \\"_inv\\"] = _csr(n, n, f[\\"dst\\"], f[\\"src\\"], f[\\"w_inv\\"])\\n",\n    "        rs, rd = related[\\"src\\"], related[\\"dst\\"]\\n",\n    "        A[\\"related\\"] = _csr(n, n, np.r_[rs, rd], np.r_[rd, rs], np.r_[related[\\"w_ab\\"], related[\\"w_ba\\"]])\\n",\n    "        A[\\"isa\\"] = _csr(n, n, isa[\\"src\\"], isa[\\"dst\\"], np.ones(len(isa[\\"src\\"])))\\n",\n    "        A[\\"isa_inv\\"] = A[\\"isa\\"].T.tocsr()\\n",\n    "        for t in self.masked:\\n",\n    "            if t != \\"contains\\" and t != \\"contains_inv\\":\\n",\n    "                A[t] = sp.csr_matrix((n, n), dtype=np.float32)\\n",\n    "        self.A = A\\n",\n    "        if \\"contains\\" in self.masked:\\n",\n    "            raise ValueError(\\"không tắt contains trong ablation cạnh quan hệ\\")\\n",\n    "        # pdist[(họ, a, b)] = {vị từ: số video}; pshare[họ][(a, b)] = {pred_id: tỉ phần} cho họ có vị từ tự do\\n",\n    "        self.pdist = {(rel, int(a), int(b)): d for rel in REL_CATS for a, b, d in zip(fwd[rel][\\"src\\"], fwd[rel][\\"dst\\"], fwd[rel][\\"pdist\\"])}\\n",\n    "        self.pshare = {}\\n",\n    "        for rel in PRED_FAMILIES:\\n",\n    "            sh = {}\\n",\n    "            for (r_, a, b), d in self.pdist.items():\\n",\n    "                if r_ == rel and A[rel][a, b] != 0:\\n",\n    "                    tot = float(sum(d.values()))\\n",\n    "                    sh[(a, b)] = {PRED_ID[(rel, p)]: c / tot for p, c in sorted(d.items()) if (rel, p) in PRED_ID}\\n",\n    "            self.pshare[rel] = sh\\n",\n    "\\n",\n    "        # ---------- IDF (taxonomy, chỉ TRAIN) — KHÔNG phụ thuộc mask/rewiring ----------\\n",\n    "        df_i = np.asarray(((Cb @ isa_true)[tr_rows] > 0).sum(0)).ravel()\\n",\n    "        df = np.where(np.arange(n) < N + inv.n_cls, cls_df, df_i)\\n",\n    "        idf = np.log((n_tr + 1.0) / (df + 1.0)).astype(np.float32)\\n",\n    "        idf[:N] = 0.0\\n",\n    "        self.idf = idf\\n",\n    "        self.PM = self.path_mats()\\n",\n    "        self.R = self.compute_R()\\n",\n    "\\n",\n    "    # ---------- đường đi ----------\\n",\n    "    def pred_expand(self, A, rel):\\n",\n    "        \\"\\"\\"X[u, pid·n + v] = A_t[u, v] · tỉ phần vị từ pid của cặp, cho t ∈ {rel, rel_inv} (đường đi khớp vị từ).\\"\\"\\"\\n",\n    "        n, rows, cols, vals = self.n_nodes, [], [], []\\n",\n    "        sh = self.pshare.get(rel, {})\\n",\n    "        for t, inv_dir in ((rel, False), (rel + \\"_inv\\", True)):\\n",\n    "            m = A[t].tocoo()\\n",\n    "            for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):\\n",\n    "                for pid, s in sh.get((v, u) if inv_dir else (u, v), {}).items():\\n",\n    "                    rows.append(u); cols.append(pid * n + v); vals.append(w * s)\\n",\n    "        return _csr(n, len(PRED_VOCAB) * n, rows, cols, vals)\\n",\n    "\\n",\n    "    def path_mats(self, A=None, A_con=None):\\n",\n    "        A = A or self.A\\n",\n    "        C = (A_con if A_con is not None else self.A_con).tocsr().astype(np.float32)\\n",\n    "        if self.path_mode == \\"count\\":\\n",\n    "            C = (C > 0).astype(np.float32)\\n",\n    "        PM = {\\"C\\": C}\\n",\n    "        for t in PATH_TYPES:\\n",\n    "            M = A[t].tocsr().astype(np.float32)\\n",\n    "            if self.path_mode == \\"count\\":\\n",\n    "                M = (M > 0).astype(np.float32)\\n",\n    "            elif self.path_mode == \\"degnorm\\":\\n",\n    "                M = _symnorm(M)\\n",\n    "            PM[t] = M\\n",\n    "        if self.path_mode == \\"pred_match\\":\\n",\n    "            for rel in PRED_FAMILIES:\\n",\n    "                PM[f\\"X_{rel}\\"] = self.pred_expand(A, rel)\\n",\n    "        return PM\\n",\n    "\\n",\n    "    def _fam(self, PM, f):\\n",\n    "        a, b = PM[FAMILY_REL[f]], PM[FAMILY_REL[f] + \\"_inv\\"]\\n",\n    "        return a.maximum(b) if self.path_mode == \\"dedup\\" else (a + b)\\n",\n    "\\n",\n    "    def compute_R(self, rows=None, PM=None):\\n",\n    "        PM = PM or self.PM\\n",\n    "        C = PM[\\"C\\"] if rows is None else PM[\\"C\\"][np.asarray(rows)]\\n",\n    "        I, Rl = PM[\\"isa\\"], PM[\\"related\\"]\\n",\n    "        prod = _maxprod if self.path_mode == \\"dedup\\" else (lambda X, Y: (X @ Y).tocsr())\\n",\n    "        R = {\\"P0_contains\\": C.tocsr()}\\n",\n    "        for f in FAMILIES:\\n",\n    "            CF = prod(C, self._fam(PM, f))\\n",\n    "            R[f\\"P1_{f}\\"] = CF\\n",\n    "            R[f\\"P4_{f}_isa\\"] = prod(CF, I)\\n",\n    "        R[\\"P2_isa\\"] = prod(C, I)\\n",\n    "        CR = prod(C, Rl)\\n",\n    "        R[\\"P3_related\\"] = CR\\n",\n    "        R[\\"P5_related_isa\\"] = prod(CR, I)\\n",\n    "        direct = (C > 0).astype(np.float32)\\n",\n    "        if CFG.drop_redundant_indirect:\\n",\n    "            for p in PATTERNS[1:]:\\n",\n    "                if p in R:\\n",\n    "                    R[p] = (R[p] - R[p].multiply(direct)).tocsr()\\n",\n    "        D = sp.diags(self.idf)\\n",\n    "        out = {}\\n",\n    "        for p in PATTERNS:\\n",\n    "            if p.startswith(\\"P1m_\\"):\\n",\n    "                continue\\n",\n    "            M = (R[p] @ D).tocsr(); M.eliminate_zeros(); out[p] = M\\n",\n    "        n, n_p = self.n_nodes, len(PRED_VOCAB)\\n",\n    "        for rel in PRED_FAMILIES:\\n",\n    "            key = f\\"X_{rel}\\"\\n",\n    "            if key not in PM:\\n",\n    "                out[f\\"P1m_{rel}\\"] = sp.csr_matrix((C.shape[0], n_p * n), dtype=np.float32)\\n",\n    "                continue\\n",\n    "            M = (C @ PM[key]).tocoo()\\n",\n    "            b = M.col % n\\n",\n    "            keep = np.ones(M.nnz, bool)\\n",\n    "            if CFG.drop_redundant_indirect and M.nnz:\\n",\n    "                keep = np.asarray(direct[M.row, b]).ravel() == 0\\n",\n    "            X = sp.csr_matrix(((M.data * self.idf[b])[keep], (M.row[keep], M.col[keep])), shape=M.shape, dtype=np.float32)\\n",\n    "            X.eliminate_zeros()\\n",\n    "            out[f\\"P1m_{rel}\\"] = X\\n",\n    "        return out\\n",\n    "\\n",\n    "    # ---------- cạnh cho GNN (có loại, có trọng số, có chỉ số vị từ) ----------\\n",\n    "    def edge_arrays(self):\\n",\n    "        \\"\\"\\"(src, dst, type, weight, pidx): pidx = hàng trong self.pmix (phân phối vị từ của cạnh) hoặc −1.\\"\\"\\"\\n",\n    "        if not hasattr(self, \\"_edges\\"):\\n",\n    "            parts, mix_rows = [], []\\n",\n    "            c = self.A_con.tocoo()\\n",\n    "            parts.append((c.row, c.col, np.full(c.nnz, ETYPE_ID[\\"contains\\"]), c.data, np.full(c.nnz, -1)))\\n",\n    "            parts.append((c.col, c.row, np.full(c.nnz, ETYPE_ID[\\"contains_inv\\"]), c.data, np.full(c.nnz, -1)))\\n",\n    "            for t in RELATION_TYPES:\\n",\n    "                m = self.A[t].tocoo()\\n",\n    "                if not m.nnz:\\n",\n    "                    continue\\n",\n    "                pidx = np.full(m.nnz, -1)\\n",\n    "                if t in PRED_TYPES:\\n",\n    "                    base, inv_dir = (t[:-4], True) if t.endswith(\\"_inv\\") else (t, False)\\n",\n    "                    for k, (u, v) in enumerate(zip(m.row.tolist(), m.col.tolist())):\\n",\n    "                        d = self.pshare[base].get((v, u) if inv_dir else (u, v), {})\\n",\n    "                        if d:\\n",\n    "                            pidx[k] = len(mix_rows); mix_rows.append(d)\\n",\n    "                parts.append((m.row, m.col, np.full(m.nnz, ETYPE_ID[t]), m.data, pidx))\\n",\n    "            s, d_, e, w, pi = (np.concatenate([p[i] for p in parts]) for i in range(5))\\n",\n    "            r_, c_, v_ = [], [], []\\n",\n    "            for i, dd in enumerate(mix_rows):\\n",\n    "                for pid, sh in dd.items():\\n",\n    "                    r_.append(i); c_.append(pid); v_.append(sh)\\n",\n    "            self.pmix = _csr(len(mix_rows), len(PRED_VOCAB), r_, c_, v_)\\n",\n    "            self._edges = (s.astype(np.int64), d_.astype(np.int64), e.astype(np.int64), w.astype(np.float32), pi.astype(np.int64))\\n",\n    "        return self._edges\\n",\n    "\\n",\n    "    def edge_counts(self):\\n",\n    "        out = {\\"contains\\": int(self.A_con.nnz), \\"contains_inv\\": int(self.A_con.nnz)}\\n",\n    "        out.update({t: int(self.A[t].nnz) for t in RELATION_TYPES})\\n",\n    "        return out\\n",\n    "\\n",\n    "    def predicates_of(self, rel, a, b):\\n",\n    "        \\"\\"\\"{vị từ: số video} của cạnh (rel, a → b) THEO ĐÚNG CHIỀU lưu trong đồ thị (đã tính rewiring / hoán vị vị từ).\\"\\"\\"\\n",\n    "        base = rel[:-4] if rel.endswith(\\"_inv\\") else rel\\n",\n    "        return self.pdist.get((base, b, a) if rel.endswith(\\"_inv\\") else (base, a, b), {})\\n",\n    "\\n",\n    "    def predicate(self, rel, a, b):\\n",\n    "        d = self.predicates_of(rel, a, b)\\n",\n    "        return max(sorted(d), key=lambda p: d[p]) if d else \\"\\"\\n",\n    "\\n",\n    "    def summary(self):\\n",\n    "        return dict(name=self.name, nodes=self.n_nodes, videos=self.n_vid, classes=self.inv.n_cls, concepts=self.inv.n_con,\\n",\n    "                    active_classes=len(self.active), edges=self.edge_counts(), contains=self.contains_mode,\\n",\n    "                    path_mode=self.path_mode, rel_policy=self.rel_policy, related_cfg=list(self.related_cfg),\\n",\n    "                    masked=list(self.masked), rewired=self.rewire_seed is not None,\\n",\n    "                    predicate_shuffled=self.pred_shuffle_seed is not None, excluded=len(self.excluded),\\n",\n    "                    paths={p: int(self.R[p].nnz) for p in PATTERNS})"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c031",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def rewire_directed(E, inv, rng, attrs, rel):\\n",\n    "    \\"\\"\\"Directed double-edge swap TRONG một loại quan hệ: (a→b),(c→d) ⇒ (a→d),(c→b) khi a,c cùng kiểu và b,d cùng\\n",\n    "    kiểu (semantic type: noun / verb / noun_concept / verb_concept). Giữ out-degree, in-degree, kiểu đầu/cuối,\\n",\n    "    số cạnh; cấm self-loop và cạnh trùng. Thuộc tính (trọng số, support, phân phối vị từ) gắn với cạnh theo NGUỒN.\\"\\"\\"\\n",\n    "    src, dst = E[\\"src\\"].copy(), E[\\"dst\\"].copy()\\n",\n    "    m = len(src)\\n",\n    "    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)\\n",\n    "    if m < 2:\\n",\n    "        st.update(frac_changed=0.0, success_rate=0.0)\\n",\n    "        return dict(E, src=src, dst=dst), st\\n",\n    "    key = np.array([f\\"{inv.semantic_type(a)}>{inv.semantic_type(b)}\\" for a, b in zip(src, dst)])\\n",\n    "    edges = set(zip(src.tolist(), dst.tolist()))\\n",\n    "    for g in np.unique(key):\\n",\n    "        I = np.where(key == g)[0]\\n",\n    "        if len(I) < 2:\\n",\n    "            continue\\n",\n    "        for _ in range(CFG.rewire_swaps_per_edge * len(I)):\\n",\n    "            i, j = rng.choice(I, 2, replace=False)\\n",\n    "            a, b, c, d = src[i], dst[i], src[j], dst[j]\\n",\n    "            st[\\"attempts\\"] += 1\\n",\n    "            if a == d or c == b:\\n",\n    "                st[\\"fail_self\\"] += 1; continue\\n",\n    "            if (a, d) in edges or (c, b) in edges:\\n",\n    "                st[\\"fail_dup\\"] += 1; continue\\n",\n    "            edges -= {(a, b), (c, d)}; edges |= {(a, d), (c, b)}\\n",\n    "            dst[i], dst[j] = d, b\\n",\n    "            st[\\"swaps\\"] += 1\\n",\n    "    orig = set(zip(E[\\"src\\"].tolist(), E[\\"dst\\"].tolist()))\\n",\n    "    st[\\"frac_changed\\"] = 1 - len(orig & set(zip(src.tolist(), dst.tolist()))) / m\\n",\n    "    st[\\"success_rate\\"] = st[\\"swaps\\"] / max(st[\\"attempts\\"], 1)\\n",\n    "    return dict(E, src=src, dst=dst), st\\n",\n    "\\n",\n    "\\n",\n    "def rewire_undirected(E, inv, rng, rel):\\n",\n    "    \\"\\"\\"Double-edge swap cho quan hệ ĐỐI XỨNG (related): giữ bậc vô hướng, cặp kiểu, không self-loop/trùng.\\n",\n    "    Cạnh lưu (a, b) có kiểu(a) ≤ kiểu(b); số đếm đồng xuất hiện gắn với cạnh.\\"\\"\\"\\n",\n    "    src, dst = E[\\"src\\"].copy(), E[\\"dst\\"].copy()\\n",\n    "    wab, wba = E[\\"w_ab\\"].copy(), E[\\"w_ba\\"].copy()            # P(b|a), P(a|b) gắn với cạnh, đổi chỗ khi lật hướng\\n",\n    "    m = len(src)\\n",\n    "    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)\\n",\n    "    if m < 2:\\n",\n    "        st.update(frac_changed=0.0, success_rate=0.0)\\n",\n    "        return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st\\n",\n    "    T = lambda x: inv.semantic_type(x)\\n",\n    "\\n",\n    "    def flip(k):\\n",\n    "        src[k], dst[k] = dst[k], src[k]\\n",\n    "        wab[k], wba[k] = wba[k], wab[k]\\n",\n    "\\n",\n    "    for k in range(m):                                  # định hướng theo kiểu để swap giữ cặp kiểu\\n",\n    "        if T(src[k]) > T(dst[k]):\\n",\n    "            flip(k)\\n",\n    "    key = np.array([f\\"{T(a)}~{T(b)}\\" for a, b in zip(src, dst)])\\n",\n    "    edges = {(min(a, b), max(a, b)) for a, b in zip(src.tolist(), dst.tolist())}\\n",\n    "    for g in np.unique(key):\\n",\n    "        I = np.where(key == g)[0]\\n",\n    "        if len(I) < 2:\\n",\n    "            continue\\n",\n    "        same_type = g.split(\\"~\\")[0] == g.split(\\"~\\")[1]\\n",\n    "        for _ in range(CFG.rewire_swaps_per_edge * len(I)):\\n",\n    "            i, j = rng.choice(I, 2, replace=False)\\n",\n    "            if same_type and rng.rand() < 0.5:\\n",\n    "                flip(j)                                 # cùng kiểu ⇒ cho phép cả hai cách ghép\\n",\n    "            a, b, c, d = src[i], dst[i], src[j], dst[j]\\n",\n    "            st[\\"attempts\\"] += 1\\n",\n    "            n1, n2 = (min(a, d), max(a, d)), (min(c, b), max(c, b))\\n",\n    "            if a == d or c == b or n1 == n2:\\n",\n    "                st[\\"fail_self\\"] += 1; continue\\n",\n    "            if n1 in edges or n2 in edges:\\n",\n    "                st[\\"fail_dup\\"] += 1; continue\\n",\n    "            edges -= {(min(a, b), max(a, b)), (min(c, d), max(c, d))}; edges |= {n1, n2}\\n",\n    "            dst[i], dst[j] = d, b                       # (a, d) và (c, b); trọng số theo nguồn a, c\\n",\n    "            st[\\"swaps\\"] += 1\\n",\n    "    orig = {(min(a, b), max(a, b)) for a, b in zip(E[\\"src\\"].tolist(), E[\\"dst\\"].tolist())}\\n",\n    "    st[\\"frac_changed\\"] = 1 - len(orig & edges) / m\\n",\n    "    st[\\"success_rate\\"] = st[\\"swaps\\"] / max(st[\\"attempts\\"], 1)\\n",\n    "    return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st\\n",\n    "\\n",\n    "\\n",\n    "def _pred_multiset(g, rel):\\n",\n    "    \\"\\"\\"Đa tập phân phối vị từ của các cạnh thuộc họ rel (chỉ cạnh còn trong đồ thị).\\"\\"\\"\\n",\n    "    if rel not in REL_CATS:\\n",\n    "        return []\\n",\n    "    return sorted(json.dumps(sorted(d.items())) for (r_, a, b), d in g.pdist.items() if r_ == rel and g.A[rel][a, b] != 0)\\n",\n    "\\n",\n    "\\n",\n    "def randomization_audit(g0, g1):\\n",\n    "    \\"\\"\\"So sánh đồ thị gốc và đồ thị rewired theo từng loại cạnh; áp tiêu chí chấp nhận đăng ký trước.\\"\\"\\"\\n",\n    "    inv, rows = g0.inv, []\\n",\n    "    stats = {s[\\"relation\\"]: s for s in g1.rewire_audit}\\n",\n    "    for t in [\\"contains\\"] + RELATION_TYPES:\\n",\n    "        M0 = g0.A_con if t == \\"contains\\" else g0.A[t]\\n",\n    "        M1 = g1.A_con if t == \\"contains\\" else g1.A[t]\\n",\n    "        c0, c1 = M0.tocoo(), M1.tocoo()\\n",\n    "        e0, e1 = set(zip(c0.row.tolist(), c0.col.tolist())), set(zip(c1.row.tolist(), c1.col.tolist()))\\n",\n    "        tp = lambda c: Counter(f\\"{inv.semantic_type(a) if t != \'contains\' else \'video\'}>{inv.semantic_type(b)}\\"\\n",\n    "                               for a, b in zip(c.row, c.col))\\n",\n    "        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n",\n    "        s = stats.get(base, {})\\n",\n    "        row = dict(relation=t, edges_orig=len(e0), edges_rand=len(e1),\\n",\n    "                   out_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(1)).ravel(), np.asarray((M1 > 0).sum(1)).ravel())),\\n",\n    "                   in_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(0)).ravel(), np.asarray((M1 > 0).sum(0)).ravel())),\\n",\n    "                   type_pairs_equal=tp(c0) == tp(c1), weights_multiset_equal=bool(np.allclose(np.sort(c0.data), np.sort(c1.data))),\\n",\n    "                   self_loops=int(sum(a == b for a, b in e1)), duplicates=int(c1.nnz - len(e1)),\\n",\n    "                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),\\n",\n    "                   frac_changed=1 - len(e0 & e1) / max(len(e0), 1), swaps=s.get(\\"swaps\\"), attempts=s.get(\\"attempts\\"),\\n",\n    "                   success_rate=s.get(\\"success_rate\\"), fail_dup=s.get(\\"fail_dup\\"), fail_self=s.get(\\"fail_self\\"))\\n",\n    "        n_s, n_t = len({a for a, _ in e0}), len({b for _, b in e0})\\n",\n    "        row[\\"density\\"] = len(e0) / max(n_s * n_t, 1)      # ≈ 1 ⇒ gần như KHÔNG có cách sắp xếp khác giữ nguyên bậc\\n",\n    "        must_change = t != \\"contains\\" and len(e0) >= 10\\n",\n    "        row[\\"accepted\\"] = bool(row[\\"edges_orig\\"] == row[\\"edges_rand\\"] and row[\\"out_degree_equal\\"] and row[\\"in_degree_equal\\"]\\n",\n    "                               and row[\\"type_pairs_equal\\"] and row[\\"weights_multiset_equal\\"] and row[\\"self_loops\\"] == 0\\n",\n    "                               and row[\\"duplicates\\"] == 0 and row[\\"predicates_multiset_equal\\"]\\n",\n    "                               and (row[\\"frac_changed\\"] >= CFG.rewire_min_changed or not must_change)\\n",\n    "                               and (t != \\"contains\\" or row[\\"frac_changed\\"] == 0))\\n",\n    "        row[\\"note\\"] = (\\"contains phải giữ nguyên\\" if t == \\"contains\\" else \\"< 10 cạnh: không yêu cầu đổi\\" if not must_change\\n",\n    "                       else (\\"đổi < ngưỡng: đồ thị quá đặc (mật độ cao) — không có đủ cách sắp xếp giữ bậc\\"\\n",\n    "                             if not row[\\"accepted\\"] and row[\\"density\\"] > 0.5 else \\"\\"))\\n",\n    "        rows.append(row)\\n",\n    "    df = pd.DataFrame(rows)\\n",\n    "    return df, bool(df.accepted.all())\\n",\n    "\\n",\n    "\\n",\n    "def predicate_shuffle_audit(g0, g1):\\n",\n    "    \\"\\"\\"Đối chứng hoán vị vị từ: topology, trọng số, bậc của MỌI loại cạnh phải giống hệt; chỉ phân phối vị từ của cạnh\\n",\n    "    thuộc họ có vị từ tự do được đổi chỗ (đa tập giữ nguyên).\\"\\"\\"\\n",\n    "    rows = []\\n",\n    "    stats = {s[\\"family\\"]: s for s in g1.shuffle_audit}\\n",\n    "    for t in [\\"contains\\"] + RELATION_TYPES:\\n",\n    "        M0 = g0.A_con if t == \\"contains\\" else g0.A[t]\\n",\n    "        M1 = g1.A_con if t == \\"contains\\" else g1.A[t]\\n",\n    "        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n",\n    "        same_topology = (M0 != M1).nnz == 0\\n",\n    "        moved = 0\\n",\n    "        if base in PRED_FAMILIES:\\n",\n    "            c = M0.tocoo()\\n",\n    "            for u, v in zip(c.row.tolist(), c.col.tolist()):\\n",\n    "                moved += int(g0.predicates_of(t, u, v) != g1.predicates_of(t, u, v))\\n",\n    "        st = stats.get(base, {})\\n",\n    "        row = dict(relation=t, n_edges=int(M0.nnz), topology_and_weights_equal=bool(same_topology),\\n",\n    "                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),\\n",\n    "                   edges_with_changed_predicates=moved, frac_edges_predicates_changed=moved / max(int(M0.nnz), 1),\\n",\n    "                   shuffled=base in PRED_FAMILIES, family_frac_changed=st.get(\\"frac_edges_predicates_changed\\"))\\n",\n    "        row[\\"accepted\\"] = bool(same_topology and row[\\"predicates_multiset_equal\\"]\\n",\n    "                               and (moved == 0 if base not in PRED_FAMILIES else (M0.nnz < 10 or row[\\"frac_edges_predicates_changed\\"] >= CFG.rewire_min_changed)))\\n",\n    "        rows.append(row)\\n",\n    "    df = pd.DataFrame(rows)\\n",\n    "    return df, bool(df.accepted.all())"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c032",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def query_tokens_v2(text):\\n",\n    "    \\"\\"\\"[(POS dự đoán, lemma danh từ, lemma động từ)] — POS tagger trên CHÍNH câu truy vấn (input của hệ thống).\\"\\"\\"\\n",\n    "    toks = re.findall(r\\"[a-z]+\\", str(text).lower())\\n",\n    "    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n",\n    "    out = []\\n",\n    "    for w, t in tags:\\n",\n    "        if w in STOP or len(w) < 2:\\n",\n    "            continue\\n",\n    "        p = \\"v\\" if t.startswith(\\"VB\\") else (\\"n\\" if t.startswith(\\"NN\\") else \\"x\\")\\n",\n    "        out.append((p, lemma(w, \\"n\\"), lemma(w, \\"v\\")))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def parse_tokens_v2(toks, inv):\\n",\n    "    \\"\\"\\"Khớp token với node: lớp đúng POS trước (danh từ → \'x#n\', động từ → \'x#v\'), không có thì POS còn lại,\\n",\n    "    không có lớp thì khái niệm taxonomy cùng lemma + POS. Trả về [(node_id, tên node, token)].\\"\\"\\"\\n",\n    "    hits, seen = [], set()\\n",\n    "    for p, ln, lv in toks:\\n",\n    "        order = [(\\"v\\", lv), (\\"n\\", ln)] if p == \\"v\\" else [(\\"n\\", ln), (\\"v\\", lv)]\\n",\n    "        found = None\\n",\n    "        for pos, lm in order:\\n",\n    "            i = inv.key2id.get(ckey(lm, pos))\\n",\n    "            if i is not None:\\n",\n    "                found = [i]; break\\n",\n    "        if found is None:\\n",\n    "            for pos, lm in order:\\n",\n    "                if inv.concepts_by_lemma.get((lm, pos)):\\n",\n    "                    found = inv.concepts_by_lemma[(lm, pos)]; break\\n",\n    "        for i in found or []:\\n",\n    "            if i not in seen:\\n",\n    "                seen.add(i); hits.append((i, inv.name[i], lv if p == \\"v\\" else ln))\\n",\n    "    return hits\\n",\n    "\\n",\n    "\\n",\n    "def query_pred_tokens(text):\\n",\n    "    \\"\\"\\"Tập token của truy vấn để khớp VỊ TỪ: từ gốc + lemma động từ (giới từ giữ nguyên; KHÔNG bỏ stop-word).\\"\\"\\"\\n",\n    "    w = re.findall(r\\"[a-z]+\\", str(text).lower())\\n",\n    "    return frozenset(w) | frozenset(lemma(x, \\"v\\") for x in w)\\n",\n    "\\n",\n    "\\n",\n    "_PRED_BY_FIRST = {}\\n",\n    "\\n",\n    "\\n",\n    "def match_predicates(qset, rel):\\n",\n    "    \\"\\"\\"pred_id của họ rel có MỌI token nằm trong truy vấn (\'ride on\' khớp \'a man is riding on a horse\').\\"\\"\\"\\n",\n    "    if rel not in _PRED_BY_FIRST:\\n",\n    "        idx = defaultdict(list)\\n",\n    "        for i, (r, p) in enumerate(PRED_VOCAB):\\n",\n    "            if r == rel:\\n",\n    "                t = predicate_tokens(p)\\n",\n    "                idx[t[0]].append((i, t))\\n",\n    "        _PRED_BY_FIRST[rel] = idx\\n",\n    "    idx = _PRED_BY_FIRST[rel]\\n",\n    "    return sorted({i for tok in qset for i, t in idx.get(tok, ()) if all(x in qset for x in t)})"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c033",\n   "metadata": {},\n   "source": [\n    "## 6. Mô hình v3\\n",\n    "\\n",\n    "**`TypedSAGEv3`.** Công thức là `h\'_v = W₀h_v + Σ_t σ(g_t)·[W_t·AGG_t(h_u) + U_t·AGG_t(e_uv)]`.\\n",\n    "* `e_uv` là biểu diễn vị từ của cạnh. Nó chỉ có ở các loại cạnh svo, svo_inv, prep, prep_inv, và chỉ khi `pred_mode ≠ none`.\\n",\n    "* `AGG_t` dùng **cùng hệ số** cho `h_u` và `e_uv`, nên trọng số cạnh chỉ nhân một lần.\\n",\n    "* Vị từ của agent_of/object_of chính là node động từ đích, nên không đưa vào message, tránh trùng thông tin.\\n",\n    "\\n",\n    "**`pred_mode`:**\\n",\n    "* `none` = relation-family-only, đúng mô hình v2;\\n",\n    "* `clip` (phương án A) = CLIP-text **đóng băng** của vị từ theo prompt, qua một phép chiếu tuyến tính;\\n",\n    "* `clip_adapt` (phương án B) = A cộng MLP dư **dùng chung** cho mọi vị từ, lớp cuối khởi tạo 0. Không có ID riêng cho từng vị từ, nên vị từ hiếm hoặc chưa gặp vẫn mã hoá được.\\n",\n    "\\n",\n    "`mp_weight` (`binary` / `weighted_mean` / `weighted_symnorm`) và `init` như v2.\\n",\n    "### 6b. Mô hình v4 — QC-RGAT (GNN quan hệ có attention điều kiện theo truy vấn)\\n",\n    "\\n",\n    "**Vì sao đổi mô hình.** Ở v3, truy vấn chỉ đi vào GNN dưới dạng **một vector CLIP toàn cục**. Cạnh quan hệ nối lớp với lớp và là thống kê **dùng chung** cho mọi video, nên mọi video có cùng nhãn đều nhận cùng \\"ngữ cảnh quan hệ\\". Vì vậy quan hệ chỉ làm mượt embedding lớp, và kết quả Kaggle v3 cho G3 ≈ G2 ≈ G4 ≈ G5. Ở v4, **chính truy vấn quyết định cạnh quan hệ nào được truyền tin**.\\n",\n    "\\n",\n    "**Công thức** (T = 1 hoặc 2 lớp; chọn trên VAL):\\n",\n    "* `H_q⁽⁰⁾(c) = E_c` và `H_q⁽ᵗ⁺¹⁾(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q⁽ᵗ⁾(u)`. Ở đây `z_{t,τ}` là toán tử của loại quan hệ τ, phép nhân theo chiều kiểu CompGCN/DistMult.\\n",\n    "* `a_q = σ(β_τ + κ_τ·log w + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ\'_τ·cos(e_p, e_P(q)))` gồm năm thành phần:\\n",\n    "  * cổng theo loại quan hệ do truy vấn chọn (`W_g u_q`);\\n",\n    "  * **vai** của node nguồn trong truy vấn (chủ thể / động từ / đối tượng / tân ngữ giới từ, do parser caption áp lên chính truy vấn);\\n",\n    "  * **khớp vị từ** giữa cạnh và truy vấn, cả khớp đúng lẫn khớp mềm bằng CLIP-text;\\n",\n    "  * support của cạnh (`κ_τ·log w`);\\n",\n    "  * độ lệch `β_τ`.\\n",\n    "* `s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v,c] · H_q⁽ᵀ⁾(c)⟩`. Video **chỉ nhận** qua nhãn của chính nó (inductive). Điểm cuối = `α·z(cos CLIP) + (1−α)·z(s)`, với α chọn trên VAL.\\n",\n    "\\n",\n    "**Khởi tạo và học các cổng.** γ_τ khởi tạo bằng 1, tức prior \\"ưu tiên cạnh có vị từ khớp truy vấn\\"; dữ liệu có thể làm γ giảm. Các cổng vô hướng (β, κ, ρ, γ, γ\') học với lr × `scalar_lr_mult`, như cổng của v3. Nếu khởi tạo γ = 0 và học với lr thường, γ gần như không đổi sau huấn luyện. Khi đó G2, G4, G6 cho kết quả trùng G3, và các đối chứng không còn đo được gì; lần thử dev đã gặp đúng trường hợp này.\\n",\n    "\\n",\n    "**Giải thích từ chính cơ chế.** Attention chỉ phụ thuộc truy vấn và thuộc tính tĩnh của cạnh, nên phép truyền là **tuyến tính theo E**. Vì vậy `s` tách **chính xác** thành tổng các số hạng: thị giác + từng nhãn contains + từng **đường đi quan hệ** (`u –τ[p]→ c ←chứa– v`, có thêm một bước khi T = 2). Giải thích là top-m số hạng quan hệ. Xoá một cạnh làm điểm đổi **đúng bằng** tổng đóng góp của các đường đi qua cạnh đó (T = 1); điều này có kiểm thử Q3 và X3.\\n",\n    "\\n",\n    "**Đối chứng tách từng thành phần** (cùng mô hình, cùng giao thức huấn luyện):\\n",\n    "* G1 `v4_rel=False`: không cạnh quan hệ;\\n",\n    "* G2 `v4_pred=False`: không khớp vị từ;\\n",\n    "* G4: hoán vị vị từ giữa các cạnh cùng họ;\\n",\n    "* G5: rewiring có ràng buộc;\\n",\n    "* G6 `v4_qcond=False`: attention tĩnh, không theo truy vấn — tức là quan hệ như v3;\\n",\n    "* R3: chính mô hình v3 cuối.\\n",\n    "\\n",\n    "Liên hệ tài liệu: NBFNet (Zhu et al., NeurIPS 2021) với message điều kiện theo truy vấn; KGAT (Wang et al., KDD 2019) với attention nhận biết quan hệ; CompGCN (Vashishth et al., ICLR 2020) với toán tử quan hệ. Thuộc họ **GAT quan hệ** của cấp độ 1.\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c034",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def scatter_sum(src, index, n):\\n",\n    "    out = torch.zeros((n,) + tuple(src.shape[1:]), device=src.device, dtype=src.dtype)\\n",\n    "    return out.index_add_(0, index, src)\\n",\n    "\\n",\n    "\\n",\n    "PRED_MODES = (\\"none\\", \\"clip\\", \\"clip_adapt\\")\\n",\n    "\\n",\n    "\\n",\n    "class TypedSAGEv3(nn.Module):\\n",\n    "    \\"\\"\\"h\'_v = W₀ h_v + Σ_t σ(g_t) · [ W_t · AGG_t(h_u) + U_t · AGG_t(e_uv) ]\\n",\n    "    e_uv = embedding vị từ của cạnh (chỉ loại cạnh thuộc họ có vị từ tự do, khi pred_mode ≠ \'none\'); AGG_t dùng CÙNG hệ số\\n",\n    "    chuẩn hoá cho h_u và e_uv ⇒ message(u → v) = W_t h_u + U_t e_uv đúng như thiết kế, trọng số cạnh chỉ nhân MỘT lần.\\n",\n    "    AGG theo mp_weight: \'binary\' (trung bình, bỏ trọng số) | \'weighted_mean\' (Σ w·x / Σ w) | \'weighted_symnorm\' (Σ w·x / √(d_u d_v)).\\n",\n    "    Nhánh W₀ h_v giữ biểu diễn của node không có hàng xóm (không sinh NaN).\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, din, dout, n_types, open_types, gate_init, mp_weight, pred_type_ids=()):\\n",\n    "        super().__init__()\\n",\n    "        assert mp_weight in (\\"binary\\", \\"weighted_mean\\", \\"weighted_symnorm\\"), mp_weight\\n",\n    "        self.mp_weight, self.n_types = mp_weight, n_types\\n",\n    "        self.self_w = nn.Linear(din, dout)\\n",\n    "        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(n_types)])\\n",\n    "        self.pred_type_ids = list(pred_type_ids)\\n",\n    "        self.pred_w = nn.ModuleDict({str(t): nn.Linear(din, dout, bias=False) for t in self.pred_type_ids})\\n",\n    "        g = torch.full((n_types,), float(gate_init))\\n",\n    "        for t in open_types:\\n",\n    "            g[t] = 3.0\\n",\n    "        self.gate = nn.Parameter(g)\\n",\n    "\\n",\n    "    def coef(self, src, dst, w, n):\\n",\n    "        \\"\\"\\"Hệ số của từng cạnh trong AGG (cùng cho h_u và e_uv).\\"\\"\\"\\n",\n    "        if self.mp_weight == \\"binary\\":\\n",\n    "            w = torch.ones_like(w)\\n",\n    "        if self.mp_weight in (\\"binary\\", \\"weighted_mean\\"):\\n",\n    "            return w / scatter_sum(w, dst, n)[dst].clamp(min=1e-12)\\n",\n    "        d_out, d_in = scatter_sum(w, src, n), scatter_sum(w, dst, n)\\n",\n    "        return w / torch.sqrt(d_out[src] * d_in[dst]).clamp(min=1e-12)\\n",\n    "\\n",\n    "    def forward(self, x, edge_index, edge_type, edge_weight, edge_pidx=None, pred_h=None):\\n",\n    "        src, dst = edge_index\\n",\n    "        n = x.size(0)\\n",\n    "        out = self.self_w(x)\\n",\n    "        for t in range(self.n_types):\\n",\n    "            m = edge_type == t\\n",\n    "            if not bool(m.any()):\\n",\n    "                continue\\n",\n    "            s, d = src[m], dst[m]\\n",\n    "            c = self.coef(s, d, edge_weight[m], n).unsqueeze(-1)\\n",\n    "            msg = self.neigh_w[t](scatter_sum(x[s] * c, d, n))\\n",\n    "            if pred_h is not None and t in self.pred_type_ids:\\n",\n    "                pi = edge_pidx[m]\\n",\n    "                has = pi >= 0\\n",\n    "                if bool(has.any()):\\n",\n    "                    msg = msg + self.pred_w[str(t)](scatter_sum(pred_h[pi[has]] * c[has], d[has], n))\\n",\n    "            out = out + torch.sigmoid(self.gate[t]) * msg\\n",\n    "        return out\\n",\n    "\\n",\n    "\\n",\n    "class PredicateEncoder(nn.Module):\\n",\n    "    \\"\\"\\"Biểu diễn vị từ của cạnh = Σ_p tỉ phần(p) · φ(e_p), e_p = CLIP-text ĐÓNG BĂNG của vị từ theo prompt.\\n",\n    "      \'clip\'       (phương án A): φ = phép chiếu tuyến tính học được (về chiều GNN)\\n",\n    "      \'clip_adapt\' (phương án B): φ(e) = chiếu(e + MLP(e)); MLP DÙNG CHUNG cho mọi vị từ, lớp cuối khởi tạo 0\\n",\n    "                                  ⇒ không có tham số riêng cho vị từ hiếm; vị từ mới (ngoài từ vựng) vẫn mã hoá được.\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, clip_dim, dout, mode, hidden):\\n",\n    "        super().__init__()\\n",\n    "        self.mode = mode\\n",\n    "        self.proj = nn.Linear(clip_dim, dout)\\n",\n    "        if mode == \\"clip_adapt\\":\\n",\n    "            self.res = nn.Sequential(nn.Linear(clip_dim, hidden), nn.GELU(), nn.Linear(hidden, clip_dim))\\n",\n    "            nn.init.zeros_(self.res[2].weight); nn.init.zeros_(self.res[2].bias)\\n",\n    "\\n",\n    "    def forward(self, pmix, pemb):\\n",\n    "        e = pemb + self.res(pemb) if self.mode == \\"clip_adapt\\" else pemb\\n",\n    "        return torch.sparse.mm(pmix, self.proj(e)) if pmix.is_sparse else pmix @ self.proj(e)\\n",\n    "\\n",\n    "\\n",\n    "class GraphScorerV3(nn.Module):\\n",\n    "    \\"\\"\\"Như GraphScorer v2 (12 mẫu đường đi + 2 mẫu khớp vị từ; khởi tạo node \'learned\' | \'clip_frozen\' | \'clip_residual\')\\n",\n    "    cộng nhánh vị từ (pred_mode). pred_mode = \'none\' ⇒ đúng mô hình v2 (đối chứng relation-family-only).\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, n_nodes, clip_dim, vid_feat, sym_feat, pred_emb=None, cfg=CFG, mp_weight=\\"binary\\", init=\\"learned\\",\\n",\n    "                 pred_mode=\\"none\\"):\\n",\n    "        super().__init__()\\n",\n    "        assert pred_mode in PRED_MODES, pred_mode\\n",\n    "        self.n_vid, self.init, self.pred_mode = vid_feat.shape[0], init, pred_mode\\n",\n    "        self.register_buffer(\\"vid_feat\\", vid_feat.clone())\\n",\n    "        self.vid_proj = nn.Linear(clip_dim, cfg.gnn_dim)\\n",\n    "        n_sym = n_nodes - self.n_vid\\n",\n    "        if init == \\"learned\\":\\n",\n    "            self.sym_emb = nn.Embedding(n_sym, cfg.gnn_dim)\\n",\n    "            nn.init.normal_(self.sym_emb.weight, std=0.02)\\n",\n    "        else:\\n",\n    "            self.register_buffer(\\"sym_feat\\", torch.as_tensor(sym_feat, dtype=torch.float32).clone())\\n",\n    "            self.sym_proj = nn.Linear(sym_feat.shape[1], cfg.gnn_dim)\\n",\n    "            if init == \\"clip_residual\\":\\n",\n    "                self.sym_res = nn.Embedding(n_sym, cfg.gnn_dim)\\n",\n    "                nn.init.zeros_(self.sym_res.weight)\\n",\n    "        pred_ids = [ETYPE_ID[t] for t in PRED_TYPES] if pred_mode != \\"none\\" else []\\n",\n    "        if pred_mode != \\"none\\":\\n",\n    "            self.register_buffer(\\"pred_emb\\", torch.as_tensor(pred_emb, dtype=torch.float32).clone())\\n",\n    "            self.pred_enc = PredicateEncoder(self.pred_emb.shape[1], cfg.gnn_dim, pred_mode, cfg.pred_adapt_hidden)\\n",\n    "        open_ids = [ETYPE_ID[t] for t in OPEN_TYPES]\\n",\n    "        self.layers = nn.ModuleList([TypedSAGEv3(cfg.gnn_dim, cfg.gnn_dim, len(EDGE_TYPES), open_ids, cfg.gate_init, mp_weight, pred_ids)\\n",\n    "                                     for _ in range(cfg.gnn_layers)])\\n",\n    "        self.norms = nn.ModuleList([nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])\\n",\n    "        self.drop = nn.Dropout(cfg.gnn_dropout)\\n",\n    "        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(), nn.Linear(cfg.gnn_dim, cfg.gnn_dim))\\n",\n    "        init_w = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init_w[0] = 0.0\\n",\n    "        self.path_w = nn.Parameter(init_w)\\n",\n    "        self.decay = nn.Parameter(torch.tensor(0.5))\\n",\n    "        self.register_buffer(\\"path_gain\\", torch.tensor(1.0))\\n",\n    "        self.clip_logit = nn.Parameter(torch.tensor(0.5413))\\n",\n    "\\n",\n    "    def node_features(self):\\n",\n    "        if self.init == \\"learned\\":\\n",\n    "            sym = self.sym_emb.weight\\n",\n    "        else:\\n",\n    "            sym = self.sym_proj(self.sym_feat)\\n",\n    "            if self.init == \\"clip_residual\\":\\n",\n    "                sym = sym + self.sym_res.weight\\n",\n    "        return torch.cat([self.vid_proj(self.vid_feat), sym], 0)\\n",\n    "\\n",\n    "    def edge_gates(self):\\n",\n    "        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers]\\n",\n    "\\n",\n    "    def predicate_features(self, pmix):\\n",\n    "        return self.pred_enc(pmix, self.pred_emb) if self.pred_mode != \\"none\\" and pmix is not None else None\\n",\n    "\\n",\n    "    def encode_graph(self, edge_index, edge_type, edge_weight, edge_pidx=None, pmix=None):\\n",\n    "        h = self.node_features()\\n",\n    "        ph = self.predicate_features(pmix)\\n",\n    "        for i, (l, nrm) in enumerate(zip(self.layers, self.norms)):\\n",\n    "            h = nrm(l(h, edge_index, edge_type, edge_weight, edge_pidx, ph))\\n",\n    "            if i < len(self.layers) - 1:\\n",\n    "                h = self.drop(F.gelu(h))\\n",\n    "        return F.normalize(h, dim=-1)\\n",\n    "\\n",\n    "    def pattern_weights(self):\\n",\n    "        d = torch.sigmoid(self.decay)\\n",\n    "        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)\\n",\n    "        w = F.softplus(self.path_w) * d.pow(lens - 1)\\n",\n    "        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])\\n",\n    "\\n",\n    "    def gnn_score(self, h, q_clip, cand_idx):\\n",\n    "        q = F.normalize(self.q_proj(q_clip), dim=-1)\\n",\n    "        return torch.einsum(\\"bd,bkd->bk\\", q, h[cand_idx])\\n",\n    "\\n",\n    "    def path_score(self, path_mat):\\n",\n    "        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)\\n",\n    "\\n",\n    "\\n",\n    "_SYM_FEAT = {}\\n",\n    "\\n",\n    "\\n",\n    "def symbolic_features(inv, vocab):\\n",\n    "    \\"\\"\\"CLIP-text của node lớp (= prompt ensemble của tagger) và khái niệm (\'lemma: định nghĩa WordNet\').\\"\\"\\"\\n",\n    "    key = inv.sha1\\n",\n    "    if key not in _SYM_FEAT:\\n",\n    "        T = term_text_features_v2(vocab)\\n",\n    "        cls = np.stack([T[vocab[\\"terms\\"].index(k)] for k in inv.class_keys])\\n",\n    "        texts = [f\\"{inv.concept_meta[k][\'lemma\']}: {inv.concept_meta[k][\'gloss\']}\\".strip(\\": \\") for k in inv.concept_keys]\\n",\n    "        con = clip_encode_texts(texts) if texts else np.zeros((0, cls.shape[1]), np.float32)\\n",\n    "        _SYM_FEAT[key] = np.concatenate([cls, con]).astype(np.float32)\\n",\n    "    return _SYM_FEAT[key]\\n",\n    "\\n",\n    "\\n",\n    "def predicate_phrase(rel, p, vocab):\\n",\n    "    \\"\\"\\"Cụm chữ cho prompt: svo → dạng V-ing của động từ đầu + phần còn lại (\'ride on\' → \'riding on\'); prep → nguyên văn.\\"\\"\\"\\n",\n    "    toks = p.split()\\n",\n    "    if rel == \\"svo\\" and toks:\\n",\n    "        g = vocab[\\"gerund\\"].get(ckey(toks[0], \\"v\\")) or naive_gerund(toks[0])\\n",\n    "        toks = [g] + toks[1:]\\n",\n    "    return \\" \\".join(toks)\\n",\n    "\\n",\n    "\\n",\n    "def predicate_embeddings(pred_vocab, vocab):\\n",\n    "    \\"\\"\\"[n_vị_từ, D] CLIP-text ĐÓNG BĂNG (trung bình các prompt CFG.pred_prompts[họ], chuẩn L2).\\"\\"\\"\\n",\n    "    out = None\\n",\n    "    for rel in PRED_FAMILIES:\\n",\n    "        idx = [i for i, (r, _) in enumerate(pred_vocab) if r == rel]\\n",\n    "        if not idx:\\n",\n    "            continue\\n",\n    "        words = [predicate_phrase(rel, pred_vocab[i][1], vocab) for i in idx]\\n",\n    "        E = sum(clip_encode_texts([pr.format(w) for w in words]) for pr in CFG.pred_prompts[rel])\\n",\n    "        if out is None:\\n",\n    "            out = np.zeros((len(pred_vocab), E.shape[1]), np.float32)\\n",\n    "        out[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n",\n    "    return out if out is not None else np.zeros((len(pred_vocab), 1), np.float32)\\n",\n    "\\n",\n    "# ---------------------------------- v4: QC-RGAT (GNN quan hệ có attention ĐIỀU KIỆN THEO TRUY VẤN) ----------------------------------\\n",\n    "# H_q^(0)(c) = E_c ;  H_q^(t+1)(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q^(t)(u)            (cạnh giữa lớp/khái niệm)\\n",\n    "# a_q = σ( β_τ + κ_τ·log w_uc + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ\'_τ·cos(e_p, e_P(q)) )\\n",\n    "# s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v, c] · H_q^(T)(c)⟩        (Ā_con = contains chuẩn hoá theo hàng, video CHỈ nhận)\\n",\n    "# Phép truyền TUYẾN TÍNH theo E (attention chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh) ⇒ s = Σ đóng góp của từng\\n",\n    "# thành phần: thị giác + từng nhãn contains + từng ĐƯỜNG ĐI quan hệ — phân rã CHÍNH XÁC, giải thích = đúng các số hạng của điểm.\\n",\n    "V4_REL_TYPES = list(RELATION_TYPES)\\n",\n    "V4_TID = {ETYPE_ID[t]: i for i, t in enumerate(V4_REL_TYPES)}\\n",\n    "Q_ROLES = (\\"none\\", \\"subj\\", \\"verb\\", \\"obj\\", \\"pobj\\", \\"other\\")\\n",\n    "\\n",\n    "\\n",\n    "def query_roles_v4(texts, inv):\\n",\n    "    \\"\\"\\"Vai của node lớp trong CHÍNH truy vấn (parser caption áp lên văn bản truy vấn lúc suy luận — mục đích query_input).\\"\\"\\"\\n",\n    "    out = []\\n",\n    "    for text in texts:\\n",\n    "        _, _, trip = parse_caption_v2(text)\\n",\n    "        role = {}\\n",\n    "        for s_, rel, o_, _p in trip:\\n",\n    "            a, b = inv.key2id.get(s_), inv.key2id.get(o_)\\n",\n    "            if a is None or b is None:\\n",\n    "                continue\\n",\n    "            if rel == \\"agent_of\\":\\n",\n    "                role.setdefault(a, \\"subj\\"); role.setdefault(b, \\"verb\\")\\n",\n    "            elif rel == \\"object_of\\":\\n",\n    "                role.setdefault(a, \\"obj\\"); role.setdefault(b, \\"verb\\")\\n",\n    "            elif rel == \\"svo\\":\\n",\n    "                role.setdefault(a, \\"subj\\")\\n",\n    "                role[b] = role[b] if role.get(b) not in (None, \\"other\\") else \\"obj\\"\\n",\n    "            elif rel == \\"prep\\":\\n",\n    "                role.setdefault(b, \\"pobj\\")\\n",\n    "        out.append({k: Q_ROLES.index(r) for k, r in role.items()})\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "class RelDataV4:\\n",\n    "    \\"\\"\\"Tensor của một đồ thị cho QC-RGAT: cạnh quan hệ (giữa node lớp/khái niệm), vị từ của cạnh, contains chuẩn hoá theo hàng.\\"\\"\\"\\n",\n    "\\n",\n    "    def __init__(self, graph, drop_edges=None):\\n",\n    "        self.graph = graph\\n",\n    "        self.N, self.n = graph.n_vid, graph.n_nodes\\n",\n    "        self.ns = self.n - self.N\\n",\n    "        s, d, e, w, pi = graph.edge_arrays()\\n",\n    "        keep = np.isin(e, list(V4_TID))\\n",\n    "        assert (s[keep] >= self.N).all() and (d[keep] >= self.N).all(), \\"cạnh quan hệ chỉ được nối node lớp/khái niệm\\"\\n",\n    "        self.gidx = np.where(keep)[0]                       # chỉ số cạnh trong graph.edge_arrays() (để giải thích / xoá cạnh)\\n",\n    "        if drop_edges is not None:\\n",\n    "            self.gidx = np.setdiff1d(self.gidx, np.asarray(list(drop_edges), np.int64))\\n",\n    "        s, d, e, w, pi = s[self.gidx], d[self.gidx], e[self.gidx], w[self.gidx], pi[self.gidx]\\n",\n    "        self.E = len(self.gidx)\\n",\n    "        self.src_np, self.dst_np, self.pidx_np = s - self.N, d - self.N, pi\\n",\n    "        self.src = torch.from_numpy(self.src_np).to(DEVICE)\\n",\n    "        self.dst = torch.from_numpy(self.dst_np).to(DEVICE)\\n",\n    "        self.et = torch.tensor([V4_TID[int(x)] for x in e], dtype=torch.long, device=DEVICE)\\n",\n    "        self.logw = torch.from_numpy(np.log(np.maximum(w, 1e-3)).astype(np.float32)).to(DEVICE)\\n",\n    "        PE = np.zeros((self.E, PRED_EMB.shape[1]), np.float32)\\n",\n    "        rows = np.where(pi >= 0)[0]\\n",\n    "        if len(rows):\\n",\n    "            X = np.asarray(graph.pmix[pi[rows]] @ PRED_EMB)\\n",\n    "            PE[rows] = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)\\n",\n    "        self.PE = torch.from_numpy(PE).to(DEVICE)\\n",\n    "        M = (graph.pmix[np.maximum(pi, 0)].multiply((pi >= 0)[:, None]) if self.E and (pi >= 0).any()\\n",\n    "             else sp.csr_matrix((self.E, len(PRED_VOCAB)), dtype=np.float32))   # G8: không cạnh nào mang vị từ\\n",\n    "        self.Mex = sp.csr_matrix(M).T.tocsr()               # [n_vị_từ, E]: phần vị từ p trên cạnh e\\n",\n    "        A = (graph.A_con.tocsr()[:, self.N:] > 0).astype(np.float32).toarray()\\n",\n    "        self.Acon = torch.from_numpy(A / np.maximum(A.sum(1, keepdims=True), 1)).to(DEVICE)\\n",\n    "\\n",\n    "    def query_batch(self, q_idx, use_pred=True):\\n",\n    "        \\"\\"\\"Đầu vào phía truy vấn: node khớp + vai, vị từ truy vấn (khớp đúng + CLIP mềm) trên TỪNG cạnh, CLIP-text.\\"\\"\\"\\n",\n    "        q_idx = np.asarray(q_idx, np.int64)\\n",\n    "        B = len(q_idx)\\n",\n    "        bi, ci, ri = [], [], []\\n",\n    "        role_src = torch.zeros(B, self.ns, dtype=torch.long)\\n",\n    "        for b, q in enumerate(q_idx):\\n",\n    "            for h in V4_HITS[q]:\\n",\n    "                r = V4_ROLES[q].get(h, Q_ROLES.index(\\"verb\\" if INV.pos[h] == \\"v\\" else \\"other\\"))\\n",\n    "                bi.append(b); ci.append(h - self.N); ri.append(r)\\n",\n    "                role_src[b, h - self.N] = r\\n",\n    "        ex = np.zeros((B, self.E), np.float32)\\n",\n    "        qp = np.zeros((B, PRED_EMB.shape[1]), np.float32)\\n",\n    "        if use_pred and self.E:\\n",\n    "            for b, q in enumerate(q_idx):\\n",\n    "                ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]\\n",\n    "                if ids:\\n",\n    "                    ex[b] = np.minimum(np.asarray(self.Mex[ids].sum(0)).ravel(), 1.0)\\n",\n    "                    m = PRED_EMB[ids].mean(0)\\n",\n    "                    qp[b] = m / max(np.linalg.norm(m), 1e-8)\\n",\n    "        qp_t = torch.from_numpy(qp).to(DEVICE)\\n",\n    "        return dict(q=q_idx, bi=torch.tensor(bi, dtype=torch.long, device=DEVICE), ci=torch.tensor(ci, dtype=torch.long, device=DEVICE),\\n",\n    "                    ri=torch.tensor(ri, dtype=torch.long, device=DEVICE), role_src=role_src.to(DEVICE),\\n",\n    "                    ex=torch.from_numpy(ex).to(DEVICE), soft=qp_t @ self.PE.T,\\n",\n    "                    tq=torch.from_numpy(np.ascontiguousarray(TXT_FEAT[q_idx], dtype=np.float32)).to(DEVICE))\\n",\n    "\\n",\n    "\\n",\n    "class QCRGAT(nn.Module):\\n",\n    "    \\"\\"\\"rel=False ⇒ G1 (chỉ contains); use_pred=False ⇒ G2 (không vị từ); qcond=False ⇒ G6 (attention tĩnh, không theo truy vấn).\\"\\"\\"\\n",\n    "\\n",\n    "    SCALARS = (\\"beta\\", \\"kappa\\", \\"rgate\\", \\"g_ex\\", \\"g_soft\\")      # cổng vô hướng: lr × CFG.scalar_lr_mult (như cổng của v3)\\n",\n    "\\n",\n    "    def __init__(self, data, clip_dim, d=64, T=1, rel=True, qcond=True, use_pred=True):\\n",\n    "        super().__init__()\\n",\n    "        assert T in (1, 2), \\"phân rã đường đi hiện cài cho T = 1 hoặc 2\\"\\n",\n    "        self.D, self.T, self.rel, self.qcond, self.use_pred = data, T, rel, qcond, use_pred\\n",\n    "        nt = len(V4_REL_TYPES)\\n",\n    "        self.vid = nn.Linear(clip_dim, d)\\n",\n    "        self.q = nn.Sequential(nn.Linear(clip_dim, d), nn.GELU(), nn.Linear(d, d))\\n",\n    "        self.qo = nn.Linear(d, d)\\n",\n    "        self.cls = nn.Embedding(data.ns, d)\\n",\n    "        nn.init.normal_(self.cls.weight, std=0.02)\\n",\n    "        self.z = nn.Parameter(1 + 0.1 * torch.randn(T, nt, d))\\n",\n    "        self.beta = nn.Parameter(torch.full((nt,), -1.0))\\n",\n    "        self.kappa = nn.Parameter(torch.zeros(nt))\\n",\n    "        self.qgate = nn.Linear(d, nt)\\n",\n    "        self.rgate = nn.Parameter(torch.zeros(len(Q_ROLES), nt))\\n",\n    "        self.g_ex = nn.Parameter(torch.ones(nt))         # prior: cạnh có vị từ KHỚP truy vấn được ưu tiên (học tiếp từ dữ liệu)\\n",\n    "        self.g_soft = nn.Parameter(torch.zeros(nt))\\n",\n    "        self.clip_logit = nn.Parameter(torch.tensor(0.5413))\\n",\n    "\\n",\n    "    def attention(self, qb, u, D=None):\\n",\n    "        \\"\\"\\"a_q(cạnh) ∈ (0, 1), [B, E]. Chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh (KHÔNG phụ thuộc trạng thái node).\\"\\"\\"\\n",\n    "        D = D or self.D\\n",\n    "        a = (self.beta[D.et] + self.kappa[D.et] * D.logw)[None].expand(u.shape[0], -1)\\n",\n    "        if self.qcond:\\n",\n    "            a = a + self.qgate(u)[:, D.et] + self.rgate[qb[\\"role_src\\"][:, D.src], D.et]\\n",\n    "            if self.use_pred:\\n",\n    "                a = a + self.g_ex[D.et][None] * qb[\\"ex\\"] + self.g_soft[D.et][None] * qb[\\"soft\\"]\\n",\n    "        return torch.sigmoid(a)\\n",\n    "\\n",\n    "    def class_states(self, qb, u, D=None):\\n",\n    "        D = D or self.D\\n",\n    "        E = self.cls.weight[None].expand(u.shape[0], -1, -1)\\n",\n    "        if not self.rel or D.E == 0:\\n",\n    "            return E, None\\n",\n    "        a = self.attention(qb, u, D)\\n",\n    "        H = E\\n",\n    "        for t in range(self.T):\\n",\n    "            H = E.index_add(1, D.dst, a[:, :, None] * self.z[t][D.et][None] * H[:, D.src, :])\\n",\n    "        return H, a\\n",\n    "\\n",\n    "    def score(self, qb, cand, D=None):\\n",\n    "        \\"\\"\\"cand: [B, K] video id (tensor) ⇒ [B, K].\\"\\"\\"\\n",\n    "        D = D or self.D\\n",\n    "        u = self.q(qb[\\"tq\\"])\\n",\n    "        H, _ = self.class_states(qb, u, D)\\n",\n    "        V = self.vid(VID_T[cand]) + torch.einsum(\\"bkc,bcd->bkd\\", D.Acon[cand], H)\\n",\n    "        return torch.einsum(\\"bd,bkd->bk\\", F.normalize(self.qo(u), dim=-1), V)\\n",\n    "\\n",\n    "    @torch.no_grad()\\n",\n    "    def decompose(self, q, vid, D=None):\\n",\n    "        \\"\\"\\"Phân rã CHÍNH XÁC s(q, vid) = thị giác + Σ nhãn + Σ đường đi quan hệ. Mỗi đường đi: list chỉ số cạnh (trong D)\\n",\n    "        theo chiều đi từ node nguồn tới node nhãn c của video, kèm đóng góp.\\"\\"\\"\\n",\n    "        D = D or self.D\\n",\n    "        qb = D.query_batch([q], self.use_pred)\\n",\n    "        u = self.q(qb[\\"tq\\"])\\n",\n    "        uo = F.normalize(self.qo(u), dim=-1)[0]\\n",\n    "        E = self.cls.weight\\n",\n    "        vt = int(vid)\\n",\n    "        terms = [dict(kind=\\"visual\\", node=None, edges=(), contrib=float(uo @ self.vid(VID_T[vt])))]\\n",\n    "        acon = D.Acon[vt]\\n",\n    "        tags = torch.nonzero(acon > 0).flatten().tolist()\\n",\n    "        for c in tags:\\n",\n    "            terms.append(dict(kind=\\"contains\\", node=c, edges=(), contrib=float(acon[c] * (uo @ E[c]))))\\n",\n    "        if self.rel and D.E:\\n",\n    "            a = self.attention(qb, u, D)[0]\\n",\n    "            inc = defaultdict(list)\\n",\n    "            for k in range(D.E):\\n",\n    "                inc[int(D.dst_np[k])].append(k)\\n",\n    "            for c in tags:\\n",\n    "                for e1 in inc.get(c, []):                     # đường đi 1 bước: u –e1→ c   (dùng z của lớp cuối)\\n",\n    "                    zz = self.z[self.T - 1][D.et[e1]]\\n",\n    "                    terms.append(dict(kind=\\"relation\\", node=c, edges=(e1,),\\n",\n    "                                      contrib=float(acon[c] * a[e1] * (uo @ (zz * E[D.src_np[e1]])))))\\n",\n    "                    if self.T == 2:                           # đường đi 2 bước: w –e0→ u –e1→ c\\n",\n    "                        for e0 in inc.get(int(D.src_np[e1]), []):\\n",\n    "                            zz2 = self.z[1][D.et[e1]] * self.z[0][D.et[e0]]\\n",\n    "                            terms.append(dict(kind=\\"relation\\", node=c, edges=(e0, e1),\\n",\n    "                                              contrib=float(acon[c] * a[e1] * a[e0] * (uo @ (zz2 * E[D.src_np[e0]])))))\\n",\n    "        return terms\\n",\n    "\\n",\n    "\\n",\n    "VID_T = None\\n",\n    "\\n",\n    "\\n",\n    "def v4_tensors():\\n",\n    "    global VID_T\\n",\n    "    VID_T = torch.from_numpy(np.ascontiguousarray(VID_FEAT, dtype=np.float32)).to(DEVICE)\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c035",\n   "metadata": {},\n   "source": [\n    "## 7. Engine và hai giao thức suy luận\\n",\n    "* **strict inductive** (mặc định): video val, test và tập ngoài **chỉ nhận** message qua `contains_inv`, nên không ảnh hưởng node dùng chung hay video khác. Có unit test cho cả video test MSVD và video MSR-VTT.\\n",\n    "* **transductive**: chỉ để báo cáo.\\n",\n    "\\n",\n    "Engine mang thêm chỉ số vị từ của từng cạnh (`edge_pidx`) và bảng phân phối vị từ (`pmix`) cho GNN. Nó cũng có ma trận truy vấn × (vị từ, node) cho mẫu P1m. Vị từ của truy vấn được khớp khi mọi token của vị từ có mặt trong truy vấn, ví dụ `ride on` khớp \\"a man is riding on a horse\\"."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c036",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def znorm_per_row(x, eps=1e-6):\\n",\n    "    if isinstance(x, torch.Tensor):\\n",\n    "        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)\\n",\n    "    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)\\n",\n    "\\n",\n    "\\n",\n    "def combine(s_clip, s_gnn, s_path, alpha, lam):\\n",\n    "    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)\\n",\n    "\\n",\n    "\\n",\n    "PROTOCOLS = (\\"inductive\\", \\"transductive\\")\\n",\n    "\\n",\n    "\\n",\n    "def _torch_sparse(M):\\n",\n    "    M = M.tocoo()\\n",\n    "    idx = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))\\n",\n    "    return torch.sparse_coo_tensor(idx, torch.from_numpy(M.data.astype(np.float32)), M.shape).coalesce()\\n",\n    "\\n",\n    "\\n",\n    "class EngineV3:\\n",\n    "    \\"\\"\\"Truy vấn = caption; ứng viên = video. Ba tập cạnh cho GNN (như v2):\\n",\n    "      train        : bỏ MỌI cạnh chạm video không thuộc train;\\n",\n    "      inductive    : cạnh của train + cạnh lớp → video (contains_inv) của video val/test/ext, KHÔNG có chiều video → lớp\\n",\n    "                     ⇒ video mới chỉ NHẬN message, không ảnh hưởng node lớp/khái niệm dùng chung hay video khác;\\n",\n    "      transductive : đồ thị đầy đủ.\\n",\n    "    Thêm ở v3: chỉ số vị từ của từng cạnh (edge_pidx) + bảng phân phối vị từ (pmix) cho GNN; ma trận truy vấn × (vị từ, node)\\n",\n    "    cho mẫu đường đi khớp vị từ P1m.\\"\\"\\"\\n",\n    "\\n",\n    "    _QCACHE = {}\\n",\n    "\\n",\n    "    def __init__(self, graph, vid_feat, txt_feat, qtok, qpreds):\\n",\n    "        self.graph = graph\\n",\n    "        self.vid_feat = torch.from_numpy(np.ascontiguousarray(vid_feat, dtype=np.float32))\\n",\n    "        self.txt_feat = torch.from_numpy(np.ascontiguousarray(txt_feat, dtype=np.float32))\\n",\n    "        n = graph.n_nodes\\n",\n    "        key = (graph.inv.sha1, len(qtok), id(qtok), id(qpreds))\\n",\n    "        if key not in EngineV3._QCACHE:                      # chỉ phụ thuộc inventory + truy vấn ⇒ dùng chung mọi biến thể đồ thị\\n",\n    "            qc = [parse_tokens_v2(t, graph.inv) for t in qtok]\\n",\n    "            rows, cols = [], []\\n",\n    "            for j, hits in enumerate(qc):\\n",\n    "                for nid, _, _ in hits:\\n",\n    "                    rows.append(j); cols.append(nid)\\n",\n    "            QT = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(qtok), n))\\n",\n    "            QPB = {}\\n",\n    "            for rel in PRED_FAMILIES:\\n",\n    "                r_, c_ = [], []\\n",\n    "                for j, (hits, qp) in enumerate(zip(qc, qpreds)):\\n",\n    "                    for pid in qp[rel]:\\n",\n    "                        for nid, _, _ in hits:\\n",\n    "                            r_.append(j); c_.append(pid * n + nid)\\n",\n    "                QPB[rel] = sp.csr_matrix((np.ones(len(r_), np.float32), (r_, c_)), shape=(len(qtok), len(PRED_VOCAB) * n))\\n",\n    "            EngineV3._QCACHE[key] = (qc, QT, QPB)\\n",\n    "        self.q_concepts, self.QT, self.QPB = EngineV3._QCACHE[key]\\n",\n    "        self.q_preds = qpreds\\n",\n    "        s, d, e, w, pi = graph.edge_arrays()\\n",\n    "        N = graph.n_vid\\n",\n    "        is_train_vid = np.zeros(n, bool)\\n",\n    "        is_train_vid[np.array(SPLIT[\\"train\\"])] = True\\n",\n    "        other_vid = (np.arange(n) < N) & ~is_train_vid\\n",\n    "        touch = other_vid[s] | other_vid[d]\\n",\n    "        keep = {\\"train\\": ~touch, \\"transductive\\": np.ones(len(s), bool),\\n",\n    "                \\"inductive\\": ~touch | ((e == ETYPE_ID[\\"contains_inv\\"]) & other_vid[d] & ~other_vid[s])}\\n",\n    "        self.edges = {k: (torch.from_numpy(np.stack([s[m], d[m]])), torch.from_numpy(e[m]), torch.from_numpy(w[m]),\\n",\n    "                          torch.from_numpy(pi[m])) for k, m in keep.items()}\\n",\n    "        self.edge_stats = {k: int(m.sum()) for k, m in keep.items()}\\n",\n    "        self.pmix = _torch_sparse(graph.pmix)\\n",\n    "\\n",\n    "    def edges_on(self, name):\\n",\n    "        ei, et, ew, epi = self.edges[name]\\n",\n    "        return ei.to(DEVICE), et.to(DEVICE), ew.to(DEVICE), epi.to(DEVICE), self.pmix.to(DEVICE)\\n",\n    "\\n",\n    "    def path_feats(self, cand, q_idx):\\n",\n    "        cand, q_idx = np.asarray(cand), np.asarray(q_idx)\\n",\n    "        if cand.ndim == 1:\\n",\n    "            cand = cand[:, None]\\n",\n    "        B, K = cand.shape\\n",\n    "        out = np.zeros((B, K, len(PATTERNS)), np.float32)\\n",\n    "        for s0 in range(0, B, CFG.eval_chunk):\\n",\n    "            c, q = cand[s0:s0 + CFG.eval_chunk], q_idx[s0:s0 + CFG.eval_chunk]\\n",\n    "            rows, inv_ = np.unique(c.ravel(), return_inverse=True)\\n",\n    "            inv_ = inv_.reshape(c.shape)\\n",\n    "            col = np.arange(len(q))[:, None]\\n",\n    "            Qb = self.QT[q].T.tocsr()\\n",\n    "            for p, name in enumerate(PATTERNS):\\n",\n    "                if name.startswith(\\"P1m_\\"):\\n",\n    "                    M = self.graph.R[name]\\n",\n    "                    if M.nnz == 0:\\n",\n    "                        continue\\n",\n    "                    out[s0:s0 + len(q), :, p] = (M[rows] @ self.QPB[name[4:]][q].T).toarray()[inv_, col]\\n",\n    "                else:\\n",\n    "                    out[s0:s0 + len(q), :, p] = (self.graph.R[name][rows] @ Qb).toarray()[inv_, col]\\n",\n    "        return np.log1p(out)\\n",\n    "\\n",\n    "    def clip_candidates(self, q_idx, gallery, k):\\n",\n    "        gallery, q_idx = np.asarray(gallery), np.asarray(q_idx)\\n",\n    "        G = self.vid_feat[torch.from_numpy(gallery)]\\n",\n    "        k = min(int(k), len(gallery))\\n",\n    "        C, S = [], []\\n",\n    "        for s0 in range(0, len(q_idx), CFG.eval_chunk):\\n",\n    "            top = (self.txt_feat[torch.from_numpy(q_idx[s0:s0 + CFG.eval_chunk])] @ G.T).topk(k, dim=1)\\n",\n    "            C.append(gallery[top.indices.numpy()]); S.append(top.values.numpy())\\n",\n    "        return np.concatenate(C), np.concatenate(S)\\n",\n    "\\n",\n    "\\n",\n    "def make_model(engine, sym_feat, mp_weight=\\"binary\\", init=\\"learned\\", pred_mode=\\"none\\"):\\n",\n    "    return GraphScorerV3(engine.graph.n_nodes, engine.txt_feat.shape[1], engine.vid_feat, sym_feat, pred_emb=PRED_EMB,\\n",\n    "                         mp_weight=mp_weight, init=init, pred_mode=pred_mode).to(DEVICE)\\n",\n    "\\n",\n    "\\n",\n    "def train_graph_scorer_v3(engine, train_q, sym_feat, seed=0, mp_weight=\\"binary\\", init=\\"learned\\", pred_mode=\\"none\\", verbose=False):\\n",\n    "    \\"\\"\\"InfoNCE như v1/v2 (âm trong batch không trùng video + âm khó CLIP trong gallery TRAIN); chỉ dùng cạnh \'train\'.\\"\\"\\"\\n",\n    "    train_q = np.asarray(train_q)\\n",\n    "    CAPS.use(train_q, \\"gnn_train\\")\\n",\n    "    set_seed(seed)\\n",\n    "    gold_all = CAP_VID[train_q]\\n",\n    "    train_vids = np.unique(gold_all)\\n",\n    "    assert all(SPLIT_OF[int(v)] == \\"train\\" for v in train_vids)\\n",\n    "    model = make_model(engine, sym_feat, mp_weight, init, pred_mode)\\n",\n    "    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))\\n",\n    "    params = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]\\n",\n    "    scalar = [p for n_, p in params if n_.endswith(\\"gate\\") or n_ in (\\"path_w\\", \\"decay\\", \\"clip_logit\\")]\\n",\n    "    sid = {id(p) for p in scalar}\\n",\n    "    opt = torch.optim.AdamW([dict(params=[p for _, p in params if id(p) not in sid]),\\n",\n    "                             dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],\\n",\n    "                            lr=CFG.lr, weight_decay=CFG.weight_decay)\\n",\n    "    edges = engine.edges_on(\\"train\\")\\n",\n    "    vid_dev = engine.vid_feat.to(DEVICE)\\n",\n    "    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)\\n",\n    "    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])\\n",\n    "\\n",\n    "    def logits_for(cand_np, qi, h, qf):\\n",\n    "        cand = torch.from_numpy(cand_np).to(DEVICE)\\n",\n    "        pm = torch.from_numpy(engine.path_feats(cand_np, qi)).to(DEVICE)\\n",\n    "        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))\\n",\n    "        if CFG.train_with_clip:\\n",\n    "            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum(\\"bd,bkd->bk\\", qf, vid_dev[cand]))\\n",\n    "        return z / CFG.temperature\\n",\n    "\\n",\n    "    by_vid = defaultdict(list)\\n",\n    "    for j, v in enumerate(gold_all):\\n",\n    "        by_vid[int(v)].append(j)\\n",\n    "    rng, hist = np.random.RandomState(seed), []\\n",\n    "    for ep in range(CFG.epochs):\\n",\n    "        model.train()\\n",\n    "        tot, nb = 0.0, 0\\n",\n    "        for _ in range(CFG.caption_passes):\\n",\n    "            perm = rng.permutation(train_vids)\\n",\n    "            for s0 in range(0, len(perm), CFG.train_bs):\\n",\n    "                bv = perm[s0:s0 + CFG.train_bs]\\n",\n    "                if len(bv) < 8:\\n",\n    "                    continue\\n",\n    "                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])\\n",\n    "                qi, gi = train_q[sel], gold_all[sel]\\n",\n    "                qf = engine.txt_feat[torch.from_numpy(qi)].to(DEVICE)\\n",\n    "                h = model.encode_graph(*edges)\\n",\n    "                loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf), torch.arange(len(gi), device=DEVICE))\\n",\n    "                loss = loss + F.cross_entropy(logits_for(np.concatenate([gi[:, None], hard[sel]], 1), qi, h, qf),\\n",\n    "                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))\\n",\n    "                opt.zero_grad(); loss.backward()\\n",\n    "                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n",\n    "                opt.step()\\n",\n    "                tot += loss.item(); nb += 1\\n",\n    "        hist.append(tot / max(nb, 1))\\n",\n    "        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):\\n",\n    "            print(f\\"    epoch {ep:2d} | loss {hist[-1]:.4f} | w_path \\" +\\n",\n    "                  \\" \\".join(f\\"{x:.2f}\\" for x in model.pattern_weights().detach().cpu().numpy()))\\n",\n    "    model.eval()\\n",\n    "    return model, hist\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def score_queries_v3(model, engine, q_idx, gallery, k, protocol=\\"inductive\\"):\\n",\n    "    if isinstance(model, QCRGAT):                       # v4: cùng đầu ra, s_path = 0 (λ = 1)\\n",\n    "        return score_queries_v4(model, engine, q_idx, gallery, k, protocol)\\n",\n    "    model.eval()\\n",\n    "    h = model.encode_graph(*engine.edges_on(protocol))\\n",\n    "    q_idx = np.asarray(q_idx)\\n",\n    "    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)\\n",\n    "    s_gnn, s_path = np.zeros_like(s_clip), np.zeros_like(s_clip)\\n",\n    "    for s0 in range(0, len(q_idx), CFG.eval_chunk):\\n",\n    "        sl = slice(s0, s0 + CFG.eval_chunk)\\n",\n    "        c = torch.from_numpy(cand[sl]).to(DEVICE)\\n",\n    "        qf = engine.txt_feat[torch.from_numpy(q_idx[sl])].to(DEVICE)\\n",\n    "        s_gnn[sl] = model.gnn_score(h, qf, c).cpu().numpy()\\n",\n    "        s_path[sl] = model.path_score(torch.from_numpy(engine.path_feats(cand[sl], q_idx[sl])).to(DEVICE)).cpu().numpy()\\n",\n    "    return dict(cand=cand, clip_raw=s_clip, path_raw=s_path, gnn_raw=s_gnn,\\n",\n    "                s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s_gnn), s_path=znorm_per_row(s_path))\\n",\n    "\\n",\n    "\\n",\n    "def tune_on_val_v3(model, engine, part, k, clip_rank, lam_grid=None, protocol=\\"inductive\\"):\\n",\n    "    \\"\\"\\"(α, λ) tối đa MRR trên VAL; lam_grid cố định cho ablation path-only (λ=0) / GNN-only (λ=1).\\"\\"\\"\\n",\n    "    assert part[\\"name\\"] == \\"val\\", \\"tune chỉ nhận VAL\\"\\n",\n    "    CAPS.use(part[\\"q_idx\\"], \\"tune\\")\\n",\n    "    sc = score_queries_v3(model, engine, part[\\"q_idx\\"], part[\\"gallery\\"], k, protocol)\\n",\n    "    grid, best = [], None\\n",\n    "    for a in CFG.alpha_grid:\\n",\n    "        for l in (lam_grid or CFG.lambda_grid):\\n",\n    "            r = ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], a, l), part[\\"gold\\"], clip_rank)\\n",\n    "            m = metrics_from_ranks(r, part[\\"gold\\"])\\n",\n    "            grid.append(dict(alpha=a, lam=l, k=k, **{x: m[x] for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}, rank=r))\\n",\n    "            if best is None or m[\\"MRR\\"] > best[\\"MRR\\"]:\\n",\n    "                best = grid[-1]\\n",\n    "    return (best[\\"alpha\\"], best[\\"lam\\"]), [{k_: v for k_, v in g_.items() if k_ != \\"rank\\"} for g_ in grid], best\\n",\n    "\\n",\n    "# ---------------------------------- v4: huấn luyện + chấm điểm QC-RGAT (cùng giao thức v3: InfoNCE, âm khó CLIP, chỉ video TRAIN) ----------------------------------\\n",\n    "_RELDATA = {}\\n",\n    "\\n",\n    "\\n",\n    "def reldata_v4(engine):\\n",\n    "    if id(engine) not in _RELDATA:\\n",\n    "        _RELDATA[id(engine)] = RelDataV4(engine.graph)\\n",\n    "    return _RELDATA[id(engine)]\\n",\n    "\\n",\n    "\\n",\n    "def make_model_v4(engine, design):\\n",\n    "    return QCRGAT(reldata_v4(engine), TXT_FEAT.shape[1], d=CFG.v4_dim, T=design[\\"v4_layers\\"], rel=design[\\"v4_rel\\"],\\n",\n    "                  qcond=design[\\"v4_qcond\\"], use_pred=design[\\"v4_pred\\"]).to(DEVICE)\\n",\n    "\\n",\n    "\\n",\n    "def train_v4(engine, train_q, design, seed=0, verbose=False):\\n",\n    "    \\"\\"\\"Như train_graph_scorer_v3: in-batch (video không trùng) + 31 âm khó CLIP trong gallery TRAIN; logit = z(s) + c·z(cos CLIP).\\"\\"\\"\\n",\n    "    train_q = np.asarray(train_q)\\n",\n    "    CAPS.use(train_q, \\"gnn_train\\")\\n",\n    "    set_seed(seed)\\n",\n    "    gold_all = CAP_VID[train_q]\\n",\n    "    train_vids = np.unique(gold_all)\\n",\n    "    assert all(SPLIT_OF[int(v)] == \\"train\\" for v in train_vids)\\n",\n    "    model = make_model_v4(engine, design)\\n",\n    "    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))\\n",\n    "    named = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]\\n",\n    "    scal = [p for n_, p in named if n_ in QCRGAT.SCALARS]\\n",\n    "    opt = torch.optim.AdamW([dict(params=[p for n_, p in named if n_ not in QCRGAT.SCALARS]),\\n",\n    "                             dict(params=scal, lr=CFG.v4_lr * CFG.scalar_lr_mult, weight_decay=0.0)],\\n",\n    "                            lr=CFG.v4_lr, weight_decay=CFG.weight_decay)\\n",\n    "    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)\\n",\n    "    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])\\n",\n    "    txt_dev = torch.from_numpy(np.ascontiguousarray(TXT_FEAT, dtype=np.float32)).to(DEVICE)\\n",\n    "\\n",\n    "    def logits_for(qb, cand_np):\\n",\n    "        cand = torch.from_numpy(cand_np).to(DEVICE)\\n",\n    "        z = znorm_per_row(model.score(qb, cand))\\n",\n    "        if CFG.train_with_clip:\\n",\n    "            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum(\\"bd,bkd->bk\\", txt_dev[qb[\\"q\\"]], VID_T[cand]))\\n",\n    "        return z / CFG.temperature\\n",\n    "\\n",\n    "    by_vid = defaultdict(list)\\n",\n    "    for j, v in enumerate(gold_all):\\n",\n    "        by_vid[int(v)].append(j)\\n",\n    "    rng, hist = np.random.RandomState(seed), []\\n",\n    "    for ep in range(CFG.epochs):\\n",\n    "        model.train()\\n",\n    "        tot, nb = 0.0, 0\\n",\n    "        for _ in range(CFG.caption_passes):\\n",\n    "            perm = rng.permutation(train_vids)\\n",\n    "            for s0 in range(0, len(perm), CFG.train_bs):\\n",\n    "                bv = perm[s0:s0 + CFG.train_bs]\\n",\n    "                if len(bv) < 8:\\n",\n    "                    continue\\n",\n    "                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])\\n",\n    "                gi = gold_all[sel]\\n",\n    "                qb = model.D.query_batch(train_q[sel], model.use_pred)\\n",\n    "                loss = F.cross_entropy(logits_for(qb, np.tile(gi, (len(gi), 1))), torch.arange(len(gi), device=DEVICE))\\n",\n    "                loss = loss + F.cross_entropy(logits_for(qb, np.concatenate([gi[:, None], hard[sel]], 1)),\\n",\n    "                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))\\n",\n    "                opt.zero_grad(); loss.backward()\\n",\n    "                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n",\n    "                opt.step()\\n",\n    "                tot += loss.item(); nb += 1\\n",\n    "        hist.append(tot / max(nb, 1))\\n",\n    "        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):\\n",\n    "            print(f\\"    epoch {ep:2d} | loss {hist[-1]:.4f}\\")\\n",\n    "    model.eval()\\n",\n    "    return model, hist\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def score_queries_v4(model, engine, q_idx, gallery, k, protocol=\\"inductive\\", D=None):\\n",\n    "    \\"\\"\\"Cùng đầu ra với score_queries_v3; s_gnn = điểm QC-RGAT, s_path = 0 (λ cố định = 1).\\"\\"\\"\\n",\n    "    model.eval()\\n",\n    "    D = D or model.D\\n",\n    "    q_idx = np.asarray(q_idx)\\n",\n    "    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)\\n",\n    "    s = np.zeros_like(s_clip)\\n",\n    "    bs = max(1, CFG.v4_score_bs)\\n",\n    "    for s0 in range(0, len(q_idx), bs):\\n",\n    "        qb = D.query_batch(q_idx[s0:s0 + bs], model.use_pred)\\n",\n    "        s[s0:s0 + bs] = model.score(qb, torch.from_numpy(cand[s0:s0 + bs]).to(DEVICE), D).cpu().numpy()\\n",\n    "    z = np.zeros_like(s)\\n",\n    "    return dict(cand=cand, clip_raw=s_clip, path_raw=z, gnn_raw=s, s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s), s_path=z)\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c037",\n   "metadata": {},\n   "source": [\n    "## 9. Chỉ số đánh giá\\n",\n    "\\n",\n    "* Mỗi **caption là một truy vấn**, có đúng một video dương. Notebook báo cáo R@1, R@5, R@10, MRR, nDCG@10, MedR và `MRR_video_macro` (macro-MRR theo video).\\n",\n    "* Thứ hạng tính trên **toàn bộ gallery**: top-K được sắp lại, ngoài top-K giữ thứ hạng CLIP.\\n",\n    "* Kiểm định cặp dùng **bootstrap theo cụm video**. Họ so sánh chính được hiệu chỉnh **Holm–Bonferroni**; luật chọn từng stage cũng hiệu chỉnh Holm trên các ứng viên của stage.\\n",\n    "* Notebook báo cáo thêm: candidate recall, nhóm direct/indirect/other, khái niệm hiếm, vị từ hiếm/phổ biến/không có vị từ, số node/cạnh/bậc, thời gian chạy và bộ nhớ GPU."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c038",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def ranks_with_fallback(cand, scores, gold, clip_rank):\\n",\n    "    \\"\\"\\"Thứ hạng trên TOÀN gallery: trong top-K theo điểm xếp lại; ngoài top-K giữ thứ hạng CLIP (> K).\\"\\"\\"\\n",\n    "    order = np.argsort(-scores, axis=1, kind=\\"stable\\")\\n",\n    "    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold)[:, None]\\n",\n    "    return np.where(hit.any(1), hit.argmax(1) + 1, np.maximum(clip_rank, cand.shape[1] + 1))\\n",\n    "\\n",\n    "\\n",\n    "def metrics_from_ranks(rank, gold=None, ks=(1, 5, 10)):\\n",\n    "    rank = np.asarray(rank, np.float64)\\n",\n    "    out = {f\\"R@{k}\\": float((rank <= k).mean()) for k in ks}\\n",\n    "    out[\\"MRR\\"] = float((1.0 / rank).mean())\\n",\n    "    out[\\"nDCG@10\\"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())\\n",\n    "    out[\\"MedR\\"] = float(np.median(rank))\\n",\n    "    out[\\"n_queries\\"] = int(len(rank))\\n",\n    "    if gold is not None:\\n",\n    "        _, inv = np.unique(np.asarray(gold), return_inverse=True)\\n",\n    "        out[\\"MRR_video_macro\\"] = float((np.bincount(inv, 1.0 / rank) / np.bincount(inv)).mean())\\n",\n    "        out[\\"n_videos\\"] = int(inv.max() + 1) if len(inv) else 0\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "METRIC_KEYS = [\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\", \\"nDCG@10\\", \\"MedR\\", \\"MRR_video_macro\\"]\\n",\n    "\\n",\n    "\\n",\n    "def fmt_mean_std(vals):\\n",\n    "    vals = np.asarray(vals, float)\\n",\n    "    return f\\"{vals.mean():.4f} ± {vals.std():.4f}\\"\\n",\n    "\\n",\n    "\\n",\n    "def cluster_bootstrap_diff(rr_a, rr_b, gold, n_boot=5000, seed=0):\\n",\n    "    \\"\\"\\"Δ = mean(rr_a − rr_b) theo truy vấn; khoảng tin cậy 95% bằng bootstrap THEO VIDEO (cụm caption).\\"\\"\\"\\n",\n    "    d = np.asarray(rr_a) - np.asarray(rr_b)\\n",\n    "    _, inv = np.unique(np.asarray(gold), return_inverse=True)\\n",\n    "    S, C = np.bincount(inv, d), np.bincount(inv).astype(float)\\n",\n    "    idx = np.random.RandomState(seed).randint(0, len(S), (n_boot, len(S)))\\n",\n    "    boot = S[idx].sum(1) / C[idx].sum(1)\\n",\n    "    return dict(diff=float(d.mean()), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),\\n",\n    "                p_not_better=float((boot <= 0).mean()), a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum()))\\n",\n    "\\n",\n    "\\n",\n    "def holm(pvals):\\n",\n    "    \\"\\"\\"p đã hiệu chỉnh Holm–Bonferroni (đơn điệu), cùng thứ tự với đầu vào.\\"\\"\\"\\n",\n    "    p = np.asarray(pvals, float)\\n",\n    "    m = len(p)\\n",\n    "    if m == 0:\\n",\n    "        return p\\n",\n    "    order = np.argsort(p)\\n",\n    "    adj = np.empty(m)\\n",\n    "    run = 0.0\\n",\n    "    for k, i in enumerate(order):\\n",\n    "        run = max(run, min(1.0, (m - k) * p[i]))\\n",\n    "        adj[i] = run\\n",\n    "    return adj"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c039",\n   "metadata": {},\n   "source": [\n    "## 8. Giải thích v3\\n",\n    "* Đường đi được liệt kê theo **chiều** cạnh, kèm loại cạnh và **vị từ** ở bước quan hệ.\\n",\n    "  * Mẫu P1m chỉ dùng vị từ có trong truy vấn.\\n",\n    "  * Giải thích hợp lệ khi: cạnh có thật, đúng loại, đúng chiều, đúng kiểu node; vị từ thuộc **đúng** cạnh theo **đúng** chiều và có trong truy vấn.\\n",\n    "  * Có đối chứng âm: sửa pred_id thì phải bị phát hiện.\\n",\n    "* Điểm liệt kê phải khớp ma trận ở mọi `path_mode`.\\n",\n    "* Fidelity báo riêng ba mức: thay đổi **điểm**, **thứ hạng**, **quyết định** top-1 / top-10.\\n",\n    "* Có phần đóng góp theo mẫu đường đi và độ nhạy theo seed (Jaccard).\\n",\n    "* Xoá cạnh giải thích được so với xoá cạnh **đối chứng** cùng loại.\\n",\n    "* Notebook báo cáo trọng số hiệu dụng `(1−α)(1−λ)` của nhánh đường đi."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c040",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "def enumerate_paths_v3(g, vid, concept_ids, qpred=None, PM=None):\\n",\n    "    \\"\\"\\"Đường đi CỤ THỂ từ video tới khái niệm truy vấn, đi theo CHIỀU cạnh, ghi loại cạnh từng bước và VỊ TỪ của bước\\n",\n    "    quan hệ. Trọng số = idf(đích) × Π trọng số trong ma trận đường đi (PM) của chế độ path hiện tại ⇒ khớp graph.R.\\n",\n    "    Mẫu P1m_f (path_mode \'pred_match\'): bước a –f→ b chỉ tính phần vị từ có trong truy vấn (qpred[f] = tập pred_id).\\"\\"\\"\\n",\n    "    PM = PM or g.PM\\n",\n    "    target = {int(c) for c in concept_ids}\\n",\n    "    C = PM[\\"C\\"]\\n",\n    "    own = set(C.indices[C.indptr[vid]:C.indptr[vid + 1]].tolist())\\n",\n    "    out = []\\n",\n    "\\n",\n    "    def nbrs(M, x):\\n",\n    "        sl = slice(M.indptr[x], M.indptr[x + 1])\\n",\n    "        return zip(M.indices[sl].tolist(), M.data[sl].tolist())\\n",\n    "\\n",\n    "    def add(pattern, nodes, rels, w, pred=None):\\n",\n    "        c = nodes[-1]\\n",\n    "        if pattern != \\"P0_contains\\" and CFG.drop_redundant_indirect and c in own:\\n",\n    "            return\\n",\n    "        w = float(g.idf[c]) * float(w)\\n",\n    "        if w > 0:\\n",\n    "            out.append(dict(pattern=pattern, nodes=[int(x) for x in nodes], rels=list(rels), w=w, pred=pred))\\n",\n    "\\n",\n    "    for o, w0 in nbrs(C, vid):\\n",\n    "        if o in target:\\n",\n    "            add(\\"P0_contains\\", [o], [\\"contains\\"], w0)\\n",\n    "        for f in FAMILIES:\\n",\n    "            for t in (FAMILY_REL[f], FAMILY_REL[f] + \\"_inv\\"):\\n",\n    "                for o2, w1 in nbrs(PM[t], o):\\n",\n    "                    if o2 in target:\\n",\n    "                        add(f\\"P1_{f}\\", [o, o2], [\\"contains\\", t], w0 * w1)\\n",\n    "                    for c, w2 in nbrs(PM[\\"isa\\"], o2):\\n",\n    "                        if c in target:\\n",\n    "                            add(f\\"P4_{f}_isa\\", [o, o2, c], [\\"contains\\", t, \\"isa\\"], w0 * w1 * w2)\\n",\n    "        for c, w1 in nbrs(PM[\\"isa\\"], o):\\n",\n    "            if c in target:\\n",\n    "                add(\\"P2_isa\\", [o, c], [\\"contains\\", \\"isa\\"], w0 * w1)\\n",\n    "        for o2, w1 in nbrs(PM[\\"related\\"], o):\\n",\n    "            if o2 in target:\\n",\n    "                add(\\"P3_related\\", [o, o2], [\\"contains\\", \\"related\\"], w0 * w1)\\n",\n    "            for c, w2 in nbrs(PM[\\"isa\\"], o2):\\n",\n    "                if c in target:\\n",\n    "                    add(\\"P5_related_isa\\", [o, o2, c], [\\"contains\\", \\"related\\", \\"isa\\"], w0 * w1 * w2)\\n",\n    "        if g.path_mode == \\"pred_match\\" and qpred:\\n",\n    "            for rel in PRED_FAMILIES:\\n",\n    "                want = set(qpred.get(rel, ()))\\n",\n    "                if not want:\\n",\n    "                    continue\\n",\n    "                for t in (rel, rel + \\"_inv\\"):\\n",\n    "                    for o2, w1 in nbrs(PM[t], o):\\n",\n    "                        if o2 not in target:\\n",\n    "                            continue\\n",\n    "                        a, b = (o2, o) if t.endswith(\\"_inv\\") else (o, o2)\\n",\n    "                        for pid, s in g.pshare[rel].get((a, b), {}).items():\\n",\n    "                            if pid in want:\\n",\n    "                                add(f\\"P1m_{rel}\\", [o, o2], [\\"contains\\", t], w0 * w1 * s, pred=(rel, int(pid), float(s)))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def paths_to_counts_v3(paths, mode):\\n",\n    "    \\"\\"\\"log1p(tổng theo mẫu) — với path_dedup: tổng theo khái niệm của đường đi MẠNH NHẤT (khớp _maxprod).\\"\\"\\"\\n",\n    "    v = np.zeros(len(PATTERNS), np.float64)\\n",\n    "    if mode == \\"dedup\\":\\n",\n    "        best = defaultdict(float)\\n",\n    "        for p in paths:\\n",\n    "            k = (p[\\"pattern\\"], p[\\"nodes\\"][-1])\\n",\n    "            best[k] = max(best[k], p[\\"w\\"])\\n",\n    "        for (pat, _), w in best.items():\\n",\n    "            v[PIDX[pat]] += w\\n",\n    "    else:\\n",\n    "        for p in paths:\\n",\n    "            v[PIDX[p[\\"pattern\\"]]] += p[\\"w\\"]\\n",\n    "    return np.log1p(v).astype(np.float32)\\n",\n    "\\n",\n    "\\n",\n    "def path_edges_v3(p, vid):\\n",\n    "    \\"\\"\\"[(loại, u, v)] gồm cạnh contains đầu tiên.\\"\\"\\"\\n",\n    "    nodes = [int(vid)] + p[\\"nodes\\"]\\n",\n    "    return list(zip(p[\\"rels\\"], nodes[:-1], nodes[1:]))\\n",\n    "\\n",\n    "\\n",\n    "def edge_valid_v3(g, rel, u, v):\\n",\n    "    \\"\\"\\"Cạnh có thật trong ma trận GỐC đúng loại + đúng chiều, và kiểu đầu/cuối khớp REL_SPEC.\\"\\"\\"\\n",\n    "    M = g.A_con if rel == \\"contains\\" else g.A[rel]\\n",\n    "    if M[u, v] == 0:\\n",\n    "        return False\\n",\n    "    ok_src = REL_SPEC[rel][\\"src\\"].split(\\":\\")[0].split(\\"|\\")\\n",\n    "    ok_tgt = REL_SPEC[rel][\\"tgt\\"].split(\\" \\")[0].split(\\":\\")[0].split(\\"|\\")\\n",\n    "    return g.inv.node_type[u] in ok_src and g.inv.node_type[v] in ok_tgt\\n",\n    "\\n",\n    "\\n",\n    "def pred_valid_v3(g, p, vid, qtokens):\\n",\n    "    \\"\\"\\"Đường đi khớp vị từ: vị từ ghi trong giải thích phải (1) thuộc phân phối vị từ của ĐÚNG cạnh theo ĐÚNG chiều và\\n",\n    "    (2) có mặt trong truy vấn. Đường đi không mang vị từ ⇒ True.\\"\\"\\"\\n",\n    "    if p.get(\\"pred\\") is None:\\n",\n    "        return True\\n",\n    "    rel, pid, share = p[\\"pred\\"]\\n",\n    "    t, u, v = path_edges_v3(p, vid)[1]\\n",\n    "    a, b = (v, u) if t.endswith(\\"_inv\\") else (u, v)\\n",\n    "    ok_edge = abs(g.pshare[rel].get((a, b), {}).get(pid, -1.0) - share) < 1e-9\\n",\n    "    return bool(ok_edge and all(x in qtokens for x in predicate_tokens(PRED_VOCAB[pid][1])) and PRED_VOCAB[pid][0] == rel)\\n",\n    "\\n",\n    "\\n",\n    "def path_text_v3(g, p):\\n",\n    "    nm = lambda i: g.inv.name[i]\\n",\n    "    s = \\"video\\"\\n",\n    "    nodes = p[\\"nodes\\"]\\n",\n    "    for rel, a, b in zip(p[\\"rels\\"], [None] + nodes[:-1], nodes):\\n",\n    "        lab = REL_TEXT.get(rel, rel)\\n",\n    "        if \\"{pred}\\" in lab:\\n",\n    "            if p.get(\\"pred\\") is not None:\\n",\n    "                pr = f\\"{PRED_VOCAB[p[\'pred\'][1]][1]}({p[\'pred\'][2]:.2f})\\"\\n",\n    "            else:\\n",\n    "                d = g.predicates_of(rel, a, b)\\n",\n    "                tot = max(sum(d.values()), 1)\\n",\n    "                pr = \\"/\\".join(f\\"{k}({c / tot:.2f})\\" for k, c in sorted(d.items(), key=lambda x: (-x[1], x[0]))[:3]) or \\"?\\"\\n",\n    "            lab = lab.format(pred=pr.replace(\\" \\", \\"_\\"))\\n",\n    "        s += f\\" –{lab}→ {nm(b)}\\"\\n",\n    "    return s\\n",\n    "\\n",\n    "\\n",\n    "def q_args(eng, q):\\n",\n    "    \\"\\"\\"(node khái niệm của truy vấn, {họ: pred_id khớp}, tập token truy vấn).\\"\\"\\"\\n",\n    "    return [nid for nid, _, _ in eng.q_concepts[q]], eng.q_preds[q], QPRED_TOK[q]\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def explain_and_score_v3(model, g, vid, concept_ids, qpred=None, qtokens=frozenset(), top_m=CFG.top_m_paths, rng_seed=0):\\n",\n    "    paths = enumerate_paths_v3(g, vid, concept_ids, qpred)\\n",\n    "    w = model.pattern_weights().detach().cpu().numpy()\\n",\n    "    gain = F.softplus(model.path_gain).item()\\n",\n    "    score = lambda ps: float(gain * (paths_to_counts_v3(ps, g.path_mode) * w).sum())\\n",\n    "    full = score(paths)\\n",\n    "    contrib = [full - score(paths[:i] + paths[i + 1:]) for i in range(len(paths))]\\n",\n    "    order = np.argsort(-np.array(contrib), kind=\\"stable\\") if contrib else np.array([], int)\\n",\n    "    ex = [int(i) for i in order[:top_m]]\\n",\n    "    expl = [dict(text=path_text_v3(g, paths[i]), pattern=paths[i][\\"pattern\\"], contribution=float(contrib[i]),\\n",\n    "                 nodes=paths[i][\\"nodes\\"], rels=paths[i][\\"rels\\"], pred=paths[i][\\"pred\\"]) for i in ex]\\n",\n    "    keep = [paths[i] for i in ex]\\n",\n    "    drop = [p for i, p in enumerate(paths) if i not in set(ex)]\\n",\n    "    rng, rnd = np.random.RandomState(rng_seed), []\\n",\n    "    for _ in range(10):\\n",\n    "        r = set(rng.choice(len(paths), size=min(len(ex), len(paths)), replace=False).tolist()) if paths else set()\\n",\n    "        rnd.append(score([p for i, p in enumerate(paths) if i not in r]))\\n",\n    "    cset = {int(c) for c in concept_ids}\\n",\n    "    valid = [p[\\"nodes\\"][-1] in cset and all(edge_valid_v3(g, *e) for e in path_edges_v3(p, vid)) for p in keep]\\n",\n    "    pvalid = [pred_valid_v3(g, p, vid, qtokens) for p in keep]\\n",\n    "    share = np.zeros(len(PATTERNS))\\n",\n    "    for i in range(len(paths)):\\n",\n    "        share[PIDX[paths[i][\\"pattern\\"]]] += max(contrib[i], 0)\\n",\n    "    detail = dict(n_paths=len(paths), score_full=full, score_without_explanation=score(drop),\\n",\n    "                  score_only_explanation=score(keep), score_without_random=float(np.mean(rnd)) if rnd else full,\\n",\n    "                  sparsity=1 - len(ex) / max(len(paths), 1), validity=float(np.mean(valid)) if valid else 0.0,\\n",\n    "                  predicate_validity=float(np.mean(pvalid)) if pvalid else 1.0,\\n",\n    "                  n_predicate_paths=int(sum(p.get(\\"pred\\") is not None for p in keep)),\\n",\n    "                  pattern_share={p: float(s / max(share.sum(), 1e-12)) for p, s in zip(PATTERNS, share) if s > 0})\\n",\n    "    return expl, full, detail\\n",\n    "\\n",\n    "\\n",\n    "def _rank_in(fin, j):\\n",\n    "    return int((fin > fin[j]).sum()) + 1\\n",\n    "\\n",\n    "\\n",\n    "def path_score_from_R(g, R, cids, qpred, gain, w):\\n",\n    "    \\"\\"\\"s_path (chưa chuẩn hoá) từ một R đã tính lại (hàng = ứng viên) — cùng công thức với engine.path_feats.\\"\\"\\"\\n",\n    "    n, cols = g.n_nodes, np.array(sorted({int(c) for c in cids}))\\n",\n    "    feats = []\\n",\n    "    for p in PATTERNS:\\n",\n    "        if p.startswith(\\"P1m_\\"):\\n",\n    "            pc = np.array(sorted(pid * n + c for pid in qpred.get(p[4:], ()) for c in cols), np.int64)\\n",\n    "            feats.append(np.asarray(R[p][:, pc].sum(1)).ravel() if len(pc) and R[p].nnz else np.zeros(R[\\"P0_contains\\"].shape[0]))\\n",\n    "        else:\\n",\n    "            feats.append(np.asarray(R[p][:, cols].sum(1)).ravel())\\n",\n    "    return gain * (np.log1p(np.stack(feats, 1)) * w).sum(1)\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def edge_deletion_v3(run, part, groups, n=CFG.n_edge_deletion, seed=0, protocol=\\"inductive\\"):\\n",\n    "    \\"\\"\\"Với mỗi truy vấn (video đúng trong top-K): xoá THẬT (a) cạnh then chốt của đường đi giải thích top-1 và\\n",\n    "    (b) đường đi QUAN HỆ đứng đầu; đối chứng: xoá một cạnh CÙNG LOẠI, KHÔNG nằm trên đường đi nào tới khái niệm\\n",\n    "    của truy vấn. Tính lại R cho mọi ứng viên từ ma trận đã sửa (cùng công thức), ghi Δ s_path, Δ điểm cuối, hạng,\\n",\n    "    và quyết định top-1 / top-10. GNN giữ cố định để cô lập nhánh đường đi.\\"\\"\\"\\n",\n    "    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n",\n    "    a, l, z = run[\\"alpha\\"], run[\\"lam\\"], znorm_per_row\\n",\n    "    h = model.encode_graph(*eng.edges_on(protocol))\\n",\n    "    w = model.pattern_weights().detach().cpu().numpy()\\n",\n    "    gain = F.softplus(model.path_gain).item()\\n",\n    "    rng = np.random.RandomState(seed)\\n",\n    "    rows, used, mism = [], set(), 0\\n",\n    "\\n",\n    "    def delete(rel, u, v):\\n",\n    "        A2 = dict(g.A)\\n",\n    "        Acon = g.A_con\\n",\n    "        if rel == \\"contains\\":\\n",\n    "            M = g.A_con.tolil(copy=True); M[u, v] = 0; Acon = M.tocsr(); Acon.eliminate_zeros()\\n",\n    "        else:\\n",\n    "            M = g.A[rel].tolil(copy=True); M[u, v] = 0; A2[rel] = M.tocsr(); A2[rel].eliminate_zeros()\\n",\n    "        return g.path_mats(A=A2, A_con=Acon)\\n",\n    "\\n",\n    "    for i in rng.permutation(len(part[\\"q_idx\\"])):\\n",\n    "        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n",\n    "        cids, qpred, qtok = q_args(eng, q)\\n",\n    "        if gold in used or not cids:\\n",\n    "            continue\\n",\n    "        cand, s_clip = eng.clip_candidates(np.array([q]), part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "        cand, s_clip = cand[0], s_clip[0]\\n",\n    "        hit = np.where(cand == gold)[0]\\n",\n    "        if len(hit) == 0:\\n",\n    "            continue\\n",\n    "        expl, _, _ = explain_and_score_v3(model, g, gold, cids, qpred, qtok)\\n",\n    "        expl = [e for e in expl if e[\\"contribution\\"] > 0]\\n",\n    "        if not expl:\\n",\n    "            continue\\n",\n    "        j = int(hit[0])\\n",\n    "        sp0 = path_score_from_R(g, g.compute_R(rows=cand), cids, qpred, gain, w)\\n",\n    "        sp_eng = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)\\n",\n    "        mism += int(not np.allclose(sp0, sp_eng, atol=1e-4))\\n",\n    "        sg = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()\\n",\n    "        f0 = combine(z(s_clip[None]), z(sg), z(sp0[None]), a, l)[0]\\n",\n    "        r0, top0 = _rank_in(f0, j), int(cand[np.argmax(f0)])\\n",\n    "        targets = [(\\"top1\\", expl[0])]\\n",\n    "        relp = next((e for e in expl if e[\\"pattern\\"] != \\"P0_contains\\"), None)\\n",\n    "        if relp is not None and relp is not expl[0]:\\n",\n    "            targets.append((\\"top_relational\\", relp))\\n",\n    "        on_path = {(t_, x, y) for p in enumerate_paths_v3(g, gold, cids, qpred) for t_, x, y in path_edges_v3(p, gold)}\\n",\n    "        for which, p in targets:\\n",\n    "            edges = path_edges_v3(p, gold)\\n",\n    "            rel, u, v = edges[1] if len(edges) > 1 else edges[0]\\n",\n    "            ctrl = None\\n",\n    "            if rel != \\"contains\\":                        # cạnh đối chứng cùng loại, không nằm trên đường đi nào\\n",\n    "                cand_e = [(x, y) for x, y in zip(*g.A[rel].nonzero()) if (rel, x, y) not in on_path]\\n",\n    "                if cand_e:\\n",\n    "                    ctrl = cand_e[rng.randint(len(cand_e))]\\n",\n    "            todo = [(\\"explanation\\", (u, v))] + ([(\\"control\\", ctrl)] if ctrl is not None else [])\\n",\n    "            for kind, (uu, vv) in todo:\\n",\n    "                sp1 = path_score_from_R(g, g.compute_R(rows=cand, PM=delete(rel, uu, vv)), cids, qpred, gain, w)\\n",\n    "                f1 = combine(z(s_clip[None]), z(sg), z(sp1[None]), a, l)[0]\\n",\n    "                r1 = _rank_in(f1, j)\\n",\n    "                rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], group=str(groups[i]), which=which, deleted=kind,\\n",\n    "                                 explanation=p[\\"text\\"], pattern=p[\\"pattern\\"], relation=rel,\\n",\n    "                                 predicate=PRED_VOCAB[p[\\"pred\\"][1]][1] if p.get(\\"pred\\") else \\"\\",\\n",\n    "                                 edge=f\\"{g.inv.name[uu]} → {g.inv.name[vv]}\\", edge_valid=edge_valid_v3(g, rel, uu, vv),\\n",\n    "                                 path_contribution=p[\\"contribution\\"], s_path_before=float(sp0[j]), s_path_after=float(sp1[j]),\\n",\n    "                                 d_s_path=float(sp1[j] - sp0[j]), d_final=float(f1[j] - f0[j]), rank_before=r0, rank_after=r1,\\n",\n    "                                 rank_changed=r1 != r0, top1_changed=int(cand[np.argmax(f1)]) != top0,\\n",\n    "                                 top10_changed=(r0 <= 10) != (r1 <= 10),\\n",\n    "                                 w_path_effective=(1 - a) * (1 - l), alpha=a, lam=l))\\n",\n    "        used.add(gold)\\n",\n    "        if len(used) >= n:\\n",\n    "            break\\n",\n    "    return pd.DataFrame(rows), mism\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def path_fidelity_v3(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0, protocol=\\"inductive\\"):\\n",\n    "    \\"\\"\\"Fidelity mức đường đi (score) và mức thứ hạng của điểm cuối cho cùng một tập truy vấn; đóng góp theo mẫu đường đi.\\"\\"\\"\\n",\n    "    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n",\n    "    a, l, z = run[\\"alpha\\"], run[\\"lam\\"], znorm_per_row\\n",\n    "    h = model.encode_graph(*eng.edges_on(protocol))\\n",\n    "    w = model.pattern_weights().detach().cpu().numpy()\\n",\n    "    gain = F.softplus(model.path_gain).item()\\n",\n    "    sc = lambda ps: gain * float((paths_to_counts_v3(ps, g.path_mode) * w).sum())\\n",\n    "    rng, rows, used = np.random.RandomState(seed), [], set()\\n",\n    "    for i in rng.permutation(len(part[\\"q_idx\\"])):\\n",\n    "        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n",\n    "        cids, qpred, qtok = q_args(eng, q)\\n",\n    "        if gold in used or not cids:\\n",\n    "            continue\\n",\n    "        cand, s_clip = eng.clip_candidates(np.array([q]), part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "        cand, s_clip = cand[0], s_clip[0]\\n",\n    "        hit = np.where(cand == gold)[0]\\n",\n    "        expl, s, det = explain_and_score_v3(model, g, gold, cids, qpred, qtok, top_m=top_m)\\n",\n    "        if len(hit) == 0 or det[\\"n_paths\\"] <= top_m:\\n",\n    "            continue\\n",\n    "        paths = enumerate_paths_v3(g, gold, cids, qpred)\\n",\n    "        j = int(hit[0])\\n",\n    "        sp_raw = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)\\n",\n    "        s_gnn = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()\\n",\n    "\\n",\n    "        def rank_with(kept):\\n",\n    "            s_ = sp_raw.copy(); s_[j] = sc(kept)\\n",\n    "            return _rank_in(combine(z(s_clip[None]), z(s_gnn), z(s_[None]), a, l)[0], j)\\n",\n    "\\n",\n    "        contrib = np.array([sc(paths) - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])\\n",\n    "        top = set(np.argsort(-contrib, kind=\\"stable\\")[:top_m].tolist())\\n",\n    "        rr = [rank_with([p for k, p in enumerate(paths) if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])\\n",\n    "              for _ in range(10)]\\n",\n    "        rf, rd = rank_with(paths), rank_with([p for k, p in enumerate(paths) if k not in top])\\n",\n    "        rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], group=str(groups[i]),\\n",\n    "                         **{k_: v for k_, v in det.items() if k_ != \\"pattern_share\\"},\\n",\n    "                         **{f\\"share_{k_}\\": v for k_, v in det[\\"pattern_share\\"].items()},\\n",\n    "                         matrix_vs_enumeration_ok=bool(abs(sc(paths) - sp_raw[j]) < 1e-4),\\n",\n    "                         drop_explanation=det[\\"score_full\\"] - det[\\"score_without_explanation\\"],\\n",\n    "                         drop_random=det[\\"score_full\\"] - det[\\"score_without_random\\"],\\n",\n    "                         rank_full=rf, rank_drop_expl=rd, rank_drop_rand=float(np.mean(rr)),\\n",\n    "                         rank_only_expl=rank_with([paths[k] for k in top]),\\n",\n    "                         top10_changed_drop_expl=(rf <= 10) != (rd <= 10), w_path_effective=(1 - a) * (1 - l),\\n",\n    "                         explanation=\\" || \\".join(e[\\"text\\"] for e in expl[:3])))\\n",\n    "        used.add(gold)\\n",\n    "        if len(rows) >= n_queries:\\n",\n    "            break\\n",\n    "    return pd.DataFrame(rows)\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def explanation_sensitivity(runs_by_seed, part, n=CFG.n_seed_sensitivity, top_m=CFG.top_m_paths, seed=0):\\n",\n    "    \\"\\"\\"Jaccard của tập đường đi giải thích top-m cho CÙNG (truy vấn, video đúng) giữa các seed / cấu hình.\\"\\"\\"\\n",\n    "    keys = list(runs_by_seed)\\n",\n    "    if len(keys) < 2:\\n",\n    "        return pd.DataFrame()\\n",\n    "    eng0 = runs_by_seed[keys[0]][\\"engine\\"]\\n",\n    "    rows, used = [], set()\\n",\n    "    for i in np.random.RandomState(seed).permutation(len(part[\\"q_idx\\"])):\\n",\n    "        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n",\n    "        cids, qpred, qtok = q_args(eng0, q)\\n",\n    "        if gold in used or not cids:\\n",\n    "            continue\\n",\n    "        sets = {}\\n",\n    "        for k in keys:\\n",\n    "            r = runs_by_seed[k]\\n",\n    "            ex, _, _ = explain_and_score_v3(r[\\"model\\"], r[\\"graph\\"], gold, cids, qpred, qtok, top_m=top_m)\\n",\n    "            sets[k] = {e[\\"text\\"] for e in ex}\\n",\n    "        if not any(sets.values()):\\n",\n    "            continue\\n",\n    "        js = [len(sets[x] & sets[y]) / max(len(sets[x] | sets[y]), 1) for x, y in itertools.combinations(keys, 2)]\\n",\n    "        rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], n_configs=len(keys), mean_jaccard=float(np.mean(js)),\\n",\n    "                         min_jaccard=float(np.min(js))))\\n",\n    "        used.add(gold)\\n",\n    "        if len(rows) >= n:\\n",\n    "            break\\n",\n    "    return pd.DataFrame(rows)\\n",\n    "\\n",\n    "# ---------------------------------- v4: giải thích = CHÍNH các số hạng của điểm (phân rã chính xác) ----------------------------------\\n",\n    "def v4_edge_info(D, k):\\n",\n    "    \\"\\"\\"(nguồn, loại, đích, vị từ) của cạnh k trong RelDataV4 — đọc lại từ đồ thị gốc.\\"\\"\\"\\n",\n    "    g = D.graph\\n",\n    "    s, d, e, w, pi = g.edge_arrays()\\n",\n    "    j = int(D.gidx[k])\\n",\n    "    preds = {}\\n",\n    "    if pi[j] >= 0:\\n",\n    "        row = g.pmix[pi[j]]\\n",\n    "        preds = {PRED_VOCAB[c][1]: float(v) for c, v in zip(row.indices, row.data)}\\n",\n    "    return dict(src=int(s[j]), rel=EDGE_TYPES[int(e[j])], dst=int(d[j]), weight=float(w[j]), predicates=preds)\\n",\n    "\\n",\n    "\\n",\n    "def v4_path_text(D, t, vid):\\n",\n    "    inv = D.graph.inv\\n",\n    "    if t[\\"kind\\"] == \\"visual\\":\\n",\n    "        return f\\"[thị giác] {inv.name[vid]}\\"\\n",\n    "    c = D.N + t[\\"node\\"]\\n",\n    "    if t[\\"kind\\"] == \\"contains\\":\\n",\n    "        return f\\"{inv.name[vid]} –chứa→ {inv.name[c]}\\"\\n",\n    "    hops = []\\n",\n    "    for k in t[\\"edges\\"]:\\n",\n    "        ei = v4_edge_info(D, k)\\n",\n    "        p = max(ei[\\"predicates\\"], key=ei[\\"predicates\\"].get) if ei[\\"predicates\\"] else \\"\\"\\n",\n    "        hops.append((inv.name[ei[\\"src\\"]], ei[\\"rel\\"] + (f\\"[{p}]\\" if p else \\"\\"), inv.name[ei[\\"dst\\"]]))\\n",\n    "    chain = hops[0][0] + \\"\\".join(f\\" –{r}→ {b}\\" for _, r, b in hops)\\n",\n    "    return f\\"{chain} ←chứa– {inv.name[vid]}\\"\\n",\n    "\\n",\n    "\\n",\n    "def v4_edge_valid(D, k, q):\\n",\n    "    \\"\\"\\"Cạnh có thật (đúng loại, chiều, kiểu node theo REL_SPEC); nếu truy vấn có vị từ khớp cạnh thì vị từ đó thuộc ĐÚNG cạnh.\\"\\"\\"\\n",\n    "    ei = v4_edge_info(D, k)\\n",\n    "    g, inv = D.graph, D.graph.inv\\n",\n    "    ok = ei[\\"rel\\"] in g.A and g.A[ei[\\"rel\\"]][ei[\\"src\\"], ei[\\"dst\\"]] > 0\\n",\n    "    ok = ok and inv.node_type[ei[\\"src\\"]] != \\"video\\" and inv.node_type[ei[\\"dst\\"]] != \\"video\\"\\n",\n    "    ids = {PRED_VOCAB[p][1] for rel in PRED_FAMILIES for p in QPREDS[q][rel]}\\n",\n    "    matched = ids & set(ei[\\"predicates\\"])\\n",\n    "    return bool(ok), bool(matched), bool(not matched or matched <= set(ei[\\"predicates\\"]))\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def explain_v4(model, q, vid, top_m=CFG.top_m_paths):\\n",\n    "    \\"\\"\\"Giải thích \\"vì sao video được trả về\\" = top-m đường đi quan hệ ỦNG HỘ (đóng góp > 0); đường đi PHẢN ĐỐI (< 0) ghi riêng.\\n",\n    "    Σ mọi số hạng (thị giác + nhãn + mọi đường đi) = điểm mô hình.\\"\\"\\"\\n",\n    "    terms = model.decompose(q, vid)\\n",\n    "    tot = {k: sum(t[\\"contrib\\"] for t in terms if t[\\"kind\\"] == k) for k in (\\"visual\\", \\"contains\\", \\"relation\\")}\\n",\n    "    rel = [t for t in terms if t[\\"kind\\"] == \\"relation\\"]\\n",\n    "    sup = sorted([t for t in rel if t[\\"contrib\\"] > 0], key=lambda t: -t[\\"contrib\\"])[:top_m]\\n",\n    "    opp = sorted([t for t in rel if t[\\"contrib\\"] < 0], key=lambda t: t[\\"contrib\\"])[:top_m]\\n",\n    "    return dict(terms=terms, top_relation=sup, opposing=opp, totals=tot, score=sum(t[\\"contrib\\"] for t in terms))\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def v4_rank_after(run, part_q, gold, cand, s_clip_z, D2):\\n",\n    "    \\"\\"\\"Thứ hạng video đúng trong top-K (điểm cuối với α đã đóng băng) khi mô hình dùng dữ liệu đồ thị D2.\\"\\"\\"\\n",\n    "    m = run[\\"model\\"]\\n",\n    "    qb = D2.query_batch([part_q], m.use_pred)\\n",\n    "    s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE), D2).cpu().numpy()\\n",\n    "    fin = run[\\"alpha\\"] * s_clip_z[None] + (1 - run[\\"alpha\\"]) * znorm_per_row(s)\\n",\n    "    j = int(np.where(cand == gold)[0][0])\\n",\n    "    return int((fin[0] > fin[0, j]).sum()) + 1, s[0, j], fin[0]\\n",\n    "\\n",\n    "\\n",\n    "def xai_v4(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0):\\n",\n    "    \\"\\"\\"Với ≥ n truy vấn có video đúng trong top-K và có đường đi quan hệ ỦNG HỘ: (1) hợp lệ (cạnh thật, vị từ đúng cạnh), (2) độ thưa,\\n",\n    "    (3) fidelity: xoá THẬT các cạnh của top-m đường đi giải thích ⇒ Δ điểm mô hình so với dự đoán từ phân rã (khớp = chính xác),\\n",\n    "    so với xoá cùng số cạnh quan hệ NGẪU NHIÊN nối vào nhãn của video; (4) thứ hạng / top-1 / top-10 đổi (báo riêng).\\"\\"\\"\\n",\n    "    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n",\n    "    D = m.D\\n",\n    "    rng = np.random.RandomState(seed)\\n",\n    "    q_all = part[\\"q_idx\\"]\\n",\n    "    order = rng.permutation(len(q_all))\\n",\n    "    cand_all, s_clip = eng.clip_candidates(q_all, part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "    rows = []\\n",\n    "    for i in order:\\n",\n    "        if len(rows) >= n_queries:\\n",\n    "            break\\n",\n    "        q, gold = int(q_all[i]), int(part[\\"gold\\"][i])\\n",\n    "        cand = cand_all[i]\\n",\n    "        if gold not in cand:\\n",\n    "            continue\\n",\n    "        ex = explain_v4(m, q, gold, top_m)\\n",\n    "        if not ex[\\"top_relation\\"]:\\n",\n    "            continue\\n",\n    "        zc = znorm_per_row(s_clip[i:i + 1])[0]\\n",\n    "        r_full, s_full, _ = v4_rank_after(run, q, gold, cand, zc, D)\\n",\n    "        e_del = sorted({k for t in ex[\\"top_relation\\"] for k in t[\\"edges\\"]})\\n",\n    "        D_ex = RelDataV4(D.graph, drop_edges=D.gidx[e_del])\\n",\n    "        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, D_ex)\\n",\n    "        tags = set(torch.nonzero(D.Acon[gold] > 0).flatten().tolist())\\n",\n    "        pool = [k for k in range(D.E) if int(D.dst_np[k]) in tags and k not in e_del]\\n",\n    "        e_rand = list(rng.choice(pool, min(len(e_del), len(pool)), replace=False)) if pool else []\\n",\n    "        D_rd = RelDataV4(D.graph, drop_edges=D.gidx[e_rand]) if e_rand else D\\n",\n    "        r_rd, s_rd, _ = v4_rank_after(run, q, gold, cand, zc, D_rd)\\n",\n    "        expect = sum(t[\\"contrib\\"] for t in ex[\\"terms\\"] if t[\\"kind\\"] == \\"relation\\" and set(t[\\"edges\\"]) & set(e_del))\\n",\n    "        val = [v4_edge_valid(D, k, q) for k in e_del]\\n",\n    "        _, _, fin_full = v4_rank_after(run, q, gold, cand, zc, D)\\n",\n    "        rows.append(dict(q=q, gold=gold, group=groups[i], n_terms=len(ex[\\"terms\\"]), n_relation_terms=sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"terms\\"]),\\n",\n    "                         n_expl_paths=len(ex[\\"top_relation\\"]), n_expl_edges=len(e_del), sparsity=len(ex[\\"top_relation\\"]) / max(1, sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"terms\\"])),\\n",\n    "                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[2] for v in val])),\\n",\n    "                         n_predicate_edges=int(sum(v[1] for v in val)),\\n",\n    "                         share_relation=ex[\\"totals\\"][\\"relation\\"] / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n",\n    "                         s_full=s_full, d_s_expl=s_ex - s_full, d_s_expected=-expect, d_s_rand=s_rd - s_full,\\n",\n    "                         decomposition_ok=float(abs(ex[\\"score\\"] - s_full) < 1e-4), deletion_matches_decomposition=float(abs((s_ex - s_full) + expect) < 1e-4) if m.T == 1 else float(\\"nan\\"),\\n",\n    "                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,\\n",\n    "                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),\\n",\n    "                         top10_changed_expl=float(set(np.argsort(-fin_ex)[:10]) != set(np.argsort(-fin_full)[:10])),\\n",\n    "                         explanation=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {v4_path_text(D, t, gold)}\\" for t in ex[\\"top_relation\\"]),\\n",\n    "                         opposing=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {v4_path_text(D, t, gold)}\\" for t in ex[\\"opposing\\"][:3])))\\n",\n    "    return pd.DataFrame(rows)\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def relation_analysis_v4(run, part, max_q=3000, seed=0):\\n",\n    "    \\"\\"\\"Theo họ quan hệ: phần đóng góp vào điểm của video đúng, knockout lúc suy luận (bỏ mọi cạnh của họ) ⇒ ΔMRR;\\n",\n    "    attention trung bình trên cạnh có vị từ KHỚP truy vấn vs không khớp; theo vai của node nguồn trong truy vấn.\\"\\"\\"\\n",\n    "    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n",\n    "    D = m.D\\n",\n    "    q = part[\\"q_idx\\"]\\n",\n    "    if len(q) > max_q:\\n",\n    "        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))\\n",\n    "    gold = CAP_VID[q]\\n",\n    "    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n",\n    "    sc = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "    base = metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n",\n    "                                                  gold, clip_rank))[\\"MRR\\"]\\n",\n    "    rows = []\\n",\n    "    fam_of = np.array([REL_SPEC[V4_REL_TYPES[int(t)]][\\"family\\"] for t in D.et.cpu().numpy()])\\n",\n    "    for fam in sorted(set(fam_of)):\\n",\n    "        D2 = RelDataV4(D.graph, drop_edges=D.gidx[np.where(fam_of == fam)[0]])\\n",\n    "        sc2 = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k, D=D2)\\n",\n    "        mrr = metrics_from_ranks(ranks_with_fallback(sc2[\\"cand\\"], combine(sc2[\\"s_clip\\"], sc2[\\"s_gnn\\"], sc2[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n",\n    "                                                     gold, clip_rank))[\\"MRR\\"]\\n",\n    "        rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=fam, n_edges=int((fam_of == fam).sum()),\\n",\n    "                         knockout_MRR=mrr, d_MRR_knockout=mrr - base))\\n",\n    "    D0 = RelDataV4(D.graph, drop_edges=D.gidx)\\n",\n    "    sc0 = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k, D=D0)\\n",\n    "    mrr0 = metrics_from_ranks(ranks_with_fallback(sc0[\\"cand\\"], combine(sc0[\\"s_clip\\"], sc0[\\"s_gnn\\"], sc0[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n",\n    "                                                  gold, clip_rank))[\\"MRR\\"]\\n",\n    "    rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=\\"ALL_relations\\", n_edges=D.E, knockout_MRR=mrr0, d_MRR_knockout=mrr0 - base))\\n",\n    "    att_m, att_n, by_role = [], [], defaultdict(list)\\n",\n    "    for s0 in range(0, min(len(q), 1000), 128):\\n",\n    "        qb = D.query_batch(q[s0:s0 + 128], m.use_pred)\\n",\n    "        a = m.attention(qb, m.q(qb[\\"tq\\"])) if m.rel and D.E else None\\n",\n    "        if a is None:\\n",\n    "            break\\n",\n    "        pe = (D.pidx_np >= 0)\\n",\n    "        exm = qb[\\"ex\\"].cpu().numpy() > 0\\n",\n    "        A = a.cpu().numpy()\\n",\n    "        att_m.extend(A[exm & pe[None]].tolist()); att_n.extend(A[~exm & pe[None]].tolist())\\n",\n    "        rs = qb[\\"role_src\\"][:, D.src].cpu().numpy()\\n",\n    "        for r in range(len(Q_ROLES)):\\n",\n    "            if (rs == r).any():\\n",\n    "                by_role[Q_ROLES[r]].extend(A[rs == r].tolist())\\n",\n    "    rows.append(dict(section=\\"predicate_attention\\", split=part[\\"name\\"], item=\\"pred_edges\\",\\n",\n    "                     att_pred_match=float(np.mean(att_m)) if att_m else float(\\"nan\\"),\\n",\n    "                     att_pred_nomatch=float(np.mean(att_n)) if att_n else float(\\"nan\\"), n_match=len(att_m)))\\n",\n    "    for r, v in by_role.items():\\n",\n    "        rows.append(dict(section=\\"role_attention\\", split=part[\\"name\\"], item=r, att_mean=float(np.mean(v)), n=len(v)))\\n",\n    "    return pd.DataFrame(rows), dict(base_MRR=base, all_relations_knockout_MRR=mrr0,\\n",\n    "                                    att_pred_match=float(np.mean(att_m)) if att_m else float(\\"nan\\"),\\n",\n    "                                    att_pred_nomatch=float(np.mean(att_n)) if att_n else float(\\"nan\\"))\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c041",\n   "metadata": {},\n   "source": [\n    "## 9. Khung thí nghiệm\\n",\n    "Mỗi thí nghiệm có: `experiment_id`, stage, giả thuyết, thiết kế đầy đủ, seed, checkpoint fp16, số cạnh theo loại, số node, số tham số, (α, λ) chọn trên val, thời gian chạy, đỉnh bộ nhớ GPU, trạng thái. Thí nghiệm không chạy được ghi `NOT RUN` kèm lý do.\\n",\n    "\\n",\n    "`decide()` áp luật đăng ký trước trên **val**; incumbent là phương án đơn giản hơn. `PRIMARY_COMPARISONS` là họ so sánh chính ở mục 0c."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c042",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "DESIGN_BASE = dict(taxonomy=\\"heuristic\\", contains=\\"binary\\", conf_mode=\\"sigmoid\\", rel_policy=\\"all\\", path_mode=\\"weighted\\",\\n",\n    "                   mp_weight=\\"binary\\", init=\\"learned\\", mask=\\"none\\", rewire=False, pred_shuffle=False, pred_mode=\\"none\\",\\n",\n    "                   related=\\"base\\", protocol=\\"inductive\\", tags=\\"base\\",\\n",\n    "                   model=\\"v3\\", v4_layers=1, v4_rel=True, v4_qcond=True, v4_pred=True)        # model: v3 (TypedSAGE) | v4 (QC-RGAT)\\n",\n    "V3_FINAL = dict(DESIGN_BASE, taxonomy=\\"none\\", mask=\\"no_related\\")   # cấu hình cuối của v3 (E44, chọn trên VAL ở Kaggle) = đồ thị của v4\\n",\n    "RELATED_CFGS = {\\"base\\": (CFG.cooc_min_count, CFG.cooc_min_npmi), \\"npmi20\\": (CFG.cooc_min_count, 0.20),\\n",\n    "                \\"count10\\": (10, CFG.cooc_min_npmi), \\"conceptnet\\": (\\"conceptnet\\", 0.0)}\\n",\n    "GRAPHS, EXPS, DECISIONS, NOT_RUN = {}, {}, [], []\\n",\n    "REWIRE_SEED_BASE, SHUFFLE_SEED_BASE = 1000, 2000\\n",\n    "VERIFIED_CELLS = None            # {(họ, mức phổ biến)} đạt ngưỡng precision — chỉ có khi có nhãn người chấm\\n",\n    "\\n",\n    "\\n",\n    "def masked_types(mask):\\n",\n    "    \\"\\"\\"mask = tên trong MASKS, có thể ghép bằng \'+\' (ví dụ \'no_related+no_taxonomy\').\\"\\"\\"\\n",\n    "    return tuple(sorted({t for m in str(mask).split(\\"+\\") for t in MASKS[m]}))\\n",\n    "\\n",\n    "\\n",\n    "def add_mask(mask, extra):\\n",\n    "    parts = sorted({m for m in str(mask).split(\\"+\\") + [extra] if m != \\"none\\"})\\n",\n    "    return \\"+\\".join(parts) or \\"none\\"\\n",\n    "\\n",\n    "\\n",\n    "def graph_key(d, seed):\\n",\n    "    return (d[\\"taxonomy\\"], d[\\"contains\\"], d[\\"conf_mode\\"] if d[\\"contains\\"] == \\"conf\\" else \\"\\", d[\\"rel_policy\\"],\\n",\n    "            d[\\"path_mode\\"], d[\\"mask\\"], (REWIRE_SEED_BASE + seed) if d[\\"rewire\\"] else None,\\n",\n    "            (SHUFFLE_SEED_BASE + seed) if d[\\"pred_shuffle\\"] else None, d[\\"tags\\"], d[\\"related\\"])\\n",\n    "\\n",\n    "\\n",\n    "def graph_for(d, seed):\\n",\n    "    key = graph_key(d, seed)\\n",\n    "    if key not in GRAPHS:\\n",\n    "        tags = ALT_TAGS[d[\\"tags\\"]] if d[\\"tags\\"] != \\"base\\" else None\\n",\n    "        scores = ALT_SCORES[d[\\"tags\\"]] if d[\\"tags\\"] != \\"base\\" else TAG_SCORES\\n",\n    "        g = GraphV3(INV, RECORDS, SPLIT[\\"train\\"], RELATIONS2, TAXONOMY[d[\\"taxonomy\\"]], tag_scores=scores,\\n",\n    "                    contains=d[\\"contains\\"], conf_mode=d[\\"conf_mode\\"], conf_cal=CONF_CAL, rel_policy=d[\\"rel_policy\\"],\\n",\n    "                    path_mode=d[\\"path_mode\\"], masked=masked_types(d[\\"mask\\"]), rewire_seed=key[6], pred_shuffle_seed=key[7],\\n",\n    "                    tags_override=tags, related_cfg=RELATED_CFGS[d[\\"related\\"]], verified_cells=VERIFIED_CELLS,\\n",\n    "                    name=\\"|\\".join(str(x) for x in key))\\n",\n    "        assert g.n_nodes == INV.n_nodes and g.inv is INV, \\"inventory phải cố định\\"\\n",\n    "        GRAPHS[key] = (g, EngineV3(g, VID_FEAT, TXT_FEAT, QTOK2, QPREDS))\\n",\n    "    return GRAPHS[key]\\n",\n    "\\n",\n    "\\n",\n    "def make_part(name, std_only=False):\\n",\n    "    \\"\\"\\"Tập truy vấn + gallery. \'test\' (MSVD, thăm dò) và \'ext\' (xác nhận) chỉ mở sau khi đóng băng.\\n",\n    "    std_only: chỉ caption chuẩn 1k-A của tập ngoài (truy vấn PHỤ).\\"\\"\\"\\n",\n    "    if name in (\\"test\\", \\"ext\\"):\\n",\n    "        TEST_GATE.check(f\\"make_part({name})\\", name)\\n",\n    "    q = np.where((CAP_SPLIT == name) & (CAP_STD if std_only else True))[0]\\n",\n    "    return dict(name=name + (\\"_std\\" if std_only else \\"\\"), split=name, q_idx=q, gold=CAP_VID[q], gallery=np.array(SPLIT[name]))\\n",\n    "\\n",\n    "\\n",\n    "def _mean_std(vals):\\n",\n    "    return float(np.mean(vals)), float(np.std(vals))\\n",\n    "\\n",\n    "\\n",\n    "def not_run(exp_id, stage, desc, reason, design=None, hypothesis=\\"\\"):\\n",\n    "    \\"\\"\\"Ghi thí nghiệm KHÔNG chạy được (kèm lý do) vào manifest — không tạo số liệu thay thế.\\"\\"\\"\\n",\n    "    EXPS[exp_id] = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=design, seeds=[],\\n",\n    "                        runs={}, status=\\"NOT RUN\\", reason=reason, val_MRR=(float(\\"nan\\"), float(\\"nan\\")),\\n",\n    "                        val_R1=(float(\\"nan\\"), float(\\"nan\\")), edges={}, n_params=0, seconds=0.0)\\n",\n    "    NOT_RUN.append(dict(stage=stage, experiment_id=exp_id, reason=reason))\\n",\n    "    print(f\\"  [{stage}] {exp_id:28s} NOT RUN — {reason}\\")\\n",\n    "    return EXPS[exp_id]\\n",\n    "\\n",\n    "\\n",\n    "def run_exp(exp_id, stage, design, desc, hypothesis=\\"\\", seeds=None, lam_fixed=None, reuse=None):\\n",\n    "    \\"\\"\\"Huấn luyện (hoặc tái dùng mô hình của `reuse`) cho từng seed; chọn (α, λ) trên VAL; ghi manifest.\\"\\"\\"\\n",\n    "    seeds = seeds or CFG.seeds\\n",\n    "    rec = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=dict(design),\\n",\n    "               seeds=list(seeds), lam_fixed=lam_fixed, reuse=reuse, status=\\"RUNNING\\", runs={})\\n",\n    "    EXPS[exp_id] = rec\\n",\n    "    for seed in seeds:\\n",\n    "        g, eng = graph_for(design, seed)\\n",\n    "        t0 = time.time()\\n",\n    "        if DEVICE == \\"cuda\\":\\n",\n    "            torch.cuda.reset_peak_memory_stats()\\n",\n    "        if reuse is not None:\\n",\n    "            model, hist = EXPS[reuse][\\"runs\\"][seed][\\"model\\"], EXPS[reuse][\\"runs\\"][seed][\\"hist\\"]\\n",\n    "        elif design.get(\\"model\\") == \\"v4\\":\\n",\n    "            model, hist = train_v4(eng, TRAIN_Q, design, seed=seed)\\n",\n    "        else:\\n",\n    "            model, hist = train_graph_scorer_v3(eng, TRAIN_Q, SYM_FEAT, seed=seed, mp_weight=design[\\"mp_weight\\"],\\n",\n    "                                                init=design[\\"init\\"], pred_mode=design[\\"pred_mode\\"])\\n",\n    "        lam_v = 1.0 if design.get(\\"model\\") == \\"v4\\" else lam_fixed           # v4: một điểm đồ thị duy nhất (s_path = 0)\\n",\n    "        (a, l), grid, best = tune_on_val_v3(model, eng, PV, CFG.rerank_k, CLIP_RANK_VAL,\\n",\n    "                                            lam_grid=[lam_v] if lam_v is not None else None, protocol=design[\\"protocol\\"])\\n",\n    "        ck = None\\n",\n    "        if CFG.save_checkpoints and reuse is None:\\n",\n    "            ck = CFG.out_dir / \\"checkpoints\\" / f\\"{exp_id}_s{seed}.pt\\"\\n",\n    "            torch.save({k_: v.half() if v.is_floating_point() else v for k_, v in model.state_dict().items()}, ck)\\n",\n    "        rec[\\"runs\\"][seed] = dict(graph=g, engine=eng, model=model, hist=hist, alpha=a, lam=l, grid=grid,\\n",\n    "                                 val={k_: best[k_] for k_ in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}, rr_val=1.0 / best[\\"rank\\"],\\n",\n    "                                 seconds=round(time.time() - t0, 1), checkpoint=str(ck) if ck else None,\\n",\n    "                                 gpu_peak_mb=round(torch.cuda.max_memory_allocated() / 2 ** 20, 1) if DEVICE == \\"cuda\\" else None)\\n",\n    "    r0 = rec[\\"runs\\"][seeds[0]]\\n",\n    "    rec.update(status=\\"DONE\\", val_MRR=_mean_std([r[\\"val\\"][\\"MRR\\"] for r in rec[\\"runs\\"].values()]),\\n",\n    "               val_R1=_mean_std([r[\\"val\\"][\\"R@1\\"] for r in rec[\\"runs\\"].values()]),\\n",\n    "               rr_val=np.mean([r[\\"rr_val\\"] for r in rec[\\"runs\\"].values()], 0),\\n",\n    "               edges=r0[\\"graph\\"].edge_counts(), n_nodes=r0[\\"graph\\"].n_nodes,\\n",\n    "               n_params=int(sum(p.numel() for p in r0[\\"model\\"].parameters())),\\n",\n    "               alpha_lam=[(r[\\"alpha\\"], r[\\"lam\\"]) for r in rec[\\"runs\\"].values()],\\n",\n    "               seconds=float(sum(r[\\"seconds\\"] for r in rec[\\"runs\\"].values())),\\n",\n    "               gpu_peak_mb=max([r[\\"gpu_peak_mb\\"] or 0 for r in rec[\\"runs\\"].values()]) or None)\\n",\n    "    m, s = rec[\\"val_MRR\\"]\\n",\n    "    print(f\\"  [{stage}] {exp_id:28s} VAL MRR {m:.4f} ± {s:.4f} | α,λ {rec[\'alpha_lam\']} | \\"\\n",\n    "          f\\"{rec[\'seconds\']:.0f}s | cạnh quan hệ {sum(v for k_, v in rec[\'edges\'].items() if not k_.startswith(\'contains\'))}\\")\\n",\n    "    return rec\\n",\n    "\\n",\n    "\\n",\n    "def design_value(d, field):\\n",\n    "    return \\"/\\".join(str((d or {}).get(f)) for f in field.split(\\"|\\"))\\n",\n    "\\n",\n    "\\n",\n    "def decide(stage, incumbent, candidates, field):\\n",\n    "    \\"\\"\\"Luật đăng ký trước: incumbent = phương án ĐƠN GIẢN hơn (hoặc thiết kế đang dùng). Ứng viên chỉ được chấp nhận khi\\n",\n    "    Δ MRR val (trung bình seed) ≥ adopt_min_delta VÀ p(Δ ≤ 0) — bootstrap theo video, hiệu chỉnh Holm trên các ứng viên\\n",\n    "    của stage — < adopt_max_p. Nhiều ứng viên đạt ⇒ chọn Δ lớn nhất. Ứng viên NOT RUN bị bỏ qua (ghi lại).\\"\\"\\"\\n",\n    "    base = EXPS[incumbent]\\n",\n    "    cands = [c for c in candidates if EXPS[c][\\"status\\"] == \\"DONE\\"]\\n",\n    "    rows = []\\n",\n    "    for c in cands:\\n",\n    "        e = EXPS[c]\\n",\n    "        d = e[\\"val_MRR\\"][0] - base[\\"val_MRR\\"][0]\\n",\n    "        bs = cluster_bootstrap_diff(e[\\"rr_val\\"], base[\\"rr_val\\"], PV[\\"gold\\"])\\n",\n    "        rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=design_value(e[\\"design\\"], field),\\n",\n    "                         delta_val_MRR=d, ci_lo=bs[\\"ci_lo\\"], ci_hi=bs[\\"ci_hi\\"], p_not_better=bs[\\"p_not_better\\"]))\\n",\n    "    adj = holm([r[\\"p_not_better\\"] for r in rows])\\n",\n    "    best = None\\n",\n    "    for r, pa in zip(rows, adj):\\n",\n    "        r[\\"p_holm\\"] = float(pa)\\n",\n    "        r[\\"passes\\"] = bool(r[\\"delta_val_MRR\\"] >= CFG.adopt_min_delta and pa < CFG.adopt_max_p)\\n",\n    "        if r[\\"passes\\"] and (best is None or r[\\"delta_val_MRR\\"] > best[1]):\\n",\n    "            best = (r[\\"candidate\\"], r[\\"delta_val_MRR\\"])\\n",\n    "    for r in rows:\\n",\n    "        r[\\"adopted\\"] = best is not None and r[\\"candidate\\"] == best[0]\\n",\n    "    for c in candidates:\\n",\n    "        if EXPS[c][\\"status\\"] != \\"DONE\\":\\n",\n    "            rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=None, delta_val_MRR=float(\\"nan\\"),\\n",\n    "                             ci_lo=float(\\"nan\\"), ci_hi=float(\\"nan\\"), p_not_better=float(\\"nan\\"), p_holm=float(\\"nan\\"),\\n",\n    "                             passes=False, adopted=False, note=f\\"NOT RUN: {EXPS[c].get(\'reason\', \'\')}\\"))\\n",\n    "    DECISIONS.extend(rows)\\n",\n    "    winner = best[0] if best else incumbent\\n",\n    "    print(f\\"  → {stage}: {\'CHẤP NHẬN \' + winner if best else \'giữ \' + incumbent} ({field} = {design_value(EXPS[winner][\'design\'], field)})\\")\\n",\n    "    return winner\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def evaluate_exp(exp_id, part, clip_rank, protocol=None):\\n",\n    "    \\"\\"\\"Đánh giá MỘT lần mỗi seed với (α, λ) đã đóng băng; trả về thứ hạng theo truy vấn.\\"\\"\\"\\n",\n    "    e = EXPS[exp_id]\\n",\n    "    out = {}\\n",\n    "    for seed, r in e[\\"runs\\"].items():\\n",\n    "        sc = score_queries_v3(r[\\"model\\"], r[\\"engine\\"], part[\\"q_idx\\"], part[\\"gallery\\"], CFG.rerank_k,\\n",\n    "                              protocol or e[\\"design\\"][\\"protocol\\"])\\n",\n    "        fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], r[\\"alpha\\"], r[\\"lam\\"])\\n",\n    "        out[seed] = dict(rank=ranks_with_fallback(sc[\\"cand\\"], fin, part[\\"gold\\"], clip_rank),\\n",\n    "                         cand_hash=hashlib.sha1(sc[\\"cand\\"].tobytes()).hexdigest()[:12])\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def query_groups_v3(engine, q_idx):\\n",\n    "    \\"\\"\\"direct | indirect | other (như v1/v2) trên đồ thị của engine.\\"\\"\\"\\n",\n    "    g, q_idx = engine.graph, np.asarray(q_idx)\\n",\n    "    gold = CAP_VID[q_idx]\\n",\n    "    pf = engine.path_feats(gold[:, None], q_idx)[:, 0]\\n",\n    "    out = []\\n",\n    "    for j, q in enumerate(q_idx):\\n",\n    "        cids = {n_ for n_, _, _ in engine.q_concepts[q]}\\n",\n    "        out.append(\\"direct\\" if cids & set(g.vid_tags[gold[j]]) else (\\"indirect\\" if pf[j, 1:].sum() > 0 else \\"other\\"))\\n",\n    "    return np.array(out)\\n",\n    "\\n",\n    "\\n",\n    "def rare_mask(engine, q_idx, thr=5):\\n",\n    "    \\"\\"\\"Truy vấn \'khái niệm hiếm\': mọi node lớp khớp được có tần suất gắn nhãn trên video TRAIN ≤ thr (hoặc chỉ khớp khái niệm).\\"\\"\\"\\n",\n    "    g = engine.graph\\n",\n    "    out = []\\n",\n    "    for q in q_idx:\\n",\n    "        cls = [n_ for n_, _, _ in engine.q_concepts[q] if g.inv.is_class(n_)]\\n",\n    "        out.append(bool(engine.q_concepts[q]) and (not cls or max(g.cls_df[c] for c in cls) <= thr))\\n",\n    "    return np.array(out)\\n",\n    "\\n",\n    "\\n",\n    "def predicate_groups(engine, q_idx, thr=None):\\n",\n    "    \\"\\"\\"Chỉ xét vị từ của truy vấn CÓ trên cạnh của đồ thị (svo/prep): \'no_pred\' nếu không có; \'pred_rare\' nếu MỌI vị từ khớp\\n",\n    "    đều có support TRAIN ≤ thr (truy vấn chỉ dựa vào vị từ hiếm); ngược lại \'pred_common\'.\\"\\"\\"\\n",\n    "    thr = thr or 2 * CFG.pred_min_support\\n",\n    "    used = {pid for rel in PRED_FAMILIES for d in engine.graph.pshare[rel].values() for pid in d}\\n",\n    "    out = []\\n",\n    "    for q in q_idx:\\n",\n    "        pids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel] if p in used]\\n",\n    "        if not pids:\\n",\n    "            out.append(\\"no_pred\\")\\n",\n    "        else:\\n",\n    "            out.append(\\"pred_rare\\" if max(PRED_SUPPORT.get(PRED_VOCAB[p], 0) for p in pids) <= thr else \\"pred_common\\")\\n",\n    "    return np.array(out)\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def path_analysis(run, part, protocol=\\"inductive\\", max_q=4000, seed=0):\\n",\n    "    \\"\\"\\"Theo mẫu: phần đóng góp, knockout từng mẫu và từng HỌ quan hệ lúc suy luận; theo node: bậc, số đường đi đi qua,\\n",\n    "    phạm vi video; tổng hợp: ρ(degree, s_path), ρ(degree, điểm cuối), P(degree video đúng > video âm).\\"\\"\\"\\n",\n    "    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n",\n    "    q = part[\\"q_idx\\"]\\n",\n    "    if len(q) > max_q:\\n",\n    "        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))\\n",\n    "    gold = CAP_VID[q]\\n",\n    "    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n",\n    "    sc = score_queries_v3(model, eng, q, part[\\"gallery\\"], CFG.rerank_k, protocol)\\n",\n    "    pf = eng.path_feats(sc[\\"cand\\"], q)\\n",\n    "    w = model.pattern_weights().detach().cpu().numpy()\\n",\n    "    gain = F.softplus(model.path_gain).item()\\n",\n    "    contrib = gain * pf * w\\n",\n    "    tot = contrib.sum(-1, keepdims=True)\\n",\n    "    share = np.where(tot > 0, contrib / np.maximum(tot, 1e-12), 0).mean((0, 1))\\n",\n    "    reldeg_out = sum(np.asarray((g.A[t] > 0).sum(1)).ravel() for t in PATH_TYPES)\\n",\n    "    reldeg_in = sum(np.asarray((g.A[t] > 0).sum(0)).ravel() for t in PATH_TYPES)\\n",\n    "    con_in = np.asarray((g.A_con > 0).sum(0)).ravel()\\n",\n    "    vdeg = np.array([reldeg_out[g.vid_tags[v]].sum() for v in range(g.n_vid)])\\n",\n    "    cand_deg = vdeg[sc[\\"cand\\"]]\\n",\n    "    from scipy.stats import spearmanr\\n",\n    "    is_gold = (sc[\\"cand\\"] == gold[:, None])\\n",\n    "    base_fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], run[\\"lam\\"])\\n",\n    "    base_mrr = metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], base_fin, gold, clip_rank))[\\"MRR\\"]\\n",\n    "\\n",\n    "    def knock(idx):\\n",\n    "        pf2 = pf.copy(); pf2[..., idx] = 0\\n",\n    "        sp2 = znorm_per_row(gain * (pf2 * w).sum(-1))\\n",\n    "        return metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sp2, run[\\"alpha\\"], run[\\"lam\\"]),\\n",\n    "                                                      gold, clip_rank))[\\"MRR\\"]\\n",\n    "\\n",\n    "    rows = []\\n",\n    "    for p_i, p in enumerate(PATTERNS):\\n",\n    "        mrr = knock([p_i])\\n",\n    "        reach = np.asarray((g.R[p] > 0).sum(0)).ravel()\\n",\n    "        rows.append(dict(section=\\"pattern\\", split=part[\\"name\\"], item=p, weight=float(w[p_i]), nnz=int(g.R[p].nnz),\\n",\n    "                         frac_pairs_nonzero=float((pf[..., p_i] > 0).mean()), contribution_share=float(share[p_i]),\\n",\n    "                         knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr, max_video_reach=int(reach.max()) if reach.size else 0))\\n",\n    "    groups = {f\\"family:{f}\\": [PIDX[p] for p in PATTERNS if p.startswith(f\\"P1_{f}\\") or p.startswith(f\\"P4_{f}\\")\\n",\n    "                              or p == f\\"P1m_{FAMILY_REL[f]}\\"] for f in FAMILIES}\\n",\n    "    groups.update({\\"family:related\\": [PIDX[\\"P3_related\\"], PIDX[\\"P5_related_isa\\"]],\\n",\n    "                   \\"family:isa\\": [PIDX[\\"P2_isa\\"]] + [PIDX[f\\"P4_{f}_isa\\"] for f in FAMILIES] + [PIDX[\\"P5_related_isa\\"]],\\n",\n    "                   \\"all_indirect\\": list(range(1, len(PATTERNS)))})\\n",\n    "    for nm, idx in groups.items():\\n",\n    "        mrr = knock(idx)\\n",\n    "        rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=nm, knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr,\\n",\n    "                         contribution_share=float(share[idx].sum())))\\n",\n    "    reach_all = sum(np.asarray((g.R[p] > 0).sum(0)).ravel()[:g.n_nodes] for p in PATTERNS[1:] if not p.startswith(\\"P1m_\\"))\\n",\n    "    through = con_in * reldeg_out                         # số đường đi 2 bước video → node → lớp/khái niệm đi QUA node\\n",\n    "    for i in np.argsort(-through)[:50]:\\n",\n    "        if through[i] <= 0:\\n",\n    "            break\\n",\n    "        rows.append(dict(section=\\"node\\", split=part[\\"name\\"], item=g.inv.name[i], node_type=g.inv.semantic_type(i),\\n",\n    "                         videos_tagged=int(con_in[i]), rel_out_degree=int(reldeg_out[i]), rel_in_degree=int(reldeg_in[i]),\\n",\n    "                         paths_through=int(through[i]), videos_reached_indirect=int(reach_all[i]), idf=float(g.idf[i])))\\n",\n    "    flat_deg, flat_sp, flat_fin = cand_deg.ravel(), sc[\\"path_raw\\"].ravel(), base_fin.ravel()\\n",\n    "    gold_rank = ranks_with_fallback(sc[\\"cand\\"], base_fin, gold, clip_rank)\\n",\n    "    gold_deg = vdeg[gold]\\n",\n    "    deg_auc = float(np.mean([(cand_deg[k][is_gold[k]] > cand_deg[k][~is_gold[k]]).mean()\\n",\n    "                              for k in range(len(q)) if is_gold[k].any()])) if is_gold.any() else float(\\"nan\\")\\n",\n    "    summary = dict(split=part[\\"name\\"], n_queries=len(q), base_MRR=base_mrr,\\n",\n    "                   spearman_degree_vs_s_path=float(spearmanr(flat_deg, flat_sp).correlation),\\n",\n    "                   spearman_degree_vs_final=float(spearmanr(flat_deg, flat_fin).correlation),\\n",\n    "                   spearman_gold_degree_vs_gold_rank=float(spearmanr(gold_deg, gold_rank).correlation),\\n",\n    "                   p_gold_degree_gt_negative=deg_auc, w_path_effective=(1 - run[\\"alpha\\"]) * (1 - run[\\"lam\\"]),\\n",\n    "                   hubs=[dict(node=g.inv.name[i], paths_through=int(through[i])) for i in np.argsort(-through)[:10]])\\n",\n    "    rows.append(dict(section=\\"summary\\", split=part[\\"name\\"], item=\\"degree_bias\\",\\n",\n    "                     **{k: v for k, v in summary.items() if k not in (\\"hubs\\", \\"split\\")}))\\n",\n    "    return pd.DataFrame(rows), summary\\n",\n    "\\n",\n    "\\n",\n    "# ---- họ so sánh CHÍNH (đăng ký trước, một phía: a > b), hiệu chỉnh Holm; chỉ trên tập ngoài mới là XÁC NHẬN ----\\n",\n    "PRIMARY_COMPARISONS = [(\\"H1_graph_vs_clip\\", \\"G3\\", \\"G0\\"), (\\"H2_relations_vs_contains_only\\", \\"G3\\", \\"G1\\"),\\n",\n    "                       (\\"H3_predicate_vs_family_only\\", \\"G3\\", \\"G2\\"), (\\"H4_true_vs_shuffled_predicates\\", \\"G3\\", \\"G4\\"),\\n",\n    "                       (\\"H5_true_vs_rewired_topology\\", \\"G3\\", \\"G5\\"), (\\"H6_query_conditioned_vs_static\\", \\"G3\\", \\"G6\\"),\\n",\n    "                       (\\"H7_v4_vs_v3\\", \\"G3\\", \\"V3REF\\"),\\n",\n    "                       (\\"H8_role_edges\\", \\"G3\\", \\"G7\\"), (\\"H9_svo_prep_edges\\", \\"G3\\", \\"G8\\"),\\n",\n    "                       (\\"H10_kg_vs_contains_only\\", \\"G9\\", \\"G1\\"), (\\"H11_kg_true_vs_rewired\\", \\"G9\\", \\"G10\\"),\\n",\n    "                       (\\"H12_kg_adds_to_graph\\", \\"G9\\", \\"G3\\"), (\\"H13_learned_vs_zeroshot_tags\\", \\"G3\\", \\"G11\\")]\\n",\n    "H_V4 = (\\"quan hệ được dùng CÓ ĐIỀU KIỆN THEO TRUY VẤN (loại cạnh, vị từ khớp truy vấn, vai của node trong truy vấn) cải thiện \\"\\n",\n    "        \\"retrieval so với cùng mô hình không có quan hệ / quan hệ sai\\")"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c043",\n   "metadata": {},\n   "source": [\n    "## 10. Self-test v3 (mục 12)\\n",\n    "| mục 12 | kiểm tra | id |\\n",\n    "|---|---|---|\\n",\n    "| 1 | embedding vị từ tác động forward; pred_mode none (≡ v2) **không** bị ảnh hưởng | P1 |\\n",\n    "| 2 | hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác | P2 |\\n",\n    "| 3 | rewiring giữ kiểu node, họ, bậc, trọng số, đa tập vị từ | G6 |\\n",\n    "| 4 | hoán vị vị từ không đổi topology / trọng số | G7 |\\n",\n    "| 5 | mỗi ablation tắt đúng thành phần; inventory cố định | AB, G1 |\\n",\n    "| 6 | trọng số chỉ nhân một lần (khớp tính tay, bất biến tỉ lệ) | G4 |\\n",\n    "| 7 | node không có hàng xóm không sinh NaN; cô lập inductive | G4, G5 |\\n",\n    "| 8 | không rò rỉ caption train/val/test/tập ngoài | C1, C1b, C2, C3, C8, C9 |\\n",\n    "| 9 | giải thích đúng vị từ, đúng chiều; điểm khớp ma trận | G3, X1 |\\n",\n    "| 10 | số trong báo cáo khớp CSV đầu ra | R1 |\\n",\n    "\\n",\n    "Một kiểm thử phần mềm PASS **không** có nghĩa giả thuyết khoa học đã được xác nhận."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c044",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "CANARY_NOUNS = [\\"flamingo\\", \\"accordion\\", \\"kayak\\", \\"harmonica\\", \\"trombone\\", \\"anvil\\", \\"walrus\\", \\"xylophone\\",\\n",\n    "                \\"gondola\\", \\"hedgehog\\", \\"unicycle\\", \\"periscope\\", \\"tuba\\", \\"iguana\\", \\"llama\\"]\\n",\n    "CANARY_VERBS = [\\"juggle\\", \\"knit\\", \\"yodel\\", \\"whittle\\", \\"embroider\\", \\"lasso\\"]\\n",\n    "\\n",\n    "\\n",\n    "def _same_graph(g1, g2):\\n",\n    "    if g1.n_nodes != g2.n_nodes or g1.inv.sha1 != g2.inv.sha1:\\n",\n    "        return False\\n",\n    "    if (g1.A_con != g2.A_con).nnz:\\n",\n    "        return False\\n",\n    "    return (all((g1.A[t] != g2.A[t]).nnz == 0 for t in RELATION_TYPES) and np.array_equal(g1.idf, g2.idf)\\n",\n    "            and g1.pdist == g2.pdist)\\n",\n    "\\n",\n    "\\n",\n    "def _pred_vocab_of(rel):\\n",\n    "    s = Counter()\\n",\n    "    for v in SPLIT[\\"train\\"]:\\n",\n    "        s.update({(r, p) for _, r, _, p in rel[v]})\\n",\n    "    return sorted(k for k in s if k[0] in PRED_FAMILIES)\\n",\n    "\\n",\n    "\\n",\n    "def test_canary(splits=(\\"test\\", \\"ext\\"), n_videos=40):\\n",\n    "    \\"\\"\\"Canary chỉ có trong caption TEST / tập ngoài không được lọt vào từ vựng, quan hệ, từ vựng vị từ; đối chứng dương:\\n",\n    "    đặt canary vào TRAIN thì phải phát hiện được.\\"\\"\\"\\n",\n    "    terms = {k.rsplit(\\"#\\", 1)[0] for k in VOCAB2[\\"terms\\"]}\\n",\n    "    a, b = [w for w in CANARY_NOUNS if w not in terms and is_visual_noun(w)][:2]\\n",\n    "    seen_pred = {t for _, p in PRED_VOCAB for t in p.split()}           # động từ canary cũng không được là vị từ train\\n",\n    "    v = next((w for w in CANARY_VERBS if w not in terms and w not in seen_pred and is_action_verb(w)),\\n",\n    "             next(w for w in CANARY_VERBS if w not in terms and is_action_verb(w)))\\n",\n    "    canary = [f\\"a {a} is {naive_gerund(v)} a {b}\\"] * 3\\n",\n    "    tv = [x for s in splits for x in SPLIT[s][:n_videos]]\\n",\n    "    store = CAPS.replaced({x: CAPS._caps[x] + canary for x in tv})\\n",\n    "    voc = build_vocab_v2(store, SPLIT)\\n",\n    "    rel = build_relations_v2(store, SPLIT, voc)\\n",\n    "    keys = {ckey(a, \\"n\\"), ckey(b, \\"n\\"), ckey(v, \\"v\\")}\\n",\n    "    leak_v = sorted(keys & set(voc[\\"terms\\"]))\\n",\n    "    leak_r = [t for trip in rel.values() for t in trip if {t[0], t[2]} & keys]\\n",\n    "    leak_p = sorted(set(_pred_vocab_of(rel)) - set(PRED_VOCAB))          # vị từ MỚI so với từ vựng vị từ train\\n",\n    "    fake = dict(SPLIT, train=list(SPLIT[\\"train\\"]) + list(tv))\\n",\n    "    store_pos = store.replaced({}, new_split={x: \\"train\\" for x in tv})\\n",\n    "    voc_pos = build_vocab_v2(store_pos, fake)\\n",\n    "    rel_pos = build_relations_v2(store_pos, fake, voc_pos)\\n",\n    "    detected = {ckey(a, \\"n\\"), ckey(b, \\"n\\")} <= set(voc_pos[\\"terms\\"]) and any({t[0], t[2]} & keys for trip in rel_pos.values() for t in trip)\\n",\n    "    return dict(canary=f\\"{a} –{v}→ {b}\\", splits=list(splits), n_videos=len(tv), leak_vocab=leak_v, leak_relations=leak_r[:3],\\n",\n    "                leak_predicates=leak_p[:3], positive_control_detected=bool(detected))\\n",\n    "\\n",\n    "\\n",\n    "def test_invariance():\\n",\n    "    \\"\\"\\"Xáo / thay caption TEST và tập ngoài rồi dựng lại TOÀN BỘ: từ vựng, quan hệ, vị từ, tagger, taxonomy, đồ thị.\\"\\"\\"\\n",\n    "    tv = list(SPLIT[\\"test\\"]) + list(SPLIT[\\"ext\\"])\\n",\n    "    perm = np.random.RandomState(0).permutation(tv)\\n",\n    "    junk = [\\"a purple elephant is skiing on the moon\\", \\"zebra piano astronaut volcano juggling\\"] * 4\\n",\n    "    store = CAPS.replaced({x: (CAPS._caps[int(p)] if k % 2 == 0 else junk) for k, (x, p) in enumerate(zip(tv, perm))})\\n",\n    "    voc = build_vocab_v2(store, SPLIT)\\n",\n    "    rel = build_relations_v2(store, SPLIT, voc)\\n",\n    "    tags, scores, sel, _ = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL[\\"n_frames\\"], voc, SPLIT, store)\\n",\n    "    tax, cmeta, _, _ = build_taxonomy_v2(voc, store, SPLIT, {v: tags[v] for v in SPLIT[\\"train\\"]})\\n",\n    "    recs = [dict(r, tags=tags[r[\\"vid\\"]]) for r in RECORDS]\\n",\n    "    inv = NodeInventory(recs, voc, cmeta)\\n",\n    "    g2 = GraphV3(inv, recs, SPLIT[\\"train\\"], rel, tax[\\"heuristic\\"], tag_scores=scores)\\n",\n    "    g1 = graph_for(DESIGN_BASE, CFG.seeds[0])[0]\\n",\n    "    return dict(vocab=voc[\\"terms\\"] == VOCAB2[\\"terms\\"], relations=rel == RELATIONS2, predicate_vocab=_pred_vocab_of(rel) == PRED_VOCAB,\\n",\n    "                tagger_selection=sel == TAG_SELECT2, tags=tags == TAGS2, taxonomy=tax == TAXONOMY, inventory=inv.sha1 == INV.sha1,\\n",\n    "                graph=_same_graph(g1, g2), annotation_sample=build_annotation_sample(store, SPLIT, voc, RECORDS)[0]\\n",\n    "                .sample_id.tolist() == ANN_SAMPLE.sample_id.tolist(),\\n",\n    "                automatic_verification=(build_agreement(store, SPLIT, voc)[0] == AGREE) if AGREE else True)\\n",\n    "\\n",\n    "\\n",\n    "VARIANT_DESIGNS = ([dict(DESIGN_BASE, mask=m) for m in MASKS] + [dict(DESIGN_BASE, rewire=True), dict(DESIGN_BASE, pred_shuffle=True),\\n",\n    "                   dict(DESIGN_BASE, rel_policy=\\"support\\"), dict(DESIGN_BASE, taxonomy=\\"none\\"), dict(DESIGN_BASE, contains=\\"conf\\"),\\n",\n    "                   dict(DESIGN_BASE, related=\\"npmi20\\"), dict(DESIGN_BASE, path_mode=\\"pred_match\\")])\\n",\n    "\\n",\n    "\\n",\n    "def test_inventory_constant():\\n",\n    "    \\"\\"\\"Mọi biến thể đồ thị dùng CÙNG inventory; với CÙNG cấu hình mô hình, số tham số không đổi giữa các biến thể đồ thị.\\"\\"\\"\\n",\n    "    gs = [graph_for(d, CFG.seeds[0]) for d in VARIANT_DESIGNS]\\n",\n    "    params = {pm: sorted({sum(p.numel() for p in make_model(e, SYM_FEAT, pred_mode=pm).parameters()) for _, e in gs[:4]})\\n",\n    "              for pm in PRED_MODES}\\n",\n    "    return dict(n_graphs=len(gs), node_counts=sorted({g.n_nodes for g, _ in gs}), same_inventory=all(g.inv is INV for g, _ in gs),\\n",\n    "                param_counts_by_pred_mode=params)\\n",\n    "\\n",\n    "\\n",\n    "def test_schema_conformance(g):\\n",\n    "    \\"\\"\\"Mọi cạnh: kiểu nút đầu/cuối + POS đúng REL_SPEC; không self-loop; trọng số trong (0, 1]; vị từ đúng họ:\\n",\n    "    svo/prep ⇒ mọi vị từ thuộc từ vựng vị từ của họ đó (không lẫn tên loại quan hệ); agent/object ⇒ vị từ = lemma động từ.\\"\\"\\"\\n",\n    "    bad = Counter()\\n",\n    "    inv = g.inv\\n",\n    "    fam_names = set(REL_SPEC) | set(FAMILIES)\\n",\n    "    for t in [\\"contains\\"] + RELATION_TYPES:\\n",\n    "        M = (g.A_con if t == \\"contains\\" else g.A[t]).tocoo()\\n",\n    "        spec = REL_SPEC[t]\\n",\n    "        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n",\n    "        for u, v, w in zip(M.row.tolist(), M.col.tolist(), M.data.tolist()):\\n",\n    "            src_ok = spec[\\"src\\"].split(\\":\\")[0].split(\\"|\\")\\n",\n    "            tgt_ok = spec[\\"tgt\\"].split(\\" \\")[0].split(\\":\\")[0].split(\\"|\\")\\n",\n    "            if inv.node_type[u] not in src_ok or inv.node_type[v] not in tgt_ok:\\n",\n    "                bad[f\\"{t}:type\\"] += 1\\n",\n    "            if \\":\\" in spec[\\"src\\"] and inv.semantic_type(u) != spec[\\"src\\"].split(\\":\\")[1]:\\n",\n    "                bad[f\\"{t}:src_pos\\"] += 1\\n",\n    "            if \\":\\" in spec[\\"tgt\\"] and inv.semantic_type(v) != spec[\\"tgt\\"].split(\\":\\")[1]:\\n",\n    "                bad[f\\"{t}:tgt_pos\\"] += 1\\n",\n    "            if t in (\\"isa\\", \\"isa_inv\\") and inv.pos[u] != inv.pos[v]:\\n",\n    "                bad[f\\"{t}:pos_mismatch\\"] += 1\\n",\n    "            if u == v:\\n",\n    "                bad[f\\"{t}:self_loop\\"] += 1\\n",\n    "            if not (0 < w <= 1 + 1e-6):\\n",\n    "                bad[f\\"{t}:weight\\"] += 1\\n",\n    "            if base in REL_CATS:\\n",\n    "                d = g.predicates_of(t, u, v)\\n",\n    "                if not d:\\n",\n    "                    bad[f\\"{t}:no_predicate\\"] += 1\\n",\n    "                if any(p in fam_names for p in d):\\n",\n    "                    bad[f\\"{t}:predicate_is_relation_name\\"] += 1\\n",\n    "                if base in PRED_FAMILIES and any((base, p) not in PRED_ID for p in d):\\n",\n    "                    bad[f\\"{t}:predicate_not_in_vocab\\"] += 1\\n",\n    "                if base in (\\"agent_of\\", \\"object_of\\"):\\n",\n    "                    verb = inv.lemma[v if not t.endswith(\\"_inv\\") else u]\\n",\n    "                    if set(d) != {verb}:\\n",\n    "                        bad[f\\"{t}:predicate_not_verb_node\\"] += 1\\n",\n    "    return dict(violations=dict(bad), n_checked=int(g.A_con.nnz + sum(g.A[t].nnz for t in RELATION_TYPES)))\\n",\n    "\\n",\n    "\\n",\n    "def test_path_identity(n=200):\\n",\n    "    \\"\\"\\"Điểm ma trận (engine) = tổng/max liệt kê đường đi ở MỌI path_mode (kể cả pred_match); tính lại R khớp graph.R;\\n",\n    "    mọi đường đi khớp vị từ ghi đúng vị từ của đúng cạnh theo đúng chiều; đối chứng âm: sửa pred_id ⇒ bị phát hiện.\\"\\"\\"\\n",\n    "    out = {}\\n",\n    "    rng = np.random.RandomState(0)\\n",\n    "    pool = np.where(np.isin(CAP_SPLIT, [\\"train\\", \\"val\\"]))[0]\\n",\n    "    qs = rng.choice(pool, min(n, len(pool)), replace=False)\\n",\n    "    pairs = [(q, int(CAP_VID[q])) for q in qs] + [(q, int(v)) for q in qs for v in rng.choice(SPLIT[\\"train\\"], 2)]\\n",\n    "    for mode in (\\"weighted\\", \\"count\\", \\"degnorm\\", \\"dedup\\", \\"pred_match\\"):\\n",\n    "        g, eng = graph_for(dict(DESIGN_BASE, path_mode=mode), CFG.seeds[0])\\n",\n    "        bad = cnt = npred = pbad = neg = 0\\n",\n    "        for q, v in pairs:                               # video đúng + video ngẫu nhiên (có nhiều đường đi gián tiếp hơn)\\n",\n    "            cids, qpred, qtok = q_args(eng, q)\\n",\n    "            if not cids:\\n",\n    "                continue\\n",\n    "            cnt += 1\\n",\n    "            paths = enumerate_paths_v3(g, v, cids, qpred)\\n",\n    "            bad += int(not np.allclose(eng.path_feats(np.array([[v]]), np.array([q]))[0, 0],\\n",\n    "                                       paths_to_counts_v3(paths, mode), atol=1e-5))\\n",\n    "            for p in paths:\\n",\n    "                if p.get(\\"pred\\") is not None:\\n",\n    "                    npred += 1\\n",\n    "                    pbad += int(not pred_valid_v3(g, p, v, qtok))\\n",\n    "                    wrong = dict(p, pred=(p[\\"pred\\"][0], (p[\\"pred\\"][1] + 1) % len(PRED_VOCAB), p[\\"pred\\"][2]))\\n",\n    "                    neg += int(not pred_valid_v3(g, wrong, v, qtok))\\n",\n    "        R2 = g.compute_R(rows=np.arange(g.n_vid), PM=g.path_mats())\\n",\n    "        diff = max(float(abs(R2[p] - g.R[p]).max()) if (R2[p] - g.R[p]).nnz else 0.0 for p in PATTERNS)\\n",\n    "        out[mode] = dict(pairs=cnt, mismatches=bad, recompute_maxdiff=diff, predicate_paths=npred, predicate_invalid=pbad,\\n",\n    "                         negative_control_detected=neg)\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_mp_weights():\\n",\n    "    \\"\\"\\"Unit test TypedSAGEv3: trọng số cạnh dùng trong forward (weighted), bị bỏ qua (binary), w=0 ≡ không cạnh, node cô lập\\n",\n    "    hữu hạn, bất biến thứ tự cạnh, gradient hữu hạn; trọng số chỉ nhân MỘT lần: (i) khớp công thức tính tay, (ii) weighted_mean\\n",\n    "    bất biến khi nhân mọi trọng số với hằng số (nhân lặp sẽ làm đổi), (iii) cùng hệ số cho h_u và vị từ e_uv.\\"\\"\\"\\n",\n    "    torch.manual_seed(0)\\n",\n    "    res = {}\\n",\n    "    x = torch.randn(4, 8)\\n",\n    "    ei = torch.tensor([[1, 2, 3], [0, 0, 0]])\\n",\n    "    et = torch.zeros(3, dtype=torch.long)\\n",\n    "    pidx = torch.tensor([0, 1, 2])\\n",\n    "    ph = torch.randn(3, 8)\\n",\n    "    for mode in (\\"binary\\", \\"weighted_mean\\", \\"weighted_symnorm\\"):\\n",\n    "        L = TypedSAGEv3(8, 8, 1, [0], 0.0, mode, pred_type_ids=[0])\\n",\n    "        w1 = torch.tensor([1.0, 1.0, 1.0]); w2 = torch.tensor([1.0, 0.0, 1.0]); w3 = torch.tensor([3.0, 1.0, 1.0])\\n",\n    "        o1, o2, o3 = L(x, ei, et, w1, pidx, ph), L(x, ei, et, w2, pidx, ph), L(x, ei, et, w3, pidx, ph)\\n",\n    "        perm = torch.tensor([2, 0, 1])\\n",\n    "        o_perm = L(x, ei[:, perm], et[perm], w3[perm], pidx[perm], ph)\\n",\n    "        o_drop = L(x, ei[:, [0, 2]], et[[0, 2]], torch.tensor([1.0, 1.0]), pidx[[0, 2]], ph)\\n",\n    "        o_scaled = L(x, ei, et, 5 * w3, pidx, ph)\\n",\n    "        ww = torch.ones(3) if mode == \\"binary\\" else w3\\n",\n    "        coef = ww / ww.sum() if mode != \\"weighted_symnorm\\" else ww / torch.sqrt(ww * ww.sum())\\n",\n    "        manual = L.self_w(x[0]) + torch.sigmoid(L.gate[0]) * (L.neigh_w[0]((coef[:, None] * x[1:4]).sum(0))\\n",\n    "                                                               + L.pred_w[\\"0\\"]((coef[:, None] * ph).sum(0)))\\n",\n    "        xg = x.clone().requires_grad_(True)\\n",\n    "        L(xg, ei, et, w3, pidx, ph).sum().backward()\\n",\n    "        changes = bool((o1[0] - o3[0]).abs().max() > 1e-6)\\n",\n    "        zero_ok = bool((o2[0] - o_drop[0]).abs().max() < 1e-5)\\n",\n    "        res[mode] = dict(weight_changes_output=changes, zero_weight_equals_edge_removed=zero_ok,\\n",\n    "                         isolated_nodes_finite=bool(torch.isfinite(o1[1:]).all()),\\n",\n    "                         edge_order_invariant=bool((o3 - o_perm).abs().max() < 1e-5),\\n",\n    "                         grad_finite=bool(torch.isfinite(xg.grad).all()),\\n",\n    "                         matches_formula=bool((o3[0] - manual).abs().max() < 1e-5),\\n",\n    "                         scale_invariant=bool((o3[0] - o_scaled[0]).abs().max() < 1e-5),\\n",\n    "                         as_designed=((not changes) if mode == \\"binary\\" else (changes and zero_ok))\\n",\n    "                         and bool((o3[0] - manual).abs().max() < 1e-5)\\n",\n    "                         and (mode != \\"weighted_mean\\" or bool((o3[0] - o_scaled[0]).abs().max() < 1e-5)))\\n",\n    "    return res\\n",\n    "\\n",\n    "\\n",\n    "def test_predicate_forward():\\n",\n    "    \\"\\"\\"(1) Embedding vị từ thật sự tác động forward: đổi embedding của các vị từ đang dùng ⇒ biểu diễn node đổi khi\\n",\n    "    pred_mode ∈ {clip, clip_adapt}; pred_mode = none (relation-family-only, ≡ v2) PHẢI không đổi (đối chứng âm).\\n",\n    "    (2) Hai vị từ khác nhau trên CÙNG cặp node tạo thông điệp khác nhau (trường hợp kiểm soát, embedding CLIP thật).\\"\\"\\"\\n",\n    "    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])\\n",\n    "    used = sorted({pid for rel in PRED_FAMILIES for d in g.pshare[rel].values() for pid in d})\\n",\n    "    out = {}\\n",\n    "    for pm in PRED_MODES:\\n",\n    "        torch.manual_seed(0)\\n",\n    "        model = make_model(eng, SYM_FEAT, pred_mode=pm).eval()\\n",\n    "        if pm == \\"clip_adapt\\":                         # MLP dư khởi tạo 0 ⇒ cho trọng số khác 0 để kiểm tra cả nhánh dư\\n",\n    "            with torch.no_grad():\\n",\n    "                for prm in model.pred_enc.res.parameters():\\n",\n    "                    prm.normal_(0, 0.05)\\n",\n    "        with torch.no_grad():\\n",\n    "            h1 = model.encode_graph(*eng.edges_on(\\"train\\"))\\n",\n    "            if pm != \\"none\\" and used:\\n",\n    "                old = model.pred_emb.clone()\\n",\n    "                model.pred_emb[used] = torch.roll(model.pred_emb[used], 1, 0)\\n",\n    "                h2 = model.encode_graph(*eng.edges_on(\\"train\\"))\\n",\n    "                model.pred_emb.copy_(old)\\n",\n    "            else:\\n",\n    "                ed = list(eng.edges_on(\\"train\\"))\\n",\n    "                ed[3] = torch.full_like(ed[3], -1)                       # gỡ mọi chỉ số vị từ\\n",\n    "                h2 = model.encode_graph(*ed)\\n",\n    "        out[pm] = dict(max_change=float((h1 - h2).abs().max()), finite=bool(torch.isfinite(h1).all()))\\n",\n    "    out[\\"predicate_used_as_designed\\"] = (out[\\"clip\\"][\\"max_change\\"] > 1e-5 and out[\\"clip_adapt\\"][\\"max_change\\"] > 1e-5\\n",\n    "                                         and out[\\"none\\"][\\"max_change\\"] < 1e-7)\\n",\n    "    pick = {}\\n",\n    "    for want in (\\"ride\\", \\"feed\\", \\"play\\", \\"hold\\"):\\n",\n    "        if (\\"svo\\", want) in PRED_ID:\\n",\n    "            pick[want] = PRED_ID[(\\"svo\\", want)]\\n",\n    "    pair = list(pick.items())[:2] if len(pick) >= 2 else [(PRED_VOCAB[i][1], i) for i in used[:2]]\\n",\n    "    torch.manual_seed(0)\\n",\n    "    L = TypedSAGEv3(8, 8, 1, [0], 0.0, \\"binary\\", pred_type_ids=[0])\\n",\n    "    enc = PredicateEncoder(PRED_EMB.shape[1], 8, \\"clip\\", 16)\\n",\n    "    pe = enc(torch.eye(len(PRED_VOCAB))[[pair[0][1], pair[1][1]]], torch.from_numpy(PRED_EMB))\\n",\n    "    x = torch.randn(2, 8)\\n",\n    "    ei, et, w = torch.tensor([[0], [1]]), torch.zeros(1, dtype=torch.long), torch.ones(1)\\n",\n    "    with torch.no_grad():\\n",\n    "        a = L(x, ei, et, w, torch.tensor([0]), pe)[1]\\n",\n    "        b = L(x, ei, et, w, torch.tensor([1]), pe)[1]\\n",\n    "    out[\\"two_predicates\\"] = dict(predicates=[pair[0][0], pair[1][0]], message_diff=float((a - b).abs().max()),\\n",\n    "                                 differ=bool((a - b).abs().max() > 1e-6))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_ablations(g0):\\n",\n    "    \\"\\"\\"Mỗi ablation tắt ĐÚNG thành phần đã mô tả: mask ⇒ loại cạnh bị tắt = 0, loại khác giữ nguyên; support ⇒ mọi vị từ còn\\n",\n    "    lại đạt ngưỡng; pred_mode none ⇒ mô hình không có tham số vị từ; contains_only ⇒ chỉ còn contains.\\"\\"\\"\\n",\n    "    out = {}\\n",\n    "    for m, types in MASKS.items():\\n",\n    "        g = graph_for(dict(DESIGN_BASE, mask=m), CFG.seeds[0])[0]\\n",\n    "        off_ok = all(g.A[t].nnz == 0 for t in types)\\n",\n    "        rest_ok = all((g.A[t] != g0.A[t]).nnz == 0 for t in RELATION_TYPES if t not in types) and (g.A_con != g0.A_con).nnz == 0\\n",\n    "        out[f\\"mask:{m}\\"] = bool(off_ok and rest_ok)\\n",\n    "    gs = graph_for(dict(DESIGN_BASE, rel_policy=\\"support\\"), CFG.seeds[0])[0]\\n",\n    "    out[\\"policy:support\\"] = all(PRED_SUPPORT[(r_, p)] >= CFG.pred_min_support for (r_, a, b), d in gs.pdist.items()\\n",\n    "                                if gs.A[r_][a, b] != 0 for p in d)\\n",\n    "    if AGREE:\\n",\n    "        ga = graph_for(dict(DESIGN_BASE, rel_policy=\\"agreement\\"), CFG.seeds[0])[0]\\n",\n    "        out[\\"policy:agreement\\"] = all(agreement_rate(r_, INV.id2key[a], INV.id2key[b], p) >= CFG.agree_min\\n",\n    "                                      for (r_, a, b), d in ga.pdist.items() if ga.A[r_][a, b] != 0 for p in d)\\n",\n    "    m0 = make_model(E_BASE, SYM_FEAT, pred_mode=\\"none\\")\\n",\n    "    out[\\"pred_mode:none_has_no_predicate_params\\"] = not any(\\"pred\\" in n_ for n_, _ in m0.named_parameters())\\n",\n    "    m1 = make_model(E_BASE, SYM_FEAT, pred_mode=\\"clip\\")\\n",\n    "    out[\\"pred_mode:clip_has_predicate_params\\"] = any(\\"pred\\" in n_ for n_, _ in m1.named_parameters())\\n",\n    "    out[\\"pred_mode:clip_no_adapter\\"] = not any(\\"pred_enc.res\\" in n_ for n_, _ in m1.named_parameters())\\n",\n    "    gp = graph_for(dict(DESIGN_BASE, path_mode=\\"pred_match\\"), CFG.seeds[0])[0]\\n",\n    "    out[\\"path:weighted_has_no_P1m\\"] = all(g0.R[f\\"P1m_{r}\\"].nnz == 0 for r in PRED_FAMILIES)\\n",\n    "    out[\\"path:pred_match_has_P1m\\"] = any(gp.R[f\\"P1m_{r}\\"].nnz > 0 for r in PRED_FAMILIES)\\n",\n    "    out[\\"path:pred_match_same_other_patterns\\"] = all((gp.R[p] != g0.R[p]).nnz == 0 for p in PATTERNS if not p.startswith(\\"P1m_\\"))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_inductive_isolation():\\n",\n    "    \\"\\"\\"Strict inductive: đổi embedding của MỘT video đánh giá không làm đổi node lớp/khái niệm hay video khác;\\n",\n    "    transductive thì có (để chứng minh phép thử nhạy). Chạy cho cả video test MSVD và video tập ngoài.\\"\\"\\"\\n",\n    "    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])\\n",\n    "    model = make_model(eng, SYM_FEAT, pred_mode=\\"clip\\").eval()\\n",\n    "    out = {}\\n",\n    "    for s in (\\"test\\", \\"ext\\"):\\n",\n    "        tv = np.array(SPLIT[s])\\n",\n    "        if len(tv) < 2:\\n",\n    "            continue\\n",\n    "        v0, v1 = int(tv[0]), int(tv[1])\\n",\n    "        for proto in PROTOCOLS:\\n",\n    "            with torch.no_grad():\\n",\n    "                h1 = model.encode_graph(*eng.edges_on(proto))\\n",\n    "                old = model.vid_feat[v0].clone()\\n",\n    "                model.vid_feat[v0] = torch.randn_like(old)\\n",\n    "                h2 = model.encode_graph(*eng.edges_on(proto))\\n",\n    "                model.vid_feat[v0] = old\\n",\n    "            sym = slice(g.n_vid, g.n_nodes)\\n",\n    "            out[f\\"{s}:{proto}\\"] = dict(symbolic_changed=float((h1[sym] - h2[sym]).abs().max()),\\n",\n    "                                       other_video_changed=float((h1[v1] - h2[v1]).abs().max()),\\n",\n    "                                       own_video_changed=float((h1[v0] - h2[v0]).abs().max()),\\n",\n    "                                       all_finite=bool(torch.isfinite(h1).all()))\\n",\n    "    out[\\"edges\\"] = eng.edge_stats\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_guards():\\n",\n    "    out = {}\\n",\n    "    for s in (\\"test\\", \\"ext\\"):\\n",\n    "        if not SPLIT[s]:\\n",\n    "            continue\\n",\n    "        try:\\n",\n    "            make_part(s); out[f\\"{s}_before_freeze_blocked\\"] = False\\n",\n    "        except RuntimeError:\\n",\n    "            out[f\\"{s}_before_freeze_blocked\\"] = True\\n",\n    "        sq = np.where(CAP_SPLIT == s)[0][:3]\\n",\n    "        for name, fn in ((f\\"{s}→gnn_train\\", lambda: CAPS.use(sq, \\"gnn_train\\")), (f\\"{s}→tune\\", lambda: CAPS.use(sq, \\"tune\\")),\\n",\n    "                         (f\\"{s}→vocab\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"vocab\\")),\\n",\n    "                         (f\\"{s}→relations\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"relations\\")),\\n",\n    "                         (f\\"{s}→annotation\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"annotation\\")),\\n",\n    "                         (f\\"{s}→agreement\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"agreement\\"))):\\n",\n    "            try:\\n",\n    "                fn(); out[name + \\"_blocked\\"] = False\\n",\n    "            except PermissionError:\\n",\n    "                out[name + \\"_blocked\\"] = True\\n",\n    "        try:\\n",\n    "            CAPS.use(sq, \\"eval\\"); out[f\\"{s}_eval_before_freeze_blocked\\"] = False\\n",\n    "        except RuntimeError:\\n",\n    "            out[f\\"{s}_eval_before_freeze_blocked\\"] = True\\n",\n    "    vq = np.where(CAP_SPLIT == \\"val\\")[0][:3]\\n",\n    "    for name, fn in ((\\"val→gnn_train\\", lambda: CAPS.use(vq, \\"gnn_train\\")),\\n",\n    "                     (\\"val→relations\\", lambda: CAPS.by_video(SPLIT[\\"val\\"][:2], \\"relations\\")),\\n",\n    "                     (\\"train→tune\\", lambda: CAPS.use(np.where(CAP_SPLIT == \\"train\\")[0][:3], \\"tune\\"))):\\n",\n    "        try:\\n",\n    "            fn(); out[name + \\"_blocked\\"] = False\\n",\n    "        except PermissionError:\\n",\n    "            out[name + \\"_blocked\\"] = True\\n",\n    "    return out\\n",\n    "\\n",\n    "# ---------------------------------- Self-test v4 (QC-RGAT) — nhanh (< 1 phút CPU), không huấn luyện ----------------------------------\\n",\n    "def _v4_model_for_test(design, seed=0):\\n",\n    "    \\"\\"\\"Mô hình KHỞI TẠO NGẪU NHIÊN nhưng mọi cổng đều khác 0 (để phép thử không qua vì tham số bằng 0).\\"\\"\\"\\n",\n    "    g, eng = graph_for(design, CFG.seeds[0])\\n",\n    "    set_seed(seed)\\n",\n    "    m = make_model_v4(eng, design).eval()\\n",\n    "    with torch.no_grad():\\n",\n    "        for p in (m.beta, m.kappa, m.g_ex, m.g_soft, m.rgate):\\n",\n    "            p.copy_(torch.randn_like(p))\\n",\n    "    return g, eng, m\\n",\n    "\\n",\n    "\\n",\n    "def _queries_with_pred_edges(D, n, seed=0):\\n",\n    "    \\"\\"\\"Truy vấn TRAIN có ≥ 1 cạnh mang vị từ khớp vị từ của truy vấn (để thử nhánh vị từ).\\"\\"\\"\\n",\n    "    rng = np.random.RandomState(seed)\\n",\n    "    out = []\\n",\n    "    for q in rng.permutation(TRAIN_Q)[:4000]:\\n",\n    "        ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]\\n",\n    "        if ids and D.E and np.asarray(D.Mex[ids].sum(0)).ravel().max() > 0 and V4_HITS[q]:\\n",\n    "            out.append(int(q))\\n",\n    "            if len(out) >= n:\\n",\n    "                break\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_v4_decomposition(n=30):\\n",\n    "    \\"\\"\\"Σ đóng góp (thị giác + nhãn + đường đi) = điểm của mô hình, cho T = 1 và 2; và xoá một cạnh ⇒ điểm giảm ĐÚNG bằng tổng đóng\\n",\n    "    góp các đường đi qua cạnh đó (T = 1: tuyệt đối; T = 2: tính lại trên đồ thị đã xoá).\\"\\"\\"\\n",\n    "    out = {}\\n",\n    "    for T in (1, 2):\\n",\n    "        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=T))\\n",\n    "        qs = _queries_with_pred_edges(m.D, n)\\n",\n    "        maxdiff, del_diff, n_rel, n_del = 0.0, 0.0, 0, 0\\n",\n    "        vids = [int(v) for v in SPLIT[\\"train\\"] if m.D.Acon[int(v)].gt(0).any()]\\n",\n    "        for q in qs:\\n",\n    "            v = vids[q % len(vids)]\\n",\n    "            terms = m.decompose(q, v)\\n",\n    "            with torch.no_grad():\\n",\n    "                s = float(m.score(m.D.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE))[0, 0])\\n",\n    "            maxdiff = max(maxdiff, abs(s - sum(t[\\"contrib\\"] for t in terms)))\\n",\n    "            rel = [t for t in terms if t[\\"kind\\"] == \\"relation\\"]\\n",\n    "            n_rel += len(rel)\\n",\n    "            if rel:\\n",\n    "                e = max(rel, key=lambda t: abs(t[\\"contrib\\"]))[\\"edges\\"][-1]\\n",\n    "                D2 = RelDataV4(g, drop_edges=[int(m.D.gidx[e])])\\n",\n    "                with torch.no_grad():\\n",\n    "                    s2 = float(m.score(D2.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE), D2)[0, 0])\\n",\n    "                expect = sum(t[\\"contrib\\"] for t in rel if e in t[\\"edges\\"])\\n",\n    "                del_diff = max(del_diff, abs((s - s2) - expect)); n_del += 1\\n",\n    "        out[f\\"T{T}\\"] = dict(queries=len(qs), relation_terms=n_rel, max_abs_diff=maxdiff, deletions=n_del, deletion_max_abs_diff=del_diff)\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_v4_predicate_and_query():\\n",\n    "    \\"\\"\\"(1) Đổi vị từ của truy vấn ⇒ attention trên cạnh mang vị từ đó đổi; use_pred = False ⇒ KHÔNG đổi (đối chứng âm).\\n",\n    "    (2) Hai truy vấn khác nhau ⇒ attention khác (qcond); qcond = False ⇒ giống hệt. (3) Hoán vị vị từ (G4) đổi tín hiệu khớp.\\"\\"\\"\\n",\n    "    out = {}\\n",\n    "    for use_pred in (True, False):\\n",\n    "        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=1, v4_pred=use_pred))\\n",\n    "        qs = _queries_with_pred_edges(m.D, 10)\\n",\n    "        diffs = []\\n",\n    "        for q in qs:\\n",\n    "            qb = m.D.query_batch([q], True)\\n",\n    "            qb2 = dict(qb, ex=torch.zeros_like(qb[\\"ex\\"]), soft=torch.zeros_like(qb[\\"soft\\"]))\\n",\n    "            with torch.no_grad():\\n",\n    "                u = m.q(qb[\\"tq\\"])\\n",\n    "                diffs.append(float((m.attention(qb, u) - m.attention(qb2, u)).abs().max()))\\n",\n    "        out[f\\"pred_change_use_pred={use_pred}\\"] = max(diffs) if diffs else float(\\"nan\\")\\n",\n    "    for qcond in (True, False):\\n",\n    "        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=1, v4_qcond=qcond))\\n",\n    "        qs = _queries_with_pred_edges(m.D, 2)\\n",\n    "        with torch.no_grad():\\n",\n    "            qb = m.D.query_batch(qs, True)\\n",\n    "            a = m.attention(qb, m.q(qb[\\"tq\\"]))\\n",\n    "        out[f\\"two_queries_attention_diff_qcond={qcond}\\"] = float((a[0] - a[1]).abs().max())\\n",\n    "    g0 = graph_for(dict(V3_FINAL, model=\\"v4\\"), CFG.seeds[0])[0]\\n",\n    "    gs = graph_for(dict(V3_FINAL, model=\\"v4\\", pred_shuffle=True), CFG.seeds[0])[0]\\n",\n    "    D0, Ds = RelDataV4(g0), RelDataV4(gs)\\n",\n    "    qs = _queries_with_pred_edges(D0, 50)\\n",\n    "    out[\\"pred_shuffle_changes_match\\"] = float((D0.query_batch(qs)[\\"ex\\"] - Ds.query_batch(qs)[\\"ex\\"]).abs().sum())\\n",\n    "    out[\\"pred_shuffle_same_topology\\"] = bool(np.array_equal(D0.src_np, Ds.src_np) and np.array_equal(D0.dst_np, Ds.dst_np))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def test_v4_inductive_and_masks():\\n",\n    "    \\"\\"\\"Video chỉ NHẬN (node lớp không phụ thuộc video) ⇒ đổi đặc trưng / nhãn của MỘT video test (hoặc tập ngoài) không đổi điểm\\n",\n    "    của video khác; rel = False ⇒ không có số hạng quan hệ; điểm hữu hạn, gradient hữu hạn.\\n",\n    "    So với dung sai 1e-6 (như G5 của v3): trên GPU, index_add_ cộng dồn bằng atomic nên hai lần gọi cùng đầu vào có thể lệch ~1e-8.\\"\\"\\"\\n",\n    "    g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=2))\\n",\n    "    out = {}\\n",\n    "    q = _queries_with_pred_edges(m.D, 1)[0]\\n",\n    "    qb = m.D.query_batch([q])\\n",\n    "    for s in (\\"test\\", \\"ext\\"):\\n",\n    "        tv = [int(v) for v in SPLIT.get(s, [])]\\n",\n    "        if len(tv) < 2:\\n",\n    "            continue\\n",\n    "        v0, v1 = tv[0], tv[1]\\n",\n    "        cand = torch.tensor([[v0, v1]], device=DEVICE)\\n",\n    "        with torch.no_grad():\\n",\n    "            s1 = m.score(qb, cand)\\n",\n    "            s1b = m.score(qb, cand)                          # cùng đầu vào ⇒ đo sai số số học khi chạy lại (GPU không tất định)\\n",\n    "            old_x, old_a = VID_T[v0].clone(), m.D.Acon[v0].clone()\\n",\n    "            VID_T[v0] = torch.randn_like(old_x); m.D.Acon[v0] = torch.roll(old_a, 7)\\n",\n    "            s2 = m.score(qb, cand)\\n",\n    "            VID_T[v0] = old_x; m.D.Acon[v0] = old_a\\n",\n    "        out[s] = dict(other_video_changed=float((s1[0, 1] - s2[0, 1]).abs()), own_video_changed=float((s1[0, 0] - s2[0, 0]).abs()),\\n",\n    "                      rerun_noise=float((s1 - s1b).abs().max()))\\n",\n    "    g1, e1, m1 = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_rel=False))\\n",\n    "    v = int(SPLIT[\\"train\\"][0])\\n",\n    "    out[\\"no_relation_terms_when_rel_false\\"] = not any(t[\\"kind\\"] == \\"relation\\" for t in m1.decompose(q, v))\\n",\n    "    m.train()\\n",\n    "    qb = m.D.query_batch(TRAIN_Q[:16])\\n",\n    "    loss = m.score(qb, torch.from_numpy(np.tile(CAP_VID[TRAIN_Q[:16]], (16, 1))).to(DEVICE)).logsumexp(1).mean()\\n",\n    "    loss.backward()\\n",\n    "    out[\\"grad_finite\\"] = bool(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\\n",\n    "    out[\\"score_finite\\"] = bool(torch.isfinite(loss).item())\\n",\n    "    m.eval()\\n",\n    "    return out\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 11. Level 4(b): bộ truy vấn đăng ký trước, xAI theo thứ hạng, phân tích ca, demo, kiểm kê\\n",\n    "\\n",\n    "Các hàm dưới đây chỉ là **định nghĩa**, được gọi ở phần THỰC THI. Mọi file đầu ra sinh từ chính lần chạy; không có số liệu điền tay."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================== Level 4(b): công cụ bổ sung (chỉ ĐỊNH NGHĨA — chạy ở phần THỰC THI) ==============================\\n",\n    "import copy as _copy\\n",\n    "\\n",\n    "# Nguồn / giấy phép: dataset card HF (kiểm qua API ngày 2026-10-25) KHÔNG khai báo license ⇒ ghi đúng như vậy, không tự điền.\\n",\n    "DATA_SOURCES = {\\n",\n    "    \\"msvd\\": dict(name=\\"MSVD (Microsoft Research Video Description Corpus)\\", paper=\\"Chen & Dolan, ACL 2011\\",\\n",\n    "                 repo=CFG.hf_repo, revision=CFG.hf_revision,\\n",\n    "                 license=\\"KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc (clip YouTube) phát hành cho nghiên cứu\\",\\n",\n    "                 role=\\"train: từ vựng, quan hệ, huấn luyện GNN | val: chọn cấu hình | test: THĂM DÒ\\"),\\n",\n    "    \\"msrvtt\\": dict(name=\\"MSR-VTT\\", paper=\\"Xu et al., CVPR 2016\\", repo=CFG.msrvtt_repo, revision=CFG.msrvtt_revision,\\n",\n    "                   license=\\"KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc của Microsoft Research cho nghiên cứu\\",\\n",\n    "                   role=\\"chỉ đánh giá (chuyển giao zero-shot): không caption nào dùng để học hay chọn\\"),\\n",\n    "}\\n",\n    "# Số lần mỗi benchmark đã được nhìn TRƯỚC notebook này (lịch sử v1–v4, xem README). Đã nhìn ⇒ kết quả chỉ là THĂM DÒ.\\n",\n    "BENCHMARK_LOOKS_BEFORE = {\\"msvd_test\\": 5, \\"msrvtt_1kA\\": 1, \\"msrvtt_fresh1k\\": 1, \\"msrvtt_fresh1kB\\": 1, \\"msrvtt_fresh1kC\\": 0}\\n",\n    "_PROV_RULES[\\"evaluation\\"] = {\\"val\\", \\"test\\", \\"ext\\"}   # bộ truy vấn đánh giá: dựng từ văn bản truy vấn của tập đánh giá, không học / chọn gì\\n",\n    "FINDINGS = []          # kết quả KHOA HỌC được báo cáo (không phải kiểm thử phần mềm): id, câu hỏi, kết quả, diễn giải\\n",\n    "\\n",\n    "\\n",\n    "def finding(fid, question, result, interpretation):\\n",\n    "    FINDINGS.append(dict(id=fid, question=question, result=str(result)[:900], interpretation=interpretation))\\n",\n    "    print(f\\"  [FINDING] {fid} — {question}: {result}\\\\n            ⇒ {interpretation}\\")\\n",\n    "\\n",\n    "\\n",\n    "def benchmark_status():\\n",\n    "    \\"\\"\\"Trạng thái từng tập đánh giá của lần chạy này: CONFIRMATORY chỉ khi tập CHƯA từng được nhìn.\\"\\"\\"\\n",\n    "    ext_key = {\\"fresh1kC\\": \\"msrvtt_fresh1kC\\", \\"fresh1kB\\": \\"msrvtt_fresh1kB\\", \\"fresh1k\\": \\"msrvtt_fresh1k\\", \\"1kA\\": \\"msrvtt_1kA\\"}.get(EXT_REPORT.get(\\"set\\"))\\n",\n    "    out = {\\"msvd_test\\": dict(looks_before=BENCHMARK_LOOKS_BEFORE[\\"msvd_test\\"], evaluated=not CFG.dev_mode,\\n",\n    "                             status=\\"EXPLORATORY\\" if not CFG.dev_mode else \\"NOT RUN (dev_mode: test khoá)\\")}\\n",\n    "    if EXT_RECORDS and not EXT_REPORT.get(\\"dev\\"):\\n",\n    "        lb = BENCHMARK_LOOKS_BEFORE[ext_key]\\n",\n    "        out[ext_key] = dict(looks_before=lb, evaluated=True, status=\\"CONFIRMATORY\\" if lb == 0 else \\"EXPLORATORY\\",\\n",\n    "                            label=EXT_REPORT.get(\\"label\\"))\\n",\n    "    elif EXT_RECORDS:\\n",\n    "        out[\\"msrvtt_dev\\"] = dict(looks_before=None, evaluated=True, status=\\"DEV (thử đường ống)\\", label=EXT_REPORT.get(\\"label\\"))\\n",\n    "    else:\\n",\n    "        out[\\"msrvtt\\"] = dict(evaluated=False, status=EXT_STATUS)\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def node_name(i):\\n",\n    "    return INV.name[int(i)]\\n",\n    "\\n",\n    "\\n",\n    "def edges_into_tags(g, src, tags):\\n",\n    "    \\"\\"\\"Cạnh quan hệ src –t→ c với c ∈ tags (đúng chiều QC-RGAT truyền tin: nguồn → nhãn của video).\\"\\"\\"\\n",\n    "    out = []\\n",\n    "    for t in RELATION_TYPES:\\n",\n    "        A = g.A[t]\\n",\n    "        if A.nnz == 0:\\n",\n    "            continue\\n",\n    "        row = A.getrow(int(src))\\n",\n    "        for c in row.indices:\\n",\n    "            if int(c) in tags:\\n",\n    "                base = t[:-4] if t.endswith(\\"_inv\\") else t\\n",\n    "                key = (base, int(c), int(src)) if t.endswith(\\"_inv\\") else (base, int(src), int(c))\\n",\n    "                pd_ = g.pdist.get(key, {}) if base in PRED_FAMILIES else {}\\n",\n    "                p = max(pd_, key=pd_.get) if pd_ else \\"\\"\\n",\n    "                out.append((int(src), t, int(c), p))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def edge_str(e):\\n",\n    "    s_, t, d_, p = e\\n",\n    "    return f\\"{node_name(s_)} –{t}{\'[\' + p + \']\' if p else \'\'}→ {node_name(d_)}\\"\\n",\n    "\\n",\n    "\\n",\n    "def build_eval_queries(part, g, n_per_group=None, seed=None):\\n",\n    "    \\"\\"\\"Bộ truy vấn ĐĂNG KÝ TRƯỚC (mục 7.3). Chỉ dùng: văn bản truy vấn (mục đích query_input), nhãn CLIP zero-shot của video đúng,\\n",\n    "    cạnh đồ thị dựng từ caption TRAIN. KHÔNG dùng điểm hay thứ hạng của bất kỳ mô hình nào.\\n",\n    "    trực tiếp : mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.\\n",\n    "    gián tiếp : truy vấn nêu ≥ 1 quan hệ (bộ ba), nhãn video phủ MỘT PHẦN khái niệm truy vấn, và có cạnh quan hệ nối một khái\\n",\n    "                niệm còn thiếu vào một nhãn của video (= quan hệ dự kiến phải khai thác).\\"\\"\\"\\n",\n    "    n_per_group = n_per_group or CFG.n_eval_queries\\n",\n    "    seed = CFG.eval_query_seed if seed is None else seed\\n",\n    "    q_all, gold_all = part[\\"q_idx\\"], part[\\"gold\\"]\\n",\n    "    texts = CAPS.texts(q_all, \\"query_input\\")\\n",\n    "    pools = {\\"direct\\": [], \\"indirect\\": []}\\n",\n    "    for j, q in enumerate(q_all):\\n",\n    "        v, hits = int(gold_all[j]), set(V4_HITS[int(q)])\\n",\n    "        tags = set(g.vid_tags[v])\\n",\n    "        if not hits:\\n",\n    "            continue\\n",\n    "        seen, miss = hits & tags, hits - tags\\n",\n    "        base = dict(q_idx=int(q), text=texts[j], gold_vid=v, gold_video_id=RECORDS[v][\\"video_id\\"],\\n",\n    "                    query_concepts=\\"; \\".join(sorted(node_name(h) for h in hits)),\\n",\n    "                    concepts_seen_in_video=\\"; \\".join(sorted(node_name(h) for h in seen)),\\n",\n    "                    concepts_missing=\\"; \\".join(sorted(node_name(h) for h in miss)))\\n",\n    "        if not miss:\\n",\n    "            pools[\\"direct\\"].append(dict(base, type=\\"direct\\", expected_relations=\\"\\",\\n",\n    "                                        reason=f\\"mọi khái niệm của truy vấn ({base[\'query_concepts\']}) là nhãn CLIP zero-shot của video đúng\\"))\\n",\n    "            continue\\n",\n    "        trip = [t for t in parse_caption_v2(texts[j])[2] if INV.key2id.get(t[0]) is not None and INV.key2id.get(t[2]) is not None]\\n",\n    "        if not trip or not seen:\\n",\n    "            continue\\n",\n    "        bridge = [e for m_ in sorted(miss) for e in edges_into_tags(g, m_, tags)]\\n",\n    "        if not bridge:\\n",\n    "            continue\\n",\n    "        rel_txt = \\"; \\".join(f\\"{s_.split(\'#\')[0]} –{r_}{\'[\' + p_ + \']\' if p_ else \'\'}→ {o_.split(\'#\')[0]}\\" for s_, r_, o_, p_ in sorted(trip))\\n",\n    "        pools[\\"indirect\\"].append(dict(base, type=\\"indirect\\", expected_relations=\\" || \\".join(edge_str(e) for e in bridge[:5]),\\n",\n    "                                      reason=(f\\"truy vấn nêu quan hệ ({rel_txt}); video chỉ có nhãn {base[\'concepts_seen_in_video\']}, \\"\\n",\n    "                                              f\\"thiếu {base[\'concepts_missing\']}; {len(bridge)} cạnh quan hệ nối khái niệm thiếu vào nhãn video\\")))\\n",\n    "    rows, info = [], {}\\n",\n    "    for k, (grp, pool) in enumerate(pools.items()):\\n",\n    "        order = np.random.RandomState(seed + k).permutation(len(pool))\\n",\n    "        used, take = set(), []\\n",\n    "        for i in order:\\n",\n    "            if pool[i][\\"gold_vid\\"] in used:\\n",\n    "                continue\\n",\n    "            used.add(pool[i][\\"gold_vid\\"]); take.append(pool[i])\\n",\n    "            if len(take) >= n_per_group:\\n",\n    "                break\\n",\n    "        for n_, r in enumerate(take, 1):\\n",\n    "            rows.append(dict(query_id=f\\"{part[\'name\']}-{grp[0].upper()}{n_:02d}\\", part=part[\\"name\\"], **r))\\n",\n    "        info[grp] = dict(eligible=len(pool), eligible_videos=len({r[\\"gold_vid\\"] for r in pool}), selected=len(take))\\n",\n    "    return pd.DataFrame(rows), info\\n",\n    "\\n",\n    "\\n",\n    "def eval_query_metrics(E, QS):\\n",\n    "    \\"\\"\\"R@1/5/10, MRR (trung bình ± std theo seed) của MỌI thí nghiệm trên bộ truy vấn đăng ký trước, theo nhóm.\\"\\"\\"\\n",\n    "    part = E[\\"part\\"]\\n",\n    "    pos = {int(q): i for i, q in enumerate(part[\\"q_idx\\"])}\\n",\n    "    rows, pairs = [], []\\n",\n    "    for grp in (\\"all\\", \\"direct\\", \\"indirect\\"):\\n",\n    "        sel = QS if grp == \\"all\\" else QS[QS.type == grp]\\n",\n    "        if not len(sel):\\n",\n    "            continue\\n",\n    "        idx = np.array([pos[int(q)] for q in sel.q_idx])\\n",\n    "        gold = part[\\"gold\\"][idx]\\n",\n    "        for k, ev_k in E[\\"ev\\"].items():\\n",\n    "            ms = [metrics_from_ranks(v[\\"rank\\"][idx], gold) for v in ev_k.values()]\\n",\n    "            rows.append(dict(part=part[\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], experiment_id=k, group=grp, n_queries=len(idx),\\n",\n    "                             n_videos=len(set(gold.tolist())), n_seeds=len(ms),\\n",\n    "                             **{f\\"{x}_mean\\": float(np.mean([mm[x] for mm in ms])) for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")},\\n",\n    "                             **{f\\"{x}_std\\": float(np.std([mm[x] for mm in ms])) for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}))\\n",\n    "        for a in (G_IDS.get(\\"G3\\"), G_IDS.get(\\"V3REF\\")):                 # QC-RGAT và GraphSAGE có kiểu cạnh, mỗi cái vs CLIP-only\\n",\n    "            if a in E[\\"ev\\"]:\\n",\n    "                rr = lambda k_: np.mean([1.0 / v[\\"rank\\"][idx] for v in E[\\"ev\\"][k_].values()], 0)\\n",\n    "                bs = cluster_bootstrap_diff(rr(a), rr(\\"G0_clip_only\\"), gold)\\n",\n    "                pairs.append(dict(part=part[\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], group=grp, a=a, b=\\"G0_clip_only\\", n=len(idx),\\n",\n    "                                  MRR_a=float(rr(a).mean()), MRR_b=float(rr(\\"G0_clip_only\\").mean()), **bs))\\n",\n    "    return pd.DataFrame(rows), pd.DataFrame(pairs)\\n",\n    "\\n",\n    "\\n",\n    "def drop_graph_parts(D, vid, cls_drop=(), rel_drop=()):\\n",\n    "    \\"\\"\\"Bản sao dữ liệu đồ thị với (1) cạnh contains video –chứa→ c (c ∈ cls_drop) của RIÊNG video `vid` bị xoá (chuẩn hoá lại hàng\\n",\n    "    như lúc dựng), (2) cạnh quan hệ (chỉ số trong D) bị xoá. Mô hình và tham số giữ nguyên.\\"\\"\\"\\n",\n    "    D2 = RelDataV4(D.graph, drop_edges=D.gidx[list(rel_drop)]) if len(rel_drop) else _copy.copy(D)\\n",\n    "    if len(cls_drop):\\n",\n    "        A = D2.Acon.clone()\\n",\n    "        row = (A[vid] > 0).float()\\n",\n    "        row[list(cls_drop)] = 0.0\\n",\n    "        A[vid] = row / row.sum().clamp(min=1.0)\\n",\n    "        D2 = _copy.copy(D2)\\n",\n    "        D2.Acon = A\\n",\n    "    return D2\\n",\n    "\\n",\n    "\\n",\n    "def term_text(D, t, vid):\\n",\n    "    if t[\\"kind\\"] == \\"contains\\":\\n",\n    "        return f\\"{RECORDS[vid][\'video_id\']} –chứa→ {node_name(D.N + t[\'node\'])}\\"\\n",\n    "    return v4_path_text(D, t, vid)\\n",\n    "\\n",\n    "\\n",\n    "def term_valid(D, t, vid, q):\\n",\n    "    \\"\\"\\"Cạnh của số hạng có thật trong đồ thị đang chạy, đúng loại, đúng chiều, đúng kiểu node; vị từ khớp (nếu có) nằm trên ĐÚNG cạnh.\\"\\"\\"\\n",\n    "    if t[\\"kind\\"] == \\"contains\\":\\n",\n    "        c = D.N + t[\\"node\\"]\\n",\n    "        ok = D.graph.A_con[vid, c] > 0 and INV.node_type[c] != \\"video\\" and INV.node_type[vid] == \\"video\\"\\n",\n    "        return bool(ok), True\\n",\n    "    v = [v4_edge_valid(D, k, q) for k in t[\\"edges\\"]]\\n",\n    "    return all(x[0] for x in v), all(x[2] for x in v)\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def explain_full(model, q, vid, top_m=None):\\n",\n    "    \\"\\"\\"Giải thích ĐẦY ĐỦ \\"vì sao video được xếp hạng như vậy\\" = top-m số hạng ĐỒ THỊ ủng hộ (contains + đường đi quan hệ).\\n",\n    "    Số hạng thị giác (W_v x_v) không phải đồ thị nên được báo riêng. Σ mọi số hạng = điểm đồ thị của mô hình.\\"\\"\\"\\n",\n    "    top_m = top_m or CFG.top_m_paths\\n",\n    "    terms = model.decompose(q, vid)\\n",\n    "    graph_terms = [t for t in terms if t[\\"kind\\"] != \\"visual\\"]\\n",\n    "    sup = sorted([t for t in graph_terms if t[\\"contrib\\"] > 0], key=lambda t: -t[\\"contrib\\"])[:top_m]\\n",\n    "    opp = sorted([t for t in graph_terms if t[\\"contrib\\"] < 0], key=lambda t: t[\\"contrib\\"])[:top_m]\\n",\n    "    tot = {k: sum(t[\\"contrib\\"] for t in terms if t[\\"kind\\"] == k) for k in (\\"visual\\", \\"contains\\", \\"relation\\")}\\n",\n    "    return dict(terms=terms, support=sup, oppose=opp, totals=tot, score=sum(t[\\"contrib\\"] for t in terms),\\n",\n    "                n_graph_terms=len(graph_terms), pos_graph=sum(t[\\"contrib\\"] for t in graph_terms if t[\\"contrib\\"] > 0))\\n",\n    "\\n",\n    "\\n",\n    "def _deletion_sets(sup):\\n",\n    "    return (sorted({t[\\"node\\"] for t in sup if t[\\"kind\\"] == \\"contains\\"}),\\n",\n    "            sorted({k for t in sup if t[\\"kind\\"] == \\"relation\\" for k in t[\\"edges\\"]}))\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def xai_rank_fidelity(run, part, n_queries=None, top_m=None, seed=0):\\n",\n    "    \\"\\"\\"Fidelity theo QUYẾT ĐỊNH xếp hạng (mục 8.3): với ≥ n truy vấn có video đúng trong top-K ứng viên, xoá THẬT các cạnh của\\n",\n    "    giải thích đầy đủ (contains của video + cạnh quan hệ) rồi chấm lại TOÀN BỘ ứng viên ⇒ Δ điểm đồ thị, Δ điểm cuối, Δ hạng,\\n",\n    "    đổi top-1 / top-10; đối chứng = xoá cùng số cạnh contains khác của video + cùng số cạnh quan hệ ngẫu nhiên vào nhãn của video.\\n",\n    "    Báo riêng: hạng không đổi thì ghi đúng là không đổi.\\"\\"\\"\\n",\n    "    n_queries, top_m = n_queries or CFG.n_explain_queries, top_m or CFG.top_m_paths\\n",\n    "    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n",\n    "    D = m.D\\n",\n    "    rng = np.random.RandomState(seed)\\n",\n    "    q_all = part[\\"q_idx\\"]\\n",\n    "    cand_all, s_clip = eng.clip_candidates(q_all, part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "    rows = []\\n",\n    "    for i in rng.permutation(len(q_all)):\\n",\n    "        if len(rows) >= n_queries:\\n",\n    "            break\\n",\n    "        q, gold = int(q_all[i]), int(part[\\"gold\\"][i])\\n",\n    "        cand = cand_all[i]\\n",\n    "        if gold not in cand:\\n",\n    "            continue\\n",\n    "        ex = explain_full(m, q, gold, top_m)\\n",\n    "        if not ex[\\"support\\"]:\\n",\n    "            continue\\n",\n    "        zc = znorm_per_row(s_clip[i:i + 1])[0]\\n",\n    "        r_full, s_full, fin_full = v4_rank_after(run, q, gold, cand, zc, D)\\n",\n    "        j = int(np.where(cand == gold)[0][0])\\n",\n    "        con_del, rel_del = _deletion_sets(ex[\\"support\\"])\\n",\n    "        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, con_del, rel_del))\\n",\n    "        tags = torch.nonzero(D.Acon[gold] > 0).flatten().tolist()\\n",\n    "        other = [c for c in tags if c not in con_del]\\n",\n    "        rc = [int(x) for x in rng.choice(other, min(len(con_del), len(other)), replace=False)] if (other and con_del) else []\\n",\n    "        pool = [k for k in range(D.E) if int(D.dst_np[k]) in set(tags) and k not in rel_del]\\n",\n    "        rr = [int(x) for x in rng.choice(pool, min(len(rel_del), len(pool)), replace=False)] if (pool and rel_del) else []\\n",\n    "        r_rd, s_rd, fin_rd = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, rc, rr))\\n",\n    "        val = [term_valid(D, t, gold, q) for t in ex[\\"support\\"]]\\n",\n    "        expl_edges = len(con_del) + len(rel_del)\\n",\n    "        top10 = lambda f: set(np.argsort(-f)[:10].tolist())\\n",\n    "        rows.append(dict(q=q, gold=gold, gold_video_id=RECORDS[gold][\\"video_id\\"], n_graph_terms=ex[\\"n_graph_terms\\"],\\n",\n    "                         n_expl_terms=len(ex[\\"support\\"]), n_expl_contains=sum(t[\\"kind\\"] == \\"contains\\" for t in ex[\\"support\\"]),\\n",\n    "                         n_expl_relation=sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"support\\"]), n_expl_edges=expl_edges,\\n",\n    "                         n_rand_edges=len(rc) + len(rr), sparsity=len(ex[\\"support\\"]) / max(1, ex[\\"n_graph_terms\\"]),\\n",\n    "                         coverage=sum(t[\\"contrib\\"] for t in ex[\\"support\\"]) / max(1e-9, ex[\\"pos_graph\\"]),\\n",\n    "                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[1] for v in val])),\\n",\n    "                         decomposition_ok=float(abs(ex[\\"score\\"] - s_full) < 1e-4),\\n",\n    "                         s_graph=float(s_full), d_s_expl=float(s_ex - s_full), d_s_rand=float(s_rd - s_full),\\n",\n    "                         d_final_expl=float(fin_ex[j] - fin_full[j]), d_final_rand=float(fin_rd[j] - fin_full[j]),\\n",\n    "                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,\\n",\n    "                         rank_worse_expl=float(r_ex > r_full), rank_worse_rand=float(r_rd > r_full),\\n",\n    "                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),\\n",\n    "                         top1_changed_rand=float(np.argmax(fin_rd) != np.argmax(fin_full)),\\n",\n    "                         top10_changed_expl=float(top10(fin_ex) != top10(fin_full)), top10_changed_rand=float(top10(fin_rd) != top10(fin_full)),\\n",\n    "                         share_graph=(ex[\\"totals\\"][\\"contains\\"] + ex[\\"totals\\"][\\"relation\\"]) / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n",\n    "                         share_relation=ex[\\"totals\\"][\\"relation\\"] / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n",\n    "                         explanation=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {term_text(D, t, gold)}\\" for t in ex[\\"support\\"]),\\n",\n    "                         opposing=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {term_text(D, t, gold)}\\" for t in ex[\\"oppose\\"][:3])))\\n",\n    "    return pd.DataFrame(rows)\\n",\n    "\\n",\n    "\\n",\n    "def diagnose_case(run, q, gold, cand, rank, clip_rank, text):\\n",\n    "    \\"\\"\\"Nguyên nhân (luật cố định): candidate_recall_low | tag_missing | missing_edge | predicate_not_in_graph | wrong_predicate |\\n",\n    "    noisy_relation | distractor_shares_concepts | graph_helped | clip_already_correct.\\"\\"\\"\\n",\n    "    g, D = run[\\"graph\\"], run[\\"model\\"].D\\n",\n    "    hits, tags = set(V4_HITS[q]), set(g.vid_tags[gold])\\n",\n    "    causes, notes = [], []\\n",\n    "    if gold not in cand:\\n",\n    "        causes.append(\\"candidate_recall_low\\"); notes.append(f\\"video đúng ở hạng CLIP {clip_rank} > k = {CFG.rerank_k}\\")\\n",\n    "    if hits and not hits & tags:\\n",\n    "        causes.append(\\"tag_missing\\"); notes.append(\\"không khái niệm nào của truy vấn là nhãn của video đúng\\")\\n",\n    "    for s_, r_, o_, p_ in sorted(parse_caption_v2(text)[2]):\\n",\n    "        a, b = INV.key2id.get(s_), INV.key2id.get(o_)\\n",\n    "        if a is None or b is None or a == b:\\n",\n    "            continue\\n",\n    "        on = [t for t in RELATION_TYPES if g.A[t][a, b] != 0 or g.A[t][b, a] != 0]\\n",\n    "        if not on:\\n",\n    "            causes.append(\\"missing_edge\\"); notes.append(f\\"không có cạnh nào giữa {s_} và {o_} (quan hệ \'{r_}\' của truy vấn)\\")\\n",\n    "        elif r_ in PRED_FAMILIES and p_:\\n",\n    "            if (r_, p_) not in PRED_ID:\\n",\n    "                causes.append(\\"predicate_not_in_graph\\"); notes.append(f\\"vị từ \'{p_}\' không có trong từ vựng vị từ (train)\\")\\n",\n    "            else:\\n",\n    "                pd_ = g.pdist.get((r_, a, b), {}) or g.pdist.get((r_, b, a), {})\\n",\n    "                if pd_ and p_ not in pd_:\\n",\n    "                    causes.append(\\"wrong_predicate\\")\\n",\n    "                    notes.append(f\\"cạnh {s_}–{r_}–{o_} mang vị từ {sorted(pd_)[:3]}, không có \'{p_}\' của truy vấn\\")\\n",\n    "    if gold in cand:\\n",\n    "        ex = explain_full(run[\\"model\\"], q, gold)\\n",\n    "        if ex[\\"totals\\"][\\"relation\\"] < 0 and rank > clip_rank:\\n",\n    "            causes.append(\\"noisy_relation\\"); notes.append(f\\"tổng đóng góp quan hệ cho video đúng âm ({ex[\'totals\'][\'relation\']:+.4f})\\")\\n",\n    "        if rank < clip_rank:\\n",\n    "            causes.append(\\"graph_helped\\"); notes.append(f\\"đồ thị kéo video đúng từ hạng {clip_rank} lên {rank}\\")\\n",\n    "    if clip_rank == 1 and rank == 1:\\n",\n    "        causes.append(\\"clip_already_correct\\")\\n",\n    "    return list(dict.fromkeys(causes)), notes\\n",\n    "\\n",\n    "\\n",\n    "FAIL_CAUSES = (\\"candidate_recall_low\\", \\"missing_edge\\", \\"wrong_predicate\\", \\"noisy_relation\\", \\"tag_missing\\", \\"predicate_not_in_graph\\")\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def case_analysis_l4b(run, part, n_each=None, seed=0, max_q=4000):\\n",\n    "    \\"\\"\\"5 thành công (hạng 1) + 5 thất bại (hạng > 10), chọn theo LUẬT: thành công ưu tiên ca đồ thị kéo hạng lên rồi tới ca CLIP đã đúng;\\n",\n    "    thất bại lấy mỗi nguyên nhân trong FAIL_CAUSES một ca (nếu có) rồi bổ sung ngẫu nhiên — không chọn ví dụ đẹp.\\"\\"\\"\\n",\n    "    n_each = n_each or CFG.n_cases\\n",\n    "    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n",\n    "    rng = np.random.RandomState(seed)\\n",\n    "    q = part[\\"q_idx\\"]\\n",\n    "    sel = np.sort(rng.choice(len(q), min(max_q, len(q)), replace=False))\\n",\n    "    q, gold = q[sel], part[\\"gold\\"][sel]\\n",\n    "    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n",\n    "    sc = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k)\\n",\n    "    fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], 1.0)\\n",\n    "    rank = ranks_with_fallback(sc[\\"cand\\"], fin, gold, clip_rank)\\n",\n    "    texts = CAPS.texts(q, \\"report\\")\\n",\n    "    order = rng.permutation(len(q))\\n",\n    "    succ = [i for i in order if rank[i] == 1 and clip_rank[i] > 1][:3]\\n",\n    "    succ += [i for i in order if rank[i] == 1 and i not in succ][:n_each - len(succ)]\\n",\n    "    fail_pool = [i for i in order if rank[i] > 10]\\n",\n    "    diag = {}\\n",\n    "    for i in fail_pool[:400]:\\n",\n    "        diag[i] = diagnose_case(run, int(q[i]), int(gold[i]), sc[\\"cand\\"][i], int(rank[i]), int(clip_rank[i]), texts[i])\\n",\n    "    fail = []\\n",\n    "    for c in FAIL_CAUSES:\\n",\n    "        hit = next((i for i in fail_pool[:400] if c in diag[i][0] and i not in fail), None)\\n",\n    "        if hit is not None and len(fail) < n_each:\\n",\n    "            fail.append(hit)\\n",\n    "    fail += [i for i in fail_pool if i not in fail][:n_each - len(fail)]\\n",\n    "    cases = []\\n",\n    "    for kind, idx in ((\\"THÀNH CÔNG\\", succ), (\\"THẤT BẠI\\", fail)):\\n",\n    "        for i in idx:\\n",\n    "            qi, gi = int(q[i]), int(gold[i])\\n",\n    "            causes, notes = diag.get(i) or diagnose_case(run, qi, gi, sc[\\"cand\\"][i], int(rank[i]), int(clip_rank[i]), texts[i])\\n",\n    "            top = sc[\\"cand\\"][i][np.argsort(-fin[i])[:3]].tolist()\\n",\n    "            ex = explain_full(m, qi, gi) if gi in sc[\\"cand\\"][i] else None\\n",\n    "            ex_top1 = explain_full(m, qi, top[0]) if top and top[0] != gi else None\\n",\n    "            cases.append(dict(kind=kind, q=qi, query=texts[i], gold_vid=gi, gold_video_id=RECORDS[gi][\\"video_id\\"],\\n",\n    "                              rank_clip=int(clip_rank[i]), rank_final=int(rank[i]), in_candidates=bool(gi in sc[\\"cand\\"][i]),\\n",\n    "                              top3=[RECORDS[v][\\"video_id\\"] for v in top], causes=causes, notes=notes,\\n",\n    "                              gold_totals={k_: round(v_, 4) for k_, v_ in ex[\\"totals\\"].items()} if ex else None,\\n",\n    "                              gold_explanation=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, gi)}\\" for t in ex[\\"support\\"]] if ex else [],\\n",\n    "                              gold_opposing=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, gi)}\\" for t in ex[\\"oppose\\"][:3]] if ex else [],\\n",\n    "                              top1_explanation=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, top[0])}\\" for t in ex_top1[\\"support\\"]] if ex_top1 else []))\\n",\n    "    stats = dict(n_queries=len(q), top1=float(np.mean(rank == 1)), fail_gt10=float(np.mean(rank > 10)),\\n",\n    "                 cause_counts_in_fail_sample=dict(Counter(c for i in diag for c in diag[i][0])), n_fail_diagnosed=len(diag))\\n",\n    "    return cases, stats\\n",\n    "\\n",\n    "\\n",\n    "def cases_markdown(cases, stats, part_name):\\n",\n    "    L = [f\\"# Phân tích 5 thành công + 5 thất bại — {part_name}\\", \\"\\",\\n",\n    "         \\"Chọn theo luật cố định (xem `case_analysis_l4b`), KHÔNG chọn tay. Thành công = video đúng ở hạng 1; thất bại = hạng > 10.\\",\\n",\n    "         f\\"Thống kê trên {stats[\'n_queries\']} truy vấn mẫu: hạng 1 = {stats[\'top1\']:.1%}, hạng > 10 = {stats[\'fail_gt10\']:.1%}; \\"\\n",\n    "         f\\"nguyên nhân trong mẫu thất bại đã chẩn đoán ({stats[\'n_fail_diagnosed\']}): {stats[\'cause_counts_in_fail_sample\']}\\", \\"\\"]\\n",\n    "    for n_, c in enumerate(cases, 1):\\n",\n    "        L += [f\\"## {n_}. [{c[\'kind\']}] \\\\\\"{c[\'query\']}\\\\\\"\\", \\"\\",\\n",\n    "              f\\"- Video đúng: `{c[\'gold_video_id\']}` — hạng CLIP **{c[\'rank_clip\']}** → hạng cuối **{c[\'rank_final\']}**\\"\\n",\n    "              f\\"{\'\' if c[\'in_candidates\'] else \' (KHÔNG nằm trong top-k ứng viên CLIP ⇒ đồ thị không thể cứu)\'}\\",\\n",\n    "              f\\"- Top-3 trả về: {\', \'.join(\'`\' + v + \'`\' for v in c[\'top3\'])}\\",\\n",\n    "              f\\"- Chẩn đoán (luật cố định{\'; nguyên nhân thất bại\' if c[\'kind\'] == \'THẤT BẠI\' else \'\'}): {\', \'.join(c[\'causes\']) or \'—\'}\\"] + \\\\\\n",\n    "            [f\\"  - {x}\\" for x in c[\\"notes\\"]]\\n",\n    "        if c[\\"gold_totals\\"]:\\n",\n    "            L.append(f\\"- Phân rã điểm đồ thị của video đúng: {c[\'gold_totals\']}\\")\\n",\n    "        L += [\\"- Đường đi / quan hệ ỦNG HỘ video đúng (đóng góp thật vào điểm):\\"] + [f\\"  - {x}\\" for x in c[\\"gold_explanation\\"] or [\\"(không có số hạng đồ thị dương)\\"]]\\n",\n    "        if c[\\"gold_opposing\\"]:\\n",\n    "            L += [\\"- Số hạng PHẢN ĐỐI:\\"] + [f\\"  - {x}\\" for x in c[\\"gold_opposing\\"]]\\n",\n    "        if c[\\"top1_explanation\\"]:\\n",\n    "            L += [f\\"- Vì sao `{c[\'top3\'][0]}` (hạng 1, sai) được xếp trên:\\"] + [f\\"  - {x}\\" for x in c[\\"top1_explanation\\"]]\\n",\n    "        L.append(\\"\\")\\n",\n    "    return \\"\\\\n\\".join(L)\\n",\n    "\\n",\n    "\\n",\n    "# ---------------------------------- Demo: văn bản tự do → top-K video + giải thích ----------------------------------\\n",\n    "def thumb_path(vid):\\n",\n    "    \\"\\"\\"Thumbnail khung giữa: dùng file đã lưu lúc trích đặc trưng; nếu không có (dùng cache) thì đọc lại từ video.\\"\\"\\"\\n",\n    "    r = RECORDS[vid]\\n",\n    "    p = CFG.out_dir / \\"thumbs\\" / f\\"{r[\'video_id\']}.jpg\\"\\n",\n    "    if p.exists():\\n",\n    "        return p\\n",\n    "    try:\\n",\n    "        src = SOURCE if vid < N_MSVD else EXT_SOURCE\\n",\n    "        with src.open(r[\\"video_id\\"]) as path:\\n",\n    "            frames, _ = read_sampled_frames(path, 3)\\n",\n    "        mid = frames[1]\\n",\n    "        h, w = mid.shape[:2]\\n",\n    "        th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)\\n",\n    "        cv2.imwrite(str(p), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))\\n",\n    "        return p\\n",\n    "    except Exception:\\n",\n    "        return None\\n",\n    "\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def demo_query(text, run=None, part=None, k=None, show=True, save_png=None):\\n",\n    "    \\"\\"\\"Nhập văn bản tự do ⇒ top-k video của gallery `part` (mặc định tập xác nhận, nếu không thì tập thăm dò) với điểm CLIP, điểm đồ thị,\\n",\n    "    điểm cuối (α đã đóng băng) và giải thích = số hạng đồ thị ủng hộ. Văn bản demo KHÔNG đi vào CaptionStore và không được lưu làm dữ liệu.\\"\\"\\"\\n",\n    "    global TXT_FEAT\\n",\n    "    run = run or EXPS[G_IDS[\\"G3\\"]][\\"runs\\"][CFG.seeds[0]]\\n",\n    "    part = part or (EVALS[\\"confirm\\"][\\"part\\"] if \\"confirm\\" in EVALS else EVALS[\\"explore\\"][\\"part\\"])\\n",\n    "    k = k or CFG.demo_topk\\n",\n    "    m = run[\\"model\\"]\\n",\n    "    D = m.D\\n",\n    "    t = clip_encode_texts([text])[0].astype(TXT_FEAT.dtype)\\n",\n    "    q = len(TXT_FEAT)\\n",\n    "    toks = query_tokens_v2(text)\\n",\n    "    saved = (TXT_FEAT,)\\n",\n    "    TXT_FEAT = np.vstack([TXT_FEAT, t[None]])\\n",\n    "    V4_HITS.append(sorted({nid for nid, _, _ in parse_tokens_v2(toks, INV)}))\\n",\n    "    V4_ROLES.append(query_roles_v4([text], INV)[0])\\n",\n    "    QPREDS.append({rel: match_predicates(query_pred_tokens(text), rel) for rel in PRED_FAMILIES})\\n",\n    "    try:\\n",\n    "        gal = np.asarray(part[\\"gallery\\"])\\n",\n    "        cos = VID_FEAT[gal] @ t\\n",\n    "        top = np.argsort(-cos)[:CFG.rerank_k]\\n",\n    "        cand, s_clip = gal[top], cos[top]\\n",\n    "        qb = D.query_batch([q], m.use_pred)\\n",\n    "        s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()[0]\\n",\n    "        fin = run[\\"alpha\\"] * znorm_per_row(s_clip[None])[0] + (1 - run[\\"alpha\\"]) * znorm_per_row(s[None])[0]\\n",\n    "        order = np.argsort(-fin)[:k]\\n",\n    "        out = dict(query=text, gallery=part[\\"name\\"], alpha=run[\\"alpha\\"], matched_concepts=[node_name(h) for h in V4_HITS[q]],\\n",\n    "                   results=[])\\n",\n    "        for r_, j in enumerate(order, 1):\\n",\n    "            v = int(cand[j])\\n",\n    "            ex = explain_full(m, q, v)\\n",\n    "            out[\\"results\\"].append(dict(rank=r_, video_id=RECORDS[v][\\"video_id\\"], clip_cos=float(s_clip[j]), graph_score=float(s[j]),\\n",\n    "                                       final=float(fin[j]), clip_rank=int(j) + 1,          # cand đã xếp theo cosine giảm dần\\n",\n    "                                       totals={k_: round(v_, 4) for k_, v_ in ex[\\"totals\\"].items()},\\n",\n    "                                       explanation=[f\\"{tt[\'contrib\']:+.4f} {term_text(D, tt, v)}\\" for tt in ex[\\"support\\"]],\\n",\n    "                                       note=\\"\\" if ex[\\"support\\"] else \\"KHÔNG có đường đi giải thích hợp lệ (không số hạng đồ thị nào ủng hộ)\\"))\\n",\n    "    finally:\\n",\n    "        TXT_FEAT = saved[0]\\n",\n    "        V4_HITS.pop(); V4_ROLES.pop(); QPREDS.pop()\\n",\n    "    if show:\\n",\n    "        print(f\\"\\\\nTruy vấn: \'{text}\' | gallery {out[\'gallery\']} ({len(gal)} video) | khái niệm khớp: {out[\'matched_concepts\'] or \'—\'} | α = {out[\'alpha\']}\\")\\n",\n    "        for r in out[\\"results\\"]:\\n",\n    "            print(f\\"  #{r[\'rank\']} {r[\'video_id\']:>12s} | CLIP cos {r[\'clip_cos\']:.3f} (hạng CLIP {r[\'clip_rank\']}) | đồ thị {r[\'graph_score\']:+.3f} \\"\\n",\n    "                  f\\"| cuối {r[\'final\']:+.3f} | phân rã {r[\'totals\']}\\")\\n",\n    "            for x in r[\\"explanation\\"]:\\n",\n    "                print(f\\"       • {x}\\")\\n",\n    "            if r[\\"note\\"]:\\n",\n    "                print(f\\"       ⚠ {r[\'note\']}\\")\\n",\n    "        try:\\n",\n    "            import matplotlib.pyplot as plt\\n",\n    "            fig, axes = plt.subplots(1, len(out[\\"results\\"]), figsize=(3 * len(out[\\"results\\"]), 2.6))\\n",\n    "            for ax, r in zip(np.atleast_1d(axes), out[\\"results\\"]):\\n",\n    "                v = next(x[\\"vid\\"] for x in RECORDS if x[\\"video_id\\"] == r[\\"video_id\\"])\\n",\n    "                p = thumb_path(v)\\n",\n    "                if p is not None:\\n",\n    "                    ax.imshow(plt.imread(str(p)))\\n",\n    "                ax.set_title(f\\"#{r[\'rank\']} {r[\'video_id\']}\\\\nCLIP {r[\'clip_cos\']:.2f} | cuối {r[\'final\']:+.2f}\\", fontsize=8)\\n",\n    "                ax.axis(\\"off\\")\\n",\n    "            fig.suptitle(text, fontsize=9)\\n",\n    "            fig.tight_layout()\\n",\n    "            if save_png:\\n",\n    "                fig.savefig(save_png, dpi=110)\\n",\n    "            plt.show()\\n",\n    "            plt.close(fig)\\n",\n    "        except Exception as e:\\n",\n    "            print(f\\"  (không vẽ được thumbnail: {type(e).__name__}: {e})\\")\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def demo_widget():\\n",\n    "    \\"\\"\\"Ô nhập truy vấn tương tác (Jupyter / Colab / Kaggle interactive). Không tự chạy khi Save & Run All.\\"\\"\\"\\n",\n    "    import ipywidgets as w\\n",\n    "    box, btn, out = w.Text(placeholder=\\"ví dụ: a man is riding a horse\\", layout=w.Layout(width=\\"60%\\")), w.Button(description=\\"Tìm\\"), w.Output()\\n",\n    "\\n",\n    "    def go(_):\\n",\n    "        with out:\\n",\n    "            out.clear_output()\\n",\n    "            demo_query(box.value)\\n",\n    "    btn.on_click(go)\\n",\n    "    display(w.HBox([box, btn]), out)\\n",\n    "\\n",\n    "\\n",\n    "# ---------------------------------- Kiểm kê, audit, kết luận, manifest ----------------------------------\\n",\n    "def file_sha1(p, n=1 << 20):\\n",\n    "    h = hashlib.sha1()\\n",\n    "    with open(p, \\"rb\\") as f:\\n",\n    "        while True:\\n",\n    "            b = f.read(n)\\n",\n    "            if not b:\\n",\n    "                break\\n",\n    "            h.update(b)\\n",\n    "    return h.hexdigest()\\n",\n    "\\n",\n    "\\n",\n    "def split_counts():\\n",\n    "    out = {}\\n",\n    "    for s in sorted(set(CAP_SPLIT.tolist())):\\n",\n    "        vids = sorted(set(CAP_VID[CAP_SPLIT == s].tolist()))\\n",\n    "        out[s] = dict(videos=len(vids), captions=int((CAP_SPLIT == s).sum()),\\n",\n    "                      video_ids_sha1=sha1_of([RECORDS[v][\\"video_id\\"] for v in vids]))\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def graph_inventory(g):\\n",\n    "    s, d, e, w, pi = g.edge_arrays()\\n",\n    "    et = Counter(EDGE_TYPES[int(x)] for x in e)\\n",\n    "    nt = Counter(INV.node_type)\\n",\n    "    deg = np.bincount(np.concatenate([s, d]), minlength=INV.n_nodes) if len(s) else np.zeros(INV.n_nodes, int)\\n",\n    "    nonvid = np.array([t != \\"video\\" for t in INV.node_type])\\n",\n    "    return dict(nodes_by_type=dict(nt), edges_by_type=dict(et), n_edges=int(len(e)), self_loops=int((s == d).sum()),\\n",\n    "                duplicate_edges=int(len(e) - len(set(zip(s.tolist(), d.tolist(), e.tolist())))),\\n",\n    "                isolated_non_video_nodes=int(((deg == 0) & nonvid).sum()), videos_without_tag=int(sum(len(x) == 0 for x in g.vid_tags)),\\n",\n    "                degree_non_video=dict(mean=float(deg[nonvid].mean()), median=float(np.median(deg[nonvid])), max=int(deg[nonvid].max())),\\n",\n    "                edges_with_predicate=int((pi >= 0).sum()), schema_sha1=SCHEMA[\\"sha1\\"], inventory_sha1=INV.sha1)"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================== Level 4(b) v2: tagger HỌC từ caption TRAIN + tri thức ngoài ConceptNet (chỉ ĐỊNH NGHĨA) ==============================\\n",\n    "import gzip, urllib.request\\n",\n    "\\n",\n    "CaptionStore.ALLOWED[\\"tagger_train\\"] = {\\"train\\"}      # nhãn huấn luyện tagger học: CHỈ caption TRAIN\\n",\n    "KG_EDGES = []                                         # cạnh ConceptNet (khoá từ vựng) — nạp ở stage V8\\n",\n    "KG_KEEP = (\\"RelatedTo\\", \\"IsA\\", \\"PartOf\\", \\"HasA\\", \\"UsedFor\\", \\"CapableOf\\", \\"AtLocation\\", \\"Causes\\", \\"HasSubevent\\", \\"HasFirstSubevent\\",\\n",\n    "           \\"HasLastSubevent\\", \\"HasPrerequisite\\", \\"MotivatedByGoal\\", \\"Desires\\", \\"CreatedBy\\", \\"Synonym\\", \\"MadeOf\\", \\"ReceivesAction\\",\\n",\n    "           \\"LocatedNear\\", \\"SimilarTo\\", \\"MannerOf\\", \\"Entails\\", \\"InstanceOf\\")   # quan hệ DƯƠNG; bỏ Antonym, Not*, DistinctFrom, FormOf, ExternalURL…\\n",\n    "\\n",\n    "\\n",\n    "def drop_mask(mask, name):\\n",\n    "    parts = [m for m in str(mask).split(\\"+\\") if m not in (name, \\"none\\")]\\n",\n    "    return \\"+\\".join(sorted(parts)) or \\"none\\"\\n",\n    "\\n",\n    "\\n",\n    "def _probe_topk(S, rows, cols, kn, kv, terms):\\n",\n    "    out = []\\n",\n    "    for r in rows:\\n",\n    "        d = {}\\n",\n    "        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n",\n    "            c = cols[typ]\\n",\n    "            if k > 0 and len(c):\\n",\n    "                top = c[np.argsort(-S[r, c])[:k]]\\n",\n    "                d.update({terms[i]: float(S[r, i]) for i in top})\\n",\n    "        out.append(d)\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def run_tagger_probe(vid_feat, vocab, split, store, seed=0):\\n",\n    "    \\"\\"\\"Tagger HỌC: hồi quy logistic đa nhãn (một lớp tuyến tính) trên embedding CLIP đóng băng của video (chuẩn hoá theo TRAIN).\\n",\n    "    Nhãn = khái niệm xuất hiện ở ≥ truth_min_caps caption của video TRAIN. Chọn (weight decay, k danh từ, k động từ) theo F1 trên VAL.\\n",\n    "    Video TRAIN nhận nhãn OUT-OF-FOLD (k-fold) để nhiễu giống video mới; val/test/tập ngoài dùng mô hình học trên toàn bộ TRAIN.\\n",\n    "    Chỉ dùng frame của chính video lúc suy luận ⇒ không rò rỉ caption val/test.\\"\\"\\"\\n",\n    "    terms = vocab[\\"terms\\"]\\n",\n    "    idx = {t: i for i, t in enumerate(terms)}\\n",\n    "    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n",\n    "    tr, va = np.array(split[\\"train\\"]), np.array(split[\\"val\\"])\\n",\n    "    truth_tr = caption_truth_v2(store, tr, vocab, \\"tagger_train\\")\\n",\n    "    truth_va = caption_truth_v2(store, va, vocab, \\"tagger_select\\")\\n",\n    "    Y = np.zeros((len(tr), len(terms)), np.float32)\\n",\n    "    for j, v in enumerate(tr):\\n",\n    "        Y[j, [idx[t] for t in truth_tr[v]]] = 1.0\\n",\n    "    X = np.asarray(vid_feat, np.float32)\\n",\n    "    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6\\n",\n    "    Xt = torch.from_numpy((X - mu) / sd).to(DEVICE)\\n",\n    "\\n",\n    "    def fit(rows, wd):\\n",\n    "        set_seed(seed)\\n",\n    "        y = torch.from_numpy(Y[rows]).to(DEVICE)\\n",\n    "        prev = y.mean(0).clamp(1e-4, 1 - 1e-4)\\n",\n    "        lin = nn.Linear(X.shape[1], len(terms)).to(DEVICE)\\n",\n    "        nn.init.zeros_(lin.weight)\\n",\n    "        with torch.no_grad():\\n",\n    "            lin.bias.copy_(torch.log(prev / (1 - prev)))\\n",\n    "        opt = torch.optim.Adam(lin.parameters(), lr=CFG.probe_lr)\\n",\n    "        xr = Xt[torch.from_numpy(tr[rows]).to(DEVICE)]\\n",\n    "        for _ in range(CFG.probe_epochs):\\n",\n    "            opt.zero_grad()\\n",\n    "            loss = F.binary_cross_entropy_with_logits(lin(xr), y, reduction=\\"sum\\") / len(rows) + wd * lin.weight.pow(2).sum()\\n",\n    "            loss.backward()\\n",\n    "            opt.step()\\n",\n    "        with torch.no_grad():\\n",\n    "            return torch.sigmoid(lin(Xt)).cpu().numpy()\\n",\n    "\\n",\n    "    allrows = np.arange(len(tr))\\n",\n    "    grid, best = [], None\\n",\n    "    for wd in CFG.probe_wd_grid:\\n",\n    "        S = fit(allrows, wd)\\n",\n    "        for kn in CFG.tag_k_noun_grid:\\n",\n    "            for kv in CFG.tag_k_verb_grid:\\n",\n    "                P, R, F1 = _prf([set(d) for d in _probe_topk(S, va, cols, kn, kv, terms)], [truth_va[v] for v in va])\\n",\n    "                g = dict(wd=wd, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1)\\n",\n    "                grid.append(g)\\n",\n    "                if best is None or (round(F1, 9), -(kn + kv)) > (round(best[\\"val_F1\\"], 9), -(best[\\"k_noun\\"] + best[\\"k_verb\\"])):\\n",\n    "                    best = g\\n",\n    "    S = fit(allrows, best[\\"wd\\"])                                      # val / test / tập ngoài: mô hình trên toàn bộ TRAIN\\n",\n    "    fold = np.random.RandomState(seed).permutation(len(tr)) % CFG.probe_folds\\n",\n    "    for f in range(CFG.probe_folds):                                   # TRAIN: out-of-fold\\n",\n    "        S_f = fit(allrows[fold != f], best[\\"wd\\"])\\n",\n    "        S[tr[fold == f]] = S_f[tr[fold == f]]\\n",\n    "    tagged = _probe_topk(S, np.arange(len(X)), cols, best[\\"k_noun\\"], best[\\"k_verb\\"], terms)\\n",\n    "    tags = {v: sorted(d) for v, d in enumerate(tagged)}\\n",\n    "    scores = {v: d for v, d in enumerate(tagged)}\\n",\n    "    sel = dict(best, method=\\"logistic đa nhãn trên CLIP video (TRAIN, out-of-fold cho TRAIN)\\", folds=CFG.probe_folds,\\n",\n    "               epochs=CFG.probe_epochs, n_terms_with_train_positive=int((Y.sum(0) > 0).sum()))\\n",\n    "    return tags, scores, sel, grid\\n",\n    "\\n",\n    "\\n",\n    "def _cn_term(uri):\\n",\n    "    p = uri.split(\\"/\\")\\n",\n    "    if len(p) < 4 or p[2] != \\"en\\" or \\"_\\" in p[3]:\\n",\n    "        return None\\n",\n    "    return p[3], (p[4] if len(p) > 4 and p[4] in (\\"n\\", \\"v\\") else None)\\n",\n    "\\n",\n    "\\n",\n    "def _cn_keys(t, vset):\\n",\n    "    \\"\\"\\"Khoá từ vựng của một term ConceptNet: có POS ⇒ đúng POS; không POS ⇒ ưu tiên danh từ, chỉ dùng động từ khi không có danh từ.\\"\\"\\"\\n",\n    "    if t[1]:\\n",\n    "        k = f\\"{t[0]}#{t[1]}\\"\\n",\n    "        return [k] if k in vset else []\\n",\n    "    for x in (\\"n\\", \\"v\\"):\\n",\n    "        if f\\"{t[0]}#{x}\\" in vset:\\n",\n    "            return [f\\"{t[0]}#{x}\\"]\\n",\n    "    return []\\n",\n    "\\n",\n    "\\n",\n    "def load_conceptnet(vocab_terms):\\n",\n    "    \\"\\"\\"Cạnh ConceptNet 5.7 (en–en, một từ, quan hệ DƯƠNG trong KG_KEEP) mà HAI đầu đều là khoá từ vựng TRAIN. Tri thức NGOÀI:\\n",\n    "    không dùng caption hay nhãn nào ⇒ không rò rỉ. Nguồn: CFG.kg_file | /kaggle/input/**/<tên file> | tải luồng từ CFG.kg_url.\\n",\n    "    Kết quả lọc được cache (cache/conceptnet_*.json, dùng lại qua Add Input).\\"\\"\\"\\n",\n    "    vset = set(vocab_terms)\\n",\n    "    name = f\\"conceptnet_{CFG.kg_version}_{sha1_of(sorted(vset))}_{CFG.kg_max_lines}.json\\"\\n",\n    "    p = find_cache(name)\\n",\n    "    if p is not None:\\n",\n    "        print(f\\"Dùng lại cache ConceptNet: {p}\\")\\n",\n    "        return json.load(open(p))\\n",\n    "    fname = Path(CFG.kg_url).name\\n",\n    "    src = [Path(CFG.kg_file)] if CFG.kg_file else []\\n",\n    "    if RUN_ON_KAGGLE:\\n",\n    "        src += [Path(x) for x in glob.glob(f\\"/kaggle/input/**/{fname}\\", recursive=True)]\\n",\n    "    src = [x for x in src if x.exists()]\\n",\n    "    t0, edges, n = time.time(), {}, 0\\n",\n    "    if src:\\n",\n    "        fh, label = gzip.open(src[0], \\"rt\\", encoding=\\"utf-8\\"), str(src[0])\\n",\n    "    else:\\n",\n    "        if not INTERNET:\\n",\n    "            raise FileNotFoundError(f\\"không có {fname} và không có Internet\\")\\n",\n    "        fh, label = io.TextIOWrapper(gzip.GzipFile(fileobj=urllib.request.urlopen(CFG.kg_url, timeout=120)), encoding=\\"utf-8\\"), CFG.kg_url\\n",\n    "    print(f\\"Đọc ConceptNet từ {label} …\\")\\n",\n    "    with fh:\\n",\n    "        for n, line in enumerate(fh, 1):\\n",\n    "            if CFG.kg_max_lines and n > CFG.kg_max_lines:\\n",\n    "                break\\n",\n    "            parts = line.split(\\"\\\\t\\")\\n",\n    "            if len(parts) < 5 or parts[1][3:] not in KG_KEEP:\\n",\n    "                continue\\n",\n    "            a, b = _cn_term(parts[2]), _cn_term(parts[3])\\n",\n    "            if a is None or b is None:\\n",\n    "                continue\\n",\n    "            ka, kb = _cn_keys(a, vset), _cn_keys(b, vset)\\n",\n    "            if not ka or not kb:\\n",\n    "                continue\\n",\n    "            m = re.search(r\'\\"weight\\":\\\\s*([0-9.]+)\', parts[4])\\n",\n    "            w = float(m.group(1)) if m else 1.0\\n",\n    "            for x in ka:\\n",\n    "                for y in kb:\\n",\n    "                    if x != y:\\n",\n    "                        k = (x, y, parts[1][3:])\\n",\n    "                        edges[k] = max(edges.get(k, 0.0), w)\\n",\n    "    out = [dict(src=x, dst=y, rel=r, weight=w) for (x, y, r), w in sorted(edges.items())]\\n",\n    "    save_json(out, CFG.out_dir / \\"cache\\" / name)\\n",\n    "    print(f\\"ConceptNet: {n} dòng đọc, {len(out)} cạnh giữa khoá từ vựng ({time.time() - t0:.0f}s)\\")\\n",\n    "    return out"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "c045",\n   "metadata": {},\n   "source": [\n    "---\\n",\n    "# THỰC THI\\n",\n    "\\n",\n    "Thứ tự chạy:\\n",\n    "1. Dữ liệu: MSVD và MSR-VTT → split → đặc trưng → pooling.\\n",\n    "2. Từ vựng, vị từ, tagger, mẫu cho người chấm và nhãn.\\n",\n    "3. Đồ thị v3, schema, audit → self-test.\\n",\n    "4. Các stage V0–V7 trên VAL → **đóng băng**.\\n",\n    "5. **Bộ truy vấn đăng ký trước** (trực tiếp / gián tiếp) → đánh giá test MSVD (thăm dò) và MSR-VTT fresh1k-B (xác nhận).\\n",\n    "6. Phân tích quan hệ → xAI → metric bộ truy vấn → xAI theo thứ hạng → 5 + 5 ca → demo.\\n",\n    "7. Xuất file → báo cáo, kết luận → kiểm kê dữ liệu, audit rò rỉ, nghiệm thu, 7 câu kết luận, manifest → nén zip."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c046",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# ============================== CỔNG A: dữ liệu thật, đọc được ==============================\\n",\n    "mark_stage(\\"data\\", requires=(\\"config\\",))\\n",\n    "if CFG.dataset == \\"msvd\\":\\n",\n    "    RECORDS, _caps, SOURCE, DATA_REPORT = load_msvd()\\n",\n    "else:\\n",\n    "    RECORDS, _caps, SOURCE, DATA_REPORT = make_synthetic_msvd()\\n",\n    "N_MSVD = len(RECORDS)\\n",\n    "\\n",\n    "# ---- benchmark xác nhận độc lập (MSR-VTT, tập theo CFG.confirm_set — mặc định fresh1k); lỗi tải ⇒ NOT RUN, ghi rõ lý do, KHÔNG thay bằng tập khác ----\\n",\n    "EXT_RECORDS, EXT_SOURCE, EXT_REPORT, EXT_STATUS = [], None, {}, \\"NOT RUN — CFG.confirm = none\\"\\n",\n    "if CFG.confirm == \\"msrvtt\\":\\n",\n    "    try:\\n",\n    "        if CFG.dataset == \\"msvd\\":\\n",\n    "            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = load_msrvtt(N_MSVD)\\n",\n    "        else:\\n",\n    "            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = make_synthetic_ext(N_MSVD)\\n",\n    "        _caps.update(_ext_caps)\\n",\n    "        EXT_STATUS = \\"DEV (không phải tập xác nhận)\\" if EXT_REPORT[\\"dev\\"] else \\"OK\\"\\n",\n    "    except Exception as e:\\n",\n    "        EXT_RECORDS, EXT_SOURCE, EXT_STATUS = [], None, f\\"NOT RUN — không nạp được MSR-VTT: {type(e).__name__}: {e}\\"\\n",\n    "        print(\\"⚠\\", EXT_STATUS)\\n",\n    "RECORDS = RECORDS + EXT_RECORDS\\n",\n    "SPLIT_OF = {r[\\"vid\\"]: r[\\"split\\"] for r in RECORDS}\\n",\n    "CAPS = CaptionStore(_caps, SPLIT_OF)\\n",\n    "del _caps                                                  # caption chỉ còn trong CAPS\\n",\n    "CAP_VID, CAP_SPLIT = CAPS.vid_of_cap, CAPS.split_of_cap\\n",\n    "CAP_STD = np.zeros(len(CAP_VID), bool)                     # caption chuẩn 1k-A (truy vấn PHỤ của tập ngoài)\\n",\n    "_first_cap = {v: i for i, v in reversed(list(enumerate(CAP_VID.tolist())))}\\n",\n    "for r in EXT_RECORDS:\\n",\n    "    CAP_STD[_first_cap[r[\\"vid\\"]] + r[\\"std_caption\\"]] = True\\n",\n    "\\n",\n    "rep = DATA_REPORT[\\"splits\\"]\\n",\n    "print(f\\"Nguồn video: {DATA_REPORT[\'source\']} ({DATA_REPORT[\'source_label\']}) | {DATA_REPORT[\'n_video_files\']} file video\\")\\n",\n    "display(pd.DataFrame(rep).T)\\n",\n    "if CFG.dataset == \\"msvd\\":\\n",\n    "    got = {s: (rep[s][\\"videos\\"], rep[s][\\"captions\\"]) for s in rep}\\n",\n    "    check(\\"A1\\", \\"số video / caption theo split khớp dataset card MSVD\\", got == {s: tuple(v) for s, v in CFG.expected.items()},\\n",\n    "          f\\"thực tế {got} | kỳ vọng {CFG.expected}\\")\\n",\n    "    check(\\"A1b\\", \\"cột \'video\' của HF chỉ là tên file .avi (video thật lấy từ zip/thư mục)\\",\\n",\n    "          all(rep[s][\\"video_field_is_filename\\"] for s in rep), DATA_REPORT[\\"source_label\\"])\\n",\n    "else:\\n",\n    "    check(\\"A1\\", \\"số video / caption khớp MSVD\\", False, \\"dữ liệu SYNTHETIC — không áp dụng\\", hard=False, status=\\"NOT RUN\\")\\n",\n    "check(\\"A2\\", \\"mỗi video_id có đúng một file video (không thiếu file)\\",\\n",\n    "      all(rep[s][\\"missing_files\\"] == 0 for s in rep), {s: rep[s][\\"missing_files\\"] for s in rep})\\n",\n    "print(f\\"Tập ngoài: {EXT_STATUS} | {EXT_REPORT.get(\'label\', \'—\')} | {EXT_REPORT.get(\'videos\', 0)} video / \\"\\n",\n    "      f\\"{EXT_REPORT.get(\'captions\', 0)} caption\\")\\n",\n    "if EXT_RECORDS:\\n",\n    "    if EXT_REPORT[\\"dev\\"]:\\n",\n    "        ok_ext = EXT_REPORT[\\"overlap_with_test_1kA\\"] == 0\\n",\n    "    elif EXT_REPORT.get(\\"set\\") in (\\"fresh1k\\", \\"fresh1kB\\", \\"fresh1kC\\"):\\n",\n    "        ok_ext = (EXT_REPORT[\\"videos\\"] == CFG.fresh_n and EXT_REPORT[\\"overlap_with_test_1kA\\"] == 0\\n",\n    "                  and EXT_REPORT[\\"overlap_with_dev\\"] == 0\\n",\n    "                  and (EXT_REPORT[\\"set\\"] == \\"fresh1k\\" or EXT_REPORT[\\"overlap_with_fresh1k\\"] == 0)\\n",\n    "                  and (EXT_REPORT[\\"set\\"] != \\"fresh1kC\\" or EXT_REPORT[\\"overlap_with_fresh1kB\\"] == 0))\\n",\n    "    else:\\n",\n    "        ok_ext = EXT_REPORT[\\"videos\\"] == 1000 and EXT_REPORT[\\"overlap_with_test_1kA\\"] == 1000\\n",\n    "    check(\\"A6\\", \\"tập ngoài: đủ file video; fresh1k / fresh1k-B = đúng CFG.fresh_n video, rời 1k-A, rời 200 video dev (fresh1k-B rời cả fresh1k); \\"\\n",\n    "                \\"1kA = đúng 1000 video test 1k-A; dev = rời hẳn 1k-A\\",\\n",\n    "          EXT_REPORT[\\"missing_files\\"] == 0 and ok_ext,\\n",\n    "          {k: EXT_REPORT.get(k) for k in (\\"label\\", \\"set\\", \\"videos\\", \\"captions\\", \\"captions_per_video\\", \\"std_captions_added\\",\\n",\n    "                                          \\"missing_files\\", \\"overlap_with_test_1kA\\", \\"overlap_with_dev\\", \\"overlap_with_fresh1k\\", \\"overlap_with_fresh1kB\\")})\\n",\n    "else:\\n",\n    "    check(\\"A6\\", \\"tập ngoài (benchmark xác nhận độc lập)\\", False, EXT_STATUS, hard=False, status=\\"NOT RUN\\")\\n",\n    "\\n",\n    "probes = []\\n",\n    "for s in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\"):\\n",\n    "    src = EXT_SOURCE if s == \\"ext\\" else SOURCE\\n",\n    "    for r in [r for r in RECORDS if r[\\"split\\"] == s][:3]:\\n",\n    "        with src.open(r[\\"video_id\\"]) as p:\\n",\n    "            probes.append(dict(split=s, video_id=r[\\"video_id\\"], **probe_video(p)))\\n",\n    "display(pd.DataFrame(probes))\\n",\n    "check(\\"A3\\", \\"giải mã được video mẫu của TỪNG split (3 video/split, đếm frame bằng grab)\\",\\n",\n    "      all(p[\\"ok\\"] for p in probes),\\n",\n    "      f\\"{sum(p[\'ok\'] for p in probes)}/{len(probes)} video | codec {sorted({str(p.get(\'codec\')) for p in probes})} | \\"\\n",\n    "      f\\"frame {[p.get(\'frames_decoded\') for p in probes]}\\")\\n",\n    "DATA_REPORT[\\"probes\\"], DATA_REPORT[\\"ext\\"] = probes, dict(EXT_REPORT, status=EXT_STATUS)\\n",\n    "save_json(DATA_REPORT, RESULTS / \\"gateA_dataset_report.json\\")\\n",\n    "provenance(\\"RECORDS + CaptionStore\\", \\"external\\", f\\"{CFG.hf_repo}@{CFG.hf_revision[:8]}\\" if CFG.dataset == \\"msvd\\"\\n",\n    "           else \\"synthetic\\", note=\\"metadata + caption gốc; caption chỉ đọc qua CaptionStore\\")\\n",\n    "if EXT_RECORDS:\\n",\n    "    provenance(\\"tập ngoài (MSR-VTT)\\", \\"external\\", EXT_REPORT[\\"label\\"], note=\\"chỉ frame (encode) trước đóng băng; caption chỉ để đánh giá\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c047",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "mark_stage(\\"split\\", requires=(\\"data\\",))\\n",\n    "SPLIT = make_split(RECORDS)\\n",\n    "check(\\"A5\\", \\"split THEO VIDEO, không video nào ở hai split; manifest đã lưu\\",\\n",\n    "      sum(len(SPLIT[k]) for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\")) == len(RECORDS), f\\"sha1={SPLIT[\'sha1\']} → split_manifest.json\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c048",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# ============================== CỔNG B (1/2): frame + đặc trưng ==============================\\n",\n    "mark_stage(\\"features\\", requires=(\\"split\\",))\\n",\n    "FRAME_FEAT, FRAME_META, TXT_FEAT, FEAT_CACHES = all_features()\\n",\n    "n_bad = len(FRAME_META[\\"bad\\"])\\n",\n    "check(\\"A4\\", \\"TOÀN BỘ video giải mã được và lấy đủ F_max frame\\", n_bad == 0,\\n",\n    "      f\\"{len(RECORDS) - n_bad}/{len(RECORDS)} video đọc được | nguồn {FRAME_META[\'source\']} | {FRAME_META[\'seconds\']}s\\")\\n",\n    "save_json(dict(policy=CFG.frame_policy, n_frames_max=CFG.n_frames_max,\\n",\n    "               formula=\\"idx_k = floor((k + 0.5) / F_max · n_decoded), t_k = idx_k / fps\\",\\n",\n    "               per_video=FRAME_META[\\"per_video\\"], bad=FRAME_META[\\"bad\\"]), RESULTS / \\"frame_sampling.json\\")\\n",\n    "provenance(\\"FRAME_FEAT [N,12,D]\\", \\"frames\\", f\\"CLIP-image đóng băng {CFG.clip_model}/{CFG.clip_weights} trên 12 frame/video\\",\\n",\n    "           video_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]), note=\\" | \\".join(map(str, FEAT_CACHES)))\\n",\n    "provenance(\\"TXT_FEAT [N_caption,D]\\", \\"encoded\\", \\"CLIP-text đóng băng — mã hoá để cache, không học\\",\\n",\n    "           caption_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]), note=\\"việc DÙNG tuân theo CaptionStore\\")\\n",\n    "dur = np.array([m[\\"duration_s\\"] for m in FRAME_META[\\"per_video\\"].values() if m])\\n",\n    "print(f\\"Thời lượng video: min {dur.min():.1f}s | trung vị {np.median(dur):.1f}s | max {dur.max():.1f}s\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c049",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# ============================== CỔNG B (2/2): gộp theo thời gian + kiểm tra ==============================\\n",\n    "mark_stage(\\"pooling\\", requires=(\\"features\\",))\\n",\n    "POOL_SEL, POOL_TABLE = select_pooling_on_val(FRAME_FEAT, TXT_FEAT, SPLIT)\\n",\n    "VID_FEAT = pool_frames(FRAME_FEAT, POOL_SEL[\\"n_frames\\"], POOL_SEL[\\"pooling\\"])\\n",\n    "POOL_TABLE.to_csv(RESULTS / \\"pooling_selection_val.csv\\", index=False)\\n",\n    "display(POOL_TABLE.round(4))\\n",\n    "print(f\\"→ chọn trên VAL: {POOL_SEL}\\")\\n",\n    "provenance(\\"n_frames + pooling\\", \\"selected\\", \\"MRR CLIP-only text→video trên VAL\\", caption_splits=(\\"val\\",))\\n",\n    "provenance(\\"VID_FEAT [N,D]\\", \\"frames\\", f\\"gộp {POOL_SEL[\'n_frames\']} frame ({POOL_SEL[\'pooling\']})\\",\\n",\n    "           video_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]))\\n",\n    "gate_b_checks(FRAME_FEAT, FRAME_META, VID_FEAT, TXT_FEAT, SPLIT)\\n",\n    "save_json([c for c in CHECKS if c[\\"id\\"].startswith(\\"B\\")], RESULTS / \\"gateB_checks.json\\")"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## E1 — đồ thị bằng chứng đa nguồn (giai đoạn 1, chọn trên VAL)\\n",\n    "\\n",\n    "Không chạm test. Caption TRAIN chỉ dùng làm ngân hàng cạnh (mục đích `relations`).\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================== E1 (Level 4(b) v5) — ĐỒ THỊ BẰNG CHỨNG ĐA NGUỒN: cạnh encoder + cạnh hub + đường cầu nối qua TRAIN ==============================\\n",\n    "# Ứng viên = top-K của CLIP (đúng quy trình của đề). Điểm đồ thị = tổ hợp tuyến tính của các cạnh có kiểu, mỗi cạnh là một sự thật về\\n",\n    "# CHÍNH video/truy vấn đó (không còn cạnh lớp–lớp). Trọng số + siêu tham số chọn trên VAL (2-fold theo video). Ô này KHÔNG chạm test.\\n",\n    "mark_stage(\\"evidence_graph\\", requires=(\\"pooling\\",))\\n",\n    "import gc\\n",\n    "import pickle\\n",\n    "from scipy.optimize import minimize\\n",\n    "\\n",\n    "V5 = dict(pe_model=_env(\\"MSVD_V5_PE_MODEL\\", \\"PE-Core-L-14-336\\"), pe_weights=_env(\\"MSVD_V5_PE_W\\", \\"meta\\"),\\n",\n    "          K=int(_env(\\"MSVD_V5_K\\", \\"30\\")), hub_k_grid=[8, 32, 128], bridge_m_grid=[16, 64], bridge_tau_grid=[0.02, 0.05],\\n",\n    "          l2=1e-3, seed=20261101, n_boot=2000, dedup_source=True, chunk=2048)\\n",\n    "REC_BY_VID = {int(r[\\"vid\\"]): r for r in RECORDS}\\n",\n    "assert all(int(r[\\"vid\\"]) == i for i, r in enumerate(RECORDS)), \\"RECORDS phải xếp theo vid\\"\\n",\n    "\\n",\n    "# ---------------- 1. encoder thứ hai, đóng băng: Perception Encoder (PE-Core, NeurIPS 2025) trên CÙNG 12 frame ----------------\\n",\n    "_saved = (CFG.clip_model, CFG.clip_weights, CFG.clip_bs)\\n",\n    "_CLIP.clear(); gc.collect()\\n",\n    "CFG.clip_model, CFG.clip_weights, CFG.clip_bs = V5[\\"pe_model\\"], V5[\\"pe_weights\\"], min(CFG.clip_bs, 64)\\n",\n    "try:\\n",\n    "    FRAME_FEAT_PE, PE_META, TXT_FEAT_PE, PE_CACHES = all_features()\\n",\n    "finally:\\n",\n    "    CFG.clip_model, CFG.clip_weights, CFG.clip_bs = _saved\\n",\n    "    _CLIP.clear(); gc.collect()\\n",\n    "    if DEVICE == \\"cuda\\":\\n",\n    "        torch.cuda.empty_cache()\\n",\n    "VID_PE = pool_frames(FRAME_FEAT_PE, POOL_SEL[\\"n_frames\\"], POOL_SEL[\\"pooling\\"])\\n",\n    "_nrm = lambda x: (x / np.linalg.norm(x, axis=-1, keepdims=True).clip(1e-8)).astype(np.float32)\\n",\n    "VID_CL, TXT_CL, TXT_PE = _nrm(VID_FEAT), _nrm(TXT_FEAT), _nrm(TXT_FEAT_PE)\\n",\n    "check(\\"E0\\", \\"đặc trưng PE đủ cho mọi video / caption, cùng thứ tự với CLIP\\",\\n",\n    "      VID_PE.shape[0] == VID_CL.shape[0] and TXT_PE.shape[0] == TXT_CL.shape[0], f\\"video {VID_PE.shape} | caption {TXT_PE.shape}\\")\\n",\n    "provenance(\\"VID_PE / TXT_PE\\", \\"frames\\", f\\"{V5[\'pe_model\']}/{V5[\'pe_weights\']} đóng băng trên 12 frame/video\\",\\n",\n    "           video_splits=(\\"train\\", \\"val\\", \\"test\\"), note=\\" | \\".join(map(str, PE_CACHES)))\\n",\n    "\\n",\n    "# ---------------- 2. ngân hàng TRAIN: caption + video train (cạnh thống kê chỉ từ TRAIN) ----------------\\n",\n    "TR_VIDS = np.array(sorted(SPLIT[\\"train\\"]))\\n",\n    "BANK = np.where(np.isin(CAP_VID, TR_VIDS))[0]\\n",\n    "CAPS.use(BANK, \\"relations\\")\\n",\n    "BANK_POS = np.searchsorted(TR_VIDS, CAP_VID[BANK])              # caption TRAIN → vị trí video train của nó (cạnh \'mô tả\', nhãn thật)\\n",\n    "_SRC_RE = re.compile(r\\"^(.+)_\\\\d+_\\\\d+$\\")\\n",\n    "\\n",\n    "\\n",\n    "def source_id(v):\\n",\n    "    m = _SRC_RE.match(str(REC_BY_VID[int(v)][\\"video_id\\"]))\\n",\n    "    return m.group(1) if m else f\\"__{int(v)}\\"\\n",\n    "\\n",\n    "\\n",\n    "_src_codes = {s: i for i, s in enumerate(sorted({source_id(r[\\"vid\\"]) for r in RECORDS}))}\\n",\n    "SRC = np.array([_src_codes[source_id(r[\\"vid\\"])] for r in RECORDS])\\n",\n    "_shared = {s: sorted({SPLIT_OF[int(v)] for v in np.where(SRC == c)[0]}) for s, c in _src_codes.items()}\\n",\n    "N_SHARED = sum(1 for s in _shared.values() if len(s) > 1)\\n",\n    "print(f\\"Ngân hàng TRAIN: {len(BANK)} caption / {len(TR_VIDS)} video | {N_SHARED} nguồn YouTube xuất hiện ở > 1 split \\"\\n",\n    "      f\\"⇒ loại cạnh hub/cầu nối tới video train CÙNG nguồn (dedup={V5[\'dedup_source\']})\\")\\n",\n    "\\n",\n    "\\n",\n    "def _t(x):\\n",\n    "    return torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(DEVICE)\\n",\n    "\\n",\n    "\\n",\n    "# ---------------- 3. cạnh hub (NNN, EMNLP 2024): video ứng viên → k caption TRAIN gần nhất ----------------\\n",\n    "_HUB = {}\\n",\n    "\\n",\n    "\\n",\n    "def hub_bias(gal, enc, k, mode=\\"knn\\"):\\n",\n    "    \\"\\"\\"b(v) = trung bình k cosine lớn nhất giữa video v và caption TRAIN (bỏ caption của video train cùng nguồn).\\n",\n    "    mode=\'random\': k caption TRAIN ngẫu nhiên — đối chứng \'cạnh hub ngẫu nhiên\', cùng bậc k.\\"\\"\\"\\n",\n    "    key = (tuple(np.asarray(gal).tolist()), enc, k, mode)\\n",\n    "    if key in _HUB:\\n",\n    "        return _HUB[key]\\n",\n    "    V, T = (VID_CL, TXT_CL) if enc == \\"clip\\" else (VID_PE, TXT_PE)\\n",\n    "    out, nb = np.zeros(len(gal)), {}\\n",\n    "    bank_src = _t(SRC[CAP_VID[BANK]])\\n",\n    "    TB = _t(T[BANK])\\n",\n    "    rng = np.random.RandomState(V5[\\"seed\\"] + k)\\n",\n    "    for s in range(0, len(gal), 256):\\n",\n    "        g = np.asarray(gal[s:s + 256])\\n",\n    "        S = _t(V[g]) @ TB.T\\n",\n    "        if V5[\\"dedup_source\\"]:\\n",\n    "            S = S.masked_fill(_t(SRC[g])[:, None] == bank_src[None, :], float(\\"-inf\\"))\\n",\n    "        if mode == \\"knn\\":\\n",\n    "            vals, idx = S.topk(k, dim=1)\\n",\n    "            for j, v in enumerate(g):\\n",\n    "                nb[int(v)] = BANK[idx[j, :3].cpu().numpy()]\\n",\n    "        else:\\n",\n    "            idx = torch.from_numpy(rng.randint(0, len(BANK), (len(g), k))).to(DEVICE)\\n",\n    "            vals = S.gather(1, idx)\\n",\n    "            vals = torch.where(torch.isfinite(vals), vals, torch.zeros_like(vals))\\n",\n    "        out[s:s + len(g)] = vals.mean(1).cpu().numpy()\\n",\n    "    _HUB[key] = (dict(zip(np.asarray(gal).tolist(), out)), nb)\\n",\n    "    return _HUB[key]\\n",\n    "\\n",\n    "\\n",\n    "# ---------------- 4. đường cầu nối: truy vấn → caption TRAIN → video TRAIN (nhãn thật) → ứng viên ----------------\\n",\n    "_BR = {}\\n",\n    "\\n",\n    "\\n",\n    "def bridge_feature(C, enc, m, tau, shuffle=False):\\n",\n    "    \\"\\"\\"B[q, ứng viên] (gộp theo top-K của C), có cache.\\"\\"\\"\\n",\n    "    key = (hashlib.sha1(np.ascontiguousarray(C[\\"qidx\\"]).tobytes()).hexdigest(), hashlib.sha1(np.ascontiguousarray(C[\\"cand\\"]).tobytes()).hexdigest(),\\n",\n    "           enc, m, tau, shuffle)\\n",\n    "    if key not in _BR:\\n",\n    "        _BR[key] = np.take_along_axis(bridge_matrix(C[\\"qidx\\"], C[\\"gal\\"], enc, m, tau, shuffle), C[\\"top\\"], 1)\\n",\n    "    return _BR[key]\\n",\n    "\\n",\n    "\\n",\n    "def bridge_matrix(qidx, gal, enc, m, tau, shuffle=False):\\n",\n    "    \\"\\"\\"B[q, v] = Σ_u a_u(q) · max(cos(u, v), 0)³, a_u(q) = Σ_{c → u} softmax_τ(cos(q, c)) trên m caption TRAIN gần q nhất.\\n",\n    "    shuffle=True: hoán vị cạnh \'mô tả\' caption → video train (giữ số caption mỗi video) — đối chứng cấu trúc.\\"\\"\\"\\n",\n    "    V, T = (VID_CL, TXT_CL) if enc == \\"clip\\" else (VID_PE, TXT_PE)\\n",\n    "    pos = np.random.RandomState(V5[\\"seed\\"]).permutation(BANK_POS) if shuffle else BANK_POS\\n",\n    "    posT = torch.from_numpy(pos).to(DEVICE)\\n",\n    "    VV = (_t(V[TR_VIDS]) @ _t(V[gal]).T).clamp(min=0) ** 3                   # [n_train, |gal|]\\n",\n    "    if V5[\\"dedup_source\\"]:\\n",\n    "        VV = VV.masked_fill(_t(SRC[TR_VIDS])[:, None] == _t(SRC[gal])[None, :], 0.0)\\n",\n    "    TB = _t(T[BANK])\\n",\n    "    out = np.zeros((len(qidx), len(gal)), np.float32)\\n",\n    "    for s in range(0, len(qidx), V5[\\"chunk\\"]):\\n",\n    "        S = _t(T[qidx[s:s + V5[\\"chunk\\"]]]) @ TB.T\\n",\n    "        vals, idx = S.topk(m, dim=1)\\n",\n    "        w = torch.softmax(vals / tau, dim=1)\\n",\n    "        A = torch.zeros((len(vals), len(TR_VIDS)), device=DEVICE).scatter_add_(1, posT[idx], w)\\n",\n    "        out[s:s + len(vals)] = (A @ VV).cpu().numpy()\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def bridge_paths(q, v, enc, m, tau, top=3):\\n",\n    "    \\"\\"\\"Các đường q → c → u → v đóng góp nhiều nhất (giải thích): [(caption TRAIN idx, video train, đóng góp)].\\"\\"\\"\\n",\n    "    V, T = (VID_CL, TXT_CL) if enc == \\"clip\\" else (VID_PE, TXT_PE)\\n",\n    "    s = T[BANK] @ T[q]\\n",\n    "    idx = np.argsort(-s)[:m]\\n",\n    "    w = np.exp((s[idx] - s[idx].max()) / tau); w /= w.sum()\\n",\n    "    out = []\\n",\n    "    for c, wc in zip(idx, w):\\n",\n    "        u = TR_VIDS[BANK_POS[c]]\\n",\n    "        vv = max(float(V[u] @ V[v]), 0.0) ** 3 if not (V5[\\"dedup_source\\"] and SRC[u] == SRC[v]) else 0.0\\n",\n    "        out.append((int(BANK[c]), int(u), float(wc * vv)))\\n",\n    "    return sorted(out, key=lambda x: -x[2])[:top]\\n",\n    "\\n",\n    "\\n",\n    "# ---------------- 5. đặc trưng cạnh cho mọi (truy vấn, ứng viên top-K CLIP) ----------------\\n",\n    "FEATS_ALL = [\\"clip\\", \\"pe\\", \\"hub_clip\\", \\"hub_pe\\", \\"br_clip\\", \\"br_pe\\"]\\n",\n    "\\n",\n    "\\n",\n    "def candidate_set(qidx, gal, K):\\n",\n    "    gal = np.asarray(gal)\\n",\n    "    S = TXT_CL[qidx] @ VID_CL[gal].T\\n",\n    "    gold = CAP_VID[qidx]\\n",\n    "    gpos = np.searchsorted(gal, gold)\\n",\n    "    clip_rank = 1 + (S > S[np.arange(len(qidx)), gpos][:, None]).sum(1)\\n",\n    "    top = np.argsort(-S, 1, kind=\\"stable\\")[:, :K]\\n",\n    "    return dict(qidx=np.asarray(qidx), gal=gal, gold=gold, clip_rank=clip_rank, top=top, cand=gal[top],\\n",\n    "                clip=np.take_along_axis(S, top, 1))\\n",\n    "\\n",\n    "\\n",\n    "def edge_features(C, hp, hub_mode=\\"knn\\", bridge_shuffle=False):\\n",\n    "    \\"\\"\\"dict loại cạnh → [Q, K]. hp = (hub_k, bridge_m, bridge_tau).\\"\\"\\"\\n",\n    "    k, m, tau = hp\\n",\n    "    cand, F = C[\\"cand\\"], {\\"clip\\": C[\\"clip\\"]}\\n",\n    "    F[\\"pe\\"] = np.einsum(\\"qd,qkd->qk\\", TXT_PE[C[\\"qidx\\"]], VID_PE[cand])\\n",\n    "    for enc in (\\"clip\\", \\"pe\\"):\\n",\n    "        hb, _ = hub_bias(C[\\"gal\\"], enc, k, hub_mode)\\n",\n    "        F[f\\"hub_{enc}\\"] = np.vectorize(lambda v: hb[int(v)])(cand)\\n",\n    "        F[f\\"br_{enc}\\"] = bridge_feature(C, enc, m, tau, bridge_shuffle)\\n",\n    "    return F\\n",\n    "\\n",\n    "\\n",\n    "def zq(x):\\n",\n    "    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)\\n",\n    "\\n",\n    "\\n",\n    "def stack(F, feats):\\n",\n    "    return np.stack([zq(F[f]) for f in feats], -1)\\n",\n    "\\n",\n    "\\n",\n    "def gold_pos(C):\\n",\n    "    hit = C[\\"cand\\"] == C[\\"gold\\"][:, None]\\n",\n    "    return np.where(hit.any(1), hit.argmax(1), -1)\\n",\n    "\\n",\n    "\\n",\n    "def fit_w(Z, g, l2=None):\\n",\n    "    \\"\\"\\"Hồi quy softmax theo danh sách (ListNet top-1) trên các truy vấn có video đúng trong top-K; khởi tạo = chỉ CLIP.\\"\\"\\"\\n",\n    "    l2 = V5[\\"l2\\"] if l2 is None else l2\\n",\n    "    keep = g >= 0\\n",\n    "    Zk, gk = Z[keep], g[keep]\\n",\n    "    ar = np.arange(len(gk))\\n",\n    "\\n",\n    "    def f(w):\\n",\n    "        s = Zk @ w\\n",\n    "        s = s - s.max(1, keepdims=True)\\n",\n    "        p = np.exp(s); p /= p.sum(1, keepdims=True)\\n",\n    "        loss = -np.log(p[ar, gk] + 1e-12).mean() + l2 * w @ w\\n",\n    "        grad = ((p[..., None] * Zk).sum(1) - Zk[ar, gk]).mean(0) + 2 * l2 * w\\n",\n    "        return loss, grad\\n",\n    "\\n",\n    "    w0 = np.zeros(Z.shape[-1]); w0[0] = 1.0\\n",\n    "    return minimize(f, w0, jac=True, method=\\"L-BFGS-B\\").x\\n",\n    "\\n",\n    "\\n",\n    "QKEYS = (\\"qidx\\", \\"gold\\", \\"clip_rank\\", \\"top\\", \\"cand\\", \\"clip\\")\\n",\n    "\\n",\n    "\\n",\n    "def sub(C, mask):\\n",\n    "    return {k: (v[mask] if k in QKEYS else v) for k, v in C.items()}\\n",\n    "\\n",\n    "\\n",\n    "def feats_of(name):\\n",\n    "    return VARIANTS[name] if name in VARIANTS else VARIANTS[CONTROLS[name][0]]\\n",\n    "\\n",\n    "\\n",\n    "def ranks_of(C, score):\\n",\n    "    return ranks_with_fallback(C[\\"cand\\"], score, C[\\"gold\\"], C[\\"clip_rank\\"])\\n",\n    "\\n",\n    "\\n",\n    "def cv_rr(C, Z, folds):\\n",\n    "    \\"\\"\\"2-fold theo video: học w trên fold này, chấm fold kia.\\"\\"\\"\\n",\n    "    g = gold_pos(C)\\n",\n    "    rr, W = np.zeros(len(g)), []\\n",\n    "    for f in (0, 1):\\n",\n    "        tr, te = folds != f, folds == f\\n",\n    "        w = fit_w(Z[tr], g[tr])\\n",\n    "        W.append(w)\\n",\n    "        rr[te] = 1.0 / ranks_of(sub(C, te), Z[te] @ w)\\n",\n    "    return rr, W\\n",\n    "\\n",\n    "\\n",\n    "VARIANTS = {\\"M0_clip\\": [\\"clip\\"], \\"G1_clip_graph\\": [\\"clip\\", \\"hub_clip\\", \\"br_clip\\"], \\"M1_clip_pe_flat\\": [\\"clip\\", \\"pe\\"],\\n",\n    "            \\"M2_clip_pe_hub\\": [\\"clip\\", \\"pe\\", \\"hub_clip\\", \\"hub_pe\\"], \\"M3_clip_pe_bridge\\": [\\"clip\\", \\"pe\\", \\"br_clip\\", \\"br_pe\\"],\\n",\n    "            \\"M4_full_graph\\": FEATS_ALL}\\n",\n    "CONTROLS = {\\"M4_ctrl_hub_random\\": (\\"M4_full_graph\\", \\"random\\", False), \\"M4_ctrl_bridge_shuffled\\": (\\"M4_full_graph\\", \\"knn\\", True),\\n",\n    "            \\"M4_ctrl_both\\": (\\"M4_full_graph\\", \\"random\\", True), \\"G1_ctrl_both\\": (\\"G1_clip_graph\\", \\"random\\", True)}\\n",\n    "\\n",\n    "# ---------------- 6. chọn siêu tham số + trọng số trên VAL (2-fold theo video) ----------------\\n",\n    "VAL_Q = np.where(np.isin(CAP_VID, SPLIT[\\"val\\"]))[0]\\n",\n    "CAPS.use(VAL_Q, \\"tune\\")\\n",\n    "C_VAL = candidate_set(VAL_Q, np.array(sorted(SPLIT[\\"val\\"])), V5[\\"K\\"])\\n",\n    "FOLD_VAL = np.searchsorted(np.unique(C_VAL[\\"gold\\"]), C_VAL[\\"gold\\"]) % 2\\n",\n    "print(f\\"VAL: {len(VAL_Q)} truy vấn / {len(C_VAL[\'gal\'])} video | CLIP R@{V5[\'K\']} = {(C_VAL[\'clip_rank\'] <= V5[\'K\']).mean():.3f}\\")\\n",\n    "HP_ROWS = []\\n",\n    "for hp in itertools.product(V5[\\"hub_k_grid\\"], V5[\\"bridge_m_grid\\"], V5[\\"bridge_tau_grid\\"]):\\n",\n    "    F = edge_features(C_VAL, hp)\\n",\n    "    rr, _ = cv_rr(C_VAL, stack(F, FEATS_ALL), FOLD_VAL)\\n",\n    "    HP_ROWS.append(dict(hub_k=hp[0], bridge_m=hp[1], bridge_tau=hp[2], val_cv_MRR=rr.mean()))\\n",\n    "HP_TABLE = pd.DataFrame(HP_ROWS).sort_values(\\"val_cv_MRR\\", ascending=False)\\n",\n    "HP_TABLE.to_csv(RESULTS / \\"v5_hp_selection_val.csv\\", index=False)\\n",\n    "_b = HP_TABLE.iloc[0]\\n",\n    "HP = (int(_b[\\"hub_k\\"]), int(_b[\\"bridge_m\\"]), float(_b[\\"bridge_tau\\"]))\\n",\n    "print(f\\"→ siêu tham số chọn trên VAL: hub_k={HP[0]}, bridge_m={HP[1]}, bridge_tau={HP[2]} (val-CV MRR {_b[\'val_cv_MRR\']:.4f})\\")\\n",\n    "provenance(\\"hub_k, bridge_m, bridge_tau\\", \\"selected\\", \\"MRR 2-fold theo video trên VAL\\", caption_splits=(\\"val\\",))\\n",\n    "\\n",\n    "F_VAL = edge_features(C_VAL, HP)\\n",\n    "F_VAL_CTRL = {(hm, bs): edge_features(C_VAL, HP, hm, bs) for _, hm, bs in CONTROLS.values()}\\n",\n    "W_FINAL, VAL_ROWS, RR_VAL = {}, [], {}\\n",\n    "for name, feats in VARIANTS.items():\\n",\n    "    Z = stack(F_VAL, feats)\\n",\n    "    RR_VAL[name], _ = cv_rr(C_VAL, Z, FOLD_VAL)\\n",\n    "    W_FINAL[name] = fit_w(Z, gold_pos(C_VAL))\\n",\n    "for name, (base, hm, bs) in CONTROLS.items():\\n",\n    "    Z = stack(F_VAL_CTRL[(hm, bs)], VARIANTS[base])\\n",\n    "    RR_VAL[name], _ = cv_rr(C_VAL, Z, FOLD_VAL)\\n",\n    "    W_FINAL[name] = fit_w(Z, gold_pos(C_VAL))\\n",\n    "for name in RR_VAL:\\n",\n    "    VAL_ROWS.append(dict(model=name, val_cv_MRR=RR_VAL[name].mean(), val_cv_R1=(RR_VAL[name] == 1).mean(),\\n",\n    "                         weights=json.dumps({f: round(float(w), 4) for f, w in zip(feats_of(name), W_FINAL[name])})))\\n",\n    "VAL_TABLE = pd.DataFrame(VAL_ROWS)\\n",\n    "VAL_TABLE.to_csv(RESULTS / \\"v5_val_cv.csv\\", index=False)\\n",\n    "display(VAL_TABLE.round(4))\\n",\n    "provenance(\\"trọng số cạnh (W_FINAL)\\", \\"selected\\", \\"softmax theo danh sách trên truy vấn VAL (không dùng caption TRAIN làm truy vấn)\\",\\n",\n    "           caption_splits=(\\"val\\",))\\n",\n    "check(\\"E1\\", \\"trọng số học được hữu hạn; mô hình đầy đủ không kém CLIP trên VAL-CV\\",\\n",\n    "      all(np.isfinite(w).all() for w in W_FINAL.values()) and RR_VAL[\\"M4_full_graph\\"].mean() >= RR_VAL[\\"M0_clip\\"].mean() - 0.01,\\n",\n    "      f\\"M0 {RR_VAL[\'M0_clip\'].mean():.4f} | M1 {RR_VAL[\'M1_clip_pe_flat\'].mean():.4f} | M4 {RR_VAL[\'M4_full_graph\'].mean():.4f}\\", hard=False)\\n"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## E2 — node VLM, đóng băng, chấm test MSVD một lần\\n",\n    "\\n",\n    "Luật đọc kết quả đã đăng ký trước ở đầu notebook.\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================== E2 (Level 4(b) v5) — node VLM \'toàn câu\' + ĐÓNG BĂNG + chấm TEST MSVD một lần ==============================\\n",\n    "# (1) VLM (Qwen2.5-VL, zero-shot) chấm top-M của đồ thị trên một tập con VAL → chọn β. (2) Đóng băng mọi cấu hình. (3) Chấm test MSVD:\\n",\n    "# giai đoạn 1 trên MỌI caption test; giai đoạn 2 (VLM) trên 1 caption / video test. Test MSVD đã được nhìn ở v1–v4 ⇒ nhãn THĂM DÒ.\\n",\n    "mark_stage(\\"vlm_test\\", requires=(\\"evidence_graph\\",))\\n",\n    "import importlib.metadata as _ilm\\n",\n    "from PIL import Image\\n",\n    "\\n",\n    "VLM = dict(model=_env(\\"MSVD_VLM_MODEL\\", \\"\\"), M=int(_env(\\"MSVD_V5_M\\", \\"10\\")), bs=int(_env(\\"MSVD_VLM_BS\\", \\"8\\")),\\n",\n    "           grid=3, cell=168, val_per_video=int(_env(\\"MSVD_V5_VAL_PER_VIDEO\\", \\"3\\")),\\n",\n    "           val_budget_min=float(_env(\\"MSVD_V5_VAL_BUDGET_MIN\\", \\"40\\")), test_budget_min=float(_env(\\"MSVD_V5_TEST_BUDGET_MIN\\", \\"120\\")),\\n",\n    "           seed=20261101, sanity_n=32, sanity_min_auc=0.70, beta_grid=[round(float(b), 2) for b in np.linspace(0, 1, 11)])\\n",\n    "REAL = CFG.dataset == \\"msvd\\"\\n",\n    "CaptionStore.ALLOWED[\\"vlm_probe\\"] = {\\"train\\", \\"val\\"}\\n",\n    "\\n",\n    "\\n",\n    "def _ver(s):\\n",\n    "    return tuple(int(x) for x in re.findall(r\\"\\\\d+\\", s)[:2])\\n",\n    "\\n",\n    "\\n",\n    "try:\\n",\n    "    _tv = _ilm.version(\\"transformers\\")\\n",\n    "except _ilm.PackageNotFoundError:\\n",\n    "    _tv = \\"0\\"\\n",\n    "if _ver(_tv) < (4, 49):\\n",\n    "    subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", \\"transformers>=4.49\\", \\"accelerate\\"], check=True)\\n",\n    "    if \\"transformers\\" in sys.modules:\\n",\n    "        raise RuntimeError(\\"transformers cũ đã được import trước cell này — Restart kernel rồi chạy lại (bản mới đã cài).\\")\\n",\n    "import transformers\\n",\n    "ENV_INFO[\\"transformers\\"] = transformers.__version__\\n",\n    "N_GPU = torch.cuda.device_count() if DEVICE == \\"cuda\\" else 0\\n",\n    "if not VLM[\\"model\\"]:\\n",\n    "    VLM[\\"model\\"] = \\"Qwen/Qwen2.5-VL-7B-Instruct\\" if N_GPU >= 2 else \\"Qwen/Qwen2.5-VL-3B-Instruct\\"\\n",\n    "PREFIX = f\\"The image shows {VLM[\'grid\'] ** 2} frames of one video in temporal order (left to right, top to bottom). \\"\\n",\n    "SUFFIX = \\" Answer yes or no.\\"\\n",\n    "\\n",\n    "\\n",\n    "def holistic_q(text):\\n",\n    "    return f\'Does the video show the following: \\"{str(text).strip().rstrip(\\".\\")}\\"?\'\\n",\n    "\\n",\n    "\\n",\n    "def video_grid(v):\\n",\n    "    r, g, c = REC_BY_VID[int(v)], VLM[\\"grid\\"], VLM[\\"cell\\"]\\n",\n    "    with SOURCE.open(r[\\"video_id\\"]) as path:\\n",\n    "        frames, _ = read_sampled_frames(path, g * g)\\n",\n    "    img = Image.new(\\"RGB\\", (c * g, c * g))\\n",\n    "    for k, fr in enumerate(frames):\\n",\n    "        img.paste(Image.fromarray(np.ascontiguousarray(fr)).convert(\\"RGB\\").resize((c, c), Image.BICUBIC), ((k % g) * c, (k // g) * c))\\n",\n    "    return img\\n",\n    "\\n",\n    "\\n",\n    "class YesNoVLM:\\n",\n    "    def __init__(self, name, vis_fp32=False):\\n",\n    "        from transformers import AutoProcessor\\n",\n    "        try:\\n",\n    "            from transformers import Qwen2_5_VLForConditionalGeneration as _M\\n",\n    "        except ImportError:\\n",\n    "            from transformers import AutoModelForImageTextToText as _M\\n",\n    "        self.name, self.dtype = name, (torch.float16 if DEVICE == \\"cuda\\" else torch.float32)\\n",\n    "        kw = dict(attn_implementation=\\"sdpa\\")\\n",\n    "        if N_GPU:\\n",\n    "            kw[\\"device_map\\"] = \\"auto\\"\\n",\n    "            kw[\\"max_memory\\"] = {i: f\\"{max(1, int(torch.cuda.get_device_properties(i).total_memory / 2**30 - 2.5))}GiB\\" for i in range(N_GPU)}\\n",\n    "        kw[\\"dtype\\" if _ver(transformers.__version__) >= (4, 56) else \\"torch_dtype\\"] = self.dtype\\n",\n    "        self.model = _M.from_pretrained(name, **kw).eval()\\n",\n    "        if not N_GPU:\\n",\n    "            self.model.to(DEVICE)\\n",\n    "        dmap = getattr(self.model, \\"hf_device_map\\", {}) or {}\\n",\n    "        assert not N_GPU or not any(str(d) in (\\"cpu\\", \\"disk\\") for d in dmap.values()), f\\"VLM bị offload: {set(dmap.values())}\\"\\n",\n    "        P = (VLM[\\"cell\\"] * VLM[\\"grid\\"]) ** 2\\n",\n    "        self.proc = AutoProcessor.from_pretrained(name, min_pixels=P, max_pixels=P)\\n",\n    "        ip = self.proc.image_processor\\n",\n    "        for a in (\\"min_pixels\\", \\"max_pixels\\"):\\n",\n    "            if hasattr(ip, a):\\n",\n    "                setattr(ip, a, P)\\n",\n    "        if isinstance(getattr(ip, \\"size\\", None), dict) and \\"shortest_edge\\" in ip.size:\\n",\n    "            ip.size = {\\"shortest_edge\\": P, \\"longest_edge\\": P}\\n",\n    "        self.proc.tokenizer.padding_side = \\"left\\"\\n",\n    "        tok = self.proc.tokenizer\\n",\n    "        one = lambda ws: sorted({t[0] for t in (tok.encode(w, add_special_tokens=False) for w in ws) if len(t) == 1})\\n",\n    "        self.yes, self.no = one([\\"Yes\\", \\"yes\\", \\" Yes\\", \\" yes\\"]), one([\\"No\\", \\"no\\", \\" No\\", \\" no\\"])\\n",\n    "        assert self.yes and self.no and not set(self.yes) & set(self.no)\\n",\n    "        self.dev = self.model.get_input_embeddings().weight.device\\n",\n    "        self.visual = getattr(self.model, \\"visual\\", None) or getattr(getattr(self.model, \\"model\\", None), \\"visual\\", None)\\n",\n    "        self.vis_fp32 = False\\n",\n    "        if vis_fp32:\\n",\n    "            self.set_vis_fp32()\\n",\n    "        self.n_forward, self.seconds = 0, 0.0\\n",\n    "\\n",\n    "    def set_vis_fp32(self):\\n",\n    "        self.visual.float()\\n",\n    "        self.vis_fp32 = True\\n",\n    "\\n",\n    "    def logodds(self, items):\\n",\n    "        texts = [self.proc.apply_chat_template([{\\"role\\": \\"user\\", \\"content\\": [{\\"type\\": \\"image\\"}, {\\"type\\": \\"text\\", \\"text\\": q}]}],\\n",\n    "                                               tokenize=False, add_generation_prompt=True) for _, q in items]\\n",\n    "        enc = self.proc(text=texts, images=[im for im, _ in items], padding=True, return_tensors=\\"pt\\")\\n",\n    "        enc = {k: (v.to(self.dev) if hasattr(v, \\"to\\") else v) for k, v in enc.items()}\\n",\n    "        if \\"pixel_values\\" in enc:\\n",\n    "            enc[\\"pixel_values\\"] = enc[\\"pixel_values\\"].to(torch.float32 if self.vis_fp32 else self.dtype)\\n",\n    "        t0 = time.time()\\n",\n    "        with torch.no_grad():\\n",\n    "            try:\\n",\n    "                out = self.model(**enc, logits_to_keep=1)\\n",\n    "            except TypeError:\\n",\n    "                out = self.model(**enc)\\n",\n    "        last = out.logits[:, -1, :].float()\\n",\n    "        e = torch.logsumexp(last[:, self.yes], 1) - torch.logsumexp(last[:, self.no], 1)\\n",\n    "        self.n_forward += len(items); self.seconds += time.time() - t0\\n",\n    "        return e.cpu().numpy().astype(np.float64)\\n",\n    "\\n",\n    "    def run(self, items):\\n",\n    "        out = [self.logodds(items[s:s + VLM[\\"bs\\"]]) for s in range(0, len(items), VLM[\\"bs\\"])]\\n",\n    "        return np.concatenate(out) if out else np.zeros(0)\\n",\n    "\\n",\n    "    def close(self):\\n",\n    "        del self.model\\n",\n    "        gc.collect()\\n",\n    "        if DEVICE == \\"cuda\\":\\n",\n    "            torch.cuda.empty_cache()\\n",\n    "\\n",\n    "\\n",\n    "def pair_auc(pos, neg):\\n",\n    "    pos, neg = np.asarray(pos), np.asarray(neg)\\n",\n    "    return float(((pos[:, None] > neg[None]) + 0.5 * (pos[:, None] == neg[None])).mean())\\n",\n    "\\n",\n    "\\n",\n    "GRID, EVID = {}, {}\\n",\n    "\\n",\n    "\\n",\n    "def ensure_grids(vids):\\n",\n    "    for v in sorted(set(int(x) for x in vids) - set(GRID)):\\n",\n    "        GRID[v] = video_grid(v)\\n",\n    "\\n",\n    "\\n",\n    "def score_pairs(pairs, texts, budget_s, t0):\\n",\n    "    \\"\\"\\"pairs = [(video, caption idx)] → EVID[(video, caption idx)] = log-odds yes/no; dừng khi hết ngân sách.\\"\\"\\"\\n",\n    "    need = [p for p in dict.fromkeys(pairs) if p not in EVID]\\n",\n    "    for s in range(0, len(need), VLM[\\"bs\\"] * 4):\\n",\n    "        if time.time() - t0 > budget_s:\\n",\n    "            return False\\n",\n    "        chunk = need[s:s + VLM[\\"bs\\"] * 4]\\n",\n    "        ensure_grids([v for v, _ in chunk])\\n",\n    "        e = SCORER.run([(GRID[v], PREFIX + holistic_q(texts[c]) + SUFFIX) for v, c in chunk])\\n",\n    "        EVID.update({k: float(x) for k, x in zip(chunk, e)})\\n",\n    "    return True\\n",\n    "\\n",\n    "\\n",\n    "# ---------------- 1. nạp VLM + kiểm tra (caption đúng vs caption video khác, chỉ VAL) ----------------\\n",\n    "_sv = np.array(sorted(SPLIT[\\"val\\"]))[:VLM[\\"sanity_n\\"]]\\n",\n    "_own = CAPS.by_video(_sv, \\"vlm_probe\\")\\n",\n    "ensure_grids(_sv)\\n",\n    "SANITY_ITEMS = [(GRID[int(v)], PREFIX + holistic_q(_own[int(v)][0]) + SUFFIX) for v in _sv] + \\\\\\n",\n    "               [(GRID[int(v)], PREFIX + holistic_q(_own[int(w)][0]) + SUFFIX) for v, w in zip(_sv, np.roll(_sv, 1))]\\n",\n    "ATTEMPTS = [(VLM[\\"model\\"], False), (VLM[\\"model\\"], True)] + ([(\\"Qwen/Qwen2.5-VL-3B-Instruct\\", True)] if N_GPU >= 2 and \\"7B\\" in VLM[\\"model\\"] else [])\\n",\n    "SCORER, SANITY_LOG = None, []\\n",\n    "for name, vis32 in ATTEMPTS:\\n",\n    "    try:\\n",\n    "        if SCORER is not None and SCORER.name == name and vis32 and not SCORER.vis_fp32:\\n",\n    "            SCORER.set_vis_fp32()\\n",\n    "        else:\\n",\n    "            if SCORER is not None:\\n",\n    "                SCORER.close(); SCORER = None\\n",\n    "            SCORER = YesNoVLM(name, vis_fp32=vis32)\\n",\n    "        e = SCORER.run(SANITY_ITEMS); n = len(_sv)\\n",\n    "        sc = dict(finite=bool(np.isfinite(e).all()), auc=pair_auc(e[:n], e[n:]) if np.isfinite(e).all() else float(\\"nan\\"))\\n",\n    "    except Exception as ex:\\n",\n    "        sc = dict(finite=False, auc=float(\\"nan\\"), error=f\\"{type(ex).__name__}: {ex}\\")\\n",\n    "    SANITY_LOG.append(dict(model=name, vis_fp32=vis32, **sc))\\n",\n    "    print(f\\"Kiểm tra VLM {name} (visual fp32={vis32}): {sc}\\")\\n",\n    "    if sc.get(\\"finite\\") and (sc[\\"auc\\"] >= VLM[\\"sanity_min_auc\\"] or not REAL):\\n",\n    "        break\\n",\n    "else:\\n",\n    "    if REAL and SCORER is not None:\\n",\n    "        SCORER.close(); SCORER = None\\n",\n    "VLM_OK = SCORER is not None\\n",\n    "check(\\"V1\\", f\\"VLM phân biệt caption đúng / sai của video VAL (AUC ≥ {VLM[\'sanity_min_auc\']})\\",\\n",\n    "      VLM_OK and SANITY_LOG[-1].get(\\"auc\\", 0) >= VLM[\\"sanity_min_auc\\"], SANITY_LOG[-1], hard=False, status=None if REAL else \\"NOT RUN\\")\\n",\n    "\\n",\n    "\\n",\n    "# ---------------- 2. giai đoạn 2 trên VAL: chọn β (2-fold theo video) ----------------\\n",\n    "def stage1_scores(C, F, name):\\n",\n    "    return stack(F, feats_of(name)) @ W_FINAL[name]\\n",\n    "\\n",\n    "\\n",\n    "def topM(C, s1):\\n",\n    "    return np.argsort(-s1, 1, kind=\\"stable\\")[:, :VLM[\\"M\\"]]\\n",\n    "\\n",\n    "\\n",\n    "def stage2_rank(C, s1, rows, vlm, beta, top=None):\\n",\n    "    \\"\\"\\"Top-M theo s1 được xếp lại bằng (1−β)·z(s1) + β·z(VLM); phần còn lại của top-K giữ thứ tự s1; ngoài top-K giữ hạng CLIP.\\"\\"\\"\\n",\n    "    s1r = s1[rows]\\n",\n    "    top = topM(C, s1) [rows] if top is None else top\\n",\n    "    blk = (1 - beta) * zq(np.take_along_axis(s1r, top, 1)) + beta * zq(vlm)\\n",\n    "    full = s1r - s1r.max() - 1e3                               # dưới mọi phần tử của khối top-M\\n",\n    "    np.put_along_axis(full, top, 1e3 + blk, 1)\\n",\n    "    return ranks_of(sub(C, rows), full)\\n",\n    "\\n",\n    "\\n",\n    "def vlm_matrix(C, rows, top, texts):\\n",\n    "    return np.array([[EVID.get((int(C[\\"cand\\"][r, j]), int(C[\\"qidx\\"][r])), np.nan) for j in t] for r, t in zip(rows, top)])\\n",\n    "\\n",\n    "\\n",\n    "def choose_beta(C, s1, rows, vlm, top=None):\\n",\n    "    \\"\\"\\"β chọn trên fold kia (2-fold theo video), hoà ⇒ β nhỏ hơn (gần giai đoạn 1 hơn). Trả RR CV và β cuối (học trên mọi hàng).\\"\\"\\"\\n",\n    "    gold = C[\\"gold\\"][rows]\\n",\n    "    folds = np.searchsorted(np.unique(gold), gold) % 2\\n",\n    "    rr = np.zeros(len(rows))\\n",\n    "    mrr = lambda b, m: (1.0 / stage2_rank(C, s1, rows[m], vlm[m], b, None if top is None else top[m])).mean()\\n",\n    "    for f in (0, 1):\\n",\n    "        tr, te = folds != f, folds == f\\n",\n    "        b = max(VLM[\\"beta_grid\\"], key=lambda b: (mrr(b, tr), -b))\\n",\n    "        rr[te] = 1.0 / stage2_rank(C, s1, rows[te], vlm[te], b, None if top is None else top[te])\\n",\n    "    return rr, max(VLM[\\"beta_grid\\"], key=lambda b: (mrr(b, np.ones(len(rows), bool)), -b))\\n",\n    "\\n",\n    "\\n",\n    "_rng = np.random.RandomState(VLM[\\"seed\\"])\\n",\n    "_vrows = np.concatenate([_rng.choice(np.where(C_VAL[\\"gold\\"] == v)[0], min(VLM[\\"val_per_video\\"], (C_VAL[\\"gold\\"] == v).sum()), replace=False)\\n",\n    "                         for v in np.unique(C_VAL[\\"gold\\"])])\\n",\n    "_vrows = np.sort(_vrows)\\n",\n    "VAL_TEXT = dict(zip(C_VAL[\\"qidx\\"][_vrows].tolist(), CAPS.texts(C_VAL[\\"qidx\\"][_vrows], \\"tune\\")))\\n",\n    "S1_VAL = stage1_scores(C_VAL, F_VAL, \\"M4_full_graph\\")\\n",\n    "TOP_VAL = topM(C_VAL, S1_VAL)[_vrows]\\n",\n    "TOPC_VAL = np.tile(np.arange(VLM[\\"M\\"]), (len(_vrows), 1))               # top-M của CLIP (ứng viên đã xếp theo CLIP)\\n",\n    "BETA, BETA_CLIP, VAL_STAGE2 = 0.0, 0.0, {}\\n",\n    "if VLM_OK:\\n",\n    "    t0 = time.time()\\n",\n    "    pairs = [(int(C_VAL[\\"cand\\"][r, j]), int(C_VAL[\\"qidx\\"][r])) for r, t in zip(_vrows, TOP_VAL) for j in t] + \\\\\\n",\n    "            [(int(C_VAL[\\"cand\\"][r, j]), int(C_VAL[\\"qidx\\"][r])) for r, t in zip(_vrows, TOPC_VAL) for j in t]\\n",\n    "    done = score_pairs(pairs, VAL_TEXT, VLM[\\"val_budget_min\\"] * 60, t0)\\n",\n    "    ok = np.array([all((int(C_VAL[\\"cand\\"][r, j]), int(C_VAL[\\"qidx\\"][r])) in EVID for j in np.r_[t, tc])\\n",\n    "                   for r, t, tc in zip(_vrows, TOP_VAL, TOPC_VAL)])\\n",\n    "    vr, tv, tcv = _vrows[ok], TOP_VAL[ok], TOPC_VAL[ok]\\n",\n    "    VLM_VAL, VLMC_VAL = vlm_matrix(C_VAL, vr, tv, VAL_TEXT), vlm_matrix(C_VAL, vr, tcv, VAL_TEXT)\\n",\n    "    rr5, BETA = choose_beta(C_VAL, S1_VAL, vr, VLM_VAL, tv)\\n",\n    "    s0 = C_VAL[\\"clip\\"].astype(np.float64)                                 # giai đoạn 1 = chỉ CLIP (kiểu probe v4)\\n",\n    "    rr5b, BETA_CLIP = choose_beta(C_VAL, s0, vr, VLMC_VAL, tcv)\\n",\n    "    VAL_STAGE2 = dict(n=int(ok.sum()), M4=float((1 / ranks_of(sub(C_VAL, vr), S1_VAL[vr])).mean()), M5=float(rr5.mean()),\\n",\n    "                      M5b_clip_topM_vlm=float(rr5b.mean()), beta=BETA, beta_clip=BETA_CLIP, finished=bool(done),\\n",\n    "                      minutes=round((time.time() - t0) / 60, 1))\\n",\n    "    print(f\\"VAL giai đoạn 2 ({VAL_STAGE2[\'n\']} truy vấn): M4 {VAL_STAGE2[\'M4\']:.4f} → M5 {VAL_STAGE2[\'M5\']:.4f} (β={BETA}) | \\"\\n",\n    "          f\\"CLIP top-{VLM[\'M\']}+VLM {VAL_STAGE2[\'M5b_clip_topM_vlm\']:.4f} (β={BETA_CLIP}) | {VAL_STAGE2[\'minutes\']} phút\\")\\n",\n    "    provenance(\\"β (trọng số node VLM)\\", \\"selected\\", f\\"MRR 2-fold theo video trên {VAL_STAGE2[\'n\']} truy vấn VAL\\", caption_splits=(\\"val\\",))\\n",\n    "\\n",\n    "# ---------------- 3. ĐÓNG BĂNG ----------------\\n",\n    "FROZEN_V5 = dict(K=V5[\\"K\\"], hp=dict(hub_k=HP[0], bridge_m=HP[1], bridge_tau=HP[2]), dedup_source=V5[\\"dedup_source\\"],\\n",\n    "                 pe=f\\"{V5[\'pe_model\']}/{V5[\'pe_weights\']}\\", clip=f\\"{CFG.clip_model}/{CFG.clip_weights}\\", pooling=POOL_SEL,\\n",\n    "                 weights={k: [float(x) for x in w] for k, w in W_FINAL.items()}, feats={k: feats_of(k) for k in W_FINAL},\\n",\n    "                 vlm=dict(model=SCORER.name if VLM_OK else None, vis_fp32=SCORER.vis_fp32 if VLM_OK else None, M=VLM[\\"M\\"],\\n",\n    "                          beta=BETA, beta_clip=BETA_CLIP, prompt=PREFIX + holistic_q(\\"<query>\\") + SUFFIX, grid=VLM[\\"grid\\"]))\\n",\n    "TEST_GATE.freeze(FROZEN_V5)\\n",\n    "\\n",\n    "# ---------------- 4. TEST giai đoạn 1: MỌI caption test ----------------\\n",\n    "TEST_GATE.check(\\"eval: v5 stage-1 (mọi caption test)\\", \\"test\\")\\n",\n    "TEST_Q = np.where(np.isin(CAP_VID, SPLIT[\\"test\\"]))[0]\\n",\n    "CAPS.use(TEST_Q, \\"eval\\")\\n",\n    "C_TEST = candidate_set(TEST_Q, np.array(sorted(SPLIT[\\"test\\"])), V5[\\"K\\"])\\n",\n    "F_TEST = edge_features(C_TEST, HP)\\n",\n    "F_TEST_CTRL = {(hm, bs): edge_features(C_TEST, HP, hm, bs) for _, hm, bs in CONTROLS.values()}\\n",\n    "RR_TEST, RANK_TEST, S1_TEST = {}, {}, {}\\n",\n    "for name in list(VARIANTS) + list(CONTROLS):\\n",\n    "    F = F_TEST if name in VARIANTS else F_TEST_CTRL[CONTROLS[name][1:]]\\n",\n    "    S1_TEST[name] = stack(F, feats_of(name)) @ W_FINAL[name]\\n",\n    "    RANK_TEST[name] = ranks_of(C_TEST, S1_TEST[name])\\n",\n    "    RR_TEST[name] = 1.0 / RANK_TEST[name]\\n",\n    "T_ROWS = [dict(model=n, set=\\"test_all\\", **metrics_from_ranks(RANK_TEST[n], C_TEST[\\"gold\\"])) for n in RANK_TEST]\\n",\n    "print(f\\"TEST: {len(TEST_Q)} truy vấn / {len(C_TEST[\'gal\'])} video | CLIP R@{V5[\'K\']} = {(C_TEST[\'clip_rank\'] <= V5[\'K\']).mean():.3f}\\")\\n",\n    "\\n",\n    "# ---------------- 5. TEST giai đoạn 2: 1 caption / video test, VLM trên top-M ----------------\\n",\n    "_trows = np.sort(np.array([np.random.RandomState(VLM[\\"seed\\"] + 7).choice(np.where(C_TEST[\\"gold\\"] == v)[0]) for v in np.unique(C_TEST[\\"gold\\"])]))\\n",\n    "_trows = _trows[np.random.RandomState(VLM[\\"seed\\"] + 8).permutation(len(_trows))]   # thứ tự ngẫu nhiên ⇒ cắt theo ngân sách không thiên lệch\\n",\n    "TEST_TEXT = dict(zip(C_TEST[\\"qidx\\"][_trows].tolist(), CAPS.texts(C_TEST[\\"qidx\\"][_trows], \\"query_input\\")))\\n",\n    "TOP_TEST = topM(C_TEST, S1_TEST[\\"M4_full_graph\\"])[_trows]\\n",\n    "TOPC_TEST = np.tile(np.arange(VLM[\\"M\\"]), (len(_trows), 1))\\n",\n    "SUB_ROWS = np.zeros(0, int)\\n",\n    "if VLM_OK:\\n",\n    "    TEST_GATE.check(\\"eval: v5 stage-2 VLM (tập con test)\\", \\"test\\")\\n",\n    "    t0, done_rows = time.time(), []\\n",\n    "    for r, t, tc in zip(_trows, TOP_TEST, TOPC_TEST):\\n",\n    "        pr = [(int(C_TEST[\\"cand\\"][r, j]), int(C_TEST[\\"qidx\\"][r])) for j in np.r_[t, tc]]\\n",\n    "        if not score_pairs(pr, TEST_TEXT, VLM[\\"test_budget_min\\"] * 60, t0):\\n",\n    "            print(f\\"⏱ hết ngân sách test VLM sau {len(done_rows)} truy vấn\\")\\n",\n    "            break\\n",\n    "        done_rows.append(r)\\n",\n    "        if len(done_rows) % 100 == 0:\\n",\n    "            print(f\\"  {len(done_rows)}/{len(_trows)} truy vấn test | {SCORER.n_forward} lượt VLM | {(time.time() - t0) / 60:.1f} phút\\")\\n",\n    "    SUB_ROWS = np.array(sorted(done_rows), dtype=int)\\n",\n    "    order = np.argsort(_trows)\\n",\n    "    pos_of = {int(r): i for i, r in enumerate(_trows)}\\n",\n    "    tt = np.array([TOP_TEST[pos_of[int(r)]] for r in SUB_ROWS])\\n",\n    "    ttc = np.array([TOPC_TEST[pos_of[int(r)]] for r in SUB_ROWS])\\n",\n    "    VLM_T, VLMC_T = vlm_matrix(C_TEST, SUB_ROWS, tt, TEST_TEXT), vlm_matrix(C_TEST, SUB_ROWS, ttc, TEST_TEXT)\\n",\n    "    sh = np.random.RandomState(VLM[\\"seed\\"] + 9)\\n",\n    "    VLM_T_SHUF = np.array([sh.permutation(x) for x in VLM_T])               # đối chứng: bằng chứng VLM hoán vị giữa các ứng viên\\n",\n    "    sub_r = {n: RR_TEST[n][SUB_ROWS] for n in (\\"M0_clip\\", \\"M1_clip_pe_flat\\", \\"M4_full_graph\\", \\"G1_clip_graph\\")}\\n",\n    "    sub_r[\\"M5_graph_plus_vlm\\"] = 1.0 / stage2_rank(C_TEST, S1_TEST[\\"M4_full_graph\\"], SUB_ROWS, VLM_T, BETA, tt)\\n",\n    "    sub_r[\\"M5_ctrl_vlm_shuffled\\"] = 1.0 / stage2_rank(C_TEST, S1_TEST[\\"M4_full_graph\\"], SUB_ROWS, VLM_T_SHUF, BETA, tt)\\n",\n    "    sub_r[\\"M5b_clip_topM_vlm\\"] = 1.0 / stage2_rank(C_TEST, C_TEST[\\"clip\\"].astype(np.float64), SUB_ROWS, VLMC_T, BETA_CLIP, ttc)\\n",\n    "    for n, rr in sub_r.items():\\n",\n    "        T_ROWS.append(dict(model=n, set=\\"test_vlm_subset\\", **metrics_from_ranks(1.0 / rr, C_TEST[\\"gold\\"][SUB_ROWS])))\\n",\n    "METRICS_V5 = pd.DataFrame(T_ROWS)\\n",\n    "METRICS_V5.to_csv(RESULTS / \\"v5_test_metrics.csv\\", index=False)\\n",\n    "display(METRICS_V5[[\\"model\\", \\"set\\", \\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\", \\"n_queries\\"]].round(4))\\n",\n    "\\n",\n    "# ---------------- 6. so sánh đăng ký trước (bootstrap theo video, Holm) ----------------\\n",\n    "CMP = [(\\"T1_total_full_vs_clip\\", \\"M4_full_graph\\", \\"M0_clip\\", \\"test_all\\"),\\n",\n    "       (\\"T2_graph_on_clip_only\\", \\"G1_clip_graph\\", \\"M0_clip\\", \\"test_all\\"),\\n",\n    "       (\\"T3_structure_over_flat_clip_pe\\", \\"M4_full_graph\\", \\"M1_clip_pe_flat\\", \\"test_all\\"),\\n",\n    "       (\\"T4_structure_vs_controls\\", \\"M4_full_graph\\", \\"M4_ctrl_both\\", \\"test_all\\"),\\n",\n    "       (\\"T4b_clip_graph_vs_controls\\", \\"G1_clip_graph\\", \\"G1_ctrl_both\\", \\"test_all\\"),\\n",\n    "       (\\"T5_vlm_node_over_graph\\", \\"M5_graph_plus_vlm\\", \\"M4_full_graph\\", \\"test_vlm_subset\\"),\\n",\n    "       (\\"T6_full_vs_clip_subset\\", \\"M5_graph_plus_vlm\\", \\"M0_clip\\", \\"test_vlm_subset\\"),\\n",\n    "       (\\"T7_vlm_edges_vs_shuffled\\", \\"M5_graph_plus_vlm\\", \\"M5_ctrl_vlm_shuffled\\", \\"test_vlm_subset\\"),\\n",\n    "       (\\"T8_full_vs_clip_topM_vlm\\", \\"M5_graph_plus_vlm\\", \\"M5b_clip_topM_vlm\\", \\"test_vlm_subset\\")]\\n",\n    "crow = []\\n",\n    "for cid, a, b, st in CMP:\\n",\n    "    if st == \\"test_all\\":\\n",\n    "        ra, rb, gold = RR_TEST[a], RR_TEST[b], C_TEST[\\"gold\\"]\\n",\n    "    elif len(SUB_ROWS):\\n",\n    "        ra, rb, gold = sub_r[a], sub_r[b], C_TEST[\\"gold\\"][SUB_ROWS]\\n",\n    "    else:\\n",\n    "        continue\\n",\n    "    crow.append(dict(comparison=cid, a=a, b=b, set=st, n=len(ra), MRR_a=ra.mean(), MRR_b=rb.mean(), R1_a=(ra == 1).mean(), R1_b=(rb == 1).mean(),\\n",\n    "                     **cluster_bootstrap_diff(ra, rb, gold, n_boot=V5[\\"n_boot\\"], seed=1)))\\n",\n    "TESTS_V5 = pd.DataFrame(crow)\\n",\n    "if len(TESTS_V5):\\n",\n    "    TESTS_V5[\\"p_holm\\"] = holm(TESTS_V5[\\"p_not_better\\"].values)\\n",\n    "TESTS_V5.to_csv(RESULTS / \\"v5_test_comparisons.csv\\", index=False)\\n",\n    "display(TESTS_V5[[\\"comparison\\", \\"MRR_a\\", \\"MRR_b\\", \\"diff\\", \\"ci_lo\\", \\"ci_hi\\", \\"p_holm\\"]].round(4))\\n",\n    "\\n",\n    "# ---------------- 7. giải thích + fidelity theo đường cầu nối (xoá top-3 đường vs 3 đường ngẫu nhiên) ----------------\\n",\n    "def path_contribs(q, v, enc):\\n",\n    "    m, tau = HP[1], HP[2]\\n",\n    "    V, T = (VID_CL, TXT_CL) if enc == \\"clip\\" else (VID_PE, TXT_PE)\\n",\n    "    s = T[BANK] @ T[q]\\n",\n    "    idx = np.argsort(-s)[:m]\\n",\n    "    w = np.exp((s[idx] - s[idx].max()) / tau); w /= w.sum()\\n",\n    "    a = defaultdict(float)\\n",\n    "    for c, wc in zip(idx, w):\\n",\n    "        a[int(BANK_POS[c])] += float(wc)\\n",\n    "    out = {}\\n",\n    "    for u, au in a.items():\\n",\n    "        uv = TR_VIDS[u]\\n",\n    "        out[u] = 0.0 if (V5[\\"dedup_source\\"] and SRC[uv] == SRC[v]) else au * max(float(V[uv] @ V[v]), 0.0) ** 3\\n",\n    "    return out\\n",\n    "\\n",\n    "\\n",\n    "def contributions(r, j, name=\\"M4_full_graph\\"):\\n",\n    "    Z = stack(F_TEST, feats_of(name))[r]\\n",\n    "    return {f: float(w * Z[j, i]) for i, (f, w) in enumerate(zip(feats_of(name), W_FINAL[name]))}\\n",\n    "\\n",\n    "\\n",\n    "FID, EXAMPLES = [], []\\n",\n    "_frng = np.random.RandomState(VLM[\\"seed\\"] + 11)\\n",\n    "_pool = [r for r in (SUB_ROWS if len(SUB_ROWS) else _trows) if gold_pos(sub(C_TEST, np.array([r])))[0] >= 0][:60]\\n",\n    "for r in _pool:\\n",\n    "    j = int(gold_pos(sub(C_TEST, np.array([r])))[0])\\n",\n    "    q, v = int(C_TEST[\\"qidx\\"][r]), int(C_TEST[\\"cand\\"][r, j])\\n",\n    "    base_rank = int(RANK_TEST[\\"M4_full_graph\\"][r])\\n",\n    "    res = {}\\n",\n    "    for mode in (\\"top3\\", \\"random3\\"):\\n",\n    "        Fm = {f: F_TEST[f][r:r + 1].copy() for f in FEATS_ALL}\\n",\n    "        for enc in (\\"clip\\", \\"pe\\"):\\n",\n    "            pc = path_contribs(q, v, enc)\\n",\n    "            keys = sorted(pc, key=lambda u: -pc[u])[:3] if mode == \\"top3\\" else list(_frng.choice(list(pc), min(3, len(pc)), replace=False))\\n",\n    "            Fm[f\\"br_{enc}\\"][0, j] -= sum(pc[u] for u in keys)\\n",\n    "        s = stack(Fm, FEATS_ALL)[0] @ W_FINAL[\\"M4_full_graph\\"]\\n",\n    "        res[mode] = int(ranks_of(sub(C_TEST, np.array([r])), s[None])[0])\\n",\n    "    FID.append(dict(row=int(r), base_rank=base_rank, rank_del_top3=res[\\"top3\\"], rank_del_random3=res[\\"random3\\"]))\\n",\n    "FID_DF = pd.DataFrame(FID)\\n",\n    "if len(FID_DF):\\n",\n    "    FID_SUM = dict(n=len(FID_DF), worse_top3=float((FID_DF.rank_del_top3 > FID_DF.base_rank).mean()),\\n",\n    "                   worse_random3=float((FID_DF.rank_del_random3 > FID_DF.base_rank).mean()),\\n",\n    "                   mean_drank_top3=float((FID_DF.rank_del_top3 - FID_DF.base_rank).mean()),\\n",\n    "                   mean_drank_random3=float((FID_DF.rank_del_random3 - FID_DF.base_rank).mean()))\\n",\n    "else:\\n",\n    "    FID_SUM = {}\\n",\n    "FID_DF.to_csv(RESULTS / \\"v5_bridge_fidelity.csv\\", index=False)\\n",\n    "print(\\"Fidelity đường cầu nối (xoá 3 đường của video đúng):\\", FID_SUM)\\n",\n    "for r in _pool[:6]:\\n",\n    "    q = int(C_TEST[\\"qidx\\"][r]); j1 = int(np.argmax(S1_TEST[\\"M4_full_graph\\"][r])); v1 = int(C_TEST[\\"cand\\"][r, j1])\\n",\n    "    paths = []\\n",\n    "    for c, u, w in bridge_paths(q, v1, \\"pe\\", HP[1], HP[2]):\\n",\n    "        paths.append(dict(train_caption=CAPS.texts([c], \\"display\\")[0], train_video=REC_BY_VID[u][\\"video_id\\"], weight=round(w, 4)))\\n",\n    "    _, nb = hub_bias(C_TEST[\\"gal\\"], \\"pe\\", HP[0])\\n",\n    "    EXAMPLES.append(dict(query=TEST_TEXT.get(q) or CAPS.texts([q], \\"query_input\\")[0], clip_rank=int(C_TEST[\\"clip_rank\\"][r]),\\n",\n    "                         graph_rank=int(RANK_TEST[\\"M4_full_graph\\"][r]), top1_video=REC_BY_VID[v1][\\"video_id\\"],\\n",\n    "                         top1_is_gold=v1 == int(C_TEST[\\"gold\\"][r]), top1_edge_contributions={k: round(x, 3) for k, x in contributions(r, j1).items()},\\n",\n    "                         top1_bridge_paths=paths, top1_hub_neighbours=CAPS.texts(nb.get(v1, [])[:3], \\"display\\") if v1 in nb else []))\\n",\n    "save_json(EXAMPLES, RESULTS / \\"v5_explanations.json\\")\\n",\n    "for ex in EXAMPLES[:3]:\\n",\n    "    print(f\\"\\\\n«{ex[\'query\']}» — hạng CLIP {ex[\'clip_rank\']} → hạng đồ thị {ex[\'graph_rank\']} | top-1 {\'ĐÚNG\' if ex[\'top1_is_gold\'] else \'sai\'}\\")\\n",\n    "    print(\\"   cạnh:\\", ex[\\"top1_edge_contributions\\"])\\n",\n    "    for p in ex[\\"top1_bridge_paths\\"]:\\n",\n    "        print(f\\"   đường: «{p[\'train_caption\']}» → video train {p[\'train_video\']} → ứng viên (w={p[\'weight\']})\\")\\n",\n    "\\n",\n    "V5_SUMMARY = dict(frozen_sha=TEST_GATE.frozen_sha, hp=FROZEN_V5[\\"hp\\"], val_cv=VAL_TABLE.to_dict(\\"records\\"), val_stage2=VAL_STAGE2,\\n",\n    "                  test=METRICS_V5.to_dict(\\"records\\"), comparisons=TESTS_V5.to_dict(\\"records\\"), fidelity=FID_SUM,\\n",\n    "                  vlm=dict(model=FROZEN_V5[\\"vlm\\"][\\"model\\"], forwards=SCORER.n_forward if VLM_OK else 0,\\n",\n    "                           s_per_forward=(SCORER.seconds / max(1, SCORER.n_forward)) if VLM_OK else None, sanity=SANITY_LOG,\\n",\n    "                           test_subset_scored=int(len(SUB_ROWS)), test_subset_planned=int(len(_trows))),\\n",\n    "                  test_looks_msvd=\\"lần thứ 8 ⇒ THĂM DÒ\\", n_shared_sources=N_SHARED)\\n",\n    "save_json(V5_SUMMARY, RESULTS / \\"v5_summary.json\\")\\n",\n    "save_json([c for c in CHECKS if c[\\"id\\"][:1] in (\\"E\\", \\"V\\")], RESULTS / \\"v5_checks.json\\")\\n",\n    "CAPS.access_table().to_csv(RESULTS / \\"caption_access_log.csv\\", index=False)\\n",\n    "if VLM_OK:\\n",\n    "    SCORER.close()\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "c062",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# ============================== GÓI KẾT QUẢ ĐỂ TẢI VỀ (không gồm dữ liệu) ==============================\\n",\n    "mark_stage(\\"pack\\", requires=(\\"vlm_test\\",))\\n",\n    "_z = pack_review(\\"hoàn tất\\")\\n",\n    "assert _z is not None, \\"Không tạo được file zip xem xét — xem cảnh báo phía trên.\\"\\n",\n    "with zipfile.ZipFile(_z) as _zf:\\n",\n    "    _names = _zf.namelist()\\n",\n    "    _man = json.loads(_zf.read(f\\"{_z.stem}/pack_manifest.json\\"))\\n",\n    "print(f\\"📦 {_z} — {_z.stat().st_size / 2**20:.1f} MB, {len(_names)} file. Tải file này về (Kaggle: Output → {_z.name}).\\")\\n",\n    "print(\\"   Notebook đã chạy:\\", \\"có (executed_notebook.ipynb, đủ output tới cell trước)\\" if _man[\\"executed_notebook\\"]\\n",\n    "      else \\"KHÔNG tìm thấy (chạy ngoài Kaggle/papermill) — tải notebook riêng nếu cần\\")\\n",\n    "print(\\"   Không gói:\\", \\"; \\".join(f\\"{k} {v[\'n_files\']} file, {v[\'mb\']} MB\\" if isinstance(v, dict) else k for k, v in _man[\\"excluded\\"].items()))"\n   ]\n  }\n ],\n "metadata": {\n  "kernelspec": {\n   "display_name": "Python 3",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python"\n  },\n  "level4b_build": {\n   "source": "v4_qc_rgat_graph.ipynb",\n   "source_sha1": "8ba65a19a414"\n  },\n  "level4b_v2_build": {\n   "source": "Project03_Level4B_TextToVideo.ipynb",\n   "source_sha1": "d35da75ec6cd"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}'
VIDEO_NB_MD5 = '4b47afd53d61cfd962332486b73dbf8b'   # notebook video Level4B v5, nhúng nguyên văn
REF = {"exp1_MRR": {"CLIP": 0.7182186436896805, "G-full": 0.7211552931663113, "G-obj": 0.721908345890363, "G-rand": 0.7174007442577174, "G-shuf": 0.7199568882232281}, "exp2_acc": {"ARO_all": {"CLIP": 0.5014062834184786, "graph_only[dg]": 0.6584979616420165, "fusion[dg]": 0.6602698707119439}, "ARO_geo": {"CLIP": 0.4971140070084001, "graph_only[dg]": 0.6627371946710509, "fusion[dg]": 0.6672055162800242}, "ARO_pred": {"CLIP": 0.50715660925085, "graph_only[dg]": 0.6633357774719031, "fusion[dg]": 0.6520314221884903}, "SC_replace_rel": {"CLIP": 0.6987984939709078, "graph_only[dg]": 0.5444779486158796, "fusion[dg]": 0.7032363039259591}, "SC_swap_obj": {"CLIP": 0.6352534703112579, "graph_only[dg]": 0.4965541490857947, "fusion[dg]": 0.6132727939827555}}, "xai": {"all": {"graph_acc": 0.8569131832797428, "fidelity_del_changed": 0.1111111111111111, "validity_both_iou50": 0.5492424242424242, "counterfactual_graph_reversed": 0.8813559322033898}, "geo": {"graph_acc": 0.8411016949152542, "fidelity_del_changed": 0.125, "validity_both_iou50": 0.5063613231552163, "counterfactual_graph_reversed": 0.8813559322033898}, "pred": {"graph_acc": 0.9066666666666666, "fidelity_del_changed": 0.06756756756756757, "validity_both_iou50": 0.674074074074074}}}   # số của các lần chạy riêng trước (để đối chiếu tái lập)
print(f'Nhúng: {len(VAL_IDS)} id VAL, {len(G0_IDS)} id G0, {len(V3_BENCH_IDS)} id benchmark v3')


## 3. Lõi đồ thị
Scene graph là đồ thị có hướng: đỉnh = đối tượng (lớp VG150), cạnh = vị từ (và cạnh hình học suy từ box). Điểm đồ thị của một ứng viên là độ khớp của các **đường đi** từ truy vấn tới ảnh; giải thích chính là các cạnh / đỉnh đã khớp, nên giải thích đến từ cơ chế xếp hạng.

In [ ]:
# ============================== 3. LÕI ĐỒ THỊ: chuẩn hoá · scene graph · tách caption · điểm đồ thị · kết hợp · thống kê ==============================
# Hàm thuần (không đọc file, không mô hình). Gộp nguyên văn logic của g0b_core (phần dùng chung), v5_core và v6_core; chỉ đổi tên hàm trùng.

# ---- 3.1 Bảng VG150 + quan hệ vị từ (tương đương / kéo theo) ----
CLS_IDX = {c: i for i, c in enumerate(TABLES["VG_CLASSES"]) if i > 0}        # chỉ số = cột xác suất lớp của RelTR (cột 0 = 'N/A')
PRED_IDX = {p: i for i, p in enumerate(TABLES["VG_PREDS"]) if i > 0}         # chỉ số = cột xác suất vị từ của RelTR
CANON = {"wears": "wearing", "laying on": "lying on", "watching": "looking at"}   # vị từ tương đương
ENT_RAW = {"on": ["sitting on", "standing on", "lying on", "laying on", "walking on", "parked on", "mounted on", "riding"],
           "in": ["walking in"], "holding": ["carrying"]}                        # vị từ kéo theo: ảnh có p' ⇒ thoả truy vấn p
canon = lambda p: CANON.get(p, p)
ENT = {canon(p): sorted({canon(q) for q in L}) for p, L in ENT_RAW.items()}
FAM = {p: {p} | set(ENT.get(p, [])) for p in {canon(p) for p in PRED_IDX}}


def norm_name(n):
    """tên đối tượng → lớp VG150 (số nhiều đơn giản: shoes → shoe), như v3."""
    n = " ".join(str(n).lower().strip().split())
    if n in CLS_IDX: return n
    for cut in (1, 2):
        if n.endswith("s") and n[:-cut] in CLS_IDX: return n[:-cut]
    return None


def norm_pred(p):
    p = " ".join(str(p).lower().strip().split())
    return canon(p) if p in PRED_IDX else None


def nname(d):
    if d.get("name") is not None: return norm_name(d["name"])
    for n in d.get("names") or []:
        r = norm_name(n)
        if r: return r
    return None


# ---- 3.2 Split (cố định bằng sha1, tái lập được) ----
def dev_g0_split(i):          # pool G0: DEV (35%) / IV khoá (65%)
    return "DEV" if int(hashlib.sha1(("p3-dir2:" + str(i)).encode()).hexdigest(), 16) % 1000 < 350 else "IV"


def p5_split(i):              # Thí nghiệm 1: chia DEV-G0 làm đôi
    return "P-TUNE" if int(hashlib.sha1(("p5-pilot:" + str(i)).encode()).hexdigest(), 16) % 2 == 0 else "P-CHECK"


def v6_split(key, s):         # Thí nghiệm 2: chia theo ảnh, 3 seed
    return "TUNE" if int(hashlib.sha1(f"v6-{s}:{key}".encode()).hexdigest(), 16) % 2 == 0 else "CHECK"


class IdGuard:
    """Hàng rào rò rỉ: chỉ id thuộc tập cho phép mới được đọc / nạp ảnh."""

    def __init__(self, allowed, name):
        self.allowed, self.name, self.n_checked = {int(i) for i in allowed}, name, 0

    def check(self, ids, purpose=""):
        ids = [int(i) for i in ids]
        bad = [i for i in ids if i not in self.allowed]
        if bad: raise PermissionError(f"GUARD[{self.name}]: {len(bad)} id ngoài tập cho phép ({purpose}): {bad[:5]}")
        self.n_checked += len(ids)


# ---- 3.3 Scene graph GT của một ảnh (đỉnh = lớp, cạnh = bộ ba VG150) ----
class Img:
    __slots__ = ("id", "triples", "between", "classes")

    def __init__(s, i):
        s.id = i; s.triples = set(); s.between = defaultdict(list); s.classes = set()


def build_image(i, rel_entries, obj_entries, stats):
    """Dựng scene graph từ annotation VG: bỏ quan hệ trùng, self-loop, thiếu object_id; chuẩn hoá về VG150."""
    im = Img(i)
    for e in obj_entries:
        for o in e.get("objects", []):
            c = nname(o)
            if c: im.classes.add(c)
    seen_rid, seen_key = set(), set()
    for rec in rel_entries:
        for r in rec.get("relationships", []):
            stats["rel_total"] += 1
            rid = r.get("relationship_id")
            if rid in seen_rid: stats["rel_dup_relationship_id"] += 1; continue
            seen_rid.add(rid)
            su, ob = r.get("subject") or {}, r.get("object") or {}
            sid, oid = su.get("object_id", su.get("id")), ob.get("object_id", ob.get("id"))
            if sid is None or oid is None: stats["rel_missing_object_id"] += 1; continue
            if int(sid) == int(oid): stats["rel_self_loop"] += 1; continue
            p_raw = " ".join(str(r.get("predicate", "")).lower().split())
            key = (int(sid), p_raw, int(oid))
            if key in seen_key: stats["rel_duplicate_same_triple_ids"] += 1; continue
            seen_key.add(key)
            s150, o150 = nname(su), nname(ob)
            if s150 and o150:
                im.classes.add(s150); im.classes.add(o150)
                im.between[(s150, o150)].append(p_raw)
                p150 = norm_pred(p_raw)
                if p150: im.triples.add((s150, p150, o150)); stats["edges_vg150"] += 1
    return im


# ---- 3.4 Tách caption thành thực thể + bộ ba (luật, tất định) ----
PRED_PHRASES = {"riding": "riding", "rides": "riding", "ride": "riding", "holding": "holding", "holds": "holding",
                "sitting on": "sitting on", "sits on": "sitting on", "standing on": "standing on", "stands on": "standing on",
                "wearing": "wearing", "wears": "wearing", "next to": "near", "beside": "near", "near": "near",
                "on top of": "on", "on": "on", "in": "in", "inside": "in", "in front of": "in front of", "behind": "behind",
                "under": "under", "underneath": "under", "with": "with", "has": "has", "have": "has", "eating": "eating", "eats": "eating",
                "playing": "playing", "plays": "playing", "looking at": "looking at", "looks at": "looking at",
                "carrying": "carrying", "carries": "carrying", "walking on": "walking on", "walking in": "walking in",
                "laying on": "lying on", "lying on": "lying on", "parked on": "parked on", "hanging from": "hanging from",
                "flying in": "flying in", "covered in": "covered in", "above": "above", "over": "above"}


def _spans(tokens, lookup, maxn=3):
    out, i = [], 0
    while i < len(tokens):
        hit = None
        for n in range(min(maxn, len(tokens) - i), 0, -1):
            v = lookup(" ".join(tokens[i:i + n]))
            if v is not None: hit = (i, i + n, v); break
        if hit: out.append(hit); i = hit[1]
        else: i += 1
    return out


def parse_caption(cap):
    """→ (E: tập lớp, T: tập bộ ba). subject = thực thể gần nhất bên trái vị từ, object = gần nhất bên phải."""
    toks = re.findall(r"[a-z0-9]+", str(cap).lower())
    ents, preds = _spans(toks, norm_name), _spans(toks, PRED_PHRASES.get)
    T = set()
    for (a, b, p) in preds:
        left, right = [e for e in ents if e[1] <= a], [e for e in ents if e[0] >= b]
        if left and right:
            T.add((max(left, key=lambda e: e[1])[2], p, min(right, key=lambda e: e[0])[2]))
    return frozenset(e[2] for e in ents), frozenset(T)


# ---- 3.5 Điểm đồ thị trên scene graph GT (Thí nghiệm 1): ba kiểu đường đi truy vấn → ảnh ----
def match_detail(Tq, Eq, im):
    """s_rel: bộ ba khớp cạnh (cho phép kéo theo) · s_pair: cặp lớp nối bởi cạnh bất kỳ · s_obj: lớp có mặt. Kèm bằng chứng (giải thích)."""
    edges, pairs = [], []
    for (s, p, o) in sorted(Tq):
        for q in sorted(FAM.get(p, {p})):
            if (s, q, o) in im.triples: edges.append(f"{s}|{p}|{o}<={s}|{q}|{o}"); break
        if (s, o) in im.between or (o, s) in im.between: pairs.append(f"{s}|{o}")
    nodes = sorted(c for c in Eq if c in im.classes)
    nt = len(Tq)
    return dict(s_rel=len(edges) / nt if nt else 0.0, s_pair=len(pairs) / nt if nt else 0.0,
                s_obj=len(nodes) / len(Eq) if Eq else 0.0, edges=edges, pairs=pairs, nodes=nodes)


def candidate_components(parsed, topk, graphs):
    Q, K = topk.shape
    out = np.zeros((Q, K, 3))
    for qi, (Eq, Tq) in enumerate(parsed):
        if not Eq and not Tq: continue
        for kj in range(K):
            d = match_detail(Tq, Eq, graphs[topk[qi, kj]])
            out[qi, kj] = (d["s_rel"], d["s_pair"], d["s_obj"])
    return out


def rand_permutation(n, seed=SEED_RAND):
    """hoán vị cố định không có điểm bất động (đối chứng 'đồ thị của ảnh khác')."""
    order = list(range(n)); random.Random(seed).shuffle(order)
    perm = [0] * n
    for pos, j in enumerate(order): perm[j] = order[(pos + 1) % n]
    return perm


def shuffle_predicates(im):
    e = sorted(im.triples); preds = [p for (_, p, _) in e]
    random.Random(int(im.id)).shuffle(preds)
    new = Img(im.id); new.triples = {(s, p, o) for (s, _, o), p in zip(e, preds)}; new.between = im.between; new.classes = im.classes
    return new


# ---- 3.6 Kết hợp CLIP + đồ thị và xếp hạng (Thí nghiệm 1) ----
def zrows(x):
    mu, sd = x.mean(1, keepdims=True), x.std(1, keepdims=True)
    return np.where(sd > 0, (x - mu) / np.where(sd > 0, sd, 1), 0.0)


def fuse_scores(zc, comps, w, alpha):
    g = comps @ np.asarray(w, dtype=np.float64)
    flat = g.max(1) == g.min(1)                  # điểm đồ thị bằng nhau ⇒ giữ thứ tự CLIP
    zg = zrows(g); zg[flat] = 0.0
    return alpha * zc + (1 - alpha) * zg


def rank_of_relevant(scores, rel_pos, clip_rank):
    Q, K = scores.shape
    pos = np.arange(K)[None, :]; rp = np.where(rel_pos >= 0, rel_pos, 0)
    sr = scores[np.arange(Q), rp][:, None]
    better = (scores > sr).sum(1) + ((scores == sr) & (pos < rp[:, None])).sum(1)
    return np.where(rel_pos >= 0, 1 + better, clip_rank).astype(np.int64)


def select_params(grid_mrr):
    return min(grid_mrr, key=lambda k: (-round(grid_mrr[k], 12), -k[1], k[0]))


def tune_grid(zc, comps, rel_pos, clip_rank, w_list, alphas):
    return {(tuple(w), a): float(np.mean(1.0 / rank_of_relevant(fuse_scores(zc, comps, w, a), rel_pos, clip_rank)))
            for w in w_list for a in alphas}


def clip_topk(cos, k):
    Q, N = cos.shape; K = min(k, N)
    order = np.argsort(-cos, axis=1, kind="stable"); topk = order[:, :K]
    pos_rel = np.array([int(np.where(order[i] == i)[0][0]) for i in range(Q)])
    return topk, np.take_along_axis(cos, topk, 1), np.where(pos_rel < K, pos_rel, -1), pos_rel + 1


def metrics_from_ranks(r):
    r = np.asarray(r, dtype=np.float64)
    return dict(R1=float((r <= 1).mean()), R5=float((r <= 5).mean()), R10=float((r <= 10).mean()), MRR=float((1 / r).mean()),
                nDCG10=float(np.where(r <= 10, 1 / np.log2(r + 1), 0.0).mean()))


def bootstrap_delta(d_by_name, n_boot=N_BOOT, seed=SEED_RAND, chunk=1000):
    names = list(d_by_name); n = len(d_by_name[names[0]])
    rng = np.random.default_rng(seed); means = {k: [] for k in names}; done = 0
    while done < n_boot:
        b = min(chunk, n_boot - done); idx = rng.integers(0, n, size=(b, n))
        for k in names: means[k].append(np.asarray(d_by_name[k])[idx].mean(1))
        done += b
    return {k: dict(mean=float(np.mean(d_by_name[k])), lo=float(np.percentile(np.concatenate(means[k]), 2.5)),
                    hi=float(np.percentile(np.concatenate(means[k]), 97.5))) for k in names}


def mean_std(xs):
    a = np.asarray(xs, dtype=np.float64)
    return dict(mean=float(a.mean()), std=float(a.std(ddof=1)) if len(a) > 1 else 0.0, per_seed=[float(x) for x in a])


# ---- 3.7 Điểm đồ thị trên scene graph RelTR (Thí nghiệm 2, 3): cạnh vị từ + cạnh hình học ----
ARO_RX = re.compile(r"^the (.+?) is (.+) the (.+)$")
GEO = {"to the left of": "L", "to the right of": "R"}
ARO_PRED = {"on": "on", "on top of": "on", "wearing": "wearing", "in": "in", "inside": "in", "behind": "behind",
            "in front of": "in front of", "near": "near", "next to": "near", "beside": "near", "by": "near", "of": "of",
            "sitting on": "sitting on", "above": "above", "below": "under", "under": "under", "underneath": "under",
            "holding": "holding", "at": "at", "with": "with", "riding": "riding", "standing on": "standing on", "eating": "eating",
            "walking on": "walking on", "parked on": "parked on", "watching": "looking at", "looking at": "looking at",
            "lying on": "lying on", "walking in": "walking in", "covering": "covering"}


def aro_parse(cap):
    m = ARO_RX.match(cap)
    if not m: return (None, "other", None, None)
    X, r, Y = norm_name(m.group(1)), m.group(2), norm_name(m.group(3))
    if not X or not Y: return (None, "other", r, None)
    if r in GEO: return (X, "geo", GEO[r], Y)
    if r in ARO_PRED: return (X, "pred", ARO_PRED[r], Y)
    return (None, "other", r, None)


def aro_subset(rel_name):
    return "geo" if rel_name in GEO else ("pred" if rel_name in ARO_PRED else "other")


def rel_score(F, X, p, Y, pr=None):
    """cạnh vị từ: log max_j P(sub=X) · P(pred=p) · P(obj=Y) trên 200 truy vấn bộ ba của RelTR (điểm 'rel' của v3)."""
    pr = F["pr"] if pr is None else pr
    v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * pr[:, PRED_IDX[p]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
    j = int(np.argmax(v))
    return float(np.log(max(v[j], FLOOR))), dict(kind="pred", j=j, p=float(v[j]), sub_box=[float(x) for x in F["sb"][j]],
                                                 obj_box=[float(x) for x in F["ob"][j]])


def geo_score(F, X, Y, d, nogeo=False):
    """cạnh hình học: max_{u≠v} P(u=X) · P(v=Y) · σ((cx_v − cx_u)/τ) trên 400 hộp đối tượng (subject + object của 200 truy vấn)."""
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, cx = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0]
    if nogeo: s = np.ones((len(cx), len(cx)))
    else:
        diff = (cx[None, :] - cx[:, None]) if d == "L" else (cx[:, None] - cx[None, :])
        s = 1.0 / (1.0 + np.exp(-diff / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]),
                                                    x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


def aro_graph_score(F, parsed, pr=None, nogeo=False):
    X, kind, r, Y = parsed
    if kind == "geo": return geo_score(F, X, Y, r, nogeo=nogeo)
    if kind == "pred": return rel_score(F, X, r, Y, pr=pr)
    return None, None


def triples_graph_score(F, T, pr=None):
    if not T: return None, None
    vals, ev = [], []
    for (s, p, o) in sorted(T):
        if p not in PRED_IDX: return None, None
        v, e = rel_score(F, s, p, o, pr=pr); vals.append(v); ev.append(dict(triple=f"{s}|{p}|{o}", **e))
    return float(np.mean(vals)), ev


def shuffled_pr(F, key):
    perm = np.arange(F["pr"].shape[1])
    perm[1:] = 1 + np.random.RandomState(int(key) % (2 ** 31)).permutation(F["pr"].shape[1] - 1)
    return F["pr"][:, perm]


def correct(delta):
    d = np.asarray(delta, dtype=np.float64)
    return np.where(d > 0, 1.0, np.where(d == 0, 0.5, 0.0))


def fuse2(dc, dg, valid, alpha, sc, sg):
    dc, dg = np.asarray(dc, float), np.asarray(dg, float)
    g = np.where(valid, dg, 0.0) / sg if sg > 0 else np.zeros_like(dg)
    return np.where(valid, alpha * dc / sc + (1 - alpha) * g, dc / sc)


def tune_alpha(dc, dg, valid):
    sc = float(np.std(dc)) or 1.0
    sg = (float(np.std(np.asarray(dg)[valid])) if np.any(valid) else 1.0) or 1.0
    acc = {a: float(correct(fuse2(dc, dg, valid, a, sc, sg)).mean()) for a in ALPHAS}
    return min(acc, key=lambda a: (-round(acc[a], 12), -a)), sc, sg, acc


def bootstrap_diff_by_cluster(x, y, clusters, n_boot=N_BOOT, seed=SEED_RAND):
    x, y = np.asarray(x, float), np.asarray(y, float)
    keys, inv = np.unique(np.asarray(clusters).astype(str), return_inverse=True); K = len(keys)
    sx, sy, cnt = np.bincount(inv, x, K), np.bincount(inv, y, K), np.bincount(inv, minlength=K).astype(float)
    rng = np.random.default_rng(seed); out = np.empty(n_boot)
    for b0 in range(0, n_boot, 1000):
        b = min(1000, n_boot - b0); idx = rng.integers(0, K, size=(b, K))
        out[b0:b0 + b] = (sx[idx].sum(1) - sy[idx].sum(1)) / cnt[idx].sum(1)
    return dict(mean=float(x.mean() - y.mean()), lo=float(np.percentile(out, 2.5)), hi=float(np.percentile(out, 97.5)))


print(f"Lõi đồ thị sẵn sàng: {len(CLS_IDX)} lớp, {len(PRED_IDX)} vị từ VG150 | {elapsed()}")


## 4. Dữ liệu và luật chống rò rỉ

In [ ]:
# ============================== 4. DỮ LIỆU: tìm input · tải benchmark · lọc chống rò rỉ · scene graph GT · caption COCO ==============================
# Luật rò rỉ (khoá trước):
#  - Mọi ảnh Visual Genome dùng tới phải thuộc VAL của v3 (chỉ số lọc ≥ 83000 ⇒ RelTR chưa huấn luyện trên chúng; heldout KHÔNG dùng).
#  - TN1 chỉ dùng DEV-G0: VAL trừ ảnh benchmark v3, cache v4 và 99 ảnh Phase 5 (danh sách G0, kiểm MD5).
#  - SugarCrepe: ảnh COCO không thuộc VG, hoặc thuộc VG và nằm trong VAL.
assert hashlib.md5(",".join(map(str, VAL_IDS)).encode()).hexdigest() == VAL_MD5 and len(VAL_IDS) == 15128
assert hashlib.md5(",".join(map(str, G0_IDS)).encode()).hexdigest() == G0_MD5 and len(G0_IDS) == 13586
VALSET = set(VAL_IDS)
DEV_G0 = sorted(i for i in G0_IDS if dev_g0_split(i) == "DEV")
assert len(DEV_G0) == 4752 and set(DEV_G0) <= VALSET
GUARD_VAL = IdGuard(VAL_IDS, "VAL")
GUARD_DEV = IdGuard(DEV_G0, "DEV-G0")
p3_log(f"VAL {len(VAL_IDS)} ảnh | DEV-G0 {len(DEV_G0)} ảnh (TN1)")


def find_inputs():
    if os.environ.get("P3_PATHS"): return json.loads(os.environ["P3_PATHS"])
    found, vgdirs, cocoval = defaultdict(list), [], []
    for dp, dn, fn in os.walk("/kaggle/input", followlinks=True):
        b = os.path.basename(dp)
        if b.startswith("VG_100K"):
            if "dannywu375" in dp or "/input/visualgenome/" in dp: vgdirs.append(dp)
            dn[:] = []; continue
        if b == "val2017" and any(f.endswith(".jpg") for f in fn[:50]):
            cocoval.append(dp); dn[:] = []; continue
        if b in ("train2017", "test2017", "unlabeled2017"):
            dn[:] = []; continue
        for f in fn:
            if f in ("image_data.json", "relationships.json", "objects.json", "captions_train2017.json", "captions_val2017.json"):
                found[f].append(os.path.join(dp, f))
    # merged: Kaggle có thể gắn dataset ở /kaggle/input/<owner>/<slug>/ hoặc /kaggle/input/<slug>/ (không có tên chủ sở hữu)
    pick = lambda name, slug: sorted(p for p in found[name] if f"/{slug}/" in p or f"/input/{slug.split('/')[-1]}/" in p)
    r = dict(image_data=pick("image_data.json", "dannywu375/visualgenome"), rel=pick("relationships.json", "mathurinache/visual-genome"),
             obj_dw=pick("objects.json", "dannywu375/visualgenome"), cap_train=sorted(found["captions_train2017.json"]),
             cap_val=sorted(found["captions_val2017.json"]))
    for k, v in r.items(): assert v, f"thiếu input {k} — đã thấy: { {kk: vv[:3] for kk, vv in found.items()} }"
    r = {k: v[0] for k, v in r.items()}
    r.update(vg_dirs=sorted(vgdirs), coco_val=sorted(cocoval)[0])
    return r


P = find_inputs()
SHA = {k: sha256(P[k]) for k in ("image_data", "rel", "obj_dw")}
if not SMOKE:
    for k, v in G0_SHA.items(): assert SHA[k] == v, f"input {k} khác manifest G0"
p3_log("Input VG khớp SHA-256 của G0" if not SMOKE else "SMOKE: bỏ kiểm SHA", "|", {k: P[k] for k in ("image_data", "rel", "obj_dw")})

# ---- benchmark công khai (tải lúc chạy) ----
if SMOKE:
    BENCH = json.loads(os.environ["P3_BENCH"])
else:
    BENCH = dict(aro=fetch(f"{WORK}/visual_genome_relation.json", gdrive=ARO_GDRIVE),
                 replace_rel=fetch(f"{WORK}/replace_rel.json", url=SC_URL + "replace_rel.json"),
                 swap_obj=fetch(f"{WORK}/swap_obj.json", url=SC_URL + "swap_obj.json"))
BSHA = {k: sha256(v) for k, v in BENCH.items()}
p3_log("Benchmark:", BSHA)

# ---- image_data: ánh xạ COCO ↔ VG; chỉ mục file ảnh VG thuộc VAL ----
COCO_OF_VG, VG_OF_COCO = {}, {}
for e in json.load(open(P["image_data"], encoding="utf-8")):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if e.get("coco_id") is not None:
        VG_OF_COCO[int(e["coco_id"])] = i
        if i in VALSET: COCO_OF_VG[i] = int(e["coco_id"])
VG_JPG = {}
for d in P["vg_dirs"]:
    for f in os.listdir(d):
        if f.endswith(".jpg") and f[:-4].isdigit() and int(f[:-4]) in VALSET: VG_JPG.setdefault(int(f[:-4]), os.path.join(d, f))
p3_log(f"Ảnh VG thuộc VAL có file: {len(VG_JPG)}")

# ---- mục benchmark, lọc rò rỉ TRƯỚC khi mở ảnh ----
ITEMS2, EXCL2 = [], Counter()
for k, x in enumerate(json.load(open(BENCH["aro"], encoding="utf-8"))):
    vid = int(x["image_id"])
    if vid not in VALSET: EXCL2["aro_not_VAL"] += 1; continue
    if vid not in VG_JPG: EXCL2["aro_no_jpg"] += 1; continue
    box = (int(x["bbox_x"]), int(x["bbox_y"]), int(x["bbox_x"]) + int(x["bbox_w"]), int(x["bbox_y"]) + int(x["bbox_h"]))
    ITEMS2.append(dict(bench="aro", item=f"aro{k}", key=str(vid), vg_id=vid, path=VG_JPG[vid], crop=box, true=x["true_caption"],
                       false=x["false_caption"], subset=aro_subset(x["relation_name"]), rel=x["relation_name"],
                       xid=int(x["primary_object_id"]), yid=int(x["relation_info"]["object"])))
for b in ("replace_rel", "swap_obj"):
    for k, x in json.load(open(BENCH[b], encoding="utf-8")).items():
        coco = int(x["filename"].split(".")[0]); vid = VG_OF_COCO.get(coco)
        if vid is not None and vid not in VALSET: EXCL2[f"{b}_vg_not_VAL"] += 1; continue
        path = os.path.join(P["coco_val"], x["filename"])
        if not os.path.exists(path): EXCL2[f"{b}_no_jpg"] += 1; continue
        ITEMS2.append(dict(bench=b, item=f"{b}{k}", key=f"coco{coco}", vg_id=vid, path=path, crop=None, true=x["caption"],
                           false=x["negative_caption"], subset="vg_VAL" if vid else "not_in_VG", rel="", xid=None, yid=None))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2 if it["vg_id"] is not None], "benchmark items")
p3_log("Mục benchmark:", dict(Counter(i["bench"] for i in ITEMS2)), "| loại:", dict(EXCL2))

# ---- annotation VG: chỉ cho ảnh cần (DEV-G0 cho TN1; ảnh ARO cho box GT của XAI) ----
NEED_REL = set(DEV_G0)
NEED_OBJ = set(DEV_G0) | {it["vg_id"] for it in ITEMS2 if it["bench"] == "aro"}
GUARD_VAL.check(NEED_OBJ, "annotation")
REL, OBJ = defaultdict(list), defaultdict(list)
for e in stream_array(P["rel"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_REL: REL[i].append(e)                 # mọi ảnh khác bị bỏ ngay, không lưu, không đếm
for e in stream_array(P["obj_dw"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_OBJ: OBJ[i].append(e)
GTBOX = {int(o["object_id"]): (int(o["x"]), int(o["y"]), int(o["x"]) + int(o["w"]), int(o["y"]) + int(o["h"]))
         for i in NEED_OBJ for e in OBJ.get(i, []) for o in e.get("objects", [])}
p3_log(f"Annotation: relationships {len(REL)} ảnh, objects {len(OBJ)} ảnh, {len(GTBOX)} hộp GT | {elapsed()}")

# ---- caption COCO cho TN1 ----
CAPS = defaultdict(list)
for k in ("cap_train", "cap_val"):
    for a in json.load(open(P[k], encoding="utf-8"))["annotations"]:
        CAPS[int(a["image_id"])].append((int(a["id"]), " ".join(str(a["caption"]).split())))
for v in CAPS.values(): v.sort()
EXCL1 = Counter()
ELIG1 = []
for i in DEV_G0:
    if i not in COCO_OF_VG: continue
    if len(CAPS.get(COCO_OF_VG[i], [])) < 3: EXCL1["lt3_captions"] += 1; continue
    if i not in VG_JPG: EXCL1["no_jpg"] += 1; continue
    ELIG1.append(i)
GUARD_DEV.check(ELIG1, "TN1 ảnh")
p3_log(f"TN1: {len(ELIG1)} ảnh DEV-G0 có caption COCO | loại {dict(EXCL1)} | {elapsed()}")


## 4v. Cấp 4 (b) — Mở rộng video MSVD (text-to-video)
Chạy nguyên vẹn notebook video Level4B v5 "EvidenceGraph" (đã chạy trên Kaggle, mã trùng khớp) trong tiến trình riêng, **trước** phần ảnh để GPU còn trống cho Qwen2.5-VL. Kết quả được đưa vào bảng tổng hợp ở mục 11. Khoảng 2–2,5 giờ.

## 4u. Phần C — Level 4(b) v2-graphfix (text → video, MSVD + MSR-VTT)
Notebook `lamvo30ai/project03-level4b-v2-graphfix` (version 1) được **nhúng nguyên văn** và chạy trong tiến trình riêng (papermill), giống cách mục 4v chạy notebook video v5, để không đụng biến của notebook này. Nó dựng đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học từ caption train, cạnh ConceptNet, xAI theo thứ hạng; và tạo **cache đặc trưng frame CLIP** của MSVD / MSR-VTT mà notebook video v5 (mục 4v) dùng lại.

Chế độ chạy (`P3_RUN_L4B_V2`): `auto` (mặc định) — nếu output của notebook này đã được Add Input (thấy cache `frames_msvd_*.npz`) thì **không chạy lại**, chỉ chép kết quả; nếu không thấy thì chạy đầy đủ (~2,4 giờ trên T4). `1` = luôn chạy, `0` = không chạy.


In [ ]:
# ============================== 4u. PHẦN C — Level 4(b) v2-graphfix: nhúng nguyên văn, chạy trong tiến trình riêng ==============================
# Phải chạy TRƯỚC mục 4v: notebook video v5 dùng lại cache đặc trưng frame mà notebook này tạo ra (qua MSVD_L4_CACHE_DIRS).
L4B_V2_NB_JSON = '{"cells": [{"cell_type": "markdown", "id": "c000", "metadata": {}, "source": "# Project 3 · Cấp độ 4(b) · v2 — sửa hai nguyên nhân gốc của đồ thị (text → video)\\n\\n**Vì sao có v2.** Lần chạy Level 4(b) v1 trên Kaggle cho thấy:\\n* đồ thị hơn CLIP: fresh1k-B, ΔMRR +0,0095;\\n* nhưng **chỉ-contains (G1) ≥ đồ thị đầy đủ (G3)**: 0,5157 so với 0,5142;\\n* bỏ mọi cạnh quan hệ lúc suy luận làm MRR tăng;\\n* H2–H9 đều không được ủng hộ.\\n\\nChẩn đoán có hai nguyên nhân:\\n1. **Nhãn video (contains) nhiễu.** Tagger CLIP zero-shot chỉ đạt F1 0,30 trên MSVD test và 0,23 trên fresh1k-B. Toàn bộ phần đồ thị giúp được lại đến từ contains: xoá giải thích đầy đủ làm tụt hạng ở 35% truy vấn, so với 0% khi xoá ngẫu nhiên.\\n2. **Quan hệ là thống kê cấp lớp từ caption train**, dùng chung cho mọi video cùng nhãn. Vì vậy nó không thêm thông tin ngoài túi nhãn, và không bắc cầu được truy vấn gián tiếp khi khái niệm của truy vấn không phải nhãn của video.\\n\\n**v2 thay đổi đúng hai chỗ đó.** Mỗi thay đổi chỉ được nhận khi qua luật đăng ký trước, và đều có đối chứng:\\n* **V2 — tagger HỌC từ caption TRAIN.** Hồi quy logistic đa nhãn trên embedding CLIP đóng băng của video. Video train nhận nhãn out-of-fold. Chọn trên VAL bằng luật nhận thay đổi: Δ MRR val ≥ 0,002 và p Holm < 0,10. Đối chứng **G11** dùng lại nhãn zero-shot.\\n* **V8 — tri thức NGOÀI ConceptNet 5.7.** Cạnh giữa các lớp (RelatedTo, IsA, UsedFor, AtLocation, CapableOf, …), cả hai đầu phải thuộc từ vựng train. Đây là tri thức ngoài, không dùng caption nên không rò rỉ. Đối chứng **G10** rewired có ràng buộc.\\n* **Tập xác nhận mới MSR-VTT fresh1k-C** (seed 20261031), rời 1k-A, dev, fresh1k và fresh1k-B. Test MSVD ở đây là lần nhìn thứ 6, chỉ mang tính thăm dò.\\n\\nMọi phần còn lại giữ như v1:\\n* G0–G8 và R3 (GraphSAGE);\\n* bộ truy vấn đăng ký trước;\\n* xAI theo thứ hạng, 5 + 5 ca, demo;\\n* các file kiểm kê, audit và kết luận tự sinh.\\n\\nKhông có GNN hay loss mới; chỉ đổi cách dựng đồ thị (nhãn và nguồn cạnh), đúng phạm vi cấp 1–2 và 4(b).\\n\\n**Chạy trên Kaggle:**\\n1. Settings: GPU T4, **Internet On**. Cần Internet để tải MSR-VTT và ConceptNet (khoảng 475 MB, đọc luồng, không lưu).\\n2. *(Nên làm)* Add Input output của Level 4(b) v1 hoặc v4 để dùng lại cache MSVD.\\n3. Save & Run All.\\n4. Tải `/kaggle/working/project03_level4b_v2_review.zip`.\\n\\n**Biến môi trường:**\\n* `MSVD_V4_STAGES`: mặc định `V1,V2,V7,V8`.\\n* `MSVD_V4_CONFIRM_SET`: mặc định `fresh1kC`.\\n* `MSVD_KG_FILE`: file ConceptNet có sẵn.\\n* `MSVD_KG_MAX_LINES`: chỉ đặt khi thử mã.\\n"}, {"cell_type": "code", "execution_count": null, "id": "c001", "metadata": {}, "outputs": [], "source": "import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re, io, socket\\nimport shutil, zipfile, tempfile, platform, copy, threading, contextlib\\nfrom pathlib import Path\\nfrom collections import defaultdict, Counter\\nfrom contextlib import contextmanager\\n\\nimport numpy as np\\n\\nRUN_ON_KAGGLE = Path(\\"/kaggle/input\\").exists()\\nNB_PID = os.getpid()\\ntry:\\n    _EXEC_COUNT_START = get_ipython().execution_count      # = 1 khi chạy từ kernel mới (Restart & Run All)\\nexcept NameError:\\n    _EXEC_COUNT_START = None\\nSTAGES = []\\n\\n\\ndef mark_stage(name, requires=()):\\n    \\"\\"\\"Ghi thứ tự chạy các cell chính. Cell chạy lệch thứ tự (hoặc dựa vào biến sót từ phiên trước)\\n    dừng ngay với thông báo rõ ràng thay vì cho ra số liệu sai.\\"\\"\\"\\n    done = {s[\\"name\\"] for s in STAGES}\\n    missing = [r for r in requires if r not in done]\\n    assert not missing, f\\"Cell \'{name}\' phải chạy sau {missing} — hãy Restart & Run All.\\"\\n    assert os.getpid() == NB_PID, \\"PID kernel đã đổi giữa chừng — hãy Restart & Run All.\\"\\n    STAGES.append(dict(name=name, t=time.time(), pid=os.getpid()))\\n\\n\\ndef has_internet(host=\\"huggingface.co\\", port=443, timeout=5.0):\\n    try:\\n        socket.create_connection((host, port), timeout=timeout).close()\\n        return True\\n    except OSError:\\n        return False\\n\\n\\nINTERNET = has_internet()\\nmark_stage(\\"env\\")\\nprint(f\\"Kaggle: {RUN_ON_KAGGLE} | Internet: {INTERNET} | Python {platform.python_version()} | \\"\\n      f\\"execution_count đầu = {_EXEC_COUNT_START}\\")\\nif not INTERNET:\\n    print(\\"⚠ Không có Internet: chỉ chạy được nếu open_clip, trọng số CLIP, dữ liệu NLTK và MSVD đã có sẵn \\"\\n          \\"(Add Input). Bật: Kaggle → Settings → Internet → On.\\")"}, {"cell_type": "code", "execution_count": null, "id": "c002", "metadata": {}, "outputs": [], "source": "# Phụ thuộc tối thiểu. Kaggle đã có torch/torchvision/opencv/nltk/pandas/scipy; chỉ cài open_clip khi thiếu.\\n# Cài thất bại thì DỪNG kèm log (không bỏ qua lỗi cài đặt).\\ndef ensure_package(module, pip_spec):\\n    import importlib\\n    try:\\n        return importlib.import_module(module)\\n    except ImportError:\\n        pass\\n    if not INTERNET:\\n        raise RuntimeError(f\\"Thiếu gói \'{pip_spec}\' và không có Internet — bật Internet hoặc Add Input wheel.\\")\\n    print(f\\"→ pip install {pip_spec}\\")\\n    r = subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", pip_spec], capture_output=True, text=True)\\n    if r.returncode != 0:\\n        print(r.stdout[-3000:]); print(r.stderr[-3000:])\\n        raise RuntimeError(f\\"pip install {pip_spec} thất bại (mã {r.returncode}) — xem log phía trên.\\")\\n    importlib.invalidate_caches()\\n    return importlib.import_module(module)\\n\\n\\ncv2 = ensure_package(\\"cv2\\", \\"opencv-python-headless\\")\\nopen_clip = ensure_package(\\"open_clip\\", \\"open_clip_torch==3.2.0\\")\\nnltk = ensure_package(\\"nltk\\", \\"nltk\\")\\n\\nimport scipy\\nimport scipy.sparse as sp\\nimport pandas as pd\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom PIL import Image\\nfrom tqdm.auto import tqdm\\nfrom IPython.display import display\\n\\nDEVICE = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\nENV_INFO = dict(python=platform.python_version(), platform=platform.platform(), torch=torch.__version__,\\n                cuda=torch.version.cuda, device=DEVICE, open_clip=getattr(open_clip, \\"__version__\\", \\"?\\"),\\n                opencv=cv2.__version__, numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,\\n                nltk=nltk.__version__, kaggle=RUN_ON_KAGGLE, internet=INTERNET)\\nif DEVICE == \\"cuda\\":\\n    ENV_INFO[\\"gpu\\"] = torch.cuda.get_device_name(0)\\n    print(f\\"GPU: {ENV_INFO[\'gpu\']} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\\")\\nelse:\\n    print(\\"⚠ KHÔNG có GPU: trích đặc trưng ViT-L-14 cho ~23,6k frame + 80,8k caption trên CPU rất chậm (hàng giờ).\\\\n\\"\\n          \\"  Kaggle: Settings → Accelerator → GPU T4/P100. Chạy thử trên CPU: MSVD_L4_CLIP=ViT-B-32, \\"\\n          \\"MSVD_L4_CLIP_W=laion2b_s34b_b79k.\\")\\nprint(json.dumps(ENV_INFO, indent=1))\\nmark_stage(\\"deps\\", requires=(\\"env\\",))"}, {"cell_type": "markdown", "id": "c003", "metadata": {}, "source": "## 0. Cấu hình\\n\\nMọi siêu tham số nằm ở cell dưới. Phần dữ liệu, CLIP, lấy mẫu frame và tagger giữ nguyên như v1/v2, nên cache đặc trưng MSVD của v2 dùng lại được. Các khối mới của v3 gồm:\\n* vị từ: họ có vị từ tự do, prompt, ngưỡng support;\\n* người chấm: mẫu, ngưỡng precision, file nhãn;\\n* benchmark xác nhận: MSR-VTT ở commit ghim;\\n* thống kê: Holm."}, {"cell_type": "code", "execution_count": null, "id": "c004", "metadata": {}, "outputs": [], "source": "_env = os.environ.get\\n\\n\\nclass CFG:\\n    # ---------------- chế độ chạy ----------------\\n    # \\"msvd\\": dữ liệu thật (mặc định). \\"synthetic\\": smoke-test mã bằng video tổng hợp — phải chọn TƯỜNG MINH.\\n    # Notebook KHÔNG bao giờ tự rơi về synthetic khi thiếu MSVD: chỉ có caption thì không phải text-to-video.\\n    dataset  = _env(\\"MSVD_L4_DATASET\\", \\"msvd\\")\\n    # dev_mode: chạy đủ pipeline trên dữ liệu thật nhưng KHOÁ test; phân tích cuối chạy trên val, gắn nhãn DEV.\\n    dev_mode = _env(\\"MSVD_L4_DEV\\", \\"0\\") == \\"1\\"\\n\\n    # ---------------- dữ liệu MSVD ----------------\\n    hf_repo      = \\"friedrichor/MSVD\\"\\n    hf_revision  = \\"9aa95a3039c106ee9e0969d82d7ce5a9efab1196\\"     # ghim commit ⇒ tái lập được\\n    msvd_root    = _env(\\"MSVD_ROOT\\", \\"\\")             # thư mục chứa msvd_*.json + YouTubeClips/ hoặc MSVD_Videos.zip\\n    video_source = _env(\\"MSVD_VIDEO_SOURCE\\", \\"auto\\")  # auto | dir | zip\\n    expected     = dict(train=(1200, 48774), val=(100, 4290), test=(670, 27763))   # (video, caption) — dataset card\\n    max_bad_video_frac = 0.0                         # MSVD thật: không chấp nhận video nào giải mã lỗi\\n\\n    # ---------------- lấy mẫu khung hình ----------------\\n    n_frames_max   = 12\\n    frame_policy   = \\"uniform-segment-midpoint\\"      # frame k = floor((k + 0.5) / F · n_frame_giải_mã_được)\\n    n_frames_grid  = [4, 8, 12]                       # chọn trên VAL; F=1 chỉ in ra để tham chiếu\\n    pool_grid      = [\\"mean\\", \\"centrality\\"]           # chọn trên VAL\\n    centrality_tau = 0.1\\n    decode_workers = 4\\n    thumb_width    = 160\\n\\n    # ---------------- CLIP (đóng băng hoàn toàn) ----------------\\n    clip_model   = _env(\\"MSVD_L4_CLIP\\", \\"ViT-L-14\\")\\n    clip_weights = _env(\\"MSVD_L4_CLIP_W\\", \\"laion2b_s32b_b82k\\")\\n    clip_bs      = 128\\n    text_bs      = 512\\n\\n    # ---------------- từ vựng (TRAIN) + tagger (chọn trên VAL) ----------------\\n    vocab_min_caps  = 2        # một từ được tính cho video nếu xuất hiện trong ≥ 2 caption của video đó ...\\n    min_term_df     = 3        # ... và ở ≥ 3 video TRAIN\\n    truth_min_caps  = 3        # nhãn tham chiếu (chỉ để chấm F1 của tagger trên VAL)\\n    tag_norms       = [\\"raw\\", \\"z\\", \\"raw+z\\"]\\n    tag_pools       = [\\"video\\", \\"frame_max\\"]\\n    tag_k_noun_grid = [3, 5, 8, 10, 12]\\n    tag_k_verb_grid = [1, 2, 3, 5]\\n    noun_prompts = [\\"a photo of a {}.\\", \\"a video frame showing a {}.\\", \\"a photo containing a {}.\\"]\\n    verb_prompts = [\\"a photo of someone {}.\\", \\"a video frame of a person {}.\\", \\"someone is {}.\\"]\\n    min_tag_freq = 2           # lớp phải được gắn cho ≥ 2 video TRAIN mới thành node\\n    max_concepts = 4000\\n\\n    # ---------------- đồ thị (cùng ý nghĩa với notebook gốc) ----------------\\n    max_path_len = 3\\n    scene_min_count, scene_top_k = 2, 10\\n    cooc_min_count, cooc_top_k = 5, 10\\n    cooc_score, cooc_min_npmi = \\"npmi\\", 0.10\\n    use_wordnet, isa_min_depth = True, 4\\n    use_idf, idf_from = True, \\"taxonomy\\"\\n    drop_redundant_indirect = True\\n    edge_weighting = True\\n\\n    # ---------------- GNN ----------------\\n    gnn_type, gnn_typed = \\"sage\\", True\\n    gate_init, path_init_indirect, scalar_lr_mult = -1.5, -1.0, 10.0\\n    hard_neg_k = 31\\n    train_with_clip = True\\n    learn_clip_w = False       # chọn lại trên VAL\\n    gnn_dim, gnn_layers, gnn_dropout = 256, 2, 0.1\\n    lr, weight_decay, temperature = 1e-3, 1e-4, 0.07\\n    epochs = int(_env(\\"MSVD_L4_EPOCHS\\", \\"15\\"))\\n    caption_passes = 4         # mỗi epoch: 4 lượt × (1 caption ngẫu nhiên / video TRAIN), batch không trùng video\\n    train_bs = 128\\n\\n    # ---------------- xếp hạng lại ----------------\\n    rerank_k_grid = [10, 20, 50]                       # chọn trên VAL (cấu hình \'none\', seed đầu)\\n    rerank_k      = None\\n    alpha_grid    = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]\\n    lambda_grid   = [0.0, 0.25, 0.5, 0.75]             # bỏ λ=1: khi α<1, s_path luôn có trọng số > 0\\n    eval_chunk    = 2048\\n\\n    # ---------------- thí nghiệm (v2: theo stage, mỗi stage một câu hỏi) ----------------\\n    seeds  = [int(s) for s in _env(\\"MSVD_L4_SEEDS\\", \\"0,1,2,3,4\\").split(\\",\\")]\\n    stages = [s for s in _env(\\"MSVD_V4_STAGES\\", \\"V1,V2,V7,V8\\").split(\\",\\") if s]\\n    # Luật CHẤP NHẬN một thay đổi (đăng ký trước, chỉ dùng VAL): Δ MRR val trung bình seed ≥ adopt_min_delta\\n    # VÀ bootstrap theo cụm video trên val cho P(Δ ≤ 0) < adopt_max_p. Không đạt ⇒ giữ thiết kế đang dùng.\\n    adopt_min_delta, adopt_max_p = 0.002, 0.10\\n    save_checkpoints = _env(\\"MSVD_G2_CKPT\\", \\"1\\") == \\"1\\"\\n\\n    # ---------------- đồ thị v2 ----------------\\n    reliable_min_support = 3          # quan hệ \'đáng tin\' = được ≥ 3 video TRAIN hỗ trợ (không có nhãn người ⇒ chỉ là lọc theo tần suất)\\n    tax_hub_frac = 0.15               # taxonomy \'lọc hub\': bỏ khái niệm mà > 15% video TRAIN với tới\\n    tax_min_depth_filtered = 6        # ... và khái niệm danh từ quá nông (độ sâu WordNet < 6)\\n    sense_max_candidates, sense_margin = 6, 1.0   # chọn sense kiểu Lesk trên ngữ cảnh caption TRAIN\\n    conf_modes = [\\"sigmoid\\", \\"minmax\\"]            # chuẩn hoá trọng số contains — chọn trên VAL theo MRR\\n    conf_floor = 0.05\\n    rewire_swaps_per_edge = 10        # số lần thử swap / cạnh trong constrained rewiring\\n    rewire_min_changed = 0.5          # đối chứng hợp lệ ⇔ ≥ 50% cạnh của mỗi loại (≥ 10 cạnh) bị đổi\\n    n_audit_captions = 200            # mẫu caption TRAIN cho audit quan hệ (để người chấm)\\n\\n    # ---------------- v3: vị từ (predicate) là thành phần của cạnh ----------------\\n    # Họ có vị từ TỰ DO: svo (vị từ = cụm động từ), prep (vị từ = giới từ). Ở agent_of / object_of vị từ chính là node\\n    # động từ đích ⇒ không mang thông tin riêng của cạnh; vẫn ghi vào schema nhưng không đưa embedding vị từ vào message.\\n    pred_families = (\\"svo\\", \\"prep\\")\\n    pred_min_support = 5              # chính sách \'support\': (họ, vị từ) xuất hiện ở ≥ 5 video TRAIN\\n    pred_prompts = {\\"svo\\": [\\"someone {} something.\\", \\"a video of a person {} something.\\"],\\n                    \\"prep\\": [\\"something {} something else.\\", \\"a photo of an object {} another object.\\"]}\\n    pred_adapt_hidden = 128           # phương án B: MLP dư DÙNG CHUNG cho mọi vị từ (không có ID riêng cho từng vị từ)\\n    # ---------------- chất lượng quan hệ: kiểm chứng TỰ ĐỘNG (mặc định) + nhãn người chấm (TUỲ CHỌN) ----------------\\n    dep_model = \\"en_core_web_sm\\"      # parser phụ thuộc cú pháp độc lập (spaCy) để đối chiếu với parser heuristic\\n    agree_min = 0.5                   # chính sách \'agreement\': giữ (họ, nguồn, vị từ, đích) nếu ≥ 50% caption TRAIN của nó được spaCy xác nhận\\n    annot_n_per_cell, annot_n_double_per_cell, annot_seed = 30, 10, 20261024\\n    annot_strata = ((2, 2, \\"low\\"), (3, 5, \\"mid\\"), (6, 10 ** 9, \\"high\\"))   # support = số video TRAIN có quan hệ\\n    annot_file = _env(\\"MSVD_G3_LABELS\\", \\"\\")                               # trống ⇒ tự tìm relation_annotation*.csv trong /kaggle/input\\n    prec_min_lower = 0.60             # chính sách \'precision\': giữ ô (họ × mức phổ biến) có cận dưới Wilson 95% ≥ 0.60\\n    # ---------------- benchmark xác nhận ĐỘC LẬP (chưa dùng ở v1/v2) ----------------\\n    confirm = _env(\\"MSVD_G3_CONFIRM\\", \\"msrvtt\\")                           # \\"msrvtt\\" | \\"none\\"\\n    msrvtt_repo = \\"friedrichor/MSR-VTT\\"\\n    msrvtt_revision = \\"c1af215a96934854f42683c19c51391aaee6f962\\"\\n    msrvtt_dev_n = 200                # dev_mode: 200 video TRAIN-9k (KHÔNG thuộc test 1k-A) chỉ để thử đường ống\\n    msrvtt_root = _env(\\"MSRVTT_ROOT\\", \\"\\")  # thư mục có sẵn msrvtt_*.json / MSRVTT_Videos.zip (trống ⇒ tải từ HF)\\n    confirm_set = _env(\\"MSVD_V4_CONFIRM_SET\\", \\"fresh1kC\\")  # \\"fresh1kC\\" (mặc định, CHƯA nhìn) | \\"fresh1kB\\" (Level 4(b) v1) | \\"fresh1k\\" (v4) | \\"1kA\\" (v3)\\n    fresh_n, fresh_seed = 1000, 20261025  # fresh1k: 1000 video train-9k, ngoài 1k-A và ngoài 200 video dev — chọn ngẫu nhiên cố định\\n    fresh_b_seed = 20261026           # fresh1k-B (Level 4(b)): 1000 video train-9k KHÁC — rời 1k-A, rời dev, rời fresh1k\\n    fresh_c_seed = 20261031           # fresh1k-C (Level 4(b) v2): rời 1k-A, dev, fresh1k VÀ fresh1k-B\\n    # ---------------- Level 4(b) v2: tagger học + ConceptNet ----------------\\n    probe_wd_grid = [1e-4, 1e-3, 1e-2, 1e-1]   # weight decay của tagger học — chọn trên VAL (F1)\\n    probe_epochs, probe_lr, probe_folds = 300, 1e-2, 5\\n    kg_version = \\"5.7.0\\"\\n    kg_url = \\"https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz\\"\\n    kg_file = _env(\\"MSVD_KG_FILE\\", \\"\\")       # file assertions .csv.gz có sẵn (trống ⇒ /kaggle/input hoặc tải từ kg_url)\\n    kg_max_lines = int(_env(\\"MSVD_KG_MAX_LINES\\", \\"0\\"))   # 0 = đọc hết (chỉ giới hạn khi thử mã)\\n    # ---------------- Level 4(b) ----------------\\n    n_eval_queries = 20               # bộ truy vấn đăng ký trước: số truy vấn / nhóm (trực tiếp, gián tiếp) / tập đánh giá\\n    eval_query_seed = 20261026\\n    n_cases = 5                       # 5 thành công + 5 thất bại\\n    demo_topk = 5\\n    demo_queries = [\\"a man is playing a guitar\\", \\"a child is playing football on the grass\\", \\"a woman is slicing a tomato\\",\\n                    \\"a dog is running on the beach\\", \\"two people are dancing on a stage\\"]   # cố định trước, không chỉnh theo kết quả\\n    # ---------------- v4: QC-RGAT ----------------\\n    v4_dim = 64                       # chiều embedding của QC-RGAT\\n    v4_lr = float(_env(\\"MSVD_V4_LR\\", \\"2e-3\\"))\\n    v4_score_bs = 256                 # số truy vấn / lô khi chấm điểm\\n    v4_rerank_k = 50                  # k chọn trên VAL ở v3 (Kaggle), giữ cố định\\n    # ---------------- thống kê ----------------\\n    holm_alpha = 0.05                 # Holm–Bonferroni cho họ so sánh chính đăng ký trước\\n\\n    # ---------------- xAI ----------------\\n    n_explain_queries, top_m_paths = 20, 5\\n    n_rank_fidelity, n_edge_deletion = 50, 15\\n    n_seed_sensitivity = 20\\n\\n    # ---------------- đường dẫn ----------------\\n    _base      = Path(\\"/kaggle/working\\") if RUN_ON_KAGGLE else Path(\\".\\")\\n    out_dir    = Path(_env(\\"MSVD_L4_OUT\\", str(_base / (\\"project03_level4b_v2\\" if dataset == \\"msvd\\" else \\"project03_level4b_v2_synthetic\\"))))\\n    data_cache = Path(_env(\\"MSVD_L4_DATA_CACHE\\", \\"/tmp/msvd_hf\\" if RUN_ON_KAGGLE else str(_base / \\"msvd_hf\\")))\\n    extra_cache_dirs = [d for d in _env(\\"MSVD_L4_CACHE_DIRS\\", \\"\\").split(\\":\\") if d]\\n\\n\\nassert CFG.dataset in (\\"msvd\\", \\"synthetic\\"), f\\"CFG.dataset không hợp lệ: {CFG.dataset}\\"\\nRESULTS = CFG.out_dir / \\"results\\"\\nfor _d in (CFG.out_dir / \\"cache\\", RESULTS, CFG.out_dir / \\"thumbs\\", CFG.out_dir / \\"checkpoints\\"):\\n    _d.mkdir(parents=True, exist_ok=True)\\nRES_TAG = \\"DEV_val\\" if CFG.dev_mode else (\\"test\\" if CFG.dataset == \\"msvd\\" else \\"SYNTHETIC_test\\")\\n\\n\\ndef set_seed(seed):\\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n\\n\\ndef _jsonable(o):\\n    if isinstance(o, np.integer): return int(o)\\n    if isinstance(o, np.floating): return float(o)\\n    if isinstance(o, np.bool_): return bool(o)\\n    if isinstance(o, np.ndarray): return o.tolist()\\n    if isinstance(o, (set, frozenset)): return sorted(o)\\n    return str(o)\\n\\n\\ndef save_json(obj, path):\\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\\n    with open(path, \\"w\\") as f:\\n        json.dump(obj, f, ensure_ascii=False, indent=2, default=_jsonable)\\n    return path\\n\\n\\ndef sha1_of(obj):\\n    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_jsonable).encode()).hexdigest()[:12]\\n\\n\\ndef cfg_snapshot():\\n    return {k: v for k, v in vars(CFG).items() if not k.startswith(\\"_\\") and not callable(v)}\\n\\n\\nCHECKS = []\\n\\n\\ndef check(cid, desc, passed, evidence=\\"\\", hard=True, status=None):\\n    \\"\\"\\"Ghi một kiểm tra vào sổ audit. hard=True ⇒ FAIL dừng notebook ngay (không chạy tiếp trên dữ liệu sai).\\"\\"\\"\\n    st = status or (\\"PASS\\" if passed else \\"FAIL\\")\\n    CHECKS.append(dict(id=cid, check=desc, status=st, evidence=str(evidence)[:900]))\\n    print(f\\"  [{st}] {cid} — {desc}: {evidence}\\")\\n    if hard and st == \\"FAIL\\":\\n        raise AssertionError(f\\"Kiểm tra {cid} THẤT BẠI: {desc} — {evidence}\\")\\n    return st == \\"PASS\\"\\n\\n\\nsave_json(dict(cfg=cfg_snapshot(), env=ENV_INFO), RESULTS / \\"config_initial.json\\")\\nprint(f\\"dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | CLIP={CFG.clip_model}/{CFG.clip_weights} | \\"\\n      f\\"seeds={CFG.seeds} | epochs={CFG.epochs} | out={CFG.out_dir.resolve()}\\")\\nif CFG.dataset == \\"synthetic\\":\\n    print(\\"⚠⚠ CHẾ ĐỘ SYNTHETIC: chỉ smoke-test mã. Mọi con số dưới đây KHÔNG phải kết quả MSVD.\\")\\nif CFG.dev_mode:\\n    print(\\"⚠ dev_mode: test bị KHOÁ; mọi phân tích cuối chạy trên VAL, gắn nhãn DEV — không dùng để báo cáo.\\")\\nmark_stage(\\"config\\", requires=(\\"deps\\",))"}, {"cell_type": "code", "execution_count": null, "id": "c005", "metadata": {}, "outputs": [], "source": "# --- Gói kết quả để TẢI VỀ xem xét: MỘT file zip nhẹ <out_dir>_review.zip, nằm cạnh thư mục output. ---\\n# Gồm: results/** (mọi CSV/JSON/MD, log truy cập, pip freeze), notebook đã chạy (nếu tìm được), pack_manifest.json.\\n# KHÔNG gồm dữ liệu và các file nặng suy ra từ dữ liệu: video MSVD (CFG.data_cache), cache/ (đặc trưng CLIP),\\n# thumbs/ (khung hình video), checkpoints/, synthetic_videos/. Manifest vẫn ghi số file và dung lượng của chúng.\\n# Cell cuối gọi pack_review(); nếu một cell bị LỖI, hook bên dưới gói ngay những gì đã có, kèm traceback.\\nimport traceback\\n\\nREVIEW_ZIP = CFG.out_dir.parent / f\\"{CFG.out_dir.name}_review.zip\\"\\nPACK_MAX_FILE_MB = 50\\n\\n\\ndef _find_executed_notebook():\\n    \\"\\"\\"Kaggle chạy bằng papermill (input = output = __notebook__.ipynb) và lưu notebook sau mỗi cell, nên bản này\\n    có output của mọi cell đã chạy xong. Chỉ nhận file được ghi SAU khi kernel này khởi động (tránh bản cũ).\\"\\"\\"\\n    t0 = STAGES[0][\\"t\\"] if STAGES else 0.0\\n    cands = [Path.cwd() / \\"__notebook__.ipynb\\", *sorted(Path(\\"/kaggle\\").glob(\\"*/__notebook__.ipynb\\")),\\n             *sorted(Path(\\"/kaggle/src\\").glob(\\"*.ipynb\\"))]\\n    for p in cands:\\n        try:\\n            if p.is_file() and p.stat().st_mtime >= t0 and json.loads(p.read_text()).get(\\"cells\\"):\\n                return p\\n        except Exception:\\n            continue\\n    return None\\n\\n\\ndef _dir_size(d):\\n    fs = [f for f in d.rglob(\\"*\\") if f.is_file()] if d.is_dir() else [d]\\n    return dict(n_files=len(fs), mb=round(sum(f.stat().st_size for f in fs) / 2**20, 1))\\n\\n\\ndef pack_review(reason, error=None):\\n    \\"\\"\\"Ghi lại REVIEW_ZIP từ đầu (gọi nhiều lần được). Không bao giờ ném lỗi ra ngoài: trả về None nếu thất bại.\\"\\"\\"\\n    try:\\n        root, inc, too_large = REVIEW_ZIP.stem, [], []\\n        nb_path = _find_executed_notebook()\\n        tmp = REVIEW_ZIP.with_name(REVIEW_ZIP.name + \\".tmp\\")\\n        with zipfile.ZipFile(tmp, \\"w\\", zipfile.ZIP_DEFLATED, compresslevel=6) as z:\\n            for p in sorted(p for p in RESULTS.rglob(\\"*\\") if p.is_file()):\\n                rel, size = p.relative_to(CFG.out_dir).as_posix(), p.stat().st_size\\n                if size > PACK_MAX_FILE_MB * 2**20:\\n                    too_large.append(dict(path=rel, mb=round(size / 2**20, 1)))\\n                    continue\\n                z.write(p, f\\"{root}/{rel}\\")\\n                inc.append(dict(path=rel, bytes=size))\\n            if nb_path is not None:\\n                z.write(nb_path, f\\"{root}/executed_notebook.ipynb\\")\\n            excluded = {d.name + (\\"/\\" if d.is_dir() else \\"\\"): _dir_size(d) for d in sorted(CFG.out_dir.iterdir()) if d != RESULTS}\\n            excluded[f\\"video MSVD ({CFG.data_cache})\\"] = \\"không gói\\"\\n            manifest = dict(created=time.strftime(\\"%Y-%m-%d %H:%M:%S\\"), reason=reason, status=\\"error\\" if error else \\"ok\\",\\n                            error_traceback=error, dataset=CFG.dataset, dev_mode=CFG.dev_mode, res_tag=RES_TAG, kaggle=RUN_ON_KAGGLE,\\n                            stages_done=[s[\\"name\\"] for s in STAGES], checks=dict(Counter(c[\\"status\\"] for c in CHECKS)),\\n                            checks_fail=[c[\\"id\\"] for c in CHECKS if c[\\"status\\"] == \\"FAIL\\"],\\n                            executed_notebook=str(nb_path) if nb_path else None, included=inc, skipped_too_large=too_large,\\n                            excluded=excluded)\\n            z.writestr(f\\"{root}/pack_manifest.json\\", json.dumps(manifest, ensure_ascii=False, indent=2, default=_jsonable))\\n        tmp.replace(REVIEW_ZIP)\\n        return REVIEW_ZIP\\n    except Exception as e:\\n        print(f\\"⚠ Không đóng gói được {REVIEW_ZIP}: {type(e).__name__}: {e}\\")\\n        return None\\n\\n\\ndef _pack_on_error(result):\\n    err = result.error_in_exec or result.error_before_exec\\n    if err is None:\\n        return\\n    tb = \\"\\".join(traceback.format_exception(type(err), err, err.__traceback__))[-8000:]\\n    z = pack_review(f\\"cell lỗi (execution_count={result.execution_count})\\", error=tb)\\n    if z is not None:\\n        print(f\\"⚠ Notebook dừng vì lỗi — đã gói những gì đã có (kèm traceback) → {z} ({z.stat().st_size / 2**20:.1f} MB)\\")\\n\\n\\ntry:\\n    _IP = get_ipython()\\n    for _cb in [c for c in _IP.events.callbacks[\\"post_run_cell\\"] if getattr(c, \\"__name__\\", \\"\\") == \\"_pack_on_error\\"]:\\n        _IP.events.unregister(\\"post_run_cell\\", _cb)          # chạy lại cell này thì không đăng ký trùng\\n    _IP.events.register(\\"post_run_cell\\", _pack_on_error)\\nexcept NameError:\\n    pass\\nprint(f\\"Gói xem xét: {REVIEW_ZIP} (tạo ở cell cuối; tự tạo nếu một cell bị lỗi)\\")"}, {"cell_type": "markdown", "id": "c006", "metadata": {}, "source": "## 0c. Giao thức đăng ký trước (v1 viết trước fresh1k-B; phần v2 ở cuối mục này viết TRƯỚC khi chạm fresh1k-C)\\n\\n**Đã biết trước khi viết notebook này** (v4, Kaggle, fresh1k):\\n* v4 hơn CLIP-only: ΔMRR +0,0109, CI [+0,0072; +0,0145].\\n* v4 **không** hơn chỉ-contains, không vị từ, vị từ hoán vị, rewired, attention tĩnh hay v3.\\n* Bỏ mọi cạnh quan hệ lúc suy luận làm MRR **tăng** nhẹ.\\n* Xoá giải thích (chỉ đường đi quan hệ) làm điểm giảm nhưng **không** đổi thứ hạng nào.\\n\\nVì vậy notebook này **không kỳ vọng** quan hệ giúp. Mục tiêu là đo lại một cách trung thực trên tập chưa nhìn, kèm các đối chứng còn thiếu.\\n\\n**Họ so sánh chính** (một phía, `a > b`, MRR trên mọi caption, bootstrap theo video, Holm–Bonferroni α = 0,05). Chỉ **fresh1k-B** mới là xác nhận; test MSVD là thăm dò.\\n\\n| | So sánh | Câu hỏi |\\n|---|---|---|\\n| H1 | G3 vs G0 | đồ thị (QC-RGAT) có hơn CLIP-only? |\\n| H2 | G3 vs G1 | cạnh quan hệ có đóng góp ngoài contains? |\\n| H3 | G3 vs G2 | khớp vị từ có ích? |\\n| H4 | G3 vs G4 | ghép **đúng** vị từ vào **đúng** cạnh có ích? |\\n| H5 | G3 vs G5 | topology thật có hơn rewiring có ràng buộc? |\\n| H6 | G3 vs G6 | attention theo truy vấn có hơn attention tĩnh? |\\n| H7 | G3 vs R3 | QC-RGAT có hơn GraphSAGE có kiểu cạnh (v3)? |\\n| H8 | G3 vs G7 | họ cạnh **vai** (`agent_of`, `object_of`) có giá trị? (G7 huấn luyện lại không có họ này) |\\n| H9 | G3 vs G8 | họ cạnh **`svo`/`prep`** có giá trị? (G8 huấn luyện lại không có họ này) |\\n\\n**Luật kết luận:**\\n* \\"Quan hệ có giá trị thật\\" chỉ được kết luận khi **H2, H4 và H5 cùng được ủng hộ** và đối chứng đạt audit.\\n* Một họ cạnh \\"có giá trị\\" khi H8 hoặc H9 tương ứng được ủng hộ.\\n\\n**So sánh phụ** (không vào họ Holm, chỉ báo cáo kèm CI):\\n* G3 vs G0 theo nhóm truy vấn trực tiếp / gián tiếp;\\n* trên bộ truy vấn đăng ký trước (≥ 20 / nhóm).\\n\\n**Chọn cấu hình** (chỉ VAL):\\n* Incumbent là T = 1 (đơn giản hơn).\\n* T = 2 chỉ được nhận khi Δ MRR val ≥ 0,002 **và** p Holm < 0,10.\\n* Cố định trước: d = 64, lr = 2e-3, k = 50.\\n* α (trọng số CLIP trong điểm cuối) chọn trên VAL trong lưới có α = 1 (CLIP-only). Nếu α = 1 thắng trên VAL thì đồ thị không được dùng ở điểm cuối, và kết quả được báo đúng như vậy.\\n\\n**Ngân sách:** 5 seed × 15 epoch cho mọi thí nghiệm. Trên cùng một tập đánh giá, mọi thí nghiệm dùng chung tập ứng viên CLIP top-k.\\n\\n**Bộ truy vấn đăng ký trước** (mục 7.3). Lấy từ caption của tập đánh giá, chỉ dựa vào hai thứ: nhãn CLIP zero-shot của video đúng và cạnh đồ thị dựng từ train. **Không** dựa vào điểm của mô hình.\\n* **Trực tiếp:** mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.\\n* **Gián tiếp:** truy vấn nêu một quan hệ (bộ ba chủ thể–vị từ–đối tượng); nhãn video chỉ phủ **một phần** khái niệm của truy vấn; có ít nhất một cạnh quan hệ nối khái niệm còn thiếu với một nhãn của video. Đó là các quan hệ dự kiến phải khai thác.\\n* Mỗi nhóm lấy 20 truy vấn (mỗi video tối đa 1) bằng seed cố định.\\n\\n### v2 — bổ sung đăng ký trước (viết TRƯỚC khi chạm MSR-VTT fresh1k-C)\\n\\nChọn cấu hình (chỉ VAL):\\n* **V2** là tagger học vs zero-shot. Incumbent là zero-shot. Tagger học chỉ được nhận khi Δ MRR val ≥ 0,002 **và** p Holm < 0,10.\\n* **V8 không phải bước chọn.** G9 (G3 + ConceptNet) là một **đối chứng / giả thuyết**, không thay G3. Cách này tránh việc chọn đồ thị theo kết quả rồi tự xác nhận nó.\\n\\nHọ so sánh chính được mở rộng (vẫn một phía, Holm α = 0,05, trên **fresh1k-C**):\\n\\n| | So sánh | Câu hỏi |\\n|---|---|---|\\n| H10 | G9 vs G1 | đồ thị có tri thức ngoài có hơn chỉ-contains? Đây là câu hỏi \\"cải thiện đến từ cấu trúc quan hệ\\" của đề |\\n| H11 | G9 vs G10 | topology thật của KG có hơn rewired có ràng buộc? |\\n| H12 | G9 vs G3 | ConceptNet có thêm giá trị so với đồ thị hiện tại? |\\n| H13 | G3 vs G11 | nhãn học từ caption train có hơn zero-shot? (NOT RUN nếu V2 không nhận) |\\n\\nKết luận \\"tri thức quan hệ (ConceptNet) có giá trị thật\\" chỉ được viết khi **H10 và H11 cùng được ủng hộ** và KG1 PASS. So sánh phụ: G9 vs G0 trên nhóm truy vấn gián tiếp.\\n"}, {"cell_type": "markdown", "id": "c007", "metadata": {}, "source": "## 0b. Hàng rào chống rò rỉ\\n\\n* **`CaptionStore`**: caption không được lưu trong `RECORDS`. Mọi lần đọc caption, hoặc dùng đặc trưng của caption, đều phải nêu **mục đích**; một ma trận quyết định split nào được dùng cho mục đích nào.\\n  * `vocab`, `relations`, `annotation` và `gnn_train` chỉ dùng train; `tune` và `tagger_select` chỉ dùng val.\\n  * Caption test MSVD và tập ngoài (`ext`, MSR-VTT) chỉ dùng để `encode`, `query_input` và (sau đóng băng) `eval`, `report`, `display`.\\n  * Truy cập sai sẽ ném `PermissionError` ngay.\\n* **`TestGate`**: test MSVD và tập ngoài chỉ mở sau `freeze()`, và mọi lần chạm đều được ghi log kèm tên split. Ở `dev_mode`, test MSVD bị khoá; tập ngoài khi đó là video **train-9k** của MSR-VTT, rời hẳn 1k-A.\\n* **`provenance()`**: mỗi artifact được ghi lại nguồn, split caption và split video đã dùng; khai báo sai loại thì `assert` thất bại."}, {"cell_type": "code", "execution_count": null, "id": "c008", "metadata": {}, "outputs": [], "source": "class TestGate:\\n    \\"\\"\\"Test chỉ mở SAU khi cấu hình đã đóng băng trên validation. Mọi lần chạm test được ghi log.\\"\\"\\"\\n\\n    def __init__(self):\\n        self.frozen, self.frozen_sha, self.frozen_at, self.log = None, None, None, []\\n\\n    def freeze(self, config):\\n        assert self.frozen is None, \\"Cấu hình đã đóng băng — không được chọn lại sau khi mở test.\\"\\n        payload = json.dumps(config, sort_keys=True, default=_jsonable)\\n        self.frozen, self.frozen_at = json.loads(payload), time.time()\\n        self.frozen_sha = hashlib.sha1(payload.encode()).hexdigest()[:12]\\n        save_json(dict(config=self.frozen, sha1=self.frozen_sha, frozen_at=self.frozen_at,\\n                       frozen_at_str=time.strftime(\\"%Y-%m-%d %H:%M:%S\\")), RESULTS / \\"frozen_config.json\\")\\n        print(f\\"🔒 Đã đóng băng cấu hình (sha1={self.frozen_sha}) → {RESULTS / \'frozen_config.json\'}\\")\\n\\n    def check(self, what, split=\\"test\\"):\\n        \\"\\"\\"split \'test\' = test MSVD (đã nhìn ở v1/v2 ⇒ ở v3 chỉ là THĂM DÒ); \'ext\' = benchmark xác nhận độc lập (MSR-VTT).\\n        Cả hai chỉ mở sau khi đóng băng. dev_mode: test MSVD bị khoá; \'ext\' khi đó là video TRAIN-9k (không phải tập xác nhận).\\"\\"\\"\\n        if self.frozen is None:\\n            raise RuntimeError(f\\"Truy cập {split.upper()} (\'{what}\') trước khi đóng băng cấu hình — vi phạm giao thức.\\")\\n        if CFG.dev_mode and split == \\"test\\":\\n            raise RuntimeError(f\\"dev_mode: test bị khoá (\'{what}\')\\")\\n        self.log.append(dict(what=what, split=split, t=time.time(), frozen_sha=self.frozen_sha, dev=bool(CFG.dev_mode)))\\n\\n\\nTEST_GATE = TestGate()\\n\\n\\nclass CaptionStore:\\n    \\"\\"\\"Nơi DUY NHẤT giữ caption. Mỗi lần đọc phải nêu mục đích; split không được phép ⇒ PermissionError.\\"\\"\\"\\n    ALLOWED = {\\n        \\"vocab\\":         {\\"train\\"},                  # từ vựng khái niệm\\n        \\"relations\\":     {\\"train\\"},                  # cạnh quan hệ thống kê từ caption\\n        \\"gnn_train\\":     {\\"train\\"},                  # truy vấn huấn luyện GNN\\n        \\"tagger_select\\": {\\"val\\"},                    # chọn (pool, norm, k) của tagger theo F1\\n        \\"tune\\":          {\\"val\\"},                    # chọn n_frames/pooling/learn_clip_w/k/α/λ\\n        \\"encode\\":        {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # CLIP-text đóng băng (không học) — chỉ để cache đặc trưng\\n        \\"query_input\\":   {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # đọc văn bản của CHÍNH truy vấn đang được trả lời\\n        \\"eval\\":          {\\"val\\", \\"test\\", \\"ext\\"},\\n        \\"display\\":       {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # chỉ hiển thị SAU khi đã xếp hạng\\n        \\"report\\":        {\\"val\\", \\"test\\", \\"ext\\"},            # báo cáo P/R/F1 của tagger sau khi đóng băng\\n    }\\n    GATED = {\\"eval\\", \\"report\\", \\"display\\"}            # với split test / ext: chỉ sau khi đóng băng\\n\\n    def __init__(self, caps_by_vid, split_by_vid):\\n        self._caps = {int(v): list(c) for v, c in caps_by_vid.items()}\\n        self.split_of = {int(v): s for v, s in split_by_vid.items()}\\n        vid_of_cap, self._text = [], []\\n        for v in sorted(self._caps):\\n            for c in self._caps[v]:\\n                vid_of_cap.append(v); self._text.append(c)\\n        self.vid_of_cap = np.array(vid_of_cap, np.int64)\\n        self.split_of_cap = np.array([self.split_of[v] for v in vid_of_cap])\\n        self.log = Counter()\\n\\n    def _authorize(self, splits, purpose):\\n        assert purpose in self.ALLOWED, f\\"mục đích lạ: {purpose}\\"\\n        for s in splits:\\n            if s not in self.ALLOWED[purpose]:\\n                raise PermissionError(f\\"RÒ RỈ bị chặn: caption split \'{s}\' không được dùng cho mục đích \'{purpose}\'\\")\\n            if s in (\\"test\\", \\"ext\\") and purpose in self.GATED:\\n                TEST_GATE.check(f\\"captions:{purpose}\\", s)\\n        for s, n in splits.items():\\n            self.log[(purpose, s)] += int(n)\\n\\n    def by_video(self, vids, purpose):\\n        vids = [int(v) for v in np.atleast_1d(vids)]\\n        self._authorize(Counter(self.split_of[v] for v in vids), purpose)\\n        return {v: list(self._caps[v]) for v in vids}\\n\\n    def texts(self, cap_idx, purpose):\\n        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))\\n        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)\\n        return [self._text[i] for i in cap_idx]\\n\\n    def use(self, cap_idx, purpose):\\n        \\"\\"\\"Khai báo dùng ĐẶC TRƯNG của caption (không lấy văn bản) — cùng luật cho phép.\\"\\"\\"\\n        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))\\n        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)\\n\\n    def replaced(self, new_caps, new_split=None):\\n        \\"\\"\\"Bản sao với caption (và tuỳ chọn split) của một số video bị thay — CHỈ dùng trong self-test.\\"\\"\\"\\n        caps = {v: list(c) for v, c in self._caps.items()}\\n        caps.update({int(v): list(c) for v, c in new_caps.items()})\\n        return CaptionStore(caps, {**self.split_of, **(new_split or {})})\\n\\n    def access_table(self):\\n        return pd.DataFrame([dict(purpose=p, split=s, n=n) for (p, s), n in sorted(self.log.items())])\\n\\n\\nPROVENANCE = []\\n_PROV_RULES = {            # loại artifact → split CAPTION được phép dùng\\n    \\"learned\\":  {\\"train\\"},             # thống kê / trọng số học được\\n    \\"selected\\": {\\"val\\"},               # siêu tham số chọn trên validation\\n    \\"frames\\":   set(),                 # chỉ từ frame (không caption)\\n    \\"external\\": set(),                 # tri thức ngoài / mô hình tiền huấn luyện\\n    \\"encoded\\":  {\\"train\\", \\"val\\", \\"test\\", \\"ext\\"},   # mã hoá bằng encoder đóng băng, không học gì\\n    \\"evaluated\\": {\\"val\\", \\"test\\", \\"ext\\"},\\n}\\n\\n\\ndef provenance(artifact, kind, source, caption_splits=(), video_splits=(), note=\\"\\"):\\n    cs = set(caption_splits)\\n    bad = cs - _PROV_RULES[kind]\\n    assert not bad, f\\"RÒ RỈ: \'{artifact}\' (loại {kind}) dùng caption của {sorted(bad)}\\"\\n    PROVENANCE.append(dict(artifact=artifact, kind=kind, source=source,\\n                           caption_splits=\\",\\".join(sorted(cs)) or \\"—\\",\\n                           video_splits=\\",\\".join(sorted(set(video_splits))) or \\"—\\",\\n                           note=note, stage=STAGES[-1][\\"name\\"], time=time.strftime(\\"%H:%M:%S\\")))"}, {"cell_type": "markdown", "id": "c009", "metadata": {}, "source": "## 1. Dữ liệu: MSVD (phát triển) + MSR-VTT 1k-A (xác nhận) — Cổng A\\n\\n**MSVD.** Như v1/v2. Video thật nằm trong `MSVD_Videos.zip`. Notebook dò `MSVD_ROOT`, rồi `/kaggle/input`, rồi cache, rồi tải từ HF ở commit ghim. Không có video thì **dừng**.\\n\\n**MSR-VTT** (`friedrichor/MSR-VTT`, commit ghim). Dùng `msrvtt_test_1k.json` (1000 video, caption chuẩn JSFusion), `raw_data/MSRVTT_data.json` (20 caption/video) và `MSRVTT_Videos.zip` (2,2 GB, mp4).\\n* Video tập ngoài được nối **sau** MSVD trong `RECORDS` với split `ext`. Vì vậy cache MSVD giữ nguyên tên và thứ tự.\\n* Truy vấn chính là mọi caption của 1000 video; truy vấn phụ là caption chuẩn 1k-A. Nếu caption chuẩn không nằm trong 20 caption thì nó được thêm vào và đếm riêng.\\n* Không nạp được MSR-VTT thì tập xác nhận ghi **NOT RUN** kèm lý do; notebook không thay bằng tập khác."}, {"cell_type": "code", "execution_count": null, "id": "c010", "metadata": {}, "outputs": [], "source": "HF_JSON = {\\"train\\": \\"msvd_train.json\\", \\"val\\": \\"msvd_val.json\\", \\"test\\": \\"msvd_test.json\\"}\\nHF_ZIP = \\"MSVD_Videos.zip\\"\\n\\n\\nclass DirVideoSource:\\n    kind = \\"dir\\"\\n\\n    def __init__(self, root):\\n        self.root = Path(root)\\n        self.files = {p.stem: p for p in self.root.rglob(\\"*\\") if p.suffix.lower() == \\".avi\\"}\\n        self.label = str(self.root)\\n\\n    def has(self, vid): return vid in self.files\\n    def size(self, vid): return self.files[vid].stat().st_size\\n\\n    @contextmanager\\n    def open(self, vid):\\n        yield self.files[vid]\\n\\n\\nclass ZipVideoSource:\\n    \\"\\"\\"Đọc video thẳng từ zip cục bộ (không giải nén toàn bộ): mỗi video được ghi ra file tạm, giải mã, rồi xoá.\\"\\"\\"\\n    kind = \\"zip\\"\\n\\n    def __init__(self, zip_file, label, exts=(\\".avi\\",)):\\n        self.z = zipfile.ZipFile(zip_file)\\n        self.members = {Path(i.filename).stem: i for i in self.z.infolist()\\n                        if i.filename.lower().endswith(exts) and not Path(i.filename).name.startswith(\\"._\\")}\\n        self.lock = threading.Lock()\\n        self.tmp = Path(tempfile.mkdtemp(prefix=\\"msvd_zip_\\"))\\n        self.label = label\\n\\n    def has(self, vid): return vid in self.members\\n    def size(self, vid): return self.members[vid].file_size\\n\\n    @contextmanager\\n    def open(self, vid):\\n        with self.lock:                                   # ZipFile không an toàn khi đọc song song\\n            data = self.z.read(self.members[vid])\\n        p = self.tmp / f\\"{vid}{Path(self.members[vid].filename).suffix.lower()}\\"\\n        p.write_bytes(data)\\n        try:\\n            yield p\\n        finally:\\n            p.unlink(missing_ok=True)\\n\\n\\ndef _first(root, name):\\n    hits = sorted(root.rglob(name), key=lambda p: len(str(p)))\\n    return hits[0] if hits else None\\n\\n\\ndef locate_msvd():\\n    roots = [Path(CFG.msvd_root)] if CFG.msvd_root else []\\n    if Path(\\"/kaggle/input\\").exists():\\n        roots.append(Path(\\"/kaggle/input\\"))\\n    roots.append(CFG.data_cache)\\n    found = dict(json={}, video_dir=None, zip=None, roots=[str(r) for r in roots])\\n    for root in roots:\\n        if not root.exists():\\n            continue\\n        for s, name in HF_JSON.items():\\n            if s not in found[\\"json\\"]:\\n                p = _first(root, name)\\n                if p is not None:\\n                    found[\\"json\\"][s] = p\\n        if found[\\"video_dir\\"] is None:\\n            for d in root.rglob(\\"YouTubeClips\\"):\\n                if d.is_dir() and any(d.glob(\\"*.avi\\")):\\n                    found[\\"video_dir\\"] = d\\n                    break\\n        if found[\\"zip\\"] is None:\\n            found[\\"zip\\"] = _first(root, HF_ZIP)\\n    return found\\n\\n\\ndef hf_download(filename):\\n    from huggingface_hub import hf_hub_download\\n    CFG.data_cache.mkdir(parents=True, exist_ok=True)\\n    print(f\\"  tải {CFG.hf_repo}@{CFG.hf_revision[:8]}/{filename} → {CFG.data_cache}\\")\\n    return Path(hf_hub_download(CFG.hf_repo, filename, repo_type=\\"dataset\\", revision=CFG.hf_revision,\\n                                local_dir=str(CFG.data_cache)))\\n\\n\\ndef open_video_source(found):\\n    mode = CFG.video_source\\n    assert mode in (\\"auto\\", \\"dir\\", \\"zip\\"), mode\\n    if mode in (\\"auto\\", \\"dir\\") and found[\\"video_dir\\"] is not None:\\n        return DirVideoSource(found[\\"video_dir\\"])\\n    if mode in (\\"auto\\", \\"zip\\") and found[\\"zip\\"] is not None:\\n        return ZipVideoSource(found[\\"zip\\"], str(found[\\"zip\\"]))\\n    if mode in (\\"auto\\", \\"zip\\") and INTERNET:\\n        return ZipVideoSource(hf_download(HF_ZIP), \\"hf_hub_download\\")\\n    raise FileNotFoundError(\\n        \\"KHÔNG tìm thấy file video MSVD (YouTubeClips/*.avi hoặc MSVD_Videos.zip) và không tải được. DỪNG: chỉ có \\"\\n        \\"caption thì KHÔNG được chạy/gọi là thí nghiệm text-to-video. Khắc phục: bật Internet, hoặc Add Input \\"\\n        f\\"dataset chứa video, hoặc đặt MSVD_ROOT. Đã dò: {found[\'roots\']}\\")\\n\\n\\ndef load_msvd():\\n    found = locate_msvd()\\n    for s, name in HF_JSON.items():\\n        if s not in found[\\"json\\"]:\\n            if not INTERNET:\\n                raise FileNotFoundError(f\\"Thiếu {name} trong {found[\'roots\']} và không có Internet để tải.\\")\\n            found[\\"json\\"][s] = hf_download(name)\\n    source = open_video_source(found)\\n    records, caps = [], {}\\n    report = dict(source=source.kind, source_label=source.label, n_video_files=len(\\n        source.files if source.kind == \\"dir\\" else source.members),\\n        json={k: str(v) for k, v in found[\\"json\\"].items()}, splits={})\\n    for s in (\\"train\\", \\"val\\", \\"test\\"):\\n        rows = json.load(open(found[\\"json\\"][s]))\\n        fields = sorted({k for r in rows for k in r})\\n        bad = [r.get(\\"video_id\\") for r in rows if not r.get(\\"video_id\\") or not isinstance(r.get(\\"caption\\"), list)]\\n        assert not bad, f\\"Schema {s} sai (thiếu video_id hoặc caption không phải list): {bad[:5]}\\"\\n        n_cap, missing, ncaps = 0, [], []\\n        for r in rows:\\n            vid = len(records)\\n            c = [str(x).strip() for x in r[\\"caption\\"] if str(x).strip()]\\n            records.append(dict(vid=vid, video_id=r[\\"video_id\\"], file=r.get(\\"video\\") or f\\"{r[\'video_id\']}.avi\\",\\n                                split=s, n_captions=len(c)))\\n            caps[vid] = c; n_cap += len(c); ncaps.append(len(c))\\n            if not source.has(r[\\"video_id\\"]):\\n                missing.append(r[\\"video_id\\"])\\n        report[\\"splits\\"][s] = dict(videos=len(rows), captions=n_cap, fields=fields, missing_files=len(missing),\\n                                   missing_examples=missing[:5],\\n                                   captions_per_video=[int(min(ncaps)), round(float(np.mean(ncaps)), 1), int(max(ncaps))],\\n                                   video_field_is_filename=all(str(r.get(\\"video\\", \\"\\")).endswith(\\".avi\\") for r in rows))\\n    ids = [r[\\"video_id\\"] for r in records]\\n    assert len(set(ids)) == len(ids), \\"video_id trùng lặp giữa các bản ghi\\"\\n    return records, caps, source, report\\n\\n\\ndef probe_video(path):\\n    \\"\\"\\"Codec / fps / kích thước / số frame giải mã thật sự (đếm bằng grab, không tin FRAME_COUNT).\\"\\"\\"\\n    cap = cv2.VideoCapture(str(path))\\n    if not cap.isOpened():\\n        return dict(ok=False, error=\\"cv2 không mở được\\")\\n    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))\\n    info = dict(codec=\\"\\".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)), fps=round(cap.get(cv2.CAP_PROP_FPS), 3),\\n                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),\\n                frames_meta=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), backend=cap.getBackendName())\\n    n = 0\\n    while cap.grab():\\n        n += 1\\n    cap.release()\\n    info.update(ok=n > 0, frames_decoded=n, duration_s=round(n / info[\\"fps\\"], 2) if info[\\"fps\\"] > 0 else None)\\n    return info"}, {"cell_type": "code", "execution_count": null, "id": "c011", "metadata": {}, "outputs": [], "source": "# --- Level 4(b): benchmark xác nhận ĐỘC LẬP = MSR-VTT fresh1k-B (CHƯA dùng ở v1–v4); fresh1k (v4) và 1k-A (v3) đã nhìn ⇒ chỉ THĂM DÒ.\\n# (Ghi chú lịch sử của v4:) Benchmark xác nhận cho v4: MSR-VTT fresh1k (CHƯA dùng ở v1–v3). Video nối vào đồ thị theo giao thức inductive. ---\\n# 1k-A (JSFusion) đã được nhìn MỘT lần ở v3 ⇒ v4 dùng 1000 video KHÁC: lấy ngẫu nhiên (seed cố định) trong train-9k, loại 1k-A và\\n# loại 200 video dev (đã chạy ở máy local). Truy vấn CHÍNH = mọi caption (MSRVTT_data.json); truy vấn PHỤ = caption đầu tiên.\\n# CFG.confirm_set = \\"1kA\\" chạy lại 1k-A (lần nhìn thứ 2 — phải khai báo). Chỉ đánh giá MỘT lần, sau khi đóng băng.\\n# dev_mode: msrvtt_dev_n video TRAIN-9k đầu theo id — chỉ để thử đường ống, không phải kết quả xác nhận.\\nMSRVTT_FILES = dict(test=\\"msrvtt_test_1k.json\\", train=\\"msrvtt_train_9k.json\\", all_caps=\\"raw_data/MSRVTT_data.json\\",\\n                    zip=\\"MSRVTT_Videos.zip\\")\\n\\n\\ndef msrvtt_file(key):\\n    name = MSRVTT_FILES[key]\\n    roots = ([Path(CFG.msrvtt_root)] if CFG.msrvtt_root else []) + ([Path(\\"/kaggle/input\\")] if RUN_ON_KAGGLE else []) \\\\\\n        + [CFG.data_cache / \\"msrvtt\\"]\\n    for root in roots:\\n        if root.exists():\\n            p = _first(root, Path(name).name)\\n            if p is not None:\\n                return p\\n    if not INTERNET:\\n        raise FileNotFoundError(f\\"Thiếu {name} (đã dò {roots}) và không có Internet.\\")\\n    from huggingface_hub import hf_hub_download\\n    d = CFG.data_cache / \\"msrvtt\\"\\n    d.mkdir(parents=True, exist_ok=True)\\n    print(f\\"  tải {CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}/{name} → {d}\\")\\n    return Path(hf_hub_download(CFG.msrvtt_repo, name, repo_type=\\"dataset\\", revision=CFG.msrvtt_revision, local_dir=str(d)))\\n\\n\\ndef load_msrvtt(vid_offset):\\n    test = json.load(open(msrvtt_file(\\"test\\")))\\n    test_ids = sorted({r[\\"video_id\\"] for r in test}, key=lambda x: int(x[5:]))\\n    tr_all = sorted([r for r in json.load(open(msrvtt_file(\\"train\\"))) if r[\\"video_id\\"] not in set(test_ids)],\\n                    key=lambda r: int(r[\\"video_id\\"][5:]))\\n    dev_ids = {r[\\"video_id\\"] for r in tr_all[:CFG.msrvtt_dev_n]}\\n    fresh1k_ids, fresh1kB_ids = set(), set()\\n    set_name = \\"dev\\" if CFG.dev_mode else CFG.confirm_set\\n    if CFG.dev_mode:\\n        rows = tr_all[:CFG.msrvtt_dev_n]\\n        caps_of = {r[\\"video_id\\"]: [str(c).strip() for c in r[\\"caption\\"] if str(c).strip()] for r in rows}\\n        std = {v: c[0] for v, c in caps_of.items()}\\n        label = f\\"MSR-VTT train-9k, {len(rows)} video đầu theo id (DEV — không phải tập xác nhận)\\"\\n    else:\\n        assert CFG.confirm_set in (\\"fresh1kC\\", \\"fresh1kB\\", \\"fresh1k\\", \\"1kA\\"), CFG.confirm_set\\n        pool = [r for r in tr_all if r[\\"video_id\\"] not in dev_ids]\\n        pick = np.random.RandomState(CFG.fresh_seed).choice(len(pool), CFG.fresh_n, replace=False)\\n        fresh1k_ids = {pool[i][\\"video_id\\"] for i in pick}          # tập xác nhận của v4 — ĐÃ nhìn một lần\\n        pool_b = [r for r in pool if r[\\"video_id\\"] not in fresh1k_ids]\\n        pick_b = np.random.RandomState(CFG.fresh_b_seed).choice(len(pool_b), CFG.fresh_n, replace=False)\\n        fresh1kB_ids = {pool_b[i][\\"video_id\\"] for i in pick_b}      # tập xác nhận của Level 4(b) v1 — ĐÃ nhìn một lần\\n        if CFG.confirm_set == \\"fresh1kC\\":\\n            pool_c = [r for r in pool_b if r[\\"video_id\\"] not in fresh1kB_ids]\\n            pick_c = np.random.RandomState(CFG.fresh_c_seed).choice(len(pool_c), CFG.fresh_n, replace=False)\\n            rows = sorted([pool_c[i] for i in pick_c], key=lambda r: int(r[\\"video_id\\"][5:]))\\n            label = (f\\"MSR-VTT fresh1k-C: {len(rows)} video train-9k (ngoài 1k-A, dev, fresh1k, fresh1k-B; seed {CFG.fresh_c_seed}) \\"\\n                     \\"— CHƯA dùng ở v1–v4 và Level 4(b) v1\\")\\n        elif CFG.confirm_set == \\"fresh1kB\\":\\n            rows = sorted([pool_b[i] for i in pick_b], key=lambda r: int(r[\\"video_id\\"][5:]))\\n            label = f\\"MSR-VTT fresh1k-B (seed {CFG.fresh_b_seed}) — ĐÃ nhìn ở Level 4(b) v1 ⇒ chỉ THĂM DÒ\\"\\n        elif CFG.confirm_set == \\"fresh1k\\":\\n            rows = sorted([pool[i] for i in pick], key=lambda r: int(r[\\"video_id\\"][5:]))\\n            label = (f\\"MSR-VTT fresh1k: {len(rows)} video train-9k (seed {CFG.fresh_seed}) — ĐÃ nhìn ở v4 ⇒ chỉ THĂM DÒ\\")\\n        else:\\n            rows = sorted(test, key=lambda r: int(r[\\"video_id\\"][5:]))\\n            label = \\"MSR-VTT test 1k-A (JSFusion) — LẦN NHÌN THỨ 2 (đã dùng ở v3)\\"\\n        data = json.load(open(msrvtt_file(\\"all_caps\\")))\\n        allc = defaultdict(list)\\n        for x in sorted(data[\\"sentences\\"], key=lambda x: int(x[\\"sen_id\\"])):\\n            allc[x[\\"video_id\\"]].append(str(x[\\"caption\\"]).strip())\\n        caps_of = {r[\\"video_id\\"]: [c for c in allc[r[\\"video_id\\"]] if c] for r in rows}\\n        std = ({r[\\"video_id\\"]: str(r[\\"caption\\"]).strip() for r in rows} if CFG.confirm_set == \\"1kA\\"\\n               else {v: c[0] for v, c in caps_of.items()})\\n    n_std_added = 0\\n    for v, c in caps_of.items():                       # caption chuẩn phải nằm trong tập truy vấn chính\\n        if std[v] not in c:\\n            c.append(std[v]); n_std_added += 1\\n    source = ZipVideoSource(msrvtt_file(\\"zip\\"), f\\"{CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}\\", exts=(\\".mp4\\",))\\n    records, caps, missing = [], {}, []\\n    for j, r in enumerate(rows):\\n        vid, c = vid_offset + j, caps_of[r[\\"video_id\\"]]\\n        records.append(dict(vid=vid, video_id=r[\\"video_id\\"], file=r.get(\\"video\\") or f\\"{r[\'video_id\']}.mp4\\", split=\\"ext\\",\\n                            n_captions=len(c), std_caption=c.index(std[r[\\"video_id\\"]]), dataset=\\"msrvtt\\"))\\n        caps[vid] = c\\n        if not source.has(r[\\"video_id\\"]):\\n            missing.append(r[\\"video_id\\"])\\n    ids = {r[\\"video_id\\"] for r in records}\\n    report = dict(label=label, set=set_name, dev=bool(CFG.dev_mode), videos=len(records), captions=sum(len(c) for c in caps.values()),\\n                  captions_per_video=[min(map(len, caps.values())), round(float(np.mean(list(map(len, caps.values())))), 1),\\n                                      max(map(len, caps.values()))],\\n                  std_captions_added=n_std_added, missing_files=len(missing), missing_examples=missing[:5],\\n                  n_test_1kA_ids=len(test_ids), overlap_with_test_1kA=len(ids & set(test_ids)),\\n                  overlap_with_dev=len(ids & dev_ids) if not CFG.dev_mode else None,\\n                  overlap_with_fresh1k=len(ids & fresh1k_ids) if not CFG.dev_mode else None,\\n                  overlap_with_fresh1kB=len(ids & fresh1kB_ids) if not CFG.dev_mode else None, zip_members=len(source.members))\\n    if set_name in (\\"fresh1k\\", \\"fresh1kB\\", \\"fresh1kC\\"):\\n        assert report[\\"overlap_with_test_1kA\\"] == 0 and report[\\"overlap_with_dev\\"] == 0, \\"fresh1k / fresh1k-B phải rời 1k-A và tập dev\\"\\n    if set_name in (\\"fresh1kB\\", \\"fresh1kC\\"):\\n        assert report[\\"overlap_with_fresh1k\\"] == 0 and report[\\"videos\\"] == CFG.fresh_n, \\"fresh1k-B/C phải rời fresh1k của v4\\"\\n    if set_name == \\"fresh1kC\\":\\n        assert report[\\"overlap_with_fresh1kB\\"] == 0, \\"fresh1k-C phải rời fresh1k-B\\"\\n    return records, caps, source, report\\n\\n\\ndef make_synthetic_ext(vid_offset, n_videos=30, seed=7):\\n    \\"\\"\\"Tập \'ext\' tổng hợp cho smoke-test: cùng thế giới đối tượng, câu chữ khác (kiểm tra đường ống, không phải kết quả).\\"\\"\\"\\n    recs, caps, source, _ = make_synthetic_msvd(n_videos=n_videos, seed=seed, prefix=\\"ext\\", all_split=\\"ext\\")\\n    out_r, out_c = [], {}\\n    for j, r in enumerate(recs):\\n        out_r.append(dict(r, vid=vid_offset + j, split=\\"ext\\", std_caption=0, dataset=\\"synthetic_ext\\"))\\n        out_c[vid_offset + j] = caps[r[\\"vid\\"]]\\n    report = dict(label=\\"synthetic ext\\", dev=True, videos=len(out_r), captions=sum(len(c) for c in out_c.values()),\\n                  missing_files=0, overlap_with_test_1kA=0, std_captions_added=0)\\n    return out_r, out_c, source, report"}, {"cell_type": "code", "execution_count": null, "id": "c012", "metadata": {}, "outputs": [], "source": "# Video TỔNG HỢP — chỉ dùng khi CFG.dataset == \\"synthetic\\" (smoke-test mã, không phải kết quả).\\n# Mỗi video có 3 cảnh (chủ thể → hành động → đồ vật), chữ in trên frame để CLIP đọc được; được ghi\\n# thành AVI (MJPG) thật để kiểm tra luôn đường ống đọc video bằng OpenCV.\\nSYN_SUBJ = {\\"man\\": \\"person\\", \\"woman\\": \\"person\\", \\"child\\": \\"person\\", \\"dog\\": \\"animal\\", \\"cat\\": \\"animal\\"}\\nSYN_ACT = {\\"play\\": ([\\"guitar\\", \\"piano\\", \\"ball\\"], \\"playing\\"), \\"ride\\": ([\\"horse\\", \\"bicycle\\"], \\"riding\\"),\\n           \\"eat\\": ([\\"apple\\", \\"pizza\\"], \\"eating\\"), \\"cut\\": ([\\"onion\\", \\"bread\\"], \\"cutting\\")}\\nSYN_COLORS = [(40, 40, 160), (40, 140, 40), (160, 60, 40), (120, 40, 120), (30, 120, 150), (150, 150, 30)]\\n\\n\\ndef make_synthetic_msvd(n_videos=150, seed=0, prefix=\\"syn\\", all_split=None):\\n    rng = random.Random(seed)\\n    vdir = CFG.out_dir / \\"synthetic_videos\\"\\n    vdir.mkdir(parents=True, exist_ok=True)\\n    records, caps = [], {}\\n    n_tr, n_va = int(n_videos * 0.6), int(n_videos * 0.15)\\n    fourcc = cv2.VideoWriter_fourcc(*\\"MJPG\\")\\n    for i in range(n_videos):\\n        s = rng.choice(sorted(SYN_SUBJ)); a = rng.choice(sorted(SYN_ACT))\\n        objs, ing = SYN_ACT[a]; o = rng.choice(objs)\\n        split_name = all_split or (\\"train\\" if i < n_tr else (\\"val\\" if i < n_tr + n_va else \\"test\\"))\\n        name = f\\"{prefix}{i:04d}_{s}_{a}_{o}\\"\\n        path = vdir / f\\"{name}.avi\\"\\n        if not path.exists():\\n            w = cv2.VideoWriter(str(path), fourcc, 12.0, (224, 224))\\n            assert w.isOpened(), \\"cv2.VideoWriter không mở được (MJPG/AVI)\\"\\n            for k in range(36):\\n                word = (s, ing, o)[k // 12]\\n                fr = np.full((224, 224, 3), SYN_COLORS[(i + k // 12) % len(SYN_COLORS)], np.uint8)\\n                cv2.putText(fr, word, (8 + (k % 12) * 2, 122), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3)\\n                w.write(fr)\\n            w.release()\\n        c = [f\\"a {s} is {ing} a {o}\\", f\\"a {s} {a}s the {o}\\", f\\"someone is {ing} a {o}\\", f\\"a {s} with a {o}\\",\\n             f\\"the {s} is {ing}\\", f\\"a {s} is {ing} a {o} slowly\\", f\\"a {SYN_SUBJ[s]} is {ing}\\", f\\"a {o} and a {s}\\"]\\n        if all_split == \\"ext\\":                         # tập ngoài: câu chữ khác với tập huấn luyện\\n            c = [f\\"there is a {s} {ing} the {o}\\", f\\"{s} {ing} {o}\\", f\\"video of a {o} being used by a {s}\\",\\n                 f\\"the {SYN_SUBJ[s]} keeps {ing} a {o}\\"]\\n        rng.shuffle(c)\\n        records.append(dict(vid=i, video_id=name, file=path.name, split=split_name, n_captions=len(c)))\\n        caps[i] = c\\n    source = DirVideoSource(vdir)\\n    report = dict(source=\\"synthetic-dir\\", source_label=str(vdir), n_video_files=len(source.files), json={}, splits={\\n        s: dict(videos=sum(r[\\"split\\"] == s for r in records),\\n                captions=sum(len(caps[r[\\"vid\\"]]) for r in records if r[\\"split\\"] == s),\\n                fields=[\\"caption\\", \\"video\\", \\"video_id\\"], missing_files=sum(\\n                    not source.has(r[\\"video_id\\"]) for r in records if r[\\"split\\"] == s))\\n        for s in (\\"train\\", \\"val\\", \\"test\\")})\\n    return records, caps, source, report"}, {"cell_type": "markdown", "id": "c013", "metadata": {}, "source": "## 2. Split chính thức và manifest\\n\\nSplit MSVD giữ đúng `msvd_{train,val,test}.json` (1200 / 100 / 670 video), chia **theo video**. sha1 chỉ tính trên ba split MSVD, nên **trùng với v2**; kiểm tra C7 xác nhận điều này. Tập ngoài `ext` có sha1 riêng và không được trùng video nào với MSVD."}, {"cell_type": "code", "execution_count": null, "id": "c014", "metadata": {}, "outputs": [], "source": "SPLITS_MSVD = (\\"train\\", \\"val\\", \\"test\\")\\n\\n\\ndef make_split(records, write=True):\\n    \\"\\"\\"Split MSVD giữ NGUYÊN như v1/v2 (sha1 chỉ tính trên train/val/test MSVD ⇒ so sánh được với v2);\\n    \'ext\' = benchmark xác nhận độc lập (MSR-VTT), có sha1 riêng.\\"\\"\\"\\n    keys = SPLITS_MSVD + (\\"ext\\",)\\n    split = {k: [r[\\"vid\\"] for r in records if r[\\"split\\"] == k] for k in keys}\\n    ids = {k: [records[v][\\"video_id\\"] for v in split[k]] for k in keys}\\n    for a, b in itertools.combinations(keys, 2):\\n        assert not set(split[a]) & set(split[b]) and not set(ids[a]) & set(ids[b]), f\\"{a}/{b} chồng lấn video!\\"\\n    payload = {k: sorted(ids[k]) for k in SPLITS_MSVD}\\n    split[\\"sha1\\"] = sha1_of(payload)\\n    split[\\"ext_sha1\\"] = sha1_of(sorted(ids[\\"ext\\"])) if ids[\\"ext\\"] else None\\n    if write:\\n        manifest = dict(\\n            dataset=CFG.dataset, hf_repo=CFG.hf_repo, hf_revision=CFG.hf_revision, sha1=split[\\"sha1\\"],\\n            policy=(\\"split chính thức của MSVD theo msvd_{train,val,test}.json; chia THEO VIDEO (không theo caption); \\"\\n                    \\"1 video = 1 ứng viên trong gallery của split đó; mỗi caption = 1 truy vấn có đúng 1 video dương\\"),\\n            counts={k: dict(videos=len(split[k]), captions=int(sum(records[v][\\"n_captions\\"] for v in split[k]))) for k in keys},\\n            video_ids=payload, ext=dict(sha1=split[\\"ext_sha1\\"], source=EXT_REPORT.get(\\"label\\") if EXT_REPORT else None,\\n                                        video_ids=sorted(ids[\\"ext\\"])))\\n        save_json(manifest, RESULTS / \\"split_manifest.json\\")\\n        print(f\\"Split (sha1={split[\'sha1\']}): \\" + \\" | \\".join(\\n            f\\"{k}={manifest[\'counts\'][k][\'videos\']} video / {manifest[\'counts\'][k][\'captions\']} caption\\" for k in keys))\\n    return split"}, {"cell_type": "markdown", "id": "c015", "metadata": {}, "source": "## 3. Lấy mẫu khung hình và đặc trưng CLIP đóng băng (Cổng B)\\n\\n* **Chính sách lấy mẫu `uniform-segment-midpoint`:** video được chia thành `F_max = 12` đoạn bằng nhau, mỗi đoạn lấy frame ở giữa: `idx_k = floor((k + 0.5)/12 · n)`, với `n` là số frame **giải mã được thực sự**. Số này được đếm bằng `grab()`, vì `CAP_PROP_FRAME_COUNT` của AVI có thể sai. Ở lượt đọc thứ hai, notebook chỉ `retrieve()` đúng 12 frame cần lấy, nên bộ nhớ chỉ chứa 12 frame/video.\\n* Mọi frame được mã hoá **một lần** bằng CLIP-image đóng băng, chuẩn hoá L2, và lưu cache `[N_video, 12, D]` (float16) cùng chỉ số frame, timestamp, fps và thời lượng. Cache được dò ở `msvd_l4/cache` và ở `/kaggle/input/**/cache`.\\n* Mọi caption được mã hoá **một lần** bằng CLIP-text của cùng mô hình. Bước này không học gì, chỉ để cache đặc trưng; việc dùng đặc trưng caption về sau vẫn phải theo luật của `CaptionStore`.\\n* Thumbnail của frame giữa video (rộng 160 px) được lưu để demo hiển thị kết quả."}, {"cell_type": "code", "execution_count": null, "id": "c016", "metadata": {}, "outputs": [], "source": "_CLIP = {}\\n\\n\\ndef get_clip():\\n    \\"\\"\\"CLIP/OpenCLIP tiền huấn luyện, đóng băng tuyệt đối (eval + requires_grad=False). Nạp một lần.\\"\\"\\"\\n    if \\"m\\" not in _CLIP:\\n        m, _, pre = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)\\n        m = m.to(DEVICE).eval()\\n        for p in m.parameters():\\n            p.requires_grad_(False)\\n        _CLIP.update(m=m, pre=pre, tok=open_clip.get_tokenizer(CFG.clip_model))\\n    return _CLIP[\\"m\\"], _CLIP[\\"pre\\"], _CLIP[\\"tok\\"]\\n\\n\\ndef _amp():\\n    return torch.autocast(\\"cuda\\", dtype=torch.float16) if DEVICE == \\"cuda\\" else contextlib.nullcontext()\\n\\n\\n@torch.no_grad()\\ndef clip_encode_images(x):\\n    m, _, _ = get_clip()\\n    with _amp():\\n        f = m.encode_image(x.to(DEVICE))\\n    return F.normalize(f.float(), dim=-1).cpu().numpy()\\n\\n\\n@torch.no_grad()\\ndef clip_encode_texts(texts, bs=None):\\n    m, _, tok = get_clip()\\n    bs = bs or CFG.text_bs\\n    out = []\\n    for s in range(0, len(texts), bs):\\n        with _amp():\\n            f = m.encode_text(tok(texts[s:s + bs]).to(DEVICE))\\n        out.append(F.normalize(f.float(), dim=-1).cpu().numpy())\\n    return np.concatenate(out)\\n\\n\\ndef segment_midpoints(n, n_frames):\\n    \\"\\"\\"Chỉ số frame tại điểm giữa của n_frames đoạn đều nhau trên n frame giải mã được.\\"\\"\\"\\n    return np.minimum(np.floor((np.arange(n_frames) + 0.5) / n_frames * n).astype(int), n - 1)\\n\\n\\ndef read_sampled_frames(path, n_frames):\\n    \\"\\"\\"Lượt 1: đếm frame GIẢI MÃ ĐƯỢC. Lượt 2: chỉ retrieve các frame cần lấy (RGB). Trả về (frames, meta).\\"\\"\\"\\n    cap = cv2.VideoCapture(str(path))\\n    if not cap.isOpened():\\n        raise IOError(\\"cv2 không mở được video\\")\\n    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)\\n    n_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\\n    n = 0\\n    while cap.grab():\\n        n += 1\\n    cap.release()\\n    if n == 0:\\n        raise IOError(\\"0 frame giải mã được\\")\\n    idx = segment_midpoints(n, n_frames)\\n    want, got = set(idx.tolist()), {}\\n    cap = cv2.VideoCapture(str(path))\\n    i = 0\\n    while i <= idx[-1] and cap.grab():\\n        if i in want:\\n            ok, fr = cap.retrieve()\\n            if ok:\\n                got[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)\\n        i += 1\\n    cap.release()\\n    lost = sorted(j for j in want if j not in got)\\n    if lost:\\n        raise IOError(f\\"không đọc được frame {lost[:5]}\\")\\n    fps_eff = fps if fps > 0 else 30.0\\n    meta = dict(n_decoded=n, n_meta=n_meta, fps=round(fps, 3), fps_assumed=fps <= 0,\\n                duration_s=round(n / fps_eff, 3), frame_idx=idx.tolist(),\\n                t_sec=[round(float(j) / fps_eff, 3) for j in idx])\\n    return [got[int(j)] for j in idx], meta\\n\\n\\ndef _cache_name(prefix, n, data_tag=None):\\n    data_tag = data_tag or f\\"{CFG.dataset}\\"\\n    rev = CFG.hf_revision[:8] if data_tag == CFG.dataset else \\"\\"\\n    tag = f\\"{data_tag}_{CFG.clip_model}_{CFG.clip_weights}_F{CFG.n_frames_max}_{CFG.frame_policy}_{rev}_{n}\\".replace(\\"__\\", \\"_\\")\\n    return f\\"{prefix}_\\" + re.sub(r\\"[^A-Za-z0-9_-]\\", \\"-\\", tag)\\n\\n\\ndef find_cache(name):\\n    \\"\\"\\"Cache của v2, cache của v1 (MSVD_L4_CACHE_DIRS, ví dụ output v1 đã Add Input) và /kaggle/input/**/cache.\\"\\"\\"\\n    dirs = [CFG.out_dir / \\"cache\\"] + [Path(d) for d in CFG.extra_cache_dirs]\\n    if RUN_ON_KAGGLE:\\n        dirs += [p for p in Path(\\"/kaggle/input\\").glob(\\"*/**/cache\\") if p.is_dir()]\\n    for d in dirs:\\n        if (d / name).exists():\\n            return d / name\\n    return None\\n\\n\\ndef extract_frame_features(records, source, allow_encode=True, data_tag=None):\\n    \\"\\"\\"[N_video, F_max, D] float16 (mỗi frame đã chuẩn hoá L2) + meta lấy mẫu. Dùng cache nếu có.\\n    Hàng thứ j ứng với records[j] (không giả định vid = j: tập ngoài nối sau MSVD).\\"\\"\\"\\n    name = _cache_name(\\"frames\\", len(records), data_tag) + \\".npz\\"\\n    vids = [r[\\"video_id\\"] for r in records]\\n    p = find_cache(name)\\n    if p is not None:\\n        z = np.load(p, allow_pickle=False)\\n        assert z[\\"video_ids\\"].tolist() == vids, \\"cache đặc trưng frame không khớp thứ tự video\\"\\n        meta = json.load(open(p.with_suffix(\\".json\\")))\\n        print(f\\"Dùng lại cache đặc trưng frame: {p} {z[\'feat\'].shape}\\")\\n        return z[\\"feat\\"].astype(np.float32), meta, p\\n    if not allow_encode:\\n        raise FileNotFoundError(name)\\n    _, pre, _ = get_clip()\\n    Fm, N = CFG.n_frames_max, len(records)\\n    feats, metas, bad = None, [None] * N, {}\\n    thumbs = CFG.out_dir / \\"thumbs\\"\\n\\n    def work(j):\\n        r = records[j]\\n        try:\\n            with source.open(r[\\"video_id\\"]) as path:\\n                frames, meta = read_sampled_frames(path, Fm)\\n            mid = frames[Fm // 2]\\n            h, w = mid.shape[:2]\\n            th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)\\n            cv2.imwrite(str(thumbs / f\\"{r[\'video_id\']}.jpg\\"), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))\\n            return j, torch.stack([pre(Image.fromarray(f)) for f in frames]), meta\\n        except Exception as e:                          # ghi lại, quyết định dừng hay không ở dưới\\n            return j, None, f\\"{type(e).__name__}: {e}\\"\\n\\n    from concurrent.futures import ThreadPoolExecutor\\n    t0, chunk = time.time(), max(8, 4 * CFG.clip_bs // Fm)\\n    with ThreadPoolExecutor(CFG.decode_workers) as ex:\\n        for s in tqdm(range(0, N, chunk), desc=f\\"giải mã + CLIP ({N} video)\\", unit=\\"lô\\"):\\n            out = list(ex.map(work, range(s, min(s + chunk, N))))     # theo lô ⇒ bộ nhớ có chặn trên\\n            ok = [(v, x, m) for v, x, m in out if x is not None]\\n            bad.update({v: m for v, x, m in out if x is None})\\n            if not ok:\\n                continue\\n            X = torch.cat([x for _, x, _ in ok])\\n            E = np.concatenate([clip_encode_images(X[i:i + CFG.clip_bs]) for i in range(0, len(X), CFG.clip_bs)])\\n            if feats is None:\\n                feats = np.zeros((N, Fm, E.shape[1]), np.float16)\\n            for j, (v, _, m) in enumerate(ok):\\n                feats[v] = E[j * Fm:(j + 1) * Fm]\\n                metas[v] = m\\n    if bad:\\n        print(f\\"⚠ {len(bad)} video giải mã lỗi:\\", list(bad.items())[:5])\\n    assert feats is not None, \\"Không giải mã được video nào\\"\\n    assert len(bad) / N <= CFG.max_bad_video_frac, (\\n        f\\"{len(bad)}/{N} video lỗi > ngưỡng {CFG.max_bad_video_frac:.0%} — dừng, không chạy thí nghiệm trên dữ liệu hỏng\\")\\n    meta = dict(policy=CFG.frame_policy, n_frames_max=Fm, clip=f\\"{CFG.clip_model}/{CFG.clip_weights}\\",\\n                source=source.kind, seconds=round(time.time() - t0, 1),\\n                per_video={records[v][\\"video_id\\"]: m for v, m in enumerate(metas)},\\n                bad={records[v][\\"video_id\\"]: e for v, e in bad.items()})\\n    p = CFG.out_dir / \\"cache\\" / name\\n    np.savez(p, feat=feats, video_ids=np.array(vids))\\n    save_json(meta, p.with_suffix(\\".json\\"))\\n    print(f\\"Đặc trưng frame {feats.shape} sau {meta[\'seconds\']}s → {p}\\")\\n    return feats.astype(np.float32), meta, p\\n\\n\\ndef extract_text_features(store, cap_idx=None, allow_encode=True, data_tag=None):\\n    \\"\\"\\"[N_caption, D]: CLIP-text đóng băng của các caption cap_idx (mặc định: mọi caption; mục đích \'encode\' — không học gì).\\"\\"\\"\\n    cap_idx = np.arange(len(store.vid_of_cap)) if cap_idx is None else np.asarray(cap_idx)\\n    texts = store.texts(cap_idx, \\"encode\\")\\n    digest = hashlib.sha1(\\"\\\\n\\".join(texts).encode()).hexdigest()[:10]\\n    name = _cache_name(\\"text\\", len(texts), data_tag) + f\\"_{digest}.npz\\"\\n    p = find_cache(name)\\n    if p is not None:\\n        feat = np.load(p)[\\"feat\\"].astype(np.float32)\\n        assert feat.shape[0] == len(texts)\\n        print(f\\"Dùng lại cache đặc trưng caption: {p} {feat.shape}\\")\\n        return feat, p\\n    if not allow_encode:\\n        raise FileNotFoundError(name)\\n    t0 = time.time()\\n    feat = clip_encode_texts(texts).astype(np.float16)\\n    p = CFG.out_dir / \\"cache\\" / name\\n    np.savez(p, feat=feat)\\n    print(f\\"Đặc trưng caption {feat.shape} sau {time.time() - t0:.0f}s → {p}\\")\\n    return feat.astype(np.float32), p\\n\\n\\ndef ext_data_tag():\\n    if not EXT_RECORDS:\\n        return None\\n    if CFG.dataset != \\"msvd\\":\\n        return \\"synthetic_ext\\"\\n    return f\\"msrvtt_{CFG.msrvtt_revision[:8]}_{\'dev\' + str(len(EXT_RECORDS)) if EXT_REPORT[\'dev\'] else EXT_REPORT.get(\'set\', \'1kA\')}\\"\\n\\n\\ndef all_features(allow_encode=True):\\n    \\"\\"\\"MSVD (cache v1/v2 dùng lại được: cùng tên, cùng thứ tự) + tập ngoài (cache riêng), nối theo thứ tự RECORDS / caption.\\"\\"\\"\\n    fm, mm, pm = extract_frame_features(RECORDS[:N_MSVD], SOURCE, allow_encode)\\n    in_msvd = CAP_VID < N_MSVD\\n    assert in_msvd[:in_msvd.sum()].all(), \\"caption MSVD phải đứng trước caption tập ngoài\\"\\n    tm, ptm = extract_text_features(CAPS, np.where(in_msvd)[0], allow_encode)\\n    caches = [pm, ptm]\\n    if EXT_RECORDS:\\n        fe, me, pe = extract_frame_features(RECORDS[N_MSVD:], EXT_SOURCE, allow_encode, data_tag=ext_data_tag())\\n        te, pte = extract_text_features(CAPS, np.where(~in_msvd)[0], allow_encode, data_tag=ext_data_tag())\\n        fm, tm = np.concatenate([fm, fe]), np.concatenate([tm, te])\\n        mm = dict(mm, per_video={**mm[\\"per_video\\"], **me[\\"per_video\\"]}, bad={**mm[\\"bad\\"], **me[\\"bad\\"]},\\n                  seconds=round(mm[\\"seconds\\"] + me[\\"seconds\\"], 1) if \\"seconds\\" in mm and \\"seconds\\" in me else None,\\n                  source=f\\"{mm[\'source\']}+{me[\'source\']}\\")\\n        caches += [pe, pte]\\n    return fm, mm, tm, caches"}, {"cell_type": "markdown", "id": "c017", "metadata": {}, "source": "### 3b. Gộp theo thời gian (chọn trên VAL)\\n\\nNotebook dùng một embedding duy nhất cho mỗi video, gộp từ `F ∈ {4, 8, 12}` frame con, cách đều nhau trong 12 frame đã lấy mẫu. Có hai cách gộp:\\n\\n* `mean`: chuẩn hoá L2 từng frame, lấy trung bình, rồi chuẩn hoá lại.\\n* `centrality`: trọng số `softmax(cos(frame, trung bình)/τ)`. Cách này giảm trọng số các frame lạc chủ đề, chẳng hạn frame chuyển cảnh hoặc frame tối; trọng số không phụ thuộc truy vấn, nên mỗi video vẫn chỉ có một vector.\\n\\nCặp `(F, cách gộp)` được chọn theo **MRR text→video của CLIP thuần trên VAL**, xếp hạng trong toàn bộ gallery val. Dòng `F=1` (frame giữa) chỉ được in ra để tham chiếu và không được chọn, để bảo đảm embedding thật sự gộp từ nhiều frame."}, {"cell_type": "code", "execution_count": null, "id": "c018", "metadata": {}, "outputs": [], "source": "def frame_subset(n_frames):\\n    \\"\\"\\"F frame con cách đều trong F_max frame đã lấy mẫu (F=4 ⇒ đúng điểm giữa của 4 đoạn: 1, 4, 7, 10).\\"\\"\\"\\n    n_frames = min(int(n_frames), CFG.n_frames_max)\\n    pos = np.round((np.arange(n_frames) + 0.5) * CFG.n_frames_max / n_frames - 0.5)\\n    return np.unique(np.clip(pos, 0, CFG.n_frames_max - 1).astype(int))\\n\\n\\ndef pool_frames(frame_feat, n_frames, mode):\\n    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n    if mode == \\"mean\\" or X.shape[1] == 1:\\n        v = X.mean(1)\\n    elif mode == \\"centrality\\":\\n        c = X.mean(1, keepdims=True)\\n        c /= np.linalg.norm(c, axis=-1, keepdims=True).clip(1e-8)\\n        a = (X * c).sum(-1) / CFG.centrality_tau\\n        w = np.exp(a - a.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)\\n        v = (w[..., None] * X).sum(1)\\n    else:\\n        raise ValueError(mode)\\n    return (v / np.linalg.norm(v, axis=-1, keepdims=True).clip(1e-8)).astype(np.float32)\\n\\n\\ndef full_gallery_ranks(txt, vid, q_idx, gallery, gold):\\n    \\"\\"\\"Thứ hạng (1 = tốt nhất) của video đúng trong TOÀN BỘ gallery theo cosine CLIP.\\"\\"\\"\\n    G = torch.from_numpy(np.ascontiguousarray(vid[np.asarray(gallery)]))\\n    pos = {int(v): i for i, v in enumerate(gallery)}\\n    gi = torch.tensor([pos[int(v)] for v in gold], dtype=torch.long)\\n    out = np.empty(len(q_idx), np.int64)\\n    for s in range(0, len(q_idx), CFG.eval_chunk):\\n        S = torch.from_numpy(np.ascontiguousarray(txt[np.asarray(q_idx[s:s + CFG.eval_chunk])])) @ G.T\\n        g = S[torch.arange(S.shape[0]), gi[s:s + CFG.eval_chunk]]\\n        out[s:s + CFG.eval_chunk] = (S > g[:, None]).sum(1).numpy() + 1\\n    return out\\n\\n\\ndef select_pooling_on_val(frame_feat, txt, split):\\n    va = np.where(CAP_SPLIT == \\"val\\")[0]\\n    CAPS.use(va, \\"tune\\")\\n    gal, gold = np.array(split[\\"val\\"]), CAP_VID[va]\\n    rows = []\\n    for nf in sorted(set(CFG.n_frames_grid) | {1}):\\n        for mode in CFG.pool_grid:\\n            if nf == 1 and mode != \\"mean\\":\\n                continue\\n            m = metrics_from_ranks(full_gallery_ranks(txt, pool_frames(frame_feat, nf, mode), va, gal, gold), gold)\\n            rows.append(dict(n_frames=nf, pooling=mode, frames=frame_subset(nf).tolist(),\\n                             selectable=nf in CFG.n_frames_grid, **{k: m[k] for k in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}))\\n    df = pd.DataFrame(rows)\\n    best = df[df.selectable].sort_values([\\"MRR\\", \\"n_frames\\"], ascending=[False, True]).iloc[0]\\n    return dict(n_frames=int(best.n_frames), pooling=str(best.pooling), val_MRR=float(best.MRR)), df\\n\\n\\ndef gate_b_checks(frame_feat, frame_meta, vid_feat, txt, split):\\n    N, Fm, D = frame_feat.shape\\n    ff = frame_feat.reshape(-1, D)\\n    fn = np.linalg.norm(ff, axis=1)\\n    m, _, _ = get_clip()\\n    check(\\"B0\\", \\"CLIP đóng băng (eval, không tham số nào requires_grad)\\",\\n          not m.training and not any(p.requires_grad for p in m.parameters()), f\\"{CFG.clip_model}/{CFG.clip_weights}\\")\\n    check(\\"B1\\", \\"đặc trưng frame có shape [N_video, F_max, D]\\", (N, Fm) == (len(RECORDS), CFG.n_frames_max),\\n          f\\"{frame_feat.shape}\\")\\n    check(\\"B2\\", \\"đặc trưng frame hữu hạn, chuẩn L2 ≈ 1\\", bool(np.isfinite(ff).all() and np.abs(fn - 1).max() < 2e-2),\\n          f\\"max|‖f‖−1| = {np.abs(fn - 1).max():.2e}\\")\\n    vn = np.linalg.norm(vid_feat, axis=1)\\n    check(\\"B3\\", \\"đúng 1 embedding / video, hữu hạn, chuẩn 1, video_id duy nhất\\",\\n          vid_feat.shape == (N, D) and bool(np.isfinite(vid_feat).all()) and np.abs(vn - 1).max() < 1e-4\\n          and len({r[\\"video_id\\"] for r in RECORDS}) == N, f\\"{vid_feat.shape}\\")\\n    check(\\"B4\\", \\"đúng 1 embedding / caption, hữu hạn, cùng chiều với video\\",\\n          txt.shape == (len(CAP_VID), D) and bool(np.isfinite(txt).all()), f\\"{txt.shape}\\")\\n    qs = np.random.RandomState(0).choice(len(txt), min(3000, len(txt)), replace=False)\\n    S = txt[qs] @ vid_feat.T\\n    check(\\"B5\\", \\"cosine văn bản–video nằm trong [−1, 1]\\", S.min() >= -1 - 1e-3 and S.max() <= 1 + 1e-3,\\n          f\\"[{S.min():.3f}, {S.max():.3f}]\\")\\n    idx = frame_subset(POOL_SEL[\\"n_frames\\"])\\n    Xn = frame_feat[:, idx] / np.linalg.norm(frame_feat[:, idx], axis=-1, keepdims=True).clip(1e-8)\\n    cos_fv = (Xn * vid_feat[:, None]).sum(-1)\\n    frac_single = float((cos_fv.max(1) > 0.9999).mean())\\n    pair = np.einsum(\\"nfd,ngd->nfg\\", Xn, Xn)\\n    off = pair[:, ~np.eye(len(idx), dtype=bool)].mean() if len(idx) > 1 else 1.0\\n    check(\\"B6\\", \\"embedding video thật sự gộp từ NHIỀU frame (không vô tình dùng 1 frame)\\",\\n          len(idx) > 1 and frac_single < 0.05,\\n          f\\"F={len(idx)} frame {idx.tolist()} ({POOL_SEL[\'pooling\']}); tỉ lệ video ≡ 1 frame = {frac_single:.3f}; \\"\\n          f\\"cos TB giữa các frame = {off:.3f}\\")\\n    bad_ts = []\\n    for name, mt in frame_meta[\\"per_video\\"].items():\\n        if mt is None:\\n            bad_ts.append(name); continue\\n        t = np.array(mt[\\"t_sec\\"])\\n        ok = (len(t) == Fm and (t >= 0).all() and (np.diff(t) >= 0).all() and t.max() < mt[\\"duration_s\\"] + 1e-6\\n              and max(mt[\\"frame_idx\\"]) < mt[\\"n_decoded\\"])\\n        if not ok:\\n            bad_ts.append(name)\\n    check(\\"B7\\", \\"lấy mẫu ổn định: đủ F_max frame / video, timestamp tăng dần trong [0, thời lượng)\\", not bad_ts,\\n          f\\"{len(frame_meta[\'per_video\'])} video; lỗi {len(bad_ts)} {bad_ts[:3]}\\")\\n    cnt = np.bincount(CAP_VID, minlength=N)\\n    vsplit = np.array([r[\\"split\\"] for r in RECORDS])\\n    probe = np.random.RandomState(1).choice(len(CAP_VID), min(500, len(CAP_VID)), replace=False)\\n    ok_text = all(CAPS._text[q] in CAPS._caps[int(CAP_VID[q])] for q in probe)        # kiểm tra chỉ số nội bộ\\n    galleries_ok = all(len(set(split[k])) == len(split[k]) and set(CAP_VID[CAP_SPLIT == k]) == set(split[k])\\n                       for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\"))\\n    check(\\"B8\\", \\"mọi caption-truy vấn ánh xạ đúng 1 video dương cùng split; gallery mỗi split không nhân bản video\\",\\n          all(cnt[r[\\"vid\\"]] == r[\\"n_captions\\"] for r in RECORDS) and bool((vsplit[CAP_VID] == CAP_SPLIT).all())\\n          and ok_text and galleries_ok,\\n          f\\"{len(CAP_VID)} truy vấn → {N} video; caption/video TB {cnt.mean():.1f}; gallery = \\"\\n          + \\", \\".join(f\\"{k}:{len(split[k])}\\" for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\")))\\n    f2, _, t2, paths = all_features(allow_encode=False)\\n    check(\\"B9\\", \\"cache đặc trưng nạp lại được mà KHÔNG chạy encoder, giống hệt bản trong bộ nhớ\\",\\n          np.array_equal(f2, frame_feat) and np.array_equal(t2, txt), \\" | \\".join(p.name for p in paths))"}, {"cell_type": "markdown", "id": "c019", "metadata": {}, "source": "## 4. Từ vựng (chỉ TRAIN), quan hệ, tagger và mẫu cho người chấm\\n\\nParser quan hệ của v2 được **giữ nguyên**, để đo đúng chất lượng của nó. Mỗi quan hệ là `(nguồn, họ, đích, vị từ)`:\\n* `agent_of`: chủ thể → động từ;\\n* `object_of`: đồ vật → động từ. Ở hai họ này vị từ chính là động từ đích;\\n* `svo`: danh từ –cụm động từ→ danh từ, ví dụ `man –ride→ horse`;\\n* `prep`: danh từ –giới từ→ danh từ, ví dụ `cat –on→ table`.\\n\\n**Kiểm chứng tự động (Nhiệm vụ B), không cần người chấm.**\\n* spaCy `en_core_web_sm` là parser phụ thuộc cú pháp, độc lập với heuristic POS. Nó trích lại quan hệ cùng schema trên **chính caption train**:\\n  * chủ ngữ (`nsubj`, kể cả kế thừa qua `conj`, `xcomp`, `acl`) → `agent_of`;\\n  * tân ngữ (`dobj`, `nsubjpass`) → `object_of`;\\n  * chủ–động–tân → `svo`;\\n  * giới từ gắn vào **danh từ** → `prep`.\\n* Một quan hệ heuristic được xác nhận khi spaCy trích cùng (nguồn, họ, đích), cùng động từ đầu (svo) hoặc cùng giới từ (prep). Tỉ lệ xác nhận được tính cho từng quan hệ, từng họ và từng ô (họ × mức phổ biến), kèm khoảng Wilson.\\n* Độ phủ ngược (spaCy tìm được mà heuristic cũng tìm được) chỉ là so sánh tương đối, **không** phải recall thật.\\n* Đây là nhãn **silver**: hai parser có thể cùng sai, nên báo cáo không gọi nó là precision.\\n\\n**Mẫu phân tầng** (họ × mức phổ biến; 360 dòng) vẫn được xuất kèm cột `auto_parser_agrees`. Nếu có file nhãn người chấm (`relation_annotation*`, **tuỳ chọn**), notebook tính thêm precision, Wilson và Cohen\'s kappa."}, {"cell_type": "code", "execution_count": null, "id": "c020", "metadata": {}, "outputs": [], "source": "# --- NLTK: WordNet + POS tagger. Không tải được thì cảnh báo rõ và dùng phương án dự phòng (ghi vào ENV_INFO). ---\\nNLTK_STATUS = {}\\nfor _pkg in (\\"wordnet\\", \\"omw-1.4\\", \\"averaged_perceptron_tagger_eng\\", \\"averaged_perceptron_tagger\\"):\\n    try:\\n        NLTK_STATUS[_pkg] = bool(nltk.download(_pkg, quiet=True, raise_on_error=False))\\n    except Exception as e:\\n        NLTK_STATUS[_pkg] = f\\"lỗi {type(e).__name__}\\"\\nfrom nltk.stem import WordNetLemmatizer\\n\\n_LEM, _LEM_CACHE = WordNetLemmatizer(), {}\\n\\n\\ndef _verb_lemma(w):\\n    \\"\\"\\"WordNetLemmatizer trả về ứng viên NGẮN NHẤT (\'riding\' → \'rid\', \'shining\' → \'shin\'); ở đây chọn ứng viên\\n    có tần suất WordNet cao nhất (\'riding\' → \'ride\').\\"\\"\\"\\n    wn = _wordnet()\\n    cands = wn._morphy(w, \\"v\\") if wn is not None and hasattr(wn, \\"_morphy\\") else []\\n    if len(cands) > 1:\\n        cnt = lambda c: sum(l.count() for s in wn.synsets(c, pos=\\"v\\") for l in s.lemmas() if l.name().lower() == c)\\n        return max(sorted(cands), key=lambda c: (cnt(c), len(c)))      # sorted: không phụ thuộc PYTHONHASHSEED\\n    return sorted(cands)[0] if cands else _LEM.lemmatize(w, \\"v\\")\\n\\n\\ndef lemma(w, pos=\\"n\\"):\\n    k = (w, pos)\\n    r = _LEM_CACHE.get(k)\\n    if r is None:\\n        try:\\n            if pos == \\"v\\":\\n                r = _verb_lemma(w)\\n            else:                                   # như NLTK (ứng viên ngắn nhất) nhưng hoà thì theo thứ tự chữ\\n                wn_ = _wordnet()\\n                cands = wn_._morphy(w, pos) if wn_ is not None and hasattr(wn_, \\"_morphy\\") else []\\n                r = min(sorted(cands), key=len) if cands else _LEM.lemmatize(w, pos)\\n        except LookupError:\\n            r = w[:-1] if (pos == \\"n\\" and w.endswith(\\"s\\") and len(w) > 3) else w\\n        _LEM_CACHE[k] = r\\n    return r\\n\\n\\ntry:\\n    from nltk.tag import PerceptronTagger\\n    _TAGGER = PerceptronTagger()                    # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)\\n    _TAGGER.tag([\\"a\\", \\"man\\", \\"is\\", \\"playing\\"])\\nexcept Exception as e:\\n    _TAGGER = None\\n    print(\\"⚠ Không nạp được POS tagger NLTK → phân loại từ theo WordNet (kém chính xác hơn):\\", type(e).__name__)\\n\\n# --- Lớp khái niệm dự phòng (notebook gốc) — dùng khi WordNet không có ---\\nFALLBACK_ISA = {\\n    \\"ball\\": [\\"sports equipment\\", \\"toy\\"], \\"field\\": [\\"open area\\", \\"sports venue\\"],\\n    \\"grass\\": [\\"vegetation\\", \\"ground\\"], \\"goal\\": [\\"sports equipment\\"],\\n    \\"shoe\\": [\\"clothing\\", \\"footwear\\"], \\"child\\": [\\"person\\"], \\"boy\\": [\\"person\\", \\"child\\"],\\n    \\"girl\\": [\\"person\\", \\"child\\"], \\"man\\": [\\"person\\"], \\"woman\\": [\\"person\\"],\\n    \\"person\\": [\\"human\\"], \\"keeper\\": [\\"person\\"],\\n    \\"pot\\": [\\"cookware\\", \\"container\\"], \\"stove\\": [\\"appliance\\", \\"cooking device\\"],\\n    \\"plate\\": [\\"tableware\\", \\"container\\"], \\"spoon\\": [\\"tableware\\", \\"utensil\\"],\\n    \\"table\\": [\\"furniture\\"], \\"bed\\": [\\"furniture\\"], \\"lamp\\": [\\"furniture\\", \\"light source\\"],\\n    \\"pillow\\": [\\"bedding\\"], \\"curtain\\": [\\"window covering\\"], \\"window\\": [\\"building part\\"],\\n    \\"book\\": [\\"reading material\\"], \\"sand\\": [\\"ground\\"], \\"umbrella\\": [\\"shade device\\"],\\n    \\"wave\\": [\\"water\\"], \\"towel\\": [\\"cloth\\"], \\"boat\\": [\\"vehicle\\", \\"watercraft\\"],\\n    \\"car\\": [\\"vehicle\\"], \\"truck\\": [\\"vehicle\\"], \\"bus\\": [\\"vehicle\\"], \\"bike\\": [\\"vehicle\\"],\\n    \\"sign\\": [\\"signage\\"], \\"road\\": [\\"open area\\", \\"transport way\\"],\\n    \\"building\\": [\\"structure\\"], \\"light\\": [\\"light source\\"], \\"elephant\\": [\\"animal\\"],\\n    \\"dog\\": [\\"animal\\"], \\"cat\\": [\\"animal\\"], \\"horse\\": [\\"animal\\"], \\"bird\\": [\\"animal\\"],\\n    \\"tree\\": [\\"vegetation\\", \\"plant\\"], \\"fence\\": [\\"barrier\\", \\"structure\\"],\\n    \\"water\\": [\\"liquid\\"], \\"rock\\": [\\"natural object\\"], \\"shirt\\": [\\"clothing\\"],\\n    \\"hat\\": [\\"clothing\\"], \\"chair\\": [\\"furniture\\"], \\"food\\": [\\"edible\\"],\\n    \\"pizza\\": [\\"food\\", \\"edible\\"], \\"plane\\": [\\"vehicle\\"], \\"train\\": [\\"vehicle\\"],\\n    \\"guitar\\": [\\"musical instrument\\"], \\"piano\\": [\\"musical instrument\\"], \\"onion\\": [\\"vegetable\\"],\\n    \\"apple\\": [\\"fruit\\"], \\"bread\\": [\\"food\\"], \\"bicycle\\": [\\"vehicle\\"],\\n}\\nFALLBACK_VERB_ISA = {\\"slice\\": [\\"cut\\"], \\"chop\\": [\\"cut\\"], \\"dice\\": [\\"cut\\"], \\"strum\\": [\\"play\\"], \\"gallop\\": [\\"ride\\"],\\n                     \\"jog\\": [\\"run\\"], \\"sprint\\": [\\"run\\"], \\"stir\\": [\\"mix\\"], \\"whisk\\": [\\"mix\\"], \\"fry\\": [\\"cook\\"],\\n                     \\"bake\\": [\\"cook\\"], \\"boil\\": [\\"cook\\"], \\"grill\\": [\\"cook\\"], \\"sip\\": [\\"drink\\"], \\"munch\\": [\\"eat\\"]}\\n\\n# --- Lọc WordNet (notebook gốc): chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC ---\\nVISUAL_LEX = {\\"noun.person\\", \\"noun.animal\\", \\"noun.artifact\\", \\"noun.object\\", \\"noun.food\\",\\n              \\"noun.plant\\", \\"noun.substance\\", \\"noun.body\\", \\"noun.location\\", \\"noun.group\\",\\n              \\"noun.phenomenon\\"}\\nSPORT_ROOTS = {\\"sport.n.01\\", \\"game.n.01\\"}\\nGENERIC_CONCEPTS = {\\"entity\\", \\"physical entity\\", \\"abstraction\\", \\"object\\", \\"whole\\", \\"thing\\",\\n                    \\"matter\\", \\"artifact\\", \\"instrumentality\\", \\"organism\\", \\"living thing\\",\\n                    \\"causal agent\\", \\"location\\", \\"region\\", \\"group\\", \\"act\\", \\"event\\",\\n                    \\"unit\\", \\"part\\", \\"piece\\", \\"material\\", \\"substance\\", \\"device\\", \\"commodity\\",\\n                    \\"consumer goods\\", \\"covering\\", \\"creation\\", \\"structure\\", \\"area\\"}\\nBASIC_CATEGORIES = {\\"person\\", \\"animal\\", \\"dog\\", \\"bird\\", \\"vehicle\\", \\"boat\\", \\"aircraft\\",\\n                    \\"clothing\\", \\"footwear\\", \\"headdress\\", \\"food\\", \\"plant\\", \\"tree\\", \\"flower\\",\\n                    \\"furniture\\", \\"building\\", \\"toy\\", \\"sport\\", \\"ball\\", \\"musical instrument\\",\\n                    \\"body of water\\", \\"geological formation\\", \\"precipitation\\", \\"child\\",\\n                    \\"game equipment\\", \\"sports equipment\\", \\"vegetable\\", \\"fruit\\"}\\nGENERIC_VERBS = {\\"act\\", \\"move\\", \\"change\\", \\"make\\", \\"be\\", \\"have\\", \\"do\\", \\"get\\", \\"go\\", \\"cause\\", \\"interact\\",\\n                 \\"transfer\\", \\"use\\", \\"take\\", \\"give\\", \\"put\\", \\"perform\\", \\"modify\\", \\"remove\\", \\"create\\", \\"travel\\",\\n                 \\"come\\", \\"displace\\", \\"connect\\", \\"communicate\\", \\"express\\", \\"refer\\", \\"think\\", \\"consume\\"}\\n\\n_WN = None\\n\\n_syn_key = lambda x: x.name()\\n_path_key = lambda path: [x.name() for x in path]      # NLTK lưu con trỏ WordNet trong set ⇒ thứ tự phụ thuộc PYTHONHASHSEED\\n\\n\\ndef _wordnet():\\n    \\"\\"\\"Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA).\\"\\"\\"\\n    global _WN\\n    if _WN is None:\\n        try:\\n            from nltk.corpus import wordnet as wn\\n            wn.synsets(\\"dog\\"); _WN = wn\\n        except Exception:\\n            _WN = False\\n    return _WN or None\\n\\n\\ndef _main_noun_synset(w):\\n    wn = _wordnet()\\n    if wn is None:\\n        return None\\n    syns = wn.synsets(w.replace(\\" \\", \\"_\\"), pos=\\"n\\")\\n    return syns[0] if syns else None\\n\\n\\n_NOUN_CACHE, _VERB_CACHE = {}, {}\\n\\n\\ndef is_visual_noun(w):\\n    if w in _NOUN_CACHE:\\n        return _NOUN_CACHE[w]\\n    wn = _wordnet()\\n    if wn is None:\\n        _NOUN_CACHE[w] = True\\n        return True\\n    s, ok = _main_noun_synset(w), False\\n    if s is not None:\\n        lex = s.lexname()\\n        if lex in VISUAL_LEX:\\n            ok = True\\n        elif lex == \\"noun.act\\":\\n            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}\\n            ok = bool(anc & SPORT_ROOTS)\\n        if ok:\\n            key = w.replace(\\" \\", \\"_\\")\\n            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)\\n                                  for l in x.lemmas() if l.name().lower() == key)\\n            ok = cnt(\\"n\\") >= cnt(\\"v\\")\\n    _NOUN_CACHE[w] = ok\\n    return ok\\n\\n\\ndef is_action_verb(v):\\n    if v not in _VERB_CACHE:\\n        wn = _wordnet()\\n        _VERB_CACHE[v] = True if wn is None else bool(wn.synsets(v, pos=\\"v\\"))\\n    return _VERB_CACHE[v]\\n\\n\\ndef get_wordnet_hypernyms(words):\\n    \\"\\"\\"IsA danh từ (notebook gốc): cha + ông (đủ sâu, không quá chung) + tổ tiên thuộc BASIC_CATEGORIES.\\"\\"\\"\\n    if not CFG.use_wordnet:\\n        return {}\\n    wn = _wordnet()\\n    if wn is None:\\n        return {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}\\n    out = {}\\n    for w in words:\\n        s = _main_noun_synset(w)\\n        if s is None:\\n            continue\\n        hy = []\\n        for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n            for depth in range(len(path) - 2, -1, -1):\\n                h = path[depth]\\n                name = h.lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n                near = depth >= len(path) - 3\\n                if name == w or name in GENERIC_CONCEPTS or name in hy:\\n                    continue\\n                if (near and depth >= CFG.isa_min_depth and\\n                        (h.lexname() in VISUAL_LEX or h.lexname() == \\"noun.Tops\\")) or name in BASIC_CATEGORIES:\\n                    hy.append(name)\\n        if hy:\\n            out[w] = hy[:5]\\n    for w in words:\\n        if w not in out and w in FALLBACK_ISA:\\n            out[w] = FALLBACK_ISA[w]\\n    return out\\n\\n\\ndef get_verb_hypernyms(verbs):\\n    \\"\\"\\"IsA động từ: cha/ông của nghĩa động từ phổ biến nhất (slice → cut, gallop → ride), bỏ động từ quá chung.\\"\\"\\"\\n    if not CFG.use_wordnet:\\n        return {}\\n    wn, out = _wordnet(), {}\\n    for v in verbs:\\n        hy = []\\n        syns = wn.synsets(v, pos=\\"v\\") if wn is not None else []\\n        if syns:\\n            for path in sorted(syns[0].hypernym_paths(), key=_path_key)[:2]:\\n                for depth in (len(path) - 2, len(path) - 3):\\n                    if depth < 0:\\n                        continue\\n                    name = path[depth].lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n                    if name != v and name not in GENERIC_VERBS and \\" \\" not in name and name not in hy:\\n                        hy.append(name)\\n        for h in FALLBACK_VERB_ISA.get(v, []):\\n            if h != v and h not in hy:\\n                hy.append(h)\\n        if hy:\\n            out[v] = hy[:3]\\n    return out\\n\\n\\ndef get_hypernyms(classes, types):\\n    nouns = [c for c in classes if types.get(c, \\"noun\\") == \\"noun\\"]\\n    verbs = [c for c in classes if types.get(c) == \\"verb\\"]\\n    out = get_wordnet_hypernyms(nouns)\\n    out.update(get_verb_hypernyms(verbs))\\n    return out\\n\\n\\nENV_INFO.update(nltk_status=NLTK_STATUS, wordnet=_wordnet() is not None, pos_tagger=_TAGGER is not None)\\nprint(\\"NLTK:\\", NLTK_STATUS, \\"| WordNet:\\", _wordnet() is not None, \\"| POS tagger:\\", _TAGGER is not None)"}, {"cell_type": "code", "execution_count": null, "id": "c021", "metadata": {}, "outputs": [], "source": "STOP = set(\\"a an the of on in at to with and or is are was were this that these those \\"\\n           \\"there here it its his her their some any for from by as be being been very \\"\\n           \\"photo image picture showing shows scene view near next video clip\\".split())\\nNOT_OBJECTS = set(\\"front background foreground top bottom side end corner air group area region place space \\"\\n                  \\"picture photo image one other something someone somebody anything thing stuff view middle edge \\"\\n                  \\"part way lot pair couple bunch number while each both several many few kind type video clip \\"\\n                  \\"footage scene time minute second\\".split())\\n_SKIP_P = set(\\"is are was were be been being has have had while as that who which and or\\".split())\\nAUX_VERBS = set(\\"be have do can will would shall should may might must seem appear become begin start try \\"\\n                \\"keep continue let get go come\\".split())\\n\\n\\ndef _norm(s):\\n    s = str(s).lower().strip()\\n    s = re.sub(r\\"[^a-z0-9 ]+\\", \\" \\", s)\\n    return re.sub(r\\"\\\\s+\\", \\" \\", s).strip()\\n\\n\\ndef _fallback_pos(toks):\\n    wn, out = _wordnet(), []\\n    for t in toks:\\n        if wn is None:\\n            out.append((t, \\"NN\\" if len(t) > 2 and t not in STOP else \\"DT\\"))\\n        elif t.endswith(\\"ing\\") and wn.synsets(lemma(t, \\"v\\"), pos=\\"v\\"):\\n            out.append((t, \\"VBG\\"))\\n        elif wn.synsets(t, pos=\\"n\\"):\\n            out.append((t, \\"NN\\"))\\n        elif wn.synsets(lemma(t, \\"v\\"), pos=\\"v\\"):\\n            out.append((t, \\"VB\\"))\\n        else:\\n            out.append((t, \\"DT\\"))\\n    return out\\n\\n\\n_PARSE_CACHE = {}\\n\\n\\ndef parse_caption(cap):\\n    \\"\\"\\"(danh từ nhìn thấy được, {động từ: dạng V-ing}, bộ ba quan hệ) của MỘT caption — heuristic POS.\\n    Bộ ba gồm (danh từ, vị từ, danh từ) như notebook gốc + (chủ thể, agent_of, động từ), (đồ vật, object_of,\\n    động từ) và chiều ngược (done_by, acts_on). Nên đọc thử ~30 mẫu để báo cáo độ chính xác.\\"\\"\\"\\n    if cap in _PARSE_CACHE:\\n        return _PARSE_CACHE[cap]\\n    toks = re.findall(r\\"[a-z]+\\", str(cap).lower())\\n    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n    nouns, verbs, triples = set(), {}, set()\\n    subj, last_n, preds, cur_v = None, None, [], None\\n    for w, t in tags:\\n        if t in (\\"NN\\", \\"NNS\\") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:\\n            n = lemma(w, \\"n\\")\\n            if n in NOT_OBJECTS or not is_visual_noun(n):\\n                continue\\n            nouns.add(n)\\n            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:\\n                triples.add((last_n, \\" \\".join(preds), n))\\n            if cur_v is not None and n != subj:\\n                triples.add((n, \\"object_of\\", cur_v)); triples.add((cur_v, \\"acts_on\\", n))\\n                cur_v = None\\n            subj = subj or n\\n            last_n, preds = n, []\\n        elif t.startswith(\\"VB\\"):\\n            v = lemma(w, \\"v\\")\\n            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):\\n                continue\\n            if t == \\"VBG\\":\\n                verbs[v] = w\\n            else:\\n                verbs.setdefault(v, None)\\n            if subj is not None and v != subj:\\n                triples.add((subj, \\"agent_of\\", v)); triples.add((v, \\"done_by\\", subj))\\n            if last_n is not None:\\n                preds.append(v)\\n            cur_v = v\\n        elif t in (\\"IN\\", \\"TO\\", \\"RP\\") and w not in _SKIP_P and last_n is not None:\\n            preds.append(w)\\n    res = (frozenset(nouns), dict(verbs), frozenset(triples))\\n    _PARSE_CACHE[cap] = res\\n    return res\\n\\n\\ndef naive_gerund(v):\\n    if v.endswith(\\"ie\\"):\\n        return v[:-2] + \\"ying\\"\\n    if v.endswith(\\"e\\") and not v.endswith((\\"ee\\", \\"ye\\", \\"oe\\")):\\n        return v[:-1] + \\"ing\\"\\n    return v + \\"ing\\"\\n\\n\\ndef build_vocab(store, split):\\n    \\"\\"\\"Từ vựng khái niệm từ caption TRAIN (mục đích \'vocab\' — CaptionStore chặn mọi split khác).\\"\\"\\"\\n    tr = split[\\"train\\"]\\n    caps = store.by_video(tr, \\"vocab\\")\\n    df, pos_cnt, ger = Counter(), defaultdict(Counter), defaultdict(Counter)\\n    for v in tr:\\n        cnt = Counter()\\n        for c in caps[v]:\\n            nouns, verbs, _ = parse_caption(c)\\n            for n in nouns:\\n                cnt[n] += 1; pos_cnt[n][\\"noun\\"] += 1\\n            for vb, surf in verbs.items():\\n                cnt[vb] += 1; pos_cnt[vb][\\"verb\\"] += 1\\n                if surf:\\n                    ger[vb][surf] += 1\\n        df.update(t for t, k in cnt.items() if k >= CFG.vocab_min_caps)\\n    terms = sorted(t for t, k in df.items() if k >= CFG.min_term_df)\\n    types = {t: (\\"verb\\" if pos_cnt[t][\\"verb\\"] > pos_cnt[t][\\"noun\\"] else \\"noun\\") for t in terms}\\n    gerund = {t: (ger[t].most_common(1)[0][0] if ger[t] else naive_gerund(t)) for t in terms if types[t] == \\"verb\\"}\\n    return dict(terms=terms, types=types, gerund=gerund, df={t: int(df[t]) for t in terms},\\n                built_from=\\"train\\", n_train_videos=len(tr), sha1=sha1_of([terms, types]))\\n\\n\\ndef build_relations(store, split, vocab):\\n    \\"\\"\\"Bộ ba quan hệ của từng video TRAIN (mục đích \'relations\'). Video val/test KHÔNG có quan hệ từ caption.\\"\\"\\"\\n    vset = set(vocab[\\"terms\\"])\\n    caps = store.by_video(split[\\"train\\"], \\"relations\\")\\n    rel = {}\\n    for v in split[\\"train\\"]:\\n        trip = set()\\n        for c in caps[v]:\\n            trip |= {t for t in parse_caption(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n        rel[v] = sorted(trip)\\n    return rel\\n\\n\\ndef caption_truth(store, vids, vocab, purpose):\\n    \\"\\"\\"Nhãn tham chiếu: từ của từ vựng xuất hiện ≥ truth_min_caps caption của video (chỉ để CHẤM tagger).\\"\\"\\"\\n    vset, caps, out = set(vocab[\\"terms\\"]), store.by_video(vids, purpose), {}\\n    for v in vids:\\n        cnt = Counter()\\n        for c in caps[v]:\\n            n, vb, _ = parse_caption(c)\\n            cnt.update(set(n) | set(vb))\\n        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}\\n    return out\\n\\n\\n_TERM_FEAT = {}\\n\\n\\ndef term_text_features(vocab):\\n    \\"\\"\\"CLIP-text của prompt cho từng từ (ensemble 3 prompt; danh từ / động từ dạng V-ing).\\"\\"\\"\\n    key = vocab[\\"sha1\\"] + sha1_of(vocab[\\"gerund\\"])\\n    if key not in _TERM_FEAT:\\n        terms, T = vocab[\\"terms\\"], None\\n        for typ, prompts in ((\\"noun\\", CFG.noun_prompts), (\\"verb\\", CFG.verb_prompts)):\\n            idx = [i for i, t in enumerate(terms) if vocab[\\"types\\"][t] == typ]\\n            if not idx:\\n                continue\\n            words = [terms[i] if typ == \\"noun\\" else vocab[\\"gerund\\"][terms[i]] for i in idx]\\n            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)\\n            if T is None:\\n                T = np.zeros((len(terms), E.shape[1]), np.float32)\\n            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n        _TERM_FEAT[key] = T\\n    return _TERM_FEAT[key]\\n\\n\\ndef _prf(pred, truth):\\n    tp = sum(len(p & t) for p, t in zip(pred, truth))\\n    P = tp / max(sum(map(len, pred)), 1)\\n    R = tp / max(sum(map(len, truth)), 1)\\n    return P, R, 2 * P * R / max(P + R, 1e-9)\\n\\n\\ndef run_tagger(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):\\n    \\"\\"\\"Gắn nhãn MỌI video bằng CLIP zero-shot trên frame với từ vựng TRAIN — không đọc caption của video được gắn.\\n    select=None ⇒ chọn (pool, norm, k_noun, k_verb) theo F1 trên VAL. Trả về (tags theo vid, lựa chọn, lưới).\\"\\"\\"\\n    terms = vocab[\\"terms\\"]\\n    T = term_text_features(vocab)\\n    tr = np.array(split[\\"train\\"])\\n    N, D = vid_feat.shape\\n    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n    S_raw = {\\"video\\": vid_feat @ T.T, \\"frame_max\\": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}\\n    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n\\n    def normed(S, mode):\\n        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên video TRAIN\\n        return S if mode == \\"raw\\" else ((S - mu) / sd if mode == \\"z\\" else S / S[tr].std() + (S - mu) / sd)\\n\\n    def topk(Sc, rows, kn, kv):\\n        out = [set() for _ in rows]\\n        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n            c = cols[typ]\\n            if k <= 0 or len(c) == 0:\\n                continue\\n            k = min(k, len(c))\\n            top = np.argpartition(-Sc[np.ix_(rows, c)], k - 1, axis=1)[:, :k]\\n            for j, row in enumerate(top):\\n                out[j].update(terms[c[i]] for i in row)\\n        return out\\n\\n    grid = []\\n    if select is None:\\n        va = np.array(split[\\"val\\"])\\n        truth = caption_truth(store, va, vocab, \\"tagger_select\\")\\n        tv = [truth[v] for v in va]\\n        for pool in CFG.tag_pools:\\n            for mode in CFG.tag_norms:\\n                Sc = normed(S_raw[pool], mode)\\n                for kn in CFG.tag_k_noun_grid:\\n                    for kv in CFG.tag_k_verb_grid:\\n                        P, R, F1 = _prf(topk(Sc, va, kn, kv), tv)\\n                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))\\n        best = max(grid, key=lambda g: (round(g[\\"val_F1\\"], 9), -(g[\\"k_noun\\"] + g[\\"k_verb\\"])))\\n        select = {k: best[k] for k in (\\"pool\\", \\"norm\\", \\"k_noun\\", \\"k_verb\\", \\"val_P\\", \\"val_R\\", \\"val_F1\\")}\\n    Sc = normed(S_raw[select[\\"pool\\"]], select[\\"norm\\"])\\n    tags = topk(Sc, np.arange(N), select[\\"k_noun\\"], select[\\"k_verb\\"])\\n    return {v: sorted(t) for v, t in enumerate(tags)}, select, grid\\n\\n\\ndef tagger_report(tags, vids, vocab, purpose):\\n    truth = caption_truth(CAPS, vids, vocab, purpose)\\n    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])\\n    return dict(P=P, R=R, F1=F1, n_videos=len(vids))"}, {"cell_type": "code", "execution_count": null, "id": "c022", "metadata": {}, "outputs": [], "source": "REL_CATS = (\\"agent_of\\", \\"object_of\\", \\"svo\\", \\"prep\\")\\n_PARSE2_CACHE = {}\\n\\n\\ndef ckey(lemma_, pos):\\n    \\"\\"\\"Khoá node lớp: \'dog#n\', \'play#v\'.\\"\\"\\"\\n    return f\\"{lemma_}#{pos}\\"\\n\\n\\ndef parse_caption_v2(cap):\\n    \\"\\"\\"Như parse_caption (v1) nhưng (1) khoá theo (lemma, POS), (2) mỗi quan hệ giữ LOẠI và VỊ TỪ.\\n    Trả về (danh từ, {động từ: dạng V-ing}, {(nguồn, loại, đích, vị từ)}).\\"\\"\\"\\n    if cap in _PARSE2_CACHE:\\n        return _PARSE2_CACHE[cap]\\n    toks = re.findall(r\\"[a-z]+\\", str(cap).lower())\\n    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n    nouns, verbs, triples = set(), {}, set()\\n    subj, last_n, preds, pred_has_verb, cur_v = None, None, [], False, None\\n    for w, t in tags:\\n        if t in (\\"NN\\", \\"NNS\\") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:\\n            n = lemma(w, \\"n\\")\\n            if n in NOT_OBJECTS or not is_visual_noun(n):\\n                continue\\n            nouns.add(n)\\n            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:\\n                triples.add((ckey(last_n, \\"n\\"), \\"svo\\" if pred_has_verb else \\"prep\\", ckey(n, \\"n\\"), \\" \\".join(preds)))\\n            if cur_v is not None and n != subj:\\n                triples.add((ckey(n, \\"n\\"), \\"object_of\\", ckey(cur_v, \\"v\\"), cur_v))\\n                cur_v = None\\n            subj = subj or n\\n            last_n, preds, pred_has_verb = n, [], False\\n        elif t.startswith(\\"VB\\"):\\n            v = lemma(w, \\"v\\")\\n            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):\\n                continue\\n            if t == \\"VBG\\":\\n                verbs[v] = w\\n            else:\\n                verbs.setdefault(v, None)\\n            if subj is not None:\\n                triples.add((ckey(subj, \\"n\\"), \\"agent_of\\", ckey(v, \\"v\\"), v))\\n            if last_n is not None:\\n                preds.append(v); pred_has_verb = True\\n            cur_v = v\\n        elif t in (\\"IN\\", \\"TO\\", \\"RP\\") and w not in _SKIP_P and last_n is not None:\\n            preds.append(w)\\n    res = (frozenset(nouns), dict(verbs), frozenset(triples))\\n    _PARSE2_CACHE[cap] = res\\n    return res\\n\\n\\ndef build_vocab_v2(store, split):\\n    \\"\\"\\"Từ vựng khoá (lemma, POS) từ caption TRAIN: một lemma có thể là cả \'x#n\' và \'x#v\' nếu cả hai đạt ngưỡng.\\"\\"\\"\\n    tr = split[\\"train\\"]\\n    caps = store.by_video(tr, \\"vocab\\")\\n    df, ger = Counter(), defaultdict(Counter)\\n    for v in tr:\\n        cnt = Counter()\\n        for c in caps[v]:\\n            nouns, verbs, _ = parse_caption_v2(c)\\n            cnt.update(ckey(n, \\"n\\") for n in nouns)\\n            for vb, surf in verbs.items():\\n                cnt[ckey(vb, \\"v\\")] += 1\\n                if surf:\\n                    ger[vb][surf] += 1\\n        df.update(k for k, m in cnt.items() if m >= CFG.vocab_min_caps)\\n    terms = sorted(k for k, m in df.items() if m >= CFG.min_term_df)\\n    lem = {k: k.rsplit(\\"#\\", 1)[0] for k in terms}\\n    types = {k: (\\"verb\\" if k.endswith(\\"#v\\") else \\"noun\\") for k in terms}\\n    gerund = {k: (ger[lem[k]].most_common(1)[0][0] if ger[lem[k]] else naive_gerund(lem[k])) for k in terms if types[k] == \\"verb\\"}\\n    both = sorted({lem[k] for k in terms if types[k] == \\"noun\\"} & {lem[k] for k in terms if types[k] == \\"verb\\"})\\n    return dict(terms=terms, types=types, lemma=lem, gerund=gerund, df={k: int(df[k]) for k in terms},\\n                lemmas_with_two_pos=both, built_from=\\"train\\", n_train_videos=len(tr), sha1=sha1_of([terms, types]))\\n\\n\\ndef build_relations_v2(store, split, vocab):\\n    \\"\\"\\"{vid TRAIN: [(nguồn, loại, đích, vị từ)]} — chỉ video TRAIN (mục đích \'relations\').\\"\\"\\"\\n    vset = set(vocab[\\"terms\\"])\\n    caps = store.by_video(split[\\"train\\"], \\"relations\\")\\n    rel = {}\\n    for v in split[\\"train\\"]:\\n        trip = set()\\n        for c in caps[v]:\\n            trip |= {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n        rel[v] = sorted(trip)\\n    return rel\\n\\n\\ndef caption_truth_v2(store, vids, vocab, purpose):\\n    vset, caps, out = set(vocab[\\"terms\\"]), store.by_video(vids, purpose), {}\\n    for v in vids:\\n        cnt = Counter()\\n        for c in caps[v]:\\n            n, vb, _ = parse_caption_v2(c)\\n            cnt.update({ckey(x, \\"n\\") for x in n} | {ckey(x, \\"v\\") for x in vb})\\n        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}\\n    return out\\n\\n\\n_TERM_FEAT2 = {}\\n\\n\\ndef term_text_features_v2(vocab):\\n    \\"\\"\\"CLIP-text (ensemble 3 prompt) cho từng khoá: danh từ dùng lemma, động từ dùng dạng V-ing.\\"\\"\\"\\n    key = vocab[\\"sha1\\"] + sha1_of(vocab[\\"gerund\\"])\\n    if key not in _TERM_FEAT2:\\n        terms, T = vocab[\\"terms\\"], None\\n        for typ, prompts in ((\\"noun\\", CFG.noun_prompts), (\\"verb\\", CFG.verb_prompts)):\\n            idx = [i for i, t in enumerate(terms) if vocab[\\"types\\"][t] == typ]\\n            if not idx:\\n                continue\\n            words = [vocab[\\"lemma\\"][terms[i]] if typ == \\"noun\\" else vocab[\\"gerund\\"][terms[i]] for i in idx]\\n            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)\\n            if T is None:\\n                T = np.zeros((len(terms), E.shape[1]), np.float32)\\n            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n        _TERM_FEAT2[key] = T\\n    return _TERM_FEAT2[key]\\n\\n\\ndef run_tagger_v2(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):\\n    \\"\\"\\"Như run_tagger (v1) với khoá (lemma, POS). Trả thêm điểm đã chuẩn hoá của từng nhãn được gắn\\n    (dùng làm trọng số contains ở S3). Thống kê chuẩn hoá chỉ trên video TRAIN; lựa chọn theo F1 trên VAL.\\"\\"\\"\\n    terms = vocab[\\"terms\\"]\\n    T = term_text_features_v2(vocab)\\n    tr = np.array(split[\\"train\\"])\\n    N, D = vid_feat.shape\\n    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)\\n    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)\\n    S_raw = {\\"video\\": vid_feat @ T.T, \\"frame_max\\": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}\\n    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n\\n    def normed(S, mode):\\n        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6\\n        return S if mode == \\"raw\\" else ((S - mu) / sd if mode == \\"z\\" else S / S[tr].std() + (S - mu) / sd)\\n\\n    def topk(Sc, rows, kn, kv):\\n        out = [dict() for _ in rows]\\n        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n            c = cols[typ]\\n            if k <= 0 or len(c) == 0:\\n                continue\\n            k = min(k, len(c))\\n            sub = Sc[np.ix_(rows, c)]\\n            top = np.argpartition(-sub, k - 1, axis=1)[:, :k]\\n            for j, row in enumerate(top):\\n                out[j].update({terms[c[i]]: float(sub[j, i]) for i in row})\\n        return out\\n\\n    grid = []\\n    if select is None:\\n        va = np.array(split[\\"val\\"])\\n        truth = caption_truth_v2(store, va, vocab, \\"tagger_select\\")\\n        tv = [truth[v] for v in va]\\n        for pool in CFG.tag_pools:\\n            for mode in CFG.tag_norms:\\n                Sc = normed(S_raw[pool], mode)\\n                for kn in CFG.tag_k_noun_grid:\\n                    for kv in CFG.tag_k_verb_grid:\\n                        P, R, F1 = _prf([set(d) for d in topk(Sc, va, kn, kv)], tv)\\n                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))\\n        best = max(grid, key=lambda g: (round(g[\\"val_F1\\"], 9), -(g[\\"k_noun\\"] + g[\\"k_verb\\"])))\\n        select = {k: best[k] for k in (\\"pool\\", \\"norm\\", \\"k_noun\\", \\"k_verb\\", \\"val_P\\", \\"val_R\\", \\"val_F1\\")}\\n    Sc = normed(S_raw[select[\\"pool\\"]], select[\\"norm\\"])\\n    tagged = topk(Sc, np.arange(N), select[\\"k_noun\\"], select[\\"k_verb\\"])\\n    tags = {v: sorted(d) for v, d in enumerate(tagged)}\\n    scores = {v: d for v, d in enumerate(tagged)}\\n    return tags, scores, select, grid\\n\\n\\ndef tagger_report_v2(tags, vids, vocab, purpose):\\n    truth = caption_truth_v2(CAPS, vids, vocab, purpose)\\n    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])\\n    return dict(P=P, R=R, F1=F1, n_videos=len(vids))"}, {"cell_type": "code", "execution_count": null, "id": "c023", "metadata": {}, "outputs": [], "source": "# --- Chất lượng quan hệ (Nhiệm vụ B): mẫu cho NGƯỜI chấm + đọc nhãn đã chấm. Chỉ caption TRAIN. ---\\n# Mẫu KHÔNG phụ thuộc CLIP/tagger (chỉ parser + từ vựng train) ⇒ giống nhau giữa máy local và Kaggle.\\n# Đơn vị mẫu = MỘT quan hệ (họ, nguồn, vị từ, đích) đủ điều kiện thành cạnh (support ≥ scene_min_count video TRAIN),\\n# kèm MỘT caption TRAIN chọn ngẫu nhiên có chứa nó; người chấm trả lời: caption này có diễn đạt đúng quan hệ đó không?\\nCaptionStore.ALLOWED[\\"annotation\\"] = {\\"train\\"}\\nANNOT_LABEL_COLS = (\\"annotator_1\\", \\"annotator_2\\", \\"adjudicated\\")\\n\\n\\ndef relation_instances(store, split, vocab):\\n    \\"\\"\\"[(vid, caption, (nguồn, họ, đích, vị từ))] trên caption TRAIN; support = số video TRAIN có quan hệ (đếm 1 lần / video);\\n    pred_support[(họ, vị từ)] = số video TRAIN có ít nhất một quan hệ thuộc họ đó với vị từ đó.\\"\\"\\"\\n    vset = set(vocab[\\"terms\\"])\\n    caps = store.by_video(split[\\"train\\"], \\"annotation\\")\\n    inst, sup, psup = [], Counter(), Counter()\\n    for v in split[\\"train\\"]:\\n        seen = set()\\n        for c in caps[v]:\\n            for t in sorted(parse_caption_v2(c)[2]):\\n                if t[0] in vset and t[2] in vset and t[0] != t[2]:\\n                    inst.append((v, c, t)); seen.add(t)\\n        sup.update(seen)\\n        psup.update({(t[1], t[3]) for t in seen})\\n    return inst, sup, psup\\n\\n\\ndef support_stratum(s):\\n    for lo, hi, name in CFG.annot_strata:\\n        if lo <= s <= hi:\\n            return name\\n    return \\"below_threshold\\"\\n\\n\\ndef _display_relation(src, rel, dst, pred):\\n    a, b = src.rsplit(\\"#\\", 1)[0], dst.rsplit(\\"#\\", 1)[0]\\n    if rel == \\"agent_of\\":\\n        return a, pred, \\"\\", f\\"\'{a}\' là CHỦ THỂ thực hiện hành động \'{pred}\'\\"\\n    if rel == \\"object_of\\":\\n        return \\"\\", pred, a, f\\"\'{a}\' là ĐỐI TƯỢNG chịu hành động \'{pred}\'\\"\\n    return a, pred, b, f\\"{a} —{pred}→ {b}\\"\\n\\n\\ndef build_annotation_sample(store, split, vocab, records):\\n    \\"\\"\\"Phân tầng: họ quan hệ × mức phổ biến (support); trong mỗi ô chọn luân phiên theo vị từ để đa dạng vị từ.\\"\\"\\"\\n    inst, sup, psup = relation_instances(store, split, vocab)\\n    by_trip = defaultdict(list)\\n    for v, c, t in inst:\\n        if sup[t] >= CFG.scene_min_count:\\n            by_trip[t].append((v, c))\\n    rng = np.random.RandomState(CFG.annot_seed)\\n    rows, cells = [], []\\n    for rel in REL_CATS:\\n        for lo, hi, name in CFG.annot_strata:\\n            trips = sorted(t for t in by_trip if t[1] == rel and lo <= sup[t] <= hi)\\n            by_pred = defaultdict(list)\\n            for t in trips:\\n                by_pred[t[3]].append(t)\\n            preds = sorted(by_pred)\\n            rng.shuffle(preds)\\n            for p in preds:\\n                rng.shuffle(by_pred[p])\\n            chosen = []\\n            while len(chosen) < CFG.annot_n_per_cell and any(by_pred[p] for p in preds):\\n                for p in preds:                              # luân phiên vị từ ⇒ không vị từ nào chiếm cả ô\\n                    if by_pred[p] and len(chosen) < CFG.annot_n_per_cell:\\n                        chosen.append(by_pred[p].pop())\\n            dbl = set(rng.choice(len(chosen), min(CFG.annot_n_double_per_cell, len(chosen)), replace=False).tolist()) if chosen else set()\\n            cells.append(dict(family=rel, stratum=name, support_lo=lo, support_hi=hi, n_relations_in_cell=len(trips),\\n                              n_predicates_in_cell=len(preds), n_sampled=len(chosen)))\\n            for j, t in enumerate(chosen):\\n                v, c = by_trip[t][rng.randint(len(by_trip[t]))]\\n                s_, p_, o_, txt = _display_relation(t[0], t[1], t[2], t[3])\\n                vid = records[v][\\"video_id\\"]\\n                rows.append(dict(sample_id=hashlib.sha1(\\"|\\".join([vid, c, t[1], t[0], t[3], t[2]]).encode()).hexdigest()[:12],\\n                                 family=rel, caption=c, subject=s_, predicate=p_, object=o_, relation=txt,\\n                                 double_annotate=int(j in dbl), annotator_1=\\"\\", annotator_2=\\"\\", adjudicated=\\"\\", note=\\"\\",\\n                                 video_id=vid, src_node=t[0], dst_node=t[2], stratum=name, relation_support_videos=int(sup[t]),\\n                                 predicate_support_videos=int(psup[(t[1], t[3])])))\\n    df = pd.DataFrame(rows)\\n    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)      # trộn thứ tự: người chấm không thấy khối theo ô\\n    return df, pd.DataFrame(cells), sup, psup\\n\\n\\ndef _label(x):\\n    x = str(x).strip().lower()\\n    return 1 if x in (\\"1\\", \\"1.0\\", \\"đúng\\", \\"dung\\", \\"true\\", \\"yes\\", \\"y\\") else (0 if x in (\\"0\\", \\"0.0\\", \\"sai\\", \\"false\\", \\"no\\", \\"n\\") else None)\\n\\n\\ndef _read_table(p):\\n    if p.suffix.lower() == \\".xlsx\\":\\n        return pd.read_excel(p, dtype=str, keep_default_na=False, sheet_name=0)\\n    for enc in (\\"utf-8-sig\\", \\"cp1252\\"):                # Excel có thể lưu \';\' hoặc mã hoá khác — nhãn chỉ là 1/0/?\\n        try:\\n            return pd.read_csv(p, dtype=str, keep_default_na=False, sep=None, engine=\\"python\\", encoding=enc)\\n        except UnicodeDecodeError:\\n            continue\\n    raise ValueError(f\\"không đọc được {p}\\")\\n\\n\\ndef find_label_file():\\n    \\"\\"\\"CFG.annot_file, hoặc relation_annotation*.(csv|xlsx) trong /kaggle/input có ít nhất một nhãn ở annotator_1.\\"\\"\\"\\n    cands = [Path(CFG.annot_file)] if CFG.annot_file else []\\n    if RUN_ON_KAGGLE:\\n        cands += sorted(p for p in Path(\\"/kaggle/input\\").rglob(\\"relation_annotation*\\") if p.suffix.lower() in (\\".csv\\", \\".xlsx\\"))\\n    for p in cands:\\n        try:\\n            df = _read_table(p)\\n            if \\"annotator_1\\" in df and df[\\"annotator_1\\"].map(_label).notna().any():\\n                return p, df\\n        except Exception as e:\\n            print(f\\"⚠ Không đọc được {p}: {type(e).__name__}: {e}\\")\\n    return None, None\\n\\n\\ndef wilson(k, n, z=1.96):\\n    if n == 0:\\n        return float(\\"nan\\"), float(\\"nan\\"), float(\\"nan\\")\\n    p = k / n\\n    d = 1 + z * z / n\\n    c = (p + z * z / (2 * n)) / d\\n    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d\\n    return p, c - h, c + h\\n\\n\\ndef cohen_kappa(a, b):\\n    a, b = np.asarray(a), np.asarray(b)\\n    if len(a) == 0:\\n        return float(\\"nan\\")\\n    po = float((a == b).mean())\\n    pe = float(a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean()))\\n    return (po - pe) / (1 - pe) if pe < 1 else float(\\"nan\\")\\n\\n\\ndef relation_quality(labels):\\n    \\"\\"\\"Precision theo họ và theo ô (họ × mức phổ biến) từ nhãn NGƯỜI chấm. Nhãn cuối = adjudicated nếu có; nếu không:\\n    annotator_1 khi chỉ có một người chấm hoặc hai người đồng ý; hai người bất đồng mà chưa phân xử ⇒ loại khỏi precision.\\"\\"\\"\\n    L = labels.copy()\\n    for c in ANNOT_LABEL_COLS:\\n        L[c + \\"_v\\"] = L[c].map(_label) if c in L else None\\n    final, status = [], []\\n    for a1, a2, ad in zip(L[\\"annotator_1_v\\"], L[\\"annotator_2_v\\"], L[\\"adjudicated_v\\"]):\\n        if not pd.isna(ad):\\n            final.append(ad); status.append(\\"adjudicated\\")\\n        elif pd.isna(a1):\\n            final.append(None); status.append(\\"unlabeled_or_unclear\\")\\n        elif pd.isna(a2):\\n            final.append(a1); status.append(\\"single\\")\\n        elif a1 == a2:\\n            final.append(a1); status.append(\\"agree\\")\\n        else:\\n            final.append(None); status.append(\\"disagree_unresolved\\")\\n    L[\\"final\\"], L[\\"label_status\\"] = final, status\\n    both = L[L[\\"annotator_1_v\\"].notna() & L[\\"annotator_2_v\\"].notna()]\\n    rows = []\\n    for keys, sub in [((\\"ALL\\", \\"ALL\\"), L)] + [((f, \\"ALL\\"), g) for f, g in L.groupby(\\"family\\")] + \\\\\\n            [((f, s), g) for (f, s), g in L.groupby([\\"family\\", \\"stratum\\"])]:\\n        lab = sub[sub[\\"final\\"].notna()]\\n        k, n = int(lab[\\"final\\"].sum()), len(lab)\\n        p, lo, hi = wilson(k, n)\\n        rows.append(dict(family=keys[0], stratum=keys[1], n_rows=len(sub), n_labeled=n, n_correct=k, precision=p, wilson_lo=lo,\\n                         wilson_hi=hi, n_unclear_or_unlabeled=int((sub[\\"label_status\\"] == \\"unlabeled_or_unclear\\").sum()),\\n                         n_disagree_unresolved=int((sub[\\"label_status\\"] == \\"disagree_unresolved\\").sum()),\\n                         verified=bool(keys[1] != \\"ALL\\" and n > 0 and lo >= CFG.prec_min_lower)))\\n    q = pd.DataFrame(rows)\\n    agree = dict(n_double=len(both), kappa=cohen_kappa(both[\\"annotator_1_v\\"].astype(int), both[\\"annotator_2_v\\"].astype(int))\\n                 if len(both) else float(\\"nan\\"), raw_agreement=float((both[\\"annotator_1_v\\"] == both[\\"annotator_2_v\\"]).mean())\\n                 if len(both) else float(\\"nan\\"))\\n    return q, agree, L"}, {"cell_type": "code", "execution_count": null, "id": "c024", "metadata": {}, "outputs": [], "source": "# --- Kiểm chứng TỰ ĐỘNG chất lượng quan hệ (không cần người chấm): đồng thuận với một parser phụ thuộc cú pháp ĐỘC LẬP. ---\\n# Parser của đồ thị là heuristic POS (v2). spaCy (dependency parse, mô hình thống kê khác hẳn) trích lại quan hệ CÙNG schema trên\\n# CHÍNH caption TRAIN đó; một quan hệ được \\"xác nhận\\" nếu spaCy cũng trích ra nó. Đây là nhãn SILVER (đồng thuận giữa hai\\n# phương pháp) — KHÔNG phải precision do người chấm: hai parser có thể cùng sai. Chỉ caption TRAIN (mục đích \'agreement\').\\nimport importlib\\n\\nCaptionStore.ALLOWED[\\"agreement\\"] = {\\"train\\"}\\n_NLP, _DEP_CACHE, DEP_STATUS = None, {}, \\"chưa nạp\\"\\nAGREE, AGREE_COVER = None, None                 # gán ở cell chạy từ vựng (chỉ TRAIN)\\n_NOUN_POS, _SUBJ, _OBJ = (\\"NOUN\\", \\"PROPN\\"), (\\"nsubj\\",), (\\"dobj\\", \\"obj\\", \\"nsubjpass\\")\\n\\n\\ndef dep_parser():\\n    \\"\\"\\"Nạp spaCy + en_core_web_sm (tải nếu thiếu và có Internet). Không được ⇒ None (kiểm chứng tự động = NOT RUN).\\"\\"\\"\\n    global _NLP, DEP_STATUS\\n    if _NLP is None:\\n        try:\\n            spacy = ensure_package(\\"spacy\\", \\"spacy\\")\\n            try:\\n                _NLP = spacy.load(CFG.dep_model, disable=[\\"ner\\", \\"lemmatizer\\"])\\n            except OSError:\\n                if not INTERNET:\\n                    raise\\n                r = subprocess.run([sys.executable, \\"-m\\", \\"spacy\\", \\"download\\", CFG.dep_model], capture_output=True, text=True)\\n                if r.returncode != 0:\\n                    raise RuntimeError(r.stderr[-500:])\\n                importlib.invalidate_caches()\\n                _NLP = spacy.load(CFG.dep_model, disable=[\\"ner\\", \\"lemmatizer\\"])\\n            DEP_STATUS = f\\"spaCy {spacy.__version__} / {CFG.dep_model} {_NLP.meta.get(\'version\')}\\"\\n        except Exception as e:\\n            _NLP, DEP_STATUS = False, f\\"NOT RUN — không nạp được spaCy/{CFG.dep_model}: {type(e).__name__}: {str(e)[:200]}\\"\\n        ENV_INFO[\\"dep_parser\\"] = DEP_STATUS\\n    return _NLP or None\\n\\n\\ndef _nkey(t):\\n    return ckey(lemma(t.text.lower(), \\"n\\"), \\"n\\")\\n\\n\\ndef _vkey(t):\\n    return ckey(lemma(t.text.lower(), \\"v\\"), \\"v\\")\\n\\n\\ndef _with_conj(t):\\n    return [t] + [c for c in t.conjuncts if c.pos_ in _NOUN_POS]\\n\\n\\ndef _subjects(v, depth=0):\\n    s = [c for c in v.children if c.dep_ in _SUBJ and c.pos_ in _NOUN_POS]\\n    if not s and depth < 3:\\n        if v.dep_ in (\\"conj\\", \\"xcomp\\", \\"advcl\\") and v.head.pos_ in (\\"VERB\\", \\"AUX\\"):\\n            return _subjects(v.head, depth + 1)                  # \\"sitting and wagging\\", \\"tries to ride\\"\\n        if v.dep_ in (\\"acl\\", \\"relcl\\") and v.head.pos_ in _NOUN_POS:\\n            return _with_conj(v.head)                            # \\"a lady making a dish\\"\\n    return [x for t in s for x in _with_conj(t)]\\n\\n\\ndef dep_relations(doc):\\n    \\"\\"\\"Quan hệ theo CÙNG schema với parser heuristic: agent {(danh, động)}, object {(danh, động)},\\n    svo {(chủ, đối tượng, động từ)}, prep {(danh, danh, giới từ)} — giới từ phải gắn vào DANH TỪ (không vào động từ).\\"\\"\\"\\n    out = dict(agent=set(), object=set(), svo=set(), prep=set())\\n    for t in doc:\\n        if t.pos_ == \\"VERB\\":\\n            v = lemma(t.text.lower(), \\"v\\")\\n            subs = _subjects(t)\\n            objs = [x for c in t.children if c.dep_ in _OBJ and c.pos_ in _NOUN_POS for x in _with_conj(c)]\\n            pobjs = [(p.text, x) for p in t.children if p.dep_ == \\"prep\\" for c in p.children\\n                     if c.dep_ == \\"pobj\\" and c.pos_ in _NOUN_POS for x in _with_conj(c)]\\n            out[\\"agent\\"] |= {(_nkey(s), _vkey(t)) for s in subs}\\n            out[\\"object\\"] |= {(_nkey(o), _vkey(t)) for o in objs}\\n            out[\\"svo\\"] |= {(_nkey(s), _nkey(o), v) for s in subs for o in objs + [x for _, x in pobjs]}\\n        elif t.dep_ == \\"prep\\" and t.head.pos_ in _NOUN_POS:\\n            for c in t.children:\\n                if c.dep_ == \\"pobj\\" and c.pos_ in _NOUN_POS:\\n                    out[\\"prep\\"] |= {(_nkey(t.head), _nkey(x), t.text.lower()) for x in _with_conj(c)}\\n    return out\\n\\n\\ndef dep_parse_all(texts):\\n    nlp = dep_parser()\\n    if nlp is None:\\n        return False\\n    todo = sorted({x.lower() for x in texts} - set(_DEP_CACHE))\\n    for txt, doc in zip(todo, nlp.pipe(todo, batch_size=512)):\\n        _DEP_CACHE[txt] = dep_relations(doc)\\n    return True\\n\\n\\ndef triple_confirmed(t, dep):\\n    \\"\\"\\"Quan hệ heuristic (nguồn, họ, đích, vị từ) được spaCy xác nhận trên cùng caption?\\n    svo: cùng (chủ, đối tượng) và CÙNG động từ đầu của vị từ; prep: cùng (danh, danh) và giới từ thuộc vị từ.\\"\\"\\"\\n    s, rel, o, p = t\\n    if rel == \\"agent_of\\":\\n        return (s, o) in dep[\\"agent\\"]\\n    if rel == \\"object_of\\":\\n        return (s, o) in dep[\\"object\\"]\\n    if rel == \\"svo\\":\\n        return (s, o, p.split()[0]) in dep[\\"svo\\"]\\n    return any((s, o, w) in dep[\\"prep\\"] for w in p.split())\\n\\n\\ndef build_agreement(store, split, vocab):\\n    \\"\\"\\"AGREE[(họ, nguồn, đích, vị từ)] = [số caption TRAIN spaCy xác nhận, số caption TRAIN có quan hệ]; cùng điều kiện từ\\n    vựng với build_relations_v2. Kèm độ phủ ngược: tỉ lệ quan hệ spaCy (trong từ vựng) mà heuristic cũng trích được.\\"\\"\\"\\n    vset = set(vocab[\\"terms\\"])\\n    caps = store.by_video(split[\\"train\\"], \\"agreement\\")\\n    texts = [c for v in split[\\"train\\"] for c in caps[v]]\\n    if not dep_parse_all(texts):\\n        return None, None\\n    agree, cover = defaultdict(lambda: [0, 0]), Counter()\\n    for c in texts:\\n        dep = _DEP_CACHE[c.lower()]\\n        heur = {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}\\n        for t in heur:\\n            a = agree[(t[1], t[0], t[2], t[3])]\\n            a[0] += int(triple_confirmed(t, dep)); a[1] += 1\\n        hk = {(t[1], t[0], t[2]) for t in heur}\\n        for fam, rel in ((\\"agent\\", \\"agent_of\\"), (\\"object\\", \\"object_of\\")):\\n            for s, o in dep[fam]:\\n                if s in vset and o in vset:\\n                    cover[(rel, \\"total\\")] += 1; cover[(rel, \\"found\\")] += int((rel, s, o) in hk)\\n        for fam in (\\"svo\\", \\"prep\\"):\\n            for s, o, _ in dep[fam]:\\n                if s in vset and o in vset and s != o:\\n                    cover[(fam, \\"total\\")] += 1; cover[(fam, \\"found\\")] += int((fam, s, o) in hk)\\n    return {k: tuple(v) for k, v in agree.items()}, dict(cover)\\n\\n\\ndef agreement_rate(rel, s, o, p):\\n    a = AGREE.get((rel, s, o, p)) if AGREE else None\\n    return a[0] / a[1] if a and a[1] else float(\\"nan\\")\\n\\n\\ndef agreement_quality(agree, cover, sup):\\n    \\"\\"\\"Bảng SILVER theo họ và theo ô (họ × mức phổ biến): tỉ lệ caption-instance được parser độc lập xác nhận (Wilson 95%).\\"\\"\\"\\n    rows = []\\n    cells = defaultdict(lambda: [0, 0])\\n    for (rel, s, o, p), (k, n) in agree.items():\\n        st = support_stratum(sup.get((s, rel, o, p), 0))\\n        for key in ((rel, \\"ALL\\"), (rel, st), (\\"ALL\\", \\"ALL\\")):\\n            cells[key][0] += k; cells[key][1] += n\\n    for (rel, st), (k, n) in sorted(cells.items()):\\n        p_, lo, hi = wilson(k, n)\\n        cv = (cover.get((rel, \\"found\\"), 0) / cover[(rel, \\"total\\")]) if st == \\"ALL\\" and cover.get((rel, \\"total\\")) else float(\\"nan\\")\\n        rows.append(dict(kind=\\"silver_parser_agreement\\", family=rel, stratum=st, n_instances=n, n_confirmed=k, agreement=p_,\\n                         wilson_lo=lo, wilson_hi=hi, heuristic_covers_spacy=cv,\\n                         note=\\"đồng thuận heuristic ↔ spaCy trên caption TRAIN; KHÔNG phải precision do người chấm\\"))\\n    return pd.DataFrame(rows)"}, {"cell_type": "markdown", "id": "c025", "metadata": {}, "source": "## 5. Đặc tả đồ thị v3 (schema)\\n\\n**Node.** Canonical inventory **cố định**: video MSVD + video tập ngoài, lớp `(lemma, POS)` từ train, khái niệm theo synset WordNet. Ablation chỉ **tắt cạnh**.\\n\\n**Cạnh.** Mỗi bản ghi cạnh trong `graph_edges_v3.csv` gồm `(src, relation_family, relation, predicate, dst, direction, weight, confidence, source, status)`. Mỗi dòng là **một** quan hệ: một cặp có nhiều vị từ thành nhiều dòng.\\n* Trường `predicate` **không bao giờ** chứa tên loại quan hệ; có kiểm tra G2 cho điều này.\\n* `weight` là support (tần suất), **không** phải độ tin cậy ngữ nghĩa.\\n* `confidence` là `silver <tỉ lệ> (k/n)`: tỉ lệ caption train của quan hệ được parser độc lập xác nhận. Nếu có nhãn người chấm thì là `human <precision>`; không có cả hai thì là `unverified`.\\n* Cạnh bị loại vẫn được ghi với `status = excluded:<lý do>`.\\n\\n**Mẫu đường đi.** Gồm 12 mẫu của v2 cộng `P1m_svo` và `P1m_prep`: đường đi `video → a –f[p]→ b` chỉ được tính khi truy vấn nhắc tới vị từ `p`."}, {"cell_type": "code", "execution_count": null, "id": "c026", "metadata": {}, "outputs": [], "source": "FAMILIES = (\\"agent\\", \\"object\\", \\"svo\\", \\"prep\\")\\nFAMILY_REL = {\\"agent\\": \\"agent_of\\", \\"object\\": \\"object_of\\", \\"svo\\": \\"svo\\", \\"prep\\": \\"prep\\"}\\n_SCENE_SRC = \\"caption TRAIN (POS parser heuristic) — đếm số video TRAIN hỗ trợ; giữ nếu ≥ scene_min_count video, ≤ scene_top_k cạnh / (nguồn, loại)\\"\\n_SCENE_W = (\\"support weight = min(1, support / df_train(nguồn)); chiều _inv: min(1, support / df_train(đích)). \\"\\n            \\"Đây là TẦN SUẤT, KHÔNG phải độ tin cậy ngữ nghĩa\\")\\n_CONF_REL = \\"precision của ô (họ × mức phổ biến) theo nhãn NGƯỜI chấm (Wilson 95%); không có nhãn ⇒ \'unverified\'\\"\\n_PRED_VERB = \\"lemma động từ = CHÍNH node đích/nguồn động từ ⇒ không mang thông tin riêng của cạnh; ghi lại, KHÔNG đưa vào message\\"\\n_PRED_FREE = (\\"phân phối vị từ của cặp (vị từ → số video TRAIN); embedding vị từ = CLIP-text đóng băng theo prompt \\"\\n              \\"(pred_mode \'clip\') hoặc + MLP dư dùng chung (pred_mode \'clip_adapt\'); vào message của GNN và mẫu đường đi P1m\\")\\nREL_SPEC = {\\n    \\"contains\\":      dict(src=\\"video\\", tgt=\\"class\\", direction=\\"video → lớp được gắn\\", family=\\"contains\\", inverse=None,\\n                          source=\\"CLIP zero-shot trên frame của CHÍNH video × từ vựng train (tagger chọn trên val)\\",\\n                          weight=\\"1 (binary) | độ tin cậy tagger đã chuẩn hoá (contains = conf, chọn trên val)\\",\\n                          predicate=\\"—\\", confidence=\\"điểm tagger CLIP (chỉ khi contains = conf)\\",\\n                          split=\\"frame mọi video; từ vựng + thống kê chuẩn hoá: train\\", usage=\\"gnn+path\\"),\\n    \\"contains_inv\\":  dict(src=\\"class\\", tgt=\\"video\\", direction=\\"lớp → video chứa nó\\", family=\\"contains\\", inverse=\\"contains\\",\\n                          source=\\"chiều ngược của contains\\", weight=\\"= contains\\", predicate=\\"—\\", confidence=\\"= contains\\",\\n                          split=\\"như contains\\", usage=\\"gnn\\"),\\n    \\"agent_of\\":      dict(src=\\"class:noun\\", tgt=\\"class:verb\\", direction=\\"chủ thể → hành động\\", family=\\"agent\\", inverse=None,\\n                          source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"agent_of_inv\\":  dict(src=\\"class:verb\\", tgt=\\"class:noun\\", direction=\\"hành động → chủ thể (done_by)\\", family=\\"agent\\",\\n                          inverse=\\"agent_of\\", source=\\"chiều ngược của agent_of\\", weight=_SCENE_W, predicate=_PRED_VERB,\\n                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"object_of\\":     dict(src=\\"class:noun\\", tgt=\\"class:verb\\", direction=\\"đồ vật → hành động tác động lên nó\\", family=\\"object\\",\\n                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL,\\n                          split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"object_of_inv\\": dict(src=\\"class:verb\\", tgt=\\"class:noun\\", direction=\\"hành động → đồ vật (acts_on)\\", family=\\"object\\",\\n                          inverse=\\"object_of\\", source=\\"chiều ngược của object_of\\", weight=_SCENE_W, predicate=_PRED_VERB,\\n                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"svo\\":           dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"chủ thể –vị từ→ đối tượng\\", family=\\"svo\\",\\n                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n                          split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"svo_inv\\":       dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"đối tượng –bị vị từ bởi→ chủ thể\\", family=\\"svo\\",\\n                          inverse=\\"svo\\", source=\\"chiều ngược của svo\\", weight=_SCENE_W, predicate=_PRED_FREE,\\n                          confidence=_CONF_REL, split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"prep\\":          dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"danh từ –giới từ→ danh từ\\", family=\\"prep\\",\\n                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n                          split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"prep_inv\\":      dict(src=\\"class:noun\\", tgt=\\"class:noun\\", direction=\\"chiều ngược của prep\\", family=\\"prep\\", inverse=\\"prep\\",\\n                          source=\\"chiều ngược của prep\\", weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,\\n                          split=\\"train\\", usage=\\"gnn+path\\"),\\n    \\"related\\":       dict(src=\\"class\\", tgt=\\"class\\", direction=\\"đối xứng (lưu cả hai chiều)\\", family=\\"related\\", inverse=None,\\n                          source=\\"NPMI ≥ ngưỡng của nhãn tagger trên video TRAIN; ≥ min_count; ≤ cooc_top_k / nguồn | biến thể related=conceptnet: ConceptNet 5.7 (tri thức NGOÀI, quan hệ dương en–en, hai đầu ∈ từ vựng train, ≤ cooc_top_k / nguồn theo weight)\\",\\n                          weight=\\"P(đích | nguồn) = count / df_train(nguồn)\\", predicate=\\"—\\",\\n                          confidence=\\"— (đồng xuất hiện ≠ quan hệ ngữ nghĩa trực tiếp)\\", split=\\"train (nhãn tagger từ frame)\\",\\n                          usage=\\"gnn+path\\"),\\n    \\"isa\\":           dict(src=\\"class\\", tgt=\\"concept|class (cùng POS)\\", direction=\\"lớp → khái niệm tổng quát hơn\\",\\n                          family=\\"isa\\", inverse=None, source=\\"WordNet (hypernym theo biến thể taxonomy)\\", weight=\\"1\\",\\n                          predicate=\\"—\\", confidence=\\"— (sense mơ hồ ⇒ KHÔNG nối)\\",\\n                          split=\\"tri thức ngoài (chọn sense: ngữ cảnh caption TRAIN)\\", usage=\\"gnn+path\\"),\\n    \\"isa_inv\\":       dict(src=\\"concept|class\\", tgt=\\"class\\", direction=\\"khái niệm → lớp cụ thể\\", family=\\"isa\\", inverse=\\"isa\\",\\n                          source=\\"chiều ngược của isa\\", weight=\\"1\\", predicate=\\"—\\", confidence=\\"—\\", split=\\"như isa\\", usage=\\"gnn\\"),\\n}\\nEDGE_TYPES = list(REL_SPEC)\\nETYPE_ID = {t: i for i, t in enumerate(EDGE_TYPES)}\\nOPEN_TYPES = (\\"contains\\", \\"contains_inv\\")            # cổng mở sẵn (như v1)\\nRELATION_TYPES = [t for t in EDGE_TYPES if REL_SPEC[t][\\"family\\"] != \\"contains\\"]\\nPRED_FAMILIES = tuple(CFG.pred_families)             # họ có vị từ tự do (svo, prep)\\nPRED_TYPES = [t for f in PRED_FAMILIES for t in (f, f + \\"_inv\\")]\\nMASKS = {\\n    \\"none\\": (),\\n    \\"no_scene\\": tuple(t for t in EDGE_TYPES if REL_SPEC[t][\\"family\\"] in FAMILIES),\\n    \\"no_related\\": (\\"related\\",),\\n    \\"no_taxonomy\\": (\\"isa\\", \\"isa_inv\\"),\\n    \\"contains_only\\": tuple(RELATION_TYPES),\\n    \\"no_roles\\": (\\"agent_of\\", \\"agent_of_inv\\", \\"object_of\\", \\"object_of_inv\\"),   # Level 4(b): họ cạnh vai\\n    \\"no_svo_prep\\": tuple(PRED_TYPES),                                          # Level 4(b): họ cạnh có vị từ tự do\\n}\\n# P1m_f: đường đi video → lớp a –f[vị từ p]→ lớp b CHỈ được tính khi truy vấn nhắc tới p (khớp vị từ); dùng ở path_mode \'pred_match\'\\nPATTERNS = ([\\"P0_contains\\"] + [f\\"P1_{f}\\" for f in FAMILIES] + [\\"P2_isa\\", \\"P3_related\\"]\\n            + [f\\"P4_{f}_isa\\" for f in FAMILIES] + [\\"P5_related_isa\\"] + [f\\"P1m_{f}\\" for f in PRED_FAMILIES])\\nPATTERN_LEN = [1] + [2] * 4 + [2, 2] + [3] * 4 + [3] + [2] * len(PRED_FAMILIES)\\nPIDX = {p: i for i, p in enumerate(PATTERNS)}\\nREL_TEXT = {\\"contains\\": \\"chứa\\", \\"agent_of\\": \\"là_chủ_thể_của\\", \\"agent_of_inv\\": \\"do\\", \\"object_of\\": \\"là_đối_tượng_của\\",\\n            \\"object_of_inv\\": \\"tác_động_lên\\", \\"svo\\": \\"{pred}\\", \\"svo_inv\\": \\"bị_{pred}_bởi\\", \\"prep\\": \\"{pred}\\",\\n            \\"prep_inv\\": \\"có_{pred}\\", \\"related\\": \\"đồng_xuất_hiện\\", \\"isa\\": \\"là_một\\"}"}, {"cell_type": "markdown", "id": "c027", "metadata": {}, "source": "### 5b. Taxonomy (giữ nguyên cách dựng của v2)\\nKhái niệm được khoá theo synset WordNet (ví dụ `animal.n.01`), nên danh từ và động từ không bao giờ bị trộn với nhau. Có bốn biến thể trên **cùng** một tập node (tập node là hợp của các biến thể):\\n\\n| Biến thể | Cách chọn hypernym |\\n|---|---|\\n| `heuristic` | quy tắc v1, lấy sense đầu tiên |\\n| `filtered` | `heuristic` bỏ các khái niệm \\"hub\\" (hơn 15% video train với tới) và khái niệm danh từ quá nông |\\n| `sense` | chọn sense bằng Extended Lesk trên ngữ cảnh caption **train**; mapping **mơ hồ** thì không nối `isa` và được ghi rõ |\\n| `none` | không có `isa` |\\n\\nMọi mapping (synset, gloss, điểm Lesk, cờ mơ hồ, hub) được xuất ra `graph_taxonomy_audit.csv`.\\n\\nỞ v3, `filtered` thắng val nhưng thua trên test đã nhìn, nên **không** còn là ứng viên. Stage V4 so sánh `none` (mặc định, đơn giản hơn) với taxonomy không lọc (`heuristic`). `heuristic` chỉ được giữ khi thắng theo luật đăng ký trước. Kiểm tra mơ hồ, tương thích danh/động từ, hypernym quá chung, hub và chu trình vẫn được ghi trong audit."}, {"cell_type": "code", "execution_count": null, "id": "c028", "metadata": {}, "outputs": [], "source": "CaptionStore.ALLOWED[\\"taxonomy\\"] = {\\"train\\"}          # chọn sense WordNet theo ngữ cảnh caption TRAIN\\n_WTOK = re.compile(r\\"[a-z]+\\")\\n\\n\\ndef _head(s):\\n    return s.lemmas()[0].name().replace(\\"_\\", \\" \\").lower()\\n\\n\\ndef _noun_targets(s, word):\\n    \\"\\"\\"Quy tắc hypernym danh từ của v1 (cha/ông đủ sâu, nhìn thấy được + BASIC_CATEGORIES) — trả về synset.\\"\\"\\"\\n    hy = []\\n    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n        for depth in range(len(path) - 2, -1, -1):\\n            h = path[depth]\\n            name = _head(h)\\n            near = depth >= len(path) - 3\\n            if name == word or name in GENERIC_CONCEPTS or h in hy:\\n                continue\\n            if (near and depth >= CFG.isa_min_depth and (h.lexname() in VISUAL_LEX or h.lexname() == \\"noun.Tops\\")) \\\\\\n                    or name in BASIC_CATEGORIES:\\n                hy.append(h)\\n    return hy[:5]\\n\\n\\ndef _verb_targets(s, word):\\n    \\"\\"\\"Quy tắc hypernym động từ của v1 (cha/ông, bỏ động từ quá chung).\\"\\"\\"\\n    hy = []\\n    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:\\n        for depth in (len(path) - 2, len(path) - 3):\\n            if depth < 0:\\n                continue\\n            h = path[depth]\\n            name = _head(h)\\n            if name != word and name not in GENERIC_VERBS and \\" \\" not in name and h not in hy:\\n                hy.append(h)\\n    return hy[:3]\\n\\n\\ndef _signature(s):\\n    \\"\\"\\"Extended Lesk: từ trong định nghĩa, ví dụ, lemma của synset + cha + 10 con (đã lemma hoá).\\"\\"\\"\\n    words = set(_WTOK.findall(s.definition().lower()))\\n    for e in s.examples():\\n        words |= set(_WTOK.findall(e.lower()))\\n    for r in [s] + sorted(s.hypernyms(), key=_syn_key) + sorted(s.hyponyms(), key=_syn_key)[:10]:\\n        words |= {l.name().lower() for l in r.lemmas() if \\"_\\" not in l.name()}\\n    return ({lemma(w, \\"n\\") for w in words} | {lemma(w, \\"v\\") for w in words}) - STOP\\n\\n\\ndef build_sense_context(store, split, vset):\\n    \\"\\"\\"Ngữ cảnh của mỗi lớp = các danh từ/động từ khác cùng xuất hiện trong caption TRAIN chứa lớp đó.\\"\\"\\"\\n    caps = store.by_video(split[\\"train\\"], \\"taxonomy\\")\\n    ctx = defaultdict(Counter)\\n    for v in split[\\"train\\"]:\\n        for c in caps[v]:\\n            nouns, verbs, _ = parse_caption_v2(c)\\n            words = set(nouns) | set(verbs)\\n            for k in {ckey(n, \\"n\\") for n in nouns} | {ckey(x, \\"v\\") for x in verbs}:\\n                if k in vset:\\n                    ctx[k].update(words - {k.rsplit(\\"#\\", 1)[0]})\\n    return ctx\\n\\n\\ndef choose_sense(lem, pos, ctx):\\n    wn = _wordnet()\\n    syns = wn.synsets(lem.replace(\\" \\", \\"_\\"), pos=pos)[:CFG.sense_max_candidates] if wn is not None else []\\n    if not syns:\\n        return None, True, []\\n    scores = [sum(math.log1p(ctx[w]) for w in sorted(_signature(s)) if w in ctx) for s in syns]   # thứ tự cố định\\n    order = np.argsort(-np.array(scores), kind=\\"stable\\")\\n    b = scores[order[0]]\\n    s2 = scores[order[1]] if len(syns) > 1 else -1e9\\n    ambiguous = len(syns) > 1 and (b <= 0 or b - s2 < CFG.sense_margin)\\n    return syns[order[0]], ambiguous, [(syns[i].name(), round(scores[i], 2)) for i in order[:3]]\\n\\n\\ndef build_taxonomy_v2(vocab, store, split, train_tags):\\n    \\"\\"\\"Ba biến thể taxonomy trên cùng một tập khái niệm (hợp) ⇒ inventory không đổi giữa các biến thể:\\n    heuristic (= quy tắc v1, sense đầu tiên), filtered (bỏ hub/khái niệm nông), sense (chọn sense theo ngữ cảnh\\n    caption TRAIN; mơ hồ ⇒ KHÔNG nối isa và đánh dấu), none (không isa).\\"\\"\\"\\n    wn = _wordnet()\\n    vset = set(vocab[\\"terms\\"])\\n    ctx = build_sense_context(store, split, vset)\\n    concept_meta, var = {}, {\\"heuristic\\": {}, \\"sense\\": {}, \\"filtered\\": {}, \\"none\\": {}}\\n    rows = []\\n\\n    def target_key(h, pos):\\n        k = ckey(_head(h), pos)\\n        if k in vset:\\n            return k                                   # hypernym trùng một lớp cùng POS ⇒ nối thẳng vào lớp đó\\n        concept_meta.setdefault(h.name(), dict(key=h.name(), lemma=_head(h), pos=pos, synset=h.name(),\\n                                               gloss=h.definition(), depth=h.min_depth(), source=\\"wordnet\\"))\\n        return h.name()\\n\\n    def fallback(lem, pos):\\n        names = (FALLBACK_ISA if pos == \\"n\\" else FALLBACK_VERB_ISA).get(lem, [])\\n        out = []\\n        for nm in names:\\n            k = ckey(nm, pos)\\n            if k not in vset:\\n                k = f\\"{nm}.{pos}.fallback\\"\\n                concept_meta.setdefault(k, dict(key=k, lemma=nm, pos=pos, synset=\\"\\", gloss=\\"\\", depth=-1, source=\\"fallback\\"))\\n            out.append(k)\\n        return out\\n\\n    for k in vocab[\\"terms\\"]:\\n        lem, pos = k.rsplit(\\"#\\", 1)\\n        syns = wn.synsets(lem.replace(\\" \\", \\"_\\"), pos=pos) if wn is not None else []\\n        tf = _noun_targets if pos == \\"n\\" else _verb_targets\\n        heur = [target_key(h, pos) for h in tf(syns[0], lem)] if syns else fallback(lem, pos)\\n        s_syn, amb, scores = choose_sense(lem, pos, ctx[k])\\n        sense = [] if (amb or s_syn is None) else [target_key(h, pos) for h in tf(s_syn, lem)]\\n        heur = list(dict.fromkeys(t for t in heur if t != k))\\n        sense = list(dict.fromkeys(t for t in sense if t != k))\\n        var[\\"heuristic\\"][k], var[\\"sense\\"][k] = heur, sense\\n        rows.append(dict(class_key=k, pos=pos, train_df=vocab[\\"df\\"][k], n_synsets=len(syns),\\n                         heuristic_synset=syns[0].name() if syns else \\"\\", heuristic_gloss=syns[0].definition() if syns else \\"\\",\\n                         sense_synset=s_syn.name() if s_syn is not None else \\"\\", sense_ambiguous=bool(amb),\\n                         sense_differs=bool(syns) and s_syn is not None and s_syn != syns[0], lesk_top3=scores,\\n                         heuristic_targets=heur, sense_targets=sense))\\n    # filtered: bỏ khái niệm \'hub\' (với tới > tax_hub_frac video TRAIN) và khái niệm danh từ quá nông\\n    tr = list(split[\\"train\\"])\\n    reach = defaultdict(set)\\n    for v in tr:\\n        for c in train_tags.get(v, []):\\n            for t in var[\\"heuristic\\"].get(c, []):\\n                reach[t].add(v)\\n    hub = {t for t, s in reach.items() if len(s) / max(len(tr), 1) > CFG.tax_hub_frac}\\n    shallow = {t for t, m in concept_meta.items() if m[\\"pos\\"] == \\"n\\" and 0 <= m[\\"depth\\"] < CFG.tax_min_depth_filtered}\\n    for row in rows:\\n        k = row[\\"class_key\\"]\\n        var[\\"filtered\\"][k] = [t for t in var[\\"heuristic\\"][k] if t not in hub and t not in shallow]\\n        row[\\"filtered_targets\\"] = var[\\"filtered\\"][k]\\n        row[\\"dropped_hub\\"] = [t for t in var[\\"heuristic\\"][k] if t in hub]\\n        row[\\"dropped_shallow\\"] = [t for t in var[\\"heuristic\\"][k] if t in shallow]\\n    var[\\"none\\"] = {k: [] for k in vocab[\\"terms\\"]}\\n    hub_rows = sorted(((t, len(s), len(s) / max(len(tr), 1)) for t, s in reach.items()), key=lambda x: -x[1])\\n    cycles = {}\\n    for name, tax in var.items():                       # chu trình trên các cạnh lớp → lớp\\n        nxt = {k: [t for t in ts if t in vset] for k, ts in tax.items()}\\n        seen, stack, cyc = set(), set(), 0\\n\\n        def dfs(u):\\n            nonlocal cyc\\n            seen.add(u); stack.add(u)\\n            for w in nxt.get(u, []):\\n                if w in stack:\\n                    cyc += 1\\n                elif w not in seen:\\n                    dfs(w)\\n            stack.discard(u)\\n        for u in nxt:\\n            if u not in seen:\\n                dfs(u)\\n        cycles[name] = cyc\\n    stats = dict(n_concepts=len(concept_meta), hubs=[dict(target=t, train_videos=n, frac=round(f, 3)) for t, n, f in hub_rows[:25]],\\n                 n_hub=len(hub), n_shallow=len(shallow), cycles=cycles,\\n                 n_ambiguous=int(sum(r[\\"sense_ambiguous\\"] for r in rows)),\\n                 n_sense_differs=int(sum(r[\\"sense_differs\\"] for r in rows)),\\n                 isa_edges={n: int(sum(len(v) for v in t.values())) for n, t in var.items()})\\n    return var, concept_meta, pd.DataFrame(rows), stats"}, {"cell_type": "markdown", "id": "c029", "metadata": {}, "source": "### 5c. Đồ thị v3 và hai đối chứng ngẫu nhiên có ràng buộc\\n**`GraphV3`** giữ topology cặp của v2, nên relation-family-only (G2) trùng với `v2_base`. Mỗi cặp (nguồn, họ, đích) mang **phân phối vị từ** `{vị từ: số video train}`.\\n\\n**Chính sách cạnh:**\\n* `all`: như v2;\\n* `support`: chỉ vị từ có ≥ 5 video train;\\n* `agreement`: chỉ vị từ của quan hệ được parser độc lập xác nhận ở ≥ 50% caption train của nó (kiểm chứng tự động);\\n* `precision`: chỉ ô đã kiểm chứng bằng nhãn người (tuỳ chọn).\\n\\nMột cặp không còn vị từ hợp lệ thì bị loại và lưu lại.\\n\\n**Đối chứng:**\\n* **Rewiring** (G5): directed double-edge swap trong từng họ và nhóm kiểu node. Phép này giữ bậc vào/ra, trọng số và **đa tập phân phối vị từ**, vì vị từ đi theo cạnh. `contains` không đổi.\\n* **Hoán vị vị từ** (G4): hoán vị phân phối vị từ giữa các cạnh **cùng họ** svo/prep. Topology, trọng số và bậc giống hệt; chỉ việc ghép vị từ ↔ cạnh bị phá.\\n\\n`path_mode` gồm `weighted`, `count`, `degnorm`, `dedup` như v2, và `pred_match` (= `weighted` cộng mẫu P1m khớp vị từ). Engine, `compute_R` và phần giải thích dùng **chung** các ma trận này."}, {"cell_type": "code", "execution_count": null, "id": "c030", "metadata": {}, "outputs": [], "source": "class NodeInventory:\\n    \\"\\"\\"Tập node CỐ ĐỊNH cho mọi cấu hình so sánh được: video | lớp (lemma, POS) | khái niệm (synset).\\n    Ablation chỉ tắt cạnh ⇒ node id, số node, kích thước embedding không đổi.\\"\\"\\"\\n\\n    def __init__(self, records, vocab, concept_meta):\\n        self.n_vid = len(records)\\n        self.class_keys = list(vocab[\\"terms\\"])\\n        self.concept_keys = sorted(concept_meta)\\n        self.n_cls, self.n_con = len(self.class_keys), len(self.concept_keys)\\n        self.n_nodes = self.n_vid + self.n_cls + self.n_con\\n        self.key2id = {k: self.n_vid + i for i, k in enumerate(self.class_keys)}\\n        self.key2id.update({k: self.n_vid + self.n_cls + i for i, k in enumerate(self.concept_keys)})\\n        self.id2key = {i: k for k, i in self.key2id.items()}\\n        self.node_type = np.array([\\"video\\"] * self.n_vid + [\\"class\\"] * self.n_cls + [\\"concept\\"] * self.n_con)\\n        self.pos = np.array([\\"\\"] * self.n_vid + [k.rsplit(\\"#\\", 1)[1] for k in self.class_keys]\\n                            + [concept_meta[k][\\"pos\\"] for k in self.concept_keys])\\n        self.lemma = {**{self.key2id[k]: vocab[\\"lemma\\"][k] for k in self.class_keys},\\n                      **{self.key2id[k]: concept_meta[k][\\"lemma\\"] for k in self.concept_keys}}\\n        self.name = {i: (records[i][\\"video_id\\"] if i < self.n_vid else self.id2key[i]) for i in range(self.n_nodes)}\\n        self.concepts_by_lemma = defaultdict(list)\\n        for k in self.concept_keys:\\n            self.concepts_by_lemma[(concept_meta[k][\\"lemma\\"], concept_meta[k][\\"pos\\"])].append(self.key2id[k])\\n        self.concept_meta = concept_meta\\n        self.sha1 = sha1_of([self.n_vid, self.class_keys, self.concept_keys])\\n\\n    def semantic_type(self, i):\\n        t = self.node_type[i]\\n        return \\"video\\" if t == \\"video\\" else ({\\"n\\": \\"noun\\", \\"v\\": \\"verb\\"}[self.pos[i]] + (\\"_concept\\" if t == \\"concept\\" else \\"\\"))\\n\\n    def is_class(self, i):\\n        return self.n_vid <= i < self.n_vid + self.n_cls\\n\\n\\ndef contains_calibration(tag_scores, train_vids, inv):\\n    \\"\\"\\"Thống kê chuẩn hoá điểm tagger của các nhãn ĐÃ GẮN trên video TRAIN, tách danh từ / động từ.\\"\\"\\"\\n    by = defaultdict(list)\\n    for v in train_vids:\\n        for k, s in tag_scores[v].items():\\n            by[k.rsplit(\\"#\\", 1)[1]].append(s)\\n    return {p: dict(mu=float(np.mean(x)), sd=float(np.std(x) + 1e-6), lo=float(np.percentile(x, 1)),\\n                    hi=float(np.percentile(x, 99))) for p, x in by.items()}\\n\\n\\ndef _conf_weight(s, pos, cal, mode):\\n    c = cal[pos]\\n    if mode == \\"sigmoid\\":                      # z bị chặn ±3 để hạn chế ngoại lai\\n        w = 1.0 / (1.0 + math.exp(-float(np.clip((s - c[\\"mu\\"]) / c[\\"sd\\"], -3, 3))))\\n    else:                                      # minmax giữa phân vị 1–99 của train\\n        w = (s - c[\\"lo\\"]) / max(c[\\"hi\\"] - c[\\"lo\\"], 1e-6)\\n    return float(np.clip(w, CFG.conf_floor, 1.0))\\n\\n\\ndef _csr(n_rows, n_cols, rows, cols, vals):\\n    if len(rows) == 0:\\n        return sp.csr_matrix((n_rows, n_cols), dtype=np.float32)\\n    M = sp.csr_matrix((np.asarray(vals, np.float32), (np.asarray(rows), np.asarray(cols))), shape=(n_rows, n_cols))\\n    M.sum_duplicates()\\n    return M\\n\\n\\ndef _symnorm(M):\\n    if M.nnz == 0:\\n        return M.copy()\\n    dout = np.asarray(M.sum(1)).ravel(); din = np.asarray(M.sum(0)).ravel()\\n    a = np.where(dout > 0, 1 / np.sqrt(np.maximum(dout, 1e-12)), 0)\\n    b = np.where(din > 0, 1 / np.sqrt(np.maximum(din, 1e-12)), 0)\\n    return (sp.diags(a) @ M @ sp.diags(b)).tocsr().astype(np.float32)\\n\\n\\ndef _maxprod(C, B):\\n    \\"\\"\\"out[v, c] = max_o C[v, o] · B[o, c] (thưa) — tổ hợp \'max\' thay cho tổng (path_dedup).\\"\\"\\"\\n    C, B = C.tocsr(), B.tocsr()\\n    rows = []\\n    for v in range(C.shape[0]):\\n        sl = slice(C.indptr[v], C.indptr[v + 1])\\n        idx, val = C.indices[sl], C.data[sl]\\n        if len(idx) == 0:\\n            rows.append(sp.csr_matrix((1, B.shape[1]), dtype=np.float32))\\n            continue\\n        sub = (sp.diags(val) @ B[idx]).tocsr()\\n        rows.append(sp.csr_matrix(sub.max(axis=0)))\\n    return sp.vstack(rows).tocsr()\\n\\n\\n\\n\\nPATH_TYPES = [\\"agent_of\\", \\"agent_of_inv\\", \\"object_of\\", \\"object_of_inv\\", \\"svo\\", \\"svo_inv\\", \\"prep\\", \\"prep_inv\\", \\"related\\", \\"isa\\"]\\nREL_POLICIES = (\\"all\\", \\"support\\", \\"agreement\\", \\"precision\\")\\n\\n\\ndef predicate_tokens(p):\\n    return tuple(p.split())\\n\\n\\nclass GraphV3:\\n    \\"\\"\\"Đồ thị dị thể có kiểu, có hướng, có trọng số trên NodeInventory cố định; cạnh quan hệ mang VỊ TỪ.\\n    Cạnh quan hệ = cặp (nguồn, họ, đích) như v2 (cùng topology) + phân phối vị từ của cặp {vị từ: số video TRAIN}.\\n      rel_policy : \'all\' | \'support\' (vị từ có ≥ pred_min_support video TRAIN) | \'agreement\' (≥ agree_min caption TRAIN được\\n                   parser độc lập xác nhận — kiểm chứng TỰ ĐỘNG) | \'precision\' (ô đã kiểm chứng bằng nhãn người, tuỳ chọn)\\n                   — cặp không còn vị từ hợp lệ bị LOẠI và lưu trong self.excluded\\n      path_mode  : \'weighted\' | \'count\' | \'degnorm\' | \'dedup\' | \'pred_match\' (= weighted + mẫu P1m khớp vị từ truy vấn)\\n      rewire_seed: constrained rewiring (vị từ, trọng số đi theo cạnh) ; pred_shuffle_seed: hoán vị phân phối vị từ giữa\\n                   các cạnh CÙNG họ (topology, trọng số giữ nguyên) ; related_cfg: (min_count, min_npmi).\\"\\"\\"\\n\\n    def __init__(self, inv, records, train_vids, relations, taxonomy, *, tag_scores=None, contains=\\"binary\\",\\n                 conf_mode=\\"sigmoid\\", conf_cal=None, rel_policy=\\"all\\", path_mode=\\"weighted\\", masked=(),\\n                 rewire_seed=None, pred_shuffle_seed=None, tags_override=None, related_cfg=None, verified_cells=None, name=\\"\\"):\\n        assert rel_policy in REL_POLICIES, rel_policy\\n        if rel_policy == \\"precision\\" and verified_cells is None:\\n            raise ValueError(\\"rel_policy = precision cần nhãn người chấm (verified_cells) — NOT RUN khi chưa có nhãn\\")\\n        if rel_policy == \\"agreement\\" and not AGREE:\\n            raise ValueError(\\"rel_policy = agreement cần kiểm chứng tự động (spaCy) — NOT RUN khi không nạp được parser\\")\\n        self.inv, self.name, self.path_mode, self.masked = inv, name, path_mode, tuple(sorted(set(masked)))\\n        self.contains_mode, self.conf_mode, self.rel_policy = contains, conf_mode, rel_policy\\n        self.rewire_seed, self.pred_shuffle_seed = rewire_seed, pred_shuffle_seed\\n        self.related_cfg = tuple(related_cfg or (CFG.cooc_min_count, CFG.cooc_min_npmi))\\n        n, N = inv.n_nodes, inv.n_vid\\n        self.n_nodes, self.n_vid = n, N\\n        train_set = set(int(v) for v in train_vids)\\n        tr_rows = np.array(sorted(train_set))\\n        n_tr = max(len(tr_rows), 1)\\n        tags = tags_override if tags_override is not None else {r[\\"vid\\"]: r[\\"tags\\"] for r in records}\\n        freq = Counter(t for v in train_set for t in tags[v])\\n        self.active = {inv.key2id[k] for k, c in freq.items() if c >= CFG.min_tag_freq and k in inv.key2id}\\n\\n        # ---------- contains (video → lớp), từ nhãn tagger của chính video ----------\\n        rows, cols, vals, self.vid_tags = [], [], [], []\\n        for v in range(N):\\n            ids = []\\n            for k in tags[v]:\\n                i = inv.key2id.get(k)\\n                if i is None or i not in self.active:\\n                    continue\\n                w = 1.0 if contains == \\"binary\\" else _conf_weight(tag_scores[v][k], k.rsplit(\\"#\\", 1)[1], conf_cal, conf_mode)\\n                rows.append(v); cols.append(i); vals.append(w); ids.append(i)\\n            self.vid_tags.append(sorted(ids))\\n        self.A_con = _csr(N, n, rows, cols, vals)\\n        Cb = (self.A_con > 0).astype(np.float32)\\n        cls_df = np.asarray(Cb[tr_rows].sum(0)).ravel()\\n        self.cls_df = cls_df\\n\\n        # ---------- quan hệ từ caption TRAIN, TÁCH theo họ, giữ phân phối vị từ ----------\\n        sup, preds = defaultdict(Counter), defaultdict(Counter)\\n        for v in train_set:\\n            seen = set()\\n            for s, rel, o, pred in relations.get(v, []):\\n                a, b = inv.key2id.get(s), inv.key2id.get(o)\\n                if a is None or b is None or a not in self.active or b not in self.active or a == b:\\n                    continue\\n                preds[(rel, a, b)][pred] += 1             # = số video TRAIN có đúng (nguồn, họ, đích, vị từ)\\n                if (rel, a, b) not in seen:\\n                    sup[rel][(a, b)] += 1; seen.add((rel, a, b))\\n        self.preds = preds\\n\\n        def kept_preds(rel, a, b):\\n            pc = preds[(rel, a, b)]\\n            if rel_policy == \\"support\\":\\n                return {p: c for p, c in pc.items() if PRED_SUPPORT.get((rel, p), 0) >= CFG.pred_min_support}\\n            if rel_policy == \\"precision\\":\\n                return {p: c for p, c in pc.items() if (rel, support_stratum(c)) in verified_cells}\\n            if rel_policy == \\"agreement\\":\\n                ka, kb = inv.id2key[a], inv.id2key[b]\\n                return {p: c for p, c in pc.items() if agreement_rate(rel, ka, kb, p) >= CFG.agree_min}\\n            return dict(pc)\\n\\n        fwd, self.excluded = {}, []\\n        for rel in REL_CATS:\\n            by_src = defaultdict(list)\\n            for (a, b), c in sup[rel].items():\\n                if c < CFG.scene_min_count:\\n                    continue\\n                kp = kept_preds(rel, a, b)\\n                if not kp:\\n                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=dict(preds[(rel, a, b)]),\\n                                              reason=f\\"policy:{rel_policy}\\"))\\n                    continue\\n                by_src[a].append((c, b, kp))\\n            E = [(a, b, c, kp) for a, lst in by_src.items() for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[:CFG.scene_top_k]]\\n            for a, lst in by_src.items():\\n                for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[CFG.scene_top_k:]:\\n                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=kp, reason=\\"top_k\\"))\\n            f = dict(src=np.array([e[0] for e in E], int), dst=np.array([e[1] for e in E], int),\\n                     support=np.array([e[2] for e in E], float))\\n            f[\\"w\\"] = np.minimum(1.0, f[\\"support\\"] / np.maximum(cls_df[f[\\"src\\"]], 1)) if E else np.zeros(0)\\n            f[\\"w_inv\\"] = np.minimum(1.0, f[\\"support\\"] / np.maximum(cls_df[f[\\"dst\\"]], 1)) if E else np.zeros(0)\\n            f[\\"pdist\\"] = [e[3] for e in E]\\n            fwd[rel] = f\\n        self.n_candidate_relations = {r: len(sup[r]) for r in REL_CATS}\\n\\n        # ---------- related: NPMI đồng xuất hiện nhãn tagger trên video TRAIN (đối xứng) ----------\\n        if self.related_cfg[0] == \\"conceptnet\\":                   # tri thức NGOÀI: không dùng caption hay nhãn\\n            und = {}\\n            for e in KG_EDGES:\\n                a_, b_ = inv.key2id.get(e[\\"src\\"]), inv.key2id.get(e[\\"dst\\"])\\n                if a_ is None or b_ is None or a_ == b_ or a_ not in self.active or b_ not in self.active:\\n                    continue\\n                k_ = (min(a_, b_), max(a_, b_))\\n                w_ = max(und.get(k_, (0.0, 0.0))[0], float(e[\\"weight\\"]))\\n                und[k_] = (w_, w_)\\n        else:\\n            min_count, min_npmi = self.related_cfg\\n            cooc = Counter()\\n            for v in train_set:\\n                for a, b in itertools.combinations(sorted(self.vid_tags[v]), 2):\\n                    cooc[(a, b)] += 1\\n            und = {}\\n            for (a, b), c in cooc.items():\\n                if c < min_count:\\n                    continue\\n                p_ab = c / n_tr\\n                npmi = math.log(p_ab / ((cls_df[a] / n_tr) * (cls_df[b] / n_tr))) / max(-math.log(p_ab), 1e-6)\\n                if npmi >= min_npmi:\\n                    und[(a, b)] = (c, npmi)\\n        keep, by_src = set(), defaultdict(list)\\n        for (a, b), (c, s) in und.items():\\n            by_src[a].append((s, b)); by_src[b].append((s, a))\\n        for a, lst in by_src.items():\\n            for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]:\\n                keep.add((min(a, b), max(a, b)))\\n        RE = sorted(keep)\\n        related = dict(src=np.array([e[0] for e in RE], int), dst=np.array([e[1] for e in RE], int),\\n                       count=np.array([und[e][0] for e in RE], float))\\n        if self.related_cfg[0] == \\"conceptnet\\":\\n            related[\\"w_ab\\"] = np.minimum(1.0, related[\\"count\\"] / 2.0) if RE else np.zeros(0)   # weight ConceptNet / 2, chặn 1\\n            related[\\"w_ba\\"] = related[\\"w_ab\\"].copy()\\n        else:\\n            related[\\"w_ab\\"] = np.minimum(1.0, related[\\"count\\"] / np.maximum(cls_df[related[\\"src\\"]], 1)) if RE else np.zeros(0)\\n            related[\\"w_ba\\"] = np.minimum(1.0, related[\\"count\\"] / np.maximum(cls_df[related[\\"dst\\"]], 1)) if RE else np.zeros(0)\\n\\n        # ---------- isa (biến thể taxonomy) ----------\\n        IE = sorted({(inv.key2id[c], inv.key2id[t]) for c, ts in taxonomy.items() if c in inv.key2id\\n                     and inv.key2id[c] in self.active for t in ts if t in inv.key2id\\n                     and (inv.node_type[inv.key2id[t]] == \\"concept\\" or inv.key2id[t] in self.active)})\\n        isa = dict(src=np.array([e[0] for e in IE], int), dst=np.array([e[1] for e in IE], int))\\n        isa_true = _csr(n, n, isa[\\"src\\"], isa[\\"dst\\"], np.ones(len(IE)))\\n\\n        # ---------- đối chứng 1: constrained rewiring (contains giữ nguyên; vị từ + trọng số đi theo cạnh) ----------\\n        self.rewire_audit = []\\n        if rewire_seed is not None:\\n            rng = np.random.RandomState(rewire_seed)\\n            for rel in REL_CATS:\\n                fwd[rel], st = rewire_directed(fwd[rel], inv, rng, [\\"w\\", \\"w_inv\\", \\"support\\", \\"pdist\\"], rel)\\n                self.rewire_audit.append(st)\\n            related, st = rewire_undirected(related, inv, rng, \\"related\\")\\n            self.rewire_audit.append(st)\\n            isa, st = rewire_directed(isa, inv, rng, [], \\"isa\\")\\n            self.rewire_audit.append(st)\\n\\n        # ---------- đối chứng 2: hoán vị phân phối vị từ giữa các cạnh CÙNG họ (topology, trọng số giữ nguyên) ----------\\n        self.shuffle_audit = []\\n        if pred_shuffle_seed is not None:\\n            rng = np.random.RandomState(pred_shuffle_seed)\\n            for rel in PRED_FAMILIES:\\n                old = fwd[rel][\\"pdist\\"]\\n                perms = [rng.permutation(len(old)) for _ in range(10)]     # chọn hoán vị đổi được nhiều cạnh nhất\\n                perm = max(perms, key=lambda pm: sum(old[i] != old[j] for i, j in enumerate(pm)))\\n                new = [old[i] for i in perm]\\n                fwd[rel] = dict(fwd[rel], pdist=new)\\n                ms = lambda L: sorted(json.dumps(sorted(d.items())) for d in L)\\n                self.shuffle_audit.append(dict(family=rel, n_edges=len(old),\\n                                               frac_edges_predicates_changed=float(np.mean([a != b for a, b in zip(old, new)])) if old else 0.0,\\n                                               predicate_multiset_equal=ms(old) == ms(new)))\\n        self._fwd, self._related, self._isa = fwd, related, isa\\n\\n        # ---------- ma trận theo từng loại cạnh + bảng vị từ theo cặp ----------\\n        A = {}\\n        for rel in REL_CATS:\\n            f = fwd[rel]\\n            A[rel] = _csr(n, n, f[\\"src\\"], f[\\"dst\\"], f[\\"w\\"])\\n            A[rel + \\"_inv\\"] = _csr(n, n, f[\\"dst\\"], f[\\"src\\"], f[\\"w_inv\\"])\\n        rs, rd = related[\\"src\\"], related[\\"dst\\"]\\n        A[\\"related\\"] = _csr(n, n, np.r_[rs, rd], np.r_[rd, rs], np.r_[related[\\"w_ab\\"], related[\\"w_ba\\"]])\\n        A[\\"isa\\"] = _csr(n, n, isa[\\"src\\"], isa[\\"dst\\"], np.ones(len(isa[\\"src\\"])))\\n        A[\\"isa_inv\\"] = A[\\"isa\\"].T.tocsr()\\n        for t in self.masked:\\n            if t != \\"contains\\" and t != \\"contains_inv\\":\\n                A[t] = sp.csr_matrix((n, n), dtype=np.float32)\\n        self.A = A\\n        if \\"contains\\" in self.masked:\\n            raise ValueError(\\"không tắt contains trong ablation cạnh quan hệ\\")\\n        # pdist[(họ, a, b)] = {vị từ: số video}; pshare[họ][(a, b)] = {pred_id: tỉ phần} cho họ có vị từ tự do\\n        self.pdist = {(rel, int(a), int(b)): d for rel in REL_CATS for a, b, d in zip(fwd[rel][\\"src\\"], fwd[rel][\\"dst\\"], fwd[rel][\\"pdist\\"])}\\n        self.pshare = {}\\n        for rel in PRED_FAMILIES:\\n            sh = {}\\n            for (r_, a, b), d in self.pdist.items():\\n                if r_ == rel and A[rel][a, b] != 0:\\n                    tot = float(sum(d.values()))\\n                    sh[(a, b)] = {PRED_ID[(rel, p)]: c / tot for p, c in sorted(d.items()) if (rel, p) in PRED_ID}\\n            self.pshare[rel] = sh\\n\\n        # ---------- IDF (taxonomy, chỉ TRAIN) — KHÔNG phụ thuộc mask/rewiring ----------\\n        df_i = np.asarray(((Cb @ isa_true)[tr_rows] > 0).sum(0)).ravel()\\n        df = np.where(np.arange(n) < N + inv.n_cls, cls_df, df_i)\\n        idf = np.log((n_tr + 1.0) / (df + 1.0)).astype(np.float32)\\n        idf[:N] = 0.0\\n        self.idf = idf\\n        self.PM = self.path_mats()\\n        self.R = self.compute_R()\\n\\n    # ---------- đường đi ----------\\n    def pred_expand(self, A, rel):\\n        \\"\\"\\"X[u, pid·n + v] = A_t[u, v] · tỉ phần vị từ pid của cặp, cho t ∈ {rel, rel_inv} (đường đi khớp vị từ).\\"\\"\\"\\n        n, rows, cols, vals = self.n_nodes, [], [], []\\n        sh = self.pshare.get(rel, {})\\n        for t, inv_dir in ((rel, False), (rel + \\"_inv\\", True)):\\n            m = A[t].tocoo()\\n            for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):\\n                for pid, s in sh.get((v, u) if inv_dir else (u, v), {}).items():\\n                    rows.append(u); cols.append(pid * n + v); vals.append(w * s)\\n        return _csr(n, len(PRED_VOCAB) * n, rows, cols, vals)\\n\\n    def path_mats(self, A=None, A_con=None):\\n        A = A or self.A\\n        C = (A_con if A_con is not None else self.A_con).tocsr().astype(np.float32)\\n        if self.path_mode == \\"count\\":\\n            C = (C > 0).astype(np.float32)\\n        PM = {\\"C\\": C}\\n        for t in PATH_TYPES:\\n            M = A[t].tocsr().astype(np.float32)\\n            if self.path_mode == \\"count\\":\\n                M = (M > 0).astype(np.float32)\\n            elif self.path_mode == \\"degnorm\\":\\n                M = _symnorm(M)\\n            PM[t] = M\\n        if self.path_mode == \\"pred_match\\":\\n            for rel in PRED_FAMILIES:\\n                PM[f\\"X_{rel}\\"] = self.pred_expand(A, rel)\\n        return PM\\n\\n    def _fam(self, PM, f):\\n        a, b = PM[FAMILY_REL[f]], PM[FAMILY_REL[f] + \\"_inv\\"]\\n        return a.maximum(b) if self.path_mode == \\"dedup\\" else (a + b)\\n\\n    def compute_R(self, rows=None, PM=None):\\n        PM = PM or self.PM\\n        C = PM[\\"C\\"] if rows is None else PM[\\"C\\"][np.asarray(rows)]\\n        I, Rl = PM[\\"isa\\"], PM[\\"related\\"]\\n        prod = _maxprod if self.path_mode == \\"dedup\\" else (lambda X, Y: (X @ Y).tocsr())\\n        R = {\\"P0_contains\\": C.tocsr()}\\n        for f in FAMILIES:\\n            CF = prod(C, self._fam(PM, f))\\n            R[f\\"P1_{f}\\"] = CF\\n            R[f\\"P4_{f}_isa\\"] = prod(CF, I)\\n        R[\\"P2_isa\\"] = prod(C, I)\\n        CR = prod(C, Rl)\\n        R[\\"P3_related\\"] = CR\\n        R[\\"P5_related_isa\\"] = prod(CR, I)\\n        direct = (C > 0).astype(np.float32)\\n        if CFG.drop_redundant_indirect:\\n            for p in PATTERNS[1:]:\\n                if p in R:\\n                    R[p] = (R[p] - R[p].multiply(direct)).tocsr()\\n        D = sp.diags(self.idf)\\n        out = {}\\n        for p in PATTERNS:\\n            if p.startswith(\\"P1m_\\"):\\n                continue\\n            M = (R[p] @ D).tocsr(); M.eliminate_zeros(); out[p] = M\\n        n, n_p = self.n_nodes, len(PRED_VOCAB)\\n        for rel in PRED_FAMILIES:\\n            key = f\\"X_{rel}\\"\\n            if key not in PM:\\n                out[f\\"P1m_{rel}\\"] = sp.csr_matrix((C.shape[0], n_p * n), dtype=np.float32)\\n                continue\\n            M = (C @ PM[key]).tocoo()\\n            b = M.col % n\\n            keep = np.ones(M.nnz, bool)\\n            if CFG.drop_redundant_indirect and M.nnz:\\n                keep = np.asarray(direct[M.row, b]).ravel() == 0\\n            X = sp.csr_matrix(((M.data * self.idf[b])[keep], (M.row[keep], M.col[keep])), shape=M.shape, dtype=np.float32)\\n            X.eliminate_zeros()\\n            out[f\\"P1m_{rel}\\"] = X\\n        return out\\n\\n    # ---------- cạnh cho GNN (có loại, có trọng số, có chỉ số vị từ) ----------\\n    def edge_arrays(self):\\n        \\"\\"\\"(src, dst, type, weight, pidx): pidx = hàng trong self.pmix (phân phối vị từ của cạnh) hoặc −1.\\"\\"\\"\\n        if not hasattr(self, \\"_edges\\"):\\n            parts, mix_rows = [], []\\n            c = self.A_con.tocoo()\\n            parts.append((c.row, c.col, np.full(c.nnz, ETYPE_ID[\\"contains\\"]), c.data, np.full(c.nnz, -1)))\\n            parts.append((c.col, c.row, np.full(c.nnz, ETYPE_ID[\\"contains_inv\\"]), c.data, np.full(c.nnz, -1)))\\n            for t in RELATION_TYPES:\\n                m = self.A[t].tocoo()\\n                if not m.nnz:\\n                    continue\\n                pidx = np.full(m.nnz, -1)\\n                if t in PRED_TYPES:\\n                    base, inv_dir = (t[:-4], True) if t.endswith(\\"_inv\\") else (t, False)\\n                    for k, (u, v) in enumerate(zip(m.row.tolist(), m.col.tolist())):\\n                        d = self.pshare[base].get((v, u) if inv_dir else (u, v), {})\\n                        if d:\\n                            pidx[k] = len(mix_rows); mix_rows.append(d)\\n                parts.append((m.row, m.col, np.full(m.nnz, ETYPE_ID[t]), m.data, pidx))\\n            s, d_, e, w, pi = (np.concatenate([p[i] for p in parts]) for i in range(5))\\n            r_, c_, v_ = [], [], []\\n            for i, dd in enumerate(mix_rows):\\n                for pid, sh in dd.items():\\n                    r_.append(i); c_.append(pid); v_.append(sh)\\n            self.pmix = _csr(len(mix_rows), len(PRED_VOCAB), r_, c_, v_)\\n            self._edges = (s.astype(np.int64), d_.astype(np.int64), e.astype(np.int64), w.astype(np.float32), pi.astype(np.int64))\\n        return self._edges\\n\\n    def edge_counts(self):\\n        out = {\\"contains\\": int(self.A_con.nnz), \\"contains_inv\\": int(self.A_con.nnz)}\\n        out.update({t: int(self.A[t].nnz) for t in RELATION_TYPES})\\n        return out\\n\\n    def predicates_of(self, rel, a, b):\\n        \\"\\"\\"{vị từ: số video} của cạnh (rel, a → b) THEO ĐÚNG CHIỀU lưu trong đồ thị (đã tính rewiring / hoán vị vị từ).\\"\\"\\"\\n        base = rel[:-4] if rel.endswith(\\"_inv\\") else rel\\n        return self.pdist.get((base, b, a) if rel.endswith(\\"_inv\\") else (base, a, b), {})\\n\\n    def predicate(self, rel, a, b):\\n        d = self.predicates_of(rel, a, b)\\n        return max(sorted(d), key=lambda p: d[p]) if d else \\"\\"\\n\\n    def summary(self):\\n        return dict(name=self.name, nodes=self.n_nodes, videos=self.n_vid, classes=self.inv.n_cls, concepts=self.inv.n_con,\\n                    active_classes=len(self.active), edges=self.edge_counts(), contains=self.contains_mode,\\n                    path_mode=self.path_mode, rel_policy=self.rel_policy, related_cfg=list(self.related_cfg),\\n                    masked=list(self.masked), rewired=self.rewire_seed is not None,\\n                    predicate_shuffled=self.pred_shuffle_seed is not None, excluded=len(self.excluded),\\n                    paths={p: int(self.R[p].nnz) for p in PATTERNS})"}, {"cell_type": "code", "execution_count": null, "id": "c031", "metadata": {}, "outputs": [], "source": "def rewire_directed(E, inv, rng, attrs, rel):\\n    \\"\\"\\"Directed double-edge swap TRONG một loại quan hệ: (a→b),(c→d) ⇒ (a→d),(c→b) khi a,c cùng kiểu và b,d cùng\\n    kiểu (semantic type: noun / verb / noun_concept / verb_concept). Giữ out-degree, in-degree, kiểu đầu/cuối,\\n    số cạnh; cấm self-loop và cạnh trùng. Thuộc tính (trọng số, support, phân phối vị từ) gắn với cạnh theo NGUỒN.\\"\\"\\"\\n    src, dst = E[\\"src\\"].copy(), E[\\"dst\\"].copy()\\n    m = len(src)\\n    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)\\n    if m < 2:\\n        st.update(frac_changed=0.0, success_rate=0.0)\\n        return dict(E, src=src, dst=dst), st\\n    key = np.array([f\\"{inv.semantic_type(a)}>{inv.semantic_type(b)}\\" for a, b in zip(src, dst)])\\n    edges = set(zip(src.tolist(), dst.tolist()))\\n    for g in np.unique(key):\\n        I = np.where(key == g)[0]\\n        if len(I) < 2:\\n            continue\\n        for _ in range(CFG.rewire_swaps_per_edge * len(I)):\\n            i, j = rng.choice(I, 2, replace=False)\\n            a, b, c, d = src[i], dst[i], src[j], dst[j]\\n            st[\\"attempts\\"] += 1\\n            if a == d or c == b:\\n                st[\\"fail_self\\"] += 1; continue\\n            if (a, d) in edges or (c, b) in edges:\\n                st[\\"fail_dup\\"] += 1; continue\\n            edges -= {(a, b), (c, d)}; edges |= {(a, d), (c, b)}\\n            dst[i], dst[j] = d, b\\n            st[\\"swaps\\"] += 1\\n    orig = set(zip(E[\\"src\\"].tolist(), E[\\"dst\\"].tolist()))\\n    st[\\"frac_changed\\"] = 1 - len(orig & set(zip(src.tolist(), dst.tolist()))) / m\\n    st[\\"success_rate\\"] = st[\\"swaps\\"] / max(st[\\"attempts\\"], 1)\\n    return dict(E, src=src, dst=dst), st\\n\\n\\ndef rewire_undirected(E, inv, rng, rel):\\n    \\"\\"\\"Double-edge swap cho quan hệ ĐỐI XỨNG (related): giữ bậc vô hướng, cặp kiểu, không self-loop/trùng.\\n    Cạnh lưu (a, b) có kiểu(a) ≤ kiểu(b); số đếm đồng xuất hiện gắn với cạnh.\\"\\"\\"\\n    src, dst = E[\\"src\\"].copy(), E[\\"dst\\"].copy()\\n    wab, wba = E[\\"w_ab\\"].copy(), E[\\"w_ba\\"].copy()            # P(b|a), P(a|b) gắn với cạnh, đổi chỗ khi lật hướng\\n    m = len(src)\\n    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)\\n    if m < 2:\\n        st.update(frac_changed=0.0, success_rate=0.0)\\n        return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st\\n    T = lambda x: inv.semantic_type(x)\\n\\n    def flip(k):\\n        src[k], dst[k] = dst[k], src[k]\\n        wab[k], wba[k] = wba[k], wab[k]\\n\\n    for k in range(m):                                  # định hướng theo kiểu để swap giữ cặp kiểu\\n        if T(src[k]) > T(dst[k]):\\n            flip(k)\\n    key = np.array([f\\"{T(a)}~{T(b)}\\" for a, b in zip(src, dst)])\\n    edges = {(min(a, b), max(a, b)) for a, b in zip(src.tolist(), dst.tolist())}\\n    for g in np.unique(key):\\n        I = np.where(key == g)[0]\\n        if len(I) < 2:\\n            continue\\n        same_type = g.split(\\"~\\")[0] == g.split(\\"~\\")[1]\\n        for _ in range(CFG.rewire_swaps_per_edge * len(I)):\\n            i, j = rng.choice(I, 2, replace=False)\\n            if same_type and rng.rand() < 0.5:\\n                flip(j)                                 # cùng kiểu ⇒ cho phép cả hai cách ghép\\n            a, b, c, d = src[i], dst[i], src[j], dst[j]\\n            st[\\"attempts\\"] += 1\\n            n1, n2 = (min(a, d), max(a, d)), (min(c, b), max(c, b))\\n            if a == d or c == b or n1 == n2:\\n                st[\\"fail_self\\"] += 1; continue\\n            if n1 in edges or n2 in edges:\\n                st[\\"fail_dup\\"] += 1; continue\\n            edges -= {(min(a, b), max(a, b)), (min(c, d), max(c, d))}; edges |= {n1, n2}\\n            dst[i], dst[j] = d, b                       # (a, d) và (c, b); trọng số theo nguồn a, c\\n            st[\\"swaps\\"] += 1\\n    orig = {(min(a, b), max(a, b)) for a, b in zip(E[\\"src\\"].tolist(), E[\\"dst\\"].tolist())}\\n    st[\\"frac_changed\\"] = 1 - len(orig & edges) / m\\n    st[\\"success_rate\\"] = st[\\"swaps\\"] / max(st[\\"attempts\\"], 1)\\n    return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st\\n\\n\\ndef _pred_multiset(g, rel):\\n    \\"\\"\\"Đa tập phân phối vị từ của các cạnh thuộc họ rel (chỉ cạnh còn trong đồ thị).\\"\\"\\"\\n    if rel not in REL_CATS:\\n        return []\\n    return sorted(json.dumps(sorted(d.items())) for (r_, a, b), d in g.pdist.items() if r_ == rel and g.A[rel][a, b] != 0)\\n\\n\\ndef randomization_audit(g0, g1):\\n    \\"\\"\\"So sánh đồ thị gốc và đồ thị rewired theo từng loại cạnh; áp tiêu chí chấp nhận đăng ký trước.\\"\\"\\"\\n    inv, rows = g0.inv, []\\n    stats = {s[\\"relation\\"]: s for s in g1.rewire_audit}\\n    for t in [\\"contains\\"] + RELATION_TYPES:\\n        M0 = g0.A_con if t == \\"contains\\" else g0.A[t]\\n        M1 = g1.A_con if t == \\"contains\\" else g1.A[t]\\n        c0, c1 = M0.tocoo(), M1.tocoo()\\n        e0, e1 = set(zip(c0.row.tolist(), c0.col.tolist())), set(zip(c1.row.tolist(), c1.col.tolist()))\\n        tp = lambda c: Counter(f\\"{inv.semantic_type(a) if t != \'contains\' else \'video\'}>{inv.semantic_type(b)}\\"\\n                               for a, b in zip(c.row, c.col))\\n        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n        s = stats.get(base, {})\\n        row = dict(relation=t, edges_orig=len(e0), edges_rand=len(e1),\\n                   out_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(1)).ravel(), np.asarray((M1 > 0).sum(1)).ravel())),\\n                   in_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(0)).ravel(), np.asarray((M1 > 0).sum(0)).ravel())),\\n                   type_pairs_equal=tp(c0) == tp(c1), weights_multiset_equal=bool(np.allclose(np.sort(c0.data), np.sort(c1.data))),\\n                   self_loops=int(sum(a == b for a, b in e1)), duplicates=int(c1.nnz - len(e1)),\\n                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),\\n                   frac_changed=1 - len(e0 & e1) / max(len(e0), 1), swaps=s.get(\\"swaps\\"), attempts=s.get(\\"attempts\\"),\\n                   success_rate=s.get(\\"success_rate\\"), fail_dup=s.get(\\"fail_dup\\"), fail_self=s.get(\\"fail_self\\"))\\n        n_s, n_t = len({a for a, _ in e0}), len({b for _, b in e0})\\n        row[\\"density\\"] = len(e0) / max(n_s * n_t, 1)      # ≈ 1 ⇒ gần như KHÔNG có cách sắp xếp khác giữ nguyên bậc\\n        must_change = t != \\"contains\\" and len(e0) >= 10\\n        row[\\"accepted\\"] = bool(row[\\"edges_orig\\"] == row[\\"edges_rand\\"] and row[\\"out_degree_equal\\"] and row[\\"in_degree_equal\\"]\\n                               and row[\\"type_pairs_equal\\"] and row[\\"weights_multiset_equal\\"] and row[\\"self_loops\\"] == 0\\n                               and row[\\"duplicates\\"] == 0 and row[\\"predicates_multiset_equal\\"]\\n                               and (row[\\"frac_changed\\"] >= CFG.rewire_min_changed or not must_change)\\n                               and (t != \\"contains\\" or row[\\"frac_changed\\"] == 0))\\n        row[\\"note\\"] = (\\"contains phải giữ nguyên\\" if t == \\"contains\\" else \\"< 10 cạnh: không yêu cầu đổi\\" if not must_change\\n                       else (\\"đổi < ngưỡng: đồ thị quá đặc (mật độ cao) — không có đủ cách sắp xếp giữ bậc\\"\\n                             if not row[\\"accepted\\"] and row[\\"density\\"] > 0.5 else \\"\\"))\\n        rows.append(row)\\n    df = pd.DataFrame(rows)\\n    return df, bool(df.accepted.all())\\n\\n\\ndef predicate_shuffle_audit(g0, g1):\\n    \\"\\"\\"Đối chứng hoán vị vị từ: topology, trọng số, bậc của MỌI loại cạnh phải giống hệt; chỉ phân phối vị từ của cạnh\\n    thuộc họ có vị từ tự do được đổi chỗ (đa tập giữ nguyên).\\"\\"\\"\\n    rows = []\\n    stats = {s[\\"family\\"]: s for s in g1.shuffle_audit}\\n    for t in [\\"contains\\"] + RELATION_TYPES:\\n        M0 = g0.A_con if t == \\"contains\\" else g0.A[t]\\n        M1 = g1.A_con if t == \\"contains\\" else g1.A[t]\\n        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n        same_topology = (M0 != M1).nnz == 0\\n        moved = 0\\n        if base in PRED_FAMILIES:\\n            c = M0.tocoo()\\n            for u, v in zip(c.row.tolist(), c.col.tolist()):\\n                moved += int(g0.predicates_of(t, u, v) != g1.predicates_of(t, u, v))\\n        st = stats.get(base, {})\\n        row = dict(relation=t, n_edges=int(M0.nnz), topology_and_weights_equal=bool(same_topology),\\n                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),\\n                   edges_with_changed_predicates=moved, frac_edges_predicates_changed=moved / max(int(M0.nnz), 1),\\n                   shuffled=base in PRED_FAMILIES, family_frac_changed=st.get(\\"frac_edges_predicates_changed\\"))\\n        row[\\"accepted\\"] = bool(same_topology and row[\\"predicates_multiset_equal\\"]\\n                               and (moved == 0 if base not in PRED_FAMILIES else (M0.nnz < 10 or row[\\"frac_edges_predicates_changed\\"] >= CFG.rewire_min_changed)))\\n        rows.append(row)\\n    df = pd.DataFrame(rows)\\n    return df, bool(df.accepted.all())"}, {"cell_type": "code", "execution_count": null, "id": "c032", "metadata": {}, "outputs": [], "source": "def query_tokens_v2(text):\\n    \\"\\"\\"[(POS dự đoán, lemma danh từ, lemma động từ)] — POS tagger trên CHÍNH câu truy vấn (input của hệ thống).\\"\\"\\"\\n    toks = re.findall(r\\"[a-z]+\\", str(text).lower())\\n    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []\\n    out = []\\n    for w, t in tags:\\n        if w in STOP or len(w) < 2:\\n            continue\\n        p = \\"v\\" if t.startswith(\\"VB\\") else (\\"n\\" if t.startswith(\\"NN\\") else \\"x\\")\\n        out.append((p, lemma(w, \\"n\\"), lemma(w, \\"v\\")))\\n    return out\\n\\n\\ndef parse_tokens_v2(toks, inv):\\n    \\"\\"\\"Khớp token với node: lớp đúng POS trước (danh từ → \'x#n\', động từ → \'x#v\'), không có thì POS còn lại,\\n    không có lớp thì khái niệm taxonomy cùng lemma + POS. Trả về [(node_id, tên node, token)].\\"\\"\\"\\n    hits, seen = [], set()\\n    for p, ln, lv in toks:\\n        order = [(\\"v\\", lv), (\\"n\\", ln)] if p == \\"v\\" else [(\\"n\\", ln), (\\"v\\", lv)]\\n        found = None\\n        for pos, lm in order:\\n            i = inv.key2id.get(ckey(lm, pos))\\n            if i is not None:\\n                found = [i]; break\\n        if found is None:\\n            for pos, lm in order:\\n                if inv.concepts_by_lemma.get((lm, pos)):\\n                    found = inv.concepts_by_lemma[(lm, pos)]; break\\n        for i in found or []:\\n            if i not in seen:\\n                seen.add(i); hits.append((i, inv.name[i], lv if p == \\"v\\" else ln))\\n    return hits\\n\\n\\ndef query_pred_tokens(text):\\n    \\"\\"\\"Tập token của truy vấn để khớp VỊ TỪ: từ gốc + lemma động từ (giới từ giữ nguyên; KHÔNG bỏ stop-word).\\"\\"\\"\\n    w = re.findall(r\\"[a-z]+\\", str(text).lower())\\n    return frozenset(w) | frozenset(lemma(x, \\"v\\") for x in w)\\n\\n\\n_PRED_BY_FIRST = {}\\n\\n\\ndef match_predicates(qset, rel):\\n    \\"\\"\\"pred_id của họ rel có MỌI token nằm trong truy vấn (\'ride on\' khớp \'a man is riding on a horse\').\\"\\"\\"\\n    if rel not in _PRED_BY_FIRST:\\n        idx = defaultdict(list)\\n        for i, (r, p) in enumerate(PRED_VOCAB):\\n            if r == rel:\\n                t = predicate_tokens(p)\\n                idx[t[0]].append((i, t))\\n        _PRED_BY_FIRST[rel] = idx\\n    idx = _PRED_BY_FIRST[rel]\\n    return sorted({i for tok in qset for i, t in idx.get(tok, ()) if all(x in qset for x in t)})"}, {"cell_type": "markdown", "id": "c033", "metadata": {}, "source": "## 6. Mô hình v3\\n\\n**`TypedSAGEv3`.** Công thức là `h\'_v = W₀h_v + Σ_t σ(g_t)·[W_t·AGG_t(h_u) + U_t·AGG_t(e_uv)]`.\\n* `e_uv` là biểu diễn vị từ của cạnh. Nó chỉ có ở các loại cạnh svo, svo_inv, prep, prep_inv, và chỉ khi `pred_mode ≠ none`.\\n* `AGG_t` dùng **cùng hệ số** cho `h_u` và `e_uv`, nên trọng số cạnh chỉ nhân một lần.\\n* Vị từ của agent_of/object_of chính là node động từ đích, nên không đưa vào message, tránh trùng thông tin.\\n\\n**`pred_mode`:**\\n* `none` = relation-family-only, đúng mô hình v2;\\n* `clip` (phương án A) = CLIP-text **đóng băng** của vị từ theo prompt, qua một phép chiếu tuyến tính;\\n* `clip_adapt` (phương án B) = A cộng MLP dư **dùng chung** cho mọi vị từ, lớp cuối khởi tạo 0. Không có ID riêng cho từng vị từ, nên vị từ hiếm hoặc chưa gặp vẫn mã hoá được.\\n\\n`mp_weight` (`binary` / `weighted_mean` / `weighted_symnorm`) và `init` như v2.\\n### 6b. Mô hình v4 — QC-RGAT (GNN quan hệ có attention điều kiện theo truy vấn)\\n\\n**Vì sao đổi mô hình.** Ở v3, truy vấn chỉ đi vào GNN dưới dạng **một vector CLIP toàn cục**. Cạnh quan hệ nối lớp với lớp và là thống kê **dùng chung** cho mọi video, nên mọi video có cùng nhãn đều nhận cùng \\"ngữ cảnh quan hệ\\". Vì vậy quan hệ chỉ làm mượt embedding lớp, và kết quả Kaggle v3 cho G3 ≈ G2 ≈ G4 ≈ G5. Ở v4, **chính truy vấn quyết định cạnh quan hệ nào được truyền tin**.\\n\\n**Công thức** (T = 1 hoặc 2 lớp; chọn trên VAL):\\n* `H_q⁽⁰⁾(c) = E_c` và `H_q⁽ᵗ⁺¹⁾(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q⁽ᵗ⁾(u)`. Ở đây `z_{t,τ}` là toán tử của loại quan hệ τ, phép nhân theo chiều kiểu CompGCN/DistMult.\\n* `a_q = σ(β_τ + κ_τ·log w + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ\'_τ·cos(e_p, e_P(q)))` gồm năm thành phần:\\n  * cổng theo loại quan hệ do truy vấn chọn (`W_g u_q`);\\n  * **vai** của node nguồn trong truy vấn (chủ thể / động từ / đối tượng / tân ngữ giới từ, do parser caption áp lên chính truy vấn);\\n  * **khớp vị từ** giữa cạnh và truy vấn, cả khớp đúng lẫn khớp mềm bằng CLIP-text;\\n  * support của cạnh (`κ_τ·log w`);\\n  * độ lệch `β_τ`.\\n* `s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v,c] · H_q⁽ᵀ⁾(c)⟩`. Video **chỉ nhận** qua nhãn của chính nó (inductive). Điểm cuối = `α·z(cos CLIP) + (1−α)·z(s)`, với α chọn trên VAL.\\n\\n**Khởi tạo và học các cổng.** γ_τ khởi tạo bằng 1, tức prior \\"ưu tiên cạnh có vị từ khớp truy vấn\\"; dữ liệu có thể làm γ giảm. Các cổng vô hướng (β, κ, ρ, γ, γ\') học với lr × `scalar_lr_mult`, như cổng của v3. Nếu khởi tạo γ = 0 và học với lr thường, γ gần như không đổi sau huấn luyện. Khi đó G2, G4, G6 cho kết quả trùng G3, và các đối chứng không còn đo được gì; lần thử dev đã gặp đúng trường hợp này.\\n\\n**Giải thích từ chính cơ chế.** Attention chỉ phụ thuộc truy vấn và thuộc tính tĩnh của cạnh, nên phép truyền là **tuyến tính theo E**. Vì vậy `s` tách **chính xác** thành tổng các số hạng: thị giác + từng nhãn contains + từng **đường đi quan hệ** (`u –τ[p]→ c ←chứa– v`, có thêm một bước khi T = 2). Giải thích là top-m số hạng quan hệ. Xoá một cạnh làm điểm đổi **đúng bằng** tổng đóng góp của các đường đi qua cạnh đó (T = 1); điều này có kiểm thử Q3 và X3.\\n\\n**Đối chứng tách từng thành phần** (cùng mô hình, cùng giao thức huấn luyện):\\n* G1 `v4_rel=False`: không cạnh quan hệ;\\n* G2 `v4_pred=False`: không khớp vị từ;\\n* G4: hoán vị vị từ giữa các cạnh cùng họ;\\n* G5: rewiring có ràng buộc;\\n* G6 `v4_qcond=False`: attention tĩnh, không theo truy vấn — tức là quan hệ như v3;\\n* R3: chính mô hình v3 cuối.\\n\\nLiên hệ tài liệu: NBFNet (Zhu et al., NeurIPS 2021) với message điều kiện theo truy vấn; KGAT (Wang et al., KDD 2019) với attention nhận biết quan hệ; CompGCN (Vashishth et al., ICLR 2020) với toán tử quan hệ. Thuộc họ **GAT quan hệ** của cấp độ 1.\\n"}, {"cell_type": "code", "execution_count": null, "id": "c034", "metadata": {}, "outputs": [], "source": "def scatter_sum(src, index, n):\\n    out = torch.zeros((n,) + tuple(src.shape[1:]), device=src.device, dtype=src.dtype)\\n    return out.index_add_(0, index, src)\\n\\n\\nPRED_MODES = (\\"none\\", \\"clip\\", \\"clip_adapt\\")\\n\\n\\nclass TypedSAGEv3(nn.Module):\\n    \\"\\"\\"h\'_v = W₀ h_v + Σ_t σ(g_t) · [ W_t · AGG_t(h_u) + U_t · AGG_t(e_uv) ]\\n    e_uv = embedding vị từ của cạnh (chỉ loại cạnh thuộc họ có vị từ tự do, khi pred_mode ≠ \'none\'); AGG_t dùng CÙNG hệ số\\n    chuẩn hoá cho h_u và e_uv ⇒ message(u → v) = W_t h_u + U_t e_uv đúng như thiết kế, trọng số cạnh chỉ nhân MỘT lần.\\n    AGG theo mp_weight: \'binary\' (trung bình, bỏ trọng số) | \'weighted_mean\' (Σ w·x / Σ w) | \'weighted_symnorm\' (Σ w·x / √(d_u d_v)).\\n    Nhánh W₀ h_v giữ biểu diễn của node không có hàng xóm (không sinh NaN).\\"\\"\\"\\n\\n    def __init__(self, din, dout, n_types, open_types, gate_init, mp_weight, pred_type_ids=()):\\n        super().__init__()\\n        assert mp_weight in (\\"binary\\", \\"weighted_mean\\", \\"weighted_symnorm\\"), mp_weight\\n        self.mp_weight, self.n_types = mp_weight, n_types\\n        self.self_w = nn.Linear(din, dout)\\n        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(n_types)])\\n        self.pred_type_ids = list(pred_type_ids)\\n        self.pred_w = nn.ModuleDict({str(t): nn.Linear(din, dout, bias=False) for t in self.pred_type_ids})\\n        g = torch.full((n_types,), float(gate_init))\\n        for t in open_types:\\n            g[t] = 3.0\\n        self.gate = nn.Parameter(g)\\n\\n    def coef(self, src, dst, w, n):\\n        \\"\\"\\"Hệ số của từng cạnh trong AGG (cùng cho h_u và e_uv).\\"\\"\\"\\n        if self.mp_weight == \\"binary\\":\\n            w = torch.ones_like(w)\\n        if self.mp_weight in (\\"binary\\", \\"weighted_mean\\"):\\n            return w / scatter_sum(w, dst, n)[dst].clamp(min=1e-12)\\n        d_out, d_in = scatter_sum(w, src, n), scatter_sum(w, dst, n)\\n        return w / torch.sqrt(d_out[src] * d_in[dst]).clamp(min=1e-12)\\n\\n    def forward(self, x, edge_index, edge_type, edge_weight, edge_pidx=None, pred_h=None):\\n        src, dst = edge_index\\n        n = x.size(0)\\n        out = self.self_w(x)\\n        for t in range(self.n_types):\\n            m = edge_type == t\\n            if not bool(m.any()):\\n                continue\\n            s, d = src[m], dst[m]\\n            c = self.coef(s, d, edge_weight[m], n).unsqueeze(-1)\\n            msg = self.neigh_w[t](scatter_sum(x[s] * c, d, n))\\n            if pred_h is not None and t in self.pred_type_ids:\\n                pi = edge_pidx[m]\\n                has = pi >= 0\\n                if bool(has.any()):\\n                    msg = msg + self.pred_w[str(t)](scatter_sum(pred_h[pi[has]] * c[has], d[has], n))\\n            out = out + torch.sigmoid(self.gate[t]) * msg\\n        return out\\n\\n\\nclass PredicateEncoder(nn.Module):\\n    \\"\\"\\"Biểu diễn vị từ của cạnh = Σ_p tỉ phần(p) · φ(e_p), e_p = CLIP-text ĐÓNG BĂNG của vị từ theo prompt.\\n      \'clip\'       (phương án A): φ = phép chiếu tuyến tính học được (về chiều GNN)\\n      \'clip_adapt\' (phương án B): φ(e) = chiếu(e + MLP(e)); MLP DÙNG CHUNG cho mọi vị từ, lớp cuối khởi tạo 0\\n                                  ⇒ không có tham số riêng cho vị từ hiếm; vị từ mới (ngoài từ vựng) vẫn mã hoá được.\\"\\"\\"\\n\\n    def __init__(self, clip_dim, dout, mode, hidden):\\n        super().__init__()\\n        self.mode = mode\\n        self.proj = nn.Linear(clip_dim, dout)\\n        if mode == \\"clip_adapt\\":\\n            self.res = nn.Sequential(nn.Linear(clip_dim, hidden), nn.GELU(), nn.Linear(hidden, clip_dim))\\n            nn.init.zeros_(self.res[2].weight); nn.init.zeros_(self.res[2].bias)\\n\\n    def forward(self, pmix, pemb):\\n        e = pemb + self.res(pemb) if self.mode == \\"clip_adapt\\" else pemb\\n        return torch.sparse.mm(pmix, self.proj(e)) if pmix.is_sparse else pmix @ self.proj(e)\\n\\n\\nclass GraphScorerV3(nn.Module):\\n    \\"\\"\\"Như GraphScorer v2 (12 mẫu đường đi + 2 mẫu khớp vị từ; khởi tạo node \'learned\' | \'clip_frozen\' | \'clip_residual\')\\n    cộng nhánh vị từ (pred_mode). pred_mode = \'none\' ⇒ đúng mô hình v2 (đối chứng relation-family-only).\\"\\"\\"\\n\\n    def __init__(self, n_nodes, clip_dim, vid_feat, sym_feat, pred_emb=None, cfg=CFG, mp_weight=\\"binary\\", init=\\"learned\\",\\n                 pred_mode=\\"none\\"):\\n        super().__init__()\\n        assert pred_mode in PRED_MODES, pred_mode\\n        self.n_vid, self.init, self.pred_mode = vid_feat.shape[0], init, pred_mode\\n        self.register_buffer(\\"vid_feat\\", vid_feat.clone())\\n        self.vid_proj = nn.Linear(clip_dim, cfg.gnn_dim)\\n        n_sym = n_nodes - self.n_vid\\n        if init == \\"learned\\":\\n            self.sym_emb = nn.Embedding(n_sym, cfg.gnn_dim)\\n            nn.init.normal_(self.sym_emb.weight, std=0.02)\\n        else:\\n            self.register_buffer(\\"sym_feat\\", torch.as_tensor(sym_feat, dtype=torch.float32).clone())\\n            self.sym_proj = nn.Linear(sym_feat.shape[1], cfg.gnn_dim)\\n            if init == \\"clip_residual\\":\\n                self.sym_res = nn.Embedding(n_sym, cfg.gnn_dim)\\n                nn.init.zeros_(self.sym_res.weight)\\n        pred_ids = [ETYPE_ID[t] for t in PRED_TYPES] if pred_mode != \\"none\\" else []\\n        if pred_mode != \\"none\\":\\n            self.register_buffer(\\"pred_emb\\", torch.as_tensor(pred_emb, dtype=torch.float32).clone())\\n            self.pred_enc = PredicateEncoder(self.pred_emb.shape[1], cfg.gnn_dim, pred_mode, cfg.pred_adapt_hidden)\\n        open_ids = [ETYPE_ID[t] for t in OPEN_TYPES]\\n        self.layers = nn.ModuleList([TypedSAGEv3(cfg.gnn_dim, cfg.gnn_dim, len(EDGE_TYPES), open_ids, cfg.gate_init, mp_weight, pred_ids)\\n                                     for _ in range(cfg.gnn_layers)])\\n        self.norms = nn.ModuleList([nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])\\n        self.drop = nn.Dropout(cfg.gnn_dropout)\\n        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(), nn.Linear(cfg.gnn_dim, cfg.gnn_dim))\\n        init_w = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init_w[0] = 0.0\\n        self.path_w = nn.Parameter(init_w)\\n        self.decay = nn.Parameter(torch.tensor(0.5))\\n        self.register_buffer(\\"path_gain\\", torch.tensor(1.0))\\n        self.clip_logit = nn.Parameter(torch.tensor(0.5413))\\n\\n    def node_features(self):\\n        if self.init == \\"learned\\":\\n            sym = self.sym_emb.weight\\n        else:\\n            sym = self.sym_proj(self.sym_feat)\\n            if self.init == \\"clip_residual\\":\\n                sym = sym + self.sym_res.weight\\n        return torch.cat([self.vid_proj(self.vid_feat), sym], 0)\\n\\n    def edge_gates(self):\\n        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers]\\n\\n    def predicate_features(self, pmix):\\n        return self.pred_enc(pmix, self.pred_emb) if self.pred_mode != \\"none\\" and pmix is not None else None\\n\\n    def encode_graph(self, edge_index, edge_type, edge_weight, edge_pidx=None, pmix=None):\\n        h = self.node_features()\\n        ph = self.predicate_features(pmix)\\n        for i, (l, nrm) in enumerate(zip(self.layers, self.norms)):\\n            h = nrm(l(h, edge_index, edge_type, edge_weight, edge_pidx, ph))\\n            if i < len(self.layers) - 1:\\n                h = self.drop(F.gelu(h))\\n        return F.normalize(h, dim=-1)\\n\\n    def pattern_weights(self):\\n        d = torch.sigmoid(self.decay)\\n        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)\\n        w = F.softplus(self.path_w) * d.pow(lens - 1)\\n        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])\\n\\n    def gnn_score(self, h, q_clip, cand_idx):\\n        q = F.normalize(self.q_proj(q_clip), dim=-1)\\n        return torch.einsum(\\"bd,bkd->bk\\", q, h[cand_idx])\\n\\n    def path_score(self, path_mat):\\n        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)\\n\\n\\n_SYM_FEAT = {}\\n\\n\\ndef symbolic_features(inv, vocab):\\n    \\"\\"\\"CLIP-text của node lớp (= prompt ensemble của tagger) và khái niệm (\'lemma: định nghĩa WordNet\').\\"\\"\\"\\n    key = inv.sha1\\n    if key not in _SYM_FEAT:\\n        T = term_text_features_v2(vocab)\\n        cls = np.stack([T[vocab[\\"terms\\"].index(k)] for k in inv.class_keys])\\n        texts = [f\\"{inv.concept_meta[k][\'lemma\']}: {inv.concept_meta[k][\'gloss\']}\\".strip(\\": \\") for k in inv.concept_keys]\\n        con = clip_encode_texts(texts) if texts else np.zeros((0, cls.shape[1]), np.float32)\\n        _SYM_FEAT[key] = np.concatenate([cls, con]).astype(np.float32)\\n    return _SYM_FEAT[key]\\n\\n\\ndef predicate_phrase(rel, p, vocab):\\n    \\"\\"\\"Cụm chữ cho prompt: svo → dạng V-ing của động từ đầu + phần còn lại (\'ride on\' → \'riding on\'); prep → nguyên văn.\\"\\"\\"\\n    toks = p.split()\\n    if rel == \\"svo\\" and toks:\\n        g = vocab[\\"gerund\\"].get(ckey(toks[0], \\"v\\")) or naive_gerund(toks[0])\\n        toks = [g] + toks[1:]\\n    return \\" \\".join(toks)\\n\\n\\ndef predicate_embeddings(pred_vocab, vocab):\\n    \\"\\"\\"[n_vị_từ, D] CLIP-text ĐÓNG BĂNG (trung bình các prompt CFG.pred_prompts[họ], chuẩn L2).\\"\\"\\"\\n    out = None\\n    for rel in PRED_FAMILIES:\\n        idx = [i for i, (r, _) in enumerate(pred_vocab) if r == rel]\\n        if not idx:\\n            continue\\n        words = [predicate_phrase(rel, pred_vocab[i][1], vocab) for i in idx]\\n        E = sum(clip_encode_texts([pr.format(w) for w in words]) for pr in CFG.pred_prompts[rel])\\n        if out is None:\\n            out = np.zeros((len(pred_vocab), E.shape[1]), np.float32)\\n        out[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)\\n    return out if out is not None else np.zeros((len(pred_vocab), 1), np.float32)\\n\\n# ---------------------------------- v4: QC-RGAT (GNN quan hệ có attention ĐIỀU KIỆN THEO TRUY VẤN) ----------------------------------\\n# H_q^(0)(c) = E_c ;  H_q^(t+1)(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q^(t)(u)            (cạnh giữa lớp/khái niệm)\\n# a_q = σ( β_τ + κ_τ·log w_uc + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ\'_τ·cos(e_p, e_P(q)) )\\n# s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v, c] · H_q^(T)(c)⟩        (Ā_con = contains chuẩn hoá theo hàng, video CHỈ nhận)\\n# Phép truyền TUYẾN TÍNH theo E (attention chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh) ⇒ s = Σ đóng góp của từng\\n# thành phần: thị giác + từng nhãn contains + từng ĐƯỜNG ĐI quan hệ — phân rã CHÍNH XÁC, giải thích = đúng các số hạng của điểm.\\nV4_REL_TYPES = list(RELATION_TYPES)\\nV4_TID = {ETYPE_ID[t]: i for i, t in enumerate(V4_REL_TYPES)}\\nQ_ROLES = (\\"none\\", \\"subj\\", \\"verb\\", \\"obj\\", \\"pobj\\", \\"other\\")\\n\\n\\ndef query_roles_v4(texts, inv):\\n    \\"\\"\\"Vai của node lớp trong CHÍNH truy vấn (parser caption áp lên văn bản truy vấn lúc suy luận — mục đích query_input).\\"\\"\\"\\n    out = []\\n    for text in texts:\\n        _, _, trip = parse_caption_v2(text)\\n        role = {}\\n        for s_, rel, o_, _p in trip:\\n            a, b = inv.key2id.get(s_), inv.key2id.get(o_)\\n            if a is None or b is None:\\n                continue\\n            if rel == \\"agent_of\\":\\n                role.setdefault(a, \\"subj\\"); role.setdefault(b, \\"verb\\")\\n            elif rel == \\"object_of\\":\\n                role.setdefault(a, \\"obj\\"); role.setdefault(b, \\"verb\\")\\n            elif rel == \\"svo\\":\\n                role.setdefault(a, \\"subj\\")\\n                role[b] = role[b] if role.get(b) not in (None, \\"other\\") else \\"obj\\"\\n            elif rel == \\"prep\\":\\n                role.setdefault(b, \\"pobj\\")\\n        out.append({k: Q_ROLES.index(r) for k, r in role.items()})\\n    return out\\n\\n\\nclass RelDataV4:\\n    \\"\\"\\"Tensor của một đồ thị cho QC-RGAT: cạnh quan hệ (giữa node lớp/khái niệm), vị từ của cạnh, contains chuẩn hoá theo hàng.\\"\\"\\"\\n\\n    def __init__(self, graph, drop_edges=None):\\n        self.graph = graph\\n        self.N, self.n = graph.n_vid, graph.n_nodes\\n        self.ns = self.n - self.N\\n        s, d, e, w, pi = graph.edge_arrays()\\n        keep = np.isin(e, list(V4_TID))\\n        assert (s[keep] >= self.N).all() and (d[keep] >= self.N).all(), \\"cạnh quan hệ chỉ được nối node lớp/khái niệm\\"\\n        self.gidx = np.where(keep)[0]                       # chỉ số cạnh trong graph.edge_arrays() (để giải thích / xoá cạnh)\\n        if drop_edges is not None:\\n            self.gidx = np.setdiff1d(self.gidx, np.asarray(list(drop_edges), np.int64))\\n        s, d, e, w, pi = s[self.gidx], d[self.gidx], e[self.gidx], w[self.gidx], pi[self.gidx]\\n        self.E = len(self.gidx)\\n        self.src_np, self.dst_np, self.pidx_np = s - self.N, d - self.N, pi\\n        self.src = torch.from_numpy(self.src_np).to(DEVICE)\\n        self.dst = torch.from_numpy(self.dst_np).to(DEVICE)\\n        self.et = torch.tensor([V4_TID[int(x)] for x in e], dtype=torch.long, device=DEVICE)\\n        self.logw = torch.from_numpy(np.log(np.maximum(w, 1e-3)).astype(np.float32)).to(DEVICE)\\n        PE = np.zeros((self.E, PRED_EMB.shape[1]), np.float32)\\n        rows = np.where(pi >= 0)[0]\\n        if len(rows):\\n            X = np.asarray(graph.pmix[pi[rows]] @ PRED_EMB)\\n            PE[rows] = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)\\n        self.PE = torch.from_numpy(PE).to(DEVICE)\\n        M = (graph.pmix[np.maximum(pi, 0)].multiply((pi >= 0)[:, None]) if self.E and (pi >= 0).any()\\n             else sp.csr_matrix((self.E, len(PRED_VOCAB)), dtype=np.float32))   # G8: không cạnh nào mang vị từ\\n        self.Mex = sp.csr_matrix(M).T.tocsr()               # [n_vị_từ, E]: phần vị từ p trên cạnh e\\n        A = (graph.A_con.tocsr()[:, self.N:] > 0).astype(np.float32).toarray()\\n        self.Acon = torch.from_numpy(A / np.maximum(A.sum(1, keepdims=True), 1)).to(DEVICE)\\n\\n    def query_batch(self, q_idx, use_pred=True):\\n        \\"\\"\\"Đầu vào phía truy vấn: node khớp + vai, vị từ truy vấn (khớp đúng + CLIP mềm) trên TỪNG cạnh, CLIP-text.\\"\\"\\"\\n        q_idx = np.asarray(q_idx, np.int64)\\n        B = len(q_idx)\\n        bi, ci, ri = [], [], []\\n        role_src = torch.zeros(B, self.ns, dtype=torch.long)\\n        for b, q in enumerate(q_idx):\\n            for h in V4_HITS[q]:\\n                r = V4_ROLES[q].get(h, Q_ROLES.index(\\"verb\\" if INV.pos[h] == \\"v\\" else \\"other\\"))\\n                bi.append(b); ci.append(h - self.N); ri.append(r)\\n                role_src[b, h - self.N] = r\\n        ex = np.zeros((B, self.E), np.float32)\\n        qp = np.zeros((B, PRED_EMB.shape[1]), np.float32)\\n        if use_pred and self.E:\\n            for b, q in enumerate(q_idx):\\n                ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]\\n                if ids:\\n                    ex[b] = np.minimum(np.asarray(self.Mex[ids].sum(0)).ravel(), 1.0)\\n                    m = PRED_EMB[ids].mean(0)\\n                    qp[b] = m / max(np.linalg.norm(m), 1e-8)\\n        qp_t = torch.from_numpy(qp).to(DEVICE)\\n        return dict(q=q_idx, bi=torch.tensor(bi, dtype=torch.long, device=DEVICE), ci=torch.tensor(ci, dtype=torch.long, device=DEVICE),\\n                    ri=torch.tensor(ri, dtype=torch.long, device=DEVICE), role_src=role_src.to(DEVICE),\\n                    ex=torch.from_numpy(ex).to(DEVICE), soft=qp_t @ self.PE.T,\\n                    tq=torch.from_numpy(np.ascontiguousarray(TXT_FEAT[q_idx], dtype=np.float32)).to(DEVICE))\\n\\n\\nclass QCRGAT(nn.Module):\\n    \\"\\"\\"rel=False ⇒ G1 (chỉ contains); use_pred=False ⇒ G2 (không vị từ); qcond=False ⇒ G6 (attention tĩnh, không theo truy vấn).\\"\\"\\"\\n\\n    SCALARS = (\\"beta\\", \\"kappa\\", \\"rgate\\", \\"g_ex\\", \\"g_soft\\")      # cổng vô hướng: lr × CFG.scalar_lr_mult (như cổng của v3)\\n\\n    def __init__(self, data, clip_dim, d=64, T=1, rel=True, qcond=True, use_pred=True):\\n        super().__init__()\\n        assert T in (1, 2), \\"phân rã đường đi hiện cài cho T = 1 hoặc 2\\"\\n        self.D, self.T, self.rel, self.qcond, self.use_pred = data, T, rel, qcond, use_pred\\n        nt = len(V4_REL_TYPES)\\n        self.vid = nn.Linear(clip_dim, d)\\n        self.q = nn.Sequential(nn.Linear(clip_dim, d), nn.GELU(), nn.Linear(d, d))\\n        self.qo = nn.Linear(d, d)\\n        self.cls = nn.Embedding(data.ns, d)\\n        nn.init.normal_(self.cls.weight, std=0.02)\\n        self.z = nn.Parameter(1 + 0.1 * torch.randn(T, nt, d))\\n        self.beta = nn.Parameter(torch.full((nt,), -1.0))\\n        self.kappa = nn.Parameter(torch.zeros(nt))\\n        self.qgate = nn.Linear(d, nt)\\n        self.rgate = nn.Parameter(torch.zeros(len(Q_ROLES), nt))\\n        self.g_ex = nn.Parameter(torch.ones(nt))         # prior: cạnh có vị từ KHỚP truy vấn được ưu tiên (học tiếp từ dữ liệu)\\n        self.g_soft = nn.Parameter(torch.zeros(nt))\\n        self.clip_logit = nn.Parameter(torch.tensor(0.5413))\\n\\n    def attention(self, qb, u, D=None):\\n        \\"\\"\\"a_q(cạnh) ∈ (0, 1), [B, E]. Chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh (KHÔNG phụ thuộc trạng thái node).\\"\\"\\"\\n        D = D or self.D\\n        a = (self.beta[D.et] + self.kappa[D.et] * D.logw)[None].expand(u.shape[0], -1)\\n        if self.qcond:\\n            a = a + self.qgate(u)[:, D.et] + self.rgate[qb[\\"role_src\\"][:, D.src], D.et]\\n            if self.use_pred:\\n                a = a + self.g_ex[D.et][None] * qb[\\"ex\\"] + self.g_soft[D.et][None] * qb[\\"soft\\"]\\n        return torch.sigmoid(a)\\n\\n    def class_states(self, qb, u, D=None):\\n        D = D or self.D\\n        E = self.cls.weight[None].expand(u.shape[0], -1, -1)\\n        if not self.rel or D.E == 0:\\n            return E, None\\n        a = self.attention(qb, u, D)\\n        H = E\\n        for t in range(self.T):\\n            H = E.index_add(1, D.dst, a[:, :, None] * self.z[t][D.et][None] * H[:, D.src, :])\\n        return H, a\\n\\n    def score(self, qb, cand, D=None):\\n        \\"\\"\\"cand: [B, K] video id (tensor) ⇒ [B, K].\\"\\"\\"\\n        D = D or self.D\\n        u = self.q(qb[\\"tq\\"])\\n        H, _ = self.class_states(qb, u, D)\\n        V = self.vid(VID_T[cand]) + torch.einsum(\\"bkc,bcd->bkd\\", D.Acon[cand], H)\\n        return torch.einsum(\\"bd,bkd->bk\\", F.normalize(self.qo(u), dim=-1), V)\\n\\n    @torch.no_grad()\\n    def decompose(self, q, vid, D=None):\\n        \\"\\"\\"Phân rã CHÍNH XÁC s(q, vid) = thị giác + Σ nhãn + Σ đường đi quan hệ. Mỗi đường đi: list chỉ số cạnh (trong D)\\n        theo chiều đi từ node nguồn tới node nhãn c của video, kèm đóng góp.\\"\\"\\"\\n        D = D or self.D\\n        qb = D.query_batch([q], self.use_pred)\\n        u = self.q(qb[\\"tq\\"])\\n        uo = F.normalize(self.qo(u), dim=-1)[0]\\n        E = self.cls.weight\\n        vt = int(vid)\\n        terms = [dict(kind=\\"visual\\", node=None, edges=(), contrib=float(uo @ self.vid(VID_T[vt])))]\\n        acon = D.Acon[vt]\\n        tags = torch.nonzero(acon > 0).flatten().tolist()\\n        for c in tags:\\n            terms.append(dict(kind=\\"contains\\", node=c, edges=(), contrib=float(acon[c] * (uo @ E[c]))))\\n        if self.rel and D.E:\\n            a = self.attention(qb, u, D)[0]\\n            inc = defaultdict(list)\\n            for k in range(D.E):\\n                inc[int(D.dst_np[k])].append(k)\\n            for c in tags:\\n                for e1 in inc.get(c, []):                     # đường đi 1 bước: u –e1→ c   (dùng z của lớp cuối)\\n                    zz = self.z[self.T - 1][D.et[e1]]\\n                    terms.append(dict(kind=\\"relation\\", node=c, edges=(e1,),\\n                                      contrib=float(acon[c] * a[e1] * (uo @ (zz * E[D.src_np[e1]])))))\\n                    if self.T == 2:                           # đường đi 2 bước: w –e0→ u –e1→ c\\n                        for e0 in inc.get(int(D.src_np[e1]), []):\\n                            zz2 = self.z[1][D.et[e1]] * self.z[0][D.et[e0]]\\n                            terms.append(dict(kind=\\"relation\\", node=c, edges=(e0, e1),\\n                                              contrib=float(acon[c] * a[e1] * a[e0] * (uo @ (zz2 * E[D.src_np[e0]])))))\\n        return terms\\n\\n\\nVID_T = None\\n\\n\\ndef v4_tensors():\\n    global VID_T\\n    VID_T = torch.from_numpy(np.ascontiguousarray(VID_FEAT, dtype=np.float32)).to(DEVICE)\\n"}, {"cell_type": "markdown", "id": "c035", "metadata": {}, "source": "## 7. Engine và hai giao thức suy luận\\n* **strict inductive** (mặc định): video val, test và tập ngoài **chỉ nhận** message qua `contains_inv`, nên không ảnh hưởng node dùng chung hay video khác. Có unit test cho cả video test MSVD và video MSR-VTT.\\n* **transductive**: chỉ để báo cáo.\\n\\nEngine mang thêm chỉ số vị từ của từng cạnh (`edge_pidx`) và bảng phân phối vị từ (`pmix`) cho GNN. Nó cũng có ma trận truy vấn × (vị từ, node) cho mẫu P1m. Vị từ của truy vấn được khớp khi mọi token của vị từ có mặt trong truy vấn, ví dụ `ride on` khớp \\"a man is riding on a horse\\"."}, {"cell_type": "code", "execution_count": null, "id": "c036", "metadata": {}, "outputs": [], "source": "def znorm_per_row(x, eps=1e-6):\\n    if isinstance(x, torch.Tensor):\\n        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)\\n    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)\\n\\n\\ndef combine(s_clip, s_gnn, s_path, alpha, lam):\\n    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)\\n\\n\\nPROTOCOLS = (\\"inductive\\", \\"transductive\\")\\n\\n\\ndef _torch_sparse(M):\\n    M = M.tocoo()\\n    idx = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))\\n    return torch.sparse_coo_tensor(idx, torch.from_numpy(M.data.astype(np.float32)), M.shape).coalesce()\\n\\n\\nclass EngineV3:\\n    \\"\\"\\"Truy vấn = caption; ứng viên = video. Ba tập cạnh cho GNN (như v2):\\n      train        : bỏ MỌI cạnh chạm video không thuộc train;\\n      inductive    : cạnh của train + cạnh lớp → video (contains_inv) của video val/test/ext, KHÔNG có chiều video → lớp\\n                     ⇒ video mới chỉ NHẬN message, không ảnh hưởng node lớp/khái niệm dùng chung hay video khác;\\n      transductive : đồ thị đầy đủ.\\n    Thêm ở v3: chỉ số vị từ của từng cạnh (edge_pidx) + bảng phân phối vị từ (pmix) cho GNN; ma trận truy vấn × (vị từ, node)\\n    cho mẫu đường đi khớp vị từ P1m.\\"\\"\\"\\n\\n    _QCACHE = {}\\n\\n    def __init__(self, graph, vid_feat, txt_feat, qtok, qpreds):\\n        self.graph = graph\\n        self.vid_feat = torch.from_numpy(np.ascontiguousarray(vid_feat, dtype=np.float32))\\n        self.txt_feat = torch.from_numpy(np.ascontiguousarray(txt_feat, dtype=np.float32))\\n        n = graph.n_nodes\\n        key = (graph.inv.sha1, len(qtok), id(qtok), id(qpreds))\\n        if key not in EngineV3._QCACHE:                      # chỉ phụ thuộc inventory + truy vấn ⇒ dùng chung mọi biến thể đồ thị\\n            qc = [parse_tokens_v2(t, graph.inv) for t in qtok]\\n            rows, cols = [], []\\n            for j, hits in enumerate(qc):\\n                for nid, _, _ in hits:\\n                    rows.append(j); cols.append(nid)\\n            QT = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(qtok), n))\\n            QPB = {}\\n            for rel in PRED_FAMILIES:\\n                r_, c_ = [], []\\n                for j, (hits, qp) in enumerate(zip(qc, qpreds)):\\n                    for pid in qp[rel]:\\n                        for nid, _, _ in hits:\\n                            r_.append(j); c_.append(pid * n + nid)\\n                QPB[rel] = sp.csr_matrix((np.ones(len(r_), np.float32), (r_, c_)), shape=(len(qtok), len(PRED_VOCAB) * n))\\n            EngineV3._QCACHE[key] = (qc, QT, QPB)\\n        self.q_concepts, self.QT, self.QPB = EngineV3._QCACHE[key]\\n        self.q_preds = qpreds\\n        s, d, e, w, pi = graph.edge_arrays()\\n        N = graph.n_vid\\n        is_train_vid = np.zeros(n, bool)\\n        is_train_vid[np.array(SPLIT[\\"train\\"])] = True\\n        other_vid = (np.arange(n) < N) & ~is_train_vid\\n        touch = other_vid[s] | other_vid[d]\\n        keep = {\\"train\\": ~touch, \\"transductive\\": np.ones(len(s), bool),\\n                \\"inductive\\": ~touch | ((e == ETYPE_ID[\\"contains_inv\\"]) & other_vid[d] & ~other_vid[s])}\\n        self.edges = {k: (torch.from_numpy(np.stack([s[m], d[m]])), torch.from_numpy(e[m]), torch.from_numpy(w[m]),\\n                          torch.from_numpy(pi[m])) for k, m in keep.items()}\\n        self.edge_stats = {k: int(m.sum()) for k, m in keep.items()}\\n        self.pmix = _torch_sparse(graph.pmix)\\n\\n    def edges_on(self, name):\\n        ei, et, ew, epi = self.edges[name]\\n        return ei.to(DEVICE), et.to(DEVICE), ew.to(DEVICE), epi.to(DEVICE), self.pmix.to(DEVICE)\\n\\n    def path_feats(self, cand, q_idx):\\n        cand, q_idx = np.asarray(cand), np.asarray(q_idx)\\n        if cand.ndim == 1:\\n            cand = cand[:, None]\\n        B, K = cand.shape\\n        out = np.zeros((B, K, len(PATTERNS)), np.float32)\\n        for s0 in range(0, B, CFG.eval_chunk):\\n            c, q = cand[s0:s0 + CFG.eval_chunk], q_idx[s0:s0 + CFG.eval_chunk]\\n            rows, inv_ = np.unique(c.ravel(), return_inverse=True)\\n            inv_ = inv_.reshape(c.shape)\\n            col = np.arange(len(q))[:, None]\\n            Qb = self.QT[q].T.tocsr()\\n            for p, name in enumerate(PATTERNS):\\n                if name.startswith(\\"P1m_\\"):\\n                    M = self.graph.R[name]\\n                    if M.nnz == 0:\\n                        continue\\n                    out[s0:s0 + len(q), :, p] = (M[rows] @ self.QPB[name[4:]][q].T).toarray()[inv_, col]\\n                else:\\n                    out[s0:s0 + len(q), :, p] = (self.graph.R[name][rows] @ Qb).toarray()[inv_, col]\\n        return np.log1p(out)\\n\\n    def clip_candidates(self, q_idx, gallery, k):\\n        gallery, q_idx = np.asarray(gallery), np.asarray(q_idx)\\n        G = self.vid_feat[torch.from_numpy(gallery)]\\n        k = min(int(k), len(gallery))\\n        C, S = [], []\\n        for s0 in range(0, len(q_idx), CFG.eval_chunk):\\n            top = (self.txt_feat[torch.from_numpy(q_idx[s0:s0 + CFG.eval_chunk])] @ G.T).topk(k, dim=1)\\n            C.append(gallery[top.indices.numpy()]); S.append(top.values.numpy())\\n        return np.concatenate(C), np.concatenate(S)\\n\\n\\ndef make_model(engine, sym_feat, mp_weight=\\"binary\\", init=\\"learned\\", pred_mode=\\"none\\"):\\n    return GraphScorerV3(engine.graph.n_nodes, engine.txt_feat.shape[1], engine.vid_feat, sym_feat, pred_emb=PRED_EMB,\\n                         mp_weight=mp_weight, init=init, pred_mode=pred_mode).to(DEVICE)\\n\\n\\ndef train_graph_scorer_v3(engine, train_q, sym_feat, seed=0, mp_weight=\\"binary\\", init=\\"learned\\", pred_mode=\\"none\\", verbose=False):\\n    \\"\\"\\"InfoNCE như v1/v2 (âm trong batch không trùng video + âm khó CLIP trong gallery TRAIN); chỉ dùng cạnh \'train\'.\\"\\"\\"\\n    train_q = np.asarray(train_q)\\n    CAPS.use(train_q, \\"gnn_train\\")\\n    set_seed(seed)\\n    gold_all = CAP_VID[train_q]\\n    train_vids = np.unique(gold_all)\\n    assert all(SPLIT_OF[int(v)] == \\"train\\" for v in train_vids)\\n    model = make_model(engine, sym_feat, mp_weight, init, pred_mode)\\n    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))\\n    params = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]\\n    scalar = [p for n_, p in params if n_.endswith(\\"gate\\") or n_ in (\\"path_w\\", \\"decay\\", \\"clip_logit\\")]\\n    sid = {id(p) for p in scalar}\\n    opt = torch.optim.AdamW([dict(params=[p for _, p in params if id(p) not in sid]),\\n                             dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],\\n                            lr=CFG.lr, weight_decay=CFG.weight_decay)\\n    edges = engine.edges_on(\\"train\\")\\n    vid_dev = engine.vid_feat.to(DEVICE)\\n    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)\\n    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])\\n\\n    def logits_for(cand_np, qi, h, qf):\\n        cand = torch.from_numpy(cand_np).to(DEVICE)\\n        pm = torch.from_numpy(engine.path_feats(cand_np, qi)).to(DEVICE)\\n        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))\\n        if CFG.train_with_clip:\\n            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum(\\"bd,bkd->bk\\", qf, vid_dev[cand]))\\n        return z / CFG.temperature\\n\\n    by_vid = defaultdict(list)\\n    for j, v in enumerate(gold_all):\\n        by_vid[int(v)].append(j)\\n    rng, hist = np.random.RandomState(seed), []\\n    for ep in range(CFG.epochs):\\n        model.train()\\n        tot, nb = 0.0, 0\\n        for _ in range(CFG.caption_passes):\\n            perm = rng.permutation(train_vids)\\n            for s0 in range(0, len(perm), CFG.train_bs):\\n                bv = perm[s0:s0 + CFG.train_bs]\\n                if len(bv) < 8:\\n                    continue\\n                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])\\n                qi, gi = train_q[sel], gold_all[sel]\\n                qf = engine.txt_feat[torch.from_numpy(qi)].to(DEVICE)\\n                h = model.encode_graph(*edges)\\n                loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf), torch.arange(len(gi), device=DEVICE))\\n                loss = loss + F.cross_entropy(logits_for(np.concatenate([gi[:, None], hard[sel]], 1), qi, h, qf),\\n                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))\\n                opt.zero_grad(); loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                opt.step()\\n                tot += loss.item(); nb += 1\\n        hist.append(tot / max(nb, 1))\\n        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):\\n            print(f\\"    epoch {ep:2d} | loss {hist[-1]:.4f} | w_path \\" +\\n                  \\" \\".join(f\\"{x:.2f}\\" for x in model.pattern_weights().detach().cpu().numpy()))\\n    model.eval()\\n    return model, hist\\n\\n\\n@torch.no_grad()\\ndef score_queries_v3(model, engine, q_idx, gallery, k, protocol=\\"inductive\\"):\\n    if isinstance(model, QCRGAT):                       # v4: cùng đầu ra, s_path = 0 (λ = 1)\\n        return score_queries_v4(model, engine, q_idx, gallery, k, protocol)\\n    model.eval()\\n    h = model.encode_graph(*engine.edges_on(protocol))\\n    q_idx = np.asarray(q_idx)\\n    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)\\n    s_gnn, s_path = np.zeros_like(s_clip), np.zeros_like(s_clip)\\n    for s0 in range(0, len(q_idx), CFG.eval_chunk):\\n        sl = slice(s0, s0 + CFG.eval_chunk)\\n        c = torch.from_numpy(cand[sl]).to(DEVICE)\\n        qf = engine.txt_feat[torch.from_numpy(q_idx[sl])].to(DEVICE)\\n        s_gnn[sl] = model.gnn_score(h, qf, c).cpu().numpy()\\n        s_path[sl] = model.path_score(torch.from_numpy(engine.path_feats(cand[sl], q_idx[sl])).to(DEVICE)).cpu().numpy()\\n    return dict(cand=cand, clip_raw=s_clip, path_raw=s_path, gnn_raw=s_gnn,\\n                s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s_gnn), s_path=znorm_per_row(s_path))\\n\\n\\ndef tune_on_val_v3(model, engine, part, k, clip_rank, lam_grid=None, protocol=\\"inductive\\"):\\n    \\"\\"\\"(α, λ) tối đa MRR trên VAL; lam_grid cố định cho ablation path-only (λ=0) / GNN-only (λ=1).\\"\\"\\"\\n    assert part[\\"name\\"] == \\"val\\", \\"tune chỉ nhận VAL\\"\\n    CAPS.use(part[\\"q_idx\\"], \\"tune\\")\\n    sc = score_queries_v3(model, engine, part[\\"q_idx\\"], part[\\"gallery\\"], k, protocol)\\n    grid, best = [], None\\n    for a in CFG.alpha_grid:\\n        for l in (lam_grid or CFG.lambda_grid):\\n            r = ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], a, l), part[\\"gold\\"], clip_rank)\\n            m = metrics_from_ranks(r, part[\\"gold\\"])\\n            grid.append(dict(alpha=a, lam=l, k=k, **{x: m[x] for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}, rank=r))\\n            if best is None or m[\\"MRR\\"] > best[\\"MRR\\"]:\\n                best = grid[-1]\\n    return (best[\\"alpha\\"], best[\\"lam\\"]), [{k_: v for k_, v in g_.items() if k_ != \\"rank\\"} for g_ in grid], best\\n\\n# ---------------------------------- v4: huấn luyện + chấm điểm QC-RGAT (cùng giao thức v3: InfoNCE, âm khó CLIP, chỉ video TRAIN) ----------------------------------\\n_RELDATA = {}\\n\\n\\ndef reldata_v4(engine):\\n    if id(engine) not in _RELDATA:\\n        _RELDATA[id(engine)] = RelDataV4(engine.graph)\\n    return _RELDATA[id(engine)]\\n\\n\\ndef make_model_v4(engine, design):\\n    return QCRGAT(reldata_v4(engine), TXT_FEAT.shape[1], d=CFG.v4_dim, T=design[\\"v4_layers\\"], rel=design[\\"v4_rel\\"],\\n                  qcond=design[\\"v4_qcond\\"], use_pred=design[\\"v4_pred\\"]).to(DEVICE)\\n\\n\\ndef train_v4(engine, train_q, design, seed=0, verbose=False):\\n    \\"\\"\\"Như train_graph_scorer_v3: in-batch (video không trùng) + 31 âm khó CLIP trong gallery TRAIN; logit = z(s) + c·z(cos CLIP).\\"\\"\\"\\n    train_q = np.asarray(train_q)\\n    CAPS.use(train_q, \\"gnn_train\\")\\n    set_seed(seed)\\n    gold_all = CAP_VID[train_q]\\n    train_vids = np.unique(gold_all)\\n    assert all(SPLIT_OF[int(v)] == \\"train\\" for v in train_vids)\\n    model = make_model_v4(engine, design)\\n    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))\\n    named = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]\\n    scal = [p for n_, p in named if n_ in QCRGAT.SCALARS]\\n    opt = torch.optim.AdamW([dict(params=[p for n_, p in named if n_ not in QCRGAT.SCALARS]),\\n                             dict(params=scal, lr=CFG.v4_lr * CFG.scalar_lr_mult, weight_decay=0.0)],\\n                            lr=CFG.v4_lr, weight_decay=CFG.weight_decay)\\n    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)\\n    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])\\n    txt_dev = torch.from_numpy(np.ascontiguousarray(TXT_FEAT, dtype=np.float32)).to(DEVICE)\\n\\n    def logits_for(qb, cand_np):\\n        cand = torch.from_numpy(cand_np).to(DEVICE)\\n        z = znorm_per_row(model.score(qb, cand))\\n        if CFG.train_with_clip:\\n            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum(\\"bd,bkd->bk\\", txt_dev[qb[\\"q\\"]], VID_T[cand]))\\n        return z / CFG.temperature\\n\\n    by_vid = defaultdict(list)\\n    for j, v in enumerate(gold_all):\\n        by_vid[int(v)].append(j)\\n    rng, hist = np.random.RandomState(seed), []\\n    for ep in range(CFG.epochs):\\n        model.train()\\n        tot, nb = 0.0, 0\\n        for _ in range(CFG.caption_passes):\\n            perm = rng.permutation(train_vids)\\n            for s0 in range(0, len(perm), CFG.train_bs):\\n                bv = perm[s0:s0 + CFG.train_bs]\\n                if len(bv) < 8:\\n                    continue\\n                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])\\n                gi = gold_all[sel]\\n                qb = model.D.query_batch(train_q[sel], model.use_pred)\\n                loss = F.cross_entropy(logits_for(qb, np.tile(gi, (len(gi), 1))), torch.arange(len(gi), device=DEVICE))\\n                loss = loss + F.cross_entropy(logits_for(qb, np.concatenate([gi[:, None], hard[sel]], 1)),\\n                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))\\n                opt.zero_grad(); loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                opt.step()\\n                tot += loss.item(); nb += 1\\n        hist.append(tot / max(nb, 1))\\n        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):\\n            print(f\\"    epoch {ep:2d} | loss {hist[-1]:.4f}\\")\\n    model.eval()\\n    return model, hist\\n\\n\\n@torch.no_grad()\\ndef score_queries_v4(model, engine, q_idx, gallery, k, protocol=\\"inductive\\", D=None):\\n    \\"\\"\\"Cùng đầu ra với score_queries_v3; s_gnn = điểm QC-RGAT, s_path = 0 (λ cố định = 1).\\"\\"\\"\\n    model.eval()\\n    D = D or model.D\\n    q_idx = np.asarray(q_idx)\\n    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)\\n    s = np.zeros_like(s_clip)\\n    bs = max(1, CFG.v4_score_bs)\\n    for s0 in range(0, len(q_idx), bs):\\n        qb = D.query_batch(q_idx[s0:s0 + bs], model.use_pred)\\n        s[s0:s0 + bs] = model.score(qb, torch.from_numpy(cand[s0:s0 + bs]).to(DEVICE), D).cpu().numpy()\\n    z = np.zeros_like(s)\\n    return dict(cand=cand, clip_raw=s_clip, path_raw=z, gnn_raw=s, s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s), s_path=z)\\n"}, {"cell_type": "markdown", "id": "c037", "metadata": {}, "source": "## 9. Chỉ số đánh giá\\n\\n* Mỗi **caption là một truy vấn**, có đúng một video dương. Notebook báo cáo R@1, R@5, R@10, MRR, nDCG@10, MedR và `MRR_video_macro` (macro-MRR theo video).\\n* Thứ hạng tính trên **toàn bộ gallery**: top-K được sắp lại, ngoài top-K giữ thứ hạng CLIP.\\n* Kiểm định cặp dùng **bootstrap theo cụm video**. Họ so sánh chính được hiệu chỉnh **Holm–Bonferroni**; luật chọn từng stage cũng hiệu chỉnh Holm trên các ứng viên của stage.\\n* Notebook báo cáo thêm: candidate recall, nhóm direct/indirect/other, khái niệm hiếm, vị từ hiếm/phổ biến/không có vị từ, số node/cạnh/bậc, thời gian chạy và bộ nhớ GPU."}, {"cell_type": "code", "execution_count": null, "id": "c038", "metadata": {}, "outputs": [], "source": "def ranks_with_fallback(cand, scores, gold, clip_rank):\\n    \\"\\"\\"Thứ hạng trên TOÀN gallery: trong top-K theo điểm xếp lại; ngoài top-K giữ thứ hạng CLIP (> K).\\"\\"\\"\\n    order = np.argsort(-scores, axis=1, kind=\\"stable\\")\\n    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold)[:, None]\\n    return np.where(hit.any(1), hit.argmax(1) + 1, np.maximum(clip_rank, cand.shape[1] + 1))\\n\\n\\ndef metrics_from_ranks(rank, gold=None, ks=(1, 5, 10)):\\n    rank = np.asarray(rank, np.float64)\\n    out = {f\\"R@{k}\\": float((rank <= k).mean()) for k in ks}\\n    out[\\"MRR\\"] = float((1.0 / rank).mean())\\n    out[\\"nDCG@10\\"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())\\n    out[\\"MedR\\"] = float(np.median(rank))\\n    out[\\"n_queries\\"] = int(len(rank))\\n    if gold is not None:\\n        _, inv = np.unique(np.asarray(gold), return_inverse=True)\\n        out[\\"MRR_video_macro\\"] = float((np.bincount(inv, 1.0 / rank) / np.bincount(inv)).mean())\\n        out[\\"n_videos\\"] = int(inv.max() + 1) if len(inv) else 0\\n    return out\\n\\n\\nMETRIC_KEYS = [\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\", \\"nDCG@10\\", \\"MedR\\", \\"MRR_video_macro\\"]\\n\\n\\ndef fmt_mean_std(vals):\\n    vals = np.asarray(vals, float)\\n    return f\\"{vals.mean():.4f} ± {vals.std():.4f}\\"\\n\\n\\ndef cluster_bootstrap_diff(rr_a, rr_b, gold, n_boot=5000, seed=0):\\n    \\"\\"\\"Δ = mean(rr_a − rr_b) theo truy vấn; khoảng tin cậy 95% bằng bootstrap THEO VIDEO (cụm caption).\\"\\"\\"\\n    d = np.asarray(rr_a) - np.asarray(rr_b)\\n    _, inv = np.unique(np.asarray(gold), return_inverse=True)\\n    S, C = np.bincount(inv, d), np.bincount(inv).astype(float)\\n    idx = np.random.RandomState(seed).randint(0, len(S), (n_boot, len(S)))\\n    boot = S[idx].sum(1) / C[idx].sum(1)\\n    return dict(diff=float(d.mean()), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),\\n                p_not_better=float((boot <= 0).mean()), a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum()))\\n\\n\\ndef holm(pvals):\\n    \\"\\"\\"p đã hiệu chỉnh Holm–Bonferroni (đơn điệu), cùng thứ tự với đầu vào.\\"\\"\\"\\n    p = np.asarray(pvals, float)\\n    m = len(p)\\n    if m == 0:\\n        return p\\n    order = np.argsort(p)\\n    adj = np.empty(m)\\n    run = 0.0\\n    for k, i in enumerate(order):\\n        run = max(run, min(1.0, (m - k) * p[i]))\\n        adj[i] = run\\n    return adj"}, {"cell_type": "markdown", "id": "c039", "metadata": {}, "source": "## 8. Giải thích v3\\n* Đường đi được liệt kê theo **chiều** cạnh, kèm loại cạnh và **vị từ** ở bước quan hệ.\\n  * Mẫu P1m chỉ dùng vị từ có trong truy vấn.\\n  * Giải thích hợp lệ khi: cạnh có thật, đúng loại, đúng chiều, đúng kiểu node; vị từ thuộc **đúng** cạnh theo **đúng** chiều và có trong truy vấn.\\n  * Có đối chứng âm: sửa pred_id thì phải bị phát hiện.\\n* Điểm liệt kê phải khớp ma trận ở mọi `path_mode`.\\n* Fidelity báo riêng ba mức: thay đổi **điểm**, **thứ hạng**, **quyết định** top-1 / top-10.\\n* Có phần đóng góp theo mẫu đường đi và độ nhạy theo seed (Jaccard).\\n* Xoá cạnh giải thích được so với xoá cạnh **đối chứng** cùng loại.\\n* Notebook báo cáo trọng số hiệu dụng `(1−α)(1−λ)` của nhánh đường đi."}, {"cell_type": "code", "execution_count": null, "id": "c040", "metadata": {}, "outputs": [], "source": "def enumerate_paths_v3(g, vid, concept_ids, qpred=None, PM=None):\\n    \\"\\"\\"Đường đi CỤ THỂ từ video tới khái niệm truy vấn, đi theo CHIỀU cạnh, ghi loại cạnh từng bước và VỊ TỪ của bước\\n    quan hệ. Trọng số = idf(đích) × Π trọng số trong ma trận đường đi (PM) của chế độ path hiện tại ⇒ khớp graph.R.\\n    Mẫu P1m_f (path_mode \'pred_match\'): bước a –f→ b chỉ tính phần vị từ có trong truy vấn (qpred[f] = tập pred_id).\\"\\"\\"\\n    PM = PM or g.PM\\n    target = {int(c) for c in concept_ids}\\n    C = PM[\\"C\\"]\\n    own = set(C.indices[C.indptr[vid]:C.indptr[vid + 1]].tolist())\\n    out = []\\n\\n    def nbrs(M, x):\\n        sl = slice(M.indptr[x], M.indptr[x + 1])\\n        return zip(M.indices[sl].tolist(), M.data[sl].tolist())\\n\\n    def add(pattern, nodes, rels, w, pred=None):\\n        c = nodes[-1]\\n        if pattern != \\"P0_contains\\" and CFG.drop_redundant_indirect and c in own:\\n            return\\n        w = float(g.idf[c]) * float(w)\\n        if w > 0:\\n            out.append(dict(pattern=pattern, nodes=[int(x) for x in nodes], rels=list(rels), w=w, pred=pred))\\n\\n    for o, w0 in nbrs(C, vid):\\n        if o in target:\\n            add(\\"P0_contains\\", [o], [\\"contains\\"], w0)\\n        for f in FAMILIES:\\n            for t in (FAMILY_REL[f], FAMILY_REL[f] + \\"_inv\\"):\\n                for o2, w1 in nbrs(PM[t], o):\\n                    if o2 in target:\\n                        add(f\\"P1_{f}\\", [o, o2], [\\"contains\\", t], w0 * w1)\\n                    for c, w2 in nbrs(PM[\\"isa\\"], o2):\\n                        if c in target:\\n                            add(f\\"P4_{f}_isa\\", [o, o2, c], [\\"contains\\", t, \\"isa\\"], w0 * w1 * w2)\\n        for c, w1 in nbrs(PM[\\"isa\\"], o):\\n            if c in target:\\n                add(\\"P2_isa\\", [o, c], [\\"contains\\", \\"isa\\"], w0 * w1)\\n        for o2, w1 in nbrs(PM[\\"related\\"], o):\\n            if o2 in target:\\n                add(\\"P3_related\\", [o, o2], [\\"contains\\", \\"related\\"], w0 * w1)\\n            for c, w2 in nbrs(PM[\\"isa\\"], o2):\\n                if c in target:\\n                    add(\\"P5_related_isa\\", [o, o2, c], [\\"contains\\", \\"related\\", \\"isa\\"], w0 * w1 * w2)\\n        if g.path_mode == \\"pred_match\\" and qpred:\\n            for rel in PRED_FAMILIES:\\n                want = set(qpred.get(rel, ()))\\n                if not want:\\n                    continue\\n                for t in (rel, rel + \\"_inv\\"):\\n                    for o2, w1 in nbrs(PM[t], o):\\n                        if o2 not in target:\\n                            continue\\n                        a, b = (o2, o) if t.endswith(\\"_inv\\") else (o, o2)\\n                        for pid, s in g.pshare[rel].get((a, b), {}).items():\\n                            if pid in want:\\n                                add(f\\"P1m_{rel}\\", [o, o2], [\\"contains\\", t], w0 * w1 * s, pred=(rel, int(pid), float(s)))\\n    return out\\n\\n\\ndef paths_to_counts_v3(paths, mode):\\n    \\"\\"\\"log1p(tổng theo mẫu) — với path_dedup: tổng theo khái niệm của đường đi MẠNH NHẤT (khớp _maxprod).\\"\\"\\"\\n    v = np.zeros(len(PATTERNS), np.float64)\\n    if mode == \\"dedup\\":\\n        best = defaultdict(float)\\n        for p in paths:\\n            k = (p[\\"pattern\\"], p[\\"nodes\\"][-1])\\n            best[k] = max(best[k], p[\\"w\\"])\\n        for (pat, _), w in best.items():\\n            v[PIDX[pat]] += w\\n    else:\\n        for p in paths:\\n            v[PIDX[p[\\"pattern\\"]]] += p[\\"w\\"]\\n    return np.log1p(v).astype(np.float32)\\n\\n\\ndef path_edges_v3(p, vid):\\n    \\"\\"\\"[(loại, u, v)] gồm cạnh contains đầu tiên.\\"\\"\\"\\n    nodes = [int(vid)] + p[\\"nodes\\"]\\n    return list(zip(p[\\"rels\\"], nodes[:-1], nodes[1:]))\\n\\n\\ndef edge_valid_v3(g, rel, u, v):\\n    \\"\\"\\"Cạnh có thật trong ma trận GỐC đúng loại + đúng chiều, và kiểu đầu/cuối khớp REL_SPEC.\\"\\"\\"\\n    M = g.A_con if rel == \\"contains\\" else g.A[rel]\\n    if M[u, v] == 0:\\n        return False\\n    ok_src = REL_SPEC[rel][\\"src\\"].split(\\":\\")[0].split(\\"|\\")\\n    ok_tgt = REL_SPEC[rel][\\"tgt\\"].split(\\" \\")[0].split(\\":\\")[0].split(\\"|\\")\\n    return g.inv.node_type[u] in ok_src and g.inv.node_type[v] in ok_tgt\\n\\n\\ndef pred_valid_v3(g, p, vid, qtokens):\\n    \\"\\"\\"Đường đi khớp vị từ: vị từ ghi trong giải thích phải (1) thuộc phân phối vị từ của ĐÚNG cạnh theo ĐÚNG chiều và\\n    (2) có mặt trong truy vấn. Đường đi không mang vị từ ⇒ True.\\"\\"\\"\\n    if p.get(\\"pred\\") is None:\\n        return True\\n    rel, pid, share = p[\\"pred\\"]\\n    t, u, v = path_edges_v3(p, vid)[1]\\n    a, b = (v, u) if t.endswith(\\"_inv\\") else (u, v)\\n    ok_edge = abs(g.pshare[rel].get((a, b), {}).get(pid, -1.0) - share) < 1e-9\\n    return bool(ok_edge and all(x in qtokens for x in predicate_tokens(PRED_VOCAB[pid][1])) and PRED_VOCAB[pid][0] == rel)\\n\\n\\ndef path_text_v3(g, p):\\n    nm = lambda i: g.inv.name[i]\\n    s = \\"video\\"\\n    nodes = p[\\"nodes\\"]\\n    for rel, a, b in zip(p[\\"rels\\"], [None] + nodes[:-1], nodes):\\n        lab = REL_TEXT.get(rel, rel)\\n        if \\"{pred}\\" in lab:\\n            if p.get(\\"pred\\") is not None:\\n                pr = f\\"{PRED_VOCAB[p[\'pred\'][1]][1]}({p[\'pred\'][2]:.2f})\\"\\n            else:\\n                d = g.predicates_of(rel, a, b)\\n                tot = max(sum(d.values()), 1)\\n                pr = \\"/\\".join(f\\"{k}({c / tot:.2f})\\" for k, c in sorted(d.items(), key=lambda x: (-x[1], x[0]))[:3]) or \\"?\\"\\n            lab = lab.format(pred=pr.replace(\\" \\", \\"_\\"))\\n        s += f\\" –{lab}→ {nm(b)}\\"\\n    return s\\n\\n\\ndef q_args(eng, q):\\n    \\"\\"\\"(node khái niệm của truy vấn, {họ: pred_id khớp}, tập token truy vấn).\\"\\"\\"\\n    return [nid for nid, _, _ in eng.q_concepts[q]], eng.q_preds[q], QPRED_TOK[q]\\n\\n\\n@torch.no_grad()\\ndef explain_and_score_v3(model, g, vid, concept_ids, qpred=None, qtokens=frozenset(), top_m=CFG.top_m_paths, rng_seed=0):\\n    paths = enumerate_paths_v3(g, vid, concept_ids, qpred)\\n    w = model.pattern_weights().detach().cpu().numpy()\\n    gain = F.softplus(model.path_gain).item()\\n    score = lambda ps: float(gain * (paths_to_counts_v3(ps, g.path_mode) * w).sum())\\n    full = score(paths)\\n    contrib = [full - score(paths[:i] + paths[i + 1:]) for i in range(len(paths))]\\n    order = np.argsort(-np.array(contrib), kind=\\"stable\\") if contrib else np.array([], int)\\n    ex = [int(i) for i in order[:top_m]]\\n    expl = [dict(text=path_text_v3(g, paths[i]), pattern=paths[i][\\"pattern\\"], contribution=float(contrib[i]),\\n                 nodes=paths[i][\\"nodes\\"], rels=paths[i][\\"rels\\"], pred=paths[i][\\"pred\\"]) for i in ex]\\n    keep = [paths[i] for i in ex]\\n    drop = [p for i, p in enumerate(paths) if i not in set(ex)]\\n    rng, rnd = np.random.RandomState(rng_seed), []\\n    for _ in range(10):\\n        r = set(rng.choice(len(paths), size=min(len(ex), len(paths)), replace=False).tolist()) if paths else set()\\n        rnd.append(score([p for i, p in enumerate(paths) if i not in r]))\\n    cset = {int(c) for c in concept_ids}\\n    valid = [p[\\"nodes\\"][-1] in cset and all(edge_valid_v3(g, *e) for e in path_edges_v3(p, vid)) for p in keep]\\n    pvalid = [pred_valid_v3(g, p, vid, qtokens) for p in keep]\\n    share = np.zeros(len(PATTERNS))\\n    for i in range(len(paths)):\\n        share[PIDX[paths[i][\\"pattern\\"]]] += max(contrib[i], 0)\\n    detail = dict(n_paths=len(paths), score_full=full, score_without_explanation=score(drop),\\n                  score_only_explanation=score(keep), score_without_random=float(np.mean(rnd)) if rnd else full,\\n                  sparsity=1 - len(ex) / max(len(paths), 1), validity=float(np.mean(valid)) if valid else 0.0,\\n                  predicate_validity=float(np.mean(pvalid)) if pvalid else 1.0,\\n                  n_predicate_paths=int(sum(p.get(\\"pred\\") is not None for p in keep)),\\n                  pattern_share={p: float(s / max(share.sum(), 1e-12)) for p, s in zip(PATTERNS, share) if s > 0})\\n    return expl, full, detail\\n\\n\\ndef _rank_in(fin, j):\\n    return int((fin > fin[j]).sum()) + 1\\n\\n\\ndef path_score_from_R(g, R, cids, qpred, gain, w):\\n    \\"\\"\\"s_path (chưa chuẩn hoá) từ một R đã tính lại (hàng = ứng viên) — cùng công thức với engine.path_feats.\\"\\"\\"\\n    n, cols = g.n_nodes, np.array(sorted({int(c) for c in cids}))\\n    feats = []\\n    for p in PATTERNS:\\n        if p.startswith(\\"P1m_\\"):\\n            pc = np.array(sorted(pid * n + c for pid in qpred.get(p[4:], ()) for c in cols), np.int64)\\n            feats.append(np.asarray(R[p][:, pc].sum(1)).ravel() if len(pc) and R[p].nnz else np.zeros(R[\\"P0_contains\\"].shape[0]))\\n        else:\\n            feats.append(np.asarray(R[p][:, cols].sum(1)).ravel())\\n    return gain * (np.log1p(np.stack(feats, 1)) * w).sum(1)\\n\\n\\n@torch.no_grad()\\ndef edge_deletion_v3(run, part, groups, n=CFG.n_edge_deletion, seed=0, protocol=\\"inductive\\"):\\n    \\"\\"\\"Với mỗi truy vấn (video đúng trong top-K): xoá THẬT (a) cạnh then chốt của đường đi giải thích top-1 và\\n    (b) đường đi QUAN HỆ đứng đầu; đối chứng: xoá một cạnh CÙNG LOẠI, KHÔNG nằm trên đường đi nào tới khái niệm\\n    của truy vấn. Tính lại R cho mọi ứng viên từ ma trận đã sửa (cùng công thức), ghi Δ s_path, Δ điểm cuối, hạng,\\n    và quyết định top-1 / top-10. GNN giữ cố định để cô lập nhánh đường đi.\\"\\"\\"\\n    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n    a, l, z = run[\\"alpha\\"], run[\\"lam\\"], znorm_per_row\\n    h = model.encode_graph(*eng.edges_on(protocol))\\n    w = model.pattern_weights().detach().cpu().numpy()\\n    gain = F.softplus(model.path_gain).item()\\n    rng = np.random.RandomState(seed)\\n    rows, used, mism = [], set(), 0\\n\\n    def delete(rel, u, v):\\n        A2 = dict(g.A)\\n        Acon = g.A_con\\n        if rel == \\"contains\\":\\n            M = g.A_con.tolil(copy=True); M[u, v] = 0; Acon = M.tocsr(); Acon.eliminate_zeros()\\n        else:\\n            M = g.A[rel].tolil(copy=True); M[u, v] = 0; A2[rel] = M.tocsr(); A2[rel].eliminate_zeros()\\n        return g.path_mats(A=A2, A_con=Acon)\\n\\n    for i in rng.permutation(len(part[\\"q_idx\\"])):\\n        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n        cids, qpred, qtok = q_args(eng, q)\\n        if gold in used or not cids:\\n            continue\\n        cand, s_clip = eng.clip_candidates(np.array([q]), part[\\"gallery\\"], CFG.rerank_k)\\n        cand, s_clip = cand[0], s_clip[0]\\n        hit = np.where(cand == gold)[0]\\n        if len(hit) == 0:\\n            continue\\n        expl, _, _ = explain_and_score_v3(model, g, gold, cids, qpred, qtok)\\n        expl = [e for e in expl if e[\\"contribution\\"] > 0]\\n        if not expl:\\n            continue\\n        j = int(hit[0])\\n        sp0 = path_score_from_R(g, g.compute_R(rows=cand), cids, qpred, gain, w)\\n        sp_eng = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)\\n        mism += int(not np.allclose(sp0, sp_eng, atol=1e-4))\\n        sg = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()\\n        f0 = combine(z(s_clip[None]), z(sg), z(sp0[None]), a, l)[0]\\n        r0, top0 = _rank_in(f0, j), int(cand[np.argmax(f0)])\\n        targets = [(\\"top1\\", expl[0])]\\n        relp = next((e for e in expl if e[\\"pattern\\"] != \\"P0_contains\\"), None)\\n        if relp is not None and relp is not expl[0]:\\n            targets.append((\\"top_relational\\", relp))\\n        on_path = {(t_, x, y) for p in enumerate_paths_v3(g, gold, cids, qpred) for t_, x, y in path_edges_v3(p, gold)}\\n        for which, p in targets:\\n            edges = path_edges_v3(p, gold)\\n            rel, u, v = edges[1] if len(edges) > 1 else edges[0]\\n            ctrl = None\\n            if rel != \\"contains\\":                        # cạnh đối chứng cùng loại, không nằm trên đường đi nào\\n                cand_e = [(x, y) for x, y in zip(*g.A[rel].nonzero()) if (rel, x, y) not in on_path]\\n                if cand_e:\\n                    ctrl = cand_e[rng.randint(len(cand_e))]\\n            todo = [(\\"explanation\\", (u, v))] + ([(\\"control\\", ctrl)] if ctrl is not None else [])\\n            for kind, (uu, vv) in todo:\\n                sp1 = path_score_from_R(g, g.compute_R(rows=cand, PM=delete(rel, uu, vv)), cids, qpred, gain, w)\\n                f1 = combine(z(s_clip[None]), z(sg), z(sp1[None]), a, l)[0]\\n                r1 = _rank_in(f1, j)\\n                rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], group=str(groups[i]), which=which, deleted=kind,\\n                                 explanation=p[\\"text\\"], pattern=p[\\"pattern\\"], relation=rel,\\n                                 predicate=PRED_VOCAB[p[\\"pred\\"][1]][1] if p.get(\\"pred\\") else \\"\\",\\n                                 edge=f\\"{g.inv.name[uu]} → {g.inv.name[vv]}\\", edge_valid=edge_valid_v3(g, rel, uu, vv),\\n                                 path_contribution=p[\\"contribution\\"], s_path_before=float(sp0[j]), s_path_after=float(sp1[j]),\\n                                 d_s_path=float(sp1[j] - sp0[j]), d_final=float(f1[j] - f0[j]), rank_before=r0, rank_after=r1,\\n                                 rank_changed=r1 != r0, top1_changed=int(cand[np.argmax(f1)]) != top0,\\n                                 top10_changed=(r0 <= 10) != (r1 <= 10),\\n                                 w_path_effective=(1 - a) * (1 - l), alpha=a, lam=l))\\n        used.add(gold)\\n        if len(used) >= n:\\n            break\\n    return pd.DataFrame(rows), mism\\n\\n\\n@torch.no_grad()\\ndef path_fidelity_v3(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0, protocol=\\"inductive\\"):\\n    \\"\\"\\"Fidelity mức đường đi (score) và mức thứ hạng của điểm cuối cho cùng một tập truy vấn; đóng góp theo mẫu đường đi.\\"\\"\\"\\n    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n    a, l, z = run[\\"alpha\\"], run[\\"lam\\"], znorm_per_row\\n    h = model.encode_graph(*eng.edges_on(protocol))\\n    w = model.pattern_weights().detach().cpu().numpy()\\n    gain = F.softplus(model.path_gain).item()\\n    sc = lambda ps: gain * float((paths_to_counts_v3(ps, g.path_mode) * w).sum())\\n    rng, rows, used = np.random.RandomState(seed), [], set()\\n    for i in rng.permutation(len(part[\\"q_idx\\"])):\\n        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n        cids, qpred, qtok = q_args(eng, q)\\n        if gold in used or not cids:\\n            continue\\n        cand, s_clip = eng.clip_candidates(np.array([q]), part[\\"gallery\\"], CFG.rerank_k)\\n        cand, s_clip = cand[0], s_clip[0]\\n        hit = np.where(cand == gold)[0]\\n        expl, s, det = explain_and_score_v3(model, g, gold, cids, qpred, qtok, top_m=top_m)\\n        if len(hit) == 0 or det[\\"n_paths\\"] <= top_m:\\n            continue\\n        paths = enumerate_paths_v3(g, gold, cids, qpred)\\n        j = int(hit[0])\\n        sp_raw = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)\\n        s_gnn = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()\\n\\n        def rank_with(kept):\\n            s_ = sp_raw.copy(); s_[j] = sc(kept)\\n            return _rank_in(combine(z(s_clip[None]), z(s_gnn), z(s_[None]), a, l)[0], j)\\n\\n        contrib = np.array([sc(paths) - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])\\n        top = set(np.argsort(-contrib, kind=\\"stable\\")[:top_m].tolist())\\n        rr = [rank_with([p for k, p in enumerate(paths) if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])\\n              for _ in range(10)]\\n        rf, rd = rank_with(paths), rank_with([p for k, p in enumerate(paths) if k not in top])\\n        rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], group=str(groups[i]),\\n                         **{k_: v for k_, v in det.items() if k_ != \\"pattern_share\\"},\\n                         **{f\\"share_{k_}\\": v for k_, v in det[\\"pattern_share\\"].items()},\\n                         matrix_vs_enumeration_ok=bool(abs(sc(paths) - sp_raw[j]) < 1e-4),\\n                         drop_explanation=det[\\"score_full\\"] - det[\\"score_without_explanation\\"],\\n                         drop_random=det[\\"score_full\\"] - det[\\"score_without_random\\"],\\n                         rank_full=rf, rank_drop_expl=rd, rank_drop_rand=float(np.mean(rr)),\\n                         rank_only_expl=rank_with([paths[k] for k in top]),\\n                         top10_changed_drop_expl=(rf <= 10) != (rd <= 10), w_path_effective=(1 - a) * (1 - l),\\n                         explanation=\\" || \\".join(e[\\"text\\"] for e in expl[:3])))\\n        used.add(gold)\\n        if len(rows) >= n_queries:\\n            break\\n    return pd.DataFrame(rows)\\n\\n\\n@torch.no_grad()\\ndef explanation_sensitivity(runs_by_seed, part, n=CFG.n_seed_sensitivity, top_m=CFG.top_m_paths, seed=0):\\n    \\"\\"\\"Jaccard của tập đường đi giải thích top-m cho CÙNG (truy vấn, video đúng) giữa các seed / cấu hình.\\"\\"\\"\\n    keys = list(runs_by_seed)\\n    if len(keys) < 2:\\n        return pd.DataFrame()\\n    eng0 = runs_by_seed[keys[0]][\\"engine\\"]\\n    rows, used = [], set()\\n    for i in np.random.RandomState(seed).permutation(len(part[\\"q_idx\\"])):\\n        q, gold = int(part[\\"q_idx\\"][i]), int(part[\\"gold\\"][i])\\n        cids, qpred, qtok = q_args(eng0, q)\\n        if gold in used or not cids:\\n            continue\\n        sets = {}\\n        for k in keys:\\n            r = runs_by_seed[k]\\n            ex, _, _ = explain_and_score_v3(r[\\"model\\"], r[\\"graph\\"], gold, cids, qpred, qtok, top_m=top_m)\\n            sets[k] = {e[\\"text\\"] for e in ex}\\n        if not any(sets.values()):\\n            continue\\n        js = [len(sets[x] & sets[y]) / max(len(sets[x] | sets[y]), 1) for x, y in itertools.combinations(keys, 2)]\\n        rows.append(dict(q=q, video_id=RECORDS[gold][\\"video_id\\"], n_configs=len(keys), mean_jaccard=float(np.mean(js)),\\n                         min_jaccard=float(np.min(js))))\\n        used.add(gold)\\n        if len(rows) >= n:\\n            break\\n    return pd.DataFrame(rows)\\n\\n# ---------------------------------- v4: giải thích = CHÍNH các số hạng của điểm (phân rã chính xác) ----------------------------------\\ndef v4_edge_info(D, k):\\n    \\"\\"\\"(nguồn, loại, đích, vị từ) của cạnh k trong RelDataV4 — đọc lại từ đồ thị gốc.\\"\\"\\"\\n    g = D.graph\\n    s, d, e, w, pi = g.edge_arrays()\\n    j = int(D.gidx[k])\\n    preds = {}\\n    if pi[j] >= 0:\\n        row = g.pmix[pi[j]]\\n        preds = {PRED_VOCAB[c][1]: float(v) for c, v in zip(row.indices, row.data)}\\n    return dict(src=int(s[j]), rel=EDGE_TYPES[int(e[j])], dst=int(d[j]), weight=float(w[j]), predicates=preds)\\n\\n\\ndef v4_path_text(D, t, vid):\\n    inv = D.graph.inv\\n    if t[\\"kind\\"] == \\"visual\\":\\n        return f\\"[thị giác] {inv.name[vid]}\\"\\n    c = D.N + t[\\"node\\"]\\n    if t[\\"kind\\"] == \\"contains\\":\\n        return f\\"{inv.name[vid]} –chứa→ {inv.name[c]}\\"\\n    hops = []\\n    for k in t[\\"edges\\"]:\\n        ei = v4_edge_info(D, k)\\n        p = max(ei[\\"predicates\\"], key=ei[\\"predicates\\"].get) if ei[\\"predicates\\"] else \\"\\"\\n        hops.append((inv.name[ei[\\"src\\"]], ei[\\"rel\\"] + (f\\"[{p}]\\" if p else \\"\\"), inv.name[ei[\\"dst\\"]]))\\n    chain = hops[0][0] + \\"\\".join(f\\" –{r}→ {b}\\" for _, r, b in hops)\\n    return f\\"{chain} ←chứa– {inv.name[vid]}\\"\\n\\n\\ndef v4_edge_valid(D, k, q):\\n    \\"\\"\\"Cạnh có thật (đúng loại, chiều, kiểu node theo REL_SPEC); nếu truy vấn có vị từ khớp cạnh thì vị từ đó thuộc ĐÚNG cạnh.\\"\\"\\"\\n    ei = v4_edge_info(D, k)\\n    g, inv = D.graph, D.graph.inv\\n    ok = ei[\\"rel\\"] in g.A and g.A[ei[\\"rel\\"]][ei[\\"src\\"], ei[\\"dst\\"]] > 0\\n    ok = ok and inv.node_type[ei[\\"src\\"]] != \\"video\\" and inv.node_type[ei[\\"dst\\"]] != \\"video\\"\\n    ids = {PRED_VOCAB[p][1] for rel in PRED_FAMILIES for p in QPREDS[q][rel]}\\n    matched = ids & set(ei[\\"predicates\\"])\\n    return bool(ok), bool(matched), bool(not matched or matched <= set(ei[\\"predicates\\"]))\\n\\n\\n@torch.no_grad()\\ndef explain_v4(model, q, vid, top_m=CFG.top_m_paths):\\n    \\"\\"\\"Giải thích \\"vì sao video được trả về\\" = top-m đường đi quan hệ ỦNG HỘ (đóng góp > 0); đường đi PHẢN ĐỐI (< 0) ghi riêng.\\n    Σ mọi số hạng (thị giác + nhãn + mọi đường đi) = điểm mô hình.\\"\\"\\"\\n    terms = model.decompose(q, vid)\\n    tot = {k: sum(t[\\"contrib\\"] for t in terms if t[\\"kind\\"] == k) for k in (\\"visual\\", \\"contains\\", \\"relation\\")}\\n    rel = [t for t in terms if t[\\"kind\\"] == \\"relation\\"]\\n    sup = sorted([t for t in rel if t[\\"contrib\\"] > 0], key=lambda t: -t[\\"contrib\\"])[:top_m]\\n    opp = sorted([t for t in rel if t[\\"contrib\\"] < 0], key=lambda t: t[\\"contrib\\"])[:top_m]\\n    return dict(terms=terms, top_relation=sup, opposing=opp, totals=tot, score=sum(t[\\"contrib\\"] for t in terms))\\n\\n\\n@torch.no_grad()\\ndef v4_rank_after(run, part_q, gold, cand, s_clip_z, D2):\\n    \\"\\"\\"Thứ hạng video đúng trong top-K (điểm cuối với α đã đóng băng) khi mô hình dùng dữ liệu đồ thị D2.\\"\\"\\"\\n    m = run[\\"model\\"]\\n    qb = D2.query_batch([part_q], m.use_pred)\\n    s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE), D2).cpu().numpy()\\n    fin = run[\\"alpha\\"] * s_clip_z[None] + (1 - run[\\"alpha\\"]) * znorm_per_row(s)\\n    j = int(np.where(cand == gold)[0][0])\\n    return int((fin[0] > fin[0, j]).sum()) + 1, s[0, j], fin[0]\\n\\n\\ndef xai_v4(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0):\\n    \\"\\"\\"Với ≥ n truy vấn có video đúng trong top-K và có đường đi quan hệ ỦNG HỘ: (1) hợp lệ (cạnh thật, vị từ đúng cạnh), (2) độ thưa,\\n    (3) fidelity: xoá THẬT các cạnh của top-m đường đi giải thích ⇒ Δ điểm mô hình so với dự đoán từ phân rã (khớp = chính xác),\\n    so với xoá cùng số cạnh quan hệ NGẪU NHIÊN nối vào nhãn của video; (4) thứ hạng / top-1 / top-10 đổi (báo riêng).\\"\\"\\"\\n    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n    D = m.D\\n    rng = np.random.RandomState(seed)\\n    q_all = part[\\"q_idx\\"]\\n    order = rng.permutation(len(q_all))\\n    cand_all, s_clip = eng.clip_candidates(q_all, part[\\"gallery\\"], CFG.rerank_k)\\n    rows = []\\n    for i in order:\\n        if len(rows) >= n_queries:\\n            break\\n        q, gold = int(q_all[i]), int(part[\\"gold\\"][i])\\n        cand = cand_all[i]\\n        if gold not in cand:\\n            continue\\n        ex = explain_v4(m, q, gold, top_m)\\n        if not ex[\\"top_relation\\"]:\\n            continue\\n        zc = znorm_per_row(s_clip[i:i + 1])[0]\\n        r_full, s_full, _ = v4_rank_after(run, q, gold, cand, zc, D)\\n        e_del = sorted({k for t in ex[\\"top_relation\\"] for k in t[\\"edges\\"]})\\n        D_ex = RelDataV4(D.graph, drop_edges=D.gidx[e_del])\\n        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, D_ex)\\n        tags = set(torch.nonzero(D.Acon[gold] > 0).flatten().tolist())\\n        pool = [k for k in range(D.E) if int(D.dst_np[k]) in tags and k not in e_del]\\n        e_rand = list(rng.choice(pool, min(len(e_del), len(pool)), replace=False)) if pool else []\\n        D_rd = RelDataV4(D.graph, drop_edges=D.gidx[e_rand]) if e_rand else D\\n        r_rd, s_rd, _ = v4_rank_after(run, q, gold, cand, zc, D_rd)\\n        expect = sum(t[\\"contrib\\"] for t in ex[\\"terms\\"] if t[\\"kind\\"] == \\"relation\\" and set(t[\\"edges\\"]) & set(e_del))\\n        val = [v4_edge_valid(D, k, q) for k in e_del]\\n        _, _, fin_full = v4_rank_after(run, q, gold, cand, zc, D)\\n        rows.append(dict(q=q, gold=gold, group=groups[i], n_terms=len(ex[\\"terms\\"]), n_relation_terms=sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"terms\\"]),\\n                         n_expl_paths=len(ex[\\"top_relation\\"]), n_expl_edges=len(e_del), sparsity=len(ex[\\"top_relation\\"]) / max(1, sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"terms\\"])),\\n                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[2] for v in val])),\\n                         n_predicate_edges=int(sum(v[1] for v in val)),\\n                         share_relation=ex[\\"totals\\"][\\"relation\\"] / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n                         s_full=s_full, d_s_expl=s_ex - s_full, d_s_expected=-expect, d_s_rand=s_rd - s_full,\\n                         decomposition_ok=float(abs(ex[\\"score\\"] - s_full) < 1e-4), deletion_matches_decomposition=float(abs((s_ex - s_full) + expect) < 1e-4) if m.T == 1 else float(\\"nan\\"),\\n                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,\\n                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),\\n                         top10_changed_expl=float(set(np.argsort(-fin_ex)[:10]) != set(np.argsort(-fin_full)[:10])),\\n                         explanation=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {v4_path_text(D, t, gold)}\\" for t in ex[\\"top_relation\\"]),\\n                         opposing=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {v4_path_text(D, t, gold)}\\" for t in ex[\\"opposing\\"][:3])))\\n    return pd.DataFrame(rows)\\n\\n\\n@torch.no_grad()\\ndef relation_analysis_v4(run, part, max_q=3000, seed=0):\\n    \\"\\"\\"Theo họ quan hệ: phần đóng góp vào điểm của video đúng, knockout lúc suy luận (bỏ mọi cạnh của họ) ⇒ ΔMRR;\\n    attention trung bình trên cạnh có vị từ KHỚP truy vấn vs không khớp; theo vai của node nguồn trong truy vấn.\\"\\"\\"\\n    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n    D = m.D\\n    q = part[\\"q_idx\\"]\\n    if len(q) > max_q:\\n        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))\\n    gold = CAP_VID[q]\\n    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n    sc = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k)\\n    base = metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n                                                  gold, clip_rank))[\\"MRR\\"]\\n    rows = []\\n    fam_of = np.array([REL_SPEC[V4_REL_TYPES[int(t)]][\\"family\\"] for t in D.et.cpu().numpy()])\\n    for fam in sorted(set(fam_of)):\\n        D2 = RelDataV4(D.graph, drop_edges=D.gidx[np.where(fam_of == fam)[0]])\\n        sc2 = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k, D=D2)\\n        mrr = metrics_from_ranks(ranks_with_fallback(sc2[\\"cand\\"], combine(sc2[\\"s_clip\\"], sc2[\\"s_gnn\\"], sc2[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n                                                     gold, clip_rank))[\\"MRR\\"]\\n        rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=fam, n_edges=int((fam_of == fam).sum()),\\n                         knockout_MRR=mrr, d_MRR_knockout=mrr - base))\\n    D0 = RelDataV4(D.graph, drop_edges=D.gidx)\\n    sc0 = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k, D=D0)\\n    mrr0 = metrics_from_ranks(ranks_with_fallback(sc0[\\"cand\\"], combine(sc0[\\"s_clip\\"], sc0[\\"s_gnn\\"], sc0[\\"s_path\\"], run[\\"alpha\\"], 1.0),\\n                                                  gold, clip_rank))[\\"MRR\\"]\\n    rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=\\"ALL_relations\\", n_edges=D.E, knockout_MRR=mrr0, d_MRR_knockout=mrr0 - base))\\n    att_m, att_n, by_role = [], [], defaultdict(list)\\n    for s0 in range(0, min(len(q), 1000), 128):\\n        qb = D.query_batch(q[s0:s0 + 128], m.use_pred)\\n        a = m.attention(qb, m.q(qb[\\"tq\\"])) if m.rel and D.E else None\\n        if a is None:\\n            break\\n        pe = (D.pidx_np >= 0)\\n        exm = qb[\\"ex\\"].cpu().numpy() > 0\\n        A = a.cpu().numpy()\\n        att_m.extend(A[exm & pe[None]].tolist()); att_n.extend(A[~exm & pe[None]].tolist())\\n        rs = qb[\\"role_src\\"][:, D.src].cpu().numpy()\\n        for r in range(len(Q_ROLES)):\\n            if (rs == r).any():\\n                by_role[Q_ROLES[r]].extend(A[rs == r].tolist())\\n    rows.append(dict(section=\\"predicate_attention\\", split=part[\\"name\\"], item=\\"pred_edges\\",\\n                     att_pred_match=float(np.mean(att_m)) if att_m else float(\\"nan\\"),\\n                     att_pred_nomatch=float(np.mean(att_n)) if att_n else float(\\"nan\\"), n_match=len(att_m)))\\n    for r, v in by_role.items():\\n        rows.append(dict(section=\\"role_attention\\", split=part[\\"name\\"], item=r, att_mean=float(np.mean(v)), n=len(v)))\\n    return pd.DataFrame(rows), dict(base_MRR=base, all_relations_knockout_MRR=mrr0,\\n                                    att_pred_match=float(np.mean(att_m)) if att_m else float(\\"nan\\"),\\n                                    att_pred_nomatch=float(np.mean(att_n)) if att_n else float(\\"nan\\"))\\n"}, {"cell_type": "markdown", "id": "c041", "metadata": {}, "source": "## 9. Khung thí nghiệm\\nMỗi thí nghiệm có: `experiment_id`, stage, giả thuyết, thiết kế đầy đủ, seed, checkpoint fp16, số cạnh theo loại, số node, số tham số, (α, λ) chọn trên val, thời gian chạy, đỉnh bộ nhớ GPU, trạng thái. Thí nghiệm không chạy được ghi `NOT RUN` kèm lý do.\\n\\n`decide()` áp luật đăng ký trước trên **val**; incumbent là phương án đơn giản hơn. `PRIMARY_COMPARISONS` là họ so sánh chính ở mục 0c."}, {"cell_type": "code", "execution_count": null, "id": "c042", "metadata": {}, "outputs": [], "source": "DESIGN_BASE = dict(taxonomy=\\"heuristic\\", contains=\\"binary\\", conf_mode=\\"sigmoid\\", rel_policy=\\"all\\", path_mode=\\"weighted\\",\\n                   mp_weight=\\"binary\\", init=\\"learned\\", mask=\\"none\\", rewire=False, pred_shuffle=False, pred_mode=\\"none\\",\\n                   related=\\"base\\", protocol=\\"inductive\\", tags=\\"base\\",\\n                   model=\\"v3\\", v4_layers=1, v4_rel=True, v4_qcond=True, v4_pred=True)        # model: v3 (TypedSAGE) | v4 (QC-RGAT)\\nV3_FINAL = dict(DESIGN_BASE, taxonomy=\\"none\\", mask=\\"no_related\\")   # cấu hình cuối của v3 (E44, chọn trên VAL ở Kaggle) = đồ thị của v4\\nRELATED_CFGS = {\\"base\\": (CFG.cooc_min_count, CFG.cooc_min_npmi), \\"npmi20\\": (CFG.cooc_min_count, 0.20),\\n                \\"count10\\": (10, CFG.cooc_min_npmi), \\"conceptnet\\": (\\"conceptnet\\", 0.0)}\\nGRAPHS, EXPS, DECISIONS, NOT_RUN = {}, {}, [], []\\nREWIRE_SEED_BASE, SHUFFLE_SEED_BASE = 1000, 2000\\nVERIFIED_CELLS = None            # {(họ, mức phổ biến)} đạt ngưỡng precision — chỉ có khi có nhãn người chấm\\n\\n\\ndef masked_types(mask):\\n    \\"\\"\\"mask = tên trong MASKS, có thể ghép bằng \'+\' (ví dụ \'no_related+no_taxonomy\').\\"\\"\\"\\n    return tuple(sorted({t for m in str(mask).split(\\"+\\") for t in MASKS[m]}))\\n\\n\\ndef add_mask(mask, extra):\\n    parts = sorted({m for m in str(mask).split(\\"+\\") + [extra] if m != \\"none\\"})\\n    return \\"+\\".join(parts) or \\"none\\"\\n\\n\\ndef graph_key(d, seed):\\n    return (d[\\"taxonomy\\"], d[\\"contains\\"], d[\\"conf_mode\\"] if d[\\"contains\\"] == \\"conf\\" else \\"\\", d[\\"rel_policy\\"],\\n            d[\\"path_mode\\"], d[\\"mask\\"], (REWIRE_SEED_BASE + seed) if d[\\"rewire\\"] else None,\\n            (SHUFFLE_SEED_BASE + seed) if d[\\"pred_shuffle\\"] else None, d[\\"tags\\"], d[\\"related\\"])\\n\\n\\ndef graph_for(d, seed):\\n    key = graph_key(d, seed)\\n    if key not in GRAPHS:\\n        tags = ALT_TAGS[d[\\"tags\\"]] if d[\\"tags\\"] != \\"base\\" else None\\n        scores = ALT_SCORES[d[\\"tags\\"]] if d[\\"tags\\"] != \\"base\\" else TAG_SCORES\\n        g = GraphV3(INV, RECORDS, SPLIT[\\"train\\"], RELATIONS2, TAXONOMY[d[\\"taxonomy\\"]], tag_scores=scores,\\n                    contains=d[\\"contains\\"], conf_mode=d[\\"conf_mode\\"], conf_cal=CONF_CAL, rel_policy=d[\\"rel_policy\\"],\\n                    path_mode=d[\\"path_mode\\"], masked=masked_types(d[\\"mask\\"]), rewire_seed=key[6], pred_shuffle_seed=key[7],\\n                    tags_override=tags, related_cfg=RELATED_CFGS[d[\\"related\\"]], verified_cells=VERIFIED_CELLS,\\n                    name=\\"|\\".join(str(x) for x in key))\\n        assert g.n_nodes == INV.n_nodes and g.inv is INV, \\"inventory phải cố định\\"\\n        GRAPHS[key] = (g, EngineV3(g, VID_FEAT, TXT_FEAT, QTOK2, QPREDS))\\n    return GRAPHS[key]\\n\\n\\ndef make_part(name, std_only=False):\\n    \\"\\"\\"Tập truy vấn + gallery. \'test\' (MSVD, thăm dò) và \'ext\' (xác nhận) chỉ mở sau khi đóng băng.\\n    std_only: chỉ caption chuẩn 1k-A của tập ngoài (truy vấn PHỤ).\\"\\"\\"\\n    if name in (\\"test\\", \\"ext\\"):\\n        TEST_GATE.check(f\\"make_part({name})\\", name)\\n    q = np.where((CAP_SPLIT == name) & (CAP_STD if std_only else True))[0]\\n    return dict(name=name + (\\"_std\\" if std_only else \\"\\"), split=name, q_idx=q, gold=CAP_VID[q], gallery=np.array(SPLIT[name]))\\n\\n\\ndef _mean_std(vals):\\n    return float(np.mean(vals)), float(np.std(vals))\\n\\n\\ndef not_run(exp_id, stage, desc, reason, design=None, hypothesis=\\"\\"):\\n    \\"\\"\\"Ghi thí nghiệm KHÔNG chạy được (kèm lý do) vào manifest — không tạo số liệu thay thế.\\"\\"\\"\\n    EXPS[exp_id] = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=design, seeds=[],\\n                        runs={}, status=\\"NOT RUN\\", reason=reason, val_MRR=(float(\\"nan\\"), float(\\"nan\\")),\\n                        val_R1=(float(\\"nan\\"), float(\\"nan\\")), edges={}, n_params=0, seconds=0.0)\\n    NOT_RUN.append(dict(stage=stage, experiment_id=exp_id, reason=reason))\\n    print(f\\"  [{stage}] {exp_id:28s} NOT RUN — {reason}\\")\\n    return EXPS[exp_id]\\n\\n\\ndef run_exp(exp_id, stage, design, desc, hypothesis=\\"\\", seeds=None, lam_fixed=None, reuse=None):\\n    \\"\\"\\"Huấn luyện (hoặc tái dùng mô hình của `reuse`) cho từng seed; chọn (α, λ) trên VAL; ghi manifest.\\"\\"\\"\\n    seeds = seeds or CFG.seeds\\n    rec = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=dict(design),\\n               seeds=list(seeds), lam_fixed=lam_fixed, reuse=reuse, status=\\"RUNNING\\", runs={})\\n    EXPS[exp_id] = rec\\n    for seed in seeds:\\n        g, eng = graph_for(design, seed)\\n        t0 = time.time()\\n        if DEVICE == \\"cuda\\":\\n            torch.cuda.reset_peak_memory_stats()\\n        if reuse is not None:\\n            model, hist = EXPS[reuse][\\"runs\\"][seed][\\"model\\"], EXPS[reuse][\\"runs\\"][seed][\\"hist\\"]\\n        elif design.get(\\"model\\") == \\"v4\\":\\n            model, hist = train_v4(eng, TRAIN_Q, design, seed=seed)\\n        else:\\n            model, hist = train_graph_scorer_v3(eng, TRAIN_Q, SYM_FEAT, seed=seed, mp_weight=design[\\"mp_weight\\"],\\n                                                init=design[\\"init\\"], pred_mode=design[\\"pred_mode\\"])\\n        lam_v = 1.0 if design.get(\\"model\\") == \\"v4\\" else lam_fixed           # v4: một điểm đồ thị duy nhất (s_path = 0)\\n        (a, l), grid, best = tune_on_val_v3(model, eng, PV, CFG.rerank_k, CLIP_RANK_VAL,\\n                                            lam_grid=[lam_v] if lam_v is not None else None, protocol=design[\\"protocol\\"])\\n        ck = None\\n        if CFG.save_checkpoints and reuse is None:\\n            ck = CFG.out_dir / \\"checkpoints\\" / f\\"{exp_id}_s{seed}.pt\\"\\n            torch.save({k_: v.half() if v.is_floating_point() else v for k_, v in model.state_dict().items()}, ck)\\n        rec[\\"runs\\"][seed] = dict(graph=g, engine=eng, model=model, hist=hist, alpha=a, lam=l, grid=grid,\\n                                 val={k_: best[k_] for k_ in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}, rr_val=1.0 / best[\\"rank\\"],\\n                                 seconds=round(time.time() - t0, 1), checkpoint=str(ck) if ck else None,\\n                                 gpu_peak_mb=round(torch.cuda.max_memory_allocated() / 2 ** 20, 1) if DEVICE == \\"cuda\\" else None)\\n    r0 = rec[\\"runs\\"][seeds[0]]\\n    rec.update(status=\\"DONE\\", val_MRR=_mean_std([r[\\"val\\"][\\"MRR\\"] for r in rec[\\"runs\\"].values()]),\\n               val_R1=_mean_std([r[\\"val\\"][\\"R@1\\"] for r in rec[\\"runs\\"].values()]),\\n               rr_val=np.mean([r[\\"rr_val\\"] for r in rec[\\"runs\\"].values()], 0),\\n               edges=r0[\\"graph\\"].edge_counts(), n_nodes=r0[\\"graph\\"].n_nodes,\\n               n_params=int(sum(p.numel() for p in r0[\\"model\\"].parameters())),\\n               alpha_lam=[(r[\\"alpha\\"], r[\\"lam\\"]) for r in rec[\\"runs\\"].values()],\\n               seconds=float(sum(r[\\"seconds\\"] for r in rec[\\"runs\\"].values())),\\n               gpu_peak_mb=max([r[\\"gpu_peak_mb\\"] or 0 for r in rec[\\"runs\\"].values()]) or None)\\n    m, s = rec[\\"val_MRR\\"]\\n    print(f\\"  [{stage}] {exp_id:28s} VAL MRR {m:.4f} ± {s:.4f} | α,λ {rec[\'alpha_lam\']} | \\"\\n          f\\"{rec[\'seconds\']:.0f}s | cạnh quan hệ {sum(v for k_, v in rec[\'edges\'].items() if not k_.startswith(\'contains\'))}\\")\\n    return rec\\n\\n\\ndef design_value(d, field):\\n    return \\"/\\".join(str((d or {}).get(f)) for f in field.split(\\"|\\"))\\n\\n\\ndef decide(stage, incumbent, candidates, field):\\n    \\"\\"\\"Luật đăng ký trước: incumbent = phương án ĐƠN GIẢN hơn (hoặc thiết kế đang dùng). Ứng viên chỉ được chấp nhận khi\\n    Δ MRR val (trung bình seed) ≥ adopt_min_delta VÀ p(Δ ≤ 0) — bootstrap theo video, hiệu chỉnh Holm trên các ứng viên\\n    của stage — < adopt_max_p. Nhiều ứng viên đạt ⇒ chọn Δ lớn nhất. Ứng viên NOT RUN bị bỏ qua (ghi lại).\\"\\"\\"\\n    base = EXPS[incumbent]\\n    cands = [c for c in candidates if EXPS[c][\\"status\\"] == \\"DONE\\"]\\n    rows = []\\n    for c in cands:\\n        e = EXPS[c]\\n        d = e[\\"val_MRR\\"][0] - base[\\"val_MRR\\"][0]\\n        bs = cluster_bootstrap_diff(e[\\"rr_val\\"], base[\\"rr_val\\"], PV[\\"gold\\"])\\n        rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=design_value(e[\\"design\\"], field),\\n                         delta_val_MRR=d, ci_lo=bs[\\"ci_lo\\"], ci_hi=bs[\\"ci_hi\\"], p_not_better=bs[\\"p_not_better\\"]))\\n    adj = holm([r[\\"p_not_better\\"] for r in rows])\\n    best = None\\n    for r, pa in zip(rows, adj):\\n        r[\\"p_holm\\"] = float(pa)\\n        r[\\"passes\\"] = bool(r[\\"delta_val_MRR\\"] >= CFG.adopt_min_delta and pa < CFG.adopt_max_p)\\n        if r[\\"passes\\"] and (best is None or r[\\"delta_val_MRR\\"] > best[1]):\\n            best = (r[\\"candidate\\"], r[\\"delta_val_MRR\\"])\\n    for r in rows:\\n        r[\\"adopted\\"] = best is not None and r[\\"candidate\\"] == best[0]\\n    for c in candidates:\\n        if EXPS[c][\\"status\\"] != \\"DONE\\":\\n            rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=None, delta_val_MRR=float(\\"nan\\"),\\n                             ci_lo=float(\\"nan\\"), ci_hi=float(\\"nan\\"), p_not_better=float(\\"nan\\"), p_holm=float(\\"nan\\"),\\n                             passes=False, adopted=False, note=f\\"NOT RUN: {EXPS[c].get(\'reason\', \'\')}\\"))\\n    DECISIONS.extend(rows)\\n    winner = best[0] if best else incumbent\\n    print(f\\"  → {stage}: {\'CHẤP NHẬN \' + winner if best else \'giữ \' + incumbent} ({field} = {design_value(EXPS[winner][\'design\'], field)})\\")\\n    return winner\\n\\n\\n@torch.no_grad()\\ndef evaluate_exp(exp_id, part, clip_rank, protocol=None):\\n    \\"\\"\\"Đánh giá MỘT lần mỗi seed với (α, λ) đã đóng băng; trả về thứ hạng theo truy vấn.\\"\\"\\"\\n    e = EXPS[exp_id]\\n    out = {}\\n    for seed, r in e[\\"runs\\"].items():\\n        sc = score_queries_v3(r[\\"model\\"], r[\\"engine\\"], part[\\"q_idx\\"], part[\\"gallery\\"], CFG.rerank_k,\\n                              protocol or e[\\"design\\"][\\"protocol\\"])\\n        fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], r[\\"alpha\\"], r[\\"lam\\"])\\n        out[seed] = dict(rank=ranks_with_fallback(sc[\\"cand\\"], fin, part[\\"gold\\"], clip_rank),\\n                         cand_hash=hashlib.sha1(sc[\\"cand\\"].tobytes()).hexdigest()[:12])\\n    return out\\n\\n\\ndef query_groups_v3(engine, q_idx):\\n    \\"\\"\\"direct | indirect | other (như v1/v2) trên đồ thị của engine.\\"\\"\\"\\n    g, q_idx = engine.graph, np.asarray(q_idx)\\n    gold = CAP_VID[q_idx]\\n    pf = engine.path_feats(gold[:, None], q_idx)[:, 0]\\n    out = []\\n    for j, q in enumerate(q_idx):\\n        cids = {n_ for n_, _, _ in engine.q_concepts[q]}\\n        out.append(\\"direct\\" if cids & set(g.vid_tags[gold[j]]) else (\\"indirect\\" if pf[j, 1:].sum() > 0 else \\"other\\"))\\n    return np.array(out)\\n\\n\\ndef rare_mask(engine, q_idx, thr=5):\\n    \\"\\"\\"Truy vấn \'khái niệm hiếm\': mọi node lớp khớp được có tần suất gắn nhãn trên video TRAIN ≤ thr (hoặc chỉ khớp khái niệm).\\"\\"\\"\\n    g = engine.graph\\n    out = []\\n    for q in q_idx:\\n        cls = [n_ for n_, _, _ in engine.q_concepts[q] if g.inv.is_class(n_)]\\n        out.append(bool(engine.q_concepts[q]) and (not cls or max(g.cls_df[c] for c in cls) <= thr))\\n    return np.array(out)\\n\\n\\ndef predicate_groups(engine, q_idx, thr=None):\\n    \\"\\"\\"Chỉ xét vị từ của truy vấn CÓ trên cạnh của đồ thị (svo/prep): \'no_pred\' nếu không có; \'pred_rare\' nếu MỌI vị từ khớp\\n    đều có support TRAIN ≤ thr (truy vấn chỉ dựa vào vị từ hiếm); ngược lại \'pred_common\'.\\"\\"\\"\\n    thr = thr or 2 * CFG.pred_min_support\\n    used = {pid for rel in PRED_FAMILIES for d in engine.graph.pshare[rel].values() for pid in d}\\n    out = []\\n    for q in q_idx:\\n        pids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel] if p in used]\\n        if not pids:\\n            out.append(\\"no_pred\\")\\n        else:\\n            out.append(\\"pred_rare\\" if max(PRED_SUPPORT.get(PRED_VOCAB[p], 0) for p in pids) <= thr else \\"pred_common\\")\\n    return np.array(out)\\n\\n\\n@torch.no_grad()\\ndef path_analysis(run, part, protocol=\\"inductive\\", max_q=4000, seed=0):\\n    \\"\\"\\"Theo mẫu: phần đóng góp, knockout từng mẫu và từng HỌ quan hệ lúc suy luận; theo node: bậc, số đường đi đi qua,\\n    phạm vi video; tổng hợp: ρ(degree, s_path), ρ(degree, điểm cuối), P(degree video đúng > video âm).\\"\\"\\"\\n    g, eng, model = run[\\"graph\\"], run[\\"engine\\"], run[\\"model\\"]\\n    q = part[\\"q_idx\\"]\\n    if len(q) > max_q:\\n        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))\\n    gold = CAP_VID[q]\\n    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n    sc = score_queries_v3(model, eng, q, part[\\"gallery\\"], CFG.rerank_k, protocol)\\n    pf = eng.path_feats(sc[\\"cand\\"], q)\\n    w = model.pattern_weights().detach().cpu().numpy()\\n    gain = F.softplus(model.path_gain).item()\\n    contrib = gain * pf * w\\n    tot = contrib.sum(-1, keepdims=True)\\n    share = np.where(tot > 0, contrib / np.maximum(tot, 1e-12), 0).mean((0, 1))\\n    reldeg_out = sum(np.asarray((g.A[t] > 0).sum(1)).ravel() for t in PATH_TYPES)\\n    reldeg_in = sum(np.asarray((g.A[t] > 0).sum(0)).ravel() for t in PATH_TYPES)\\n    con_in = np.asarray((g.A_con > 0).sum(0)).ravel()\\n    vdeg = np.array([reldeg_out[g.vid_tags[v]].sum() for v in range(g.n_vid)])\\n    cand_deg = vdeg[sc[\\"cand\\"]]\\n    from scipy.stats import spearmanr\\n    is_gold = (sc[\\"cand\\"] == gold[:, None])\\n    base_fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], run[\\"lam\\"])\\n    base_mrr = metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], base_fin, gold, clip_rank))[\\"MRR\\"]\\n\\n    def knock(idx):\\n        pf2 = pf.copy(); pf2[..., idx] = 0\\n        sp2 = znorm_per_row(gain * (pf2 * w).sum(-1))\\n        return metrics_from_ranks(ranks_with_fallback(sc[\\"cand\\"], combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sp2, run[\\"alpha\\"], run[\\"lam\\"]),\\n                                                      gold, clip_rank))[\\"MRR\\"]\\n\\n    rows = []\\n    for p_i, p in enumerate(PATTERNS):\\n        mrr = knock([p_i])\\n        reach = np.asarray((g.R[p] > 0).sum(0)).ravel()\\n        rows.append(dict(section=\\"pattern\\", split=part[\\"name\\"], item=p, weight=float(w[p_i]), nnz=int(g.R[p].nnz),\\n                         frac_pairs_nonzero=float((pf[..., p_i] > 0).mean()), contribution_share=float(share[p_i]),\\n                         knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr, max_video_reach=int(reach.max()) if reach.size else 0))\\n    groups = {f\\"family:{f}\\": [PIDX[p] for p in PATTERNS if p.startswith(f\\"P1_{f}\\") or p.startswith(f\\"P4_{f}\\")\\n                              or p == f\\"P1m_{FAMILY_REL[f]}\\"] for f in FAMILIES}\\n    groups.update({\\"family:related\\": [PIDX[\\"P3_related\\"], PIDX[\\"P5_related_isa\\"]],\\n                   \\"family:isa\\": [PIDX[\\"P2_isa\\"]] + [PIDX[f\\"P4_{f}_isa\\"] for f in FAMILIES] + [PIDX[\\"P5_related_isa\\"]],\\n                   \\"all_indirect\\": list(range(1, len(PATTERNS)))})\\n    for nm, idx in groups.items():\\n        mrr = knock(idx)\\n        rows.append(dict(section=\\"family_knockout\\", split=part[\\"name\\"], item=nm, knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr,\\n                         contribution_share=float(share[idx].sum())))\\n    reach_all = sum(np.asarray((g.R[p] > 0).sum(0)).ravel()[:g.n_nodes] for p in PATTERNS[1:] if not p.startswith(\\"P1m_\\"))\\n    through = con_in * reldeg_out                         # số đường đi 2 bước video → node → lớp/khái niệm đi QUA node\\n    for i in np.argsort(-through)[:50]:\\n        if through[i] <= 0:\\n            break\\n        rows.append(dict(section=\\"node\\", split=part[\\"name\\"], item=g.inv.name[i], node_type=g.inv.semantic_type(i),\\n                         videos_tagged=int(con_in[i]), rel_out_degree=int(reldeg_out[i]), rel_in_degree=int(reldeg_in[i]),\\n                         paths_through=int(through[i]), videos_reached_indirect=int(reach_all[i]), idf=float(g.idf[i])))\\n    flat_deg, flat_sp, flat_fin = cand_deg.ravel(), sc[\\"path_raw\\"].ravel(), base_fin.ravel()\\n    gold_rank = ranks_with_fallback(sc[\\"cand\\"], base_fin, gold, clip_rank)\\n    gold_deg = vdeg[gold]\\n    deg_auc = float(np.mean([(cand_deg[k][is_gold[k]] > cand_deg[k][~is_gold[k]]).mean()\\n                              for k in range(len(q)) if is_gold[k].any()])) if is_gold.any() else float(\\"nan\\")\\n    summary = dict(split=part[\\"name\\"], n_queries=len(q), base_MRR=base_mrr,\\n                   spearman_degree_vs_s_path=float(spearmanr(flat_deg, flat_sp).correlation),\\n                   spearman_degree_vs_final=float(spearmanr(flat_deg, flat_fin).correlation),\\n                   spearman_gold_degree_vs_gold_rank=float(spearmanr(gold_deg, gold_rank).correlation),\\n                   p_gold_degree_gt_negative=deg_auc, w_path_effective=(1 - run[\\"alpha\\"]) * (1 - run[\\"lam\\"]),\\n                   hubs=[dict(node=g.inv.name[i], paths_through=int(through[i])) for i in np.argsort(-through)[:10]])\\n    rows.append(dict(section=\\"summary\\", split=part[\\"name\\"], item=\\"degree_bias\\",\\n                     **{k: v for k, v in summary.items() if k not in (\\"hubs\\", \\"split\\")}))\\n    return pd.DataFrame(rows), summary\\n\\n\\n# ---- họ so sánh CHÍNH (đăng ký trước, một phía: a > b), hiệu chỉnh Holm; chỉ trên tập ngoài mới là XÁC NHẬN ----\\nPRIMARY_COMPARISONS = [(\\"H1_graph_vs_clip\\", \\"G3\\", \\"G0\\"), (\\"H2_relations_vs_contains_only\\", \\"G3\\", \\"G1\\"),\\n                       (\\"H3_predicate_vs_family_only\\", \\"G3\\", \\"G2\\"), (\\"H4_true_vs_shuffled_predicates\\", \\"G3\\", \\"G4\\"),\\n                       (\\"H5_true_vs_rewired_topology\\", \\"G3\\", \\"G5\\"), (\\"H6_query_conditioned_vs_static\\", \\"G3\\", \\"G6\\"),\\n                       (\\"H7_v4_vs_v3\\", \\"G3\\", \\"V3REF\\"),\\n                       (\\"H8_role_edges\\", \\"G3\\", \\"G7\\"), (\\"H9_svo_prep_edges\\", \\"G3\\", \\"G8\\"),\\n                       (\\"H10_kg_vs_contains_only\\", \\"G9\\", \\"G1\\"), (\\"H11_kg_true_vs_rewired\\", \\"G9\\", \\"G10\\"),\\n                       (\\"H12_kg_adds_to_graph\\", \\"G9\\", \\"G3\\"), (\\"H13_learned_vs_zeroshot_tags\\", \\"G3\\", \\"G11\\")]\\nH_V4 = (\\"quan hệ được dùng CÓ ĐIỀU KIỆN THEO TRUY VẤN (loại cạnh, vị từ khớp truy vấn, vai của node trong truy vấn) cải thiện \\"\\n        \\"retrieval so với cùng mô hình không có quan hệ / quan hệ sai\\")"}, {"cell_type": "markdown", "id": "c043", "metadata": {}, "source": "## 10. Self-test v3 (mục 12)\\n| mục 12 | kiểm tra | id |\\n|---|---|---|\\n| 1 | embedding vị từ tác động forward; pred_mode none (≡ v2) **không** bị ảnh hưởng | P1 |\\n| 2 | hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác | P2 |\\n| 3 | rewiring giữ kiểu node, họ, bậc, trọng số, đa tập vị từ | G6 |\\n| 4 | hoán vị vị từ không đổi topology / trọng số | G7 |\\n| 5 | mỗi ablation tắt đúng thành phần; inventory cố định | AB, G1 |\\n| 6 | trọng số chỉ nhân một lần (khớp tính tay, bất biến tỉ lệ) | G4 |\\n| 7 | node không có hàng xóm không sinh NaN; cô lập inductive | G4, G5 |\\n| 8 | không rò rỉ caption train/val/test/tập ngoài | C1, C1b, C2, C3, C8, C9 |\\n| 9 | giải thích đúng vị từ, đúng chiều; điểm khớp ma trận | G3, X1 |\\n| 10 | số trong báo cáo khớp CSV đầu ra | R1 |\\n\\nMột kiểm thử phần mềm PASS **không** có nghĩa giả thuyết khoa học đã được xác nhận."}, {"cell_type": "code", "execution_count": null, "id": "c044", "metadata": {}, "outputs": [], "source": "CANARY_NOUNS = [\\"flamingo\\", \\"accordion\\", \\"kayak\\", \\"harmonica\\", \\"trombone\\", \\"anvil\\", \\"walrus\\", \\"xylophone\\",\\n                \\"gondola\\", \\"hedgehog\\", \\"unicycle\\", \\"periscope\\", \\"tuba\\", \\"iguana\\", \\"llama\\"]\\nCANARY_VERBS = [\\"juggle\\", \\"knit\\", \\"yodel\\", \\"whittle\\", \\"embroider\\", \\"lasso\\"]\\n\\n\\ndef _same_graph(g1, g2):\\n    if g1.n_nodes != g2.n_nodes or g1.inv.sha1 != g2.inv.sha1:\\n        return False\\n    if (g1.A_con != g2.A_con).nnz:\\n        return False\\n    return (all((g1.A[t] != g2.A[t]).nnz == 0 for t in RELATION_TYPES) and np.array_equal(g1.idf, g2.idf)\\n            and g1.pdist == g2.pdist)\\n\\n\\ndef _pred_vocab_of(rel):\\n    s = Counter()\\n    for v in SPLIT[\\"train\\"]:\\n        s.update({(r, p) for _, r, _, p in rel[v]})\\n    return sorted(k for k in s if k[0] in PRED_FAMILIES)\\n\\n\\ndef test_canary(splits=(\\"test\\", \\"ext\\"), n_videos=40):\\n    \\"\\"\\"Canary chỉ có trong caption TEST / tập ngoài không được lọt vào từ vựng, quan hệ, từ vựng vị từ; đối chứng dương:\\n    đặt canary vào TRAIN thì phải phát hiện được.\\"\\"\\"\\n    terms = {k.rsplit(\\"#\\", 1)[0] for k in VOCAB2[\\"terms\\"]}\\n    a, b = [w for w in CANARY_NOUNS if w not in terms and is_visual_noun(w)][:2]\\n    seen_pred = {t for _, p in PRED_VOCAB for t in p.split()}           # động từ canary cũng không được là vị từ train\\n    v = next((w for w in CANARY_VERBS if w not in terms and w not in seen_pred and is_action_verb(w)),\\n             next(w for w in CANARY_VERBS if w not in terms and is_action_verb(w)))\\n    canary = [f\\"a {a} is {naive_gerund(v)} a {b}\\"] * 3\\n    tv = [x for s in splits for x in SPLIT[s][:n_videos]]\\n    store = CAPS.replaced({x: CAPS._caps[x] + canary for x in tv})\\n    voc = build_vocab_v2(store, SPLIT)\\n    rel = build_relations_v2(store, SPLIT, voc)\\n    keys = {ckey(a, \\"n\\"), ckey(b, \\"n\\"), ckey(v, \\"v\\")}\\n    leak_v = sorted(keys & set(voc[\\"terms\\"]))\\n    leak_r = [t for trip in rel.values() for t in trip if {t[0], t[2]} & keys]\\n    leak_p = sorted(set(_pred_vocab_of(rel)) - set(PRED_VOCAB))          # vị từ MỚI so với từ vựng vị từ train\\n    fake = dict(SPLIT, train=list(SPLIT[\\"train\\"]) + list(tv))\\n    store_pos = store.replaced({}, new_split={x: \\"train\\" for x in tv})\\n    voc_pos = build_vocab_v2(store_pos, fake)\\n    rel_pos = build_relations_v2(store_pos, fake, voc_pos)\\n    detected = {ckey(a, \\"n\\"), ckey(b, \\"n\\")} <= set(voc_pos[\\"terms\\"]) and any({t[0], t[2]} & keys for trip in rel_pos.values() for t in trip)\\n    return dict(canary=f\\"{a} –{v}→ {b}\\", splits=list(splits), n_videos=len(tv), leak_vocab=leak_v, leak_relations=leak_r[:3],\\n                leak_predicates=leak_p[:3], positive_control_detected=bool(detected))\\n\\n\\ndef test_invariance():\\n    \\"\\"\\"Xáo / thay caption TEST và tập ngoài rồi dựng lại TOÀN BỘ: từ vựng, quan hệ, vị từ, tagger, taxonomy, đồ thị.\\"\\"\\"\\n    tv = list(SPLIT[\\"test\\"]) + list(SPLIT[\\"ext\\"])\\n    perm = np.random.RandomState(0).permutation(tv)\\n    junk = [\\"a purple elephant is skiing on the moon\\", \\"zebra piano astronaut volcano juggling\\"] * 4\\n    store = CAPS.replaced({x: (CAPS._caps[int(p)] if k % 2 == 0 else junk) for k, (x, p) in enumerate(zip(tv, perm))})\\n    voc = build_vocab_v2(store, SPLIT)\\n    rel = build_relations_v2(store, SPLIT, voc)\\n    tags, scores, sel, _ = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL[\\"n_frames\\"], voc, SPLIT, store)\\n    tax, cmeta, _, _ = build_taxonomy_v2(voc, store, SPLIT, {v: tags[v] for v in SPLIT[\\"train\\"]})\\n    recs = [dict(r, tags=tags[r[\\"vid\\"]]) for r in RECORDS]\\n    inv = NodeInventory(recs, voc, cmeta)\\n    g2 = GraphV3(inv, recs, SPLIT[\\"train\\"], rel, tax[\\"heuristic\\"], tag_scores=scores)\\n    g1 = graph_for(DESIGN_BASE, CFG.seeds[0])[0]\\n    return dict(vocab=voc[\\"terms\\"] == VOCAB2[\\"terms\\"], relations=rel == RELATIONS2, predicate_vocab=_pred_vocab_of(rel) == PRED_VOCAB,\\n                tagger_selection=sel == TAG_SELECT2, tags=tags == TAGS2, taxonomy=tax == TAXONOMY, inventory=inv.sha1 == INV.sha1,\\n                graph=_same_graph(g1, g2), annotation_sample=build_annotation_sample(store, SPLIT, voc, RECORDS)[0]\\n                .sample_id.tolist() == ANN_SAMPLE.sample_id.tolist(),\\n                automatic_verification=(build_agreement(store, SPLIT, voc)[0] == AGREE) if AGREE else True)\\n\\n\\nVARIANT_DESIGNS = ([dict(DESIGN_BASE, mask=m) for m in MASKS] + [dict(DESIGN_BASE, rewire=True), dict(DESIGN_BASE, pred_shuffle=True),\\n                   dict(DESIGN_BASE, rel_policy=\\"support\\"), dict(DESIGN_BASE, taxonomy=\\"none\\"), dict(DESIGN_BASE, contains=\\"conf\\"),\\n                   dict(DESIGN_BASE, related=\\"npmi20\\"), dict(DESIGN_BASE, path_mode=\\"pred_match\\")])\\n\\n\\ndef test_inventory_constant():\\n    \\"\\"\\"Mọi biến thể đồ thị dùng CÙNG inventory; với CÙNG cấu hình mô hình, số tham số không đổi giữa các biến thể đồ thị.\\"\\"\\"\\n    gs = [graph_for(d, CFG.seeds[0]) for d in VARIANT_DESIGNS]\\n    params = {pm: sorted({sum(p.numel() for p in make_model(e, SYM_FEAT, pred_mode=pm).parameters()) for _, e in gs[:4]})\\n              for pm in PRED_MODES}\\n    return dict(n_graphs=len(gs), node_counts=sorted({g.n_nodes for g, _ in gs}), same_inventory=all(g.inv is INV for g, _ in gs),\\n                param_counts_by_pred_mode=params)\\n\\n\\ndef test_schema_conformance(g):\\n    \\"\\"\\"Mọi cạnh: kiểu nút đầu/cuối + POS đúng REL_SPEC; không self-loop; trọng số trong (0, 1]; vị từ đúng họ:\\n    svo/prep ⇒ mọi vị từ thuộc từ vựng vị từ của họ đó (không lẫn tên loại quan hệ); agent/object ⇒ vị từ = lemma động từ.\\"\\"\\"\\n    bad = Counter()\\n    inv = g.inv\\n    fam_names = set(REL_SPEC) | set(FAMILIES)\\n    for t in [\\"contains\\"] + RELATION_TYPES:\\n        M = (g.A_con if t == \\"contains\\" else g.A[t]).tocoo()\\n        spec = REL_SPEC[t]\\n        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n        for u, v, w in zip(M.row.tolist(), M.col.tolist(), M.data.tolist()):\\n            src_ok = spec[\\"src\\"].split(\\":\\")[0].split(\\"|\\")\\n            tgt_ok = spec[\\"tgt\\"].split(\\" \\")[0].split(\\":\\")[0].split(\\"|\\")\\n            if inv.node_type[u] not in src_ok or inv.node_type[v] not in tgt_ok:\\n                bad[f\\"{t}:type\\"] += 1\\n            if \\":\\" in spec[\\"src\\"] and inv.semantic_type(u) != spec[\\"src\\"].split(\\":\\")[1]:\\n                bad[f\\"{t}:src_pos\\"] += 1\\n            if \\":\\" in spec[\\"tgt\\"] and inv.semantic_type(v) != spec[\\"tgt\\"].split(\\":\\")[1]:\\n                bad[f\\"{t}:tgt_pos\\"] += 1\\n            if t in (\\"isa\\", \\"isa_inv\\") and inv.pos[u] != inv.pos[v]:\\n                bad[f\\"{t}:pos_mismatch\\"] += 1\\n            if u == v:\\n                bad[f\\"{t}:self_loop\\"] += 1\\n            if not (0 < w <= 1 + 1e-6):\\n                bad[f\\"{t}:weight\\"] += 1\\n            if base in REL_CATS:\\n                d = g.predicates_of(t, u, v)\\n                if not d:\\n                    bad[f\\"{t}:no_predicate\\"] += 1\\n                if any(p in fam_names for p in d):\\n                    bad[f\\"{t}:predicate_is_relation_name\\"] += 1\\n                if base in PRED_FAMILIES and any((base, p) not in PRED_ID for p in d):\\n                    bad[f\\"{t}:predicate_not_in_vocab\\"] += 1\\n                if base in (\\"agent_of\\", \\"object_of\\"):\\n                    verb = inv.lemma[v if not t.endswith(\\"_inv\\") else u]\\n                    if set(d) != {verb}:\\n                        bad[f\\"{t}:predicate_not_verb_node\\"] += 1\\n    return dict(violations=dict(bad), n_checked=int(g.A_con.nnz + sum(g.A[t].nnz for t in RELATION_TYPES)))\\n\\n\\ndef test_path_identity(n=200):\\n    \\"\\"\\"Điểm ma trận (engine) = tổng/max liệt kê đường đi ở MỌI path_mode (kể cả pred_match); tính lại R khớp graph.R;\\n    mọi đường đi khớp vị từ ghi đúng vị từ của đúng cạnh theo đúng chiều; đối chứng âm: sửa pred_id ⇒ bị phát hiện.\\"\\"\\"\\n    out = {}\\n    rng = np.random.RandomState(0)\\n    pool = np.where(np.isin(CAP_SPLIT, [\\"train\\", \\"val\\"]))[0]\\n    qs = rng.choice(pool, min(n, len(pool)), replace=False)\\n    pairs = [(q, int(CAP_VID[q])) for q in qs] + [(q, int(v)) for q in qs for v in rng.choice(SPLIT[\\"train\\"], 2)]\\n    for mode in (\\"weighted\\", \\"count\\", \\"degnorm\\", \\"dedup\\", \\"pred_match\\"):\\n        g, eng = graph_for(dict(DESIGN_BASE, path_mode=mode), CFG.seeds[0])\\n        bad = cnt = npred = pbad = neg = 0\\n        for q, v in pairs:                               # video đúng + video ngẫu nhiên (có nhiều đường đi gián tiếp hơn)\\n            cids, qpred, qtok = q_args(eng, q)\\n            if not cids:\\n                continue\\n            cnt += 1\\n            paths = enumerate_paths_v3(g, v, cids, qpred)\\n            bad += int(not np.allclose(eng.path_feats(np.array([[v]]), np.array([q]))[0, 0],\\n                                       paths_to_counts_v3(paths, mode), atol=1e-5))\\n            for p in paths:\\n                if p.get(\\"pred\\") is not None:\\n                    npred += 1\\n                    pbad += int(not pred_valid_v3(g, p, v, qtok))\\n                    wrong = dict(p, pred=(p[\\"pred\\"][0], (p[\\"pred\\"][1] + 1) % len(PRED_VOCAB), p[\\"pred\\"][2]))\\n                    neg += int(not pred_valid_v3(g, wrong, v, qtok))\\n        R2 = g.compute_R(rows=np.arange(g.n_vid), PM=g.path_mats())\\n        diff = max(float(abs(R2[p] - g.R[p]).max()) if (R2[p] - g.R[p]).nnz else 0.0 for p in PATTERNS)\\n        out[mode] = dict(pairs=cnt, mismatches=bad, recompute_maxdiff=diff, predicate_paths=npred, predicate_invalid=pbad,\\n                         negative_control_detected=neg)\\n    return out\\n\\n\\ndef test_mp_weights():\\n    \\"\\"\\"Unit test TypedSAGEv3: trọng số cạnh dùng trong forward (weighted), bị bỏ qua (binary), w=0 ≡ không cạnh, node cô lập\\n    hữu hạn, bất biến thứ tự cạnh, gradient hữu hạn; trọng số chỉ nhân MỘT lần: (i) khớp công thức tính tay, (ii) weighted_mean\\n    bất biến khi nhân mọi trọng số với hằng số (nhân lặp sẽ làm đổi), (iii) cùng hệ số cho h_u và vị từ e_uv.\\"\\"\\"\\n    torch.manual_seed(0)\\n    res = {}\\n    x = torch.randn(4, 8)\\n    ei = torch.tensor([[1, 2, 3], [0, 0, 0]])\\n    et = torch.zeros(3, dtype=torch.long)\\n    pidx = torch.tensor([0, 1, 2])\\n    ph = torch.randn(3, 8)\\n    for mode in (\\"binary\\", \\"weighted_mean\\", \\"weighted_symnorm\\"):\\n        L = TypedSAGEv3(8, 8, 1, [0], 0.0, mode, pred_type_ids=[0])\\n        w1 = torch.tensor([1.0, 1.0, 1.0]); w2 = torch.tensor([1.0, 0.0, 1.0]); w3 = torch.tensor([3.0, 1.0, 1.0])\\n        o1, o2, o3 = L(x, ei, et, w1, pidx, ph), L(x, ei, et, w2, pidx, ph), L(x, ei, et, w3, pidx, ph)\\n        perm = torch.tensor([2, 0, 1])\\n        o_perm = L(x, ei[:, perm], et[perm], w3[perm], pidx[perm], ph)\\n        o_drop = L(x, ei[:, [0, 2]], et[[0, 2]], torch.tensor([1.0, 1.0]), pidx[[0, 2]], ph)\\n        o_scaled = L(x, ei, et, 5 * w3, pidx, ph)\\n        ww = torch.ones(3) if mode == \\"binary\\" else w3\\n        coef = ww / ww.sum() if mode != \\"weighted_symnorm\\" else ww / torch.sqrt(ww * ww.sum())\\n        manual = L.self_w(x[0]) + torch.sigmoid(L.gate[0]) * (L.neigh_w[0]((coef[:, None] * x[1:4]).sum(0))\\n                                                               + L.pred_w[\\"0\\"]((coef[:, None] * ph).sum(0)))\\n        xg = x.clone().requires_grad_(True)\\n        L(xg, ei, et, w3, pidx, ph).sum().backward()\\n        changes = bool((o1[0] - o3[0]).abs().max() > 1e-6)\\n        zero_ok = bool((o2[0] - o_drop[0]).abs().max() < 1e-5)\\n        res[mode] = dict(weight_changes_output=changes, zero_weight_equals_edge_removed=zero_ok,\\n                         isolated_nodes_finite=bool(torch.isfinite(o1[1:]).all()),\\n                         edge_order_invariant=bool((o3 - o_perm).abs().max() < 1e-5),\\n                         grad_finite=bool(torch.isfinite(xg.grad).all()),\\n                         matches_formula=bool((o3[0] - manual).abs().max() < 1e-5),\\n                         scale_invariant=bool((o3[0] - o_scaled[0]).abs().max() < 1e-5),\\n                         as_designed=((not changes) if mode == \\"binary\\" else (changes and zero_ok))\\n                         and bool((o3[0] - manual).abs().max() < 1e-5)\\n                         and (mode != \\"weighted_mean\\" or bool((o3[0] - o_scaled[0]).abs().max() < 1e-5)))\\n    return res\\n\\n\\ndef test_predicate_forward():\\n    \\"\\"\\"(1) Embedding vị từ thật sự tác động forward: đổi embedding của các vị từ đang dùng ⇒ biểu diễn node đổi khi\\n    pred_mode ∈ {clip, clip_adapt}; pred_mode = none (relation-family-only, ≡ v2) PHẢI không đổi (đối chứng âm).\\n    (2) Hai vị từ khác nhau trên CÙNG cặp node tạo thông điệp khác nhau (trường hợp kiểm soát, embedding CLIP thật).\\"\\"\\"\\n    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])\\n    used = sorted({pid for rel in PRED_FAMILIES for d in g.pshare[rel].values() for pid in d})\\n    out = {}\\n    for pm in PRED_MODES:\\n        torch.manual_seed(0)\\n        model = make_model(eng, SYM_FEAT, pred_mode=pm).eval()\\n        if pm == \\"clip_adapt\\":                         # MLP dư khởi tạo 0 ⇒ cho trọng số khác 0 để kiểm tra cả nhánh dư\\n            with torch.no_grad():\\n                for prm in model.pred_enc.res.parameters():\\n                    prm.normal_(0, 0.05)\\n        with torch.no_grad():\\n            h1 = model.encode_graph(*eng.edges_on(\\"train\\"))\\n            if pm != \\"none\\" and used:\\n                old = model.pred_emb.clone()\\n                model.pred_emb[used] = torch.roll(model.pred_emb[used], 1, 0)\\n                h2 = model.encode_graph(*eng.edges_on(\\"train\\"))\\n                model.pred_emb.copy_(old)\\n            else:\\n                ed = list(eng.edges_on(\\"train\\"))\\n                ed[3] = torch.full_like(ed[3], -1)                       # gỡ mọi chỉ số vị từ\\n                h2 = model.encode_graph(*ed)\\n        out[pm] = dict(max_change=float((h1 - h2).abs().max()), finite=bool(torch.isfinite(h1).all()))\\n    out[\\"predicate_used_as_designed\\"] = (out[\\"clip\\"][\\"max_change\\"] > 1e-5 and out[\\"clip_adapt\\"][\\"max_change\\"] > 1e-5\\n                                         and out[\\"none\\"][\\"max_change\\"] < 1e-7)\\n    pick = {}\\n    for want in (\\"ride\\", \\"feed\\", \\"play\\", \\"hold\\"):\\n        if (\\"svo\\", want) in PRED_ID:\\n            pick[want] = PRED_ID[(\\"svo\\", want)]\\n    pair = list(pick.items())[:2] if len(pick) >= 2 else [(PRED_VOCAB[i][1], i) for i in used[:2]]\\n    torch.manual_seed(0)\\n    L = TypedSAGEv3(8, 8, 1, [0], 0.0, \\"binary\\", pred_type_ids=[0])\\n    enc = PredicateEncoder(PRED_EMB.shape[1], 8, \\"clip\\", 16)\\n    pe = enc(torch.eye(len(PRED_VOCAB))[[pair[0][1], pair[1][1]]], torch.from_numpy(PRED_EMB))\\n    x = torch.randn(2, 8)\\n    ei, et, w = torch.tensor([[0], [1]]), torch.zeros(1, dtype=torch.long), torch.ones(1)\\n    with torch.no_grad():\\n        a = L(x, ei, et, w, torch.tensor([0]), pe)[1]\\n        b = L(x, ei, et, w, torch.tensor([1]), pe)[1]\\n    out[\\"two_predicates\\"] = dict(predicates=[pair[0][0], pair[1][0]], message_diff=float((a - b).abs().max()),\\n                                 differ=bool((a - b).abs().max() > 1e-6))\\n    return out\\n\\n\\ndef test_ablations(g0):\\n    \\"\\"\\"Mỗi ablation tắt ĐÚNG thành phần đã mô tả: mask ⇒ loại cạnh bị tắt = 0, loại khác giữ nguyên; support ⇒ mọi vị từ còn\\n    lại đạt ngưỡng; pred_mode none ⇒ mô hình không có tham số vị từ; contains_only ⇒ chỉ còn contains.\\"\\"\\"\\n    out = {}\\n    for m, types in MASKS.items():\\n        g = graph_for(dict(DESIGN_BASE, mask=m), CFG.seeds[0])[0]\\n        off_ok = all(g.A[t].nnz == 0 for t in types)\\n        rest_ok = all((g.A[t] != g0.A[t]).nnz == 0 for t in RELATION_TYPES if t not in types) and (g.A_con != g0.A_con).nnz == 0\\n        out[f\\"mask:{m}\\"] = bool(off_ok and rest_ok)\\n    gs = graph_for(dict(DESIGN_BASE, rel_policy=\\"support\\"), CFG.seeds[0])[0]\\n    out[\\"policy:support\\"] = all(PRED_SUPPORT[(r_, p)] >= CFG.pred_min_support for (r_, a, b), d in gs.pdist.items()\\n                                if gs.A[r_][a, b] != 0 for p in d)\\n    if AGREE:\\n        ga = graph_for(dict(DESIGN_BASE, rel_policy=\\"agreement\\"), CFG.seeds[0])[0]\\n        out[\\"policy:agreement\\"] = all(agreement_rate(r_, INV.id2key[a], INV.id2key[b], p) >= CFG.agree_min\\n                                      for (r_, a, b), d in ga.pdist.items() if ga.A[r_][a, b] != 0 for p in d)\\n    m0 = make_model(E_BASE, SYM_FEAT, pred_mode=\\"none\\")\\n    out[\\"pred_mode:none_has_no_predicate_params\\"] = not any(\\"pred\\" in n_ for n_, _ in m0.named_parameters())\\n    m1 = make_model(E_BASE, SYM_FEAT, pred_mode=\\"clip\\")\\n    out[\\"pred_mode:clip_has_predicate_params\\"] = any(\\"pred\\" in n_ for n_, _ in m1.named_parameters())\\n    out[\\"pred_mode:clip_no_adapter\\"] = not any(\\"pred_enc.res\\" in n_ for n_, _ in m1.named_parameters())\\n    gp = graph_for(dict(DESIGN_BASE, path_mode=\\"pred_match\\"), CFG.seeds[0])[0]\\n    out[\\"path:weighted_has_no_P1m\\"] = all(g0.R[f\\"P1m_{r}\\"].nnz == 0 for r in PRED_FAMILIES)\\n    out[\\"path:pred_match_has_P1m\\"] = any(gp.R[f\\"P1m_{r}\\"].nnz > 0 for r in PRED_FAMILIES)\\n    out[\\"path:pred_match_same_other_patterns\\"] = all((gp.R[p] != g0.R[p]).nnz == 0 for p in PATTERNS if not p.startswith(\\"P1m_\\"))\\n    return out\\n\\n\\ndef test_inductive_isolation():\\n    \\"\\"\\"Strict inductive: đổi embedding của MỘT video đánh giá không làm đổi node lớp/khái niệm hay video khác;\\n    transductive thì có (để chứng minh phép thử nhạy). Chạy cho cả video test MSVD và video tập ngoài.\\"\\"\\"\\n    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])\\n    model = make_model(eng, SYM_FEAT, pred_mode=\\"clip\\").eval()\\n    out = {}\\n    for s in (\\"test\\", \\"ext\\"):\\n        tv = np.array(SPLIT[s])\\n        if len(tv) < 2:\\n            continue\\n        v0, v1 = int(tv[0]), int(tv[1])\\n        for proto in PROTOCOLS:\\n            with torch.no_grad():\\n                h1 = model.encode_graph(*eng.edges_on(proto))\\n                old = model.vid_feat[v0].clone()\\n                model.vid_feat[v0] = torch.randn_like(old)\\n                h2 = model.encode_graph(*eng.edges_on(proto))\\n                model.vid_feat[v0] = old\\n            sym = slice(g.n_vid, g.n_nodes)\\n            out[f\\"{s}:{proto}\\"] = dict(symbolic_changed=float((h1[sym] - h2[sym]).abs().max()),\\n                                       other_video_changed=float((h1[v1] - h2[v1]).abs().max()),\\n                                       own_video_changed=float((h1[v0] - h2[v0]).abs().max()),\\n                                       all_finite=bool(torch.isfinite(h1).all()))\\n    out[\\"edges\\"] = eng.edge_stats\\n    return out\\n\\n\\ndef test_guards():\\n    out = {}\\n    for s in (\\"test\\", \\"ext\\"):\\n        if not SPLIT[s]:\\n            continue\\n        try:\\n            make_part(s); out[f\\"{s}_before_freeze_blocked\\"] = False\\n        except RuntimeError:\\n            out[f\\"{s}_before_freeze_blocked\\"] = True\\n        sq = np.where(CAP_SPLIT == s)[0][:3]\\n        for name, fn in ((f\\"{s}→gnn_train\\", lambda: CAPS.use(sq, \\"gnn_train\\")), (f\\"{s}→tune\\", lambda: CAPS.use(sq, \\"tune\\")),\\n                         (f\\"{s}→vocab\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"vocab\\")),\\n                         (f\\"{s}→relations\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"relations\\")),\\n                         (f\\"{s}→annotation\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"annotation\\")),\\n                         (f\\"{s}→agreement\\", lambda: CAPS.by_video(SPLIT[s][:2], \\"agreement\\"))):\\n            try:\\n                fn(); out[name + \\"_blocked\\"] = False\\n            except PermissionError:\\n                out[name + \\"_blocked\\"] = True\\n        try:\\n            CAPS.use(sq, \\"eval\\"); out[f\\"{s}_eval_before_freeze_blocked\\"] = False\\n        except RuntimeError:\\n            out[f\\"{s}_eval_before_freeze_blocked\\"] = True\\n    vq = np.where(CAP_SPLIT == \\"val\\")[0][:3]\\n    for name, fn in ((\\"val→gnn_train\\", lambda: CAPS.use(vq, \\"gnn_train\\")),\\n                     (\\"val→relations\\", lambda: CAPS.by_video(SPLIT[\\"val\\"][:2], \\"relations\\")),\\n                     (\\"train→tune\\", lambda: CAPS.use(np.where(CAP_SPLIT == \\"train\\")[0][:3], \\"tune\\"))):\\n        try:\\n            fn(); out[name + \\"_blocked\\"] = False\\n        except PermissionError:\\n            out[name + \\"_blocked\\"] = True\\n    return out\\n\\n# ---------------------------------- Self-test v4 (QC-RGAT) — nhanh (< 1 phút CPU), không huấn luyện ----------------------------------\\ndef _v4_model_for_test(design, seed=0):\\n    \\"\\"\\"Mô hình KHỞI TẠO NGẪU NHIÊN nhưng mọi cổng đều khác 0 (để phép thử không qua vì tham số bằng 0).\\"\\"\\"\\n    g, eng = graph_for(design, CFG.seeds[0])\\n    set_seed(seed)\\n    m = make_model_v4(eng, design).eval()\\n    with torch.no_grad():\\n        for p in (m.beta, m.kappa, m.g_ex, m.g_soft, m.rgate):\\n            p.copy_(torch.randn_like(p))\\n    return g, eng, m\\n\\n\\ndef _queries_with_pred_edges(D, n, seed=0):\\n    \\"\\"\\"Truy vấn TRAIN có ≥ 1 cạnh mang vị từ khớp vị từ của truy vấn (để thử nhánh vị từ).\\"\\"\\"\\n    rng = np.random.RandomState(seed)\\n    out = []\\n    for q in rng.permutation(TRAIN_Q)[:4000]:\\n        ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]\\n        if ids and D.E and np.asarray(D.Mex[ids].sum(0)).ravel().max() > 0 and V4_HITS[q]:\\n            out.append(int(q))\\n            if len(out) >= n:\\n                break\\n    return out\\n\\n\\ndef test_v4_decomposition(n=30):\\n    \\"\\"\\"Σ đóng góp (thị giác + nhãn + đường đi) = điểm của mô hình, cho T = 1 và 2; và xoá một cạnh ⇒ điểm giảm ĐÚNG bằng tổng đóng\\n    góp các đường đi qua cạnh đó (T = 1: tuyệt đối; T = 2: tính lại trên đồ thị đã xoá).\\"\\"\\"\\n    out = {}\\n    for T in (1, 2):\\n        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=T))\\n        qs = _queries_with_pred_edges(m.D, n)\\n        maxdiff, del_diff, n_rel, n_del = 0.0, 0.0, 0, 0\\n        vids = [int(v) for v in SPLIT[\\"train\\"] if m.D.Acon[int(v)].gt(0).any()]\\n        for q in qs:\\n            v = vids[q % len(vids)]\\n            terms = m.decompose(q, v)\\n            with torch.no_grad():\\n                s = float(m.score(m.D.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE))[0, 0])\\n            maxdiff = max(maxdiff, abs(s - sum(t[\\"contrib\\"] for t in terms)))\\n            rel = [t for t in terms if t[\\"kind\\"] == \\"relation\\"]\\n            n_rel += len(rel)\\n            if rel:\\n                e = max(rel, key=lambda t: abs(t[\\"contrib\\"]))[\\"edges\\"][-1]\\n                D2 = RelDataV4(g, drop_edges=[int(m.D.gidx[e])])\\n                with torch.no_grad():\\n                    s2 = float(m.score(D2.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE), D2)[0, 0])\\n                expect = sum(t[\\"contrib\\"] for t in rel if e in t[\\"edges\\"])\\n                del_diff = max(del_diff, abs((s - s2) - expect)); n_del += 1\\n        out[f\\"T{T}\\"] = dict(queries=len(qs), relation_terms=n_rel, max_abs_diff=maxdiff, deletions=n_del, deletion_max_abs_diff=del_diff)\\n    return out\\n\\n\\ndef test_v4_predicate_and_query():\\n    \\"\\"\\"(1) Đổi vị từ của truy vấn ⇒ attention trên cạnh mang vị từ đó đổi; use_pred = False ⇒ KHÔNG đổi (đối chứng âm).\\n    (2) Hai truy vấn khác nhau ⇒ attention khác (qcond); qcond = False ⇒ giống hệt. (3) Hoán vị vị từ (G4) đổi tín hiệu khớp.\\"\\"\\"\\n    out = {}\\n    for use_pred in (True, False):\\n        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=1, v4_pred=use_pred))\\n        qs = _queries_with_pred_edges(m.D, 10)\\n        diffs = []\\n        for q in qs:\\n            qb = m.D.query_batch([q], True)\\n            qb2 = dict(qb, ex=torch.zeros_like(qb[\\"ex\\"]), soft=torch.zeros_like(qb[\\"soft\\"]))\\n            with torch.no_grad():\\n                u = m.q(qb[\\"tq\\"])\\n                diffs.append(float((m.attention(qb, u) - m.attention(qb2, u)).abs().max()))\\n        out[f\\"pred_change_use_pred={use_pred}\\"] = max(diffs) if diffs else float(\\"nan\\")\\n    for qcond in (True, False):\\n        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=1, v4_qcond=qcond))\\n        qs = _queries_with_pred_edges(m.D, 2)\\n        with torch.no_grad():\\n            qb = m.D.query_batch(qs, True)\\n            a = m.attention(qb, m.q(qb[\\"tq\\"]))\\n        out[f\\"two_queries_attention_diff_qcond={qcond}\\"] = float((a[0] - a[1]).abs().max())\\n    g0 = graph_for(dict(V3_FINAL, model=\\"v4\\"), CFG.seeds[0])[0]\\n    gs = graph_for(dict(V3_FINAL, model=\\"v4\\", pred_shuffle=True), CFG.seeds[0])[0]\\n    D0, Ds = RelDataV4(g0), RelDataV4(gs)\\n    qs = _queries_with_pred_edges(D0, 50)\\n    out[\\"pred_shuffle_changes_match\\"] = float((D0.query_batch(qs)[\\"ex\\"] - Ds.query_batch(qs)[\\"ex\\"]).abs().sum())\\n    out[\\"pred_shuffle_same_topology\\"] = bool(np.array_equal(D0.src_np, Ds.src_np) and np.array_equal(D0.dst_np, Ds.dst_np))\\n    return out\\n\\n\\ndef test_v4_inductive_and_masks():\\n    \\"\\"\\"Video chỉ NHẬN (node lớp không phụ thuộc video) ⇒ đổi đặc trưng / nhãn của MỘT video test (hoặc tập ngoài) không đổi điểm\\n    của video khác; rel = False ⇒ không có số hạng quan hệ; điểm hữu hạn, gradient hữu hạn.\\n    So với dung sai 1e-6 (như G5 của v3): trên GPU, index_add_ cộng dồn bằng atomic nên hai lần gọi cùng đầu vào có thể lệch ~1e-8.\\"\\"\\"\\n    g, eng, m = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_layers=2))\\n    out = {}\\n    q = _queries_with_pred_edges(m.D, 1)[0]\\n    qb = m.D.query_batch([q])\\n    for s in (\\"test\\", \\"ext\\"):\\n        tv = [int(v) for v in SPLIT.get(s, [])]\\n        if len(tv) < 2:\\n            continue\\n        v0, v1 = tv[0], tv[1]\\n        cand = torch.tensor([[v0, v1]], device=DEVICE)\\n        with torch.no_grad():\\n            s1 = m.score(qb, cand)\\n            s1b = m.score(qb, cand)                          # cùng đầu vào ⇒ đo sai số số học khi chạy lại (GPU không tất định)\\n            old_x, old_a = VID_T[v0].clone(), m.D.Acon[v0].clone()\\n            VID_T[v0] = torch.randn_like(old_x); m.D.Acon[v0] = torch.roll(old_a, 7)\\n            s2 = m.score(qb, cand)\\n            VID_T[v0] = old_x; m.D.Acon[v0] = old_a\\n        out[s] = dict(other_video_changed=float((s1[0, 1] - s2[0, 1]).abs()), own_video_changed=float((s1[0, 0] - s2[0, 0]).abs()),\\n                      rerun_noise=float((s1 - s1b).abs().max()))\\n    g1, e1, m1 = _v4_model_for_test(dict(V3_FINAL, model=\\"v4\\", v4_rel=False))\\n    v = int(SPLIT[\\"train\\"][0])\\n    out[\\"no_relation_terms_when_rel_false\\"] = not any(t[\\"kind\\"] == \\"relation\\" for t in m1.decompose(q, v))\\n    m.train()\\n    qb = m.D.query_batch(TRAIN_Q[:16])\\n    loss = m.score(qb, torch.from_numpy(np.tile(CAP_VID[TRAIN_Q[:16]], (16, 1))).to(DEVICE)).logsumexp(1).mean()\\n    loss.backward()\\n    out[\\"grad_finite\\"] = bool(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\\n    out[\\"score_finite\\"] = bool(torch.isfinite(loss).item())\\n    m.eval()\\n    return out\\n"}, {"cell_type": "markdown", "metadata": {}, "source": "## 11. Level 4(b): bộ truy vấn đăng ký trước, xAI theo thứ hạng, phân tích ca, demo, kiểm kê\\n\\nCác hàm dưới đây chỉ là **định nghĩa**, được gọi ở phần THỰC THI. Mọi file đầu ra sinh từ chính lần chạy; không có số liệu điền tay."}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b): công cụ bổ sung (chỉ ĐỊNH NGHĨA — chạy ở phần THỰC THI) ==============================\\nimport copy as _copy\\n\\n# Nguồn / giấy phép: dataset card HF (kiểm qua API ngày 2026-10-25) KHÔNG khai báo license ⇒ ghi đúng như vậy, không tự điền.\\nDATA_SOURCES = {\\n    \\"msvd\\": dict(name=\\"MSVD (Microsoft Research Video Description Corpus)\\", paper=\\"Chen & Dolan, ACL 2011\\",\\n                 repo=CFG.hf_repo, revision=CFG.hf_revision,\\n                 license=\\"KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc (clip YouTube) phát hành cho nghiên cứu\\",\\n                 role=\\"train: từ vựng, quan hệ, huấn luyện GNN | val: chọn cấu hình | test: THĂM DÒ\\"),\\n    \\"msrvtt\\": dict(name=\\"MSR-VTT\\", paper=\\"Xu et al., CVPR 2016\\", repo=CFG.msrvtt_repo, revision=CFG.msrvtt_revision,\\n                   license=\\"KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc của Microsoft Research cho nghiên cứu\\",\\n                   role=\\"chỉ đánh giá (chuyển giao zero-shot): không caption nào dùng để học hay chọn\\"),\\n}\\n# Số lần mỗi benchmark đã được nhìn TRƯỚC notebook này (lịch sử v1–v4, xem README). Đã nhìn ⇒ kết quả chỉ là THĂM DÒ.\\nBENCHMARK_LOOKS_BEFORE = {\\"msvd_test\\": 5, \\"msrvtt_1kA\\": 1, \\"msrvtt_fresh1k\\": 1, \\"msrvtt_fresh1kB\\": 1, \\"msrvtt_fresh1kC\\": 0}\\n_PROV_RULES[\\"evaluation\\"] = {\\"val\\", \\"test\\", \\"ext\\"}   # bộ truy vấn đánh giá: dựng từ văn bản truy vấn của tập đánh giá, không học / chọn gì\\nFINDINGS = []          # kết quả KHOA HỌC được báo cáo (không phải kiểm thử phần mềm): id, câu hỏi, kết quả, diễn giải\\n\\n\\ndef finding(fid, question, result, interpretation):\\n    FINDINGS.append(dict(id=fid, question=question, result=str(result)[:900], interpretation=interpretation))\\n    print(f\\"  [FINDING] {fid} — {question}: {result}\\\\n            ⇒ {interpretation}\\")\\n\\n\\ndef benchmark_status():\\n    \\"\\"\\"Trạng thái từng tập đánh giá của lần chạy này: CONFIRMATORY chỉ khi tập CHƯA từng được nhìn.\\"\\"\\"\\n    ext_key = {\\"fresh1kC\\": \\"msrvtt_fresh1kC\\", \\"fresh1kB\\": \\"msrvtt_fresh1kB\\", \\"fresh1k\\": \\"msrvtt_fresh1k\\", \\"1kA\\": \\"msrvtt_1kA\\"}.get(EXT_REPORT.get(\\"set\\"))\\n    out = {\\"msvd_test\\": dict(looks_before=BENCHMARK_LOOKS_BEFORE[\\"msvd_test\\"], evaluated=not CFG.dev_mode,\\n                             status=\\"EXPLORATORY\\" if not CFG.dev_mode else \\"NOT RUN (dev_mode: test khoá)\\")}\\n    if EXT_RECORDS and not EXT_REPORT.get(\\"dev\\"):\\n        lb = BENCHMARK_LOOKS_BEFORE[ext_key]\\n        out[ext_key] = dict(looks_before=lb, evaluated=True, status=\\"CONFIRMATORY\\" if lb == 0 else \\"EXPLORATORY\\",\\n                            label=EXT_REPORT.get(\\"label\\"))\\n    elif EXT_RECORDS:\\n        out[\\"msrvtt_dev\\"] = dict(looks_before=None, evaluated=True, status=\\"DEV (thử đường ống)\\", label=EXT_REPORT.get(\\"label\\"))\\n    else:\\n        out[\\"msrvtt\\"] = dict(evaluated=False, status=EXT_STATUS)\\n    return out\\n\\n\\ndef node_name(i):\\n    return INV.name[int(i)]\\n\\n\\ndef edges_into_tags(g, src, tags):\\n    \\"\\"\\"Cạnh quan hệ src –t→ c với c ∈ tags (đúng chiều QC-RGAT truyền tin: nguồn → nhãn của video).\\"\\"\\"\\n    out = []\\n    for t in RELATION_TYPES:\\n        A = g.A[t]\\n        if A.nnz == 0:\\n            continue\\n        row = A.getrow(int(src))\\n        for c in row.indices:\\n            if int(c) in tags:\\n                base = t[:-4] if t.endswith(\\"_inv\\") else t\\n                key = (base, int(c), int(src)) if t.endswith(\\"_inv\\") else (base, int(src), int(c))\\n                pd_ = g.pdist.get(key, {}) if base in PRED_FAMILIES else {}\\n                p = max(pd_, key=pd_.get) if pd_ else \\"\\"\\n                out.append((int(src), t, int(c), p))\\n    return out\\n\\n\\ndef edge_str(e):\\n    s_, t, d_, p = e\\n    return f\\"{node_name(s_)} –{t}{\'[\' + p + \']\' if p else \'\'}→ {node_name(d_)}\\"\\n\\n\\ndef build_eval_queries(part, g, n_per_group=None, seed=None):\\n    \\"\\"\\"Bộ truy vấn ĐĂNG KÝ TRƯỚC (mục 7.3). Chỉ dùng: văn bản truy vấn (mục đích query_input), nhãn CLIP zero-shot của video đúng,\\n    cạnh đồ thị dựng từ caption TRAIN. KHÔNG dùng điểm hay thứ hạng của bất kỳ mô hình nào.\\n    trực tiếp : mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.\\n    gián tiếp : truy vấn nêu ≥ 1 quan hệ (bộ ba), nhãn video phủ MỘT PHẦN khái niệm truy vấn, và có cạnh quan hệ nối một khái\\n                niệm còn thiếu vào một nhãn của video (= quan hệ dự kiến phải khai thác).\\"\\"\\"\\n    n_per_group = n_per_group or CFG.n_eval_queries\\n    seed = CFG.eval_query_seed if seed is None else seed\\n    q_all, gold_all = part[\\"q_idx\\"], part[\\"gold\\"]\\n    texts = CAPS.texts(q_all, \\"query_input\\")\\n    pools = {\\"direct\\": [], \\"indirect\\": []}\\n    for j, q in enumerate(q_all):\\n        v, hits = int(gold_all[j]), set(V4_HITS[int(q)])\\n        tags = set(g.vid_tags[v])\\n        if not hits:\\n            continue\\n        seen, miss = hits & tags, hits - tags\\n        base = dict(q_idx=int(q), text=texts[j], gold_vid=v, gold_video_id=RECORDS[v][\\"video_id\\"],\\n                    query_concepts=\\"; \\".join(sorted(node_name(h) for h in hits)),\\n                    concepts_seen_in_video=\\"; \\".join(sorted(node_name(h) for h in seen)),\\n                    concepts_missing=\\"; \\".join(sorted(node_name(h) for h in miss)))\\n        if not miss:\\n            pools[\\"direct\\"].append(dict(base, type=\\"direct\\", expected_relations=\\"\\",\\n                                        reason=f\\"mọi khái niệm của truy vấn ({base[\'query_concepts\']}) là nhãn CLIP zero-shot của video đúng\\"))\\n            continue\\n        trip = [t for t in parse_caption_v2(texts[j])[2] if INV.key2id.get(t[0]) is not None and INV.key2id.get(t[2]) is not None]\\n        if not trip or not seen:\\n            continue\\n        bridge = [e for m_ in sorted(miss) for e in edges_into_tags(g, m_, tags)]\\n        if not bridge:\\n            continue\\n        rel_txt = \\"; \\".join(f\\"{s_.split(\'#\')[0]} –{r_}{\'[\' + p_ + \']\' if p_ else \'\'}→ {o_.split(\'#\')[0]}\\" for s_, r_, o_, p_ in sorted(trip))\\n        pools[\\"indirect\\"].append(dict(base, type=\\"indirect\\", expected_relations=\\" || \\".join(edge_str(e) for e in bridge[:5]),\\n                                      reason=(f\\"truy vấn nêu quan hệ ({rel_txt}); video chỉ có nhãn {base[\'concepts_seen_in_video\']}, \\"\\n                                              f\\"thiếu {base[\'concepts_missing\']}; {len(bridge)} cạnh quan hệ nối khái niệm thiếu vào nhãn video\\")))\\n    rows, info = [], {}\\n    for k, (grp, pool) in enumerate(pools.items()):\\n        order = np.random.RandomState(seed + k).permutation(len(pool))\\n        used, take = set(), []\\n        for i in order:\\n            if pool[i][\\"gold_vid\\"] in used:\\n                continue\\n            used.add(pool[i][\\"gold_vid\\"]); take.append(pool[i])\\n            if len(take) >= n_per_group:\\n                break\\n        for n_, r in enumerate(take, 1):\\n            rows.append(dict(query_id=f\\"{part[\'name\']}-{grp[0].upper()}{n_:02d}\\", part=part[\\"name\\"], **r))\\n        info[grp] = dict(eligible=len(pool), eligible_videos=len({r[\\"gold_vid\\"] for r in pool}), selected=len(take))\\n    return pd.DataFrame(rows), info\\n\\n\\ndef eval_query_metrics(E, QS):\\n    \\"\\"\\"R@1/5/10, MRR (trung bình ± std theo seed) của MỌI thí nghiệm trên bộ truy vấn đăng ký trước, theo nhóm.\\"\\"\\"\\n    part = E[\\"part\\"]\\n    pos = {int(q): i for i, q in enumerate(part[\\"q_idx\\"])}\\n    rows, pairs = [], []\\n    for grp in (\\"all\\", \\"direct\\", \\"indirect\\"):\\n        sel = QS if grp == \\"all\\" else QS[QS.type == grp]\\n        if not len(sel):\\n            continue\\n        idx = np.array([pos[int(q)] for q in sel.q_idx])\\n        gold = part[\\"gold\\"][idx]\\n        for k, ev_k in E[\\"ev\\"].items():\\n            ms = [metrics_from_ranks(v[\\"rank\\"][idx], gold) for v in ev_k.values()]\\n            rows.append(dict(part=part[\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], experiment_id=k, group=grp, n_queries=len(idx),\\n                             n_videos=len(set(gold.tolist())), n_seeds=len(ms),\\n                             **{f\\"{x}_mean\\": float(np.mean([mm[x] for mm in ms])) for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")},\\n                             **{f\\"{x}_std\\": float(np.std([mm[x] for mm in ms])) for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\")}))\\n        for a in (G_IDS.get(\\"G3\\"), G_IDS.get(\\"V3REF\\")):                 # QC-RGAT và GraphSAGE có kiểu cạnh, mỗi cái vs CLIP-only\\n            if a in E[\\"ev\\"]:\\n                rr = lambda k_: np.mean([1.0 / v[\\"rank\\"][idx] for v in E[\\"ev\\"][k_].values()], 0)\\n                bs = cluster_bootstrap_diff(rr(a), rr(\\"G0_clip_only\\"), gold)\\n                pairs.append(dict(part=part[\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], group=grp, a=a, b=\\"G0_clip_only\\", n=len(idx),\\n                                  MRR_a=float(rr(a).mean()), MRR_b=float(rr(\\"G0_clip_only\\").mean()), **bs))\\n    return pd.DataFrame(rows), pd.DataFrame(pairs)\\n\\n\\ndef drop_graph_parts(D, vid, cls_drop=(), rel_drop=()):\\n    \\"\\"\\"Bản sao dữ liệu đồ thị với (1) cạnh contains video –chứa→ c (c ∈ cls_drop) của RIÊNG video `vid` bị xoá (chuẩn hoá lại hàng\\n    như lúc dựng), (2) cạnh quan hệ (chỉ số trong D) bị xoá. Mô hình và tham số giữ nguyên.\\"\\"\\"\\n    D2 = RelDataV4(D.graph, drop_edges=D.gidx[list(rel_drop)]) if len(rel_drop) else _copy.copy(D)\\n    if len(cls_drop):\\n        A = D2.Acon.clone()\\n        row = (A[vid] > 0).float()\\n        row[list(cls_drop)] = 0.0\\n        A[vid] = row / row.sum().clamp(min=1.0)\\n        D2 = _copy.copy(D2)\\n        D2.Acon = A\\n    return D2\\n\\n\\ndef term_text(D, t, vid):\\n    if t[\\"kind\\"] == \\"contains\\":\\n        return f\\"{RECORDS[vid][\'video_id\']} –chứa→ {node_name(D.N + t[\'node\'])}\\"\\n    return v4_path_text(D, t, vid)\\n\\n\\ndef term_valid(D, t, vid, q):\\n    \\"\\"\\"Cạnh của số hạng có thật trong đồ thị đang chạy, đúng loại, đúng chiều, đúng kiểu node; vị từ khớp (nếu có) nằm trên ĐÚNG cạnh.\\"\\"\\"\\n    if t[\\"kind\\"] == \\"contains\\":\\n        c = D.N + t[\\"node\\"]\\n        ok = D.graph.A_con[vid, c] > 0 and INV.node_type[c] != \\"video\\" and INV.node_type[vid] == \\"video\\"\\n        return bool(ok), True\\n    v = [v4_edge_valid(D, k, q) for k in t[\\"edges\\"]]\\n    return all(x[0] for x in v), all(x[2] for x in v)\\n\\n\\n@torch.no_grad()\\ndef explain_full(model, q, vid, top_m=None):\\n    \\"\\"\\"Giải thích ĐẦY ĐỦ \\"vì sao video được xếp hạng như vậy\\" = top-m số hạng ĐỒ THỊ ủng hộ (contains + đường đi quan hệ).\\n    Số hạng thị giác (W_v x_v) không phải đồ thị nên được báo riêng. Σ mọi số hạng = điểm đồ thị của mô hình.\\"\\"\\"\\n    top_m = top_m or CFG.top_m_paths\\n    terms = model.decompose(q, vid)\\n    graph_terms = [t for t in terms if t[\\"kind\\"] != \\"visual\\"]\\n    sup = sorted([t for t in graph_terms if t[\\"contrib\\"] > 0], key=lambda t: -t[\\"contrib\\"])[:top_m]\\n    opp = sorted([t for t in graph_terms if t[\\"contrib\\"] < 0], key=lambda t: t[\\"contrib\\"])[:top_m]\\n    tot = {k: sum(t[\\"contrib\\"] for t in terms if t[\\"kind\\"] == k) for k in (\\"visual\\", \\"contains\\", \\"relation\\")}\\n    return dict(terms=terms, support=sup, oppose=opp, totals=tot, score=sum(t[\\"contrib\\"] for t in terms),\\n                n_graph_terms=len(graph_terms), pos_graph=sum(t[\\"contrib\\"] for t in graph_terms if t[\\"contrib\\"] > 0))\\n\\n\\ndef _deletion_sets(sup):\\n    return (sorted({t[\\"node\\"] for t in sup if t[\\"kind\\"] == \\"contains\\"}),\\n            sorted({k for t in sup if t[\\"kind\\"] == \\"relation\\" for k in t[\\"edges\\"]}))\\n\\n\\n@torch.no_grad()\\ndef xai_rank_fidelity(run, part, n_queries=None, top_m=None, seed=0):\\n    \\"\\"\\"Fidelity theo QUYẾT ĐỊNH xếp hạng (mục 8.3): với ≥ n truy vấn có video đúng trong top-K ứng viên, xoá THẬT các cạnh của\\n    giải thích đầy đủ (contains của video + cạnh quan hệ) rồi chấm lại TOÀN BỘ ứng viên ⇒ Δ điểm đồ thị, Δ điểm cuối, Δ hạng,\\n    đổi top-1 / top-10; đối chứng = xoá cùng số cạnh contains khác của video + cùng số cạnh quan hệ ngẫu nhiên vào nhãn của video.\\n    Báo riêng: hạng không đổi thì ghi đúng là không đổi.\\"\\"\\"\\n    n_queries, top_m = n_queries or CFG.n_explain_queries, top_m or CFG.top_m_paths\\n    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n    D = m.D\\n    rng = np.random.RandomState(seed)\\n    q_all = part[\\"q_idx\\"]\\n    cand_all, s_clip = eng.clip_candidates(q_all, part[\\"gallery\\"], CFG.rerank_k)\\n    rows = []\\n    for i in rng.permutation(len(q_all)):\\n        if len(rows) >= n_queries:\\n            break\\n        q, gold = int(q_all[i]), int(part[\\"gold\\"][i])\\n        cand = cand_all[i]\\n        if gold not in cand:\\n            continue\\n        ex = explain_full(m, q, gold, top_m)\\n        if not ex[\\"support\\"]:\\n            continue\\n        zc = znorm_per_row(s_clip[i:i + 1])[0]\\n        r_full, s_full, fin_full = v4_rank_after(run, q, gold, cand, zc, D)\\n        j = int(np.where(cand == gold)[0][0])\\n        con_del, rel_del = _deletion_sets(ex[\\"support\\"])\\n        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, con_del, rel_del))\\n        tags = torch.nonzero(D.Acon[gold] > 0).flatten().tolist()\\n        other = [c for c in tags if c not in con_del]\\n        rc = [int(x) for x in rng.choice(other, min(len(con_del), len(other)), replace=False)] if (other and con_del) else []\\n        pool = [k for k in range(D.E) if int(D.dst_np[k]) in set(tags) and k not in rel_del]\\n        rr = [int(x) for x in rng.choice(pool, min(len(rel_del), len(pool)), replace=False)] if (pool and rel_del) else []\\n        r_rd, s_rd, fin_rd = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, rc, rr))\\n        val = [term_valid(D, t, gold, q) for t in ex[\\"support\\"]]\\n        expl_edges = len(con_del) + len(rel_del)\\n        top10 = lambda f: set(np.argsort(-f)[:10].tolist())\\n        rows.append(dict(q=q, gold=gold, gold_video_id=RECORDS[gold][\\"video_id\\"], n_graph_terms=ex[\\"n_graph_terms\\"],\\n                         n_expl_terms=len(ex[\\"support\\"]), n_expl_contains=sum(t[\\"kind\\"] == \\"contains\\" for t in ex[\\"support\\"]),\\n                         n_expl_relation=sum(t[\\"kind\\"] == \\"relation\\" for t in ex[\\"support\\"]), n_expl_edges=expl_edges,\\n                         n_rand_edges=len(rc) + len(rr), sparsity=len(ex[\\"support\\"]) / max(1, ex[\\"n_graph_terms\\"]),\\n                         coverage=sum(t[\\"contrib\\"] for t in ex[\\"support\\"]) / max(1e-9, ex[\\"pos_graph\\"]),\\n                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[1] for v in val])),\\n                         decomposition_ok=float(abs(ex[\\"score\\"] - s_full) < 1e-4),\\n                         s_graph=float(s_full), d_s_expl=float(s_ex - s_full), d_s_rand=float(s_rd - s_full),\\n                         d_final_expl=float(fin_ex[j] - fin_full[j]), d_final_rand=float(fin_rd[j] - fin_full[j]),\\n                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,\\n                         rank_worse_expl=float(r_ex > r_full), rank_worse_rand=float(r_rd > r_full),\\n                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),\\n                         top1_changed_rand=float(np.argmax(fin_rd) != np.argmax(fin_full)),\\n                         top10_changed_expl=float(top10(fin_ex) != top10(fin_full)), top10_changed_rand=float(top10(fin_rd) != top10(fin_full)),\\n                         share_graph=(ex[\\"totals\\"][\\"contains\\"] + ex[\\"totals\\"][\\"relation\\"]) / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n                         share_relation=ex[\\"totals\\"][\\"relation\\"] / max(1e-9, sum(abs(x) for x in ex[\\"totals\\"].values())),\\n                         explanation=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {term_text(D, t, gold)}\\" for t in ex[\\"support\\"]),\\n                         opposing=\\" || \\".join(f\\"{t[\'contrib\']:+.4f} {term_text(D, t, gold)}\\" for t in ex[\\"oppose\\"][:3])))\\n    return pd.DataFrame(rows)\\n\\n\\ndef diagnose_case(run, q, gold, cand, rank, clip_rank, text):\\n    \\"\\"\\"Nguyên nhân (luật cố định): candidate_recall_low | tag_missing | missing_edge | predicate_not_in_graph | wrong_predicate |\\n    noisy_relation | distractor_shares_concepts | graph_helped | clip_already_correct.\\"\\"\\"\\n    g, D = run[\\"graph\\"], run[\\"model\\"].D\\n    hits, tags = set(V4_HITS[q]), set(g.vid_tags[gold])\\n    causes, notes = [], []\\n    if gold not in cand:\\n        causes.append(\\"candidate_recall_low\\"); notes.append(f\\"video đúng ở hạng CLIP {clip_rank} > k = {CFG.rerank_k}\\")\\n    if hits and not hits & tags:\\n        causes.append(\\"tag_missing\\"); notes.append(\\"không khái niệm nào của truy vấn là nhãn của video đúng\\")\\n    for s_, r_, o_, p_ in sorted(parse_caption_v2(text)[2]):\\n        a, b = INV.key2id.get(s_), INV.key2id.get(o_)\\n        if a is None or b is None or a == b:\\n            continue\\n        on = [t for t in RELATION_TYPES if g.A[t][a, b] != 0 or g.A[t][b, a] != 0]\\n        if not on:\\n            causes.append(\\"missing_edge\\"); notes.append(f\\"không có cạnh nào giữa {s_} và {o_} (quan hệ \'{r_}\' của truy vấn)\\")\\n        elif r_ in PRED_FAMILIES and p_:\\n            if (r_, p_) not in PRED_ID:\\n                causes.append(\\"predicate_not_in_graph\\"); notes.append(f\\"vị từ \'{p_}\' không có trong từ vựng vị từ (train)\\")\\n            else:\\n                pd_ = g.pdist.get((r_, a, b), {}) or g.pdist.get((r_, b, a), {})\\n                if pd_ and p_ not in pd_:\\n                    causes.append(\\"wrong_predicate\\")\\n                    notes.append(f\\"cạnh {s_}–{r_}–{o_} mang vị từ {sorted(pd_)[:3]}, không có \'{p_}\' của truy vấn\\")\\n    if gold in cand:\\n        ex = explain_full(run[\\"model\\"], q, gold)\\n        if ex[\\"totals\\"][\\"relation\\"] < 0 and rank > clip_rank:\\n            causes.append(\\"noisy_relation\\"); notes.append(f\\"tổng đóng góp quan hệ cho video đúng âm ({ex[\'totals\'][\'relation\']:+.4f})\\")\\n        if rank < clip_rank:\\n            causes.append(\\"graph_helped\\"); notes.append(f\\"đồ thị kéo video đúng từ hạng {clip_rank} lên {rank}\\")\\n    if clip_rank == 1 and rank == 1:\\n        causes.append(\\"clip_already_correct\\")\\n    return list(dict.fromkeys(causes)), notes\\n\\n\\nFAIL_CAUSES = (\\"candidate_recall_low\\", \\"missing_edge\\", \\"wrong_predicate\\", \\"noisy_relation\\", \\"tag_missing\\", \\"predicate_not_in_graph\\")\\n\\n\\n@torch.no_grad()\\ndef case_analysis_l4b(run, part, n_each=None, seed=0, max_q=4000):\\n    \\"\\"\\"5 thành công (hạng 1) + 5 thất bại (hạng > 10), chọn theo LUẬT: thành công ưu tiên ca đồ thị kéo hạng lên rồi tới ca CLIP đã đúng;\\n    thất bại lấy mỗi nguyên nhân trong FAIL_CAUSES một ca (nếu có) rồi bổ sung ngẫu nhiên — không chọn ví dụ đẹp.\\"\\"\\"\\n    n_each = n_each or CFG.n_cases\\n    m, eng = run[\\"model\\"], run[\\"engine\\"]\\n    rng = np.random.RandomState(seed)\\n    q = part[\\"q_idx\\"]\\n    sel = np.sort(rng.choice(len(q), min(max_q, len(q)), replace=False))\\n    q, gold = q[sel], part[\\"gold\\"][sel]\\n    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part[\\"gallery\\"], gold)\\n    sc = score_queries_v4(m, eng, q, part[\\"gallery\\"], CFG.rerank_k)\\n    fin = combine(sc[\\"s_clip\\"], sc[\\"s_gnn\\"], sc[\\"s_path\\"], run[\\"alpha\\"], 1.0)\\n    rank = ranks_with_fallback(sc[\\"cand\\"], fin, gold, clip_rank)\\n    texts = CAPS.texts(q, \\"report\\")\\n    order = rng.permutation(len(q))\\n    succ = [i for i in order if rank[i] == 1 and clip_rank[i] > 1][:3]\\n    succ += [i for i in order if rank[i] == 1 and i not in succ][:n_each - len(succ)]\\n    fail_pool = [i for i in order if rank[i] > 10]\\n    diag = {}\\n    for i in fail_pool[:400]:\\n        diag[i] = diagnose_case(run, int(q[i]), int(gold[i]), sc[\\"cand\\"][i], int(rank[i]), int(clip_rank[i]), texts[i])\\n    fail = []\\n    for c in FAIL_CAUSES:\\n        hit = next((i for i in fail_pool[:400] if c in diag[i][0] and i not in fail), None)\\n        if hit is not None and len(fail) < n_each:\\n            fail.append(hit)\\n    fail += [i for i in fail_pool if i not in fail][:n_each - len(fail)]\\n    cases = []\\n    for kind, idx in ((\\"THÀNH CÔNG\\", succ), (\\"THẤT BẠI\\", fail)):\\n        for i in idx:\\n            qi, gi = int(q[i]), int(gold[i])\\n            causes, notes = diag.get(i) or diagnose_case(run, qi, gi, sc[\\"cand\\"][i], int(rank[i]), int(clip_rank[i]), texts[i])\\n            top = sc[\\"cand\\"][i][np.argsort(-fin[i])[:3]].tolist()\\n            ex = explain_full(m, qi, gi) if gi in sc[\\"cand\\"][i] else None\\n            ex_top1 = explain_full(m, qi, top[0]) if top and top[0] != gi else None\\n            cases.append(dict(kind=kind, q=qi, query=texts[i], gold_vid=gi, gold_video_id=RECORDS[gi][\\"video_id\\"],\\n                              rank_clip=int(clip_rank[i]), rank_final=int(rank[i]), in_candidates=bool(gi in sc[\\"cand\\"][i]),\\n                              top3=[RECORDS[v][\\"video_id\\"] for v in top], causes=causes, notes=notes,\\n                              gold_totals={k_: round(v_, 4) for k_, v_ in ex[\\"totals\\"].items()} if ex else None,\\n                              gold_explanation=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, gi)}\\" for t in ex[\\"support\\"]] if ex else [],\\n                              gold_opposing=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, gi)}\\" for t in ex[\\"oppose\\"][:3]] if ex else [],\\n                              top1_explanation=[f\\"{t[\'contrib\']:+.4f} {term_text(m.D, t, top[0])}\\" for t in ex_top1[\\"support\\"]] if ex_top1 else []))\\n    stats = dict(n_queries=len(q), top1=float(np.mean(rank == 1)), fail_gt10=float(np.mean(rank > 10)),\\n                 cause_counts_in_fail_sample=dict(Counter(c for i in diag for c in diag[i][0])), n_fail_diagnosed=len(diag))\\n    return cases, stats\\n\\n\\ndef cases_markdown(cases, stats, part_name):\\n    L = [f\\"# Phân tích 5 thành công + 5 thất bại — {part_name}\\", \\"\\",\\n         \\"Chọn theo luật cố định (xem `case_analysis_l4b`), KHÔNG chọn tay. Thành công = video đúng ở hạng 1; thất bại = hạng > 10.\\",\\n         f\\"Thống kê trên {stats[\'n_queries\']} truy vấn mẫu: hạng 1 = {stats[\'top1\']:.1%}, hạng > 10 = {stats[\'fail_gt10\']:.1%}; \\"\\n         f\\"nguyên nhân trong mẫu thất bại đã chẩn đoán ({stats[\'n_fail_diagnosed\']}): {stats[\'cause_counts_in_fail_sample\']}\\", \\"\\"]\\n    for n_, c in enumerate(cases, 1):\\n        L += [f\\"## {n_}. [{c[\'kind\']}] \\\\\\"{c[\'query\']}\\\\\\"\\", \\"\\",\\n              f\\"- Video đúng: `{c[\'gold_video_id\']}` — hạng CLIP **{c[\'rank_clip\']}** → hạng cuối **{c[\'rank_final\']}**\\"\\n              f\\"{\'\' if c[\'in_candidates\'] else \' (KHÔNG nằm trong top-k ứng viên CLIP ⇒ đồ thị không thể cứu)\'}\\",\\n              f\\"- Top-3 trả về: {\', \'.join(\'`\' + v + \'`\' for v in c[\'top3\'])}\\",\\n              f\\"- Chẩn đoán (luật cố định{\'; nguyên nhân thất bại\' if c[\'kind\'] == \'THẤT BẠI\' else \'\'}): {\', \'.join(c[\'causes\']) or \'—\'}\\"] + \\\\\\n            [f\\"  - {x}\\" for x in c[\\"notes\\"]]\\n        if c[\\"gold_totals\\"]:\\n            L.append(f\\"- Phân rã điểm đồ thị của video đúng: {c[\'gold_totals\']}\\")\\n        L += [\\"- Đường đi / quan hệ ỦNG HỘ video đúng (đóng góp thật vào điểm):\\"] + [f\\"  - {x}\\" for x in c[\\"gold_explanation\\"] or [\\"(không có số hạng đồ thị dương)\\"]]\\n        if c[\\"gold_opposing\\"]:\\n            L += [\\"- Số hạng PHẢN ĐỐI:\\"] + [f\\"  - {x}\\" for x in c[\\"gold_opposing\\"]]\\n        if c[\\"top1_explanation\\"]:\\n            L += [f\\"- Vì sao `{c[\'top3\'][0]}` (hạng 1, sai) được xếp trên:\\"] + [f\\"  - {x}\\" for x in c[\\"top1_explanation\\"]]\\n        L.append(\\"\\")\\n    return \\"\\\\n\\".join(L)\\n\\n\\n# ---------------------------------- Demo: văn bản tự do → top-K video + giải thích ----------------------------------\\ndef thumb_path(vid):\\n    \\"\\"\\"Thumbnail khung giữa: dùng file đã lưu lúc trích đặc trưng; nếu không có (dùng cache) thì đọc lại từ video.\\"\\"\\"\\n    r = RECORDS[vid]\\n    p = CFG.out_dir / \\"thumbs\\" / f\\"{r[\'video_id\']}.jpg\\"\\n    if p.exists():\\n        return p\\n    try:\\n        src = SOURCE if vid < N_MSVD else EXT_SOURCE\\n        with src.open(r[\\"video_id\\"]) as path:\\n            frames, _ = read_sampled_frames(path, 3)\\n        mid = frames[1]\\n        h, w = mid.shape[:2]\\n        th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)\\n        cv2.imwrite(str(p), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))\\n        return p\\n    except Exception:\\n        return None\\n\\n\\n@torch.no_grad()\\ndef demo_query(text, run=None, part=None, k=None, show=True, save_png=None):\\n    \\"\\"\\"Nhập văn bản tự do ⇒ top-k video của gallery `part` (mặc định tập xác nhận, nếu không thì tập thăm dò) với điểm CLIP, điểm đồ thị,\\n    điểm cuối (α đã đóng băng) và giải thích = số hạng đồ thị ủng hộ. Văn bản demo KHÔNG đi vào CaptionStore và không được lưu làm dữ liệu.\\"\\"\\"\\n    global TXT_FEAT\\n    run = run or EXPS[G_IDS[\\"G3\\"]][\\"runs\\"][CFG.seeds[0]]\\n    part = part or (EVALS[\\"confirm\\"][\\"part\\"] if \\"confirm\\" in EVALS else EVALS[\\"explore\\"][\\"part\\"])\\n    k = k or CFG.demo_topk\\n    m = run[\\"model\\"]\\n    D = m.D\\n    t = clip_encode_texts([text])[0].astype(TXT_FEAT.dtype)\\n    q = len(TXT_FEAT)\\n    toks = query_tokens_v2(text)\\n    saved = (TXT_FEAT,)\\n    TXT_FEAT = np.vstack([TXT_FEAT, t[None]])\\n    V4_HITS.append(sorted({nid for nid, _, _ in parse_tokens_v2(toks, INV)}))\\n    V4_ROLES.append(query_roles_v4([text], INV)[0])\\n    QPREDS.append({rel: match_predicates(query_pred_tokens(text), rel) for rel in PRED_FAMILIES})\\n    try:\\n        gal = np.asarray(part[\\"gallery\\"])\\n        cos = VID_FEAT[gal] @ t\\n        top = np.argsort(-cos)[:CFG.rerank_k]\\n        cand, s_clip = gal[top], cos[top]\\n        qb = D.query_batch([q], m.use_pred)\\n        s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()[0]\\n        fin = run[\\"alpha\\"] * znorm_per_row(s_clip[None])[0] + (1 - run[\\"alpha\\"]) * znorm_per_row(s[None])[0]\\n        order = np.argsort(-fin)[:k]\\n        out = dict(query=text, gallery=part[\\"name\\"], alpha=run[\\"alpha\\"], matched_concepts=[node_name(h) for h in V4_HITS[q]],\\n                   results=[])\\n        for r_, j in enumerate(order, 1):\\n            v = int(cand[j])\\n            ex = explain_full(m, q, v)\\n            out[\\"results\\"].append(dict(rank=r_, video_id=RECORDS[v][\\"video_id\\"], clip_cos=float(s_clip[j]), graph_score=float(s[j]),\\n                                       final=float(fin[j]), clip_rank=int(j) + 1,          # cand đã xếp theo cosine giảm dần\\n                                       totals={k_: round(v_, 4) for k_, v_ in ex[\\"totals\\"].items()},\\n                                       explanation=[f\\"{tt[\'contrib\']:+.4f} {term_text(D, tt, v)}\\" for tt in ex[\\"support\\"]],\\n                                       note=\\"\\" if ex[\\"support\\"] else \\"KHÔNG có đường đi giải thích hợp lệ (không số hạng đồ thị nào ủng hộ)\\"))\\n    finally:\\n        TXT_FEAT = saved[0]\\n        V4_HITS.pop(); V4_ROLES.pop(); QPREDS.pop()\\n    if show:\\n        print(f\\"\\\\nTruy vấn: \'{text}\' | gallery {out[\'gallery\']} ({len(gal)} video) | khái niệm khớp: {out[\'matched_concepts\'] or \'—\'} | α = {out[\'alpha\']}\\")\\n        for r in out[\\"results\\"]:\\n            print(f\\"  #{r[\'rank\']} {r[\'video_id\']:>12s} | CLIP cos {r[\'clip_cos\']:.3f} (hạng CLIP {r[\'clip_rank\']}) | đồ thị {r[\'graph_score\']:+.3f} \\"\\n                  f\\"| cuối {r[\'final\']:+.3f} | phân rã {r[\'totals\']}\\")\\n            for x in r[\\"explanation\\"]:\\n                print(f\\"       • {x}\\")\\n            if r[\\"note\\"]:\\n                print(f\\"       ⚠ {r[\'note\']}\\")\\n        try:\\n            import matplotlib.pyplot as plt\\n            fig, axes = plt.subplots(1, len(out[\\"results\\"]), figsize=(3 * len(out[\\"results\\"]), 2.6))\\n            for ax, r in zip(np.atleast_1d(axes), out[\\"results\\"]):\\n                v = next(x[\\"vid\\"] for x in RECORDS if x[\\"video_id\\"] == r[\\"video_id\\"])\\n                p = thumb_path(v)\\n                if p is not None:\\n                    ax.imshow(plt.imread(str(p)))\\n                ax.set_title(f\\"#{r[\'rank\']} {r[\'video_id\']}\\\\nCLIP {r[\'clip_cos\']:.2f} | cuối {r[\'final\']:+.2f}\\", fontsize=8)\\n                ax.axis(\\"off\\")\\n            fig.suptitle(text, fontsize=9)\\n            fig.tight_layout()\\n            if save_png:\\n                fig.savefig(save_png, dpi=110)\\n            plt.show()\\n            plt.close(fig)\\n        except Exception as e:\\n            print(f\\"  (không vẽ được thumbnail: {type(e).__name__}: {e})\\")\\n    return out\\n\\n\\ndef demo_widget():\\n    \\"\\"\\"Ô nhập truy vấn tương tác (Jupyter / Colab / Kaggle interactive). Không tự chạy khi Save & Run All.\\"\\"\\"\\n    import ipywidgets as w\\n    box, btn, out = w.Text(placeholder=\\"ví dụ: a man is riding a horse\\", layout=w.Layout(width=\\"60%\\")), w.Button(description=\\"Tìm\\"), w.Output()\\n\\n    def go(_):\\n        with out:\\n            out.clear_output()\\n            demo_query(box.value)\\n    btn.on_click(go)\\n    display(w.HBox([box, btn]), out)\\n\\n\\n# ---------------------------------- Kiểm kê, audit, kết luận, manifest ----------------------------------\\ndef file_sha1(p, n=1 << 20):\\n    h = hashlib.sha1()\\n    with open(p, \\"rb\\") as f:\\n        while True:\\n            b = f.read(n)\\n            if not b:\\n                break\\n            h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef split_counts():\\n    out = {}\\n    for s in sorted(set(CAP_SPLIT.tolist())):\\n        vids = sorted(set(CAP_VID[CAP_SPLIT == s].tolist()))\\n        out[s] = dict(videos=len(vids), captions=int((CAP_SPLIT == s).sum()),\\n                      video_ids_sha1=sha1_of([RECORDS[v][\\"video_id\\"] for v in vids]))\\n    return out\\n\\n\\ndef graph_inventory(g):\\n    s, d, e, w, pi = g.edge_arrays()\\n    et = Counter(EDGE_TYPES[int(x)] for x in e)\\n    nt = Counter(INV.node_type)\\n    deg = np.bincount(np.concatenate([s, d]), minlength=INV.n_nodes) if len(s) else np.zeros(INV.n_nodes, int)\\n    nonvid = np.array([t != \\"video\\" for t in INV.node_type])\\n    return dict(nodes_by_type=dict(nt), edges_by_type=dict(et), n_edges=int(len(e)), self_loops=int((s == d).sum()),\\n                duplicate_edges=int(len(e) - len(set(zip(s.tolist(), d.tolist(), e.tolist())))),\\n                isolated_non_video_nodes=int(((deg == 0) & nonvid).sum()), videos_without_tag=int(sum(len(x) == 0 for x in g.vid_tags)),\\n                degree_non_video=dict(mean=float(deg[nonvid].mean()), median=float(np.median(deg[nonvid])), max=int(deg[nonvid].max())),\\n                edges_with_predicate=int((pi >= 0).sum()), schema_sha1=SCHEMA[\\"sha1\\"], inventory_sha1=INV.sha1)"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) v2: tagger HỌC từ caption TRAIN + tri thức ngoài ConceptNet (chỉ ĐỊNH NGHĨA) ==============================\\nimport gzip, urllib.request\\n\\nCaptionStore.ALLOWED[\\"tagger_train\\"] = {\\"train\\"}      # nhãn huấn luyện tagger học: CHỈ caption TRAIN\\nKG_EDGES = []                                         # cạnh ConceptNet (khoá từ vựng) — nạp ở stage V8\\nKG_KEEP = (\\"RelatedTo\\", \\"IsA\\", \\"PartOf\\", \\"HasA\\", \\"UsedFor\\", \\"CapableOf\\", \\"AtLocation\\", \\"Causes\\", \\"HasSubevent\\", \\"HasFirstSubevent\\",\\n           \\"HasLastSubevent\\", \\"HasPrerequisite\\", \\"MotivatedByGoal\\", \\"Desires\\", \\"CreatedBy\\", \\"Synonym\\", \\"MadeOf\\", \\"ReceivesAction\\",\\n           \\"LocatedNear\\", \\"SimilarTo\\", \\"MannerOf\\", \\"Entails\\", \\"InstanceOf\\")   # quan hệ DƯƠNG; bỏ Antonym, Not*, DistinctFrom, FormOf, ExternalURL…\\n\\n\\ndef drop_mask(mask, name):\\n    parts = [m for m in str(mask).split(\\"+\\") if m not in (name, \\"none\\")]\\n    return \\"+\\".join(sorted(parts)) or \\"none\\"\\n\\n\\ndef _probe_topk(S, rows, cols, kn, kv, terms):\\n    out = []\\n    for r in rows:\\n        d = {}\\n        for typ, k in ((\\"noun\\", kn), (\\"verb\\", kv)):\\n            c = cols[typ]\\n            if k > 0 and len(c):\\n                top = c[np.argsort(-S[r, c])[:k]]\\n                d.update({terms[i]: float(S[r, i]) for i in top})\\n        out.append(d)\\n    return out\\n\\n\\ndef run_tagger_probe(vid_feat, vocab, split, store, seed=0):\\n    \\"\\"\\"Tagger HỌC: hồi quy logistic đa nhãn (một lớp tuyến tính) trên embedding CLIP đóng băng của video (chuẩn hoá theo TRAIN).\\n    Nhãn = khái niệm xuất hiện ở ≥ truth_min_caps caption của video TRAIN. Chọn (weight decay, k danh từ, k động từ) theo F1 trên VAL.\\n    Video TRAIN nhận nhãn OUT-OF-FOLD (k-fold) để nhiễu giống video mới; val/test/tập ngoài dùng mô hình học trên toàn bộ TRAIN.\\n    Chỉ dùng frame của chính video lúc suy luận ⇒ không rò rỉ caption val/test.\\"\\"\\"\\n    terms = vocab[\\"terms\\"]\\n    idx = {t: i for i, t in enumerate(terms)}\\n    cols = {t: np.array([i for i, w in enumerate(terms) if vocab[\\"types\\"][w] == t], int) for t in (\\"noun\\", \\"verb\\")}\\n    tr, va = np.array(split[\\"train\\"]), np.array(split[\\"val\\"])\\n    truth_tr = caption_truth_v2(store, tr, vocab, \\"tagger_train\\")\\n    truth_va = caption_truth_v2(store, va, vocab, \\"tagger_select\\")\\n    Y = np.zeros((len(tr), len(terms)), np.float32)\\n    for j, v in enumerate(tr):\\n        Y[j, [idx[t] for t in truth_tr[v]]] = 1.0\\n    X = np.asarray(vid_feat, np.float32)\\n    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6\\n    Xt = torch.from_numpy((X - mu) / sd).to(DEVICE)\\n\\n    def fit(rows, wd):\\n        set_seed(seed)\\n        y = torch.from_numpy(Y[rows]).to(DEVICE)\\n        prev = y.mean(0).clamp(1e-4, 1 - 1e-4)\\n        lin = nn.Linear(X.shape[1], len(terms)).to(DEVICE)\\n        nn.init.zeros_(lin.weight)\\n        with torch.no_grad():\\n            lin.bias.copy_(torch.log(prev / (1 - prev)))\\n        opt = torch.optim.Adam(lin.parameters(), lr=CFG.probe_lr)\\n        xr = Xt[torch.from_numpy(tr[rows]).to(DEVICE)]\\n        for _ in range(CFG.probe_epochs):\\n            opt.zero_grad()\\n            loss = F.binary_cross_entropy_with_logits(lin(xr), y, reduction=\\"sum\\") / len(rows) + wd * lin.weight.pow(2).sum()\\n            loss.backward()\\n            opt.step()\\n        with torch.no_grad():\\n            return torch.sigmoid(lin(Xt)).cpu().numpy()\\n\\n    allrows = np.arange(len(tr))\\n    grid, best = [], None\\n    for wd in CFG.probe_wd_grid:\\n        S = fit(allrows, wd)\\n        for kn in CFG.tag_k_noun_grid:\\n            for kv in CFG.tag_k_verb_grid:\\n                P, R, F1 = _prf([set(d) for d in _probe_topk(S, va, cols, kn, kv, terms)], [truth_va[v] for v in va])\\n                g = dict(wd=wd, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1)\\n                grid.append(g)\\n                if best is None or (round(F1, 9), -(kn + kv)) > (round(best[\\"val_F1\\"], 9), -(best[\\"k_noun\\"] + best[\\"k_verb\\"])):\\n                    best = g\\n    S = fit(allrows, best[\\"wd\\"])                                      # val / test / tập ngoài: mô hình trên toàn bộ TRAIN\\n    fold = np.random.RandomState(seed).permutation(len(tr)) % CFG.probe_folds\\n    for f in range(CFG.probe_folds):                                   # TRAIN: out-of-fold\\n        S_f = fit(allrows[fold != f], best[\\"wd\\"])\\n        S[tr[fold == f]] = S_f[tr[fold == f]]\\n    tagged = _probe_topk(S, np.arange(len(X)), cols, best[\\"k_noun\\"], best[\\"k_verb\\"], terms)\\n    tags = {v: sorted(d) for v, d in enumerate(tagged)}\\n    scores = {v: d for v, d in enumerate(tagged)}\\n    sel = dict(best, method=\\"logistic đa nhãn trên CLIP video (TRAIN, out-of-fold cho TRAIN)\\", folds=CFG.probe_folds,\\n               epochs=CFG.probe_epochs, n_terms_with_train_positive=int((Y.sum(0) > 0).sum()))\\n    return tags, scores, sel, grid\\n\\n\\ndef _cn_term(uri):\\n    p = uri.split(\\"/\\")\\n    if len(p) < 4 or p[2] != \\"en\\" or \\"_\\" in p[3]:\\n        return None\\n    return p[3], (p[4] if len(p) > 4 and p[4] in (\\"n\\", \\"v\\") else None)\\n\\n\\ndef _cn_keys(t, vset):\\n    \\"\\"\\"Khoá từ vựng của một term ConceptNet: có POS ⇒ đúng POS; không POS ⇒ ưu tiên danh từ, chỉ dùng động từ khi không có danh từ.\\"\\"\\"\\n    if t[1]:\\n        k = f\\"{t[0]}#{t[1]}\\"\\n        return [k] if k in vset else []\\n    for x in (\\"n\\", \\"v\\"):\\n        if f\\"{t[0]}#{x}\\" in vset:\\n            return [f\\"{t[0]}#{x}\\"]\\n    return []\\n\\n\\ndef load_conceptnet(vocab_terms):\\n    \\"\\"\\"Cạnh ConceptNet 5.7 (en–en, một từ, quan hệ DƯƠNG trong KG_KEEP) mà HAI đầu đều là khoá từ vựng TRAIN. Tri thức NGOÀI:\\n    không dùng caption hay nhãn nào ⇒ không rò rỉ. Nguồn: CFG.kg_file | /kaggle/input/**/<tên file> | tải luồng từ CFG.kg_url.\\n    Kết quả lọc được cache (cache/conceptnet_*.json, dùng lại qua Add Input).\\"\\"\\"\\n    vset = set(vocab_terms)\\n    name = f\\"conceptnet_{CFG.kg_version}_{sha1_of(sorted(vset))}_{CFG.kg_max_lines}.json\\"\\n    p = find_cache(name)\\n    if p is not None:\\n        print(f\\"Dùng lại cache ConceptNet: {p}\\")\\n        return json.load(open(p))\\n    fname = Path(CFG.kg_url).name\\n    src = [Path(CFG.kg_file)] if CFG.kg_file else []\\n    if RUN_ON_KAGGLE:\\n        src += [Path(x) for x in glob.glob(f\\"/kaggle/input/**/{fname}\\", recursive=True)]\\n    src = [x for x in src if x.exists()]\\n    t0, edges, n = time.time(), {}, 0\\n    if src:\\n        fh, label = gzip.open(src[0], \\"rt\\", encoding=\\"utf-8\\"), str(src[0])\\n    else:\\n        if not INTERNET:\\n            raise FileNotFoundError(f\\"không có {fname} và không có Internet\\")\\n        fh, label = io.TextIOWrapper(gzip.GzipFile(fileobj=urllib.request.urlopen(CFG.kg_url, timeout=120)), encoding=\\"utf-8\\"), CFG.kg_url\\n    print(f\\"Đọc ConceptNet từ {label} …\\")\\n    with fh:\\n        for n, line in enumerate(fh, 1):\\n            if CFG.kg_max_lines and n > CFG.kg_max_lines:\\n                break\\n            parts = line.split(\\"\\\\t\\")\\n            if len(parts) < 5 or parts[1][3:] not in KG_KEEP:\\n                continue\\n            a, b = _cn_term(parts[2]), _cn_term(parts[3])\\n            if a is None or b is None:\\n                continue\\n            ka, kb = _cn_keys(a, vset), _cn_keys(b, vset)\\n            if not ka or not kb:\\n                continue\\n            m = re.search(r\'\\"weight\\":\\\\s*([0-9.]+)\', parts[4])\\n            w = float(m.group(1)) if m else 1.0\\n            for x in ka:\\n                for y in kb:\\n                    if x != y:\\n                        k = (x, y, parts[1][3:])\\n                        edges[k] = max(edges.get(k, 0.0), w)\\n    out = [dict(src=x, dst=y, rel=r, weight=w) for (x, y, r), w in sorted(edges.items())]\\n    save_json(out, CFG.out_dir / \\"cache\\" / name)\\n    print(f\\"ConceptNet: {n} dòng đọc, {len(out)} cạnh giữa khoá từ vựng ({time.time() - t0:.0f}s)\\")\\n    return out"}, {"cell_type": "markdown", "id": "c045", "metadata": {}, "source": "---\\n# THỰC THI\\n\\nThứ tự chạy:\\n1. Dữ liệu: MSVD và MSR-VTT → split → đặc trưng → pooling.\\n2. Từ vựng, vị từ, tagger, mẫu cho người chấm và nhãn.\\n3. Đồ thị v3, schema, audit → self-test.\\n4. Các stage V0–V7 trên VAL → **đóng băng**.\\n5. **Bộ truy vấn đăng ký trước** (trực tiếp / gián tiếp) → đánh giá test MSVD (thăm dò) và MSR-VTT fresh1k-B (xác nhận).\\n6. Phân tích quan hệ → xAI → metric bộ truy vấn → xAI theo thứ hạng → 5 + 5 ca → demo.\\n7. Xuất file → báo cáo, kết luận → kiểm kê dữ liệu, audit rò rỉ, nghiệm thu, 7 câu kết luận, manifest → nén zip."}, {"cell_type": "code", "execution_count": null, "id": "c046", "metadata": {}, "outputs": [], "source": "# ============================== CỔNG A: dữ liệu thật, đọc được ==============================\\nmark_stage(\\"data\\", requires=(\\"config\\",))\\nif CFG.dataset == \\"msvd\\":\\n    RECORDS, _caps, SOURCE, DATA_REPORT = load_msvd()\\nelse:\\n    RECORDS, _caps, SOURCE, DATA_REPORT = make_synthetic_msvd()\\nN_MSVD = len(RECORDS)\\n\\n# ---- benchmark xác nhận độc lập (MSR-VTT, tập theo CFG.confirm_set — mặc định fresh1k); lỗi tải ⇒ NOT RUN, ghi rõ lý do, KHÔNG thay bằng tập khác ----\\nEXT_RECORDS, EXT_SOURCE, EXT_REPORT, EXT_STATUS = [], None, {}, \\"NOT RUN — CFG.confirm = none\\"\\nif CFG.confirm == \\"msrvtt\\":\\n    try:\\n        if CFG.dataset == \\"msvd\\":\\n            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = load_msrvtt(N_MSVD)\\n        else:\\n            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = make_synthetic_ext(N_MSVD)\\n        _caps.update(_ext_caps)\\n        EXT_STATUS = \\"DEV (không phải tập xác nhận)\\" if EXT_REPORT[\\"dev\\"] else \\"OK\\"\\n    except Exception as e:\\n        EXT_RECORDS, EXT_SOURCE, EXT_STATUS = [], None, f\\"NOT RUN — không nạp được MSR-VTT: {type(e).__name__}: {e}\\"\\n        print(\\"⚠\\", EXT_STATUS)\\nRECORDS = RECORDS + EXT_RECORDS\\nSPLIT_OF = {r[\\"vid\\"]: r[\\"split\\"] for r in RECORDS}\\nCAPS = CaptionStore(_caps, SPLIT_OF)\\ndel _caps                                                  # caption chỉ còn trong CAPS\\nCAP_VID, CAP_SPLIT = CAPS.vid_of_cap, CAPS.split_of_cap\\nCAP_STD = np.zeros(len(CAP_VID), bool)                     # caption chuẩn 1k-A (truy vấn PHỤ của tập ngoài)\\n_first_cap = {v: i for i, v in reversed(list(enumerate(CAP_VID.tolist())))}\\nfor r in EXT_RECORDS:\\n    CAP_STD[_first_cap[r[\\"vid\\"]] + r[\\"std_caption\\"]] = True\\n\\nrep = DATA_REPORT[\\"splits\\"]\\nprint(f\\"Nguồn video: {DATA_REPORT[\'source\']} ({DATA_REPORT[\'source_label\']}) | {DATA_REPORT[\'n_video_files\']} file video\\")\\ndisplay(pd.DataFrame(rep).T)\\nif CFG.dataset == \\"msvd\\":\\n    got = {s: (rep[s][\\"videos\\"], rep[s][\\"captions\\"]) for s in rep}\\n    check(\\"A1\\", \\"số video / caption theo split khớp dataset card MSVD\\", got == {s: tuple(v) for s, v in CFG.expected.items()},\\n          f\\"thực tế {got} | kỳ vọng {CFG.expected}\\")\\n    check(\\"A1b\\", \\"cột \'video\' của HF chỉ là tên file .avi (video thật lấy từ zip/thư mục)\\",\\n          all(rep[s][\\"video_field_is_filename\\"] for s in rep), DATA_REPORT[\\"source_label\\"])\\nelse:\\n    check(\\"A1\\", \\"số video / caption khớp MSVD\\", False, \\"dữ liệu SYNTHETIC — không áp dụng\\", hard=False, status=\\"NOT RUN\\")\\ncheck(\\"A2\\", \\"mỗi video_id có đúng một file video (không thiếu file)\\",\\n      all(rep[s][\\"missing_files\\"] == 0 for s in rep), {s: rep[s][\\"missing_files\\"] for s in rep})\\nprint(f\\"Tập ngoài: {EXT_STATUS} | {EXT_REPORT.get(\'label\', \'—\')} | {EXT_REPORT.get(\'videos\', 0)} video / \\"\\n      f\\"{EXT_REPORT.get(\'captions\', 0)} caption\\")\\nif EXT_RECORDS:\\n    if EXT_REPORT[\\"dev\\"]:\\n        ok_ext = EXT_REPORT[\\"overlap_with_test_1kA\\"] == 0\\n    elif EXT_REPORT.get(\\"set\\") in (\\"fresh1k\\", \\"fresh1kB\\", \\"fresh1kC\\"):\\n        ok_ext = (EXT_REPORT[\\"videos\\"] == CFG.fresh_n and EXT_REPORT[\\"overlap_with_test_1kA\\"] == 0\\n                  and EXT_REPORT[\\"overlap_with_dev\\"] == 0\\n                  and (EXT_REPORT[\\"set\\"] == \\"fresh1k\\" or EXT_REPORT[\\"overlap_with_fresh1k\\"] == 0)\\n                  and (EXT_REPORT[\\"set\\"] != \\"fresh1kC\\" or EXT_REPORT[\\"overlap_with_fresh1kB\\"] == 0))\\n    else:\\n        ok_ext = EXT_REPORT[\\"videos\\"] == 1000 and EXT_REPORT[\\"overlap_with_test_1kA\\"] == 1000\\n    check(\\"A6\\", \\"tập ngoài: đủ file video; fresh1k / fresh1k-B = đúng CFG.fresh_n video, rời 1k-A, rời 200 video dev (fresh1k-B rời cả fresh1k); \\"\\n                \\"1kA = đúng 1000 video test 1k-A; dev = rời hẳn 1k-A\\",\\n          EXT_REPORT[\\"missing_files\\"] == 0 and ok_ext,\\n          {k: EXT_REPORT.get(k) for k in (\\"label\\", \\"set\\", \\"videos\\", \\"captions\\", \\"captions_per_video\\", \\"std_captions_added\\",\\n                                          \\"missing_files\\", \\"overlap_with_test_1kA\\", \\"overlap_with_dev\\", \\"overlap_with_fresh1k\\", \\"overlap_with_fresh1kB\\")})\\nelse:\\n    check(\\"A6\\", \\"tập ngoài (benchmark xác nhận độc lập)\\", False, EXT_STATUS, hard=False, status=\\"NOT RUN\\")\\n\\nprobes = []\\nfor s in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\"):\\n    src = EXT_SOURCE if s == \\"ext\\" else SOURCE\\n    for r in [r for r in RECORDS if r[\\"split\\"] == s][:3]:\\n        with src.open(r[\\"video_id\\"]) as p:\\n            probes.append(dict(split=s, video_id=r[\\"video_id\\"], **probe_video(p)))\\ndisplay(pd.DataFrame(probes))\\ncheck(\\"A3\\", \\"giải mã được video mẫu của TỪNG split (3 video/split, đếm frame bằng grab)\\",\\n      all(p[\\"ok\\"] for p in probes),\\n      f\\"{sum(p[\'ok\'] for p in probes)}/{len(probes)} video | codec {sorted({str(p.get(\'codec\')) for p in probes})} | \\"\\n      f\\"frame {[p.get(\'frames_decoded\') for p in probes]}\\")\\nDATA_REPORT[\\"probes\\"], DATA_REPORT[\\"ext\\"] = probes, dict(EXT_REPORT, status=EXT_STATUS)\\nsave_json(DATA_REPORT, RESULTS / \\"gateA_dataset_report.json\\")\\nprovenance(\\"RECORDS + CaptionStore\\", \\"external\\", f\\"{CFG.hf_repo}@{CFG.hf_revision[:8]}\\" if CFG.dataset == \\"msvd\\"\\n           else \\"synthetic\\", note=\\"metadata + caption gốc; caption chỉ đọc qua CaptionStore\\")\\nif EXT_RECORDS:\\n    provenance(\\"tập ngoài (MSR-VTT)\\", \\"external\\", EXT_REPORT[\\"label\\"], note=\\"chỉ frame (encode) trước đóng băng; caption chỉ để đánh giá\\")"}, {"cell_type": "code", "execution_count": null, "id": "c047", "metadata": {}, "outputs": [], "source": "mark_stage(\\"split\\", requires=(\\"data\\",))\\nSPLIT = make_split(RECORDS)\\ncheck(\\"A5\\", \\"split THEO VIDEO, không video nào ở hai split; manifest đã lưu\\",\\n      sum(len(SPLIT[k]) for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\")) == len(RECORDS), f\\"sha1={SPLIT[\'sha1\']} → split_manifest.json\\")"}, {"cell_type": "code", "execution_count": null, "id": "c048", "metadata": {}, "outputs": [], "source": "# ============================== CỔNG B (1/2): frame + đặc trưng ==============================\\nmark_stage(\\"features\\", requires=(\\"split\\",))\\nFRAME_FEAT, FRAME_META, TXT_FEAT, FEAT_CACHES = all_features()\\nn_bad = len(FRAME_META[\\"bad\\"])\\ncheck(\\"A4\\", \\"TOÀN BỘ video giải mã được và lấy đủ F_max frame\\", n_bad == 0,\\n      f\\"{len(RECORDS) - n_bad}/{len(RECORDS)} video đọc được | nguồn {FRAME_META[\'source\']} | {FRAME_META[\'seconds\']}s\\")\\nsave_json(dict(policy=CFG.frame_policy, n_frames_max=CFG.n_frames_max,\\n               formula=\\"idx_k = floor((k + 0.5) / F_max · n_decoded), t_k = idx_k / fps\\",\\n               per_video=FRAME_META[\\"per_video\\"], bad=FRAME_META[\\"bad\\"]), RESULTS / \\"frame_sampling.json\\")\\nprovenance(\\"FRAME_FEAT [N,12,D]\\", \\"frames\\", f\\"CLIP-image đóng băng {CFG.clip_model}/{CFG.clip_weights} trên 12 frame/video\\",\\n           video_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]), note=\\" | \\".join(map(str, FEAT_CACHES)))\\nprovenance(\\"TXT_FEAT [N_caption,D]\\", \\"encoded\\", \\"CLIP-text đóng băng — mã hoá để cache, không học\\",\\n           caption_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]), note=\\"việc DÙNG tuân theo CaptionStore\\")\\ndur = np.array([m[\\"duration_s\\"] for m in FRAME_META[\\"per_video\\"].values() if m])\\nprint(f\\"Thời lượng video: min {dur.min():.1f}s | trung vị {np.median(dur):.1f}s | max {dur.max():.1f}s\\")"}, {"cell_type": "code", "execution_count": null, "id": "c049", "metadata": {}, "outputs": [], "source": "# ============================== CỔNG B (2/2): gộp theo thời gian + kiểm tra ==============================\\nmark_stage(\\"pooling\\", requires=(\\"features\\",))\\nPOOL_SEL, POOL_TABLE = select_pooling_on_val(FRAME_FEAT, TXT_FEAT, SPLIT)\\nVID_FEAT = pool_frames(FRAME_FEAT, POOL_SEL[\\"n_frames\\"], POOL_SEL[\\"pooling\\"])\\nPOOL_TABLE.to_csv(RESULTS / \\"pooling_selection_val.csv\\", index=False)\\ndisplay(POOL_TABLE.round(4))\\nprint(f\\"→ chọn trên VAL: {POOL_SEL}\\")\\nprovenance(\\"n_frames + pooling\\", \\"selected\\", \\"MRR CLIP-only text→video trên VAL\\", caption_splits=(\\"val\\",))\\nprovenance(\\"VID_FEAT [N,D]\\", \\"frames\\", f\\"gộp {POOL_SEL[\'n_frames\']} frame ({POOL_SEL[\'pooling\']})\\",\\n           video_splits=tuple(k for k in (\\"train\\", \\"val\\", \\"test\\", \\"ext\\") if SPLIT[k]))\\ngate_b_checks(FRAME_FEAT, FRAME_META, VID_FEAT, TXT_FEAT, SPLIT)\\nsave_json([c for c in CHECKS if c[\\"id\\"].startswith(\\"B\\")], RESULTS / \\"gateB_checks.json\\")"}, {"cell_type": "code", "execution_count": null, "id": "c050", "metadata": {}, "outputs": [], "source": "mark_stage(\\"vocab_tagger\\", requires=(\\"pooling\\",))\\nt0 = time.time()\\nVOCAB2 = build_vocab_v2(CAPS, SPLIT)\\nRELATIONS2 = build_relations_v2(CAPS, SPLIT, VOCAB2)\\nTAGS2, TAG_SCORES, TAG_SELECT2, TAG_GRID2 = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL[\\"n_frames\\"], VOCAB2, SPLIT, CAPS)\\nfor r in RECORDS:\\n    r[\\"tags\\"] = TAGS2[r[\\"vid\\"]]\\nTAXONOMY, CONCEPT_META, TAX_AUDIT, TAX_STATS = build_taxonomy_v2(VOCAB2, CAPS, SPLIT, {v: TAGS2[v] for v in SPLIT[\\"train\\"]})\\nINV = NodeInventory(RECORDS, VOCAB2, CONCEPT_META)\\nCONF_CAL = contains_calibration(TAG_SCORES, SPLIT[\\"train\\"], INV)\\n_qtexts = CAPS.texts(np.arange(len(CAP_VID)), \\"query_input\\")\\nQTOK2 = [query_tokens_v2(t) for t in _qtexts]\\nSYM_FEAT = symbolic_features(INV, VOCAB2)\\nTRAIN_Q = np.where(CAP_SPLIT == \\"train\\")[0]\\n\\n# ---- vị từ (v3): từ vựng + support chỉ từ quan hệ caption TRAIN; embedding = CLIP-text đóng băng theo prompt ----\\nPRED_SUPPORT = Counter()\\nfor v in SPLIT[\\"train\\"]:\\n    PRED_SUPPORT.update({(rel, p) for _, rel, _, p in RELATIONS2[v]})\\nPRED_VOCAB = sorted(k for k in PRED_SUPPORT if k[0] in PRED_FAMILIES)\\nPRED_ID = {k: i for i, k in enumerate(PRED_VOCAB)}\\nPRED_EMB = predicate_embeddings(PRED_VOCAB, VOCAB2)\\nQPRED_TOK = [query_pred_tokens(t) for t in _qtexts]\\nQPREDS = [{rel: match_predicates(qs, rel) for rel in PRED_FAMILIES} for qs in QPRED_TOK]\\ndel _qtexts\\nn_v = sum(t == \\"verb\\" for t in VOCAB2[\\"types\\"].values())\\nprint(f\\"Từ vựng: {len(VOCAB2[\'terms\'])} khoá = {len(VOCAB2[\'terms\']) - n_v} danh từ + {n_v} động từ; \\"\\n      f\\"{len(VOCAB2[\'lemmas_with_two_pos\'])} lemma có CẢ hai POS\\")\\nprint(f\\"Tagger (chọn trên VAL): {TAG_SELECT2}\\")\\nprint(f\\"Taxonomy: {TAX_STATS[\'n_concepts\']} khái niệm | isa theo biến thể {TAX_STATS[\'isa_edges\']} | mơ hồ {TAX_STATS[\'n_ambiguous\']} | \\"\\n      f\\"sense khác sense đầu {TAX_STATS[\'n_sense_differs\']} | hub {TAX_STATS[\'n_hub\']} | chu trình {TAX_STATS[\'cycles\']}\\")\\nprint(f\\"Vị từ: {len(PRED_VOCAB)} (họ có vị từ tự do {PRED_FAMILIES}; \\"\\n      f\\"{ {f: sum(r == f for r, _ in PRED_VOCAB) for f in PRED_FAMILIES} }); ≥ {CFG.pred_min_support} video TRAIN: \\"\\n      f\\"{ {f: sum(r == f and PRED_SUPPORT[(r, p)] >= CFG.pred_min_support for r, p in PRED_VOCAB) for f in PRED_FAMILIES} } | \\"\\n      f\\"truy vấn khớp ≥ 1 vị từ: {np.mean([any(q.values()) for q in QPREDS]):.1%}\\")\\nprint(f\\"Inventory: {INV.n_nodes} node = {INV.n_vid} video ({N_MSVD} MSVD + {INV.n_vid - N_MSVD} tập ngoài) + {INV.n_cls} lớp + \\"\\n      f\\"{INV.n_con} khái niệm (sha1 {INV.sha1}) | {time.time() - t0:.0f}s\\")\\n\\n# ---- Nhiệm vụ B: kiểm chứng TỰ ĐỘNG (parser độc lập, chỉ TRAIN) + mẫu phân tầng; nhãn người chấm chỉ là TUỲ CHỌN ----\\nt1 = time.time()\\nAGREE, AGREE_COVER = build_agreement(CAPS, SPLIT, VOCAB2)\\nANN_SAMPLE, ANN_CELLS, TRIPLE_SUP, _ = build_annotation_sample(CAPS, SPLIT, VOCAB2, RECORDS)\\nif AGREE is not None:\\n    ANN_SAMPLE.insert(7, \\"auto_parser_agrees\\", [int(triple_confirmed((r.src_node, r.family, r.dst_node, r.predicate),\\n                                                                       _DEP_CACHE[r.caption.lower()])) for r in ANN_SAMPLE.itertuples()])\\n    REL_QUALITY_AUTO = agreement_quality(AGREE, AGREE_COVER, TRIPLE_SUP)\\n    _fam = REL_QUALITY_AUTO[(REL_QUALITY_AUTO.stratum == \\"ALL\\") & (REL_QUALITY_AUTO.family != \\"ALL\\")]\\n    AUTO_STATUS = (f\\"kiểm chứng TỰ ĐỘNG ({DEP_STATUS}): tỉ lệ quan hệ heuristic được parser độc lập xác nhận — \\"\\n                   + \\"; \\".join(f\\"{r.family} {r.agreement:.2f} [{r.wilson_lo:.2f}, {r.wilson_hi:.2f}]\\" for r in _fam.itertuples())\\n                   + f\\" ({time.time() - t1:.0f}s). SILVER — không phải precision do người chấm.\\")\\nelse:\\n    REL_QUALITY_AUTO = pd.DataFrame([dict(kind=\\"silver_parser_agreement\\", family=\\"ALL\\", stratum=\\"ALL\\", status=\\"NOT RUN\\", note=DEP_STATUS)])\\n    AUTO_STATUS = DEP_STATUS\\nprint(\\"Kiểm chứng tự động:\\", AUTO_STATUS)\\nANN_SAMPLE.to_csv(RESULTS / \\"relation_annotation_sample.csv\\", index=False, encoding=\\"utf-8-sig\\")\\nANN_CELLS.to_csv(RESULTS / \\"relation_annotation_cells.csv\\", index=False)\\nLABEL_FILE, LABELS = find_label_file()           # tuỳ chọn: chỉ dùng nếu có file nhãn trong /kaggle/input hoặc MSVD_G3_LABELS\\nif LABELS is not None:\\n    REL_QUALITY, KAPPA, LABELED = relation_quality(LABELS)\\n    VERIFIED_CELLS = {(r.family, r.stratum) for r in REL_QUALITY.itertuples() if r.verified}\\n    match = float(LABELS[\\"sample_id\\"].isin(set(ANN_SAMPLE[\\"sample_id\\"])).mean()) if \\"sample_id\\" in LABELS else float(\\"nan\\")\\n    REL_QUALITY_STATUS = (f\\"nhãn người chấm: {LABEL_FILE} | {int(REL_QUALITY.iloc[0].n_labeled)} dòng có nhãn | \\"\\n                          f\\"kappa {KAPPA[\'kappa\']:.3f} trên {KAPPA[\'n_double\']} dòng chấm đôi | ô đạt ngưỡng: {sorted(VERIFIED_CELLS)} | \\"\\n                          f\\"sample_id khớp mẫu tái tạo: {match:.1%}\\")\\n    LABELED.to_csv(RESULTS / \\"relation_annotation_labeled.csv\\", index=False)\\nelse:\\n    KAPPA, REL_QUALITY = dict(n_double=0, kappa=float(\\"nan\\"), raw_agreement=float(\\"nan\\")), None\\n    REL_QUALITY_STATUS = \\"không có nhãn người chấm (tuỳ chọn) — chỉ có kiểm chứng tự động\\"\\npd.concat([REL_QUALITY_AUTO] + ([REL_QUALITY.assign(kind=\\"human_precision\\")] if REL_QUALITY is not None else []),\\n          ignore_index=True).to_csv(RESULTS / \\"relation_quality_audit.csv\\", index=False)\\nprint(\\"Nhãn người chấm:\\", REL_QUALITY_STATUS)\\n\\npd.DataFrame(TAG_GRID2).to_csv(RESULTS / \\"tagger_selection_val_v2.csv\\", index=False)\\nsave_json(dict(VOCAB2, relations_example={RECORDS[v][\\"video_id\\"]: RELATIONS2[v][:8] for v in SPLIT[\\"train\\"][:5]}),\\n          RESULTS / \\"vocab_v2.json\\")\\nsave_json(dict(selection=TAG_SELECT2, calibration=CONF_CAL), RESULTS / \\"tagger_info_v2.json\\")\\nTAX_AUDIT.to_csv(RESULTS / \\"graph_taxonomy_audit.csv\\", index=False)\\nsave_json(TAX_STATS, RESULTS / \\"graph_taxonomy_stats.json\\")\\nprovenance(\\"từ vựng (lemma, POS)\\", \\"learned\\", \\"caption TRAIN (POS + WordNet)\\", caption_splits=(\\"train\\",))\\nprovenance(\\"quan hệ (loại + vị từ)\\", \\"learned\\", \\"caption TRAIN\\", caption_splits=(\\"train\\",))\\nprovenance(\\"từ vựng + support vị từ\\", \\"learned\\", \\"quan hệ caption TRAIN\\", caption_splits=(\\"train\\",))\\nprovenance(\\"embedding vị từ (CLIP-text đóng băng)\\", \\"external\\", \\"prompt CFG.pred_prompts × vị từ train\\")\\nprovenance(\\"mẫu quan hệ phân tầng\\", \\"learned\\", \\"caption TRAIN (phân tầng họ × mức phổ biến)\\", caption_splits=(\\"train\\",))\\nif AGREE is not None:\\n    provenance(\\"đồng thuận heuristic ↔ spaCy (kiểm chứng tự động)\\", \\"learned\\", f\\"caption TRAIN; {DEP_STATUS}\\", caption_splits=(\\"train\\",))\\nif LABELS is not None:\\n    provenance(\\"precision quan hệ + ô đã kiểm chứng\\", \\"external\\", f\\"nhãn NGƯỜI chấm trên caption TRAIN ({LABEL_FILE.name})\\")\\nprovenance(\\"tham số tagger\\", \\"selected\\", \\"F1 trên caption VAL\\", caption_splits=(\\"val\\",))\\nprovenance(\\"thống kê hiệu chỉnh contains (μ, σ, phân vị)\\", \\"frames\\", \\"điểm tagger trên video TRAIN\\", video_splits=(\\"train\\",))\\nprovenance(\\"taxonomy: sense theo ngữ cảnh + hub\\", \\"learned\\", \\"caption TRAIN + nhãn tagger TRAIN + WordNet\\",\\n           caption_splits=(\\"train\\",), video_splits=(\\"train\\",))\\nprovenance(\\"đặc trưng ngữ nghĩa node (CLIP-text)\\", \\"external\\", \\"prompt tagger / \'lemma: định nghĩa WordNet\'\\")\\n# ---- v4: node khớp + VAI của từng node trong CHÍNH truy vấn (đọc văn bản truy vấn — mục đích query_input, như QTOK2) ----\\nV4_HITS = [sorted({nid for nid, _, _ in parse_tokens_v2(t, INV)}) for t in QTOK2]\\nV4_ROLES = query_roles_v4(CAPS.texts(np.arange(len(CAP_VID)), \\"query_input\\"), INV)\\nprint(f\\"v4: truy vấn có ≥ 1 node khớp {np.mean([bool(h) for h in V4_HITS]):.3f}; có vai chủ thể/đối tượng \\"\\n      f\\"{np.mean([any(r in (1, 3) for r in d.values()) for d in V4_ROLES]):.3f}\\")\\n"}, {"cell_type": "code", "execution_count": null, "id": "c051", "metadata": {}, "outputs": [], "source": "# ============================== ĐỒ THỊ v3: schema · node · cạnh có vị từ · audit cấu trúc · audit vị từ ==============================\\nmark_stage(\\"graph_v3\\", requires=(\\"vocab_tagger\\",))\\nG_BASE, E_BASE = graph_for(DESIGN_BASE, CFG.seeds[0])\\nprint(\\"Đồ thị v3 (thiết kế gốc = v2_base):\\", json.dumps(G_BASE.summary(), ensure_ascii=False))\\nSCHEMA = dict(version=\\"v3\\", node_types=dict(video=\\"video MSVD / tập ngoài (embedding CLIP gộp frame, đóng băng)\\",\\n                                            **{\\"class\\": \\"lớp (lemma, POS) từ caption TRAIN\\", \\"concept\\": \\"synset WordNet\\"}),\\n              semantic_types=[\\"video\\", \\"noun\\", \\"verb\\", \\"noun_concept\\", \\"verb_concept\\"], relations=REL_SPEC,\\n              edge_record=[\\"src\\", \\"relation_family\\", \\"relation\\", \\"predicate\\", \\"dst\\", \\"direction\\", \\"weight\\", \\"confidence\\",\\n                           \\"source\\", \\"status\\"],\\n              predicate_families=list(PRED_FAMILIES), predicate_vocab_size=len(PRED_VOCAB),\\n              predicate_prompts=CFG.pred_prompts, pred_modes=list(PRED_MODES), rel_policies=list(REL_POLICIES),\\n              patterns=dict(zip(PATTERNS, PATTERN_LEN)), masks={k: list(v) for k, v in MASKS.items()},\\n              inventory=dict(n_nodes=INV.n_nodes, videos=INV.n_vid, msvd_videos=N_MSVD, ext_videos=INV.n_vid - N_MSVD,\\n                             classes=INV.n_cls, concepts=INV.n_con, sha1=INV.sha1),\\n              path_modes=[\\"weighted\\", \\"count\\", \\"degnorm\\", \\"dedup\\", \\"pred_match\\"],\\n              mp_weight_modes=[\\"binary\\", \\"weighted_mean\\", \\"weighted_symnorm\\"], init_modes=[\\"learned\\", \\"clip_frozen\\", \\"clip_residual\\"],\\n              related_cfgs=RELATED_CFGS, protocols=list(PROTOCOLS))\\nSCHEMA[\\"sha1\\"] = sha1_of(SCHEMA)\\nsave_json(SCHEMA, RESULTS / \\"graph_schema_v3.json\\")\\n\\ndeg_out = {t: np.asarray((G_BASE.A[t] > 0).sum(1)).ravel() for t in RELATION_TYPES}\\ndeg_in = {t: np.asarray((G_BASE.A[t] > 0).sum(0)).ravel() for t in RELATION_TYPES}\\ncon_in = np.asarray((G_BASE.A_con > 0).sum(0)).ravel()\\nnode_rows = []\\nfor i in range(INV.n_nodes):\\n    t = INV.node_type[i]\\n    node_rows.append(dict(node_id=i, node_type=t, canonical_name=INV.name[i], semantic_type=INV.semantic_type(i),\\n                          lemma=INV.lemma.get(i, \\"\\"), split_provenance=SPLIT_OF.get(i, \\"\\") if t == \\"video\\" else\\n                          (\\"train\\" if t == \\"class\\" else \\"external\\"),\\n                          synset=INV.concept_meta.get(INV.name[i], {}).get(\\"synset\\", \\"\\"),\\n                          train_df=int(G_BASE.cls_df[i]) if t == \\"class\\" else \\"\\", active=i in G_BASE.active,\\n                          videos_tagged=int(con_in[i]), rel_out_degree=int(sum(deg_out[x][i] for x in RELATION_TYPES)),\\n                          rel_in_degree=int(sum(deg_in[x][i] for x in RELATION_TYPES))))\\nNODES_DF = pd.DataFrame(node_rows)\\nNODES_DF.to_csv(RESULTS / \\"graph_nodes_v3.csv\\", index=False)\\n\\n\\ndef _confidence(rel, c, s=None, o=None, p=None):\\n    \\"\\"\\"Nhãn người chấm (nếu có) của ô (họ × mức phổ biến); ngược lại tỉ lệ đồng thuận SILVER với parser độc lập của đúng\\n    quan hệ đó; không có cả hai ⇒ \'unverified\'.\\"\\"\\"\\n    if rel not in REL_CATS:\\n        return \\"\\"\\n    if REL_QUALITY is not None:\\n        r = REL_QUALITY[(REL_QUALITY.family == rel) & (REL_QUALITY.stratum == support_stratum(c))]\\n        if len(r) and r.iloc[0].n_labeled:\\n            return f\\"human {r.iloc[0].precision:.3f} [{r.iloc[0].wilson_lo:.3f}, {r.iloc[0].wilson_hi:.3f}]\\"\\n    a = AGREE.get((rel, s, o, p)) if AGREE and s is not None else None\\n    return f\\"silver {a[0] / a[1]:.2f} ({a[0]}/{a[1]})\\" if a and a[1] else \\"unverified\\"\\n\\n\\ndef edge_table(g):\\n    \\"\\"\\"Một dòng = MỘT quan hệ (nguồn, họ, vị từ, đích, chiều): cạnh quan hệ tách theo từng vị từ của cặp; cạnh bị loại\\n    (chính sách / top-k) vẫn được ghi với status = excluded:<lý do>.\\"\\"\\"\\n    rows = []\\n    c = g.A_con.tocoo()\\n    for u, v, w in zip(c.row.tolist(), c.col.tolist(), c.data.tolist()):\\n        for rel, a, b, dr in ((\\"contains\\", u, v, \\"forward\\"), (\\"contains_inv\\", v, u, \\"inverse\\")):\\n            rows.append(dict(src=INV.name[a], relation_family=\\"contains\\", relation=rel, predicate=\\"\\", dst=INV.name[b],\\n                             direction=dr, weight=w, confidence=\\"tagger\\" if g.contains_mode == \\"conf\\" else \\"\\",\\n                             source=REL_SPEC[rel][\\"source\\"][:60], status=\\"kept\\", pair_support=\\"\\", predicate_videos=\\"\\",\\n                             predicate_share=\\"\\"))\\n    for t in RELATION_TYPES:\\n        m = g.A[t].tocoo()\\n        base = t[:-4] if t.endswith(\\"_inv\\") else t\\n        dr = \\"symmetric\\" if t == \\"related\\" else (\\"inverse\\" if t.endswith(\\"_inv\\") else \\"forward\\")\\n        for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):\\n            d = g.predicates_of(t, u, v) if base in REL_CATS else {}\\n            tot = float(sum(d.values())) or 1.0\\n            ka, kb = (INV.name[v], INV.name[u]) if t.endswith(\\"_inv\\") else (INV.name[u], INV.name[v])\\n            for p, cnt in (sorted(d.items(), key=lambda x: (-x[1], x[0])) or [(\\"\\", \\"\\")]):\\n                rows.append(dict(src=INV.name[u], relation_family=REL_SPEC[t][\\"family\\"], relation=t, predicate=p,\\n                                 dst=INV.name[v], direction=dr, weight=w, confidence=_confidence(base, cnt, ka, kb, p) if p else \\"\\",\\n                                 source=REL_SPEC[t][\\"source\\"][:60], status=\\"kept\\",\\n                                 pair_support=int(sum(d.values())) if d else \\"\\", predicate_videos=cnt,\\n                                 predicate_share=round(cnt / tot, 4) if p else \\"\\"))\\n    for x in g.excluded:\\n        for p, cnt in sorted(x[\\"predicates\\"].items(), key=lambda y: (-y[1], y[0])):\\n            rows.append(dict(src=INV.name[x[\\"src\\"]], relation_family=REL_SPEC[x[\\"family\\"]][\\"family\\"], relation=x[\\"family\\"],\\n                             predicate=p, dst=INV.name[x[\\"dst\\"]], direction=\\"forward\\", weight=\\"\\",\\n                             confidence=_confidence(x[\\"family\\"], cnt, INV.name[x[\\"src\\"]], INV.name[x[\\"dst\\"]], p),\\n                             source=REL_SPEC[x[\\"family\\"]][\\"source\\"][:60],\\n                             status=f\\"excluded:{x[\'reason\']}\\", pair_support=x[\\"support\\"], predicate_videos=cnt, predicate_share=\\"\\"))\\n    return pd.DataFrame(rows)\\n\\n\\nEDGES_DF = edge_table(G_BASE)\\nEDGES_DF.to_csv(RESULTS / \\"graph_edges_v3.csv\\", index=False)\\n\\nstruct = []\\nfor t in [\\"contains\\"] + RELATION_TYPES:\\n    M = G_BASE.A_con if t == \\"contains\\" else G_BASE.A[t]\\n    m = M.tocoo()\\n    do, di = np.asarray((M > 0).sum(1)).ravel(), np.asarray((M > 0).sum(0)).ravel()\\n    srcs, tgts = np.unique(m.row), np.unique(m.col)\\n    base = t[:-4] if t.endswith(\\"_inv\\") else t\\n    npred = [len(G_BASE.predicates_of(t, u, v)) for u, v in zip(m.row.tolist(), m.col.tolist())] if base in REL_CATS else []\\n    struct.append(dict(section=\\"v3_relation\\", relation=t, family=REL_SPEC[t][\\"family\\"], usage=REL_SPEC[t][\\"usage\\"], n_edges=int(M.nnz),\\n                       src_types=dict(Counter(INV.semantic_type(i) for i in srcs)) if t != \\"contains\\" else {\\"video\\": len(srcs)},\\n                       tgt_types=dict(Counter(INV.semantic_type(i) for i in tgts)),\\n                       w_mean=float(m.data.mean()) if m.nnz else None, w_nonbinary_frac=float((m.data < 1 - 1e-9).mean()) if m.nnz else 0,\\n                       out_deg_mean=float(do[do > 0].mean()) if (do > 0).any() else 0, out_deg_max=int(do.max()) if do.size else 0,\\n                       in_deg_mean=float(di[di > 0].mean()) if (di > 0).any() else 0, in_deg_max=int(di.max()) if di.size else 0,\\n                       predicates_per_edge_mean=float(np.mean(npred)) if npred else None,\\n                       frac_edges_multi_predicate=float(np.mean([k > 1 for k in npred])) if npred else None,\\n                       predicate_in_message=t in PRED_TYPES, self_loops=int((m.row == m.col).sum()),\\n                       duplicates=int(m.nnz - len(set(zip(m.row, m.col))))))\\nact = np.array(sorted(G_BASE.active))\\nrel_deg = sum(deg_out[t] + deg_in[t] for t in RELATION_TYPES)\\nvid_no_tag = int(sum(len(x) == 0 for x in G_BASE.vid_tags))\\nstruct.append(dict(section=\\"v3_nodes\\", relation=\\"—\\", note=(\\n    f\\"lớp active {len(act)}/{INV.n_cls}; lớp active KHÔNG có cạnh quan hệ: {int((rel_deg[act] == 0).sum())}; video không có contains: \\"\\n    f\\"{vid_no_tag}; khái niệm không có isa vào: {int((deg_in[\'isa\'][INV.n_vid + INV.n_cls:] == 0).sum())}/{INV.n_con}; \\"\\n    f\\"cặp quan hệ bị loại (chính sách/top-k): {len(G_BASE.excluded)}\\")))\\nSTRUCT_DF = pd.DataFrame(struct)\\nSTRUCT_DF.to_csv(RESULTS / \\"graph_structure_audit.csv\\", index=False)\\ndisplay(STRUCT_DF[STRUCT_DF.section == \\"v3_relation\\"][[\\"relation\\", \\"n_edges\\", \\"w_mean\\", \\"predicates_per_edge_mean\\",\\n                                                       \\"frac_edges_multi_predicate\\", \\"predicate_in_message\\", \\"self_loops\\", \\"duplicates\\"]])\\n\\n# ---- audit embedding vị từ: vị từ nào dùng trong đồ thị gốc, support, láng giềng gần nhất (kiểm tra ngữ nghĩa) ----\\n_used = Counter()\\nfor rel in PRED_FAMILIES:\\n    for d in G_BASE.pshare[rel].values():\\n        _used.update({PRED_VOCAB[pid] for pid in d})\\n_E = PRED_EMB / np.linalg.norm(PRED_EMB, axis=1, keepdims=True).clip(1e-8)\\npe_rows = []\\nfor rel in PRED_FAMILIES:\\n    idx = np.array([i for i, (r, _) in enumerate(PRED_VOCAB) if r == rel])\\n    if not len(idx):\\n        continue\\n    used_idx = np.array([i for i in idx if _used[PRED_VOCAB[i]] > 0])\\n    S = _E[idx] @ _E[idx].T\\n    off = S[~np.eye(len(idx), dtype=bool)] if len(idx) > 1 else np.array([1.0])\\n    for i in (used_idx if len(used_idx) else idx[:0]):\\n        s = _E[idx] @ _E[i]\\n        nn_ = [PRED_VOCAB[idx[j]][1] for j in np.argsort(-s) if idx[j] != i][:3]\\n        pe_rows.append(dict(family=rel, predicate=PRED_VOCAB[i][1], phrase=predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2),\\n                            prompt=CFG.pred_prompts[rel][0].format(predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2)),\\n                            train_videos=int(PRED_SUPPORT[PRED_VOCAB[i]]), graph_edges=int(_used[PRED_VOCAB[i]]),\\n                            passes_support_policy=PRED_SUPPORT[PRED_VOCAB[i]] >= CFG.pred_min_support,\\n                            nearest=\\", \\".join(nn_), family_mean_cos=float(off.mean()), family_min_cos=float(off.min()),\\n                            family_max_cos=float(off.max())))\\nPRED_AUDIT = pd.DataFrame(pe_rows)\\nPRED_AUDIT.to_csv(RESULTS / \\"predicate_embedding_audit.csv\\", index=False)\\nif len(PRED_AUDIT):\\n    print(\\"Embedding vị từ:\\", PRED_AUDIT.groupby(\\"family\\").agg(n=(\\"predicate\\", \\"size\\"), mean_cos=(\\"family_mean_cos\\", \\"first\\"),\\n                                                              min_cos=(\\"family_min_cos\\", \\"first\\")).to_dict(\\"index\\"))\\n\\n# ---- thống kê quan hệ caption TRAIN ----\\ntq = np.where(CAP_SPLIT == \\"train\\")[0]\\ntr_texts = CAPS.texts(tq, \\"relations\\")\\nall_trip, oov = Counter(), 0\\nvset = set(VOCAB2[\\"terms\\"])\\nfor txt in tr_texts:\\n    for s, rel, o, pred in parse_caption_v2(txt)[2]:\\n        all_trip[rel] += 1\\n        oov += int(s not in vset or o not in vset)\\nREL_STATS = dict(train_captions=len(tq), triples_extracted=int(sum(all_trip.values())), by_category=dict(all_trip),\\n                 frac_dropped_oov=oov / max(sum(all_trip.values()), 1),\\n                 kept_relation_pairs=sum(int(G_BASE.A[rel].nnz) for rel in REL_CATS),\\n                 kept_relation_predicate_rows=int(((EDGES_DF.status == \\"kept\\") & EDGES_DF.relation.isin(REL_CATS)).sum()),\\n                 excluded_pairs=len(G_BASE.excluded), annotation_sample_rows=len(ANN_SAMPLE),\\n                 automatic_verification=AUTO_STATUS, human_precision=REL_QUALITY_STATUS,\\n                 heuristic_covers_spacy={k: round(AGREE_COVER.get((k, \\"found\\"), 0) / max(AGREE_COVER.get((k, \\"total\\"), 0), 1), 3)\\n                                         for k in REL_CATS} if AGREE_COVER else \\"NOT RUN\\",\\n                 recall_f1=\\"NOT RUN — không có tập quan hệ đúng đầy đủ; \'heuristic_covers_spacy\' chỉ là độ phủ tương đối so với spaCy\\")\\nsave_json(REL_STATS, RESULTS / \\"graph_relation_quality_stats.json\\")\\nprint(\\"Quan hệ caption TRAIN:\\", json.dumps(REL_STATS, ensure_ascii=False))\\nprovenance(\\"cạnh quan hệ v3 (agent/object/svo/prep + _inv, phân phối vị từ)\\", \\"learned\\", \\"caption TRAIN\\", caption_splits=(\\"train\\",))\\nprovenance(\\"cạnh related (NPMI)\\", \\"learned\\", \\"nhãn tagger của video TRAIN\\", video_splits=(\\"train\\",))\\nprovenance(\\"IDF\\", \\"learned\\", \\"contains + isa thật của video TRAIN\\", video_splits=(\\"train\\",))\\nprovenance(\\"cạnh isa\\", \\"learned\\", \\"WordNet + chọn sense từ ngữ cảnh caption TRAIN\\", caption_splits=(\\"train\\",))"}, {"cell_type": "code", "execution_count": null, "id": "c052", "metadata": {}, "outputs": [], "source": "# ============================== SELF-TEST v3 (mục 12: kiểm thử 1–10) ==============================\\n# PASS ở đây chỉ xác nhận PHẦN MỀM làm đúng như mô tả — KHÔNG xác nhận giả thuyết khoa học.\\nmark_stage(\\"self_tests\\", requires=(\\"graph_v3\\",))\\nST = {}\\nt = ST[\\"canary\\"] = test_canary()\\ncheck(\\"C1\\", \\"[12.8] canary chỉ có trong caption TEST / tập ngoài không lọt vào từ vựng, quan hệ, từ vựng vị từ\\",\\n      not t[\\"leak_vocab\\"] and not t[\\"leak_relations\\"] and not t[\\"leak_predicates\\"], t)\\ncheck(\\"C1b\\", \\"đối chứng dương: canary đặt vào TRAIN thì bị phát hiện\\", t[\\"positive_control_detected\\"], t)\\nt = ST[\\"invariance\\"] = test_invariance()\\ncheck(\\"C2\\", \\"[12.8] xáo/thay caption TEST + tập ngoài: từ vựng, quan hệ, vị từ, tagger, taxonomy, inventory, đồ thị, mẫu gán nhãn GIỐNG HỆT\\",\\n      all(t.values()), t)\\nt = ST[\\"guards\\"] = test_guards()\\ncheck(\\"C3\\", \\"[12.8] hàng rào: chặn mở test / tập ngoài trước đóng băng + caption sai mục đích\\", all(t.values()), t)\\nt = ST[\\"inventory\\"] = test_inventory_constant()\\ncheck(\\"G1\\", \\"[12.5] inventory CỐ ĐỊNH giữa mọi biến thể đồ thị; cùng cấu hình mô hình ⇒ cùng số tham số\\",\\n      len(t[\\"node_counts\\"]) == 1 and t[\\"same_inventory\\"] and all(len(v) == 1 for v in t[\\"param_counts_by_pred_mode\\"].values()), t)\\nt = ST[\\"schema\\"] = test_schema_conformance(G_BASE)\\ncheck(\\"G2\\", \\"mọi cạnh tuân thủ schema (kiểu node, POS, trọng số, vị từ đúng họ, không lẫn tên loại quan hệ)\\", not t[\\"violations\\"], t)\\nt = ST[\\"path_identity\\"] = test_path_identity()\\ncheck(\\"G3\\", \\"[12.9] điểm đường đi ma trận = liệt kê ở cả 5 path_mode; tính lại R khớp; đường đi khớp vị từ ghi đúng vị từ + chiều\\",\\n      all(v[\\"mismatches\\"] == 0 and v[\\"recompute_maxdiff\\"] < 1e-5 and v[\\"pairs\\"] > 0 and v[\\"predicate_invalid\\"] == 0\\n          and v[\\"negative_control_detected\\"] == v[\\"predicate_paths\\"] for v in t.values())\\n      and t[\\"pred_match\\"][\\"predicate_paths\\"] > 0, t)\\nt = ST[\\"mp_weights\\"] = test_mp_weights()\\ncheck(\\"G4\\", \\"[12.6, 12.7] message passing: trọng số dùng đúng công thức, chỉ nhân MỘT lần (khớp tính tay, bất biến tỉ lệ), \\"\\n      \\"w=0 ≡ không cạnh, node cô lập hữu hạn, bất biến thứ tự, gradient hữu hạn\\",\\n      all(v[\\"as_designed\\"] and v[\\"isolated_nodes_finite\\"] and v[\\"edge_order_invariant\\"] and v[\\"grad_finite\\"] for v in t.values()), t)\\nt = ST[\\"predicate_forward\\"] = test_predicate_forward()\\ncheck(\\"P1\\", \\"[12.1] embedding vị từ TÁC ĐỘNG forward khi pred_mode ≠ none; pred_mode none (≡ v2) KHÔNG dùng vị từ\\",\\n      t[\\"predicate_used_as_designed\\"] and all(t[m][\\"finite\\"] for m in PRED_MODES), t)\\ncheck(\\"P2\\", \\"[12.2] hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác nhau\\", t[\\"two_predicates\\"][\\"differ\\"], t[\\"two_predicates\\"])\\nt = ST[\\"ablations\\"] = test_ablations(G_BASE)\\ncheck(\\"AB\\", \\"[12.5] mỗi ablation tắt ĐÚNG thành phần mô tả (mask, chính sách support, pred_mode, path_mode)\\", all(t.values()), t)\\nt = ST[\\"inductive\\"] = test_inductive_isolation()\\nind = [v for k, v in t.items() if k.endswith(\\":inductive\\")]\\ntra = [v for k, v in t.items() if k.endswith(\\":transductive\\")]\\ncheck(\\"G5\\", \\"[12.7] strict inductive (video test MSVD + tập ngoài): KHÔNG đổi node lớp/khái niệm hay video khác; transductive thì đổi; \\"\\n      \\"không NaN\\", all(v[\\"symbolic_changed\\"] < 1e-6 and v[\\"other_video_changed\\"] < 1e-6 and v[\\"own_video_changed\\"] > 1e-6\\n                        and v[\\"all_finite\\"] for v in ind) and all(v[\\"symbolic_changed\\"] > 1e-6 for v in tra), t)\\nR_AUD, ok_list = [], []\\nfor s in CFG.seeds:\\n    df_, ok_ = randomization_audit(G_BASE, graph_for(dict(DESIGN_BASE, rewire=True), s)[0])\\n    R_AUD.append(df_.assign(control=\\"rewire\\", seed=s, rand_seed=REWIRE_SEED_BASE + s)); ok_list.append(ok_)\\nS_AUD, ok_s = [], []\\nfor s in CFG.seeds:\\n    df_, ok_ = predicate_shuffle_audit(G_BASE, graph_for(dict(DESIGN_BASE, pred_shuffle=True), s)[0])\\n    S_AUD.append(df_.assign(control=\\"predicate_shuffle\\", seed=s, rand_seed=SHUFFLE_SEED_BASE + s)); ok_s.append(ok_)\\nRAND_AUDIT = pd.concat(R_AUD + S_AUD, ignore_index=True)\\nRAND_AUDIT.to_csv(RESULTS / \\"graph_randomization_audit.csv\\", index=False)\\nRANDOM_VALID, SHUFFLE_VALID = all(ok_list), all(ok_s)\\ndisplay(RAND_AUDIT[(RAND_AUDIT.seed == CFG.seeds[0]) & (RAND_AUDIT.control == \\"rewire\\")][\\n    [\\"relation\\", \\"edges_orig\\", \\"edges_rand\\", \\"out_degree_equal\\", \\"in_degree_equal\\", \\"type_pairs_equal\\", \\"weights_multiset_equal\\",\\n     \\"predicates_multiset_equal\\", \\"duplicates\\", \\"self_loops\\", \\"frac_changed\\", \\"accepted\\"]].round(3))\\ncheck(\\"G6\\", \\"[12.3] đối chứng rewiring giữ kiểu node, họ quan hệ, bậc, trọng số, đa tập vị từ; đổi ≥ ngưỡng (nếu không: KHÔNG kết luận về topology)\\",\\n      RANDOM_VALID, f\\"{int((~RAND_AUDIT[RAND_AUDIT.control == \'rewire\'].accepted).sum())} dòng không đạt\\", hard=False)\\ncheck(\\"G7\\", \\"[12.4] đối chứng hoán vị vị từ KHÔNG đổi topology / trọng số; chỉ đổi vị từ trong họ svo/prep (nếu không: KHÔNG kết luận về vị từ)\\",\\n      SHUFFLE_VALID, RAND_AUDIT[RAND_AUDIT.control == \\"predicate_shuffle\\"].groupby(\\"relation\\").frac_edges_predicates_changed.mean()\\n      .round(3).to_dict(), hard=False)\\ncheck(\\"C7\\", \\"split MSVD tái lập được (sha1 giống v2)\\", make_split(RECORDS, write=False)[\\"sha1\\"] == SPLIT[\\"sha1\\"], SPLIT[\\"sha1\\"])\\nv4_tensors()\\nt = ST[\\"v4_decomposition\\"] = test_v4_decomposition()\\ncheck(\\"Q3\\", \\"[v4] Σ số hạng (thị giác + nhãn + đường đi quan hệ) = điểm mô hình (T = 1, 2); xoá một cạnh ⇒ điểm đổi ĐÚNG bằng phần \\"\\n      \\"đóng góp các đường đi qua cạnh đó\\", all(v[\\"max_abs_diff\\"] < 1e-4 and v[\\"deletion_max_abs_diff\\"] < 1e-4 and v[\\"relation_terms\\"] > 0\\n                                             for v in t.values()), t)\\nt = ST[\\"v4_predicate_query\\"] = test_v4_predicate_and_query()\\ncheck(\\"Q1\\", \\"[v4] vị từ của truy vấn tác động attention trên ĐÚNG cạnh mang vị từ đó; v4_pred = False thì không (đối chứng âm); \\"\\n      \\"hoán vị vị từ đổi tín hiệu khớp nhưng giữ topology\\",\\n      t[\\"pred_change_use_pred=True\\"] > 1e-6 and t[\\"pred_change_use_pred=False\\"] < 1e-6 and t[\\"pred_shuffle_changes_match\\"] > 0\\n      and t[\\"pred_shuffle_same_topology\\"], t)\\ncheck(\\"Q2\\", \\"[v4] attention phụ thuộc truy vấn khi v4_qcond = True; giống hệt giữa các truy vấn khi False (đối chứng âm)\\",\\n      t[\\"two_queries_attention_diff_qcond=True\\"] > 1e-6 and t[\\"two_queries_attention_diff_qcond=False\\"] < 1e-6, t)\\nt = ST[\\"v4_inductive\\"] = test_v4_inductive_and_masks()\\ncheck(\\"Q4\\", \\"[v4] inductive: đổi đặc trưng/nhãn của MỘT video test (tập ngoài) không đổi điểm video khác (dung sai 1e-6 như G5: GPU cộng dồn atomic không tất định); rel = False ⇒ không số hạng \\"\\n      \\"quan hệ; điểm + gradient hữu hạn\\",\\n      all(v[\\"other_video_changed\\"] < 1e-6 and v[\\"own_video_changed\\"] > 1e-6 for k, v in t.items() if k in (\\"test\\", \\"ext\\"))\\n      and t[\\"no_relation_terms_when_rel_false\\"] and t[\\"grad_finite\\"] and t[\\"score_finite\\"], t)\\nsave_json(ST, RESULTS / \\"self_test_v4.json\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== V0 — CLIP-only (G0) + v3 cuối (tham chiếu R3) ==============================\\nmark_stage(\\"V0\\", requires=(\\"self_tests\\",))\\nPV = make_part(\\"val\\")\\nCLIP_RANK_VAL = full_gallery_ranks(TXT_FEAT, VID_FEAT, PV[\\"q_idx\\"], PV[\\"gallery\\"], PV[\\"gold\\"])\\nCLIP_VAL = metrics_from_ranks(CLIP_RANK_VAL, PV[\\"gold\\"])\\nALT_TAGS, ALT_SCORES = {}, {}\\nEXPS[\\"G0_clip_only\\"] = dict(experiment_id=\\"G0_clip_only\\", stage=\\"V0\\", description=\\"CLIP-only (không xếp lại bằng đồ thị, α = 1)\\",\\n                            design=None, hypothesis=\\"baseline chính\\", seeds=[], runs={}, status=\\"DONE\\",\\n                            val_MRR=(CLIP_VAL[\\"MRR\\"], 0.0), val_R1=(CLIP_VAL[\\"R@1\\"], 0.0), rr_val=1.0 / CLIP_RANK_VAL,\\n                            edges={}, n_params=0, seconds=0.0)\\nprint(f\\"CLIP-only VAL: MRR {CLIP_VAL[\'MRR\']:.4f} R@1 {CLIP_VAL[\'R@1\']:.4f}\\")\\nCFG.rerank_k = CFG.v4_rerank_k                       # k = 50: chọn trên VAL ở v3 (Kaggle), giữ cố định ở v4 — không nhìn test\\nCAND_RECALL_VAL = float(np.mean(CLIP_RANK_VAL <= CFG.rerank_k))\\nprovenance(\\"k (số ứng viên rerank)\\", \\"selected\\", \\"chọn trên VAL ở v3 (k = 50), giữ nguyên ở v4\\", caption_splits=(\\"val\\",))\\nv4_tensors()\\nrun_exp(\\"R3_v3_final\\", \\"V0\\", V3_FINAL, \\"v3 cuối (TypedSAGE + 12 mẫu đường đi, đồ thị E44_no_related) — tham chiếu\\", \\"v4 tốt hơn v3\\")\\n\\n", "id": "c053"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== V1 — v4 (QC-RGAT): chọn số lớp trên VAL ==============================\\nmark_stage(\\"V1\\", requires=(\\"V0\\",))\\nV4_T1 = dict(V3_FINAL, model=\\"v4\\", v4_layers=1)\\nrun_exp(\\"E10_v4_T1\\", \\"V1\\", V4_T1, \\"QC-RGAT 1 lớp (đường đi quan hệ ≤ 1 bước tới nhãn của video)\\", H_V4)\\nif \\"V1\\" in CFG.stages:\\n    run_exp(\\"E11_v4_T2\\", \\"V1\\", dict(V4_T1, v4_layers=2), \\"QC-RGAT 2 lớp (đường đi ≤ 2 bước)\\", \\"đường đi 2 bước có thêm thông tin\\")\\n    FINAL = decide(\\"V1\\", \\"E10_v4_T1\\", [\\"E11_v4_T2\\"], \\"v4_layers\\")\\nelse:\\n    NOT_RUN.append(dict(stage=\\"V1\\", experiment_id=\\"E11_v4_T2\\", reason=\\"không có trong CFG.stages\\"))\\n    FINAL = \\"E10_v4_T1\\"\\nCURRENT = dict(EXPS[FINAL][\\"design\\"])\\n\\n", "id": "c054"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== V2 (Level 4(b) v2) — nhãn video từ tagger HỌC từ caption TRAIN (chọn trên VAL) ==============================\\nmark_stage(\\"V2\\", requires=(\\"V1\\",))\\nPT_SELECT = None\\nif \\"V2\\" in CFG.stages:\\n    PT_TAGS, PT_SCORES, PT_SELECT, PT_GRID = run_tagger_probe(VID_FEAT, VOCAB2, SPLIT, CAPS)\\n    pd.DataFrame(PT_GRID).to_csv(RESULTS / \\"tagger_probe_selection_val.csv\\", index=False)\\n    save_json(dict(PT_SELECT, zero_shot_val=TAG_SELECT2), RESULTS / \\"tagger_probe_info.json\\")\\n    provenance(\\"tagger học (logistic đa nhãn trên CLIP video)\\", \\"learned\\", \\"nhãn = khái niệm trong caption TRAIN; train out-of-fold\\",\\n               caption_splits=(\\"train\\",))\\n    provenance(\\"siêu tham số tagger học (weight decay, k)\\", \\"selected\\", \\"F1 trên VAL\\", caption_splits=(\\"val\\",))\\n    ALT_TAGS[\\"probe\\"], ALT_SCORES[\\"probe\\"] = PT_TAGS, PT_SCORES\\n    print(f\\"Tagger VAL F1: zero-shot {TAG_SELECT2[\'val_F1\']:.3f} → học từ train {PT_SELECT[\'val_F1\']:.3f} \\"\\n          f\\"(wd {PT_SELECT[\'wd\']}, k danh từ {PT_SELECT[\'k_noun\']}, k động từ {PT_SELECT[\'k_verb\']})\\")\\n    run_exp(\\"E20_tags_probe\\", \\"V2\\", dict(CURRENT, tags=\\"probe\\"), \\"QC-RGAT, nhãn video từ tagger HỌC từ caption TRAIN\\",\\n            \\"nhãn video tốt hơn ⇒ đồ thị (contains + quan hệ) giúp hơn\\")\\n    FINAL = decide(\\"V2\\", FINAL, [\\"E20_tags_probe\\"], \\"tags\\")\\n    CURRENT = dict(EXPS[FINAL][\\"design\\"])\\n    print(f\\"→ V2: dùng {\'tagger HỌC\' if CURRENT.get(\'tags\') == \'probe\' else \'tagger zero-shot (incumbent)\'} ({FINAL})\\")\\nelse:\\n    NOT_RUN.append(dict(stage=\\"V2\\", experiment_id=\\"E20_tags_probe\\", reason=\\"không có trong CFG.stages\\"))"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== V7 — ma trận đối chứng dưới CÙNG cấu hình v4 (G3 = v4 đầy đủ) ==============================\\nmark_stage(\\"V7\\", requires=(\\"V1\\",))\\nG3_DESIGN = dict(CURRENT)\\nG_IDS = {\\"G0\\": \\"G0_clip_only\\", \\"V3REF\\": \\"R3_v3_final\\"}\\n\\n\\ndef control(gid, design, desc, hyp, same_as=None):\\n    G_IDS[gid.split(\\"_\\")[0]] = gid\\n    src = same_as or next((k for k, e in EXPS.items() if e[\\"status\\"] == \\"DONE\\" and e[\\"design\\"] == design\\n                           and sorted(e[\\"runs\\"]) == sorted(CFG.seeds)), None)\\n    if src is not None:\\n        return run_exp(gid, \\"V7\\", EXPS[src][\\"design\\"], f\\"{desc} (≡ {src}, dùng lại mô hình)\\", hyp, reuse=src)\\n    return run_exp(gid, \\"V7\\", design, desc, hyp)\\n\\n\\ncontrol(\\"G3_v4_full\\", G3_DESIGN, \\"v4 đầy đủ: quan hệ thật + vị từ + attention theo truy vấn\\", H_V4)\\nif \\"V7\\" in CFG.stages:\\n    control(\\"G1_contains_only\\", dict(G3_DESIGN, v4_rel=False), \\"cùng mô hình, KHÔNG cạnh quan hệ (chỉ contains + thị giác)\\",\\n            \\"quan hệ đóng góp ngoài contains\\")\\n    control(\\"G2_no_predicate\\", dict(G3_DESIGN, v4_pred=False), \\"cùng topology + vai + cổng theo truy vấn, KHÔNG khớp vị từ\\", H_V4)\\n    control(\\"G4_pred_shuffled\\", dict(G3_DESIGN, pred_shuffle=True), \\"phân phối vị từ hoán vị giữa các cạnh cùng họ\\",\\n            \\"ghép ĐÚNG vị từ vào ĐÚNG cạnh có ích\\")\\n    control(\\"G5_rewired\\", dict(G3_DESIGN, rewire=True), \\"constrained rewiring (giữ kiểu node, họ, bậc, trọng số, đa tập vị từ)\\",\\n            \\"cấu trúc liên kết thật có ích hơn ngẫu nhiên có ràng buộc\\")\\n    control(\\"G6_static_attention\\", dict(G3_DESIGN, v4_qcond=False), \\"attention tĩnh (chỉ loại cạnh + trọng số), KHÔNG theo truy vấn\\",\\n            \\"điều kiện theo truy vấn làm quan hệ có ích\\")\\n    control(\\"G7_no_role_edges\\", dict(G3_DESIGN, mask=add_mask(G3_DESIGN[\\"mask\\"], \\"no_roles\\")),\\n            \\"HUẤN LUYỆN LẠI không có họ cạnh vai (agent_of, object_of); cùng inventory, các họ khác, giao thức\\", \\"họ cạnh vai có giá trị\\")\\n    control(\\"G8_no_svo_prep_edges\\", dict(G3_DESIGN, mask=add_mask(G3_DESIGN[\\"mask\\"], \\"no_svo_prep\\")),\\n            \\"HUẤN LUYỆN LẠI không có họ cạnh svo / prep (có vị từ); cùng inventory, các họ khác, giao thức\\", \\"họ cạnh svo/prep có giá trị\\")\\n    if G3_DESIGN.get(\\"tags\\", \\"base\\") != \\"base\\":\\n        control(\\"G11_zeroshot_tags\\", dict(G3_DESIGN, tags=\\"base\\"), \\"như G3 nhưng nhãn video từ tagger CLIP zero-shot (v1)\\",\\n                \\"nhãn học từ caption train tốt hơn zero-shot\\")\\n    else:\\n        not_run(\\"G11_zeroshot_tags\\", \\"V7\\", \\"nhãn zero-shot\\", \\"V2 không chọn tagger học ⇒ G3 đã dùng nhãn zero-shot\\")\\n        G_IDS[\\"G11\\"] = \\"G11_zeroshot_tags\\"\\n    control(\\"V3REF\\", V3_FINAL, \\"v3 cuối (tham chiếu)\\", \\"v4 tốt hơn v3\\", same_as=\\"R3_v3_final\\")\\n    g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]\\n    a1, RANDOM_VALID_G3 = randomization_audit(g3, graph_for(dict(G3_DESIGN, rewire=True), CFG.seeds[0])[0])\\n    a2, SHUFFLE_VALID_G3 = predicate_shuffle_audit(g3, graph_for(dict(G3_DESIGN, pred_shuffle=True), CFG.seeds[0])[0])\\n    RAND_AUDIT = pd.concat([RAND_AUDIT, a1.assign(control=\\"rewire\\", config=\\"G3\\", seed=CFG.seeds[0]),\\n                            a2.assign(control=\\"predicate_shuffle\\", config=\\"G3\\", seed=CFG.seeds[0])], ignore_index=True)\\n    RAND_AUDIT.to_csv(RESULTS / \\"graph_randomization_audit.csv\\", index=False)\\nelse:\\n    NOT_RUN.append(dict(stage=\\"V7\\", experiment_id=\\"G1–G6\\", reason=\\"không có trong CFG.stages\\"))\\n    RANDOM_VALID_G3, SHUFFLE_VALID_G3 = RANDOM_VALID, SHUFFLE_VALID\\nDEC_DF = pd.DataFrame(DECISIONS)\\nDEC_DF.to_csv(RESULTS / \\"graph_stage_decisions.csv\\", index=False)\\nVAL_TABLE = pd.DataFrame([dict(experiment_id=k, stage=e[\\"stage\\"], status=e[\\"status\\"], description=e[\\"description\\"],\\n                               val_MRR=e[\\"val_MRR\\"][0], val_MRR_std=e[\\"val_MRR\\"][1], val_R1=e[\\"val_R1\\"][0],\\n                               n_seeds=len(e[\\"seeds\\"]) or (1 if e[\\"status\\"] == \\"DONE\\" else 0), seconds=e.get(\\"seconds\\"))\\n                          for k, e in EXPS.items()])\\ndisplay(VAL_TABLE.round(4))\\n", "id": "c055"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== V8 (Level 4(b) v2) — tri thức NGOÀI: cạnh ConceptNet giữa các lớp + đối chứng rewired ==============================\\nmark_stage(\\"V8\\", requires=(\\"V7\\",))\\nKG_STATUS = \\"NOT RUN — V8 không có trong CFG.stages\\"\\nif \\"V8\\" in CFG.stages:\\n    try:\\n        KG_EDGES[:] = load_conceptnet(VOCAB2[\\"terms\\"])\\n        KG_STATUS = f\\"OK — {len(KG_EDGES)} cạnh ConceptNet giữa khoá từ vựng TRAIN\\"\\n    except Exception as e:\\n        KG_STATUS = f\\"NOT RUN — không nạp được ConceptNet: {type(e).__name__}: {e}\\"\\nprint(\\"ConceptNet:\\", KG_STATUS)\\nif KG_EDGES:\\n    provenance(\\"cạnh related = ConceptNet 5.7\\", \\"external\\", f\\"{CFG.kg_url} (quan hệ dương, en–en, hai đầu ∈ từ vựng train)\\")\\n    G9_DESIGN = dict(G3_DESIGN, mask=drop_mask(G3_DESIGN[\\"mask\\"], \\"no_related\\"), related=\\"conceptnet\\")\\n    control(\\"G9_conceptnet\\", G9_DESIGN, \\"G3 + cạnh tri thức ngoài ConceptNet giữa các lớp (related = conceptnet)\\",\\n            \\"tri thức ngoài có giá trị (nhất là truy vấn gián tiếp)\\")\\n    control(\\"G10_conceptnet_rewired\\", dict(G9_DESIGN, rewire=True),\\n            \\"như G9 nhưng mọi cạnh quan hệ (gồm ConceptNet) rewired có ràng buộc\\", \\"topology thật của KG có ích hơn ngẫu nhiên\\")\\n    _g9 = graph_for(G9_DESIGN, CFG.seeds[0])[0]\\n    _a9, RANDOM_VALID_G9 = randomization_audit(_g9, graph_for(dict(G9_DESIGN, rewire=True), CFG.seeds[0])[0])\\n    RAND_AUDIT = pd.concat([RAND_AUDIT, _a9.assign(control=\\"rewire\\", config=\\"G9\\", seed=CFG.seeds[0])], ignore_index=True)\\n    RAND_AUDIT.to_csv(RESULTS / \\"graph_randomization_audit.csv\\", index=False)\\n    _R = _g9.A[\\"related\\"].tocoo()\\n    _used = {(min(a, b), max(a, b)) for a, b in zip(_R.row.tolist(), _R.col.tolist())}\\n    _kg = pd.DataFrame([dict(e, src_id=INV.key2id.get(e[\\"src\\"]), dst_id=INV.key2id.get(e[\\"dst\\"])) for e in KG_EDGES])\\n    _kg[\\"in_graph\\"] = [(min(a, b), max(a, b)) in _used if a is not None and b is not None else False\\n                       for a, b in zip(_kg.src_id, _kg.dst_id)]\\n    _kg.to_csv(RESULTS / \\"conceptnet_edges.csv\\", index=False)\\n    _types_ok = all(INV.node_type[a] != \\"video\\" and INV.node_type[b] != \\"video\\" and a != b for a, b in _used)\\n    check(\\"KG1\\", \\"cạnh ConceptNet trong đồ thị G9: chỉ nối lớp–lớp, không self-loop, hai đầu đều là node đang hoạt động; rewiring hợp lệ\\",\\n          _types_ok and bool(_used) and RANDOM_VALID_G9,\\n          dict(kg_edges_loaded=len(KG_EDGES), related_pairs_in_G9=len(_used), rewire_valid=RANDOM_VALID_G9,\\n               relations=dict(Counter(_kg[_kg.in_graph].rel))), hard=False)\\nelse:\\n    RANDOM_VALID_G9 = None\\n    for _g, _d in ((\\"G9_conceptnet\\", \\"G3 + ConceptNet\\"), (\\"G10_conceptnet_rewired\\", \\"G9 rewired\\")):\\n        not_run(_g, \\"V8\\", _d, KG_STATUS)\\n        G_IDS[_g.split(\\"_\\")[0]] = _g"}, {"cell_type": "code", "execution_count": null, "id": "c056", "metadata": {}, "outputs": [], "source": "# ============================== ĐÓNG BĂNG (trước mọi lần chạm test MSVD / tập ngoài của v4) ==============================\\nmark_stage(\\"freeze\\", requires=(\\"V7\\",))\\nuntouched = not TEST_GATE.log and all(CAPS.log.get((p, s), 0) == 0 for p in (\\"eval\\", \\"report\\", \\"display\\") for s in (\\"test\\", \\"ext\\"))\\nFROZEN = dict(version=\\"level4b_v2\\", kg_status=KG_STATUS, tagger_probe=PT_SELECT, model=\\"QC-RGAT\\", v4=dict(dim=CFG.v4_dim, lr=CFG.v4_lr), v3_final=V3_FINAL, final_experiment=FINAL, final_design=CURRENT, g3_design=G3_DESIGN, g_ids=G_IDS, design_base=DESIGN_BASE,\\n              rerank_k=CFG.rerank_k, learn_clip_w=CFG.learn_clip_w, seeds=CFG.seeds, split_sha1=SPLIT[\\"sha1\\"],\\n              ext_sha1=SPLIT[\\"ext_sha1\\"], ext_status=EXT_STATUS, inventory_sha1=INV.sha1, schema_sha1=SCHEMA[\\"sha1\\"],\\n              clip=f\\"{CFG.clip_model}/{CFG.clip_weights}\\", n_frames=POOL_SEL[\\"n_frames\\"], pooling=POOL_SEL[\\"pooling\\"],\\n              tagger=TAG_SELECT2, predicate_vocab=len(PRED_VOCAB), relation_quality=dict(automatic=AUTO_STATUS, human=REL_QUALITY_STATUS),\\n              verified_cells=sorted(VERIFIED_CELLS) if VERIFIED_CELLS is not None else None,\\n              random_control_valid=dict(base=RANDOM_VALID, g3=RANDOM_VALID_G3),\\n              predicate_shuffle_valid=dict(base=SHUFFLE_VALID, g3=SHUFFLE_VALID_G3),\\n              adoption_rule=dict(min_delta=CFG.adopt_min_delta, max_p_holm=CFG.adopt_max_p, default=\\"phương án đơn giản hơn\\"),\\n              confirmatory=PRIMARY_COMPARISONS, holm_alpha=CFG.holm_alpha,\\n              decisions=DECISIONS, experiments={k: dict(stage=e[\\"stage\\"], status=e[\\"status\\"], design=e[\\"design\\"],\\n                                                        alpha_lam=e.get(\\"alpha_lam\\"), val_MRR=e[\\"val_MRR\\"]) for k, e in EXPS.items()},\\n              stages_not_run=NOT_RUN, msvd_test_looks_before=BENCHMARK_LOOKS_BEFORE[\\"msvd_test\\"], benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,\\n              confirm_set=CFG.confirm_set, eval_query_rule=dict(n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed),\\n              note=(\\"test MSVD đã được nhìn ở v1–v4 và Level 4(b) v1 ⇒ ở đây chỉ là THĂM DÒ (lần 6), không dùng để chọn; \\"\\n                    \\"xác nhận = MSR-VTT fresh1k-C (1000 video CHƯA dùng ở v1–v4, Level 4(b) v1), đánh giá MỘT lần sau đóng băng\\"))\\nTEST_GATE.freeze(FROZEN)\\ncheck(\\"D1\\", \\"mọi lựa chọn của v4 chỉ dựa trên VAL và đóng băng TRƯỚC khi chạm test MSVD / tập ngoài\\", untouched,\\n      f\\"sha1={TEST_GATE.frozen_sha}; final={FINAL}; G3={G_IDS.get(\'G3\')}; quyết định={[(d[\'stage\'], d[\'candidate\'], d[\'adopted\']) for d in DECISIONS]}\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC (dựng sau đóng băng, TRƯỚC khi chấm bất kỳ mô hình nào) ==============================\\nmark_stage(\\"eval_queries\\", requires=(\\"freeze\\",))\\n_scored_before = sum(CAPS.log.get((\\"eval\\", s_), 0) for s_ in (\\"val\\", \\"test\\", \\"ext\\"))\\n_G3_GRAPH = graph_for(G3_DESIGN, CFG.seeds[0])[0]\\n_qs_parts = [(\\"explore\\", make_part(\\"val\\" if CFG.dev_mode else \\"test\\"))] + ([(\\"confirm\\", make_part(\\"ext\\"))] if EXT_RECORDS else [])\\nQS_ALL, QS_INFO = [], {}\\nfor _k, _p in _qs_parts:\\n    _df, _inf = build_eval_queries(_p, _G3_GRAPH)\\n    QS_ALL.append(_df.assign(part_key=_k))\\n    QS_INFO[_k] = dict(part=_p[\\"name\\"], **_inf)\\nQS = pd.concat(QS_ALL, ignore_index=True)\\nQS.to_csv(RESULTS / \\"eval_queries_level4b.csv\\", index=False)\\nQS_SHA1 = sha1_of(QS[[\\"query_id\\", \\"q_idx\\", \\"type\\"]].values.tolist())\\nsave_json(dict(rule=build_eval_queries.__doc__, n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed, sha1=QS_SHA1,\\n               model_scores_used=False, eval_calls_before_build=_scored_before, graph=\\"G3 (đồ thị cuối, cạnh từ caption TRAIN)\\",\\n               tags=\\"CLIP zero-shot trên frame của chính video (tagger chọn trên VAL)\\", info=QS_INFO), RESULTS / \\"eval_queries_level4b.json\\")\\nprovenance(\\"bộ truy vấn đánh giá Level 4(b)\\", \\"evaluation\\", \\"luật cố định: nhãn CLIP của video đúng + cạnh đồ thị train; không dùng điểm mô hình\\",\\n           caption_splits=tuple(sorted({p_[\\"split\\"] for _, p_ in _qs_parts})))\\nprint(f\\"Bộ truy vấn đăng ký trước (sha1 {QS_SHA1}): {QS_INFO}\\")\\ndisplay(QS[[\\"query_id\\", \\"type\\", \\"text\\", \\"gold_video_id\\", \\"concepts_seen_in_video\\", \\"concepts_missing\\", \\"expected_relations\\"]].head(8))\\n_main_k = \\"confirm\\" if \\"confirm\\" in QS_INFO else \\"explore\\"\\ncheck(\\"QS1\\", f\\"bộ truy vấn trực tiếp / gián tiếp ≥ {CFG.n_eval_queries} / nhóm trên tập đánh giá chính, theo luật đăng ký trước, \\"\\n             \\"dựng TRƯỚC khi chấm mô hình (không dùng điểm mô hình)\\",\\n      _scored_before == 0 and all(QS_INFO[_main_k][g_][\\"selected\\"] >= CFG.n_eval_queries for g_ in (\\"direct\\", \\"indirect\\")),\\n      {k_: {g_: v_[g_][\\"selected\\"] for g_ in (\\"direct\\", \\"indirect\\")} for k_, v_ in QS_INFO.items()}, hard=False)"}, {"cell_type": "code", "execution_count": null, "id": "c057", "metadata": {}, "outputs": [], "source": "# ============================== ĐÁNH GIÁ sau đóng băng: test MSVD (THĂM DÒ) + tập ngoài (XÁC NHẬN) ==============================\\n# dev_mode: test MSVD bị khoá ⇒ phần \'thăm dò\' chạy trên VAL (gắn nhãn DEV); tập ngoài = video TRAIN-9k của MSR-VTT (DEV).\\nmark_stage(\\"evaluate\\", requires=(\\"freeze\\",))\\nEVAL_SPLIT = \\"val\\" if CFG.dev_mode else \\"test\\"\\n_, E_G3 = graph_for(G3_DESIGN, CFG.seeds[0])\\nresolve = lambda x: FINAL if x == \\"FINAL\\" else G_IDS.get(x, x)\\n\\n\\ndef eval_part(part, kind):\\n    \\"\\"\\"Đánh giá MỌI thí nghiệm đã chạy trên một tập truy vấn; trả về thứ hạng, nhóm truy vấn, bảng metric.\\"\\"\\"\\n    CAPS.use(part[\\"q_idx\\"], \\"eval\\")\\n    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, part[\\"q_idx\\"], part[\\"gallery\\"], part[\\"gold\\"])\\n    groups = query_groups_v3(E_G3, part[\\"q_idx\\"])\\n    pg = predicate_groups(E_G3, part[\\"q_idx\\"])\\n    masks = {\\"all\\": np.ones(len(groups), bool), \\"direct\\": groups == \\"direct\\", \\"indirect\\": groups == \\"indirect\\",\\n             \\"other\\": groups == \\"other\\", \\"rare_concepts\\": rare_mask(E_G3, part[\\"q_idx\\"]),\\n             \\"pred_rare\\": pg == \\"pred_rare\\", \\"pred_common\\": pg == \\"pred_common\\", \\"no_pred\\": pg == \\"no_pred\\"}\\n    ev, hashes = {\\"G0_clip_only\\": {None: dict(rank=clip_rank)}}, {}\\n    t0 = time.time()\\n    for k, e in EXPS.items():\\n        if e[\\"runs\\"]:\\n            ev[k] = evaluate_exp(k, part, clip_rank)\\n            hashes[k] = {r[\\"cand_hash\\"] for r in ev[k].values()}\\n    rows = []\\n    for k, ev_k in ev.items():\\n        e = EXPS[k]\\n        for grp, m in masks.items():\\n            if m.sum() < 5:\\n                continue\\n            ms = [metrics_from_ranks(v[\\"rank\\"][m], part[\\"gold\\"][m]) for v in ev_k.values()]\\n            rows.append(dict(part=part[\\"name\\"], kind=kind, experiment_id=k, stage=e[\\"stage\\"], description=e[\\"description\\"], group=grp,\\n                             n_queries=int(m.sum()), n_videos=ms[0][\\"n_videos\\"], n_seeds=len(ms),\\n                             **{f\\"{x}_mean\\": float(np.mean([mm[x] for mm in ms])) for x in METRIC_KEYS},\\n                             **{f\\"{x}_std\\": float(np.std([mm[x] for mm in ms])) for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\", \\"MRR_video_macro\\")},\\n                             val_MRR=e[\\"val_MRR\\"][0]))\\n    info = dict(part=part[\\"name\\"], kind=kind, n_queries=len(part[\\"q_idx\\"]), n_videos=len(part[\\"gallery\\"]),\\n                candidate_recall=float(np.mean(clip_rank <= CFG.rerank_k)), groups=dict(Counter(groups.tolist())),\\n                predicate_groups=dict(Counter(pg.tolist())), seconds=round(time.time() - t0, 1),\\n                cand_hashes=sorted(set().union(*hashes.values())) if hashes else [])\\n    return dict(part=part, ev=ev, masks=masks, rows=pd.DataFrame(rows), info=info)\\n\\n\\ndef rr_of(E, k, m=None):\\n    rr = np.mean([1.0 / v[\\"rank\\"] for v in E[\\"ev\\"][k].values()], 0)\\n    return rr if m is None else rr[m]\\n\\n\\ndef paired(E, a, b, grp=\\"all\\"):\\n    m = E[\\"masks\\"][grp]\\n    bs = cluster_bootstrap_diff(rr_of(E, a, m), rr_of(E, b, m), E[\\"part\\"][\\"gold\\"][m])\\n    return dict(part=E[\\"part\\"][\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], a=a, b=b, group=grp, n=int(m.sum()),\\n                MRR_a=float(rr_of(E, a, m).mean()), MRR_b=float(rr_of(E, b, m).mean()), **bs)\\n\\n\\nEVALS = {}\\nEVALS[\\"explore\\"] = eval_part(make_part(EVAL_SPLIT), \\"DEV_val\\" if CFG.dev_mode else \\"exploratory_test_6th_look\\")\\nif EXT_RECORDS:\\n    EVALS[\\"confirm\\"] = eval_part(make_part(\\"ext\\"), \\"DEV_ext_train9k\\" if EXT_REPORT[\\"dev\\"] else\\n                                  (\\"confirmatory\\" if EXT_REPORT.get(\\"set\\") == \\"fresh1kC\\" else \\"exploratory_seen_benchmark\\"))\\n    EVALS[\\"confirm_std\\"] = eval_part(make_part(\\"ext\\", std_only=True), \\"DEV_ext_train9k_std\\" if EXT_REPORT[\\"dev\\"] else\\n                                      (\\"confirmatory_secondary\\" if EXT_REPORT.get(\\"set\\") == \\"fresh1kC\\" else \\"exploratory_seen_benchmark_std\\"))\\nfor k_, E in EVALS.items():\\n    print(f\\"[{k_}] {E[\'info\']}\\")\\ncheck(\\"D2\\", \\"mọi thí nghiệm dùng chung split, gallery, truy vấn, tập ứng viên CLIP và hàm metric (từng tập đánh giá)\\",\\n      all(len(E[\\"info\\"][\\"cand_hashes\\"]) == 1 for E in EVALS.values()), {k_: E[\\"info\\"][\\"cand_hashes\\"] for k_, E in EVALS.items()})\\n\\nCOMPARE = []\\nfor d in DECISIONS:\\n    if d.get(\\"adopted\\") is not None and d[\\"candidate\\"] in EXPS and EXPS[d[\\"candidate\\"]][\\"runs\\"] and EXPS[d[\\"incumbent\\"]].get(\\"runs\\") is not None:\\n        COMPARE.append((f\\"{d[\'stage\']}_variant\\", d[\\"candidate\\"], d[\\"incumbent\\"]))\\nfor gid in (\\"G1\\", \\"G2\\", \\"G4\\", \\"G5\\", \\"G6\\", \\"G7\\", \\"G8\\", \\"G9\\", \\"G10\\", \\"G11\\", \\"V3REF\\"):\\n    if gid in G_IDS and EXPS.get(G_IDS[gid], {}).get(\\"runs\\"):\\n        COMPARE.append((\\"G3_vs_control\\", G_IDS[\\"G3\\"], G_IDS[gid]))\\nif \\"G3\\" in G_IDS:\\n    COMPARE.append((\\"G3_vs_clip\\", G_IDS[\\"G3\\"], \\"G0_clip_only\\"))       # so sánh PHỤ theo nhóm trực tiếp / gián tiếp\\nif EXPS.get(G_IDS.get(\\"G9\\"), {}).get(\\"runs\\"):\\n    COMPARE.append((\\"G9_vs_clip\\", G_IDS[\\"G9\\"], \\"G0_clip_only\\"))       # v2: đồ thị + ConceptNet vs CLIP theo nhóm\\nSIG_ROWS, CONF_ROWS = [], []\\nfor k_, E in EVALS.items():\\n    for c, a, b in COMPARE:\\n        if a in E[\\"ev\\"] and b in E[\\"ev\\"]:\\n            for g_ in (\\"all\\", \\"direct\\", \\"indirect\\", \\"rare_concepts\\", \\"pred_rare\\", \\"pred_common\\"):\\n                if E[\\"masks\\"][g_].sum() >= 5:\\n                    SIG_ROWS.append(dict(comparison=c, **paired(E, a, b, g_)))\\n    fam = [(h, resolve(a), resolve(b)) for h, a, b in PRIMARY_COMPARISONS if not (h == \\"H0_final_vs_clip\\" and FINAL == G_IDS.get(\\"G3\\"))]\\n    ok = [(h, a, b) for h, a, b in fam if a in E[\\"ev\\"] and b in E[\\"ev\\"]]\\n    res = [dict(hypothesis=h, **paired(E, a, b)) for h, a, b in ok]\\n    adj = holm([r[\\"p_not_better\\"] for r in res])\\n    for r, pa in zip(res, adj):\\n        r.update(p_holm=float(pa), supported=bool(pa < CFG.holm_alpha and r[\\"diff\\"] > 0),\\n                 status=\\"CONFIRMATORY\\" if E[\\"info\\"][\\"kind\\"] == \\"confirmatory\\" else\\n                 (\\"EXPLORATORY (benchmark đã được nhìn trước đây)\\" if E[\\"info\\"][\\"kind\\"].startswith(\\"exploratory\\") else \\"DEV\\"))\\n        CONF_ROWS.append(r)\\n    for h, a, b in fam:\\n        if (h, a, b) not in ok:\\n            CONF_ROWS.append(dict(hypothesis=h, part=E[\\"part\\"][\\"name\\"], kind=E[\\"info\\"][\\"kind\\"], a=a, b=b, status=\\"NOT RUN\\",\\n                                  note=f\\"{a if a not in E[\'ev\'] else b}: {EXPS.get(a if a not in E[\'ev\'] else b, {}).get(\'reason\', \'không có\')}\\"))\\nif not EXT_RECORDS:\\n    for h, a, b in PRIMARY_COMPARISONS:\\n        CONF_ROWS.append(dict(hypothesis=h, part=\\"ext\\", kind=\\"confirmatory\\", a=resolve(a), b=resolve(b), status=\\"NOT RUN\\", note=EXT_STATUS))\\nSIG_T = pd.DataFrame(SIG_ROWS)\\nCONF_T = pd.DataFrame(CONF_ROWS)\\nRES_ALL = pd.concat([E[\\"rows\\"] for E in EVALS.values()], ignore_index=True)\\nRES_ALL.to_csv(RESULTS / \\"graph_experiment_results.csv\\", index=False)\\nSIG_T.to_csv(RESULTS / \\"graph_paired_tests.csv\\", index=False)\\nCONF_T.to_csv(RESULTS / \\"graph_confirmatory_tests.csv\\", index=False)\\nsave_json({k_: E[\\"info\\"] for k_, E in EVALS.items()}, RESULTS / \\"graph_eval_parts.json\\")\\nshow = [\\"part\\", \\"experiment_id\\", \\"n_seeds\\", \\"R@1_mean\\", \\"R@5_mean\\", \\"R@10_mean\\", \\"MRR_mean\\", \\"MRR_std\\", \\"MRR_video_macro_mean\\", \\"val_MRR\\"]\\nprint(\\"\\\\nKẾT QUẢ (nhóm \'all\'):\\")\\ndisplay(RES_ALL[(RES_ALL.group == \\"all\\") & RES_ALL.experiment_id.isin(list(G_IDS.values()) + [FINAL])][show].round(4))\\nprint(\\"Họ so sánh chính (một phía, Holm):\\")\\ndisplay(CONF_T[[c for c in (\\"hypothesis\\", \\"part\\", \\"status\\", \\"a\\", \\"b\\", \\"MRR_a\\", \\"MRR_b\\", \\"diff\\", \\"ci_lo\\", \\"ci_hi\\", \\"p_not_better\\",\\n                            \\"p_holm\\", \\"supported\\") if c in CONF_T]].round(4))\\n_TAGS_USED = ALT_TAGS[G3_DESIGN[\\"tags\\"]] if G3_DESIGN.get(\\"tags\\", \\"base\\") != \\"base\\" else TAGS2\\nTAGGER_REPORT_ZS = {k_: tagger_report_v2(TAGS2, SPLIT[E[\\"part\\"][\\"split\\"]], VOCAB2, \\"report\\")\\n                    for k_, E in EVALS.items() if k_ != \\"confirm_std\\"}\\nTAGGER_REPORT = {k_: tagger_report_v2(_TAGS_USED, SPLIT[E[\\"part\\"][\\"split\\"]], VOCAB2, \\"report\\")\\n                 for k_, E in EVALS.items() if k_ != \\"confirm_std\\"}\\nprint(f\\"Tagger trên tập đánh giá — dùng trong G3 ({G3_DESIGN.get(\'tags\', \'base\')}):\\", TAGGER_REPORT, \\"| zero-shot:\\", TAGGER_REPORT_ZS)\\ncheck(\\"D3\\", \\"R@1/5/10, MRR, macro-MRR theo video + kiểm định cặp cho G0–G6 + v3 trên tập xác nhận độc lập\\",\\n      \\"confirm\\" in EVALS and EVALS[\\"confirm\\"][\\"info\\"][\\"kind\\"] == \\"confirmatory\\",\\n      f\\"{len(RES_ALL)} dòng → graph_experiment_results.csv | xác nhận: {EXT_STATUS}\\", hard=False,\\n      status=None if (\\"confirm\\" in EVALS and EVALS[\\"confirm\\"][\\"info\\"][\\"kind\\"] == \\"confirmatory\\") else \\"NOT RUN\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Phân tích quan hệ v4: knockout theo họ, attention theo vị từ / vai ==============================\\nmark_stage(\\"relations\\", requires=(\\"evaluate\\",))\\nP_MAIN = EVALS[\\"confirm\\"][\\"part\\"] if \\"confirm\\" in EVALS else EVALS[\\"explore\\"][\\"part\\"]\\nG3_ID = G_IDS[\\"G3\\"]\\nRA_ROWS, RA_SUMM = [], []\\nfor part in (PV, P_MAIN):\\n    df_, s_ = relation_analysis_v4(EXPS[G3_ID][\\"runs\\"][CFG.seeds[0]], part)\\n    RA_ROWS.append(df_.assign(experiment_id=G3_ID)); RA_SUMM.append(dict(experiment_id=G3_ID, split=part[\\"name\\"], **s_))\\nREL_DF = pd.concat(RA_ROWS, ignore_index=True)\\nREL_DF.to_csv(RESULTS / \\"graph_relation_analysis.csv\\", index=False)\\nsave_json(RA_SUMM, RESULTS / \\"graph_relation_analysis_summary.json\\")\\ndisplay(REL_DF[REL_DF.split == P_MAIN[\\"name\\"]].round(4))\\nfor s_ in RA_SUMM:\\n    print(f\\"{s_[\'split\']:8s} MRR {s_[\'base_MRR\']:.4f} | bỏ MỌI cạnh quan hệ lúc suy luận ⇒ {s_[\'all_relations_knockout_MRR\']:.4f} | \\"\\n          f\\"attention cạnh có vị từ KHỚP truy vấn {s_[\'att_pred_match\']:.3f} vs không khớp {s_[\'att_pred_nomatch\']:.3f}\\")\\n\\n", "id": "c058"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== xAI v4: giải thích = số hạng của điểm; fidelity bằng xoá cạnh THẬT ==============================\\nmark_stage(\\"xai\\", requires=(\\"relations\\",))\\nRUN_F = EXPS[G3_ID][\\"runs\\"][CFG.seeds[0]]\\nGROUPS_X = query_groups_v3(RUN_F[\\"engine\\"], P_MAIN[\\"q_idx\\"])\\nXAI_DF = xai_v4(RUN_F, P_MAIN, GROUPS_X)\\nXAI_DF.to_csv(RESULTS / \\"graph_xai_fidelity.csv\\", index=False)\\nSENS = []\\nif len(CFG.seeds) > 1 and len(XAI_DF):\\n    for _, r in XAI_DF.head(CFG.n_seed_sensitivity).iterrows():\\n        sets = [{tuple(t[\\"edges\\"]) for t in explain_v4(EXPS[G3_ID][\\"runs\\"][s][\\"model\\"], int(r.q), int(r.gold))[\\"top_relation\\"]}\\n                for s in CFG.seeds]\\n        js = [len(a & b) / max(1, len(a | b)) for a, b in itertools.combinations(sets, 2)]\\n        SENS.append(dict(q=int(r.q), gold=int(r.gold), mean_jaccard=float(np.mean(js))))\\nSENS_DF = pd.DataFrame(SENS)\\nSENS_DF.to_csv(RESULTS / \\"graph_xai_seed_sensitivity.csv\\", index=False)\\n_f = lambda d, c: float(d[c].mean()) if len(d) and c in d else float(\\"nan\\")\\nXAI_SUMM = dict(experiment_id=G3_ID, part=P_MAIN[\\"name\\"], n_queries=len(XAI_DF), validity=_f(XAI_DF, \\"validity\\"),\\n                predicate_validity=_f(XAI_DF, \\"predicate_validity\\"), predicate_edges=int(XAI_DF.n_predicate_edges.sum()) if len(XAI_DF) else 0,\\n                sparsity=_f(XAI_DF, \\"sparsity\\"), share_relation=_f(XAI_DF, \\"share_relation\\"),\\n                decomposition_ok=_f(XAI_DF, \\"decomposition_ok\\"), deletion_matches_decomposition=_f(XAI_DF, \\"deletion_matches_decomposition\\"),\\n                d_s_expl=_f(XAI_DF, \\"d_s_expl\\"), d_s_rand=_f(XAI_DF, \\"d_s_rand\\"),\\n                frac_score_decreased_expl=float((XAI_DF.d_s_expl < -1e-9).mean()) if len(XAI_DF) else float(\\"nan\\"),\\n                rank_full=_f(XAI_DF, \\"rank_full\\"), rank_drop_expl=_f(XAI_DF, \\"rank_drop_expl\\"), rank_drop_rand=_f(XAI_DF, \\"rank_drop_rand\\"),\\n                frac_rank_changed_expl=float((XAI_DF.rank_drop_expl != XAI_DF.rank_full).mean()) if len(XAI_DF) else float(\\"nan\\"),\\n                frac_rank_changed_rand=float((XAI_DF.rank_drop_rand != XAI_DF.rank_full).mean()) if len(XAI_DF) else float(\\"nan\\"),\\n                frac_top1_changed_expl=_f(XAI_DF, \\"top1_changed_expl\\"), frac_top10_changed_expl=_f(XAI_DF, \\"top10_changed_expl\\"),\\n                seed_sensitivity_mean_jaccard=_f(SENS_DF, \\"mean_jaccard\\"), alpha=RUN_F[\\"alpha\\"])\\nsave_json(XAI_SUMM, RESULTS / \\"graph_xai_summary.json\\")\\nprint(json.dumps(XAI_SUMM, indent=1, ensure_ascii=False))\\nif len(XAI_DF):\\n    for _, r in XAI_DF.head(3).iterrows():\\n        print(f\\"  q={r.q} (nhóm {r.group}) hạng {r.rank_full}→{r.rank_drop_expl} khi xoá giải thích: {r.explanation[:300]}\\")\\ncheck(\\"X1\\", \\"[12.9] giải thích hợp lệ (cạnh thật đúng loại/chiều; vị từ khớp nằm trên ĐÚNG cạnh); Σ số hạng = điểm; ≥ 10 truy vấn\\",\\n      len(XAI_DF) >= 10 and XAI_SUMM[\\"validity\\"] == 1.0 and XAI_SUMM[\\"predicate_validity\\"] == 1.0 and XAI_SUMM[\\"decomposition_ok\\"] == 1.0,\\n      XAI_SUMM, hard=False)\\ncheck(\\"X2\\", \\"xoá cạnh của các đường đi ỦNG HỘ (giải thích) làm điểm mô hình giảm nhiều hơn xoá cùng số cạnh quan hệ ngẫu nhiên (vào nhãn của cùng video)\\",\\n      XAI_SUMM[\\"d_s_expl\\"] < XAI_SUMM[\\"d_s_rand\\"], f\\"{XAI_SUMM[\'d_s_expl\']:.4f} vs {XAI_SUMM[\'d_s_rand\']:.4f}\\", hard=False)\\ncheck(\\"X3\\", \\"T = 1: Δ điểm khi xoá cạnh = đúng tổng đóng góp đã báo (fidelity chính xác theo cấu tạo)\\",\\n      RUN_F[\\"model\\"].T != 1 or XAI_SUMM[\\"deletion_matches_decomposition\\"] == 1.0, XAI_SUMM[\\"deletion_matches_decomposition\\"], hard=False)\\n# X4 (thứ hạng có đổi khi xoá giải thích?) là KẾT QUẢ khoa học, không phải kiểm thử phần mềm ⇒ báo ở FINDING F-X4 (ô xAI theo thứ hạng).\\n", "id": "c059"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — METRIC TRÊN BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC + nhóm trực tiếp / gián tiếp ==============================\\nmark_stage(\\"eval_queries_metrics\\", requires=(\\"evaluate\\", \\"eval_queries\\"))\\nQM, QP = [], []\\nfor _k, E in EVALS.items():\\n    _q = QS[QS.part_key == _k] if _k in (\\"explore\\", \\"confirm\\") else QS.iloc[:0]\\n    if len(_q):\\n        _a, _b = eval_query_metrics(E, _q)\\n        QM.append(_a); QP.append(_b)\\nQS_METRICS = pd.concat(QM, ignore_index=True) if QM else pd.DataFrame()\\nQS_PAIRED = pd.concat(QP, ignore_index=True) if QP else pd.DataFrame()\\nQS_METRICS.to_csv(RESULTS / \\"eval_queries_metrics.csv\\", index=False)\\nQS_PAIRED.to_csv(RESULTS / \\"eval_queries_paired.csv\\", index=False)\\n_show = [G_IDS[g_] for g_ in (\\"G0\\", \\"G1\\", \\"G3\\", \\"G7\\", \\"G8\\", \\"V3REF\\") if g_ in G_IDS]\\nfor _k in (\\"explore\\", \\"confirm\\"):\\n    if _k in EVALS and len(QS_METRICS):\\n        _t = QS_METRICS[(QS_METRICS.part == EVALS[_k][\\"part\\"][\\"name\\"]) & QS_METRICS.experiment_id.isin(_show)]\\n        print(f\\"\\\\n[{_k}] bộ truy vấn đăng ký trước — MRR (trung bình theo seed):\\")\\n        display(_t.pivot_table(index=\\"experiment_id\\", columns=\\"group\\", values=\\"MRR_mean\\").round(4))\\nif len(QS_PAIRED):\\n    display(QS_PAIRED[[\\"part\\", \\"group\\", \\"a\\", \\"b\\", \\"n\\", \\"MRR_a\\", \\"MRR_b\\", \\"diff\\", \\"ci_lo\\", \\"ci_hi\\"]].round(4))\\n_mk = \\"confirm\\" if \\"confirm\\" in EVALS else \\"explore\\"\\n_sig = SIG_T[(SIG_T.comparison == \\"G3_vs_clip\\") & (SIG_T.part == EVALS[_mk][\\"part\\"][\\"name\\"])] if len(SIG_T) else SIG_T\\nfor _, r in _sig[_sig.group.isin([\\"all\\", \\"direct\\", \\"indirect\\"])].iterrows():\\n    finding(f\\"F-GRP-{r.group}\\", f\\"QC-RGAT (G3) vs CLIP-only trên nhóm \'{r.group}\' của {r.part} (mọi caption; so sánh PHỤ, không Holm)\\",\\n            f\\"n={r.n}: ΔMRR {r[\'diff\']:+.4f} [CI95 theo video {r.ci_lo:+.4f}, {r.ci_hi:+.4f}]\\",\\n            \\"đồ thị giúp nhóm này\\" if r.ci_lo > 0 else (\\"đồ thị làm kém nhóm này\\" if r.ci_hi < 0 else \\"không khác biệt có ý nghĩa\\"))\\n_qi = QS_PAIRED[(QS_PAIRED.group == \\"indirect\\") & (QS_PAIRED.a == G_IDS.get(\\"G3\\")) & (QS_PAIRED.part == EVALS[_mk][\\"part\\"][\\"name\\"])] \\\\\\n    if len(QS_PAIRED) else QS_PAIRED\\nif len(_qi):\\n    r = _qi.iloc[0]\\n    finding(\\"F-QS-indirect\\", f\\"G3 vs CLIP-only trên {r.n} truy vấn GIÁN TIẾP đăng ký trước ({r.part})\\",\\n            f\\"MRR {r.MRR_b:.4f} → {r.MRR_a:.4f} (Δ {r[\'diff\']:+.4f}, CI95 [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}])\\",\\n            \\"n nhỏ ⇒ CI rộng; chỉ là bằng chứng bổ sung cho kết quả trên mọi caption\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — xAI theo THỨ HẠNG: giải thích đầy đủ (contains + đường đi quan hệ) ==============================\\nmark_stage(\\"xai_rank\\", requires=(\\"xai\\",))\\nXR = xai_rank_fidelity(RUN_F, P_MAIN)\\nXR.to_csv(RESULTS / \\"xai_rank_fidelity.csv\\", index=False)\\n_m = lambda c: float(XR[c].mean()) if len(XR) else float(\\"nan\\")\\nXR_SUMM = dict(experiment_id=G3_ID, part=P_MAIN[\\"name\\"], n_queries=len(XR), top_m=CFG.top_m_paths, alpha=RUN_F[\\"alpha\\"],\\n               validity=_m(\\"validity\\"), predicate_validity=_m(\\"predicate_validity\\"), decomposition_ok=_m(\\"decomposition_ok\\"),\\n               size_edges=_m(\\"n_expl_edges\\"), sparsity=_m(\\"sparsity\\"), coverage=_m(\\"coverage\\"),\\n               expl_contains_terms=_m(\\"n_expl_contains\\"), expl_relation_terms=_m(\\"n_expl_relation\\"),\\n               d_s_expl=_m(\\"d_s_expl\\"), d_s_rand=_m(\\"d_s_rand\\"), d_final_expl=_m(\\"d_final_expl\\"), d_final_rand=_m(\\"d_final_rand\\"),\\n               rank_full=_m(\\"rank_full\\"), rank_drop_expl=_m(\\"rank_drop_expl\\"), rank_drop_rand=_m(\\"rank_drop_rand\\"),\\n               frac_rank_worse_expl=_m(\\"rank_worse_expl\\"), frac_rank_worse_rand=_m(\\"rank_worse_rand\\"),\\n               frac_top1_changed_expl=_m(\\"top1_changed_expl\\"), frac_top1_changed_rand=_m(\\"top1_changed_rand\\"),\\n               frac_top10_changed_expl=_m(\\"top10_changed_expl\\"), frac_top10_changed_rand=_m(\\"top10_changed_rand\\"),\\n               share_graph=_m(\\"share_graph\\"), share_relation=_m(\\"share_relation\\"),\\n               definitions=dict(explanation=\\"top-m số hạng ĐỒ THỊ có đóng góp dương (contains: video –chứa→ nhãn; quan hệ: u –r→ nhãn ←chứa– video)\\",\\n                                validity=\\"tỉ lệ số hạng giải thích mà MỌI cạnh có thật trong đồ thị đang chạy, đúng loại, chiều, kiểu node\\",\\n                                sparsity=\\"số hạng giải thích / số hạng đồ thị khác 0 của video (nhỏ = gọn)\\", size_edges=\\"số cạnh bị xoá\\",\\n                                coverage=\\"Σ đóng góp của giải thích / Σ đóng góp đồ thị dương\\",\\n                                fidelity=\\"xoá THẬT các cạnh của giải thích rồi chấm lại mọi ứng viên: Δ điểm đồ thị, Δ điểm cuối, Δ hạng, top-1/top-10; \\"\\n                                         \\"đối chứng = xoá cùng số cạnh contains khác + cạnh quan hệ ngẫu nhiên vào nhãn của cùng video\\"))\\nsave_json(XR_SUMM, RESULTS / \\"xai_level4b_summary.json\\")\\nprint(json.dumps({k_: (round(v_, 4) if isinstance(v_, float) else v_) for k_, v_ in XR_SUMM.items() if k_ != \\"definitions\\"}, ensure_ascii=False))\\nif len(XR):\\n    display(XR[[\\"gold_video_id\\", \\"rank_full\\", \\"rank_drop_expl\\", \\"rank_drop_rand\\", \\"d_s_expl\\", \\"d_s_rand\\", \\"d_final_expl\\", \\"explanation\\"]].head(6))\\ncheck(\\"X5\\", \\"giải thích đầy đủ hợp lệ (mọi cạnh contains / quan hệ có thật, đúng loại, chiều, kiểu node; vị từ đúng cạnh); \\"\\n            \\"Σ số hạng = điểm; ≥ 10 truy vấn\\", len(XR) >= 10 and XR_SUMM[\\"validity\\"] == 1.0 and XR_SUMM[\\"predicate_validity\\"] == 1.0\\n      and XR_SUMM[\\"decomposition_ok\\"] == 1.0, {k_: XR_SUMM[k_] for k_ in (\\"n_queries\\", \\"validity\\", \\"predicate_validity\\", \\"decomposition_ok\\")},\\n      hard=False)\\nfinding(\\"F-X4\\", \\"xoá các đường đi QUAN HỆ của giải thích (v4) có đổi thứ hạng video đúng?\\",\\n        f\\"{XAI_SUMM[\'n_queries\']} truy vấn: hạng đổi {XAI_SUMM[\'frac_rank_changed_expl\']:.0%} (xoá ngẫu nhiên {XAI_SUMM[\'frac_rank_changed_rand\']:.0%}); \\"\\n        f\\"Δ điểm đồ thị {XAI_SUMM[\'d_s_expl\']:+.4f} vs ngẫu nhiên {XAI_SUMM[\'d_s_rand\']:+.4f}\\",\\n        \\"điểm giảm đúng như phân rã nhưng quyết định xếp hạng KHÔNG đổi ⇒ quan hệ không phải nguyên nhân của thứ hạng\\"\\n        if XAI_SUMM[\\"frac_rank_changed_expl\\"] == 0 else \\"xoá quan hệ của giải thích có làm đổi thứ hạng ở một phần truy vấn\\")\\n_better = XR_SUMM[\\"frac_rank_worse_expl\\"] > XR_SUMM[\\"frac_rank_worse_rand\\"] and XR_SUMM[\\"d_final_expl\\"] < XR_SUMM[\\"d_final_rand\\"]\\nfinding(\\"F-X6\\", \\"xoá giải thích ĐẦY ĐỦ (contains + quan hệ) có làm video đúng tụt hạng nhiều hơn xoá ngẫu nhiên cùng số cạnh?\\",\\n        f\\"{XR_SUMM[\'n_queries\']} truy vấn: tụt hạng {XR_SUMM[\'frac_rank_worse_expl\']:.0%} vs {XR_SUMM[\'frac_rank_worse_rand\']:.0%}; \\"\\n        f\\"hạng TB {XR_SUMM[\'rank_full\']:.2f} → {XR_SUMM[\'rank_drop_expl\']:.2f} (ngẫu nhiên {XR_SUMM[\'rank_drop_rand\']:.2f}); \\"\\n        f\\"Δ điểm cuối {XR_SUMM[\'d_final_expl\']:+.3f} vs {XR_SUMM[\'d_final_rand\']:+.3f}; top-1 đổi {XR_SUMM[\'frac_top1_changed_expl\']:.0%}\\",\\n        \\"giải thích chỉ ra đúng phần đồ thị quyết định thứ hạng (mạnh hơn ngẫu nhiên)\\" if _better else\\n        \\"KHÔNG mạnh hơn xoá ngẫu nhiên ⇒ không kết luận được giải thích phản ánh nguyên nhân của thứ hạng\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — 5 THÀNH CÔNG + 5 THẤT BẠI (chọn theo luật, có đường đi thật + nguyên nhân) ==============================\\nmark_stage(\\"cases\\", requires=(\\"xai_rank\\",))\\nCASES, CASE_STATS = case_analysis_l4b(RUN_F, P_MAIN)\\nsave_json(dict(experiment_id=G3_ID, part=P_MAIN[\\"name\\"], alpha=RUN_F[\\"alpha\\"], stats=CASE_STATS, cases=CASES), RESULTS / \\"case_analysis.json\\")\\n(RESULTS / \\"case_analysis.md\\").write_text(cases_markdown(CASES, CASE_STATS, P_MAIN[\\"name\\"]))\\nfor c in CASES:\\n    print(f\\"[{c[\'kind\']}] hạng CLIP {c[\'rank_clip\']} → {c[\'rank_final\']} | {c[\'query\'][:70]} | nguyên nhân: {\', \'.join(c[\'causes\']) or \'—\'}\\")\\n    for x in c[\\"gold_explanation\\"][:3]:\\n        print(f\\"      • {x}\\")\\n_ns, _nf = sum(c[\\"kind\\"] == \\"THÀNH CÔNG\\" for c in CASES), sum(c[\\"kind\\"] == \\"THẤT BẠI\\" for c in CASES)\\ncheck(\\"K1\\", f\\"phân tích ≥ {CFG.n_cases} thành công + ≥ {CFG.n_cases} thất bại, chọn theo luật, có đường đi thật, thứ hạng và nguyên nhân\\",\\n      _ns >= CFG.n_cases and _nf >= CFG.n_cases,\\n      f\\"{_ns} thành công, {_nf} thất bại; nguyên nhân có mặt: {sorted({x for c in CASES for x in c[\'causes\']})}\\", hard=False)"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — DEMO: văn bản tự do → top-K video + điểm CLIP / đồ thị / cuối + giải thích ==============================\\nmark_stage(\\"demo\\", requires=(\\"cases\\",))\\nDEMO_OUT = []\\nfor _i, _t in enumerate(CFG.demo_queries, 1):\\n    DEMO_OUT.append(demo_query(_t, save_png=RESULTS / f\\"demo_{_i}.png\\"))\\nfor _kind in (\\"THÀNH CÔNG\\", \\"THẤT BẠI\\"):                     # một ca có đáp án mỗi loại (văn bản của tập đánh giá, chỉ hiển thị)\\n    c = next((c for c in CASES if c[\\"kind\\"] == _kind), None)\\n    if c is not None:\\n        _o = demo_query(c[\\"query\\"], save_png=RESULTS / f\\"demo_case_{\'ok\' if _kind == \'THÀNH CÔNG\' else \'fail\'}.png\\")\\n        _o.update(gold_video_id=c[\\"gold_video_id\\"], case_kind=_kind)\\n        DEMO_OUT.append(_o)\\nsave_json(DEMO_OUT, RESULTS / \\"demo_outputs.json\\")\\n_ok = all(len(o[\\"results\\"]) == CFG.demo_topk and all(r[\\"explanation\\"] or r[\\"note\\"] for r in o[\\"results\\"]) for o in DEMO_OUT)\\ncheck(\\"DM1\\", \\"demo: văn bản tự do → top-k video + điểm CLIP / đồ thị / cuối + giải thích (hoặc báo rõ không có đường đi hợp lệ)\\",\\n      _ok and len(DEMO_OUT) >= len(CFG.demo_queries), f\\"{len(DEMO_OUT)} truy vấn demo → demo_outputs.json, demo_*.png\\", hard=False)\\nprint(\\"\\\\nThử truy vấn khác: demo_query(\'a man is riding a horse\')  |  ô nhập tương tác: demo_widget()\\")"}, {"cell_type": "code", "execution_count": null, "id": "c060", "metadata": {}, "outputs": [], "source": "# ============================== XUẤT KẾT QUẢ THEO CÂU HỎI + MANIFEST ==============================\\nmark_stage(\\"export\\", requires=(\\"xai\\",))\\nPARTS = [k_ for k_ in (\\"explore\\", \\"confirm\\", \\"confirm_std\\") if k_ in EVALS]\\n\\n\\ndef result_table(ids, groups=(\\"all\\", \\"direct\\", \\"indirect\\")):\\n    \\"\\"\\"Một dòng / thí nghiệm: thiết kế, trạng thái, VAL, và metric trên từng tập đánh giá (thăm dò / xác nhận) theo nhóm.\\"\\"\\"\\n    rows = []\\n    label = {v: k for k, v in G_IDS.items()}\\n    for k in dict.fromkeys(ids):\\n        if k not in EXPS:\\n            continue\\n        e = EXPS[k]\\n        base = dict(experiment_id=k, control=label.get(k, \\"\\"), is_final=k == FINAL, stage=e[\\"stage\\"], status=e[\\"status\\"],\\n                    reason=e.get(\\"reason\\", \\"\\"), description=e[\\"description\\"], hypothesis=e.get(\\"hypothesis\\", \\"\\"),\\n                    design=json.dumps(e[\\"design\\"], ensure_ascii=False), n_seeds=len(e[\\"seeds\\"]),\\n                    edges=json.dumps(e.get(\\"edges\\", {})), n_params=e.get(\\"n_params\\"), runtime_s=e.get(\\"seconds\\"),\\n                    gpu_peak_mb=e.get(\\"gpu_peak_mb\\"), val_MRR=e[\\"val_MRR\\"][0], val_MRR_std=e[\\"val_MRR\\"][1], val_R1=e[\\"val_R1\\"][0])\\n        for pk in PARTS:\\n            sub = EVALS[pk][\\"rows\\"]\\n            sub = sub[sub.experiment_id == k]\\n            for grp in groups:\\n                r = sub[sub.group == grp]\\n                if len(r):\\n                    r = r.iloc[0]\\n                    tag = f\\"{pk}_{grp}\\"\\n                    base.update({f\\"{tag}_{x}\\": r[f\\"{x}_mean\\"] for x in (\\"R@1\\", \\"R@5\\", \\"R@10\\", \\"MRR\\", \\"MRR_video_macro\\")})\\n                    base[f\\"{tag}_MRR_std\\"] = r[\\"MRR_std\\"]\\n        rows.append(base)\\n    return pd.DataFrame(rows)\\n\\n\\nG_ORDER = [G_IDS[g] for g in (\\"G0\\", \\"G1\\", \\"G2\\", \\"G3\\", \\"G4\\", \\"G5\\", \\"G6\\", \\"G7\\", \\"G8\\", \\"G9\\", \\"G10\\", \\"G11\\", \\"V3REF\\") if g in G_IDS]\\nABL_DF = result_table(G_ORDER + [FINAL] + [k for k, e in EXPS.items() if e[\\"stage\\"] == \\"V1\\"],\\n                      (\\"all\\", \\"direct\\", \\"indirect\\", \\"pred_rare\\", \\"pred_common\\", \\"no_pred\\"))\\nABL_DF.to_csv(RESULTS / \\"graph_ablation_results.csv\\", index=False)\\n\\nMANIFEST = []\\nfor k, e in EXPS.items():\\n    MANIFEST.append(dict(experiment_id=k, control={v: g for g, v in G_IDS.items()}.get(k, \\"\\"), stage=e[\\"stage\\"], status=e[\\"status\\"],\\n                         reason=e.get(\\"reason\\"), description=e[\\"description\\"], hypothesis=e.get(\\"hypothesis\\", \\"\\"), design=e[\\"design\\"],\\n                         seeds=e[\\"seeds\\"], reuse=e.get(\\"reuse\\"), lam_fixed=e.get(\\"lam_fixed\\"),\\n                         checkpoints=[r.get(\\"checkpoint\\") for r in e[\\"runs\\"].values()], edges_by_type=e.get(\\"edges\\"),\\n                         n_nodes=e.get(\\"n_nodes\\"), n_params=e.get(\\"n_params\\"), alpha_lam=e.get(\\"alpha_lam\\"),\\n                         val=dict(MRR=e[\\"val_MRR\\"], R1=e[\\"val_R1\\"]), runtime_s=e.get(\\"seconds\\"), gpu_peak_mb=e.get(\\"gpu_peak_mb\\"),\\n                         eval={pk: {r_.group: dict(MRR=r_[\\"MRR_mean\\"], MRR_std=r_[\\"MRR_std\\"], R1=r_[\\"R@1_mean\\"])\\n                                    for _, r_ in EVALS[pk][\\"rows\\"][EVALS[pk][\\"rows\\"].experiment_id == k].iterrows()} for pk in PARTS},\\n                         per_seed={str(s): dict(val=r[\\"val\\"], alpha=r[\\"alpha\\"], lam=r[\\"lam\\"], seconds=r[\\"seconds\\"],\\n                                                final_loss=r[\\"hist\\"][-1] if r[\\"hist\\"] else None) for s, r in e[\\"runs\\"].items()}))\\nsave_json(dict(version=\\"level4b_v2\\", kg_status=KG_STATUS, tagger_probe=PT_SELECT, final=FINAL, benchmark_status=benchmark_status(), g_ids=G_IDS, rerank_k=CFG.rerank_k, eval_parts={k_: E[\\"info\\"] for k_, E in EVALS.items()},\\n               candidate_recall_val=CAND_RECALL_VAL, not_run=NOT_RUN, primary_comparisons=PRIMARY_COMPARISONS,\\n               experiments=MANIFEST, env=ENV_INFO), RESULTS / \\"experiment_manifest.json\\")\\nPROV = pd.DataFrame(PROVENANCE)\\nPROV.to_csv(RESULTS / \\"provenance.csv\\", index=False)\\nACCESS = CAPS.access_table()\\nACCESS.to_csv(RESULTS / \\"caption_access_log.csv\\", index=False)\\nbad_access = ACCESS[ACCESS.split.isin([\\"test\\", \\"ext\\"]) & ~ACCESS.purpose.isin([\\"encode\\", \\"query_input\\", \\"eval\\", \\"display\\", \\"report\\"])]\\ncheck(\\"C8\\", \\"[12.8] nhật ký: caption TEST / tập ngoài chỉ dùng cho encode / query_input / eval / report\\", bad_access.empty,\\n      {f\\"{r.purpose}:{r.split}\\": int(r.n) for r in ACCESS.itertuples()})\\nbad_prov = PROV[((PROV.kind == \\"learned\\") & PROV.caption_splits.str.contains(\\"val|test|ext\\"))\\n                | ((PROV.kind == \\"selected\\") & PROV.caption_splits.str.contains(\\"test|ext\\"))]\\ncheck(\\"C9\\", \\"[12.8] provenance: artifact học được chỉ từ caption TRAIN; lựa chọn chỉ từ VAL\\", bad_prov.empty, f\\"{len(PROV)} artifact\\")\\nsave_json(TEST_GATE.log, RESULTS / \\"test_access_log.json\\")\\nfirst = min((x[\\"t\\"] for x in TEST_GATE.log), default=None)\\ncheck(\\"D4\\", \\"mọi lần chạm test MSVD / tập ngoài sau thời điểm đóng băng\\", first is None or first >= TEST_GATE.frozen_at,\\n      f\\"{len(TEST_GATE.log)} lần ({dict(Counter(x[\'split\'] for x in TEST_GATE.log))})\\")\\nr_ = subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"freeze\\"], capture_output=True, text=True)\\n(RESULTS / \\"pip_freeze.txt\\").write_text(r_.stdout if r_.returncode == 0 else r_.stderr[-2000:])\\nsave_json(dict(ENV_INFO, cfg=cfg_snapshot(), stages=STAGES), RESULTS / \\"env_versions.json\\")"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== BÁO CÁO AUDIT + KẾT LUẬN v4 (tự sinh từ số liệu) ==============================\\nmark_stage(\\"report\\", requires=(\\"export\\",))\\nofficial = CFG.dataset == \\"msvd\\" and not CFG.dev_mode\\n_by = {c[\\"id\\"]: c for c in CHECKS}\\n\\n\\ndef _st(*ids):\\n    rows = [_by[i] for i in ids if i in _by]\\n    if not rows:\\n        return \\"NOT RUN\\", \\"không chạy\\"\\n    st = \\"FAIL\\" if any(r[\\"status\\"] == \\"FAIL\\" for r in rows) else (\\"NOT RUN\\" if any(r[\\"status\\"] == \\"NOT RUN\\" for r in rows) else \\"PASS\\")\\n    return st, \\" | \\".join(f\\"{r[\'id\']}={r[\'status\']}\\" for r in rows)\\n\\n\\ndef _verdict(lo, hi):\\n    return \\"TỐT HƠN có ý nghĩa\\" if lo > 0 else (\\"KÉM HƠN có ý nghĩa\\" if hi < 0 else \\"không khác biệt có ý nghĩa\\")\\n\\n\\ndef _conf(hyp, part_key):\\n    if part_key not in EVALS:\\n        return \\"NOT RUN\\" + (f\\" ({EXT_STATUS})\\" if part_key.startswith(\\"confirm\\") else \\"\\")\\n    r = CONF_T[(CONF_T.hypothesis == hyp) & (CONF_T.part == EVALS[part_key][\\"part\\"][\\"name\\"])]\\n    if r.empty or r.iloc[0].get(\\"status\\") == \\"NOT RUN\\":\\n        return \\"NOT RUN\\" + (f\\" — {r.iloc[0].get(\'note\', \'\')}\\" if len(r) else \\"\\")\\n    r = r.iloc[0]\\n    return (f\\"{r.a} vs {r.b}: ΔMRR {r[\'diff\']:+.4f} (CI95 theo video [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}], p Holm {r.p_holm:.3f}) \\"\\n            f\\"⇒ {_verdict(r.ci_lo, r.ci_hi)}{\' — ỦNG HỘ giả thuyết\' if r.supported else \'\'}\\")\\n\\n\\ndef _val(a, b):\\n    if not (EXPS.get(a, {}).get(\\"runs\\") or a == \\"G0_clip_only\\") or not (EXPS.get(b, {}).get(\\"runs\\") or b == \\"G0_clip_only\\"):\\n        return \\"NOT RUN\\"\\n    bs = cluster_bootstrap_diff(EXPS[a][\\"rr_val\\"], EXPS[b][\\"rr_val\\"], PV[\\"gold\\"])\\n    return f\\"{a} vs {b} (VAL): ΔMRR {bs[\'diff\']:+.4f} [{bs[\'ci_lo\']:+.4f}, {bs[\'ci_hi\']:+.4f}]\\"\\n\\n\\nCK = \\"confirm\\" if \\"confirm\\" in EVALS else None\\nEX = \\"explore\\"\\nL = []\\nif not official:\\n    L.append(f\\"⚠ KHÔNG phải lần chạy chính thức (dataset={CFG.dataset}, dev_mode={CFG.dev_mode}) — số liệu chỉ để kiểm tra mã.\\")\\nL.append(f\\"⚠ Khai báo: test MSVD đã được nhìn ở v1–v4 và Level 4(b) v1 ⇒ số test MSVD ở đây là THĂM DÒ (lần nhìn thứ 6). Xác nhận độc lập: \\"\\n         f\\"{EXT_STATUS} ({EXT_REPORT.get(\'label\', \'—\')}).\\")\\nL.append(f\\"Mô hình v4 = QC-RGAT ({CFG.v4_dim} chiều, {CURRENT[\'v4_layers\']} lớp), đồ thị = cấu hình cuối của v3 \\"\\n         f\\"({json.dumps({k: V3_FINAL[k] for k in (\'taxonomy\', \'mask\', \'rel_policy\', \'contains\')}, ensure_ascii=False)}); k = {CFG.rerank_k}; \\"\\n         f\\"chọn số lớp trên VAL: {FINAL}.\\")\\nL.append(f\\"Chất lượng quan hệ — {AUTO_STATUS}\\")\\nL.append(f\\"v2: tagger học {PT_SELECT and round(PT_SELECT[\'val_F1\'], 3)} vs zero-shot {TAG_SELECT2[\'val_F1\']:.3f} (VAL F1); \\"\\n         f\\"G3 dùng nhãn \'{G3_DESIGN.get(\'tags\', \'base\')}\'; ConceptNet: {KG_STATUS}\\")\\nHYPS = ((\\"H1_graph_vs_clip\\", \\"v4 (G3) vs CLIP-only (G0)\\"), (\\"H2_relations_vs_contains_only\\", \\"Quan hệ vs chỉ contains (G3 vs G1)\\"),\\n        (\\"H3_predicate_vs_family_only\\", \\"Khớp vị từ vs không vị từ (G3 vs G2)\\"),\\n        (\\"H4_true_vs_shuffled_predicates\\", \\"Vị từ đúng cạnh vs hoán vị (G3 vs G4)\\"),\\n        (\\"H5_true_vs_rewired_topology\\", \\"Topology thật vs rewiring có ràng buộc (G3 vs G5)\\"),\\n        (\\"H6_query_conditioned_vs_static\\", \\"Attention theo truy vấn vs tĩnh (G3 vs G6)\\"),\\n        (\\"H7_v4_vs_v3\\", \\"QC-RGAT vs GraphSAGE có kiểu cạnh (G3 vs R3)\\"),\\n        (\\"H8_role_edges\\", \\"Họ cạnh vai agent_of / object_of (G3 vs G7 huấn luyện lại)\\"),\\n        (\\"H9_svo_prep_edges\\", \\"Họ cạnh svo / prep (G3 vs G8 huấn luyện lại)\\"),\\n        (\\"H10_kg_vs_contains_only\\", \\"ConceptNet: G9 (đồ thị + KG) vs chỉ-contains G1\\"),\\n        (\\"H11_kg_true_vs_rewired\\", \\"ConceptNet: topology thật vs rewired (G9 vs G10)\\"),\\n        (\\"H12_kg_adds_to_graph\\", \\"ConceptNet thêm vào đồ thị hiện tại (G9 vs G3)\\"),\\n        (\\"H13_learned_vs_zeroshot_tags\\", \\"Nhãn video học từ caption train vs zero-shot (G3 vs G11)\\"))\\nfor hyp, txt in HYPS:\\n    a, b = PRIMARY_COMPARISONS[[h for h, _, _ in PRIMARY_COMPARISONS].index(hyp)][1:]\\n    L.append(f\\"{hyp} — {txt}:\\")\\n    L.append(f\\"     VAL (phát triển): {_val(resolve(a), resolve(b))}\\")\\n    L.append(f\\"     {\'test MSVD (THĂM DÒ)\' if not CFG.dev_mode else \'VAL lặp lại (DEV)\'}: {_conf(hyp, EX)}\\")\\n    L.append(f\\"     tập ngoài {\'(XÁC NHẬN)\' if CK and EVALS[CK][\'info\'][\'kind\'] == \'confirmatory\' else \'(DEV)\' if CK else \'\'}: \\"\\n             f\\"{_conf(hyp, CK) if CK else \'NOT RUN — \' + EXT_STATUS}\\")\\nL.append(f\\"Đối chứng (dưới cấu hình G3): rewiring hợp lệ: {RANDOM_VALID_G3}; hoán vị vị từ hợp lệ: {SHUFFLE_VALID_G3}. \\"\\n         \\"Đối chứng không hợp lệ ⇒ KHÔNG rút kết luận tương ứng.\\")\\nfor s_ in RA_SUMM:\\n    L.append(f\\"Phân tích quan hệ ({s_[\'split\']}): bỏ mọi cạnh quan hệ lúc suy luận: MRR {s_[\'base_MRR\']:.4f} → \\"\\n             f\\"{s_[\'all_relations_knockout_MRR\']:.4f}; attention cạnh có vị từ khớp truy vấn {s_[\'att_pred_match\']:.3f} vs không khớp \\"\\n             f\\"{s_[\'att_pred_nomatch\']:.3f}.\\")\\nL.append(f\\"Giải thích ({XAI_SUMM[\'experiment_id\']}, {XAI_SUMM[\'part\']}, {XAI_SUMM[\'n_queries\']} truy vấn): hợp lệ {XAI_SUMM[\'validity\']:.2f}, \\"\\n         f\\"vị từ đúng cạnh {XAI_SUMM[\'predicate_validity\']:.2f}, Σ số hạng = điểm {XAI_SUMM[\'decomposition_ok\']:.2f}, xoá cạnh khớp phân rã \\"\\n         f\\"{XAI_SUMM[\'deletion_matches_decomposition\']:.2f}; Δ điểm khi xoá giải thích {XAI_SUMM[\'d_s_expl\']:+.4f} vs ngẫu nhiên \\"\\n         f\\"{XAI_SUMM[\'d_s_rand\']:+.4f}; hạng đổi {XAI_SUMM[\'frac_rank_changed_expl\']:.0%} (ngẫu nhiên {XAI_SUMM[\'frac_rank_changed_rand\']:.0%}); \\"\\n         f\\"phần điểm từ quan hệ {XAI_SUMM[\'share_relation\']:.1%}; Jaccard theo seed {XAI_SUMM[\'seed_sensitivity_mean_jaccard\']:.2f}.\\")\\nconf_sup = {r.hypothesis: bool(r.supported) for r in CONF_T.itertuples()\\n            if CK and r.part == EVALS[CK][\\"part\\"][\\"name\\"] and getattr(r, \\"status\\", \\"\\") == \\"CONFIRMATORY\\"}\\nREL_KEYS = (\\"H2_relations_vs_contains_only\\", \\"H4_true_vs_shuffled_predicates\\", \\"H5_true_vs_rewired_topology\\")\\nif conf_sup and all(conf_sup.get(h) for h in REL_KEYS) and RANDOM_VALID_G3 and SHUFFLE_VALID_G3:\\n    VERDICT = (\\"Trên tập xác nhận độc lập, v4 vượt có ý nghĩa (Holm) cả đối chứng chỉ-contains, vị từ hoán vị và topology rewired ⇒ \\"\\n               \\"quan hệ (đúng cạnh, đúng vị từ) có giá trị thật cho xếp hạng.\\")\\nelif conf_sup:\\n    VERDICT = (\\"Quan hệ CHƯA được xác nhận đầy đủ: trên tập xác nhận, v4 không vượt có ý nghĩa mọi đối chứng quan hệ \\"\\n               f\\"(ủng hộ: {[h for h, v in conf_sup.items() if v] or \'—\'}). Cơ chế dùng quan hệ có thật (P/Q/X-test) nhưng tác động lên MRR \\"\\n               \\"không đủ để kết luận.\\")\\nelse:\\n    VERDICT = \\"Chưa có kết luận xác nhận: tập xác nhận độc lập NOT RUN hoặc DEV. Số VAL / test MSVD chỉ là phát triển / thăm dò.\\"\\nL.append(\\"KẾT LUẬN: \\" + VERDICT)\\n_tf1 = \\", \\".join(f\\"{k_} {v_[\'F1\']:.3f}\\" for k_, v_ in TAGGER_REPORT.items())\\nL.append(f\\"Giới hạn: nhãn video từ CLIP zero-shot (F1 {_tf1}); quan hệ là thống kê cấp LỚP từ caption train (không có quan hệ cấp video \\"\\n         \\"cho video mới); parser quan hệ heuristic; val chỉ 100 video; PASS của kiểm thử phần mềm không xác nhận giả thuyết khoa học.\\")\\nCONCLUSION = \\"\\\\n\\".join(L)\\n(RESULTS / \\"graph_conclusion.md\\").write_text(\\"# Kết luận đồ thị — Level 4(b) (tự sinh từ số liệu)\\\\n\\\\n\\" + \\"\\\\n\\".join(f\\"- {x}\\" for x in L) + \\"\\\\n\\")\\n\\n_csv_conf = pd.read_csv(RESULTS / \\"graph_confirmatory_tests.csv\\")\\n_csv_abl = pd.read_csv(RESULTS / \\"graph_ablation_results.csv\\")\\n_m1 = CONF_T.dropna(subset=[\\"diff\\"]) if \\"diff\\" in CONF_T else CONF_T.iloc[:0]\\n_m2 = _csv_conf.dropna(subset=[\\"diff\\"]) if \\"diff\\" in _csv_conf else _csv_conf.iloc[:0]\\nok_conf = len(_m1) == len(_m2) and np.allclose(_m1[\\"diff\\"].values, _m2[\\"diff\\"].values) and \\\\\\n    np.allclose(_m1[\\"p_holm\\"].values, _m2[\\"p_holm\\"].values)\\nok_abl = all(abs(float(_csv_abl.loc[_csv_abl.experiment_id == k, \\"val_MRR\\"].iloc[0]) - EXPS[k][\\"val_MRR\\"][0]) < 1e-9\\n             for k in _csv_abl.experiment_id if EXPS[k][\\"status\\"] == \\"DONE\\")\\nok_text = all(f\\"{r[\'diff\']:+.4f}\\" in CONCLUSION for _, r in _m1.iterrows()\\n              if r[\\"hypothesis\\"] in {h for h, _, _ in PRIMARY_COMPARISONS} and r[\\"part\\"] in {EVALS[k_][\\"part\\"][\\"name\\"] for k_ in (EX, CK) if k_})\\ncheck(\\"R1\\", \\"[12.10] kết quả trong báo cáo / kết luận khớp các file CSV đầu ra (đọc lại từ đĩa)\\", ok_conf and ok_abl and ok_text,\\n      dict(confirmatory=ok_conf, ablation_val=ok_abl, conclusion_text=ok_text))\\n\\nREQUIRED = [\\"graph_schema_v3.json\\", \\"graph_edges_v3.csv\\", \\"relation_quality_audit.csv\\", \\"graph_randomization_audit.csv\\",\\n            \\"graph_ablation_results.csv\\", \\"graph_relation_analysis.csv\\", \\"graph_xai_fidelity.csv\\", \\"frozen_config.json\\",\\n            \\"experiment_manifest.json\\", \\"graph_conclusion.md\\"]\\n\\n\\ndef _nonempty(p):\\n    if not p.exists() or p.stat().st_size == 0:\\n        return False\\n    if p.suffix == \\".csv\\":\\n        try:\\n            return len(pd.read_csv(p)) > 0\\n        except Exception:\\n            return False\\n    return True\\n\\n\\nfiles = {f: _nonempty(RESULTS / f) for f in REQUIRED}\\ncriteria = [\\n    (\\"1\\", \\"Quan hệ được DÙNG có điều kiện theo truy vấn (vị từ, vai, loại cạnh tác động attention)\\", *_st(\\"Q1\\", \\"Q2\\")),\\n    (\\"2\\", \\"Điểm = Σ số hạng giải thích (phân rã chính xác); xoá cạnh ⇒ đúng phần đóng góp\\", *_st(\\"Q3\\", \\"X1\\", \\"X3\\")),\\n    (\\"3\\", \\"Có đối chứng tách quan hệ / vị từ / topology / điều kiện truy vấn (G1, G2, G4, G5, G6 + audit)\\", *_st(\\"G6\\", \\"G7\\")),\\n    (\\"4\\", \\"Ablation công bằng (inventory cố định, cùng ứng viên, cùng giao thức huấn luyện)\\", *_st(\\"G1\\", \\"D2\\")),\\n    (\\"5\\", \\"Không dùng test đã xem để chọn mô hình\\", *_st(\\"D1\\", \\"D4\\", \\"C3\\")),\\n    (\\"6\\", \\"Đủ bằng chứng xác định tốt hơn / tương đương / kém hơn (tập xác nhận độc lập, Holm)\\", *_st(\\"D3\\")),\\n    (\\"7\\", \\"Mọi kết luận có nguồn số liệu kiểm chứng\\", *_st(\\"R1\\")),\\n]\\nmd = [\\"# Audit đồ thị — Level 4(b): QC-RGAT + đối chứng cho text→video (MSVD → MSR-VTT)\\", \\"\\",\\n      f\\"{time.strftime(\'%Y-%m-%d %H:%M:%S\')} | dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | Kaggle={RUN_ON_KAGGLE} | \\"\\n      f\\"{ENV_INFO.get(\'gpu\', DEVICE)} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | tập ngoài: {EXT_STATUS}\\", \\"\\",\\n      \\"## Tiêu chí nghiệm thu\\", \\"\\", \\"| # | Tiêu chí | Trạng thái | Bằng chứng |\\", \\"|---|---|---|---|\\"]\\nmd += [f\\"| {a} | {b} | **{c}** | {d} |\\" for a, b, c, d in criteria]\\nmd += [\\"\\", \\"## File bàn giao\\", \\"\\", \\"| File | Có dữ liệu |\\", \\"|---|---|\\"] + [f\\"| {f} | {\'PASS\' if ok else \'FAIL\'} |\\" for f, ok in files.items()]\\nmd += [\\"\\", \\"## Quyết định theo stage (chỉ VAL; incumbent = phương án đơn giản hơn)\\", \\"\\",\\n       \\"| stage | incumbent | ứng viên | Δ val MRR | p Holm | chấp nhận |\\", \\"|---|---|---|---|---|---|\\"]\\nmd += [f\\"| {d[\'stage\']} | {d[\'incumbent\']} | {d[\'candidate\']} | {d[\'delta_val_MRR\']:+.4f} | {d[\'p_holm\']:.3f} | {d[\'adopted\']} |\\"\\n       if d.get(\\"value\\") is not None else f\\"| {d[\'stage\']} | {d[\'incumbent\']} | {d[\'candidate\']} | NOT RUN | — | False |\\" for d in DECISIONS]\\nmd += [\\"\\", \\"## Thí nghiệm NOT RUN\\", \\"\\"] + ([f\\"- {n[\'stage\']} {n.get(\'experiment_id\', \'\')}: {n[\'reason\']}\\" for n in NOT_RUN] or [\\"- (không có)\\"])\\nmd += [\\"\\", \\"## Kết luận\\", \\"\\"] + [f\\"- {x}\\" for x in L]\\nmd += [\\"\\", \\"## Mọi kiểm tra\\", \\"\\", \\"| id | kiểm tra | trạng thái |\\", \\"|---|---|---|\\"] + [f\\"| {c[\'id\']} | {c[\'check\']} | {c[\'status\']} |\\" for c in CHECKS]\\n(RESULTS / \\"graph_audit_report.md\\").write_text(\\"\\\\n\\".join(md) + \\"\\\\n\\")\\ndisplay(pd.DataFrame(criteria, columns=[\\"#\\", \\"tiêu chí\\", \\"trạng thái\\", \\"bằng chứng\\"]))\\nprint(\\"\\\\n\\" + CONCLUSION)\\nprint(f\\"\\\\nFile bàn giao có dữ liệu: {sum(files.values())}/{len(files)} | kiểm tra FAIL: {sum(c[\'status\'] == \'FAIL\' for c in CHECKS)} \\"\\n      f\\"→ {RESULTS / \'graph_audit_report.md\'}\\")\\n", "id": "c061"}, {"cell_type": "code", "metadata": {}, "execution_count": null, "outputs": [], "source": "# ============================== Level 4(b) — KIỂM KÊ DỮ LIỆU · AUDIT RÒ RỈ · NGHIỆM THU · KẾT LUẬN · MANIFEST (tự sinh từ lần chạy này) ==============================\\nmark_stage(\\"level4b_report\\", requires=(\\"report\\", \\"demo\\"))\\nBENCH = benchmark_status()\\n_g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]\\nGRAPH_INV = graph_inventory(_g3)\\n_cache_dirs = [CFG.out_dir / \\"cache\\"] + [Path(d) for d in CFG.extra_cache_dirs] + \\\\\\n    ([p for p in Path(\\"/kaggle/input\\").glob(\\"*/**/cache\\") if p.is_dir()] if RUN_ON_KAGGLE else [])\\n_caches = [dict(file=p.name, dir=str(p.parent), mb=round(p.stat().st_size / 2 ** 20, 1), used_by_this_run=CFG.clip_model in p.name and\\n                CFG.clip_weights in p.name) for d in _cache_dirs if d.exists() for p in sorted(d.glob(\\"*.npz\\"))]\\n_ck = sorted((CFG.out_dir / \\"checkpoints\\").glob(\\"*\\")) if (CFG.out_dir / \\"checkpoints\\").exists() else []\\nDATA_INVENTORY = dict(\\n    created=time.strftime(\\"%Y-%m-%d %H:%M:%S\\"), official_run=CFG.dataset == \\"msvd\\" and not CFG.dev_mode, dev_mode=CFG.dev_mode,\\n    datasets=[dict(key=\\"msvd\\", **DATA_SOURCES[\\"msvd\\"], videos_loaded=N_MSVD, video_source=DATA_REPORT.get(\\"source_label\\"),\\n                   splits={s: DATA_REPORT[\\"splits\\"][s] for s in DATA_REPORT[\\"splits\\"]}),\\n              dict(key=\\"msrvtt\\", **DATA_SOURCES[\\"msrvtt\\"], status=EXT_STATUS,\\n                   report={k_: v_ for k_, v_ in EXT_REPORT.items() if k_ != \\"missing_examples\\"})],\\n    loaded_counts_by_split=split_counts(), benchmark_status=BENCH, benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,\\n    split_manifest=dict(file=\\"split_manifest.json\\", msvd_sha1=SPLIT[\\"sha1\\"], ext_sha1=SPLIT.get(\\"ext_sha1\\"),\\n                        rule=\\"split CHÍNH THỨC của MSVD theo video; tập ngoài theo luật seed cố định (load_msrvtt)\\"),\\n    features=dict(clip=f\\"{CFG.clip_model}/{CFG.clip_weights} (đóng băng)\\", frames_per_video=CFG.n_frames_max,\\n                  frame_policy=CFG.frame_policy, pooling_selected_on_val=POOL_SEL, video_dim=int(VID_FEAT.shape[1]),\\n                  n_video_embeddings=int(VID_FEAT.shape[0]), n_text_embeddings=int(TXT_FEAT.shape[0]), caches=_caches),\\n    tagger=dict(selected_on_val=TAG_SELECT2, report=TAGGER_REPORT),\\n    graph=dict(design=G3_DESIGN, **GRAPH_INV), checkpoints=dict(n_files=len(_ck), mb=round(sum(p.stat().st_size for p in _ck) / 2 ** 20, 1),\\n                                                               names=[p.name for p in _ck][:60]),\\n    environment=ENV_INFO)\\nsave_json(DATA_INVENTORY, RESULTS / \\"data_inventory.json\\")\\ncheck(\\"I1\\", \\"kiểm kê: số video / caption theo split khớp dữ liệu đã nạp; manifest split có sha1; cache có metadata mô hình\\",\\n      all(DATA_INVENTORY[\\"loaded_counts_by_split\\"][s][\\"videos\\"] == len(SPLIT[s]) for s in DATA_INVENTORY[\\"loaded_counts_by_split\\"] if s in SPLIT)\\n      and bool(SPLIT[\\"sha1\\"]), {s: (v[\\"videos\\"], v[\\"captions\\"]) for s, v in DATA_INVENTORY[\\"loaded_counts_by_split\\"].items()}, hard=False)\\ncheck(\\"I2\\", \\"đồ thị đang chạy: không self-loop, không cạnh trùng (cùng nguồn, đích, loại); có thống kê bậc và node cô lập\\",\\n      GRAPH_INV[\\"self_loops\\"] == 0 and GRAPH_INV[\\"duplicate_edges\\"] == 0,\\n      {k_: GRAPH_INV[k_] for k_ in (\\"n_edges\\", \\"self_loops\\", \\"duplicate_edges\\", \\"isolated_non_video_nodes\\", \\"videos_without_tag\\")}, hard=False)\\n\\n# ---------------- data_leakage_audit.md ----------------\\n_by = {c[\\"id\\"]: c for c in CHECKS}\\n_LEAK = [(\\"A5\\", \\"split theo video, không video nào ở hai split\\"), (\\"C7\\", \\"split MSVD tái lập được\\"),\\n         (\\"C1\\", \\"canary chỉ có ở caption test / tập ngoài KHÔNG lọt vào từ vựng, quan hệ, vị từ\\"), (\\"C1b\\", \\"đối chứng dương của canary\\"),\\n         (\\"C2\\", \\"xáo caption test / tập ngoài ⇒ mọi artifact học được GIỐNG HỆT\\"), (\\"C3\\", \\"hàng rào chặn mở test / dùng caption sai mục đích\\"),\\n         (\\"C8\\", \\"nhật ký: caption test / tập ngoài chỉ dùng cho encode / query_input / eval / report\\"),\\n         (\\"C9\\", \\"provenance: học chỉ từ train, chọn chỉ từ val\\"), (\\"D1\\", \\"mọi lựa chọn trước đóng băng, chỉ dựa trên val\\"),\\n         (\\"D4\\", \\"mọi lần chạm test / tập ngoài sau đóng băng\\"), (\\"A6\\", \\"tập ngoài rời các tập đã dùng\\"), (\\"QS1\\", \\"bộ truy vấn dựng trước khi chấm mô hình\\")]\\n_acc = CAPS.access_table()\\nL = [\\"# Audit rò rỉ dữ liệu — Level 4(b) (tự sinh)\\", \\"\\",\\n     f\\"{DATA_INVENTORY[\'created\']} | chạy chính thức: {DATA_INVENTORY[\'official_run\']} | tập ngoài: {EXT_STATUS}\\", \\"\\",\\n     \\"## Trạng thái từng benchmark\\", \\"\\", \\"| Tập | Đã nhìn trước | Trạng thái |\\", \\"|---|---|---|\\"]\\nL += [f\\"| {k_} | {v_.get(\'looks_before\', \'—\')} | **{v_[\'status\']}** |\\" for k_, v_ in BENCH.items()]\\nL += [\\"\\", \\"## Kiểm tra chống rò rỉ\\", \\"\\", \\"| id | nội dung | trạng thái | bằng chứng |\\", \\"|---|---|---|---|\\"]\\nL += [f\\"| {i} | {d} | {_by[i][\'status\'] if i in _by else \'NOT RUN\'} | {(_by[i][\'evidence\'][:220] if i in _by else \'\').replace(\'|\', \'/\')} |\\" for i, d in _LEAK]\\nL += [\\"\\", \\"## Nhật ký dùng caption (mục đích × split)\\", \\"\\", \\"| mục đích | split | số lần |\\", \\"|---|---|---|\\"]\\nL += [f\\"| {r.purpose} | {r.split} | {int(r.n)} |\\" for r in _acc.itertuples()]\\nL += [\\"\\", \\"## Chất lượng ngữ nghĩa của cạnh\\", \\"\\",\\n      f\\"- Kiểm chứng tự động (parser độc lập, SILVER — không phải precision): {AUTO_STATUS}\\",\\n      f\\"- Nhãn người chấm: {REL_QUALITY_STATUS}\\",\\n      \\"- ⇒ Chất lượng ngữ nghĩa của cạnh quan hệ **CHƯA được xác nhận bởi người**. Tỉ lệ đồng thuận giữa hai parser không phải precision thật.\\", \\"\\",\\n      \\"## Nguồn và giấy phép\\", \\"\\"] + [f\\"- **{v_[\'name\']}** ({v_[\'paper\']}) — `{v_[\'repo\']}@{v_[\'revision\'][:8]}` — {v_[\'license\']} — vai trò: {v_[\'role\']}\\"\\n                                     for v_ in DATA_SOURCES.values()]\\n(RESULTS / \\"data_leakage_audit.md\\").write_text(\\"\\\\n\\".join(L) + \\"\\\\n\\")\\n\\n# ---------------- audit_report.md: ma trận nghiệm thu (mục 12 của master instruction) ----------------\\ndef _req_st(*ids):\\n    st, ev = _st(*ids)\\n    return st, ev\\n\\n\\ndef _done(*gids):\\n    ks = [G_IDS.get(g_, g_) for g_ in gids]\\n    ok = all(EXPS.get(k_, {}).get(\\"status\\") == \\"DONE\\" and (EXPS[k_].get(\\"runs\\") or k_ == \\"G0_clip_only\\") for k_ in ks)   # CLIP-only không huấn luyện\\n    return (\\"PASS\\" if ok else \\"NOT RUN\\"), \\", \\".join(f\\"{k_}={EXPS.get(k_, {}).get(\'status\', \'không có\')}\\" for k_ in ks)\\n\\n\\ndef _file(*names):\\n    ok = all(_nonempty(RESULTS / n) for n in names)\\n    return (\\"PASS\\" if ok else \\"FAIL\\"), \\", \\".join(names)\\n\\n\\ndef _all(*pairs):\\n    sts = [p[0] for p in pairs]\\n    st = \\"FAIL\\" if \\"FAIL\\" in sts else (\\"NOT RUN\\" if \\"NOT RUN\\" in sts else \\"PASS\\")\\n    return st, \\" ; \\".join(p[1] for p in pairs)\\n\\n\\n_n_seeds_ok = (\\"PASS\\" if len(CFG.seeds) >= 3 else \\"FAIL\\", f\\"{len(CFG.seeds)} seed {CFG.seeds}\\")\\n_conf_ok = (\\"PASS\\" if \\"confirm\\" in EVALS and EVALS[\\"confirm\\"][\\"info\\"][\\"kind\\"] == \\"confirmatory\\" else \\"NOT RUN\\",\\n            f\\"tập ngoài: {EXT_STATUS}; kind={EVALS[\'confirm\'][\'info\'][\'kind\'] if \'confirm\' in EVALS else \'—\'}\\")\\nREQ = [\\n    (\\"A1\\", \\"Baseline CLIP đóng băng\\", *_all(_req_st(\\"B0\\"), _done(\\"G0_clip_only\\"))),\\n    (\\"A2\\", \\"Đồ thị từ nguồn hợp lệ (caption TRAIN + CLIP zero-shot trên frame), đúng schema\\", *_req_st(\\"G2\\", \\"C1\\", \\"C9\\")),\\n    (\\"A3\\", \\"Ít nhất một GNN đã học (GraphSAGE có kiểu cạnh = R3; QC-RGAT = G3)\\", *_done(\\"V3REF\\", \\"G3\\")),\\n    (\\"A4\\", \\"CLIP-only và CLIP + Graph so sánh trên cùng split, cùng ứng viên\\", *_all(_req_st(\\"D2\\"), _done(\\"G0_clip_only\\", \\"G3\\"))),\\n    (\\"A5\\", \\"Ablation quan hệ (G1, G7, G8 huấn luyện lại) + đồ thị ngẫu nhiên hợp lệ (G4, G5)\\", *_all(_done(\\"G1\\", \\"G4\\", \\"G5\\", \\"G7\\", \\"G8\\"), _req_st(\\"G6\\", \\"G7\\", \\"AB\\"))),\\n    (\\"A7\\", \\"v2: tri thức ngoài ConceptNet (G9) + đối chứng rewired (G10) hợp lệ\\", *_all(_done(\\"G9\\", \\"G10\\"), _req_st(\\"KG1\\"))),\\n    (\\"A8\\", \\"v2: tagger học từ caption TRAIN được đánh giá trên VAL (V2) và đối chứng zero-shot\\", \\"PASS\\" if PT_SELECT is not None else \\"NOT RUN\\", f\\"VAL F1 học {PT_SELECT and round(PT_SELECT[\'val_F1\'], 3)} vs zero-shot {TAG_SELECT2[\'val_F1\']:.3f}\\"),\\n    (\\"A6\\", \\"R@1, R@5, R@10, MRR; ≥ 3 seed cho thí nghiệm chính\\", *_all(_n_seeds_ok, _req_st(\\"D3\\"))),\\n    (\\"B1\\", \\"Schema node / cạnh rõ ràng\\", *_file(\\"graph_schema_v3.json\\")),\\n    (\\"B2\\", \\"Nguồn gốc cạnh được ghi (bảng cạnh + provenance)\\", *_file(\\"graph_edges_v3.csv\\", \\"provenance.csv\\")),\\n    (\\"B3\\", \\"Không rò rỉ test\\", *_req_st(\\"A5\\", \\"C1\\", \\"C1b\\", \\"C2\\", \\"C3\\", \\"C8\\", \\"C9\\", \\"D1\\", \\"D4\\")),\\n    (\\"B4\\", \\"Audit cấu trúc (inventory cố định, schema, trọng số, self-loop / trùng / bậc)\\", *_req_st(\\"G1\\", \\"G2\\", \\"G4\\", \\"I2\\")),\\n    (\\"B5\\", \\"Đối chứng ngẫu nhiên đúng thuộc tính bảo toàn đã công bố\\", *_req_st(\\"G6\\", \\"G7\\")),\\n    (\\"B6\\", \\"Chất lượng ngữ nghĩa của cạnh được người xác minh\\", \\"NOT RUN\\",\\n     \\"chưa có nhãn người chấm ⇒ chưa xác nhận; chỉ có kiểm chứng tự động SILVER (spaCy)\\"),\\n    (\\"C1\\", \\"Pipeline text-to-video thật (giải mã video, lấy mẫu frame, gộp, gallery)\\", *_req_st(\\"A3\\", \\"A4\\", \\"B1\\", \\"B3\\", \\"B6\\", \\"B7\\", \\"B8\\")),\\n    (\\"C2\\", \\"Encoder tiền huấn luyện đóng băng + lấy mẫu frame\\", *_req_st(\\"B0\\", \\"B6\\")),\\n    (\\"C3\\", \\"Gallery, bộ truy vấn và metric xác định rõ (gồm bộ ≥ 20 truy vấn / nhóm)\\", *_req_st(\\"B8\\", \\"QS1\\")),\\n    (\\"C4\\", \\"Xếp hạng lại bằng đồ thị + baseline, đánh giá trên tập CHƯA nhìn\\", *_all(_req_st(\\"D3\\"), _conf_ok)),\\n    (\\"C5\\", \\"Giải thích lấy từ chính cơ chế xếp hạng\\", *_req_st(\\"Q3\\", \\"X1\\", \\"X3\\", \\"X5\\")),\\n    (\\"C6\\", \\"Phân tích thành công / thất bại và giới hạn\\", *_req_st(\\"K1\\")),\\n    (\\"D1\\", \\"Đường đi / quan hệ giải thích là thành phần thật của phép tính\\", *_req_st(\\"Q3\\", \\"X3\\")),\\n    (\\"D2\\", \\"Validity, sparsity, fidelity (điểm + thứ hạng) trên ≥ 10 truy vấn\\", *_all(_req_st(\\"X1\\", \\"X5\\"), _file(\\"xai_rank_fidelity.csv\\"))),\\n    (\\"D3\\", \\"≥ 5 thành công + ≥ 5 thất bại được phân tích\\", *_req_st(\\"K1\\")),\\n    (\\"D4\\", \\"Demo: nhập truy vấn → top-k + điểm + đường đi giải thích\\", *_req_st(\\"DM1\\")),\\n    (\\"E1\\", \\"Notebook chạy hết (hoặc báo rõ giới hạn)\\", \\"PASS\\" if CFG.dataset == \\"msvd\\" and not CFG.dev_mode else \\"NOT RUN\\",\\n     \\"chạy chính thức tới ô này\\" if CFG.dataset == \\"msvd\\" and not CFG.dev_mode else f\\"dataset={CFG.dataset}, dev_mode={CFG.dev_mode}\\"),\\n    (\\"E2\\", \\"File môi trường + manifest\\", *_file(\\"pip_freeze.txt\\", \\"env_versions.json\\", \\"experiment_manifest.json\\", \\"data_inventory.json\\")),\\n    (\\"E3\\", \\"Kết quả báo cáo khớp file nguồn\\", *_req_st(\\"R1\\")),\\n]\\nREQ_DF = pd.DataFrame(REQ, columns=[\\"id\\", \\"yêu cầu\\", \\"trạng thái\\", \\"bằng chứng\\"])\\nREQ_DF.to_csv(RESULTS / \\"requirements_status.csv\\", index=False)\\nmd = [\\"# Audit nghiệm thu — Project 3 · Level 4(b) text-to-video (tự sinh)\\", \\"\\",\\n      f\\"{DATA_INVENTORY[\'created\']} | chính thức: {DATA_INVENTORY[\'official_run\']} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | \\"\\n      f\\"tập ngoài: {EXT_STATUS} | Kaggle={RUN_ON_KAGGLE}\\", \\"\\",\\n      \\"PASS ở đây là PASS của **kiểm thử / sản phẩm**, không phải kết luận khoa học. Kết luận khoa học nằm ở `level4b_conclusions.md`.\\", \\"\\",\\n      \\"| id | yêu cầu | trạng thái | bằng chứng |\\", \\"|---|---|---|---|\\"]\\nmd += [f\\"| {a} | {b} | **{c}** | {str(d)[:240].replace(\'|\', \'/\')} |\\" for a, b, c, d in REQ]\\nmd += [\\"\\", \\"## Kết quả khoa học được báo cáo (FINDINGS)\\", \\"\\", \\"| id | câu hỏi | kết quả | diễn giải |\\", \\"|---|---|---|---|\\"]\\nmd += [f\\"| {f[\'id\']} | {f[\'question\']} | {f[\'result\'].replace(\'|\', \'/\')} | {f[\'interpretation\']} |\\" for f in FINDINGS]\\nmd += [\\"\\", \\"## Thí nghiệm NOT RUN\\", \\"\\"] + ([f\\"- {n[\'stage\']} {n.get(\'experiment_id\', \'\')}: {n[\'reason\']}\\" for n in NOT_RUN] or [\\"- (không có)\\"])\\n(RESULTS / \\"audit_report.md\\").write_text(\\"\\\\n\\".join(md) + \\"\\\\n\\")\\ndisplay(REQ_DF)\\n\\n# ---------------- level4b_conclusions.md: 7 câu hỏi bắt buộc (mục 13) ----------------\\n_P = EVALS[CK][\\"part\\"][\\"name\\"] if CK else EVALS[EX][\\"part\\"][\\"name\\"]\\n_kind = EVALS[CK][\\"info\\"][\\"kind\\"] if CK else EVALS[EX][\\"info\\"][\\"kind\\"]\\n\\n\\ndef _row(h):\\n    r = CONF_T[(CONF_T.hypothesis == h) & (CONF_T.part == _P)]\\n    return r.iloc[0] if len(r) and \\"diff\\" in r and pd.notna(r.iloc[0].get(\\"diff\\")) else None\\n\\n\\ndef _fmt(h):\\n    r = _row(h)\\n    if r is None:\\n        return \\"NOT RUN\\"\\n    return (f\\"ΔMRR {r[\'diff\']:+.4f} (CI95 theo video, chưa hiệu chỉnh bội [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}] ⇒ {_verdict(r.ci_lo, r.ci_hi)}; \\"\\n            f\\"giả thuyết một phía sau Holm: p {r.p_holm:.3f} ⇒ {\'ỦNG HỘ\' if r.supported else \'KHÔNG ủng hộ\'})\\")\\n\\n\\ndef _mrr(k_, part=_P, grp=\\"all\\"):\\n    r = RES_ALL[(RES_ALL.experiment_id == k_) & (RES_ALL.part == part) & (RES_ALL.group == grp)]\\n    return f\\"{r.iloc[0][\'MRR_mean\']:.4f} ± {r.iloc[0][\'MRR_std\']:.4f}\\" if len(r) else \\"NOT RUN\\"\\n\\n\\n_fam = REL_DF[(REL_DF.section == \\"family_knockout\\") & (REL_DF.split == P_MAIN[\\"name\\"])] if len(REL_DF) else REL_DF\\nQ = []\\nQ.append((\\"1. CLIP + Graph có tốt hơn CLIP-only không?\\",\\n          f\\"Trên {_P} ({_kind}): CLIP-only MRR {_mrr(\'G0_clip_only\')}; QC-RGAT (G3) {_mrr(G_IDS.get(\'G3\'))}; GraphSAGE có kiểu cạnh (R3) \\"\\n          f\\"{_mrr(G_IDS.get(\'V3REF\'))}; đồ thị + ConceptNet (G9) {_mrr(G_IDS.get(\'G9\'))}. H1 (G3 > CLIP): {_fmt(\'H1_graph_vs_clip\')}. \\"\\n          f\\"Nhãn học vs zero-shot (H13): {_fmt(\'H13_learned_vs_zeroshot_tags\')}.\\"))\\n_gi = SIG_T[(SIG_T.comparison == \\"G3_vs_clip\\") & (SIG_T.part == _P) & (SIG_T.group == \\"indirect\\")] if len(SIG_T) else SIG_T\\n_qs = QS_PAIRED[(QS_PAIRED.part == _P) & (QS_PAIRED.group == \\"indirect\\") & (QS_PAIRED.a == G_IDS.get(\\"G3\\"))] if len(QS_PAIRED) else QS_PAIRED\\nQ.append((\\"2. Đồ thị có giúp nhóm truy vấn gián tiếp không?\\",\\n          (f\\"Mọi caption nhóm gián tiếp (n={int(_gi.iloc[0].n)}): ΔMRR {_gi.iloc[0][\'diff\']:+.4f} [CI {_gi.iloc[0].ci_lo:+.4f}, {_gi.iloc[0].ci_hi:+.4f}]\\"\\n           if len(_gi) else \\"NOT RUN\\") + \\"; \\" +\\n          (f\\"bộ {int(_qs.iloc[0].n)} truy vấn gián tiếp đăng ký trước: ΔMRR {_qs.iloc[0][\'diff\']:+.4f} [CI {_qs.iloc[0].ci_lo:+.4f}, {_qs.iloc[0].ci_hi:+.4f}]\\"\\n           if len(_qs) else \\"bộ truy vấn: NOT RUN\\") + \\". So sánh PHỤ (không Holm).\\"))\\nQ.append((\\"3. Những loại cạnh nào thực sự có giá trị?\\",\\n          f\\"Quan hệ vs chỉ-contains (H2): {_fmt(\'H2_relations_vs_contains_only\')}. Họ vai agent_of/object_of (H8, huấn luyện lại không có họ): \\"\\n          f\\"{_fmt(\'H8_role_edges\')}. Họ svo/prep (H9): {_fmt(\'H9_svo_prep_edges\')}. Vị từ đúng cạnh (H4): {_fmt(\'H4_true_vs_shuffled_predicates\')}. \\"\\n          f\\"Topology thật (H5): {_fmt(\'H5_true_vs_rewired_topology\')}. ConceptNet vs chỉ-contains (H10): {_fmt(\'H10_kg_vs_contains_only\')}; \\"\\n          f\\"ConceptNet thật vs rewired (H11): {_fmt(\'H11_kg_true_vs_rewired\')}; thêm ConceptNet vào G3 (H12): {_fmt(\'H12_kg_adds_to_graph\')}. \\"\\n          f\\"Knockout lúc suy luận ({P_MAIN[\'name\']}): \\"\\n          + (\\"; \\".join(f\\"{r.item} {r.d_MRR_knockout:+.4f}\\" for r in _fam.itertuples()) if len(_fam) else \\"NOT RUN\\")\\n          + \\". Một họ cạnh chỉ được coi là có giá trị khi H tương ứng được ủng hộ.\\"))\\nQ.append((\\"4. Dùng đồ thị có làm lời giải thích đáng tin cậy hơn không?\\",\\n          f\\"Giải thích = số hạng thật của điểm (Σ = điểm: {XR_SUMM[\'decomposition_ok\']:.2f}; hợp lệ {XR_SUMM[\'validity\']:.2f}). Xoá giải thích đầy đủ: \\"\\n          f\\"tụt hạng {XR_SUMM[\'frac_rank_worse_expl\']:.0%} truy vấn vs ngẫu nhiên {XR_SUMM[\'frac_rank_worse_rand\']:.0%}; chỉ xoá quan hệ: hạng đổi \\"\\n          f\\"{XAI_SUMM[\'frac_rank_changed_expl\']:.0%} (ngẫu nhiên {XAI_SUMM[\'frac_rank_changed_rand\']:.0%}). Jaccard giải thích giữa seed \\"\\n          f\\"{XAI_SUMM[\'seed_sensitivity_mean_jaccard\']:.2f}. ⇒ Giải thích trung thực với ĐIỂM theo cấu tạo (phân rã chính xác). Về QUYẾT ĐỊNH xếp hạng: \\"\\n          + (\\"xoá giải thích đầy đủ làm tụt hạng nhiều hơn xoá ngẫu nhiên, nên giải thích có chỉ ra phần đồ thị gây ra thứ hạng\\"\\n             if _better else \\"xoá giải thích đầy đủ KHÔNG mạnh hơn xoá ngẫu nhiên, nên không kết luận được giải thích phản ánh nguyên nhân thứ hạng\\")\\n          + (\\"; riêng phần QUAN HỆ hầu như không đổi thứ hạng\\" if XAI_SUMM[\\"frac_rank_changed_expl\\"] <= XAI_SUMM[\\"frac_rank_changed_rand\\"] + 0.05\\n             else \\"; phần quan hệ có đổi thứ hạng ở một số truy vấn\\")\\n          + f\\". Độ ổn định giữa seed thấp (Jaccard {XAI_SUMM[\'seed_sensitivity_mean_jaccard\']:.2f}) ⇒ đường đi cụ thể phụ thuộc lần huấn luyện.\\"))\\nQ.append((\\"5. Phần video tái sử dụng kiến trúc retrieval và graph explanation đến mức nào?\\",\\n          \\"Tái sử dụng nguyên: CLIP đóng băng (ảnh → frame), top-k theo cosine, điểm cuối α·z(CLIP) + (1−α)·z(đồ thị) chọn α trên val, \\"\\n          \\"schema node / cạnh (video ~ ảnh), GNN (GraphSAGE có kiểu cạnh, QC-RGAT), giải thích bằng phân rã điểm và xoá cạnh. \\"\\n          \\"Phần riêng của video: lấy mẫu 12 frame, gộp theo thời gian (chọn trên val), nhãn video = CLIP zero-shot trên frame, \\"\\n          f\\"quan hệ từ caption train cấp lớp. Bằng chứng chạy: {_st(\'A4\', \'B6\', \'B8\')[1]}.\\"))\\n_cnt = Counter(REQ_DF[\\"trạng thái\\"])\\nQ.append((\\"6. Yêu cầu nào PASS, FAIL hay NOT RUN?\\",\\n          f\\"{dict(_cnt)}. FAIL: {\', \'.join(REQ_DF[REQ_DF[\'trạng thái\'] == \'FAIL\'].id) or \'—\'}; NOT RUN: \\"\\n          f\\"{\', \'.join(REQ_DF[REQ_DF[\'trạng thái\'] == \'NOT RUN\'].id) or \'—\'} (chi tiết: audit_report.md).\\"))\\nQ.append((\\"7. Level 4(b) được thực hiện đến đâu và giới hạn là gì?\\",\\n          f\\"Đã có pipeline text→video đầy đủ, xếp hạng lại bằng đồ thị, đối chứng, bộ truy vấn trực tiếp/gián tiếp, xAI theo thứ hạng, 5+5 ca, demo. \\"\\n          f\\"Tập xác nhận: {EXT_STATUS} ({EXT_REPORT.get(\'label\', \'—\')}). Giới hạn: nhãn video là CLIP zero-shot (F1 \\"\\n          + \\", \\".join(f\\"{k_} {v_[\'F1\']:.3f}\\" for k_, v_ in TAGGER_REPORT.items()) +\\n          \\"); quan hệ là thống kê cấp lớp từ caption train (video mới không có quan hệ riêng); parser heuristic; chất lượng ngữ nghĩa \\"\\n          \\"cạnh chưa được người xác minh; val chỉ 100 video; test MSVD đã nhìn 4 lần trước (thăm dò); fresh1k-B cùng miền với fresh1k đã ảnh \\"\\n          \\"hưởng tới thiết kế. PASS kiểm thử phần mềm không phải kết luận khoa học.\\"))\\nCONCL = [\\"# Kết luận Level 4(b) — trả lời 7 câu hỏi bắt buộc (tự sinh từ số liệu của lần chạy này)\\", \\"\\",\\n         f\\"Tập chính: **{_P}** ({_kind}). Kết luận chung về quan hệ: {VERDICT}\\", \\"\\"]\\nfor q_, a_ in Q:\\n    CONCL += [f\\"## {q_}\\", \\"\\", a_, \\"\\"]\\n(RESULTS / \\"level4b_conclusions.md\\").write_text(\\"\\\\n\\".join(CONCL))\\nprint(\\"\\\\n\\".join(CONCL))\\n\\n# ---------------- artifacts_manifest.json ----------------\\n_desc = {\\"eval_queries_level4b.csv\\": \\"bộ truy vấn trực tiếp / gián tiếp đăng ký trước\\", \\"eval_queries_metrics.csv\\": \\"metric trên bộ truy vấn\\",\\n         \\"xai_rank_fidelity.csv\\": \\"fidelity theo thứ hạng (giải thích đầy đủ)\\", \\"case_analysis.md\\": \\"5 thành công + 5 thất bại\\",\\n         \\"demo_outputs.json\\": \\"đầu ra demo\\", \\"data_inventory.json\\": \\"kiểm kê dữ liệu\\", \\"data_leakage_audit.md\\": \\"audit rò rỉ\\",\\n         \\"audit_report.md\\": \\"ma trận nghiệm thu\\", \\"level4b_conclusions.md\\": \\"7 câu kết luận\\", \\"graph_ablation_results.csv\\": \\"ablation (1 dòng / thí nghiệm)\\",\\n         \\"graph_experiment_results.csv\\": \\"metric mọi thí nghiệm × tập × nhóm\\", \\"graph_confirmatory_tests.csv\\": \\"họ so sánh chính (Holm)\\",\\n         \\"graph_paired_tests.csv\\": \\"so sánh phụ theo nhóm\\", \\"experiment_manifest.json\\": \\"cấu hình, seed, checkpoint mọi thí nghiệm\\",\\n         \\"split_manifest.json\\": \\"split tái lập\\", \\"graph_schema_v3.json\\": \\"schema đồ thị\\", \\"provenance.csv\\": \\"nguồn gốc artifact\\"}\\nARTIFACTS = [dict(path=p.relative_to(CFG.out_dir).as_posix(), bytes=p.stat().st_size, sha1=file_sha1(p), description=_desc.get(p.name, \\"\\"))\\n             for p in sorted(RESULTS.rglob(\\"*\\")) if p.is_file() and p.name != \\"artifacts_manifest.json\\"]\\nsave_json(dict(created=time.strftime(\\"%Y-%m-%d %H:%M:%S\\"), out_dir=str(CFG.out_dir), n_files=len(ARTIFACTS),\\n               not_packed=dict(thumbs=len(list((CFG.out_dir / \\"thumbs\\").glob(\\"*.jpg\\"))), checkpoints=len(_ck)), files=ARTIFACTS),\\n          RESULTS / \\"artifacts_manifest.json\\")\\nprint(f\\"\\\\nartifacts_manifest.json: {len(ARTIFACTS)} file | yêu cầu: {dict(_cnt)} | FINDINGS: {len(FINDINGS)}\\")"}, {"cell_type": "code", "execution_count": null, "id": "c062", "metadata": {}, "outputs": [], "source": "# ============================== GÓI KẾT QUẢ ĐỂ TẢI VỀ (không gồm dữ liệu) ==============================\\nmark_stage(\\"pack\\", requires=(\\"report\\",))\\n_z = pack_review(\\"hoàn tất\\")\\nassert _z is not None, \\"Không tạo được file zip xem xét — xem cảnh báo phía trên.\\"\\nwith zipfile.ZipFile(_z) as _zf:\\n    _names = _zf.namelist()\\n    _man = json.loads(_zf.read(f\\"{_z.stem}/pack_manifest.json\\"))\\nprint(f\\"📦 {_z} — {_z.stat().st_size / 2**20:.1f} MB, {len(_names)} file. Tải file này về (Kaggle: Output → {_z.name}).\\")\\nprint(\\"   Notebook đã chạy:\\", \\"có (executed_notebook.ipynb, đủ output tới cell trước)\\" if _man[\\"executed_notebook\\"]\\n      else \\"KHÔNG tìm thấy (chạy ngoài Kaggle/papermill) — tải notebook riêng nếu cần\\")\\nprint(\\"   Không gói:\\", \\"; \\".join(f\\"{k} {v[\'n_files\']} file, {v[\'mb\']} MB\\" if isinstance(v, dict) else k for k, v in _man[\\"excluded\\"].items()))"}], "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python"}, "level4b_build": {"source": "v4_qc_rgat_graph.ipynb", "source_sha1": "8ba65a19a414"}, "level4b_v2_build": {"source": "Project03_Level4B_TextToVideo.ipynb", "source_sha1": "d35da75ec6cd"}}, "nbformat": 4, "nbformat_minor": 5}'
L4B_V2_NB_MD5 = 'f4b127817568994b192bbd399f9fc44b'
L4B_V2_DIR = "/kaggle/working/project03_level4b_v2" if ON_KAGGLE else f"{WORK}/project03_level4b_v2"
L4B_V2_RES = f"{OUT}/video_v2"
os.makedirs(L4B_V2_RES, exist_ok=True)
RES_L4B_V2 = dict(status="not run")
_t_v2 = time.time()


def _copy_v2_results(src_dir):
    n = 0
    for f in glob.glob(f"{src_dir}/*.csv") + glob.glob(f"{src_dir}/*.json") + glob.glob(f"{src_dir}/*.md"):
        try:
            open(f"{L4B_V2_RES}/{os.path.basename(f)}", "wb").write(open(f, "rb").read()); n += 1
        except Exception:
            pass
    return n


try:
    _mode = os.environ.get("P3_RUN_L4B_V2", "auto")
    # chỉ dò vài mức thư mục cố định (không rglob cả /kaggle/input — COCO/VG có hàng trăm nghìn file)
    _cached = sorted(set(glob.glob("/kaggle/input/*/project03_level4b_v2/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/*/project03_level4b_v2/cache/frames_msvd_*.npz"))) if ON_KAGGLE else []
    if SMOKE:
        RES_L4B_V2 = dict(status="skipped (SMOKE)")
    elif _mode == "0" or (_mode == "auto" and _cached):
        _root = os.path.dirname(os.path.dirname(_cached[0])) if _cached else None
        _n = _copy_v2_results(f"{_root}/results") if _root else 0
        RES_L4B_V2 = dict(status="reused" if _cached else "skipped", cache=_cached[:3], results_copied=_n, source=_root)
        p3_log("Phần C: không chạy lại —", ("dùng output đã Add Input: " + _root) if _root else "P3_RUN_L4B_V2=0", f"| chép {_n} file kết quả")
    else:
        _nb_in = f"{WORK}/level4b_v2_graphfix_input.ipynb"
        _nb_out = f"{L4B_V2_RES}/project03_level4b_v2_graphfix_executed.ipynb"
        assert hashlib.md5(L4B_V2_NB_JSON.encode()).hexdigest() == L4B_V2_NB_MD5, "notebook v2-graphfix nhúng bị thay đổi"
        open(_nb_in, "w", encoding="utf-8").write(L4B_V2_NB_JSON)
        try:
            import papermill as pm
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
        _old_out = os.environ.get("MSVD_L4_OUT")
        os.environ["MSVD_L4_OUT"] = L4B_V2_DIR          # chỉ cho tiến trình v2; khôi phục ngay sau đó để 4v (v5) dùng thư mục của nó
        try:
            p3_log("Phần C: bắt đầu chạy notebook Level 4(b) v2-graphfix (tiến trình riêng)…")
            pm.execute_notebook(_nb_in, _nb_out, cwd="/kaggle/working" if ON_KAGGLE else WORK,
                                kernel_name="python3", log_output=True, progress_bar=False)
        finally:
            if _old_out is None:
                os.environ.pop("MSVD_L4_OUT", None)
            else:
                os.environ["MSVD_L4_OUT"] = _old_out
        _n = _copy_v2_results(f"{L4B_V2_DIR}/results")
        RES_L4B_V2 = dict(status="ok", out_dir=L4B_V2_DIR, results_copied=_n)
    # cache của Phần C → notebook video v5 (mục 4v) tìm thấy qua MSVD_L4_CACHE_DIRS
    _cdirs = [d for d in os.environ.get("MSVD_L4_CACHE_DIRS", "").split(":") if d]
    for _d in ([f"{L4B_V2_DIR}/cache"] + [os.path.dirname(p) for p in _cached[:1]]):
        if os.path.isdir(_d) and _d not in _cdirs:
            _cdirs.append(_d)
    os.environ["MSVD_L4_CACHE_DIRS"] = ":".join(_cdirs)
    RES_L4B_V2["cache_dirs"] = _cdirs
except Exception as ex:
    RES_L4B_V2 = dict(status="error", error=repr(ex)[:1500])
    p3_log("Phần C: LỖI — các phần sau vẫn chạy tiếp:", repr(ex)[:300])
RES_L4B_V2["minutes"] = round((time.time() - _t_v2) / 60, 1)
p3_log("Phần C:", {k: v for k, v in RES_L4B_V2.items() if k != "error"})


In [ ]:
# ============================== 4v. CẤP 4 (b) — MỞ RỘNG VIDEO (text-to-video, MSVD): chạy nguyên vẹn notebook video v5 đã kiểm chứng ==============================
# Notebook video Level4B v5 "EvidenceGraph" (project/code/project03_level4b/video_v5/, đã chạy trên Kaggle — mã trùng khớp) được NHÚNG nguyên văn
# và chạy trong một tiến trình riêng (papermill) để không đụng biến của notebook này. Chạy TRƯỚC phần ảnh để GPU còn trống cho Qwen2.5-VL-7B.
# Ô này KHÔNG import torch (không khởi tạo CUDA ở tiến trình chính). Lỗi ở nhánh video được ghi lại, phần ảnh vẫn chạy tiếp.
VIDEO_DIR = f"{OUT}/video"
os.makedirs(VIDEO_DIR, exist_ok=True)
RES_VIDEO = dict(status="not run")
_t_video = time.time()
try:
    _nb_in = f"{WORK}/video_v5_input.ipynb"
    _nb_out = f"{VIDEO_DIR}/Project03_Level4B_v5_EvidenceGraph_executed.ipynb"
    if SMOKE:   # chạy thử: notebook giả chỉ ghi một summary giả đúng cấu trúc (KHÔNG phải kết quả MSVD)
        _fake = {"cells": [{"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
            "import json, os\n", f"d = {VIDEO_DIR!r} + '/fake_run/results'\n", "os.makedirs(d, exist_ok=True)\n",
            "json.dump({'test': [{'model': 'M0_clip', 'set': 'test_all', 'MRR': 0.5, 'R@1': 0.4, 'R@5': 0.7, 'R@10': 0.8, 'n_queries': 10},"
            "{'model': 'M4_full_graph', 'set': 'test_all', 'MRR': 0.6, 'R@1': 0.5, 'R@5': 0.8, 'R@10': 0.9, 'n_queries': 10}],"
            "'comparisons': [], 'fidelity': {}, 'test_looks_msvd': 'smoke'}, open(d + '/v5_summary.json', 'w'))\n"]}],
            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}}, "nbformat": 4, "nbformat_minor": 5}
        json.dump(_fake, open(_nb_in, "w"))
        _cwd = VIDEO_DIR
    else:
        open(_nb_in, "w", encoding="utf-8").write(VIDEO_NB_JSON)
        assert hashlib.md5(VIDEO_NB_JSON.encode()).hexdigest() == VIDEO_NB_MD5, "notebook video nhúng bị thay đổi"
        _cwd = "/kaggle/working" if ON_KAGGLE else VIDEO_DIR
    try:
        import papermill as pm
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
    p3_log("Video: bắt đầu chạy notebook video v5 (tiến trình riêng)…")
    pm.execute_notebook(_nb_in, _nb_out, cwd=_cwd, kernel_name="python3", log_output=True, progress_bar=False)
    _hits = sorted(glob.glob(f"{_cwd}/**/results/v5_summary.json", recursive=True), key=os.path.getmtime)
    assert _hits, "không thấy v5_summary.json sau khi chạy notebook video"
    _vs = json.load(open(_hits[-1], encoding="utf-8"))
    _res_dir = os.path.dirname(_hits[-1])
    for f in glob.glob(f"{_res_dir}/*.csv") + glob.glob(f"{_res_dir}/*.json"):
        try:
            open(f"{VIDEO_DIR}/{os.path.basename(f)}", "wb").write(open(f, "rb").read())
        except Exception:
            pass
    RES_VIDEO = dict(status="ok", summary_path=_hits[-1], test=_vs.get("test", []), comparisons=_vs.get("comparisons", []),
                     fidelity=_vs.get("fidelity", {}), test_looks_msvd=_vs.get("test_looks_msvd"), vlm=_vs.get("vlm", {}))
except Exception as ex:
    RES_VIDEO = dict(status="error", error=repr(ex)[:1500])
    p3_log("Video: LỖI — phần ảnh vẫn chạy tiếp:", repr(ex)[:300])
RES_VIDEO["minutes"] = round((time.time() - _t_video) / 60, 1)
if RES_VIDEO["status"] == "ok":
    _tv = {r["model"]: r for r in RES_VIDEO["test"] if r.get("set") == "test_all"}
    p3_log("Video xong:", {k: round(v["MRR"], 4) for k, v in _tv.items()}, f"| {RES_VIDEO['minutes']} phút | lần nhìn test MSVD: {RES_VIDEO['test_looks_msvd']}")


## 5. Mô hình đóng băng (CLIP, RelTR)

In [ ]:
# ============================== 5. MÔ HÌNH ĐÓNG BĂNG: CLIP ViT-L/14 · RelTR (scene graph từ điểm ảnh) — không huấn luyện ==============================
NQ = 200   # số truy vấn bộ ba của RelTR

if SMOKE:
    from PIL import Image   # chỉ để có kiểu; ảnh giả không được mở
    MODEL_INFO = "fake (smoke)"
    _h = lambda s: int(hashlib.md5(str(s).encode()).hexdigest(), 16) % 2 ** 32

    def clip_image_paths(paths):
        v = np.stack([np.random.default_rng(_h(p)).standard_normal(16) for p in paths])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def clip_text(caps):
        v = np.stack([np.random.default_rng(_h(c)).standard_normal(16) for c in caps])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def reltr_clip_features(items, flip=False):
        """giả lập: cài bằng chứng cho caption ĐÚNG của mục ARO (để kiểm tra luồng xử lý, KHÔNG phải kết quả)."""
        F = []
        for it in items:
            r = np.random.default_rng(_h(it["item"]))
            ps, po, pr = r.random((NQ, 151)) * 0.01, r.random((NQ, 151)) * 0.01, r.random((NQ, 51)) * 0.01
            sb, ob = r.random((NQ, 4)) * 0.5 + 0.25, r.random((NQ, 4)) * 0.5 + 0.25
            par = aro_parse(it["true"]) if it["bench"] == "aro" else None
            if par and par[1] != "other":
                X, kind, rel, Y = par
                ps[0, CLS_IDX[X]] = po[0, CLS_IDX[Y]] = 0.9
                if kind == "pred": pr[0, PRED_IDX[rel]] = 0.9
                sb[0, :2], ob[0, :2] = ((0.2, 0.5), (0.8, 0.5)) if rel == "L" else ((0.8, 0.5), (0.2, 0.5))
                sb[0, 2:] = ob[0, 2:] = 0.1
            if flip: sb[:, 0], ob[:, 0] = 1 - sb[:, 0], 1 - ob[:, 0]
            img = r.standard_normal(16)
            F.append(dict(ps=ps.astype(np.float16), po=po.astype(np.float16), pr=pr.astype(np.float16), sb=sb.astype(np.float16),
                          ob=ob.astype(np.float16), img=img / np.linalg.norm(img)))
        return F

    def pil_of(it, flip=False):
        return Image.new("RGB", (100, 100), (128, 128, 128))

    def pil_path(p):
        return Image.new("RGB", (100, 100), (128, 128, 128))
else:
    try:
        import open_clip
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"]); import open_clip
    import torch, torchvision
    import torchvision.transforms as TT
    from PIL import Image, ImageOps
    from concurrent.futures import ThreadPoolExecutor
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    # RelTR: mã nguồn GitHub + checkpoint VG công khai (cùng cách v3)
    RDIR = f"{WORK}/RelTR"
    if not os.path.isdir(f"{RDIR}/models"):
        assert sh(["git", "clone", "-q", "--depth", "1", "https://github.com/yrcong/RelTR.git", RDIR]) == 0
    CK = fetch(f"{WORK}/reltr_vg_checkpoint0149.pth", gdrive=RELTR_GDRIVE)
    CK_SHA = sha256(CK)
    _orig = torchvision.models.resnet50
    torchvision.models.resnet50 = lambda *a, pretrained=False, **kw: _orig(*a, weights=None, **kw)
    sys.path.insert(0, RDIR)
    try:
        from models import build_model as _build
    finally:
        sys.path.remove(RDIR); torchvision.models.resnet50 = _orig
    _args = argparse.Namespace(lr_backbone=1e-5, dataset="vg", backbone="resnet50", dilation=False, position_embedding="sine",
                               enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8,
                               num_entities=100, num_triplets=NQ, pre_norm=False, aux_loss=False, device=DEVICE,
                               set_cost_class=1, set_cost_bbox=5, set_cost_giou=2, set_iou_threshold=0.7, bbox_loss_coef=5,
                               giou_loss_coef=2, rel_loss_coef=1, eos_coef=0.1, return_interm_layers=False)
    RELTR, _, _ = _build(_args)
    _sd = torch.load(CK, map_location="cpu", weights_only=False)
    p3_log("RelTR:", RELTR.load_state_dict(_sd["model"])); del _sd
    RELTR = RELTR.to(DEVICE).eval()
    RTF = TT.Compose([TT.Resize(800, max_size=1333), TT.ToTensor(), TT.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    CLIP, _, PRE = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_WEIGHTS)
    CLIP = CLIP.to(DEVICE).eval(); TOK = open_clip.get_tokenizer(CLIP_MODEL)
    MODEL_INFO = f"RelTR ckpt sha256 {CK_SHA} | CLIP {CLIP_MODEL}/{CLIP_WEIGHTS} | open_clip {open_clip.__version__} | torch {torch.__version__} | {DEVICE}"
    p3_log(MODEL_INFO)

    def pil_path(p):
        return Image.open(p).convert("RGB")

    def pil_of(it, flip=False):
        im = pil_path(it["path"])
        if it["crop"]: im = im.crop(it["crop"])                  # ARO: cắt theo bbox của mục (giao thức ARO)
        return ImageOps.mirror(im) if flip else im

    @torch.no_grad()
    def clip_image_paths(paths, bs=64):
        out = []
        with ThreadPoolExecutor(8) as ex:
            for s in range(0, len(paths), bs):
                x = list(ex.map(lambda p: PRE(pil_path(p)), paths[s:s + bs]))
                with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                    f = CLIP.encode_image(torch.stack(x).to(DEVICE)).float()
                out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def clip_text(caps, bs=256):
        out = []
        for s in range(0, len(caps), bs):
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                f = CLIP.encode_text(TOK(caps[s:s + bs]).to(DEVICE)).float()
            out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def reltr_clip_features(items, flip=False, bs=4):
        """RelTR: softmax rồi bỏ cột nền (cột cuối), 200 truy vấn bộ ba, hộp cxcywh chuẩn hoá — như v3. Kèm đặc trưng CLIP của cùng ảnh."""
        F = []
        for s in range(0, len(items), bs):
            pils = [pil_of(it, flip) for it in items[s:s + bs]]
            try:
                o = RELTR([RTF(im).to(DEVICE) for im in pils])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                parts = [RELTR([RTF(im).to(DEVICE)]) for im in pils]
                o = {k: torch.cat([q[k] for q in parts]) for k in ("sub_logits", "obj_logits", "rel_logits", "sub_boxes", "obj_boxes")}
            sm = lambda k: o[k].float().softmax(-1)[..., :-1].cpu().numpy().astype(np.float16)
            bx = lambda k: o[k].float().cpu().numpy().astype(np.float16)
            ps, po, pr, sb, ob = sm("sub_logits"), sm("obj_logits"), sm("rel_logits"), bx("sub_boxes"), bx("obj_boxes")
            assert ps.shape[1:] == (NQ, 151) and pr.shape[1:] == (NQ, 51)
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                ci = CLIP.encode_image(torch.stack([PRE(im) for im in pils]).to(DEVICE)).float()
            ci = torch.nn.functional.normalize(ci, dim=-1).cpu().numpy()
            for k in range(len(pils)):
                F.append(dict(ps=ps[k], po=po[k], pr=pr[k], sb=sb[k], ob=ob[k], img=ci[k]))
            if (s // bs) % 200 == 0: p3_log(f"  RelTR+CLIP{' (lật)' if flip else ''} {len(F)}/{len(items)}")
        return F


## 6. Thí nghiệm 1 — văn bản → ảnh, full-gallery (bộ chấm đường đi trên scene graph)

In [ ]:
# ============================== 6. THÍ NGHIỆM 1 — Văn bản → ảnh (nhánh A), full-gallery: CLIP top-K → điểm đồ thị → xếp hạng lại ==============================
# Đăng ký trước: v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md. Truy vấn = caption COCO (người viết khác VG); đồ thị = scene graph GT của VG.
# Tham số (w, α) chọn trên P-TUNE; P-CHECK chỉ đánh giá. Đối chứng: G-obj (chỉ đối tượng), G-rand (đồ thị ảnh khác), G-shuf (hoán vị vị từ).
GSTATS = Counter()
GRAPH = {i: build_image(i, REL.get(i, []), OBJ.get(i, []), GSTATS) for i in ELIG1}
FEAT1 = dict(zip(ELIG1, clip_image_paths([VG_JPG[i] for i in ELIG1])))
SPL1 = {"P-TUNE": [i for i in ELIG1 if p5_split(i) == "P-TUNE"], "P-CHECK": [i for i in ELIG1 if p5_split(i) == "P-CHECK"]}
CAPTION = {(i, c): CAPS[COCO_OF_VG[i]][c][1] for i in ELIG1 for c in SEEDS}
METHODS1 = ("G-full", "G-obj", "G-rand", "G-shuf")
W_LISTS = {"G-full": W_GRID, "G-obj": [W_OBJ], "G-rand": W_GRID, "G-shuf": W_GRID}
p3_log(f"TN1: P-TUNE {len(SPL1['P-TUNE'])} / P-CHECK {len(SPL1['P-CHECK'])} ảnh | {elapsed()}")


def exp1_split(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    parsed = [parse_caption(x) for x in caps]
    g_true = [GRAPH[i] for i in ids]
    perm = rand_permutation(len(ids))
    comps = {"G-full": candidate_components(parsed, topk, g_true),
             "G-rand": candidate_components(parsed, topk, [g_true[perm[j]] for j in range(len(ids))]),
             "G-shuf": candidate_components(parsed, topk, [shuffle_predicates(g) for g in g_true])}
    comps["G-obj"] = comps["G-full"]
    return dict(ids=ids, caps=caps, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=parsed, comps=comps,
                g_true=g_true, T=T, I=I)


RES1 = {"seeds": {}}
EXP1_KEEP = {}
for c in SEEDS:
    tune, check = exp1_split("P-TUNE", c), exp1_split("P-CHECK", c)
    params, alpha_curve = {}, {}
    for m in METHODS1:
        grid = tune_grid(tune["zc"], tune["comps"][m], tune["rel_pos"], tune["clip_rank"], W_LISTS[m], ALPHAS)
        params[m] = (select_params(grid), grid[select_params(grid)])
        alpha_curve[m] = {a: max(v for (w, aa), v in grid.items() if aa == a) for a in ALPHAS}   # độ nhạy α: MRR tốt nhất theo w, trên TUNE
    ranks, fused = {"CLIP": check["clip_rank"]}, {}
    for m in METHODS1:
        (w, a), _ = params[m]
        fused[m] = fuse_scores(check["zc"], check["comps"][m], w, a)
        ranks[m] = rank_of_relevant(fused[m], check["rel_pos"], check["clip_rank"])
    rr = {k: 1.0 / v for k, v in ranks.items()}
    has_T = np.array([len(T) > 0 for E, T in check["parsed"]])
    RES1["seeds"][c] = dict(
        params={m: dict(w=list(params[m][0][0]), alpha=params[m][0][1], tune_MRR=params[m][1]) for m in METHODS1},
        n_tune=len(tune["ids"]), n_check=len(check["ids"]), check={k: metrics_from_ranks(v) for k, v in ranks.items()},
        candidate_recall_at_K=float((check["clip_rank"] <= K_TOP).mean()), alpha_curve_tune_MRR=alpha_curve,
        parse_coverage=dict(entity=float(np.mean([len(E) > 0 for E, T in check["parsed"]])), triple=float(has_T.mean())),
        bootstrap=bootstrap_delta({"H1 full-CLIP": rr["G-full"] - rr["CLIP"], "H2 full-obj": rr["G-full"] - rr["G-obj"],
                                   "H3 full-rand": rr["G-full"] - rr["G-rand"], "H4 full-shuf": rr["G-full"] - rr["G-shuf"]}),
        subgroups={nm: dict(n=int(mk.sum()), **({k: metrics_from_ranks(v[mk]) for k, v in ranks.items()} if mk.any() else {}))
                   for nm, mk in (("co_bo_ba", has_T), ("khong_bo_ba", ~has_T))})
    if c == 0: EXP1_KEEP = dict(check=check, params=params, fused=fused, ranks=ranks)
    p3_log(f"TN1 seed {c}: MRR", {k: round(v["MRR"], 4) for k, v in RES1["seeds"][c]["check"].items()}, "| H1", RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"])

H1 = {c: RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"] for c in SEEDS}
RES1["decision"] = "GO" if all(H1[c]["lo"] > 0 for c in SEEDS) else "NO-GO"
RES1["mean_std"] = {k: {m: mean_std([RES1["seeds"][c]["check"][k][m] for c in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                    for k in ["CLIP", *METHODS1]}
RES1.update(n_eligible=len(ELIG1), excluded=dict(EXCL1), graph_stats=dict(GSTATS))
p3_log("TN1 quyết định (đăng ký trước):", RES1["decision"], "| MRR trung bình:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1["mean_std"].items()})


## 6b. Thí nghiệm 1b — GraphSAGE (GNN đã học)
Đăng ký trước: `final_pipeline/GNN_PREREGISTRATION_VI.md`. Đồ thị dị thể ảnh – khái niệm; học dự đoán liên kết trên ảnh không thuộc P-CHECK; suy diễn quy nạp một chiều; ablation cạnh ngẫu nhiên và bỏ cạnh.

In [ ]:
# ============================== 6b. THÍ NGHIỆM 1b — GraphSAGE (GNN đã học) làm điểm đồ thị trong cùng pipeline văn bản → ảnh ==============================
# Đăng ký trước: final_pipeline/GNN_PREREGISTRATION_VI.md. Đồ thị dị thể ảnh – khái niệm (lớp VG150, bộ ba VG150) từ scene graph GT của chính ảnh.
# Huấn luyện chỉ trên ảnh DEV-G0 KHÔNG thuộc P-CHECK; suy diễn quy nạp một chiều (ảnh P-CHECK nhận tin, không gửi tin). α chọn trên P-TUNE.
import torch, torch.nn as nn, torch.nn.functional as TF_
GDEV = "cuda" if torch.cuda.is_available() else "cpu"
SAGE_DIM, SAGE_EPOCHS, SAGE_LR, SAGE_WD, SAGE_DROP, SAGE_MSG_FRAC, SAGE_LOGIT_SCALE = 256, 300, 1e-3, 1e-5, 0.1, 0.8, 10.0

EVAL_IMGS = list(SPL1["P-CHECK"]); EVAL_SET = set(EVAL_IMGS)
TRAIN_IMGS = sorted(i for i in DEV_G0 if i in VG_JPG and i not in EVAL_SET)
GUARD_DEV.check(TRAIN_IMGS + EVAL_IMGS, "GraphSAGE")
assert not set(TRAIN_IMGS) & set(EVAL_IMGS)
for i in TRAIN_IMGS:
    if i not in GRAPH: GRAPH[i] = build_image(i, REL.get(i, []), OBJ.get(i, []), Counter())
_miss = [i for i in TRAIN_IMGS if i not in FEAT1]
FEAT1.update(dict(zip(_miss, clip_image_paths([VG_JPG[i] for i in _miss]))) if _miss else {})
IMG_NODES = TRAIN_IMGS + EVAL_IMGS
IMG_IDX = {i: k for k, i in enumerate(IMG_NODES)}
CLS_LIST = sorted(CLS_IDX)
TRI_LIST = sorted({t for i in TRAIN_IMGS for t in GRAPH[i].triples})          # chỉ bộ ba có trong ảnh huấn luyện
CON = {("c", c): k for k, c in enumerate(CLS_LIST)}
CON.update({("t", t): len(CLS_LIST) + k for k, t in enumerate(TRI_LIST)})
NI, NC = len(IMG_NODES), len(CON)
X_IMG = torch.tensor(np.stack([FEAT1[i] for i in IMG_NODES]), dtype=torch.float32)
X_CON = torch.tensor(clip_text([f"a photo of a {c}" for c in CLS_LIST] + [f"a photo of a {s} {p} a {o}" for s, p, o in TRI_LIST]), dtype=torch.float32)
p3_log(f"GraphSAGE: {len(TRAIN_IMGS)} ảnh huấn luyện, {len(EVAL_IMGS)} ảnh P-CHECK, {len(CLS_LIST)} lớp + {len(TRI_LIST)} bộ ba | {elapsed()}")


def image_edges(rand_seed=None):
    """danh sách (ảnh_idx, khái_niệm_idx). rand_seed ≠ None ⇒ cạnh ngẫu nhiên giữ nguyên bậc từng loại (ablation SAGE-rand)."""
    E = []
    rr = np.random.default_rng(rand_seed) if rand_seed is not None else None
    for i in IMG_NODES:
        im = GRAPH[i]
        cl = [CON[("c", c)] for c in sorted(im.classes) if ("c", c) in CON]
        tr = [CON[("t", t)] for t in sorted(im.triples) if ("t", t) in CON]
        if rr is not None:
            cl = list(rr.choice(len(CLS_LIST), size=len(cl), replace=False)) if cl else []
            tr = list(len(CLS_LIST) + rr.choice(len(TRI_LIST), size=min(len(tr), len(TRI_LIST)), replace=False)) if tr else []
        E += [(IMG_IDX[i], int(c)) for c in cl + tr]
    return np.array(E, dtype=np.int64).reshape(-1, 2)


def mean_adj(edges_img_recv, edges_con_recv, n):
    """ma trận thưa chuẩn hoá theo hàng (mean aggregator). Đỉnh khái niệm có chỉ số NI + k."""
    rows = np.concatenate([edges_img_recv[:, 0], NI + edges_con_recv[:, 1]])
    cols = np.concatenate([NI + edges_img_recv[:, 1], edges_con_recv[:, 0]])
    deg = np.bincount(rows, minlength=n).astype(np.float32)
    vals = 1.0 / np.maximum(deg[rows], 1.0)
    return torch.sparse_coo_tensor(torch.tensor(np.stack([rows, cols])), torch.tensor(vals), (n, n)).coalesce()


class SAGE(nn.Module):
    def __init__(self, d_img, d_con, d=SAGE_DIM, layers=2):
        super().__init__()
        self.pi, self.pc = nn.Linear(d_img, d), nn.Linear(d_con, d)
        self.ws = nn.ModuleList(nn.Linear(d, d) for _ in range(layers))
        self.wn = nn.ModuleList(nn.Linear(d, d, bias=False) for _ in range(layers))

    def forward(self, xi, xc, M):
        h = torch.cat([self.pi(xi), self.pc(xc)])
        for l in range(len(self.ws)):
            h = self.ws[l](h) + self.wn[l](torch.sparse.mm(M, h))
            if l < len(self.ws) - 1: h = TF_.dropout(TF_.relu(h), SAGE_DROP, self.training)
        return TF_.normalize(h, dim=-1)


def train_sage(seed, layers=2, rand=False):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    E = image_edges(SEED_RAND + seed if rand else None)
    is_train = np.array([IMG_NODES[a] not in EVAL_SET for a in E[:, 0]]) if len(E) else np.zeros(0, bool)
    Etr = E[is_train]
    xi, xc = X_IMG.to(GDEV), X_CON.to(GDEV)
    model = SAGE(xi.shape[1], xc.shape[1], layers=layers).to(GDEV)
    opt = torch.optim.Adam(model.parameters(), lr=SAGE_LR, weight_decay=SAGE_WD)
    is_cls = Etr[:, 1] < len(CLS_LIST)
    losses = []
    for ep in range(SAGE_EPOCHS):
        model.train()
        perm = rng.permutation(len(Etr)); cut = int(SAGE_MSG_FRAC * len(Etr))
        msg, sup = Etr[perm[:cut]], Etr[perm[cut:]]
        M = mean_adj(msg, msg, NI + NC).to(GDEV)          # chỉ ảnh huấn luyện gửi/nhận tin trong lúc học
        h = model(xi, xc, M)
        neg = np.where(sup[:, 1] < len(CLS_LIST), rng.integers(0, len(CLS_LIST), len(sup)),
                       len(CLS_LIST) + rng.integers(0, max(len(TRI_LIST), 1), len(sup)))
        a = torch.tensor(sup[:, 0], device=GDEV)
        pos = (h[a] * h[NI + torch.tensor(sup[:, 1], device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        ng = (h[a] * h[NI + torch.tensor(neg, device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        loss = TF_.binary_cross_entropy_with_logits(torch.cat([pos, ng]), torch.cat([torch.ones_like(pos), torch.zeros_like(ng)]))
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(loss))
    model.eval()
    with torch.no_grad():
        # suy diễn: khái niệm chỉ nhận tin từ ảnh huấn luyện; mọi ảnh (kể cả P-CHECK) nhận tin từ khái niệm của chính nó
        H = model(xi, xc, mean_adj(E, Etr, NI + NC).to(GDEV)).cpu().numpy()
    return H, dict(loss_first=losses[0], loss_last=losses[-1], n_edges=int(len(E)), n_train_edges=int(len(Etr)))


def query_concepts(parsed):
    E, T = parsed
    return [CON[("c", c)] for c in sorted(E) if ("c", c) in CON] + [CON[("t", t)] for t in sorted(T) if ("t", t) in CON]


def exp1_clip(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    return dict(ids=ids, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=[parse_caption(x) for x in caps])


def sage_components(d, H):
    Q, K = d["topk"].shape
    out = np.zeros((Q, K, 1))
    for q in range(Q):
        cs = query_concepts(d["parsed"][q])
        if not cs: continue
        v = H[NI + np.array(cs)].mean(0); v = v / (np.linalg.norm(v) + 1e-12)
        out[q, :, 0] = H[[IMG_IDX[d["ids"][j]] for j in d["topk"][q]]] @ v
    return out


VARIANTS = {"SAGE": dict(layers=2, rand=False), "SAGE-rand": dict(layers=2, rand=True), "SAGE-0": dict(layers=0, rand=False)}
RES1G = {"seeds": {}}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, cfg in VARIANTS.items():
        H, tr = train_sage(s, **cfg)
        ct, cc = sage_components(tune, H), sage_components(check, H)
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1G["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_concept_coverage=float(np.mean([len(query_concepts(p)) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"G1 SAGE-CLIP": rr["SAGE"] - rr["CLIP"], "G2 SAGE-rand": rr["SAGE"] - rr["SAGE-rand"],
                                                        "G3 SAGE-noedge": rr["SAGE"] - rr["SAGE-0"]}))
    p3_log(f"TN1b seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1G["seeds"][s]["check"].items()},
        "| α", {k: v["alpha"] for k, v in info.items()}, "| loss", {k: (round(v["loss_first"], 3), round(v["loss_last"], 3)) for k, v in info.items()})
RES1G["mean_std"] = {k: {m: mean_std([RES1G["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *VARIANTS]}
RES1G["decision"] = {h: ("DƯƠNG" if all(RES1G["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h in RES1G["seeds"][0]["bootstrap"]}
RES1G.update(n_train_images=len(TRAIN_IMGS), n_eval_images=len(EVAL_IMGS), n_class_nodes=len(CLS_LIST), n_triple_nodes=len(TRI_LIST),
             hyper=dict(dim=SAGE_DIM, epochs=SAGE_EPOCHS, lr=SAGE_LR, wd=SAGE_WD, dropout=SAGE_DROP, msg_frac=SAGE_MSG_FRAC, logit_scale=SAGE_LOGIT_SCALE))
p3_log("TN1b quyết định (đăng ký trước):", RES1G["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1G["mean_std"].items()}, "|", elapsed())


## 6c. Thí nghiệm 1c — SCENIR (ICML 2025), phương pháp từ 2024
Đối chiếu với mã gốc: `final_pipeline/SCENIR_VERIFICATION_VI.md`. Học không giám sát trên scene graph của ảnh huấn luyện; truy vấn = scene graph tách từ caption; ablation cạnh ngẫu nhiên / bỏ cạnh.

In [ ]:
# ============================== 6c. THÍ NGHIỆM 1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (phương pháp từ 2024 cho cấp 2) ==============================
# Xác minh và kế hoạch: final_pipeline/SCENIR_VERIFICATION_VI.md. Cài lại ARGVA của mã gốc (github.com/nickhaidos/scenir-icml2025, MIT) bằng PyTorch thuần:
# encoder GIN 2 nhánh (μ, log σ), feature decoder (MSE), edge decoder MLP + tích vô hướng, discriminator đối kháng; siêu tham số = config.json.
# Huấn luyện KHÔNG giám sát trên scene graph của ảnh huấn luyện (giống GraphSAGE, không chứa P-CHECK). Truy vấn = scene graph tách từ caption.
SC_CFG = dict(epochs=30, lr=1e-3, gamma=0.95, bs=64, latent=1000, layers=3, disc_hidden=32, edge_hidden=256, edge_out=32, disc_steps=5)
SC_EPS, SC_MAX_LOGSTD = 1e-15, 10.0

# ---- đặc trưng đỉnh: Sentence-Transformer 768 chiều như bài báo (giả định all-mpnet-base-v2); dự phòng CLIP-text ViT-L/14 (768) ----
SC_FEAT_SRC = None
if not SMOKE:
    try:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"]); from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=GDEV)
        SC_X = torch.tensor(_st.encode(CLS_LIST, convert_to_numpy=True, normalize_embeddings=False), dtype=torch.float32)
        SC_FEAT_SRC = "sentence-transformers/all-mpnet-base-v2"; del _st
    except Exception as ex:
        p3_log("SCENIR: không tải được Sentence-Transformer, dùng CLIP-text:", repr(ex)[:200])
if SC_FEAT_SRC is None:
    SC_X = torch.tensor(clip_text(CLS_LIST), dtype=torch.float32); SC_FEAT_SRC = f"CLIP-text {CLIP_MODEL}" if not SMOKE else "fake (smoke)"
SC_CI = {c: k for k, c in enumerate(CLS_LIST)}
p3_log(f"SCENIR: đặc trưng đỉnh {SC_FEAT_SRC}, {tuple(SC_X.shape)}")


def sg_of_pairs(pairs):
    """đồ thị lớp-level: đỉnh = lớp tham gia ≥ 1 quan hệ (bỏ đỉnh cô lập như bài báo), cạnh vô hướng (lưu hai chiều)."""
    pairs = sorted({tuple(sorted((a, b))) for a, b in pairs if a != b and a in SC_CI and b in SC_CI})
    nodes = sorted({c for p in pairs for c in p})
    loc = {c: k for k, c in enumerate(nodes)}
    ei = [(loc[a], loc[b]) for a, b in pairs] + [(loc[b], loc[a]) for a, b in pairs]
    return [SC_CI[c] for c in nodes], np.array(ei, dtype=np.int64).reshape(-1, 2).T


def image_sg(i, variant, seed):
    pairs = list(GRAPH[i].between.keys())
    nodes, ei = sg_of_pairs(pairs)
    if variant == "noedge":
        return nodes, np.zeros((2, 0), np.int64)
    if variant == "rand" and len(nodes) > 1:
        rr = np.random.default_rng(SEED_RAND + seed * 100003 + int(i))
        allp = [(a, b) for a in range(len(nodes)) for b in range(a + 1, len(nodes))]
        pick = [allp[k] for k in rr.choice(len(allp), size=min(ei.shape[1] // 2, len(allp)), replace=False)]
        ei = np.array([(a, b) for a, b in pick] + [(b, a) for a, b in pick], dtype=np.int64).reshape(-1, 2).T
    return nodes, ei


def query_sg(parsed):
    return sg_of_pairs([(s, o) for (s, p, o) in parsed[1]])


class GIN(nn.Module):
    """GINConv(Linear), ε = 0: h'_v = W·(h_v + Σ_{u→v} h_u)."""
    def __init__(self, a, b):
        super().__init__(); self.lin = nn.Linear(a, b)

    def forward(self, x, ei):
        agg = torch.zeros_like(x).index_add_(0, ei[1], x[ei[0]]) if ei.shape[1] else torch.zeros_like(x)
        return self.lin(x + agg)


class ScenirModel(nn.Module):
    def __init__(self, din):
        super().__init__()
        L, H = SC_CFG["layers"], SC_CFG["latent"]
        dims = [din] + [H] * L
        self.mu = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.ls = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.disc = nn.Sequential(nn.Linear(H, SC_CFG["disc_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["disc_hidden"], SC_CFG["disc_hidden"]), nn.ReLU(),
                                  nn.Linear(SC_CFG["disc_hidden"], 1))
        self.fdec = nn.Sequential(nn.Linear(H, din), nn.ReLU(), nn.Linear(din, din))
        self.edec = nn.Sequential(nn.Linear(H, SC_CFG["edge_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["edge_hidden"], SC_CFG["edge_out"]))

    @staticmethod
    def _run(layers, x, ei):
        for k, l in enumerate(layers):
            x = l(x, ei)
            if k < len(layers) - 1: x = TF_.relu(x)
        return x

    def encode(self, x, ei):
        self._mu, self._ls = self._run(self.mu, x, ei), self._run(self.ls, x, ei).clamp(max=SC_MAX_LOGSTD)
        return self._mu + torch.randn_like(self._ls) * torch.exp(self._ls) if self.training else self._mu

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.disc(z)) + SC_EPS).mean()

    def disc_loss(self, z):
        real, fake = torch.sigmoid(self.disc(torch.randn_like(z))), torch.sigmoid(self.disc(z.detach()))
        return -torch.log(real + SC_EPS).mean() - torch.log(1 - fake + SC_EPS).mean()

    def kl_loss(self):
        return -0.5 * torch.mean(torch.sum(1 + 2 * self._ls - self._mu ** 2 - self._ls.exp() ** 2, dim=1))

    def recon_loss(self, x, ei, neg):
        z = self.encode(x, ei)
        feat = TF_.mse_loss(self.fdec(z), x)
        zd = self.edec(z)
        dec = lambda e: torch.sigmoid((zd[e[0]] * zd[e[1]]).sum(-1))
        pos = -torch.log(dec(ei) + SC_EPS).mean() if ei.shape[1] else torch.zeros((), device=x.device)
        ngl = -torch.log(1 - dec(neg) + SC_EPS).mean() if neg.shape[1] else torch.zeros((), device=x.device)
        return pos + ngl + feat


def sc_batch(graphs):
    xs, eis, ptr, off = [], [], [0], 0
    for nodes, ei in graphs:
        xs.append(SC_X[nodes]); eis.append(ei + off); off += len(nodes); ptr.append(off)
    return torch.cat(xs), torch.tensor(np.concatenate(eis, axis=1) if eis else np.zeros((2, 0), np.int64)), np.array(ptr)


def sc_negatives(ei, ptr, rng):
    """cặp (u, v) cùng đồ thị, không tự nối, không trùng cạnh thật; số lượng = số cạnh dương (như batched_negative_sampling)."""
    n = ei.shape[1]
    if n == 0: return np.zeros((2, 0), np.int64)
    exist = set(map(tuple, ei.T.tolist()))
    g = np.searchsorted(ptr, ei[0], side="right") - 1
    out = []
    for gi in rng.choice(g, size=n):
        lo, hi = ptr[gi], ptr[gi + 1]
        for _ in range(10):
            u, v = rng.integers(lo, hi, 2)
            if u != v and (u, v) not in exist: out.append((u, v)); break
    return np.array(out, dtype=np.int64).reshape(-1, 2).T


def train_scenir(seed, variant):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    G = [image_sg(i, variant, seed) for i in TRAIN_IMGS]
    G = [g for g in G if len(g[0]) > 0]
    model = ScenirModel(SC_X.shape[1]).to(GDEV)
    d_opt = torch.optim.AdamW(model.disc.parameters(), lr=SC_CFG["lr"])
    e_opt = torch.optim.AdamW(list(model.mu.parameters()) + list(model.ls.parameters()) + list(model.fdec.parameters()) + list(model.edec.parameters()), lr=SC_CFG["lr"])
    sched = torch.optim.lr_scheduler.ExponentialLR(e_opt, gamma=SC_CFG["gamma"])
    hist = []
    for ep in range(SC_CFG["epochs"]):
        model.train(); perm = rng.permutation(len(G)); tot = 0.0
        for s in range(0, len(G), SC_CFG["bs"]):
            x, ei, ptr = sc_batch([G[k] for k in perm[s:s + SC_CFG["bs"]]])
            neg = torch.tensor(sc_negatives(ei.numpy(), ptr, rng)).to(GDEV); x, ei = x.to(GDEV), ei.to(GDEV)
            e_opt.zero_grad()
            z = model.encode(x, ei)
            for _ in range(SC_CFG["disc_steps"]):
                d_opt.zero_grad(); dl = model.disc_loss(z); dl.backward(); d_opt.step()
            loss = (1 / 6) * model.reg_loss(z) + 3 * model.recon_loss(x, ei, neg) + (1 / 3) * (1 / x.shape[0]) * model.kl_loss()
            loss.backward(); e_opt.step(); tot += float(loss)
        sched.step(); hist.append(tot)
    model.eval()
    return model, dict(n_train_graphs=len(G), loss_first=hist[0], loss_last=hist[-1])


@torch.no_grad()
def sc_embed(model, graphs):
    """nhúng đồ thị = Σ μ của các đỉnh (sum-pool), chuẩn hoá L2; đồ thị rỗng → vector 0."""
    out = np.zeros((len(graphs), SC_CFG["latent"]), np.float32)
    for k, (nodes, ei) in enumerate(graphs):
        if len(nodes) == 0: continue
        z = model.encode(SC_X[nodes].to(GDEV), torch.tensor(ei).to(GDEV)).sum(0)
        out[k] = (z / (z.norm() + 1e-12)).cpu().numpy()
    return out


SC_VARIANTS = {"SCENIR": "full", "SCENIR-rand": "rand", "SCENIR-noedge": "noedge"}
RES1S = {"seeds": {}, "node_features": SC_FEAT_SRC, "config": SC_CFG}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, var in SC_VARIANTS.items():
        model, tr = train_scenir(s, var)
        emb_img = dict(zip(SPL1["P-TUNE"] + SPL1["P-CHECK"],
                           sc_embed(model, [image_sg(i, var, s) for i in SPL1["P-TUNE"] + SPL1["P-CHECK"]])))
        qsg = lambda d: sc_embed(model, [query_sg(p) if var != "noedge" else (query_sg(p)[0], np.zeros((2, 0), np.int64)) for p in d["parsed"]])

        def comps_of(d, qe):
            Q, K = d["topk"].shape
            return np.stack([np.stack([emb_img[d["ids"][j]] for j in d["topk"][q]]) @ qe[q] for q in range(Q)])[:, :, None]
        ct, cc = comps_of(tune, qsg(tune)), comps_of(check, qsg(check))
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
        del model
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1S["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_graph_coverage=float(np.mean([len(query_sg(p)[0]) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"S1 SCENIR-CLIP": rr["SCENIR"] - rr["CLIP"], "S2 SCENIR-rand": rr["SCENIR"] - rr["SCENIR-rand"],
                                                        "S3 SCENIR-noedge": rr["SCENIR"] - rr["SCENIR-noedge"]}))
    p3_log(f"TN1c seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1S["seeds"][s]["check"].items()}, "| α", {k: v["alpha"] for k, v in info.items()},
        "| loss", {k: (round(v["loss_first"], 2), round(v["loss_last"], 2)) for k, v in info.items()})
RES1S["mean_std"] = {k: {m: mean_std([RES1S["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *SC_VARIANTS]}
RES1S["decision"] = {h: ("DƯƠNG" if all(RES1S["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG") for h in RES1S["seeds"][0]["bootstrap"]}
p3_log("TN1c quyết định (đăng ký trước):", RES1S["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1S["mean_std"].items()}, "|", elapsed())


## 7. Thí nghiệm 2 — benchmark kết hợp công khai (ARO, SugarCrepe)

In [ ]:
# ============================== 7. THÍ NGHIỆM 2 — Benchmark kết hợp công khai (ARO VG-Relation, SugarCrepe): ảnh → chọn caption đúng ==============================
# Đăng ký trước: v6_compositional_benchmarks/PREREGISTRATION_VI.md. Đồ thị = scene graph RelTR suy từ điểm ảnh (KHÔNG dùng GT vì caption ARO sinh từ GT).
# Đối chứng: C-rand (đặc trưng mục khác), C-shuf (hoán vị vị từ), C-nogeo (bỏ vị trí). α chọn trên TUNE, đánh giá CHECK, 3 seed chia.
FEATS2 = reltr_clip_features(ITEMS2)
TT2, TF2 = clip_text([it["true"] for it in ITEMS2]), clip_text([it["false"] for it in ITEMS2])
V3B = set(V3_BENCH_IDS)
p3_log(f"TN2: đặc trưng {len(FEATS2)} ảnh | {elapsed()}")

RAND2 = {}
for b in ("aro", "replace_rel", "swap_obj"):
    idx = [n for n, it in enumerate(ITEMS2) if it["bench"] == b]
    perm = rand_permutation(len(idx))
    for j, n in enumerate(idx): RAND2[n] = idx[perm[j]]

ROWS2 = []
for n, it in enumerate(ITEMS2):
    F, Fr = FEATS2[n], FEATS2[RAND2[n]]
    row = dict(bench=it["bench"], item=it["item"], key=it["key"], subset=it["subset"], rel=it["rel"],
               dc=float(F["img"] @ TT2[n] - F["img"] @ TF2[n]), in_v3_bench=bool(it["vg_id"] in V3B))
    if it["bench"] == "aro":
        pt, pf = aro_parse(it["true"]), aro_parse(it["false"])
        valid = pt[1] != "other" and pf[1] != "other"
        if valid:
            prs = shuffled_pr(F, it["key"])
            gt, et = aro_graph_score(F, pt); gf, ef = aro_graph_score(F, pf)
            row.update(dg=gt - gf, dg_rand=aro_graph_score(Fr, pt)[0] - aro_graph_score(Fr, pf)[0],
                       dg_shuf=(aro_graph_score(F, pt, pr=prs)[0] - aro_graph_score(F, pf, pr=prs)[0]) if pt[1] == "pred" else np.nan,
                       dg_nogeo=(aro_graph_score(F, pt, nogeo=True)[0] - aro_graph_score(F, pf, nogeo=True)[0]) if pt[1] == "geo" else np.nan,
                       ev_true=json.dumps(et), ev_false=json.dumps(ef))
    else:
        Tt, Tf = parse_caption(it["true"])[1], parse_caption(it["false"])[1]
        valid = bool(Tt) and bool(Tf) and Tt != Tf
        if valid:
            prs = shuffled_pr(F, int(it["key"][4:]))
            gt, et = triples_graph_score(F, Tt); gf, ef = triples_graph_score(F, Tf)
            if gt is None or gf is None: valid = False
            else:
                row.update(dg=gt - gf, dg_rand=triples_graph_score(Fr, Tt)[0] - triples_graph_score(Fr, Tf)[0],
                           dg_shuf=triples_graph_score(F, Tt, pr=prs)[0] - triples_graph_score(F, Tf, pr=prs)[0], dg_nogeo=np.nan,
                           ev_true=json.dumps(et), ev_false=json.dumps(ef))
    row["valid"] = bool(valid)
    for k in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(k, np.nan)
    ROWS2.append(row)


def exp2_evaluate(rows, s):
    sp = np.array([v6_split(r["key"], s) for r in rows]); ck, tu = sp == "CHECK", sp == "TUNE"
    dc = np.array([r["dc"] for r in rows]); valid = np.array([r["valid"] for r in rows]); clusters = np.array([r["key"] for r in rows])
    out = dict(n_tune=int(tu.sum()), n_check=int(ck.sum()), coverage=float(valid.mean()))
    res = {"CLIP": correct(dc)}
    for var in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"):
        dg = np.array([r[var] for r in rows], dtype=float); v = valid & ~np.isnan(dg)
        if not v.any(): continue
        dgz = np.where(v, dg, 0.0)
        res[f"graph_only[{var}]"] = np.where(v, correct(dgz), 0.5)
        a, sc, sg, acc = tune_alpha(dc[tu], dgz[tu], v[tu])
        res[f"fusion[{var}]"] = correct(fuse2(dc, dgz, v, a, sc, sg)); out[f"alpha[{var}]"] = a
        if var == "dg": out["alpha_grid_tune"] = acc
    out["acc_check"] = {k: float(v[ck].mean()) for k, v in res.items()}
    out["acc_check_valid_only"] = {k: float(v[ck & valid].mean()) for k, v in res.items()} if (ck & valid).any() else {}
    out["n_check_valid"] = int((ck & valid).sum())
    comps = {"H fusion-CLIP": ("fusion[dg]", "CLIP"), "graph-rand": ("graph_only[dg]", "graph_only[dg_rand]"),
             "fusion-fusion_rand": ("fusion[dg]", "fusion[dg_rand]"), "graph-shuf": ("graph_only[dg]", "graph_only[dg_shuf]"),
             "graph-nogeo": ("graph_only[dg]", "graph_only[dg_nogeo]"), "graph-CLIP": ("graph_only[dg]", "CLIP")}
    out["bootstrap_check"] = {}
    for nm, (a, b) in comps.items():
        if a in res and b in res:
            m = ck.copy()
            if "shuf" in nm: m &= ~np.isnan(np.array([r["dg_shuf"] for r in rows], dtype=float)) & valid
            if "nogeo" in nm: m &= ~np.isnan(np.array([r["dg_nogeo"] for r in rows], dtype=float)) & valid
            if m.sum() > 1: out["bootstrap_check"][nm] = dict(n=int(m.sum()), **bootstrap_diff_by_cluster(res[a][m], res[b][m], clusters[m]))
    return out


GROUPS2 = {"ARO_all": lambda r: r["bench"] == "aro", "ARO_geo": lambda r: r["bench"] == "aro" and r["subset"] == "geo",
           "ARO_pred": lambda r: r["bench"] == "aro" and r["subset"] == "pred", "ARO_other": lambda r: r["bench"] == "aro" and r["subset"] == "other",
           "ARO_all_excl_v3bench": lambda r: r["bench"] == "aro" and not r["in_v3_bench"],
           "SC_replace_rel": lambda r: r["bench"] == "replace_rel", "SC_swap_obj": lambda r: r["bench"] == "swap_obj"}
RES2 = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2 if f(r)], s) for g, f in GROUPS2.items() if any(f(r) for r in ROWS2)} for s in SEEDS}}
RES2["mean_std"] = {g: {m: mean_std([RES2["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2["seeds"][0][g]["acc_check"]}
                    for g in RES2["seeds"][0]}
RES2["decision"] = {g: ("DƯƠNG" if all(RES2["seeds"][s][g]["bootstrap_check"].get("H fusion-CLIP", {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                    for g in ("ARO_all", "SC_replace_rel") if g in RES2["seeds"][0]}
RES2.update(items=dict(Counter(i["bench"] for i in ITEMS2)), excluded=dict(EXCL2), overlap_v3_benchmark_items=sum(it["vg_id"] in V3B for it in ITEMS2))
for g in RES2["mean_std"]:
    p3_log(f"TN2 {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "fusion[dg_rand]")})
p3_log("TN2 quyết định (đăng ký trước):", RES2["decision"], "|", elapsed())


## 7b. Thí nghiệm 2b — Cấp 4 (a): cạnh hình học trên COCO-spatial
Đăng ký trước: `final_pipeline/LEVEL4A_PREREGISTRATION_VI.md`. Cạnh trái / phải / trên / dưới suy từ box RelTR; đối chứng bỏ vị trí, đặc trưng mục khác, lật ảnh.

In [ ]:
# ============================== 7b. THÍ NGHIỆM 2b — Cấp 4 hướng (a): cạnh hình học trên tập công khai thứ hai (COCO-spatial, What'sUp EMNLP 2023) ==============================
# Đăng ký trước: final_pipeline/LEVEL4A_PREREGISTRATION_VI.md. Cùng RelTR, CLIP, cách kết hợp và chọn α như TN2. Lọc rò rỉ như SugarCrepe.
CSP_GDRIVE = "1TCEoM0mgFmz8T4cF7PQ3XJmO6JjtiQ-s"
CSP_PATH = BENCH["cocosp"] if SMOKE else fetch(f"{WORK}/coco_qa_two_obj.json", gdrive=CSP_GDRIVE)
CSP_SHA = sha256(CSP_PATH)
CSP_RX = re.compile(r"^A photo of an? (.+?) (?:to the )?(left of|right of|above|below) (?:an? |the )?(.+)$")
CSP_DIR = {"left of": ("x", 1), "right of": ("x", -1), "above": ("y", 1), "below": ("y", -1)}


def csp_name(phrase):
    return norm_name(phrase) or norm_name(phrase.split()[-1])


def csp_parse(cap):
    m = CSP_RX.match(cap)
    if not m: return None
    X, Y = csp_name(m.group(1)), csp_name(m.group(3))
    return (X, m.group(2), Y) if X and Y else ("?", m.group(2), "?")


def geo_axis_score(F, X, Y, rel, nogeo=False):
    """như geo_score nhưng cho cả trục ngang (left/right) và trục dọc (above/below): σ(sign·(c_v − c_u)/τ)."""
    axis, sign = CSP_DIR[rel]
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, c = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0 if axis == "x" else 1]
    s = np.ones((len(c), len(c))) if nogeo else 1.0 / (1.0 + np.exp(-sign * (c[None, :] - c[:, None]) / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]), x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


ITEMS2B, EXCL2B = [], Counter()
for k, x in enumerate(json.load(open(CSP_PATH, encoding="utf-8"))):
    coco = int(x[0]); vid = VG_OF_COCO.get(coco)
    if vid is not None and vid not in VALSET: EXCL2B["vg_not_VAL"] += 1; continue
    path = os.path.join(P["coco_val"], f"{coco:012d}.jpg")
    if not os.path.exists(path): EXCL2B["no_jpg"] += 1; continue
    pt = csp_parse(x[1]); pf = csp_parse(x[2])
    if pt is None or pf is None: EXCL2B["unparsed"] += 1; continue
    ITEMS2B.append(dict(bench="cocosp", item=f"cocosp{k}", key=f"cocosp{coco}", vg_id=vid, path=path, crop=None, true=x[1], false=x[2],
                        pt=pt, pf=pf, axis=CSP_DIR[pt[1]][0], valid=pt[0] != "?" and pf[0] != "?"))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2B if it["vg_id"] is not None], "COCO-spatial")
p3_log(f"TN2b COCO-spatial: {len(ITEMS2B)} mục giữ lại, loại {dict(EXCL2B)}, đồ thị chấm được {sum(it['valid'] for it in ITEMS2B)} | {elapsed()}")

FEATS2B = reltr_clip_features(ITEMS2B)
T2B, F2B = clip_text([it["true"] for it in ITEMS2B]), clip_text([it["false"] for it in ITEMS2B])
PERM2B = rand_permutation(len(ITEMS2B))
_flip_idx = [n for n, it in enumerate(ITEMS2B) if it["valid"]]
FLIP2B = {}
if _flip_idx:
    _hidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "x"]; _vidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "y"]
    FLIP2B.update(dict(zip(_hidx, reltr_clip_features([ITEMS2B[n] for n in _hidx], flip=True))))
    if _vidx:
        _orig_pil_of = pil_of
        if not SMOKE:
            pil_of = lambda it, flip=False: ImageOps.flip(pil_path(it["path"])) if flip else pil_path(it["path"])     # lật DỌC cho mục trên/dưới
        FLIP2B.update(dict(zip(_vidx, reltr_clip_features([ITEMS2B[n] for n in _vidx], flip=True))))
        pil_of = _orig_pil_of
p3_log(f"TN2b: đặc trưng {len(FEATS2B)} ảnh + {len(FLIP2B)} ảnh lật | {elapsed()}")

ROWS2B = []
for n, it in enumerate(ITEMS2B):
    F, Fr = FEATS2B[n], FEATS2B[PERM2B[n]]
    row = dict(item=it["item"], key=it["key"], axis=it["axis"], rel=it["pt"][1], valid=it["valid"], dc=float(F["img"] @ T2B[n] - F["img"] @ F2B[n]))
    if it["valid"]:
        (X, r, Y), (Xf, rf, Yf) = it["pt"], it["pf"]
        gt, et = geo_axis_score(F, X, Y, r); gf, ef = geo_axis_score(F, Xf, Yf, rf)
        row.update(dg=gt - gf, dg_rand=geo_axis_score(Fr, X, Y, r)[0] - geo_axis_score(Fr, Xf, Yf, rf)[0],
                   dg_nogeo=geo_axis_score(F, X, Y, r, nogeo=True)[0] - geo_axis_score(F, Xf, Yf, rf, nogeo=True)[0],
                   dg_shuf=np.nan, ev_true=json.dumps(et), ev_false=json.dumps(ef))
        if n in FLIP2B:
            Ff = FLIP2B[n]
            dgf = geo_axis_score(Ff, X, Y, r)[0] - geo_axis_score(Ff, Xf, Yf, rf)[0]
            dcf = float(Ff["img"] @ T2B[n] - Ff["img"] @ F2B[n])
            row.update(flip_graph_reversed=float(np.sign(dgf) == -np.sign(row["dg"]) and row["dg"] != 0),
                       flip_clip_reversed=float(np.sign(dcf) == -np.sign(row["dc"]) and row["dc"] != 0))
    for kk in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(kk, np.nan)
    ROWS2B.append(row)

GROUPS2B = {"COCOSP_all": lambda r: True, "COCOSP_leftright": lambda r: r["axis"] == "x", "COCOSP_abovebelow": lambda r: r["axis"] == "y"}
RES2B = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2B if f(r)], s) for g, f in GROUPS2B.items() if any(f(r) for r in ROWS2B)} for s in SEEDS}}
RES2B["mean_std"] = {g: {m: mean_std([RES2B["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2B["seeds"][0][g]["acc_check"]}
                     for g in RES2B["seeds"][0]}
RES2B["decision"] = {h: ("DƯƠNG" if all(RES2B["seeds"][s]["COCOSP_all"]["bootstrap_check"].get(k, {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h, k in (("A1 fusion-CLIP", "H fusion-CLIP"), ("A2 graph-nogeo", "graph-nogeo"), ("A3 graph-rand", "graph-rand"))}
_fl = [r for r in ROWS2B if "flip_graph_reversed" in r]
if _fl:
    fg, fc = np.array([r["flip_graph_reversed"] for r in _fl]), np.array([r["flip_clip_reversed"] for r in _fl])
    RES2B["counterfactual"] = dict(n=len(_fl), graph_reversed=float(fg.mean()), clip_reversed=float(fc.mean()),
                                   gap=bootstrap_diff_by_cluster(fg, fc, [r["key"] for r in _fl]))
RES2B.update(n_items=len(ITEMS2B), excluded=dict(EXCL2B), n_scorable=int(sum(it["valid"] for it in ITEMS2B)), sha256=CSP_SHA)
for g in RES2B["mean_std"]:
    p3_log(f"TN2b {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2B["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "graph_only[dg_rand]", "graph_only[dg_nogeo]")})
p3_log("TN2b quyết định (đăng ký trước):", RES2B["decision"], "| phản thực tế:", RES2B.get("counterfactual"), "|", elapsed())


## 8. Giải thích (XAI, cấp 3)
Đăng ký trước: `v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md`. Sparsity gồm kích thước bằng chứng (hằng số theo thiết kế) và **độ tập trung bằng chứng** (thay đổi theo mục).

In [ ]:
# ============================== 8. GIẢI THÍCH (XAI, cấp 3): fidelity · sufficiency · phản thực tế (lật ảnh) · validity (khớp GT) · sparsity ==============================
# Đăng ký trước: v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md. Đơn vị: mục ARO đồ thị chấm được, phần CHECK của seed 0.
# Giải thích = đúng phần tử tạo ra điểm đồ thị: truy vấn bộ ba j* (vị từ) hoặc cặp hộp u*, v* (trái/phải).
XI = [n for n, it in enumerate(ITEMS2) if it["bench"] == "aro" and aro_parse(it["true"])[1] != "other" and aro_parse(it["false"])[1] != "other"
      and v6_split(it["key"], 0) == "CHECK"]
XI.sort(key=lambda n: int(ITEMS2[n]["item"][3:]))
GEO_N = [n for n in XI if aro_parse(ITEMS2[n]["true"])[1] == "geo"]
FLIP = dict(zip(GEO_N, reltr_clip_features([ITEMS2[n] for n in GEO_N], flip=True)))
p3_log(f"XAI: {len(XI)} mục ({len(GEO_N)} trái/phải) | đặc trưng ảnh lật xong | {elapsed()}")


def zero_rows(F, kind, rows):
    G = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in F.items()}
    for r in rows:
        if kind == "pred": G["ps"][r] = 0; G["po"][r] = 0; G["pr"][r] = 0
        else: (G["ps"] if r < NQ else G["po"])[r % NQ] = 0
    return G


def keep_rows(F, kind, rows):
    return zero_rows(F, kind, [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows)])


ev_rows = lambda e: [e["j"]] if e["kind"] == "pred" else [e["u"], e["v"]]
sgn = lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
cxcywh = lambda b: (b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2)


def dgraph(F, it):
    gt, et = aro_graph_score(F, aro_parse(it["true"])); gf, ef = aro_graph_score(F, aro_parse(it["false"]))
    return gt - gf, et, ef


def evidence_concentration(F, parsed):
    """sparsity không cố định: tỷ lệ điểm của phần tử giải thích (max) trên tổng điểm mọi ứng viên (truy vấn bộ ba hoặc cặp hộp).
    Gần 1 ⇒ giải thích gọn (một bằng chứng áp đảo); gần 0 ⇒ bằng chứng dàn trải trên nhiều ứng viên."""
    X, kind, r, Y = parsed
    if kind == "pred":
        v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * F["pr"][:, PRED_IDX[r]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
        return float(v.max() / max(v.sum(), FLOOR))
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    diff = (B[None, :, 0] - B[:, None, 0]) if r == "L" else (B[:, None, 0] - B[None, :, 0])
    M = P[:, CLS_IDX[X]][:, None] * P[:, CLS_IDX[Y]][None, :] / (1.0 + np.exp(-diff / TAU)); np.fill_diagonal(M, 0.0)
    return float(M.max() / max(M.sum(), FLOOR))


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix * iy
    return ix * iy / u if u > 0 else 0.0


def crop_norm(gt, crop):
    cw, ch = crop[2] - crop[0], crop[3] - crop[1]
    return ((gt[0] - crop[0]) / cw, (gt[1] - crop[1]) / ch, (gt[2] - crop[0]) / cw, (gt[3] - crop[1]) / ch)


ROWS3 = []
for n in XI:
    it, F = ITEMS2[n], FEATS2[n]
    kind = aro_parse(it["true"])[1]
    dg, et, ef = dgraph(F, it)
    dc = float(F["img"] @ TT2[n] - F["img"] @ TF2[n])
    row = dict(item=it["item"], n=n, vg_id=it["vg_id"], kind=kind, true=it["true"], false=it["false"], dg=dg, dc=dc,
               graph_correct=float(correct([dg])[0]), clip_correct=float(correct([dc])[0]), ev_true=json.dumps(et), ev_false=json.dumps(ef))
    if dg != 0:
        rows_ev = ev_rows(et if dg > 0 else ef)
        row["del_changed"] = float(sgn(dgraph(zero_rows(F, kind, rows_ev), it)[0]) != sgn(dg))
        pool = [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows_ev)]
        rr = np.random.default_rng(int(hashlib.md5(it["item"].encode()).hexdigest(), 16) % 2 ** 32)
        row["rand_changed"] = float(np.mean([sgn(dgraph(zero_rows(F, kind, list(rr.choice(pool, size=len(rows_ev), replace=False))), it)[0]) != sgn(dg)
                                             for _ in range(N_RAND_DEL)]))
        row["suff_kept"] = float(sgn(dgraph(keep_rows(F, kind, sorted(set(ev_rows(et)) | set(ev_rows(ef)))), it)[0]) == sgn(dg))
        row["evidence_fraction"] = len(rows_ev) / (NQ if kind == "pred" else 2 * NQ)
        row["evidence_concentration"] = evidence_concentration(F, aro_parse(it["true"] if dg > 0 else it["false"]))
        if dg > 0 and it["xid"] in GTBOX and it["yid"] in GTBOX:
            gx, gy = crop_norm(GTBOX[it["xid"]], it["crop"]), crop_norm(GTBOX[it["yid"]], it["crop"])
            bx_, by_ = (et["sub_box"], et["obj_box"]) if kind == "pred" else (et["x_box"], et["y_box"])
            row["iou_x"], row["iou_y"] = iou(cxcywh(bx_), gx), iou(cxcywh(by_), gy)
            row["valid_both"] = float(row["iou_x"] >= 0.5 and row["iou_y"] >= 0.5); row["valid_any"] = float(row["iou_x"] >= 0.5 or row["iou_y"] >= 0.5)
    if n in FLIP:
        dgf = dgraph(FLIP[n], it)[0]; dcf = float(FLIP[n]["img"] @ TT2[n] - FLIP[n]["img"] @ TF2[n])
        row["flip_graph_reversed"] = float(sgn(dgf) == -sgn(dg) and dg != 0); row["flip_clip_reversed"] = float(sgn(dcf) == -sgn(dc) and dc != 0)
    ROWS3.append(row)


def _col(name, rows):
    return np.array([r[name] for r in rows if name in r], float), [r["vg_id"] for r in rows if name in r]


RES3 = {}
for kind in ("all", "geo", "pred"):
    sub = [r for r in ROWS3 if kind == "all" or r["kind"] == kind]
    d, cl = _col("del_changed", sub); rnd, _ = _col("rand_changed", sub)
    o = dict(n=len(sub), graph_acc=float(np.mean([r["graph_correct"] for r in sub])), clip_acc=float(np.mean([r["clip_correct"] for r in sub])))
    if len(d):
        vb, _ = _col("valid_both", sub); va, _ = _col("valid_any", sub)
        o.update(fidelity_del_changed=float(d.mean()), fidelity_rand_changed=float(rnd.mean()), fidelity_gap=bootstrap_diff_by_cluster(d, rnd, cl),
                 sufficiency_kept=float(_col("suff_kept", sub)[0].mean()), sparsity_evidence_fraction=float(_col("evidence_fraction", sub)[0].mean()),
                 sparsity_concentration_mean=float(_col("evidence_concentration", sub)[0].mean()),
                 sparsity_concentration_median=float(np.median(_col("evidence_concentration", sub)[0])),
                 validity_both_iou50=float(vb.mean()) if len(vb) else None, validity_any_iou50=float(va.mean()) if len(va) else None, n_validity=len(vb))
    fg, fcl = _col("flip_graph_reversed", sub); fc, _ = _col("flip_clip_reversed", sub)
    if len(fg):
        o.update(counterfactual_graph_reversed=float(fg.mean()), counterfactual_clip_reversed=float(fc.mean()), n_counterfactual=len(fg),
                 counterfactual_gap=bootstrap_diff_by_cluster(fg, fc, fcl))
    RES3[kind] = o
p3_log("XAI:", json.dumps({k: {m: (round(v, 3) if isinstance(v, float) else v) for m, v in o.items() if not isinstance(v, dict)} for k, o in RES3.items()}, ensure_ascii=False)[:900])

# ---- 10 ví dụ (5 đồ thị đúng, 5 đồ thị sai; 5 mục đầu theo id — luật khai báo trước) ----
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
EX3 = [r for r in ROWS3 if r["graph_correct"] == 1.0][:5] + [r for r in ROWS3 if r["graph_correct"] == 0.0][:5]


def draw_explanation(r, path):
    it = ITEMS2[r["n"]]; im = pil_of(it); W, H = im.size
    fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(im); ax.axis("off")
    for ev, color, lab in ((json.loads(r["ev_true"]), "lime", "đúng"), (json.loads(r["ev_false"]), "red", "sai")):
        bxs = [ev["sub_box"], ev["obj_box"]] if ev["kind"] == "pred" else [ev["x_box"], ev["y_box"]]
        for b, nm in zip(bxs, ("X", "Y")):
            x0, y0, x1, y1 = cxcywh(b)
            ax.add_patch(mpatches.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H, fill=False, ec=color, lw=2, ls="-" if lab == "đúng" else "--"))
            ax.text(x0 * W, y0 * H, f"{nm}({lab})", color=color, fontsize=8, va="bottom")
    ax.set_title(f"{r['item']} | {'ĐỒ THỊ ĐÚNG' if r['graph_correct'] == 1 else 'ĐỒ THỊ SAI'}\nđúng: {r['true']} (p={json.loads(r['ev_true'])['p']:.2e})\n"
                 f"sai: {r['false']} (p={json.loads(r['ev_false'])['p']:.2e})\nΔCLIP={r['dc']:+.3f}", fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


for k, r in enumerate(EX3): draw_explanation(r, f"{FIG}/xai_ex{k:02d}_{r['item']}.png")
p3_log(f"XAI: đã vẽ {len(EX3)} ví dụ vào {FIG} | {elapsed()}")


## 10. Demo: nhập truy vấn → top-k + giải thích
Gọi `show(search("a child holding a kite"))` để thử truy vấn khác.

In [ ]:
# ============================== 10. DEMO: nhập truy vấn → top-k ảnh + quan hệ/đường đi giải thích (đề cương mục 8) ==============================
# Dùng gallery P-CHECK của TN1 (ảnh VG có scene graph GT), tham số (w, α) đã chọn trên P-TUNE (seed 0). Không tinh chỉnh gì thêm.
DEMO_IDS = EXP1_KEEP["check"]["ids"]
DEMO_I = EXP1_KEEP["check"]["I"]
(DEMO_W, DEMO_A), _ = EXP1_KEEP["params"]["G-full"]


def search(query, k=5, alpha=None, w=None):
    """CLIP top-K theo cosine → điểm đồ thị (3 kiểu đường đi) → xếp hạng lại α·z(cos) + (1−α)·z(graph). Trả về top-k + giải thích."""
    alpha = DEMO_A if alpha is None else alpha; w = DEMO_W if w is None else w
    t = clip_text([query])[0]
    cos = DEMO_I @ t
    top = np.argsort(-cos, kind="stable")[:K_TOP]
    E, T = parse_caption(query)
    comps = np.array([[(lambda d: (d["s_rel"], d["s_pair"], d["s_obj"]))(match_detail(T, E, GRAPH[DEMO_IDS[j]])) for j in top]])
    final = fuse_scores(zrows(cos[top][None, :]), comps, w, alpha)[0]
    order = np.lexsort((np.arange(len(top)), -final))[:k]
    res = []
    for r, kk in enumerate(order):
        i = DEMO_IDS[top[kk]]; d = match_detail(T, E, GRAPH[i])
        res.append(dict(rank=r + 1, clip_rank=int(kk) + 1, image_id=i, score=float(final[kk]), cos=float(cos[top[kk]]),
                        matched_edges=d["edges"], matched_pairs=d["pairs"], matched_nodes=d["nodes"]))
    return dict(query=query, parsed_entities=sorted(E), parsed_triples=sorted("|".join(x) for x in T), alpha=alpha, w=list(w), results=res)


def show(out, path=None):
    n = len(out["results"])
    fig, axs = plt.subplots(1, n, figsize=(3.3 * n, 3.9)); axs = np.atleast_1d(axs)
    for ax, r in zip(axs, out["results"]):
        ax.imshow(pil_path(VG_JPG[r["image_id"]])); ax.axis("off")
        why = "; ".join(e.split("<=")[0] for e in r["matched_edges"]) or ("cặp: " + ", ".join(r["matched_pairs"]) if r["matched_pairs"] else "đối tượng: " + ", ".join(r["matched_nodes"]))
        ax.set_title(f"#{r['rank']} (CLIP #{r['clip_rank']}) id {r['image_id']}\n{why[:60]}", fontsize=7)
    fig.suptitle(f"'{out['query']}' | bộ ba: {out['parsed_triples']} | α={out['alpha']}", fontsize=8); fig.tight_layout()
    path = path or f"{FIG}/_last_search.png"
    fig.savefig(path, dpi=100); plt.close(fig)
    try:
        from IPython.display import Image as _IImg, display
        display(_IImg(filename=path))                       # hiện hình khi chạy trong notebook
    except Exception:
        pass


DEMO = []
for q in ["a man riding a horse on the beach", "a dog sitting on a bench", "a woman holding an umbrella on the street"]:
    out = search(q); DEMO.append(out); show(out, f"{FIG}/demo_{len(DEMO)}.png")
    p3_log(f"DEMO '{q}':", [(r["image_id"], r["clip_rank"], r["matched_edges"][:1]) for r in out["results"]])
# Dùng tương tác trong notebook:  show(search("a child holding a kite"))


## 10b. Năm truy vấn thành công và năm truy vấn thất bại (văn bản → ảnh)
Luật chọn cố định; phân tích chi tiết trong `final_pipeline/CASE_ANALYSIS_LEVEL1_VI.md`.

In [ ]:
# ============================== 10b. 5 TRUY VẤN THÀNH CÔNG + 5 THẤT BẠI (văn bản → ảnh) — luật chọn cố định ==============================
# Luật (CASE_ANALYSIS_LEVEL1_VI.md): seed 0, P-CHECK; thành công = 5 truy vấn tăng hạng nhiều nhất khi thêm đồ thị (hạng CLIP − hạng CLIP+đồ thị),
# thất bại = 5 truy vấn giảm nhiều nhất; hoà xếp theo id ảnh. Mỗi hình: ảnh đúng (kèm hạng trước/sau) + top-3 sau xếp hạng lại kèm bằng chứng đồ thị.
_ck, _rk = EXP1_KEEP["check"], EXP1_KEEP["ranks"]
_d = _rk["CLIP"] - _rk["G-full"]
_ord_up = sorted(range(len(_d)), key=lambda q: (-_d[q], _ck["ids"][q]))[:5]
_ord_dn = sorted(range(len(_d)), key=lambda q: (_d[q], _ck["ids"][q]))[:5]
CASES1 = []
for tag, L in (("thanh_cong", _ord_up), ("that_bai", _ord_dn)):
    for q in L:
        E, T = _ck["parsed"][q]
        order = np.lexsort((np.arange(_ck["topk"].shape[1]), -EXP1_KEEP["fused"]["G-full"][q]))[:3]
        top = []
        for kk in order:
            i = _ck["ids"][_ck["topk"][q, kk]]; dd = match_detail(T, E, GRAPH[i])
            top.append(dict(image_id=i, clip_pos=int(kk) + 1, relevant=bool(i == _ck["ids"][q]), edges=dd["edges"], nodes=dd["nodes"]))
        drel = match_detail(T, E, GRAPH[_ck["ids"][q]])
        CASES1.append(dict(kind=tag, query_image_id=_ck["ids"][q], caption=_ck["caps"][q], rank_clip=int(_rk["CLIP"][q]), rank_graph=int(_rk["G-full"][q]),
                           rank_obj_only=int(_rk["G-obj"][q]), parsed_triples=sorted("|".join(t) for t in T), parsed_entities=sorted(E),
                           relevant_evidence=dict(edges=drel["edges"], nodes=drel["nodes"]), top3=top))
        fig, axs = plt.subplots(1, 4, figsize=(13, 3.9))
        panels = [(_ck["ids"][q], f"ẢNH ĐÚNG: hạng CLIP {CASES1[-1]['rank_clip']} → {CASES1[-1]['rank_graph']}\nkhớp: {drel['nodes']} {[e.split('<=')[0] for e in drel['edges']]}")]
        panels += [(t["image_id"], f"#{n + 1} sau xếp hạng (CLIP #{t['clip_pos']}){' ✓' if t['relevant'] else ''}\nkhớp: {t['nodes']} {[e.split('<=')[0] for e in t['edges']]}")
                   for n, t in enumerate(top)]
        for ax, (i, title) in zip(axs, panels):
            ax.imshow(pil_path(VG_JPG[i])); ax.axis("off"); ax.set_title(title[:110], fontsize=7)
        fig.suptitle(f"[{tag}] '{_ck['caps'][q]}' | bộ ba tách được: {CASES1[-1]['parsed_triples']}", fontsize=8); fig.tight_layout()
        fig.savefig(f"{FIG}/case_{tag}_{len(CASES1):02d}_{_ck['ids'][q]}.png", dpi=95); plt.close(fig)
json.dump(CASES1, open(f"{OUT}/cases_text_to_image.json", "w"), indent=1, ensure_ascii=False, default=str)
p3_log(f"Ca văn bản → ảnh: {[(c['kind'], c['rank_clip'], c['rank_graph']) for c in CASES1]} | tăng {int((_d > 0).sum())}, giảm {int((_d < 0).sum())}, "
    f"không đổi {int((_d == 0).sum())}; trùng hạng với bản chỉ đối tượng: {float(np.mean(_rk['G-obj'] == _rk['G-full'])):.3f}")


## 11a. Phân tích bổ sung: độ nhạy α, so sánh cách tạo cạnh, file split

In [ ]:
# ============================== 11a. PHÂN TÍCH BỔ SUNG: độ nhạy α · so sánh cách tạo cạnh · xuất file split · hiển thị hình ==============================
from IPython.display import Image as _IImg, Markdown as _IMd, display as _idisp


def show_png(path, width=900):
    try:
        _idisp(_IImg(filename=path, width=width))
    except Exception:
        pass


# ---- 11a.1 Độ nhạy theo α (đề cương mục 7: "phân tích độ nhạy theo trọng số kết hợp α"), đo trên tập TUNE (validation) ----
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, res, key in (("TN1 luật (G-full)", RES1, None), ("TN1b GraphSAGE", RES1G, "SAGE"), ("TN1c SCENIR", RES1S, "SCENIR")):
    M = np.array([[(res["seeds"][s]["alpha_curve_tune_MRR"]["G-full"] if key is None else res["seeds"][s]["train"][key]["alpha_curve_tune_MRR"])[a]
                   for a in ALPHAS] for s in SEEDS])
    axs[0].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=name)
axs[0].set_title("TN1 văn bản → ảnh: MRR trên P-TUNE theo α", fontsize=9); axs[0].set_xlabel("α (1 = chỉ CLIP)"); axs[0].set_ylabel("MRR")
for g, res in (("ARO_all", RES2), ("ARO_geo", RES2), ("ARO_pred", RES2), ("SC_replace_rel", RES2), ("COCOSP_all", RES2B)):
    if g not in res["seeds"][0] or "alpha_grid_tune" not in res["seeds"][0][g]: continue
    M = np.array([[res["seeds"][s][g]["alpha_grid_tune"][a] for a in ALPHAS] for s in SEEDS]) * 100
    axs[1].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=g)
axs[1].set_title("TN2/2b ảnh → caption: độ chính xác (%) trên TUNE theo α", fontsize=9); axs[1].set_xlabel("α (1 = chỉ CLIP)")
for ax in axs: ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIG}/alpha_sensitivity.png", dpi=120); plt.close(fig)
show_png(f"{FIG}/alpha_sensitivity.png")

# ---- 11a.2 So sánh các cách tạo cạnh / kiểu đường đi (đề cương mục 7, mở rộng) ----
EDGE_TABLE = [("TN1", "Không đồ thị (CLIP)", RES1["mean_std"]["CLIP"]["MRR"], "MRR"),
              ("TN1", "Chỉ đỉnh đối tượng (G-obj)", RES1["mean_std"]["G-obj"]["MRR"], "MRR"),
              ("TN1", "Bộ ba + cặp + đối tượng (G-full)", RES1["mean_std"]["G-full"]["MRR"], "MRR"),
              ("TN1", "G-full, vị từ hoán vị (G-shuf)", RES1["mean_std"]["G-shuf"]["MRR"], "MRR"),
              ("TN1", "G-full, đồ thị ảnh khác (G-rand)", RES1["mean_std"]["G-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE: ảnh – lớp – bộ ba", RES1G["mean_std"]["SAGE"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, cạnh ngẫu nhiên", RES1G["mean_std"]["SAGE-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, bỏ cạnh", RES1G["mean_std"]["SAGE-0"]["MRR"], "MRR"),
              ("TN1c", "SCENIR: scene graph lớp", RES1S["mean_std"]["SCENIR"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, cạnh ngẫu nhiên", RES1S["mean_std"]["SCENIR-rand"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, bỏ cạnh", RES1S["mean_std"]["SCENIR-noedge"]["MRR"], "MRR")]
for g, res, lab in (("ARO_geo", RES2, "ARO trái/phải"), ("ARO_pred", RES2, "ARO vị từ"), ("COCOSP_all", RES2B, "COCO-spatial")):
    v = res["mean_std"][g]
    EDGE_TABLE += [(lab, "Cạnh vị từ / hình học (RelTR)", v["graph_only[dg]"], "acc"), (lab, "Đồ thị của mục khác", v["graph_only[dg_rand]"], "acc")]
    for m, d in (("graph_only[dg_nogeo]", "Bỏ vị trí (cạnh hình học = hằng)"), ("graph_only[dg_shuf]", "Vị từ hoán vị")):
        if m in v: EDGE_TABLE.append((lab, d, v[m], "acc"))
_md = ["| Thí nghiệm | Cách tạo cạnh / biến thể | Giá trị (trung bình ± std, 3 seed) |", "|---|---|---|"]
_md += [f"| {e} | {d} | {v['mean']:.4f} ± {v['std']:.4f} |" if u == "MRR" else f"| {e} | {d} | {100 * v['mean']:.1f}% ± {100 * v['std']:.1f} |" for e, d, v, u in EDGE_TABLE]
EDGE_MD = "\n".join(_md)
_idisp(_IMd("**So sánh các cách tạo cạnh** (TN1: MRR trên P-CHECK; TN2/2b: độ chính xác 'chỉ đồ thị' trên CHECK)\n\n" + EDGE_MD))

# ---- 11a.3 Xuất file split cố định (sản phẩm nộp: "tập chia dữ liệu cố định hoặc mã tạo split") ----
SPLIT_DIR = f"{OUT}/splits"; os.makedirs(SPLIT_DIR, exist_ok=True)
with open(f"{SPLIT_DIR}/tn1_vg_coco_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["vg_image_id", "coco_id", "split", "role"])
    for i in ELIG1: w.writerow([i, COCO_OF_VG[i], p5_split(i), "eval (P-CHECK)" if p5_split(i) == "P-CHECK" else "tune (P-TUNE) + GNN train"])
    for i in TRAIN_IMGS:
        if i not in set(ELIG1): w.writerow([i, "", "GNN-train-only", "GraphSAGE / SCENIR train"])
with open(f"{SPLIT_DIR}/tn2_tn2b_items_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["bench", "item", "image_key", "split_seed0", "split_seed1", "split_seed2"])
    for it in ITEMS2 + ITEMS2B: w.writerow([it["bench"], it["item"], it["key"]] + [v6_split(it["key"], s) for s in SEEDS])
json.dump(dict(rules={"VAL (v3)": "sha1(id) theo v3, chỉ số lọc ≥ 83000; heldout KHÔNG dùng",
                      "DEV-G0": "sha1('p3-dir2:'+id) mod 1000 < 350, trong pool G0 (VAL trừ benchmark v3, cache v4, 99 ảnh Phase 5)",
                      "TN1 P-TUNE/P-CHECK": "sha1('p5-pilot:'+id) mod 2", "TN2/2b TUNE/CHECK": "sha1(f'v6-{seed}:{image_key}') mod 2, seed ∈ {0,1,2}",
                      "lọc rò rỉ benchmark": "ảnh VG phải thuộc VAL; ảnh COCO phải không thuộc VG hoặc thuộc VG-VAL"},
               md5=dict(VAL=VAL_MD5, G0=G0_MD5), counts=dict(tn1=len(ELIG1), gnn_train=len(TRAIN_IMGS), tn2=len(ITEMS2), tn2b=len(ITEMS2B))),
          open(f"{SPLIT_DIR}/SPLIT_RULES.json", "w"), indent=1, ensure_ascii=False)
p3_log(f"Đã xuất file split vào {SPLIT_DIR}")

# ---- 11a.4 Hình chính (hiển thị trong notebook) ----
for p in [f"{FIG}/demo_1.png"] + sorted(glob.glob(f"{FIG}/case_*.png"))[:2] + sorted(glob.glob(f"{FIG}/xai_ex*.png"))[:2]:
    show_png(p, width=800)


## 11. Tổng hợp và đối chiếu yêu cầu đề cương

In [ ]:
# ============================== 11. TỔNG HỢP: bảng kết quả · đối chiếu tái lập với các lần chạy trước · lưu đầu ra ==============================
def pct(x): return f"{100 * x:.1f}"


lines = ["# Kết quả pipeline Project 3", "", f"Chế độ: {'SMOKE (dữ liệu giả — KHÔNG phải kết quả)' if SMOKE else 'REAL'} · {MODEL_INFO} · {elapsed()}", "",
         "## TN1 — Văn bản → ảnh, full-gallery (P-CHECK, trung bình ± std qua 3 seed caption)", "",
         "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1['decision']}**", "",
          "## TN1b — GraphSAGE (GNN đã học) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed)", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1G["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1G['decision']}** (SAGE-rand = cạnh ngẫu nhiên, SAGE-0 = bỏ cạnh)", "",
          f"## TN1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed; đặc trưng đỉnh: {RES1S['node_features']})", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1S["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1S['decision']}**", "",
          "## TN2 — Benchmark kết hợp (độ chính xác CHECK %, trung bình ± std qua 3 seed chia)", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
lines += ["", f"Quyết định đăng ký trước: **{RES2['decision']}**", "",
          f"## TN2b — Cấp 4 (a): cạnh hình học trên COCO-spatial ({RES2B['n_items']} mục sau lọc, {RES2B['n_scorable']} mục đồ thị chấm được; loại {RES2B['excluded']})", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2B["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
_cf = RES2B.get("counterfactual")
lines += ["", f"Quyết định đăng ký trước: **{RES2B['decision']}**" + (f" · Lật ảnh: đồ thị đảo {pct(_cf['graph_reversed'])}%, CLIP đảo {pct(_cf['clip_reversed'])}% (n={_cf['n']})" if _cf else ""), "",
          "## XAI (ARO, CHECK seed 0)", "",
          "| Nhóm | n | Lật ảnh: đồ thị đảo / CLIP đảo | Xoá bằng chứng: đổi / ngẫu nhiên | Validity cả hai / ít nhất một | Sufficiency | Sparsity: kích thước / độ tập trung |",
          "|---|---|---|---|---|---|---|"]
for k, o in RES3.items():
    cf = f"{pct(o['counterfactual_graph_reversed'])} / {pct(o['counterfactual_clip_reversed'])}" if "counterfactual_graph_reversed" in o else "—"
    lines.append(f"| {k} | {o['n']} | {cf} | {pct(o.get('fidelity_del_changed', 0))} / {pct(o.get('fidelity_rand_changed', 0))} | "
                 f"{pct(o['validity_both_iou50'] or 0)} / {pct(o['validity_any_iou50'] or 0)} | {pct(o.get('sufficiency_kept', 0))} (theo cấu trúc) | "
                 f"{pct(o.get('sparsity_evidence_fraction', 0))}% (thiết kế) / {o.get('sparsity_concentration_mean', 0):.2f} (trung vị {o.get('sparsity_concentration_median', 0):.2f}) |")

# ---- đối chiếu tái lập với số đã chạy riêng (v5, v6, v6-XAI) ----
REPRO = []
if not SMOKE:
    for k, ref in REF["exp1_MRR"].items():
        REPRO.append(("TN1 MRR " + k, ref, RES1["mean_std"][k]["MRR"]["mean"]))
    for g, d in REF["exp2_acc"].items():
        for m, ref in d.items(): REPRO.append((f"TN2 {g} {m}", ref, RES2["mean_std"][g][m]["mean"]))
    for k, d in REF["xai"].items():
        for m, ref in d.items(): REPRO.append((f"XAI {k} {m}", ref, RES3[k][m]))
    lines += ["", "## Đối chiếu tái lập với các lần chạy riêng trước đó", "", "| Chỉ số | Lần trước | Pipeline này | Chênh |", "|---|---|---|---|"]
    lines += [f"| {n} | {a:.4f} | {b:.4f} | {b - a:+.4f} |" for n, a, b in REPRO]
lines += ["", "## So sánh các cách tạo cạnh", "", EDGE_MD]

# ---- Cấp 4 (b): video MSVD (notebook video v5 chạy nguyên vẹn ở mục 4v) ----
lines += ["", f"## Cấp 4 (b) — Video MSVD (notebook Level4B v5, trạng thái: {RES_VIDEO['status']}, {RES_VIDEO.get('minutes')} phút)", ""]
if RES_VIDEO["status"] == "ok":
    lines += [f"Lần nhìn test MSVD: **{RES_VIDEO.get('test_looks_msvd')}** — số test chỉ là thăm dò, không phải xác nhận độc lập.", "",
              "| Mô hình | Tập | R@1 | R@5 | R@10 | MRR | n truy vấn |", "|---|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r.get('set')} | {pct(r['R@1'])} | {pct(r['R@5'])} | {pct(r['R@10'])} | {r['MRR']:.4f} | {r.get('n_queries')} |" for r in RES_VIDEO["test"]]
    if RES_VIDEO["comparisons"]:
        lines += ["", "| So sánh | a vs b | ΔMRR [CI 95%] | p (Holm) |", "|---|---|---|---|"]
        lines += [f"| {c.get('comparison')} | {c.get('a')} vs {c.get('b')} | {c.get('diff', 0):+.4f} [{c.get('ci_lo', 0):+.4f}, {c.get('ci_hi', 0):+.4f}] | {c.get('p_holm')} |"
                  for c in RES_VIDEO["comparisons"]]
    if RES_VIDEO.get("fidelity"): lines += ["", f"Fidelity (video): {RES_VIDEO['fidelity']}"]
else:
    lines += [f"Nhánh video không hoàn tất: {RES_VIDEO.get('error', '')[:500]}"]

# ---- bảng đối chiếu yêu cầu đề cương (sinh tự động từ kết quả của chính lần chạy này) ----
_d = lambda dec: "DƯƠNG" if (dec == "GO" or (isinstance(dec, str) and dec.startswith("DƯƠNG"))) else "không dương"
CHECKLIST = [
    ("Cấp 1", "Baseline CLIP đóng băng trên ≥ 1 tập công khai", "✅", f"CLIP {CLIP_MODEL}; TN1 MRR {RES1['mean_std']['CLIP']['MRR']['mean']:.4f}"),
    ("Cấp 1", "Đồ thị từ dữ liệu có sẵn (scene graph VG)", "✅", "Scene graph GT (TN1), scene graph RelTR (TN2, 2b)"),
    ("Cấp 1", "≥ 1 thuật toán đã học (GraphSAGE)", "✅", f"TN1b: SAGE {RES1G['mean_std']['SAGE']['MRR']['mean']:.4f} vs CLIP {RES1G['mean_std']['CLIP']['MRR']['mean']:.4f} ({_d(RES1G['decision']['G1 SAGE-CLIP'])})"),
    ("Cấp 1", "Xếp hạng lại α·cos + (1−α)·graph, so CLIP vs CLIP+Graph cùng split", "✅", f"TN1 {RES1['decision']}; ARO {RES2['decision'].get('ARO_all')}"),
    ("Cấp 1", "Ablation bỏ cạnh / cạnh ngẫu nhiên", "✅", f"ARO: mục khác {pct(RES2['mean_std']['ARO_all']['graph_only[dg_rand]']['mean'])}%, bỏ vị trí {pct(RES2['mean_std']['ARO_all']['graph_only[dg_nogeo]']['mean'])}%"),
    ("Cấp 1", "Hiển thị quan hệ/đường đi giải thích", "✅", "Demo search() + hình XAI"),
    ("Cấp 1", "5 truy vấn thành công + 5 thất bại", "✅", f"{len(CASES1)} ca văn bản → ảnh (case_*.png) + 10 ca ARO (xai_ex*.png); phân tích: CASE_ANALYSIS_LEVEL1_VI.md"),
    ("Cấp 1", "R@1/5/10, MRR, ≥ 3 seed, α chọn trên validation", "✅", "Mọi thí nghiệm: 3 seed, α trên TUNE"),
    ("Cấp 2", "≥ 2 tập công khai", "✅", "Visual Genome + COCO, ARO, SugarCrepe, COCO-spatial"),
    ("Cấp 2", "1 phương pháp mới từ 2024, cùng pipeline/split", "✅", f"SCENIR (ICML 2025) TN1c: {RES1S['mean_std']['SCENIR']['MRR']['mean']:.4f} vs CLIP ({_d(RES1S['decision']['S1 SCENIR-CLIP'])})"),
    ("Cấp 2", "Bộ ≥ 20 truy vấn có nhóm gián tiếp, báo cáo tách nhóm", "✅", "Ở phần A (eval_queries_20.json, tách direct/indirect) và MMKG (mmkg/)"),
    ("Cấp 2", "Ablation xáo trộn / đồ thị ngẫu nhiên; so sánh cách tạo cạnh; độ nhạy α", "✅", "Bảng so sánh cạnh + alpha_sensitivity.png"),
    ("Cấp 3", "Fidelity định lượng ≥ 10 truy vấn", "✅", f"Lật ảnh {pct(RES3['all'].get('counterfactual_graph_reversed', 0))}% vs CLIP {pct(RES3['all'].get('counterfactual_clip_reversed', 0))}%; xoá {pct(RES3['all']['fidelity_del_changed'])}% vs ngẫu nhiên {pct(RES3['all']['fidelity_rand_changed'])}% (n={RES3['all']['n']})"),
    ("Cấp 3", "Validity", "✅", f"Khớp GT IoU≥0.5: cả hai {pct(RES3['all']['validity_both_iou50'] or 0)}%, ít nhất một {pct(RES3['all']['validity_any_iou50'] or 0)}%"),
    ("Cấp 3", "Sparsity", "✅", f"Kích thước {pct(RES3['all']['sparsity_evidence_fraction'])}% (thiết kế); độ tập trung trung bình {RES3['all']['sparsity_concentration_mean']:.2f}"),
    ("Cấp 3", "MMKG tự xây + data card", "✅", "Ở mmkg/: Project3_MMKG_VN.ipynb + DATA_CARD.md"),
    ("Cấp 4 (a)", "Cải tiến của nhóm + giả thuyết + ablation + ≥ 2 tập công khai", "✅" if RES2B["decision"]["A1 fusion-CLIP"].startswith("DƯƠNG") else "🟡",
     f"Cạnh hình học: ARO trái/phải {pct(RES2['mean_std']['ARO_geo']['CLIP']['mean'])}→{pct(RES2['mean_std']['ARO_geo']['fusion[dg]']['mean'])}%; COCO-spatial {RES2B['decision']}"),
    ("Cấp 4 (b)", "Mở rộng video (text-to-video)", "✅" if RES_VIDEO["status"] == "ok" else "❌",
     (lambda tv: f"MSVD test: CLIP {tv['M0_clip']['MRR']:.4f} → đồ thị đầy đủ {tv['M4_full_graph']['MRR']:.4f} (lần nhìn test: {RES_VIDEO.get('test_looks_msvd')})"
      if RES_VIDEO["status"] == "ok" and "M0_clip" in tv and "M4_full_graph" in tv else f"trạng thái {RES_VIDEO['status']}")(
         {r["model"]: r for r in RES_VIDEO.get("test", []) if r.get("set") == "test_all"})),
    ("Mở rộng", "Chiều ảnh → văn bản (nhánh B)", "✅", "TN2, TN2b"),
    ("Nộp", "Mã tạo split / file split cố định", "✅", "splits/ (CSV + SPLIT_RULES.json)"),
    ("Nộp", "Notebook nhập truy vấn → top-k + giải thích", "✅", "Hàm search() (mục 10)")]
lines += ["", "## Đối chiếu yêu cầu đề cương", "", "| Cấp | Yêu cầu | Trạng thái | Bằng chứng (lần chạy này) |", "|---|---|---|---|"]
lines += [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in CHECKLIST]
REPORT = "\n".join(lines)
open(f"{OUT}/RESULTS.md", "w", encoding="utf-8").write(REPORT)
json.dump(dict(exp1=RES1, exp1_graphsage=RES1G, exp1_scenir=RES1S, exp2=RES2, exp2b_cocospatial=RES2B, xai=RES3, video=RES_VIDEO, demo=DEMO, checklist=CHECKLIST, models=MODEL_INFO, smoke=SMOKE, inputs_sha256=SHA, bench_sha256=BSHA,
               reproduction=[dict(metric=n, previous=a, now=b) for n, a, b in REPRO], runtime_s=round(time.time() - T0, 1)),
          open(f"{OUT}/summary.json", "w"), indent=1, default=str, ensure_ascii=False)
for name, rows in (("exp2_items.csv", ROWS2), ("exp2b_cocospatial_items.csv", ROWS2B), ("xai_items.csv", ROWS3)):
    with open(f"{OUT}/{name}", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r})); w.writeheader(); w.writerows(rows)
try:
    _idisp(_IMd(REPORT))
except Exception:
    print(REPORT)
p3_log("Xong toàn bộ pipeline:", elapsed())


In [ ]:
# ============ TỔNG KẾT: nơi lưu kết quả của cả hai phần ============
import os, glob
for _p in ["/kaggle/working/results", "/kaggle/working/output/action_genome/results", "/kaggle/working/p3_out", "/kaggle/working/p3_out/video_v2"]:
    _fs = sorted(glob.glob(os.path.join(_p, "*")))
    print(f"{_p}: {len(_fs)} tệp")
    for _f in _fs[:40]:
        print("   ", os.path.basename(_f))

print("Phần C (Level 4(b) v2-graphfix):", {k: v for k, v in RES_L4B_V2.items() if k != "error"} if "RES_L4B_V2" in globals() else "chưa chạy")
